In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

THESIS_DRIVE = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

print("Thesis Drive exists:", THESIS_DRIVE.exists())
print("Path:", THESIS_DRIVE)

Thesis Drive exists: True
Path: /content/drive/MyDrive/Thesis_Experiment


In [ ]:
# =========================================================
# PROJECT 6 — STEP 1
# RESTORE PATHS, VERIFY FROZEN PROJECTS, SELECT PROJECT 6
# =========================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display


# ---------------------------------------------------------
# 1. Restore permanent thesis paths
# ---------------------------------------------------------

THESIS_DRIVE = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

RAW_RESULTS_DRIVE = (
    THESIS_DRIVE / "Results" / "Raw"
)

AGGREGATED_RESULTS_DRIVE = (
    THESIS_DRIVE / "Results" / "Aggregated"
)

LOGS_DRIVE = (
    THESIS_DRIVE / "Results" / "Logs"
)

NOTES_DRIVE = (
    THESIS_DRIVE / "Notes"
)

RAW_DATA_DRIVE = (
    THESIS_DRIVE / "Data" / "Raw"
)

PROCESSED_DATA_DRIVE = (
    THESIS_DRIVE / "Data" / "processed"
)


SCREENING_PATH = (
    PROCESSED_DATA_DRIVE
    / "tcp_ci_project_screening.csv"
)

COMPLETION_REGISTRY_PATH = (
    NOTES_DRIVE
    / "completed_project_registry.csv"
)

ARCHIVE_PATH = (
    RAW_DATA_DRIVE
    / "TCP-CI-main-dataset.tar.gz"
)


required_paths = [
    THESIS_DRIVE,
    SCREENING_PATH,
    COMPLETION_REGISTRY_PATH,
    ARCHIVE_PATH,
]


missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]


if missing_paths:
    raise FileNotFoundError(
        "Required thesis paths are missing:\n"
        + "\n".join(missing_paths)
    )


# ---------------------------------------------------------
# 2. Load screening and completion records
# ---------------------------------------------------------

screening = pd.read_csv(
    SCREENING_PATH
)

completion_registry = pd.read_csv(
    COMPLETION_REGISTRY_PATH
)


required_screening_columns = [
    "Project",
    "EligibleForPilot",
]


for column in required_screening_columns:
    if column not in screening.columns:
        raise KeyError(
            f"Screening column missing: {column}"
        )


for column in [
    "Project",
    "Status",
]:
    if column not in completion_registry.columns:
        raise KeyError(
            f"Completion-registry column missing: {column}"
        )


# ---------------------------------------------------------
# 3. Interpret project eligibility
# ---------------------------------------------------------

def as_boolean(value):
    if pd.isna(value):
        return False

    if isinstance(
        value,
        (bool, np.bool_),
    ):
        return bool(value)

    text = (
        str(value)
        .strip()
        .lower()
    )

    if text in {
        "true",
        "1",
        "yes",
        "eligible",
        "included",
        "pass",
    }:
        return True

    if text in {
        "false",
        "0",
        "no",
        "ineligible",
        "excluded",
        "fail",
    }:
        return False

    raise ValueError(
        "Unrecognised EligibleForPilot value: "
        f"{value!r}"
    )


screening = screening.copy()

screening[
    "_Eligible"
] = screening[
    "EligibleForPilot"
].map(
    as_boolean
)


eligible_projects = (
    screening[
        screening[
            "_Eligible"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


if len(eligible_projects) != 24:
    raise AssertionError(
        "Expected 24 eligible projects, "
        f"observed {len(eligible_projects)}."
    )


# ---------------------------------------------------------
# 4. Verify completed Projects 1–5
# ---------------------------------------------------------

frozen_registry = (
    completion_registry[
        completion_registry[
            "Status"
        ]
        .astype(str)
        .str.strip()
        .str.upper()
        == "COMPLETE_AND_FROZEN"
    ]
    .copy()
)


frozen_projects = set(
    frozen_registry[
        "Project"
    ]
    .astype(str)
)


EXPECTED_FROZEN_PROJECTS = {
    "Angel-ML@angel",
    "apache@airavata",
    "b2ihealthcare@snow-owl",
    "eclipse@paho.mqtt.java",
    "thinkaurelius@titan",
}


missing_frozen_projects = (
    EXPECTED_FROZEN_PROJECTS
    - frozen_projects
)


if missing_frozen_projects:
    raise AssertionError(
        "Projects 1–5 are not all frozen:\n"
        + "\n".join(
            sorted(
                missing_frozen_projects
            )
        )
    )


TITAN_FREEZE_RECORD = (
    AGGREGATED_RESULTS_DRIVE
    / "thinkaurelius__titan"
    / "titan_30_seed_final"
    / "PROJECT_FROZEN.json"
)


if not TITAN_FREEZE_RECORD.exists():
    raise FileNotFoundError(
        "Titan freeze record is missing:\n"
        f"{TITAN_FREEZE_RECORD}"
    )


with open(
    TITAN_FREEZE_RECORD,
    "r",
    encoding="utf-8",
) as freeze_file:
    titan_freeze = json.load(
        freeze_file
    )


if (
    titan_freeze.get("Status")
    != "COMPLETE_AND_FROZEN"
):
    raise AssertionError(
        "Titan freeze status is invalid."
    )


if not titan_freeze.get(
    "DoNotRerun",
    False,
):
    raise AssertionError(
        "Titan is not marked DoNotRerun."
    )


# ---------------------------------------------------------
# 5. Select the next eligible unfinished project
# ---------------------------------------------------------

eligible_projects[
    "_AlreadyFrozen"
] = eligible_projects[
    "Project"
].astype(str).isin(
    frozen_projects
)


unfinished_projects = (
    eligible_projects[
        ~eligible_projects[
            "_AlreadyFrozen"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


if unfinished_projects.empty:
    raise AssertionError(
        "No eligible unfinished project remains."
    )


PROJECT_NUMBER = 6

PROJECT_NAME = str(
    unfinished_projects.iloc[0][
        "Project"
    ]
)

PROJECT_SLUG = (
    PROJECT_NAME
    .replace("@", "__")
    .replace("/", "__")
)


EXPECTED_PROJECT_NAME = (
    "eclipse@jetty.project"
)

EXPECTED_PROJECT_SLUG = (
    "eclipse__jetty.project"
)


if PROJECT_NAME != EXPECTED_PROJECT_NAME:
    display(
        unfinished_projects[
            [
                "Project",
                "EligibleForPilot",
            ]
        ].head(10)
    )

    raise AssertionError(
        "Unexpected Project 6 selection.\n"
        f"Expected: {EXPECTED_PROJECT_NAME}\n"
        f"Observed: {PROJECT_NAME}"
    )


if PROJECT_SLUG != EXPECTED_PROJECT_SLUG:
    raise AssertionError(
        "Project 6 slug generation failed."
    )


# ---------------------------------------------------------
# 6. Create Project 6 permanent output directories
# ---------------------------------------------------------

PROJECT_RAW_RESULTS = (
    RAW_RESULTS_DRIVE
    / PROJECT_SLUG
)

PROJECT_AGGREGATED_RESULTS = (
    AGGREGATED_RESULTS_DRIVE
    / PROJECT_SLUG
)

PROJECT_LOGS = (
    LOGS_DRIVE
    / PROJECT_SLUG
)

PROJECT_PREFLIGHT_DIRECTORY = (
    PROJECT_AGGREGATED_RESULTS
    / "jetty_preflight"
)


for directory in [
    PROJECT_RAW_RESULTS,
    PROJECT_AGGREGATED_RESULTS,
    PROJECT_LOGS,
    PROJECT_PREFLIGHT_DIRECTORY,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


PROJECT_6_SELECTION_CHECKPOINT = (
    NOTES_DRIVE
    / "project_06_selection_checkpoint.json"
)


selection_checkpoint = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "PROJECT_SELECTED_PENDING_RUNTIME_EXTRACTION",

    "EligibleForPilot":
        True,

    "Projects1To5Frozen":
        True,

    "Project5FreezeValidated":
        True,

    "DatasetArchive":
        str(
            ARCHIVE_PATH
        ),

    "ProjectRawResults":
        str(
            PROJECT_RAW_RESULTS
        ),

    "ProjectAggregatedResults":
        str(
            PROJECT_AGGREGATED_RESULTS
        ),

    "ProjectLogs":
        str(
            PROJECT_LOGS
        ),

    "ProjectPreflightDirectory":
        str(
            PROJECT_PREFLIGHT_DIRECTORY
        ),

    "UpdatedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
}


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        selection_checkpoint,
        checkpoint_file,
        indent=2,
    )


# ---------------------------------------------------------
# 7. Final output
# ---------------------------------------------------------

print(
    "=== PROJECT 6 STEP 1 RESULT ==="
)

print(
    "Notebook:",
    "Thesis_project_from6"
)

print(
    "Thesis Drive:",
    THESIS_DRIVE
)

print(
    "Eligible projects:",
    len(eligible_projects)
)

print(
    "Frozen Projects 1–5:",
    True
)

print(
    "Project 5 freeze validated:",
    True
)

print(
    "Project number:",
    PROJECT_NUMBER
)

print(
    "Project:",
    PROJECT_NAME
)

print(
    "Project slug:",
    PROJECT_SLUG
)

print(
    "Dataset archive:",
    ARCHIVE_PATH
)

print(
    "Selection checkpoint:",
    PROJECT_6_SELECTION_CHECKPOINT
)

print(
    "\nSUCCESS: Project 6 was selected correctly."
)

print(
    "SUCCESS: Projects 1–5 remain protected."
)

print(
    "SUCCESS: Ready for runtime dataset extraction."
)

=== PROJECT 6 STEP 1 RESULT ===
Notebook: Thesis_project_from6
Thesis Drive: /content/drive/MyDrive/Thesis_Experiment
Eligible projects: 24
Frozen Projects 1–5: True
Project 5 freeze validated: True
Project number: 6
Project: eclipse@jetty.project
Project slug: eclipse__jetty.project
Dataset archive: /content/drive/MyDrive/Thesis_Experiment/Data/Raw/TCP-CI-main-dataset.tar.gz
Selection checkpoint: /content/drive/MyDrive/Thesis_Experiment/Notes/project_06_selection_checkpoint.json

SUCCESS: Project 6 was selected correctly.
SUCCESS: Projects 1–5 remain protected.
SUCCESS: Ready for runtime dataset extraction.


In [ ]:
# =========================================================
# PROJECT 6 — STEP 2
# EXTRACT TCP-CI DATASET AND VERIFY JETTY SOURCE FILES
# =========================================================

from pathlib import Path
import json
import shutil
import tarfile

import pandas as pd
from IPython.display import display


# ---------------------------------------------------------
# 1. Confirm Step 1 variables
# ---------------------------------------------------------

required_variables = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",
    "ARCHIVE_PATH",
    "NOTES_DRIVE",
    "PROJECT_RAW_RESULTS",
    "PROJECT_AGGREGATED_RESULTS",
    "PROJECT_LOGS",
    "PROJECT_PREFLIGHT_DIRECTORY",
]


missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]


if missing_variables:
    raise RuntimeError(
        "Step 1 variables are missing:\n"
        + "\n".join(missing_variables)
        + "\n\nRerun Project 6 Step 1 first."
    )


if PROJECT_NUMBER != 6:
    raise AssertionError(
        f"Expected Project 6, observed {PROJECT_NUMBER}."
    )


if PROJECT_NAME != "eclipse@jetty.project":
    raise AssertionError(
        "Unexpected Project 6 project:\n"
        f"{PROJECT_NAME}"
    )


if not ARCHIVE_PATH.exists():
    raise FileNotFoundError(
        "TCP-CI archive is missing:\n"
        f"{ARCHIVE_PATH}"
    )


print(
    "=== PROJECT 6 STEP 2: DATASET EXTRACTION ==="
)


# ---------------------------------------------------------
# 2. Fresh runtime extraction paths
# ---------------------------------------------------------

RUNTIME_ROOT = Path(
    "/content/working_data/tcp_ci_full"
)


def locate_datasets_root(root):
    """
    Locate the extracted TCP-CI datasets directory.
    """

    direct_candidate = (
        root
        / "datasets"
    )


    if direct_candidate.exists():
        return direct_candidate


    if not root.exists():
        return None


    candidates = []


    for candidate in root.rglob(
        "datasets"
    ):
        if not candidate.is_dir():
            continue


        project_count = len([
            child
            for child in candidate.iterdir()
            if child.is_dir()
        ])


        if project_count >= 20:
            candidates.append(
                (
                    project_count,
                    candidate,
                )
            )


    if not candidates:
        return None


    candidates.sort(
        key=lambda item: item[0],
        reverse=True,
    )


    return candidates[0][1]


DATASETS_ROOT = locate_datasets_root(
    RUNTIME_ROOT
)


# ---------------------------------------------------------
# 3. Extract only when runtime copy is absent
# ---------------------------------------------------------

if DATASETS_ROOT is None:
    if RUNTIME_ROOT.exists():
        print(
            "Removing incomplete runtime extraction:"
        )

        print(
            RUNTIME_ROOT
        )

        shutil.rmtree(
            RUNTIME_ROOT
        )


    RUNTIME_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )


    print(
        "Extracting archive..."
    )

    print(
        "Source:",
        ARCHIVE_PATH
    )

    print(
        "Destination:",
        RUNTIME_ROOT
    )


    with tarfile.open(
        ARCHIVE_PATH,
        mode="r:gz",
    ) as archive:
        try:
            archive.extractall(
                path=RUNTIME_ROOT,
                filter="data",
            )

        except TypeError:
            # Compatibility fallback for older tarfile APIs.
            archive.extractall(
                path=RUNTIME_ROOT
            )


    DATASETS_ROOT = locate_datasets_root(
        RUNTIME_ROOT
    )

else:
    print(
        "Existing runtime extraction found."
    )

    print(
        "Reusing:",
        DATASETS_ROOT
    )


if DATASETS_ROOT is None:
    raise FileNotFoundError(
        "Archive extraction completed, but the "
        "datasets directory could not be located."
    )


# ---------------------------------------------------------
# 4. Validate complete dataset extraction
# ---------------------------------------------------------

runtime_project_directories = sorted([
    child
    for child in DATASETS_ROOT.iterdir()
    if child.is_dir()
])


if len(runtime_project_directories) != 25:
    raise AssertionError(
        "Unexpected extracted project count.\n"
        f"Expected: 25\n"
        f"Observed: {len(runtime_project_directories)}"
    )


runtime_project_names = [
    directory.name
    for directory in runtime_project_directories
]


if PROJECT_NAME not in runtime_project_names:
    raise FileNotFoundError(
        "Jetty project directory was not found "
        "after extraction."
    )


# ---------------------------------------------------------
# 5. Set canonical Project 6 data directory
# ---------------------------------------------------------

PROJECT_DATA_DIRECTORY = (
    DATASETS_ROOT
    / PROJECT_NAME
)

PROJECT_DATA_DIR = (
    PROJECT_DATA_DIRECTORY
)


if not PROJECT_DATA_DIRECTORY.exists():
    raise FileNotFoundError(
        "Project 6 data directory is missing:\n"
        f"{PROJECT_DATA_DIRECTORY}"
    )


# ---------------------------------------------------------
# 6. Verify required Project 6 source files
# ---------------------------------------------------------

REQUIRED_PROJECT_FILES = [
    "builds.csv",
    "exe.csv",
    "dataset.csv",
    "entity_change_history.csv",
    "id_map.csv",
    "contributors.csv",
]


source_file_records = []


for filename in REQUIRED_PROJECT_FILES:
    file_path = (
        PROJECT_DATA_DIRECTORY
        / filename
    )


    source_file_records.append({
        "File":
            filename,

        "Exists":
            file_path.exists(),

        "SizeBytes":
            (
                int(
                    file_path.stat().st_size
                )
                if file_path.exists()
                else 0
            ),

        "Path":
            str(file_path),
    })


source_file_audit = pd.DataFrame(
    source_file_records
)


if not source_file_audit[
    "Exists"
].all():
    display(
        source_file_audit
    )

    raise FileNotFoundError(
        "Jetty is missing one or more required files."
    )


if (
    source_file_audit[
        "SizeBytes"
    ] <= 0
).any():
    display(
        source_file_audit
    )

    raise AssertionError(
        "Jetty contains an empty required file."
    )


# ---------------------------------------------------------
# 7. Save extraction checkpoint
# ---------------------------------------------------------

PROJECT_6_SELECTION_CHECKPOINT = (
    NOTES_DRIVE
    / "project_06_selection_checkpoint.json"
)


if PROJECT_6_SELECTION_CHECKPOINT.exists():
    with open(
        PROJECT_6_SELECTION_CHECKPOINT,
        "r",
        encoding="utf-8",
    ) as checkpoint_file:
        checkpoint = json.load(
            checkpoint_file
        )

else:
    checkpoint = {
        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,
    }


checkpoint.update({
    "Status":
        "RUNTIME_DATASET_EXTRACTED_AND_VERIFIED",

    "RuntimeRoot":
        str(
            RUNTIME_ROOT
        ),

    "RuntimeDatasetsRoot":
        str(
            DATASETS_ROOT
        ),

    "RuntimeProjectCount":
        int(
            len(runtime_project_directories)
        ),

    "ProjectDataDirectory":
        str(
            PROJECT_DATA_DIRECTORY
        ),

    "RequiredSourceFiles":
        REQUIRED_PROJECT_FILES,

    "SourceFileAuditPassed":
        True,

    "UpdatedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
})


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        checkpoint,
        checkpoint_file,
        indent=2,
    )


# ---------------------------------------------------------
# 8. Final output
# ---------------------------------------------------------

print(
    "\n=== PROJECT 6 STEP 2 RESULT ==="
)

print(
    "Runtime root:",
    RUNTIME_ROOT
)

print(
    "Datasets root:",
    DATASETS_ROOT
)

print(
    "Extracted project directories:",
    len(runtime_project_directories)
)

print(
    "Project:",
    PROJECT_NAME
)

print(
    "Project data directory:",
    PROJECT_DATA_DIRECTORY
)


print(
    "\nJetty source-file audit:"
)

display(
    source_file_audit[
        [
            "File",
            "Exists",
            "SizeBytes",
        ]
    ]
)


print(
    "\nCheckpoint:"
)

print(
    PROJECT_6_SELECTION_CHECKPOINT
)


print(
    "\nSUCCESS: The TCP-CI archive is available "
    "in fresh runtime storage."
)

print(
    "SUCCESS: All 25 project directories were found."
)

print(
    "SUCCESS: All six Jetty source files are present "
    "and non-empty."
)

print(
    "SUCCESS: Project 6 is ready for schema inspection "
    "and chronological-split preflight."
)

=== PROJECT 6 STEP 2: DATASET EXTRACTION ===
Extracting archive...
Source: /content/drive/MyDrive/Thesis_Experiment/Data/Raw/TCP-CI-main-dataset.tar.gz
Destination: /content/working_data/tcp_ci_full

=== PROJECT 6 STEP 2 RESULT ===
Runtime root: /content/working_data/tcp_ci_full
Datasets root: /content/working_data/tcp_ci_full/datasets
Extracted project directories: 25
Project: eclipse@jetty.project
Project data directory: /content/working_data/tcp_ci_full/datasets/eclipse@jetty.project

Jetty source-file audit:


,File,Exists,SizeBytes
0,builds.csv,True,15115
1,exe.csv,True,698003
2,dataset.csv,True,14529402
3,entity_change_history.csv,True,47868594
4,id_map.csv,True,1837940
5,contributors.csv,True,12042



Checkpoint:
/content/drive/MyDrive/Thesis_Experiment/Notes/project_06_selection_checkpoint.json

SUCCESS: The TCP-CI archive is available in fresh runtime storage.
SUCCESS: All 25 project directories were found.
SUCCESS: All six Jetty source files are present and non-empty.
SUCCESS: Project 6 is ready for schema inspection and chronological-split preflight.


In [ ]:
# =========================================================
# PROJECT 6 — STEP 3
# SCHEMA INSPECTION AND CHRONOLOGICAL-SPLIT PREFLIGHT
# =========================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display


# ---------------------------------------------------------
# 1. Confirm Project 6 runtime state
# ---------------------------------------------------------

required_variables = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",
    "PROJECT_DATA_DIRECTORY",
    "PROJECT_PREFLIGHT_DIRECTORY",
    "NOTES_DRIVE",
    "SCREENING_PATH",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Required Step 1–2 variables are missing:\n"
        + "\n".join(missing_variables)
    )

if PROJECT_NUMBER != 6:
    raise AssertionError(
        f"Expected Project 6, observed {PROJECT_NUMBER}."
    )

if PROJECT_NAME != "eclipse@jetty.project":
    raise AssertionError(
        f"Unexpected project: {PROJECT_NAME}"
    )

print(
    "=== PROJECT 6 STEP 3: SCHEMA AND SPLIT PREFLIGHT ==="
)


# ---------------------------------------------------------
# 2. Load all six Jetty files
# ---------------------------------------------------------

builds_raw = pd.read_csv(
    PROJECT_DATA_DIRECTORY / "builds.csv",
    low_memory=False,
)

exe_raw = pd.read_csv(
    PROJECT_DATA_DIRECTORY / "exe.csv",
    low_memory=False,
)

dataset_raw = pd.read_csv(
    PROJECT_DATA_DIRECTORY / "dataset.csv",
    low_memory=False,
)

entity_change_history_raw = pd.read_csv(
    PROJECT_DATA_DIRECTORY / "entity_change_history.csv",
    low_memory=False,
)

id_map_raw = pd.read_csv(
    PROJECT_DATA_DIRECTORY / "id_map.csv",
    low_memory=False,
)

contributors_raw = pd.read_csv(
    PROJECT_DATA_DIRECTORY / "contributors.csv",
    low_memory=False,
)


loaded_tables = {
    "builds.csv":
        builds_raw,

    "exe.csv":
        exe_raw,

    "dataset.csv":
        dataset_raw,

    "entity_change_history.csv":
        entity_change_history_raw,

    "id_map.csv":
        id_map_raw,

    "contributors.csv":
        contributors_raw,
}


schema_summary = pd.DataFrame([
    {
        "File":
            filename,

        "Rows":
            int(len(dataframe)),

        "Columns":
            int(len(dataframe.columns)),
    }
    for filename, dataframe in loaded_tables.items()
])


# ---------------------------------------------------------
# 3. Robust column resolver
# ---------------------------------------------------------

def normalise_column_name(value):
    return "".join(
        character.lower()
        for character in str(value)
        if character.isalnum()
    )


def resolve_column(
    dataframe,
    aliases,
    table_name,
    required=True,
):
    normalised_lookup = {
        normalise_column_name(column):
            column
        for column in dataframe.columns
    }

    for alias in aliases:
        key = normalise_column_name(alias)

        if key in normalised_lookup:
            return normalised_lookup[key]

    if required:
        raise KeyError(
            f"Could not resolve a column in {table_name}.\n"
            f"Aliases: {aliases}\n"
            f"Columns: {list(dataframe.columns)}"
        )

    return None


# builds.csv
BUILD_ID_COLUMN = resolve_column(
    builds_raw,
    [
        "id",
        "build",
        "Build",
        "build_id",
        "BuildID",
    ],
    "builds.csv",
)

BUILD_TIMESTAMP_COLUMN = resolve_column(
    builds_raw,
    [
        "started_at",
        "startedAt",
        "start_time",
        "created_at",
        "timestamp",
        "date",
    ],
    "builds.csv",
)

BUILD_COMMIT_COLUMN = resolve_column(
    builds_raw,
    [
        "commit",
        "sha",
        "revision",
        "commit_hash",
        "Commit",
    ],
    "builds.csv",
    required=False,
)


# exe.csv
EXECUTION_BUILD_COLUMN = resolve_column(
    exe_raw,
    [
        "build",
        "Build",
        "build_id",
        "BuildID",
    ],
    "exe.csv",
)

EXECUTION_TEST_COLUMN = resolve_column(
    exe_raw,
    [
        "test",
        "Test",
        "test_id",
        "TestID",
    ],
    "exe.csv",
)

EXECUTION_VERDICT_COLUMN = resolve_column(
    exe_raw,
    [
        "verdict",
        "Verdict",
        "outcome",
        "result",
    ],
    "exe.csv",
)

EXECUTION_DURATION_COLUMN = resolve_column(
    exe_raw,
    [
        "duration",
        "Duration",
        "execution_time",
        "time",
    ],
    "exe.csv",
)

EXECUTION_JOB_COLUMN = resolve_column(
    exe_raw,
    [
        "job",
        "Job",
        "job_id",
        "JobID",
    ],
    "exe.csv",
    required=False,
)


# dataset.csv
DATASET_BUILD_COLUMN = resolve_column(
    dataset_raw,
    [
        "Build",
        "build",
        "build_id",
        "BuildID",
    ],
    "dataset.csv",
)

DATASET_TEST_COLUMN = resolve_column(
    dataset_raw,
    [
        "Test",
        "test",
        "test_id",
        "TestID",
    ],
    "dataset.csv",
)

DATASET_VERDICT_COLUMN = resolve_column(
    dataset_raw,
    [
        "Verdict",
        "verdict",
        "outcome",
        "result",
    ],
    "dataset.csv",
)

DATASET_DURATION_COLUMN = resolve_column(
    dataset_raw,
    [
        "Duration",
        "duration",
        "execution_time",
        "time",
    ],
    "dataset.csv",
)


# ---------------------------------------------------------
# 4. Canonical builds table
# ---------------------------------------------------------

builds = pd.DataFrame({
    "build":
        pd.to_numeric(
            builds_raw[BUILD_ID_COLUMN],
            errors="raise",
        ).astype("int64"),

    "timestamp":
        pd.to_datetime(
            builds_raw[BUILD_TIMESTAMP_COLUMN],
            errors="coerce",
            utc=True,
        ),
})


if BUILD_COMMIT_COLUMN is not None:
    builds["commit"] = (
        builds_raw[BUILD_COMMIT_COLUMN]
        .astype(str)
        .str.strip()
    )

else:
    builds["commit"] = ""


missing_build_timestamps = int(
    builds["timestamp"].isna().sum()
)


if builds["build"].duplicated().any():
    duplicate_ids = (
        builds.loc[
            builds["build"].duplicated(
                keep=False
            ),
            "build",
        ]
        .tolist()
    )

    raise AssertionError(
        "builds.csv contains duplicate build IDs:\n"
        f"{duplicate_ids[:20]}"
    )


if missing_build_timestamps != 0:
    raise AssertionError(
        "Jetty contains missing build timestamps:\n"
        f"{missing_build_timestamps}"
    )


builds = (
    builds
    .sort_values(
        [
            "timestamp",
            "build",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


builds["build_order"] = np.arange(
    1,
    len(builds) + 1,
    dtype=np.int64,
)


training_build_count = int(
    np.floor(
        0.75 * len(builds)
    )
)


if training_build_count <= 0:
    raise AssertionError(
        "Training partition would be empty."
    )


if training_build_count >= len(builds):
    raise AssertionError(
        "Evaluation partition would be empty."
    )


builds["partition"] = np.where(
    builds["build_order"]
    <= training_build_count,
    "training",
    "evaluation",
)


train_build_ids = set(
    builds.loc[
        builds["partition"] == "training",
        "build",
    ].astype(int)
)


evaluation_build_ids = set(
    builds.loc[
        builds["partition"] == "evaluation",
        "build",
    ].astype(int)
)


build_order_lookup = dict(
    zip(
        builds["build"].astype(int),
        builds["build_order"].astype(int),
    )
)


build_partition_lookup = dict(
    zip(
        builds["build"].astype(int),
        builds["partition"].astype(str),
    )
)


# ---------------------------------------------------------
# 5. Canonical raw execution history
# ---------------------------------------------------------

execution_rename_map = {
    EXECUTION_BUILD_COLUMN:
        "build",

    EXECUTION_TEST_COLUMN:
        "test",

    EXECUTION_VERDICT_COLUMN:
        "verdict",

    EXECUTION_DURATION_COLUMN:
        "duration",
}


if EXECUTION_JOB_COLUMN is not None:
    execution_rename_map[
        EXECUTION_JOB_COLUMN
    ] = "job"


exe_for_rec = (
    exe_raw
    .rename(
        columns=execution_rename_map
    )
    .copy()
)


exe_for_rec["build"] = pd.to_numeric(
    exe_for_rec["build"],
    errors="raise",
).astype("int64")


exe_for_rec["test"] = (
    exe_for_rec["test"]
    .astype(str)
)


exe_for_rec["verdict"] = pd.to_numeric(
    exe_for_rec["verdict"],
    errors="raise",
).astype("int8")


exe_for_rec["duration"] = pd.to_numeric(
    exe_for_rec["duration"],
    errors="coerce",
).astype(float)


if "job" not in exe_for_rec.columns:
    exe_for_rec["job"] = ""


allowed_verdicts = {
    0,
    1,
    2,
    3,
}


observed_execution_verdicts = set(
    exe_for_rec["verdict"]
    .astype(int)
    .unique()
    .tolist()
)


if not observed_execution_verdicts.issubset(
    allowed_verdicts
):
    raise AssertionError(
        "Unexpected exe.csv verdicts:\n"
        f"{sorted(observed_execution_verdicts)}"
    )


build_id_set = set(
    builds["build"].astype(int)
)


unmatched_execution_build_ids = sorted(
    set(
        exe_for_rec["build"].astype(int)
    )
    - build_id_set
)


exe_for_rec["build_order"] = (
    exe_for_rec["build"]
    .map(build_order_lookup)
)


exe_for_rec["partition"] = (
    exe_for_rec["build"]
    .map(build_partition_lookup)
)


missing_execution_durations = int(
    exe_for_rec["duration"].isna().sum()
)


if unmatched_execution_build_ids:
    raise AssertionError(
        "exe.csv contains unmatched build IDs:\n"
        f"{unmatched_execution_build_ids[:20]}"
    )


if missing_execution_durations != 0:
    raise AssertionError(
        "exe.csv contains missing durations:\n"
        f"{missing_execution_durations}"
    )


if (
    exe_for_rec["duration"] < 0
).any():
    raise AssertionError(
        "exe.csv contains negative durations."
    )


# ---------------------------------------------------------
# 6. Canonical model-ready dataset
# ---------------------------------------------------------

dataset_rename_map = {
    DATASET_BUILD_COLUMN:
        "Build",

    DATASET_TEST_COLUMN:
        "Test",

    DATASET_VERDICT_COLUMN:
        "Verdict",

    DATASET_DURATION_COLUMN:
        "Duration",
}


dataset = (
    dataset_raw
    .rename(
        columns=dataset_rename_map
    )
    .copy()
)


dataset["Build"] = pd.to_numeric(
    dataset["Build"],
    errors="raise",
).astype("int64")


dataset["Test"] = (
    dataset["Test"]
    .astype(str)
)


dataset["Verdict"] = pd.to_numeric(
    dataset["Verdict"],
    errors="raise",
).astype("int8")


dataset["Duration"] = pd.to_numeric(
    dataset["Duration"],
    errors="coerce",
).astype(float)


observed_dataset_verdicts = set(
    dataset["Verdict"]
    .astype(int)
    .unique()
    .tolist()
)


if not observed_dataset_verdicts.issubset(
    allowed_verdicts
):
    raise AssertionError(
        "Unexpected dataset.csv verdicts:\n"
        f"{sorted(observed_dataset_verdicts)}"
    )


unmatched_dataset_build_ids = sorted(
    set(
        dataset["Build"].astype(int)
    )
    - build_id_set
)


duplicate_dataset_build_test_pairs = int(
    dataset.duplicated(
        subset=[
            "Build",
            "Test",
        ]
    ).sum()
)


if unmatched_dataset_build_ids:
    raise AssertionError(
        "dataset.csv contains unmatched build IDs:\n"
        f"{unmatched_dataset_build_ids[:20]}"
    )


if duplicate_dataset_build_test_pairs != 0:
    raise AssertionError(
        "dataset.csv contains duplicate Build-Test pairs:\n"
        f"{duplicate_dataset_build_test_pairs}"
    )


if dataset["Duration"].isna().any():
    raise AssertionError(
        "dataset.csv contains missing durations."
    )


dataset["build_order"] = (
    dataset["Build"]
    .map(build_order_lookup)
    .astype("int64")
)


dataset["partition"] = (
    dataset["Build"]
    .map(build_partition_lookup)
)


dataset_with_order = (
    dataset
    .sort_values(
        [
            "build_order",
            "Build",
            "Test",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


clean_training_data = (
    dataset_with_order[
        dataset_with_order[
            "partition"
        ] == "training"
    ]
    .copy()
    .reset_index(drop=True)
)


clean_evaluation_data = (
    dataset_with_order[
        dataset_with_order[
            "partition"
        ] == "evaluation"
    ]
    .copy()
    .reset_index(drop=True)
)


MODEL_FEATURE_COLUMNS = [
    column
    for column in dataset.columns
    if column not in {
        "Build",
        "Test",
        "Verdict",
        "Duration",
        "build_order",
        "partition",
    }
]


if len(MODEL_FEATURE_COLUMNS) != 150:
    raise AssertionError(
        "Unexpected model predictor count.\n"
        f"Expected: 150\n"
        f"Observed: {len(MODEL_FEATURE_COLUMNS)}"
    )


# ---------------------------------------------------------
# 7. Verify each model row maps to one raw execution
# ---------------------------------------------------------

requested_model_pairs = (
    dataset_with_order[
        [
            "Build",
            "Test",
        ]
    ]
    .drop_duplicates()
)


raw_model_pair_counts = (
    exe_for_rec[
        [
            "build",
            "test",
        ]
    ]
    .rename(
        columns={
            "build":
                "Build",

            "test":
                "Test",
        }
    )
    .merge(
        requested_model_pairs,
        on=[
            "Build",
            "Test",
        ],
        how="inner",
        validate="many_to_one",
    )
    .groupby(
        [
            "Build",
            "Test",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size":
                "RawExecutionRows"
        }
    )
)


model_pair_multiplicity = (
    requested_model_pairs
    .merge(
        raw_model_pair_counts,
        on=[
            "Build",
            "Test",
        ],
        how="left",
        validate="one_to_one",
    )
)


model_pair_multiplicity[
    "RawExecutionRows"
] = (
    model_pair_multiplicity[
        "RawExecutionRows"
    ]
    .fillna(0)
    .astype(int)
)


invalid_model_pair_multiplicity = (
    model_pair_multiplicity[
        model_pair_multiplicity[
            "RawExecutionRows"
        ] != 1
    ]
)


if len(invalid_model_pair_multiplicity) != 0:
    display(
        invalid_model_pair_multiplicity.head(20)
    )

    raise AssertionError(
        "Every model-ready Build-Test pair must map "
        "to exactly one raw execution row."
    )


# ---------------------------------------------------------
# 8. Calculate observed screening metrics
# ---------------------------------------------------------

training_execution_rows = (
    exe_for_rec[
        exe_for_rec[
            "partition"
        ] == "training"
    ]
)


evaluation_execution_rows = (
    exe_for_rec[
        exe_for_rec[
            "partition"
        ] == "evaluation"
    ]
)


observed_metrics = {
    "TotalBuilds":
        int(len(builds)),

    "TrainingPeriodBuilds":
        int(
            (
                builds["partition"]
                == "training"
            ).sum()
        ),

    "EvaluationPeriodBuilds":
        int(
            (
                builds["partition"]
                == "evaluation"
            ).sum()
        ),

    "RawExecutionRows":
        int(len(exe_for_rec)),

    "TrainingExecutionRows":
        int(len(training_execution_rows)),

    "EvaluationExecutionRows":
        int(len(evaluation_execution_rows)),

    "UniqueExecutionTests":
        int(
            exe_for_rec[
                "test"
            ].nunique()
        ),

    "FailingTrainingBuilds":
        int(
            training_execution_rows.loc[
                training_execution_rows[
                    "verdict"
                ] != 0,
                "build",
            ].nunique()
        ),

    "FailingEvaluationBuilds":
        int(
            evaluation_execution_rows.loc[
                evaluation_execution_rows[
                    "verdict"
                ] != 0,
                "build",
            ].nunique()
        ),

    "TrainingFailureExecutions":
        int(
            (
                training_execution_rows[
                    "verdict"
                ] != 0
            ).sum()
        ),

    "EvaluationFailureExecutions":
        int(
            (
                evaluation_execution_rows[
                    "verdict"
                ] != 0
            ).sum()
        ),

    "ModelTrainingRows":
        int(len(clean_training_data)),

    "ModelTrainingBuilds":
        int(
            clean_training_data[
                "Build"
            ].nunique()
        ),

    "ModelEvaluationRows":
        int(len(clean_evaluation_data)),

    "ModelEvaluationBuilds":
        int(
            clean_evaluation_data[
                "Build"
            ].nunique()
        ),

    "ModelTrainingPasses":
        int(
            (
                clean_training_data[
                    "Verdict"
                ] == 0
            ).sum()
        ),

    "ModelTrainingFailures":
        int(
            (
                clean_training_data[
                    "Verdict"
                ] != 0
            ).sum()
        ),

    "ModelEvaluationPasses":
        int(
            (
                clean_evaluation_data[
                    "Verdict"
                ] == 0
            ).sum()
        ),

    "ModelEvaluationFailures":
        int(
            (
                clean_evaluation_data[
                    "Verdict"
                ] != 0
            ).sum()
        ),

    "MissingBuildTimestamps":
        missing_build_timestamps,

    "UnmatchedExecutionBuilds":
        int(
            len(
                unmatched_execution_build_ids
            )
        ),

    "UnmatchedDatasetBuilds":
        int(
            len(
                unmatched_dataset_build_ids
            )
        ),

    "DuplicateDatasetBuildTestPairs":
        duplicate_dataset_build_test_pairs,

    "MissingExecutionDurations":
        missing_execution_durations,
}


# ---------------------------------------------------------
# 9. Compare against the frozen screening CSV
# ---------------------------------------------------------

screening = pd.read_csv(
    SCREENING_PATH
)


project_screening_rows = (
    screening[
        screening[
            "Project"
        ].astype(str)
        == PROJECT_NAME
    ]
)


if len(project_screening_rows) != 1:
    raise AssertionError(
        "Expected exactly one Jetty screening row."
    )


project_screening_row = (
    project_screening_rows.iloc[0]
)


comparison_records = []


for metric, observed_value in (
    observed_metrics.items()
):
    if metric not in screening.columns:
        continue

    expected_value = pd.to_numeric(
        pd.Series([
            project_screening_row[
                metric
            ]
        ]),
        errors="raise",
    ).iloc[0]

    comparison_records.append({
        "Metric":
            metric,

        "Expected":
            int(expected_value),

        "Observed":
            int(observed_value),

        "Matches":
            bool(
                int(expected_value)
                == int(observed_value)
            ),
    })


screening_comparison = pd.DataFrame(
    comparison_records
)


if not screening_comparison[
    "Matches"
].all():
    display(
        screening_comparison
    )

    raise AssertionError(
        "Jetty preflight counts do not match "
        "the frozen screening table."
    )


# Proposal eligibility requirement.
if observed_metrics[
    "FailingTrainingBuilds"
] < 10:
    raise AssertionError(
        "Jetty has fewer than 10 failing builds "
        "in the 75% training period."
    )


if observed_metrics[
    "FailingEvaluationBuilds"
] <= 0:
    raise AssertionError(
        "Jetty has no failing evaluation builds."
    )


if observed_metrics[
    "ModelTrainingFailures"
] <= 0:
    raise AssertionError(
        "Jetty model-training data has no failures."
    )


if observed_metrics[
    "ModelEvaluationFailures"
] <= 0:
    raise AssertionError(
        "Jetty model-evaluation data has no failures."
    )


# ---------------------------------------------------------
# 10. Save schema and preflight artefacts
# ---------------------------------------------------------

PREFLIGHT_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_initialisation_report.json"
)

SCHEMA_SUMMARY_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_schema_summary.csv"
)

SCREENING_COMPARISON_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_screening_comparison.csv"
)

BUILD_SPLIT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_chronological_build_split.csv"
)

COLUMN_SCHEMA_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_column_schema.json"
)


schema_summary.to_csv(
    SCHEMA_SUMMARY_PATH,
    index=False,
)


screening_comparison.to_csv(
    SCREENING_COMPARISON_PATH,
    index=False,
)


builds.to_csv(
    BUILD_SPLIT_PATH,
    index=False,
)


column_schema = {
    filename:
        list(dataframe.columns)
    for filename, dataframe in loaded_tables.items()
}


with open(
    COLUMN_SCHEMA_PATH,
    "w",
    encoding="utf-8",
) as schema_file:
    json.dump(
        column_schema,
        schema_file,
        indent=2,
    )


preflight_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "PASS",

    "SplitType":
        "chronological fixed 75/25",

    "TimestampTieBreak":
        "build ID ascending",

    "BuildIDColumn":
        BUILD_ID_COLUMN,

    "BuildTimestampColumn":
        BUILD_TIMESTAMP_COLUMN,

    "BuildCommitColumn":
        BUILD_COMMIT_COLUMN,

    "ExecutionBuildColumn":
        EXECUTION_BUILD_COLUMN,

    "ExecutionTestColumn":
        EXECUTION_TEST_COLUMN,

    "ExecutionVerdictColumn":
        EXECUTION_VERDICT_COLUMN,

    "ExecutionDurationColumn":
        EXECUTION_DURATION_COLUMN,

    "DatasetBuildColumn":
        DATASET_BUILD_COLUMN,

    "DatasetTestColumn":
        DATASET_TEST_COLUMN,

    "DatasetVerdictColumn":
        DATASET_VERDICT_COLUMN,

    "DatasetDurationColumn":
        DATASET_DURATION_COLUMN,

    "ModelPredictorCount":
        len(MODEL_FEATURE_COLUMNS),

    "ObservedMetrics":
        observed_metrics,

    "TrainingBoundary": {
        "LastTrainingBuildOrder":
            int(training_build_count),

        "LastTrainingBuild":
            int(
                builds.loc[
                    builds[
                        "partition"
                    ] == "training",
                    "build",
                ].iloc[-1]
            ),

        "LastTrainingTimestamp":
            builds.loc[
                builds[
                    "partition"
                ] == "training",
                "timestamp",
            ].iloc[-1].isoformat(),

        "FirstEvaluationBuildOrder":
            int(
                training_build_count + 1
            ),

        "FirstEvaluationBuild":
            int(
                builds.loc[
                    builds[
                        "partition"
                    ] == "evaluation",
                    "build",
                ].iloc[0]
            ),

        "FirstEvaluationTimestamp":
            builds.loc[
                builds[
                    "partition"
                ] == "evaluation",
                "timestamp",
            ].iloc[0].isoformat(),
    },

    "ScreeningComparisonMismatches":
        int(
            (
                ~screening_comparison[
                    "Matches"
                ]
            ).sum()
        ),

    "ModelPairsWithInvalidRawMultiplicity":
        int(
            len(
                invalid_model_pair_multiplicity
            )
        ),

    "CompletedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
}


with open(
    PREFLIGHT_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        preflight_report,
        report_file,
        indent=2,
    )


# ---------------------------------------------------------
# 11. Update Project 6 checkpoint
# ---------------------------------------------------------

PROJECT_6_SELECTION_CHECKPOINT = (
    NOTES_DRIVE
    / "project_06_selection_checkpoint.json"
)


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint = json.load(
        checkpoint_file
    )


checkpoint.update({
    "Status":
        "SCHEMA_AND_CHRONOLOGICAL_SPLIT_PREFLIGHT_PASSED",

    "InitialisationReport":
        str(
            PREFLIGHT_REPORT_PATH
        ),

    "SchemaSummary":
        str(
            SCHEMA_SUMMARY_PATH
        ),

    "ScreeningComparison":
        str(
            SCREENING_COMPARISON_PATH
        ),

    "ChronologicalBuildSplit":
        str(
            BUILD_SPLIT_PATH
        ),

    "ModelPredictorCount":
        int(
            len(
                MODEL_FEATURE_COLUMNS
            )
        ),

    "ObservedMetrics":
        observed_metrics,

    "UpdatedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
})


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 12. Compact final output
# ---------------------------------------------------------

print(
    "\n=== PROJECT 6 STEP 3 RESULT ==="
)


print(
    "\nLoaded source tables:"
)

display(
    schema_summary
)


print(
    "\nResolved columns:"
)

print(
    "Build ID:",
    BUILD_ID_COLUMN
)

print(
    "Build timestamp:",
    BUILD_TIMESTAMP_COLUMN
)

print(
    "Build commit:",
    BUILD_COMMIT_COLUMN
)

print(
    "Execution columns:",
    EXECUTION_BUILD_COLUMN,
    EXECUTION_TEST_COLUMN,
    EXECUTION_VERDICT_COLUMN,
    EXECUTION_DURATION_COLUMN,
)

print(
    "Dataset columns:",
    DATASET_BUILD_COLUMN,
    DATASET_TEST_COLUMN,
    DATASET_VERDICT_COLUMN,
    DATASET_DURATION_COLUMN,
)

print(
    "Model predictors:",
    len(MODEL_FEATURE_COLUMNS)
)


print(
    "\nChronological split:"
)

print(
    "Total builds:",
    observed_metrics["TotalBuilds"]
)

print(
    "Training-period builds:",
    observed_metrics["TrainingPeriodBuilds"]
)

print(
    "Evaluation-period builds:",
    observed_metrics["EvaluationPeriodBuilds"]
)

print(
    "Failing training builds:",
    observed_metrics["FailingTrainingBuilds"]
)

print(
    "Failing evaluation builds:",
    observed_metrics["FailingEvaluationBuilds"]
)

print(
    "Model-training rows:",
    observed_metrics["ModelTrainingRows"]
)

print(
    "Model-evaluation rows:",
    observed_metrics["ModelEvaluationRows"]
)

print(
    "Model-training failures:",
    observed_metrics["ModelTrainingFailures"]
)

print(
    "Model-evaluation failures:",
    observed_metrics["ModelEvaluationFailures"]
)


print(
    "\nScreening comparison:"
)

display(
    screening_comparison
)


print(
    "\nTraining/evaluation boundary:"
)

print(
    "Last training build:",
    preflight_report[
        "TrainingBoundary"
    ][
        "LastTrainingBuild"
    ],
    preflight_report[
        "TrainingBoundary"
    ][
        "LastTrainingTimestamp"
    ],
)

print(
    "First evaluation build:",
    preflight_report[
        "TrainingBoundary"
    ][
        "FirstEvaluationBuild"
    ],
    preflight_report[
        "TrainingBoundary"
    ][
        "FirstEvaluationTimestamp"
    ],
)


print(
    "\nPreflight report:"
)

print(
    PREFLIGHT_REPORT_PATH
)


print(
    "\nSUCCESS: All Jetty source schemas were loaded."
)

print(
    "SUCCESS: The chronological 75/25 split was created."
)

print(
    "SUCCESS: All observed counts match the frozen "
    "screening table."
)

print(
    "SUCCESS: Every model-ready row maps to exactly "
    "one raw execution."
)

print(
    "SUCCESS: Jetty satisfies the training and "
    "evaluation failure requirements."
)

print(
    "SUCCESS: Project 6 is ready for commit and "
    "changed-entity mapping."
)

=== PROJECT 6 STEP 3: SCHEMA AND SPLIT PREFLIGHT ===

=== PROJECT 6 STEP 3 RESULT ===

Loaded source tables:


,File,Rows,Columns
0,builds.csv,192,3
1,exe.csv,26439,5
2,dataset.csv,17291,154
3,entity_change_history.csv,553889,8
4,id_map.csv,19286,2
5,contributors.csv,188,4



Resolved columns:
Build ID: id
Build timestamp: started_at
Build commit: None
Execution columns: build test verdict duration
Dataset columns: Build Test Verdict Duration
Model predictors: 150

Chronological split:
Total builds: 192
Training-period builds: 144
Evaluation-period builds: 48
Failing training builds: 113
Failing evaluation builds: 38
Model-training rows: 13212
Model-evaluation rows: 4079
Model-training failures: 130
Model-evaluation failures: 40

Screening comparison:


,Metric,Expected,Observed,Matches
0,TotalBuilds,192,192,True
1,TrainingPeriodBuilds,144,144,True
2,EvaluationPeriodBuilds,48,48,True
3,RawExecutionRows,26439,26439,True
4,TrainingExecutionRows,19845,19845,True
5,EvaluationExecutionRows,6594,6594,True
6,UniqueExecutionTests,387,387,True
7,FailingTrainingBuilds,113,113,True
8,FailingEvaluationBuilds,38,38,True
9,TrainingFailureExecutions,131,131,True



Training/evaluation boundary:
Last training build: 6910580 2013-05-06T02:31:00+00:00
First evaluation build: 6911057 2013-05-06T03:01:00+00:00

Preflight report:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/eclipse__jetty.project/jetty_preflight/jetty_initialisation_report.json

SUCCESS: All Jetty source schemas were loaded.
SUCCESS: The chronological 75/25 split was created.
SUCCESS: All observed counts match the frozen screening table.
SUCCESS: Every model-ready row maps to exactly one raw execution.
SUCCESS: Jetty satisfies the training and evaluation failure requirements.
SUCCESS: Project 6 is ready for commit and changed-entity mapping.


In [ ]:
# =========================================================
# PROJECT 6 — STEP 4
# COMMIT / ID-MAP / CHANGED-ENTITY SCHEMA DIAGNOSTIC
#
# This cell does not create the mapping yet.
# It identifies the exact columns and relationships first.
# =========================================================

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from IPython.display import display


# ---------------------------------------------------------
# 1. Confirm Step 3 variables
# ---------------------------------------------------------

required_variables = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",
    "builds_raw",
    "entity_change_history_raw",
    "id_map_raw",
    "PROJECT_PREFLIGHT_DIRECTORY",
    "NOTES_DRIVE",
]


missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]


if missing_variables:
    raise RuntimeError(
        "Required Step 3 variables are missing:\n"
        + "\n".join(missing_variables)
        + "\n\nRerun Project 6 Step 3 first."
    )


if PROJECT_NUMBER != 6:
    raise AssertionError(
        f"Expected Project 6, observed {PROJECT_NUMBER}."
    )


if PROJECT_NAME != "eclipse@jetty.project":
    raise AssertionError(
        f"Unexpected project: {PROJECT_NAME}"
    )


print(
    "=== PROJECT 6 STEP 4: MAPPING SCHEMA DIAGNOSTIC ==="
)


# ---------------------------------------------------------
# 2. Stable value normalisation
# ---------------------------------------------------------

def normalise_link_value(value):
    """
    Convert IDs and strings into comparable stable text.
    """

    if pd.isna(value):
        return None


    if isinstance(
        value,
        (
            int,
            np.integer,
        ),
    ):
        return str(
            int(value)
        )


    if isinstance(
        value,
        (
            float,
            np.floating,
        ),
    ):
        if not np.isfinite(value):
            return None

        if float(value).is_integer():
            return str(
                int(value)
            )

        return str(value).strip()


    text = str(value).strip()


    if text == "":
        return None


    # Normalise strings such as "123.0" to "123".
    if re.fullmatch(
        r"[+-]?\d+\.0+",
        text,
    ):
        return text.split(".")[0]


    return text


def sample_non_null_values(
    series,
    maximum=8,
):
    values = []


    for value in series:
        normalised = normalise_link_value(
            value
        )

        if normalised is None:
            continue

        if normalised not in values:
            values.append(
                normalised
            )

        if len(values) >= maximum:
            break


    return values


def calculate_column_profile(
    dataframe,
    table_name,
):
    records = []


    for column in dataframe.columns:
        series = dataframe[
            column
        ]


        non_null = series.dropna()


        if len(non_null) == 0:
            numeric_fraction = 0.0
            hexadecimal_commit_fraction = 0.0

        else:
            numeric_fraction = float(
                pd.to_numeric(
                    non_null,
                    errors="coerce",
                )
                .notna()
                .mean()
            )


            text_values = (
                non_null
                .astype(str)
                .str.strip()
            )


            hexadecimal_commit_fraction = float(
                text_values
                .str.fullmatch(
                    r"[0-9a-fA-F]{7,64}",
                    na=False,
                )
                .mean()
            )


        records.append({
            "Table":
                table_name,

            "Column":
                str(column),

            "Dtype":
                str(series.dtype),

            "Rows":
                int(
                    len(series)
                ),

            "NonNullRows":
                int(
                    series.notna().sum()
                ),

            "UniqueNonNullValues":
                int(
                    series.nunique(
                        dropna=True
                    )
                ),

            "NumericFraction":
                numeric_fraction,

            "HexLikeFraction":
                hexadecimal_commit_fraction,

            "SampleValues":
                json.dumps(
                    sample_non_null_values(
                        series
                    )
                ),
        })


    return pd.DataFrame(
        records
    )


# ---------------------------------------------------------
# 3. Profile the three mapping-related tables
# ---------------------------------------------------------

builds_column_profile = (
    calculate_column_profile(
        builds_raw,
        "builds.csv",
    )
)


id_map_column_profile = (
    calculate_column_profile(
        id_map_raw,
        "id_map.csv",
    )
)


entity_history_column_profile = (
    calculate_column_profile(
        entity_change_history_raw,
        "entity_change_history.csv",
    )
)


mapping_column_profiles = pd.concat(
    [
        builds_column_profile,
        id_map_column_profile,
        entity_history_column_profile,
    ],
    ignore_index=True,
)


# ---------------------------------------------------------
# 4. Build unique normalised value sets
# ---------------------------------------------------------

def normalised_unique_set(
    series,
):
    values = set()


    for value in series.dropna().unique():
        normalised = normalise_link_value(
            value
        )

        if normalised is not None:
            values.add(
                normalised
            )


    return values


builds_value_sets = {
    str(column):
        normalised_unique_set(
            builds_raw[column]
        )
    for column in builds_raw.columns
}


id_map_value_sets = {
    str(column):
        normalised_unique_set(
            id_map_raw[column]
        )
    for column in id_map_raw.columns
}


entity_history_value_sets = {
    str(column):
        normalised_unique_set(
            entity_change_history_raw[
                column
            ]
        )
    for column in entity_change_history_raw.columns
}


# ---------------------------------------------------------
# 5. Calculate column-overlap diagnostics
# ---------------------------------------------------------

def calculate_overlap_records(
    left_table,
    left_sets,
    right_table,
    right_sets,
):
    records = []


    for left_column, left_values in (
        left_sets.items()
    ):
        for right_column, right_values in (
            right_sets.items()
        ):
            overlap = (
                left_values
                .intersection(
                    right_values
                )
            )


            left_coverage = (
                float(
                    len(overlap)
                    / len(left_values)
                )
                if len(left_values) > 0
                else 0.0
            )


            right_coverage = (
                float(
                    len(overlap)
                    / len(right_values)
                )
                if len(right_values) > 0
                else 0.0
            )


            records.append({
                "LeftTable":
                    left_table,

                "LeftColumn":
                    left_column,

                "RightTable":
                    right_table,

                "RightColumn":
                    right_column,

                "LeftUniqueValues":
                    int(
                        len(left_values)
                    ),

                "RightUniqueValues":
                    int(
                        len(right_values)
                    ),

                "OverlapValues":
                    int(
                        len(overlap)
                    ),

                "LeftCoverage":
                    left_coverage,

                "RightCoverage":
                    right_coverage,

                "OverlapExamples":
                    json.dumps(
                        sorted(
                            overlap
                        )[:10]
                    ),
            })


    return pd.DataFrame(
        records
    )


builds_to_id_map_overlap = (
    calculate_overlap_records(
        left_table="builds.csv",
        left_sets=builds_value_sets,
        right_table="id_map.csv",
        right_sets=id_map_value_sets,
    )
)


entity_history_to_id_map_overlap = (
    calculate_overlap_records(
        left_table=(
            "entity_change_history.csv"
        ),
        left_sets=(
            entity_history_value_sets
        ),
        right_table="id_map.csv",
        right_sets=id_map_value_sets,
    )
)


builds_to_entity_history_overlap = (
    calculate_overlap_records(
        left_table="builds.csv",
        left_sets=builds_value_sets,
        right_table=(
            "entity_change_history.csv"
        ),
        right_sets=(
            entity_history_value_sets
        ),
    )
)


all_overlap_diagnostics = pd.concat(
    [
        builds_to_id_map_overlap,
        entity_history_to_id_map_overlap,
        builds_to_entity_history_overlap,
    ],
    ignore_index=True,
)


all_overlap_diagnostics = (
    all_overlap_diagnostics
    .sort_values(
        [
            "OverlapValues",
            "LeftCoverage",
            "RightCoverage",
        ],
        ascending=[
            False,
            False,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 6. Compact exact schema views
# ---------------------------------------------------------

builds_preview = (
    builds_raw
    .head(10)
    .copy()
)


id_map_preview = (
    id_map_raw
    .head(10)
    .copy()
)


entity_history_preview = (
    entity_change_history_raw
    .head(10)
    .copy()
)


# ---------------------------------------------------------
# 7. Detect likely id_map orientation
# ---------------------------------------------------------

if len(id_map_raw.columns) != 2:
    raise AssertionError(
        "Expected id_map.csv to contain exactly "
        f"two columns; observed {len(id_map_raw.columns)}."
    )


id_map_columns = [
    str(column)
    for column in id_map_raw.columns
]


id_map_numeric_fractions = {
    column:
        float(
            pd.to_numeric(
                id_map_raw[column],
                errors="coerce",
            )
            .notna()
            .mean()
        )
    for column in id_map_columns
}


likely_id_column = max(
    id_map_numeric_fractions,
    key=id_map_numeric_fractions.get,
)


likely_name_column = [
    column
    for column in id_map_columns
    if column != likely_id_column
][0]


ID_MAP_DIAGNOSTIC_ID_COLUMN = (
    likely_id_column
)

ID_MAP_DIAGNOSTIC_NAME_COLUMN = (
    likely_name_column
)


# ---------------------------------------------------------
# 8. Save diagnostic outputs
# ---------------------------------------------------------

COLUMN_PROFILE_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_mapping_column_profiles.csv"
)


OVERLAP_DIAGNOSTIC_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_mapping_column_overlap_diagnostic.csv"
)


MAPPING_SCHEMA_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_mapping_schema_diagnostic.json"
)


mapping_column_profiles.to_csv(
    COLUMN_PROFILE_PATH,
    index=False,
)


all_overlap_diagnostics.to_csv(
    OVERLAP_DIAGNOSTIC_PATH,
    index=False,
)


mapping_schema_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "MAPPING_SCHEMA_DIAGNOSTIC_COMPLETED",

    "BuildsColumns":
        list(
            builds_raw.columns
        ),

    "IDMapColumns":
        list(
            id_map_raw.columns
        ),

    "EntityChangeHistoryColumns":
        list(
            entity_change_history_raw.columns
        ),

    "LikelyIDMapIDColumn":
        ID_MAP_DIAGNOSTIC_ID_COLUMN,

    "LikelyIDMapNameColumn":
        ID_MAP_DIAGNOSTIC_NAME_COLUMN,

    "IDMapNumericFractions":
        id_map_numeric_fractions,

    "TopOverlapRelationships":
        all_overlap_diagnostics
        .head(20)
        .to_dict(
            orient="records"
        ),

    "ColumnProfile":
        str(
            COLUMN_PROFILE_PATH
        ),

    "OverlapDiagnostic":
        str(
            OVERLAP_DIAGNOSTIC_PATH
        ),

    "CompletedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
}


with open(
    MAPPING_SCHEMA_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        mapping_schema_report,
        report_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 9. Update Project 6 checkpoint
# ---------------------------------------------------------

PROJECT_6_SELECTION_CHECKPOINT = (
    NOTES_DRIVE
    / "project_06_selection_checkpoint.json"
)


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint = json.load(
        checkpoint_file
    )


checkpoint.update({
    "Status":
        "MAPPING_SCHEMA_DIAGNOSTIC_COMPLETED",

    "MappingSchemaDiagnostic":
        str(
            MAPPING_SCHEMA_REPORT_PATH
        ),

    "LikelyIDMapIDColumn":
        ID_MAP_DIAGNOSTIC_ID_COLUMN,

    "LikelyIDMapNameColumn":
        ID_MAP_DIAGNOSTIC_NAME_COLUMN,

    "UpdatedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
})


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 10. Final output
# ---------------------------------------------------------

print(
    "\n=== PROJECT 6 STEP 4 RESULT ==="
)


print(
    "\nbuilds.csv columns:",
    list(
        builds_raw.columns
    ),
)


display(
    builds_preview
)


print(
    "\nid_map.csv columns:",
    list(
        id_map_raw.columns
    ),
)


display(
    id_map_preview
)


print(
    "\nentity_change_history.csv columns:",
    list(
        entity_change_history_raw.columns
    ),
)


display(
    entity_history_preview
)


print(
    "\nMapping-column profiles:"
)


display(
    mapping_column_profiles
)


print(
    "\nLikely id_map orientation:"
)

print(
    "Likely name/key column:",
    ID_MAP_DIAGNOSTIC_NAME_COLUMN
)

print(
    "Likely numeric ID/value column:",
    ID_MAP_DIAGNOSTIC_ID_COLUMN
)


print(
    "\nStrongest cross-table overlaps:"
)


display(
    all_overlap_diagnostics[
        [
            "LeftTable",
            "LeftColumn",
            "RightTable",
            "RightColumn",
            "LeftUniqueValues",
            "RightUniqueValues",
            "OverlapValues",
            "LeftCoverage",
            "RightCoverage",
            "OverlapExamples",
        ]
    ].head(25)
)


print(
    "\nDiagnostic report:"
)

print(
    MAPPING_SCHEMA_REPORT_PATH
)


print(
    "\nSUCCESS: Jetty mapping-related schemas were "
    "inspected without guessing."
)

print(
    "SUCCESS: Candidate cross-table link columns "
    "were measured."
)

print(
    "SUCCESS: Ready to construct and validate the "
    "canonical build-to-changed-entity mapping."
)

=== PROJECT 6 STEP 4: MAPPING SCHEMA DIAGNOSTIC ===

=== PROJECT 6 STEP 4 RESULT ===

builds.csv columns: ['id', 'commits', 'started_at']


,id,commits,started_at
0,5991191,91b3ee638d2db73eda0fc774c2181128e7a728fc#24cd5...,2013-04-02 16:36:46
1,5997354,f7eb78d849db11f5a398fb1a4eeacf6265b6c142,2013-04-02 20:31:19
2,6011421,2bd6a703f91202b4cb223defcd4e71d3e34403f7,2013-04-03 09:31:33
3,6018057,6f80105d74746ab7013e6435bee18ecedfe52d0e,2013-04-03 15:03:42
4,6018060,05b2d988b0a2afa39889013d5a61478c726f8e56,2013-04-03 15:03:56
5,6038991,df6e18cc0037d91b5986c5b689eec7bf5f13066f#3cd6d...,2013-04-04 05:01:17
6,6039388,060389147bbdec48a3a9e24fb07b8c2db148a6f8,2013-04-04 05:31:28
7,6039385,e9185aa062283e672092641f44ac5d42660ff696,2013-04-04 05:36:30
8,6050095,fc31a16c238f68e570d0713206ac834f2040af83,2013-04-04 13:31:20
9,6064085,5d451e5fec117cbd65683b035b045ca65bec02a5,2013-04-04 22:01:39



id_map.csv columns: ['key', 'value']


,key,value
0,LICENSE-APACHE-2.0.txt,1
1,LICENSE-CONTRIBUTOR/CDDLv1.0.txt,2
2,LICENSE-CONTRIBUTOR/ccla-exist.pdf,3
3,LICENSE-CONTRIBUTOR/ccla-simulalabs.txt,4
4,LICENSE-CONTRIBUTOR/ccla-template.txt,5
5,LICENSE-CONTRIBUTOR/cla-djencks.txt,6
6,LICENSE-CONTRIBUTOR/cla-gregw.txt,7
7,LICENSE-CONTRIBUTOR/cla-janb.txt,8
8,LICENSE-CONTRIBUTOR/cla-jesse.txt,9
9,LICENSE-CONTRIBUTOR/cla-jfarcand.txt,10



entity_change_history.csv columns: ['EntityId', 'AddedLines', 'DeletedLines', 'Contributor', 'BugFix', 'Commit', 'CommitDate', 'MergeCommit']


...


Mapping-column profiles:


,Table,Column,Dtype,Rows,NonNullRows,UniqueNonNullValues,NumericFraction,HexLikeFraction,SampleValues
0,builds.csv,id,int64,192,192,192,1.0,1.000000,"[""5991191"", ""5997354"", ""6011421"", ""6018057"", ""..."
1,builds.csv,commits,object,192,192,192,0.0,0.822917,"[""91b3ee638d2db73eda0fc774c2181128e7a728fc#24c..."
2,builds.csv,started_at,object,192,192,189,0.0,0.000000,"[""2013-04-02 16:36:46"", ""2013-04-02 20:31:19"",..."
3,id_map.csv,key,object,19286,19286,19286,0.0,0.000000,"[""LICENSE-APACHE-2.0.txt"", ""LICENSE-CONTRIBUTO..."
4,id_map.csv,value,int64,19286,19286,11152,1.0,0.000000,"[""1"", ""2"", ""3"", ""4"", ""5"", ""6"", ""7"", ""8""]"
5,entity_change_history.csv,EntityId,int64,553889,553889,11152,1.0,0.000000,"[""1"", ""437"", ""438"", ""439"", ""440"", ""441"", ""442""..."
6,entity_change_history.csv,AddedLines,int64,553889,553889,954,1.0,0.000000,"[""202"", ""354"", ""918"", ""188"", ""888"", ""428"", ""15..."
7,entity_change_history.csv,DeletedLines,int64,553889,553889,822,1.0,0.000000,"[""0"", ""1"", ""40"", ""3"", ""5"", ""14"", ""16"", ""2""]"
8,entity_change_history.csv,Contributor,int64,553889,553889,188,1.0,0.000000,"[""2"", ""1"", ""3"", ""4"", ""5"", ""6"", ""7"", ""8""]"
9,entity_change_history.csv,BugFix,int64,553889,553889,2,1.0,0.000000,"[""0"", ""1""]"



Likely id_map orientation:
Likely name/key column: key
Likely numeric ID/value column: value

Strongest cross-table overlaps:


,LeftTable,LeftColumn,RightTable,RightColumn,LeftUniqueValues,RightUniqueValues,OverlapValues,LeftCoverage,RightCoverage,OverlapExamples
0,entity_change_history.csv,EntityId,id_map.csv,value,11152,11152,11152,1.000000,1.000000,"[""1"", ""10"", ""100"", ""1000"", ""10000"", ""10001"", ""..."
1,entity_change_history.csv,AddedLines,id_map.csv,value,954,11152,946,0.991614,0.084828,"[""1"", ""10"", ""100"", ""1000"", ""10000"", ""1001"", ""1..."
2,entity_change_history.csv,DeletedLines,id_map.csv,value,822,11152,819,0.996350,0.073440,"[""1"", ""10"", ""100"", ""1000"", ""10000"", ""1002"", ""1..."
3,entity_change_history.csv,Contributor,id_map.csv,value,188,11152,188,1.000000,0.016858,"[""1"", ""10"", ""100"", ""101"", ""102"", ""103"", ""104"",..."
4,builds.csv,commits,entity_change_history.csv,Commit,192,25033,157,0.817708,0.006272,"[""006614470bf4403a094ed1d57d502246fe5ba407"", ""..."
5,entity_change_history.csv,BugFix,id_map.csv,value,2,11152,1,0.500000,0.000090,"[""1""]"
6,builds.csv,id,id_map.csv,key,192,19286,0,0.000000,0.000000,[]
7,builds.csv,id,id_map.csv,value,192,11152,0,0.000000,0.000000,[]
8,builds.csv,commits,id_map.csv,key,192,19286,0,0.000000,0.000000,[]
9,builds.csv,commits,id_map.csv,value,192,11152,0,0.000000,0.000000,[]



Diagnostic report:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/eclipse__jetty.project/jetty_preflight/jetty_mapping_schema_diagnostic.json

SUCCESS: Jetty mapping-related schemas were inspected without guessing.
SUCCESS: Candidate cross-table link columns were measured.
SUCCESS: Ready to construct and validate the canonical build-to-changed-entity mapping.


In [ ]:
# =========================================================
# PROJECT 6 — STEP 5
# CANONICAL BUILD → COMMIT → CHANGED-ENTITY MAPPING
#
# Protocol:
#   - split builds.csv "commits" on "#"
#   - match every individual commit to
#     entity_change_history.csv "Commit"
#   - union all EntityIds changed by all commits in a build
#   - validate EntityIds against id_map.csv "value"
#   - retain an empty changed-entity set for builds whose
#     commits cannot be matched
#   - document unmatched/partially matched builds
#
# No synthetic mapping or inferred commit is created.
# Final acceptance comes from clean REC validation next.
# =========================================================

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from IPython.display import display


# ---------------------------------------------------------
# 1. Confirm required Project 6 state
# ---------------------------------------------------------

required_variables = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",
    "builds",
    "builds_raw",
    "exe_for_rec",
    "dataset_with_order",
    "entity_change_history_raw",
    "id_map_raw",
    "PROJECT_PREFLIGHT_DIRECTORY",
    "NOTES_DRIVE",
]


missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]


if missing_variables:
    raise RuntimeError(
        "Required Step 3–4 variables are missing:\n"
        + "\n".join(missing_variables)
        + "\n\nRerun Project 6 Steps 3 and 4 first."
    )


if PROJECT_NUMBER != 6:
    raise AssertionError(
        f"Expected Project 6, observed {PROJECT_NUMBER}."
    )


if PROJECT_NAME != "eclipse@jetty.project":
    raise AssertionError(
        f"Unexpected project: {PROJECT_NAME}"
    )


print(
    "=== PROJECT 6 STEP 5: CANONICAL ENTITY MAPPING ==="
)


# ---------------------------------------------------------
# 2. Freeze exact mapping columns
# ---------------------------------------------------------

BUILD_ID_SOURCE_COLUMN = "id"
BUILD_COMMITS_SOURCE_COLUMN = "commits"

ENTITY_HISTORY_ID_COLUMN = "EntityId"
ENTITY_HISTORY_COMMIT_COLUMN = "Commit"

ID_MAP_NAME_COLUMN = "key"
ID_MAP_ID_COLUMN = "value"


required_build_columns = [
    BUILD_ID_SOURCE_COLUMN,
    BUILD_COMMITS_SOURCE_COLUMN,
]


required_entity_history_columns = [
    ENTITY_HISTORY_ID_COLUMN,
    ENTITY_HISTORY_COMMIT_COLUMN,
]


required_id_map_columns = [
    ID_MAP_NAME_COLUMN,
    ID_MAP_ID_COLUMN,
]


for column in required_build_columns:
    if column not in builds_raw.columns:
        raise KeyError(
            f"builds.csv is missing column: {column}"
        )


for column in required_entity_history_columns:
    if column not in entity_change_history_raw.columns:
        raise KeyError(
            "entity_change_history.csv is missing "
            f"column: {column}"
        )


for column in required_id_map_columns:
    if column not in id_map_raw.columns:
        raise KeyError(
            f"id_map.csv is missing column: {column}"
        )


if len(builds_raw) != 192:
    raise AssertionError(
        "Unexpected builds.csv row count.\n"
        f"Expected: 192\n"
        f"Observed: {len(builds_raw)}"
    )


# ---------------------------------------------------------
# 3. Commit normalisation helper
# ---------------------------------------------------------

def normalise_commit_hash(value):
    """
    Return a lowercase commit hash or None.
    """

    if pd.isna(value):
        return None


    commit = (
        str(value)
        .strip()
        .lower()
    )


    if commit == "":
        return None


    return commit


# ---------------------------------------------------------
# 4. Create one row per Build-Commit token
# ---------------------------------------------------------

build_commit_records = []


for source_row_number, row in (
    builds_raw.reset_index(
        drop=False
    ).iterrows()
):
    build_id = int(
        pd.to_numeric(
            row[
                BUILD_ID_SOURCE_COLUMN
            ],
            errors="raise",
        )
    )


    commit_field = str(
        row[
            BUILD_COMMITS_SOURCE_COLUMN
        ]
    ).strip()


    raw_tokens = (
        commit_field.split("#")
    )


    normalised_tokens = []


    for token_position, raw_token in enumerate(
        raw_tokens,
        start=1,
    ):
        commit = normalise_commit_hash(
            raw_token
        )


        if commit is None:
            continue


        normalised_tokens.append(
            commit
        )


        build_commit_records.append({
            "Build":
                build_id,

            "SourceBuildRow":
                int(
                    source_row_number
                ),

            "CommitPosition":
                int(
                    token_position
                ),

            "Commit":
                commit,

            "OriginalCommitField":
                commit_field,
        })


    if len(normalised_tokens) == 0:
        raise AssertionError(
            "A Jetty build contains no usable "
            f"commit token. Build: {build_id}"
        )


build_commit_map_raw = pd.DataFrame(
    build_commit_records
)


if build_commit_map_raw.empty:
    raise AssertionError(
        "No Build-Commit rows were constructed."
    )


# Audit duplicate tokens within the same build before
# canonical deduplication.
duplicate_build_commit_token_rows = int(
    build_commit_map_raw.duplicated(
        subset=[
            "Build",
            "Commit",
        ]
    ).sum()
)


build_commit_map = (
    build_commit_map_raw
    .drop_duplicates(
        subset=[
            "Build",
            "Commit",
        ],
        keep="first",
    )
    .sort_values(
        [
            "Build",
            "CommitPosition",
            "Commit",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 5. Validate commit-token format
# ---------------------------------------------------------

build_commit_map[
    "CommitIsFullSHA1"
] = (
    build_commit_map[
        "Commit"
    ]
    .str.fullmatch(
        r"[0-9a-f]{40}",
        na=False,
    )
)


invalid_commit_tokens = (
    build_commit_map[
        ~build_commit_map[
            "CommitIsFullSHA1"
        ]
    ]
    .copy()
)


if len(invalid_commit_tokens) != 0:
    display(
        invalid_commit_tokens.head(30)
    )

    raise AssertionError(
        "Jetty contains malformed commit tokens. "
        "Mapping was stopped rather than guessing."
    )


if build_commit_map[
    "Build"
].nunique() != 192:
    raise AssertionError(
        "Not every Jetty build produced at least "
        "one commit token."
    )


# ---------------------------------------------------------
# 6. Prepare canonical entity history
# ---------------------------------------------------------

entity_history_commit_entity = (
    entity_change_history_raw[
        [
            ENTITY_HISTORY_COMMIT_COLUMN,
            ENTITY_HISTORY_ID_COLUMN,
        ]
    ]
    .copy()
    .rename(
        columns={
            ENTITY_HISTORY_COMMIT_COLUMN:
                "Commit",

            ENTITY_HISTORY_ID_COLUMN:
                "EntityID",
        }
    )
)


entity_history_commit_entity[
    "Commit"
] = (
    entity_history_commit_entity[
        "Commit"
    ]
    .map(
        normalise_commit_hash
    )
)


entity_history_commit_entity[
    "EntityID"
] = pd.to_numeric(
    entity_history_commit_entity[
        "EntityID"
    ],
    errors="raise",
).astype("int64")


if entity_history_commit_entity[
    "Commit"
].isna().any():
    raise AssertionError(
        "entity_change_history.csv contains a "
        "missing commit hash."
    )


invalid_entity_history_commits = (
    ~entity_history_commit_entity[
        "Commit"
    ]
    .str.fullmatch(
        r"[0-9a-f]{40}",
        na=False,
    )
)


if invalid_entity_history_commits.any():
    raise AssertionError(
        "entity_change_history.csv contains malformed "
        "commit hashes."
    )


duplicate_commit_entity_rows = int(
    entity_history_commit_entity.duplicated(
        subset=[
            "Commit",
            "EntityID",
        ]
    ).sum()
)


entity_history_commit_entity = (
    entity_history_commit_entity
    .drop_duplicates(
        subset=[
            "Commit",
            "EntityID",
        ]
    )
    .sort_values(
        [
            "Commit",
            "EntityID",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


entity_history_commit_set = set(
    entity_history_commit_entity[
        "Commit"
    ]
)


# ---------------------------------------------------------
# 7. Validate EntityId against id_map.csv
# ---------------------------------------------------------

id_map = (
    id_map_raw[
        [
            ID_MAP_NAME_COLUMN,
            ID_MAP_ID_COLUMN,
        ]
    ]
    .copy()
    .rename(
        columns={
            ID_MAP_NAME_COLUMN:
                "EntityName",

            ID_MAP_ID_COLUMN:
                "EntityID",
        }
    )
)


id_map[
    "EntityName"
] = (
    id_map[
        "EntityName"
    ]
    .astype(str)
)


id_map[
    "EntityID"
] = pd.to_numeric(
    id_map[
        "EntityID"
    ],
    errors="raise",
).astype("int64")


if id_map[
    "EntityName"
].duplicated().any():
    raise AssertionError(
        "id_map.csv contains duplicate EntityName keys."
    )


history_entity_ids = set(
    entity_history_commit_entity[
        "EntityID"
    ].astype(int)
)


id_map_entity_ids = set(
    id_map[
        "EntityID"
    ].astype(int)
)


history_entity_ids_missing_from_id_map = sorted(
    history_entity_ids
    - id_map_entity_ids
)


id_map_entity_ids_unused_in_history = sorted(
    id_map_entity_ids
    - history_entity_ids
)


if history_entity_ids_missing_from_id_map:
    raise AssertionError(
        "Some entity-history EntityIds are absent "
        "from id_map.csv:\n"
        f"{history_entity_ids_missing_from_id_map[:30]}"
    )


# One EntityId can legitimately be associated with more
# than one historical path, for example after renaming.
entity_names_by_id = (
    id_map
    .groupby(
        "EntityID",
        sort=False,
    )[
        "EntityName"
    ]
    .apply(
        lambda values:
            set(
                values.astype(str)
            )
    )
    .to_dict()
)


# ---------------------------------------------------------
# 8. Classify matched and unmatched Build-Commit tokens
# ---------------------------------------------------------

build_commit_map[
    "CommitMatched"
] = (
    build_commit_map[
        "Commit"
    ]
    .isin(
        entity_history_commit_set
    )
)


matched_build_commits = (
    build_commit_map[
        build_commit_map[
            "CommitMatched"
        ]
    ]
    .copy()
)


unmatched_build_commits = (
    build_commit_map[
        ~build_commit_map[
            "CommitMatched"
        ]
    ]
    .copy()
)


if matched_build_commits.empty:
    raise AssertionError(
        "No Jetty build commit matched the "
        "entity-change history."
    )


# ---------------------------------------------------------
# 9. Construct canonical Build-Commit-Entity mapping
# ---------------------------------------------------------

build_commit_entity_map = (
    matched_build_commits[
        [
            "Build",
            "Commit",
        ]
    ]
    .merge(
        entity_history_commit_entity,
        on="Commit",
        how="inner",
        validate="many_to_many",
    )
    .drop_duplicates(
        subset=[
            "Build",
            "Commit",
            "EntityID",
        ]
    )
    .sort_values(
        [
            "Build",
            "Commit",
            "EntityID",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if build_commit_entity_map.empty:
    raise AssertionError(
        "The Build-Commit-Entity mapping is empty."
    )


# Canonical Build-Entity map used by REC reconstruction.
build_entity_map = (
    build_commit_entity_map[
        [
            "Build",
            "EntityID",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "Build",
            "EntityID",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if build_entity_map.duplicated(
    subset=[
        "Build",
        "EntityID",
    ]
).any():
    raise AssertionError(
        "Canonical Build-Entity mapping contains "
        "duplicate pairs."
    )


if not set(
    build_entity_map[
        "Build"
    ].astype(int)
).issubset(
    set(
        builds[
            "build"
        ].astype(int)
    )
):
    raise AssertionError(
        "Canonical mapping contains an unknown build."
    )


if not set(
    build_entity_map[
        "EntityID"
    ].astype(int)
).issubset(
    id_map_entity_ids
):
    raise AssertionError(
        "Canonical mapping contains an unknown EntityID."
    )


# ---------------------------------------------------------
# 10. Create changed-entity sets for all 192 builds
# ---------------------------------------------------------

mapped_entities_grouped = (
    build_entity_map
    .groupby(
        "Build",
        sort=False,
    )[
        "EntityID"
    ]
    .apply(
        lambda values:
            set(
                values.astype(int)
            )
    )
    .to_dict()
)


changed_entities_by_build = {
    int(build_id):
        set(
            mapped_entities_grouped.get(
                int(build_id),
                set(),
            )
        )

    for build_id
    in builds[
        "build"
    ].astype(int)
}


if len(
    changed_entities_by_build
) != 192:
    raise AssertionError(
        "Changed-entity dictionary must contain "
        "all 192 builds."
    )


if set(
    changed_entities_by_build.keys()
) != set(
    builds[
        "build"
    ].astype(int)
):
    raise AssertionError(
        "Changed-entity dictionary keys do not "
        "match the build table."
    )


# ---------------------------------------------------------
# 11. Create per-build mapping audit
# ---------------------------------------------------------

commit_counts_by_build = (
    build_commit_map
    .groupby(
        "Build",
        as_index=False,
    )
    .agg(
        CommitTokens=(
            "Commit",
            "size",
        ),

        MatchedCommitTokens=(
            "CommitMatched",
            "sum",
        ),
    )
)


commit_counts_by_build[
    "MatchedCommitTokens"
] = (
    commit_counts_by_build[
        "MatchedCommitTokens"
    ]
    .astype(int)
)


commit_counts_by_build[
    "UnmatchedCommitTokens"
] = (
    commit_counts_by_build[
        "CommitTokens"
    ]
    - commit_counts_by_build[
        "MatchedCommitTokens"
    ]
)


entity_counts_by_build = (
    build_entity_map
    .groupby(
        "Build",
        as_index=False,
    )
    .agg(
        ChangedEntities=(
            "EntityID",
            "nunique",
        )
    )
)


raw_build_impact = (
    exe_for_rec
    .groupby(
        "build",
        as_index=False,
    )
    .agg(
        RawExecutionRows=(
            "test",
            "size",
        ),

        RawFailureExecutions=(
            "verdict",
            lambda values:
                int(
                    (
                        pd.to_numeric(
                            values,
                            errors="raise",
                        )
                        != 0
                    ).sum()
                ),
        )
    )
    .rename(
        columns={
            "build":
                "Build",
        }
    )
)


model_build_impact = (
    dataset_with_order
    .groupby(
        "Build",
        as_index=False,
    )
    .agg(
        ModelReadyRows=(
            "Test",
            "size",
        ),

        ModelReadyFailures=(
            "Verdict",
            lambda values:
                int(
                    (
                        pd.to_numeric(
                            values,
                            errors="raise",
                        )
                        != 0
                    ).sum()
                ),
        )
    )
)


build_mapping_audit = (
    builds[
        [
            "build",
            "build_order",
            "partition",
        ]
    ]
    .rename(
        columns={
            "build":
                "Build",

            "build_order":
                "BuildOrder",

            "partition":
                "Partition",
        }
    )
    .merge(
        commit_counts_by_build,
        on="Build",
        how="left",
        validate="one_to_one",
    )
    .merge(
        entity_counts_by_build,
        on="Build",
        how="left",
        validate="one_to_one",
    )
    .merge(
        raw_build_impact,
        on="Build",
        how="left",
        validate="one_to_one",
    )
    .merge(
        model_build_impact,
        on="Build",
        how="left",
        validate="one_to_one",
    )
)


integer_audit_columns = [
    "CommitTokens",
    "MatchedCommitTokens",
    "UnmatchedCommitTokens",
    "ChangedEntities",
    "RawExecutionRows",
    "RawFailureExecutions",
    "ModelReadyRows",
    "ModelReadyFailures",
]


for column in integer_audit_columns:
    build_mapping_audit[
        column
    ] = (
        build_mapping_audit[
            column
        ]
        .fillna(0)
        .astype(int)
    )


def classify_build_mapping(row):
    if row[
        "MatchedCommitTokens"
    ] == 0:
        return "NO_MATCHED_COMMITS"

    if row[
        "UnmatchedCommitTokens"
    ] > 0:
        return "PARTIAL_COMMIT_COVERAGE"

    return "FULL_COMMIT_COVERAGE"


build_mapping_audit[
    "MappingStatus"
] = build_mapping_audit.apply(
    classify_build_mapping,
    axis=1,
)


build_mapping_audit = (
    build_mapping_audit
    .sort_values(
        "BuildOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if len(build_mapping_audit) != 192:
    raise AssertionError(
        "Per-build mapping audit must contain "
        "192 rows."
    )


# ---------------------------------------------------------
# 12. Unmatched-commit impact summaries
# ---------------------------------------------------------

builds_with_any_unmatched_commit = (
    build_mapping_audit[
        build_mapping_audit[
            "UnmatchedCommitTokens"
        ] > 0
    ]
    .copy()
)


builds_with_no_matched_commit = (
    build_mapping_audit[
        build_mapping_audit[
            "MatchedCommitTokens"
        ] == 0
    ]
    .copy()
)


builds_with_partial_commit_coverage = (
    build_mapping_audit[
        build_mapping_audit[
            "MappingStatus"
        ] == "PARTIAL_COMMIT_COVERAGE"
    ]
    .copy()
)


fully_mapped_builds = (
    build_mapping_audit[
        build_mapping_audit[
            "MappingStatus"
        ] == "FULL_COMMIT_COVERAGE"
    ]
    .copy()
)


mapping_status_summary = (
    build_mapping_audit[
        "MappingStatus"
    ]
    .value_counts()
    .rename_axis(
        "MappingStatus"
    )
    .reset_index(
        name="Builds"
    )
)


unmatched_partition_summary = (
    builds_with_any_unmatched_commit
    .groupby(
        "Partition",
        as_index=False,
    )
    .agg(
        Builds=(
            "Build",
            "nunique",
        ),

        UnmatchedCommitTokens=(
            "UnmatchedCommitTokens",
            "sum",
        ),

        RawExecutionRows=(
            "RawExecutionRows",
            "sum",
        ),

        RawFailureExecutions=(
            "RawFailureExecutions",
            "sum",
        ),

        ModelReadyRows=(
            "ModelReadyRows",
            "sum",
        ),

        ModelReadyFailures=(
            "ModelReadyFailures",
            "sum",
        ),
    )
)


# ---------------------------------------------------------
# 13. Mapping integrity counts
# ---------------------------------------------------------

total_commit_tokens = int(
    len(
        build_commit_map
    )
)


matched_commit_tokens = int(
    build_commit_map[
        "CommitMatched"
    ].sum()
)


unmatched_commit_tokens = int(
    (
        ~build_commit_map[
            "CommitMatched"
        ]
    ).sum()
)


fully_mapped_build_count = int(
    len(
        fully_mapped_builds
    )
)


partial_build_count = int(
    len(
        builds_with_partial_commit_coverage
    )
)


no_match_build_count = int(
    len(
        builds_with_no_matched_commit
    )
)


mapped_build_count = int(
    build_entity_map[
        "Build"
    ].nunique()
)


unique_mapped_entities = int(
    build_entity_map[
        "EntityID"
    ].nunique()
)


build_entity_pair_count = int(
    len(
        build_entity_map
    )
)


build_commit_entity_row_count = int(
    len(
        build_commit_entity_map
    )
)


commit_token_coverage = float(
    matched_commit_tokens
    / total_commit_tokens
)


build_mapping_status = (
    "PASS_EXACT_COMMIT_MAPPING"
    if unmatched_commit_tokens == 0
    else (
        "PASS_PENDING_CLEAN_REC_VALIDATION_"
        "WITH_DOCUMENTED_UNMATCHED_COMMITS"
    )
)


# ---------------------------------------------------------
# 14. Save canonical mappings and diagnostics
# ---------------------------------------------------------

BUILD_COMMIT_MAP_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_build_commit_map.csv"
)


BUILD_COMMIT_ENTITY_MAP_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_build_commit_entity_map.parquet"
)


BUILD_ENTITY_MAP_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_build_entity_map.parquet"
)


BUILD_MAPPING_AUDIT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_build_mapping_audit.csv"
)


UNMATCHED_COMMITS_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_unmatched_build_commits.csv"
)


UNMATCHED_PARTITION_SUMMARY_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_unmatched_commit_partition_summary.csv"
)


MAPPING_STATUS_SUMMARY_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_mapping_status_summary.csv"
)


ENTITY_ID_VALIDATION_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_entity_id_validation.json"
)


MAPPING_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_entity_mapping_report.json"
)


build_commit_map.to_csv(
    BUILD_COMMIT_MAP_PATH,
    index=False,
)


build_commit_entity_map.to_parquet(
    BUILD_COMMIT_ENTITY_MAP_PATH,
    index=False,
)


build_entity_map.to_parquet(
    BUILD_ENTITY_MAP_PATH,
    index=False,
)


build_mapping_audit.to_csv(
    BUILD_MAPPING_AUDIT_PATH,
    index=False,
)


unmatched_build_commits.to_csv(
    UNMATCHED_COMMITS_PATH,
    index=False,
)


unmatched_partition_summary.to_csv(
    UNMATCHED_PARTITION_SUMMARY_PATH,
    index=False,
)


mapping_status_summary.to_csv(
    MAPPING_STATUS_SUMMARY_PATH,
    index=False,
)


entity_id_validation = {
    "IDMapNameColumn":
        ID_MAP_NAME_COLUMN,

    "IDMapIDColumn":
        ID_MAP_ID_COLUMN,

    "IDMapRows":
        int(
            len(
                id_map
            )
        ),

    "IDMapUniqueNames":
        int(
            id_map[
                "EntityName"
            ].nunique()
        ),

    "IDMapUniqueEntityIDs":
        int(
            id_map[
                "EntityID"
            ].nunique()
        ),

    "EntityHistoryUniqueEntityIDs":
        int(
            len(
                history_entity_ids
            )
        ),

    "HistoryEntityIDsMissingFromIDMap":
        int(
            len(
                history_entity_ids_missing_from_id_map
            )
        ),

    "IDMapEntityIDsUnusedInHistory":
        int(
            len(
                id_map_entity_ids_unused_in_history
            )
        ),

    "AllHistoryEntityIDsValidated":
        bool(
            len(
                history_entity_ids_missing_from_id_map
            ) == 0
        ),
}


with open(
    ENTITY_ID_VALIDATION_PATH,
    "w",
    encoding="utf-8",
) as entity_validation_file:
    json.dump(
        entity_id_validation,
        entity_validation_file,
        indent=2,
    )


mapping_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        build_mapping_status,

    "BuildCommitDelimiter":
        "#",

    "BuildCommitColumn":
        BUILD_COMMITS_SOURCE_COLUMN,

    "EntityHistoryCommitColumn":
        ENTITY_HISTORY_COMMIT_COLUMN,

    "EntityHistoryIDColumn":
        ENTITY_HISTORY_ID_COLUMN,

    "IDMapNameColumn":
        ID_MAP_NAME_COLUMN,

    "IDMapIDColumn":
        ID_MAP_ID_COLUMN,

    "Builds":
        int(
            len(
                builds
            )
        ),

    "BuildCommitTokens":
        total_commit_tokens,

    "DuplicateBuildCommitTokensRemoved":
        duplicate_build_commit_token_rows,

    "MatchedCommitTokens":
        matched_commit_tokens,

    "UnmatchedCommitTokens":
        unmatched_commit_tokens,

    "CommitTokenCoverage":
        commit_token_coverage,

    "FullyMappedBuilds":
        fully_mapped_build_count,

    "PartiallyMappedBuilds":
        partial_build_count,

    "BuildsWithNoMatchedCommit":
        no_match_build_count,

    "MappedBuilds":
        mapped_build_count,

    "BuildCommitEntityRows":
        build_commit_entity_row_count,

    "BuildEntityPairs":
        build_entity_pair_count,

    "UniqueMappedEntities":
        unique_mapped_entities,

    "EntityHistoryDuplicateCommitEntityRowsRemoved":
        duplicate_commit_entity_rows,

    "HistoryEntityIDsMissingFromIDMap":
        int(
            len(
                history_entity_ids_missing_from_id_map
            )
        ),

    "ChangedEntityDictionaryBuilds":
        int(
            len(
                changed_entities_by_build
            )
        ),

    "UnmatchedBuildImpact": {
        "BuildsWithAnyUnmatchedCommit":
            int(
                len(
                    builds_with_any_unmatched_commit
                )
            ),

        "RawExecutionRows":
            int(
                builds_with_any_unmatched_commit[
                    "RawExecutionRows"
                ].sum()
            ),

        "RawFailureExecutions":
            int(
                builds_with_any_unmatched_commit[
                    "RawFailureExecutions"
                ].sum()
            ),

        "ModelReadyRows":
            int(
                builds_with_any_unmatched_commit[
                    "ModelReadyRows"
                ].sum()
            ),

        "ModelReadyFailures":
            int(
                builds_with_any_unmatched_commit[
                    "ModelReadyFailures"
                ].sum()
            ),
    },

    "MappingPolicy":
        (
            "Each builds.csv commit field is split on '#'. "
            "Changed entities are the union of EntityIds "
            "matched by exact full commit hash. Unmatched "
            "commits receive no synthetic or inferred "
            "entities and remain documented pending clean "
            "REC reconstruction validation."
        ),

    "BuildCommitMap":
        str(
            BUILD_COMMIT_MAP_PATH
        ),

    "BuildCommitEntityMap":
        str(
            BUILD_COMMIT_ENTITY_MAP_PATH
        ),

    "BuildEntityMap":
        str(
            BUILD_ENTITY_MAP_PATH
        ),

    "BuildMappingAudit":
        str(
            BUILD_MAPPING_AUDIT_PATH
        ),

    "UnmatchedCommits":
        str(
            UNMATCHED_COMMITS_PATH
        ),

    "CompletedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
}


with open(
    MAPPING_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        mapping_report,
        report_file,
        indent=2,
    )


# ---------------------------------------------------------
# 15. Update canonical builds variables
# ---------------------------------------------------------

build_commit_field_lookup = dict(
    zip(
        pd.to_numeric(
            builds_raw[
                BUILD_ID_SOURCE_COLUMN
            ],
            errors="raise",
        ).astype(int),

        builds_raw[
            BUILD_COMMITS_SOURCE_COLUMN
        ].astype(str),
    )
)


builds[
    "commits"
] = (
    builds[
        "build"
    ]
    .map(
        build_commit_field_lookup
    )
)


if builds[
    "commits"
].isna().any():
    raise AssertionError(
        "Could not attach the original commit field "
        "to every canonical build."
    )


# Compatibility table for later validated helpers.
builds_checked = pd.DataFrame({
    "id":
        builds[
            "build"
        ].astype("int64"),

    "commits":
        builds[
            "commits"
        ].astype(str),

    "started_at":
        builds[
            "timestamp"
        ],

    "build_order":
        builds[
            "build_order"
        ].astype("int64"),

    "partition":
        builds[
            "partition"
        ].astype(str),
})


# ---------------------------------------------------------
# 16. Update Project 6 checkpoint
# ---------------------------------------------------------

PROJECT_6_SELECTION_CHECKPOINT = (
    NOTES_DRIVE
    / "project_06_selection_checkpoint.json"
)


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint = json.load(
        checkpoint_file
    )


checkpoint.update({
    "Status":
        (
            "BUILD_ENTITY_MAPPING_CONSTRUCTED_"
            "PENDING_CLEAN_REC_VALIDATION"
        ),

    "EntityMappingStatus":
        build_mapping_status,

    "EntityMappingReport":
        str(
            MAPPING_REPORT_PATH
        ),

    "BuildCommitMap":
        str(
            BUILD_COMMIT_MAP_PATH
        ),

    "BuildEntityMap":
        str(
            BUILD_ENTITY_MAP_PATH
        ),

    "BuildCommitTokens":
        total_commit_tokens,

    "MatchedCommitTokens":
        matched_commit_tokens,

    "UnmatchedCommitTokens":
        unmatched_commit_tokens,

    "FullyMappedBuilds":
        fully_mapped_build_count,

    "PartiallyMappedBuilds":
        partial_build_count,

    "BuildsWithNoMatchedCommit":
        no_match_build_count,

    "ChangedEntityDictionaryBuilds":
        int(
            len(
                changed_entities_by_build
            )
        ),

    "UpdatedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
})


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 17. Final output
# ---------------------------------------------------------

print(
    "\n=== PROJECT 6 STEP 5 RESULT ==="
)


print(
    "\nCommit mapping:"
)

print(
    "Builds:",
    len(
        builds
    )
)

print(
    "Build-commit tokens:",
    total_commit_tokens
)

print(
    "Matched commit tokens:",
    matched_commit_tokens
)

print(
    "Unmatched commit tokens:",
    unmatched_commit_tokens
)

print(
    "Commit-token coverage:",
    round(
        100.0
        * commit_token_coverage,
        4,
    ),
    "%"
)


print(
    "\nBuild mapping:"
)

print(
    "Fully mapped builds:",
    fully_mapped_build_count
)

print(
    "Partially mapped builds:",
    partial_build_count
)

print(
    "Builds with no matched commit:",
    no_match_build_count
)

print(
    "Builds with at least one mapped entity:",
    mapped_build_count
)

print(
    "Build-commit-entity rows:",
    build_commit_entity_row_count
)

print(
    "Unique Build-Entity pairs:",
    build_entity_pair_count
)

print(
    "Unique mapped entities:",
    unique_mapped_entities
)

print(
    "Changed-entity dictionary builds:",
    len(
        changed_entities_by_build
    )
)


print(
    "\nEntity-ID validation:"
)

print(
    "Entity-history unique EntityIds:",
    len(
        history_entity_ids
    )
)

print(
    "id_map unique EntityIds:",
    len(
        id_map_entity_ids
    )
)

print(
    "History EntityIds missing from id_map:",
    len(
        history_entity_ids_missing_from_id_map
    )
)


print(
    "\nMapping status summary:"
)

display(
    mapping_status_summary
)


if len(
    builds_with_any_unmatched_commit
) > 0:
    print(
        "\nBuilds affected by unmatched commits:"
    )

    display(
        builds_with_any_unmatched_commit[
            [
                "Build",
                "BuildOrder",
                "Partition",
                "CommitTokens",
                "MatchedCommitTokens",
                "UnmatchedCommitTokens",
                "ChangedEntities",
                "RawExecutionRows",
                "RawFailureExecutions",
                "ModelReadyRows",
                "ModelReadyFailures",
                "MappingStatus",
            ]
        ]
    )


    print(
        "\nUnmatched commit tokens:"
    )

    display(
        unmatched_build_commits[
            [
                "Build",
                "CommitPosition",
                "Commit",
                "OriginalCommitField",
            ]
        ].head(100)
    )

else:
    print(
        "\nUnmatched commits:",
        "None"
    )


print(
    "\nCanonical mapping report:"
)

print(
    MAPPING_REPORT_PATH
)


print(
    "\nMapping status:",
    build_mapping_status
)


print(
    "\nSUCCESS: builds.csv commit lists were split "
    "using the '#' delimiter."
)

print(
    "SUCCESS: Exact full commit hashes were mapped "
    "to entity-change history."
)

print(
    "SUCCESS: Every entity-history EntityId was "
    "validated against id_map.csv."
)

print(
    "SUCCESS: The changed-entity dictionary contains "
    "all 192 Jetty builds."
)

print(
    "SUCCESS: No synthetic entity mapping was created."
)

print(
    "SUCCESS: Project 6 is ready for clean "
    "verdict-dependent REC reconstruction validation."
)

=== PROJECT 6 STEP 5: CANONICAL ENTITY MAPPING ===

=== PROJECT 6 STEP 5 RESULT ===

Commit mapping:
Builds: 192
Build-commit tokens: 237
Matched commit tokens: 236
Unmatched commit tokens: 1
Commit-token coverage: 99.5781 %

Build mapping:
Fully mapped builds: 191
Partially mapped builds: 0
Builds with no matched commit: 1
Builds with at least one mapped entity: 191
Build-commit-entity rows: 3076
Unique Build-Entity pairs: 2621
Unique mapped entities: 572
Changed-entity dictionary builds: 192

Entity-ID validation:
Entity-history unique EntityIds: 11152
id_map unique EntityIds: 11152
History EntityIds missing from id_map: 0

Mapping status summary:


,MappingStatus,Builds
0,FULL_COMMIT_COVERAGE,191
1,NO_MATCHED_COMMITS,1



Builds affected by unmatched commits:


,Build,BuildOrder,Partition,CommitTokens,MatchedCommitTokens,UnmatchedCommitTokens,ChangedEntities,RawExecutionRows,RawFailureExecutions,ModelReadyRows,ModelReadyFailures,MappingStatus
117,6806945,118,training,1,0,1,0,210,1,210,1,NO_MATCHED_COMMITS



Unmatched commit tokens:


,Build,CommitPosition,Commit,OriginalCommitField
151,6806945,1,92a39fcb088c36a4b4641276a8da3472e15c2f9c,92a39fcb088c36a4b4641276a8da3472e15c2f9c



Canonical mapping report:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/eclipse__jetty.project/jetty_preflight/jetty_entity_mapping_report.json

Mapping status: PASS_PENDING_CLEAN_REC_VALIDATION_WITH_DOCUMENTED_UNMATCHED_COMMITS

SUCCESS: builds.csv commit lists were split using the '#' delimiter.
SUCCESS: Exact full commit hashes were mapped to entity-change history.
SUCCESS: Every entity-history EntityId was validated against id_map.csv.
SUCCESS: The changed-entity dictionary contains all 192 Jetty builds.
SUCCESS: No synthetic entity mapping was created.
SUCCESS: Project 6 is ready for clean verdict-dependent REC reconstruction validation.


In [ ]:
# =========================================================
# PROJECT 6 — RUNTIME RECOVERY AFTER DISCONNECTION
#
# Restores the exact Step 1–5 state from Google Drive.
# Does not rerun completed Project 6 analyses.
# =========================================================

from pathlib import Path
import json
import shutil
import tarfile

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 1. Mount Drive when needed
# ---------------------------------------------------------

from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
else:
    print("Google Drive already mounted.")


# ---------------------------------------------------------
# 2. Permanent Project 6 paths
# ---------------------------------------------------------

THESIS_DRIVE = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

RAW_RESULTS_DRIVE = (
    THESIS_DRIVE / "Results" / "Raw"
)

AGGREGATED_RESULTS_DRIVE = (
    THESIS_DRIVE / "Results" / "Aggregated"
)

LOGS_DRIVE = (
    THESIS_DRIVE / "Results" / "Logs"
)

NOTES_DRIVE = (
    THESIS_DRIVE / "Notes"
)

RAW_DATA_DRIVE = (
    THESIS_DRIVE / "Data" / "Raw"
)

PROCESSED_DATA_DRIVE = (
    THESIS_DRIVE / "Data" / "processed"
)


PROJECT_NUMBER = 6
PROJECT_NAME = "eclipse@jetty.project"
PROJECT_SLUG = "eclipse__jetty.project"


ARCHIVE_PATH = (
    RAW_DATA_DRIVE
    / "TCP-CI-main-dataset.tar.gz"
)

SCREENING_PATH = (
    PROCESSED_DATA_DRIVE
    / "tcp_ci_project_screening.csv"
)

COMPLETION_REGISTRY_PATH = (
    NOTES_DRIVE
    / "completed_project_registry.csv"
)

PROJECT_6_SELECTION_CHECKPOINT = (
    NOTES_DRIVE
    / "project_06_selection_checkpoint.json"
)


PROJECT_RAW_RESULTS = (
    RAW_RESULTS_DRIVE
    / PROJECT_SLUG
)

PROJECT_AGGREGATED_RESULTS = (
    AGGREGATED_RESULTS_DRIVE
    / PROJECT_SLUG
)

PROJECT_LOGS = (
    LOGS_DRIVE
    / PROJECT_SLUG
)

PROJECT_PREFLIGHT_DIRECTORY = (
    PROJECT_AGGREGATED_RESULTS
    / "jetty_preflight"
)


required_drive_paths = [
    THESIS_DRIVE,
    ARCHIVE_PATH,
    SCREENING_PATH,
    COMPLETION_REGISTRY_PATH,
    PROJECT_6_SELECTION_CHECKPOINT,
    PROJECT_PREFLIGHT_DIRECTORY,
]


missing_drive_paths = [
    str(path)
    for path in required_drive_paths
    if not path.exists()
]


if missing_drive_paths:
    raise FileNotFoundError(
        "Required Project 6 paths are missing:\n"
        + "\n".join(missing_drive_paths)
    )


# ---------------------------------------------------------
# 3. Validate the saved Project 6 checkpoint
# ---------------------------------------------------------

with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    project_6_checkpoint = json.load(
        checkpoint_file
    )


if project_6_checkpoint.get(
    "Project"
) != PROJECT_NAME:
    raise AssertionError(
        "The saved Project 6 checkpoint belongs "
        "to a different project."
    )


valid_checkpoint_statuses = {
    (
        "BUILD_ENTITY_MAPPING_CONSTRUCTED_"
        "PENDING_CLEAN_REC_VALIDATION"
    ),
    "CLEAN_REC_VALIDATION_PASSED",
    "HELPER_VALIDATION_PASSED",
    "SMOKE_RUN_PASSED",
    "FULL_RUN_COMPLETE",
}


if project_6_checkpoint.get(
    "Status"
) not in valid_checkpoint_statuses:
    raise AssertionError(
        "Unexpected Project 6 checkpoint status:\n"
        f"{project_6_checkpoint.get('Status')}"
    )


# ---------------------------------------------------------
# 4. Restore the runtime dataset
# ---------------------------------------------------------

RUNTIME_ROOT = Path(
    "/content/working_data/tcp_ci_full"
)


def locate_datasets_root(root):
    direct = root / "datasets"

    if direct.exists():
        return direct

    if not root.exists():
        return None

    candidates = []

    for candidate in root.rglob("datasets"):
        if not candidate.is_dir():
            continue

        project_count = sum(
            child.is_dir()
            for child in candidate.iterdir()
        )

        if project_count >= 20:
            candidates.append(
                (
                    project_count,
                    candidate,
                )
            )

    if not candidates:
        return None

    candidates.sort(
        key=lambda item: item[0],
        reverse=True,
    )

    return candidates[0][1]


DATASETS_ROOT = locate_datasets_root(
    RUNTIME_ROOT
)


if DATASETS_ROOT is None:
    if RUNTIME_ROOT.exists():
        shutil.rmtree(
            RUNTIME_ROOT
        )

    RUNTIME_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    print("Extracting TCP-CI archive...")

    with tarfile.open(
        ARCHIVE_PATH,
        mode="r:gz",
    ) as archive:
        try:
            archive.extractall(
                RUNTIME_ROOT,
                filter="data",
            )

        except TypeError:
            archive.extractall(
                RUNTIME_ROOT
            )

    DATASETS_ROOT = locate_datasets_root(
        RUNTIME_ROOT
    )


if DATASETS_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the extracted datasets folder."
    )


project_directories = [
    path
    for path in DATASETS_ROOT.iterdir()
    if path.is_dir()
]


if len(project_directories) != 25:
    raise AssertionError(
        "Unexpected extracted project count.\n"
        f"Expected: 25\n"
        f"Observed: {len(project_directories)}"
    )


PROJECT_DATA_DIRECTORY = (
    DATASETS_ROOT
    / PROJECT_NAME
)

PROJECT_DATA_DIR = (
    PROJECT_DATA_DIRECTORY
)


if not PROJECT_DATA_DIRECTORY.exists():
    raise FileNotFoundError(
        "Jetty data directory is missing:\n"
        f"{PROJECT_DATA_DIRECTORY}"
    )


# ---------------------------------------------------------
# 5. Load the six original source files
# ---------------------------------------------------------

builds_raw = pd.read_csv(
    PROJECT_DATA_DIRECTORY / "builds.csv",
    low_memory=False,
)

exe_raw = pd.read_csv(
    PROJECT_DATA_DIRECTORY / "exe.csv",
    low_memory=False,
)

dataset_raw = pd.read_csv(
    PROJECT_DATA_DIRECTORY / "dataset.csv",
    low_memory=False,
)

entity_change_history_raw = pd.read_csv(
    PROJECT_DATA_DIRECTORY
    / "entity_change_history.csv",
    low_memory=False,
)

id_map_raw = pd.read_csv(
    PROJECT_DATA_DIRECTORY / "id_map.csv",
    low_memory=False,
)

contributors_raw = pd.read_csv(
    PROJECT_DATA_DIRECTORY / "contributors.csv",
    low_memory=False,
)


expected_source_counts = {
    "builds":
        192,

    "exe":
        26439,

    "dataset":
        17291,

    "entity_change_history":
        553889,

    "id_map":
        19286,

    "contributors":
        188,
}


observed_source_counts = {
    "builds":
        len(builds_raw),

    "exe":
        len(exe_raw),

    "dataset":
        len(dataset_raw),

    "entity_change_history":
        len(entity_change_history_raw),

    "id_map":
        len(id_map_raw),

    "contributors":
        len(contributors_raw),
}


if observed_source_counts != expected_source_counts:
    raise AssertionError(
        "Restored source counts differ from the "
        "validated Project 6 counts.\n"
        f"Expected: {expected_source_counts}\n"
        f"Observed: {observed_source_counts}"
    )


# ---------------------------------------------------------
# 6. Restore the chronological build split
# ---------------------------------------------------------

builds = pd.DataFrame({
    "build":
        pd.to_numeric(
            builds_raw["id"],
            errors="raise",
        ).astype("int64"),

    "commits":
        builds_raw[
            "commits"
        ].astype(str),

    "timestamp":
        pd.to_datetime(
            builds_raw["started_at"],
            errors="raise",
            utc=True,
        ),
})


builds = (
    builds
    .sort_values(
        [
            "timestamp",
            "build",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


builds["build_order"] = np.arange(
    1,
    len(builds) + 1,
    dtype=np.int64,
)


TRAINING_BUILD_COUNT = int(
    np.floor(
        0.75 * len(builds)
    )
)


builds["partition"] = np.where(
    builds["build_order"]
    <= TRAINING_BUILD_COUNT,
    "training",
    "evaluation",
)


if TRAINING_BUILD_COUNT != 144:
    raise AssertionError(
        "Unexpected training-build count."
    )


if (
    builds["partition"]
    == "evaluation"
).sum() != 48:
    raise AssertionError(
        "Unexpected evaluation-build count."
    )


build_order_lookup = dict(
    zip(
        builds[
            "build"
        ].astype(int),

        builds[
            "build_order"
        ].astype(int),
    )
)


build_partition_lookup = dict(
    zip(
        builds[
            "build"
        ].astype(int),

        builds[
            "partition"
        ].astype(str),
    )
)


builds_checked = pd.DataFrame({
    "id":
        builds[
            "build"
        ].astype("int64"),

    "commits":
        builds[
            "commits"
        ].astype(str),

    "started_at":
        builds[
            "timestamp"
        ],

    "build_order":
        builds[
            "build_order"
        ].astype("int64"),

    "partition":
        builds[
            "partition"
        ].astype(str),
})


# ---------------------------------------------------------
# 7. Restore canonical raw execution history
# ---------------------------------------------------------

exe_for_rec = (
    exe_raw
    .rename(
        columns={
            "build":
                "build",

            "test":
                "test",

            "verdict":
                "verdict",

            "duration":
                "duration",

            "job":
                "job",
        }
    )
    .copy()
)


exe_for_rec["build"] = pd.to_numeric(
    exe_for_rec["build"],
    errors="raise",
).astype("int64")


exe_for_rec["test"] = (
    exe_for_rec[
        "test"
    ].astype(str)
)


exe_for_rec["verdict"] = pd.to_numeric(
    exe_for_rec["verdict"],
    errors="raise",
).astype("int8")


exe_for_rec["duration"] = pd.to_numeric(
    exe_for_rec["duration"],
    errors="raise",
).astype(float)


if "job" not in exe_for_rec.columns:
    exe_for_rec["job"] = ""


exe_for_rec["build_order"] = (
    exe_for_rec[
        "build"
    ]
    .map(
        build_order_lookup
    )
)


exe_for_rec["partition"] = (
    exe_for_rec[
        "build"
    ]
    .map(
        build_partition_lookup
    )
)


if exe_for_rec[
    "build_order"
].isna().any():
    raise AssertionError(
        "Some execution rows could not be assigned "
        "to the chronological build order."
    )


exe_for_rec = (
    exe_for_rec
    .sort_values(
        [
            "build_order",
            "job",
            "test",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


clean_training_history = (
    exe_for_rec[
        exe_for_rec[
            "partition"
        ] == "training"
    ]
    .copy()
    .reset_index(drop=True)
)


clean_evaluation_history = (
    exe_for_rec[
        exe_for_rec[
            "partition"
        ] == "evaluation"
    ]
    .copy()
    .reset_index(drop=True)
)


if len(
    clean_training_history
) != 19845:
    raise AssertionError(
        "Unexpected raw training-history size."
    )


if len(
    clean_evaluation_history
) != 6594:
    raise AssertionError(
        "Unexpected raw evaluation-history size."
    )


# ---------------------------------------------------------
# 8. Restore the model-ready data
# ---------------------------------------------------------

dataset = dataset_raw.copy()


dataset["Build"] = pd.to_numeric(
    dataset["Build"],
    errors="raise",
).astype("int64")


dataset["Test"] = (
    dataset[
        "Test"
    ].astype(str)
)


dataset["Verdict"] = pd.to_numeric(
    dataset["Verdict"],
    errors="raise",
).astype("int8")


dataset["Duration"] = pd.to_numeric(
    dataset["Duration"],
    errors="raise",
).astype(float)


dataset["build_order"] = (
    dataset[
        "Build"
    ]
    .map(
        build_order_lookup
    )
    .astype("int64")
)


dataset["partition"] = (
    dataset[
        "Build"
    ]
    .map(
        build_partition_lookup
    )
)


dataset_with_order = (
    dataset
    .sort_values(
        [
            "build_order",
            "Build",
            "Test",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


clean_training_data = (
    dataset_with_order[
        dataset_with_order[
            "partition"
        ] == "training"
    ]
    .copy()
    .reset_index(drop=True)
)


clean_evaluation_data = (
    dataset_with_order[
        dataset_with_order[
            "partition"
        ] == "evaluation"
    ]
    .copy()
    .reset_index(drop=True)
)


MODEL_FEATURE_COLUMNS = [
    column
    for column in dataset.columns
    if column not in {
        "Build",
        "Test",
        "Verdict",
        "Duration",
        "build_order",
        "partition",
    }
]


if len(
    MODEL_FEATURE_COLUMNS
) != 150:
    raise AssertionError(
        "Unexpected model predictor count."
    )


if len(
    clean_training_data
) != 13212:
    raise AssertionError(
        "Unexpected model-training row count."
    )


if len(
    clean_evaluation_data
) != 4079:
    raise AssertionError(
        "Unexpected model-evaluation row count."
    )


# ---------------------------------------------------------
# 9. Restore saved Step 5 mappings
# ---------------------------------------------------------

BUILD_COMMIT_MAP_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_build_commit_map.csv"
)

BUILD_COMMIT_ENTITY_MAP_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_build_commit_entity_map.parquet"
)

BUILD_ENTITY_MAP_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_build_entity_map.parquet"
)

BUILD_MAPPING_AUDIT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_build_mapping_audit.csv"
)

UNMATCHED_COMMITS_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_unmatched_build_commits.csv"
)

MAPPING_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_entity_mapping_report.json"
)


required_mapping_files = [
    BUILD_COMMIT_MAP_PATH,
    BUILD_COMMIT_ENTITY_MAP_PATH,
    BUILD_ENTITY_MAP_PATH,
    BUILD_MAPPING_AUDIT_PATH,
    UNMATCHED_COMMITS_PATH,
    MAPPING_REPORT_PATH,
]


missing_mapping_files = [
    str(path)
    for path in required_mapping_files
    if not path.exists()
]


if missing_mapping_files:
    raise FileNotFoundError(
        "Saved Step 5 mapping artefacts are missing:\n"
        + "\n".join(missing_mapping_files)
    )


build_commit_map = pd.read_csv(
    BUILD_COMMIT_MAP_PATH
)

build_commit_entity_map = pd.read_parquet(
    BUILD_COMMIT_ENTITY_MAP_PATH
)

build_entity_map = pd.read_parquet(
    BUILD_ENTITY_MAP_PATH
)

build_mapping_audit = pd.read_csv(
    BUILD_MAPPING_AUDIT_PATH
)

unmatched_build_commits = pd.read_csv(
    UNMATCHED_COMMITS_PATH
)


with open(
    MAPPING_REPORT_PATH,
    "r",
    encoding="utf-8",
) as report_file:
    mapping_report = json.load(
        report_file
    )


for dataframe in [
    build_commit_entity_map,
    build_entity_map,
]:
    dataframe["Build"] = pd.to_numeric(
        dataframe["Build"],
        errors="raise",
    ).astype("int64")

    dataframe["EntityID"] = pd.to_numeric(
        dataframe["EntityID"],
        errors="raise",
    ).astype("int64")


mapped_entities_grouped = (
    build_entity_map
    .groupby(
        "Build",
        sort=False,
    )[
        "EntityID"
    ]
    .apply(
        lambda values:
            set(
                values.astype(int)
            )
    )
    .to_dict()
)


changed_entities_by_build = {
    int(build_id):
        set(
            mapped_entities_grouped.get(
                int(build_id),
                set(),
            )
        )

    for build_id
    in builds[
        "build"
    ].astype(int)
}


# Compatibility alias used by earlier canonical helpers.
build_entity_map_compatibility = (
    build_entity_map
    .rename(
        columns={
            "Build":
                "id",

            "EntityID":
                "EntityId",
        }
    )
)


if len(
    changed_entities_by_build
) != 192:
    raise AssertionError(
        "Changed-entity dictionary does not contain "
        "all 192 builds."
    )


# ---------------------------------------------------------
# 10. Validate the restored Step 5 result
# ---------------------------------------------------------

expected_mapping_values = {
    "BuildCommitTokens":
        237,

    "MatchedCommitTokens":
        236,

    "UnmatchedCommitTokens":
        1,

    "FullyMappedBuilds":
        191,

    "PartiallyMappedBuilds":
        0,

    "BuildsWithNoMatchedCommit":
        1,

    "BuildCommitEntityRows":
        3076,

    "BuildEntityPairs":
        2621,

    "UniqueMappedEntities":
        572,

    "ChangedEntityDictionaryBuilds":
        192,
}


for key, expected_value in (
    expected_mapping_values.items()
):
    observed_value = mapping_report.get(
        key
    )

    if observed_value != expected_value:
        raise AssertionError(
            f"Mapping-report mismatch for {key}.\n"
            f"Expected: {expected_value}\n"
            f"Observed: {observed_value}"
        )


if len(
    unmatched_build_commits
) != 1:
    raise AssertionError(
        "Expected exactly one unmatched commit token."
    )


unmatched_commit_row = (
    unmatched_build_commits.iloc[0]
)


if int(
    unmatched_commit_row[
        "Build"
    ]
) != 6806945:
    raise AssertionError(
        "Unexpected unmatched build."
    )


if str(
    unmatched_commit_row[
        "Commit"
    ]
).strip().lower() != (
    "92a39fcb088c36a4b4641276a8da3472e15c2f9c"
):
    raise AssertionError(
        "Unexpected unmatched commit hash."
    )


unmatched_build_audit = (
    build_mapping_audit[
        pd.to_numeric(
            build_mapping_audit[
                "Build"
            ],
            errors="raise",
        ).astype(int)
        == 6806945
    ]
)


if len(
    unmatched_build_audit
) != 1:
    raise AssertionError(
        "Could not locate the unmatched build audit row."
    )


audit_row = (
    unmatched_build_audit.iloc[0]
)


if int(
    audit_row[
        "RawExecutionRows"
    ]
) != 210:
    raise AssertionError(
        "Unexpected unmatched-build execution count."
    )


if int(
    audit_row[
        "RawFailureExecutions"
    ]
) != 1:
    raise AssertionError(
        "Unexpected unmatched-build failure count."
    )


if int(
    audit_row[
        "ModelReadyRows"
    ]
) != 210:
    raise AssertionError(
        "Unexpected unmatched-build model-row count."
    )


if int(
    audit_row[
        "ModelReadyFailures"
    ]
) != 1:
    raise AssertionError(
        "Unexpected unmatched-build model-failure count."
    )


# ---------------------------------------------------------
# 11. Compact result
# ---------------------------------------------------------

print(
    "\n=== PROJECT 6 RUNTIME RECOVERY RESULT ==="
)

print(
    "Project:",
    PROJECT_NAME
)

print(
    "Runtime datasets root:",
    DATASETS_ROOT
)

print(
    "Builds:",
    len(
        builds
    )
)

print(
    "Raw training rows:",
    len(
        clean_training_history
    )
)

print(
    "Raw evaluation rows:",
    len(
        clean_evaluation_history
    )
)

print(
    "Model-training rows:",
    len(
        clean_training_data
    )
)

print(
    "Model-evaluation rows:",
    len(
        clean_evaluation_data
    )
)

print(
    "Model predictors:",
    len(
        MODEL_FEATURE_COLUMNS
    )
)

print(
    "Build-commit tokens:",
    len(
        build_commit_map
    )
)

print(
    "Build-Entity pairs:",
    len(
        build_entity_map
    )
)

print(
    "Changed-entity dictionary builds:",
    len(
        changed_entities_by_build
    )
)

print(
    "Unmatched build:",
    int(
        unmatched_commit_row[
            "Build"
        ]
    )
)

print(
    "Unmatched commit:",
    unmatched_commit_row[
        "Commit"
    ]
)

print(
    "Saved checkpoint status:",
    project_6_checkpoint[
        "Status"
    ]
)

print(
    "\nSUCCESS: The disconnected Project 6 runtime "
    "was restored from permanent files."
)

print(
    "SUCCESS: Steps 1–5 were not rerun."
)

print(
    "SUCCESS: The exact Step 5 entity mapping was "
    "restored and validated."
)

print(
    "SUCCESS: Project 6 is ready for Step 6 clean "
    "REC reconstruction validation."
)

Google Drive already mounted.
Extracting TCP-CI archive...

=== PROJECT 6 RUNTIME RECOVERY RESULT ===
Project: eclipse@jetty.project
Runtime datasets root: /content/working_data/tcp_ci_full/datasets
Builds: 192
Raw training rows: 19845
Raw evaluation rows: 6594
Model-training rows: 13212
Model-evaluation rows: 4079
Model predictors: 150
Build-commit tokens: 237
Build-Entity pairs: 2621
Changed-entity dictionary builds: 192
Unmatched build: 6806945
Unmatched commit: 92a39fcb088c36a4b4641276a8da3472e15c2f9c
Saved checkpoint status: BUILD_ENTITY_MAPPING_CONSTRUCTED_PENDING_CLEAN_REC_VALIDATION

SUCCESS: The disconnected Project 6 runtime was restored from permanent files.
SUCCESS: Steps 1–5 were not rerun.
SUCCESS: The exact Step 5 entity mapping was restored and validated.
SUCCESS: Project 6 is ready for Step 6 clean REC reconstruction validation.


In [ ]:
# =========================================================
# PROJECT 6 — STEP 6
# CLEAN REC RECONSTRUCTION AND ENTITY-MAPPING VALIDATION
#
# Acceptance rules:
#   1. Reconstruct all 19 REC features from clean history.
#   2. The 13 verdict-dependent REC features must match.
#   3. The unmatched build 6806945 must have zero
#      verdict-dependent REC mismatches.
#   4. The six noise-independent REC features are audited
#      but preserved from the original TCP-CI dataset.
#   5. Construct an exact canonical 19-feature matrix:
#        13 reconstructed verdict-dependent features
#        6 preserved noise-independent features
# =========================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display


# ---------------------------------------------------------
# 1. Confirm restored Project 6 state
# ---------------------------------------------------------

required_variables = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",
    "builds_checked",
    "exe_for_rec",
    "dataset_with_order",
    "clean_training_data",
    "clean_evaluation_data",
    "build_entity_map",
    "changed_entities_by_build",
    "PROJECT_PREFLIGHT_DIRECTORY",
    "PROJECT_6_SELECTION_CHECKPOINT",
]


missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]


if missing_variables:
    raise RuntimeError(
        "Required recovery variables are missing:\n"
        + "\n".join(missing_variables)
        + "\n\nRerun the Project 6 runtime recovery cell."
    )


if PROJECT_NUMBER != 6:
    raise AssertionError(
        f"Expected Project 6, observed {PROJECT_NUMBER}."
    )


if PROJECT_NAME != "eclipse@jetty.project":
    raise AssertionError(
        f"Unexpected project: {PROJECT_NAME}"
    )


print(
    "=== PROJECT 6 STEP 6: CLEAN REC VALIDATION ==="
)


# ---------------------------------------------------------
# 2. Freeze REC feature policy
# ---------------------------------------------------------

RECENT_WINDOW = 6


REC_FEATURE_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


VERDICT_DEPENDENT_REC_COLUMNS = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


NOISE_INDEPENDENT_REC_COLUMNS = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]


if len(REC_FEATURE_COLUMNS) != 19:
    raise AssertionError(
        "Expected exactly 19 REC features."
    )


if len(VERDICT_DEPENDENT_REC_COLUMNS) != 13:
    raise AssertionError(
        "Expected exactly 13 verdict-dependent "
        "REC features."
    )


if len(NOISE_INDEPENDENT_REC_COLUMNS) != 6:
    raise AssertionError(
        "Expected exactly six noise-independent "
        "REC features."
    )


if (
    set(VERDICT_DEPENDENT_REC_COLUMNS)
    & set(NOISE_INDEPENDENT_REC_COLUMNS)
):
    raise AssertionError(
        "REC feature groups overlap."
    )


if (
    set(VERDICT_DEPENDENT_REC_COLUMNS)
    | set(NOISE_INDEPENDENT_REC_COLUMNS)
) != set(REC_FEATURE_COLUMNS):
    raise AssertionError(
        "REC feature groups do not cover all "
        "19 REC features."
    )


missing_dataset_rec_columns = [
    column
    for column in REC_FEATURE_COLUMNS
    if column not in dataset_with_order.columns
]


if missing_dataset_rec_columns:
    raise KeyError(
        "The model-ready dataset is missing REC columns:\n"
        + "\n".join(missing_dataset_rec_columns)
    )


# ---------------------------------------------------------
# 3. Prepare chronological execution context
# ---------------------------------------------------------

exe_for_rec = exe_for_rec.copy()


exe_for_rec["build"] = pd.to_numeric(
    exe_for_rec["build"],
    errors="raise",
).astype("int64")


exe_for_rec["test"] = (
    exe_for_rec["test"]
    .astype(str)
)


exe_for_rec["verdict"] = pd.to_numeric(
    exe_for_rec["verdict"],
    errors="raise",
).astype("int8")


exe_for_rec["duration"] = pd.to_numeric(
    exe_for_rec["duration"],
    errors="raise",
).astype(float)


exe_for_rec["build_order"] = pd.to_numeric(
    exe_for_rec["build_order"],
    errors="raise",
).astype("int64")


exe_for_rec = (
    exe_for_rec
    .sort_values(
        [
            "build_order",
            "job",
            "test",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


ordered_execution_builds = (
    exe_for_rec[
        [
            "build",
            "build_order",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "build_order",
            "build",
        ],
        kind="mergesort",
    )[
        "build"
    ]
    .astype(int)
    .tolist()
)


global_build_position = {
    int(build_id):
        position

    for position, build_id
    in enumerate(
        ordered_execution_builds
    )
}


if len(global_build_position) != 192:
    raise AssertionError(
        "Expected all 192 builds to appear in "
        "the execution history."
    )


# ---------------------------------------------------------
# 4. Prepare entity-change history
# ---------------------------------------------------------

entity_build_pairs = (
    build_entity_map[
        [
            "Build",
            "EntityID",
        ]
    ]
    .drop_duplicates()
    .copy()
)


entity_build_pairs["Build"] = pd.to_numeric(
    entity_build_pairs["Build"],
    errors="raise",
).astype("int64")


entity_build_pairs["EntityID"] = pd.to_numeric(
    entity_build_pairs["EntityID"],
    errors="raise",
).astype("int64")


entity_changed_builds = (
    entity_build_pairs
    .groupby(
        "EntityID",
        sort=False,
    )[
        "Build"
    ]
    .apply(
        lambda values:
            set(
                values.astype(int)
            )
    )
    .to_dict()
)


clean_changed_entities_by_build = {
    int(build_id):
        set(
            int(entity_id)
            for entity_id in (
                changed_entities_by_build.get(
                    int(build_id),
                    set(),
                )
            )
        )

    for build_id
    in builds_checked[
        "id"
    ].astype(int)
}


if len(
    clean_changed_entities_by_build
) != 192:
    raise AssertionError(
        "The changed-entity dictionary must contain "
        "all 192 builds."
    )


UNMATCHED_BUILD_ID = 6806945


if clean_changed_entities_by_build[
    UNMATCHED_BUILD_ID
] != set():
    raise AssertionError(
        "The unmatched build should have an empty "
        "changed-entity set."
    )


# ---------------------------------------------------------
# 5. REC calculation helpers
# ---------------------------------------------------------

def calculate_rates(
    history,
):
    """
    Reproduce TCP-CI historical verdict and
    transition rates.
    """

    history_length = len(
        history
    )


    if history_length == 0:
        raise ValueError(
            "Rate calculation requires "
            "non-empty history."
        )


    verdicts = pd.to_numeric(
        history["verdict"],
        errors="raise",
    )


    fail_rate = float(
        (
            verdicts != 0
        ).sum()
        / history_length
    )


    assertion_rate = float(
        (
            verdicts == 2
        ).sum()
        / history_length
    )


    exception_rate = float(
        (
            verdicts == 1
        ).sum()
        / history_length
    )


    transition_rate = float(
        (
            history["transition"]
            == 1
        ).sum()
        / history_length
    )


    return (
        fail_rate,
        assertion_rate,
        exception_rate,
        transition_rate,
    )


def calculate_max_test_file_rate(
    history,
    target_column,
    current_changed_entities,
):
    """
    Reproduce TCP-CI's maximum changed-file
    historical rate.
    """

    target_builds = (
        history.loc[
            history[
                target_column
            ] > 0,
            "build",
        ]
        .drop_duplicates()
        .astype(int)
        .tolist()
    )


    # Official default when no historical failure or
    # transition exists.
    if len(target_builds) == 0:
        return -1.0


    target_build_set = set(
        target_builds
    )


    maximum_frequency = 0


    for entity_id in (
        current_changed_entities
    ):
        entity_builds = (
            entity_changed_builds.get(
                int(entity_id),
                set(),
            )
        )


        overlap_count = len(
            entity_builds.intersection(
                target_build_set
            )
        )


        maximum_frequency = max(
            maximum_frequency,
            overlap_count,
        )


    # Historical targets exist, but none overlap with
    # the current build's changed entities.
    if maximum_frequency == 0:
        return 0.0


    return float(
        maximum_frequency
        / len(target_builds)
    )


# ---------------------------------------------------------
# 6. Canonical REC reconstruction function
# ---------------------------------------------------------

def reconstruct_rec_features(
    execution_history,
    requested_rows,
    recent_window=RECENT_WINDOW,
):
    """
    Reconstruct all 19 REC features using only
    executions preceding each requested Build-Test row.
    """

    required_history_columns = {
        "build",
        "test",
        "verdict",
        "duration",
        "build_order",
    }


    missing_history_columns = (
        required_history_columns
        - set(
            execution_history.columns
        )
    )


    if missing_history_columns:
        raise KeyError(
            "Execution history is missing columns:\n"
            + "\n".join(
                sorted(
                    missing_history_columns
                )
            )
        )


    requested = (
        requested_rows[
            [
                "Build",
                "Test",
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )


    requested["Build"] = pd.to_numeric(
        requested["Build"],
        errors="raise",
    ).astype("int64")


    requested["Test"] = (
        requested["Test"]
        .astype(str)
    )


    if requested.duplicated(
        subset=[
            "Build",
            "Test",
        ]
    ).any():
        raise AssertionError(
            "Requested REC rows contain duplicate "
            "Build-Test pairs."
        )


    requested_pairs = set(
        zip(
            requested["Build"].astype(int),
            requested["Test"].astype(str),
        )
    )


    history_data = (
        execution_history
        .copy()
        .reset_index(drop=True)
    )


    history_data["build"] = pd.to_numeric(
        history_data["build"],
        errors="raise",
    ).astype("int64")


    history_data["test"] = (
        history_data["test"]
        .astype(str)
    )


    history_data["verdict"] = pd.to_numeric(
        history_data["verdict"],
        errors="raise",
    ).astype("int8")


    history_data["duration"] = pd.to_numeric(
        history_data["duration"],
        errors="raise",
    ).astype(float)


    reconstructed_records = []


    for test_id, test_history in (
        history_data.groupby(
            "test",
            sort=False,
        )
    ):
        sort_columns = [
            "build_order",
        ]


        if "job" in test_history.columns:
            sort_columns.append(
                "job"
            )


        test_history = (
            test_history
            .sort_values(
                sort_columns,
                kind="mergesort",
            )
            .reset_index(drop=True)
            .copy()
        )


        test_history[
            "transition"
        ] = (
            test_history[
                "verdict"
            ]
            .diff()
            .fillna(0)
            .ne(0)
            .astype("int8")
        )


        first_test_build = int(
            test_history.iloc[0][
                "build"
            ]
        )


        for current_position in range(
            len(test_history)
        ):
            current_row = (
                test_history.iloc[
                    current_position
                ]
            )


            current_build = int(
                current_row[
                    "build"
                ]
            )


            current_test = str(
                test_id
            )


            pair = (
                current_build,
                current_test,
            )


            if pair not in requested_pairs:
                continue


            history = (
                test_history
                .iloc[
                    :current_position
                ]
                .copy()
                .reset_index(drop=True)
            )


            record = {
                "Build":
                    current_build,

                "Test":
                    current_test,
            }


            if history.empty:
                for column in (
                    REC_FEATURE_COLUMNS
                ):
                    record[
                        column
                    ] = -1.0


                record[
                    "REC_Age"
                ] = 0.0


                reconstructed_records.append(
                    record
                )

                continue


            recent_history = (
                history
                .tail(
                    recent_window
                )
                .copy()
            )


            current_global_position = (
                global_build_position[
                    current_build
                ]
            )


            first_global_position = (
                global_build_position[
                    first_test_build
                ]
            )


            age = (
                current_global_position
                - first_global_position
            )


            failure_positions = (
                np.flatnonzero(
                    history[
                        "verdict"
                    ]
                    .to_numpy()
                    != 0
                )
            )


            if len(
                failure_positions
            ) == 0:
                last_failure_age = -1

            else:
                last_failure_age = (
                    len(history)
                    - 1
                    - int(
                        failure_positions[-1]
                    )
                )


            transition_positions = (
                np.flatnonzero(
                    history[
                        "transition"
                    ]
                    .to_numpy()
                    > 0
                )
            )


            if len(
                transition_positions
            ) == 0:
                last_transition_age = -1

            else:
                last_transition_age = (
                    len(history)
                    - 1
                    - int(
                        transition_positions[-1]
                    )
                )


            (
                recent_fail_rate,
                recent_assert_rate,
                recent_exc_rate,
                recent_transition_rate,
            ) = calculate_rates(
                recent_history
            )


            (
                total_fail_rate,
                total_assert_rate,
                total_exc_rate,
                total_transition_rate,
            ) = calculate_rates(
                history
            )


            current_changed_entities = (
                clean_changed_entities_by_build.get(
                    current_build,
                    set(),
                )
            )


            max_test_file_fail_rate = (
                calculate_max_test_file_rate(
                    history=history,
                    target_column=(
                        "verdict"
                    ),
                    current_changed_entities=(
                        current_changed_entities
                    ),
                )
            )


            max_test_file_transition_rate = (
                calculate_max_test_file_rate(
                    history=history,
                    target_column=(
                        "transition"
                    ),
                    current_changed_entities=(
                        current_changed_entities
                    ),
                )
            )


            record.update({
                "REC_Age":
                    float(age),

                "REC_LastFailureAge":
                    float(
                        last_failure_age
                    ),

                "REC_LastTransitionAge":
                    float(
                        last_transition_age
                    ),

                "REC_RecentAvgExeTime":
                    float(
                        recent_history[
                            "duration"
                        ].mean()
                    ),

                "REC_RecentMaxExeTime":
                    float(
                        recent_history[
                            "duration"
                        ].max()
                    ),

                "REC_RecentFailRate":
                    recent_fail_rate,

                "REC_RecentAssertRate":
                    recent_assert_rate,

                "REC_RecentExcRate":
                    recent_exc_rate,

                "REC_RecentTransitionRate":
                    recent_transition_rate,

                "REC_TotalAvgExeTime":
                    float(
                        history[
                            "duration"
                        ].mean()
                    ),

                "REC_TotalMaxExeTime":
                    float(
                        history[
                            "duration"
                        ].max()
                    ),

                "REC_TotalFailRate":
                    total_fail_rate,

                "REC_TotalAssertRate":
                    total_assert_rate,

                "REC_TotalExcRate":
                    total_exc_rate,

                "REC_TotalTransitionRate":
                    total_transition_rate,

                "REC_LastVerdict":
                    float(
                        recent_history.iloc[
                            -1
                        ][
                            "verdict"
                        ]
                    ),

                "REC_LastExeTime":
                    float(
                        recent_history.iloc[
                            -1
                        ][
                            "duration"
                        ]
                    ),

                "REC_MaxTestFileFailRate":
                    max_test_file_fail_rate,

                "REC_MaxTestFileTransitionRate":
                    max_test_file_transition_rate,
            })


            reconstructed_records.append(
                record
            )


    reconstructed = pd.DataFrame(
        reconstructed_records
    )


    if len(reconstructed) != len(
        requested
    ):
        raise AssertionError(
            "REC reconstruction row-count mismatch.\n"
            f"Requested: {len(requested)}\n"
            f"Reconstructed: {len(reconstructed)}"
        )


    if reconstructed.duplicated(
        subset=[
            "Build",
            "Test",
        ]
    ).any():
        raise AssertionError(
            "REC reconstruction produced duplicate "
            "Build-Test pairs."
        )


    return reconstructed


# ---------------------------------------------------------
# 7. Alignment helper for later noisy conditions
# ---------------------------------------------------------

def align_reconstructed_rec_features(
    reconstructed,
    requested_rows,
    columns=None,
):
    """
    Align reconstructed REC values to the exact order
    of requested model-ready rows.
    """

    if columns is None:
        columns = (
            REC_FEATURE_COLUMNS
        )


    requested = (
        requested_rows[
            [
                "Build",
                "Test",
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )


    requested["Build"] = pd.to_numeric(
        requested["Build"],
        errors="raise",
    ).astype("int64")


    requested["Test"] = (
        requested["Test"]
        .astype(str)
    )


    requested[
        "_requested_order"
    ] = np.arange(
        len(requested),
        dtype=np.int64,
    )


    reconstructed_subset = (
        reconstructed[
            [
                "Build",
                "Test",
                *columns,
            ]
        ]
        .copy()
    )


    reconstructed_subset[
        "Build"
    ] = pd.to_numeric(
        reconstructed_subset[
            "Build"
        ],
        errors="raise",
    ).astype("int64")


    reconstructed_subset[
        "Test"
    ] = (
        reconstructed_subset[
            "Test"
        ]
        .astype(str)
    )


    if reconstructed_subset.duplicated(
        subset=[
            "Build",
            "Test",
        ]
    ).any():
        raise AssertionError(
            "Reconstructed REC data contains "
            "duplicate Build-Test pairs."
        )


    aligned = (
        requested
        .merge(
            reconstructed_subset,
            on=[
                "Build",
                "Test",
            ],
            how="left",
            validate="one_to_one",
        )
        .sort_values(
            "_requested_order",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    if aligned[
        columns
    ].isna().all(
        axis=1
    ).any():
        raise AssertionError(
            "At least one requested Build-Test pair "
            "received no reconstructed REC features."
        )


    return aligned[
        columns
    ].copy()


# ---------------------------------------------------------
# 8. Reconstruct all 19 clean REC features
# ---------------------------------------------------------

requested_model_rows = (
    dataset_with_order[
        [
            "Build",
            "Test",
        ]
    ]
    .copy()
)


rec_clean_reconstructed = (
    reconstruct_rec_features(
        execution_history=exe_for_rec,
        requested_rows=requested_model_rows,
        recent_window=RECENT_WINDOW,
    )
)


aligned_reconstructed_rec = (
    align_reconstructed_rec_features(
        reconstructed=(
            rec_clean_reconstructed
        ),
        requested_rows=(
            requested_model_rows
        ),
        columns=REC_FEATURE_COLUMNS,
    )
)


if len(
    rec_clean_reconstructed
) != 17291:
    raise AssertionError(
        "Expected 17,291 reconstructed rows."
    )


# ---------------------------------------------------------
# 9. Compare against original TCP-CI values
# ---------------------------------------------------------

comparison = (
    dataset_with_order[
        [
            "Build",
            "Test",
            "build_order",
            "partition",
            *REC_FEATURE_COLUMNS,
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


comparison[
    "Build"
] = pd.to_numeric(
    comparison["Build"],
    errors="raise",
).astype("int64")


comparison[
    "Test"
] = (
    comparison["Test"]
    .astype(str)
)


for feature in REC_FEATURE_COLUMNS:
    comparison[
        f"{feature}_reconstructed"
    ] = (
        aligned_reconstructed_rec[
            feature
        ]
        .to_numpy(dtype=float)
    )


comparison_records = []
mismatch_frames = []


for feature in REC_FEATURE_COLUMNS:
    original_values = pd.to_numeric(
        comparison[feature],
        errors="coerce",
    ).to_numpy(dtype=float)


    reconstructed_values = pd.to_numeric(
        comparison[
            f"{feature}_reconstructed"
        ],
        errors="coerce",
    ).to_numpy(dtype=float)


    matching_mask = np.isclose(
        original_values,
        reconstructed_values,
        rtol=1e-9,
        atol=1e-9,
        equal_nan=True,
    )


    mismatch_mask = (
        ~matching_mask
    )


    training_mask = (
        comparison[
            "partition"
        ].to_numpy()
        == "training"
    )


    evaluation_mask = (
        comparison[
            "partition"
        ].to_numpy()
        == "evaluation"
    )


    unmatched_build_mask = (
        comparison[
            "Build"
        ].to_numpy(dtype=int)
        == UNMATCHED_BUILD_ID
    )


    absolute_differences = np.abs(
        original_values
        - reconstructed_values
    )


    finite_differences = (
        absolute_differences[
            np.isfinite(
                absolute_differences
            )
        ]
    )


    maximum_difference = (
        float(
            finite_differences.max()
        )
        if len(
            finite_differences
        ) > 0
        else 0.0
    )


    comparison_records.append({
        "Feature":
            feature,

        "FeaturePolicy":
            (
                "RECOMPUTE_AFTER_NOISE"
                if feature in (
                    VERDICT_DEPENDENT_REC_COLUMNS
                )
                else "PRESERVE_ORIGINAL"
            ),

        "ModelRows":
            int(
                len(comparison)
            ),

        "MatchingRows":
            int(
                matching_mask.sum()
            ),

        "MismatchingRows":
            int(
                mismatch_mask.sum()
            ),

        "TrainingMismatches":
            int(
                (
                    mismatch_mask
                    & training_mask
                ).sum()
            ),

        "EvaluationMismatches":
            int(
                (
                    mismatch_mask
                    & evaluation_mask
                ).sum()
            ),

        "UnmatchedBuildMismatches":
            int(
                (
                    mismatch_mask
                    & unmatched_build_mask
                ).sum()
            ),

        "MaximumAbsoluteDifference":
            maximum_difference,
    })


    if mismatch_mask.any():
        feature_mismatches = (
            comparison.loc[
                mismatch_mask,
                [
                    "Build",
                    "Test",
                    "build_order",
                    "partition",
                    feature,
                    f"{feature}_reconstructed",
                ]
            ]
            .copy()
            .rename(
                columns={
                    feature:
                        "OriginalValue",

                    f"{feature}_reconstructed":
                        "ReconstructedValue",
                }
            )
        )


        feature_mismatches.insert(
            0,
            "Feature",
            feature,
        )


        feature_mismatches[
            "FeaturePolicy"
        ] = (
            "RECOMPUTE_AFTER_NOISE"
            if feature in (
                VERDICT_DEPENDENT_REC_COLUMNS
            )
            else "PRESERVE_ORIGINAL"
        )


        mismatch_frames.append(
            feature_mismatches
        )


comparison_summary = pd.DataFrame(
    comparison_records
)


if mismatch_frames:
    mismatch_details = pd.concat(
        mismatch_frames,
        ignore_index=True,
    )

else:
    mismatch_details = pd.DataFrame(
        columns=[
            "Feature",
            "Build",
            "Test",
            "build_order",
            "partition",
            "OriginalValue",
            "ReconstructedValue",
            "FeaturePolicy",
        ]
    )


# ---------------------------------------------------------
# 10. Apply acceptance rules
# ---------------------------------------------------------

dependent_summary = (
    comparison_summary[
        comparison_summary[
            "Feature"
        ].isin(
            VERDICT_DEPENDENT_REC_COLUMNS
        )
    ]
    .copy()
)


independent_summary = (
    comparison_summary[
        comparison_summary[
            "Feature"
        ].isin(
            NOISE_INDEPENDENT_REC_COLUMNS
        )
    ]
    .copy()
)


verdict_dependent_mismatches = int(
    dependent_summary[
        "MismatchingRows"
    ].sum()
)


noise_independent_mismatches = int(
    independent_summary[
        "MismatchingRows"
    ].sum()
)


unmatched_build_dependent_mismatches = int(
    dependent_summary[
        "UnmatchedBuildMismatches"
    ].sum()
)


evaluation_dependent_mismatches = int(
    dependent_summary[
        "EvaluationMismatches"
    ].sum()
)


dependent_value_count = int(
    len(
        dataset_with_order
    )
    * len(
        VERDICT_DEPENDENT_REC_COLUMNS
    )
)


independent_value_count = int(
    len(
        dataset_with_order
    )
    * len(
        NOISE_INDEPENDENT_REC_COLUMNS
    )
)


all_rec_value_count = int(
    len(
        dataset_with_order
    )
    * len(
        REC_FEATURE_COLUMNS
    )
)


dependent_validation_passed = bool(
    verdict_dependent_mismatches == 0
)


unmatched_commit_validation_passed = bool(
    unmatched_build_dependent_mismatches == 0
)


evaluation_validation_passed = bool(
    evaluation_dependent_mismatches == 0
)


# ---------------------------------------------------------
# 11. Construct canonical 19-feature matrix
# ---------------------------------------------------------

canonical_clean_rec_matrix = (
    dataset_with_order[
        [
            "Build",
            "Test",
            "build_order",
            "partition",
            *REC_FEATURE_COLUMNS,
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


# Replace only the 13 verdict-dependent features.
# The six noise-independent values remain exactly as
# supplied by TCP-CI.
for feature in (
    VERDICT_DEPENDENT_REC_COLUMNS
):
    canonical_clean_rec_matrix[
        feature
    ] = (
        aligned_reconstructed_rec[
            feature
        ]
        .to_numpy(dtype=float)
    )


canonical_rec_mismatches = 0


for feature in REC_FEATURE_COLUMNS:
    canonical_values = pd.to_numeric(
        canonical_clean_rec_matrix[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)


    original_values = pd.to_numeric(
        dataset_with_order[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)


    equal_mask = np.isclose(
        canonical_values,
        original_values,
        rtol=1e-9,
        atol=1e-9,
        equal_nan=True,
    )


    canonical_rec_mismatches += int(
        (
            ~equal_mask
        ).sum()
    )


canonical_matrix_validation_passed = bool(
    canonical_rec_mismatches == 0
)


# These aliases are used by later experiment helpers.
canonical_clean_training_data = (
    clean_training_data
    .copy()
    .reset_index(drop=True)
)


canonical_clean_evaluation_data = (
    clean_evaluation_data
    .copy()
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 12. Save permanent validation artefacts
# ---------------------------------------------------------

REC_COMPARISON_SUMMARY_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_clean_rec_feature_comparison.csv"
)


REC_MISMATCH_DETAILS_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_clean_rec_mismatch_details.csv"
)


REC_RECONSTRUCTED_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_clean_rec_reconstructed_all19.parquet"
)


CANONICAL_REC_MATRIX_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_canonical_clean_rec_matrix.parquet"
)


REC_VALIDATION_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_clean_rec_validation_report.json"
)


comparison_summary.to_csv(
    REC_COMPARISON_SUMMARY_PATH,
    index=False,
)


mismatch_details.to_csv(
    REC_MISMATCH_DETAILS_PATH,
    index=False,
)


rec_clean_reconstructed.to_parquet(
    REC_RECONSTRUCTED_PATH,
    index=False,
)


canonical_clean_rec_matrix.to_parquet(
    CANONICAL_REC_MATRIX_PATH,
    index=False,
)


validation_status = (
    "PASS"
    if (
        dependent_validation_passed
        and unmatched_commit_validation_passed
        and evaluation_validation_passed
        and canonical_matrix_validation_passed
    )
    else "FAIL"
)


rec_validation_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        validation_status,

    "RecentWindow":
        RECENT_WINDOW,

    "ModelReadyRows":
        int(
            len(
                dataset_with_order
            )
        ),

    "ReconstructedRows":
        int(
            len(
                rec_clean_reconstructed
            )
        ),

    "RECFeatureCount":
        int(
            len(
                REC_FEATURE_COLUMNS
            )
        ),

    "VerdictDependentRECFeatures":
        VERDICT_DEPENDENT_REC_COLUMNS,

    "NoiseIndependentRECFeatures":
        NOISE_INDEPENDENT_REC_COLUMNS,

    "VerdictDependentValuesChecked":
        dependent_value_count,

    "NoiseIndependentValuesAudited":
        independent_value_count,

    "CanonicalRECValuesChecked":
        all_rec_value_count,

    "VerdictDependentMismatches":
        verdict_dependent_mismatches,

    "NoiseIndependentMismatches":
        noise_independent_mismatches,

    "EvaluationVerdictDependentMismatches":
        evaluation_dependent_mismatches,

    "UnmatchedBuild":
        UNMATCHED_BUILD_ID,

    "UnmatchedBuildChangedEntities":
        int(
            len(
                clean_changed_entities_by_build[
                    UNMATCHED_BUILD_ID
                ]
            )
        ),

    "UnmatchedBuildVerdictDependentMismatches":
        unmatched_build_dependent_mismatches,

    "CanonicalMatrixMismatches":
        canonical_rec_mismatches,

    "VerdictDependentValidationPassed":
        dependent_validation_passed,

    "UnmatchedCommitValidationPassed":
        unmatched_commit_validation_passed,

    "EvaluationValidationPassed":
        evaluation_validation_passed,

    "CanonicalMatrixValidationPassed":
        canonical_matrix_validation_passed,

    "NoiseIndependentPolicy":
        (
            "Preserve the original TCP-CI values. "
            "Verdict flipping cannot affect age or "
            "execution-duration-only REC features."
        ),

    "EntityMappingDecision":
        (
            "ACCEPTED_AFTER_CLEAN_REC_VALIDATION"
            if unmatched_commit_validation_passed
            and dependent_validation_passed
            else "NOT_ACCEPTED"
        ),

    "ComparisonSummary":
        str(
            REC_COMPARISON_SUMMARY_PATH
        ),

    "MismatchDetails":
        str(
            REC_MISMATCH_DETAILS_PATH
        ),

    "ReconstructedAll19":
        str(
            REC_RECONSTRUCTED_PATH
        ),

    "CanonicalCleanRECMatrix":
        str(
            CANONICAL_REC_MATRIX_PATH
        ),

    "CompletedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
}


with open(
    REC_VALIDATION_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        rec_validation_report,
        report_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 13. Update checkpoint only after successful validation
# ---------------------------------------------------------

if validation_status == "PASS":
    with open(
        PROJECT_6_SELECTION_CHECKPOINT,
        "r",
        encoding="utf-8",
    ) as checkpoint_file:
        checkpoint = json.load(
            checkpoint_file
        )


    checkpoint.update({
        "Status":
            "CLEAN_REC_VALIDATION_PASSED",

        "EntityMappingStatus":
            (
                "ACCEPTED_AFTER_CLEAN_REC_VALIDATION"
            ),

        "UnmatchedCommitBuild":
            UNMATCHED_BUILD_ID,

        "UnmatchedCommitAccepted":
            True,

        "VerdictDependentRECFeatures":
            VERDICT_DEPENDENT_REC_COLUMNS,

        "NoiseIndependentRECFeatures":
            NOISE_INDEPENDENT_REC_COLUMNS,

        "VerdictDependentValuesChecked":
            dependent_value_count,

        "VerdictDependentMismatches":
            verdict_dependent_mismatches,

        "NoiseIndependentMismatches":
            noise_independent_mismatches,

        "UnmatchedBuildVerdictDependentMismatches":
            unmatched_build_dependent_mismatches,

        "CanonicalRECMatrixMismatches":
            canonical_rec_mismatches,

        "CleanRECValidationReport":
            str(
                REC_VALIDATION_REPORT_PATH
            ),

        "UpdatedAtUTC":
            pd.Timestamp.utcnow().isoformat(),
    })


    with open(
        PROJECT_6_SELECTION_CHECKPOINT,
        "w",
        encoding="utf-8",
    ) as checkpoint_file:
        json.dump(
            checkpoint,
            checkpoint_file,
            indent=2,
            default=str,
        )


# ---------------------------------------------------------
# 14. Compact final output
# ---------------------------------------------------------

print(
    "\n=== PROJECT 6 STEP 6 RESULT ==="
)


print(
    "\nREC reconstruction:"
)

print(
    "Model-ready rows:",
    len(
        dataset_with_order
    )
)

print(
    "Reconstructed rows:",
    len(
        rec_clean_reconstructed
    )
)

print(
    "REC features:",
    len(
        REC_FEATURE_COLUMNS
    )
)

print(
    "Verdict-dependent REC features:",
    len(
        VERDICT_DEPENDENT_REC_COLUMNS
    )
)

print(
    "Noise-independent REC features:",
    len(
        NOISE_INDEPENDENT_REC_COLUMNS
    )
)


print(
    "\nVerdict-dependent validation:"
)

print(
    "Values checked:",
    dependent_value_count
)

print(
    "Total mismatches:",
    verdict_dependent_mismatches
)

print(
    "Evaluation mismatches:",
    evaluation_dependent_mismatches
)

print(
    "Unmatched-build mismatches:",
    unmatched_build_dependent_mismatches
)


print(
    "\nNoise-independent audit:"
)

print(
    "Values audited:",
    independent_value_count
)

print(
    "Mismatches:",
    noise_independent_mismatches
)

print(
    "Policy:",
    "preserve original TCP-CI values"
)


print(
    "\nCanonical 19-feature matrix:"
)

print(
    "Values checked:",
    all_rec_value_count
)

print(
    "Mismatches:",
    canonical_rec_mismatches
)


print(
    "\nPer-feature comparison:"
)

display(
    comparison_summary
)


if len(
    mismatch_details
) > 0:
    print(
        "\nFirst recorded mismatches:"
    )

    display(
        mismatch_details.head(30)
    )

else:
    print(
        "\nRecorded mismatches:",
        "None"
    )


print(
    "\nValidation report:"
)

print(
    REC_VALIDATION_REPORT_PATH
)


print(
    "\nValidation status:",
    validation_status
)


if validation_status != "PASS":
    raise AssertionError(
        "Project 6 clean REC validation failed. "
        "Do not proceed to noise injection."
    )


print(
    "\nSUCCESS: All 13 verdict-dependent REC features "
    "match the clean TCP-CI dataset."
)

print(
    "SUCCESS: The unmatched build caused zero "
    "verdict-dependent REC mismatches."
)

print(
    "SUCCESS: The unmatched commit mapping is accepted "
    "without inventing changed entities."
)

print(
    "SUCCESS: The six noise-independent REC features "
    "will be preserved from TCP-CI."
)

print(
    "SUCCESS: The canonical 19-feature REC matrix "
    "matches the original dataset exactly."
)

print(
    "SUCCESS: Project 6 is ready for canonical "
    "noise-injection and helper validation."
)

=== PROJECT 6 STEP 6: CLEAN REC VALIDATION ===

=== PROJECT 6 STEP 6 RESULT ===

REC reconstruction:
Model-ready rows: 17291
Reconstructed rows: 17291
REC features: 19
Verdict-dependent REC features: 13
Noise-independent REC features: 6

Verdict-dependent validation:
Values checked: 224783
Total mismatches: 345
Evaluation mismatches: 279
Unmatched-build mismatches: 0

Noise-independent audit:
Values audited: 103746
Mismatches: 2105
Policy: preserve original TCP-CI values

Canonical 19-feature matrix:
Values checked: 328529
Mismatches: 345

Per-feature comparison:


,Feature,FeaturePolicy,ModelRows,MatchingRows,MismatchingRows,TrainingMismatches,EvaluationMismatches,UnmatchedBuildMismatches,MaximumAbsoluteDifference
0,REC_Age,PRESERVE_ORIGINAL,17291,16718,573,107,466,0,1.000000e+00
1,REC_LastFailureAge,RECOMPUTE_AFTER_NOISE,17291,17233,58,14,44,0,1.000000e+00
2,REC_LastTransitionAge,RECOMPUTE_AFTER_NOISE,17291,17234,57,12,45,0,1.000000e+00
3,REC_RecentAvgExeTime,PRESERVE_ORIGINAL,17291,16813,478,54,424,0,1.090633e+04
4,REC_RecentMaxExeTime,PRESERVE_ORIGINAL,17291,17151,140,18,122,0,1.254300e+04
5,REC_RecentFailRate,RECOMPUTE_AFTER_NOISE,17291,17290,1,0,1,0,1.666667e-01
6,REC_RecentAssertRate,RECOMPUTE_AFTER_NOISE,17291,17290,1,0,1,0,1.666667e-01
7,REC_RecentExcRate,RECOMPUTE_AFTER_NOISE,17291,17291,0,0,0,0,5.551115e-17
8,REC_RecentTransitionRate,RECOMPUTE_AFTER_NOISE,17291,17286,5,4,1,0,3.333333e-01
9,REC_TotalAvgExeTime,PRESERVE_ORIGINAL,17291,16896,395,51,344,0,6.435911e+02



First recorded mismatches:


,Feature,Build,Test,build_order,partition,OriginalValue,ReconstructedValue,FeaturePolicy
0,REC_Age,6843283,1214,127,training,127.0,126.0,PRESERVE_ORIGINAL
1,REC_Age,6843283,1215,127,training,127.0,126.0,PRESERVE_ORIGINAL
2,REC_Age,6843283,1234,127,training,127.0,126.0,PRESERVE_ORIGINAL
3,REC_Age,6843283,1382,127,training,124.0,123.0,PRESERVE_ORIGINAL
4,REC_Age,6843283,139,127,training,127.0,126.0,PRESERVE_ORIGINAL
5,REC_Age,6843283,140,127,training,127.0,126.0,PRESERVE_ORIGINAL
6,REC_Age,6843283,141,127,training,127.0,126.0,PRESERVE_ORIGINAL
7,REC_Age,6843283,142,127,training,127.0,126.0,PRESERVE_ORIGINAL
8,REC_Age,6843283,143,127,training,127.0,126.0,PRESERVE_ORIGINAL
9,REC_Age,6843283,144,127,training,127.0,126.0,PRESERVE_ORIGINAL



Validation report:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/eclipse__jetty.project/jetty_preflight/jetty_clean_rec_validation_report.json

Validation status: FAIL


AssertionError: Project 6 clean REC validation failed. Do not proceed to noise injection.

In [ ]:
# =========================================================
# PROJECT 6 — STEP 6 CORRECTION
# RESTORE UPSTREAM TIE ORDER AND REVALIDATE CLEAN REC
#
# Correction:
#   Chronology = started_at ascending
#   Equal timestamps = original builds.csv row order
#
# The previous implementation incorrectly used build ID
# ascending to break equal-timestamp ties.
# =========================================================

from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd
from IPython.display import display


# ---------------------------------------------------------
# 1. Confirm that the failed Step 6 defined its helpers
# ---------------------------------------------------------

required_objects = [
    "builds_raw",
    "exe_raw",
    "dataset_raw",
    "build_entity_map",
    "changed_entities_by_build",
    "reconstruct_rec_features",
    "align_reconstructed_rec_features",
    "REC_FEATURE_COLUMNS",
    "VERDICT_DEPENDENT_REC_COLUMNS",
    "NOISE_INDEPENDENT_REC_COLUMNS",
    "PROJECT_PREFLIGHT_DIRECTORY",
    "PROJECT_6_SELECTION_CHECKPOINT",
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nThe runtime recovery cell and the original "
        "Step 6 cell must have been run in this runtime."
    )


print(
    "=== PROJECT 6 STEP 6 CORRECTION: "
    "UPSTREAM TIE ORDER ==="
)


# ---------------------------------------------------------
# 2. Preserve the failed validation report
# ---------------------------------------------------------

REC_VALIDATION_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_clean_rec_validation_report.json"
)


FAILED_VALIDATION_ARCHIVE_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / (
        "jetty_clean_rec_validation_report_"
        "failed_build_id_tiebreak.json"
    )
)


if (
    REC_VALIDATION_REPORT_PATH.exists()
    and not FAILED_VALIDATION_ARCHIVE_PATH.exists()
):
    try:
        with open(
            REC_VALIDATION_REPORT_PATH,
            "r",
            encoding="utf-8",
        ) as failed_report_file:
            previous_report = json.load(
                failed_report_file
            )


        if previous_report.get(
            "Status"
        ) == "FAIL":
            shutil.copy2(
                REC_VALIDATION_REPORT_PATH,
                FAILED_VALIDATION_ARCHIVE_PATH,
            )

    except Exception:
        pass


# ---------------------------------------------------------
# 3. Reconstruct chronology using upstream source order
# ---------------------------------------------------------

source_builds = pd.DataFrame({
    "source_row":
        np.arange(
            len(builds_raw),
            dtype=np.int64,
        ),

    "build":
        pd.to_numeric(
            builds_raw["id"],
            errors="raise",
        ).astype("int64"),

    "commits":
        builds_raw[
            "commits"
        ].astype(str),

    "timestamp":
        pd.to_datetime(
            builds_raw["started_at"],
            errors="raise",
            utc=True,
        ),
})


if len(source_builds) != 192:
    raise AssertionError(
        "Expected 192 Jetty builds."
    )


if source_builds[
    "build"
].duplicated().any():
    raise AssertionError(
        "Duplicate build IDs were found."
    )


# Correct deterministic chronology:
# timestamp first, original CSV sequence for equal times.
corrected_builds = (
    source_builds
    .sort_values(
        [
            "timestamp",
            "source_row",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


corrected_builds[
    "build_order"
] = np.arange(
    1,
    len(corrected_builds) + 1,
    dtype=np.int64,
)


TRAINING_BUILD_COUNT = int(
    np.floor(
        0.75
        * len(corrected_builds)
    )
)


corrected_builds[
    "partition"
] = np.where(
    corrected_builds[
        "build_order"
    ] <= TRAINING_BUILD_COUNT,
    "training",
    "evaluation",
)


if TRAINING_BUILD_COUNT != 144:
    raise AssertionError(
        "Corrected chronology did not produce "
        "144 training builds."
    )


if (
    corrected_builds[
        "partition"
    ] == "evaluation"
).sum() != 48:
    raise AssertionError(
        "Corrected chronology did not produce "
        "48 evaluation builds."
    )


# ---------------------------------------------------------
# 4. Audit the tied timestamps
# ---------------------------------------------------------

timestamp_group_sizes = (
    corrected_builds
    .groupby(
        "timestamp"
    )[
        "build"
    ]
    .transform("size")
)


tied_timestamp_builds = (
    corrected_builds[
        timestamp_group_sizes > 1
    ][
        [
            "source_row",
            "build",
            "timestamp",
            "build_order",
            "partition",
        ]
    ]
    .copy()
)


id_tiebreak_order = (
    source_builds
    .sort_values(
        [
            "timestamp",
            "build",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


id_tiebreak_order[
    "OldBuildOrder"
] = np.arange(
    1,
    len(id_tiebreak_order) + 1,
    dtype=np.int64,
)


old_order_lookup = dict(
    zip(
        id_tiebreak_order[
            "build"
        ].astype(int),

        id_tiebreak_order[
            "OldBuildOrder"
        ].astype(int),
    )
)


chronology_order_comparison = (
    corrected_builds[
        [
            "build",
            "timestamp",
            "source_row",
            "build_order",
            "partition",
        ]
    ]
    .copy()
)


chronology_order_comparison[
    "OldBuildIDTieOrder"
] = (
    chronology_order_comparison[
        "build"
    ]
    .map(
        old_order_lookup
    )
    .astype(int)
)


chronology_order_comparison[
    "OrderChanged"
] = (
    chronology_order_comparison[
        "build_order"
    ]
    != chronology_order_comparison[
        "OldBuildIDTieOrder"
    ]
)


changed_order_builds = (
    chronology_order_comparison[
        chronology_order_comparison[
            "OrderChanged"
        ]
    ]
    .copy()
)


# ---------------------------------------------------------
# 5. Construct corrected build lookups
# ---------------------------------------------------------

corrected_build_order_lookup = dict(
    zip(
        corrected_builds[
            "build"
        ].astype(int),

        corrected_builds[
            "build_order"
        ].astype(int),
    )
)


corrected_partition_lookup = dict(
    zip(
        corrected_builds[
            "build"
        ].astype(int),

        corrected_builds[
            "partition"
        ].astype(str),
    )
)


# ---------------------------------------------------------
# 6. Rebuild raw execution history
# ---------------------------------------------------------

corrected_exe_for_rec = (
    exe_raw
    .copy()
)


corrected_exe_for_rec[
    "build"
] = pd.to_numeric(
    corrected_exe_for_rec[
        "build"
    ],
    errors="raise",
).astype("int64")


corrected_exe_for_rec[
    "test"
] = (
    corrected_exe_for_rec[
        "test"
    ]
    .astype(str)
)


corrected_exe_for_rec[
    "verdict"
] = pd.to_numeric(
    corrected_exe_for_rec[
        "verdict"
    ],
    errors="raise",
).astype("int8")


corrected_exe_for_rec[
    "duration"
] = pd.to_numeric(
    corrected_exe_for_rec[
        "duration"
    ],
    errors="raise",
).astype(float)


if "job" not in corrected_exe_for_rec.columns:
    corrected_exe_for_rec[
        "job"
    ] = ""


corrected_exe_for_rec[
    "build_order"
] = (
    corrected_exe_for_rec[
        "build"
    ]
    .map(
        corrected_build_order_lookup
    )
)


corrected_exe_for_rec[
    "partition"
] = (
    corrected_exe_for_rec[
        "build"
    ]
    .map(
        corrected_partition_lookup
    )
)


if corrected_exe_for_rec[
    "build_order"
].isna().any():
    raise AssertionError(
        "Some raw executions could not be assigned "
        "to the corrected chronology."
    )


corrected_exe_for_rec[
    "build_order"
] = corrected_exe_for_rec[
    "build_order"
].astype("int64")


corrected_exe_for_rec = (
    corrected_exe_for_rec
    .sort_values(
        [
            "build_order",
            "job",
            "test",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


corrected_clean_training_history = (
    corrected_exe_for_rec[
        corrected_exe_for_rec[
            "partition"
        ] == "training"
    ]
    .copy()
    .reset_index(drop=True)
)


corrected_clean_evaluation_history = (
    corrected_exe_for_rec[
        corrected_exe_for_rec[
            "partition"
        ] == "evaluation"
    ]
    .copy()
    .reset_index(drop=True)
)


if len(
    corrected_clean_training_history
) != 19845:
    raise AssertionError(
        "Corrected raw training count changed."
    )


if len(
    corrected_clean_evaluation_history
) != 6594:
    raise AssertionError(
        "Corrected raw evaluation count changed."
    )


# ---------------------------------------------------------
# 7. Rebuild model-ready data
# ---------------------------------------------------------

corrected_dataset = (
    dataset_raw
    .copy()
)


corrected_dataset[
    "Build"
] = pd.to_numeric(
    corrected_dataset[
        "Build"
    ],
    errors="raise",
).astype("int64")


corrected_dataset[
    "Test"
] = (
    corrected_dataset[
        "Test"
    ]
    .astype(str)
)


corrected_dataset[
    "Verdict"
] = pd.to_numeric(
    corrected_dataset[
        "Verdict"
    ],
    errors="raise",
).astype("int8")


corrected_dataset[
    "Duration"
] = pd.to_numeric(
    corrected_dataset[
        "Duration"
    ],
    errors="raise",
).astype(float)


corrected_dataset[
    "build_order"
] = (
    corrected_dataset[
        "Build"
    ]
    .map(
        corrected_build_order_lookup
    )
)


corrected_dataset[
    "partition"
] = (
    corrected_dataset[
        "Build"
    ]
    .map(
        corrected_partition_lookup
    )
)


if corrected_dataset[
    "build_order"
].isna().any():
    raise AssertionError(
        "Some model-ready rows could not be assigned "
        "to the corrected chronology."
    )


corrected_dataset[
    "build_order"
] = corrected_dataset[
    "build_order"
].astype("int64")


corrected_dataset_with_order = (
    corrected_dataset
    .sort_values(
        [
            "build_order",
            "Build",
            "Test",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


corrected_clean_training_data = (
    corrected_dataset_with_order[
        corrected_dataset_with_order[
            "partition"
        ] == "training"
    ]
    .copy()
    .reset_index(drop=True)
)


corrected_clean_evaluation_data = (
    corrected_dataset_with_order[
        corrected_dataset_with_order[
            "partition"
        ] == "evaluation"
    ]
    .copy()
    .reset_index(drop=True)
)


if len(
    corrected_clean_training_data
) != 13212:
    raise AssertionError(
        "Corrected model-training count changed."
    )


if len(
    corrected_clean_evaluation_data
) != 4079:
    raise AssertionError(
        "Corrected model-evaluation count changed."
    )


# ---------------------------------------------------------
# 8. Restore entity-history globals used by REC helpers
# ---------------------------------------------------------

entity_build_pairs = (
    build_entity_map[
        [
            "Build",
            "EntityID",
        ]
    ]
    .drop_duplicates()
    .copy()
)


entity_build_pairs[
    "Build"
] = pd.to_numeric(
    entity_build_pairs[
        "Build"
    ],
    errors="raise",
).astype("int64")


entity_build_pairs[
    "EntityID"
] = pd.to_numeric(
    entity_build_pairs[
        "EntityID"
    ],
    errors="raise",
).astype("int64")


entity_changed_builds = (
    entity_build_pairs
    .groupby(
        "EntityID"
    )[
        "Build"
    ]
    .apply(
        lambda values:
            set(
                values.astype(int)
            )
    )
    .to_dict()
)


clean_changed_entities_by_build = {
    int(build_id):
        set(
            int(entity_id)
            for entity_id in (
                changed_entities_by_build.get(
                    int(build_id),
                    set(),
                )
            )
        )

    for build_id
    in corrected_builds[
        "build"
    ].astype(int)
}


UNMATCHED_BUILD_ID = 6806945


if clean_changed_entities_by_build[
    UNMATCHED_BUILD_ID
] != set():
    raise AssertionError(
        "The unmatched build should still have an "
        "empty changed-entity set."
    )


# ---------------------------------------------------------
# 9. Correct global build positions used by the helper
# ---------------------------------------------------------

corrected_ordered_execution_builds = (
    corrected_exe_for_rec[
        [
            "build",
            "build_order",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "build_order",
            "build",
        ],
        kind="mergesort",
    )[
        "build"
    ]
    .astype(int)
    .tolist()
)


corrected_global_build_position = {
    int(build_id):
        position

    for position, build_id
    in enumerate(
        corrected_ordered_execution_builds
    )
}


if len(
    corrected_global_build_position
) != 192:
    raise AssertionError(
        "Corrected global build-position map does "
        "not contain 192 builds."
    )


# The reconstruction function reads this global.
previous_global_build_position = (
    global_build_position.copy()
    if "global_build_position" in globals()
    else None
)


global_build_position = (
    corrected_global_build_position
)


# ---------------------------------------------------------
# 10. Reconstruct all clean REC values again
# ---------------------------------------------------------

requested_rows = (
    corrected_dataset_with_order[
        [
            "Build",
            "Test",
        ]
    ]
    .copy()
)


corrected_rec_reconstructed = (
    reconstruct_rec_features(
        execution_history=(
            corrected_exe_for_rec
        ),
        requested_rows=requested_rows,
        recent_window=RECENT_WINDOW,
    )
)


corrected_aligned_rec = (
    align_reconstructed_rec_features(
        reconstructed=(
            corrected_rec_reconstructed
        ),
        requested_rows=requested_rows,
        columns=REC_FEATURE_COLUMNS,
    )
)


if len(
    corrected_rec_reconstructed
) != 17291:
    if (
        previous_global_build_position
        is not None
    ):
        global_build_position = (
            previous_global_build_position
        )

    raise AssertionError(
        "Corrected REC reconstruction row count "
        "is not 17,291."
    )


# ---------------------------------------------------------
# 11. Compare every REC feature
# ---------------------------------------------------------

comparison_records = []
mismatch_frames = []


for feature in REC_FEATURE_COLUMNS:
    original_values = pd.to_numeric(
        corrected_dataset_with_order[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)


    reconstructed_values = pd.to_numeric(
        corrected_aligned_rec[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)


    matching_mask = np.isclose(
        original_values,
        reconstructed_values,
        rtol=1e-9,
        atol=1e-9,
        equal_nan=True,
    )


    mismatch_mask = (
        ~matching_mask
    )


    training_mask = (
        corrected_dataset_with_order[
            "partition"
        ].to_numpy()
        == "training"
    )


    evaluation_mask = (
        corrected_dataset_with_order[
            "partition"
        ].to_numpy()
        == "evaluation"
    )


    unmatched_build_mask = (
        corrected_dataset_with_order[
            "Build"
        ].to_numpy(dtype=int)
        == UNMATCHED_BUILD_ID
    )


    differences = np.abs(
        original_values
        - reconstructed_values
    )


    finite_differences = (
        differences[
            np.isfinite(
                differences
            )
        ]
    )


    maximum_difference = (
        float(
            finite_differences.max()
        )
        if len(
            finite_differences
        ) > 0
        else 0.0
    )


    policy = (
        "RECOMPUTE_AFTER_NOISE"
        if feature in (
            VERDICT_DEPENDENT_REC_COLUMNS
        )
        else "PRESERVE_ORIGINAL"
    )


    comparison_records.append({
        "Feature":
            feature,

        "FeaturePolicy":
            policy,

        "ModelRows":
            int(
                len(
                    corrected_dataset_with_order
                )
            ),

        "MatchingRows":
            int(
                matching_mask.sum()
            ),

        "MismatchingRows":
            int(
                mismatch_mask.sum()
            ),

        "TrainingMismatches":
            int(
                (
                    mismatch_mask
                    & training_mask
                ).sum()
            ),

        "EvaluationMismatches":
            int(
                (
                    mismatch_mask
                    & evaluation_mask
                ).sum()
            ),

        "UnmatchedBuildMismatches":
            int(
                (
                    mismatch_mask
                    & unmatched_build_mask
                ).sum()
            ),

        "MaximumAbsoluteDifference":
            maximum_difference,
    })


    if mismatch_mask.any():
        details = (
            corrected_dataset_with_order.loc[
                mismatch_mask,
                [
                    "Build",
                    "Test",
                    "build_order",
                    "partition",
                ]
            ]
            .copy()
        )


        details.insert(
            0,
            "Feature",
            feature,
        )


        details[
            "OriginalValue"
        ] = original_values[
            mismatch_mask
        ]


        details[
            "ReconstructedValue"
        ] = reconstructed_values[
            mismatch_mask
        ]


        details[
            "FeaturePolicy"
        ] = policy


        mismatch_frames.append(
            details
        )


corrected_comparison_summary = (
    pd.DataFrame(
        comparison_records
    )
)


if mismatch_frames:
    corrected_mismatch_details = (
        pd.concat(
            mismatch_frames,
            ignore_index=True,
        )
    )

else:
    corrected_mismatch_details = (
        pd.DataFrame(
            columns=[
                "Feature",
                "Build",
                "Test",
                "build_order",
                "partition",
                "OriginalValue",
                "ReconstructedValue",
                "FeaturePolicy",
            ]
        )
    )


dependent_summary = (
    corrected_comparison_summary[
        corrected_comparison_summary[
            "Feature"
        ].isin(
            VERDICT_DEPENDENT_REC_COLUMNS
        )
    ]
)


independent_summary = (
    corrected_comparison_summary[
        corrected_comparison_summary[
            "Feature"
        ].isin(
            NOISE_INDEPENDENT_REC_COLUMNS
        )
    ]
)


verdict_dependent_mismatches = int(
    dependent_summary[
        "MismatchingRows"
    ].sum()
)


noise_independent_mismatches = int(
    independent_summary[
        "MismatchingRows"
    ].sum()
)


evaluation_dependent_mismatches = int(
    dependent_summary[
        "EvaluationMismatches"
    ].sum()
)


unmatched_build_dependent_mismatches = int(
    dependent_summary[
        "UnmatchedBuildMismatches"
    ].sum()
)


# ---------------------------------------------------------
# 12. Construct canonical 19-feature matrix
# ---------------------------------------------------------

canonical_clean_rec_matrix = (
    corrected_dataset_with_order[
        [
            "Build",
            "Test",
            "build_order",
            "partition",
            *REC_FEATURE_COLUMNS,
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


# Replace only features affected by verdict noise.
for feature in (
    VERDICT_DEPENDENT_REC_COLUMNS
):
    canonical_clean_rec_matrix[
        feature
    ] = (
        corrected_aligned_rec[
            feature
        ]
        .to_numpy(dtype=float)
    )


canonical_rec_mismatches = 0


for feature in REC_FEATURE_COLUMNS:
    canonical_values = pd.to_numeric(
        canonical_clean_rec_matrix[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)


    original_values = pd.to_numeric(
        corrected_dataset_with_order[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)


    canonical_equal = np.isclose(
        canonical_values,
        original_values,
        rtol=1e-9,
        atol=1e-9,
        equal_nan=True,
    )


    canonical_rec_mismatches += int(
        (
            ~canonical_equal
        ).sum()
    )


validation_status = (
    "PASS"
    if (
        verdict_dependent_mismatches == 0
        and evaluation_dependent_mismatches == 0
        and unmatched_build_dependent_mismatches == 0
        and canonical_rec_mismatches == 0
    )
    else "FAIL"
)


# Do not commit the corrected chronology unless it passes.
if validation_status != "PASS":
    if (
        previous_global_build_position
        is not None
    ):
        global_build_position = (
            previous_global_build_position
        )


    print(
        "\nCorrected per-feature comparison:"
    )

    display(
        corrected_comparison_summary
    )


    raise AssertionError(
        "The source-row tie-order correction did not "
        "produce an exact verdict-dependent match. "
        "Do not continue."
    )


# ---------------------------------------------------------
# 13. Commit corrected variables to the runtime
# ---------------------------------------------------------

builds = corrected_builds.copy()


build_order_lookup = (
    corrected_build_order_lookup
)


build_partition_lookup = (
    corrected_partition_lookup
)


builds_checked = pd.DataFrame({
    "id":
        builds[
            "build"
        ].astype("int64"),

    "commits":
        builds[
            "commits"
        ].astype(str),

    "started_at":
        builds[
            "timestamp"
        ],

    "source_row":
        builds[
            "source_row"
        ].astype("int64"),

    "build_order":
        builds[
            "build_order"
        ].astype("int64"),

    "partition":
        builds[
            "partition"
        ].astype(str),
})


exe_for_rec = (
    corrected_exe_for_rec
)


clean_training_history = (
    corrected_clean_training_history
)


clean_evaluation_history = (
    corrected_clean_evaluation_history
)


dataset = (
    corrected_dataset
)


dataset_with_order = (
    corrected_dataset_with_order
)


clean_training_data = (
    corrected_clean_training_data
)


clean_evaluation_data = (
    corrected_clean_evaluation_data
)


canonical_clean_training_data = (
    clean_training_data
    .copy()
    .reset_index(drop=True)
)


canonical_clean_evaluation_data = (
    clean_evaluation_data
    .copy()
    .reset_index(drop=True)
)


ordered_execution_builds = (
    corrected_ordered_execution_builds
)


global_build_position = (
    corrected_global_build_position
)


rec_clean_reconstructed = (
    corrected_rec_reconstructed
)


aligned_reconstructed_rec = (
    corrected_aligned_rec
)


comparison_summary = (
    corrected_comparison_summary
)


mismatch_details = (
    corrected_mismatch_details
)


MODEL_FEATURE_COLUMNS = [
    column
    for column in dataset.columns
    if column not in {
        "Build",
        "Test",
        "Verdict",
        "Duration",
        "build_order",
        "partition",
    }
]


if len(
    MODEL_FEATURE_COLUMNS
) != 150:
    raise AssertionError(
        "Corrected runtime does not contain "
        "150 model predictors."
    )


# ---------------------------------------------------------
# 14. Correct the saved mapping audit build order
# ---------------------------------------------------------

BUILD_MAPPING_AUDIT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_build_mapping_audit.csv"
)


if BUILD_MAPPING_AUDIT_PATH.exists():
    build_mapping_audit = pd.read_csv(
        BUILD_MAPPING_AUDIT_PATH
    )


    build_mapping_audit[
        "Build"
    ] = pd.to_numeric(
        build_mapping_audit[
            "Build"
        ],
        errors="raise",
    ).astype("int64")


    build_mapping_audit[
        "BuildOrder"
    ] = (
        build_mapping_audit[
            "Build"
        ]
        .map(
            build_order_lookup
        )
        .astype("int64")
    )


    build_mapping_audit[
        "Partition"
    ] = (
        build_mapping_audit[
            "Build"
        ]
        .map(
            build_partition_lookup
        )
    )


    build_mapping_audit = (
        build_mapping_audit
        .sort_values(
            "BuildOrder",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    build_mapping_audit.to_csv(
        BUILD_MAPPING_AUDIT_PATH,
        index=False,
    )


# ---------------------------------------------------------
# 15. Save corrected validation artefacts
# ---------------------------------------------------------

CHRONOLOGY_CORRECTION_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_chronology_tie_order_correction.json"
)


CHRONOLOGY_COMPARISON_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_chronology_order_comparison.csv"
)


TIED_BUILDS_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_equal_timestamp_builds.csv"
)


BUILD_SPLIT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_chronological_build_split.csv"
)


REC_COMPARISON_SUMMARY_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_clean_rec_feature_comparison.csv"
)


REC_MISMATCH_DETAILS_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_clean_rec_mismatch_details.csv"
)


REC_RECONSTRUCTED_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_clean_rec_reconstructed_all19.parquet"
)


CANONICAL_REC_MATRIX_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_canonical_clean_rec_matrix.parquet"
)


chronology_order_comparison.to_csv(
    CHRONOLOGY_COMPARISON_PATH,
    index=False,
)


tied_timestamp_builds.to_csv(
    TIED_BUILDS_PATH,
    index=False,
)


builds.to_csv(
    BUILD_SPLIT_PATH,
    index=False,
)


comparison_summary.to_csv(
    REC_COMPARISON_SUMMARY_PATH,
    index=False,
)


mismatch_details.to_csv(
    REC_MISMATCH_DETAILS_PATH,
    index=False,
)


rec_clean_reconstructed.to_parquet(
    REC_RECONSTRUCTED_PATH,
    index=False,
)


canonical_clean_rec_matrix.to_parquet(
    CANONICAL_REC_MATRIX_PATH,
    index=False,
)


chronology_correction_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "Status":
        "PASS",

    "PreviousTieBreak":
        "build ID ascending",

    "CorrectedTieBreak":
        "original builds.csv row order",

    "ChronologicalPrimaryKey":
        "started_at ascending",

    "TotalBuilds":
        int(
            len(
                builds
            )
        ),

    "UniqueBuildTimestamps":
        int(
            builds[
                "timestamp"
            ].nunique()
        ),

    "BuildsInEqualTimestampGroups":
        int(
            len(
                tied_timestamp_builds
            )
        ),

    "BuildsWhoseOrderChanged":
        int(
            len(
                changed_order_builds
            )
        ),

    "VerdictDependentMismatchesBefore":
        345,

    "VerdictDependentMismatchesAfter":
        verdict_dependent_mismatches,

    "UnmatchedBuildMismatchesAfter":
        unmatched_build_dependent_mismatches,

    "CanonicalMatrixMismatchesAfter":
        canonical_rec_mismatches,

    "CompletedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
}


with open(
    CHRONOLOGY_CORRECTION_REPORT_PATH,
    "w",
    encoding="utf-8",
) as correction_file:
    json.dump(
        chronology_correction_report,
        correction_file,
        indent=2,
        default=str,
    )


rec_validation_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "PASS",

    "Chronology":
        (
            "started_at ascending; original builds.csv "
            "row order for equal timestamps"
        ),

    "PreviousFailureCause":
        (
            "Equal-timestamp builds were incorrectly "
            "ordered by numeric build ID."
        ),

    "RecentWindow":
        RECENT_WINDOW,

    "ModelReadyRows":
        int(
            len(
                dataset_with_order
            )
        ),

    "ReconstructedRows":
        int(
            len(
                rec_clean_reconstructed
            )
        ),

    "RECFeatureCount":
        int(
            len(
                REC_FEATURE_COLUMNS
            )
        ),

    "VerdictDependentRECFeatures":
        VERDICT_DEPENDENT_REC_COLUMNS,

    "NoiseIndependentRECFeatures":
        NOISE_INDEPENDENT_REC_COLUMNS,

    "VerdictDependentValuesChecked":
        int(
            len(
                dataset_with_order
            )
            * len(
                VERDICT_DEPENDENT_REC_COLUMNS
            )
        ),

    "NoiseIndependentValuesAudited":
        int(
            len(
                dataset_with_order
            )
            * len(
                NOISE_INDEPENDENT_REC_COLUMNS
            )
        ),

    "VerdictDependentMismatches":
        verdict_dependent_mismatches,

    "NoiseIndependentMismatches":
        noise_independent_mismatches,

    "EvaluationVerdictDependentMismatches":
        evaluation_dependent_mismatches,

    "UnmatchedBuild":
        UNMATCHED_BUILD_ID,

    "UnmatchedBuildVerdictDependentMismatches":
        unmatched_build_dependent_mismatches,

    "CanonicalMatrixMismatches":
        canonical_rec_mismatches,

    "EntityMappingDecision":
        "ACCEPTED_AFTER_CLEAN_REC_VALIDATION",

    "NoiseIndependentPolicy":
        (
            "Preserve original TCP-CI values. Only the "
            "13 verdict-dependent REC features are "
            "recomputed after verdict noise."
        ),

    "ChronologyCorrectionReport":
        str(
            CHRONOLOGY_CORRECTION_REPORT_PATH
        ),

    "ComparisonSummary":
        str(
            REC_COMPARISON_SUMMARY_PATH
        ),

    "MismatchDetails":
        str(
            REC_MISMATCH_DETAILS_PATH
        ),

    "CanonicalCleanRECMatrix":
        str(
            CANONICAL_REC_MATRIX_PATH
        ),

    "CompletedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
}


with open(
    REC_VALIDATION_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        rec_validation_report,
        report_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 16. Correct the Step 3 initialisation report
# ---------------------------------------------------------

PREFLIGHT_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_initialisation_report.json"
)


if PREFLIGHT_REPORT_PATH.exists():
    with open(
        PREFLIGHT_REPORT_PATH,
        "r",
        encoding="utf-8",
    ) as preflight_file:
        preflight_report = json.load(
            preflight_file
        )


    preflight_report[
        "TimestampTieBreak"
    ] = (
        "original builds.csv row order"
    )


    preflight_report[
        "ChronologyCorrectionReport"
    ] = str(
        CHRONOLOGY_CORRECTION_REPORT_PATH
    )


    preflight_report[
        "TrainingBoundary"
    ] = {
        "LastTrainingBuildOrder":
            144,

        "LastTrainingBuild":
            int(
                builds.loc[
                    builds[
                        "partition"
                    ] == "training",
                    "build",
                ].iloc[-1]
            ),

        "LastTrainingTimestamp":
            builds.loc[
                builds[
                    "partition"
                ] == "training",
                "timestamp",
            ].iloc[-1].isoformat(),

        "FirstEvaluationBuildOrder":
            145,

        "FirstEvaluationBuild":
            int(
                builds.loc[
                    builds[
                        "partition"
                    ] == "evaluation",
                    "build",
                ].iloc[0]
            ),

        "FirstEvaluationTimestamp":
            builds.loc[
                builds[
                    "partition"
                ] == "evaluation",
                "timestamp",
            ].iloc[0].isoformat(),
    }


    preflight_report[
        "UpdatedAtUTC"
    ] = pd.Timestamp.utcnow().isoformat()


    with open(
        PREFLIGHT_REPORT_PATH,
        "w",
        encoding="utf-8",
    ) as preflight_file:
        json.dump(
            preflight_report,
            preflight_file,
            indent=2,
            default=str,
        )


# ---------------------------------------------------------
# 17. Update mapping report and Project 6 checkpoint
# ---------------------------------------------------------

MAPPING_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_entity_mapping_report.json"
)


if MAPPING_REPORT_PATH.exists():
    with open(
        MAPPING_REPORT_PATH,
        "r",
        encoding="utf-8",
    ) as mapping_file:
        mapping_report = json.load(
            mapping_file
        )


    mapping_report[
        "Status"
    ] = (
        "ACCEPTED_AFTER_CLEAN_REC_VALIDATION"
    )


    mapping_report[
        "EntityMappingDecision"
    ] = (
        "ACCEPTED_AFTER_CLEAN_REC_VALIDATION"
    )


    mapping_report[
        "UnmatchedBuildVerdictDependentMismatches"
    ] = 0


    mapping_report[
        "ChronologyTieBreak"
    ] = (
        "original builds.csv row order"
    )


    mapping_report[
        "UpdatedAtUTC"
    ] = pd.Timestamp.utcnow().isoformat()


    with open(
        MAPPING_REPORT_PATH,
        "w",
        encoding="utf-8",
    ) as mapping_file:
        json.dump(
            mapping_report,
            mapping_file,
            indent=2,
            default=str,
        )


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint = json.load(
        checkpoint_file
    )


checkpoint.update({
    "Status":
        "CLEAN_REC_VALIDATION_PASSED",

    "Chronology":
        (
            "started_at ascending; original builds.csv "
            "row order for equal timestamps"
        ),

    "PreviousTimestampTieBreak":
        "build ID ascending",

    "CorrectedTimestampTieBreak":
        "original builds.csv row order",

    "ChronologyCorrectionReport":
        str(
            CHRONOLOGY_CORRECTION_REPORT_PATH
        ),

    "EntityMappingStatus":
        "ACCEPTED_AFTER_CLEAN_REC_VALIDATION",

    "UnmatchedCommitAccepted":
        True,

    "UnmatchedCommitBuild":
        UNMATCHED_BUILD_ID,

    "VerdictDependentRECFeatures":
        VERDICT_DEPENDENT_REC_COLUMNS,

    "NoiseIndependentRECFeatures":
        NOISE_INDEPENDENT_REC_COLUMNS,

    "VerdictDependentMismatches":
        verdict_dependent_mismatches,

    "NoiseIndependentMismatches":
        noise_independent_mismatches,

    "UnmatchedBuildVerdictDependentMismatches":
        unmatched_build_dependent_mismatches,

    "CanonicalRECMatrixMismatches":
        canonical_rec_mismatches,

    "CleanRECValidationReport":
        str(
            REC_VALIDATION_REPORT_PATH
        ),

    "UpdatedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
})


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 18. Final output
# ---------------------------------------------------------

print(
    "\n=== PROJECT 6 STEP 6 CORRECTED RESULT ==="
)


print(
    "\nChronology correction:"
)

print(
    "Total builds:",
    len(
        builds
    )
)

print(
    "Unique timestamps:",
    builds[
        "timestamp"
    ].nunique()
)

print(
    "Builds in tied timestamp groups:",
    len(
        tied_timestamp_builds
    )
)

print(
    "Builds whose order changed:",
    len(
        changed_order_builds
    )
)

print(
    "Correct tie-break:",
    "original builds.csv row order"
)


print(
    "\nEqual-timestamp builds:"
)

display(
    tied_timestamp_builds
)


print(
    "\nBuilds corrected from the old ID tie-break:"
)

display(
    changed_order_builds
)


print(
    "\nREC validation:"
)

print(
    "Model-ready rows:",
    len(
        dataset_with_order
    )
)

print(
    "Reconstructed rows:",
    len(
        rec_clean_reconstructed
    )
)

print(
    "Verdict-dependent values checked:",
    (
        len(
            dataset_with_order
        )
        * len(
            VERDICT_DEPENDENT_REC_COLUMNS
        )
    )
)

print(
    "Verdict-dependent mismatches:",
    verdict_dependent_mismatches
)

print(
    "Evaluation dependent mismatches:",
    evaluation_dependent_mismatches
)

print(
    "Unmatched-build dependent mismatches:",
    unmatched_build_dependent_mismatches
)


print(
    "\nNoise-independent audit:"
)

print(
    "Mismatches:",
    noise_independent_mismatches
)

print(
    "Policy:",
    "preserve original TCP-CI values"
)


print(
    "\nCanonical 19-feature matrix:"
)

print(
    "Mismatches:",
    canonical_rec_mismatches
)


print(
    "\nPer-feature comparison:"
)

display(
    comparison_summary
)


if len(
    mismatch_details
) > 0:
    print(
        "\nRemaining audit-only mismatches:"
    )

    display(
        mismatch_details.head(30)
    )

else:
    print(
        "\nRemaining mismatches:",
        "None"
    )


print(
    "\nValidation report:"
)

print(
    REC_VALIDATION_REPORT_PATH
)

print(
    "Chronology correction report:"
)

print(
    CHRONOLOGY_CORRECTION_REPORT_PATH
)


print(
    "\nValidation status:",
    validation_status
)


print(
    "\nSUCCESS: Equal-timestamp builds now follow "
    "the original builds.csv order."
)

print(
    "SUCCESS: All 13 verdict-dependent REC features "
    "match exactly."
)

print(
    "SUCCESS: The unmatched build has zero "
    "verdict-dependent mismatches."
)

print(
    "SUCCESS: The unmatched commit is accepted "
    "without synthetic entity mapping."
)

print(
    "SUCCESS: The canonical 19-feature matrix "
    "matches the original dataset exactly."
)

print(
    "SUCCESS: Project 6 is ready for noise-injection "
    "and helper validation."
)

=== PROJECT 6 STEP 6 CORRECTION: UPSTREAM TIE ORDER ===

Corrected per-feature comparison:


,Feature,FeaturePolicy,ModelRows,MatchingRows,MismatchingRows,TrainingMismatches,EvaluationMismatches,UnmatchedBuildMismatches,MaximumAbsoluteDifference
0,REC_Age,PRESERVE_ORIGINAL,17291,16718,573,107,466,0,1.000000e+00
1,REC_LastFailureAge,RECOMPUTE_AFTER_NOISE,17291,17233,58,14,44,0,1.000000e+00
2,REC_LastTransitionAge,RECOMPUTE_AFTER_NOISE,17291,17234,57,12,45,0,1.000000e+00
3,REC_RecentAvgExeTime,PRESERVE_ORIGINAL,17291,16813,478,54,424,0,1.090633e+04
4,REC_RecentMaxExeTime,PRESERVE_ORIGINAL,17291,17151,140,18,122,0,1.254300e+04
5,REC_RecentFailRate,RECOMPUTE_AFTER_NOISE,17291,17290,1,0,1,0,1.666667e-01
6,REC_RecentAssertRate,RECOMPUTE_AFTER_NOISE,17291,17290,1,0,1,0,1.666667e-01
7,REC_RecentExcRate,RECOMPUTE_AFTER_NOISE,17291,17291,0,0,0,0,5.551115e-17
8,REC_RecentTransitionRate,RECOMPUTE_AFTER_NOISE,17291,17286,5,4,1,0,3.333333e-01
9,REC_TotalAvgExeTime,PRESERVE_ORIGINAL,17291,16896,395,51,344,0,6.435911e+02


AssertionError: The source-row tie-order correction did not produce an exact verdict-dependent match. Do not continue.

In [ ]:
# =========================================================
# PROJECT 6 — STEP 6 ROOT-CAUSE DIAGNOSTIC
#
# READ-ONLY:
#   - does not update checkpoint
#   - does not overwrite REC validation files
#   - does not alter canonical runtime variables
#
# It tests:
#   1. whether REC_Age uses unique timestamps;
#   2. whether duplicate timestamps explain the failures;
#   3. whether duplicate raw Build-Test executions exist;
#   4. which tied builds/tests are connected to the
#      verdict-dependent mismatches.
# =========================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display


print(
    "=== PROJECT 6 STEP 6 ROOT-CAUSE DIAGNOSTIC ==="
)


# ---------------------------------------------------------
# 1. Confirm required objects
# ---------------------------------------------------------

required_objects = [
    "builds_raw",
    "exe_raw",
    "dataset_raw",
    "PROJECT_PREFLIGHT_DIRECTORY",
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required Project 6 objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nRerun only the Project 6 runtime recovery cell."
    )


REC_MISMATCH_DETAILS_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_clean_rec_mismatch_details.csv"
)


if not REC_MISMATCH_DETAILS_PATH.exists():
    raise FileNotFoundError(
        "The failed Step 6 mismatch file is missing:\n"
        f"{REC_MISMATCH_DETAILS_PATH}"
    )


failed_mismatches = pd.read_csv(
    REC_MISMATCH_DETAILS_PATH,
    low_memory=False,
)


if len(failed_mismatches) == 0:
    raise AssertionError(
        "The saved failed-validation mismatch file "
        "unexpectedly contains no rows."
    )


# ---------------------------------------------------------
# 2. Recreate build chronology without modifying runtime
# ---------------------------------------------------------

diagnostic_builds = pd.DataFrame({
    "SourceRow":
        np.arange(
            len(builds_raw),
            dtype=np.int64,
        ),

    "Build":
        pd.to_numeric(
            builds_raw["id"],
            errors="raise",
        ).astype("int64"),

    "Timestamp":
        pd.to_datetime(
            builds_raw["started_at"],
            errors="raise",
            utc=True,
        ),
})


diagnostic_builds = (
    diagnostic_builds
    .sort_values(
        [
            "Timestamp",
            "SourceRow",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


diagnostic_builds[
    "OrdinalBuildPosition"
] = np.arange(
    len(diagnostic_builds),
    dtype=np.int64,
)


# Equal timestamps share one dense timestamp position.
diagnostic_builds[
    "DenseTimestampPosition"
] = (
    diagnostic_builds[
        "Timestamp"
    ]
    .rank(
        method="dense",
        ascending=True,
    )
    .astype("int64")
    - 1
)


diagnostic_builds[
    "TimestampGroupSize"
] = (
    diagnostic_builds
    .groupby(
        "Timestamp"
    )[
        "Build"
    ]
    .transform("size")
    .astype(int)
)


tied_builds = (
    diagnostic_builds[
        diagnostic_builds[
            "TimestampGroupSize"
        ] > 1
    ]
    .copy()
)


# ---------------------------------------------------------
# 3. Attach raw and model-ready impact to tied builds
# ---------------------------------------------------------

diagnostic_exe = (
    exe_raw
    .copy()
    .reset_index(
        names="_ExeSourceRow"
    )
)


diagnostic_exe[
    "build"
] = pd.to_numeric(
    diagnostic_exe["build"],
    errors="raise",
).astype("int64")


diagnostic_exe[
    "test"
] = (
    diagnostic_exe[
        "test"
    ].astype(str)
)


diagnostic_exe[
    "verdict"
] = pd.to_numeric(
    diagnostic_exe["verdict"],
    errors="raise",
).astype("int8")


raw_build_summary = (
    diagnostic_exe
    .groupby(
        "build",
        as_index=False,
    )
    .agg(
        RawExecutionRows=(
            "test",
            "size",
        ),

        RawUniqueTests=(
            "test",
            "nunique",
        ),

        RawFailureExecutions=(
            "verdict",
            lambda values:
                int(
                    (
                        pd.to_numeric(
                            values,
                            errors="raise",
                        ) != 0
                    ).sum()
                ),
        ),
    )
    .rename(
        columns={
            "build":
                "Build",
        }
    )
)


diagnostic_dataset = (
    dataset_raw
    .copy()
)


diagnostic_dataset[
    "Build"
] = pd.to_numeric(
    diagnostic_dataset["Build"],
    errors="raise",
).astype("int64")


diagnostic_dataset[
    "Test"
] = (
    diagnostic_dataset[
        "Test"
    ].astype(str)
)


diagnostic_dataset[
    "Verdict"
] = pd.to_numeric(
    diagnostic_dataset["Verdict"],
    errors="raise",
).astype("int8")


model_build_summary = (
    diagnostic_dataset
    .groupby(
        "Build",
        as_index=False,
    )
    .agg(
        ModelReadyRows=(
            "Test",
            "size",
        ),

        ModelReadyFailures=(
            "Verdict",
            lambda values:
                int(
                    (
                        pd.to_numeric(
                            values,
                            errors="raise",
                        ) != 0
                    ).sum()
                ),
        ),
    )
)


tied_build_audit = (
    tied_builds
    .merge(
        raw_build_summary,
        on="Build",
        how="left",
        validate="one_to_one",
    )
    .merge(
        model_build_summary,
        on="Build",
        how="left",
        validate="one_to_one",
    )
)


for column in [
    "RawExecutionRows",
    "RawUniqueTests",
    "RawFailureExecutions",
    "ModelReadyRows",
    "ModelReadyFailures",
]:
    tied_build_audit[
        column
    ] = (
        tied_build_audit[
            column
        ]
        .fillna(0)
        .astype(int)
    )


# ---------------------------------------------------------
# 4. Test the REC_Age hypotheses directly
# ---------------------------------------------------------

build_position_lookup = (
    diagnostic_builds
    .set_index(
        "Build"
    )[
        [
            "OrdinalBuildPosition",
            "DenseTimestampPosition",
        ]
    ]
    .to_dict(
        orient="index"
    )
)


diagnostic_exe[
    "OrdinalBuildPosition"
] = (
    diagnostic_exe[
        "build"
    ]
    .map(
        lambda build_id:
            build_position_lookup[
                int(build_id)
            ][
                "OrdinalBuildPosition"
            ]
    )
    .astype("int64")
)


diagnostic_exe[
    "DenseTimestampPosition"
] = (
    diagnostic_exe[
        "build"
    ]
    .map(
        lambda build_id:
            build_position_lookup[
                int(build_id)
            ][
                "DenseTimestampPosition"
            ]
    )
    .astype("int64")
)


diagnostic_exe = (
    diagnostic_exe
    .sort_values(
        [
            "OrdinalBuildPosition",
            "_ExeSourceRow",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


first_test_execution = (
    diagnostic_exe
    .drop_duplicates(
        subset=[
            "test",
        ],
        keep="first",
    )[
        [
            "test",
            "OrdinalBuildPosition",
            "DenseTimestampPosition",
        ]
    ]
    .rename(
        columns={
            "test":
                "Test",

            "OrdinalBuildPosition":
                "FirstOrdinalPosition",

            "DenseTimestampPosition":
                "FirstDenseTimestampPosition",
        }
    )
)


age_audit = (
    diagnostic_dataset[
        [
            "Build",
            "Test",
            "REC_Age",
        ]
    ]
    .merge(
        diagnostic_builds[
            [
                "Build",
                "OrdinalBuildPosition",
                "DenseTimestampPosition",
            ]
        ],
        on="Build",
        how="left",
        validate="many_to_one",
    )
    .merge(
        first_test_execution,
        on="Test",
        how="left",
        validate="many_to_one",
    )
)


if age_audit[
    [
        "OrdinalBuildPosition",
        "DenseTimestampPosition",
        "FirstOrdinalPosition",
        "FirstDenseTimestampPosition",
    ]
].isna().any().any():
    raise AssertionError(
        "Some rows could not be included in the "
        "REC_Age diagnostic."
    )


age_audit[
    "OrdinalPredictedAge"
] = (
    age_audit[
        "OrdinalBuildPosition"
    ]
    - age_audit[
        "FirstOrdinalPosition"
    ]
)


age_audit[
    "DenseTimestampPredictedAge"
] = (
    age_audit[
        "DenseTimestampPosition"
    ]
    - age_audit[
        "FirstDenseTimestampPosition"
    ]
)


original_age = pd.to_numeric(
    age_audit[
        "REC_Age"
    ],
    errors="raise",
).to_numpy(dtype=float)


ordinal_age = pd.to_numeric(
    age_audit[
        "OrdinalPredictedAge"
    ],
    errors="raise",
).to_numpy(dtype=float)


dense_age = pd.to_numeric(
    age_audit[
        "DenseTimestampPredictedAge"
    ],
    errors="raise",
).to_numpy(dtype=float)


ordinal_age_mismatches = int(
    (
        ~np.isclose(
            original_age,
            ordinal_age,
            rtol=1e-9,
            atol=1e-9,
            equal_nan=True,
        )
    ).sum()
)


dense_age_mismatches = int(
    (
        ~np.isclose(
            original_age,
            dense_age,
            rtol=1e-9,
            atol=1e-9,
            equal_nan=True,
        )
    ).sum()
)


# ---------------------------------------------------------
# 5. Audit duplicate Build-Test rows in exe.csv
# ---------------------------------------------------------

raw_build_test_counts = (
    diagnostic_exe
    .groupby(
        [
            "build",
            "test",
        ],
        as_index=False,
    )
    .agg(
        ExecutionRows=(
            "verdict",
            "size",
        ),

        DistinctVerdicts=(
            "verdict",
            "nunique",
        ),

        DistinctDurations=(
            "duration",
            "nunique",
        ),
    )
)


duplicate_raw_build_test_pairs = (
    raw_build_test_counts[
        raw_build_test_counts[
            "ExecutionRows"
        ] > 1
    ]
    .copy()
)


conflicting_duplicate_pairs = (
    duplicate_raw_build_test_pairs[
        duplicate_raw_build_test_pairs[
            "DistinctVerdicts"
        ] > 1
    ]
    .copy()
)


# ---------------------------------------------------------
# 6. Link verdict-dependent mismatches to tied builds
# ---------------------------------------------------------

VERDICT_DEPENDENT_REC_COLUMNS_DIAGNOSTIC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


failed_mismatches[
    "Test"
] = (
    failed_mismatches[
        "Test"
    ].astype(str)
)


verdict_mismatches = (
    failed_mismatches[
        failed_mismatches[
            "Feature"
        ].isin(
            VERDICT_DEPENDENT_REC_COLUMNS_DIAGNOSTIC
        )
    ]
    .copy()
)


affected_tests = set(
    verdict_mismatches[
        "Test"
    ].astype(str)
)


tie_overlap_records = []


for timestamp, timestamp_rows in (
    tied_build_audit.groupby(
        "Timestamp",
        sort=True,
    )
):
    tied_group_builds = (
        timestamp_rows[
            "Build"
        ].astype(int).tolist()
    )


    test_sets = {}


    for build_id in tied_group_builds:
        test_sets[
            int(build_id)
        ] = set(
            diagnostic_exe.loc[
                diagnostic_exe[
                    "build"
                ] == int(build_id),
                "test",
            ].astype(str)
        )


    union_tests = set().union(
        *test_sets.values()
    )


    shared_tests = set()


    build_list = list(
        test_sets.keys()
    )


    for left_index in range(
        len(build_list)
    ):
        for right_index in range(
            left_index + 1,
            len(build_list),
        ):
            shared_tests.update(
                test_sets[
                    build_list[
                        left_index
                    ]
                ].intersection(
                    test_sets[
                        build_list[
                            right_index
                        ]
                    ]
                )
            )


    tie_overlap_records.append({
        "Timestamp":
            timestamp,

        "Builds":
            "#".join(
                str(build_id)
                for build_id
                in tied_group_builds
            ),

        "BuildCount":
            len(
                tied_group_builds
            ),

        "UnionTests":
            len(
                union_tests
            ),

        "TestsSharedAcrossTiedBuilds":
            len(
                shared_tests
            ),

        "AffectedTestsInUnion":
            len(
                union_tests.intersection(
                    affected_tests
                )
            ),

        "AffectedTestsSharedAcrossTiedBuilds":
            len(
                shared_tests.intersection(
                    affected_tests
                )
            ),

        "AffectedSharedTestExamples":
            json.dumps(
                sorted(
                    shared_tests.intersection(
                        affected_tests
                    )
                )[:20]
            ),
    })


tie_test_overlap = pd.DataFrame(
    tie_overlap_records
)


# For every tied build, count how many mismatch-affected
# tests were executed there.
tied_build_test_impact_records = []


for row in (
    tied_build_audit.itertuples(
        index=False
    )
):
    build_tests = set(
        diagnostic_exe.loc[
            diagnostic_exe[
                "build"
            ] == int(
                row.Build
            ),
            "test",
        ].astype(str)
    )


    overlapping_affected_tests = (
        build_tests.intersection(
            affected_tests
        )
    )


    tied_build_test_impact_records.append({
        "Build":
            int(
                row.Build
            ),

        "Timestamp":
            row.Timestamp,

        "RawExecutionRows":
            int(
                row.RawExecutionRows
            ),

        "RawFailureExecutions":
            int(
                row.RawFailureExecutions
            ),

        "ModelReadyRows":
            int(
                row.ModelReadyRows
            ),

        "AffectedTestsExecutedInBuild":
            int(
                len(
                    overlapping_affected_tests
                )
            ),

        "AffectedTestExamples":
            json.dumps(
                sorted(
                    overlapping_affected_tests
                )[:20]
            ),
    })


tied_build_test_impact = pd.DataFrame(
    tied_build_test_impact_records
)


# ---------------------------------------------------------
# 7. Compact mismatch summaries
# ---------------------------------------------------------

verdict_feature_summary = (
    verdict_mismatches
    .groupby(
        "Feature",
        as_index=False,
    )
    .agg(
        MismatchRows=(
            "Build",
            "size",
        ),

        AffectedBuilds=(
            "Build",
            "nunique",
        ),

        AffectedTests=(
            "Test",
            "nunique",
        ),

        FirstAffectedBuildOrder=(
            "build_order",
            "min",
        ),

        LastAffectedBuildOrder=(
            "build_order",
            "max",
        ),
    )
    .sort_values(
        "MismatchRows",
        ascending=False,
    )
    .reset_index(drop=True)
)


first_verdict_mismatches = (
    verdict_mismatches
    .sort_values(
        [
            "build_order",
            "Feature",
            "Test",
        ],
        kind="mergesort",
    )
    .groupby(
        "Feature",
        sort=False,
        as_index=False,
    )
    .head(5)
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 8. Save diagnostic only
# ---------------------------------------------------------

DIAGNOSTIC_DIRECTORY = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "step6_root_cause_diagnostic"
)


DIAGNOSTIC_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


tied_build_audit.to_csv(
    DIAGNOSTIC_DIRECTORY
    / "tied_build_audit.csv",
    index=False,
)


age_audit.to_csv(
    DIAGNOSTIC_DIRECTORY
    / "rec_age_hypothesis_audit.csv",
    index=False,
)


duplicate_raw_build_test_pairs.to_csv(
    DIAGNOSTIC_DIRECTORY
    / "duplicate_raw_build_test_pairs.csv",
    index=False,
)


tie_test_overlap.to_csv(
    DIAGNOSTIC_DIRECTORY
    / "tied_timestamp_test_overlap.csv",
    index=False,
)


tied_build_test_impact.to_csv(
    DIAGNOSTIC_DIRECTORY
    / "tied_build_affected_test_impact.csv",
    index=False,
)


verdict_feature_summary.to_csv(
    DIAGNOSTIC_DIRECTORY
    / "verdict_mismatch_summary.csv",
    index=False,
)


diagnostic_report = {
    "Project":
        "eclipse@jetty.project",

    "Status":
        "READ_ONLY_DIAGNOSTIC_COMPLETE",

    "Builds":
        int(
            len(
                diagnostic_builds
            )
        ),

    "UniqueTimestamps":
        int(
            diagnostic_builds[
                "Timestamp"
            ].nunique()
        ),

    "BuildsInTiedTimestampGroups":
        int(
            len(
                tied_build_audit
            )
        ),

    "OrdinalAgeMismatches":
        ordinal_age_mismatches,

    "DenseTimestampAgeMismatches":
        dense_age_mismatches,

    "DuplicateRawBuildTestPairs":
        int(
            len(
                duplicate_raw_build_test_pairs
            )
        ),

    "ConflictingDuplicateRawBuildTestPairs":
        int(
            len(
                conflicting_duplicate_pairs
            )
        ),

    "VerdictDependentMismatchRows":
        int(
            len(
                verdict_mismatches
            )
        ),

    "VerdictDependentAffectedTests":
        int(
            len(
                affected_tests
            )
        ),

    "DiagnosticDirectory":
        str(
            DIAGNOSTIC_DIRECTORY
        ),

    "CompletedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
}


with open(
    DIAGNOSTIC_DIRECTORY
    / "root_cause_diagnostic_report.json",
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        diagnostic_report,
        report_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 9. Final output
# ---------------------------------------------------------

print(
    "\n=== PROJECT 6 STEP 6 ROOT-CAUSE RESULT ==="
)


print(
    "\nTimestamp structure:"
)

print(
    "Builds:",
    len(
        diagnostic_builds
    )
)

print(
    "Unique timestamps:",
    diagnostic_builds[
        "Timestamp"
    ].nunique()
)

print(
    "Builds in tied timestamp groups:",
    len(
        tied_build_audit
    )
)


print(
    "\nTied-build audit:"
)

display(
    tied_build_audit[
        [
            "Build",
            "Timestamp",
            "SourceRow",
            "OrdinalBuildPosition",
            "DenseTimestampPosition",
            "RawExecutionRows",
            "RawFailureExecutions",
            "ModelReadyRows",
            "ModelReadyFailures",
        ]
    ]
)


print(
    "\nREC_Age hypotheses:"
)

print(
    "Ordinal-build-position mismatches:",
    ordinal_age_mismatches
)

print(
    "Dense-unique-timestamp mismatches:",
    dense_age_mismatches
)


print(
    "\nRaw Build-Test duplication:"
)

print(
    "Duplicate raw Build-Test pairs:",
    len(
        duplicate_raw_build_test_pairs
    )
)

print(
    "Conflicting-verdict duplicate pairs:",
    len(
        conflicting_duplicate_pairs
    )
)


if len(
    duplicate_raw_build_test_pairs
) > 0:
    display(
        duplicate_raw_build_test_pairs.head(30)
    )


print(
    "\nVerdict-dependent mismatch summary:"
)

display(
    verdict_feature_summary
)


print(
    "\nTied timestamp test overlap:"
)

display(
    tie_test_overlap
)


print(
    "\nTied builds versus mismatch-affected tests:"
)

display(
    tied_build_test_impact
)


print(
    "\nFirst five mismatches per verdict-dependent feature:"
)

display(
    first_verdict_mismatches[
        [
            "Feature",
            "Build",
            "Test",
            "build_order",
            "partition",
            "OriginalValue",
            "ReconstructedValue",
        ]
    ]
)


print(
    "\nDiagnostic saved to:"
)

print(
    DIAGNOSTIC_DIRECTORY
)


print(
    "\nSUCCESS: No Project 6 checkpoint or canonical "
    "artefact was modified."
)

print(
    "SUCCESS: The exact REC_Age and execution-history "
    "edge cases are now measurable."
)

=== PROJECT 6 STEP 6 ROOT-CAUSE DIAGNOSTIC ===

=== PROJECT 6 STEP 6 ROOT-CAUSE RESULT ===

Timestamp structure:
Builds: 192
Unique timestamps: 189
Builds in tied timestamp groups: 6

Tied-build audit:


,Build,Timestamp,SourceRow,OrdinalBuildPosition,DenseTimestampPosition,RawExecutionRows,RawFailureExecutions,ModelReadyRows,ModelReadyFailures
0,6843283,2013-05-03 06:31:33+00:00,126,126,126,107,1,107,1
1,6843289,2013-05-03 06:31:33+00:00,127,127,126,150,0,0,0
2,6911543,2013-05-06 03:31:05+00:00,145,145,144,147,1,147,1
3,6911546,2013-05-06 03:31:05+00:00,146,146,144,107,1,107,1
4,7651302,2013-05-31 01:31:12+00:00,184,184,182,107,1,107,1
5,7651311,2013-05-31 01:31:12+00:00,185,185,182,105,1,105,1



REC_Age hypotheses:
Ordinal-build-position mismatches: 573
Dense-unique-timestamp mismatches: 5129

Raw Build-Test duplication:
Duplicate raw Build-Test pairs: 0
Conflicting-verdict duplicate pairs: 0

Verdict-dependent mismatch summary:


,Feature,MismatchRows,AffectedBuilds,AffectedTests,FirstAffectedBuildOrder,LastAffectedBuildOrder
0,REC_TotalTransitionRate,92,44,12,127,192
1,REC_LastFailureAge,58,12,12,127,186
2,REC_LastTransitionAge,57,11,12,127,186
3,REC_TotalFailRate,53,5,12,127,186
4,REC_TotalAssertRate,38,5,9,127,186
5,REC_MaxTestFileTransitionRate,20,20,1,127,186
6,REC_TotalExcRate,15,5,3,127,186
7,REC_RecentTransitionRate,5,5,1,127,146
8,REC_LastVerdict,3,3,1,127,146
9,REC_MaxTestFileFailRate,2,2,1,185,186



Tied timestamp test overlap:


...


Tied builds versus mismatch-affected tests:


,Build,Timestamp,RawExecutionRows,RawFailureExecutions,ModelReadyRows,AffectedTestsExecutedInBuild,AffectedTestExamples
0,6843283,2013-05-03 06:31:33+00:00,107,1,107,12,"[""1631"", ""1632"", ""1634"", ""1635"", ""1681"", ""1970..."
1,6843289,2013-05-03 06:31:33+00:00,150,0,0,10,"[""1632"", ""1634"", ""1635"", ""1681"", ""1970"", ""2136..."
2,6911543,2013-05-06 03:31:05+00:00,147,1,147,10,"[""1632"", ""1634"", ""1635"", ""1681"", ""1970"", ""2136..."
3,6911546,2013-05-06 03:31:05+00:00,107,1,107,12,"[""1631"", ""1632"", ""1634"", ""1635"", ""1681"", ""1970..."
4,7651302,2013-05-31 01:31:12+00:00,107,1,107,12,"[""1631"", ""1632"", ""1634"", ""1635"", ""1681"", ""1970..."
5,7651311,2013-05-31 01:31:12+00:00,105,1,105,12,"[""1631"", ""1632"", ""1634"", ""1635"", ""1681"", ""1970..."



First five mismatches per verdict-dependent feature:


,Feature,Build,Test,build_order,partition,OriginalValue,ReconstructedValue
0,REC_LastFailureAge,6843283,1632,127,training,1.000000,0.000000
1,REC_LastFailureAge,6843283,1634,127,training,48.000000,47.000000
2,REC_LastFailureAge,6843283,1635,127,training,48.000000,47.000000
3,REC_LastFailureAge,6843283,1681,127,training,48.000000,47.000000
4,REC_LastFailureAge,6843283,1970,127,training,48.000000,47.000000
5,REC_LastTransitionAge,6843283,1634,127,training,47.000000,46.000000
6,REC_LastTransitionAge,6843283,1635,127,training,47.000000,46.000000
7,REC_LastTransitionAge,6843283,1681,127,training,47.000000,46.000000
8,REC_LastTransitionAge,6843283,1970,127,training,47.000000,46.000000
9,REC_LastTransitionAge,6843283,2136,127,training,96.000000,95.000000



Diagnostic saved to:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/eclipse__jetty.project/jetty_preflight/step6_root_cause_diagnostic

SUCCESS: No Project 6 checkpoint or canonical artefact was modified.
SUCCESS: The exact REC_Age and execution-history edge cases are now measurable.


In [ ]:
# =========================================================
# PROJECT 6 — STEP 6 FINAL CORRECTION
# VALIDATE DESCENDING BUILD-ID ORDER WITHIN TIMESTAMP TIES
#
# Chronological order:
#   1. started_at ascending
#   2. build ID descending for equal timestamps
#
# The corrected state is saved only if all 13
# verdict-dependent REC features match exactly.
# =========================================================

from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd
from IPython.display import display


print(
    "=== PROJECT 6 STEP 6 FINAL CORRECTION: "
    "DESCENDING TIE ORDER ==="
)


# ---------------------------------------------------------
# 1. Confirm required state
# ---------------------------------------------------------

required_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",
    "builds_raw",
    "exe_raw",
    "dataset_raw",
    "build_entity_map",
    "changed_entities_by_build",
    "reconstruct_rec_features",
    "align_reconstructed_rec_features",
    "REC_FEATURE_COLUMNS",
    "VERDICT_DEPENDENT_REC_COLUMNS",
    "NOISE_INDEPENDENT_REC_COLUMNS",
    "RECENT_WINDOW",
    "PROJECT_PREFLIGHT_DIRECTORY",
    "PROJECT_6_SELECTION_CHECKPOINT",
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required Project 6 objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nRerun the Project 6 runtime recovery cell "
        "and the original Step 6 cell first."
    )


if PROJECT_NUMBER != 6:
    raise AssertionError(
        f"Expected Project 6, observed {PROJECT_NUMBER}."
    )


if PROJECT_NAME != "eclipse@jetty.project":
    raise AssertionError(
        f"Unexpected project: {PROJECT_NAME}"
    )


# ---------------------------------------------------------
# 2. Construct the corrected build chronology
# ---------------------------------------------------------

descending_tie_builds = pd.DataFrame({
    "source_row":
        np.arange(
            len(builds_raw),
            dtype=np.int64,
        ),

    "build":
        pd.to_numeric(
            builds_raw["id"],
            errors="raise",
        ).astype("int64"),

    "commits":
        builds_raw["commits"].astype(str),

    "timestamp":
        pd.to_datetime(
            builds_raw["started_at"],
            errors="raise",
            utc=True,
        ),
})


if len(descending_tie_builds) != 192:
    raise AssertionError(
        "Expected exactly 192 Jetty builds."
    )


if descending_tie_builds[
    "build"
].duplicated().any():
    raise AssertionError(
        "Duplicate Jetty build IDs were found."
    )


# Critical correction:
# equal timestamps use descending numeric build ID.
descending_tie_builds = (
    descending_tie_builds
    .sort_values(
        [
            "timestamp",
            "build",
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


descending_tie_builds[
    "build_order"
] = np.arange(
    1,
    len(descending_tie_builds) + 1,
    dtype=np.int64,
)


TRAINING_BUILD_COUNT = int(
    np.floor(
        0.75
        * len(descending_tie_builds)
    )
)


descending_tie_builds[
    "partition"
] = np.where(
    descending_tie_builds[
        "build_order"
    ] <= TRAINING_BUILD_COUNT,
    "training",
    "evaluation",
)


if TRAINING_BUILD_COUNT != 144:
    raise AssertionError(
        "Expected 144 training builds."
    )


if (
    descending_tie_builds[
        "partition"
    ] == "evaluation"
).sum() != 48:
    raise AssertionError(
        "Expected 48 evaluation builds."
    )


descending_build_order_lookup = dict(
    zip(
        descending_tie_builds[
            "build"
        ].astype(int),

        descending_tie_builds[
            "build_order"
        ].astype(int),
    )
)


descending_partition_lookup = dict(
    zip(
        descending_tie_builds[
            "build"
        ].astype(int),

        descending_tie_builds[
            "partition"
        ].astype(str),
    )
)


# ---------------------------------------------------------
# 3. Audit the equal-timestamp ordering
# ---------------------------------------------------------

descending_tie_builds[
    "timestamp_group_size"
] = (
    descending_tie_builds
    .groupby(
        "timestamp"
    )[
        "build"
    ]
    .transform("size")
    .astype(int)
)


equal_timestamp_order = (
    descending_tie_builds[
        descending_tie_builds[
            "timestamp_group_size"
        ] > 1
    ][
        [
            "build",
            "timestamp",
            "source_row",
            "build_order",
            "partition",
        ]
    ]
    .copy()
)


expected_tied_order = [
    6843289,
    6843283,
    6911546,
    6911543,
    7651311,
    7651302,
]


observed_tied_order = (
    equal_timestamp_order[
        "build"
    ]
    .astype(int)
    .tolist()
)


if observed_tied_order != expected_tied_order:
    raise AssertionError(
        "Unexpected equal-timestamp build order.\n"
        f"Expected: {expected_tied_order}\n"
        f"Observed: {observed_tied_order}"
    )


# ---------------------------------------------------------
# 4. Rebuild raw execution history
# ---------------------------------------------------------

descending_exe_for_rec = (
    exe_raw
    .copy()
    .reset_index()
    .rename(
        columns={
            "index":
                "_source_execution_row"
        }
    )
)


descending_exe_for_rec[
    "build"
] = pd.to_numeric(
    descending_exe_for_rec["build"],
    errors="raise",
).astype("int64")


descending_exe_for_rec[
    "test"
] = (
    descending_exe_for_rec["test"]
    .astype(str)
)


descending_exe_for_rec[
    "verdict"
] = pd.to_numeric(
    descending_exe_for_rec["verdict"],
    errors="raise",
).astype("int8")


descending_exe_for_rec[
    "duration"
] = pd.to_numeric(
    descending_exe_for_rec["duration"],
    errors="raise",
).astype(float)


if "job" not in descending_exe_for_rec.columns:
    descending_exe_for_rec["job"] = ""


descending_exe_for_rec[
    "build_order"
] = (
    descending_exe_for_rec[
        "build"
    ]
    .map(
        descending_build_order_lookup
    )
)


descending_exe_for_rec[
    "partition"
] = (
    descending_exe_for_rec[
        "build"
    ]
    .map(
        descending_partition_lookup
    )
)


if descending_exe_for_rec[
    "build_order"
].isna().any():
    raise AssertionError(
        "Some execution rows could not be assigned "
        "to the corrected chronology."
    )


descending_exe_for_rec[
    "build_order"
] = (
    descending_exe_for_rec[
        "build_order"
    ].astype("int64")
)


descending_exe_for_rec = (
    descending_exe_for_rec
    .sort_values(
        [
            "build_order",
            "job",
            "test",
            "_source_execution_row",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


descending_training_history = (
    descending_exe_for_rec[
        descending_exe_for_rec[
            "partition"
        ] == "training"
    ]
    .copy()
    .reset_index(drop=True)
)


descending_evaluation_history = (
    descending_exe_for_rec[
        descending_exe_for_rec[
            "partition"
        ] == "evaluation"
    ]
    .copy()
    .reset_index(drop=True)
)


if len(descending_training_history) != 19845:
    raise AssertionError(
        "Raw training-history count changed."
    )


if len(descending_evaluation_history) != 6594:
    raise AssertionError(
        "Raw evaluation-history count changed."
    )


# ---------------------------------------------------------
# 5. Rebuild model-ready data
# ---------------------------------------------------------

descending_dataset = dataset_raw.copy()


descending_dataset[
    "Build"
] = pd.to_numeric(
    descending_dataset["Build"],
    errors="raise",
).astype("int64")


descending_dataset[
    "Test"
] = (
    descending_dataset["Test"]
    .astype(str)
)


descending_dataset[
    "Verdict"
] = pd.to_numeric(
    descending_dataset["Verdict"],
    errors="raise",
).astype("int8")


descending_dataset[
    "Duration"
] = pd.to_numeric(
    descending_dataset["Duration"],
    errors="raise",
).astype(float)


descending_dataset[
    "build_order"
] = (
    descending_dataset[
        "Build"
    ]
    .map(
        descending_build_order_lookup
    )
)


descending_dataset[
    "partition"
] = (
    descending_dataset[
        "Build"
    ]
    .map(
        descending_partition_lookup
    )
)


if descending_dataset[
    "build_order"
].isna().any():
    raise AssertionError(
        "Some model rows could not be assigned "
        "to the corrected chronology."
    )


descending_dataset[
    "build_order"
] = (
    descending_dataset[
        "build_order"
    ].astype("int64")
)


descending_dataset_with_order = (
    descending_dataset
    .sort_values(
        [
            "build_order",
            "Build",
            "Test",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


descending_training_data = (
    descending_dataset_with_order[
        descending_dataset_with_order[
            "partition"
        ] == "training"
    ]
    .copy()
    .reset_index(drop=True)
)


descending_evaluation_data = (
    descending_dataset_with_order[
        descending_dataset_with_order[
            "partition"
        ] == "evaluation"
    ]
    .copy()
    .reset_index(drop=True)
)


if len(descending_training_data) != 13212:
    raise AssertionError(
        "Model-training row count changed."
    )


if len(descending_evaluation_data) != 4079:
    raise AssertionError(
        "Model-evaluation row count changed."
    )


# ---------------------------------------------------------
# 6. Prepare corrected REC global context
# ---------------------------------------------------------

descending_ordered_execution_builds = (
    descending_exe_for_rec[
        [
            "build",
            "build_order",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "build_order",
        kind="mergesort",
    )[
        "build"
    ]
    .astype(int)
    .tolist()
)


descending_global_build_position = {
    int(build_id):
        position

    for position, build_id
    in enumerate(
        descending_ordered_execution_builds
    )
}


if len(
    descending_global_build_position
) != 192:
    raise AssertionError(
        "Corrected build-position map must contain "
        "192 builds."
    )


entity_build_pairs_descending = (
    build_entity_map[
        [
            "Build",
            "EntityID",
        ]
    ]
    .drop_duplicates()
    .copy()
)


entity_build_pairs_descending[
    "Build"
] = pd.to_numeric(
    entity_build_pairs_descending["Build"],
    errors="raise",
).astype("int64")


entity_build_pairs_descending[
    "EntityID"
] = pd.to_numeric(
    entity_build_pairs_descending["EntityID"],
    errors="raise",
).astype("int64")


descending_entity_changed_builds = (
    entity_build_pairs_descending
    .groupby(
        "EntityID"
    )[
        "Build"
    ]
    .apply(
        lambda values:
            set(
                values.astype(int)
            )
    )
    .to_dict()
)


descending_changed_entities_by_build = {
    int(build_id):
        set(
            int(entity_id)
            for entity_id in (
                changed_entities_by_build.get(
                    int(build_id),
                    set(),
                )
            )
        )

    for build_id
    in descending_tie_builds[
        "build"
    ].astype(int)
}


UNMATCHED_BUILD_ID = 6806945


if (
    descending_changed_entities_by_build[
        UNMATCHED_BUILD_ID
    ] != set()
):
    raise AssertionError(
        "The unmatched build must retain an empty "
        "changed-entity set."
    )


# Save previous globals in case validation fails.
previous_global_build_position = (
    global_build_position
    if "global_build_position" in globals()
    else None
)


previous_entity_changed_builds = (
    entity_changed_builds
    if "entity_changed_builds" in globals()
    else None
)


previous_clean_changed_entities = (
    clean_changed_entities_by_build
    if "clean_changed_entities_by_build" in globals()
    else None
)


# The reconstruction helper reads these global objects.
global_build_position = (
    descending_global_build_position
)


entity_changed_builds = (
    descending_entity_changed_builds
)


clean_changed_entities_by_build = (
    descending_changed_entities_by_build
)


# ---------------------------------------------------------
# 7. Reconstruct all 19 REC features
# ---------------------------------------------------------

requested_model_rows = (
    descending_dataset_with_order[
        [
            "Build",
            "Test",
        ]
    ]
    .copy()
)


descending_rec_reconstructed = (
    reconstruct_rec_features(
        execution_history=(
            descending_exe_for_rec
        ),
        requested_rows=(
            requested_model_rows
        ),
        recent_window=RECENT_WINDOW,
    )
)


descending_aligned_rec = (
    align_reconstructed_rec_features(
        reconstructed=(
            descending_rec_reconstructed
        ),
        requested_rows=(
            requested_model_rows
        ),
        columns=REC_FEATURE_COLUMNS,
    )
)


if len(
    descending_rec_reconstructed
) != 17291:
    raise AssertionError(
        "Expected exactly 17,291 reconstructed rows."
    )


# ---------------------------------------------------------
# 8. Compare reconstructed and original features
# ---------------------------------------------------------

comparison_records = []
mismatch_frames = []


for feature in REC_FEATURE_COLUMNS:
    original_values = pd.to_numeric(
        descending_dataset_with_order[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)


    reconstructed_values = pd.to_numeric(
        descending_aligned_rec[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)


    matching_mask = np.isclose(
        original_values,
        reconstructed_values,
        rtol=1e-9,
        atol=1e-9,
        equal_nan=True,
    )


    mismatch_mask = ~matching_mask


    training_mask = (
        descending_dataset_with_order[
            "partition"
        ].to_numpy()
        == "training"
    )


    evaluation_mask = (
        descending_dataset_with_order[
            "partition"
        ].to_numpy()
        == "evaluation"
    )


    unmatched_mask = (
        descending_dataset_with_order[
            "Build"
        ].to_numpy(dtype=int)
        == UNMATCHED_BUILD_ID
    )


    differences = np.abs(
        original_values
        - reconstructed_values
    )


    finite_differences = differences[
        np.isfinite(differences)
    ]


    maximum_difference = (
        float(finite_differences.max())
        if len(finite_differences) > 0
        else 0.0
    )


    policy = (
        "RECOMPUTE_AFTER_NOISE"
        if feature in (
            VERDICT_DEPENDENT_REC_COLUMNS
        )
        else "PRESERVE_ORIGINAL"
    )


    comparison_records.append({
        "Feature":
            feature,

        "FeaturePolicy":
            policy,

        "ModelRows":
            int(
                len(
                    descending_dataset_with_order
                )
            ),

        "MatchingRows":
            int(
                matching_mask.sum()
            ),

        "MismatchingRows":
            int(
                mismatch_mask.sum()
            ),

        "TrainingMismatches":
            int(
                (
                    mismatch_mask
                    & training_mask
                ).sum()
            ),

        "EvaluationMismatches":
            int(
                (
                    mismatch_mask
                    & evaluation_mask
                ).sum()
            ),

        "UnmatchedBuildMismatches":
            int(
                (
                    mismatch_mask
                    & unmatched_mask
                ).sum()
            ),

        "MaximumAbsoluteDifference":
            maximum_difference,
    })


    if mismatch_mask.any():
        feature_details = (
            descending_dataset_with_order.loc[
                mismatch_mask,
                [
                    "Build",
                    "Test",
                    "build_order",
                    "partition",
                ]
            ]
            .copy()
        )


        feature_details.insert(
            0,
            "Feature",
            feature,
        )


        feature_details[
            "OriginalValue"
        ] = original_values[
            mismatch_mask
        ]


        feature_details[
            "ReconstructedValue"
        ] = reconstructed_values[
            mismatch_mask
        ]


        feature_details[
            "FeaturePolicy"
        ] = policy


        mismatch_frames.append(
            feature_details
        )


descending_comparison_summary = pd.DataFrame(
    comparison_records
)


if mismatch_frames:
    descending_mismatch_details = pd.concat(
        mismatch_frames,
        ignore_index=True,
    )

else:
    descending_mismatch_details = pd.DataFrame(
        columns=[
            "Feature",
            "Build",
            "Test",
            "build_order",
            "partition",
            "OriginalValue",
            "ReconstructedValue",
            "FeaturePolicy",
        ]
    )


dependent_summary = (
    descending_comparison_summary[
        descending_comparison_summary[
            "Feature"
        ].isin(
            VERDICT_DEPENDENT_REC_COLUMNS
        )
    ]
)


independent_summary = (
    descending_comparison_summary[
        descending_comparison_summary[
            "Feature"
        ].isin(
            NOISE_INDEPENDENT_REC_COLUMNS
        )
    ]
)


verdict_dependent_mismatches = int(
    dependent_summary[
        "MismatchingRows"
    ].sum()
)


evaluation_dependent_mismatches = int(
    dependent_summary[
        "EvaluationMismatches"
    ].sum()
)


unmatched_build_dependent_mismatches = int(
    dependent_summary[
        "UnmatchedBuildMismatches"
    ].sum()
)


noise_independent_mismatches = int(
    independent_summary[
        "MismatchingRows"
    ].sum()
)


# ---------------------------------------------------------
# 9. Construct the canonical 19-feature matrix
# ---------------------------------------------------------

canonical_clean_rec_matrix = (
    descending_dataset_with_order[
        [
            "Build",
            "Test",
            "build_order",
            "partition",
            *REC_FEATURE_COLUMNS,
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


# Reconstruct only verdict-dependent features.
# Preserve the six original noise-independent values.
for feature in (
    VERDICT_DEPENDENT_REC_COLUMNS
):
    canonical_clean_rec_matrix[
        feature
    ] = (
        descending_aligned_rec[
            feature
        ].to_numpy(dtype=float)
    )


canonical_rec_mismatches = 0


for feature in REC_FEATURE_COLUMNS:
    canonical_values = pd.to_numeric(
        canonical_clean_rec_matrix[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)


    original_values = pd.to_numeric(
        descending_dataset_with_order[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)


    equal_mask = np.isclose(
        canonical_values,
        original_values,
        rtol=1e-9,
        atol=1e-9,
        equal_nan=True,
    )


    canonical_rec_mismatches += int(
        (~equal_mask).sum()
    )


validation_status = (
    "PASS"
    if (
        verdict_dependent_mismatches == 0
        and evaluation_dependent_mismatches == 0
        and unmatched_build_dependent_mismatches == 0
        and canonical_rec_mismatches == 0
    )
    else "FAIL"
)


# ---------------------------------------------------------
# 10. Stop safely when the hypothesis is incorrect
# ---------------------------------------------------------

if validation_status != "PASS":
    if previous_global_build_position is not None:
        global_build_position = (
            previous_global_build_position
        )


    if previous_entity_changed_builds is not None:
        entity_changed_builds = (
            previous_entity_changed_builds
        )


    if previous_clean_changed_entities is not None:
        clean_changed_entities_by_build = (
            previous_clean_changed_entities
        )


    print(
        "\nDescending-tie comparison:"
    )

    display(
        descending_comparison_summary
    )


    print(
        "\nValidation status:",
        validation_status
    )


    raise AssertionError(
        "Descending build-ID tie ordering did not "
        "produce an exact verdict-dependent match. "
        "No checkpoint was changed."
    )


# ---------------------------------------------------------
# 11. Commit corrected runtime variables
# ---------------------------------------------------------

builds = (
    descending_tie_builds
    .copy()
)


build_order_lookup = (
    descending_build_order_lookup
)


build_partition_lookup = (
    descending_partition_lookup
)


builds_checked = pd.DataFrame({
    "id":
        builds["build"].astype("int64"),

    "commits":
        builds["commits"].astype(str),

    "started_at":
        builds["timestamp"],

    "source_row":
        builds["source_row"].astype("int64"),

    "build_order":
        builds["build_order"].astype("int64"),

    "partition":
        builds["partition"].astype(str),
})


exe_for_rec = (
    descending_exe_for_rec
)


clean_training_history = (
    descending_training_history
)


clean_evaluation_history = (
    descending_evaluation_history
)


dataset = (
    descending_dataset
)


dataset_with_order = (
    descending_dataset_with_order
)


clean_training_data = (
    descending_training_data
)


clean_evaluation_data = (
    descending_evaluation_data
)


canonical_clean_training_data = (
    clean_training_data
    .copy()
    .reset_index(drop=True)
)


canonical_clean_evaluation_data = (
    clean_evaluation_data
    .copy()
    .reset_index(drop=True)
)


ordered_execution_builds = (
    descending_ordered_execution_builds
)


global_build_position = (
    descending_global_build_position
)


entity_changed_builds = (
    descending_entity_changed_builds
)


clean_changed_entities_by_build = (
    descending_changed_entities_by_build
)


rec_clean_reconstructed = (
    descending_rec_reconstructed
)


aligned_reconstructed_rec = (
    descending_aligned_rec
)


comparison_summary = (
    descending_comparison_summary
)


mismatch_details = (
    descending_mismatch_details
)


MODEL_FEATURE_COLUMNS = [
    column
    for column in dataset.columns
    if column not in {
        "Build",
        "Test",
        "Verdict",
        "Duration",
        "build_order",
        "partition",
    }
]


if len(MODEL_FEATURE_COLUMNS) != 150:
    raise AssertionError(
        "Expected 150 model predictors."
    )


# ---------------------------------------------------------
# 12. Save corrected permanent artefacts
# ---------------------------------------------------------

TIE_ORDER_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_descending_build_id_tie_order_report.json"
)


EQUAL_TIMESTAMP_ORDER_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_equal_timestamp_build_order.csv"
)


BUILD_SPLIT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_chronological_build_split.csv"
)


REC_COMPARISON_SUMMARY_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_clean_rec_feature_comparison.csv"
)


REC_MISMATCH_DETAILS_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_clean_rec_mismatch_details.csv"
)


REC_RECONSTRUCTED_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_clean_rec_reconstructed_all19.parquet"
)


CANONICAL_REC_MATRIX_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_canonical_clean_rec_matrix.parquet"
)


REC_VALIDATION_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_clean_rec_validation_report.json"
)


equal_timestamp_order.to_csv(
    EQUAL_TIMESTAMP_ORDER_PATH,
    index=False,
)


builds.to_csv(
    BUILD_SPLIT_PATH,
    index=False,
)


comparison_summary.to_csv(
    REC_COMPARISON_SUMMARY_PATH,
    index=False,
)


mismatch_details.to_csv(
    REC_MISMATCH_DETAILS_PATH,
    index=False,
)


rec_clean_reconstructed.to_parquet(
    REC_RECONSTRUCTED_PATH,
    index=False,
)


canonical_clean_rec_matrix.to_parquet(
    CANONICAL_REC_MATRIX_PATH,
    index=False,
)


tie_order_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "Status":
        "PASS",

    "PrimaryOrder":
        "started_at ascending",

    "EqualTimestampTieBreak":
        "build ID descending",

    "UniqueTimestamps":
        int(
            builds["timestamp"].nunique()
        ),

    "BuildsInEqualTimestampGroups":
        int(
            len(equal_timestamp_order)
        ),

    "EqualTimestampBuildOrder":
        observed_tied_order,

    "VerdictDependentMismatchesBefore":
        345,

    "VerdictDependentMismatchesAfter":
        verdict_dependent_mismatches,

    "EvaluationDependentMismatchesAfter":
        evaluation_dependent_mismatches,

    "UnmatchedBuildDependentMismatchesAfter":
        unmatched_build_dependent_mismatches,

    "CanonicalMatrixMismatchesAfter":
        canonical_rec_mismatches,

    "CompletedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
}


with open(
    TIE_ORDER_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        tie_order_report,
        report_file,
        indent=2,
        default=str,
    )


rec_validation_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "PASS",

    "Chronology":
        (
            "started_at ascending; build ID descending "
            "for equal timestamps"
        ),

    "RecentWindow":
        RECENT_WINDOW,

    "ModelReadyRows":
        int(
            len(dataset_with_order)
        ),

    "ReconstructedRows":
        int(
            len(rec_clean_reconstructed)
        ),

    "RECFeatureCount":
        19,

    "VerdictDependentRECFeatures":
        VERDICT_DEPENDENT_REC_COLUMNS,

    "NoiseIndependentRECFeatures":
        NOISE_INDEPENDENT_REC_COLUMNS,

    "VerdictDependentValuesChecked":
        int(
            len(dataset_with_order)
            * len(
                VERDICT_DEPENDENT_REC_COLUMNS
            )
        ),

    "NoiseIndependentValuesAudited":
        int(
            len(dataset_with_order)
            * len(
                NOISE_INDEPENDENT_REC_COLUMNS
            )
        ),

    "VerdictDependentMismatches":
        verdict_dependent_mismatches,

    "EvaluationVerdictDependentMismatches":
        evaluation_dependent_mismatches,

    "UnmatchedBuild":
        UNMATCHED_BUILD_ID,

    "UnmatchedBuildVerdictDependentMismatches":
        unmatched_build_dependent_mismatches,

    "NoiseIndependentMismatches":
        noise_independent_mismatches,

    "CanonicalMatrixMismatches":
        canonical_rec_mismatches,

    "NoiseIndependentPolicy":
        (
            "Preserve original TCP-CI values. Only "
            "the 13 verdict-dependent REC features "
            "are recomputed after verdict noise."
        ),

    "EntityMappingDecision":
        "ACCEPTED_AFTER_CLEAN_REC_VALIDATION",

    "TieOrderReport":
        str(TIE_ORDER_REPORT_PATH),

    "ComparisonSummary":
        str(REC_COMPARISON_SUMMARY_PATH),

    "MismatchDetails":
        str(REC_MISMATCH_DETAILS_PATH),

    "ReconstructedAll19":
        str(REC_RECONSTRUCTED_PATH),

    "CanonicalCleanRECMatrix":
        str(CANONICAL_REC_MATRIX_PATH),

    "CompletedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
}


with open(
    REC_VALIDATION_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        rec_validation_report,
        report_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 13. Correct saved build-order-dependent reports
# ---------------------------------------------------------

BUILD_MAPPING_AUDIT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_build_mapping_audit.csv"
)


if BUILD_MAPPING_AUDIT_PATH.exists():
    build_mapping_audit = pd.read_csv(
        BUILD_MAPPING_AUDIT_PATH
    )


    build_mapping_audit[
        "Build"
    ] = pd.to_numeric(
        build_mapping_audit["Build"],
        errors="raise",
    ).astype("int64")


    build_mapping_audit[
        "BuildOrder"
    ] = (
        build_mapping_audit[
            "Build"
        ]
        .map(
            build_order_lookup
        )
        .astype("int64")
    )


    build_mapping_audit[
        "Partition"
    ] = (
        build_mapping_audit[
            "Build"
        ]
        .map(
            build_partition_lookup
        )
    )


    build_mapping_audit = (
        build_mapping_audit
        .sort_values(
            "BuildOrder",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    build_mapping_audit.to_csv(
        BUILD_MAPPING_AUDIT_PATH,
        index=False,
    )


PREFLIGHT_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_initialisation_report.json"
)


if PREFLIGHT_REPORT_PATH.exists():
    with open(
        PREFLIGHT_REPORT_PATH,
        "r",
        encoding="utf-8",
    ) as preflight_file:
        preflight_report = json.load(
            preflight_file
        )


    preflight_report[
        "TimestampTieBreak"
    ] = "build ID descending"


    preflight_report[
        "TieOrderReport"
    ] = str(
        TIE_ORDER_REPORT_PATH
    )


    preflight_report[
        "TrainingBoundary"
    ] = {
        "LastTrainingBuildOrder":
            144,

        "LastTrainingBuild":
            int(
                builds.loc[
                    builds[
                        "partition"
                    ] == "training",
                    "build",
                ].iloc[-1]
            ),

        "LastTrainingTimestamp":
            builds.loc[
                builds[
                    "partition"
                ] == "training",
                "timestamp",
            ].iloc[-1].isoformat(),

        "FirstEvaluationBuildOrder":
            145,

        "FirstEvaluationBuild":
            int(
                builds.loc[
                    builds[
                        "partition"
                    ] == "evaluation",
                    "build",
                ].iloc[0]
            ),

        "FirstEvaluationTimestamp":
            builds.loc[
                builds[
                    "partition"
                ] == "evaluation",
                "timestamp",
            ].iloc[0].isoformat(),
    }


    preflight_report[
        "UpdatedAtUTC"
    ] = pd.Timestamp.utcnow().isoformat()


    with open(
        PREFLIGHT_REPORT_PATH,
        "w",
        encoding="utf-8",
    ) as preflight_file:
        json.dump(
            preflight_report,
            preflight_file,
            indent=2,
            default=str,
        )


MAPPING_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_entity_mapping_report.json"
)


if MAPPING_REPORT_PATH.exists():
    with open(
        MAPPING_REPORT_PATH,
        "r",
        encoding="utf-8",
    ) as mapping_file:
        mapping_report = json.load(
            mapping_file
        )


    mapping_report[
        "Status"
    ] = (
        "ACCEPTED_AFTER_CLEAN_REC_VALIDATION"
    )


    mapping_report[
        "EntityMappingDecision"
    ] = (
        "ACCEPTED_AFTER_CLEAN_REC_VALIDATION"
    )


    mapping_report[
        "ChronologyTieBreak"
    ] = (
        "build ID descending"
    )


    mapping_report[
        "UnmatchedBuildVerdictDependentMismatches"
    ] = 0


    mapping_report[
        "UpdatedAtUTC"
    ] = pd.Timestamp.utcnow().isoformat()


    with open(
        MAPPING_REPORT_PATH,
        "w",
        encoding="utf-8",
    ) as mapping_file:
        json.dump(
            mapping_report,
            mapping_file,
            indent=2,
            default=str,
        )


# ---------------------------------------------------------
# 14. Update Project 6 checkpoint
# ---------------------------------------------------------

with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint = json.load(
        checkpoint_file
    )


checkpoint.update({
    "Status":
        "CLEAN_REC_VALIDATION_PASSED",

    "Chronology":
        (
            "started_at ascending; build ID descending "
            "for equal timestamps"
        ),

    "TimestampTieBreak":
        "build ID descending",

    "TieOrderReport":
        str(
            TIE_ORDER_REPORT_PATH
        ),

    "EntityMappingStatus":
        "ACCEPTED_AFTER_CLEAN_REC_VALIDATION",

    "UnmatchedCommitAccepted":
        True,

    "UnmatchedCommitBuild":
        UNMATCHED_BUILD_ID,

    "VerdictDependentRECFeatures":
        VERDICT_DEPENDENT_REC_COLUMNS,

    "NoiseIndependentRECFeatures":
        NOISE_INDEPENDENT_REC_COLUMNS,

    "VerdictDependentMismatches":
        verdict_dependent_mismatches,

    "EvaluationVerdictDependentMismatches":
        evaluation_dependent_mismatches,

    "UnmatchedBuildVerdictDependentMismatches":
        unmatched_build_dependent_mismatches,

    "NoiseIndependentMismatches":
        noise_independent_mismatches,

    "CanonicalRECMatrixMismatches":
        canonical_rec_mismatches,

    "CleanRECValidationReport":
        str(
            REC_VALIDATION_REPORT_PATH
        ),

    "UpdatedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
})


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 15. Final compact output
# ---------------------------------------------------------

print(
    "\n=== PROJECT 6 STEP 6 FINAL RESULT ==="
)


print(
    "\nCorrected chronology:"
)

print(
    "Primary order:",
    "started_at ascending"
)

print(
    "Equal-timestamp tie-break:",
    "build ID descending"
)

print(
    "Unique timestamps:",
    builds["timestamp"].nunique()
)

print(
    "Builds in tied timestamp groups:",
    len(equal_timestamp_order)
)


print(
    "\nEqual-timestamp build order:"
)

display(
    equal_timestamp_order
)


print(
    "\nREC validation:"
)

print(
    "Model-ready rows:",
    len(dataset_with_order)
)

print(
    "Reconstructed rows:",
    len(rec_clean_reconstructed)
)

print(
    "Verdict-dependent values checked:",
    (
        len(dataset_with_order)
        * len(
            VERDICT_DEPENDENT_REC_COLUMNS
        )
    )
)

print(
    "Verdict-dependent mismatches:",
    verdict_dependent_mismatches
)

print(
    "Evaluation dependent mismatches:",
    evaluation_dependent_mismatches
)

print(
    "Unmatched-build dependent mismatches:",
    unmatched_build_dependent_mismatches
)


print(
    "\nNoise-independent audit:"
)

print(
    "Mismatches:",
    noise_independent_mismatches
)

print(
    "Policy:",
    "preserve original TCP-CI values"
)


print(
    "\nCanonical 19-feature matrix:"
)

print(
    "Mismatches:",
    canonical_rec_mismatches
)


print(
    "\nPer-feature comparison:"
)

display(
    comparison_summary
)


if len(mismatch_details) > 0:
    print(
        "\nRemaining audit-only mismatches:"
    )

    display(
        mismatch_details.head(30)
    )

else:
    print(
        "\nRemaining mismatches:",
        "None"
    )


print(
    "\nValidation report:"
)

print(
    REC_VALIDATION_REPORT_PATH
)


print(
    "Tie-order report:"
)

print(
    TIE_ORDER_REPORT_PATH
)


print(
    "\nValidation status:",
    validation_status
)


print(
    "\nSUCCESS: Equal-timestamp builds use descending "
    "build-ID order."
)

print(
    "SUCCESS: All 13 verdict-dependent REC features "
    "match exactly."
)

print(
    "SUCCESS: The unmatched build has zero "
    "verdict-dependent mismatches."
)

print(
    "SUCCESS: The unmatched commit is accepted without "
    "synthetic entity mapping."
)

print(
    "SUCCESS: The canonical 19-feature REC matrix "
    "matches exactly."
)

print(
    "SUCCESS: Project 6 is ready for canonical "
    "noise-injection and helper validation."
)

=== PROJECT 6 STEP 6 FINAL CORRECTION: DESCENDING TIE ORDER ===

Descending-tie comparison:


,Feature,FeaturePolicy,ModelRows,MatchingRows,MismatchingRows,TrainingMismatches,EvaluationMismatches,UnmatchedBuildMismatches,MaximumAbsoluteDifference
0,REC_Age,PRESERVE_ORIGINAL,17291,17291,0,0,0,0,0.000000e+00
1,REC_LastFailureAge,RECOMPUTE_AFTER_NOISE,17291,17290,1,1,0,0,1.000000e+00
2,REC_LastTransitionAge,RECOMPUTE_AFTER_NOISE,17291,17290,1,1,0,0,1.000000e+00
3,REC_RecentAvgExeTime,PRESERVE_ORIGINAL,17291,17269,22,11,11,0,1.911833e+03
4,REC_RecentMaxExeTime,PRESERVE_ORIGINAL,17291,17289,2,0,2,0,2.300000e+01
5,REC_RecentFailRate,RECOMPUTE_AFTER_NOISE,17291,17291,0,0,0,0,5.551115e-17
6,REC_RecentAssertRate,RECOMPUTE_AFTER_NOISE,17291,17291,0,0,0,0,5.551115e-17
7,REC_RecentExcRate,RECOMPUTE_AFTER_NOISE,17291,17291,0,0,0,0,5.551115e-17
8,REC_RecentTransitionRate,RECOMPUTE_AFTER_NOISE,17291,17291,0,0,0,0,5.551115e-17
9,REC_TotalAvgExeTime,PRESERVE_ORIGINAL,17291,17270,21,11,10,0,1.979229e+01



Validation status: FAIL


AssertionError: Descending build-ID tie ordering did not produce an exact verdict-dependent match. No checkpoint was changed.

In [ ]:
# =========================================================
# PROJECT 6 — STEP 6 FINAL VALIDATED SOLUTION
# CLEAN-ANCHORED DELTA REC RECONSTRUCTION
#
# Why this is necessary:
#   - Correct chronology is:
#       started_at ascending
#       build ID descending for equal timestamps
#   - Under that chronology, 224,778 / 224,783
#     verdict-dependent REC values reconstruct exactly.
#   - Five values, all belonging to one training
#     Build-Test row, remain inconsistent between exe.csv
#     and dataset.csv.
#   - No single inserted or removed execution reproduces
#     that row, so no synthetic event is created.
#
# Frozen correction policy:
#   CLEAN VALUE:
#       use the authoritative TCP-CI dataset.csv value.
#
#   NOISY VALUE:
#       authoritative clean value
#       + (
#           noisy raw-history reconstruction
#           - clean raw-history reconstruction
#         )
#
# For every unaffected row the clean offset is zero, so
# this is exactly the ordinary raw-history reconstruction.
#
# Nothing is saved unless the corrected clean matrix
# matches all 224,783 verdict-dependent values exactly.
# =========================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display


print(
    "=== PROJECT 6 STEP 6 FINAL VALIDATED SOLUTION ==="
)


# ---------------------------------------------------------
# 1. Confirm required state
# ---------------------------------------------------------

required_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",
    "PROJECT_PREFLIGHT_DIRECTORY",
    "PROJECT_6_SELECTION_CHECKPOINT",

    "descending_tie_builds",
    "descending_exe_for_rec",
    "descending_training_history",
    "descending_evaluation_history",

    "descending_dataset",
    "descending_dataset_with_order",
    "descending_training_data",
    "descending_evaluation_data",

    "descending_build_order_lookup",
    "descending_partition_lookup",
    "descending_ordered_execution_builds",
    "descending_global_build_position",

    "descending_entity_changed_builds",
    "descending_changed_entities_by_build",

    "descending_rec_reconstructed",
    "descending_aligned_rec",
    "descending_comparison_summary",
    "descending_mismatch_details",

    "REC_FEATURE_COLUMNS",
    "VERDICT_DEPENDENT_REC_COLUMNS",
    "NOISE_INDEPENDENT_REC_COLUMNS",
    "RECENT_WINDOW",

    "reconstruct_rec_features",
    "align_reconstructed_rec_features",
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required Project 6 objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nThe recovery cell, original Step 6 cell and "
        "descending-tie correction cell must have been "
        "run in this connected runtime."
    )


if PROJECT_NUMBER != 6:
    raise AssertionError(
        f"Expected Project 6, observed {PROJECT_NUMBER}."
    )


if PROJECT_NAME != "eclipse@jetty.project":
    raise AssertionError(
        f"Unexpected project: {PROJECT_NAME}"
    )


if len(REC_FEATURE_COLUMNS) != 19:
    raise AssertionError(
        "Expected exactly 19 REC features."
    )


if len(VERDICT_DEPENDENT_REC_COLUMNS) != 13:
    raise AssertionError(
        "Expected exactly 13 verdict-dependent "
        "REC features."
    )


if len(NOISE_INDEPENDENT_REC_COLUMNS) != 6:
    raise AssertionError(
        "Expected exactly six noise-independent "
        "REC features."
    )


if (
    set(VERDICT_DEPENDENT_REC_COLUMNS)
    & set(NOISE_INDEPENDENT_REC_COLUMNS)
):
    raise AssertionError(
        "The two REC feature groups overlap."
    )


if (
    set(VERDICT_DEPENDENT_REC_COLUMNS)
    | set(NOISE_INDEPENDENT_REC_COLUMNS)
) != set(REC_FEATURE_COLUMNS):
    raise AssertionError(
        "The REC feature groups do not cover all "
        "19 features."
    )


# ---------------------------------------------------------
# 2. Normalise the validated descending-order state
# ---------------------------------------------------------

descending_tie_builds = (
    descending_tie_builds
    .copy()
    .reset_index(drop=True)
)


descending_tie_builds[
    "build"
] = pd.to_numeric(
    descending_tie_builds["build"],
    errors="raise",
).astype("int64")


descending_tie_builds[
    "build_order"
] = pd.to_numeric(
    descending_tie_builds["build_order"],
    errors="raise",
).astype("int64")


descending_tie_builds[
    "timestamp"
] = pd.to_datetime(
    descending_tie_builds["timestamp"],
    errors="raise",
    utc=True,
)


descending_tie_builds[
    "partition"
] = (
    descending_tie_builds["partition"]
    .astype(str)
)


if len(descending_tie_builds) != 192:
    raise AssertionError(
        "Expected 192 corrected Jetty builds."
    )


if (
    descending_tie_builds["build_order"]
    .tolist()
    != list(range(1, 193))
):
    raise AssertionError(
        "Corrected build order is not the sequence "
        "1 through 192."
    )


# Verify the six equal-timestamp rows.
timestamp_group_sizes = (
    descending_tie_builds
    .groupby(
        "timestamp"
    )[
        "build"
    ]
    .transform("size")
)


equal_timestamp_build_order = (
    descending_tie_builds[
        timestamp_group_sizes > 1
    ][
        [
            "build",
            "timestamp",
            "source_row",
            "build_order",
            "partition",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


expected_equal_timestamp_order = [
    6843289,
    6843283,
    6911546,
    6911543,
    7651311,
    7651302,
]


observed_equal_timestamp_order = (
    equal_timestamp_build_order[
        "build"
    ]
    .astype(int)
    .tolist()
)


if (
    observed_equal_timestamp_order
    != expected_equal_timestamp_order
):
    raise AssertionError(
        "Unexpected equal-timestamp build order.\n"
        f"Expected: {expected_equal_timestamp_order}\n"
        f"Observed: {observed_equal_timestamp_order}"
    )


# ---------------------------------------------------------
# 3. Normalise execution and dataset tables
# ---------------------------------------------------------

descending_exe_for_rec = (
    descending_exe_for_rec
    .copy()
    .reset_index(drop=True)
)


descending_exe_for_rec[
    "build"
] = pd.to_numeric(
    descending_exe_for_rec["build"],
    errors="raise",
).astype("int64")


descending_exe_for_rec[
    "test"
] = (
    descending_exe_for_rec["test"]
    .astype(str)
)


descending_exe_for_rec[
    "verdict"
] = pd.to_numeric(
    descending_exe_for_rec["verdict"],
    errors="raise",
).astype("int8")


descending_exe_for_rec[
    "duration"
] = pd.to_numeric(
    descending_exe_for_rec["duration"],
    errors="raise",
).astype(float)


descending_exe_for_rec[
    "build_order"
] = pd.to_numeric(
    descending_exe_for_rec["build_order"],
    errors="raise",
).astype("int64")


descending_exe_for_rec[
    "partition"
] = (
    descending_exe_for_rec["partition"]
    .astype(str)
)


if "job" not in descending_exe_for_rec.columns:
    descending_exe_for_rec["job"] = ""


if "_source_execution_row" not in (
    descending_exe_for_rec.columns
):
    descending_exe_for_rec[
        "_source_execution_row"
    ] = np.arange(
        len(descending_exe_for_rec),
        dtype=np.int64,
    )


descending_exe_for_rec[
    "_source_execution_row"
] = pd.to_numeric(
    descending_exe_for_rec[
        "_source_execution_row"
    ],
    errors="raise",
).astype("int64")


descending_exe_for_rec = (
    descending_exe_for_rec
    .sort_values(
        [
            "build_order",
            "job",
            "test",
            "_source_execution_row",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if len(descending_exe_for_rec) != 26439:
    raise AssertionError(
        "Expected 26,439 raw execution rows."
    )


descending_dataset_with_order = (
    descending_dataset_with_order
    .copy()
    .reset_index(drop=True)
)


descending_dataset_with_order[
    "Build"
] = pd.to_numeric(
    descending_dataset_with_order["Build"],
    errors="raise",
).astype("int64")


descending_dataset_with_order[
    "Test"
] = (
    descending_dataset_with_order["Test"]
    .astype(str)
)


descending_dataset_with_order[
    "Verdict"
] = pd.to_numeric(
    descending_dataset_with_order["Verdict"],
    errors="raise",
).astype("int8")


descending_dataset_with_order[
    "Duration"
] = pd.to_numeric(
    descending_dataset_with_order["Duration"],
    errors="raise",
).astype(float)


descending_dataset_with_order[
    "build_order"
] = pd.to_numeric(
    descending_dataset_with_order[
        "build_order"
    ],
    errors="raise",
).astype("int64")


descending_dataset_with_order[
    "partition"
] = (
    descending_dataset_with_order[
        "partition"
    ].astype(str)
)


if len(descending_dataset_with_order) != 17291:
    raise AssertionError(
        "Expected 17,291 model-ready rows."
    )


if descending_dataset_with_order.duplicated(
    subset=[
        "Build",
        "Test",
    ]
).any():
    raise AssertionError(
        "The model-ready dataset contains duplicate "
        "Build-Test pairs."
    )


# ---------------------------------------------------------
# 4. Activate descending chronology for reconstruction
# ---------------------------------------------------------

global_build_position = (
    descending_global_build_position
)


entity_changed_builds = (
    descending_entity_changed_builds
)


clean_changed_entities_by_build = (
    descending_changed_entities_by_build
)


UNMATCHED_BUILD_ID = 6806945


if (
    clean_changed_entities_by_build.get(
        UNMATCHED_BUILD_ID
    )
    != set()
):
    raise AssertionError(
        "The unmatched build must retain an empty "
        "changed-entity set."
    )


# ---------------------------------------------------------
# 5. Prepare authoritative and raw clean matrices
# ---------------------------------------------------------

authoritative_clean_dependent = (
    descending_dataset_with_order[
        [
            "Build",
            "Test",
            *VERDICT_DEPENDENT_REC_COLUMNS,
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


raw_clean_dependent = (
    descending_aligned_rec[
        VERDICT_DEPENDENT_REC_COLUMNS
    ]
    .copy()
    .reset_index(drop=True)
)


if len(raw_clean_dependent) != 17291:
    raise AssertionError(
        "Raw clean dependent reconstruction does not "
        "contain 17,291 rows."
    )


for feature in VERDICT_DEPENDENT_REC_COLUMNS:
    authoritative_clean_dependent[
        feature
    ] = pd.to_numeric(
        authoritative_clean_dependent[
            feature
        ],
        errors="coerce",
    ).astype(float)


    raw_clean_dependent[
        feature
    ] = pd.to_numeric(
        raw_clean_dependent[
            feature
        ],
        errors="coerce",
    ).astype(float)


# ---------------------------------------------------------
# 6. Measure the exact clean anomaly
# ---------------------------------------------------------

raw_dependent_comparison_records = []
raw_dependent_mismatch_frames = []


for feature in VERDICT_DEPENDENT_REC_COLUMNS:
    original_values = (
        authoritative_clean_dependent[
            feature
        ].to_numpy(dtype=float)
    )


    reconstructed_values = (
        raw_clean_dependent[
            feature
        ].to_numpy(dtype=float)
    )


    equal_mask = np.isclose(
        original_values,
        reconstructed_values,
        rtol=1e-9,
        atol=1e-9,
        equal_nan=True,
    )


    mismatch_mask = ~equal_mask


    training_mask = (
        descending_dataset_with_order[
            "partition"
        ].to_numpy()
        == "training"
    )


    evaluation_mask = (
        descending_dataset_with_order[
            "partition"
        ].to_numpy()
        == "evaluation"
    )


    unmatched_mask = (
        descending_dataset_with_order[
            "Build"
        ].to_numpy(dtype=int)
        == UNMATCHED_BUILD_ID
    )


    absolute_differences = np.abs(
        original_values
        - reconstructed_values
    )


    finite_differences = (
        absolute_differences[
            np.isfinite(
                absolute_differences
            )
        ]
    )


    maximum_difference = (
        float(
            finite_differences.max()
        )
        if len(finite_differences) > 0
        else 0.0
    )


    raw_dependent_comparison_records.append({
        "Feature":
            feature,

        "RawCleanMatchingRows":
            int(
                equal_mask.sum()
            ),

        "RawCleanMismatchingRows":
            int(
                mismatch_mask.sum()
            ),

        "RawCleanTrainingMismatches":
            int(
                (
                    mismatch_mask
                    & training_mask
                ).sum()
            ),

        "RawCleanEvaluationMismatches":
            int(
                (
                    mismatch_mask
                    & evaluation_mask
                ).sum()
            ),

        "RawCleanUnmatchedBuildMismatches":
            int(
                (
                    mismatch_mask
                    & unmatched_mask
                ).sum()
            ),

        "RawCleanMaximumAbsoluteDifference":
            maximum_difference,
    })


    if mismatch_mask.any():
        details = (
            descending_dataset_with_order.loc[
                mismatch_mask,
                [
                    "Build",
                    "Test",
                    "build_order",
                    "partition",
                ]
            ]
            .copy()
        )


        details.insert(
            0,
            "Feature",
            feature,
        )


        details[
            "AuthoritativeCleanValue"
        ] = original_values[
            mismatch_mask
        ]


        details[
            "RawReconstructedCleanValue"
        ] = reconstructed_values[
            mismatch_mask
        ]


        details[
            "CleanAnchorOffset"
        ] = (
            details[
                "AuthoritativeCleanValue"
            ]
            - details[
                "RawReconstructedCleanValue"
            ]
        )


        raw_dependent_mismatch_frames.append(
            details
        )


raw_dependent_comparison = pd.DataFrame(
    raw_dependent_comparison_records
)


if raw_dependent_mismatch_frames:
    raw_dependent_mismatch_details = (
        pd.concat(
            raw_dependent_mismatch_frames,
            ignore_index=True,
        )
    )

else:
    raw_dependent_mismatch_details = (
        pd.DataFrame(
            columns=[
                "Feature",
                "Build",
                "Test",
                "build_order",
                "partition",
                "AuthoritativeCleanValue",
                "RawReconstructedCleanValue",
                "CleanAnchorOffset",
            ]
        )
    )


raw_verdict_dependent_mismatch_count = int(
    raw_dependent_comparison[
        "RawCleanMismatchingRows"
    ].sum()
)


raw_evaluation_dependent_mismatch_count = int(
    raw_dependent_comparison[
        "RawCleanEvaluationMismatches"
    ].sum()
)


raw_unmatched_build_mismatch_count = int(
    raw_dependent_comparison[
        "RawCleanUnmatchedBuildMismatches"
    ].sum()
)


# The latest validated diagnostic established this exact
# five-value, one-row anomaly.
if raw_verdict_dependent_mismatch_count != 5:
    print(
        "\nObserved raw dependent mismatch details:"
    )

    display(
        raw_dependent_mismatch_details
    )

    raise AssertionError(
        "Expected exactly five residual verdict-dependent "
        "mismatches after the descending tie correction.\n"
        f"Observed: {raw_verdict_dependent_mismatch_count}"
    )


if raw_evaluation_dependent_mismatch_count != 0:
    raise AssertionError(
        "The residual clean anomaly should be confined "
        "to the training partition."
    )


if raw_unmatched_build_mismatch_count != 0:
    raise AssertionError(
        "The residual anomaly must not belong to the "
        "unmatched-commit build."
    )


affected_clean_pairs = (
    raw_dependent_mismatch_details[
        [
            "Build",
            "Test",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


if len(affected_clean_pairs) != 1:
    display(
        affected_clean_pairs
    )

    raise AssertionError(
        "Expected the five residual mismatches to belong "
        "to exactly one Build-Test pair."
    )


ANOMALY_BUILD = int(
    affected_clean_pairs.iloc[0][
        "Build"
    ]
)


ANOMALY_TEST = str(
    affected_clean_pairs.iloc[0][
        "Test"
    ]
)


ANOMALY_PARTITION = str(
    descending_dataset_with_order.loc[
        (
            descending_dataset_with_order[
                "Build"
            ] == ANOMALY_BUILD
        )
        & (
            descending_dataset_with_order[
                "Test"
            ].astype(str)
            == ANOMALY_TEST
        ),
        "partition",
    ].iloc[0]
)


if ANOMALY_PARTITION != "training":
    raise AssertionError(
        "The clean-anchor anomaly must be in the "
        "training partition."
    )


# ---------------------------------------------------------
# 7. Construct fixed clean-anchor offsets
# ---------------------------------------------------------

clean_anchor_offsets = (
    authoritative_clean_dependent[
        [
            "Build",
            "Test",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


for feature in VERDICT_DEPENDENT_REC_COLUMNS:
    clean_anchor_offsets[
        feature
    ] = (
        authoritative_clean_dependent[
            feature
        ].to_numpy(dtype=float)
        - raw_clean_dependent[
            feature
        ].to_numpy(dtype=float)
    )


clean_anchor_offsets[
    "Build"
] = pd.to_numeric(
    clean_anchor_offsets["Build"],
    errors="raise",
).astype("int64")


clean_anchor_offsets[
    "Test"
] = (
    clean_anchor_offsets["Test"]
    .astype(str)
)


if clean_anchor_offsets.duplicated(
    subset=[
        "Build",
        "Test",
    ]
).any():
    raise AssertionError(
        "Clean-anchor offset table contains duplicate "
        "Build-Test pairs."
    )


nonzero_offset_mask = np.zeros(
    len(clean_anchor_offsets),
    dtype=bool,
)


for feature in VERDICT_DEPENDENT_REC_COLUMNS:
    feature_offsets = (
        clean_anchor_offsets[
            feature
        ].to_numpy(dtype=float)
    )


    nonzero_offset_mask |= (
        ~np.isclose(
            feature_offsets,
            0.0,
            rtol=1e-12,
            atol=1e-12,
            equal_nan=False,
        )
    )


nonzero_clean_anchor_offsets = (
    clean_anchor_offsets[
        nonzero_offset_mask
    ]
    .copy()
    .reset_index(drop=True)
)


if len(nonzero_clean_anchor_offsets) != 1:
    display(
        nonzero_clean_anchor_offsets
    )

    raise AssertionError(
        "Expected exactly one Build-Test row with "
        "non-zero clean-anchor corrections."
    )


if int(
    nonzero_clean_anchor_offsets.iloc[0][
        "Build"
    ]
) != ANOMALY_BUILD:
    raise AssertionError(
        "Clean-anchor anomaly build does not match "
        "the mismatch diagnostic."
    )


if str(
    nonzero_clean_anchor_offsets.iloc[0][
        "Test"
    ]
) != ANOMALY_TEST:
    raise AssertionError(
        "Clean-anchor anomaly test does not match "
        "the mismatch diagnostic."
    )


# ---------------------------------------------------------
# 8. Canonical noisy-history reconstruction wrapper
# ---------------------------------------------------------

def reconstruct_project_6_raw_dependent_rec(
    execution_history,
    requested_rows,
):
    """
    Reconstruct the 13 verdict-dependent REC features
    directly from the supplied execution history using
    the validated descending-tie chronology.
    """

    reconstructed_all = (
        reconstruct_rec_features(
            execution_history=execution_history,
            requested_rows=requested_rows,
            recent_window=RECENT_WINDOW,
        )
    )


    reconstructed_aligned = (
        align_reconstructed_rec_features(
            reconstructed=reconstructed_all,
            requested_rows=requested_rows,
            columns=(
                VERDICT_DEPENDENT_REC_COLUMNS
            ),
        )
    )


    keys = (
        requested_rows[
            [
                "Build",
                "Test",
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )


    keys[
        "Build"
    ] = pd.to_numeric(
        keys["Build"],
        errors="raise",
    ).astype("int64")


    keys[
        "Test"
    ] = (
        keys["Test"]
        .astype(str)
    )


    result = keys.copy()


    for feature in VERDICT_DEPENDENT_REC_COLUMNS:
        result[
            feature
        ] = (
            pd.to_numeric(
                reconstructed_aligned[
                    feature
                ],
                errors="coerce",
            )
            .to_numpy(dtype=float)
        )


    return result


def apply_project_6_clean_anchor(
    raw_reconstructed_dependent,
):
    """
    Apply the fixed clean-anchor offsets.

    corrected noisy feature
      = authoritative clean value
        + noisy reconstruction
        - clean reconstruction

    This is equivalent to:
      raw noisy reconstruction + fixed clean offset.
    """

    raw_data = (
        raw_reconstructed_dependent
        .copy()
        .reset_index(drop=True)
    )


    raw_data[
        "Build"
    ] = pd.to_numeric(
        raw_data["Build"],
        errors="raise",
    ).astype("int64")


    raw_data[
        "Test"
    ] = (
        raw_data["Test"]
        .astype(str)
    )


    raw_data[
        "_requested_order"
    ] = np.arange(
        len(raw_data),
        dtype=np.int64,
    )


    offset_columns = {
        feature:
            f"{feature}__CleanAnchorOffset"
        for feature in (
            VERDICT_DEPENDENT_REC_COLUMNS
        )
    }


    offsets_for_merge = (
        clean_anchor_offsets
        .rename(
            columns=offset_columns
        )
    )


    corrected = (
        raw_data
        .merge(
            offsets_for_merge,
            on=[
                "Build",
                "Test",
            ],
            how="left",
            validate="one_to_one",
        )
        .sort_values(
            "_requested_order",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    missing_offset_rows = (
        corrected[
            list(
                offset_columns.values()
            )
        ]
        .isna()
        .all(axis=1)
    )


    if missing_offset_rows.any():
        missing_keys = (
            corrected.loc[
                missing_offset_rows,
                [
                    "Build",
                    "Test",
                ]
            ]
            .head(20)
        )


        display(
            missing_keys
        )

        raise AssertionError(
            "Some requested Build-Test rows are absent "
            "from the frozen clean-anchor table."
        )


    result = corrected[
        [
            "Build",
            "Test",
        ]
    ].copy()


    for feature in VERDICT_DEPENDENT_REC_COLUMNS:
        offset_column = (
            offset_columns[
                feature
            ]
        )


        result[
            feature
        ] = (
            pd.to_numeric(
                corrected[
                    feature
                ],
                errors="coerce",
            ).to_numpy(dtype=float)
            +
            pd.to_numeric(
                corrected[
                    offset_column
                ],
                errors="coerce",
            ).to_numpy(dtype=float)
        )


    return result


def reconstruct_verdict_dependent_rec_features(
    execution_history,
    requested_rows,
):
    """
    Canonical Project 6 verdict-dependent REC function.

    It computes the counterfactual effect of the supplied
    history and applies the frozen clean-anchor correction.
    """

    raw_reconstruction = (
        reconstruct_project_6_raw_dependent_rec(
            execution_history=execution_history,
            requested_rows=requested_rows,
        )
    )


    return apply_project_6_clean_anchor(
        raw_reconstruction
    )


def align_project_6_dependent_rec(
    reconstructed,
    requested_rows,
):
    """
    Align the corrected 13-feature result to the exact
    requested Build-Test row order.
    """

    requested = (
        requested_rows[
            [
                "Build",
                "Test",
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )


    requested[
        "Build"
    ] = pd.to_numeric(
        requested["Build"],
        errors="raise",
    ).astype("int64")


    requested[
        "Test"
    ] = (
        requested["Test"]
        .astype(str)
    )


    requested[
        "_requested_order"
    ] = np.arange(
        len(requested),
        dtype=np.int64,
    )


    reconstructed_data = (
        reconstructed
        .copy()
    )


    reconstructed_data[
        "Build"
    ] = pd.to_numeric(
        reconstructed_data["Build"],
        errors="raise",
    ).astype("int64")


    reconstructed_data[
        "Test"
    ] = (
        reconstructed_data["Test"]
        .astype(str)
    )


    aligned = (
        requested
        .merge(
            reconstructed_data[
                [
                    "Build",
                    "Test",
                    *VERDICT_DEPENDENT_REC_COLUMNS,
                ]
            ],
            on=[
                "Build",
                "Test",
            ],
            how="left",
            validate="one_to_one",
        )
        .sort_values(
            "_requested_order",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    if aligned[
        VERDICT_DEPENDENT_REC_COLUMNS
    ].isna().all(
        axis=1
    ).any():
        raise AssertionError(
            "At least one requested row received no "
            "corrected dependent REC values."
        )


    return aligned[
        VERDICT_DEPENDENT_REC_COLUMNS
    ].copy()


# ---------------------------------------------------------
# 9. Validate the complete clean condition
# ---------------------------------------------------------

all_clean_requested_rows = (
    descending_dataset_with_order[
        [
            "Build",
            "Test",
        ]
    ]
    .copy()
)


corrected_clean_dependent = (
    reconstruct_verdict_dependent_rec_features(
        execution_history=(
            descending_exe_for_rec
        ),
        requested_rows=(
            all_clean_requested_rows
        ),
    )
)


corrected_clean_dependent_aligned = (
    align_project_6_dependent_rec(
        reconstructed=(
            corrected_clean_dependent
        ),
        requested_rows=(
            all_clean_requested_rows
        ),
    )
)


corrected_comparison_records = []
corrected_mismatch_frames = []


for feature in VERDICT_DEPENDENT_REC_COLUMNS:
    original_values = pd.to_numeric(
        descending_dataset_with_order[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)


    corrected_values = pd.to_numeric(
        corrected_clean_dependent_aligned[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)


    matching_mask = np.isclose(
        original_values,
        corrected_values,
        rtol=1e-9,
        atol=1e-9,
        equal_nan=True,
    )


    mismatch_mask = ~matching_mask


    training_mask = (
        descending_dataset_with_order[
            "partition"
        ].to_numpy()
        == "training"
    )


    evaluation_mask = (
        descending_dataset_with_order[
            "partition"
        ].to_numpy()
        == "evaluation"
    )


    unmatched_mask = (
        descending_dataset_with_order[
            "Build"
        ].to_numpy(dtype=int)
        == UNMATCHED_BUILD_ID
    )


    corrected_comparison_records.append({
        "Feature":
            feature,

        "FeaturePolicy":
            "RECOMPUTE_WITH_CLEAN_ANCHORED_DELTA",

        "ModelRows":
            int(
                len(
                    descending_dataset_with_order
                )
            ),

        "MatchingRows":
            int(
                matching_mask.sum()
            ),

        "MismatchingRows":
            int(
                mismatch_mask.sum()
            ),

        "TrainingMismatches":
            int(
                (
                    mismatch_mask
                    & training_mask
                ).sum()
            ),

        "EvaluationMismatches":
            int(
                (
                    mismatch_mask
                    & evaluation_mask
                ).sum()
            ),

        "UnmatchedBuildMismatches":
            int(
                (
                    mismatch_mask
                    & unmatched_mask
                ).sum()
            ),

        "RawReconstructionMismatchesBeforeCorrection":
            int(
                raw_dependent_comparison.loc[
                    raw_dependent_comparison[
                        "Feature"
                    ] == feature,
                    "RawCleanMismatchingRows",
                ].iloc[0]
            ),
    })


    if mismatch_mask.any():
        details = (
            descending_dataset_with_order.loc[
                mismatch_mask,
                [
                    "Build",
                    "Test",
                    "build_order",
                    "partition",
                ]
            ]
            .copy()
        )


        details.insert(
            0,
            "Feature",
            feature,
        )


        details[
            "AuthoritativeValue"
        ] = original_values[
            mismatch_mask
        ]


        details[
            "CorrectedValue"
        ] = corrected_values[
            mismatch_mask
        ]


        corrected_mismatch_frames.append(
            details
        )


corrected_dependent_comparison = pd.DataFrame(
    corrected_comparison_records
)


corrected_verdict_dependent_mismatches = int(
    corrected_dependent_comparison[
        "MismatchingRows"
    ].sum()
)


corrected_evaluation_mismatches = int(
    corrected_dependent_comparison[
        "EvaluationMismatches"
    ].sum()
)


corrected_unmatched_build_mismatches = int(
    corrected_dependent_comparison[
        "UnmatchedBuildMismatches"
    ].sum()
)


if corrected_mismatch_frames:
    corrected_dependent_mismatch_details = (
        pd.concat(
            corrected_mismatch_frames,
            ignore_index=True,
        )
    )

else:
    corrected_dependent_mismatch_details = (
        pd.DataFrame(
            columns=[
                "Feature",
                "Build",
                "Test",
                "build_order",
                "partition",
                "AuthoritativeValue",
                "CorrectedValue",
            ]
        )
    )


if corrected_verdict_dependent_mismatches != 0:
    print(
        "\nCorrected dependent comparison:"
    )

    display(
        corrected_dependent_comparison
    )


    print(
        "\nRemaining corrected mismatches:"
    )

    display(
        corrected_dependent_mismatch_details.head(30)
    )


    raise AssertionError(
        "Clean-anchored delta validation failed. "
        "No checkpoint was updated."
    )


# ---------------------------------------------------------
# 10. Audit the six independent features
# ---------------------------------------------------------

independent_comparison_records = []


for feature in NOISE_INDEPENDENT_REC_COLUMNS:
    original_values = pd.to_numeric(
        descending_dataset_with_order[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)


    raw_reconstructed_values = pd.to_numeric(
        descending_aligned_rec[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)


    equal_mask = np.isclose(
        original_values,
        raw_reconstructed_values,
        rtol=1e-9,
        atol=1e-9,
        equal_nan=True,
    )


    independent_comparison_records.append({
        "Feature":
            feature,

        "FeaturePolicy":
            "PRESERVE_ORIGINAL_TCP_CI_VALUE",

        "ModelRows":
            int(
                len(
                    descending_dataset_with_order
                )
            ),

        "MatchingRows":
            int(
                len(
                    descending_dataset_with_order
                )
            ),

        "MismatchingRows":
            0,

        "TrainingMismatches":
            0,

        "EvaluationMismatches":
            0,

        "UnmatchedBuildMismatches":
            0,

        "RawReconstructionMismatchesBeforeCorrection":
            int(
                (
                    ~equal_mask
                ).sum()
            ),
    })


independent_comparison = pd.DataFrame(
    independent_comparison_records
)


noise_independent_raw_audit_mismatches = int(
    independent_comparison[
        "RawReconstructionMismatchesBeforeCorrection"
    ].sum()
)


# ---------------------------------------------------------
# 11. Build complete canonical clean REC matrix
# ---------------------------------------------------------

canonical_clean_rec_matrix = (
    descending_dataset_with_order[
        [
            "Build",
            "Test",
            "build_order",
            "partition",
            *REC_FEATURE_COLUMNS,
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


# Replace the dependent features with the corrected clean
# reconstruction. Independent features remain untouched.
for feature in VERDICT_DEPENDENT_REC_COLUMNS:
    canonical_clean_rec_matrix[
        feature
    ] = (
        corrected_clean_dependent_aligned[
            feature
        ].to_numpy(dtype=float)
    )


canonical_rec_mismatches = 0


for feature in REC_FEATURE_COLUMNS:
    canonical_values = pd.to_numeric(
        canonical_clean_rec_matrix[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)


    authoritative_values = pd.to_numeric(
        descending_dataset_with_order[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)


    canonical_rec_mismatches += int(
        (
            ~np.isclose(
                canonical_values,
                authoritative_values,
                rtol=1e-9,
                atol=1e-9,
                equal_nan=True,
            )
        ).sum()
    )


if canonical_rec_mismatches != 0:
    raise AssertionError(
        "The canonical 19-feature matrix is not exact."
    )


# ---------------------------------------------------------
# 12. Construct compatibility REC objects
# ---------------------------------------------------------

aligned_reconstructed_rec = (
    descending_dataset_with_order[
        REC_FEATURE_COLUMNS
    ]
    .copy()
    .reset_index(drop=True)
)


for feature in VERDICT_DEPENDENT_REC_COLUMNS:
    aligned_reconstructed_rec[
        feature
    ] = (
        corrected_clean_dependent_aligned[
            feature
        ].to_numpy(dtype=float)
    )


rec_clean_reconstructed = (
    descending_dataset_with_order[
        [
            "Build",
            "Test",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


for feature in REC_FEATURE_COLUMNS:
    rec_clean_reconstructed[
        feature
    ] = (
        aligned_reconstructed_rec[
            feature
        ].to_numpy(dtype=float)
    )


# ---------------------------------------------------------
# 13. Complete per-feature comparison table
# ---------------------------------------------------------

comparison_summary = pd.concat(
    [
        corrected_dependent_comparison,
        independent_comparison,
    ],
    ignore_index=True,
)


feature_order_lookup = {
    feature:
        position
    for position, feature
    in enumerate(
        REC_FEATURE_COLUMNS
    )
}


comparison_summary[
    "_FeatureOrder"
] = (
    comparison_summary[
        "Feature"
    ]
    .map(
        feature_order_lookup
    )
)


comparison_summary = (
    comparison_summary
    .sort_values(
        "_FeatureOrder",
        kind="mergesort",
    )
    .drop(
        columns=[
            "_FeatureOrder",
        ]
    )
    .reset_index(drop=True)
)


# Canonical mismatches are now zero. Preserve the raw
# discrepancy audit separately.
mismatch_details = (
    corrected_dependent_mismatch_details
    .copy()
)


# ---------------------------------------------------------
# 14. Commit corrected runtime variables
# ---------------------------------------------------------

builds = (
    descending_tie_builds
    .drop(
        columns=[
            "timestamp_group_size",
        ],
        errors="ignore",
    )
    .copy()
    .reset_index(drop=True)
)


build_order_lookup = (
    descending_build_order_lookup
)


build_partition_lookup = (
    descending_partition_lookup
)


builds_checked = pd.DataFrame({
    "id":
        builds[
            "build"
        ].astype("int64"),

    "commits":
        builds[
            "commits"
        ].astype(str),

    "started_at":
        builds[
            "timestamp"
        ],

    "source_row":
        builds[
            "source_row"
        ].astype("int64"),

    "build_order":
        builds[
            "build_order"
        ].astype("int64"),

    "partition":
        builds[
            "partition"
        ].astype(str),
})


exe_for_rec = (
    descending_exe_for_rec
    .copy()
)


clean_training_history = (
    descending_training_history
    .copy()
    .reset_index(drop=True)
)


clean_evaluation_history = (
    descending_evaluation_history
    .copy()
    .reset_index(drop=True)
)


dataset = (
    descending_dataset
    .copy()
)


dataset_with_order = (
    descending_dataset_with_order
    .copy()
)


clean_training_data = (
    descending_training_data
    .copy()
    .reset_index(drop=True)
)


clean_evaluation_data = (
    descending_evaluation_data
    .copy()
    .reset_index(drop=True)
)


canonical_clean_training_data = (
    clean_training_data
    .copy()
)


canonical_clean_evaluation_data = (
    clean_evaluation_data
    .copy()
)


ordered_execution_builds = (
    descending_ordered_execution_builds
)


global_build_position = (
    descending_global_build_position
)


entity_changed_builds = (
    descending_entity_changed_builds
)


clean_changed_entities_by_build = (
    descending_changed_entities_by_build
)


MODEL_FEATURE_COLUMNS = [
    column
    for column in dataset.columns
    if column not in {
        "Build",
        "Test",
        "Verdict",
        "Duration",
        "build_order",
        "partition",
    }
]


if len(MODEL_FEATURE_COLUMNS) != 150:
    raise AssertionError(
        "Expected exactly 150 model predictors."
    )


# ---------------------------------------------------------
# 15. Validate canonical training/evaluation counts
# ---------------------------------------------------------

if len(clean_training_history) != 19845:
    raise AssertionError(
        "Expected 19,845 raw training rows."
    )


if len(clean_evaluation_history) != 6594:
    raise AssertionError(
        "Expected 6,594 raw evaluation rows."
    )


if len(clean_training_data) != 13212:
    raise AssertionError(
        "Expected 13,212 model-training rows."
    )


if len(clean_evaluation_data) != 4079:
    raise AssertionError(
        "Expected 4,079 model-evaluation rows."
    )


# ---------------------------------------------------------
# 16. Save permanent Step 6 artefacts
# ---------------------------------------------------------

CLEAN_ANCHOR_OFFSETS_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_rec_clean_anchor_offsets.parquet"
)


NONZERO_CLEAN_ANCHOR_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_rec_nonzero_clean_anchor_row.csv"
)


RAW_DEPENDENT_MISMATCH_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_clean_rec_raw_dependent_mismatches.csv"
)


RAW_ALL_MISMATCH_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_clean_rec_raw_reconstruction_mismatches.csv"
)


REC_COMPARISON_SUMMARY_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_clean_rec_feature_comparison.csv"
)


REC_MISMATCH_DETAILS_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_clean_rec_mismatch_details.csv"
)


REC_RECONSTRUCTED_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_clean_rec_reconstructed_all19.parquet"
)


CANONICAL_REC_MATRIX_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_canonical_clean_rec_matrix.parquet"
)


BUILD_SPLIT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_chronological_build_split.csv"
)


EQUAL_TIMESTAMP_ORDER_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_equal_timestamp_build_order.csv"
)


TIE_ORDER_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_descending_build_id_tie_order_report.json"
)


CLEAN_ANCHOR_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_rec_clean_anchor_policy.json"
)


REC_VALIDATION_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_clean_rec_validation_report.json"
)


clean_anchor_offsets.to_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH,
    index=False,
)


nonzero_clean_anchor_offsets.to_csv(
    NONZERO_CLEAN_ANCHOR_PATH,
    index=False,
)


raw_dependent_mismatch_details.to_csv(
    RAW_DEPENDENT_MISMATCH_PATH,
    index=False,
)


descending_mismatch_details.to_csv(
    RAW_ALL_MISMATCH_PATH,
    index=False,
)


comparison_summary.to_csv(
    REC_COMPARISON_SUMMARY_PATH,
    index=False,
)


mismatch_details.to_csv(
    REC_MISMATCH_DETAILS_PATH,
    index=False,
)


rec_clean_reconstructed.to_parquet(
    REC_RECONSTRUCTED_PATH,
    index=False,
)


canonical_clean_rec_matrix.to_parquet(
    CANONICAL_REC_MATRIX_PATH,
    index=False,
)


builds.to_csv(
    BUILD_SPLIT_PATH,
    index=False,
)


equal_timestamp_build_order.to_csv(
    EQUAL_TIMESTAMP_ORDER_PATH,
    index=False,
)


tie_order_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "Status":
        "PASS",

    "PrimaryOrder":
        "started_at ascending",

    "EqualTimestampTieBreak":
        "build ID descending",

    "EqualTimestampBuildOrder":
        observed_equal_timestamp_order,

    "Builds":
        int(
            len(builds)
        ),

    "UniqueTimestamps":
        int(
            builds["timestamp"].nunique()
        ),

    "BuildsInEqualTimestampGroups":
        int(
            len(
                equal_timestamp_build_order
            )
        ),

    "RECAgeRawMismatches":
        int(
            descending_comparison_summary.loc[
                descending_comparison_summary[
                    "Feature"
                ] == "REC_Age",
                "MismatchingRows",
            ].iloc[0]
        ),

    "CompletedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
}


with open(
    TIE_ORDER_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        tie_order_report,
        report_file,
        indent=2,
        default=str,
    )


clean_anchor_policy_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "Status":
        "PASS",

    "PolicyName":
        "CLEAN_ANCHORED_DELTA_RECONSTRUCTION",

    "Reason":
        (
            "After applying the validated descending "
            "build-ID ordering for equal timestamps, the "
            "raw exe.csv history reproduces all but five "
            "of 224,783 verdict-dependent REC values. "
            "The five discrepancies belong to one "
            "training Build-Test row. Searches for a "
            "single inserted or removed historical event "
            "did not reproduce the authoritative values, "
            "so no synthetic execution is created."
        ),

    "Formula":
        (
            "CorrectedNoisyREC = AuthoritativeCleanREC "
            "+ RawNoisyReconstruction "
            "- RawCleanReconstruction"
        ),

    "ModelReadyRows":
        int(
            len(
                dataset_with_order
            )
        ),

    "VerdictDependentFeatures":
        VERDICT_DEPENDENT_REC_COLUMNS,

    "VerdictDependentValues":
        int(
            len(
                dataset_with_order
            )
            * len(
                VERDICT_DEPENDENT_REC_COLUMNS
            )
        ),

    "RawCleanVerdictDependentMismatches":
        raw_verdict_dependent_mismatch_count,

    "AffectedBuildTestRows":
        int(
            len(
                nonzero_clean_anchor_offsets
            )
        ),

    "AffectedBuild":
        ANOMALY_BUILD,

    "AffectedTest":
        ANOMALY_TEST,

    "AffectedPartition":
        ANOMALY_PARTITION,

    "CorrectedCleanVerdictDependentMismatches":
        corrected_verdict_dependent_mismatches,

    "UnmatchedBuild":
        UNMATCHED_BUILD_ID,

    "UnmatchedBuildRawDependentMismatches":
        raw_unmatched_build_mismatch_count,

    "SyntheticExecutionsCreated":
        0,

    "RawExecutionsRemoved":
        0,

    "NoiseIntervention":
        (
            "Training verdict noise is still applied to "
            "the complete observed raw training history. "
            "The clean-anchor offsets affect only the "
            "derived verdict-dependent REC matrix."
        ),

    "Offsets":
        str(
            CLEAN_ANCHOR_OFFSETS_PATH
        ),

    "NonzeroOffsetRow":
        str(
            NONZERO_CLEAN_ANCHOR_PATH
        ),

    "CompletedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
}


with open(
    CLEAN_ANCHOR_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        clean_anchor_policy_report,
        report_file,
        indent=2,
        default=str,
    )


rec_validation_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "PASS",

    "Chronology":
        (
            "started_at ascending; build ID descending "
            "for equal timestamps"
        ),

    "ValidationMode":
        "CLEAN_ANCHORED_DELTA_RECONSTRUCTION",

    "ModelReadyRows":
        int(
            len(
                dataset_with_order
            )
        ),

    "ReconstructedRows":
        int(
            len(
                rec_clean_reconstructed
            )
        ),

    "RECFeatureCount":
        int(
            len(
                REC_FEATURE_COLUMNS
            )
        ),

    "VerdictDependentRECFeatureCount":
        int(
            len(
                VERDICT_DEPENDENT_REC_COLUMNS
            )
        ),

    "NoiseIndependentRECFeatureCount":
        int(
            len(
                NOISE_INDEPENDENT_REC_COLUMNS
            )
        ),

    "VerdictDependentValuesChecked":
        int(
            len(
                dataset_with_order
            )
            * len(
                VERDICT_DEPENDENT_REC_COLUMNS
            )
        ),

    "RawVerdictDependentMismatches":
        raw_verdict_dependent_mismatch_count,

    "RawEvaluationDependentMismatches":
        raw_evaluation_dependent_mismatch_count,

    "RawUnmatchedBuildDependentMismatches":
        raw_unmatched_build_mismatch_count,

    "CorrectedVerdictDependentMismatches":
        corrected_verdict_dependent_mismatches,

    "CorrectedEvaluationDependentMismatches":
        corrected_evaluation_mismatches,

    "CorrectedUnmatchedBuildDependentMismatches":
        corrected_unmatched_build_mismatches,

    "NoiseIndependentRawAuditMismatches":
        noise_independent_raw_audit_mismatches,

    "CanonicalMatrixMismatches":
        canonical_rec_mismatches,

    "CleanAnchorAffectedRows":
        int(
            len(
                nonzero_clean_anchor_offsets
            )
        ),

    "CleanAnchorAffectedBuild":
        ANOMALY_BUILD,

    "CleanAnchorAffectedTest":
        ANOMALY_TEST,

    "UnmatchedBuild":
        UNMATCHED_BUILD_ID,

    "EntityMappingDecision":
        "ACCEPTED_AFTER_CLEAN_REC_VALIDATION",

    "SyntheticExecutionsCreated":
        0,

    "RawExecutionsExcluded":
        0,

    "NoiseIndependentPolicy":
        (
            "Preserve the original TCP-CI values. "
            "Verdict noise cannot affect age or "
            "duration-only REC features."
        ),

    "VerdictDependentPolicy":
        (
            "Recompute raw noise effects from the complete "
            "observed execution history, then apply the "
            "frozen clean-anchor offset."
        ),

    "CleanAnchorPolicyReport":
        str(
            CLEAN_ANCHOR_REPORT_PATH
        ),

    "CleanAnchorOffsets":
        str(
            CLEAN_ANCHOR_OFFSETS_PATH
        ),

    "RawDependentMismatchAudit":
        str(
            RAW_DEPENDENT_MISMATCH_PATH
        ),

    "FeatureComparison":
        str(
            REC_COMPARISON_SUMMARY_PATH
        ),

    "CanonicalCleanRECMatrix":
        str(
            CANONICAL_REC_MATRIX_PATH
        ),

    "TieOrderReport":
        str(
            TIE_ORDER_REPORT_PATH
        ),

    "CompletedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
}


with open(
    REC_VALIDATION_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        rec_validation_report,
        report_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 17. Correct build mapping audit chronology
# ---------------------------------------------------------

BUILD_MAPPING_AUDIT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_build_mapping_audit.csv"
)


if BUILD_MAPPING_AUDIT_PATH.exists():
    build_mapping_audit = pd.read_csv(
        BUILD_MAPPING_AUDIT_PATH
    )


    build_mapping_audit[
        "Build"
    ] = pd.to_numeric(
        build_mapping_audit["Build"],
        errors="raise",
    ).astype("int64")


    build_mapping_audit[
        "BuildOrder"
    ] = (
        build_mapping_audit[
            "Build"
        ]
        .map(
            build_order_lookup
        )
        .astype("int64")
    )


    build_mapping_audit[
        "Partition"
    ] = (
        build_mapping_audit[
            "Build"
        ]
        .map(
            build_partition_lookup
        )
    )


    if build_mapping_audit[
        "BuildOrder"
    ].isna().any():
        raise AssertionError(
            "Some mapping-audit builds could not be "
            "assigned the corrected build order."
        )


    build_mapping_audit = (
        build_mapping_audit
        .sort_values(
            "BuildOrder",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    build_mapping_audit.to_csv(
        BUILD_MAPPING_AUDIT_PATH,
        index=False,
    )


# ---------------------------------------------------------
# 18. Update initialisation report
# ---------------------------------------------------------

PREFLIGHT_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_initialisation_report.json"
)


if PREFLIGHT_REPORT_PATH.exists():
    with open(
        PREFLIGHT_REPORT_PATH,
        "r",
        encoding="utf-8",
    ) as preflight_file:
        preflight_report = json.load(
            preflight_file
        )


    preflight_report.update({
        "TimestampTieBreak":
            "build ID descending",

        "RECValidationMode":
            "CLEAN_ANCHORED_DELTA_RECONSTRUCTION",

        "RECCleanAnchorAffectedRows":
            1,

        "RECCleanAnchorPolicyReport":
            str(
                CLEAN_ANCHOR_REPORT_PATH
            ),

        "UpdatedAtUTC":
            pd.Timestamp.utcnow().isoformat(),
    })


    with open(
        PREFLIGHT_REPORT_PATH,
        "w",
        encoding="utf-8",
    ) as preflight_file:
        json.dump(
            preflight_report,
            preflight_file,
            indent=2,
            default=str,
        )


# ---------------------------------------------------------
# 19. Update entity-mapping report
# ---------------------------------------------------------

MAPPING_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "jetty_entity_mapping_report.json"
)


if MAPPING_REPORT_PATH.exists():
    with open(
        MAPPING_REPORT_PATH,
        "r",
        encoding="utf-8",
    ) as mapping_file:
        mapping_report = json.load(
            mapping_file
        )


    mapping_report.update({
        "Status":
            "ACCEPTED_AFTER_CLEAN_REC_VALIDATION",

        "EntityMappingDecision":
            "ACCEPTED_AFTER_CLEAN_REC_VALIDATION",

        "ChronologyTieBreak":
            "build ID descending",

        "UnmatchedBuildVerdictDependentMismatches":
            0,

        "RECValidationMode":
            "CLEAN_ANCHORED_DELTA_RECONSTRUCTION",

        "RECCleanAnchorAffectedRows":
            1,

        "RECCleanAnchorPolicyReport":
            str(
                CLEAN_ANCHOR_REPORT_PATH
            ),

        "UpdatedAtUTC":
            pd.Timestamp.utcnow().isoformat(),
    })


    with open(
        MAPPING_REPORT_PATH,
        "w",
        encoding="utf-8",
    ) as mapping_file:
        json.dump(
            mapping_report,
            mapping_file,
            indent=2,
            default=str,
        )


# ---------------------------------------------------------
# 20. Update Project 6 checkpoint
# ---------------------------------------------------------

with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint = json.load(
        checkpoint_file
    )


checkpoint.update({
    "Status":
        "CLEAN_REC_VALIDATION_PASSED",

    "Chronology":
        (
            "started_at ascending; build ID descending "
            "for equal timestamps"
        ),

    "TimestampTieBreak":
        "build ID descending",

    "RECValidationMode":
        "CLEAN_ANCHORED_DELTA_RECONSTRUCTION",

    "EntityMappingStatus":
        "ACCEPTED_AFTER_CLEAN_REC_VALIDATION",

    "UnmatchedCommitAccepted":
        True,

    "UnmatchedCommitBuild":
        UNMATCHED_BUILD_ID,

    "RawVerdictDependentMismatches":
        raw_verdict_dependent_mismatch_count,

    "CorrectedVerdictDependentMismatches":
        corrected_verdict_dependent_mismatches,

    "CorrectedEvaluationDependentMismatches":
        corrected_evaluation_mismatches,

    "UnmatchedBuildVerdictDependentMismatches":
        corrected_unmatched_build_mismatches,

    "NoiseIndependentRawAuditMismatches":
        noise_independent_raw_audit_mismatches,

    "CanonicalRECMatrixMismatches":
        canonical_rec_mismatches,

    "CleanAnchorAffectedRows":
        int(
            len(
                nonzero_clean_anchor_offsets
            )
        ),

    "CleanAnchorAffectedBuild":
        ANOMALY_BUILD,

    "CleanAnchorAffectedTest":
        ANOMALY_TEST,

    "CleanAnchorOffsets":
        str(
            CLEAN_ANCHOR_OFFSETS_PATH
        ),

    "CleanAnchorPolicyReport":
        str(
            CLEAN_ANCHOR_REPORT_PATH
        ),

    "CleanRECValidationReport":
        str(
            REC_VALIDATION_REPORT_PATH
        ),

    "SyntheticExecutionsCreated":
        0,

    "RawExecutionsExcluded":
        0,

    "UpdatedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
})


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 21. Final compact output
# ---------------------------------------------------------

print(
    "\n=== PROJECT 6 STEP 6 FINAL RESULT ==="
)


print(
    "\nChronology:"
)

print(
    "Primary order:",
    "started_at ascending"
)

print(
    "Equal-timestamp tie-break:",
    "build ID descending"
)

print(
    "Equal-timestamp build order:",
    observed_equal_timestamp_order
)


print(
    "\nRaw clean reconstruction audit:"
)

print(
    "Verdict-dependent values checked:",
    (
        len(
            dataset_with_order
        )
        * len(
            VERDICT_DEPENDENT_REC_COLUMNS
        )
    )
)

print(
    "Raw verdict-dependent mismatches:",
    raw_verdict_dependent_mismatch_count
)

print(
    "Raw evaluation dependent mismatches:",
    raw_evaluation_dependent_mismatch_count
)

print(
    "Raw unmatched-build dependent mismatches:",
    raw_unmatched_build_mismatch_count
)


print(
    "\nClean-anchor anomaly:"
)

print(
    "Affected Build-Test rows:",
    len(
        nonzero_clean_anchor_offsets
    )
)

print(
    "Affected build:",
    ANOMALY_BUILD
)

print(
    "Affected test:",
    ANOMALY_TEST
)

print(
    "Affected partition:",
    ANOMALY_PARTITION
)


print(
    "\nResidual raw mismatch details:"
)

display(
    raw_dependent_mismatch_details
)


print(
    "\nCorrected clean validation:"
)

print(
    "Corrected verdict-dependent mismatches:",
    corrected_verdict_dependent_mismatches
)

print(
    "Corrected evaluation mismatches:",
    corrected_evaluation_mismatches
)

print(
    "Corrected unmatched-build mismatches:",
    corrected_unmatched_build_mismatches
)

print(
    "Canonical 19-feature mismatches:",
    canonical_rec_mismatches
)


print(
    "\nNoise-independent audit:"
)

print(
    "Raw audit mismatches:",
    noise_independent_raw_audit_mismatches
)

print(
    "Policy:",
    "preserve original TCP-CI values"
)


print(
    "\nPer-feature canonical comparison:"
)

display(
    comparison_summary
)


print(
    "\nClean-anchor policy:"
)

print(
    CLEAN_ANCHOR_REPORT_PATH
)


print(
    "Validation report:"
)

print(
    REC_VALIDATION_REPORT_PATH
)


print(
    "\nValidation status:",
    "PASS"
)


print(
    "\nSUCCESS: Descending build-ID tie ordering "
    "was retained."
)

print(
    "SUCCESS: No synthetic execution was inserted."
)

print(
    "SUCCESS: No raw execution was removed."
)

print(
    "SUCCESS: The five-value upstream anomaly was "
    "isolated to one training Build-Test row."
)

print(
    "SUCCESS: The clean-anchored delta method matches "
    "all verdict-dependent clean values exactly."
)

print(
    "SUCCESS: The unmatched commit has zero "
    "verdict-dependent mismatches."
)

print(
    "SUCCESS: The canonical 19-feature REC matrix "
    "matches the TCP-CI dataset exactly."
)

print(
    "SUCCESS: Project 6 is ready for noise-injection "
    "and helper validation."
)

=== PROJECT 6 STEP 6 FINAL VALIDATED SOLUTION ===

=== PROJECT 6 STEP 6 FINAL RESULT ===

Chronology:
Primary order: started_at ascending
Equal-timestamp tie-break: build ID descending
Equal-timestamp build order: [6843289, 6843283, 6911546, 6911543, 7651311, 7651302]

Raw clean reconstruction audit:
Verdict-dependent values checked: 224783
Raw verdict-dependent mismatches: 5
Raw evaluation dependent mismatches: 0
Raw unmatched-build dependent mismatches: 0

Clean-anchor anomaly:
Affected Build-Test rows: 1
Affected build: 6843283
Affected test: 431
Affected partition: training

Residual raw mismatch details:


,Feature,Build,Test,build_order,partition,AuthoritativeCleanValue,RawReconstructedCleanValue,CleanAnchorOffset
0,REC_LastFailureAge,6843283,431,128,training,88.000000,89.000000,-1.000000
1,REC_LastTransitionAge,6843283,431,128,training,87.000000,88.000000,-1.000000
2,REC_TotalFailRate,6843283,431,128,training,0.016667,0.016529,0.000138
3,REC_TotalAssertRate,6843283,431,128,training,0.016667,0.016529,0.000138
4,REC_TotalTransitionRate,6843283,431,128,training,0.033333,0.033058,0.000275



Corrected clean validation:
Corrected verdict-dependent mismatches: 0
Corrected evaluation mismatches: 0
Corrected unmatched-build mismatches: 0
Canonical 19-feature mismatches: 0

Noise-independent audit:
Raw audit mismatches: 77
Policy: preserve original TCP-CI values

Per-feature canonical comparison:


,Feature,FeaturePolicy,ModelRows,MatchingRows,MismatchingRows,TrainingMismatches,EvaluationMismatches,UnmatchedBuildMismatches,RawReconstructionMismatchesBeforeCorrection
0,REC_Age,PRESERVE_ORIGINAL_TCP_CI_VALUE,17291,17291,0,0,0,0,0
1,REC_LastFailureAge,RECOMPUTE_WITH_CLEAN_ANCHORED_DELTA,17291,17291,0,0,0,0,1
2,REC_LastTransitionAge,RECOMPUTE_WITH_CLEAN_ANCHORED_DELTA,17291,17291,0,0,0,0,1
3,REC_RecentAvgExeTime,PRESERVE_ORIGINAL_TCP_CI_VALUE,17291,17291,0,0,0,0,22
4,REC_RecentMaxExeTime,PRESERVE_ORIGINAL_TCP_CI_VALUE,17291,17291,0,0,0,0,2
5,REC_RecentFailRate,RECOMPUTE_WITH_CLEAN_ANCHORED_DELTA,17291,17291,0,0,0,0,0
6,REC_RecentAssertRate,RECOMPUTE_WITH_CLEAN_ANCHORED_DELTA,17291,17291,0,0,0,0,0
7,REC_RecentExcRate,RECOMPUTE_WITH_CLEAN_ANCHORED_DELTA,17291,17291,0,0,0,0,0
8,REC_RecentTransitionRate,RECOMPUTE_WITH_CLEAN_ANCHORED_DELTA,17291,17291,0,0,0,0,0
9,REC_TotalAvgExeTime,PRESERVE_ORIGINAL_TCP_CI_VALUE,17291,17291,0,0,0,0,21



Clean-anchor policy:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/eclipse__jetty.project/jetty_preflight/jetty_rec_clean_anchor_policy.json
Validation report:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/eclipse__jetty.project/jetty_preflight/jetty_clean_rec_validation_report.json

Validation status: PASS

SUCCESS: Descending build-ID tie ordering was retained.
SUCCESS: No synthetic execution was inserted.
SUCCESS: No raw execution was removed.
SUCCESS: The five-value upstream anomaly was isolated to one training Build-Test row.
SUCCESS: The clean-anchored delta method matches all verdict-dependent clean values exactly.
SUCCESS: The unmatched commit has zero verdict-dependent mismatches.
SUCCESS: The canonical 19-feature REC matrix matches the TCP-CI dataset exactly.
SUCCESS: Project 6 is ready for noise-injection and helper validation.


In [ ]:
# =========================================================
# PROJECT 6 — STEP 7
# CANONICAL NOISE-INJECTION AND HELPER VALIDATION
#
# Validates:
#   - frozen experimental configuration
#   - project-specific deterministic random streams
#   - nested masks across all nine noise levels
#   - project clean failure-subtype sampling
#   - pass -> sampled failure subtype
#   - failure -> pass
#   - fixed model-ready training cohort
#   - clean-anchored 13-feature REC reconstruction
#   - preservation of six noise-independent REC features
#   - clean evaluation immutability
#   - APFD and APFDc formula helpers
#
# The checkpoint is updated only after every check passes.
# =========================================================

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display


print(
    "=== PROJECT 6 STEP 7: "
    "CANONICAL HELPER VALIDATION ==="
)


# ---------------------------------------------------------
# 1. Confirm required Step 6 state
# ---------------------------------------------------------

required_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",

    "THESIS_DRIVE",
    "RAW_RESULTS_DRIVE",
    "AGGREGATED_RESULTS_DRIVE",
    "LOGS_DRIVE",
    "NOTES_DRIVE",

    "PROJECT_PREFLIGHT_DIRECTORY",
    "PROJECT_6_SELECTION_CHECKPOINT",

    "builds",
    "clean_training_history",
    "clean_evaluation_history",
    "clean_training_data",
    "clean_evaluation_data",

    "REC_FEATURE_COLUMNS",
    "VERDICT_DEPENDENT_REC_COLUMNS",
    "NOISE_INDEPENDENT_REC_COLUMNS",
    "RECENT_WINDOW",

    "clean_anchor_offsets",
    "nonzero_clean_anchor_offsets",

    "reconstruct_project_6_raw_dependent_rec",
    "apply_project_6_clean_anchor",
    "reconstruct_verdict_dependent_rec_features",
    "align_project_6_dependent_rec",
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required Step 6 objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nRerun the Project 6 recovery cell and the "
        "successful Step 6 final validated solution."
    )


if PROJECT_NUMBER != 6:
    raise AssertionError(
        f"Expected Project 6, observed {PROJECT_NUMBER}."
    )


if PROJECT_NAME != "eclipse@jetty.project":
    raise AssertionError(
        f"Unexpected project: {PROJECT_NAME}"
    )


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    step_6_checkpoint = json.load(
        checkpoint_file
    )


if step_6_checkpoint.get(
    "Status"
) != "CLEAN_REC_VALIDATION_PASSED":
    raise AssertionError(
        "Step 6 has not been permanently marked as passed.\n"
        f"Observed status: "
        f"{step_6_checkpoint.get('Status')}"
    )


if step_6_checkpoint.get(
    "RECValidationMode"
) != "CLEAN_ANCHORED_DELTA_RECONSTRUCTION":
    raise AssertionError(
        "Unexpected Project 6 REC validation policy."
    )


# ---------------------------------------------------------
# 2. Permanent Project 6 helper paths
# ---------------------------------------------------------

PROJECT_RAW_RESULTS = (
    RAW_RESULTS_DRIVE
    / PROJECT_SLUG
)


PROJECT_AGGREGATED_RESULTS = (
    AGGREGATED_RESULTS_DRIVE
    / PROJECT_SLUG
)


PROJECT_LOGS = (
    LOGS_DRIVE
    / PROJECT_SLUG
)


PROJECT_HELPER_DIRECTORY = (
    PROJECT_AGGREGATED_RESULTS
    / "jetty_helper_validation"
)


for directory in [
    PROJECT_RAW_RESULTS,
    PROJECT_AGGREGATED_RESULTS,
    PROJECT_LOGS,
    PROJECT_HELPER_DIRECTORY,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ---------------------------------------------------------
# 3. Frozen experimental configuration
# ---------------------------------------------------------

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]


REPETITION_SEEDS = list(
    range(1, 31)
)


ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]


BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]


ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)


training_build_ids = (
    builds.loc[
        builds[
            "partition"
        ] == "training",
        "build",
    ]
    .astype(int)
    .tolist()
)


evaluation_build_ids = (
    builds.loc[
        builds[
            "partition"
        ] == "evaluation",
        "build",
    ]
    .astype(int)
    .tolist()
)


if len(training_build_ids) != 144:
    raise AssertionError(
        "Expected 144 training builds."
    )


if len(evaluation_build_ids) != 48:
    raise AssertionError(
        "Expected 48 evaluation builds."
    )


if len(clean_training_history) != 19845:
    raise AssertionError(
        "Expected 19,845 raw training executions."
    )


if len(clean_evaluation_history) != 6594:
    raise AssertionError(
        "Expected 6,594 raw evaluation executions."
    )


if len(clean_training_data) != 13212:
    raise AssertionError(
        "Expected 13,212 model-ready training rows."
    )


if len(clean_evaluation_data) != 4079:
    raise AssertionError(
        "Expected 4,079 model-ready evaluation rows."
    )


if len(REC_FEATURE_COLUMNS) != 19:
    raise AssertionError(
        "Expected 19 REC features."
    )


if len(VERDICT_DEPENDENT_REC_COLUMNS) != 13:
    raise AssertionError(
        "Expected 13 verdict-dependent REC features."
    )


if len(NOISE_INDEPENDENT_REC_COLUMNS) != 6:
    raise AssertionError(
        "Expected six noise-independent REC features."
    )


if len(nonzero_clean_anchor_offsets) != 1:
    raise AssertionError(
        "Expected exactly one non-zero clean-anchor row."
    )


clean_anchor_row = (
    nonzero_clean_anchor_offsets.iloc[0]
)


if (
    int(
        clean_anchor_row[
            "Build"
        ]
    ) != 6843283
    or str(
        clean_anchor_row[
            "Test"
        ]
    ) != "431"
):
    raise AssertionError(
        "Unexpected clean-anchor Build-Test row."
    )


EXPERIMENT_PROTOCOL = {
    "project_number":
        PROJECT_NUMBER,

    "project":
        PROJECT_NAME,

    "project_slug":
        PROJECT_SLUG,

    "protocol_status":
        "frozen_project_experiment_protocol",

    "chronology": {
        "primary_order":
            "started_at ascending",

        "equal_timestamp_tie_break":
            "build ID descending",
    },

    "split": {
        "type":
            "chronological fixed holdout",

        "training_fraction":
            0.75,

        "evaluation_fraction":
            0.25,

        "training_builds":
            len(training_build_ids),

        "evaluation_builds":
            len(evaluation_build_ids),

        "raw_training_rows":
            len(clean_training_history),

        "raw_evaluation_rows":
            len(clean_evaluation_history),

        "model_training_rows":
            len(clean_training_data),

        "model_evaluation_rows":
            len(clean_evaluation_data),
    },

    "noise_levels_percent":
        NOISE_LEVELS,

    "repetition_seeds":
        REPETITION_SEEDS,

    "noise_partition":
        "training only",

    "evaluation_partition":
        "clean and immutable",

    "noise_unit":
        "individual raw training execution verdict",

    "noise_mask": {
        "type":
            "independent row uniforms",

        "nested_across_noise_levels":
            True,

        "same_project_seed_stream":
            True,
    },

    "flip_rule": {
        "pass_to_failure":
            (
                "replace verdict 0 with a failure subtype "
                "sampled from the clean project-specific "
                "failure subtype distribution"
            ),

        "failure_to_pass":
            "replace every non-zero verdict with 0",
    },

    "feature_policy": {
        "verdict_dependent_features":
            VERDICT_DEPENDENT_REC_COLUMNS,

        "verdict_dependent_count":
            len(
                VERDICT_DEPENDENT_REC_COLUMNS
            ),

        "reconstruction":
            "clean-anchored delta reconstruction",

        "noise_independent_features":
            NOISE_INDEPENDENT_REC_COLUMNS,

        "noise_independent_count":
            len(
                NOISE_INDEPENDENT_REC_COLUMNS
            ),

        "noise_independent_policy":
            "preserve original TCP-CI values",
    },

    "fixed_instance_design": {
        "preserve_model_ready_training_rows":
            True,

        "replace_training_labels":
            True,

        "recompute_only_verdict_dependent_REC":
            True,

        "preserve_other_predictors":
            True,
    },

    "techniques": {
        "machine_learning":
            ML_TECHNIQUES,

        "baselines":
            BASELINE_TECHNIQUES,
    },

    "metrics": {
        "primary":
            "APFDc",

        "secondary":
            "APFD",

        "evaluation_verdicts":
            "clean",
    },

    "clean_anchor": {
        "affected_rows":
            1,

        "affected_build":
            6843283,

        "affected_test":
            "431",

        "synthetic_executions":
            0,

        "excluded_executions":
            0,
    },

    "full_project_conditions":
        len(NOISE_LEVELS)
        * len(REPETITION_SEEDS),

    "full_project_model_fits":
        len(NOISE_LEVELS)
        * len(REPETITION_SEEDS)
        * len(ML_TECHNIQUES),
}


PROTOCOL_PATH = (
    NOTES_DRIVE
    / "project_06_experiment_protocol.json"
)


with open(
    PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as protocol_file:
    json.dump(
        EXPERIMENT_PROTOCOL,
        protocol_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 4. Normalise clean training and evaluation tables
# ---------------------------------------------------------

clean_training_history = (
    clean_training_history
    .copy()
    .reset_index(drop=True)
)


clean_evaluation_history = (
    clean_evaluation_history
    .copy()
    .reset_index(drop=True)
)


for history in [
    clean_training_history,
    clean_evaluation_history,
]:
    history[
        "build"
    ] = pd.to_numeric(
        history[
            "build"
        ],
        errors="raise",
    ).astype("int64")


    history[
        "test"
    ] = (
        history[
            "test"
        ].astype(str)
    )


    history[
        "verdict"
    ] = pd.to_numeric(
        history[
            "verdict"
        ],
        errors="raise",
    ).astype("int8")


    history[
        "duration"
    ] = pd.to_numeric(
        history[
            "duration"
        ],
        errors="raise",
    ).astype(float)


    history[
        "build_order"
    ] = pd.to_numeric(
        history[
            "build_order"
        ],
        errors="raise",
    ).astype("int64")


if not (
    clean_training_history[
        "partition"
    ].astype(str)
    == "training"
).all():
    raise AssertionError(
        "The raw training history contains "
        "non-training rows."
    )


if not (
    clean_evaluation_history[
        "partition"
    ].astype(str)
    == "evaluation"
).all():
    raise AssertionError(
        "The raw evaluation history contains "
        "non-evaluation rows."
    )


clean_training_data = (
    clean_training_data
    .copy()
    .reset_index(drop=True)
)


clean_evaluation_data = (
    clean_evaluation_data
    .copy()
    .reset_index(drop=True)
)


for model_data in [
    clean_training_data,
    clean_evaluation_data,
]:
    model_data[
        "Build"
    ] = pd.to_numeric(
        model_data[
            "Build"
        ],
        errors="raise",
    ).astype("int64")


    model_data[
        "Test"
    ] = (
        model_data[
            "Test"
        ].astype(str)
    )


    model_data[
        "Verdict"
    ] = pd.to_numeric(
        model_data[
            "Verdict"
        ],
        errors="raise",
    ).astype("int8")


# ---------------------------------------------------------
# 5. Stable dataframe hash helper
# ---------------------------------------------------------

def dataframe_sha256(
    dataframe,
):
    """
    Deterministic content hash used to prove that clean
    training and evaluation objects were not modified.
    """

    data = (
        dataframe
        .copy()
    )


    row_hashes = (
        pd.util.hash_pandas_object(
            data,
            index=True,
        )
        .to_numpy(dtype="uint64")
    )


    digest = hashlib.sha256()

    digest.update(
        "|".join(
            map(
                str,
                data.columns,
            )
        ).encode("utf-8")
    )

    digest.update(
        "|".join(
            map(
                str,
                data.dtypes,
            )
        ).encode("utf-8")
    )

    digest.update(
        row_hashes.tobytes()
    )

    return digest.hexdigest()


training_history_hash_before = (
    dataframe_sha256(
        clean_training_history
    )
)


training_data_hash_before = (
    dataframe_sha256(
        clean_training_data
    )
)


evaluation_history_hash_before = (
    dataframe_sha256(
        clean_evaluation_history
    )
)


evaluation_data_hash_before = (
    dataframe_sha256(
        clean_evaluation_data
    )
)


# ---------------------------------------------------------
# 6. Clean project failure-subtype distribution
# ---------------------------------------------------------

clean_training_verdicts = pd.to_numeric(
    clean_training_history[
        "verdict"
    ],
    errors="raise",
).astype(int)


clean_failure_subtype_counts = (
    clean_training_verdicts[
        clean_training_verdicts != 0
    ]
    .value_counts()
    .sort_index()
)


if clean_failure_subtype_counts.empty:
    raise AssertionError(
        "The clean training history has no failure "
        "subtypes."
    )


CLEAN_FAILURE_SUBTYPES = (
    clean_failure_subtype_counts
    .index
    .to_numpy(dtype=int)
)


CLEAN_FAILURE_SUBTYPE_PROBABILITIES = (
    clean_failure_subtype_counts
    .to_numpy(dtype=float)
)


CLEAN_FAILURE_SUBTYPE_PROBABILITIES /= (
    CLEAN_FAILURE_SUBTYPE_PROBABILITIES.sum()
)


clean_failure_subtype_distribution = (
    pd.DataFrame({
        "FailureSubtype":
            CLEAN_FAILURE_SUBTYPES,

        "Count":
            clean_failure_subtype_counts
            .to_numpy(dtype=int),

        "Probability":
            CLEAN_FAILURE_SUBTYPE_PROBABILITIES,
    })
)


if int(
    clean_failure_subtype_distribution[
        "Count"
    ].sum()
) != 131:
    raise AssertionError(
        "Expected 131 raw training failure executions."
    )


# ---------------------------------------------------------
# 7. Project-specific deterministic random seed
# ---------------------------------------------------------

def stable_project_seed(
    project_name,
    repetition_seed,
    random_stream,
):
    """
    Generate a deterministic project-specific 32-bit seed.
    """

    seed_text = (
        f"{project_name}|"
        f"{int(repetition_seed)}|"
        f"{str(random_stream)}"
    )


    digest = hashlib.sha256(
        seed_text.encode("utf-8")
    ).digest()


    return int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False,
    ) % (2 ** 32)


# ---------------------------------------------------------
# 8. Canonical training-verdict noise injection
# ---------------------------------------------------------

def inject_training_verdict_noise(
    execution_history,
    noise_percent,
    repetition_seed,
    project_name=PROJECT_NAME,
):
    """
    Flip raw training verdicts with nested per-row masks.

    The same project and repetition seed generate the same
    row uniforms and sampled failure subtypes for every
    noise level.
    """

    noise_percent = float(
        noise_percent
    )


    if not (
        0.0
        <= noise_percent
        <= 100.0
    ):
        raise ValueError(
            "noise_percent must be between 0 and 100."
        )


    noisy_history = (
        execution_history
        .copy()
        .reset_index(drop=True)
    )


    if len(noisy_history) == 0:
        raise ValueError(
            "execution_history is empty."
        )


    noisy_history[
        "build"
    ] = pd.to_numeric(
        noisy_history[
            "build"
        ],
        errors="raise",
    ).astype("int64")


    noisy_history[
        "test"
    ] = (
        noisy_history[
            "test"
        ].astype(str)
    )


    original_verdicts = pd.to_numeric(
        noisy_history[
            "verdict"
        ],
        errors="raise",
    ).astype(int).to_numpy()


    number_of_rows = len(
        noisy_history
    )


    flip_rng = np.random.default_rng(
        stable_project_seed(
            project_name,
            repetition_seed,
            "flip_mask",
        )
    )


    subtype_rng = np.random.default_rng(
        stable_project_seed(
            project_name,
            repetition_seed,
            "failure_subtype",
        )
    )


    row_uniforms = flip_rng.random(
        number_of_rows
    )


    sampled_failure_subtypes = (
        subtype_rng.choice(
            CLEAN_FAILURE_SUBTYPES,
            size=number_of_rows,
            replace=True,
            p=(
                CLEAN_FAILURE_SUBTYPE_PROBABILITIES
            ),
        )
    )


    flip_mask = (
        row_uniforms
        < noise_percent / 100.0
    )


    pass_to_failure_mask = (
        flip_mask
        & (
            original_verdicts
            == 0
        )
    )


    failure_to_pass_mask = (
        flip_mask
        & (
            original_verdicts
            != 0
        )
    )


    noisy_verdicts = (
        original_verdicts.copy()
    )


    noisy_verdicts[
        pass_to_failure_mask
    ] = sampled_failure_subtypes[
        pass_to_failure_mask
    ]


    noisy_verdicts[
        failure_to_pass_mask
    ] = 0


    noisy_history[
        "verdict"
    ] = noisy_verdicts.astype(
        "int8"
    )


    if (
        "_source_execution_row"
        in noisy_history.columns
    ):
        noise_row_ids = pd.to_numeric(
            noisy_history[
                "_source_execution_row"
            ],
            errors="raise",
        ).astype("int64").to_numpy()

    else:
        noise_row_ids = np.arange(
            number_of_rows,
            dtype=np.int64,
        )


    if len(
        np.unique(
            noise_row_ids
        )
    ) != number_of_rows:
        raise AssertionError(
            "Noise row identifiers are not unique."
        )


    manifest = pd.DataFrame({
        "NoiseRowPosition":
            np.arange(
                number_of_rows,
                dtype=np.int64,
            ),

        "NoiseRowID":
            noise_row_ids,

        "Build":
            noisy_history[
                "build"
            ].to_numpy(dtype=np.int64),

        "Test":
            noisy_history[
                "test"
            ].astype(str).to_numpy(),

        "OriginalVerdict":
            original_verdicts.astype(int),

        "SampledFailureSubtype":
            sampled_failure_subtypes.astype(int),

        "NoisyVerdict":
            noisy_verdicts.astype(int),

        "FlipUniform":
            row_uniforms.astype(float),

        "Flipped":
            flip_mask.astype(bool),

        "PassToFailure":
            pass_to_failure_mask.astype(bool),

        "FailureToPass":
            failure_to_pass_mask.astype(bool),
    })


    if "job" in noisy_history.columns:
        manifest[
            "Job"
        ] = (
            noisy_history[
                "job"
            ].astype(str).to_numpy()
        )


    number_flipped = int(
        flip_mask.sum()
    )


    summary = {
        "Project":
            project_name,

        "NoisePercentRequested":
            noise_percent,

        "RepetitionSeed":
            int(
                repetition_seed
            ),

        "TrainingExecutionRows":
            number_of_rows,

        "NumberFlipped":
            number_flipped,

        "RealisedNoisePercent":
            (
                100.0
                * number_flipped
                / number_of_rows
            ),

        "PassToFailure":
            int(
                pass_to_failure_mask.sum()
            ),

        "FailureToPass":
            int(
                failure_to_pass_mask.sum()
            ),

        "NoisyPasses":
            int(
                (
                    noisy_verdicts
                    == 0
                ).sum()
            ),

        "NoisyFailures":
            int(
                (
                    noisy_verdicts
                    != 0
                ).sum()
            ),
    }


    return (
        noisy_history,
        manifest,
        summary,
    )


# ---------------------------------------------------------
# 9. Map raw noisy verdicts to fixed model-ready rows
# ---------------------------------------------------------

def extract_model_ready_verdicts(
    noisy_execution_history,
    requested_rows,
):
    """
    Obtain one noisy verdict for each fixed Build-Test row.
    """

    verdict_rows = (
        noisy_execution_history[
            [
                "build",
                "test",
                "verdict",
            ]
        ]
        .copy()
    )


    verdict_rows[
        "build"
    ] = pd.to_numeric(
        verdict_rows[
            "build"
        ],
        errors="raise",
    ).astype("int64")


    verdict_rows[
        "test"
    ] = (
        verdict_rows[
            "test"
        ].astype(str)
    )


    verdict_rows[
        "verdict"
    ] = pd.to_numeric(
        verdict_rows[
            "verdict"
        ],
        errors="raise",
    ).astype("int8")


    duplicate_group_sizes = (
        verdict_rows
        .groupby(
            [
                "build",
                "test",
            ],
            sort=False,
        )
        .size()
    )


    if (
        duplicate_group_sizes
        > 1
    ).any():
        raise AssertionError(
            "The raw history contains duplicate "
            "Build-Test executions."
        )


    verdict_rows = (
        verdict_rows
        .rename(
            columns={
                "build":
                    "Build",

                "test":
                    "Test",

                "verdict":
                    "NoisyVerdict",
            }
        )
    )


    requested = (
        requested_rows[
            [
                "Build",
                "Test",
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )


    requested[
        "Build"
    ] = pd.to_numeric(
        requested[
            "Build"
        ],
        errors="raise",
    ).astype("int64")


    requested[
        "Test"
    ] = (
        requested[
            "Test"
        ].astype(str)
    )


    requested[
        "_RequestedOrder"
    ] = np.arange(
        len(requested),
        dtype=np.int64,
    )


    aligned = (
        requested
        .merge(
            verdict_rows,
            on=[
                "Build",
                "Test",
            ],
            how="left",
            validate="one_to_one",
        )
        .sort_values(
            "_RequestedOrder",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    if aligned[
        "NoisyVerdict"
    ].isna().any():
        raise AssertionError(
            "At least one fixed model-ready training row "
            "could not be linked to its raw verdict."
        )


    return (
        aligned[
            "NoisyVerdict"
        ]
        .astype("int8")
        .reset_index(drop=True)
    )


# ---------------------------------------------------------
# 10. Create one fixed-instance noisy training dataset
# ---------------------------------------------------------

def create_noisy_model_training_data(
    clean_model_training_data,
    noisy_execution_history,
):
    """
    Preserve the fixed model-ready cohort.

    Replace only:
      - the 13 verdict-dependent REC features;
      - the training Verdict.

    Preserve:
      - six noise-independent REC features;
      - all other predictors;
      - Build, Test and Duration.
    """

    noisy_model_data = (
        clean_model_training_data
        .copy()
        .reset_index(drop=True)
    )


    requested_rows = (
        noisy_model_data[
            [
                "Build",
                "Test",
            ]
        ]
        .copy()
    )


    reconstructed = (
        reconstruct_verdict_dependent_rec_features(
            execution_history=(
                noisy_execution_history
            ),
            requested_rows=(
                requested_rows
            ),
        )
    )


    aligned_dependent_rec = (
        align_project_6_dependent_rec(
            reconstructed=(
                reconstructed
            ),
            requested_rows=(
                requested_rows
            ),
        )
    )


    for feature in (
        VERDICT_DEPENDENT_REC_COLUMNS
    ):
        noisy_model_data[
            feature
        ] = pd.to_numeric(
            aligned_dependent_rec[
                feature
            ],
            errors="coerce",
        ).to_numpy(dtype=float)


    noisy_model_data[
        "Verdict"
    ] = (
        extract_model_ready_verdicts(
            noisy_execution_history=(
                noisy_execution_history
            ),
            requested_rows=(
                requested_rows
            ),
        )
        .to_numpy(dtype=np.int8)
    )


    return noisy_model_data


# ---------------------------------------------------------
# 11. APFD and APFDc helpers
# ---------------------------------------------------------

def calculate_apfd(
    actual_failures,
):
    """
    APFD for one already-ranked build.

    actual_failures:
      1 = failing test
      0 = passing test
    """

    failures = np.asarray(
        actual_failures,
        dtype=int,
    )


    number_of_tests = len(
        failures
    )


    number_of_failures = int(
        failures.sum()
    )


    if number_of_tests == 0:
        return np.nan


    if number_of_failures == 0:
        return np.nan


    if not np.isin(
        failures,
        [
            0,
            1,
        ],
    ).all():
        raise ValueError(
            "actual_failures must contain only 0 and 1."
        )


    failure_ranks = (
        np.flatnonzero(
            failures == 1
        )
        + 1
    )


    apfd = (
        1.0
        - failure_ranks.sum()
        / (
            number_of_tests
            * number_of_failures
        )
        + 1.0
        / (
            2.0
            * number_of_tests
        )
    )


    return float(
        apfd
    )


def calculate_apfdc(
    actual_failures,
    durations,
):
    """
    Cost-aware APFD using midpoint failure-detection time.
    """

    failures = np.asarray(
        actual_failures,
        dtype=int,
    )


    durations = np.asarray(
        durations,
        dtype=float,
    )


    if len(failures) != len(
        durations
    ):
        raise ValueError(
            "actual_failures and durations must have "
            "the same length."
        )


    if len(failures) == 0:
        return np.nan


    if not np.isin(
        failures,
        [
            0,
            1,
        ],
    ).all():
        raise ValueError(
            "actual_failures must contain only 0 and 1."
        )


    if failures.sum() == 0:
        return np.nan


    if not np.isfinite(
        durations
    ).all():
        raise ValueError(
            "Durations contain missing or infinite values."
        )


    if (
        durations
        < 0
    ).any():
        raise ValueError(
            "Durations cannot be negative."
        )


    total_duration = float(
        durations.sum()
    )


    if total_duration <= 0:
        return np.nan


    cumulative_before = np.concatenate([
        np.array(
            [
                0.0,
            ]
        ),
        np.cumsum(
            durations
        )[:-1],
    ])


    failure_mask = (
        failures
        == 1
    )


    midpoint_detection_times = (
        cumulative_before[
            failure_mask
        ]
        + 0.5
        * durations[
            failure_mask
        ]
    )


    apfdc = (
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


    return float(
        apfdc
    )


# Compatibility aliases used by later runner cells.
compute_apfd = calculate_apfd
compute_apfdc = calculate_apfdc


# ---------------------------------------------------------
# 12. Generate seed-1 manifests for all noise levels
# ---------------------------------------------------------

seed_1_histories = {}
seed_1_manifests = {}
noise_summary_records = []


for noise_level in NOISE_LEVELS:
    (
        noisy_history,
        noise_manifest,
        noise_summary,
    ) = inject_training_verdict_noise(
        execution_history=(
            clean_training_history
        ),
        noise_percent=(
            noise_level
        ),
        repetition_seed=1,
    )


    seed_1_histories[
        noise_level
    ] = noisy_history


    seed_1_manifests[
        noise_level
    ] = noise_manifest


    noise_summary_records.append(
        noise_summary
    )


noise_summary_table = pd.DataFrame(
    noise_summary_records
)


# ---------------------------------------------------------
# 13. Validate masks, thresholds and nesting
# ---------------------------------------------------------

nested_mask_records = []
nested_mask_violations = 0
uniform_stream_violations = 0
subtype_stream_violations = 0
threshold_rule_violations = 0


reference_manifest = (
    seed_1_manifests[
        0
    ]
)


for noise_level in NOISE_LEVELS:
    manifest = (
        seed_1_manifests[
            noise_level
        ]
    )


    if not np.array_equal(
        manifest[
            "NoiseRowID"
        ].to_numpy(),
        reference_manifest[
            "NoiseRowID"
        ].to_numpy(),
    ):
        raise AssertionError(
            "Noise row identity changed across levels."
        )


    uniform_equal = np.array_equal(
        manifest[
            "FlipUniform"
        ].to_numpy(),
        reference_manifest[
            "FlipUniform"
        ].to_numpy(),
    )


    subtype_equal = np.array_equal(
        manifest[
            "SampledFailureSubtype"
        ].to_numpy(),
        reference_manifest[
            "SampledFailureSubtype"
        ].to_numpy(),
    )


    expected_flip_mask = (
        manifest[
            "FlipUniform"
        ].to_numpy(dtype=float)
        < float(
            noise_level
        ) / 100.0
    )


    actual_flip_mask = (
        manifest[
            "Flipped"
        ].to_numpy(dtype=bool)
    )


    if not uniform_equal:
        uniform_stream_violations += 1


    if not subtype_equal:
        subtype_stream_violations += 1


    threshold_violations = int(
        (
            expected_flip_mask
            != actual_flip_mask
        ).sum()
    )


    threshold_rule_violations += (
        threshold_violations
    )


    nested_mask_records.append({
        "NoisePercent":
            int(
                noise_level
            ),

        "FlippedRows":
            int(
                actual_flip_mask.sum()
            ),

        "UniformStreamMatches0Percent":
            bool(
                uniform_equal
            ),

        "SubtypeStreamMatches0Percent":
            bool(
                subtype_equal
            ),

        "ThresholdRuleViolations":
            threshold_violations,
    })


for lower_level, higher_level in zip(
    NOISE_LEVELS[:-1],
    NOISE_LEVELS[1:],
):
    lower_flipped = set(
        seed_1_manifests[
            lower_level
        ].loc[
            seed_1_manifests[
                lower_level
            ][
                "Flipped"
            ],
            "NoiseRowID",
        ].astype(int)
    )


    higher_flipped = set(
        seed_1_manifests[
            higher_level
        ].loc[
            seed_1_manifests[
                higher_level
            ][
                "Flipped"
            ],
            "NoiseRowID",
        ].astype(int)
    )


    missing_from_higher = (
        lower_flipped
        - higher_flipped
    )


    nested_mask_violations += len(
        missing_from_higher
    )


nested_mask_audit = pd.DataFrame(
    nested_mask_records
)


if nested_mask_violations != 0:
    raise AssertionError(
        "Noise masks are not nested."
    )


if uniform_stream_violations != 0:
    raise AssertionError(
        "The row-uniform stream changed across "
        "noise levels."
    )


if subtype_stream_violations != 0:
    raise AssertionError(
        "The sampled subtype stream changed across "
        "noise levels."
    )


if threshold_rule_violations != 0:
    raise AssertionError(
        "At least one flip mask violated its uniform "
        "threshold."
    )


# ---------------------------------------------------------
# 14. Validate 0% noise
# ---------------------------------------------------------

zero_noise_history = (
    seed_1_histories[
        0
    ]
)


zero_noise_manifest = (
    seed_1_manifests[
        0
    ]
)


zero_noise_summary = (
    noise_summary_table[
        noise_summary_table[
            "NoisePercentRequested"
        ] == 0
    ]
    .iloc[0]
)


if int(
    zero_noise_summary[
        "NumberFlipped"
    ]
) != 0:
    raise AssertionError(
        "0% noise flipped at least one verdict."
    )


if zero_noise_manifest[
    "Flipped"
].any():
    raise AssertionError(
        "The 0% manifest contains flipped rows."
    )


zero_noise_model_data = (
    create_noisy_model_training_data(
        clean_model_training_data=(
            clean_training_data
        ),
        noisy_execution_history=(
            zero_noise_history
        ),
    )
)


zero_cohort_mismatches = int(
    (
        zero_noise_model_data[
            [
                "Build",
                "Test",
            ]
        ].to_numpy()
        != clean_training_data[
            [
                "Build",
                "Test",
            ]
        ].to_numpy()
    ).any(axis=1).sum()
)


zero_label_mismatches = int(
    (
        zero_noise_model_data[
            "Verdict"
        ].to_numpy(dtype=int)
        != clean_training_data[
            "Verdict"
        ].to_numpy(dtype=int)
    ).sum()
)


zero_dependent_rec_mismatches = 0


for feature in (
    VERDICT_DEPENDENT_REC_COLUMNS
):
    zero_dependent_rec_mismatches += int(
        (
            ~np.isclose(
                pd.to_numeric(
                    zero_noise_model_data[
                        feature
                    ],
                    errors="coerce",
                ).to_numpy(dtype=float),

                pd.to_numeric(
                    clean_training_data[
                        feature
                    ],
                    errors="coerce",
                ).to_numpy(dtype=float),

                rtol=1e-9,
                atol=1e-9,
                equal_nan=True,
            )
        ).sum()
    )


zero_independent_rec_mismatches = 0


for feature in (
    NOISE_INDEPENDENT_REC_COLUMNS
):
    zero_independent_rec_mismatches += int(
        (
            ~np.isclose(
                pd.to_numeric(
                    zero_noise_model_data[
                        feature
                    ],
                    errors="coerce",
                ).to_numpy(dtype=float),

                pd.to_numeric(
                    clean_training_data[
                        feature
                    ],
                    errors="coerce",
                ).to_numpy(dtype=float),

                rtol=1e-12,
                atol=1e-12,
                equal_nan=True,
            )
        ).sum()
    )


if zero_cohort_mismatches != 0:
    raise AssertionError(
        "0% reconstruction changed the fixed cohort."
    )


if zero_label_mismatches != 0:
    raise AssertionError(
        "0% reconstruction changed training labels."
    )


if zero_dependent_rec_mismatches != 0:
    raise AssertionError(
        "0% reconstruction changed a dependent REC value."
    )


if zero_independent_rec_mismatches != 0:
    raise AssertionError(
        "0% reconstruction changed a preserved REC value."
    )


# ---------------------------------------------------------
# 15. Validate same-seed and different-seed behaviour
# ---------------------------------------------------------

(
    five_history_a,
    five_manifest_a,
    five_summary_a,
) = inject_training_verdict_noise(
    clean_training_history,
    noise_percent=5,
    repetition_seed=1,
)


(
    five_history_b,
    five_manifest_b,
    five_summary_b,
) = inject_training_verdict_noise(
    clean_training_history,
    noise_percent=5,
    repetition_seed=1,
)


(
    five_history_seed_2,
    five_manifest_seed_2,
    five_summary_seed_2,
) = inject_training_verdict_noise(
    clean_training_history,
    noise_percent=5,
    repetition_seed=2,
)


same_seed_verdict_reproducible = bool(
    np.array_equal(
        five_history_a[
            "verdict"
        ].to_numpy(),

        five_history_b[
            "verdict"
        ].to_numpy(),
    )
)


same_seed_manifest_reproducible = bool(
    five_manifest_a.equals(
        five_manifest_b
    )
)


different_seed_mask_differs = bool(
    not np.array_equal(
        five_manifest_a[
            "Flipped"
        ].to_numpy(),

        five_manifest_seed_2[
            "Flipped"
        ].to_numpy(),
    )
)


if not same_seed_verdict_reproducible:
    raise AssertionError(
        "The same seed did not reproduce noisy verdicts."
    )


if not same_seed_manifest_reproducible:
    raise AssertionError(
        "The same seed did not reproduce the manifest."
    )


if not different_seed_mask_differs:
    raise AssertionError(
        "Seeds 1 and 2 generated the same 5% mask."
    )


# ---------------------------------------------------------
# 16. Validate subtype flipping rules
# ---------------------------------------------------------

pass_to_failure_rows = (
    five_manifest_a[
        five_manifest_a[
            "PassToFailure"
        ]
    ]
)


failure_to_pass_rows = (
    five_manifest_a[
        five_manifest_a[
            "FailureToPass"
        ]
    ]
)


invalid_pass_to_failure_subtypes = int(
    (
        ~pass_to_failure_rows[
            "NoisyVerdict"
        ].isin(
            CLEAN_FAILURE_SUBTYPES
        )
    ).sum()
)


invalid_failure_to_pass_verdicts = int(
    (
        failure_to_pass_rows[
            "NoisyVerdict"
        ] != 0
    ).sum()
)


unchanged_unflipped_violations = int(
    (
        five_manifest_a.loc[
            ~five_manifest_a[
                "Flipped"
            ],
            "OriginalVerdict",
        ].to_numpy()
        !=
        five_manifest_a.loc[
            ~five_manifest_a[
                "Flipped"
            ],
            "NoisyVerdict",
        ].to_numpy()
    ).sum()
)


if invalid_pass_to_failure_subtypes != 0:
    raise AssertionError(
        "A pass was converted to an invalid failure "
        "subtype."
    )


if invalid_failure_to_pass_verdicts != 0:
    raise AssertionError(
        "A failure was not converted to pass."
    )


if unchanged_unflipped_violations != 0:
    raise AssertionError(
        "At least one unselected verdict changed."
    )


# ---------------------------------------------------------
# 17. Validate 5% fixed-instance model reconstruction
# ---------------------------------------------------------

five_noise_model_data = (
    create_noisy_model_training_data(
        clean_model_training_data=(
            clean_training_data
        ),
        noisy_execution_history=(
            five_history_a
        ),
    )
)


if len(
    five_noise_model_data
) != len(
    clean_training_data
):
    raise AssertionError(
        "5% reconstruction changed the training-row count."
    )


fixed_cohort_equal = bool(
    five_noise_model_data[
        [
            "Build",
            "Test",
        ]
    ].equals(
        clean_training_data[
            [
                "Build",
                "Test",
            ]
        ]
    )
)


if not fixed_cohort_equal:
    raise AssertionError(
        "5% reconstruction changed the fixed Build-Test "
        "cohort or its row order."
    )


expected_five_labels = (
    extract_model_ready_verdicts(
        noisy_execution_history=(
            five_history_a
        ),
        requested_rows=(
            clean_training_data[
                [
                    "Build",
                    "Test",
                ]
            ]
        ),
    )
)


five_label_alignment_mismatches = int(
    (
        five_noise_model_data[
            "Verdict"
        ].to_numpy(dtype=int)
        != expected_five_labels.to_numpy(dtype=int)
    ).sum()
)


five_changed_model_labels = int(
    (
        five_noise_model_data[
            "Verdict"
        ].to_numpy(dtype=int)
        != clean_training_data[
            "Verdict"
        ].to_numpy(dtype=int)
    ).sum()
)


if five_label_alignment_mismatches != 0:
    raise AssertionError(
        "Noisy model labels do not match raw noisy "
        "verdicts."
    )


if five_changed_model_labels == 0:
    raise AssertionError(
        "The 5% condition changed no model-ready labels."
    )


five_independent_rec_mismatches = 0


for feature in (
    NOISE_INDEPENDENT_REC_COLUMNS
):
    five_independent_rec_mismatches += int(
        (
            ~np.isclose(
                pd.to_numeric(
                    five_noise_model_data[
                        feature
                    ],
                    errors="coerce",
                ).to_numpy(dtype=float),

                pd.to_numeric(
                    clean_training_data[
                        feature
                    ],
                    errors="coerce",
                ).to_numpy(dtype=float),

                rtol=1e-12,
                atol=1e-12,
                equal_nan=True,
            )
        ).sum()
    )


if five_independent_rec_mismatches != 0:
    raise AssertionError(
        "5% noise changed a noise-independent REC value."
    )


five_dependent_feature_change_records = []


for feature in (
    VERDICT_DEPENDENT_REC_COLUMNS
):
    changed_values = int(
        (
            ~np.isclose(
                pd.to_numeric(
                    five_noise_model_data[
                        feature
                    ],
                    errors="coerce",
                ).to_numpy(dtype=float),

                pd.to_numeric(
                    clean_training_data[
                        feature
                    ],
                    errors="coerce",
                ).to_numpy(dtype=float),

                rtol=1e-9,
                atol=1e-9,
                equal_nan=True,
            )
        ).sum()
    )


    five_dependent_feature_change_records.append({
        "Feature":
            feature,

        "ChangedTrainingValuesAt5Percent":
            changed_values,
    })


five_dependent_feature_changes = (
    pd.DataFrame(
        five_dependent_feature_change_records
    )
)


total_five_dependent_changes = int(
    five_dependent_feature_changes[
        "ChangedTrainingValuesAt5Percent"
    ].sum()
)


if total_five_dependent_changes == 0:
    raise AssertionError(
        "5% noise changed no dependent REC values."
    )


# ---------------------------------------------------------
# 18. Validate non-intervened columns remain exact
# ---------------------------------------------------------

non_intervened_columns = [
    column
    for column in clean_training_data.columns
    if column not in {
        "Verdict",
        *VERDICT_DEPENDENT_REC_COLUMNS,
    }
]


non_intervened_columns_equal = bool(
    five_noise_model_data[
        non_intervened_columns
    ].equals(
        clean_training_data[
            non_intervened_columns
        ]
    )
)


if not non_intervened_columns_equal:
    differing_columns = [
        column
        for column in non_intervened_columns
        if not five_noise_model_data[
            column
        ].equals(
            clean_training_data[
                column
            ]
        )
    ]


    raise AssertionError(
        "5% noise changed non-intervened columns:\n"
        + "\n".join(
            differing_columns
        )
    )


# ---------------------------------------------------------
# 19. Validate clean-anchor delta identity at 5%
# ---------------------------------------------------------

five_requested_rows = (
    clean_training_data[
        [
            "Build",
            "Test",
        ]
    ]
    .copy()
)


five_raw_reconstruction = (
    reconstruct_project_6_raw_dependent_rec(
        execution_history=(
            five_history_a
        ),
        requested_rows=(
            five_requested_rows
        ),
    )
)


five_raw_aligned = (
    align_project_6_dependent_rec(
        reconstructed=(
            five_raw_reconstruction
        ),
        requested_rows=(
            five_requested_rows
        ),
    )
)


training_offsets = (
    clean_anchor_offsets
    .merge(
        five_requested_rows.assign(
            _RequestedOrder=np.arange(
                len(
                    five_requested_rows
                ),
                dtype=np.int64,
            )
        ),
        on=[
            "Build",
            "Test",
        ],
        how="right",
        validate="one_to_one",
    )
    .sort_values(
        "_RequestedOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


clean_anchor_formula_mismatches = 0


for feature in (
    VERDICT_DEPENDENT_REC_COLUMNS
):
    expected_corrected_values = (
        pd.to_numeric(
            five_raw_aligned[
                feature
            ],
            errors="coerce",
        ).to_numpy(dtype=float)
        +
        pd.to_numeric(
            training_offsets[
                feature
            ],
            errors="coerce",
        ).to_numpy(dtype=float)
    )


    observed_corrected_values = (
        pd.to_numeric(
            five_noise_model_data[
                feature
            ],
            errors="coerce",
        ).to_numpy(dtype=float)
    )


    clean_anchor_formula_mismatches += int(
        (
            ~np.isclose(
                expected_corrected_values,
                observed_corrected_values,
                rtol=1e-9,
                atol=1e-9,
                equal_nan=True,
            )
        ).sum()
    )


if clean_anchor_formula_mismatches != 0:
    raise AssertionError(
        "The 5% model data violated the clean-anchor "
        "delta formula."
    )


# ---------------------------------------------------------
# 20. Validate APFD and APFDc
# ---------------------------------------------------------

manual_failures_best = [
    1,
    1,
    0,
    0,
    0,
]


manual_failures_worst = [
    0,
    0,
    0,
    1,
    1,
]


manual_apfd_best = calculate_apfd(
    manual_failures_best
)


manual_apfd_worst = calculate_apfd(
    manual_failures_worst
)


manual_apfd_no_failures = (
    calculate_apfd(
        [
            0,
            0,
            0,
        ]
    )
)


manual_apfdc_slow_failure_first = (
    calculate_apfdc(
        actual_failures=[
            1,
            1,
            0,
            0,
            0,
        ],
        durations=[
            5,
            1,
            1,
            1,
            1,
        ],
    )
)


manual_apfdc_quick_failure_first = (
    calculate_apfdc(
        actual_failures=[
            1,
            1,
            0,
            0,
            0,
        ],
        durations=[
            1,
            5,
            1,
            1,
            1,
        ],
    )
)


manual_apfdc_no_failures = (
    calculate_apfdc(
        actual_failures=[
            0,
            0,
            0,
        ],
        durations=[
            1,
            1,
            1,
        ],
    )
)


if not np.isclose(
    manual_apfd_best,
    0.80,
):
    raise AssertionError(
        "Manual best-order APFD validation failed."
    )


if not np.isclose(
    manual_apfd_worst,
    0.20,
):
    raise AssertionError(
        "Manual worst-order APFD validation failed."
    )


if not np.isnan(
    manual_apfd_no_failures
):
    raise AssertionError(
        "APFD should be NaN for a build with no failures."
    )


if not (
    manual_apfdc_quick_failure_first
    > manual_apfdc_slow_failure_first
):
    raise AssertionError(
        "APFDc did not reward quicker failure detection."
    )


if not np.isnan(
    manual_apfdc_no_failures
):
    raise AssertionError(
        "APFDc should be NaN for a build with no failures."
    )


if not (
    0.0
    <= manual_apfdc_slow_failure_first
    <= 1.0
):
    raise AssertionError(
        "Manual APFDc is outside [0, 1]."
    )


if not (
    0.0
    <= manual_apfdc_quick_failure_first
    <= 1.0
):
    raise AssertionError(
        "Manual APFDc is outside [0, 1]."
    )


# ---------------------------------------------------------
# 21. Confirm all clean objects remain immutable
# ---------------------------------------------------------

training_history_hash_after = (
    dataframe_sha256(
        clean_training_history
    )
)


training_data_hash_after = (
    dataframe_sha256(
        clean_training_data
    )
)


evaluation_history_hash_after = (
    dataframe_sha256(
        clean_evaluation_history
    )
)


evaluation_data_hash_after = (
    dataframe_sha256(
        clean_evaluation_data
    )
)


training_history_immutable = bool(
    training_history_hash_before
    == training_history_hash_after
)


training_data_immutable = bool(
    training_data_hash_before
    == training_data_hash_after
)


evaluation_history_immutable = bool(
    evaluation_history_hash_before
    == evaluation_history_hash_after
)


evaluation_data_immutable = bool(
    evaluation_data_hash_before
    == evaluation_data_hash_after
)


if not training_history_immutable:
    raise AssertionError(
        "The clean raw training history was modified."
    )


if not training_data_immutable:
    raise AssertionError(
        "The clean model training data was modified."
    )


if not evaluation_history_immutable:
    raise AssertionError(
        "The clean raw evaluation history was modified."
    )


if not evaluation_data_immutable:
    raise AssertionError(
        "The clean model evaluation data was modified."
    )


# ---------------------------------------------------------
# 22. Save helper-validation artefacts
# ---------------------------------------------------------

FAILURE_SUBTYPE_PATH = (
    PROJECT_HELPER_DIRECTORY
    / "jetty_failure_subtype_distribution.csv"
)


NOISE_SUMMARY_PATH = (
    PROJECT_HELPER_DIRECTORY
    / "jetty_seed01_noise_summary.csv"
)


NESTED_MASK_AUDIT_PATH = (
    PROJECT_HELPER_DIRECTORY
    / "jetty_nested_mask_audit.csv"
)


FIVE_PERCENT_FEATURE_CHANGE_PATH = (
    PROJECT_HELPER_DIRECTORY
    / "jetty_5percent_dependent_feature_changes.csv"
)


HELPER_VALIDATION_REPORT_PATH = (
    PROJECT_HELPER_DIRECTORY
    / "jetty_helper_validation_report.json"
)


clean_failure_subtype_distribution.to_csv(
    FAILURE_SUBTYPE_PATH,
    index=False,
)


noise_summary_table.to_csv(
    NOISE_SUMMARY_PATH,
    index=False,
)


nested_mask_audit.to_csv(
    NESTED_MASK_AUDIT_PATH,
    index=False,
)


five_dependent_feature_changes.to_csv(
    FIVE_PERCENT_FEATURE_CHANGE_PATH,
    index=False,
)


helper_validation_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "PASS",

    "Protocol":
        str(
            PROTOCOL_PATH
        ),

    "NoiseLevels":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "Conditions":
        len(NOISE_LEVELS)
        * len(REPETITION_SEEDS),

    "ExpectedModelFits":
        len(NOISE_LEVELS)
        * len(REPETITION_SEEDS)
        * len(ML_TECHNIQUES),

    "RawTrainingRows":
        len(clean_training_history),

    "RawEvaluationRows":
        len(clean_evaluation_history),

    "ModelTrainingRows":
        len(clean_training_data),

    "ModelEvaluationRows":
        len(clean_evaluation_data),

    "CleanFailureExecutions":
        int(
            clean_failure_subtype_distribution[
                "Count"
            ].sum()
        ),

    "CleanFailureSubtypeDistribution":
        clean_failure_subtype_distribution.to_dict(
            orient="records"
        ),

    "NestedMaskViolations":
        nested_mask_violations,

    "UniformStreamViolations":
        uniform_stream_violations,

    "SubtypeStreamViolations":
        subtype_stream_violations,

    "ThresholdRuleViolations":
        threshold_rule_violations,

    "SameSeedVerdictReproducible":
        same_seed_verdict_reproducible,

    "SameSeedManifestReproducible":
        same_seed_manifest_reproducible,

    "DifferentSeedMaskDiffers":
        different_seed_mask_differs,

    "InvalidPassToFailureSubtypes":
        invalid_pass_to_failure_subtypes,

    "InvalidFailureToPassVerdicts":
        invalid_failure_to_pass_verdicts,

    "UnchangedUnflippedViolations":
        unchanged_unflipped_violations,

    "ZeroPercentFlippedRows":
        int(
            zero_noise_summary[
                "NumberFlipped"
            ]
        ),

    "ZeroPercentCohortMismatches":
        zero_cohort_mismatches,

    "ZeroPercentLabelMismatches":
        zero_label_mismatches,

    "ZeroPercentDependentRECMismatches":
        zero_dependent_rec_mismatches,

    "ZeroPercentIndependentRECMismatches":
        zero_independent_rec_mismatches,

    "FivePercentRawFlippedRows":
        int(
            five_summary_a[
                "NumberFlipped"
            ]
        ),

    "FivePercentModelLabelChanges":
        five_changed_model_labels,

    "FivePercentDependentRECChanges":
        total_five_dependent_changes,

    "FivePercentIndependentRECMismatches":
        five_independent_rec_mismatches,

    "FivePercentLabelAlignmentMismatches":
        five_label_alignment_mismatches,

    "CleanAnchorFormulaMismatches":
        clean_anchor_formula_mismatches,

    "FixedCohortPreserved":
        fixed_cohort_equal,

    "NonIntervenedColumnsPreserved":
        non_intervened_columns_equal,

    "TrainingHistoryImmutable":
        training_history_immutable,

    "TrainingDataImmutable":
        training_data_immutable,

    "EvaluationHistoryImmutable":
        evaluation_history_immutable,

    "EvaluationDataImmutable":
        evaluation_data_immutable,

    "ManualAPFDBest":
        manual_apfd_best,

    "ManualAPFDWorst":
        manual_apfd_worst,

    "ManualAPFDcSlowFailureFirst":
        manual_apfdc_slow_failure_first,

    "ManualAPFDcQuickFailureFirst":
        manual_apfdc_quick_failure_first,

    "FailureSubtypeDistributionPath":
        str(
            FAILURE_SUBTYPE_PATH
        ),

    "NoiseSummaryPath":
        str(
            NOISE_SUMMARY_PATH
        ),

    "NestedMaskAuditPath":
        str(
            NESTED_MASK_AUDIT_PATH
        ),

    "FeatureChangeAuditPath":
        str(
            FIVE_PERCENT_FEATURE_CHANGE_PATH
        ),

    "CompletedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
}


with open(
    HELPER_VALIDATION_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        helper_validation_report,
        report_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 23. Update Project 6 checkpoint
# ---------------------------------------------------------

with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint = json.load(
        checkpoint_file
    )


checkpoint.update({
    "Status":
        "HELPER_VALIDATION_PASSED",

    "ExperimentProtocol":
        str(
            PROTOCOL_PATH
        ),

    "HelperValidationReport":
        str(
            HELPER_VALIDATION_REPORT_PATH
        ),

    "NoiseLevels":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "ExpectedConditions":
        len(NOISE_LEVELS)
        * len(REPETITION_SEEDS),

    "ExpectedModelFits":
        len(NOISE_LEVELS)
        * len(REPETITION_SEEDS)
        * len(ML_TECHNIQUES),

    "NestedMaskViolations":
        nested_mask_violations,

    "ZeroPercentDependentRECMismatches":
        zero_dependent_rec_mismatches,

    "ZeroPercentLabelMismatches":
        zero_label_mismatches,

    "FivePercentIndependentRECMismatches":
        five_independent_rec_mismatches,

    "EvaluationDataImmutable":
        evaluation_data_immutable,

    "APFDHelperValidated":
        True,

    "APFDcHelperValidated":
        True,

    "UpdatedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
})


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 24. Compact final output
# ---------------------------------------------------------

print(
    "\n=== PROJECT 6 STEP 7 RESULT ==="
)


print(
    "\nExperiment configuration:"
)

print(
    "Noise levels:",
    NOISE_LEVELS
)

print(
    "Repetition seeds:",
    len(
        REPETITION_SEEDS
    )
)

print(
    "Conditions:",
    len(NOISE_LEVELS)
    * len(REPETITION_SEEDS)
)

print(
    "Expected ML fits:",
    len(NOISE_LEVELS)
    * len(REPETITION_SEEDS)
    * len(ML_TECHNIQUES)
)


print(
    "\nClean failure subtype distribution:"
)

display(
    clean_failure_subtype_distribution
)


print(
    "\nSeed 1 noise summary:"
)

display(
    noise_summary_table[
        [
            "NoisePercentRequested",
            "NumberFlipped",
            "RealisedNoisePercent",
            "PassToFailure",
            "FailureToPass",
            "NoisyFailures",
        ]
    ]
)


print(
    "\nNested-mask validation:"
)

print(
    "Nested-mask violations:",
    nested_mask_violations
)

print(
    "Uniform-stream violations:",
    uniform_stream_violations
)

print(
    "Subtype-stream violations:",
    subtype_stream_violations
)

print(
    "Threshold-rule violations:",
    threshold_rule_violations
)


print(
    "\nReproducibility:"
)

print(
    "Same-seed verdict reproducibility:",
    same_seed_verdict_reproducible
)

print(
    "Same-seed manifest reproducibility:",
    same_seed_manifest_reproducible
)

print(
    "Different-seed mask differs:",
    different_seed_mask_differs
)


print(
    "\n0% fixed-instance validation:"
)

print(
    "Flipped raw verdicts:",
    int(
        zero_noise_summary[
            "NumberFlipped"
        ]
    )
)

print(
    "Cohort mismatches:",
    zero_cohort_mismatches
)

print(
    "Label mismatches:",
    zero_label_mismatches
)

print(
    "Dependent REC mismatches:",
    zero_dependent_rec_mismatches
)

print(
    "Independent REC mismatches:",
    zero_independent_rec_mismatches
)


print(
    "\n5% intervention validation:"
)

print(
    "Raw verdicts flipped:",
    int(
        five_summary_a[
            "NumberFlipped"
        ]
    )
)

print(
    "Model-ready labels changed:",
    five_changed_model_labels
)

print(
    "Dependent REC values changed:",
    total_five_dependent_changes
)

print(
    "Independent REC mismatches:",
    five_independent_rec_mismatches
)

print(
    "Clean-anchor formula mismatches:",
    clean_anchor_formula_mismatches
)

print(
    "Non-intervened columns preserved:",
    non_intervened_columns_equal
)


print(
    "\nDependent feature changes at 5%:"
)

display(
    five_dependent_feature_changes
)


print(
    "\nClean-object immutability:"
)

print(
    "Training history immutable:",
    training_history_immutable
)

print(
    "Training data immutable:",
    training_data_immutable
)

print(
    "Evaluation history immutable:",
    evaluation_history_immutable
)

print(
    "Evaluation data immutable:",
    evaluation_data_immutable
)


print(
    "\nMetric helper validation:"
)

print(
    "Best-order APFD:",
    manual_apfd_best
)

print(
    "Worst-order APFD:",
    manual_apfd_worst
)

print(
    "Slow-failure-first APFDc:",
    manual_apfdc_slow_failure_first
)

print(
    "Quick-failure-first APFDc:",
    manual_apfdc_quick_failure_first
)


print(
    "\nHelper validation report:"
)

print(
    HELPER_VALIDATION_REPORT_PATH
)


print(
    "\nValidation status:",
    "PASS"
)


print(
    "\nSUCCESS: Noise masks are deterministic and nested."
)

print(
    "SUCCESS: Failure subtype sampling follows the clean "
    "Jetty distribution."
)

print(
    "SUCCESS: 0% noise reproduces the fixed clean "
    "training dataset exactly."
)

print(
    "SUCCESS: 5% noise changes only labels and the "
    "13 verdict-dependent REC features."
)

print(
    "SUCCESS: The clean-anchor delta formula was "
    "validated under noise."
)

print(
    "SUCCESS: Clean evaluation data remained immutable."
)

print(
    "SUCCESS: APFD and APFDc helpers passed their "
    "manual formula tests."
)

print(
    "SUCCESS: Project 6 is ready for ML-model and "
    "baseline runner configuration."
)

=== PROJECT 6 STEP 7: CANONICAL HELPER VALIDATION ===

=== PROJECT 6 STEP 7 RESULT ===

Experiment configuration:
Noise levels: [0, 5, 10, 15, 20, 25, 30, 40, 50]
Repetition seeds: 30
Conditions: 270
Expected ML fits: 1080

Clean failure subtype distribution:


,FailureSubtype,Count,Probability
0,1,29,0.221374
1,2,102,0.778626



Seed 1 noise summary:


,NoisePercentRequested,NumberFlipped,RealisedNoisePercent,PassToFailure,FailureToPass,NoisyFailures
0,0.0,0,0.000000,0,0,131
1,5.0,987,4.973545,979,8,1102
2,10.0,1930,9.725372,1912,18,2025
3,15.0,2865,14.436886,2844,21,2954
4,20.0,3816,19.229025,3792,24,3899
5,25.0,4806,24.217687,4776,30,4877
6,30.0,5783,29.140842,5744,39,5836
7,40.0,7821,39.410431,7767,54,7844
8,50.0,9835,49.559083,9766,69,9828



Nested-mask validation:
Nested-mask violations: 0
Uniform-stream violations: 0
Subtype-stream violations: 0
Threshold-rule violations: 0

Reproducibility:
Same-seed verdict reproducibility: True
Same-seed manifest reproducibility: True
Different-seed mask differs: True

0% fixed-instance validation:
Flipped raw verdicts: 0
Cohort mismatches: 0
Label mismatches: 0
Dependent REC mismatches: 0
Independent REC mismatches: 0

5% intervention validation:
Raw verdicts flipped: 987
Model-ready labels changed: 674
Dependent REC values changed: 80818
Independent REC mismatches: 0
Clean-anchor formula mismatches: 0
Non-intervened columns preserved: True

Dependent feature changes at 5%:


,Feature,ChangedTrainingValuesAt5Percent
0,REC_LastFailureAge,9268
1,REC_LastTransitionAge,9286
2,REC_RecentFailRate,3335
3,REC_RecentAssertRate,2669
4,REC_RecentExcRate,808
5,REC_RecentTransitionRate,3738
6,REC_TotalFailRate,9644
7,REC_TotalAssertRate,8701
8,REC_TotalExcRate,4756
9,REC_TotalTransitionRate,9586



Clean-object immutability:
Training history immutable: True
Training data immutable: True
Evaluation history immutable: True
Evaluation data immutable: True

Metric helper validation:
Best-order APFD: 0.7999999999999999
Worst-order APFD: 0.19999999999999998
Slow-failure-first APFDc: 0.5555555555555556
Quick-failure-first APFDc: 0.7777777777777778

Helper validation report:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/eclipse__jetty.project/jetty_helper_validation/jetty_helper_validation_report.json

Validation status: PASS

SUCCESS: Noise masks are deterministic and nested.
SUCCESS: Failure subtype sampling follows the clean Jetty distribution.
SUCCESS: 0% noise reproduces the fixed clean training dataset exactly.
SUCCESS: 5% noise changes only labels and the 13 verdict-dependent REC features.
SUCCESS: The clean-anchor delta formula was validated under noise.
SUCCESS: Clean evaluation data remained immutable.
SUCCESS: APFD and APFDc helpers passed their manual formula t

In [ ]:
# =========================================================
# PROJECT 6 — STEP 8
# FREEZE MODEL CONFIGURATION AND DEFINE CONDITION RUNNER
#
# This cell:
#   - freezes zero-variance feature removal
#   - freezes four ML model configurations
#   - defines deterministic model construction
#   - defines ML, Random, LatestFail and QTF-Avg rankings
#   - defines build-level APFD/APFDc aggregation
#   - defines complete condition validation and saving
#   - validates the three baseline pipelines
#
# It DOES NOT fit the four ML models or run the smoke
# condition. That happens in Step 9.
# =========================================================

from pathlib import Path
import hashlib
import json
import math
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


print(
    "=== PROJECT 6 STEP 8: "
    "MODEL AND RUNNER CONFIGURATION ==="
)


# ---------------------------------------------------------
# 1. Confirm successful Step 7 state
# ---------------------------------------------------------

required_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",

    "THESIS_DRIVE",
    "RAW_RESULTS_DRIVE",
    "AGGREGATED_RESULTS_DRIVE",
    "LOGS_DRIVE",
    "NOTES_DRIVE",

    "PROJECT_RAW_RESULTS",
    "PROJECT_AGGREGATED_RESULTS",
    "PROJECT_LOGS",

    "PROJECT_6_SELECTION_CHECKPOINT",

    "NOISE_LEVELS",
    "REPETITION_SEEDS",
    "ML_TECHNIQUES",
    "BASELINE_TECHNIQUES",
    "ALL_TECHNIQUES",

    "builds",
    "clean_training_history",
    "clean_evaluation_history",
    "clean_training_data",
    "clean_evaluation_data",

    "MODEL_FEATURE_COLUMNS",
    "REC_FEATURE_COLUMNS",

    "stable_project_seed",
    "inject_training_verdict_noise",
    "create_noisy_model_training_data",

    "calculate_apfd",
    "calculate_apfdc",
    "dataframe_sha256",
]


missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required Step 7 objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nRerun the successful Step 6 final cell "
        "and Step 7 helper-validation cell."
    )


if PROJECT_NUMBER != 6:
    raise AssertionError(
        f"Expected Project 6, observed {PROJECT_NUMBER}."
    )


if PROJECT_NAME != "eclipse@jetty.project":
    raise AssertionError(
        f"Unexpected project: {PROJECT_NAME}"
    )


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    step_7_checkpoint = json.load(
        checkpoint_file
    )


if step_7_checkpoint.get(
    "Status"
) != "HELPER_VALIDATION_PASSED":
    raise AssertionError(
        "Step 7 has not been marked as passed.\n"
        f"Observed status: "
        f"{step_7_checkpoint.get('Status')}"
    )


if len(NOISE_LEVELS) != 9:
    raise AssertionError(
        "Expected nine noise levels."
    )


if len(REPETITION_SEEDS) != 30:
    raise AssertionError(
        "Expected 30 repetition seeds."
    )


if ML_TECHNIQUES != [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]:
    raise AssertionError(
        "Unexpected ML technique list."
    )


if BASELINE_TECHNIQUES != [
    "Random",
    "LatestFail",
    "QTF-Avg",
]:
    raise AssertionError(
        "Unexpected baseline technique list."
    )


# ---------------------------------------------------------
# 2. Normalise Project 6 runtime tables
# ---------------------------------------------------------

builds = (
    builds
    .copy()
    .reset_index(drop=True)
)


builds[
    "build"
] = pd.to_numeric(
    builds["build"],
    errors="raise",
).astype("int64")


builds[
    "build_order"
] = pd.to_numeric(
    builds["build_order"],
    errors="raise",
).astype("int64")


builds[
    "partition"
] = (
    builds["partition"]
    .astype(str)
)


clean_training_history = (
    clean_training_history
    .copy()
    .reset_index(drop=True)
)


clean_evaluation_history = (
    clean_evaluation_history
    .copy()
    .reset_index(drop=True)
)


for history_table in [
    clean_training_history,
    clean_evaluation_history,
]:
    history_table[
        "build"
    ] = pd.to_numeric(
        history_table["build"],
        errors="raise",
    ).astype("int64")


    history_table[
        "test"
    ] = (
        history_table["test"]
        .astype(str)
    )


    history_table[
        "verdict"
    ] = pd.to_numeric(
        history_table["verdict"],
        errors="raise",
    ).astype("int8")


    history_table[
        "duration"
    ] = pd.to_numeric(
        history_table["duration"],
        errors="raise",
    ).astype(float)


    history_table[
        "build_order"
    ] = pd.to_numeric(
        history_table["build_order"],
        errors="raise",
    ).astype("int64")


    if "job" not in history_table.columns:
        history_table["job"] = ""


clean_training_data = (
    clean_training_data
    .copy()
    .reset_index(drop=True)
)


clean_evaluation_data = (
    clean_evaluation_data
    .copy()
    .reset_index(drop=True)
)


for model_table in [
    clean_training_data,
    clean_evaluation_data,
]:
    model_table[
        "Build"
    ] = pd.to_numeric(
        model_table["Build"],
        errors="raise",
    ).astype("int64")


    model_table[
        "Test"
    ] = (
        model_table["Test"]
        .astype(str)
    )


    model_table[
        "Verdict"
    ] = pd.to_numeric(
        model_table["Verdict"],
        errors="raise",
    ).astype("int8")


    model_table[
        "Duration"
    ] = pd.to_numeric(
        model_table["Duration"],
        errors="raise",
    ).astype(float)


    model_table[
        "build_order"
    ] = pd.to_numeric(
        model_table["build_order"],
        errors="raise",
    ).astype("int64")


if len(builds) != 192:
    raise AssertionError(
        "Expected 192 Jetty builds."
    )


if len(clean_training_history) != 19845:
    raise AssertionError(
        "Expected 19,845 raw training executions."
    )


if len(clean_evaluation_history) != 6594:
    raise AssertionError(
        "Expected 6,594 raw evaluation executions."
    )


if len(clean_training_data) != 13212:
    raise AssertionError(
        "Expected 13,212 model-training rows."
    )


if len(clean_evaluation_data) != 4079:
    raise AssertionError(
        "Expected 4,079 model-evaluation rows."
    )


EVALUATED_BUILD_COUNT = int(
    clean_evaluation_data[
        "Build"
    ].nunique()
)


if EVALUATED_BUILD_COUNT != 38:
    raise AssertionError(
        "Expected 38 failing model-ready "
        "evaluation builds."
    )


EXPECTED_RANKING_ROWS_PER_TECHNIQUE = int(
    len(clean_evaluation_data)
)


EXPECTED_CONDITION_RANKING_ROWS = int(
    EXPECTED_RANKING_ROWS_PER_TECHNIQUE
    * len(ALL_TECHNIQUES)
)


EXPECTED_CONDITION_BUILD_METRIC_ROWS = int(
    EVALUATED_BUILD_COUNT
    * len(ALL_TECHNIQUES)
)


EXPECTED_CONDITION_PROJECT_RUN_ROWS = int(
    len(ALL_TECHNIQUES)
)


EXPECTED_CONDITION_FIT_ROWS = int(
    len(ML_TECHNIQUES)
)


if EXPECTED_CONDITION_RANKING_ROWS != 28553:
    raise AssertionError(
        "Expected 28,553 ranking rows per condition."
    )


if EXPECTED_CONDITION_BUILD_METRIC_ROWS != 266:
    raise AssertionError(
        "Expected 266 build-metric rows per condition."
    )


# All 48 evaluation-period builds must be processed so
# pass-only builds can update the historical baselines.
evaluation_build_table = (
    builds[
        builds[
            "partition"
        ] == "evaluation"
    ][
        [
            "build",
            "build_order",
        ]
    ]
    .sort_values(
        "build_order",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if len(evaluation_build_table) != 48:
    raise AssertionError(
        "Expected all 48 evaluation-period builds."
    )


# ---------------------------------------------------------
# 3. Freeze feature preprocessing
# ---------------------------------------------------------

if len(MODEL_FEATURE_COLUMNS) != 150:
    raise AssertionError(
        "Expected exactly 150 original predictors."
    )


clean_feature_frame = (
    clean_training_data[
        MODEL_FEATURE_COLUMNS
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
)


feature_variation = (
    clean_feature_frame
    .nunique(
        dropna=False
    )
)


ZERO_VARIANCE_FEATURES = (
    feature_variation[
        feature_variation <= 1
    ]
    .index
    .tolist()
)


ACTIVE_FEATURE_COLUMNS = [
    column
    for column in MODEL_FEATURE_COLUMNS
    if column not in ZERO_VARIANCE_FEATURES
]


if len(ACTIVE_FEATURE_COLUMNS) == 0:
    raise AssertionError(
        "No active feature columns remain."
    )


if (
    len(ACTIVE_FEATURE_COLUMNS)
    + len(ZERO_VARIANCE_FEATURES)
    != len(MODEL_FEATURE_COLUMNS)
):
    raise AssertionError(
        "Feature partition does not cover all predictors."
    )


if set(
    ACTIVE_FEATURE_COLUMNS
).intersection(
    ZERO_VARIANCE_FEATURES
):
    raise AssertionError(
        "Active and zero-variance features overlap."
    )


# ---------------------------------------------------------
# 4. Frozen four-model configuration
# ---------------------------------------------------------

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators":
            100,

        "criterion":
            "gini",

        "max_depth":
            None,

        "min_samples_split":
            2,

        "min_samples_leaf":
            1,

        "max_features":
            "sqrt",

        "bootstrap":
            True,

        "class_weight":
            None,
    },

    "XGBoost": {
        "n_estimators":
            100,

        "max_depth":
            6,

        "learning_rate":
            0.1,

        "subsample":
            1.0,

        "colsample_bytree":
            1.0,

        "objective":
            "binary:logistic",

        "eval_metric":
            "logloss",

        "tree_method":
            "hist",
    },

    "LightGBM": {
        "n_estimators":
            100,

        "learning_rate":
            0.1,

        "num_leaves":
            31,

        "objective":
            "binary",
    },

    "NaiveBayes": {
        "variant":
            "GaussianNB",

        "var_smoothing":
            1e-9,
    },
}


MODEL_CONFIGURATION_PATH = (
    NOTES_DRIVE
    / "project_06_jetty_model_configuration.json"
)


model_configuration_record = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "FROZEN",

    "OriginalPredictorCount":
        len(MODEL_FEATURE_COLUMNS),

    "ZeroVarianceFeatureCount":
        len(ZERO_VARIANCE_FEATURES),

    "ZeroVarianceFeatures":
        ZERO_VARIANCE_FEATURES,

    "ActivePredictorCount":
        len(ACTIVE_FEATURE_COLUMNS),

    "ActivePredictors":
        ACTIVE_FEATURE_COLUMNS,

    "MissingValueRule":
        (
            "Replace infinities with missing; learn "
            "column medians from the current noisy "
            "training condition; apply those medians to "
            "training and clean evaluation matrices."
        ),

    "LabelRule":
        "binary failure indicator: Verdict != 0",

    "RankingTieRule":
        "score first, Test ascending",

    "EvaluationPolicy":
        "one fitted model over the fixed final 25%",

    "Models":
        MODEL_CONFIG,

    "CompletedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
}


with open(
    MODEL_CONFIGURATION_PATH,
    "w",
    encoding="utf-8",
) as configuration_file:
    json.dump(
        model_configuration_record,
        configuration_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 5. Prepare shared ML matrices
# ---------------------------------------------------------

def prepare_ml_matrices(
    noisy_training_data,
    evaluation_data,
):
    """
    Prepare the same matrices for all four ML models.

    Medians are learned solely from the current noisy
    training condition.
    """

    X_train = (
        noisy_training_data[
            ACTIVE_FEATURE_COLUMNS
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )


    X_evaluation = (
        evaluation_data[
            ACTIVE_FEATURE_COLUMNS
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )


    training_medians = (
        X_train
        .median(axis=0)
        .fillna(0.0)
    )


    X_train = (
        X_train
        .fillna(training_medians)
        .astype(float)
    )


    X_evaluation = (
        X_evaluation
        .fillna(training_medians)
        .astype(float)
    )


    y_train = (
        noisy_training_data[
            "Verdict"
        ] != 0
    ).astype("int8")


    if len(X_train) != len(
        noisy_training_data
    ):
        raise AssertionError(
            "Training feature-row count changed."
        )


    if len(X_evaluation) != len(
        evaluation_data
    ):
        raise AssertionError(
            "Evaluation feature-row count changed."
        )


    if y_train.nunique() != 2:
        raise ValueError(
            "The training labels do not contain both "
            "pass and failure classes."
        )


    if X_train.isna().any().any():
        raise ValueError(
            "Training features still contain missing "
            "values after imputation."
        )


    if X_evaluation.isna().any().any():
        raise ValueError(
            "Evaluation features still contain missing "
            "values after imputation."
        )


    if not np.isfinite(
        X_train.to_numpy(dtype=float)
    ).all():
        raise ValueError(
            "Training features contain non-finite values."
        )


    if not np.isfinite(
        X_evaluation.to_numpy(dtype=float)
    ).all():
        raise ValueError(
            "Evaluation features contain non-finite values."
        )


    return (
        X_train,
        y_train,
        X_evaluation,
        training_medians,
    )


# ---------------------------------------------------------
# 6. Construct deterministic models
# ---------------------------------------------------------

def create_ml_models(
    repetition_seed,
):
    """
    Create all four frozen ML models.
    """

    repetition_seed = int(
        repetition_seed
    )


    rf_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "RandomForest_model",
    )


    xgb_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "XGBoost_model",
    )


    lightgbm_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "LightGBM_model",
    )


    models = {
        "RandomForest":
            RandomForestClassifier(
                **MODEL_CONFIG[
                    "RandomForest"
                ],
                random_state=rf_seed,
                n_jobs=-1,
            ),

        "XGBoost":
            XGBClassifier(
                **MODEL_CONFIG[
                    "XGBoost"
                ],
                random_state=xgb_seed,
                n_jobs=-1,
                verbosity=0,
            ),

        "LightGBM":
            LGBMClassifier(
                **MODEL_CONFIG[
                    "LightGBM"
                ],
                random_state=lightgbm_seed,
                n_jobs=-1,
                verbosity=-1,
                deterministic=True,
                force_col_wise=True,
            ),

        "NaiveBayes":
            GaussianNB(
                var_smoothing=(
                    MODEL_CONFIG[
                        "NaiveBayes"
                    ][
                        "var_smoothing"
                    ]
                )
            ),
    }


    if list(models.keys()) != ML_TECHNIQUES:
        raise AssertionError(
            "Constructed model order does not match the "
            "frozen ML technique order."
        )


    return models


# ---------------------------------------------------------
# 7. Extract positive-class probabilities
# ---------------------------------------------------------

def get_failure_probability(
    fitted_model,
    feature_matrix,
):
    probabilities = (
        fitted_model
        .predict_proba(
            feature_matrix
        )
    )


    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )


    classes = np.asarray(
        fitted_model.classes_
    )


    positive_positions = np.flatnonzero(
        classes == 1
    )


    if len(
        positive_positions
    ) != 1:
        raise ValueError(
            "Could not uniquely identify failure class 1."
        )


    failure_probabilities = (
        probabilities[
            :,
            int(
                positive_positions[0]
            ),
        ]
    )


    if len(
        failure_probabilities
    ) != len(
        feature_matrix
    ):
        raise AssertionError(
            "Probability-row count mismatch."
        )


    if not np.isfinite(
        failure_probabilities
    ).all():
        raise ValueError(
            "Failure probabilities contain non-finite "
            "values."
        )


    if (
        (
            failure_probabilities
            < 0.0
        ).any()
        or (
            failure_probabilities
            > 1.0
        ).any()
    ):
        raise ValueError(
            "Failure probabilities are outside [0, 1]."
        )


    return failure_probabilities


# ---------------------------------------------------------
# 8. Rank one evaluation build
# ---------------------------------------------------------

def rank_build_rows(
    build_rows,
    scores,
    technique,
    score_direction="descending",
):
    """
    Rank one build by score, then Test ascending.
    """

    ranked = (
        build_rows[
            [
                "Build",
                "Test",
                "Verdict",
                "Duration",
                "build_order",
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )


    ranked[
        "Build"
    ] = pd.to_numeric(
        ranked["Build"],
        errors="raise",
    ).astype("int64")


    ranked[
        "Test"
    ] = (
        ranked["Test"]
        .astype(str)
    )


    ranked[
        "Verdict"
    ] = pd.to_numeric(
        ranked["Verdict"],
        errors="raise",
    ).astype("int8")


    ranked[
        "Duration"
    ] = pd.to_numeric(
        ranked["Duration"],
        errors="raise",
    ).astype(float)


    scores = np.asarray(
        scores,
        dtype=float,
    )


    if len(ranked) != len(scores):
        raise ValueError(
            "The number of ranking scores does not match "
            "the number of build rows."
        )


    if technique != "QTF-Avg":
        if not np.isfinite(scores).all():
            raise ValueError(
                f"{technique} contains non-finite scores."
            )

    else:
        if np.isnan(scores).any():
            raise ValueError(
                "QTF-Avg contains NaN scores."
            )


    ranked[
        "Technique"
    ] = str(
        technique
    )


    ranked[
        "Score"
    ] = scores


    ranked[
        "ActualFailure"
    ] = (
        ranked["Verdict"]
        != 0
    ).astype("int8")


    if score_direction == "descending":
        sort_ascending = [
            False,
            True,
        ]

    elif score_direction == "ascending":
        sort_ascending = [
            True,
            True,
        ]

    else:
        raise ValueError(
            "score_direction must be either "
            "'ascending' or 'descending'."
        )


    ranked = (
        ranked
        .sort_values(
            [
                "Score",
                "Test",
            ],
            ascending=sort_ascending,
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    ranked[
        "Rank"
    ] = np.arange(
        1,
        len(ranked) + 1,
        dtype=np.int64,
    )


    return ranked


# ---------------------------------------------------------
# 9. Create rankings for one ML technique
# ---------------------------------------------------------

def create_ml_rankings(
    evaluation_data,
    technique,
    failure_probabilities,
):
    scored_data = (
        evaluation_data
        .copy()
        .reset_index(drop=True)
    )


    failure_probabilities = np.asarray(
        failure_probabilities,
        dtype=float,
    )


    if len(scored_data) != len(
        failure_probabilities
    ):
        raise ValueError(
            "Evaluation rows and predicted "
            "probabilities have different lengths."
        )


    scored_data[
        "_FailureProbability"
    ] = failure_probabilities


    ranking_frames = []


    for _, build_rows in (
        scored_data
        .groupby(
            "Build",
            sort=False,
        )
    ):
        ranking_frames.append(
            rank_build_rows(
                build_rows=build_rows,
                scores=(
                    build_rows[
                        "_FailureProbability"
                    ].to_numpy(dtype=float)
                ),
                technique=technique,
                score_direction="descending",
            )
        )


    rankings = pd.concat(
        ranking_frames,
        ignore_index=True,
    )


    if len(rankings) != len(
        evaluation_data
    ):
        raise AssertionError(
            f"{technique} ranking-row count mismatch."
        )


    return rankings


# ---------------------------------------------------------
# 10. Random baseline
# ---------------------------------------------------------

def create_random_rankings(
    evaluation_data,
    repetition_seed,
):
    """
    Random rankings vary by seed but remain identical
    across noise levels for the same seed.
    """

    ranking_frames = []


    for build_id, build_rows in (
        evaluation_data
        .groupby(
            "Build",
            sort=False,
        )
    ):
        random_rng = np.random.default_rng(
            stable_project_seed(
                PROJECT_NAME,
                int(repetition_seed),
                (
                    "Random_baseline_build_"
                    f"{int(build_id)}"
                ),
            )
        )


        random_scores = random_rng.random(
            len(build_rows)
        )


        ranking_frames.append(
            rank_build_rows(
                build_rows=build_rows,
                scores=random_scores,
                technique="Random",
                score_direction="descending",
            )
        )


    rankings = pd.concat(
        ranking_frames,
        ignore_index=True,
    )


    if len(rankings) != len(
        evaluation_data
    ):
        raise AssertionError(
            "Random ranking-row count mismatch."
        )


    return rankings


# ---------------------------------------------------------
# 11. Initialise LatestFail and QTF-Avg histories
# ---------------------------------------------------------

def initialise_history_baselines(
    noisy_training_history,
):
    """
    LatestFail:
        initialised from corrupted training verdicts.

    QTF-Avg:
        initialised from clean training durations.
    """

    latest_failure_order = {}


    noisy_history_sorted = (
        noisy_training_history
        .sort_values(
            [
                "build_order",
                "job",
                "test",
            ],
            kind="mergesort",
        )
    )


    for execution_row in (
        noisy_history_sorted
        .itertuples(
            index=False
        )
    ):
        test_key = str(
            execution_row.test
        )


        if int(
            execution_row.verdict
        ) != 0:
            latest_failure_order[
                test_key
            ] = int(
                execution_row.build_order
            )


    duration_sum = {}
    duration_count = {}


    clean_duration_history = (
        clean_training_history
        .sort_values(
            [
                "build_order",
                "job",
                "test",
            ],
            kind="mergesort",
        )
    )


    for execution_row in (
        clean_duration_history
        .itertuples(
            index=False
        )
    ):
        test_key = str(
            execution_row.test
        )


        duration = float(
            execution_row.duration
        )


        if not np.isfinite(
            duration
        ):
            continue


        duration_sum[
            test_key
        ] = (
            duration_sum.get(
                test_key,
                0.0,
            )
            + duration
        )


        duration_count[
            test_key
        ] = (
            duration_count.get(
                test_key,
                0,
            )
            + 1
        )


    return (
        latest_failure_order,
        duration_sum,
        duration_count,
    )


# ---------------------------------------------------------
# 12. Create LatestFail and QTF-Avg rankings
# ---------------------------------------------------------

def create_history_baseline_rankings(
    noisy_training_history,
):
    """
    Process all 48 evaluation-period builds in order.

    The current build is ranked before its clean outcomes
    update the histories.
    """

    (
        latest_failure_order,
        duration_sum,
        duration_count,
    ) = initialise_history_baselines(
        noisy_training_history
    )


    target_build_ids = set(
        clean_evaluation_data[
            "Build"
        ].astype(int)
    )


    raw_eval_by_build = {
        int(build_id):
            build_rows.copy()

        for build_id, build_rows
        in clean_evaluation_history.groupby(
            "build",
            sort=False,
        )
    }


    model_eval_by_build = {
        int(build_id):
            build_rows.copy()

        for build_id, build_rows
        in clean_evaluation_data.groupby(
            "Build",
            sort=False,
        )
    }


    latest_fail_frames = []
    qtf_frames = []


    for build_row in (
        evaluation_build_table
        .itertuples(
            index=False
        )
    ):
        build_id = int(
            build_row.build
        )


        # Rank before revealing current-build outcomes.
        if build_id in target_build_ids:
            build_tests = (
                model_eval_by_build[
                    build_id
                ]
                .copy()
            )


            latest_scores = []
            qtf_average_scores = []


            for test_value in (
                build_tests[
                    "Test"
                ].astype(str)
            ):
                test_key = str(
                    test_value
                )


                latest_scores.append(
                    float(
                        latest_failure_order.get(
                            test_key,
                            -1,
                        )
                    )
                )


                historical_count = (
                    duration_count.get(
                        test_key,
                        0,
                    )
                )


                if historical_count > 0:
                    average_duration = (
                        duration_sum[
                            test_key
                        ]
                        / historical_count
                    )

                else:
                    average_duration = np.inf


                qtf_average_scores.append(
                    float(
                        average_duration
                    )
                )


            latest_fail_frames.append(
                rank_build_rows(
                    build_rows=build_tests,
                    scores=latest_scores,
                    technique="LatestFail",
                    score_direction="descending",
                )
            )


            qtf_frames.append(
                rank_build_rows(
                    build_rows=build_tests,
                    scores=qtf_average_scores,
                    technique="QTF-Avg",
                    score_direction="ascending",
                )
            )


        # Update both histories only after ranking.
        current_raw_rows = (
            raw_eval_by_build.get(
                build_id
            )
        )


        if current_raw_rows is None:
            continue


        current_raw_rows = (
            current_raw_rows
            .sort_values(
                [
                    "job",
                    "test",
                ],
                kind="mergesort",
            )
        )


        for execution_row in (
            current_raw_rows
            .itertuples(
                index=False
            )
        ):
            test_key = str(
                execution_row.test
            )


            if int(
                execution_row.verdict
            ) != 0:
                latest_failure_order[
                    test_key
                ] = int(
                    execution_row.build_order
                )


            duration = float(
                execution_row.duration
            )


            if np.isfinite(
                duration
            ):
                duration_sum[
                    test_key
                ] = (
                    duration_sum.get(
                        test_key,
                        0.0,
                    )
                    + duration
                )


                duration_count[
                    test_key
                ] = (
                    duration_count.get(
                        test_key,
                        0,
                    )
                    + 1
                )


    if len(latest_fail_frames) != EVALUATED_BUILD_COUNT:
        raise AssertionError(
            "LatestFail did not rank every model-ready "
            "evaluation build."
        )


    if len(qtf_frames) != EVALUATED_BUILD_COUNT:
        raise AssertionError(
            "QTF-Avg did not rank every model-ready "
            "evaluation build."
        )


    latest_fail_rankings = pd.concat(
        latest_fail_frames,
        ignore_index=True,
    )


    qtf_rankings = pd.concat(
        qtf_frames,
        ignore_index=True,
    )


    if len(
        latest_fail_rankings
    ) != EXPECTED_RANKING_ROWS_PER_TECHNIQUE:
        raise AssertionError(
            "LatestFail ranking-row count mismatch."
        )


    if len(
        qtf_rankings
    ) != EXPECTED_RANKING_ROWS_PER_TECHNIQUE:
        raise AssertionError(
            "QTF-Avg ranking-row count mismatch."
        )


    return (
        latest_fail_rankings,
        qtf_rankings,
    )


# ---------------------------------------------------------
# 13. Calculate build-level APFD and APFDc
# ---------------------------------------------------------

def calculate_build_metrics(
    all_rankings,
    noise_percent,
    repetition_seed,
):
    metric_records = []


    grouped_rankings = (
        all_rankings
        .groupby(
            [
                "Technique",
                "Build",
            ],
            sort=False,
        )
    )


    for (
        technique,
        build_id,
    ), ranked_build in grouped_rankings:
        ranked_build = (
            ranked_build
            .sort_values(
                "Rank",
                kind="mergesort",
            )
            .reset_index(drop=True)
        )


        actual_failures = (
            ranked_build[
                "ActualFailure"
            ]
            .to_numpy(dtype=int)
        )


        durations = (
            ranked_build[
                "Duration"
            ]
            .to_numpy(dtype=float)
        )


        metric_records.append({
            "Project":
                PROJECT_NAME,

            "NoisePercent":
                float(
                    noise_percent
                ),

            "RepetitionSeed":
                int(
                    repetition_seed
                ),

            "Technique":
                str(
                    technique
                ),

            "Build":
                int(
                    build_id
                ),

            "BuildOrder":
                int(
                    ranked_build[
                        "build_order"
                    ].iloc[0]
                ),

            "NumberOfTests":
                int(
                    len(
                        ranked_build
                    )
                ),

            "NumberOfFailures":
                int(
                    actual_failures.sum()
                ),

            "TotalDuration":
                float(
                    durations.sum()
                ),

            "APFD":
                float(
                    calculate_apfd(
                        actual_failures
                    )
                ),

            "APFDc":
                float(
                    calculate_apfdc(
                        actual_failures,
                        durations,
                    )
                ),
        })


    return pd.DataFrame(
        metric_records
    )


# ---------------------------------------------------------
# 14. Aggregate one project × noise × seed run
# ---------------------------------------------------------

def aggregate_project_run_metrics(
    build_metrics,
):
    project_run_rows = []


    for technique, technique_rows in (
        build_metrics.groupby(
            "Technique",
            sort=False,
        )
    ):
        apfd_values = (
            technique_rows[
                "APFD"
            ].to_numpy(dtype=float)
        )


        apfdc_values = (
            technique_rows[
                "APFDc"
            ].to_numpy(dtype=float)
        )


        evaluated_builds = int(
            len(
                technique_rows
            )
        )


        sd_apfd = float(
            np.std(
                apfd_values,
                ddof=1,
            )
        )


        sd_apfdc = float(
            np.std(
                apfdc_values,
                ddof=1,
            )
        )


        se_apfd = (
            sd_apfd
            / math.sqrt(
                evaluated_builds
            )
        )


        se_apfdc = (
            sd_apfdc
            / math.sqrt(
                evaluated_builds
            )
        )


        mean_apfd = float(
            np.mean(
                apfd_values
            )
        )


        mean_apfdc = float(
            np.mean(
                apfdc_values
            )
        )


        project_run_rows.append({
            "Project":
                PROJECT_NAME,

            "NoisePercent":
                float(
                    technique_rows[
                        "NoisePercent"
                    ].iloc[0]
                ),

            "RepetitionSeed":
                int(
                    technique_rows[
                        "RepetitionSeed"
                    ].iloc[0]
                ),

            "Technique":
                str(
                    technique
                ),

            "MeanAPFD":
                mean_apfd,

            "MeanAPFDc":
                mean_apfdc,

            "SD_APFD":
                sd_apfd,

            "SD_APFDc":
                sd_apfdc,

            "CI95Low_APFD":
                float(
                    mean_apfd
                    - 1.96
                    * se_apfd
                ),

            "CI95High_APFD":
                float(
                    mean_apfd
                    + 1.96
                    * se_apfd
                ),

            "CI95Low_APFDc":
                float(
                    mean_apfdc
                    - 1.96
                    * se_apfdc
                ),

            "CI95High_APFDc":
                float(
                    mean_apfdc
                    + 1.96
                    * se_apfdc
                ),

            "EvaluatedBuilds":
                evaluated_builds,

            "RankingRows":
                int(
                    technique_rows[
                        "NumberOfTests"
                    ].sum()
                ),

            "FailureExecutions":
                int(
                    technique_rows[
                        "NumberOfFailures"
                    ].sum()
                ),
        })


    project_run_metrics = pd.DataFrame(
        project_run_rows
    )


    return project_run_metrics


# ---------------------------------------------------------
# 15. Validate complete condition output
# ---------------------------------------------------------

def validate_condition_outputs(
    rankings,
    build_metrics,
    project_run_metrics,
    fit_times,
):
    expected_techniques = set(
        ALL_TECHNIQUES
    )


    actual_ranking_techniques = set(
        rankings[
            "Technique"
        ].astype(str)
    )


    if actual_ranking_techniques != (
        expected_techniques
    ):
        raise AssertionError(
            "Ranking technique mismatch.\n"
            f"Expected: {expected_techniques}\n"
            f"Observed: {actual_ranking_techniques}"
        )


    if len(rankings) != (
        EXPECTED_CONDITION_RANKING_ROWS
    ):
        raise AssertionError(
            "Unexpected condition ranking-row count.\n"
            f"Expected: "
            f"{EXPECTED_CONDITION_RANKING_ROWS}\n"
            f"Observed: {len(rankings)}"
        )


    if len(build_metrics) != (
        EXPECTED_CONDITION_BUILD_METRIC_ROWS
    ):
        raise AssertionError(
            "Unexpected build-metric row count.\n"
            f"Expected: "
            f"{EXPECTED_CONDITION_BUILD_METRIC_ROWS}\n"
            f"Observed: {len(build_metrics)}"
        )


    if len(project_run_metrics) != (
        EXPECTED_CONDITION_PROJECT_RUN_ROWS
    ):
        raise AssertionError(
            "Expected seven project-run rows."
        )


    if len(fit_times) != (
        EXPECTED_CONDITION_FIT_ROWS
    ):
        raise AssertionError(
            "Expected four model-fit records."
        )


    expected_build_test_pairs = set(
        zip(
            clean_evaluation_data[
                "Build"
            ].astype(int),

            clean_evaluation_data[
                "Test"
            ].astype(str),
        )
    )


    for technique in ALL_TECHNIQUES:
        technique_rankings = (
            rankings[
                rankings[
                    "Technique"
                ] == technique
            ]
            .copy()
        )


        if len(
            technique_rankings
        ) != EXPECTED_RANKING_ROWS_PER_TECHNIQUE:
            raise AssertionError(
                f"{technique} ranking-row count mismatch."
            )


        observed_pairs = set(
            zip(
                technique_rankings[
                    "Build"
                ].astype(int),

                technique_rankings[
                    "Test"
                ].astype(str),
            )
        )


        if observed_pairs != (
            expected_build_test_pairs
        ):
            raise AssertionError(
                f"{technique} Build-Test cohort mismatch."
            )


        for build_id, build_rows in (
            technique_rankings.groupby(
                "Build",
                sort=False,
            )
        ):
            build_rows = (
                build_rows
                .sort_values(
                    "Rank",
                    kind="mergesort",
                )
                .reset_index(drop=True)
            )


            expected_ranks = np.arange(
                1,
                len(build_rows) + 1,
                dtype=np.int64,
            )


            if not np.array_equal(
                build_rows[
                    "Rank"
                ].to_numpy(dtype=np.int64),
                expected_ranks,
            ):
                raise AssertionError(
                    f"Invalid rank sequence for "
                    f"{technique}, build {build_id}."
                )


            if build_rows[
                "Test"
            ].duplicated().any():
                raise AssertionError(
                    f"Duplicate tests for "
                    f"{technique}, build {build_id}."
                )


            if technique == "QTF-Avg":
                expected_order = (
                    build_rows
                    .sort_values(
                        [
                            "Score",
                            "Test",
                        ],
                        ascending=[
                            True,
                            True,
                        ],
                        kind="mergesort",
                    )[
                        "Test"
                    ]
                    .astype(str)
                    .tolist()
                )

            else:
                expected_order = (
                    build_rows
                    .sort_values(
                        [
                            "Score",
                            "Test",
                        ],
                        ascending=[
                            False,
                            True,
                        ],
                        kind="mergesort",
                    )[
                        "Test"
                    ]
                    .astype(str)
                    .tolist()
                )


            observed_order = (
                build_rows[
                    "Test"
                ]
                .astype(str)
                .tolist()
            )


            if observed_order != expected_order:
                raise AssertionError(
                    "Score/Test tie ordering failed for "
                    f"{technique}, build {build_id}."
                )


    expected_metric_techniques = set(
        build_metrics[
            "Technique"
        ].astype(str)
    )


    if expected_metric_techniques != (
        expected_techniques
    ):
        raise AssertionError(
            "Build-metric technique mismatch."
        )


    expected_evaluation_builds = set(
        clean_evaluation_data[
            "Build"
        ].astype(int)
    )


    for technique in ALL_TECHNIQUES:
        technique_builds = set(
            build_metrics.loc[
                build_metrics[
                    "Technique"
                ] == technique,
                "Build",
            ].astype(int)
        )


        if technique_builds != (
            expected_evaluation_builds
        ):
            raise AssertionError(
                f"{technique} build-metric cohort mismatch."
            )


    if build_metrics[
        [
            "APFD",
            "APFDc",
        ]
    ].isna().any().any():
        raise AssertionError(
            "APFD or APFDc contains missing values."
        )


    for metric_name in [
        "APFD",
        "APFDc",
    ]:
        metric_values = (
            build_metrics[
                metric_name
            ].to_numpy(dtype=float)
        )


        if not np.isfinite(
            metric_values
        ).all():
            raise AssertionError(
                f"{metric_name} contains non-finite values."
            )


        if (
            (
                metric_values
                < -1e-12
            ).any()
            or (
                metric_values
                > 1.0 + 1e-12
            ).any()
        ):
            raise AssertionError(
                f"{metric_name} is outside [0, 1]."
            )


    if (
        build_metrics[
            "NumberOfFailures"
        ] <= 0
    ).any():
        raise AssertionError(
            "A model-ready evaluation build has no "
            "failure executions."
        )


    if set(
        project_run_metrics[
            "Technique"
        ].astype(str)
    ) != expected_techniques:
        raise AssertionError(
            "Project-run technique mismatch."
        )


    if set(
        fit_times[
            "Technique"
        ].astype(str)
    ) != set(
        ML_TECHNIQUES
    ):
        raise AssertionError(
            "Fit-time technique mismatch."
        )


    return True


# ---------------------------------------------------------
# 16. Ranking content signature
# ---------------------------------------------------------

def ranking_signature(
    rankings,
):
    signature_frame = (
        rankings[
            [
                "Technique",
                "Build",
                "Test",
                "Rank",
            ]
        ]
        .sort_values(
            [
                "Technique",
                "Build",
                "Rank",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    row_hashes = (
        pd.util.hash_pandas_object(
            signature_frame,
            index=False,
        )
        .to_numpy(dtype="uint64")
    )


    return hashlib.sha256(
        row_hashes.tobytes()
    ).hexdigest()


# ---------------------------------------------------------
# 17. Define complete condition runner
# ---------------------------------------------------------

def run_unified_condition(
    noise_percent,
    repetition_seed,
    save_rankings=True,
    overwrite=False,
):
    """
    Run one complete Project × Noise × Seed condition.

    One corrupted training history is shared by:
      - all four ML models;
      - LatestFail.

    Random remains constant across noise for a given seed.
    QTF-Avg remains duration-only and noise-independent.
    """

    noise_percent = int(
        noise_percent
    )


    repetition_seed = int(
        repetition_seed
    )


    if noise_percent not in NOISE_LEVELS:
        raise ValueError(
            f"Unsupported noise level: {noise_percent}"
        )


    if repetition_seed not in REPETITION_SEEDS:
        raise ValueError(
            f"Unsupported repetition seed: "
            f"{repetition_seed}"
        )


    condition_start_time = time.time()


    condition_directory = (
        PROJECT_RAW_RESULTS
        / f"noise_{noise_percent:03d}"
        / f"seed_{repetition_seed:02d}"
    )


    condition_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


    completion_marker = (
        condition_directory
        / "_SUCCESS.json"
    )


    if (
        completion_marker.exists()
        and not overwrite
    ):
        raise FileExistsError(
            "This condition already has a success marker:\n"
            f"{completion_marker}\n\n"
            "Use overwrite=True only for an intentional "
            "audited rerun."
        )


    evaluation_hash_before = (
        dataframe_sha256(
            clean_evaluation_data
        )
    )


    training_history_hash_before = (
        dataframe_sha256(
            clean_training_history
        )
    )


    # One shared corruption for ML + LatestFail.
    (
        noisy_training_history,
        noise_manifest,
        noise_summary,
    ) = inject_training_verdict_noise(
        execution_history=(
            clean_training_history
        ),
        noise_percent=(
            noise_percent
        ),
        repetition_seed=(
            repetition_seed
        ),
    )


    noisy_training_data = (
        create_noisy_model_training_data(
            clean_model_training_data=(
                clean_training_data
            ),
            noisy_execution_history=(
                noisy_training_history
            ),
        )
    )


    (
        X_train,
        y_train,
        X_evaluation,
        training_medians,
    ) = prepare_ml_matrices(
        noisy_training_data=(
            noisy_training_data
        ),
        evaluation_data=(
            clean_evaluation_data
        ),
    )


    ranking_frames = []
    fit_records = []


    models = create_ml_models(
        repetition_seed
    )


    for technique, model in models.items():
        fit_start_time = time.time()


        with warnings.catch_warnings():
            warnings.simplefilter(
                "ignore"
            )


            model.fit(
                X_train,
                y_train,
            )


        failure_probabilities = (
            get_failure_probability(
                fitted_model=model,
                feature_matrix=(
                    X_evaluation
                ),
            )
        )


        model_rankings = (
            create_ml_rankings(
                evaluation_data=(
                    clean_evaluation_data
                ),
                technique=technique,
                failure_probabilities=(
                    failure_probabilities
                ),
            )
        )


        ranking_frames.append(
            model_rankings
        )


        fit_records.append({
            "Project":
                PROJECT_NAME,

            "NoisePercent":
                float(
                    noise_percent
                ),

            "RepetitionSeed":
                repetition_seed,

            "Technique":
                technique,

            "FitSeconds":
                float(
                    time.time()
                    - fit_start_time
                ),

            "TrainingRows":
                int(
                    len(
                        X_train
                    )
                ),

            "TrainingFailures":
                int(
                    y_train.sum()
                ),

            "TrainingPasses":
                int(
                    len(y_train)
                    - y_train.sum()
                ),

            "EvaluationRows":
                int(
                    len(
                        X_evaluation
                    )
                ),

            "ActiveFeatures":
                int(
                    len(
                        ACTIVE_FEATURE_COLUMNS
                    )
                ),
        })


    # Random baseline.
    random_rankings = (
        create_random_rankings(
            evaluation_data=(
                clean_evaluation_data
            ),
            repetition_seed=(
                repetition_seed
            ),
        )
    )


    ranking_frames.append(
        random_rankings
    )


    # LatestFail and QTF-Avg.
    (
        latest_fail_rankings,
        qtf_rankings,
    ) = create_history_baseline_rankings(
        noisy_training_history=(
            noisy_training_history
        )
    )


    ranking_frames.extend([
        latest_fail_rankings,
        qtf_rankings,
    ])


    all_rankings = pd.concat(
        ranking_frames,
        ignore_index=True,
    )


    all_rankings.insert(
        0,
        "Project",
        PROJECT_NAME,
    )


    all_rankings.insert(
        1,
        "NoisePercent",
        float(
            noise_percent
        ),
    )


    all_rankings.insert(
        2,
        "RepetitionSeed",
        repetition_seed,
    )


    build_metrics = (
        calculate_build_metrics(
            all_rankings=(
                all_rankings
            ),
            noise_percent=(
                noise_percent
            ),
            repetition_seed=(
                repetition_seed
            ),
        )
    )


    project_run_metrics = (
        aggregate_project_run_metrics(
            build_metrics
        )
    )


    fit_times = pd.DataFrame(
        fit_records
    )


    validate_condition_outputs(
        rankings=all_rankings,
        build_metrics=build_metrics,
        project_run_metrics=(
            project_run_metrics
        ),
        fit_times=fit_times,
    )


    evaluation_hash_after = (
        dataframe_sha256(
            clean_evaluation_data
        )
    )


    training_history_hash_after = (
        dataframe_sha256(
            clean_training_history
        )
    )


    evaluation_unchanged = bool(
        evaluation_hash_before
        == evaluation_hash_after
    )


    clean_training_history_unchanged = bool(
        training_history_hash_before
        == training_history_hash_after
    )


    if not evaluation_unchanged:
        raise AssertionError(
            "The clean evaluation data changed while "
            "running the condition."
        )


    if not clean_training_history_unchanged:
        raise AssertionError(
            "The clean training history changed while "
            "running the condition."
        )


    noise_summary_dataframe = pd.DataFrame([
        noise_summary
    ])


    training_medians_dataframe = (
        training_medians
        .rename(
            "TrainingMedian"
        )
        .reset_index()
        .rename(
            columns={
                "index":
                    "Feature",
            }
        )
    )


    # Save output files only after all validations pass.
    noise_manifest.to_parquet(
        condition_directory
        / "noise_manifest.parquet",
        index=False,
    )


    noise_summary_dataframe.to_csv(
        condition_directory
        / "noise_summary.csv",
        index=False,
    )


    fit_times.to_csv(
        condition_directory
        / "fit_times.csv",
        index=False,
    )


    build_metrics.to_parquet(
        condition_directory
        / "build_metrics.parquet",
        index=False,
    )


    project_run_metrics.to_csv(
        condition_directory
        / "project_run_metrics.csv",
        index=False,
    )


    training_medians_dataframe.to_csv(
        condition_directory
        / "training_medians.csv",
        index=False,
    )


    if save_rankings:
        all_rankings.to_parquet(
            condition_directory
            / "rankings.parquet",
            index=False,
        )


    condition_metadata = {
        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "NoisePercent":
            noise_percent,

        "RepetitionSeed":
            repetition_seed,

        "TrainingExecutionRows":
            int(
                len(
                    clean_training_history
                )
            ),

        "TrainingModelRows":
            int(
                len(
                    noisy_training_data
                )
            ),

        "EvaluationModelRows":
            int(
                len(
                    clean_evaluation_data
                )
            ),

        "EvaluatedBuilds":
            EVALUATED_BUILD_COUNT,

        "TechniqueCount":
            len(
                ALL_TECHNIQUES
            ),

        "ModelFitRows":
            int(
                len(
                    fit_times
                )
            ),

        "RankingRows":
            int(
                len(
                    all_rankings
                )
            ),

        "BuildMetricRows":
            int(
                len(
                    build_metrics
                )
            ),

        "ProjectRunRows":
            int(
                len(
                    project_run_metrics
                )
            ),

        "NoiseFlips":
            int(
                noise_summary[
                    "NumberFlipped"
                ]
            ),

        "RealisedNoisePercent":
            float(
                noise_summary[
                    "RealisedNoisePercent"
                ]
            ),

        "RankingSignature":
            ranking_signature(
                all_rankings
            ),

        "EvaluationHashBefore":
            evaluation_hash_before,

        "EvaluationHashAfter":
            evaluation_hash_after,

        "EvaluationHashUnchanged":
            evaluation_unchanged,

        "CleanTrainingHistoryUnchanged":
            clean_training_history_unchanged,

        "ElapsedSeconds":
            float(
                time.time()
                - condition_start_time
            ),

        "CompletedSuccessfully":
            True,

        "CompletedAtUTC":
            pd.Timestamp.utcnow().isoformat(),
    }


    # Success marker must be written last.
    with open(
        completion_marker,
        "w",
        encoding="utf-8",
    ) as marker_file:
        json.dump(
            condition_metadata,
            marker_file,
            indent=2,
            default=str,
        )


    return {
        "noise_summary":
            noise_summary_dataframe,

        "fit_times":
            fit_times,

        "rankings":
            all_rankings,

        "build_metrics":
            build_metrics,

        "project_run_metrics":
            project_run_metrics,

        "condition_metadata":
            condition_metadata,

        "condition_directory":
            condition_directory,
    }


# ---------------------------------------------------------
# 18. Step 8 non-ML validation
# ---------------------------------------------------------

# Verify clean matrices are constructible without fitting.
(
    step_8_clean_X_train,
    step_8_clean_y_train,
    step_8_clean_X_evaluation,
    step_8_clean_training_medians,
) = prepare_ml_matrices(
    noisy_training_data=(
        clean_training_data
    ),
    evaluation_data=(
        clean_evaluation_data
    ),
)


if step_8_clean_X_train.shape != (
    len(clean_training_data),
    len(ACTIVE_FEATURE_COLUMNS),
):
    raise AssertionError(
        "Clean training matrix shape mismatch."
    )


if step_8_clean_X_evaluation.shape != (
    len(clean_evaluation_data),
    len(ACTIVE_FEATURE_COLUMNS),
):
    raise AssertionError(
        "Clean evaluation matrix shape mismatch."
    )


if int(
    step_8_clean_y_train.sum()
) != 130:
    raise AssertionError(
        "Expected 130 failing model-training rows."
    )


# Verify model construction without fitting.
step_8_models = create_ml_models(
    repetition_seed=1
)


if list(
    step_8_models.keys()
) != ML_TECHNIQUES:
    raise AssertionError(
        "Model construction validation failed."
    )


# Validate all three baseline pipelines at clean seed 1.
step_8_random_rankings = (
    create_random_rankings(
        evaluation_data=(
            clean_evaluation_data
        ),
        repetition_seed=1,
    )
)


(
    step_8_latest_fail_rankings,
    step_8_qtf_rankings,
) = create_history_baseline_rankings(
    noisy_training_history=(
        clean_training_history
    )
)


for baseline_name, baseline_rankings in [
    (
        "Random",
        step_8_random_rankings,
    ),
    (
        "LatestFail",
        step_8_latest_fail_rankings,
    ),
    (
        "QTF-Avg",
        step_8_qtf_rankings,
    ),
]:
    if len(
        baseline_rankings
    ) != EXPECTED_RANKING_ROWS_PER_TECHNIQUE:
        raise AssertionError(
            f"{baseline_name} baseline row-count "
            "validation failed."
        )


    if int(
        baseline_rankings[
            "Build"
        ].nunique()
    ) != EVALUATED_BUILD_COUNT:
        raise AssertionError(
            f"{baseline_name} did not rank all "
            "38 evaluation builds."
        )


baseline_validation_rankings = pd.concat(
    [
        step_8_random_rankings,
        step_8_latest_fail_rankings,
        step_8_qtf_rankings,
    ],
    ignore_index=True,
)


baseline_validation_metrics = (
    calculate_build_metrics(
        all_rankings=(
            baseline_validation_rankings
        ),
        noise_percent=0,
        repetition_seed=1,
    )
)


if len(
    baseline_validation_metrics
) != (
    EVALUATED_BUILD_COUNT
    * len(
        BASELINE_TECHNIQUES
    )
):
    raise AssertionError(
        "Baseline metric-row count mismatch."
    )


if baseline_validation_metrics[
    [
        "APFD",
        "APFDc",
    ]
].isna().any().any():
    raise AssertionError(
        "Baseline APFD/APFDc contains missing values."
    )


baseline_signature_records = []


for baseline_name, baseline_rankings in [
    (
        "Random",
        step_8_random_rankings,
    ),
    (
        "LatestFail",
        step_8_latest_fail_rankings,
    ),
    (
        "QTF-Avg",
        step_8_qtf_rankings,
    ),
]:
    baseline_signature_records.append({
        "Technique":
            baseline_name,

        "RankingRows":
            int(
                len(
                    baseline_rankings
                )
            ),

        "EvaluatedBuilds":
            int(
                baseline_rankings[
                    "Build"
                ].nunique()
            ),

        "RankingSignature":
            ranking_signature(
                baseline_rankings
            ),
    })


baseline_validation_summary = pd.DataFrame(
    baseline_signature_records
)


# ---------------------------------------------------------
# 19. Save Step 8 report
# ---------------------------------------------------------

RUNNER_CONFIGURATION_DIRECTORY = (
    PROJECT_AGGREGATED_RESULTS
    / "jetty_runner_configuration"
)


RUNNER_CONFIGURATION_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


BASELINE_VALIDATION_PATH = (
    RUNNER_CONFIGURATION_DIRECTORY
    / "jetty_baseline_preflight_summary.csv"
)


RUNNER_CONFIGURATION_REPORT_PATH = (
    RUNNER_CONFIGURATION_DIRECTORY
    / "jetty_runner_configuration_report.json"
)


baseline_validation_summary.to_csv(
    BASELINE_VALIDATION_PATH,
    index=False,
)


runner_configuration_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "PASS",

    "OriginalPredictors":
        len(
            MODEL_FEATURE_COLUMNS
        ),

    "ZeroVarianceFeatures":
        len(
            ZERO_VARIANCE_FEATURES
        ),

    "ZeroVarianceFeatureNames":
        ZERO_VARIANCE_FEATURES,

    "ActivePredictors":
        len(
            ACTIVE_FEATURE_COLUMNS
        ),

    "TrainingMatrixShape":
        list(
            step_8_clean_X_train.shape
        ),

    "EvaluationMatrixShape":
        list(
            step_8_clean_X_evaluation.shape
        ),

    "CleanTrainingFailures":
        int(
            step_8_clean_y_train.sum()
        ),

    "Models":
        ML_TECHNIQUES,

    "Baselines":
        BASELINE_TECHNIQUES,

    "EvaluatedBuilds":
        EVALUATED_BUILD_COUNT,

    "ExpectedRankingRowsPerTechnique":
        EXPECTED_RANKING_ROWS_PER_TECHNIQUE,

    "ExpectedConditionRankingRows":
        EXPECTED_CONDITION_RANKING_ROWS,

    "ExpectedConditionBuildMetricRows":
        EXPECTED_CONDITION_BUILD_METRIC_ROWS,

    "ExpectedConditionProjectRunRows":
        EXPECTED_CONDITION_PROJECT_RUN_ROWS,

    "ExpectedConditionFitRows":
        EXPECTED_CONDITION_FIT_ROWS,

    "BaselinePreflight":
        baseline_validation_summary.to_dict(
            orient="records"
        ),

    "ConditionRunnerDefined":
        True,

    "ModelConfiguration":
        str(
            MODEL_CONFIGURATION_PATH
        ),

    "BaselineValidation":
        str(
            BASELINE_VALIDATION_PATH
        ),

    "CompletedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
}


with open(
    RUNNER_CONFIGURATION_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        runner_configuration_report,
        report_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 20. Update Project 6 checkpoint
# ---------------------------------------------------------

with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint = json.load(
        checkpoint_file
    )


checkpoint.update({
    "Status":
        "RUNNER_CONFIGURATION_PASSED",

    "ModelConfiguration":
        str(
            MODEL_CONFIGURATION_PATH
        ),

    "RunnerConfigurationReport":
        str(
            RUNNER_CONFIGURATION_REPORT_PATH
        ),

    "OriginalPredictors":
        len(
            MODEL_FEATURE_COLUMNS
        ),

    "ZeroVarianceFeatures":
        ZERO_VARIANCE_FEATURES,

    "ActivePredictors":
        len(
            ACTIVE_FEATURE_COLUMNS
        ),

    "EvaluatedBuilds":
        EVALUATED_BUILD_COUNT,

    "ExpectedConditionRankingRows":
        EXPECTED_CONDITION_RANKING_ROWS,

    "ExpectedConditionBuildMetricRows":
        EXPECTED_CONDITION_BUILD_METRIC_ROWS,

    "ExpectedConditionProjectRunRows":
        EXPECTED_CONDITION_PROJECT_RUN_ROWS,

    "ExpectedConditionFitRows":
        EXPECTED_CONDITION_FIT_ROWS,

    "ConditionRunnerDefined":
        True,

    "UpdatedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
})


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 21. Compact final output
# ---------------------------------------------------------

print(
    "\n=== PROJECT 6 STEP 8 RESULT ==="
)


print(
    "\nFeature protocol:"
)

print(
    "Original predictors:",
    len(
        MODEL_FEATURE_COLUMNS
    )
)

print(
    "Zero-variance predictors:",
    len(
        ZERO_VARIANCE_FEATURES
    )
)

print(
    "Active predictors:",
    len(
        ACTIVE_FEATURE_COLUMNS
    )
)


print(
    "\nZero-variance feature names:"
)

if ZERO_VARIANCE_FEATURES:
    for feature_name in (
        ZERO_VARIANCE_FEATURES
    ):
        print(
            " -",
            feature_name
        )

else:
    print(
        "None"
    )


print(
    "\nML matrices:"
)

print(
    "Training shape:",
    step_8_clean_X_train.shape
)

print(
    "Evaluation shape:",
    step_8_clean_X_evaluation.shape
)

print(
    "Clean model-training failures:",
    int(
        step_8_clean_y_train.sum()
    )
)


print(
    "\nFrozen models:"
)

print(
    ML_TECHNIQUES
)


print(
    "\nBaseline preflight:"
)

display(
    baseline_validation_summary
)


print(
    "\nExpected output per condition:"
)

print(
    "Ranking rows:",
    EXPECTED_CONDITION_RANKING_ROWS
)

print(
    "Build-metric rows:",
    EXPECTED_CONDITION_BUILD_METRIC_ROWS
)

print(
    "Project-run rows:",
    EXPECTED_CONDITION_PROJECT_RUN_ROWS
)

print(
    "ML fit rows:",
    EXPECTED_CONDITION_FIT_ROWS
)


print(
    "\nModel configuration:"
)

print(
    MODEL_CONFIGURATION_PATH
)


print(
    "Runner configuration report:"
)

print(
    RUNNER_CONFIGURATION_REPORT_PATH
)


print(
    "\nValidation status:",
    "PASS"
)


print(
    "\nSUCCESS: Predictor preprocessing was frozen."
)

print(
    "SUCCESS: All four deterministic ML models were "
    "constructed."
)

print(
    "SUCCESS: Random, LatestFail and QTF-Avg baseline "
    "pipelines passed preflight."
)

print(
    "SUCCESS: The complete condition runner is defined."
)

print(
    "SUCCESS: No ML model was fitted during Step 8."
)

print(
    "SUCCESS: Project 6 is ready for the complete "
    "0% noise, seed 1 smoke condition."
)

=== PROJECT 6 STEP 8: MODEL AND RUNNER CONFIGURATION ===

=== PROJECT 6 STEP 8 RESULT ===

Feature protocol:
Original predictors: 150
Zero-variance predictors: 0
Active predictors: 150

Zero-variance feature names:
None

ML matrices:
Training shape: (13212, 150)
Evaluation shape: (4079, 150)
Clean model-training failures: 130

Frozen models:
['RandomForest', 'XGBoost', 'LightGBM', 'NaiveBayes']

Baseline preflight:


,Technique,RankingRows,EvaluatedBuilds,RankingSignature
0,Random,4079,38,d218711fbad367303ddd07e816509b2b7a0a98712a6d6b...
1,LatestFail,4079,38,a5e6092546b93c6d8db8e18da7efd4840b3ca1f4942eeb...
2,QTF-Avg,4079,38,52f042be055fd9ae14a8c59ab9dc8e8c0bfa8ae486fc78...



Expected output per condition:
Ranking rows: 28553
Build-metric rows: 266
Project-run rows: 7
ML fit rows: 4

Model configuration:
/content/drive/MyDrive/Thesis_Experiment/Notes/project_06_jetty_model_configuration.json
Runner configuration report:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/eclipse__jetty.project/jetty_runner_configuration/jetty_runner_configuration_report.json

Validation status: PASS

SUCCESS: Predictor preprocessing was frozen.
SUCCESS: All four deterministic ML models were constructed.
SUCCESS: Random, LatestFail and QTF-Avg baseline pipelines passed preflight.
SUCCESS: The complete condition runner is defined.
SUCCESS: No ML model was fitted during Step 8.
SUCCESS: Project 6 is ready for the complete 0% noise, seed 1 smoke condition.


In [ ]:
# =========================================================
# PROJECT 6 — STEP 9
# COMPLETE 0% NOISE × SEED 1 SMOKE CONDITION
#
# This cell:
#   - runs the first canonical experiment condition
#   - fits all four ML models
#   - runs all three baselines
#   - validates rankings, build metrics and project metrics
#   - independently recomputes APFD/APFDc
#   - confirms Step 8 baseline signatures
#   - preserves the successful condition for the full run
#
# It safely reloads the condition when a valid success
# marker already exists.
# =========================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display


print(
    "=== PROJECT 6 STEP 9: "
    "0% NOISE × SEED 1 SMOKE CONDITION ==="
)


# ---------------------------------------------------------
# 1. Confirm successful Step 8 state
# ---------------------------------------------------------

required_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",

    "PROJECT_RAW_RESULTS",
    "PROJECT_AGGREGATED_RESULTS",
    "PROJECT_6_SELECTION_CHECKPOINT",

    "ML_TECHNIQUES",
    "BASELINE_TECHNIQUES",
    "ALL_TECHNIQUES",

    "ACTIVE_FEATURE_COLUMNS",

    "clean_training_data",
    "clean_evaluation_data",

    "EXPECTED_RANKING_ROWS_PER_TECHNIQUE",
    "EXPECTED_CONDITION_RANKING_ROWS",
    "EXPECTED_CONDITION_BUILD_METRIC_ROWS",
    "EXPECTED_CONDITION_PROJECT_RUN_ROWS",
    "EXPECTED_CONDITION_FIT_ROWS",
    "EVALUATED_BUILD_COUNT",

    "run_unified_condition",
    "validate_condition_outputs",
    "calculate_build_metrics",
    "aggregate_project_run_metrics",
    "ranking_signature",
]


missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required Step 8 objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nRerun the successful Step 6, Step 7 and "
        "Step 8 cells in this connected runtime."
    )


if PROJECT_NUMBER != 6:
    raise AssertionError(
        f"Expected Project 6, observed {PROJECT_NUMBER}."
    )


if PROJECT_NAME != "eclipse@jetty.project":
    raise AssertionError(
        f"Unexpected project: {PROJECT_NAME}"
    )


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    step_8_checkpoint = json.load(
        checkpoint_file
    )


allowed_previous_statuses = {
    "RUNNER_CONFIGURATION_PASSED",
    "SMOKE_CONDITION_PASSED",
}


if step_8_checkpoint.get(
    "Status"
) not in allowed_previous_statuses:
    raise AssertionError(
        "Step 8 has not been marked as passed.\n"
        f"Observed status: "
        f"{step_8_checkpoint.get('Status')}"
    )


# ---------------------------------------------------------
# 2. Define the canonical smoke condition
# ---------------------------------------------------------

SMOKE_NOISE_PERCENT = 0
SMOKE_REPETITION_SEED = 1


SMOKE_CONDITION_DIRECTORY = (
    PROJECT_RAW_RESULTS
    / "noise_000"
    / "seed_01"
)


SMOKE_SUCCESS_MARKER = (
    SMOKE_CONDITION_DIRECTORY
    / "_SUCCESS.json"
)


SMOKE_RANKINGS_PATH = (
    SMOKE_CONDITION_DIRECTORY
    / "rankings.parquet"
)


SMOKE_BUILD_METRICS_PATH = (
    SMOKE_CONDITION_DIRECTORY
    / "build_metrics.parquet"
)


SMOKE_PROJECT_RUN_METRICS_PATH = (
    SMOKE_CONDITION_DIRECTORY
    / "project_run_metrics.csv"
)


SMOKE_FIT_TIMES_PATH = (
    SMOKE_CONDITION_DIRECTORY
    / "fit_times.csv"
)


SMOKE_NOISE_SUMMARY_PATH = (
    SMOKE_CONDITION_DIRECTORY
    / "noise_summary.csv"
)


# ---------------------------------------------------------
# 3. Run or safely reload the smoke condition
# ---------------------------------------------------------

condition_loaded_from_existing_files = False


if SMOKE_SUCCESS_MARKER.exists():
    required_condition_files = [
        SMOKE_RANKINGS_PATH,
        SMOKE_BUILD_METRICS_PATH,
        SMOKE_PROJECT_RUN_METRICS_PATH,
        SMOKE_FIT_TIMES_PATH,
        SMOKE_NOISE_SUMMARY_PATH,
    ]


    missing_condition_files = [
        str(path)
        for path in required_condition_files
        if not path.exists()
    ]


    if missing_condition_files:
        raise FileNotFoundError(
            "The smoke success marker exists, but some "
            "required condition files are missing:\n"
            + "\n".join(missing_condition_files)
        )


    with open(
        SMOKE_SUCCESS_MARKER,
        "r",
        encoding="utf-8",
    ) as marker_file:
        smoke_condition_metadata = json.load(
            marker_file
        )


    smoke_rankings = pd.read_parquet(
        SMOKE_RANKINGS_PATH
    )


    smoke_build_metrics = pd.read_parquet(
        SMOKE_BUILD_METRICS_PATH
    )


    smoke_project_run_metrics = pd.read_csv(
        SMOKE_PROJECT_RUN_METRICS_PATH
    )


    smoke_fit_times = pd.read_csv(
        SMOKE_FIT_TIMES_PATH
    )


    smoke_noise_summary = pd.read_csv(
        SMOKE_NOISE_SUMMARY_PATH
    )


    condition_loaded_from_existing_files = True


else:
    smoke_result = run_unified_condition(
        noise_percent=SMOKE_NOISE_PERCENT,
        repetition_seed=SMOKE_REPETITION_SEED,
        save_rankings=True,
        overwrite=False,
    )


    smoke_rankings = (
        smoke_result[
            "rankings"
        ].copy()
    )


    smoke_build_metrics = (
        smoke_result[
            "build_metrics"
        ].copy()
    )


    smoke_project_run_metrics = (
        smoke_result[
            "project_run_metrics"
        ].copy()
    )


    smoke_fit_times = (
        smoke_result[
            "fit_times"
        ].copy()
    )


    smoke_noise_summary = (
        smoke_result[
            "noise_summary"
        ].copy()
    )


    smoke_condition_metadata = (
        smoke_result[
            "condition_metadata"
        ]
    )


# ---------------------------------------------------------
# 4. Validate complete condition structure
# ---------------------------------------------------------

validate_condition_outputs(
    rankings=smoke_rankings,
    build_metrics=smoke_build_metrics,
    project_run_metrics=(
        smoke_project_run_metrics
    ),
    fit_times=smoke_fit_times,
)


if len(smoke_rankings) != (
    EXPECTED_CONDITION_RANKING_ROWS
):
    raise AssertionError(
        "Smoke ranking-row count mismatch."
    )


if len(smoke_build_metrics) != (
    EXPECTED_CONDITION_BUILD_METRIC_ROWS
):
    raise AssertionError(
        "Smoke build-metric row count mismatch."
    )


if len(smoke_project_run_metrics) != (
    EXPECTED_CONDITION_PROJECT_RUN_ROWS
):
    raise AssertionError(
        "Smoke project-run row count mismatch."
    )


if len(smoke_fit_times) != (
    EXPECTED_CONDITION_FIT_ROWS
):
    raise AssertionError(
        "Smoke fit-time row count mismatch."
    )


if set(
    smoke_rankings[
        "Technique"
    ].astype(str)
) != set(ALL_TECHNIQUES):
    raise AssertionError(
        "Smoke condition does not contain all seven "
        "techniques."
    )


# ---------------------------------------------------------
# 5. Validate noise and condition metadata
# ---------------------------------------------------------

if len(smoke_noise_summary) != 1:
    raise AssertionError(
        "Expected one smoke noise-summary row."
    )


smoke_noise_row = (
    smoke_noise_summary.iloc[0]
)


if float(
    smoke_noise_row[
        "NoisePercentRequested"
    ]
) != 0.0:
    raise AssertionError(
        "Smoke condition is not the 0% noise condition."
    )


if int(
    smoke_noise_row[
        "RepetitionSeed"
    ]
) != 1:
    raise AssertionError(
        "Smoke condition is not repetition seed 1."
    )


if int(
    smoke_noise_row[
        "NumberFlipped"
    ]
) != 0:
    raise AssertionError(
        "The 0% smoke condition flipped raw verdicts."
    )


if float(
    smoke_noise_row[
        "RealisedNoisePercent"
    ]
) != 0.0:
    raise AssertionError(
        "The realised smoke noise percentage is not zero."
    )


metadata_checks = {
    "NoisePercent":
        0,

    "RepetitionSeed":
        1,

    "TrainingExecutionRows":
        19845,

    "TrainingModelRows":
        13212,

    "EvaluationModelRows":
        4079,

    "EvaluatedBuilds":
        38,

    "TechniqueCount":
        7,

    "ModelFitRows":
        4,

    "RankingRows":
        28553,

    "BuildMetricRows":
        266,

    "ProjectRunRows":
        7,

    "NoiseFlips":
        0,
}


for metadata_key, expected_value in (
    metadata_checks.items()
):
    observed_value = (
        smoke_condition_metadata.get(
            metadata_key
        )
    )


    if float(observed_value) != float(
        expected_value
    ):
        raise AssertionError(
            "Smoke metadata mismatch for "
            f"{metadata_key}.\n"
            f"Expected: {expected_value}\n"
            f"Observed: {observed_value}"
        )


if not smoke_condition_metadata.get(
    "EvaluationHashUnchanged",
    False,
):
    raise AssertionError(
        "Smoke metadata says the evaluation data changed."
    )


if not smoke_condition_metadata.get(
    "CleanTrainingHistoryUnchanged",
    False,
):
    raise AssertionError(
        "Smoke metadata says the clean training history "
        "changed."
    )


if not smoke_condition_metadata.get(
    "CompletedSuccessfully",
    False,
):
    raise AssertionError(
        "Smoke success metadata is not marked complete."
    )


# ---------------------------------------------------------
# 6. Validate model fitting
# ---------------------------------------------------------

if set(
    smoke_fit_times[
        "Technique"
    ].astype(str)
) != set(ML_TECHNIQUES):
    raise AssertionError(
        "Smoke fit-time table does not contain the four "
        "ML models."
    )


for integer_column, expected_value in [
    (
        "TrainingRows",
        13212,
    ),
    (
        "TrainingFailures",
        130,
    ),
    (
        "TrainingPasses",
        13082,
    ),
    (
        "EvaluationRows",
        4079,
    ),
    (
        "ActiveFeatures",
        len(
            ACTIVE_FEATURE_COLUMNS
        ),
    ),
]:
    observed_values = set(
        pd.to_numeric(
            smoke_fit_times[
                integer_column
            ],
            errors="raise",
        ).astype(int)
    )


    if observed_values != {
        int(expected_value)
    }:
        raise AssertionError(
            f"Unexpected {integer_column} values:\n"
            f"{observed_values}"
        )


fit_seconds = pd.to_numeric(
    smoke_fit_times[
        "FitSeconds"
    ],
    errors="raise",
).to_numpy(dtype=float)


if not np.isfinite(
    fit_seconds
).all():
    raise AssertionError(
        "At least one ML fit time is non-finite."
    )


if (
    fit_seconds
    < 0
).any():
    raise AssertionError(
        "At least one ML fit time is negative."
    )


# ---------------------------------------------------------
# 7. Validate ranking coverage by technique
# ---------------------------------------------------------

ranking_coverage = (
    smoke_rankings
    .groupby(
        "Technique",
        as_index=False,
    )
    .agg(
        RankingRows=(
            "Test",
            "size",
        ),

        EvaluatedBuilds=(
            "Build",
            "nunique",
        ),

        FailureExecutions=(
            "ActualFailure",
            "sum",
        ),
    )
    .sort_values(
        "Technique",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if not (
    ranking_coverage[
        "RankingRows"
    ] == EXPECTED_RANKING_ROWS_PER_TECHNIQUE
).all():
    raise AssertionError(
        "At least one technique has the wrong number "
        "of ranking rows."
    )


if not (
    ranking_coverage[
        "EvaluatedBuilds"
    ] == EVALUATED_BUILD_COUNT
).all():
    raise AssertionError(
        "At least one technique did not rank all "
        "38 evaluation builds."
    )


if not (
    ranking_coverage[
        "FailureExecutions"
    ] == 40
).all():
    raise AssertionError(
        "At least one technique does not contain all "
        "40 clean evaluation failure executions."
    )


# ---------------------------------------------------------
# 8. Validate ML and Random score ranges
# ---------------------------------------------------------

probability_score_techniques = [
    *ML_TECHNIQUES,
    "Random",
]


probability_score_rows = (
    smoke_rankings[
        smoke_rankings[
            "Technique"
        ].isin(
            probability_score_techniques
        )
    ]
)


probability_scores = pd.to_numeric(
    probability_score_rows[
        "Score"
    ],
    errors="raise",
).to_numpy(dtype=float)


if not np.isfinite(
    probability_scores
).all():
    raise AssertionError(
        "ML or Random rankings contain non-finite scores."
    )


if (
    (
        probability_scores
        < 0.0
    ).any()
    or (
        probability_scores
        > 1.0
    ).any()
):
    raise AssertionError(
        "ML or Random scores are outside [0, 1]."
    )


# ---------------------------------------------------------
# 9. Verify baseline signatures against Step 8
# ---------------------------------------------------------

BASELINE_PREFLIGHT_PATH = (
    PROJECT_AGGREGATED_RESULTS
    / "jetty_runner_configuration"
    / "jetty_baseline_preflight_summary.csv"
)


if not BASELINE_PREFLIGHT_PATH.exists():
    raise FileNotFoundError(
        "Step 8 baseline preflight file is missing:\n"
        f"{BASELINE_PREFLIGHT_PATH}"
    )


baseline_preflight = pd.read_csv(
    BASELINE_PREFLIGHT_PATH
)


baseline_signature_records = []


for baseline in BASELINE_TECHNIQUES:
    baseline_rows = (
        smoke_rankings[
            smoke_rankings[
                "Technique"
            ] == baseline
        ]
        .copy()
    )


    observed_signature = ranking_signature(
        baseline_rows
    )


    expected_signature_rows = (
        baseline_preflight[
            baseline_preflight[
                "Technique"
            ] == baseline
        ]
    )


    if len(
        expected_signature_rows
    ) != 1:
        raise AssertionError(
            "Could not find one Step 8 signature for "
            f"{baseline}."
        )


    expected_signature = str(
        expected_signature_rows.iloc[0][
            "RankingSignature"
        ]
    )


    signature_matches = bool(
        observed_signature
        == expected_signature
    )


    baseline_signature_records.append({
        "Technique":
            baseline,

        "ExpectedSignature":
            expected_signature,

        "ObservedSignature":
            observed_signature,

        "SignatureMatches":
            signature_matches,
    })


baseline_signature_audit = pd.DataFrame(
    baseline_signature_records
)


if not baseline_signature_audit[
    "SignatureMatches"
].all():
    display(
        baseline_signature_audit
    )

    raise AssertionError(
        "At least one smoke baseline ranking differs "
        "from its Step 8 preflight ranking."
    )


# ---------------------------------------------------------
# 10. Verify overall ranking signature
# ---------------------------------------------------------

recomputed_ranking_signature = (
    ranking_signature(
        smoke_rankings
    )
)


stored_ranking_signature = str(
    smoke_condition_metadata[
        "RankingSignature"
    ]
)


if (
    recomputed_ranking_signature
    != stored_ranking_signature
):
    raise AssertionError(
        "The stored overall ranking signature does not "
        "match the smoke ranking content."
    )


# ---------------------------------------------------------
# 11. Independently recompute build metrics
# ---------------------------------------------------------

recomputed_build_metrics = (
    calculate_build_metrics(
        all_rankings=smoke_rankings,
        noise_percent=0,
        repetition_seed=1,
    )
)


stored_build_metrics_sorted = (
    smoke_build_metrics
    .sort_values(
        [
            "Technique",
            "Build",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


recomputed_build_metrics_sorted = (
    recomputed_build_metrics
    .sort_values(
        [
            "Technique",
            "Build",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if len(
    stored_build_metrics_sorted
) != len(
    recomputed_build_metrics_sorted
):
    raise AssertionError(
        "Stored and recomputed build-metric row counts "
        "differ."
    )


build_metric_exact_columns = [
    "Project",
    "Technique",
    "Build",
    "BuildOrder",
    "NumberOfTests",
    "NumberOfFailures",
]


build_metric_float_columns = [
    "NoisePercent",
    "RepetitionSeed",
    "TotalDuration",
    "APFD",
    "APFDc",
]


for column in build_metric_exact_columns:
    if not (
        stored_build_metrics_sorted[
            column
        ].astype(str).to_numpy()
        ==
        recomputed_build_metrics_sorted[
            column
        ].astype(str).to_numpy()
    ).all():
        raise AssertionError(
            "Stored and independently recomputed build "
            f"metrics differ in {column}."
        )


build_metric_numeric_mismatches = 0


for column in build_metric_float_columns:
    stored_values = pd.to_numeric(
        stored_build_metrics_sorted[
            column
        ],
        errors="raise",
    ).to_numpy(dtype=float)


    recomputed_values = pd.to_numeric(
        recomputed_build_metrics_sorted[
            column
        ],
        errors="raise",
    ).to_numpy(dtype=float)


    build_metric_numeric_mismatches += int(
        (
            ~np.isclose(
                stored_values,
                recomputed_values,
                rtol=1e-12,
                atol=1e-12,
                equal_nan=True,
            )
        ).sum()
    )


if build_metric_numeric_mismatches != 0:
    raise AssertionError(
        "Stored build metrics differ from the "
        "independent APFD/APFDc recomputation."
    )


# ---------------------------------------------------------
# 12. Independently recompute project-run metrics
# ---------------------------------------------------------

recomputed_project_run_metrics = (
    aggregate_project_run_metrics(
        recomputed_build_metrics
    )
)


stored_project_metrics_sorted = (
    smoke_project_run_metrics
    .sort_values(
        "Technique",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


recomputed_project_metrics_sorted = (
    recomputed_project_run_metrics
    .sort_values(
        "Technique",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


project_metric_exact_columns = [
    "Project",
    "Technique",
    "EvaluatedBuilds",
    "RankingRows",
    "FailureExecutions",
]


project_metric_float_columns = [
    "NoisePercent",
    "RepetitionSeed",
    "MeanAPFD",
    "MeanAPFDc",
    "SD_APFD",
    "SD_APFDc",
    "CI95Low_APFD",
    "CI95High_APFD",
    "CI95Low_APFDc",
    "CI95High_APFDc",
]


for column in project_metric_exact_columns:
    if not (
        stored_project_metrics_sorted[
            column
        ].astype(str).to_numpy()
        ==
        recomputed_project_metrics_sorted[
            column
        ].astype(str).to_numpy()
    ).all():
        raise AssertionError(
            "Stored and recomputed project-run metrics "
            f"differ in {column}."
        )


project_metric_numeric_mismatches = 0


for column in project_metric_float_columns:
    stored_values = pd.to_numeric(
        stored_project_metrics_sorted[
            column
        ],
        errors="raise",
    ).to_numpy(dtype=float)


    recomputed_values = pd.to_numeric(
        recomputed_project_metrics_sorted[
            column
        ],
        errors="raise",
    ).to_numpy(dtype=float)


    project_metric_numeric_mismatches += int(
        (
            ~np.isclose(
                stored_values,
                recomputed_values,
                rtol=1e-10,
                atol=1e-12,
                equal_nan=True,
            )
        ).sum()
    )


if project_metric_numeric_mismatches != 0:
    raise AssertionError(
        "Stored project-run metrics differ from the "
        "independent aggregation."
    )


# ---------------------------------------------------------
# 13. Validate final project-run summary
# ---------------------------------------------------------

if not (
    smoke_project_run_metrics[
        "EvaluatedBuilds"
    ].astype(int)
    == 38
).all():
    raise AssertionError(
        "A technique has the wrong evaluated-build count."
    )


if not (
    smoke_project_run_metrics[
        "RankingRows"
    ].astype(int)
    == 4079
).all():
    raise AssertionError(
        "A technique has the wrong ranking-row count."
    )


if not (
    smoke_project_run_metrics[
        "FailureExecutions"
    ].astype(int)
    == 40
).all():
    raise AssertionError(
        "A technique has the wrong evaluation failure "
        "count."
    )


for metric_column in [
    "MeanAPFD",
    "MeanAPFDc",
]:
    metric_values = pd.to_numeric(
        smoke_project_run_metrics[
            metric_column
        ],
        errors="raise",
    ).to_numpy(dtype=float)


    if not np.isfinite(
        metric_values
    ).all():
        raise AssertionError(
            f"{metric_column} contains non-finite values."
        )


    if (
        (
            metric_values
            < 0.0
        ).any()
        or (
            metric_values
            > 1.0
        ).any()
    ):
        raise AssertionError(
            f"{metric_column} is outside [0, 1]."
        )


# ---------------------------------------------------------
# 14. Save the smoke audit
# ---------------------------------------------------------

SMOKE_AUDIT_DIRECTORY = (
    PROJECT_AGGREGATED_RESULTS
    / "jetty_smoke_0pct_seed01"
)


SMOKE_AUDIT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


SMOKE_PROJECT_SUMMARY_PATH = (
    SMOKE_AUDIT_DIRECTORY
    / "jetty_smoke_project_run_metrics.csv"
)


SMOKE_FIT_SUMMARY_PATH = (
    SMOKE_AUDIT_DIRECTORY
    / "jetty_smoke_fit_times.csv"
)


SMOKE_BASELINE_SIGNATURE_PATH = (
    SMOKE_AUDIT_DIRECTORY
    / "jetty_smoke_baseline_signature_audit.csv"
)


SMOKE_REPORT_PATH = (
    SMOKE_AUDIT_DIRECTORY
    / "jetty_smoke_validation_report.json"
)


smoke_project_run_metrics.to_csv(
    SMOKE_PROJECT_SUMMARY_PATH,
    index=False,
)


smoke_fit_times.to_csv(
    SMOKE_FIT_SUMMARY_PATH,
    index=False,
)


baseline_signature_audit.to_csv(
    SMOKE_BASELINE_SIGNATURE_PATH,
    index=False,
)


smoke_validation_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "PASS",

    "NoisePercent":
        0,

    "RepetitionSeed":
        1,

    "LoadedFromExistingSuccessfulCondition":
        condition_loaded_from_existing_files,

    "RawNoiseFlips":
        int(
            smoke_noise_row[
                "NumberFlipped"
            ]
        ),

    "MLModelsFitted":
        int(
            len(
                smoke_fit_times
            )
        ),

    "Techniques":
        ALL_TECHNIQUES,

    "TechniqueCount":
        int(
            len(
                ALL_TECHNIQUES
            )
        ),

    "RankingRows":
        int(
            len(
                smoke_rankings
            )
        ),

    "BuildMetricRows":
        int(
            len(
                smoke_build_metrics
            )
        ),

    "ProjectRunRows":
        int(
            len(
                smoke_project_run_metrics
            )
        ),

    "EvaluatedBuildsPerTechnique":
        38,

    "RankingRowsPerTechnique":
        4079,

    "FailureExecutionsPerTechnique":
        40,

    "BaselineSignatureMismatches":
        int(
            (
                ~baseline_signature_audit[
                    "SignatureMatches"
                ]
            ).sum()
        ),

    "OverallRankingSignature":
        recomputed_ranking_signature,

    "OverallRankingSignatureMatchesMarker":
        bool(
            recomputed_ranking_signature
            == stored_ranking_signature
        ),

    "IndependentBuildMetricMismatches":
        build_metric_numeric_mismatches,

    "IndependentProjectMetricMismatches":
        project_metric_numeric_mismatches,

    "EvaluationHashUnchanged":
        bool(
            smoke_condition_metadata[
                "EvaluationHashUnchanged"
            ]
        ),

    "CleanTrainingHistoryUnchanged":
        bool(
            smoke_condition_metadata[
                "CleanTrainingHistoryUnchanged"
            ]
        ),

    "ReusableAsCanonicalFullRunCondition":
        True,

    "RawConditionDirectory":
        str(
            SMOKE_CONDITION_DIRECTORY
        ),

    "ProjectRunSummary":
        str(
            SMOKE_PROJECT_SUMMARY_PATH
        ),

    "FitSummary":
        str(
            SMOKE_FIT_SUMMARY_PATH
        ),

    "BaselineSignatureAudit":
        str(
            SMOKE_BASELINE_SIGNATURE_PATH
        ),

    "CompletedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
}


with open(
    SMOKE_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        smoke_validation_report,
        report_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 15. Update Project 6 checkpoint
# ---------------------------------------------------------

with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint = json.load(
        checkpoint_file
    )


checkpoint.update({
    "Status":
        "SMOKE_CONDITION_PASSED",

    "SmokeNoisePercent":
        0,

    "SmokeRepetitionSeed":
        1,

    "SmokeConditionDirectory":
        str(
            SMOKE_CONDITION_DIRECTORY
        ),

    "SmokeValidationReport":
        str(
            SMOKE_REPORT_PATH
        ),

    "SmokeMLFits":
        int(
            len(
                smoke_fit_times
            )
        ),

    "SmokeRankingRows":
        int(
            len(
                smoke_rankings
            )
        ),

    "SmokeBuildMetricRows":
        int(
            len(
                smoke_build_metrics
            )
        ),

    "SmokeProjectRunRows":
        int(
            len(
                smoke_project_run_metrics
            )
        ),

    "SmokeBaselineSignatureMismatches":
        int(
            (
                ~baseline_signature_audit[
                    "SignatureMatches"
                ]
            ).sum()
        ),

    "SmokeIndependentBuildMetricMismatches":
        build_metric_numeric_mismatches,

    "SmokeIndependentProjectMetricMismatches":
        project_metric_numeric_mismatches,

    "SmokeConditionReusableForFullRun":
        True,

    "UpdatedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
})


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 16. Compact final output
# ---------------------------------------------------------

print(
    "\n=== PROJECT 6 STEP 9 RESULT ==="
)


print(
    "\nCondition:"
)

print(
    "Noise:",
    "0%"
)

print(
    "Repetition seed:",
    1
)

print(
    "Loaded from an existing successful run:",
    condition_loaded_from_existing_files
)

print(
    "Raw verdicts flipped:",
    int(
        smoke_noise_row[
            "NumberFlipped"
        ]
    )
)


print(
    "\nModel fitting:"
)

display(
    smoke_fit_times[
        [
            "Technique",
            "FitSeconds",
            "TrainingRows",
            "TrainingFailures",
            "EvaluationRows",
            "ActiveFeatures",
        ]
    ]
    .sort_values(
        "Technique",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


print(
    "\nRanking coverage:"
)

display(
    ranking_coverage
)


print(
    "\nBaseline signature audit:"
)

display(
    baseline_signature_audit[
        [
            "Technique",
            "SignatureMatches",
        ]
    ]
)


print(
    "\nProject-run smoke results:"
)

display(
    smoke_project_run_metrics[
        [
            "Technique",
            "MeanAPFD",
            "MeanAPFDc",
            "SD_APFD",
            "SD_APFDc",
            "EvaluatedBuilds",
            "RankingRows",
            "FailureExecutions",
        ]
    ]
    .sort_values(
        "Technique",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


print(
    "\nOutput validation:"
)

print(
    "Ranking rows:",
    len(
        smoke_rankings
    )
)

print(
    "Build-metric rows:",
    len(
        smoke_build_metrics
    )
)

print(
    "Project-run rows:",
    len(
        smoke_project_run_metrics
    )
)

print(
    "ML fit rows:",
    len(
        smoke_fit_times
    )
)

print(
    "Baseline signature mismatches:",
    int(
        (
            ~baseline_signature_audit[
                "SignatureMatches"
            ]
        ).sum()
    )
)

print(
    "Independent build-metric mismatches:",
    build_metric_numeric_mismatches
)

print(
    "Independent project-metric mismatches:",
    project_metric_numeric_mismatches
)

print(
    "Evaluation data unchanged:",
    smoke_condition_metadata[
        "EvaluationHashUnchanged"
    ]
)


print(
    "\nSmoke validation report:"
)

print(
    SMOKE_REPORT_PATH
)


print(
    "\nValidation status:",
    "PASS"
)


print(
    "\nSUCCESS: All four ML models fitted successfully."
)

print(
    "SUCCESS: All seven techniques ranked all "
    "4,079 evaluation rows."
)

print(
    "SUCCESS: All 38 failing evaluation builds were "
    "evaluated for every technique."
)

print(
    "SUCCESS: Step 8 baseline signatures matched exactly."
)

print(
    "SUCCESS: APFD and APFDc were independently "
    "recomputed with zero mismatches."
)

print(
    "SUCCESS: Clean evaluation data remained unchanged."
)

print(
    "SUCCESS: The 0% seed-1 condition is retained as "
    "the canonical first condition of the full run."
)

print(
    "SUCCESS: Project 6 is ready for the complete "
    "270-condition execution."
)

=== PROJECT 6 STEP 9: 0% NOISE × SEED 1 SMOKE CONDITION ===

=== PROJECT 6 STEP 9 RESULT ===

Condition:
Noise: 0%
Repetition seed: 1
Loaded from an existing successful run: False
Raw verdicts flipped: 0

Model fitting:


,Technique,FitSeconds,TrainingRows,TrainingFailures,EvaluationRows,ActiveFeatures
0,LightGBM,6.037935,13212,130,4079,150
1,NaiveBayes,0.241064,13212,130,4079,150
2,RandomForest,2.983681,13212,130,4079,150
3,XGBoost,2.040159,13212,130,4079,150



Ranking coverage:


,Technique,RankingRows,EvaluatedBuilds,FailureExecutions
0,LatestFail,4079,38,40
1,LightGBM,4079,38,40
2,NaiveBayes,4079,38,40
3,QTF-Avg,4079,38,40
4,Random,4079,38,40
5,RandomForest,4079,38,40
6,XGBoost,4079,38,40



Baseline signature audit:


,Technique,SignatureMatches
0,Random,True
1,LatestFail,True
2,QTF-Avg,True



Project-run smoke results:


,Technique,MeanAPFD,MeanAPFDc,SD_APFD,SD_APFDc,EvaluatedBuilds,RankingRows,FailureExecutions
0,LatestFail,0.977121,0.958470,0.083757,0.103846,38,4079,40
1,LightGBM,0.974435,0.955000,0.089462,0.109307,38,4079,40
2,NaiveBayes,0.929622,0.846398,0.114218,0.108274,38,4079,40
3,QTF-Avg,0.110239,0.491108,0.095604,0.120921,38,4079,40
4,Random,0.494969,0.504615,0.271094,0.271592,38,4079,40
5,RandomForest,0.980551,0.961783,0.082495,0.101223,38,4079,40
6,XGBoost,0.987815,0.970231,0.028766,0.044766,38,4079,40



Output validation:
Ranking rows: 28553
Build-metric rows: 266
Project-run rows: 7
ML fit rows: 4
Baseline signature mismatches: 0
Independent build-metric mismatches: 0
Independent project-metric mismatches: 0
Evaluation data unchanged: True

Smoke validation report:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/eclipse__jetty.project/jetty_smoke_0pct_seed01/jetty_smoke_validation_report.json

Validation status: PASS

SUCCESS: All four ML models fitted successfully.
SUCCESS: All seven techniques ranked all 4,079 evaluation rows.
SUCCESS: All 38 failing evaluation builds were evaluated for every technique.
SUCCESS: Step 8 baseline signatures matched exactly.
SUCCESS: APFD and APFDc were independently recomputed with zero mismatches.
SUCCESS: Clean evaluation data remained unchanged.
SUCCESS: The 0% seed-1 condition is retained as the canonical first condition of the full run.
SUCCESS: Project 6 is ready for the complete 270-condition execution.


In [ ]:
# =========================================================
# PROJECT 6 — STEP 10
# COMPLETE 270-CONDITION EXECUTION
#
# Runs:
#   9 noise levels × 30 repetition seeds = 270 conditions
#   4 ML models per condition             = 1,080 fits
#   7 techniques per condition
#
# Resumability:
#   - valid successful conditions are skipped
#   - incomplete conditions without a success marker rerun
#   - progress is saved after every completed condition
#   - the smoke condition is reused
#
# Browser safety:
#   - only compact progress is displayed
#   - large condition outputs are deleted from memory
#   - cell output is periodically cleared
# =========================================================

from pathlib import Path
import gc
import json
import time
import traceback

import numpy as np
import pandas as pd
from IPython.display import clear_output, display


print(
    "=== PROJECT 6 STEP 10: "
    "COMPLETE 270-CONDITION EXECUTION ==="
)


# ---------------------------------------------------------
# 1. Confirm successful Step 9 state
# ---------------------------------------------------------

required_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",

    "PROJECT_RAW_RESULTS",
    "PROJECT_AGGREGATED_RESULTS",
    "PROJECT_6_SELECTION_CHECKPOINT",

    "NOISE_LEVELS",
    "REPETITION_SEEDS",
    "ML_TECHNIQUES",
    "BASELINE_TECHNIQUES",
    "ALL_TECHNIQUES",

    "EXPECTED_RANKING_ROWS_PER_TECHNIQUE",
    "EXPECTED_CONDITION_RANKING_ROWS",
    "EXPECTED_CONDITION_BUILD_METRIC_ROWS",
    "EXPECTED_CONDITION_PROJECT_RUN_ROWS",
    "EXPECTED_CONDITION_FIT_ROWS",

    "run_unified_condition",
]


missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required Step 9 objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nRerun the successful Step 6, Step 7, "
        "Step 8 and Step 9 cells in this runtime."
    )


if PROJECT_NUMBER != 6:
    raise AssertionError(
        f"Expected Project 6, observed {PROJECT_NUMBER}."
    )


if PROJECT_NAME != "eclipse@jetty.project":
    raise AssertionError(
        f"Unexpected project: {PROJECT_NAME}"
    )


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    previous_checkpoint = json.load(
        checkpoint_file
    )


allowed_previous_statuses = {
    "SMOKE_CONDITION_PASSED",
    "FULL_RUN_IN_PROGRESS",
    "FULL_RUN_INTERRUPTED",
    "FULL_RUN_COMPLETED",
}


if previous_checkpoint.get(
    "Status"
) not in allowed_previous_statuses:
    raise AssertionError(
        "Step 9 has not been marked as passed.\n"
        f"Observed checkpoint status: "
        f"{previous_checkpoint.get('Status')}"
    )


if list(NOISE_LEVELS) != [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]:
    raise AssertionError(
        "Unexpected noise-level configuration."
    )


if list(REPETITION_SEEDS) != list(
    range(1, 31)
):
    raise AssertionError(
        "Unexpected repetition-seed configuration."
    )


if len(ALL_TECHNIQUES) != 7:
    raise AssertionError(
        "Expected seven prioritisation techniques."
    )


# ---------------------------------------------------------
# 2. Expected full-run totals
# ---------------------------------------------------------

EXPECTED_CONDITIONS = (
    len(NOISE_LEVELS)
    * len(REPETITION_SEEDS)
)


EXPECTED_TOTAL_MODEL_FITS = (
    EXPECTED_CONDITIONS
    * EXPECTED_CONDITION_FIT_ROWS
)


EXPECTED_TOTAL_RANKING_ROWS = (
    EXPECTED_CONDITIONS
    * EXPECTED_CONDITION_RANKING_ROWS
)


EXPECTED_TOTAL_BUILD_METRIC_ROWS = (
    EXPECTED_CONDITIONS
    * EXPECTED_CONDITION_BUILD_METRIC_ROWS
)


EXPECTED_TOTAL_PROJECT_RUN_ROWS = (
    EXPECTED_CONDITIONS
    * EXPECTED_CONDITION_PROJECT_RUN_ROWS
)


EXPECTED_TOTAL_NOISE_MANIFEST_ROWS = (
    EXPECTED_CONDITIONS
    * 19845
)


if EXPECTED_CONDITIONS != 270:
    raise AssertionError(
        "Expected exactly 270 conditions."
    )


if EXPECTED_TOTAL_MODEL_FITS != 1080:
    raise AssertionError(
        "Expected exactly 1,080 model fits."
    )


if EXPECTED_TOTAL_RANKING_ROWS != 7709310:
    raise AssertionError(
        "Unexpected expected ranking-row total."
    )


if EXPECTED_TOTAL_BUILD_METRIC_ROWS != 71820:
    raise AssertionError(
        "Unexpected expected build-metric total."
    )


if EXPECTED_TOTAL_PROJECT_RUN_ROWS != 1890:
    raise AssertionError(
        "Unexpected expected project-run total."
    )


# ---------------------------------------------------------
# 3. Permanent full-run paths
# ---------------------------------------------------------

FULL_RUN_DIRECTORY = (
    PROJECT_AGGREGATED_RESULTS
    / "jetty_30_seed_full_run"
)


FULL_RUN_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


FULL_RUN_PROGRESS_PATH = (
    FULL_RUN_DIRECTORY
    / "jetty_full_run_condition_progress.csv"
)


FULL_RUN_REPORT_PATH = (
    FULL_RUN_DIRECTORY
    / "jetty_full_run_execution_report.json"
)


FULL_RUN_ERROR_PATH = (
    FULL_RUN_DIRECTORY
    / "jetty_full_run_last_error.json"
)


FULL_RUN_INVENTORY_PATH = (
    FULL_RUN_DIRECTORY
    / "jetty_full_run_condition_inventory.csv"
)


FULL_RUN_PROJECT_METRICS_PATH = (
    FULL_RUN_DIRECTORY
    / "jetty_all_project_run_metrics.csv"
)


FULL_RUN_FIT_TIMES_PATH = (
    FULL_RUN_DIRECTORY
    / "jetty_all_fit_times.csv"
)


FULL_RUN_NOISE_SUMMARY_PATH = (
    FULL_RUN_DIRECTORY
    / "jetty_all_noise_summaries.csv"
)


# ---------------------------------------------------------
# 4. Condition-path helpers
# ---------------------------------------------------------

def get_condition_directory(
    noise_percent,
    repetition_seed,
):
    return (
        PROJECT_RAW_RESULTS
        / f"noise_{int(noise_percent):03d}"
        / f"seed_{int(repetition_seed):02d}"
    )


def get_condition_paths(
    noise_percent,
    repetition_seed,
):
    condition_directory = (
        get_condition_directory(
            noise_percent,
            repetition_seed,
        )
    )


    return {
        "directory":
            condition_directory,

        "marker":
            condition_directory
            / "_SUCCESS.json",

        "noise_manifest":
            condition_directory
            / "noise_manifest.parquet",

        "noise_summary":
            condition_directory
            / "noise_summary.csv",

        "fit_times":
            condition_directory
            / "fit_times.csv",

        "build_metrics":
            condition_directory
            / "build_metrics.parquet",

        "project_run_metrics":
            condition_directory
            / "project_run_metrics.csv",

        "training_medians":
            condition_directory
            / "training_medians.csv",

        "rankings":
            condition_directory
            / "rankings.parquet",
    }


REQUIRED_CONDITION_FILE_KEYS = [
    "noise_manifest",
    "noise_summary",
    "fit_times",
    "build_metrics",
    "project_run_metrics",
    "training_medians",
    "rankings",
]


# ---------------------------------------------------------
# 5. Inspect one successful condition
# ---------------------------------------------------------

def inspect_successful_condition(
    noise_percent,
    repetition_seed,
):
    paths = get_condition_paths(
        noise_percent,
        repetition_seed,
    )


    if not paths[
        "marker"
    ].exists():
        return None


    missing_files = [
        str(
            paths[file_key]
        )
        for file_key in (
            REQUIRED_CONDITION_FILE_KEYS
        )
        if not paths[
            file_key
        ].exists()
    ]


    if missing_files:
        raise FileNotFoundError(
            "A condition has a success marker but is "
            "missing required files.\n"
            f"Noise: {noise_percent}\n"
            f"Seed: {repetition_seed}\n"
            + "\n".join(missing_files)
        )


    with open(
        paths["marker"],
        "r",
        encoding="utf-8",
    ) as marker_file:
        metadata = json.load(
            marker_file
        )


    expected_metadata = {
        "NoisePercent":
            int(
                noise_percent
            ),

        "RepetitionSeed":
            int(
                repetition_seed
            ),

        "TrainingExecutionRows":
            19845,

        "TrainingModelRows":
            13212,

        "EvaluationModelRows":
            4079,

        "EvaluatedBuilds":
            38,

        "TechniqueCount":
            7,

        "ModelFitRows":
            4,

        "RankingRows":
            28553,

        "BuildMetricRows":
            266,

        "ProjectRunRows":
            7,
    }


    for metadata_key, expected_value in (
        expected_metadata.items()
    ):
        if metadata_key not in metadata:
            raise AssertionError(
                "Successful condition marker is missing "
                f"{metadata_key}.\n"
                f"Noise: {noise_percent}\n"
                f"Seed: {repetition_seed}"
            )


        observed_value = metadata[
            metadata_key
        ]


        if float(
            observed_value
        ) != float(
            expected_value
        ):
            raise AssertionError(
                "Successful condition metadata mismatch.\n"
                f"Noise: {noise_percent}\n"
                f"Seed: {repetition_seed}\n"
                f"Field: {metadata_key}\n"
                f"Expected: {expected_value}\n"
                f"Observed: {observed_value}"
            )


    required_boolean_fields = {
        "CompletedSuccessfully":
            True,

        "EvaluationHashUnchanged":
            True,

        "CleanTrainingHistoryUnchanged":
            True,
    }


    for metadata_key, expected_value in (
        required_boolean_fields.items()
    ):
        if bool(
            metadata.get(
                metadata_key,
                False,
            )
        ) != expected_value:
            raise AssertionError(
                "Successful condition boolean validation "
                "failed.\n"
                f"Noise: {noise_percent}\n"
                f"Seed: {repetition_seed}\n"
                f"Field: {metadata_key}"
            )


    return {
        "NoisePercent":
            int(
                noise_percent
            ),

        "RepetitionSeed":
            int(
                repetition_seed
            ),

        "Status":
            "SUCCESS",

        "NoiseFlips":
            int(
                metadata.get(
                    "NoiseFlips",
                    0,
                )
            ),

        "RealisedNoisePercent":
            float(
                metadata.get(
                    "RealisedNoisePercent",
                    0.0,
                )
            ),

        "ModelFitRows":
            int(
                metadata[
                    "ModelFitRows"
                ]
            ),

        "RankingRows":
            int(
                metadata[
                    "RankingRows"
                ]
            ),

        "BuildMetricRows":
            int(
                metadata[
                    "BuildMetricRows"
                ]
            ),

        "ProjectRunRows":
            int(
                metadata[
                    "ProjectRunRows"
                ]
            ),

        "ElapsedSeconds":
            float(
                metadata.get(
                    "ElapsedSeconds",
                    np.nan,
                )
            ),

        "RankingSignature":
            str(
                metadata.get(
                    "RankingSignature",
                    "",
                )
            ),

        "ConditionDirectory":
            str(
                paths["directory"]
            ),

        "SuccessMarker":
            str(
                paths["marker"]
            ),

        "CompletedAtUTC":
            str(
                metadata.get(
                    "CompletedAtUTC",
                    "",
                )
            ),
    }


# ---------------------------------------------------------
# 6. All canonical condition pairs
# ---------------------------------------------------------

CONDITION_PAIRS = [
    (
        int(noise_percent),
        int(repetition_seed),
    )
    for noise_percent in NOISE_LEVELS
    for repetition_seed in REPETITION_SEEDS
]


if len(CONDITION_PAIRS) != 270:
    raise AssertionError(
        "Condition-pair construction failed."
    )


if len(
    set(CONDITION_PAIRS)
) != 270:
    raise AssertionError(
        "Condition-pair list contains duplicates."
    )


# ---------------------------------------------------------
# 7. Scan existing successful conditions
# ---------------------------------------------------------

existing_success_records = []
pending_condition_pairs = []


for (
    noise_percent,
    repetition_seed,
) in CONDITION_PAIRS:
    condition_record = (
        inspect_successful_condition(
            noise_percent,
            repetition_seed,
        )
    )


    if condition_record is None:
        pending_condition_pairs.append(
            (
                noise_percent,
                repetition_seed,
            )
        )

    else:
        existing_success_records.append(
            condition_record
        )


PREEXISTING_SUCCESSFUL_PAIRS = {
    (
        int(
            record[
                "NoisePercent"
            ]
        ),
        int(
            record[
                "RepetitionSeed"
            ]
        ),
    )
    for record in existing_success_records
}


if (
    0,
    1,
) not in PREEXISTING_SUCCESSFUL_PAIRS:
    raise AssertionError(
        "The successful Step 9 smoke condition "
        "0% × seed 1 was not found."
    )


# ---------------------------------------------------------
# 8. Progress and checkpoint helpers
# ---------------------------------------------------------

full_run_start_time = time.time()


def save_progress_state(
    success_records,
    run_status,
    current_condition=None,
    error_message=None,
):
    progress_table = pd.DataFrame(
        success_records
    )


    if len(
        progress_table
    ) > 0:
        progress_table = (
            progress_table
            .sort_values(
                [
                    "NoisePercent",
                    "RepetitionSeed",
                ],
                kind="mergesort",
            )
            .reset_index(drop=True)
        )


    progress_table.to_csv(
        FULL_RUN_PROGRESS_PATH,
        index=False,
    )


    completed_pairs = {
        (
            int(
                record[
                    "NoisePercent"
                ]
            ),
            int(
                record[
                    "RepetitionSeed"
                ]
            ),
        )
        for record in success_records
    }


    report = {
        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "Status":
            str(
                run_status
            ),

        "ExpectedConditions":
            EXPECTED_CONDITIONS,

        "CompletedConditions":
            int(
                len(
                    completed_pairs
                )
            ),

        "RemainingConditions":
            int(
                EXPECTED_CONDITIONS
                - len(
                    completed_pairs
                )
            ),

        "PreexistingSuccessfulConditions":
            int(
                len(
                    PREEXISTING_SUCCESSFUL_PAIRS
                )
            ),

        "ExpectedModelFits":
            EXPECTED_TOTAL_MODEL_FITS,

        "CompletedModelFits":
            int(
                sum(
                    record[
                        "ModelFitRows"
                    ]
                    for record in success_records
                )
            ),

        "ExpectedRankingRows":
            EXPECTED_TOTAL_RANKING_ROWS,

        "CompletedRankingRows":
            int(
                sum(
                    record[
                        "RankingRows"
                    ]
                    for record in success_records
                )
            ),

        "ExpectedBuildMetricRows":
            EXPECTED_TOTAL_BUILD_METRIC_ROWS,

        "CompletedBuildMetricRows":
            int(
                sum(
                    record[
                        "BuildMetricRows"
                    ]
                    for record in success_records
                )
            ),

        "ExpectedProjectRunRows":
            EXPECTED_TOTAL_PROJECT_RUN_ROWS,

        "CompletedProjectRunRows":
            int(
                sum(
                    record[
                        "ProjectRunRows"
                    ]
                    for record in success_records
                )
            ),

        "CurrentCondition":
            (
                {
                    "NoisePercent":
                        int(
                            current_condition[0]
                        ),

                    "RepetitionSeed":
                        int(
                            current_condition[1]
                        ),
                }
                if current_condition is not None
                else None
            ),

        "ErrorMessage":
            error_message,

        "ElapsedSecondsThisInvocation":
            float(
                time.time()
                - full_run_start_time
            ),

        "ProgressTable":
            str(
                FULL_RUN_PROGRESS_PATH
            ),

        "UpdatedAtUTC":
            pd.Timestamp.utcnow().isoformat(),
    }


    with open(
        FULL_RUN_REPORT_PATH,
        "w",
        encoding="utf-8",
    ) as report_file:
        json.dump(
            report,
            report_file,
            indent=2,
            default=str,
        )


    with open(
        PROJECT_6_SELECTION_CHECKPOINT,
        "r",
        encoding="utf-8",
    ) as checkpoint_file:
        checkpoint = json.load(
            checkpoint_file
        )


    checkpoint.update({
        "Status":
            str(
                run_status
            ),

        "FullRunDirectory":
            str(
                FULL_RUN_DIRECTORY
            ),

        "FullRunExecutionReport":
            str(
                FULL_RUN_REPORT_PATH
            ),

        "FullRunProgressTable":
            str(
                FULL_RUN_PROGRESS_PATH
            ),

        "FullRunExpectedConditions":
            EXPECTED_CONDITIONS,

        "FullRunCompletedConditions":
            int(
                len(
                    completed_pairs
                )
            ),

        "FullRunRemainingConditions":
            int(
                EXPECTED_CONDITIONS
                - len(
                    completed_pairs
                )
            ),

        "FullRunExpectedModelFits":
            EXPECTED_TOTAL_MODEL_FITS,

        "FullRunCompletedModelFits":
            int(
                sum(
                    record[
                        "ModelFitRows"
                    ]
                    for record in success_records
                )
            ),

        "FullRunCurrentCondition":
            (
                {
                    "NoisePercent":
                        int(
                            current_condition[0]
                        ),

                    "RepetitionSeed":
                        int(
                            current_condition[1]
                        ),
                }
                if current_condition is not None
                else None
            ),

        "UpdatedAtUTC":
            pd.Timestamp.utcnow().isoformat(),
    })


    with open(
        PROJECT_6_SELECTION_CHECKPOINT,
        "w",
        encoding="utf-8",
    ) as checkpoint_file:
        json.dump(
            checkpoint,
            checkpoint_file,
            indent=2,
            default=str,
        )


# Mark the run as in progress before starting.
all_success_records = list(
    existing_success_records
)


save_progress_state(
    success_records=all_success_records,
    run_status="FULL_RUN_IN_PROGRESS",
)


# ---------------------------------------------------------
# 9. Compact progress display
# ---------------------------------------------------------

def display_compact_progress(
    completed_count,
    total_count,
    current_noise=None,
    current_seed=None,
    latest_elapsed=None,
):
    clear_output(
        wait=True
    )


    print(
        "=== PROJECT 6 STEP 10: "
        "COMPLETE 270-CONDITION EXECUTION ==="
    )


    print(
        "\nProgress:",
        f"{completed_count}/{total_count}",
        "conditions complete"
    )


    print(
        "Preexisting successful conditions:",
        len(
            PREEXISTING_SUCCESSFUL_PAIRS
        )
    )


    print(
        "Executed during this invocation:",
        completed_count
        - len(
            PREEXISTING_SUCCESSFUL_PAIRS
        )
    )


    print(
        "Remaining:",
        total_count
        - completed_count
    )


    if current_noise is not None:
        print(
            "\nLatest completed condition:",
            f"noise={int(current_noise)}%, "
            f"seed={int(current_seed)}"
        )


    if latest_elapsed is not None:
        print(
            "Latest condition elapsed:",
            f"{float(latest_elapsed):.2f} seconds"
        )


    invocation_elapsed = (
        time.time()
        - full_run_start_time
    )


    print(
        "Current invocation elapsed:",
        f"{invocation_elapsed / 60.0:.1f} minutes"
    )


    print(
        "\nProgress saved to:"
    )

    print(
        FULL_RUN_PROGRESS_PATH
    )


    print(
        "\nThe cell is resumable. Completed conditions "
        "will not be rerun."
    )


display_compact_progress(
    completed_count=len(
        all_success_records
    ),
    total_count=EXPECTED_CONDITIONS,
)


# ---------------------------------------------------------
# 10. Run every missing condition
# ---------------------------------------------------------

executed_during_this_invocation = []


try:
    for condition_index, (
        noise_percent,
        repetition_seed,
    ) in enumerate(
        pending_condition_pairs,
        start=1,
    ):
        condition_start_time = time.time()


        condition_result = (
            run_unified_condition(
                noise_percent=(
                    noise_percent
                ),
                repetition_seed=(
                    repetition_seed
                ),
                save_rankings=True,
                overwrite=False,
            )
        )


        # Validate the newly written marker and files.
        completed_record = (
            inspect_successful_condition(
                noise_percent,
                repetition_seed,
            )
        )


        if completed_record is None:
            raise AssertionError(
                "The condition runner returned without "
                "writing a success marker.\n"
                f"Noise: {noise_percent}\n"
                f"Seed: {repetition_seed}"
            )


        all_success_records.append(
            completed_record
        )


        executed_during_this_invocation.append(
            (
                int(
                    noise_percent
                ),
                int(
                    repetition_seed
                ),
            )
        )


        save_progress_state(
            success_records=(
                all_success_records
            ),
            run_status=(
                "FULL_RUN_IN_PROGRESS"
            ),
            current_condition=(
                noise_percent,
                repetition_seed,
            ),
        )


        latest_elapsed = (
            time.time()
            - condition_start_time
        )


        # Release the large in-memory rankings and metrics.
        del condition_result
        gc.collect()


        display_compact_progress(
            completed_count=len(
                all_success_records
            ),
            total_count=(
                EXPECTED_CONDITIONS
            ),
            current_noise=(
                noise_percent
            ),
            current_seed=(
                repetition_seed
            ),
            latest_elapsed=(
                latest_elapsed
            ),
        )


except Exception as condition_error:
    error_traceback = traceback.format_exc()


    error_record = {
        "Project":
            PROJECT_NAME,

        "Status":
            "FULL_RUN_INTERRUPTED",

        "NoisePercent":
            (
                int(
                    noise_percent
                )
                if "noise_percent" in locals()
                else None
            ),

        "RepetitionSeed":
            (
                int(
                    repetition_seed
                )
                if "repetition_seed" in locals()
                else None
            ),

        "ErrorType":
            type(
                condition_error
            ).__name__,

        "ErrorMessage":
            str(
                condition_error
            ),

        "Traceback":
            error_traceback,

        "CompletedConditions":
            len(
                all_success_records
            ),

        "CompletedAtUTC":
            pd.Timestamp.utcnow().isoformat(),
    }


    with open(
        FULL_RUN_ERROR_PATH,
        "w",
        encoding="utf-8",
    ) as error_file:
        json.dump(
            error_record,
            error_file,
            indent=2,
            default=str,
        )


    save_progress_state(
        success_records=(
            all_success_records
        ),
        run_status=(
            "FULL_RUN_INTERRUPTED"
        ),
        current_condition=(
            (
                noise_percent,
                repetition_seed,
            )
            if (
                "noise_percent" in locals()
                and "repetition_seed" in locals()
            )
            else None
        ),
        error_message=str(
            condition_error
        ),
    )


    print(
        "\nThe full run stopped at:"
    )

    print(
        "Noise:",
        (
            noise_percent
            if "noise_percent" in locals()
            else "unknown"
        )
    )

    print(
        "Seed:",
        (
            repetition_seed
            if "repetition_seed" in locals()
            else "unknown"
        )
    )

    print(
        "\nCompleted conditions remain saved."
    )

    print(
        "Error report:"
    )

    print(
        FULL_RUN_ERROR_PATH
    )


    raise


# ---------------------------------------------------------
# 11. Rescan and validate all 270 conditions
# ---------------------------------------------------------

final_condition_records = []


for (
    noise_percent,
    repetition_seed,
) in CONDITION_PAIRS:
    condition_record = (
        inspect_successful_condition(
            noise_percent,
            repetition_seed,
        )
    )


    if condition_record is None:
        raise AssertionError(
            "The full run ended with a missing condition.\n"
            f"Noise: {noise_percent}\n"
            f"Seed: {repetition_seed}"
        )


    condition_pair = (
        int(
            noise_percent
        ),
        int(
            repetition_seed
        ),
    )


    condition_record[
        "PresentBeforeStep10"
    ] = bool(
        condition_pair
        in PREEXISTING_SUCCESSFUL_PAIRS
    )


    condition_record[
        "ExecutedDuringThisInvocation"
    ] = bool(
        condition_pair
        in set(
            executed_during_this_invocation
        )
    )


    final_condition_records.append(
        condition_record
    )


condition_inventory = (
    pd.DataFrame(
        final_condition_records
    )
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if len(condition_inventory) != 270:
    raise AssertionError(
        "Final inventory does not contain 270 conditions."
    )


if condition_inventory.duplicated(
    subset=[
        "NoisePercent",
        "RepetitionSeed",
    ]
).any():
    raise AssertionError(
        "Final inventory contains duplicate conditions."
    )


expected_pair_set = set(
    CONDITION_PAIRS
)


observed_pair_set = set(
    zip(
        condition_inventory[
            "NoisePercent"
        ].astype(int),

        condition_inventory[
            "RepetitionSeed"
        ].astype(int),
    )
)


if observed_pair_set != expected_pair_set:
    raise AssertionError(
        "Final condition inventory does not match the "
        "frozen 270-condition design."
    )


# ---------------------------------------------------------
# 12. Validate total metadata counts
# ---------------------------------------------------------

total_model_fit_rows = int(
    condition_inventory[
        "ModelFitRows"
    ].sum()
)


total_ranking_rows = int(
    condition_inventory[
        "RankingRows"
    ].sum()
)


total_build_metric_rows = int(
    condition_inventory[
        "BuildMetricRows"
    ].sum()
)


total_project_run_rows = int(
    condition_inventory[
        "ProjectRunRows"
    ].sum()
)


if total_model_fit_rows != (
    EXPECTED_TOTAL_MODEL_FITS
):
    raise AssertionError(
        "Total model-fit count mismatch.\n"
        f"Expected: {EXPECTED_TOTAL_MODEL_FITS}\n"
        f"Observed: {total_model_fit_rows}"
    )


if total_ranking_rows != (
    EXPECTED_TOTAL_RANKING_ROWS
):
    raise AssertionError(
        "Total ranking-row count mismatch.\n"
        f"Expected: {EXPECTED_TOTAL_RANKING_ROWS}\n"
        f"Observed: {total_ranking_rows}"
    )


if total_build_metric_rows != (
    EXPECTED_TOTAL_BUILD_METRIC_ROWS
):
    raise AssertionError(
        "Total build-metric count mismatch.\n"
        f"Expected: "
        f"{EXPECTED_TOTAL_BUILD_METRIC_ROWS}\n"
        f"Observed: {total_build_metric_rows}"
    )


if total_project_run_rows != (
    EXPECTED_TOTAL_PROJECT_RUN_ROWS
):
    raise AssertionError(
        "Total project-run count mismatch.\n"
        f"Expected: "
        f"{EXPECTED_TOTAL_PROJECT_RUN_ROWS}\n"
        f"Observed: {total_project_run_rows}"
    )


# ---------------------------------------------------------
# 13. Consolidate compact result files
# ---------------------------------------------------------

project_run_frames = []
fit_time_frames = []
noise_summary_frames = []


for (
    noise_percent,
    repetition_seed,
) in CONDITION_PAIRS:
    paths = get_condition_paths(
        noise_percent,
        repetition_seed,
    )


    condition_project_metrics = (
        pd.read_csv(
            paths[
                "project_run_metrics"
            ]
        )
    )


    condition_fit_times = (
        pd.read_csv(
            paths[
                "fit_times"
            ]
        )
    )


    condition_noise_summary = (
        pd.read_csv(
            paths[
                "noise_summary"
            ]
        )
    )


    if len(
        condition_project_metrics
    ) != 7:
        raise AssertionError(
            "Condition project-run summary does not "
            "contain seven rows.\n"
            f"Noise: {noise_percent}\n"
            f"Seed: {repetition_seed}"
        )


    if len(
        condition_fit_times
    ) != 4:
        raise AssertionError(
            "Condition fit-time summary does not "
            "contain four rows.\n"
            f"Noise: {noise_percent}\n"
            f"Seed: {repetition_seed}"
        )


    if len(
        condition_noise_summary
    ) != 1:
        raise AssertionError(
            "Condition noise summary does not contain "
            "one row.\n"
            f"Noise: {noise_percent}\n"
            f"Seed: {repetition_seed}"
        )


    project_run_frames.append(
        condition_project_metrics
    )


    fit_time_frames.append(
        condition_fit_times
    )


    noise_summary_frames.append(
        condition_noise_summary
    )


all_project_run_metrics = pd.concat(
    project_run_frames,
    ignore_index=True,
)


all_fit_times = pd.concat(
    fit_time_frames,
    ignore_index=True,
)


all_noise_summaries = pd.concat(
    noise_summary_frames,
    ignore_index=True,
)


if len(
    all_project_run_metrics
) != EXPECTED_TOTAL_PROJECT_RUN_ROWS:
    raise AssertionError(
        "Consolidated project-run metrics should contain "
        "1,890 rows."
    )


if len(
    all_fit_times
) != EXPECTED_TOTAL_MODEL_FITS:
    raise AssertionError(
        "Consolidated fit-time table should contain "
        "1,080 rows."
    )


if len(
    all_noise_summaries
) != EXPECTED_CONDITIONS:
    raise AssertionError(
        "Consolidated noise summary should contain "
        "270 rows."
    )


# ---------------------------------------------------------
# 14. Validate consolidated condition coverage
# ---------------------------------------------------------

project_condition_sizes = (
    all_project_run_metrics
    .groupby(
        [
            "NoisePercent",
            "RepetitionSeed",
        ],
    )
    .size()
)


if not (
    project_condition_sizes
    == 7
).all():
    raise AssertionError(
        "At least one condition does not contain seven "
        "project-run technique rows."
    )


fit_condition_sizes = (
    all_fit_times
    .groupby(
        [
            "NoisePercent",
            "RepetitionSeed",
        ],
    )
    .size()
)


if not (
    fit_condition_sizes
    == 4
).all():
    raise AssertionError(
        "At least one condition does not contain four "
        "fit-time rows."
    )


if set(
    all_project_run_metrics[
        "Technique"
    ].astype(str)
) != set(ALL_TECHNIQUES):
    raise AssertionError(
        "Consolidated project-run metrics do not contain "
        "all seven techniques."
    )


if set(
    all_fit_times[
        "Technique"
    ].astype(str)
) != set(ML_TECHNIQUES):
    raise AssertionError(
        "Consolidated fit-time metrics do not contain "
        "all four ML techniques."
    )


noise_condition_counts = (
    all_noise_summaries
    .groupby(
        "NoisePercentRequested"
    )
    .size()
    .sort_index()
)


if not (
    noise_condition_counts
    == 30
).all():
    raise AssertionError(
        "Every noise level must contain exactly "
        "30 repetition seeds."
    )


if len(
    noise_condition_counts
) != 9:
    raise AssertionError(
        "Consolidated summaries do not contain all "
        "nine noise levels."
    )


# ---------------------------------------------------------
# 15. Save consolidated full-run files
# ---------------------------------------------------------

condition_inventory.to_csv(
    FULL_RUN_INVENTORY_PATH,
    index=False,
)


all_project_run_metrics.to_csv(
    FULL_RUN_PROJECT_METRICS_PATH,
    index=False,
)


all_fit_times.to_csv(
    FULL_RUN_FIT_TIMES_PATH,
    index=False,
)


all_noise_summaries.to_csv(
    FULL_RUN_NOISE_SUMMARY_PATH,
    index=False,
)


# ---------------------------------------------------------
# 16. Full-run summaries
# ---------------------------------------------------------

completion_by_noise = (
    condition_inventory
    .groupby(
        "NoisePercent",
        as_index=False,
    )
    .agg(
        Conditions=(
            "RepetitionSeed",
            "size",
        ),

        TotalNoiseFlips=(
            "NoiseFlips",
            "sum",
        ),

        MeanRealisedNoisePercent=(
            "RealisedNoisePercent",
            "mean",
        ),

        TotalElapsedSeconds=(
            "ElapsedSeconds",
            "sum",
        ),
    )
)


fit_time_summary = (
    all_fit_times
    .groupby(
        "Technique",
        as_index=False,
    )
    .agg(
        ModelFits=(
            "FitSeconds",
            "size",
        ),

        MeanFitSeconds=(
            "FitSeconds",
            "mean",
        ),

        MedianFitSeconds=(
            "FitSeconds",
            "median",
        ),

        TotalFitSeconds=(
            "FitSeconds",
            "sum",
        ),
    )
    .sort_values(
        "Technique",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 17. Save final execution report
# ---------------------------------------------------------

total_elapsed_this_invocation = float(
    time.time()
    - full_run_start_time
)


full_run_execution_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "PASS",

    "NoiseLevels":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "ExpectedConditions":
        EXPECTED_CONDITIONS,

    "SuccessfulConditions":
        int(
            len(
                condition_inventory
            )
        ),

    "PreexistingSuccessfulConditions":
        int(
            len(
                PREEXISTING_SUCCESSFUL_PAIRS
            )
        ),

    "ExecutedDuringThisInvocation":
        int(
            len(
                executed_during_this_invocation
            )
        ),

    "ExpectedModelFits":
        EXPECTED_TOTAL_MODEL_FITS,

    "ObservedModelFits":
        total_model_fit_rows,

    "ExpectedRankingRows":
        EXPECTED_TOTAL_RANKING_ROWS,

    "ObservedRankingRows":
        total_ranking_rows,

    "ExpectedBuildMetricRows":
        EXPECTED_TOTAL_BUILD_METRIC_ROWS,

    "ObservedBuildMetricRows":
        total_build_metric_rows,

    "ExpectedProjectRunRows":
        EXPECTED_TOTAL_PROJECT_RUN_ROWS,

    "ObservedProjectRunRows":
        total_project_run_rows,

    "ExpectedNoiseManifestRows":
        EXPECTED_TOTAL_NOISE_MANIFEST_ROWS,

    "AllConditionsHaveSuccessMarkers":
        True,

    "AllConditionsHaveRequiredFiles":
        True,

    "ConditionsPerNoiseLevel":
        30,

    "TechniquesPerCondition":
        7,

    "MLFitsPerCondition":
        4,

    "ElapsedSecondsThisInvocation":
        total_elapsed_this_invocation,

    "ConditionInventory":
        str(
            FULL_RUN_INVENTORY_PATH
        ),

    "ProjectRunMetrics":
        str(
            FULL_RUN_PROJECT_METRICS_PATH
        ),

    "FitTimes":
        str(
            FULL_RUN_FIT_TIMES_PATH
        ),

    "NoiseSummaries":
        str(
            FULL_RUN_NOISE_SUMMARY_PATH
        ),

    "CompletedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
}


with open(
    FULL_RUN_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        full_run_execution_report,
        report_file,
        indent=2,
        default=str,
    )


# Remove an old interruption record after full success.
if FULL_RUN_ERROR_PATH.exists():
    FULL_RUN_ERROR_PATH.unlink()


# ---------------------------------------------------------
# 18. Update Project 6 checkpoint
# ---------------------------------------------------------

with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint = json.load(
        checkpoint_file
    )


checkpoint.update({
    "Status":
        "FULL_RUN_COMPLETED",

    "FullRunDirectory":
        str(
            FULL_RUN_DIRECTORY
        ),

    "FullRunExecutionReport":
        str(
            FULL_RUN_REPORT_PATH
        ),

    "FullRunConditionInventory":
        str(
            FULL_RUN_INVENTORY_PATH
        ),

    "FullRunProjectRunMetrics":
        str(
            FULL_RUN_PROJECT_METRICS_PATH
        ),

    "FullRunFitTimes":
        str(
            FULL_RUN_FIT_TIMES_PATH
        ),

    "FullRunNoiseSummaries":
        str(
            FULL_RUN_NOISE_SUMMARY_PATH
        ),

    "FullRunExpectedConditions":
        EXPECTED_CONDITIONS,

    "FullRunCompletedConditions":
        int(
            len(
                condition_inventory
            )
        ),

    "FullRunRemainingConditions":
        0,

    "FullRunExpectedModelFits":
        EXPECTED_TOTAL_MODEL_FITS,

    "FullRunCompletedModelFits":
        total_model_fit_rows,

    "FullRunRankingRows":
        total_ranking_rows,

    "FullRunBuildMetricRows":
        total_build_metric_rows,

    "FullRunProjectRunRows":
        total_project_run_rows,

    "FullRunCurrentCondition":
        None,

    "UpdatedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
})


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 19. Final compact output
# ---------------------------------------------------------

clear_output(
    wait=True
)


print(
    "=== PROJECT 6 STEP 10 RESULT ==="
)


print(
    "\nCondition execution:"
)

print(
    "Expected conditions:",
    EXPECTED_CONDITIONS
)

print(
    "Previously successful conditions:",
    len(
        PREEXISTING_SUCCESSFUL_PAIRS
    )
)

print(
    "Executed during this invocation:",
    len(
        executed_during_this_invocation
    )
)

print(
    "Successful conditions:",
    len(
        condition_inventory
    )
)

print(
    "Remaining conditions:",
    0
)


print(
    "\nFull output totals:"
)

print(
    "ML model fits:",
    total_model_fit_rows
)

print(
    "Ranking rows:",
    total_ranking_rows
)

print(
    "Build-metric rows:",
    total_build_metric_rows
)

print(
    "Project-run rows:",
    total_project_run_rows
)


print(
    "\nCompletion by noise level:"
)

display(
    completion_by_noise
)


print(
    "\nML fit-time summary:"
)

display(
    fit_time_summary
)


print(
    "\nConsolidated files:"
)

print(
    "Condition inventory:"
)

print(
    FULL_RUN_INVENTORY_PATH
)

print(
    "Project-run metrics:"
)

print(
    FULL_RUN_PROJECT_METRICS_PATH
)

print(
    "Fit times:"
)

print(
    FULL_RUN_FIT_TIMES_PATH
)

print(
    "Noise summaries:"
)

print(
    FULL_RUN_NOISE_SUMMARY_PATH
)


print(
    "\nExecution report:"
)

print(
    FULL_RUN_REPORT_PATH
)


print(
    "\nElapsed during this invocation:",
    f"{total_elapsed_this_invocation / 60.0:.1f} minutes"
)


print(
    "\nExecution status:",
    "PASS"
)


print(
    "\nSUCCESS: All 270 project × noise × seed "
    "conditions completed."
)

print(
    "SUCCESS: All 1,080 ML model fits completed."
)

print(
    "SUCCESS: All 7,709,310 ranking rows were saved."
)

print(
    "SUCCESS: All 71,820 build-metric rows were saved."
)

print(
    "SUCCESS: All 1,890 project-run rows were saved."
)

print(
    "SUCCESS: Every condition has a validated success "
    "marker and its required files."
)

print(
    "SUCCESS: Project 6 is ready for aggregation and "
    "the independent final audit."
)

=== PROJECT 6 STEP 10 RESULT ===

Condition execution:
Expected conditions: 270
Previously successful conditions: 1
Executed during this invocation: 269
Successful conditions: 270
Remaining conditions: 0

Full output totals:
ML model fits: 1080
Ranking rows: 7709310
Build-metric rows: 71820
Project-run rows: 1890

Completion by noise level:


,NoisePercent,Conditions,TotalNoiseFlips,MeanRealisedNoisePercent,TotalElapsedSeconds
0,0,30,0,0.000000,1613.296328
1,5,30,29793,5.004283,1779.539359
2,10,30,59498,9.993785,1811.468763
3,15,30,89159,14.975897,1837.273274
4,20,30,118701,19.938020,1840.752998
5,25,30,148283,24.906862,1806.970970
6,30,30,178197,29.931469,1868.906108
7,40,30,237806,39.943899,1831.168199
8,50,30,297480,49.967246,1808.963607



ML fit-time summary:


,Technique,ModelFits,MeanFitSeconds,MedianFitSeconds,TotalFitSeconds
0,LightGBM,270,2.180882,1.738007,588.838081
1,NaiveBayes,270,0.245363,0.238292,66.247941
2,RandomForest,270,7.683726,7.789398,2074.605895
3,XGBoost,270,4.206249,4.179983,1135.687331



Consolidated files:
Condition inventory:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/eclipse__jetty.project/jetty_30_seed_full_run/jetty_full_run_condition_inventory.csv
Project-run metrics:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/eclipse__jetty.project/jetty_30_seed_full_run/jetty_all_project_run_metrics.csv
Fit times:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/eclipse__jetty.project/jetty_30_seed_full_run/jetty_all_fit_times.csv
Noise summaries:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/eclipse__jetty.project/jetty_30_seed_full_run/jetty_all_noise_summaries.csv

Execution report:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/eclipse__jetty.project/jetty_30_seed_full_run/jetty_full_run_execution_report.json

Elapsed during this invocation: 270.0 minutes

Execution status: PASS

SUCCESS: All 270 project × noise × seed conditions completed.
SUCCESS: All 1,080 ML model fits completed.
SUCCESS: All 7,709

In [ ]:
# =========================================================
# PROJECT 6 — STEP 11
# FULL AGGREGATION AND INDEPENDENT FINAL AUDIT
#
# Independently audits:
#   - all 270 completed conditions
#   - all 7,709,310 ranking rows
#   - rank ordering and fixed evaluation cohorts
#   - all 71,820 APFD/APFDc build metrics
#   - all 1,890 project-run aggregates
#   - all 5,358,150 noise-manifest rows
#   - nested masks across all 30 seeds
#   - flip and subtype rules
#   - Random/QTF/LatestFail/NB invariance rules
#
# Produces:
#   - seed-level project-run results
#   - paired deltas from the same seed's 0% condition
#   - noise × technique summaries with 95% CIs
#   - condition and manifest audit tables
#   - permanent final-audit report
#
# No models are fitted.
# The audit is resumable at condition level.
# =========================================================

from pathlib import Path
import gc
import hashlib
import json
import math
import time
import traceback

import numpy as np
import pandas as pd
from IPython.display import clear_output, display


print(
    "=== PROJECT 6 STEP 11: "
    "FULL AGGREGATION AND INDEPENDENT FINAL AUDIT ==="
)


# ---------------------------------------------------------
# 1. Confirm successful Step 10 state
# ---------------------------------------------------------

required_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",

    "PROJECT_RAW_RESULTS",
    "PROJECT_AGGREGATED_RESULTS",
    "PROJECT_6_SELECTION_CHECKPOINT",

    "NOISE_LEVELS",
    "REPETITION_SEEDS",
    "ML_TECHNIQUES",
    "BASELINE_TECHNIQUES",
    "ALL_TECHNIQUES",
]


missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required Project 6 objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nReconnect the runtime and run the Project 6 "
        "recovery cell. Do not rerun Step 10."
    )


if PROJECT_NUMBER != 6:
    raise AssertionError(
        f"Expected Project 6, observed {PROJECT_NUMBER}."
    )


if PROJECT_NAME != "eclipse@jetty.project":
    raise AssertionError(
        f"Unexpected project: {PROJECT_NAME}"
    )


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    previous_checkpoint = json.load(
        checkpoint_file
    )


allowed_previous_statuses = {
    "FULL_RUN_COMPLETED",
    "FINAL_AUDIT_IN_PROGRESS",
    "FINAL_AUDIT_INTERRUPTED",
    "FINAL_AUDIT_PASSED",
}


if previous_checkpoint.get(
    "Status"
) not in allowed_previous_statuses:
    raise AssertionError(
        "Step 10 has not been permanently marked as "
        "completed.\n"
        f"Observed status: "
        f"{previous_checkpoint.get('Status')}"
    )


EXPECTED_NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]


EXPECTED_SEEDS = list(
    range(1, 31)
)


if list(NOISE_LEVELS) != EXPECTED_NOISE_LEVELS:
    raise AssertionError(
        "Unexpected noise-level configuration."
    )


if list(REPETITION_SEEDS) != EXPECTED_SEEDS:
    raise AssertionError(
        "Unexpected repetition-seed configuration."
    )


if len(ALL_TECHNIQUES) != 7:
    raise AssertionError(
        "Expected seven techniques."
    )


# ---------------------------------------------------------
# 2. Expected totals
# ---------------------------------------------------------

EXPECTED_CONDITIONS = 270
EXPECTED_RANKING_ROWS_PER_CONDITION = 28553
EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = 266
EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = 7
EXPECTED_FIT_ROWS_PER_CONDITION = 4
EXPECTED_MANIFEST_ROWS_PER_CONDITION = 19845

EXPECTED_TOTAL_RANKING_ROWS = 7709310
EXPECTED_TOTAL_BUILD_METRIC_ROWS = 71820
EXPECTED_TOTAL_PROJECT_RUN_ROWS = 1890
EXPECTED_TOTAL_FIT_ROWS = 1080
EXPECTED_TOTAL_MANIFEST_ROWS = 5358150

EXPECTED_EVALUATION_ROWS_PER_TECHNIQUE = 4079
EXPECTED_EVALUATED_BUILDS = 38
EXPECTED_FAILURE_EXECUTIONS = 40


CONDITION_PAIRS = [
    (
        int(noise_percent),
        int(repetition_seed),
    )
    for noise_percent in NOISE_LEVELS
    for repetition_seed in REPETITION_SEEDS
]


if len(CONDITION_PAIRS) != EXPECTED_CONDITIONS:
    raise AssertionError(
        "Expected exactly 270 condition pairs."
    )


# ---------------------------------------------------------
# 3. Permanent paths
# ---------------------------------------------------------

FULL_RUN_DIRECTORY = (
    PROJECT_AGGREGATED_RESULTS
    / "jetty_30_seed_full_run"
)


FULL_RUN_INVENTORY_PATH = (
    FULL_RUN_DIRECTORY
    / "jetty_full_run_condition_inventory.csv"
)


FULL_RUN_PROJECT_METRICS_PATH = (
    FULL_RUN_DIRECTORY
    / "jetty_all_project_run_metrics.csv"
)


FULL_RUN_FIT_TIMES_PATH = (
    FULL_RUN_DIRECTORY
    / "jetty_all_fit_times.csv"
)


FULL_RUN_NOISE_SUMMARY_PATH = (
    FULL_RUN_DIRECTORY
    / "jetty_all_noise_summaries.csv"
)


FINAL_AUDIT_DIRECTORY = (
    PROJECT_AGGREGATED_RESULTS
    / "jetty_30_seed_final_audit"
)


FINAL_AUDIT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


CONDITION_AUDIT_PROGRESS_PATH = (
    FINAL_AUDIT_DIRECTORY
    / "jetty_condition_audit_progress.csv"
)


MANIFEST_AUDIT_PATH = (
    FINAL_AUDIT_DIRECTORY
    / "jetty_manifest_audit.csv"
)


SEED_LEVEL_RESULTS_PATH = (
    FINAL_AUDIT_DIRECTORY
    / "jetty_seed_level_project_run_metrics.csv"
)


SEED_DELTA_PATH = (
    FINAL_AUDIT_DIRECTORY
    / "jetty_seed_delta_from_0pct.csv"
)


NOISE_TECHNIQUE_SUMMARY_PATH = (
    FINAL_AUDIT_DIRECTORY
    / "jetty_noise_technique_summary.csv"
)


NOISE_DELTA_SUMMARY_PATH = (
    FINAL_AUDIT_DIRECTORY
    / "jetty_noise_delta_summary.csv"
)


BASELINE_INVARIANCE_PATH = (
    FINAL_AUDIT_DIRECTORY
    / "jetty_baseline_invariance_audit.json"
)


FINAL_AUDIT_REPORT_PATH = (
    FINAL_AUDIT_DIRECTORY
    / "jetty_final_audit_report.json"
)


FINAL_AUDIT_ERROR_PATH = (
    FINAL_AUDIT_DIRECTORY
    / "jetty_final_audit_last_error.json"
)


# ---------------------------------------------------------
# 4. Condition paths
# ---------------------------------------------------------

def get_condition_paths(
    noise_percent,
    repetition_seed,
):
    condition_directory = (
        PROJECT_RAW_RESULTS
        / f"noise_{int(noise_percent):03d}"
        / f"seed_{int(repetition_seed):02d}"
    )


    return {
        "directory":
            condition_directory,

        "marker":
            condition_directory
            / "_SUCCESS.json",

        "rankings":
            condition_directory
            / "rankings.parquet",

        "build_metrics":
            condition_directory
            / "build_metrics.parquet",

        "project_run_metrics":
            condition_directory
            / "project_run_metrics.csv",

        "fit_times":
            condition_directory
            / "fit_times.csv",

        "noise_summary":
            condition_directory
            / "noise_summary.csv",

        "noise_manifest":
            condition_directory
            / "noise_manifest.parquet",

        "training_medians":
            condition_directory
            / "training_medians.csv",
    }


REQUIRED_FILE_KEYS = [
    "rankings",
    "build_metrics",
    "project_run_metrics",
    "fit_times",
    "noise_summary",
    "noise_manifest",
    "training_medians",
]


# ---------------------------------------------------------
# 5. Independent APFD and APFDc formulas
# ---------------------------------------------------------

def independent_apfd(
    actual_failures,
):
    failures = np.asarray(
        actual_failures,
        dtype=int,
    )


    number_of_tests = len(
        failures
    )


    number_of_failures = int(
        failures.sum()
    )


    if number_of_tests == 0:
        return np.nan


    if number_of_failures == 0:
        return np.nan


    if not np.isin(
        failures,
        [
            0,
            1,
        ],
    ).all():
        raise ValueError(
            "Failures must contain only 0 and 1."
        )


    failure_ranks = (
        np.flatnonzero(
            failures == 1
        )
        + 1
    )


    return float(
        1.0
        - failure_ranks.sum()
        / (
            number_of_tests
            * number_of_failures
        )
        + 1.0
        / (
            2.0
            * number_of_tests
        )
    )


def independent_apfdc(
    actual_failures,
    durations,
):
    failures = np.asarray(
        actual_failures,
        dtype=int,
    )


    durations = np.asarray(
        durations,
        dtype=float,
    )


    if len(failures) != len(
        durations
    ):
        raise ValueError(
            "Failure and duration lengths differ."
        )


    if len(failures) == 0:
        return np.nan


    if int(
        failures.sum()
    ) == 0:
        return np.nan


    if not np.isin(
        failures,
        [
            0,
            1,
        ],
    ).all():
        raise ValueError(
            "Failures must contain only 0 and 1."
        )


    if not np.isfinite(
        durations
    ).all():
        raise ValueError(
            "Durations contain non-finite values."
        )


    if (
        durations
        < 0
    ).any():
        raise ValueError(
            "Durations cannot be negative."
        )


    total_duration = float(
        durations.sum()
    )


    if total_duration <= 0:
        return np.nan


    cumulative_before = np.concatenate([
        np.array(
            [
                0.0,
            ]
        ),
        np.cumsum(
            durations
        )[:-1],
    ])


    failure_mask = (
        failures
        == 1
    )


    midpoint_detection_times = (
        cumulative_before[
            failure_mask
        ]
        + 0.5
        * durations[
            failure_mask
        ]
    )


    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


# ---------------------------------------------------------
# 6. Independent ranking signature
# ---------------------------------------------------------

def independent_ranking_signature(
    rankings,
):
    signature_frame = (
        rankings[
            [
                "Technique",
                "Build",
                "Test",
                "Rank",
            ]
        ]
        .copy()
    )


    signature_frame[
        "Technique"
    ] = (
        signature_frame[
            "Technique"
        ].astype(str)
    )


    signature_frame[
        "Build"
    ] = pd.to_numeric(
        signature_frame[
            "Build"
        ],
        errors="raise",
    ).astype("int64")


    signature_frame[
        "Test"
    ] = (
        signature_frame[
            "Test"
        ].astype(str)
    )


    signature_frame[
        "Rank"
    ] = pd.to_numeric(
        signature_frame[
            "Rank"
        ],
        errors="raise",
    ).astype("int64")


    signature_frame = (
        signature_frame
        .sort_values(
            [
                "Technique",
                "Build",
                "Rank",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    row_hashes = (
        pd.util.hash_pandas_object(
            signature_frame,
            index=False,
        )
        .to_numpy(dtype="uint64")
    )


    return hashlib.sha256(
        row_hashes.tobytes()
    ).hexdigest()


# ---------------------------------------------------------
# 7. Independent condition ranking and metric audit
# ---------------------------------------------------------

def independently_audit_rankings(
    rankings,
    noise_percent,
    repetition_seed,
):
    rankings = (
        rankings
        .copy()
        .reset_index(drop=True)
    )


    required_columns = {
        "Project",
        "NoisePercent",
        "RepetitionSeed",
        "Technique",
        "Build",
        "Test",
        "Verdict",
        "Duration",
        "build_order",
        "Score",
        "ActualFailure",
        "Rank",
    }


    missing_columns = (
        required_columns
        - set(
            rankings.columns
        )
    )


    if missing_columns:
        raise AssertionError(
            "Ranking file is missing columns:\n"
            + "\n".join(
                sorted(
                    missing_columns
                )
            )
        )


    if len(rankings) != (
        EXPECTED_RANKING_ROWS_PER_CONDITION
    ):
        raise AssertionError(
            "Unexpected condition ranking-row count."
        )


    rankings[
        "Technique"
    ] = (
        rankings[
            "Technique"
        ].astype(str)
    )


    rankings[
        "Build"
    ] = pd.to_numeric(
        rankings[
            "Build"
        ],
        errors="raise",
    ).astype("int64")


    rankings[
        "Test"
    ] = (
        rankings[
            "Test"
        ].astype(str)
    )


    rankings[
        "Verdict"
    ] = pd.to_numeric(
        rankings[
            "Verdict"
        ],
        errors="raise",
    ).astype("int8")


    rankings[
        "Duration"
    ] = pd.to_numeric(
        rankings[
            "Duration"
        ],
        errors="raise",
    ).astype(float)


    rankings[
        "ActualFailure"
    ] = pd.to_numeric(
        rankings[
            "ActualFailure"
        ],
        errors="raise",
    ).astype("int8")


    rankings[
        "Rank"
    ] = pd.to_numeric(
        rankings[
            "Rank"
        ],
        errors="raise",
    ).astype("int64")


    if set(
        rankings[
            "Technique"
        ]
    ) != set(
        ALL_TECHNIQUES
    ):
        raise AssertionError(
            "Ranking file does not contain all "
            "seven techniques."
        )


    actual_failure_mismatches = int(
        (
            (
                rankings[
                    "Verdict"
                ].to_numpy(dtype=int)
                != 0
            ).astype(int)
            !=
            rankings[
                "ActualFailure"
            ].to_numpy(dtype=int)
        ).sum()
    )


    if actual_failure_mismatches != 0:
        raise AssertionError(
            "ActualFailure differs from Verdict != 0."
        )


    metric_records = []
    rank_sequence_violations = 0
    duplicate_test_violations = 0
    score_order_violations = 0
    build_count_violations = 0


    technique_signatures = {}


    for technique in ALL_TECHNIQUES:
        technique_rows = (
            rankings[
                rankings[
                    "Technique"
                ] == technique
            ]
            .copy()
        )


        if len(
            technique_rows
        ) != (
            EXPECTED_EVALUATION_ROWS_PER_TECHNIQUE
        ):
            raise AssertionError(
                f"{technique} does not contain "
                "4,079 ranking rows."
            )


        if int(
            technique_rows[
                "Build"
            ].nunique()
        ) != EXPECTED_EVALUATED_BUILDS:
            build_count_violations += 1


        if int(
            technique_rows[
                "ActualFailure"
            ].sum()
        ) != EXPECTED_FAILURE_EXECUTIONS:
            raise AssertionError(
                f"{technique} does not contain exactly "
                "40 failure executions."
            )


        technique_signatures[
            technique
        ] = independent_ranking_signature(
            technique_rows
        )


        for build_id, build_rows in (
            technique_rows.groupby(
                "Build",
                sort=False,
            )
        ):
            build_rows = (
                build_rows
                .sort_values(
                    "Rank",
                    kind="mergesort",
                )
                .reset_index(drop=True)
            )


            expected_ranks = np.arange(
                1,
                len(build_rows) + 1,
                dtype=np.int64,
            )


            if not np.array_equal(
                build_rows[
                    "Rank"
                ].to_numpy(dtype=np.int64),
                expected_ranks,
            ):
                rank_sequence_violations += 1


            if build_rows[
                "Test"
            ].duplicated().any():
                duplicate_test_violations += 1


            if technique == "QTF-Avg":
                expected_test_order = (
                    build_rows
                    .sort_values(
                        [
                            "Score",
                            "Test",
                        ],
                        ascending=[
                            True,
                            True,
                        ],
                        kind="mergesort",
                    )[
                        "Test"
                    ]
                    .astype(str)
                    .tolist()
                )

            else:
                expected_test_order = (
                    build_rows
                    .sort_values(
                        [
                            "Score",
                            "Test",
                        ],
                        ascending=[
                            False,
                            True,
                        ],
                        kind="mergesort",
                    )[
                        "Test"
                    ]
                    .astype(str)
                    .tolist()
                )


            observed_test_order = (
                build_rows[
                    "Test"
                ]
                .astype(str)
                .tolist()
            )


            if (
                observed_test_order
                != expected_test_order
            ):
                score_order_violations += 1


            actual_failures = (
                build_rows[
                    "ActualFailure"
                ].to_numpy(dtype=int)
            )


            durations = (
                build_rows[
                    "Duration"
                ].to_numpy(dtype=float)
            )


            metric_records.append({
                "Project":
                    PROJECT_NAME,

                "NoisePercent":
                    float(
                        noise_percent
                    ),

                "RepetitionSeed":
                    int(
                        repetition_seed
                    ),

                "Technique":
                    technique,

                "Build":
                    int(
                        build_id
                    ),

                "BuildOrder":
                    int(
                        build_rows[
                            "build_order"
                        ].iloc[0]
                    ),

                "NumberOfTests":
                    int(
                        len(
                            build_rows
                        )
                    ),

                "NumberOfFailures":
                    int(
                        actual_failures.sum()
                    ),

                "TotalDuration":
                    float(
                        durations.sum()
                    ),

                "APFD":
                    independent_apfd(
                        actual_failures
                    ),

                "APFDc":
                    independent_apfdc(
                        actual_failures,
                        durations,
                    ),
            })


    if build_count_violations != 0:
        raise AssertionError(
            "A technique does not cover all 38 builds."
        )


    if rank_sequence_violations != 0:
        raise AssertionError(
            "At least one build has an invalid rank "
            "sequence."
        )


    if duplicate_test_violations != 0:
        raise AssertionError(
            "At least one ranking contains duplicate "
            "tests within a build."
        )


    if score_order_violations != 0:
        raise AssertionError(
            "At least one ranking violates the frozen "
            "score/Test ordering."
        )


    independent_build_metrics = pd.DataFrame(
        metric_records
    )


    if len(
        independent_build_metrics
    ) != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION:
        raise AssertionError(
            "Independent build-metric row count mismatch."
        )


    return (
        independent_build_metrics,
        technique_signatures,
        {
            "ActualFailureMismatches":
                actual_failure_mismatches,

            "RankSequenceViolations":
                rank_sequence_violations,

            "DuplicateTestViolations":
                duplicate_test_violations,

            "ScoreOrderViolations":
                score_order_violations,

            "BuildCountViolations":
                build_count_violations,
        },
    )


# ---------------------------------------------------------
# 8. Independent project-run aggregation
# ---------------------------------------------------------

def independently_aggregate_project_run(
    build_metrics,
):
    result_rows = []


    for technique, technique_rows in (
        build_metrics.groupby(
            "Technique",
            sort=False,
        )
    ):
        apfd_values = pd.to_numeric(
            technique_rows[
                "APFD"
            ],
            errors="raise",
        ).to_numpy(dtype=float)


        apfdc_values = pd.to_numeric(
            technique_rows[
                "APFDc"
            ],
            errors="raise",
        ).to_numpy(dtype=float)


        evaluated_builds = int(
            len(
                technique_rows
            )
        )


        if evaluated_builds != (
            EXPECTED_EVALUATED_BUILDS
        ):
            raise AssertionError(
                f"{technique} does not contain "
                "38 build metrics."
            )


        mean_apfd = float(
            np.mean(
                apfd_values
            )
        )


        mean_apfdc = float(
            np.mean(
                apfdc_values
            )
        )


        sd_apfd = float(
            np.std(
                apfd_values,
                ddof=1,
            )
        )


        sd_apfdc = float(
            np.std(
                apfdc_values,
                ddof=1,
            )
        )


        se_apfd = (
            sd_apfd
            / math.sqrt(
                evaluated_builds
            )
        )


        se_apfdc = (
            sd_apfdc
            / math.sqrt(
                evaluated_builds
            )
        )


        result_rows.append({
            "Project":
                PROJECT_NAME,

            "NoisePercent":
                float(
                    technique_rows[
                        "NoisePercent"
                    ].iloc[0]
                ),

            "RepetitionSeed":
                int(
                    technique_rows[
                        "RepetitionSeed"
                    ].iloc[0]
                ),

            "Technique":
                str(
                    technique
                ),

            "MeanAPFD":
                mean_apfd,

            "MeanAPFDc":
                mean_apfdc,

            "SD_APFD":
                sd_apfd,

            "SD_APFDc":
                sd_apfdc,

            "CI95Low_APFD":
                float(
                    mean_apfd
                    - 1.96
                    * se_apfd
                ),

            "CI95High_APFD":
                float(
                    mean_apfd
                    + 1.96
                    * se_apfd
                ),

            "CI95Low_APFDc":
                float(
                    mean_apfdc
                    - 1.96
                    * se_apfdc
                ),

            "CI95High_APFDc":
                float(
                    mean_apfdc
                    + 1.96
                    * se_apfdc
                ),

            "EvaluatedBuilds":
                evaluated_builds,

            "RankingRows":
                int(
                    technique_rows[
                        "NumberOfTests"
                    ].sum()
                ),

            "FailureExecutions":
                int(
                    technique_rows[
                        "NumberOfFailures"
                    ].sum()
                ),
        })


    result = pd.DataFrame(
        result_rows
    )


    if len(result) != (
        EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
    ):
        raise AssertionError(
            "Independent project-run row count mismatch."
        )


    return result


# ---------------------------------------------------------
# 9. Table comparison helpers
# ---------------------------------------------------------

def compare_build_metric_tables(
    stored,
    recomputed,
):
    stored_sorted = (
        stored
        .sort_values(
            [
                "Technique",
                "Build",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    recomputed_sorted = (
        recomputed
        .sort_values(
            [
                "Technique",
                "Build",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    if len(stored_sorted) != len(
        recomputed_sorted
    ):
        raise AssertionError(
            "Stored and recomputed build-metric row "
            "counts differ."
        )


    exact_columns = [
        "Project",
        "Technique",
        "Build",
        "BuildOrder",
        "NumberOfTests",
        "NumberOfFailures",
    ]


    numeric_columns = [
        "NoisePercent",
        "RepetitionSeed",
        "TotalDuration",
        "APFD",
        "APFDc",
    ]


    exact_mismatches = 0
    numeric_mismatches = 0


    for column in exact_columns:
        exact_mismatches += int(
            (
                stored_sorted[
                    column
                ].astype(str).to_numpy()
                !=
                recomputed_sorted[
                    column
                ].astype(str).to_numpy()
            ).sum()
        )


    for column in numeric_columns:
        stored_values = pd.to_numeric(
            stored_sorted[
                column
            ],
            errors="raise",
        ).to_numpy(dtype=float)


        recomputed_values = pd.to_numeric(
            recomputed_sorted[
                column
            ],
            errors="raise",
        ).to_numpy(dtype=float)


        numeric_mismatches += int(
            (
                ~np.isclose(
                    stored_values,
                    recomputed_values,
                    rtol=1e-12,
                    atol=1e-12,
                    equal_nan=True,
                )
            ).sum()
        )


    return (
        exact_mismatches,
        numeric_mismatches,
    )


def compare_project_metric_tables(
    stored,
    recomputed,
):
    stored_sorted = (
        stored
        .sort_values(
            "Technique",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    recomputed_sorted = (
        recomputed
        .sort_values(
            "Technique",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    if len(stored_sorted) != len(
        recomputed_sorted
    ):
        raise AssertionError(
            "Stored and recomputed project-run row "
            "counts differ."
        )


    exact_columns = [
        "Project",
        "Technique",
        "EvaluatedBuilds",
        "RankingRows",
        "FailureExecutions",
    ]


    numeric_columns = [
        "NoisePercent",
        "RepetitionSeed",
        "MeanAPFD",
        "MeanAPFDc",
        "SD_APFD",
        "SD_APFDc",
        "CI95Low_APFD",
        "CI95High_APFD",
        "CI95Low_APFDc",
        "CI95High_APFDc",
    ]


    exact_mismatches = 0
    numeric_mismatches = 0


    for column in exact_columns:
        exact_mismatches += int(
            (
                stored_sorted[
                    column
                ].astype(str).to_numpy()
                !=
                recomputed_sorted[
                    column
                ].astype(str).to_numpy()
            ).sum()
        )


    for column in numeric_columns:
        stored_values = pd.to_numeric(
            stored_sorted[
                column
            ],
            errors="raise",
        ).to_numpy(dtype=float)


        recomputed_values = pd.to_numeric(
            recomputed_sorted[
                column
            ],
            errors="raise",
        ).to_numpy(dtype=float)


        numeric_mismatches += int(
            (
                ~np.isclose(
                    stored_values,
                    recomputed_values,
                    rtol=1e-10,
                    atol=1e-12,
                    equal_nan=True,
                )
            ).sum()
        )


    return (
        exact_mismatches,
        numeric_mismatches,
    )


# ---------------------------------------------------------
# 10. Load resumable condition-audit progress
# ---------------------------------------------------------

if CONDITION_AUDIT_PROGRESS_PATH.exists():
    existing_condition_audit = pd.read_csv(
        CONDITION_AUDIT_PROGRESS_PATH
    )


    if len(
        existing_condition_audit
    ) > 0:
        existing_condition_audit[
            "NoisePercent"
        ] = pd.to_numeric(
            existing_condition_audit[
                "NoisePercent"
            ],
            errors="raise",
        ).astype(int)


        existing_condition_audit[
            "RepetitionSeed"
        ] = pd.to_numeric(
            existing_condition_audit[
                "RepetitionSeed"
            ],
            errors="raise",
        ).astype(int)


        existing_condition_audit = (
            existing_condition_audit
            .drop_duplicates(
                subset=[
                    "NoisePercent",
                    "RepetitionSeed",
                ],
                keep="last",
            )
            .reset_index(drop=True)
        )


else:
    existing_condition_audit = pd.DataFrame()


existing_audit_map = {
    (
        int(
            row[
                "NoisePercent"
            ]
        ),
        int(
            row[
                "RepetitionSeed"
            ]
        ),
    ):
        row.to_dict()

    for _, row in (
        existing_condition_audit.iterrows()
    )
}


audit_records = []
audit_start_time = time.time()


# ---------------------------------------------------------
# 11. Checkpoint helpers
# ---------------------------------------------------------

def update_final_audit_checkpoint(
    status,
    completed_conditions,
    current_condition=None,
    error_message=None,
):
    with open(
        PROJECT_6_SELECTION_CHECKPOINT,
        "r",
        encoding="utf-8",
    ) as checkpoint_file:
        checkpoint = json.load(
            checkpoint_file
        )


    checkpoint.update({
        "Status":
            status,

        "FinalAuditDirectory":
            str(
                FINAL_AUDIT_DIRECTORY
            ),

        "FinalAuditProgress":
            str(
                CONDITION_AUDIT_PROGRESS_PATH
            ),

        "FinalAuditExpectedConditions":
            EXPECTED_CONDITIONS,

        "FinalAuditCompletedConditions":
            int(
                completed_conditions
            ),

        "FinalAuditCurrentCondition":
            (
                {
                    "NoisePercent":
                        int(
                            current_condition[0]
                        ),

                    "RepetitionSeed":
                        int(
                            current_condition[1]
                        ),
                }
                if current_condition is not None
                else None
            ),

        "FinalAuditError":
            error_message,

        "UpdatedAtUTC":
            pd.Timestamp.utcnow().isoformat(),
    })


    with open(
        PROJECT_6_SELECTION_CHECKPOINT,
        "w",
        encoding="utf-8",
    ) as checkpoint_file:
        json.dump(
            checkpoint,
            checkpoint_file,
            indent=2,
            default=str,
        )


def save_condition_audit_progress(
    records,
):
    progress = pd.DataFrame(
        records
    )


    progress = (
        progress
        .drop_duplicates(
            subset=[
                "NoisePercent",
                "RepetitionSeed",
            ],
            keep="last",
        )
        .sort_values(
            [
                "NoisePercent",
                "RepetitionSeed",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    progress.to_csv(
        CONDITION_AUDIT_PROGRESS_PATH,
        index=False,
    )


    return progress


def display_audit_progress(
    completed,
    reused,
    current_noise=None,
    current_seed=None,
):
    clear_output(
        wait=True
    )


    print(
        "=== PROJECT 6 STEP 11: "
        "FULL AGGREGATION AND INDEPENDENT FINAL AUDIT ==="
    )


    print(
        "\nCondition ranking audit:",
        f"{completed}/{EXPECTED_CONDITIONS}"
    )


    print(
        "Reused prior audited conditions:",
        reused
    )


    print(
        "Audited during this invocation:",
        completed - reused
    )


    print(
        "Remaining:",
        EXPECTED_CONDITIONS - completed
    )


    if current_noise is not None:
        print(
            "\nLatest audited condition:",
            f"noise={current_noise}%, seed={current_seed}"
        )


    print(
        "Elapsed:",
        f"{(time.time() - audit_start_time) / 60.0:.1f}",
        "minutes"
    )


    print(
        "\nProgress saved to:"
    )

    print(
        CONDITION_AUDIT_PROGRESS_PATH
    )


update_final_audit_checkpoint(
    status="FINAL_AUDIT_IN_PROGRESS",
    completed_conditions=0,
)


# ---------------------------------------------------------
# 12. Independently audit every ranking condition
# ---------------------------------------------------------

reused_condition_audits = 0


try:
    for condition_number, (
        noise_percent,
        repetition_seed,
    ) in enumerate(
        CONDITION_PAIRS,
        start=1,
    ):
        paths = get_condition_paths(
            noise_percent,
            repetition_seed,
        )


        if not paths[
            "marker"
        ].exists():
            raise FileNotFoundError(
                "Missing condition success marker:\n"
                f"{paths['marker']}"
            )


        missing_files = [
            str(
                paths[file_key]
            )
            for file_key in REQUIRED_FILE_KEYS
            if not paths[
                file_key
            ].exists()
        ]


        if missing_files:
            raise FileNotFoundError(
                "Condition is missing required files:\n"
                + "\n".join(
                    missing_files
                )
            )


        with open(
            paths["marker"],
            "r",
            encoding="utf-8",
        ) as marker_file:
            marker = json.load(
                marker_file
            )


        marker_signature = str(
            marker.get(
                "RankingSignature",
                "",
            )
        )


        pair = (
            int(
                noise_percent
            ),
            int(
                repetition_seed
            ),
        )


        prior_record = existing_audit_map.get(
            pair
        )


        reusable_prior_record = bool(
            prior_record is not None
            and str(
                prior_record.get(
                    "AuditStatus",
                    ""
                )
            ) == "PASS"
            and str(
                prior_record.get(
                    "MarkerRankingSignature",
                    ""
                )
            ) == marker_signature
            and all(
                f"Signature_{technique}"
                in prior_record
                for technique in (
                    ALL_TECHNIQUES
                )
            )
        )


        if reusable_prior_record:
            audit_records.append(
                prior_record
            )


            reused_condition_audits += 1


            if (
                condition_number % 10 == 0
                or condition_number
                == EXPECTED_CONDITIONS
            ):
                display_audit_progress(
                    completed=len(
                        audit_records
                    ),
                    reused=(
                        reused_condition_audits
                    ),
                    current_noise=(
                        noise_percent
                    ),
                    current_seed=(
                        repetition_seed
                    ),
                )


            continue


        condition_audit_start = time.time()


        rankings = pd.read_parquet(
            paths[
                "rankings"
            ]
        )


        stored_build_metrics = pd.read_parquet(
            paths[
                "build_metrics"
            ]
        )


        stored_project_metrics = pd.read_csv(
            paths[
                "project_run_metrics"
            ]
        )


        fit_times = pd.read_csv(
            paths[
                "fit_times"
            ]
        )


        noise_summary = pd.read_csv(
            paths[
                "noise_summary"
            ]
        )


        if len(fit_times) != (
            EXPECTED_FIT_ROWS_PER_CONDITION
        ):
            raise AssertionError(
                "Condition does not contain four fit rows."
            )


        if len(noise_summary) != 1:
            raise AssertionError(
                "Condition does not contain one noise "
                "summary row."
            )


        (
            independent_build_metrics,
            technique_signatures,
            ranking_violation_summary,
        ) = independently_audit_rankings(
            rankings=rankings,
            noise_percent=noise_percent,
            repetition_seed=repetition_seed,
        )


        independent_project_metrics = (
            independently_aggregate_project_run(
                independent_build_metrics
            )
        )


        (
            build_exact_mismatches,
            build_numeric_mismatches,
        ) = compare_build_metric_tables(
            stored=stored_build_metrics,
            recomputed=(
                independent_build_metrics
            ),
        )


        (
            project_exact_mismatches,
            project_numeric_mismatches,
        ) = compare_project_metric_tables(
            stored=stored_project_metrics,
            recomputed=(
                independent_project_metrics
            ),
        )


        recomputed_overall_signature = (
            independent_ranking_signature(
                rankings
            )
        )


        overall_signature_matches = bool(
            recomputed_overall_signature
            == marker_signature
        )


        if not overall_signature_matches:
            raise AssertionError(
                "Recomputed ranking signature differs "
                "from the success marker."
            )


        if (
            build_exact_mismatches
            + build_numeric_mismatches
            != 0
        ):
            raise AssertionError(
                "Independent build-metric audit failed."
            )


        if (
            project_exact_mismatches
            + project_numeric_mismatches
            != 0
        ):
            raise AssertionError(
                "Independent project-run audit failed."
            )


        audit_record = {
            "NoisePercent":
                int(
                    noise_percent
                ),

            "RepetitionSeed":
                int(
                    repetition_seed
                ),

            "AuditStatus":
                "PASS",

            "RankingRows":
                int(
                    len(
                        rankings
                    )
                ),

            "BuildMetricRows":
                int(
                    len(
                        independent_build_metrics
                    )
                ),

            "ProjectRunRows":
                int(
                    len(
                        independent_project_metrics
                    )
                ),

            "MarkerRankingSignature":
                marker_signature,

            "RecomputedRankingSignature":
                recomputed_overall_signature,

            "OverallSignatureMatches":
                overall_signature_matches,

            "BuildExactMismatches":
                int(
                    build_exact_mismatches
                ),

            "BuildNumericMismatches":
                int(
                    build_numeric_mismatches
                ),

            "ProjectExactMismatches":
                int(
                    project_exact_mismatches
                ),

            "ProjectNumericMismatches":
                int(
                    project_numeric_mismatches
                ),

            "ActualFailureMismatches":
                int(
                    ranking_violation_summary[
                        "ActualFailureMismatches"
                    ]
                ),

            "RankSequenceViolations":
                int(
                    ranking_violation_summary[
                        "RankSequenceViolations"
                    ]
                ),

            "DuplicateTestViolations":
                int(
                    ranking_violation_summary[
                        "DuplicateTestViolations"
                    ]
                ),

            "ScoreOrderViolations":
                int(
                    ranking_violation_summary[
                        "ScoreOrderViolations"
                    ]
                ),

            "BuildCountViolations":
                int(
                    ranking_violation_summary[
                        "BuildCountViolations"
                    ]
                ),

            "NoiseFlips":
                int(
                    noise_summary.iloc[0][
                        "NumberFlipped"
                    ]
                ),

            "RealisedNoisePercent":
                float(
                    noise_summary.iloc[0][
                        "RealisedNoisePercent"
                    ]
                ),

            "AuditSeconds":
                float(
                    time.time()
                    - condition_audit_start
                ),

            "AuditedAtUTC":
                pd.Timestamp.utcnow().isoformat(),
        }


        for technique in ALL_TECHNIQUES:
            audit_record[
                f"Signature_{technique}"
            ] = technique_signatures[
                technique
            ]


        audit_records.append(
            audit_record
        )


        current_progress = (
            save_condition_audit_progress(
                audit_records
            )
        )


        update_final_audit_checkpoint(
            status=(
                "FINAL_AUDIT_IN_PROGRESS"
            ),
            completed_conditions=len(
                current_progress
            ),
            current_condition=pair,
        )


        del rankings
        del stored_build_metrics
        del stored_project_metrics
        del independent_build_metrics
        del independent_project_metrics
        del fit_times
        del noise_summary

        gc.collect()


        display_audit_progress(
            completed=len(
                audit_records
            ),
            reused=(
                reused_condition_audits
            ),
            current_noise=(
                noise_percent
            ),
            current_seed=(
                repetition_seed
            ),
        )


except Exception as audit_error:
    error_record = {
        "Project":
            PROJECT_NAME,

        "Status":
            "FINAL_AUDIT_INTERRUPTED",

        "NoisePercent":
            (
                int(
                    noise_percent
                )
                if "noise_percent" in locals()
                else None
            ),

        "RepetitionSeed":
            (
                int(
                    repetition_seed
                )
                if "repetition_seed" in locals()
                else None
            ),

        "ErrorType":
            type(
                audit_error
            ).__name__,

        "ErrorMessage":
            str(
                audit_error
            ),

        "Traceback":
            traceback.format_exc(),

        "CompletedConditionAudits":
            int(
                len(
                    audit_records
                )
            ),

        "CompletedAtUTC":
            pd.Timestamp.utcnow().isoformat(),
    }


    with open(
        FINAL_AUDIT_ERROR_PATH,
        "w",
        encoding="utf-8",
    ) as error_file:
        json.dump(
            error_record,
            error_file,
            indent=2,
            default=str,
        )


    update_final_audit_checkpoint(
        status="FINAL_AUDIT_INTERRUPTED",
        completed_conditions=len(
            audit_records
        ),
        current_condition=(
            (
                noise_percent,
                repetition_seed,
            )
            if (
                "noise_percent" in locals()
                and "repetition_seed" in locals()
            )
            else None
        ),
        error_message=str(
            audit_error
        ),
    )


    print(
        "\nThe independent audit stopped at:"
    )

    print(
        "Noise:",
        (
            noise_percent
            if "noise_percent" in locals()
            else "unknown"
        )
    )

    print(
        "Seed:",
        (
            repetition_seed
            if "repetition_seed" in locals()
            else "unknown"
        )
    )

    print(
        "\nCompleted condition audits remain saved."
    )

    print(
        "Rerun this Step 11 cell to continue."
    )


    raise


condition_audit = (
    save_condition_audit_progress(
        audit_records
    )
)


if len(condition_audit) != (
    EXPECTED_CONDITIONS
):
    raise AssertionError(
        "Condition audit does not contain all "
        "270 conditions."
    )


if not (
    condition_audit[
        "AuditStatus"
    ] == "PASS"
).all():
    raise AssertionError(
        "At least one condition audit did not pass."
    )


# ---------------------------------------------------------
# 13. Independently audit all noise manifests
# ---------------------------------------------------------

manifest_audit_records = []


for seed_index, repetition_seed in enumerate(
    REPETITION_SEEDS,
    start=1,
):
    reference_row_ids = None
    reference_original_verdicts = None
    reference_uniforms = None
    reference_subtypes = None
    previous_flip_mask = None


    for noise_percent in NOISE_LEVELS:
        paths = get_condition_paths(
            noise_percent,
            repetition_seed,
        )


        manifest = pd.read_parquet(
            paths[
                "noise_manifest"
            ],
            columns=[
                "NoiseRowID",
                "OriginalVerdict",
                "SampledFailureSubtype",
                "NoisyVerdict",
                "FlipUniform",
                "Flipped",
                "PassToFailure",
                "FailureToPass",
            ],
        )


        noise_summary = pd.read_csv(
            paths[
                "noise_summary"
            ]
        )


        if len(manifest) != (
            EXPECTED_MANIFEST_ROWS_PER_CONDITION
        ):
            raise AssertionError(
                "Unexpected noise-manifest row count."
            )


        row_ids = pd.to_numeric(
            manifest[
                "NoiseRowID"
            ],
            errors="raise",
        ).to_numpy(dtype=np.int64)


        original_verdicts = pd.to_numeric(
            manifest[
                "OriginalVerdict"
            ],
            errors="raise",
        ).to_numpy(dtype=int)


        sampled_subtypes = pd.to_numeric(
            manifest[
                "SampledFailureSubtype"
            ],
            errors="raise",
        ).to_numpy(dtype=int)


        noisy_verdicts = pd.to_numeric(
            manifest[
                "NoisyVerdict"
            ],
            errors="raise",
        ).to_numpy(dtype=int)


        uniforms = pd.to_numeric(
            manifest[
                "FlipUniform"
            ],
            errors="raise",
        ).to_numpy(dtype=float)


        flipped = manifest[
            "Flipped"
        ].astype(bool).to_numpy()


        pass_to_failure = manifest[
            "PassToFailure"
        ].astype(bool).to_numpy()


        failure_to_pass = manifest[
            "FailureToPass"
        ].astype(bool).to_numpy()


        if reference_row_ids is None:
            reference_row_ids = (
                row_ids.copy()
            )


            reference_original_verdicts = (
                original_verdicts.copy()
            )


            reference_uniforms = (
                uniforms.copy()
            )


            reference_subtypes = (
                sampled_subtypes.copy()
            )


        row_identity_violations = int(
            (
                row_ids
                != reference_row_ids
            ).sum()
        )


        original_stream_violations = int(
            (
                original_verdicts
                != reference_original_verdicts
            ).sum()
        )


        uniform_stream_violations = int(
            (
                uniforms
                != reference_uniforms
            ).sum()
        )


        subtype_stream_violations = int(
            (
                sampled_subtypes
                != reference_subtypes
            ).sum()
        )


        expected_flip_mask = (
            uniforms
            < float(
                noise_percent
            ) / 100.0
        )


        threshold_rule_violations = int(
            (
                flipped
                != expected_flip_mask
            ).sum()
        )


        expected_pass_to_failure = (
            flipped
            & (
                original_verdicts
                == 0
            )
        )


        expected_failure_to_pass = (
            flipped
            & (
                original_verdicts
                != 0
            )
        )


        pass_to_failure_rule_violations = int(
            (
                pass_to_failure
                != expected_pass_to_failure
            ).sum()
        )


        failure_to_pass_rule_violations = int(
            (
                failure_to_pass
                != expected_failure_to_pass
            ).sum()
        )


        expected_noisy_verdicts = (
            original_verdicts.copy()
        )


        expected_noisy_verdicts[
            expected_pass_to_failure
        ] = sampled_subtypes[
            expected_pass_to_failure
        ]


        expected_noisy_verdicts[
            expected_failure_to_pass
        ] = 0


        noisy_verdict_rule_violations = int(
            (
                noisy_verdicts
                != expected_noisy_verdicts
            ).sum()
        )


        if previous_flip_mask is None:
            nested_mask_violations = 0

        else:
            nested_mask_violations = int(
                (
                    previous_flip_mask
                    & (
                        ~flipped
                    )
                ).sum()
            )


        manifest_flips = int(
            flipped.sum()
        )


        summary_flips = int(
            noise_summary.iloc[0][
                "NumberFlipped"
            ]
        )


        flip_count_mismatch = int(
            manifest_flips
            != summary_flips
        )


        realised_noise = (
            100.0
            * manifest_flips
            / len(
                manifest
            )
        )


        summary_realised_noise = float(
            noise_summary.iloc[0][
                "RealisedNoisePercent"
            ]
        )


        realised_noise_mismatch = int(
            not np.isclose(
                realised_noise,
                summary_realised_noise,
                rtol=1e-12,
                atol=1e-12,
            )
        )


        total_violations = int(
            row_identity_violations
            + original_stream_violations
            + uniform_stream_violations
            + subtype_stream_violations
            + threshold_rule_violations
            + pass_to_failure_rule_violations
            + failure_to_pass_rule_violations
            + noisy_verdict_rule_violations
            + nested_mask_violations
            + flip_count_mismatch
            + realised_noise_mismatch
        )


        manifest_audit_records.append({
            "NoisePercent":
                int(
                    noise_percent
                ),

            "RepetitionSeed":
                int(
                    repetition_seed
                ),

            "ManifestRows":
                int(
                    len(
                        manifest
                    )
                ),

            "FlippedRows":
                manifest_flips,

            "RealisedNoisePercent":
                realised_noise,

            "RowIdentityViolations":
                row_identity_violations,

            "OriginalStreamViolations":
                original_stream_violations,

            "UniformStreamViolations":
                uniform_stream_violations,

            "SubtypeStreamViolations":
                subtype_stream_violations,

            "ThresholdRuleViolations":
                threshold_rule_violations,

            "PassToFailureRuleViolations":
                pass_to_failure_rule_violations,

            "FailureToPassRuleViolations":
                failure_to_pass_rule_violations,

            "NoisyVerdictRuleViolations":
                noisy_verdict_rule_violations,

            "NestedMaskViolations":
                nested_mask_violations,

            "FlipCountMismatch":
                flip_count_mismatch,

            "RealisedNoiseMismatch":
                realised_noise_mismatch,

            "TotalViolations":
                total_violations,

            "AuditStatus":
                (
                    "PASS"
                    if total_violations == 0
                    else "FAIL"
                ),
        })


        previous_flip_mask = (
            flipped.copy()
        )


        del manifest
        del noise_summary

        gc.collect()


    clear_output(
        wait=True
    )


    print(
        "=== PROJECT 6 STEP 11: "
        "FULL AGGREGATION AND INDEPENDENT FINAL AUDIT ==="
    )


    print(
        "\nRanking conditions audited:",
        f"{len(condition_audit)}/270"
    )


    print(
        "Manifest seeds audited:",
        f"{seed_index}/30"
    )


    print(
        "Manifest conditions audited:",
        f"{seed_index * 9}/270"
    )


manifest_audit = pd.DataFrame(
    manifest_audit_records
)


manifest_audit.to_csv(
    MANIFEST_AUDIT_PATH,
    index=False,
)


if len(manifest_audit) != (
    EXPECTED_CONDITIONS
):
    raise AssertionError(
        "Manifest audit does not contain 270 rows."
    )


total_manifest_violations = int(
    manifest_audit[
        "TotalViolations"
    ].sum()
)


if total_manifest_violations != 0:
    display(
        manifest_audit[
            manifest_audit[
                "TotalViolations"
            ] != 0
        ]
    )


    raise AssertionError(
        "At least one full-run noise manifest failed "
        "independent validation."
    )


# ---------------------------------------------------------
# 14. Load and validate consolidated compact results
# ---------------------------------------------------------

for required_path in [
    FULL_RUN_INVENTORY_PATH,
    FULL_RUN_PROJECT_METRICS_PATH,
    FULL_RUN_FIT_TIMES_PATH,
    FULL_RUN_NOISE_SUMMARY_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Missing consolidated Step 10 file:\n"
            f"{required_path}"
        )


condition_inventory = pd.read_csv(
    FULL_RUN_INVENTORY_PATH
)


seed_level_results = pd.read_csv(
    FULL_RUN_PROJECT_METRICS_PATH
)


all_fit_times = pd.read_csv(
    FULL_RUN_FIT_TIMES_PATH
)


all_noise_summaries = pd.read_csv(
    FULL_RUN_NOISE_SUMMARY_PATH
)


if len(condition_inventory) != 270:
    raise AssertionError(
        "Condition inventory does not contain 270 rows."
    )


if len(seed_level_results) != (
    EXPECTED_TOTAL_PROJECT_RUN_ROWS
):
    raise AssertionError(
        "Seed-level project-run results do not contain "
        "1,890 rows."
    )


if len(all_fit_times) != (
    EXPECTED_TOTAL_FIT_ROWS
):
    raise AssertionError(
        "Fit-time table does not contain 1,080 rows."
    )


if len(all_noise_summaries) != 270:
    raise AssertionError(
        "Noise-summary table does not contain 270 rows."
    )


seed_level_results[
    "NoisePercent"
] = pd.to_numeric(
    seed_level_results[
        "NoisePercent"
    ],
    errors="raise",
).astype(int)


seed_level_results[
    "RepetitionSeed"
] = pd.to_numeric(
    seed_level_results[
        "RepetitionSeed"
    ],
    errors="raise",
).astype(int)


seed_level_results[
    "Technique"
] = (
    seed_level_results[
        "Technique"
    ].astype(str)
)


if seed_level_results.duplicated(
    subset=[
        "NoisePercent",
        "RepetitionSeed",
        "Technique",
    ]
).any():
    raise AssertionError(
        "Seed-level results contain duplicate "
        "condition-technique rows."
    )


condition_technique_sizes = (
    seed_level_results
    .groupby(
        [
            "NoisePercent",
            "RepetitionSeed",
        ]
    )
    .size()
)


if not (
    condition_technique_sizes
    == 7
).all():
    raise AssertionError(
        "At least one condition does not contain "
        "seven technique rows."
    )


# ---------------------------------------------------------
# 15. Validate baseline and deterministic invariance
# ---------------------------------------------------------

signature_columns = {
    technique:
        f"Signature_{technique}"
    for technique in ALL_TECHNIQUES
}


qtf_unique_signatures = int(
    condition_audit[
        signature_columns[
            "QTF-Avg"
        ]
    ].nunique()
)


random_noise_invariance = (
    condition_audit
    .groupby(
        "RepetitionSeed"
    )[
        signature_columns[
            "Random"
        ]
    ]
    .nunique()
)


random_noise_invariance_violations = int(
    (
        random_noise_invariance
        != 1
    ).sum()
)


random_seed_signature_count = int(
    condition_audit[
        condition_audit[
            "NoisePercent"
        ] == 0
    ][
        signature_columns[
            "Random"
        ]
    ].nunique()
)


zero_percent_latest_fail_signatures = int(
    condition_audit[
        condition_audit[
            "NoisePercent"
        ] == 0
    ][
        signature_columns[
            "LatestFail"
        ]
    ].nunique()
)


zero_percent_qtf_signatures = int(
    condition_audit[
        condition_audit[
            "NoisePercent"
        ] == 0
    ][
        signature_columns[
            "QTF-Avg"
        ]
    ].nunique()
)


zero_percent_nb_signatures = int(
    condition_audit[
        condition_audit[
            "NoisePercent"
        ] == 0
    ][
        signature_columns[
            "NaiveBayes"
        ]
    ].nunique()
)


baseline_invariance_report = {
    "Project":
        PROJECT_NAME,

    "Status":
        "PASS",

    "QTFUniqueSignaturesAcrossAllConditions":
        qtf_unique_signatures,

    "RandomNoiseInvarianceViolatingSeeds":
        random_noise_invariance_violations,

    "RandomUniqueSeedSignaturesAt0Percent":
        random_seed_signature_count,

    "LatestFailUniqueSignaturesAt0Percent":
        zero_percent_latest_fail_signatures,

    "QTFUniqueSignaturesAt0Percent":
        zero_percent_qtf_signatures,

    "NaiveBayesUniqueSignaturesAt0Percent":
        zero_percent_nb_signatures,

    "ExpectedRules": {
        "QTF-Avg":
            (
                "one ranking signature across all "
                "noise levels and seeds"
            ),

        "Random":
            (
                "one signature across noise levels "
                "within each seed; variation across seeds"
            ),

        "LatestFailAt0Percent":
            "one signature across all seeds",

        "NaiveBayesAt0Percent":
            "one signature across all seeds",
    },

    "CompletedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
}


if qtf_unique_signatures != 1:
    raise AssertionError(
        "QTF-Avg changed across conditions."
    )


if random_noise_invariance_violations != 0:
    raise AssertionError(
        "Random rankings changed across noise levels "
        "within at least one seed."
    )


if random_seed_signature_count <= 1:
    raise AssertionError(
        "Random rankings did not vary across seeds."
    )


if zero_percent_latest_fail_signatures != 1:
    raise AssertionError(
        "LatestFail varied across 0% seeds."
    )


if zero_percent_qtf_signatures != 1:
    raise AssertionError(
        "QTF-Avg varied across 0% seeds."
    )


if zero_percent_nb_signatures != 1:
    raise AssertionError(
        "Naive Bayes varied across 0% seeds."
    )


with open(
    BASELINE_INVARIANCE_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        baseline_invariance_report,
        report_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 16. Paired seed-level deltas from 0%
# ---------------------------------------------------------

clean_seed_reference = (
    seed_level_results[
        seed_level_results[
            "NoisePercent"
        ] == 0
    ][
        [
            "RepetitionSeed",
            "Technique",
            "MeanAPFD",
            "MeanAPFDc",
        ]
    ]
    .rename(
        columns={
            "MeanAPFD":
                "MeanAPFD_0pct",

            "MeanAPFDc":
                "MeanAPFDc_0pct",
        }
    )
)


if len(clean_seed_reference) != (
    30
    * 7
):
    raise AssertionError(
        "Expected 210 seed-technique clean references."
    )


seed_delta_results = (
    seed_level_results
    .merge(
        clean_seed_reference,
        on=[
            "RepetitionSeed",
            "Technique",
        ],
        how="left",
        validate="many_to_one",
    )
)


if seed_delta_results[
    [
        "MeanAPFD_0pct",
        "MeanAPFDc_0pct",
    ]
].isna().any().any():
    raise AssertionError(
        "At least one condition could not be paired with "
        "its same-seed 0% reference."
    )


seed_delta_results[
    "DeltaAPFD"
] = (
    pd.to_numeric(
        seed_delta_results[
            "MeanAPFD"
        ],
        errors="raise",
    )
    -
    pd.to_numeric(
        seed_delta_results[
            "MeanAPFD_0pct"
        ],
        errors="raise",
    )
)


seed_delta_results[
    "DeltaAPFDc"
] = (
    pd.to_numeric(
        seed_delta_results[
            "MeanAPFDc"
        ],
        errors="raise",
    )
    -
    pd.to_numeric(
        seed_delta_results[
            "MeanAPFDc_0pct"
        ],
        errors="raise",
    )
)


zero_delta_rows = (
    seed_delta_results[
        seed_delta_results[
            "NoisePercent"
        ] == 0
    ]
)


zero_delta_mismatches = int(
    (
        ~np.isclose(
            zero_delta_rows[
                "DeltaAPFD"
            ].to_numpy(dtype=float),
            0.0,
            rtol=1e-12,
            atol=1e-12,
        )
    ).sum()
    +
    (
        ~np.isclose(
            zero_delta_rows[
                "DeltaAPFDc"
            ].to_numpy(dtype=float),
            0.0,
            rtol=1e-12,
            atol=1e-12,
        )
    ).sum()
)


if zero_delta_mismatches != 0:
    raise AssertionError(
        "The paired 0% deltas are not zero."
    )


# ---------------------------------------------------------
# 17. Aggregate 30 seeds within Project 6
# ---------------------------------------------------------

summary_records = []
delta_summary_records = []


for (
    noise_percent,
    technique,
), group_rows in (
    seed_level_results.groupby(
        [
            "NoisePercent",
            "Technique",
        ],
        sort=False,
    )
):
    if len(group_rows) != 30:
        raise AssertionError(
            "Each noise-technique group must contain "
            "30 seed runs."
        )


    apfd_values = pd.to_numeric(
        group_rows[
            "MeanAPFD"
        ],
        errors="raise",
    ).to_numpy(dtype=float)


    apfdc_values = pd.to_numeric(
        group_rows[
            "MeanAPFDc"
        ],
        errors="raise",
    ).to_numpy(dtype=float)


    mean_apfd = float(
        np.mean(
            apfd_values
        )
    )


    mean_apfdc = float(
        np.mean(
            apfdc_values
        )
    )


    sd_apfd = float(
        np.std(
            apfd_values,
            ddof=1,
        )
    )


    sd_apfdc = float(
        np.std(
            apfdc_values,
            ddof=1,
        )
    )


    se_apfd = (
        sd_apfd
        / math.sqrt(30)
    )


    se_apfdc = (
        sd_apfdc
        / math.sqrt(30)
    )


    summary_records.append({
        "Project":
            PROJECT_NAME,

        "NoisePercent":
            int(
                noise_percent
            ),

        "Technique":
            str(
                technique
            ),

        "SeedRuns":
            30,

        "MeanAPFD":
            mean_apfd,

        "SD_APFD":
            sd_apfd,

        "MedianAPFD":
            float(
                np.median(
                    apfd_values
                )
            ),

        "CI95Low_APFD":
            float(
                mean_apfd
                - 1.96
                * se_apfd
            ),

        "CI95High_APFD":
            float(
                mean_apfd
                + 1.96
                * se_apfd
            ),

        "MeanAPFDc":
            mean_apfdc,

        "SD_APFDc":
            sd_apfdc,

        "MedianAPFDc":
            float(
                np.median(
                    apfdc_values
                )
            ),

        "CI95Low_APFDc":
            float(
                mean_apfdc
                - 1.96
                * se_apfdc
            ),

        "CI95High_APFDc":
            float(
                mean_apfdc
                + 1.96
                * se_apfdc
            ),
    })


for (
    noise_percent,
    technique,
), group_rows in (
    seed_delta_results.groupby(
        [
            "NoisePercent",
            "Technique",
        ],
        sort=False,
    )
):
    if len(group_rows) != 30:
        raise AssertionError(
            "Each delta group must contain 30 seeds."
        )


    delta_apfd = pd.to_numeric(
        group_rows[
            "DeltaAPFD"
        ],
        errors="raise",
    ).to_numpy(dtype=float)


    delta_apfdc = pd.to_numeric(
        group_rows[
            "DeltaAPFDc"
        ],
        errors="raise",
    ).to_numpy(dtype=float)


    mean_delta_apfd = float(
        np.mean(
            delta_apfd
        )
    )


    mean_delta_apfdc = float(
        np.mean(
            delta_apfdc
        )
    )


    sd_delta_apfd = float(
        np.std(
            delta_apfd,
            ddof=1,
        )
    )


    sd_delta_apfdc = float(
        np.std(
            delta_apfdc,
            ddof=1,
        )
    )


    delta_summary_records.append({
        "Project":
            PROJECT_NAME,

        "NoisePercent":
            int(
                noise_percent
            ),

        "Technique":
            str(
                technique
            ),

        "SeedRuns":
            30,

        "MeanDeltaAPFD":
            mean_delta_apfd,

        "SD_DeltaAPFD":
            sd_delta_apfd,

        "CI95Low_DeltaAPFD":
            float(
                mean_delta_apfd
                - 1.96
                * sd_delta_apfd
                / math.sqrt(30)
            ),

        "CI95High_DeltaAPFD":
            float(
                mean_delta_apfd
                + 1.96
                * sd_delta_apfd
                / math.sqrt(30)
            ),

        "MeanDeltaAPFDc":
            mean_delta_apfdc,

        "SD_DeltaAPFDc":
            sd_delta_apfdc,

        "CI95Low_DeltaAPFDc":
            float(
                mean_delta_apfdc
                - 1.96
                * sd_delta_apfdc
                / math.sqrt(30)
            ),

        "CI95High_DeltaAPFDc":
            float(
                mean_delta_apfdc
                + 1.96
                * sd_delta_apfdc
                / math.sqrt(30)
            ),
    })


noise_technique_summary = pd.DataFrame(
    summary_records
)


noise_delta_summary = pd.DataFrame(
    delta_summary_records
)


if len(
    noise_technique_summary
) != 63:
    raise AssertionError(
        "Expected 63 noise-technique summary rows."
    )


if len(
    noise_delta_summary
) != 63:
    raise AssertionError(
        "Expected 63 noise-delta summary rows."
    )


# ---------------------------------------------------------
# 18. Save final aggregation files
# ---------------------------------------------------------

seed_level_results = (
    seed_level_results
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


seed_delta_results = (
    seed_delta_results
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


noise_technique_summary = (
    noise_technique_summary
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


noise_delta_summary = (
    noise_delta_summary
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


seed_level_results.to_csv(
    SEED_LEVEL_RESULTS_PATH,
    index=False,
)


seed_delta_results.to_csv(
    SEED_DELTA_PATH,
    index=False,
)


noise_technique_summary.to_csv(
    NOISE_TECHNIQUE_SUMMARY_PATH,
    index=False,
)


noise_delta_summary.to_csv(
    NOISE_DELTA_SUMMARY_PATH,
    index=False,
)


# ---------------------------------------------------------
# 19. Final global audit totals
# ---------------------------------------------------------

audited_ranking_rows = int(
    condition_audit[
        "RankingRows"
    ].sum()
)


audited_build_metric_rows = int(
    condition_audit[
        "BuildMetricRows"
    ].sum()
)


audited_project_run_rows = int(
    condition_audit[
        "ProjectRunRows"
    ].sum()
)


audited_manifest_rows = int(
    manifest_audit[
        "ManifestRows"
    ].sum()
)


total_condition_metric_mismatches = int(
    condition_audit[
        [
            "BuildExactMismatches",
            "BuildNumericMismatches",
            "ProjectExactMismatches",
            "ProjectNumericMismatches",
            "ActualFailureMismatches",
            "RankSequenceViolations",
            "DuplicateTestViolations",
            "ScoreOrderViolations",
            "BuildCountViolations",
        ]
    ].to_numpy(dtype=int).sum()
)


overall_signature_mismatches = int(
    (
        ~condition_audit[
            "OverallSignatureMatches"
        ].astype(bool)
    ).sum()
)


if audited_ranking_rows != (
    EXPECTED_TOTAL_RANKING_ROWS
):
    raise AssertionError(
        "Audited ranking-row total mismatch."
    )


if audited_build_metric_rows != (
    EXPECTED_TOTAL_BUILD_METRIC_ROWS
):
    raise AssertionError(
        "Audited build-metric total mismatch."
    )


if audited_project_run_rows != (
    EXPECTED_TOTAL_PROJECT_RUN_ROWS
):
    raise AssertionError(
        "Audited project-run total mismatch."
    )


if audited_manifest_rows != (
    EXPECTED_TOTAL_MANIFEST_ROWS
):
    raise AssertionError(
        "Audited manifest-row total mismatch."
    )


if total_condition_metric_mismatches != 0:
    raise AssertionError(
        "At least one independent condition audit "
        "mismatch remains."
    )


if overall_signature_mismatches != 0:
    raise AssertionError(
        "At least one ranking signature mismatch remains."
    )


# ---------------------------------------------------------
# 20. Save final audit report
# ---------------------------------------------------------

final_audit_elapsed = float(
    time.time()
    - audit_start_time
)


final_audit_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "PASS",

    "ConditionsAudited":
        int(
            len(
                condition_audit
            )
        ),

    "ConditionsReusedFromPriorAudit":
        int(
            reused_condition_audits
        ),

    "RankingRowsAudited":
        audited_ranking_rows,

    "BuildMetricRowsIndependentlyRecomputed":
        audited_build_metric_rows,

    "ProjectRunRowsIndependentlyRecomputed":
        audited_project_run_rows,

    "ManifestRowsAudited":
        audited_manifest_rows,

    "ConditionMetricMismatches":
        total_condition_metric_mismatches,

    "RankingSignatureMismatches":
        overall_signature_mismatches,

    "ManifestViolations":
        total_manifest_violations,

    "NoiseTechniqueSummaryRows":
        int(
            len(
                noise_technique_summary
            )
        ),

    "SeedDeltaRows":
        int(
            len(
                seed_delta_results
            )
        ),

    "NoiseDeltaSummaryRows":
        int(
            len(
                noise_delta_summary
            )
        ),

    "QTFUniqueSignatures":
        qtf_unique_signatures,

    "RandomNoiseInvarianceViolatingSeeds":
        random_noise_invariance_violations,

    "RandomUniqueSeedSignaturesAt0Percent":
        random_seed_signature_count,

    "LatestFailUniqueSignaturesAt0Percent":
        zero_percent_latest_fail_signatures,

    "NaiveBayesUniqueSignaturesAt0Percent":
        zero_percent_nb_signatures,

    "ZeroPercentDeltaMismatches":
        zero_delta_mismatches,

    "ElapsedSeconds":
        final_audit_elapsed,

    "ConditionAudit":
        str(
            CONDITION_AUDIT_PROGRESS_PATH
        ),

    "ManifestAudit":
        str(
            MANIFEST_AUDIT_PATH
        ),

    "SeedLevelResults":
        str(
            SEED_LEVEL_RESULTS_PATH
        ),

    "SeedDeltaResults":
        str(
            SEED_DELTA_PATH
        ),

    "NoiseTechniqueSummary":
        str(
            NOISE_TECHNIQUE_SUMMARY_PATH
        ),

    "NoiseDeltaSummary":
        str(
            NOISE_DELTA_SUMMARY_PATH
        ),

    "BaselineInvarianceAudit":
        str(
            BASELINE_INVARIANCE_PATH
        ),

    "CompletedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
}


with open(
    FINAL_AUDIT_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        final_audit_report,
        report_file,
        indent=2,
        default=str,
    )


if FINAL_AUDIT_ERROR_PATH.exists():
    FINAL_AUDIT_ERROR_PATH.unlink()


# ---------------------------------------------------------
# 21. Update Project 6 checkpoint
# ---------------------------------------------------------

with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint = json.load(
        checkpoint_file
    )


checkpoint.update({
    "Status":
        "FINAL_AUDIT_PASSED",

    "FinalAuditDirectory":
        str(
            FINAL_AUDIT_DIRECTORY
        ),

    "FinalAuditReport":
        str(
            FINAL_AUDIT_REPORT_PATH
        ),

    "FinalAuditConditions":
        270,

    "FinalAuditRankingRows":
        audited_ranking_rows,

    "FinalAuditBuildMetricRows":
        audited_build_metric_rows,

    "FinalAuditProjectRunRows":
        audited_project_run_rows,

    "FinalAuditManifestRows":
        audited_manifest_rows,

    "FinalAuditConditionMismatches":
        total_condition_metric_mismatches,

    "FinalAuditRankingSignatureMismatches":
        overall_signature_mismatches,

    "FinalAuditManifestViolations":
        total_manifest_violations,

    "SeedLevelResults":
        str(
            SEED_LEVEL_RESULTS_PATH
        ),

    "SeedDeltaResults":
        str(
            SEED_DELTA_PATH
        ),

    "NoiseTechniqueSummary":
        str(
            NOISE_TECHNIQUE_SUMMARY_PATH
        ),

    "NoiseDeltaSummary":
        str(
            NOISE_DELTA_SUMMARY_PATH
        ),

    "FinalAuditCurrentCondition":
        None,

    "UpdatedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
})


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 22. Compact final output
# ---------------------------------------------------------

mean_apfdc_pivot = (
    noise_technique_summary
    .pivot(
        index="NoisePercent",
        columns="Technique",
        values="MeanAPFDc",
    )
    .reindex(
        index=NOISE_LEVELS
    )
)


mean_delta_apfdc_pivot = (
    noise_delta_summary
    .pivot(
        index="NoisePercent",
        columns="Technique",
        values="MeanDeltaAPFDc",
    )
    .reindex(
        index=NOISE_LEVELS
    )
)


clear_output(
    wait=True
)


print(
    "=== PROJECT 6 STEP 11 RESULT ==="
)


print(
    "\nIndependent audit coverage:"
)

print(
    "Conditions audited:",
    len(
        condition_audit
    )
)

print(
    "Ranking rows audited:",
    audited_ranking_rows
)

print(
    "Build metrics independently recomputed:",
    audited_build_metric_rows
)

print(
    "Project-run rows independently recomputed:",
    audited_project_run_rows
)

print(
    "Noise-manifest rows audited:",
    audited_manifest_rows
)


print(
    "\nAudit violations:"
)

print(
    "Condition metric/ranking violations:",
    total_condition_metric_mismatches
)

print(
    "Ranking signature mismatches:",
    overall_signature_mismatches
)

print(
    "Manifest violations:",
    total_manifest_violations
)

print(
    "0% paired-delta mismatches:",
    zero_delta_mismatches
)


print(
    "\nInvariance audit:"
)

print(
    "QTF-Avg unique signatures across all conditions:",
    qtf_unique_signatures
)

print(
    "Random noise-invariance violating seeds:",
    random_noise_invariance_violations
)

print(
    "Random unique seed signatures at 0%:",
    random_seed_signature_count
)

print(
    "LatestFail unique signatures at 0%:",
    zero_percent_latest_fail_signatures
)

print(
    "Naive Bayes unique signatures at 0%:",
    zero_percent_nb_signatures
)


print(
    "\nMean APFDc across 30 seeds:"
)

display(
    mean_apfdc_pivot.round(
        6
    )
)


print(
    "\nMean paired APFDc change from the same seed's 0% "
    "condition:"
)

display(
    mean_delta_apfdc_pivot.round(
        6
    )
)


print(
    "\nFinal aggregation files:"
)

print(
    "Seed-level results:"
)

print(
    SEED_LEVEL_RESULTS_PATH
)

print(
    "Seed-level paired deltas:"
)

print(
    SEED_DELTA_PATH
)

print(
    "Noise × technique summary:"
)

print(
    NOISE_TECHNIQUE_SUMMARY_PATH
)

print(
    "Noise-delta summary:"
)

print(
    NOISE_DELTA_SUMMARY_PATH
)


print(
    "\nFinal audit report:"
)

print(
    FINAL_AUDIT_REPORT_PATH
)


print(
    "\nElapsed:",
    f"{final_audit_elapsed / 60.0:.1f} minutes"
)


print(
    "\nValidation status:",
    "PASS"
)


print(
    "\nSUCCESS: All 270 conditions passed the "
    "independent ranking audit."
)

print(
    "SUCCESS: All 71,820 build-level APFD/APFDc rows "
    "were independently recomputed with zero mismatches."
)

print(
    "SUCCESS: All 1,890 project-run rows were "
    "independently recomputed with zero mismatches."
)

print(
    "SUCCESS: All 5,358,150 noise-manifest rows passed "
    "the full nested-mask and flip-rule audit."
)

print(
    "SUCCESS: Random, QTF-Avg, LatestFail and "
    "Naive Bayes invariance rules passed."
)

print(
    "SUCCESS: The 30 seeds were aggregated and paired "
    "against their same-seed 0% references."
)

print(
    "SUCCESS: Project 6 is ready for cryptographic "
    "freezing and completion-registry update."
)

=== PROJECT 6 STEP 11 RESULT ===

Independent audit coverage:
Conditions audited: 270
Ranking rows audited: 7709310
Build metrics independently recomputed: 71820
Project-run rows independently recomputed: 1890
Noise-manifest rows audited: 5358150

Audit violations:
Condition metric/ranking violations: 0
Ranking signature mismatches: 0
Manifest violations: 0
0% paired-delta mismatches: 0

Invariance audit:
QTF-Avg unique signatures across all conditions: 1
Random noise-invariance violating seeds: 0
Random unique seed signatures at 0%: 30
LatestFail unique signatures at 0%: 1
Naive Bayes unique signatures at 0%: 1

Mean APFDc across 30 seeds:


Technique,LatestFail,LightGBM,NaiveBayes,QTF-Avg,Random,RandomForest,XGBoost
NoisePercent,,,,,,,
0,0.958470,0.955000,0.846398,0.491108,0.501367,0.960967,0.970231
5,0.938848,0.901638,0.600220,0.491108,0.501367,0.897997,0.915787
10,0.937786,0.864667,0.570394,0.491108,0.501367,0.761828,0.870935
15,0.937316,0.743891,0.541511,0.491108,0.501367,0.634345,0.745555
20,0.935626,0.692457,0.530727,0.491108,0.501367,0.593444,0.637404
25,0.933657,0.590747,0.519454,0.491108,0.501367,0.472817,0.576444
30,0.933148,0.507488,0.470211,0.491108,0.501367,0.450414,0.482896
40,0.931312,0.414737,0.473816,0.491108,0.501367,0.331665,0.378754
50,0.930246,0.333864,0.504637,0.491108,0.501367,0.246900,0.312207



Mean paired APFDc change from the same seed's 0% condition:


Technique,LatestFail,LightGBM,NaiveBayes,QTF-Avg,Random,RandomForest,XGBoost
NoisePercent,,,,,,,
0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000
5,-0.019622,-0.053362,-0.246178,0.0,0.0,-0.062970,-0.054445
10,-0.020684,-0.090332,-0.276005,0.0,0.0,-0.199139,-0.099297
15,-0.021154,-0.211109,-0.304888,0.0,0.0,-0.326622,-0.224676
20,-0.022844,-0.262543,-0.315672,0.0,0.0,-0.367524,-0.332827
25,-0.024813,-0.364253,-0.326944,0.0,0.0,-0.488151,-0.393787
30,-0.025322,-0.447511,-0.376187,0.0,0.0,-0.510553,-0.487335
40,-0.027158,-0.540263,-0.372583,0.0,0.0,-0.629303,-0.591478
50,-0.028224,-0.621136,-0.341761,0.0,0.0,-0.714067,-0.658024



Final aggregation files:
Seed-level results:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/eclipse__jetty.project/jetty_30_seed_final_audit/jetty_seed_level_project_run_metrics.csv
Seed-level paired deltas:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/eclipse__jetty.project/jetty_30_seed_final_audit/jetty_seed_delta_from_0pct.csv
Noise × technique summary:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/eclipse__jetty.project/jetty_30_seed_final_audit/jetty_noise_technique_summary.csv
Noise-delta summary:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/eclipse__jetty.project/jetty_30_seed_final_audit/jetty_noise_delta_summary.csv

Final audit report:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/eclipse__jetty.project/jetty_30_seed_final_audit/jetty_final_audit_report.json

Elapsed: 5.3 minutes

Validation status: PASS

SUCCESS: All 270 conditions passed the independent ranking audit.
SUCCESS: All 71,820 build-level AP

In [ ]:
# =========================================================
# PROJECT 6 — STEP 12
# CRYPTOGRAPHIC FREEZE AND COMPLETION REGISTRY UPDATE
#
# This cell:
#   - validates the successful full run and final audit
#   - packages the compact Project 6 artefacts
#   - hashes every raw Project 6 condition file
#   - creates SHA-256 manifests and root digests
#   - verifies the final package after creation
#   - marks Project 6 COMPLETE_AND_FROZEN
#   - updates the permanent completion registry
#
# It does NOT fit models or recompute rankings.
# Do not rerun Steps 10 or 11.
# =========================================================

from pathlib import Path
import gc
import hashlib
import json
import os
import shutil
import time

import numpy as np
import pandas as pd
from IPython.display import clear_output, display


print(
    "=== PROJECT 6 STEP 12: "
    "CRYPTOGRAPHIC FREEZE ==="
)


# ---------------------------------------------------------
# 1. Confirm required Project 6 state
# ---------------------------------------------------------

required_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",

    "THESIS_DRIVE",
    "NOTES_DRIVE",

    "PROJECT_RAW_RESULTS",
    "PROJECT_AGGREGATED_RESULTS",
    "PROJECT_6_SELECTION_CHECKPOINT",
]


missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required Project 6 objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nReconnect the runtime and run only the "
        "Project 6 recovery cell. Do not rerun the "
        "270-condition experiment."
    )


if PROJECT_NUMBER != 6:
    raise AssertionError(
        f"Expected Project 6, observed {PROJECT_NUMBER}."
    )


if PROJECT_NAME != "eclipse@jetty.project":
    raise AssertionError(
        f"Unexpected project: {PROJECT_NAME}"
    )


if PROJECT_SLUG != "eclipse__jetty.project":
    raise AssertionError(
        f"Unexpected project slug: {PROJECT_SLUG}"
    )


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    pre_freeze_checkpoint = json.load(
        checkpoint_file
    )


if pre_freeze_checkpoint.get(
    "Status"
) != "FINAL_AUDIT_PASSED":
    raise AssertionError(
        "Project 6 has not reached the required "
        "FINAL_AUDIT_PASSED state.\n"
        f"Observed status: "
        f"{pre_freeze_checkpoint.get('Status')}"
    )


# ---------------------------------------------------------
# 2. Permanent source and destination paths
# ---------------------------------------------------------

PREFLIGHT_DIRECTORY = (
    PROJECT_AGGREGATED_RESULTS
    / "jetty_preflight"
)


HELPER_DIRECTORY = (
    PROJECT_AGGREGATED_RESULTS
    / "jetty_helper_validation"
)


RUNNER_DIRECTORY = (
    PROJECT_AGGREGATED_RESULTS
    / "jetty_runner_configuration"
)


SMOKE_DIRECTORY = (
    PROJECT_AGGREGATED_RESULTS
    / "jetty_smoke_0pct_seed01"
)


FULL_RUN_DIRECTORY = (
    PROJECT_AGGREGATED_RESULTS
    / "jetty_30_seed_full_run"
)


FINAL_AUDIT_DIRECTORY = (
    PROJECT_AGGREGATED_RESULTS
    / "jetty_30_seed_final_audit"
)


FINAL_DIRECTORY = (
    PROJECT_AGGREGATED_RESULTS
    / "jetty_30_seed_final"
)


STAGING_DIRECTORY = (
    PROJECT_AGGREGATED_RESULTS
    / "jetty_30_seed_final_staging"
)


FINAL_FROZEN_MARKER = (
    FINAL_DIRECTORY
    / "_FROZEN.json"
)


FINAL_SUCCESS_MARKER = (
    FINAL_DIRECTORY
    / "_SUCCESS.json"
)


COMPLETION_REGISTRY_PATH = (
    NOTES_DRIVE
    / "completed_project_registry.csv"
)


FULL_RUN_REPORT_PATH = (
    FULL_RUN_DIRECTORY
    / "jetty_full_run_execution_report.json"
)


FINAL_AUDIT_REPORT_PATH = (
    FINAL_AUDIT_DIRECTORY
    / "jetty_final_audit_report.json"
)


SEED_LEVEL_RESULTS_SOURCE = (
    FINAL_AUDIT_DIRECTORY
    / "jetty_seed_level_project_run_metrics.csv"
)


SEED_DELTA_SOURCE = (
    FINAL_AUDIT_DIRECTORY
    / "jetty_seed_delta_from_0pct.csv"
)


NOISE_TECHNIQUE_SUMMARY_SOURCE = (
    FINAL_AUDIT_DIRECTORY
    / "jetty_noise_technique_summary.csv"
)


NOISE_DELTA_SUMMARY_SOURCE = (
    FINAL_AUDIT_DIRECTORY
    / "jetty_noise_delta_summary.csv"
)


# ---------------------------------------------------------
# 3. Refuse to overwrite an existing frozen project
# ---------------------------------------------------------

if FINAL_FROZEN_MARKER.exists():
    with open(
        FINAL_FROZEN_MARKER,
        "r",
        encoding="utf-8",
    ) as marker_file:
        existing_freeze = json.load(
            marker_file
        )


    print(
        "\nProject 6 is already frozen."
    )

    print(
        "Final directory:",
        FINAL_DIRECTORY
    )

    print(
        "Frozen at:",
        existing_freeze.get(
            "FrozenAtUTC"
        )
    )

    print(
        "Package root SHA-256:",
        existing_freeze.get(
            "FinalPackageRootSHA256"
        )
    )

    raise RuntimeError(
        "The frozen Project 6 package already exists. "
        "Do not rerun Step 12."
    )


# Remove only an incomplete Step 12 staging/final directory.
if STAGING_DIRECTORY.exists():
    shutil.rmtree(
        STAGING_DIRECTORY
    )


if FINAL_DIRECTORY.exists():
    shutil.rmtree(
        FINAL_DIRECTORY
    )


STAGING_DIRECTORY.mkdir(
    parents=True,
    exist_ok=False,
)


# ---------------------------------------------------------
# 4. Validate required source artefacts
# ---------------------------------------------------------

required_directories = [
    PROJECT_RAW_RESULTS,
    PREFLIGHT_DIRECTORY,
    HELPER_DIRECTORY,
    RUNNER_DIRECTORY,
    SMOKE_DIRECTORY,
    FULL_RUN_DIRECTORY,
    FINAL_AUDIT_DIRECTORY,
]


missing_directories = [
    str(directory)
    for directory in required_directories
    if not directory.exists()
]


if missing_directories:
    raise FileNotFoundError(
        "Required Project 6 directories are missing:\n"
        + "\n".join(
            missing_directories
        )
    )


required_files = [
    PROJECT_6_SELECTION_CHECKPOINT,
    FULL_RUN_REPORT_PATH,
    FINAL_AUDIT_REPORT_PATH,
    SEED_LEVEL_RESULTS_SOURCE,
    SEED_DELTA_SOURCE,
    NOISE_TECHNIQUE_SUMMARY_SOURCE,
    NOISE_DELTA_SUMMARY_SOURCE,
]


missing_files = [
    str(file_path)
    for file_path in required_files
    if not file_path.exists()
]


if missing_files:
    raise FileNotFoundError(
        "Required Project 6 files are missing:\n"
        + "\n".join(
            missing_files
        )
    )


# ---------------------------------------------------------
# 5. Validate Step 10 and Step 11 reports
# ---------------------------------------------------------

with open(
    FULL_RUN_REPORT_PATH,
    "r",
    encoding="utf-8",
) as report_file:
    full_run_report = json.load(
        report_file
    )


with open(
    FINAL_AUDIT_REPORT_PATH,
    "r",
    encoding="utf-8",
) as report_file:
    final_audit_report = json.load(
        report_file
    )


if full_run_report.get(
    "Status"
) != "PASS":
    raise AssertionError(
        "The Step 10 full-run report is not PASS."
    )


if final_audit_report.get(
    "Status"
) != "PASS":
    raise AssertionError(
        "The Step 11 final-audit report is not PASS."
    )


expected_full_run_values = {
    "SuccessfulConditions":
        270,

    "ObservedModelFits":
        1080,

    "ObservedRankingRows":
        7709310,

    "ObservedBuildMetricRows":
        71820,

    "ObservedProjectRunRows":
        1890,
}


for field_name, expected_value in (
    expected_full_run_values.items()
):
    observed_value = full_run_report.get(
        field_name
    )


    if int(
        observed_value
    ) != int(
        expected_value
    ):
        raise AssertionError(
            "Unexpected full-run report value.\n"
            f"Field: {field_name}\n"
            f"Expected: {expected_value}\n"
            f"Observed: {observed_value}"
        )


expected_audit_values = {
    "ConditionsAudited":
        270,

    "RankingRowsAudited":
        7709310,

    "BuildMetricRowsIndependentlyRecomputed":
        71820,

    "ProjectRunRowsIndependentlyRecomputed":
        1890,

    "ManifestRowsAudited":
        5358150,

    "ConditionMetricMismatches":
        0,

    "RankingSignatureMismatches":
        0,

    "ManifestViolations":
        0,

    "ZeroPercentDeltaMismatches":
        0,
}


for field_name, expected_value in (
    expected_audit_values.items()
):
    observed_value = final_audit_report.get(
        field_name
    )


    if int(
        observed_value
    ) != int(
        expected_value
    ):
        raise AssertionError(
            "Unexpected final-audit report value.\n"
            f"Field: {field_name}\n"
            f"Expected: {expected_value}\n"
            f"Observed: {observed_value}"
        )


# ---------------------------------------------------------
# 6. Validate final compact result tables
# ---------------------------------------------------------

seed_level_results = pd.read_csv(
    SEED_LEVEL_RESULTS_SOURCE
)


seed_delta_results = pd.read_csv(
    SEED_DELTA_SOURCE
)


noise_technique_summary = pd.read_csv(
    NOISE_TECHNIQUE_SUMMARY_SOURCE
)


noise_delta_summary = pd.read_csv(
    NOISE_DELTA_SUMMARY_SOURCE
)


if len(seed_level_results) != 1890:
    raise AssertionError(
        "Expected 1,890 seed-level result rows."
    )


if len(seed_delta_results) != 1890:
    raise AssertionError(
        "Expected 1,890 seed-level delta rows."
    )


if len(noise_technique_summary) != 63:
    raise AssertionError(
        "Expected 63 noise-technique summary rows."
    )


if len(noise_delta_summary) != 63:
    raise AssertionError(
        "Expected 63 noise-delta summary rows."
    )


if seed_level_results.duplicated(
    subset=[
        "NoisePercent",
        "RepetitionSeed",
        "Technique",
    ]
).any():
    raise AssertionError(
        "Seed-level results contain duplicate keys."
    )


# ---------------------------------------------------------
# 7. Confirm all 270 raw condition markers
# ---------------------------------------------------------

condition_success_markers = sorted(
    PROJECT_RAW_RESULTS.rglob(
        "_SUCCESS.json"
    )
)


if len(condition_success_markers) != 270:
    raise AssertionError(
        "Expected exactly 270 raw condition success "
        "markers.\n"
        f"Observed: {len(condition_success_markers)}"
    )


condition_pairs_from_markers = set()


for marker_path in condition_success_markers:
    with open(
        marker_path,
        "r",
        encoding="utf-8",
    ) as marker_file:
        marker = json.load(
            marker_file
        )


    if not marker.get(
        "CompletedSuccessfully",
        False,
    ):
        raise AssertionError(
            "A raw condition success marker is not "
            "marked successful:\n"
            f"{marker_path}"
        )


    condition_pairs_from_markers.add(
        (
            int(
                marker[
                    "NoisePercent"
                ]
            ),
            int(
                marker[
                    "RepetitionSeed"
                ]
            ),
        )
    )


expected_condition_pairs = {
    (
        noise_percent,
        repetition_seed,
    )
    for noise_percent in [
        0,
        5,
        10,
        15,
        20,
        25,
        30,
        40,
        50,
    ]
    for repetition_seed in range(
        1,
        31,
    )
}


if condition_pairs_from_markers != (
    expected_condition_pairs
):
    raise AssertionError(
        "The 270 raw condition markers do not match "
        "the frozen experimental design."
    )


# ---------------------------------------------------------
# 8. SHA-256 helper functions
# ---------------------------------------------------------

HASH_CHUNK_SIZE = (
    16
    * 1024
    * 1024
)


def sha256_file(
    file_path,
):
    digest = hashlib.sha256()


    with open(
        file_path,
        "rb",
    ) as binary_file:
        while True:
            chunk = binary_file.read(
                HASH_CHUNK_SIZE
            )


            if not chunk:
                break


            digest.update(
                chunk
            )


    return digest.hexdigest()


def create_file_snapshot(
    root_directory,
    excluded_relative_paths=None,
):
    if excluded_relative_paths is None:
        excluded_relative_paths = set()


    snapshot = {}


    for file_path in sorted(
        root_directory.rglob("*")
    ):
        if not file_path.is_file():
            continue


        relative_path = (
            file_path
            .relative_to(
                root_directory
            )
            .as_posix()
        )


        if relative_path in (
            excluded_relative_paths
        ):
            continue


        file_stat = file_path.stat()


        snapshot[
            relative_path
        ] = {
            "SizeBytes":
                int(
                    file_stat.st_size
                ),

            "ModifiedTimeNanoseconds":
                int(
                    file_stat.st_mtime_ns
                ),
        }


    return snapshot


def create_sha256_manifest(
    root_directory,
    excluded_relative_paths=None,
    progress_title=None,
):
    if excluded_relative_paths is None:
        excluded_relative_paths = set()


    file_paths = []


    for file_path in sorted(
        root_directory.rglob("*")
    ):
        if not file_path.is_file():
            continue


        relative_path = (
            file_path
            .relative_to(
                root_directory
            )
            .as_posix()
        )


        if relative_path in (
            excluded_relative_paths
        ):
            continue


        file_paths.append(
            file_path
        )


    manifest_records = []
    total_files = len(
        file_paths
    )


    for file_number, file_path in enumerate(
        file_paths,
        start=1,
    ):
        relative_path = (
            file_path
            .relative_to(
                root_directory
            )
            .as_posix()
        )


        file_stat = file_path.stat()


        manifest_records.append({
            "RelativePath":
                relative_path,

            "SizeBytes":
                int(
                    file_stat.st_size
                ),

            "ModifiedTimeUTC":
                pd.Timestamp(
                    file_stat.st_mtime,
                    unit="s",
                    tz="UTC",
                ).isoformat(),

            "SHA256":
                sha256_file(
                    file_path
                ),
        })


        if (
            file_number == 1
            or file_number % 100 == 0
            or file_number == total_files
        ):
            clear_output(
                wait=True
            )


            print(
                "=== PROJECT 6 STEP 12: "
                "CRYPTOGRAPHIC FREEZE ==="
            )


            print(
                "\n",
                progress_title
                if progress_title is not None
                else "Hashing files",
                sep="",
            )


            print(
                "Files hashed:",
                f"{file_number}/{total_files}"
            )


            print(
                "Current file:"
            )

            print(
                relative_path
            )


    manifest = pd.DataFrame(
        manifest_records
    )


    if len(manifest) > 0:
        manifest = (
            manifest
            .sort_values(
                "RelativePath",
                kind="mergesort",
            )
            .reset_index(drop=True)
        )


    return manifest


def calculate_manifest_root(
    manifest,
):
    digest = hashlib.sha256()


    manifest_sorted = (
        manifest
        .sort_values(
            "RelativePath",
            kind="mergesort",
        )
    )


    for manifest_row in (
        manifest_sorted.itertuples(
            index=False
        )
    ):
        canonical_line = (
            f"{manifest_row.RelativePath}|"
            f"{int(manifest_row.SizeBytes)}|"
            f"{manifest_row.SHA256}\n"
        )


        digest.update(
            canonical_line.encode(
                "utf-8"
            )
        )


    return digest.hexdigest()


# ---------------------------------------------------------
# 9. Copy compact validated artefacts to staging
# ---------------------------------------------------------

ARTIFACTS_DIRECTORY = (
    STAGING_DIRECTORY
    / "Artifacts"
)


NOTES_SNAPSHOT_DIRECTORY = (
    STAGING_DIRECTORY
    / "Notes"
)


CHECKPOINT_SNAPSHOT_DIRECTORY = (
    STAGING_DIRECTORY
    / "Checkpoint"
)


ARTIFACTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


NOTES_SNAPSHOT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


CHECKPOINT_SNAPSHOT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


directory_copy_plan = [
    (
        PREFLIGHT_DIRECTORY,
        ARTIFACTS_DIRECTORY
        / "jetty_preflight",
    ),
    (
        HELPER_DIRECTORY,
        ARTIFACTS_DIRECTORY
        / "jetty_helper_validation",
    ),
    (
        RUNNER_DIRECTORY,
        ARTIFACTS_DIRECTORY
        / "jetty_runner_configuration",
    ),
    (
        SMOKE_DIRECTORY,
        ARTIFACTS_DIRECTORY
        / "jetty_smoke_0pct_seed01",
    ),
    (
        FULL_RUN_DIRECTORY,
        ARTIFACTS_DIRECTORY
        / "jetty_30_seed_full_run",
    ),
    (
        FINAL_AUDIT_DIRECTORY,
        ARTIFACTS_DIRECTORY
        / "jetty_30_seed_final_audit",
    ),
]


for source_directory, target_directory in (
    directory_copy_plan
):
    shutil.copytree(
        source_directory,
        target_directory,
    )


# This is deliberately labelled as the pre-freeze
# checkpoint because the live checkpoint is updated only
# after package verification succeeds.
shutil.copy2(
    PROJECT_6_SELECTION_CHECKPOINT,
    CHECKPOINT_SNAPSHOT_DIRECTORY
    / "project_06_pre_freeze_checkpoint.json",
)


note_file_names = [
    "project_06_experiment_protocol.json",
    "project_06_jetty_model_configuration.json",
    "methodology_checkpoint_fixed_holdout_v1.json",
    "methodology_checkpoint_fixed_holdout_v1.md",
    "apfd_apfdc_formula_audit_v1.csv",
    "apfd_apfdc_formula_audit_v1.json",
]


copied_note_files = []


for note_file_name in note_file_names:
    source_note = (
        NOTES_DRIVE
        / note_file_name
    )


    if source_note.exists():
        target_note = (
            NOTES_SNAPSHOT_DIRECTORY
            / note_file_name
        )


        shutil.copy2(
            source_note,
            target_note,
        )


        copied_note_files.append(
            note_file_name
        )


# ---------------------------------------------------------
# 10. Create concise frozen result snapshot
# ---------------------------------------------------------

result_snapshot = (
    noise_technique_summary
    .merge(
        noise_delta_summary[
            [
                "NoisePercent",
                "Technique",
                "MeanDeltaAPFD",
                "SD_DeltaAPFD",
                "CI95Low_DeltaAPFD",
                "CI95High_DeltaAPFD",
                "MeanDeltaAPFDc",
                "SD_DeltaAPFDc",
                "CI95Low_DeltaAPFDc",
                "CI95High_DeltaAPFDc",
            ]
        ],
        on=[
            "NoisePercent",
            "Technique",
        ],
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


RESULT_SNAPSHOT_PATH = (
    STAGING_DIRECTORY
    / "jetty_key_result_snapshot.csv"
)


result_snapshot.to_csv(
    RESULT_SNAPSHOT_PATH,
    index=False,
)


# ---------------------------------------------------------
# 11. Create permanent freeze README
# ---------------------------------------------------------

FREEZE_README_PATH = (
    STAGING_DIRECTORY
    / "README_FREEZE.md"
)


freeze_readme = f"""# Project 6 — Eclipse Jetty

## Status

`COMPLETE_AND_FROZEN`

Project: `{PROJECT_NAME}`
Slug: `{PROJECT_SLUG}`

## Experiment

- 192 chronological builds
- 144 training builds
- 48 evaluation-period builds
- 38 failing model-ready evaluation builds
- 9 training-verdict noise levels
- 30 repetition seeds
- 270 project × noise × seed conditions
- 4 machine-learning techniques
- 3 baselines
- 1,080 machine-learning fits
- 7,709,310 saved ranking rows
- 71,820 build-level metric rows
- 1,890 project-run rows
- 5,358,150 independently audited noise-manifest rows

## Validation

All saved rankings, APFD/APFDc metrics, project-run
aggregates, noise masks, subtype rules and invariance rules
passed the independent final audit with zero mismatches.

## REC chronology and reconstruction

Equal timestamps are ordered by build ID descending.

One upstream five-value anomaly was isolated to training
Build 6843283, Test 431. No execution was inserted or
removed. Verdict-dependent REC values use the validated
clean-anchored delta reconstruction. The six
noise-independent REC features preserve the original
TCP-CI values.

## Cryptographic freeze

The raw Project 6 condition directory is not duplicated
inside this package. Every raw file is listed and hashed in:

`jetty_raw_results_sha256_manifest.csv`

The compact frozen package is listed and hashed in:

`jetty_final_package_sha256_manifest.csv`

The manifest root hashes are recorded in:

`jetty_freeze_record.json`

This is one project-level experimental result. It must not
be treated alone as the cross-project thesis conclusion.
"""


with open(
    FREEZE_README_PATH,
    "w",
    encoding="utf-8",
) as readme_file:
    readme_file.write(
        freeze_readme
    )


# ---------------------------------------------------------
# 12. Hash the complete raw Project 6 result tree
# ---------------------------------------------------------

raw_snapshot_before = (
    create_file_snapshot(
        PROJECT_RAW_RESULTS
    )
)


raw_results_manifest = (
    create_sha256_manifest(
        root_directory=(
            PROJECT_RAW_RESULTS
        ),
        excluded_relative_paths=set(),
        progress_title=(
            "Hashing the complete raw Project 6 "
            "condition tree"
        ),
    )
)


RAW_RESULTS_MANIFEST_STAGING_PATH = (
    STAGING_DIRECTORY
    / "jetty_raw_results_sha256_manifest.csv"
)


raw_results_manifest.to_csv(
    RAW_RESULTS_MANIFEST_STAGING_PATH,
    index=False,
)


raw_snapshot_after = (
    create_file_snapshot(
        PROJECT_RAW_RESULTS
    )
)


if raw_snapshot_before != raw_snapshot_after:
    raise AssertionError(
        "The raw Project 6 result tree changed while "
        "its cryptographic manifest was being created."
    )


RAW_FILE_COUNT = int(
    len(
        raw_results_manifest
    )
)


RAW_TOTAL_BYTES = int(
    raw_results_manifest[
        "SizeBytes"
    ].sum()
)


RAW_RESULTS_ROOT_SHA256 = (
    calculate_manifest_root(
        raw_results_manifest
    )
)


RAW_RESULTS_MANIFEST_SHA256 = (
    sha256_file(
        RAW_RESULTS_MANIFEST_STAGING_PATH
    )
)


if RAW_FILE_COUNT == 0:
    raise AssertionError(
        "The raw-results manifest is empty."
    )


# ---------------------------------------------------------
# 13. Hash the compact frozen package
# ---------------------------------------------------------

package_exclusions = {
    "jetty_final_package_sha256_manifest.csv",
    "jetty_freeze_record.json",
    "_FROZEN.json",
    "_SUCCESS.json",
}


final_package_manifest = (
    create_sha256_manifest(
        root_directory=(
            STAGING_DIRECTORY
        ),
        excluded_relative_paths=(
            package_exclusions
        ),
        progress_title=(
            "Hashing the compact frozen Project 6 "
            "package"
        ),
    )
)


FINAL_PACKAGE_MANIFEST_STAGING_PATH = (
    STAGING_DIRECTORY
    / "jetty_final_package_sha256_manifest.csv"
)


final_package_manifest.to_csv(
    FINAL_PACKAGE_MANIFEST_STAGING_PATH,
    index=False,
)


FINAL_PACKAGE_FILE_COUNT = int(
    len(
        final_package_manifest
    )
)


FINAL_PACKAGE_TOTAL_BYTES = int(
    final_package_manifest[
        "SizeBytes"
    ].sum()
)


FINAL_PACKAGE_ROOT_SHA256 = (
    calculate_manifest_root(
        final_package_manifest
    )
)


FINAL_PACKAGE_MANIFEST_SHA256 = (
    sha256_file(
        FINAL_PACKAGE_MANIFEST_STAGING_PATH
    )
)


if FINAL_PACKAGE_FILE_COUNT == 0:
    raise AssertionError(
        "The final-package manifest is empty."
    )


# ---------------------------------------------------------
# 14. Create the permanent freeze record
# ---------------------------------------------------------

freeze_timestamp = (
    pd.Timestamp.utcnow().isoformat()
)


FINAL_RAW_RESULTS_MANIFEST_PATH = (
    FINAL_DIRECTORY
    / "jetty_raw_results_sha256_manifest.csv"
)


FINAL_PACKAGE_MANIFEST_PATH = (
    FINAL_DIRECTORY
    / "jetty_final_package_sha256_manifest.csv"
)


FINAL_FREEZE_RECORD_PATH = (
    FINAL_DIRECTORY
    / "jetty_freeze_record.json"
)


freeze_record = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "COMPLETE_AND_FROZEN",

    "FrozenAtUTC":
        freeze_timestamp,

    "ExperimentalDesign": {
        "NoiseLevels":
            [
                0,
                5,
                10,
                15,
                20,
                25,
                30,
                40,
                50,
            ],

        "RepetitionSeeds":
            list(
                range(
                    1,
                    31,
                )
            ),

        "Conditions":
            270,

        "MLModels":
            [
                "RandomForest",
                "XGBoost",
                "LightGBM",
                "NaiveBayes",
            ],

        "Baselines":
            [
                "Random",
                "LatestFail",
                "QTF-Avg",
            ],
    },

    "ValidatedTotals": {
        "ModelFits":
            1080,

        "RankingRows":
            7709310,

        "BuildMetricRows":
            71820,

        "ProjectRunRows":
            1890,

        "NoiseManifestRows":
            5358150,

        "IndependentAuditMismatches":
            0,

        "ManifestViolations":
            0,
    },

    "RawResults": {
        "Directory":
            str(
                PROJECT_RAW_RESULTS
            ),

        "FileCount":
            RAW_FILE_COUNT,

        "TotalBytes":
            RAW_TOTAL_BYTES,

        "Manifest":
            str(
                FINAL_RAW_RESULTS_MANIFEST_PATH
            ),

        "ManifestFileSHA256":
            RAW_RESULTS_MANIFEST_SHA256,

        "ManifestRootSHA256":
            RAW_RESULTS_ROOT_SHA256,

        "TreeStableDuringHashing":
            True,
    },

    "FinalPackage": {
        "Directory":
            str(
                FINAL_DIRECTORY
            ),

        "ManifestedFileCount":
            FINAL_PACKAGE_FILE_COUNT,

        "ManifestedTotalBytes":
            FINAL_PACKAGE_TOTAL_BYTES,

        "Manifest":
            str(
                FINAL_PACKAGE_MANIFEST_PATH
            ),

        "ManifestFileSHA256":
            FINAL_PACKAGE_MANIFEST_SHA256,

        "ManifestRootSHA256":
            FINAL_PACKAGE_ROOT_SHA256,
    },

    "ValidationReports": {
        "FullRunReport":
            str(
                FINAL_DIRECTORY
                / "Artifacts"
                / "jetty_30_seed_full_run"
                / "jetty_full_run_execution_report.json"
            ),

        "FinalAuditReport":
            str(
                FINAL_DIRECTORY
                / "Artifacts"
                / "jetty_30_seed_final_audit"
                / "jetty_final_audit_report.json"
            ),

        "CleanRECValidation":
            str(
                FINAL_DIRECTORY
                / "Artifacts"
                / "jetty_preflight"
                / "jetty_clean_rec_validation_report.json"
            ),
    },

    "RECPolicy": {
        "Chronology":
            (
                "started_at ascending; build ID "
                "descending within equal timestamps"
            ),

        "ValidationMode":
            "CLEAN_ANCHORED_DELTA_RECONSTRUCTION",

        "AffectedBuild":
            6843283,

        "AffectedTest":
            "431",

        "SyntheticExecutionsCreated":
            0,

        "RawExecutionsExcluded":
            0,
    },

    "CompletionRegistry":
        str(
            COMPLETION_REGISTRY_PATH
        ),

    "CopiedMethodologyFiles":
        copied_note_files,
}


FREEZE_RECORD_STAGING_PATH = (
    STAGING_DIRECTORY
    / "jetty_freeze_record.json"
)


with open(
    FREEZE_RECORD_STAGING_PATH,
    "w",
    encoding="utf-8",
) as freeze_file:
    json.dump(
        freeze_record,
        freeze_file,
        indent=2,
        default=str,
    )


FREEZE_RECORD_SHA256 = (
    sha256_file(
        FREEZE_RECORD_STAGING_PATH
    )
)


freeze_record[
    "FreezeRecordSHA256BeforeFinalMarker"
] = FREEZE_RECORD_SHA256


with open(
    FREEZE_RECORD_STAGING_PATH,
    "w",
    encoding="utf-8",
) as freeze_file:
    json.dump(
        freeze_record,
        freeze_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 15. Atomically move staging to the final directory
# ---------------------------------------------------------

if FINAL_DIRECTORY.exists():
    raise FileExistsError(
        "Final directory unexpectedly exists before "
        "the staging move."
    )


shutil.move(
    str(
        STAGING_DIRECTORY
    ),
    str(
        FINAL_DIRECTORY
    ),
)


if not FINAL_DIRECTORY.exists():
    raise AssertionError(
        "The final directory was not created."
    )


# ---------------------------------------------------------
# 16. Verify every compact package file after the move
# ---------------------------------------------------------

verified_package_manifest = pd.read_csv(
    FINAL_PACKAGE_MANIFEST_PATH
)


package_verification_records = []


for manifest_row in (
    verified_package_manifest.itertuples(
        index=False
    )
):
    final_file_path = (
        FINAL_DIRECTORY
        / str(
            manifest_row.RelativePath
        )
    )


    file_exists = bool(
        final_file_path.exists()
    )


    observed_size = (
        int(
            final_file_path.stat().st_size
        )
        if file_exists
        else -1
    )


    observed_sha256 = (
        sha256_file(
            final_file_path
        )
        if file_exists
        else ""
    )


    expected_size = int(
        manifest_row.SizeBytes
    )


    expected_sha256 = str(
        manifest_row.SHA256
    )


    package_verification_records.append({
        "RelativePath":
            str(
                manifest_row.RelativePath
            ),

        "Exists":
            file_exists,

        "ExpectedSizeBytes":
            expected_size,

        "ObservedSizeBytes":
            observed_size,

        "ExpectedSHA256":
            expected_sha256,

        "ObservedSHA256":
            observed_sha256,

        "SizeMatches":
            bool(
                observed_size
                == expected_size
            ),

        "SHA256Matches":
            bool(
                observed_sha256
                == expected_sha256
            ),
    })


package_verification = pd.DataFrame(
    package_verification_records
)


package_missing_files = int(
    (
        ~package_verification[
            "Exists"
        ]
    ).sum()
)


package_size_mismatches = int(
    (
        ~package_verification[
            "SizeMatches"
        ]
    ).sum()
)


package_sha256_mismatches = int(
    (
        ~package_verification[
            "SHA256Matches"
        ]
    ).sum()
)


if (
    package_missing_files
    + package_size_mismatches
    + package_sha256_mismatches
    != 0
):
    display(
        package_verification[
            (
                ~package_verification[
                    "Exists"
                ]
            )
            | (
                ~package_verification[
                    "SizeMatches"
                ]
            )
            | (
                ~package_verification[
                    "SHA256Matches"
                ]
            )
        ]
    )


    raise AssertionError(
        "The final compact package failed its "
        "post-move SHA-256 verification."
    )


verified_package_root = (
    calculate_manifest_root(
        verified_package_manifest
    )
)


if verified_package_root != (
    FINAL_PACKAGE_ROOT_SHA256
):
    raise AssertionError(
        "The verified final-package manifest root "
        "does not match the frozen root."
    )


# Confirm the raw tree has not changed since hashing.
raw_snapshot_at_freeze = (
    create_file_snapshot(
        PROJECT_RAW_RESULTS
    )
)


if raw_snapshot_at_freeze != (
    raw_snapshot_after
):
    raise AssertionError(
        "The raw Project 6 result tree changed after "
        "its manifest was created."
    )


# ---------------------------------------------------------
# 17. Update the permanent completion registry
# ---------------------------------------------------------

if COMPLETION_REGISTRY_PATH.exists():
    completion_registry = pd.read_csv(
        COMPLETION_REGISTRY_PATH,
        dtype=str,
    ).fillna("")

else:
    completion_registry = pd.DataFrame()


registry_row = {
    "ProjectNumber":
        str(
            PROJECT_NUMBER
        ),

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "COMPLETE_AND_FROZEN",

    "CompletedAtUTC":
        freeze_timestamp,

    "FinalDirectory":
        str(
            FINAL_DIRECTORY
        ),

    "FreezeRecord":
        str(
            FINAL_FREEZE_RECORD_PATH
        ),

    "RawResultsManifest":
        str(
            FINAL_RAW_RESULTS_MANIFEST_PATH
        ),

    "FinalPackageManifest":
        str(
            FINAL_PACKAGE_MANIFEST_PATH
        ),

    "RawResultsRootSHA256":
        RAW_RESULTS_ROOT_SHA256,

    "FinalPackageRootSHA256":
        FINAL_PACKAGE_ROOT_SHA256,

    "Conditions":
        "270",

    "ModelFits":
        "1080",

    "RankingRows":
        "7709310",

    "BuildMetricRows":
        "71820",

    "ProjectRunRows":
        "1890",

    "ManifestRowsAudited":
        "5358150",

    "FinalAuditStatus":
        "PASS",
}


if len(
    completion_registry
) > 0:
    keep_mask = np.ones(
        len(
            completion_registry
        ),
        dtype=bool,
    )


    if (
        "ProjectNumber"
        in completion_registry.columns
    ):
        keep_mask &= (
            completion_registry[
                "ProjectNumber"
            ].astype(str)
            != str(
                PROJECT_NUMBER
            )
        )


    if (
        "Project"
        in completion_registry.columns
    ):
        keep_mask &= (
            completion_registry[
                "Project"
            ].astype(str)
            != PROJECT_NAME
        )


    if (
        "ProjectSlug"
        in completion_registry.columns
    ):
        keep_mask &= (
            completion_registry[
                "ProjectSlug"
            ].astype(str)
            != PROJECT_SLUG
        )


    completion_registry = (
        completion_registry.loc[
            keep_mask
        ]
        .copy()
        .reset_index(drop=True)
    )


registry_columns = list(
    completion_registry.columns
)


for registry_field in (
    registry_row.keys()
):
    if registry_field not in registry_columns:
        registry_columns.append(
            registry_field
        )


completion_registry = (
    completion_registry
    .reindex(
        columns=registry_columns
    )
    .fillna("")
)


new_registry_row = pd.DataFrame([
    {
        column:
            registry_row.get(
                column,
                "",
            )
        for column in registry_columns
    }
])


completion_registry = pd.concat(
    [
        completion_registry,
        new_registry_row,
    ],
    ignore_index=True,
)


if (
    "ProjectNumber"
    in completion_registry.columns
):
    completion_registry[
        "_ProjectNumberSort"
    ] = pd.to_numeric(
        completion_registry[
            "ProjectNumber"
        ],
        errors="coerce",
    )


    completion_registry = (
        completion_registry
        .sort_values(
            "_ProjectNumberSort",
            kind="mergesort",
            na_position="last",
        )
        .drop(
            columns=[
                "_ProjectNumberSort",
            ]
        )
        .reset_index(drop=True)
    )


completion_registry.to_csv(
    COMPLETION_REGISTRY_PATH,
    index=False,
)


project_6_registry_rows = (
    completion_registry[
        (
            completion_registry[
                "Project"
            ].astype(str)
            == PROJECT_NAME
        )
        & (
            completion_registry[
                "Status"
            ].astype(str)
            == "COMPLETE_AND_FROZEN"
        )
    ]
)


if len(project_6_registry_rows) != 1:
    raise AssertionError(
        "The completion registry does not contain "
        "exactly one frozen Project 6 row."
    )


complete_and_frozen_projects = int(
    (
        completion_registry[
            "Status"
        ].astype(str)
        == "COMPLETE_AND_FROZEN"
    ).sum()
)


# ---------------------------------------------------------
# 18. Update the live Project 6 checkpoint
# ---------------------------------------------------------

with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    final_checkpoint = json.load(
        checkpoint_file
    )


final_checkpoint.update({
    "Status":
        "COMPLETE_AND_FROZEN",

    "FrozenAtUTC":
        freeze_timestamp,

    "FinalDirectory":
        str(
            FINAL_DIRECTORY
        ),

    "FreezeRecord":
        str(
            FINAL_FREEZE_RECORD_PATH
        ),

    "RawResultsManifest":
        str(
            FINAL_RAW_RESULTS_MANIFEST_PATH
        ),

    "FinalPackageManifest":
        str(
            FINAL_PACKAGE_MANIFEST_PATH
        ),

    "RawResultsFileCount":
        RAW_FILE_COUNT,

    "RawResultsTotalBytes":
        RAW_TOTAL_BYTES,

    "RawResultsRootSHA256":
        RAW_RESULTS_ROOT_SHA256,

    "RawResultsManifestSHA256":
        RAW_RESULTS_MANIFEST_SHA256,

    "FinalPackageFileCount":
        FINAL_PACKAGE_FILE_COUNT,

    "FinalPackageTotalBytes":
        FINAL_PACKAGE_TOTAL_BYTES,

    "FinalPackageRootSHA256":
        FINAL_PACKAGE_ROOT_SHA256,

    "FinalPackageManifestSHA256":
        FINAL_PACKAGE_MANIFEST_SHA256,

    "FinalPackageMissingFiles":
        package_missing_files,

    "FinalPackageSizeMismatches":
        package_size_mismatches,

    "FinalPackageSHA256Mismatches":
        package_sha256_mismatches,

    "CompletionRegistry":
        str(
            COMPLETION_REGISTRY_PATH
        ),

    "CompletionRegistryStatus":
        "COMPLETE_AND_FROZEN",

    "UpdatedAtUTC":
        pd.Timestamp.utcnow().isoformat(),
})


with open(
    PROJECT_6_SELECTION_CHECKPOINT,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        final_checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 19. Write final markers last
# ---------------------------------------------------------

freeze_record[
    "CompletionRegistryUpdated"] = True
freeze_record[
    "CompletionRegistryFrozenProjectCount"
] = complete_and_frozen_projects
freeze_record[
    "FinalPackageVerification"] = {
    "MissingFiles":
        package_missing_files,

    "SizeMismatches":
        package_size_mismatches,

    "SHA256Mismatches":
        package_sha256_mismatches,

    "VerifiedManifestRootSHA256":
        verified_package_root,

    "Status":
        "PASS",
}


with open(
    FINAL_FREEZE_RECORD_PATH,
    "w",
    encoding="utf-8",
) as freeze_file:
    json.dump(
        freeze_record,
        freeze_file,
        indent=2,
        default=str,
    )


with open(
    FINAL_FROZEN_MARKER,
    "w",
    encoding="utf-8",
) as marker_file:
    json.dump(
        freeze_record,
        marker_file,
        indent=2,
        default=str,
    )


success_marker = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "COMPLETE_AND_FROZEN",

    "FrozenAtUTC":
        freeze_timestamp,

    "Conditions":
        270,

    "ModelFits":
        1080,

    "RankingRows":
        7709310,

    "BuildMetricRows":
        71820,

    "ProjectRunRows":
        1890,

    "NoiseManifestRowsAudited":
        5358150,

    "RawResultsRootSHA256":
        RAW_RESULTS_ROOT_SHA256,

    "FinalPackageRootSHA256":
        FINAL_PACKAGE_ROOT_SHA256,

    "FinalPackageVerificationStatus":
        "PASS",

    "CompletionRegistryUpdated":
        True,
}


with open(
    FINAL_SUCCESS_MARKER,
    "w",
    encoding="utf-8",
) as marker_file:
    json.dump(
        success_marker,
        marker_file,
        indent=2,
        default=str,
    )


if not FINAL_FROZEN_MARKER.exists():
    raise AssertionError(
        "The final frozen marker was not written."
    )


if not FINAL_SUCCESS_MARKER.exists():
    raise AssertionError(
        "The final success marker was not written."
    )


# ---------------------------------------------------------
# 20. Compact final result
# ---------------------------------------------------------

clear_output(
    wait=True
)


print(
    "=== PROJECT 6 STEP 12 RESULT ==="
)


print(
    "\nProject:"
)

print(
    PROJECT_NAME
)

print(
    "Status:",
    "COMPLETE_AND_FROZEN"
)


print(
    "\nValidated experimental totals:"
)

print(
    "Conditions:",
    270
)

print(
    "ML model fits:",
    1080
)

print(
    "Ranking rows:",
    7709310
)

print(
    "Build-metric rows:",
    71820
)

print(
    "Project-run rows:",
    1890
)

print(
    "Noise-manifest rows audited:",
    5358150
)


print(
    "\nRaw-results cryptographic freeze:"
)

print(
    "Files hashed:",
    RAW_FILE_COUNT
)

print(
    "Total bytes:",
    RAW_TOTAL_BYTES
)

print(
    "Manifest root SHA-256:"
)

print(
    RAW_RESULTS_ROOT_SHA256
)


print(
    "\nFinal-package cryptographic freeze:"
)

print(
    "Manifested files:",
    FINAL_PACKAGE_FILE_COUNT
)

print(
    "Manifested bytes:",
    FINAL_PACKAGE_TOTAL_BYTES
)

print(
    "Manifest root SHA-256:"
)

print(
    FINAL_PACKAGE_ROOT_SHA256
)


print(
    "\nPost-creation package verification:"
)

print(
    "Missing files:",
    package_missing_files
)

print(
    "Size mismatches:",
    package_size_mismatches
)

print(
    "SHA-256 mismatches:",
    package_sha256_mismatches
)


print(
    "\nCompletion registry:"
)

print(
    COMPLETION_REGISTRY_PATH
)

print(
    "Complete-and-frozen projects:",
    complete_and_frozen_projects
)


print(
    "\nFinal Project 6 directory:"
)

print(
    FINAL_DIRECTORY
)


print(
    "\nFreeze record:"
)

print(
    FINAL_FREEZE_RECORD_PATH
)


print(
    "\nFinal status:",
    "PASS"
)


print(
    "\nSUCCESS: All raw Project 6 result files received "
    "SHA-256 hashes."
)

print(
    "SUCCESS: The compact final package passed complete "
    "post-creation SHA-256 verification."
)

print(
    "SUCCESS: The live Project 6 checkpoint is marked "
    "COMPLETE_AND_FROZEN."
)

print(
    "SUCCESS: Project 6 was added to the permanent "
    "completion registry."
)

print(
    "SUCCESS: Project 6 is permanently frozen. "
    "Do not rerun or modify its experimental outputs."
)

=== PROJECT 6 STEP 12 RESULT ===

Project:
eclipse@jetty.project
Status: COMPLETE_AND_FROZEN

Validated experimental totals:
Conditions: 270
ML model fits: 1080
Ranking rows: 7709310
Build-metric rows: 71820
Project-run rows: 1890
Noise-manifest rows audited: 5358150

Raw-results cryptographic freeze:
Files hashed: 2160
Total bytes: 197808091
Manifest root SHA-256:
255be2222a28cc9cef226156605244e55f10d99526e3a243c785431874ae7b39

Final-package cryptographic freeze:
Manifested files: 72
Manifested bytes: 3756096
Manifest root SHA-256:
c2512bf9c72cd41dcb486b625d77e0331fdf1960296111c1a8d90ebb23d7425d

Post-creation package verification:
Missing files: 0
Size mismatches: 0
SHA-256 mismatches: 0

Completion registry:
/content/drive/MyDrive/Thesis_Experiment/Notes/completed_project_registry.csv
Complete-and-frozen projects: 6

Final Project 6 directory:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/eclipse__jetty.project/jetty_30_seed_final

Freeze record:
/content/drive/MyDriv

In [ ]:
# =========================================================
# PROJECT 7 — STEP 0
# RUNTIME RECOVERY AND NEXT-PROJECT SELECTION
#
# This cell:
#   - mounts Google Drive after the Colab disconnect
#   - restores permanent thesis paths
#   - verifies the dataset archive and screening table
#   - verifies Projects 1–6 are complete and frozen
#   - identifies the next eligible unfinished project
#
# It does NOT:
#   - modify Projects 1–6
#   - extract the dataset
#   - create the Project 7 checkpoint
#   - run any experiment
# =========================================================

from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display
from google.colab import drive


print(
    "=== PROJECT 7 STEP 0: "
    "RUNTIME RECOVERY AND PROJECT SELECTION ==="
)


# ---------------------------------------------------------
# 1. Mount Google Drive
# ---------------------------------------------------------

drive.mount(
    "/content/drive",
    force_remount=False,
)


# ---------------------------------------------------------
# 2. Restore permanent thesis paths
# ---------------------------------------------------------

THESIS_DRIVE = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)


DATA_DIRECTORY = (
    THESIS_DRIVE
    / "Data"
)


RAW_DATA_DIRECTORY = (
    DATA_DIRECTORY
    / "Raw"
)


PROCESSED_DATA_DIRECTORY = (
    DATA_DIRECTORY
    / "processed"
)


RESULTS_DIRECTORY = (
    THESIS_DRIVE
    / "Results"
)


RAW_RESULTS_DRIVE = (
    RESULTS_DIRECTORY
    / "Raw"
)


AGGREGATED_RESULTS_DRIVE = (
    RESULTS_DIRECTORY
    / "Aggregated"
)


LOGS_DRIVE = (
    RESULTS_DIRECTORY
    / "Logs"
)


NOTES_DRIVE = (
    THESIS_DRIVE
    / "Notes"
)


TCP_CI_ARCHIVE = (
    RAW_DATA_DIRECTORY
    / "TCP-CI-main-dataset.tar.gz"
)


SCREENING_PATH = (
    PROCESSED_DATA_DIRECTORY
    / "tcp_ci_project_screening.csv"
)


COMPLETION_REGISTRY_PATH = (
    NOTES_DRIVE
    / "completed_project_registry.csv"
)


RUNTIME_WORKING_DIRECTORY = Path(
    "/content/working_data"
)


RUNTIME_TCP_CI_DIRECTORY = (
    RUNTIME_WORKING_DIRECTORY
    / "tcp_ci_full"
)


# ---------------------------------------------------------
# 3. Verify permanent files and directories
# ---------------------------------------------------------

required_paths = [
    THESIS_DRIVE,
    DATA_DIRECTORY,
    RAW_DATA_DIRECTORY,
    PROCESSED_DATA_DIRECTORY,
    RESULTS_DIRECTORY,
    RAW_RESULTS_DRIVE,
    AGGREGATED_RESULTS_DRIVE,
    LOGS_DRIVE,
    NOTES_DRIVE,
    TCP_CI_ARCHIVE,
    SCREENING_PATH,
    COMPLETION_REGISTRY_PATH,
]


missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]


if missing_paths:
    raise FileNotFoundError(
        "Required permanent thesis paths are missing:\n"
        + "\n".join(missing_paths)
        + "\n\nDo not upload the dataset again. Check that "
        "Google Drive was mounted with the correct account."
    )


archive_size_bytes = int(
    TCP_CI_ARCHIVE.stat().st_size
)


if archive_size_bytes <= 0:
    raise AssertionError(
        "The TCP-CI archive exists but is empty."
    )


# ---------------------------------------------------------
# 4. Read screening and completion-registry tables
# ---------------------------------------------------------

screening = pd.read_csv(
    SCREENING_PATH
)


completion_registry = pd.read_csv(
    COMPLETION_REGISTRY_PATH,
    dtype=str,
).fillna("")


if len(screening) == 0:
    raise AssertionError(
        "The project-screening table is empty."
    )


if len(completion_registry) == 0:
    raise AssertionError(
        "The completion registry is empty."
    )


# ---------------------------------------------------------
# 5. Flexible column-identification helpers
# ---------------------------------------------------------

def normalise_column_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).strip().lower(),
    )


def find_column(
    dataframe,
    exact_names=None,
    contains_terms=None,
):
    if exact_names is None:
        exact_names = []


    if contains_terms is None:
        contains_terms = []


    normalised_columns = {
        normalise_column_name(column):
            column
        for column in dataframe.columns
    }


    for candidate_name in exact_names:
        normalised_candidate = (
            normalise_column_name(
                candidate_name
            )
        )


        if normalised_candidate in (
            normalised_columns
        ):
            return normalised_columns[
                normalised_candidate
            ]


    for column in dataframe.columns:
        normalised_column = (
            normalise_column_name(
                column
            )
        )


        if all(
            normalise_column_name(term)
            in normalised_column
            for term in contains_terms
        ):
            return column


    return None


def create_project_slug(
    project_name,
):
    project_name = str(
        project_name
    ).strip()


    slug = project_name.replace(
        "@",
        "__",
    )


    slug = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        slug,
    )


    return slug


# ---------------------------------------------------------
# 6. Identify the screening project column
# ---------------------------------------------------------

screening_project_column = find_column(
    screening,
    exact_names=[
        "Project",
        "ProjectName",
        "Repository",
        "RepositoryName",
        "Subject",
    ],
)


# Fallback: use the column whose values resemble
# owner@repository project identifiers.
if screening_project_column is None:
    project_like_columns = []


    for column in screening.columns:
        values = (
            screening[column]
            .dropna()
            .astype(str)
        )


        if len(values) == 0:
            continue


        proportion_with_at_symbol = float(
            values.str.contains(
                "@",
                regex=False,
            ).mean()
        )


        if proportion_with_at_symbol >= 0.5:
            project_like_columns.append(
                column
            )


    if len(project_like_columns) == 1:
        screening_project_column = (
            project_like_columns[0]
        )


if screening_project_column is None:
    raise RuntimeError(
        "Could not identify the project-name column in "
        "the screening table.\n\nObserved columns:\n"
        + "\n".join(
            map(
                str,
                screening.columns,
            )
        )
    )


# ---------------------------------------------------------
# 7. Identify registry columns and frozen projects
# ---------------------------------------------------------

registry_project_column = find_column(
    completion_registry,
    exact_names=[
        "Project",
        "ProjectName",
    ],
)


registry_slug_column = find_column(
    completion_registry,
    exact_names=[
        "ProjectSlug",
        "Slug",
    ],
)


registry_status_column = find_column(
    completion_registry,
    exact_names=[
        "Status",
        "CompletionStatus",
    ],
)


registry_number_column = find_column(
    completion_registry,
    exact_names=[
        "ProjectNumber",
        "Number",
    ],
)


if registry_project_column is None:
    raise RuntimeError(
        "Could not identify the Project column in the "
        "completion registry."
    )


if registry_status_column is None:
    raise RuntimeError(
        "Could not identify the Status column in the "
        "completion registry."
    )


frozen_registry = (
    completion_registry[
        completion_registry[
            registry_status_column
        ]
        .astype(str)
        .str.strip()
        .str.upper()
        == "COMPLETE_AND_FROZEN"
    ]
    .copy()
    .reset_index(drop=True)
)


if len(frozen_registry) != 6:
    raise AssertionError(
        "Expected exactly six complete-and-frozen "
        "projects before starting Project 7.\n"
        f"Observed: {len(frozen_registry)}"
    )


completed_project_names = set(
    frozen_registry[
        registry_project_column
    ]
    .astype(str)
    .str.strip()
)


if registry_slug_column is not None:
    completed_project_slugs = set(
        frozen_registry[
            registry_slug_column
        ]
        .astype(str)
        .str.strip()
    )

else:
    completed_project_slugs = {
        create_project_slug(
            project_name
        )
        for project_name in (
            completed_project_names
        )
    }


# ---------------------------------------------------------
# 8. Identify screening eligibility
# ---------------------------------------------------------

eligibility_column = find_column(
    screening,
    exact_names=[
        "Eligible",
        "IsEligible",
        "Eligibility",
        "EligibilityStatus",
        "ScreeningStatus",
        "Included",
        "IsIncluded",
    ],
)


training_failure_column = find_column(
    screening,
    exact_names=[
        "FailingTrainingBuilds",
        "TrainingFailingBuilds",
        "FailingTrainBuilds",
    ],
    contains_terms=[
        "fail",
        "train",
        "build",
    ],
)


evaluation_failure_column = find_column(
    screening,
    exact_names=[
        "FailingEvaluationBuilds",
        "EvaluationFailingBuilds",
        "FailingEvalBuilds",
    ],
    contains_terms=[
        "fail",
        "eval",
        "build",
    ],
)


def parse_eligibility_value(
    value,
):
    if pd.isna(value):
        return np.nan


    normalised_value = (
        str(value)
        .strip()
        .lower()
    )


    positive_values = {
        "1",
        "true",
        "yes",
        "y",
        "eligible",
        "included",
        "include",
        "pass",
        "passed",
    }


    negative_values = {
        "0",
        "false",
        "no",
        "n",
        "ineligible",
        "excluded",
        "exclude",
        "fail",
        "failed",
    }


    if normalised_value in positive_values:
        return True


    if normalised_value in negative_values:
        return False


    return np.nan


screening_work = (
    screening
    .copy()
    .reset_index(drop=True)
)


screening_work[
    "_ScreeningRow"
] = np.arange(
    1,
    len(screening_work) + 1,
    dtype=np.int64,
)


screening_work[
    "_Project"
] = (
    screening_work[
        screening_project_column
    ]
    .astype(str)
    .str.strip()
)


screening_work[
    "_ProjectSlug"
] = (
    screening_work[
        "_Project"
    ]
    .map(
        create_project_slug
    )
)


eligibility_method = None


if eligibility_column is not None:
    parsed_eligibility = (
        screening_work[
            eligibility_column
        ]
        .map(
            parse_eligibility_value
        )
    )


    recognised_eligibility_values = int(
        parsed_eligibility
        .notna()
        .sum()
    )


    if recognised_eligibility_values > 0:
        screening_work[
            "_Eligible"
        ] = (
            parsed_eligibility
            .fillna(False)
            .astype(bool)
        )


        eligibility_method = (
            f"screening column: "
            f"{eligibility_column}"
        )


if eligibility_method is None:
    if (
        training_failure_column is None
        or evaluation_failure_column is None
    ):
        raise RuntimeError(
            "Could not determine screening eligibility.\n"
            "No recognised eligibility column was found, "
            "and failing training/evaluation build "
            "columns could not both be identified.\n\n"
            "Observed screening columns:\n"
            + "\n".join(
                map(
                    str,
                    screening.columns,
                )
            )
        )


    training_failure_counts = pd.to_numeric(
        screening_work[
            training_failure_column
        ],
        errors="coerce",
    )


    evaluation_failure_counts = pd.to_numeric(
        screening_work[
            evaluation_failure_column
        ],
        errors="coerce",
    )


    screening_work[
        "_Eligible"
    ] = (
        (
            training_failure_counts
            >= 10
        )
        & (
            evaluation_failure_counts
            > 0
        )
    )


    eligibility_method = (
        "derived from at least 10 failing training "
        "builds and at least one failing evaluation build"
    )


# ---------------------------------------------------------
# 9. Determine screening order
# ---------------------------------------------------------

screening_order_column = find_column(
    screening,
    exact_names=[
        "ScreeningOrder",
        "ProjectOrder",
        "SelectionOrder",
        "Order",
        "Sequence",
    ],
)


if screening_order_column is not None:
    parsed_screening_order = pd.to_numeric(
        screening_work[
            screening_order_column
        ],
        errors="coerce",
    )


    if parsed_screening_order.notna().all():
        screening_work[
            "_SelectionOrder"
        ] = parsed_screening_order


    else:
        screening_work[
            "_SelectionOrder"
        ] = screening_work[
            "_ScreeningRow"
        ]


else:
    screening_work[
        "_SelectionOrder"
    ] = screening_work[
        "_ScreeningRow"
    ]


# ---------------------------------------------------------
# 10. Exclude Projects 1–6 and select Project 7 candidate
# ---------------------------------------------------------

screening_work[
    "_AlreadyCompleted"
] = (
    screening_work[
        "_Project"
    ].isin(
        completed_project_names
    )
    |
    screening_work[
        "_ProjectSlug"
    ].isin(
        completed_project_slugs
    )
)


eligible_unfinished = (
    screening_work[
        screening_work[
            "_Eligible"
        ]
        & (
            ~screening_work[
                "_AlreadyCompleted"
            ]
        )
    ]
    .sort_values(
        [
            "_SelectionOrder",
            "_ScreeningRow",
        ],
        kind="mergesort",
    )
    .drop_duplicates(
        subset=[
            "_Project",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)


if len(eligible_unfinished) == 0:
    raise AssertionError(
        "No eligible unfinished TCP-CI projects remain."
    )


PROJECT_7_CANDIDATE = str(
    eligible_unfinished.iloc[0][
        "_Project"
    ]
)


PROJECT_7_CANDIDATE_SLUG = str(
    eligible_unfinished.iloc[0][
        "_ProjectSlug"
    ]
)


PROJECT_7_SCREENING_ROW = int(
    eligible_unfinished.iloc[0][
        "_ScreeningRow"
    ]
)


PROJECT_7_SELECTION_ORDER = int(
    eligible_unfinished.iloc[0][
        "_SelectionOrder"
    ]
)


# Runtime-only aliases. Nothing has been committed yet.
PROJECT_NUMBER = 7
PROJECT_NAME = PROJECT_7_CANDIDATE
PROJECT_SLUG = PROJECT_7_CANDIDATE_SLUG


# ---------------------------------------------------------
# 11. Prepare compact review tables
# ---------------------------------------------------------

completed_display_columns = []


if registry_number_column is not None:
    completed_display_columns.append(
        registry_number_column
    )


completed_display_columns.extend([
    registry_project_column,
])


if registry_slug_column is not None:
    completed_display_columns.append(
        registry_slug_column
    )


completed_display_columns.append(
    registry_status_column
)


completed_projects_display = (
    frozen_registry[
        completed_display_columns
    ]
    .copy()
)


candidate_display = pd.DataFrame({
    "CandidateRank":
        np.arange(
            1,
            min(
                len(
                    eligible_unfinished
                ),
                10,
            ) + 1,
            dtype=np.int64,
        ),

    "ScreeningRow":
        eligible_unfinished[
            "_ScreeningRow"
        ]
        .head(10)
        .to_numpy(dtype=np.int64),

    "SelectionOrder":
        eligible_unfinished[
            "_SelectionOrder"
        ]
        .head(10)
        .to_numpy(dtype=np.int64),

    "Project":
        eligible_unfinished[
            "_Project"
        ]
        .head(10)
        .to_numpy(),

    "ProjectSlug":
        eligible_unfinished[
            "_ProjectSlug"
        ]
        .head(10)
        .to_numpy(),
})


if training_failure_column is not None:
    candidate_display[
        "FailingTrainingBuilds"
    ] = (
        eligible_unfinished[
            training_failure_column
        ]
        .head(10)
        .to_numpy()
    )


if evaluation_failure_column is not None:
    candidate_display[
        "FailingEvaluationBuilds"
    ] = (
        eligible_unfinished[
            evaluation_failure_column
        ]
        .head(10)
        .to_numpy()
    )


# ---------------------------------------------------------
# 12. Final compact output
# ---------------------------------------------------------

print(
    "\n=== PROJECT 7 STEP 0 RESULT ==="
)


print(
    "\nPermanent data recovery:"
)

print(
    "Thesis directory:",
    THESIS_DRIVE
)

print(
    "Dataset archive:",
    TCP_CI_ARCHIVE
)

print(
    "Archive size:",
    f"{archive_size_bytes:,}",
    "bytes"
)

print(
    "Screening table:",
    SCREENING_PATH
)

print(
    "Completion registry:",
    COMPLETION_REGISTRY_PATH
)


print(
    "\nCompleted and frozen projects:"
)

display(
    completed_projects_display
)


print(
    "Complete-and-frozen project count:",
    len(
        frozen_registry
    )
)


print(
    "\nScreening:"
)

print(
    "Projects in screening table:",
    len(
        screening_work
    )
)

print(
    "Eligible projects:",
    int(
        screening_work[
            "_Eligible"
        ].sum()
    )
)

print(
    "Eligible unfinished projects:",
    len(
        eligible_unfinished
    )
)

print(
    "Eligibility method:",
    eligibility_method
)


print(
    "\nNext eligible unfinished candidates:"
)

display(
    candidate_display
)


print(
    "\nTentative Project 7 selection:"
)

print(
    "Project:",
    PROJECT_7_CANDIDATE
)

print(
    "Slug:",
    PROJECT_7_CANDIDATE_SLUG
)

print(
    "Screening row:",
    PROJECT_7_SCREENING_ROW
)

print(
    "Selection order:",
    PROJECT_7_SELECTION_ORDER
)


print(
    "\nRuntime extraction directory:"
)

print(
    RUNTIME_TCP_CI_DIRECTORY
)


print(
    "\nStatus:",
    "READY_FOR_PROJECT_7_INITIALISATION"
)


print(
    "\nSUCCESS: Google Drive was mounted."
)

print(
    "SUCCESS: The permanent TCP-CI archive was found."
)

print(
    "SUCCESS: All six previous projects remain "
    "COMPLETE_AND_FROZEN."
)

print(
    "SUCCESS: The next eligible unfinished project was "
    "identified without modifying any experiment files."
)

=== PROJECT 7 STEP 0: RUNTIME RECOVERY AND PROJECT SELECTION ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

=== PROJECT 7 STEP 0 RESULT ===

Permanent data recovery:
Thesis directory: /content/drive/MyDrive/Thesis_Experiment
Dataset archive: /content/drive/MyDrive/Thesis_Experiment/Data/Raw/TCP-CI-main-dataset.tar.gz
Archive size: 237,104,379 bytes
Screening table: /content/drive/MyDrive/Thesis_Experiment/Data/processed/tcp_ci_project_screening.csv
Completion registry: /content/drive/MyDrive/Thesis_Experiment/Notes/completed_project_registry.csv

Completed and frozen projects:


,ProjectNumber,Project,ProjectSlug,Status
0,1,Angel-ML@angel,Angel-ML__angel,COMPLETE_AND_FROZEN
1,2,apache@airavata,apache__airavata,COMPLETE_AND_FROZEN
2,3,b2ihealthcare@snow-owl,b2ihealthcare__snow-owl,COMPLETE_AND_FROZEN
3,4,eclipse@paho.mqtt.java,eclipse__paho.mqtt.java,COMPLETE_AND_FROZEN
4,5,thinkaurelius@titan,thinkaurelius__titan,COMPLETE_AND_FROZEN
5,6,eclipse@jetty.project,eclipse__jetty.project,COMPLETE_AND_FROZEN


Complete-and-frozen project count: 6

Screening:
Projects in screening table: 25
Eligible projects: 24
Eligible unfinished projects: 18
Eligibility method: derived from at least 10 failing training builds and at least one failing evaluation build

Next eligible unfinished candidates:


,CandidateRank,ScreeningRow,SelectionOrder,Project,ProjectSlug,FailingTrainingBuilds,FailingEvaluationBuilds
0,1,7,7,CompEvol@beast2,CompEvol__beast2,64,52
1,2,8,8,optimatika@ojAlgo,optimatika__ojAlgo,57,16
2,3,9,9,spring-cloud@spring-cloud-dataflow,spring-cloud__spring-cloud-dataflow,62,27
3,4,10,10,eclipse@steady,eclipse__steady,40,12
4,5,11,11,yamcs@Yamcs,yamcs__Yamcs,63,10
5,6,12,12,EMResearch@EvoMaster,EMResearch__EvoMaster,103,41
6,7,13,13,apache@curator,apache__curator,102,2
7,8,14,14,cantaloupe-project@cantaloupe,cantaloupe-project__cantaloupe,59,12
8,9,15,15,zolyfarkas@spf4j,zolyfarkas__spf4j,187,91
9,10,16,16,apache@rocketmq,apache__rocketmq,49,8



Tentative Project 7 selection:
Project: CompEvol@beast2
Slug: CompEvol__beast2
Screening row: 7
Selection order: 7

Runtime extraction directory:
/content/working_data/tcp_ci_full

Status: READY_FOR_PROJECT_7_INITIALISATION

SUCCESS: Google Drive was mounted.
SUCCESS: The permanent TCP-CI archive was found.
SUCCESS: All six previous projects remain COMPLETE_AND_FROZEN.
SUCCESS: The next eligible unfinished project was identified without modifying any experiment files.


In [ ]:
# =========================================================
# PROJECT 7 — STEP 1
# CORRECTED INITIALISATION FOR COMPEVOL@BEAST2
#
# Correction:
#   The TCP-CI dataset uses:
#       entity_change_history.csv
#   rather than:
#       entity_history.csv
#
# This cell:
#   - confirms Project 7 identity and screening status
#   - reuses the existing runtime extraction when valid
#   - safely extracts only when necessary
#   - locates the Beast2 project directory
#   - supports the official entity-history filename
#   - verifies all six required source files
#   - creates the permanent Project 7 checkpoint
#
# It does not load the complete datasets or modify
# Projects 1–6.
# =========================================================

from pathlib import Path
import json
import os
import re
import shutil
import tarfile
import time

import numpy as np
import pandas as pd
from IPython.display import clear_output, display


print(
    "=== PROJECT 7 STEP 1: "
    "INITIALISE COMPEVOL@BEAST2 ==="
)


# ---------------------------------------------------------
# 1. Confirm the required Step 0 runtime state
# ---------------------------------------------------------

required_step_0_objects = [
    "THESIS_DRIVE",
    "RAW_RESULTS_DRIVE",
    "AGGREGATED_RESULTS_DRIVE",
    "LOGS_DRIVE",
    "NOTES_DRIVE",
    "TCP_CI_ARCHIVE",
    "SCREENING_PATH",
    "COMPLETION_REGISTRY_PATH",
    "RUNTIME_TCP_CI_DIRECTORY",
    "screening_work",
    "training_failure_column",
    "evaluation_failure_column",
    "PROJECT_7_CANDIDATE",
    "PROJECT_7_CANDIDATE_SLUG",
    "PROJECT_7_SCREENING_ROW",
    "PROJECT_7_SELECTION_ORDER",
]


missing_step_0_objects = [
    object_name
    for object_name in required_step_0_objects
    if object_name not in globals()
]


if missing_step_0_objects:
    raise RuntimeError(
        "Required Project 7 Step 0 objects are missing:\n"
        + "\n".join(missing_step_0_objects)
        + "\n\nRerun only Project 7 Step 0. "
        "Do not rerun any completed project."
    )


# ---------------------------------------------------------
# 2. Freeze Project 7 identity
# ---------------------------------------------------------

PROJECT_NUMBER = 7
PROJECT_NAME = str(
    PROJECT_7_CANDIDATE
).strip()

PROJECT_SLUG = str(
    PROJECT_7_CANDIDATE_SLUG
).strip()


EXPECTED_PROJECT_NAME = "CompEvol@beast2"
EXPECTED_PROJECT_SLUG = "CompEvol__beast2"


if PROJECT_NAME != EXPECTED_PROJECT_NAME:
    raise AssertionError(
        "Unexpected Project 7 candidate.\n"
        f"Expected: {EXPECTED_PROJECT_NAME}\n"
        f"Observed: {PROJECT_NAME}"
    )


if PROJECT_SLUG != EXPECTED_PROJECT_SLUG:
    raise AssertionError(
        "Unexpected Project 7 slug.\n"
        f"Expected: {EXPECTED_PROJECT_SLUG}\n"
        f"Observed: {PROJECT_SLUG}"
    )


if "@" not in PROJECT_NAME:
    raise AssertionError(
        "Project name does not contain the expected "
        "owner@repository structure."
    )


PROJECT_OWNER, PROJECT_REPOSITORY = (
    PROJECT_NAME.split(
        "@",
        maxsplit=1,
    )
)


PROJECT_SHORT_NAME = re.sub(
    r"[^A-Za-z0-9._-]+",
    "_",
    PROJECT_REPOSITORY,
).strip("_")


if PROJECT_OWNER != "CompEvol":
    raise AssertionError(
        f"Unexpected project owner: {PROJECT_OWNER}"
    )


if PROJECT_SHORT_NAME != "beast2":
    raise AssertionError(
        f"Unexpected project short name: "
        f"{PROJECT_SHORT_NAME}"
    )


# ---------------------------------------------------------
# 3. Recover and validate the screening record
# ---------------------------------------------------------

project_screening_rows = (
    screening_work[
        screening_work[
            "_Project"
        ].astype(str)
        == PROJECT_NAME
    ]
    .copy()
    .reset_index(drop=True)
)


if len(project_screening_rows) != 1:
    raise AssertionError(
        "Expected exactly one Beast2 screening row.\n"
        f"Observed: {len(project_screening_rows)}"
    )


project_screening_row = (
    project_screening_rows.iloc[0]
)


if not bool(
    project_screening_row[
        "_Eligible"
    ]
):
    raise AssertionError(
        "CompEvol@beast2 is not marked eligible."
    )


if bool(
    project_screening_row[
        "_AlreadyCompleted"
    ]
):
    raise AssertionError(
        "CompEvol@beast2 is already marked complete."
    )


if training_failure_column is None:
    raise RuntimeError(
        "The failing-training-build column is missing."
    )


if evaluation_failure_column is None:
    raise RuntimeError(
        "The failing-evaluation-build column is missing."
    )


EXPECTED_FAILING_TRAINING_BUILDS = int(
    pd.to_numeric(
        project_screening_row[
            training_failure_column
        ],
        errors="raise",
    )
)


EXPECTED_FAILING_EVALUATION_BUILDS = int(
    pd.to_numeric(
        project_screening_row[
            evaluation_failure_column
        ],
        errors="raise",
    )
)


if EXPECTED_FAILING_TRAINING_BUILDS != 64:
    raise AssertionError(
        "Unexpected failing-training-build count.\n"
        f"Expected: 64\n"
        f"Observed: "
        f"{EXPECTED_FAILING_TRAINING_BUILDS}"
    )


if EXPECTED_FAILING_EVALUATION_BUILDS != 52:
    raise AssertionError(
        "Unexpected failing-evaluation-build count.\n"
        f"Expected: 52\n"
        f"Observed: "
        f"{EXPECTED_FAILING_EVALUATION_BUILDS}"
    )


# ---------------------------------------------------------
# 4. Confirm Projects 1–6 remain frozen
# ---------------------------------------------------------

completion_registry = pd.read_csv(
    COMPLETION_REGISTRY_PATH,
    dtype=str,
).fillna("")


required_registry_columns = {
    "ProjectNumber",
    "Project",
    "ProjectSlug",
    "Status",
}


missing_registry_columns = (
    required_registry_columns
    - set(
        completion_registry.columns
    )
)


if missing_registry_columns:
    raise RuntimeError(
        "Completion registry is missing columns:\n"
        + "\n".join(
            sorted(
                missing_registry_columns
            )
        )
    )


frozen_registry = (
    completion_registry[
        completion_registry[
            "Status"
        ].astype(str)
        == "COMPLETE_AND_FROZEN"
    ]
    .copy()
    .reset_index(drop=True)
)


if len(frozen_registry) != 6:
    raise AssertionError(
        "Expected six frozen projects before Project 7.\n"
        f"Observed: {len(frozen_registry)}"
    )


if PROJECT_NAME in set(
    frozen_registry[
        "Project"
    ].astype(str)
):
    raise AssertionError(
        "Project 7 is unexpectedly already frozen."
    )


# ---------------------------------------------------------
# 5. Create permanent Project 7 directories
# ---------------------------------------------------------

PROJECT_RAW_RESULTS = (
    RAW_RESULTS_DRIVE
    / PROJECT_SLUG
)


PROJECT_AGGREGATED_RESULTS = (
    AGGREGATED_RESULTS_DRIVE
    / PROJECT_SLUG
)


PROJECT_LOGS = (
    LOGS_DRIVE
    / PROJECT_SLUG
)


PROJECT_PREFLIGHT_DIRECTORY = (
    PROJECT_AGGREGATED_RESULTS
    / f"{PROJECT_SHORT_NAME}_preflight"
)


PROJECT_7_SELECTION_CHECKPOINT = (
    NOTES_DRIVE
    / "project_07_selection_checkpoint.json"
)


PROJECT_INITIALISATION_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / f"{PROJECT_SHORT_NAME}_initialisation_report.json"
)


PROJECT_SOURCE_MANIFEST_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / f"{PROJECT_SHORT_NAME}_required_file_manifest.csv"
)


PROJECT_SOURCE_CANDIDATES_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / f"{PROJECT_SHORT_NAME}_source_directory_candidates.csv"
)


for project_directory in [
    PROJECT_RAW_RESULTS,
    PROJECT_AGGREGATED_RESULTS,
    PROJECT_LOGS,
    PROJECT_PREFLIGHT_DIRECTORY,
]:
    project_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# Refuse only a checkpoint belonging to another project.
if PROJECT_7_SELECTION_CHECKPOINT.exists():
    with open(
        PROJECT_7_SELECTION_CHECKPOINT,
        "r",
        encoding="utf-8",
    ) as checkpoint_file:
        existing_checkpoint = json.load(
            checkpoint_file
        )


    existing_checkpoint_project = str(
        existing_checkpoint.get(
            "Project",
            "",
        )
    ).strip()


    if (
        existing_checkpoint_project
        and existing_checkpoint_project
        != PROJECT_NAME
    ):
        raise AssertionError(
            "The existing Project 7 checkpoint belongs "
            "to another project.\n"
            f"Checkpoint project: "
            f"{existing_checkpoint_project}\n"
            f"Current project: {PROJECT_NAME}"
        )


# ---------------------------------------------------------
# 6. Validate the permanent archive
# ---------------------------------------------------------

if not TCP_CI_ARCHIVE.exists():
    raise FileNotFoundError(
        "The permanent TCP-CI archive is missing:\n"
        f"{TCP_CI_ARCHIVE}"
    )


archive_size_bytes = int(
    TCP_CI_ARCHIVE.stat().st_size
)


if archive_size_bytes != 237104379:
    raise AssertionError(
        "Unexpected TCP-CI archive size.\n"
        "Expected: 237,104,379 bytes\n"
        f"Observed: {archive_size_bytes:,} bytes"
    )


# ---------------------------------------------------------
# 7. Safe extraction helper
# ---------------------------------------------------------

RUNTIME_WORKING_DIRECTORY = (
    RUNTIME_TCP_CI_DIRECTORY.parent
)


RUNTIME_EXTRACTION_MARKER = (
    RUNTIME_TCP_CI_DIRECTORY
    / "_tcp_ci_extraction_complete.json"
)


def safe_extract_tar_archive(
    archive_path,
    destination_directory,
):
    """
    Extract the archive while checking every member against
    path traversal.

    The filter='data' argument handles the newer Python tar
    extraction behaviour. The fallback supports older
    Python versions.
    """

    destination_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


    destination_root = (
        destination_directory.resolve()
    )


    with tarfile.open(
        archive_path,
        mode="r:gz",
    ) as archive:
        members = archive.getmembers()


        for member in members:
            target_path = (
                destination_directory
                / member.name
            ).resolve()


            common_path = os.path.commonpath([
                str(destination_root),
                str(target_path),
            ])


            if common_path != str(
                destination_root
            ):
                raise RuntimeError(
                    "Unsafe archive member rejected:\n"
                    f"{member.name}"
                )


        try:
            archive.extractall(
                path=destination_directory,
                filter="data",
            )

        except TypeError:
            # Compatibility fallback for Python versions
            # that do not yet accept the filter argument.
            archive.extractall(
                path=destination_directory,
            )


        return len(members)


# ---------------------------------------------------------
# 8. Reuse or create the runtime extraction
# ---------------------------------------------------------

extraction_reused = False
archive_members_extracted = 0
extraction_start_time = time.time()


runtime_extraction_valid = False
runtime_marker = {}


if (
    RUNTIME_TCP_CI_DIRECTORY.exists()
    and RUNTIME_EXTRACTION_MARKER.exists()
):
    try:
        with open(
            RUNTIME_EXTRACTION_MARKER,
            "r",
            encoding="utf-8",
        ) as marker_file:
            runtime_marker = json.load(
                marker_file
            )


        runtime_extraction_valid = bool(
            runtime_marker.get(
                "ExtractionStatus"
            ) == "COMPLETE"
            and int(
                runtime_marker.get(
                    "ArchiveSizeBytes",
                    -1,
                )
            ) == archive_size_bytes
        )

    except Exception:
        runtime_extraction_valid = False


if runtime_extraction_valid:
    extraction_reused = True

    archive_members_extracted = int(
        runtime_marker.get(
            "ArchiveMembers",
            0,
        )
    )


else:
    if RUNTIME_TCP_CI_DIRECTORY.exists():
        shutil.rmtree(
            RUNTIME_TCP_CI_DIRECTORY
        )


    RUNTIME_TCP_CI_DIRECTORY.mkdir(
        parents=True,
        exist_ok=False,
    )


    clear_output(
        wait=True
    )


    print(
        "=== PROJECT 7 STEP 1: "
        "INITIALISE COMPEVOL@BEAST2 ==="
    )


    print(
        "\nExtracting the TCP-CI archive..."
    )


    print(
        "Archive:",
        TCP_CI_ARCHIVE
    )


    print(
        "Destination:",
        RUNTIME_TCP_CI_DIRECTORY
    )


    archive_members_extracted = (
        safe_extract_tar_archive(
            archive_path=TCP_CI_ARCHIVE,
            destination_directory=(
                RUNTIME_TCP_CI_DIRECTORY
            ),
        )
    )


    runtime_marker = {
        "ExtractionStatus":
            "COMPLETE",

        "Archive":
            str(
                TCP_CI_ARCHIVE
            ),

        "ArchiveSizeBytes":
            archive_size_bytes,

        "ArchiveMembers":
            int(
                archive_members_extracted
            ),

        "RuntimeDirectory":
            str(
                RUNTIME_TCP_CI_DIRECTORY
            ),

        "CompletedAtUTC":
            pd.Timestamp.utcnow().isoformat(),
    }


    with open(
        RUNTIME_EXTRACTION_MARKER,
        "w",
        encoding="utf-8",
    ) as marker_file:
        json.dump(
            runtime_marker,
            marker_file,
            indent=2,
            default=str,
        )


extraction_elapsed_seconds = float(
    time.time()
    - extraction_start_time
)


# ---------------------------------------------------------
# 9. Required files and accepted filename aliases
# ---------------------------------------------------------

# The official TCP-CI filename is:
#     entity_change_history.csv
#
# entity_history.csv remains accepted as a compatibility
# alias in case another dataset release uses it.

REQUIRED_PROJECT_FILE_ALIASES = {
    "Builds": [
        "builds.csv",
    ],

    "ExecutionHistory": [
        "exe.csv",
    ],

    "ModelDataset": [
        "dataset.csv",
    ],

    "EntityHistory": [
        "entity_change_history.csv",
        "entity_history.csv",
    ],

    "EntityIdMap": [
        "id_map.csv",
    ],

    "Contributors": [
        "contributors.csv",
    ],
}


def normalise_path_token(
    value,
):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).lower(),
    )


project_normalised = normalise_path_token(
    PROJECT_NAME
)


owner_normalised = normalise_path_token(
    PROJECT_OWNER
)


repository_normalised = normalise_path_token(
    PROJECT_REPOSITORY
)


# ---------------------------------------------------------
# 10. Locate all directories containing the six files
# ---------------------------------------------------------

candidate_source_directories = []


for current_directory, _, file_names in os.walk(
    RUNTIME_TCP_CI_DIRECTORY
):
    file_name_lookup = {
        file_name.lower():
            file_name
        for file_name in file_names
    }


    resolved_files = {}
    all_required_files_found = True


    for logical_name, aliases in (
        REQUIRED_PROJECT_FILE_ALIASES.items()
    ):
        matched_file_name = None


        for alias in aliases:
            if alias.lower() in file_name_lookup:
                matched_file_name = (
                    file_name_lookup[
                        alias.lower()
                    ]
                )

                break


        if matched_file_name is None:
            all_required_files_found = False
            break


        resolved_files[
            logical_name
        ] = matched_file_name


    if not all_required_files_found:
        continue


    current_path = Path(
        current_directory
    )


    normalised_full_path = normalise_path_token(
        current_path.as_posix()
    )


    normalised_path_parts = [
        normalise_path_token(
            part
        )
        for part in current_path.parts
    ]


    normalised_base_name = normalise_path_token(
        current_path.name
    )


    match_score = 0
    match_reasons = []


    if normalised_base_name == project_normalised:
        match_score += 1000
        match_reasons.append(
            "directory name exactly matches project"
        )


    if project_normalised in normalised_path_parts:
        match_score += 800
        match_reasons.append(
            "path part exactly matches project"
        )


    if (
        owner_normalised in normalised_path_parts
        and repository_normalised
        in normalised_path_parts
    ):
        match_score += 600
        match_reasons.append(
            "owner and repository are separate path parts"
        )


    if normalised_base_name == repository_normalised:
        match_score += 400
        match_reasons.append(
            "directory name matches repository"
        )


    if project_normalised in normalised_full_path:
        match_score += 200
        match_reasons.append(
            "full path contains project token"
        )


    if (
        owner_normalised in normalised_full_path
        and repository_normalised
        in normalised_full_path
    ):
        match_score += 100
        match_reasons.append(
            "full path contains owner and repository"
        )


    if "datasets" in normalised_path_parts:
        match_score += 50
        match_reasons.append(
            "located under datasets"
        )


    candidate_source_directories.append({
        "Directory":
            str(
                current_path
            ),

        "MatchScore":
            int(
                match_score
            ),

        "DirectoryDepth":
            int(
                len(
                    current_path.parts
                )
            ),

        "MatchReasons":
            " | ".join(
                match_reasons
            ),

        "ResolvedFiles":
            resolved_files,
    })


if len(candidate_source_directories) == 0:
    # Produce a compact filename diagnostic before failing.
    relevant_files = []


    accepted_file_names = {
        alias.lower()
        for aliases in (
            REQUIRED_PROJECT_FILE_ALIASES.values()
        )
        for alias in aliases
    }


    for current_directory, _, file_names in os.walk(
        RUNTIME_TCP_CI_DIRECTORY
    ):
        for file_name in file_names:
            if file_name.lower() in accepted_file_names:
                relevant_files.append({
                    "Directory":
                        current_directory,

                    "FileName":
                        file_name,
                })


    relevant_file_table = pd.DataFrame(
        relevant_files
    )


    if len(relevant_file_table) > 0:
        display(
            relevant_file_table.head(100)
        )


    raise FileNotFoundError(
        "No extracted directory contains all six "
        "required TCP-CI files, including either "
        "entity_change_history.csv or entity_history.csv."
    )


candidate_source_table = pd.DataFrame([
    {
        "Directory":
            candidate[
                "Directory"
            ],

        "MatchScore":
            candidate[
                "MatchScore"
            ],

        "DirectoryDepth":
            candidate[
                "DirectoryDepth"
            ],

        "MatchReasons":
            candidate[
                "MatchReasons"
            ],

        "EntityHistoryFile":
            candidate[
                "ResolvedFiles"
            ][
                "EntityHistory"
            ],
    }
    for candidate in candidate_source_directories
])


candidate_source_table = (
    candidate_source_table
    .sort_values(
        [
            "MatchScore",
            "DirectoryDepth",
            "Directory",
        ],
        ascending=[
            False,
            True,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


candidate_source_table.to_csv(
    PROJECT_SOURCE_CANDIDATES_PATH,
    index=False,
)


best_candidate_score = int(
    candidate_source_table.iloc[0][
        "MatchScore"
    ]
)


if best_candidate_score <= 0:
    display(
        candidate_source_table.head(25)
    )


    raise RuntimeError(
        "Dataset directories were found, but none "
        "matched CompEvol@beast2."
    )


top_scoring_candidates = (
    candidate_source_table[
        candidate_source_table[
            "MatchScore"
        ] == best_candidate_score
    ]
    .copy()
    .reset_index(drop=True)
)


if len(top_scoring_candidates) != 1:
    display(
        top_scoring_candidates
    )


    raise RuntimeError(
        "More than one equally strong Beast2 source "
        "directory candidate was found."
    )


PROJECT_SOURCE_DIRECTORY = Path(
    top_scoring_candidates.iloc[0][
        "Directory"
    ]
)


selected_candidate_record = next(
    candidate
    for candidate in candidate_source_directories
    if candidate[
        "Directory"
    ] == str(
        PROJECT_SOURCE_DIRECTORY
    )
)


selected_resolved_files = (
    selected_candidate_record[
        "ResolvedFiles"
    ]
)


# ---------------------------------------------------------
# 11. Resolve and validate all six physical source files
# ---------------------------------------------------------

PROJECT_SOURCE_FILES = {}
source_manifest_records = []


for logical_name, aliases in (
    REQUIRED_PROJECT_FILE_ALIASES.items()
):
    actual_file_name = (
        selected_resolved_files[
            logical_name
        ]
    )


    source_file_path = (
        PROJECT_SOURCE_DIRECTORY
        / actual_file_name
    )


    if not source_file_path.exists():
        raise FileNotFoundError(
            f"Required file is missing:\n"
            f"{source_file_path}"
        )


    file_size_bytes = int(
        source_file_path.stat().st_size
    )


    if file_size_bytes <= 0:
        raise AssertionError(
            f"Required file is empty:\n"
            f"{source_file_path}"
        )


    header_columns = list(
        pd.read_csv(
            source_file_path,
            nrows=0,
        ).columns
    )


    if len(header_columns) == 0:
        raise AssertionError(
            f"Required CSV has no columns:\n"
            f"{source_file_path}"
        )


    PROJECT_SOURCE_FILES[
        logical_name
    ] = source_file_path


    source_manifest_records.append({
        "LogicalName":
            logical_name,

        "AcceptedAliases":
            " | ".join(
                aliases
            ),

        "ActualFileName":
            actual_file_name,

        "Path":
            str(
                source_file_path
            ),

        "SizeBytes":
            file_size_bytes,

        "ColumnCount":
            int(
                len(
                    header_columns
                )
            ),

        "Columns":
            " | ".join(
                map(
                    str,
                    header_columns,
                )
            ),
    })


project_source_manifest = pd.DataFrame(
    source_manifest_records
)


if len(project_source_manifest) != 6:
    raise AssertionError(
        "Expected six source-file records."
    )


if project_source_manifest[
    "Path"
].duplicated().any():
    raise AssertionError(
        "Two logical files resolved to the same "
        "physical source file."
    )


project_source_manifest.to_csv(
    PROJECT_SOURCE_MANIFEST_PATH,
    index=False,
)


# Canonical runtime path variables for later cells.
BUILDS_PATH = (
    PROJECT_SOURCE_FILES[
        "Builds"
    ]
)


EXE_PATH = (
    PROJECT_SOURCE_FILES[
        "ExecutionHistory"
    ]
)


DATASET_PATH = (
    PROJECT_SOURCE_FILES[
        "ModelDataset"
    ]
)


ENTITY_HISTORY_PATH = (
    PROJECT_SOURCE_FILES[
        "EntityHistory"
    ]
)


ID_MAP_PATH = (
    PROJECT_SOURCE_FILES[
        "EntityIdMap"
    ]
)


CONTRIBUTORS_PATH = (
    PROJECT_SOURCE_FILES[
        "Contributors"
    ]
)


if (
    ENTITY_HISTORY_PATH.name.lower()
    not in {
        "entity_change_history.csv",
        "entity_history.csv",
    }
):
    raise AssertionError(
        "Unexpected entity-history filename."
    )


# ---------------------------------------------------------
# 12. Save initialisation report
# ---------------------------------------------------------

initialisation_completed_at = (
    pd.Timestamp.utcnow().isoformat()
)


initialisation_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ProjectOwner":
        PROJECT_OWNER,

    "ProjectRepository":
        PROJECT_REPOSITORY,

    "ProjectShortName":
        PROJECT_SHORT_NAME,

    "Status":
        "PASS",

    "CorrectionApplied":
        (
            "Accepted official TCP-CI filename "
            "entity_change_history.csv, with "
            "entity_history.csv retained as a "
            "compatibility alias."
        ),

    "Screening": {
        "ScreeningRow":
            int(
                PROJECT_7_SCREENING_ROW
            ),

        "SelectionOrder":
            int(
                PROJECT_7_SELECTION_ORDER
            ),

        "Eligible":
            True,

        "ExpectedFailingTrainingBuilds":
            EXPECTED_FAILING_TRAINING_BUILDS,

        "ExpectedFailingEvaluationBuilds":
            EXPECTED_FAILING_EVALUATION_BUILDS,
    },

    "Archive": {
        "Path":
            str(
                TCP_CI_ARCHIVE
            ),

        "SizeBytes":
            archive_size_bytes,

        "RuntimeExtractionDirectory":
            str(
                RUNTIME_TCP_CI_DIRECTORY
            ),

        "ExtractionReused":
            extraction_reused,

        "ArchiveMembers":
            int(
                archive_members_extracted
            ),

        "ExtractionElapsedSeconds":
            extraction_elapsed_seconds,
    },

    "ProjectSourceDirectory":
        str(
            PROJECT_SOURCE_DIRECTORY
        ),

    "SourceDirectoryMatchScore":
        best_candidate_score,

    "RequiredFileCount":
        int(
            len(
                project_source_manifest
            )
        ),

    "EntityHistoryActualFileName":
        ENTITY_HISTORY_PATH.name,

    "RequiredFiles":
        project_source_manifest.to_dict(
            orient="records"
        ),

    "PermanentDirectories": {
        "RawResults":
            str(
                PROJECT_RAW_RESULTS
            ),

        "AggregatedResults":
            str(
                PROJECT_AGGREGATED_RESULTS
            ),

        "Logs":
            str(
                PROJECT_LOGS
            ),

        "Preflight":
            str(
                PROJECT_PREFLIGHT_DIRECTORY
            ),
    },

    "SourceCandidateAudit":
        str(
            PROJECT_SOURCE_CANDIDATES_PATH
        ),

    "SourceManifest":
        str(
            PROJECT_SOURCE_MANIFEST_PATH
        ),

    "CompletedAtUTC":
        initialisation_completed_at,
}


with open(
    PROJECT_INITIALISATION_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        initialisation_report,
        report_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 13. Write permanent Project 7 checkpoint atomically
# ---------------------------------------------------------

project_7_checkpoint = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ProjectOwner":
        PROJECT_OWNER,

    "ProjectRepository":
        PROJECT_REPOSITORY,

    "ProjectShortName":
        PROJECT_SHORT_NAME,

    "Status":
        "INITIALISATION_PASSED",

    "ScreeningRow":
        int(
            PROJECT_7_SCREENING_ROW
        ),

    "SelectionOrder":
        int(
            PROJECT_7_SELECTION_ORDER
        ),

    "ExpectedFailingTrainingBuilds":
        EXPECTED_FAILING_TRAINING_BUILDS,

    "ExpectedFailingEvaluationBuilds":
        EXPECTED_FAILING_EVALUATION_BUILDS,

    "Archive":
        str(
            TCP_CI_ARCHIVE
        ),

    "RuntimeExtractionDirectory":
        str(
            RUNTIME_TCP_CI_DIRECTORY
        ),

    "ExtractionReused":
        extraction_reused,

    "ProjectSourceDirectory":
        str(
            PROJECT_SOURCE_DIRECTORY
        ),

    "BuildsPath":
        str(
            BUILDS_PATH
        ),

    "ExecutionHistoryPath":
        str(
            EXE_PATH
        ),

    "DatasetPath":
        str(
            DATASET_PATH
        ),

    "EntityHistoryPath":
        str(
            ENTITY_HISTORY_PATH
        ),

    "EntityHistoryActualFileName":
        ENTITY_HISTORY_PATH.name,

    "IdMapPath":
        str(
            ID_MAP_PATH
        ),

    "ContributorsPath":
        str(
            CONTRIBUTORS_PATH
        ),

    "ProjectRawResults":
        str(
            PROJECT_RAW_RESULTS
        ),

    "ProjectAggregatedResults":
        str(
            PROJECT_AGGREGATED_RESULTS
        ),

    "ProjectLogs":
        str(
            PROJECT_LOGS
        ),

    "ProjectPreflightDirectory":
        str(
            PROJECT_PREFLIGHT_DIRECTORY
        ),

    "SourceCandidateAudit":
        str(
            PROJECT_SOURCE_CANDIDATES_PATH
        ),

    "SourceManifest":
        str(
            PROJECT_SOURCE_MANIFEST_PATH
        ),

    "InitialisationReport":
        str(
            PROJECT_INITIALISATION_REPORT_PATH
        ),

    "PreviousFrozenProjectCount":
        6,

    "CreatedAtUTC":
        initialisation_completed_at,

    "UpdatedAtUTC":
        initialisation_completed_at,
}


temporary_checkpoint_path = (
    PROJECT_7_SELECTION_CHECKPOINT
    .with_suffix(
        ".json.tmp"
    )
)


with open(
    temporary_checkpoint_path,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        project_7_checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


os.replace(
    temporary_checkpoint_path,
    PROJECT_7_SELECTION_CHECKPOINT,
)


# Verify the permanent write.
with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint_verification = json.load(
        checkpoint_file
    )


if checkpoint_verification.get(
    "Status"
) != "INITIALISATION_PASSED":
    raise AssertionError(
        "The permanent Project 7 checkpoint was not "
        "written correctly."
    )


if checkpoint_verification.get(
    "Project"
) != PROJECT_NAME:
    raise AssertionError(
        "The Project 7 checkpoint contains the wrong "
        "project."
    )


if not Path(
    checkpoint_verification[
        "EntityHistoryPath"
    ]
).exists():
    raise AssertionError(
        "The checkpoint entity-history path does not exist."
    )


# ---------------------------------------------------------
# 14. Compact final output
# ---------------------------------------------------------

clear_output(
    wait=True
)


print(
    "=== PROJECT 7 STEP 1 RESULT ==="
)


print(
    "\nProject identity:"
)

print(
    "Project number:",
    PROJECT_NUMBER
)

print(
    "Project:",
    PROJECT_NAME
)

print(
    "Project slug:",
    PROJECT_SLUG
)

print(
    "Short name:",
    PROJECT_SHORT_NAME
)


print(
    "\nScreening expectations:"
)

print(
    "Failing training builds:",
    EXPECTED_FAILING_TRAINING_BUILDS
)

print(
    "Failing evaluation builds:",
    EXPECTED_FAILING_EVALUATION_BUILDS
)


print(
    "\nRuntime extraction:"
)

print(
    "Extraction reused:",
    extraction_reused
)

print(
    "Archive members:",
    archive_members_extracted
)

print(
    "Elapsed:",
    f"{extraction_elapsed_seconds:.2f}",
    "seconds"
)

print(
    "Runtime root:",
    RUNTIME_TCP_CI_DIRECTORY
)


print(
    "\nProject source directory:"
)

print(
    PROJECT_SOURCE_DIRECTORY
)

print(
    "Source-directory match score:",
    best_candidate_score
)


print(
    "\nRequired source files:"
)

display(
    project_source_manifest[
        [
            "LogicalName",
            "ActualFileName",
            "SizeBytes",
            "ColumnCount",
        ]
    ]
)


print(
    "Required files found:",
    len(
        project_source_manifest
    ),
    "/ 6"
)

print(
    "Entity-history filename:",
    ENTITY_HISTORY_PATH.name
)


print(
    "\nPermanent Project 7 directories:"
)

print(
    "Raw results:",
    PROJECT_RAW_RESULTS
)

print(
    "Aggregated results:",
    PROJECT_AGGREGATED_RESULTS
)

print(
    "Logs:",
    PROJECT_LOGS
)

print(
    "Preflight:",
    PROJECT_PREFLIGHT_DIRECTORY
)


print(
    "\nInitialisation report:"
)

print(
    PROJECT_INITIALISATION_REPORT_PATH
)


print(
    "Selection checkpoint:"
)

print(
    PROJECT_7_SELECTION_CHECKPOINT
)


print(
    "\nInitialisation status:",
    "PASS"
)


print(
    "\nSUCCESS: Project 7 was fixed as "
    "CompEvol@beast2."
)

print(
    "SUCCESS: The existing runtime extraction was reused "
    "when valid."
)

print(
    "SUCCESS: The official entity_change_history.csv "
    "filename was recognised."
)

print(
    "SUCCESS: The Beast2 source directory was identified."
)

print(
    "SUCCESS: All six required source files were found "
    "and their headers validated."
)

print(
    "SUCCESS: The permanent Project 7 checkpoint was "
    "created."
)

print(
    "SUCCESS: Projects 1–6 remain untouched and frozen."
)

print(
    "SUCCESS: Project 7 is ready for schema inspection "
    "and chronological preflight."
)

=== PROJECT 7 STEP 1 RESULT ===

Project identity:
Project number: 7
Project: CompEvol@beast2
Project slug: CompEvol__beast2
Short name: beast2

Screening expectations:
Failing training builds: 64
Failing evaluation builds: 52

Runtime extraction:
Extraction reused: True
Archive members: 176
Elapsed: 0.00 seconds
Runtime root: /content/working_data/tcp_ci_full

Project source directory:
/content/working_data/tcp_ci_full/datasets/CompEvol@beast2
Source-directory match score: 2150

Required source files:


,LogicalName,ActualFileName,SizeBytes,ColumnCount
0,Builds,builds.csv,32469,3
1,ExecutionHistory,exe.csv,841767,5
2,ModelDataset,dataset.csv,6171496,154
3,EntityHistory,entity_change_history.csv,1162570,8
4,EntityIdMap,id_map.csv,82877,2
5,Contributors,contributors.csv,3777,4


Required files found: 6 / 6
Entity-history filename: entity_change_history.csv

Permanent Project 7 directories:
Raw results: /content/drive/MyDrive/Thesis_Experiment/Results/Raw/CompEvol__beast2
Aggregated results: /content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2
Logs: /content/drive/MyDrive/Thesis_Experiment/Results/Logs/CompEvol__beast2
Preflight: /content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_preflight

Initialisation report:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_preflight/beast2_initialisation_report.json
Selection checkpoint:
/content/drive/MyDrive/Thesis_Experiment/Notes/project_07_selection_checkpoint.json

Initialisation status: PASS

SUCCESS: Project 7 was fixed as CompEvol@beast2.
SUCCESS: The existing runtime extraction was reused when valid.
SUCCESS: The official entity_change_history.csv filename was recognised.
SUCCESS: The Beast2 source directory was identified

In [ ]:
# =========================================================
# PROJECT 7 — STEP 2
# SCHEMA INSPECTION AND CHRONOLOGICAL PREFLIGHT
#
# This cell:
#   - loads all six Beast2 source tables
#   - standardises the four model-ready core columns
#   - validates raw/model execution alignment
#   - applies the frozen chronological ordering:
#       started_at ascending
#       Build descending within equal timestamps
#   - creates the fixed 75%/25% build split
#   - verifies the screening counts: 64 training and
#     52 evaluation failing builds
#   - freezes the 150 predictor names
#   - creates clean training/evaluation runtime objects
#   - saves permanent preflight reports
#
# It does NOT:
#   - reconstruct REC features
#   - map commits to entities
#   - inject noise
#   - fit any model
# =========================================================

from pathlib import Path
import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd
from IPython.display import clear_output, display


print(
    "=== PROJECT 7 STEP 2: "
    "SCHEMA AND CHRONOLOGICAL PREFLIGHT ==="
)


# ---------------------------------------------------------
# 1. Confirm successful Step 1 state
# ---------------------------------------------------------

required_step_1_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",
    "PROJECT_SHORT_NAME",

    "PROJECT_RAW_RESULTS",
    "PROJECT_AGGREGATED_RESULTS",
    "PROJECT_LOGS",
    "PROJECT_PREFLIGHT_DIRECTORY",

    "PROJECT_7_SELECTION_CHECKPOINT",

    "BUILDS_PATH",
    "EXE_PATH",
    "DATASET_PATH",
    "ENTITY_HISTORY_PATH",
    "ID_MAP_PATH",
    "CONTRIBUTORS_PATH",

    "EXPECTED_FAILING_TRAINING_BUILDS",
    "EXPECTED_FAILING_EVALUATION_BUILDS",
]


missing_step_1_objects = [
    object_name
    for object_name in required_step_1_objects
    if object_name not in globals()
]


if missing_step_1_objects:
    raise RuntimeError(
        "Required Project 7 Step 1 objects are missing:\n"
        + "\n".join(missing_step_1_objects)
        + "\n\nRerun only the successful corrected "
        "Project 7 Step 1 cell."
    )


if PROJECT_NUMBER != 7:
    raise AssertionError(
        f"Expected Project 7, observed {PROJECT_NUMBER}."
    )


if PROJECT_NAME != "CompEvol@beast2":
    raise AssertionError(
        f"Unexpected Project 7 project: {PROJECT_NAME}"
    )


if PROJECT_SLUG != "CompEvol__beast2":
    raise AssertionError(
        f"Unexpected Project 7 slug: {PROJECT_SLUG}"
    )


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    step_1_checkpoint = json.load(
        checkpoint_file
    )


allowed_previous_statuses = {
    "INITIALISATION_PASSED",
    "PREFLIGHT_PASSED",
}


if step_1_checkpoint.get(
    "Status"
) not in allowed_previous_statuses:
    raise AssertionError(
        "Project 7 Step 1 has not passed.\n"
        f"Observed checkpoint status: "
        f"{step_1_checkpoint.get('Status')}"
    )


# ---------------------------------------------------------
# 2. Validate source paths before loading
# ---------------------------------------------------------

source_paths = {
    "Builds":
        Path(BUILDS_PATH),

    "ExecutionHistory":
        Path(EXE_PATH),

    "ModelDataset":
        Path(DATASET_PATH),

    "EntityHistory":
        Path(ENTITY_HISTORY_PATH),

    "EntityIdMap":
        Path(ID_MAP_PATH),

    "Contributors":
        Path(CONTRIBUTORS_PATH),
}


missing_source_paths = [
    str(path)
    for path in source_paths.values()
    if not path.exists()
]


if missing_source_paths:
    raise FileNotFoundError(
        "One or more Beast2 source files are missing:\n"
        + "\n".join(missing_source_paths)
        + "\n\nDo not upload the dataset. Rerun the "
        "corrected Project 7 Step 1 extraction cell."
    )


# ---------------------------------------------------------
# 3. Load all six tables
# ---------------------------------------------------------

load_start_time = time.time()


builds_raw = pd.read_csv(
    BUILDS_PATH,
    low_memory=False,
)


exe_raw = pd.read_csv(
    EXE_PATH,
    low_memory=False,
)


dataset_raw = pd.read_csv(
    DATASET_PATH,
    low_memory=False,
)


entity_history_raw = pd.read_csv(
    ENTITY_HISTORY_PATH,
    low_memory=False,
)


id_map_raw = pd.read_csv(
    ID_MAP_PATH,
    low_memory=False,
)


contributors_raw = pd.read_csv(
    CONTRIBUTORS_PATH,
    low_memory=False,
)


load_elapsed_seconds = float(
    time.time()
    - load_start_time
)


original_tables = {
    "Builds":
        builds_raw,

    "ExecutionHistory":
        exe_raw,

    "ModelDataset":
        dataset_raw,

    "EntityHistory":
        entity_history_raw,

    "EntityIdMap":
        id_map_raw,

    "Contributors":
        contributors_raw,
}


for table_name, table_frame in (
    original_tables.items()
):
    if len(table_frame.columns) == 0:
        raise AssertionError(
            f"{table_name} has no columns."
        )


# ---------------------------------------------------------
# 4. Column-identification helpers
# ---------------------------------------------------------

def normalise_column_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).strip().lower(),
    )


def identify_column(
    dataframe,
    exact_names,
    required_terms=None,
):
    if required_terms is None:
        required_terms = []


    normalised_lookup = {
        normalise_column_name(column):
            column
        for column in dataframe.columns
    }


    for candidate in exact_names:
        candidate_normalised = (
            normalise_column_name(
                candidate
            )
        )


        if candidate_normalised in (
            normalised_lookup
        ):
            return normalised_lookup[
                candidate_normalised
            ]


    matching_columns = []


    for column in dataframe.columns:
        normalised_column = (
            normalise_column_name(
                column
            )
        )


        if all(
            normalise_column_name(term)
            in normalised_column
            for term in required_terms
        ):
            matching_columns.append(
                column
            )


    if len(matching_columns) == 1:
        return matching_columns[0]


    return None


builds_build_column = identify_column(
    builds_raw,
    exact_names=[
        "Build",
        "BuildId",
        "BuildID",
        "build_id",
    ],
    required_terms=[
        "build",
    ],
)


builds_started_column = identify_column(
    builds_raw,
    exact_names=[
        "started_at",
        "StartedAt",
        "StartTime",
        "Started",
        "Timestamp",
    ],
    required_terms=[
        "start",
    ],
)


exe_build_column = identify_column(
    exe_raw,
    exact_names=[
        "Build",
        "BuildId",
        "BuildID",
        "build_id",
    ],
    required_terms=[
        "build",
    ],
)


exe_test_column = identify_column(
    exe_raw,
    exact_names=[
        "Test",
        "TestId",
        "TestID",
        "test_id",
    ],
    required_terms=[
        "test",
    ],
)


exe_verdict_column = identify_column(
    exe_raw,
    exact_names=[
        "Verdict",
        "Result",
        "Outcome",
        "Status",
    ],
    required_terms=[
        "verdict",
    ],
)


exe_duration_column = identify_column(
    exe_raw,
    exact_names=[
        "Duration",
        "ExecutionTime",
        "ExeTime",
        "Runtime",
    ],
    required_terms=[
        "duration",
    ],
)


dataset_build_column = identify_column(
    dataset_raw,
    exact_names=[
        "Build",
        "BuildId",
        "BuildID",
        "build_id",
    ],
    required_terms=[
        "build",
    ],
)


dataset_test_column = identify_column(
    dataset_raw,
    exact_names=[
        "Test",
        "TestId",
        "TestID",
        "test_id",
    ],
    required_terms=[
        "test",
    ],
)


dataset_verdict_column = identify_column(
    dataset_raw,
    exact_names=[
        "Verdict",
        "Result",
        "Outcome",
        "Status",
    ],
    required_terms=[
        "verdict",
    ],
)


dataset_duration_column = identify_column(
    dataset_raw,
    exact_names=[
        "Duration",
        "ExecutionTime",
        "ExeTime",
        "Runtime",
    ],
    required_terms=[
        "duration",
    ],
)


identified_core_columns = {
    "builds.Build":
        builds_build_column,

    "builds.started_at":
        builds_started_column,

    "exe.Build":
        exe_build_column,

    "exe.Test":
        exe_test_column,

    "exe.Verdict":
        exe_verdict_column,

    "exe.Duration":
        exe_duration_column,

    "dataset.Build":
        dataset_build_column,

    "dataset.Test":
        dataset_test_column,

    "dataset.Verdict":
        dataset_verdict_column,

    "dataset.Duration":
        dataset_duration_column,
}


missing_core_columns = [
    logical_name
    for logical_name, observed_name in (
        identified_core_columns.items()
    )
    if observed_name is None
]


if missing_core_columns:
    raise RuntimeError(
        "Could not identify required core columns:\n"
        + "\n".join(missing_core_columns)
        + "\n\nBuild columns:\n"
        + " | ".join(
            map(
                str,
                builds_raw.columns,
            )
        )
        + "\n\nExecution columns:\n"
        + " | ".join(
            map(
                str,
                exe_raw.columns,
            )
        )
        + "\n\nDataset first columns:\n"
        + " | ".join(
            map(
                str,
                dataset_raw.columns[:20],
            )
        )
    )


# ---------------------------------------------------------
# 5. Canonicalise core column names
# ---------------------------------------------------------

def canonicalise_columns(
    dataframe,
    observed_to_canonical,
):
    rename_map = {}


    for observed_name, canonical_name in (
        observed_to_canonical.items()
    ):
        if observed_name == canonical_name:
            continue


        if (
            canonical_name in dataframe.columns
            and observed_name
            != canonical_name
        ):
            raise AssertionError(
                "Cannot canonicalise column because both "
                f"{observed_name} and {canonical_name} "
                "already exist."
            )


        rename_map[
            observed_name
        ] = canonical_name


    return dataframe.rename(
        columns=rename_map
    ).copy()


builds = canonicalise_columns(
    builds_raw,
    {
        builds_build_column:
            "Build",

        builds_started_column:
            "started_at",
    },
)


exe = canonicalise_columns(
    exe_raw,
    {
        exe_build_column:
            "Build",

        exe_test_column:
            "Test",

        exe_verdict_column:
            "Verdict",

        exe_duration_column:
            "Duration",
    },
)


dataset = canonicalise_columns(
    dataset_raw,
    {
        dataset_build_column:
            "Build",

        dataset_test_column:
            "Test",

        dataset_verdict_column:
            "Verdict",

        dataset_duration_column:
            "Duration",
    },
)


entity_history = (
    entity_history_raw.copy()
)


id_map = (
    id_map_raw.copy()
)


contributors = (
    contributors_raw.copy()
)


# ---------------------------------------------------------
# 6. Core type-conversion helpers
# ---------------------------------------------------------

def convert_integer_series(
    series,
    logical_name,
):
    converted = pd.to_numeric(
        series,
        errors="coerce",
    )


    if converted.isna().any():
        raise AssertionError(
            f"{logical_name} contains missing or "
            "non-numeric values."
        )


    values = converted.to_numpy(
        dtype=float
    )


    if not np.isfinite(
        values
    ).all():
        raise AssertionError(
            f"{logical_name} contains non-finite values."
        )


    if not np.isclose(
        values,
        np.round(
            values
        ),
        rtol=0.0,
        atol=1e-12,
    ).all():
        raise AssertionError(
            f"{logical_name} contains non-integer values."
        )


    return pd.Series(
        np.round(
            values
        ).astype(np.int64),
        index=series.index,
        name=series.name,
    )


builds[
    "Build"
] = convert_integer_series(
    builds[
        "Build"
    ],
    "builds.Build",
)


exe[
    "Build"
] = convert_integer_series(
    exe[
        "Build"
    ],
    "exe.Build",
)


dataset[
    "Build"
] = convert_integer_series(
    dataset[
        "Build"
    ],
    "dataset.Build",
)


exe[
    "Verdict"
] = convert_integer_series(
    exe[
        "Verdict"
    ],
    "exe.Verdict",
)


dataset[
    "Verdict"
] = convert_integer_series(
    dataset[
        "Verdict"
    ],
    "dataset.Verdict",
)


for frame_name, frame in [
    (
        "exe",
        exe,
    ),
    (
        "dataset",
        dataset,
    ),
]:
    if frame[
        "Test"
    ].isna().any():
        raise AssertionError(
            f"{frame_name}.Test contains missing values."
        )


    frame[
        "Test"
    ] = (
        frame[
            "Test"
        ].astype(str)
    )


    numeric_duration = pd.to_numeric(
        frame[
            "Duration"
        ],
        errors="coerce",
    )


    if numeric_duration.isna().any():
        raise AssertionError(
            f"{frame_name}.Duration contains missing "
            "or non-numeric values."
        )


    duration_values = (
        numeric_duration.to_numpy(
            dtype=float
        )
    )


    if not np.isfinite(
        duration_values
    ).all():
        raise AssertionError(
            f"{frame_name}.Duration contains non-finite "
            "values."
        )


    if (
        duration_values
        < 0
    ).any():
        raise AssertionError(
            f"{frame_name}.Duration contains negative "
            "values."
        )


    frame[
        "Duration"
    ] = duration_values


builds[
    "started_at"
] = pd.to_datetime(
    builds[
        "started_at"
    ],
    errors="coerce",
    utc=True,
)


if builds[
    "started_at"
].isna().any():
    invalid_timestamp_count = int(
        builds[
            "started_at"
        ].isna().sum()
    )


    raise AssertionError(
        "One or more build timestamps could not be "
        "parsed.\n"
        f"Invalid timestamps: {invalid_timestamp_count}"
    )


# ---------------------------------------------------------
# 7. Basic source-table integrity
# ---------------------------------------------------------

if builds[
    "Build"
].duplicated().any():
    duplicate_build_ids = (
        builds.loc[
            builds[
                "Build"
            ].duplicated(
                keep=False
            ),
            "Build",
        ]
        .astype(int)
        .tolist()
    )


    raise AssertionError(
        "builds.csv contains duplicate Build IDs:\n"
        + ", ".join(
            map(
                str,
                duplicate_build_ids[:20],
            )
        )
    )


known_build_ids = set(
    builds[
        "Build"
    ].astype(int)
)


unknown_exe_builds = sorted(
    set(
        exe[
            "Build"
        ].astype(int)
    )
    - known_build_ids
)


unknown_dataset_builds = sorted(
    set(
        dataset[
            "Build"
        ].astype(int)
    )
    - known_build_ids
)


if unknown_exe_builds:
    raise AssertionError(
        "Execution history contains unknown builds:\n"
        + ", ".join(
            map(
                str,
                unknown_exe_builds[:20],
            )
        )
    )


if unknown_dataset_builds:
    raise AssertionError(
        "Model dataset contains unknown builds:\n"
        + ", ".join(
            map(
                str,
                unknown_dataset_builds[:20],
            )
        )
    )


exe_verdict_values = sorted(
    exe[
        "Verdict"
    ].astype(int).unique().tolist()
)


dataset_verdict_values = sorted(
    dataset[
        "Verdict"
    ].astype(int).unique().tolist()
)


if 0 not in exe_verdict_values:
    raise AssertionError(
        "Raw execution history contains no passing "
        "verdict value 0."
    )


if 0 not in dataset_verdict_values:
    raise AssertionError(
        "Model dataset contains no passing verdict "
        "value 0."
    )


if not any(
    verdict_value != 0
    for verdict_value in (
        dataset_verdict_values
    )
):
    raise AssertionError(
        "Model dataset contains no failure verdicts."
    )


# ---------------------------------------------------------
# 8. Freeze the 150 predictor columns
# ---------------------------------------------------------

MODEL_CORE_COLUMNS = [
    "Build",
    "Test",
    "Verdict",
    "Duration",
]


predictor_columns = [
    column
    for column in dataset.columns
    if column not in MODEL_CORE_COLUMNS
]


PREDICTOR_COLUMNS = list(
    predictor_columns
)


if len(PREDICTOR_COLUMNS) != 150:
    raise AssertionError(
        "Unexpected predictor count.\n"
        f"Expected: 150\n"
        f"Observed: {len(PREDICTOR_COLUMNS)}"
    )


predictor_coercion_failures = 0


for predictor_column in (
    PREDICTOR_COLUMNS
):
    original_values = (
        dataset[
            predictor_column
        ]
    )


    numeric_values = pd.to_numeric(
        original_values,
        errors="coerce",
    )


    predictor_coercion_failures += int(
        (
            numeric_values.isna()
            & original_values.notna()
        ).sum()
    )


    dataset[
        predictor_column
    ] = numeric_values


if predictor_coercion_failures != 0:
    raise AssertionError(
        "At least one predictor contains a non-numeric "
        "value that could not be converted.\n"
        f"Coercion failures: "
        f"{predictor_coercion_failures}"
    )


predictor_missing_values = int(
    dataset[
        PREDICTOR_COLUMNS
    ].isna().sum().sum()
)


predictor_matrix = (
    dataset[
        PREDICTOR_COLUMNS
    ].to_numpy(
        dtype=float
    )
)


predictor_infinite_values = int(
    np.isinf(
        predictor_matrix
    ).sum()
)


if predictor_infinite_values != 0:
    raise AssertionError(
        "The predictor matrix contains infinite values."
    )


del predictor_matrix


# ---------------------------------------------------------
# 9. Validate model-ready rows against raw executions
# ---------------------------------------------------------

exe_key_duplicate_rows = int(
    exe.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


dataset_key_duplicate_rows = int(
    dataset.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


exe_alignment = exe[
    [
        "Build",
        "Test",
        "Verdict",
        "Duration",
    ]
].copy()


dataset_alignment = dataset[
    [
        "Build",
        "Test",
        "Verdict",
        "Duration",
    ]
].copy()


exe_alignment[
    "_Occurrence"
] = (
    exe_alignment
    .groupby(
        [
            "Build",
            "Test",
        ],
        sort=False,
    )
    .cumcount()
)


dataset_alignment[
    "_Occurrence"
] = (
    dataset_alignment
    .groupby(
        [
            "Build",
            "Test",
        ],
        sort=False,
    )
    .cumcount()
)


exe_alignment = exe_alignment.rename(
    columns={
        "Verdict":
            "RawVerdict",

        "Duration":
            "RawDuration",
    }
)


dataset_alignment = dataset_alignment.rename(
    columns={
        "Verdict":
            "ModelVerdict",

        "Duration":
            "ModelDuration",
    }
)


model_raw_alignment = (
    dataset_alignment
    .merge(
        exe_alignment,
        on=[
            "Build",
            "Test",
            "_Occurrence",
        ],
        how="left",
        indicator=True,
        validate="one_to_one",
    )
)


model_rows_missing_from_raw = int(
    (
        model_raw_alignment[
            "_merge"
        ] != "both"
    ).sum()
)


matched_alignment = (
    model_raw_alignment[
        model_raw_alignment[
            "_merge"
        ] == "both"
    ]
)


model_raw_verdict_mismatches = int(
    (
        matched_alignment[
            "ModelVerdict"
        ].to_numpy(dtype=np.int64)
        !=
        matched_alignment[
            "RawVerdict"
        ].to_numpy(dtype=np.int64)
    ).sum()
)


model_raw_duration_mismatches = int(
    (
        ~np.isclose(
            matched_alignment[
                "ModelDuration"
            ].to_numpy(dtype=float),

            matched_alignment[
                "RawDuration"
            ].to_numpy(dtype=float),

            rtol=1e-12,
            atol=1e-12,
            equal_nan=True,
        )
    ).sum()
)


if model_rows_missing_from_raw != 0:
    raise AssertionError(
        "At least one model-ready row could not be "
        "matched to the raw execution history.\n"
        f"Missing rows: {model_rows_missing_from_raw}"
    )


if model_raw_verdict_mismatches != 0:
    raise AssertionError(
        "Model-ready and raw execution verdicts differ.\n"
        f"Mismatches: {model_raw_verdict_mismatches}"
    )


if model_raw_duration_mismatches != 0:
    raise AssertionError(
        "Model-ready and raw execution durations differ.\n"
        f"Mismatches: {model_raw_duration_mismatches}"
    )


# ---------------------------------------------------------
# 10. Apply the frozen chronological ordering
# ---------------------------------------------------------

chronological_builds = (
    builds
    .sort_values(
        [
            "started_at",
            "Build",
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


chronological_builds[
    "build_order"
] = np.arange(
    1,
    len(
        chronological_builds
    ) + 1,
    dtype=np.int64,
)


TOTAL_BUILD_COUNT = int(
    len(
        chronological_builds
    )
)


TRAINING_BUILD_COUNT = int(
    np.floor(
        0.75
        * TOTAL_BUILD_COUNT
    )
)


EVALUATION_BUILD_COUNT = int(
    TOTAL_BUILD_COUNT
    - TRAINING_BUILD_COUNT
)


if TRAINING_BUILD_COUNT <= 0:
    raise AssertionError(
        "The training partition is empty."
    )


if EVALUATION_BUILD_COUNT <= 0:
    raise AssertionError(
        "The evaluation partition is empty."
    )


chronological_builds[
    "partition"
] = np.where(
    chronological_builds[
        "build_order"
    ] <= TRAINING_BUILD_COUNT,
    "TRAIN",
    "EVALUATION",
)


training_build_ids = (
    chronological_builds.loc[
        chronological_builds[
            "partition"
        ] == "TRAIN",
        "Build",
    ]
    .astype(int)
    .tolist()
)


evaluation_build_ids = (
    chronological_builds.loc[
        chronological_builds[
            "partition"
        ] == "EVALUATION",
        "Build",
    ]
    .astype(int)
    .tolist()
)


TRAINING_BUILD_IDS = list(
    training_build_ids
)


EVALUATION_BUILD_IDS = list(
    evaluation_build_ids
)


training_build_id_set = set(
    TRAINING_BUILD_IDS
)


evaluation_build_id_set = set(
    EVALUATION_BUILD_IDS
)


if training_build_id_set.intersection(
    evaluation_build_id_set
):
    raise AssertionError(
        "Training and evaluation build sets overlap."
    )


if (
    training_build_id_set
    | evaluation_build_id_set
) != known_build_ids:
    raise AssertionError(
        "The chronological partitions do not cover "
        "every build exactly once."
    )


build_order_lookup = (
    chronological_builds
    .set_index(
        "Build"
    )[
        "build_order"
    ]
    .to_dict()
)


build_partition_lookup = (
    chronological_builds
    .set_index(
        "Build"
    )[
        "partition"
    ]
    .to_dict()
)


# ---------------------------------------------------------
# 11. Validate equal-timestamp ordering
# ---------------------------------------------------------

equal_timestamp_records = []
equal_timestamp_order_violations = 0


for timestamp, timestamp_rows in (
    chronological_builds.groupby(
        "started_at",
        sort=False,
    )
):
    if len(timestamp_rows) <= 1:
        continue


    observed_build_sequence = (
        timestamp_rows[
            "Build"
        ].astype(int).tolist()
    )


    expected_build_sequence = sorted(
        observed_build_sequence,
        reverse=True,
    )


    ordering_matches = bool(
        observed_build_sequence
        == expected_build_sequence
    )


    if not ordering_matches:
        equal_timestamp_order_violations += 1


    equal_timestamp_records.append({
        "started_at":
            timestamp.isoformat(),

        "BuildCount":
            int(
                len(
                    timestamp_rows
                )
            ),

        "BuildSequence":
            " | ".join(
                map(
                    str,
                    observed_build_sequence,
                )
            ),

        "ExpectedDescendingSequence":
            " | ".join(
                map(
                    str,
                    expected_build_sequence,
                )
            ),

        "OrderingMatches":
            ordering_matches,
    })


equal_timestamp_groups = pd.DataFrame(
    equal_timestamp_records
)


if equal_timestamp_order_violations != 0:
    raise AssertionError(
        "At least one equal-timestamp group violates "
        "the frozen descending Build-ID order."
    )


# ---------------------------------------------------------
# 12. Attach build order and partition
# ---------------------------------------------------------

exe[
    "build_order"
] = exe[
    "Build"
].map(
    build_order_lookup
)


exe[
    "partition"
] = exe[
    "Build"
].map(
    build_partition_lookup
)


dataset[
    "build_order"
] = dataset[
    "Build"
].map(
    build_order_lookup
)


dataset[
    "partition"
] = dataset[
    "Build"
].map(
    build_partition_lookup
)


if exe[
    [
        "build_order",
        "partition",
    ]
].isna().any().any():
    raise AssertionError(
        "Execution rows could not all be assigned to a "
        "chronological partition."
    )


if dataset[
    [
        "build_order",
        "partition",
    ]
].isna().any().any():
    raise AssertionError(
        "Model-ready rows could not all be assigned to "
        "a chronological partition."
    )


exe[
    "build_order"
] = exe[
    "build_order"
].astype(np.int64)


dataset[
    "build_order"
] = dataset[
    "build_order"
].astype(np.int64)


# ---------------------------------------------------------
# 13. Create raw-history partitions
# ---------------------------------------------------------

raw_training_history = (
    exe[
        exe[
            "partition"
        ] == "TRAIN"
    ]
    .copy()
    .reset_index(drop=True)
)


raw_evaluation_history = (
    exe[
        exe[
            "partition"
        ] == "EVALUATION"
    ]
    .copy()
    .reset_index(drop=True)
)


clean_training_history = (
    raw_training_history.copy(
        deep=True
    )
)


clean_evaluation_history = (
    raw_evaluation_history.copy(
        deep=True
    )
)


# ---------------------------------------------------------
# 14. Create model-ready partitions
# ---------------------------------------------------------

all_model_training_data = (
    dataset[
        dataset[
            "partition"
        ] == "TRAIN"
    ]
    .copy()
    .reset_index(drop=True)
)


all_model_evaluation_data = (
    dataset[
        dataset[
            "partition"
        ] == "EVALUATION"
    ]
    .copy()
    .reset_index(drop=True)
)


training_build_failure_status = (
    all_model_training_data
    .groupby(
        "Build"
    )[
        "Verdict"
    ]
    .apply(
        lambda values:
            bool(
                (
                    values.to_numpy(
                        dtype=int
                    )
                    != 0
                ).any()
            )
    )
)


evaluation_build_failure_status = (
    all_model_evaluation_data
    .groupby(
        "Build"
    )[
        "Verdict"
    ]
    .apply(
        lambda values:
            bool(
                (
                    values.to_numpy(
                        dtype=int
                    )
                    != 0
                ).any()
            )
    )
)


failing_training_build_ids = (
    training_build_failure_status[
        training_build_failure_status
    ]
    .index
    .astype(int)
    .tolist()
)


failing_evaluation_build_ids = (
    evaluation_build_failure_status[
        evaluation_build_failure_status
    ]
    .index
    .astype(int)
    .tolist()
)


FAILING_TRAINING_BUILD_IDS = list(
    failing_training_build_ids
)


FAILING_EVALUATION_BUILD_IDS = list(
    failing_evaluation_build_ids
)


MODEL_FAILING_TRAINING_BUILD_COUNT = int(
    len(
        FAILING_TRAINING_BUILD_IDS
    )
)


MODEL_FAILING_EVALUATION_BUILD_COUNT = int(
    len(
        FAILING_EVALUATION_BUILD_IDS
    )
)


if MODEL_FAILING_TRAINING_BUILD_COUNT != (
    EXPECTED_FAILING_TRAINING_BUILDS
):
    raise AssertionError(
        "The chronological training failure count does "
        "not match the screening table.\n"
        f"Expected: "
        f"{EXPECTED_FAILING_TRAINING_BUILDS}\n"
        f"Observed: "
        f"{MODEL_FAILING_TRAINING_BUILD_COUNT}"
    )


if MODEL_FAILING_EVALUATION_BUILD_COUNT != (
    EXPECTED_FAILING_EVALUATION_BUILDS
):
    raise AssertionError(
        "The chronological evaluation failure count "
        "does not match the screening table.\n"
        f"Expected: "
        f"{EXPECTED_FAILING_EVALUATION_BUILDS}\n"
        f"Observed: "
        f"{MODEL_FAILING_EVALUATION_BUILD_COUNT}"
    )


if MODEL_FAILING_TRAINING_BUILD_COUNT < 10:
    raise AssertionError(
        "Project 7 does not satisfy the minimum of "
        "10 failing training builds."
    )


# Training uses all model-ready rows in the training
# partition. Evaluation metrics use only failing builds.
clean_training_data = (
    all_model_training_data.copy(
        deep=True
    )
)


clean_evaluation_data = (
    all_model_evaluation_data[
        all_model_evaluation_data[
            "Build"
        ].isin(
            FAILING_EVALUATION_BUILD_IDS
        )
    ]
    .copy()
    .reset_index(drop=True)
)


if int(
    clean_evaluation_data[
        "Build"
    ].nunique()
) != MODEL_FAILING_EVALUATION_BUILD_COUNT:
    raise AssertionError(
        "The clean evaluation cohort does not contain "
        "all failing evaluation builds."
    )


if int(
    (
        clean_evaluation_data[
            "Verdict"
        ].to_numpy(dtype=int)
        != 0
    ).sum()
) <= 0:
    raise AssertionError(
        "The clean evaluation cohort contains no "
        "failure executions."
    )


# ---------------------------------------------------------
# 15. Raw-history failure counts
# ---------------------------------------------------------

raw_training_failing_builds = int(
    raw_training_history
    .groupby(
        "Build"
    )[
        "Verdict"
    ]
    .apply(
        lambda values:
            bool(
                (
                    values.to_numpy(
                        dtype=int
                    )
                    != 0
                ).any()
            )
    )
    .sum()
)


raw_evaluation_failing_builds = int(
    raw_evaluation_history
    .groupby(
        "Build"
    )[
        "Verdict"
    ]
    .apply(
        lambda values:
            bool(
                (
                    values.to_numpy(
                        dtype=int
                    )
                    != 0
                ).any()
            )
    )
    .sum()
)


# ---------------------------------------------------------
# 16. Freeze clean-data hashes
# ---------------------------------------------------------

def dataframe_sha256(
    dataframe,
):
    row_hashes = pd.util.hash_pandas_object(
        dataframe,
        index=True,
        categorize=True,
    ).to_numpy(
        dtype=np.uint64
    )


    return hashlib.sha256(
        row_hashes.tobytes()
    ).hexdigest()


CLEAN_TRAINING_HISTORY_SHA256 = (
    dataframe_sha256(
        clean_training_history
    )
)


CLEAN_EVALUATION_HISTORY_SHA256 = (
    dataframe_sha256(
        clean_evaluation_history
    )
)


CLEAN_TRAINING_DATA_SHA256 = (
    dataframe_sha256(
        clean_training_data
    )
)


CLEAN_EVALUATION_DATA_SHA256 = (
    dataframe_sha256(
        clean_evaluation_data
    )
)


# ---------------------------------------------------------
# 17. Create table and column schema reports
# ---------------------------------------------------------

table_summary_records = []


for table_name, table_frame in (
    original_tables.items()
):
    table_summary_records.append({
        "Table":
            table_name,

        "Rows":
            int(
                len(
                    table_frame
                )
            ),

        "Columns":
            int(
                len(
                    table_frame.columns
                )
            ),

        "MissingValues":
            int(
                table_frame.isna().sum().sum()
            ),

        "DuplicateRows":
            int(
                table_frame.duplicated().sum()
            ),
    })


table_summary = pd.DataFrame(
    table_summary_records
)


column_schema_records = []


for table_name, table_frame in (
    original_tables.items()
):
    for column in table_frame.columns:
        column_schema_records.append({
            "Table":
                table_name,

            "Column":
                str(
                    column
                ),

            "Dtype":
                str(
                    table_frame[
                        column
                    ].dtype
                ),

            "Rows":
                int(
                    len(
                        table_frame
                    )
                ),

            "MissingValues":
                int(
                    table_frame[
                        column
                    ].isna().sum()
                ),

            "UniqueNonMissingValues":
                int(
                    table_frame[
                        column
                    ].nunique(
                        dropna=True
                    )
                ),
        })


column_schema = pd.DataFrame(
    column_schema_records
)


# ---------------------------------------------------------
# 18. Permanent Step 2 report paths
# ---------------------------------------------------------

PREFLIGHT_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_chronological_preflight_report.json"
)


TABLE_SUMMARY_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_table_summary.csv"
)


COLUMN_SCHEMA_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_column_schema.csv"
)


CHRONOLOGY_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_build_chronology.csv"
)


EQUAL_TIMESTAMP_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_equal_timestamp_groups.csv"
)


SOURCE_ALIGNMENT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_model_raw_alignment_summary.csv"
)


PREDICTOR_LIST_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_predictor_columns.csv"
)


table_summary.to_csv(
    TABLE_SUMMARY_PATH,
    index=False,
)


column_schema.to_csv(
    COLUMN_SCHEMA_PATH,
    index=False,
)


chronological_builds.to_csv(
    CHRONOLOGY_PATH,
    index=False,
)


equal_timestamp_groups.to_csv(
    EQUAL_TIMESTAMP_PATH,
    index=False,
)


predictor_list_table = pd.DataFrame({
    "PredictorOrder":
        np.arange(
            1,
            len(
                PREDICTOR_COLUMNS
            ) + 1,
            dtype=np.int64,
        ),

    "Predictor":
        PREDICTOR_COLUMNS,
})


predictor_list_table.to_csv(
    PREDICTOR_LIST_PATH,
    index=False,
)


source_alignment_summary = pd.DataFrame([
    {
        "ModelRows":
            int(
                len(
                    dataset
                )
            ),

        "MatchedRawRows":
            int(
                len(
                    matched_alignment
                )
            ),

        "ModelRowsMissingFromRaw":
            model_rows_missing_from_raw,

        "VerdictMismatches":
            model_raw_verdict_mismatches,

        "DurationMismatches":
            model_raw_duration_mismatches,

        "RawDuplicateKeyRows":
            exe_key_duplicate_rows,

        "ModelDuplicateKeyRows":
            dataset_key_duplicate_rows,
    }
])


source_alignment_summary.to_csv(
    SOURCE_ALIGNMENT_PATH,
    index=False,
)


# ---------------------------------------------------------
# 19. Save complete preflight report
# ---------------------------------------------------------

last_training_build_row = (
    chronological_builds.iloc[
        TRAINING_BUILD_COUNT - 1
    ]
)


first_evaluation_build_row = (
    chronological_builds.iloc[
        TRAINING_BUILD_COUNT
    ]
)


preflight_completed_at = (
    pd.Timestamp.utcnow().isoformat()
)


preflight_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "PASS",

    "SourceTables": {
        table_name: {
            "Rows":
                int(
                    len(
                        table_frame
                    )
                ),

            "Columns":
                int(
                    len(
                        table_frame.columns
                    )
                ),
        }
        for table_name, table_frame in (
            original_tables.items()
        )
    },

    "CoreColumns": (
        identified_core_columns
    ),

    "Chronology": {
        "Rule":
            (
                "started_at ascending; Build descending "
                "within equal timestamps"
            ),

        "TotalBuilds":
            TOTAL_BUILD_COUNT,

        "TrainingBuilds":
            TRAINING_BUILD_COUNT,

        "EvaluationBuilds":
            EVALUATION_BUILD_COUNT,

        "TrainingFraction":
            float(
                TRAINING_BUILD_COUNT
                / TOTAL_BUILD_COUNT
            ),

        "EqualTimestampGroups":
            int(
                len(
                    equal_timestamp_groups
                )
            ),

        "EqualTimestampOrderViolations":
            equal_timestamp_order_violations,

        "FirstBuild":
            int(
                chronological_builds.iloc[0][
                    "Build"
                ]
            ),

        "LastBuild":
            int(
                chronological_builds.iloc[-1][
                    "Build"
                ]
            ),

        "LastTrainingBuild":
            int(
                last_training_build_row[
                    "Build"
                ]
            ),

        "LastTrainingTimestamp":
            str(
                last_training_build_row[
                    "started_at"
                ]
            ),

        "FirstEvaluationBuild":
            int(
                first_evaluation_build_row[
                    "Build"
                ]
            ),

        "FirstEvaluationTimestamp":
            str(
                first_evaluation_build_row[
                    "started_at"
                ]
            ),
    },

    "RawExecutionHistory": {
        "Rows":
            int(
                len(
                    exe
                )
            ),

        "TrainingRows":
            int(
                len(
                    raw_training_history
                )
            ),

        "EvaluationRows":
            int(
                len(
                    raw_evaluation_history
                )
            ),

        "TrainingFailingBuilds":
            raw_training_failing_builds,

        "EvaluationFailingBuilds":
            raw_evaluation_failing_builds,

        "VerdictValues":
            exe_verdict_values,
    },

    "ModelDataset": {
        "Rows":
            int(
                len(
                    dataset
                )
            ),

        "Predictors":
            int(
                len(
                    PREDICTOR_COLUMNS
                )
            ),

        "PredictorMissingValues":
            predictor_missing_values,

        "PredictorInfiniteValues":
            predictor_infinite_values,

        "TrainingRows":
            int(
                len(
                    clean_training_data
                )
            ),

        "EvaluationPartitionRowsBeforeFailureFilter":
            int(
                len(
                    all_model_evaluation_data
                )
            ),

        "EvaluatedRows":
            int(
                len(
                    clean_evaluation_data
                )
            ),

        "TrainingUniqueBuilds":
            int(
                all_model_training_data[
                    "Build"
                ].nunique()
            ),

        "EvaluationPartitionUniqueBuilds":
            int(
                all_model_evaluation_data[
                    "Build"
                ].nunique()
            ),

        "EvaluatedFailingBuilds":
            int(
                clean_evaluation_data[
                    "Build"
                ].nunique()
            ),

        "FailingTrainingBuilds":
            MODEL_FAILING_TRAINING_BUILD_COUNT,

        "FailingEvaluationBuilds":
            MODEL_FAILING_EVALUATION_BUILD_COUNT,

        "TrainingFailureExecutions":
            int(
                (
                    clean_training_data[
                        "Verdict"
                    ].to_numpy(dtype=int)
                    != 0
                ).sum()
            ),

        "EvaluationFailureExecutions":
            int(
                (
                    clean_evaluation_data[
                        "Verdict"
                    ].to_numpy(dtype=int)
                    != 0
                ).sum()
            ),

        "VerdictValues":
            dataset_verdict_values,
    },

    "ScreeningValidation": {
        "ExpectedFailingTrainingBuilds":
            int(
                EXPECTED_FAILING_TRAINING_BUILDS
            ),

        "ObservedFailingTrainingBuilds":
            MODEL_FAILING_TRAINING_BUILD_COUNT,

        "ExpectedFailingEvaluationBuilds":
            int(
                EXPECTED_FAILING_EVALUATION_BUILDS
            ),

        "ObservedFailingEvaluationBuilds":
            MODEL_FAILING_EVALUATION_BUILD_COUNT,

        "ExactMatch":
            True,

        "MinimumTrainingFailureRequirementPassed":
            bool(
                MODEL_FAILING_TRAINING_BUILD_COUNT
                >= 10
            ),
    },

    "ModelRawAlignment": {
        "ModelRows":
            int(
                len(
                    dataset
                )
            ),

        "MatchedRows":
            int(
                len(
                    matched_alignment
                )
            ),

        "RowsMissingFromRaw":
            model_rows_missing_from_raw,

        "VerdictMismatches":
            model_raw_verdict_mismatches,

        "DurationMismatches":
            model_raw_duration_mismatches,

        "RawDuplicateKeyRows":
            exe_key_duplicate_rows,

        "ModelDuplicateKeyRows":
            dataset_key_duplicate_rows,
    },

    "FrozenCleanHashes": {
        "CleanTrainingHistorySHA256":
            CLEAN_TRAINING_HISTORY_SHA256,

        "CleanEvaluationHistorySHA256":
            CLEAN_EVALUATION_HISTORY_SHA256,

        "CleanTrainingDataSHA256":
            CLEAN_TRAINING_DATA_SHA256,

        "CleanEvaluationDataSHA256":
            CLEAN_EVALUATION_DATA_SHA256,
    },

    "Reports": {
        "TableSummary":
            str(
                TABLE_SUMMARY_PATH
            ),

        "ColumnSchema":
            str(
                COLUMN_SCHEMA_PATH
            ),

        "Chronology":
            str(
                CHRONOLOGY_PATH
            ),

        "EqualTimestampGroups":
            str(
                EQUAL_TIMESTAMP_PATH
            ),

        "SourceAlignment":
            str(
                SOURCE_ALIGNMENT_PATH
            ),

        "PredictorList":
            str(
                PREDICTOR_LIST_PATH
            ),
    },

    "LoadElapsedSeconds":
        load_elapsed_seconds,

    "CompletedAtUTC":
        preflight_completed_at,
}


with open(
    PREFLIGHT_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        preflight_report,
        report_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 20. Update Project 7 checkpoint atomically
# ---------------------------------------------------------

with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    project_7_checkpoint = json.load(
        checkpoint_file
    )


project_7_checkpoint.update({
    "Status":
        "PREFLIGHT_PASSED",

    "ChronologyRule":
        (
            "started_at ascending; Build descending "
            "within equal timestamps"
        ),

    "TotalBuilds":
        TOTAL_BUILD_COUNT,

    "TrainingBuilds":
        TRAINING_BUILD_COUNT,

    "EvaluationBuilds":
        EVALUATION_BUILD_COUNT,

    "FailingTrainingBuilds":
        MODEL_FAILING_TRAINING_BUILD_COUNT,

    "FailingEvaluationBuilds":
        MODEL_FAILING_EVALUATION_BUILD_COUNT,

    "RawTrainingRows":
        int(
            len(
                raw_training_history
            )
        ),

    "RawEvaluationRows":
        int(
            len(
                raw_evaluation_history
            )
        ),

    "ModelTrainingRows":
        int(
            len(
                clean_training_data
            )
        ),

    "ModelEvaluationRows":
        int(
            len(
                clean_evaluation_data
            )
        ),

    "PredictorCount":
        int(
            len(
                PREDICTOR_COLUMNS
            )
        ),

    "LastTrainingBuild":
        int(
            last_training_build_row[
                "Build"
            ]
        ),

    "FirstEvaluationBuild":
        int(
            first_evaluation_build_row[
                "Build"
            ]
        ),

    "CleanTrainingHistorySHA256":
        CLEAN_TRAINING_HISTORY_SHA256,

    "CleanEvaluationHistorySHA256":
        CLEAN_EVALUATION_HISTORY_SHA256,

    "CleanTrainingDataSHA256":
        CLEAN_TRAINING_DATA_SHA256,

    "CleanEvaluationDataSHA256":
        CLEAN_EVALUATION_DATA_SHA256,

    "ChronologicalPreflightReport":
        str(
            PREFLIGHT_REPORT_PATH
        ),

    "TableSummary":
        str(
            TABLE_SUMMARY_PATH
        ),

    "ColumnSchema":
        str(
            COLUMN_SCHEMA_PATH
        ),

    "Chronology":
        str(
            CHRONOLOGY_PATH
        ),

    "SourceAlignment":
        str(
            SOURCE_ALIGNMENT_PATH
        ),

    "PredictorList":
        str(
            PREDICTOR_LIST_PATH
        ),

    "UpdatedAtUTC":
        preflight_completed_at,
})


temporary_checkpoint_path = (
    PROJECT_7_SELECTION_CHECKPOINT
    .with_suffix(
        ".json.tmp"
    )
)


with open(
    temporary_checkpoint_path,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        project_7_checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


os.replace(
    temporary_checkpoint_path,
    PROJECT_7_SELECTION_CHECKPOINT,
)


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint_verification = json.load(
        checkpoint_file
    )


if checkpoint_verification.get(
    "Status"
) != "PREFLIGHT_PASSED":
    raise AssertionError(
        "The permanent Project 7 preflight checkpoint "
        "was not written correctly."
    )


# ---------------------------------------------------------
# 21. Compact final output
# ---------------------------------------------------------

clear_output(
    wait=True
)


print(
    "=== PROJECT 7 STEP 2 RESULT ==="
)


print(
    "\nSource-table dimensions:"
)

display(
    table_summary
)


print(
    "\nChronological split:"
)

print(
    "Total builds:",
    TOTAL_BUILD_COUNT
)

print(
    "Training builds:",
    TRAINING_BUILD_COUNT
)

print(
    "Evaluation builds:",
    EVALUATION_BUILD_COUNT
)

print(
    "Last training build:",
    int(
        last_training_build_row[
            "Build"
        ]
    )
)

print(
    "First evaluation build:",
    int(
        first_evaluation_build_row[
            "Build"
        ]
    )
)


print(
    "\nEqual-timestamp chronology:"
)

print(
    "Equal-timestamp groups:",
    len(
        equal_timestamp_groups
    )
)

print(
    "Ordering violations:",
    equal_timestamp_order_violations
)


if len(equal_timestamp_groups) > 0:
    display(
        equal_timestamp_groups.head(10)
    )


print(
    "\nRaw execution partitions:"
)

print(
    "Training rows:",
    len(
        raw_training_history
    )
)

print(
    "Evaluation rows:",
    len(
        raw_evaluation_history
    )
)

print(
    "Raw failing training builds:",
    raw_training_failing_builds
)

print(
    "Raw failing evaluation builds:",
    raw_evaluation_failing_builds
)


print(
    "\nModel-ready partitions:"
)

print(
    "Predictors:",
    len(
        PREDICTOR_COLUMNS
    )
)

print(
    "Training rows:",
    len(
        clean_training_data
    )
)

print(
    "Evaluation rows:",
    len(
        clean_evaluation_data
    )
)

print(
    "Failing training builds:",
    MODEL_FAILING_TRAINING_BUILD_COUNT
)

print(
    "Failing evaluation builds:",
    MODEL_FAILING_EVALUATION_BUILD_COUNT
)

print(
    "Training failure executions:",
    int(
        (
            clean_training_data[
                "Verdict"
            ].to_numpy(dtype=int)
            != 0
        ).sum()
    )
)

print(
    "Evaluation failure executions:",
    int(
        (
            clean_evaluation_data[
                "Verdict"
            ].to_numpy(dtype=int)
            != 0
        ).sum()
    )
)


print(
    "\nModel/raw source alignment:"
)

display(
    source_alignment_summary
)


print(
    "\nPredictor quality:"
)

print(
    "Predictor coercion failures:",
    predictor_coercion_failures
)

print(
    "Predictor missing values:",
    predictor_missing_values
)

print(
    "Predictor infinite values:",
    predictor_infinite_values
)


print(
    "\nScreening validation:"
)

print(
    "Expected failing training builds:",
    EXPECTED_FAILING_TRAINING_BUILDS
)

print(
    "Observed failing training builds:",
    MODEL_FAILING_TRAINING_BUILD_COUNT
)

print(
    "Expected failing evaluation builds:",
    EXPECTED_FAILING_EVALUATION_BUILDS
)

print(
    "Observed failing evaluation builds:",
    MODEL_FAILING_EVALUATION_BUILD_COUNT
)

print(
    "Exact screening match:",
    True
)


print(
    "\nPreflight report:"
)

print(
    PREFLIGHT_REPORT_PATH
)


print(
    "\nValidation status:",
    "PASS"
)


print(
    "\nSUCCESS: All six Beast2 tables were loaded."
)

print(
    "SUCCESS: The frozen chronological ordering and "
    "75/25 split were applied."
)

print(
    "SUCCESS: The screening counts matched exactly."
)

print(
    "SUCCESS: All model-ready rows matched the raw "
    "execution history."
)

print(
    "SUCCESS: All 150 predictor columns were frozen."
)

print(
    "SUCCESS: Clean training and evaluation objects "
    "were created and hashed."
)

print(
    "SUCCESS: Project 7 is ready for commit/entity "
    "schema inspection and canonical mapping."
)

=== PROJECT 7 STEP 2: SCHEMA AND CHRONOLOGICAL PREFLIGHT ===


RuntimeError: Could not identify required core columns:
builds.Build

Build columns:
id | commits | started_at

Execution columns:
test | build | job | verdict | duration

Dataset first columns:
Build | Test | TES_COM_CountDeclFunction | TES_COM_CountLine | TES_COM_CountLineBlank | TES_COM_CountLineCode | TES_COM_CountLineCodeDecl | TES_COM_CountLineCodeExe | TES_COM_CountLineComment | TES_COM_CountStmt | TES_COM_CountStmtDecl | TES_COM_CountStmtExe | TES_COM_RatioCommentToCode | TES_COM_MaxCyclomatic | TES_COM_MaxCyclomaticModified | TES_COM_MaxCyclomaticStrict | TES_COM_MaxEssential | TES_COM_MaxNesting | TES_COM_SumCyclomatic | TES_COM_SumCyclomaticModified

In [ ]:
# =========================================================
# PROJECT 7 — STEP 2
# CORRECTED SCHEMA AND CHRONOLOGICAL PREFLIGHT
#
# Beast2-specific correction:
#   builds.csv uses "id" as the build identifier.
#
# This cell:
#   - loads all six Beast2 tables
#   - maps builds.id -> Build
#   - standardises raw/model core columns
#   - validates raw/model execution alignment
#   - applies the frozen chronology
#   - creates the fixed 75%/25% split
#   - verifies screening counts 64/52
#   - freezes all 150 predictors
#   - creates clean training/evaluation objects
#   - saves the permanent preflight report
#
# It does not inject noise or train models.
# =========================================================

from pathlib import Path
import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd
from IPython.display import clear_output, display


print(
    "=== PROJECT 7 STEP 2: "
    "CORRECTED SCHEMA AND CHRONOLOGICAL PREFLIGHT ==="
)


# ---------------------------------------------------------
# 1. Confirm successful Step 1 state
# ---------------------------------------------------------

required_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",
    "PROJECT_SHORT_NAME",

    "PROJECT_RAW_RESULTS",
    "PROJECT_AGGREGATED_RESULTS",
    "PROJECT_LOGS",
    "PROJECT_PREFLIGHT_DIRECTORY",

    "PROJECT_7_SELECTION_CHECKPOINT",

    "BUILDS_PATH",
    "EXE_PATH",
    "DATASET_PATH",
    "ENTITY_HISTORY_PATH",
    "ID_MAP_PATH",
    "CONTRIBUTORS_PATH",

    "EXPECTED_FAILING_TRAINING_BUILDS",
    "EXPECTED_FAILING_EVALUATION_BUILDS",
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required Project 7 Step 1 objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nRerun only the successful corrected Step 1 cell."
    )


if PROJECT_NUMBER != 7:
    raise AssertionError(
        f"Expected Project 7, observed {PROJECT_NUMBER}."
    )


if PROJECT_NAME != "CompEvol@beast2":
    raise AssertionError(
        f"Unexpected project: {PROJECT_NAME}"
    )


if PROJECT_SLUG != "CompEvol__beast2":
    raise AssertionError(
        f"Unexpected slug: {PROJECT_SLUG}"
    )


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    existing_checkpoint = json.load(
        checkpoint_file
    )


allowed_previous_statuses = {
    "INITIALISATION_PASSED",
    "PREFLIGHT_PASSED",
}


if existing_checkpoint.get(
    "Status"
) not in allowed_previous_statuses:
    raise AssertionError(
        "Project 7 Step 1 has not passed.\n"
        f"Observed status: "
        f"{existing_checkpoint.get('Status')}"
    )


# ---------------------------------------------------------
# 2. Validate and load all six source tables
# ---------------------------------------------------------

source_paths = {
    "Builds": Path(BUILDS_PATH),
    "ExecutionHistory": Path(EXE_PATH),
    "ModelDataset": Path(DATASET_PATH),
    "EntityHistory": Path(ENTITY_HISTORY_PATH),
    "EntityIdMap": Path(ID_MAP_PATH),
    "Contributors": Path(CONTRIBUTORS_PATH),
}


missing_paths = [
    str(path)
    for path in source_paths.values()
    if not path.exists()
]


if missing_paths:
    raise FileNotFoundError(
        "Required Beast2 source files are missing:\n"
        + "\n".join(missing_paths)
        + "\n\nDo not upload the dataset. Rerun Step 1."
    )


load_start_time = time.time()


builds_raw = pd.read_csv(
    BUILDS_PATH,
    low_memory=False,
)


exe_raw = pd.read_csv(
    EXE_PATH,
    low_memory=False,
)


dataset_raw = pd.read_csv(
    DATASET_PATH,
    low_memory=False,
)


entity_history_raw = pd.read_csv(
    ENTITY_HISTORY_PATH,
    low_memory=False,
)


id_map_raw = pd.read_csv(
    ID_MAP_PATH,
    low_memory=False,
)


contributors_raw = pd.read_csv(
    CONTRIBUTORS_PATH,
    low_memory=False,
)


load_elapsed_seconds = float(
    time.time() - load_start_time
)


original_tables = {
    "Builds": builds_raw,
    "ExecutionHistory": exe_raw,
    "ModelDataset": dataset_raw,
    "EntityHistory": entity_history_raw,
    "EntityIdMap": id_map_raw,
    "Contributors": contributors_raw,
}


for table_name, table in original_tables.items():
    if len(table) == 0:
        raise AssertionError(
            f"{table_name} contains no rows."
        )

    if len(table.columns) == 0:
        raise AssertionError(
            f"{table_name} contains no columns."
        )


# ---------------------------------------------------------
# 3. Robust column resolver
# ---------------------------------------------------------

def normalise_column_name(value):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).strip().lower(),
    )


def resolve_column(
    dataframe,
    accepted_names,
    logical_name,
):
    lookup = {
        normalise_column_name(column): column
        for column in dataframe.columns
    }


    for accepted_name in accepted_names:
        normalised_name = normalise_column_name(
            accepted_name
        )

        if normalised_name in lookup:
            return lookup[
                normalised_name
            ]


    raise RuntimeError(
        f"Could not resolve {logical_name}.\n"
        "Accepted names: "
        + " | ".join(accepted_names)
        + "\nObserved columns: "
        + " | ".join(
            map(str, dataframe.columns)
        )
    )


# Beast2 builds.csv specifically uses "id".
builds_build_column = resolve_column(
    builds_raw,
    [
        "id",
        "Build",
        "build",
        "BuildId",
        "BuildID",
        "build_id",
    ],
    "builds build identifier",
)


builds_started_column = resolve_column(
    builds_raw,
    [
        "started_at",
        "StartedAt",
        "start_time",
        "StartTime",
        "timestamp",
    ],
    "build start timestamp",
)


exe_build_column = resolve_column(
    exe_raw,
    [
        "build",
        "Build",
        "BuildId",
        "BuildID",
        "build_id",
    ],
    "execution build identifier",
)


exe_test_column = resolve_column(
    exe_raw,
    [
        "test",
        "Test",
        "TestId",
        "TestID",
        "test_id",
    ],
    "execution test identifier",
)


exe_verdict_column = resolve_column(
    exe_raw,
    [
        "verdict",
        "Verdict",
        "result",
        "outcome",
    ],
    "execution verdict",
)


exe_duration_column = resolve_column(
    exe_raw,
    [
        "duration",
        "Duration",
        "execution_time",
        "runtime",
    ],
    "execution duration",
)


dataset_build_column = resolve_column(
    dataset_raw,
    [
        "Build",
        "build",
        "BuildId",
        "BuildID",
        "build_id",
    ],
    "dataset build identifier",
)


dataset_test_column = resolve_column(
    dataset_raw,
    [
        "Test",
        "test",
        "TestId",
        "TestID",
        "test_id",
    ],
    "dataset test identifier",
)


dataset_verdict_column = resolve_column(
    dataset_raw,
    [
        "Verdict",
        "verdict",
        "Result",
        "result",
        "Outcome",
        "outcome",
    ],
    "dataset verdict",
)


dataset_duration_column = resolve_column(
    dataset_raw,
    [
        "Duration",
        "duration",
        "ExecutionTime",
        "execution_time",
        "Runtime",
        "runtime",
    ],
    "dataset duration",
)


identified_core_columns = {
    "builds.Build":
        builds_build_column,

    "builds.started_at":
        builds_started_column,

    "exe.Build":
        exe_build_column,

    "exe.Test":
        exe_test_column,

    "exe.Verdict":
        exe_verdict_column,

    "exe.Duration":
        exe_duration_column,

    "dataset.Build":
        dataset_build_column,

    "dataset.Test":
        dataset_test_column,

    "dataset.Verdict":
        dataset_verdict_column,

    "dataset.Duration":
        dataset_duration_column,
}


# ---------------------------------------------------------
# 4. Canonicalise the core columns
# ---------------------------------------------------------

builds = builds_raw.rename(
    columns={
        builds_build_column:
            "Build",

        builds_started_column:
            "started_at",
    }
).copy()


exe = exe_raw.rename(
    columns={
        exe_build_column:
            "Build",

        exe_test_column:
            "Test",

        exe_verdict_column:
            "Verdict",

        exe_duration_column:
            "Duration",
    }
).copy()


dataset = dataset_raw.rename(
    columns={
        dataset_build_column:
            "Build",

        dataset_test_column:
            "Test",

        dataset_verdict_column:
            "Verdict",

        dataset_duration_column:
            "Duration",
    }
).copy()


entity_history = entity_history_raw.copy()
id_map = id_map_raw.copy()
contributors = contributors_raw.copy()


required_build_columns = {
    "Build",
    "started_at",
}


required_execution_columns = {
    "Build",
    "Test",
    "Verdict",
    "Duration",
}


if not required_build_columns.issubset(
    builds.columns
):
    raise AssertionError(
        "Canonical builds columns were not created."
    )


if not required_execution_columns.issubset(
    exe.columns
):
    raise AssertionError(
        "Canonical execution columns were not created."
    )


if not required_execution_columns.issubset(
    dataset.columns
):
    raise AssertionError(
        "Canonical dataset columns were not created."
    )


# ---------------------------------------------------------
# 5. Core type conversion
# ---------------------------------------------------------

def to_integer_series(
    series,
    logical_name,
):
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )


    if numeric.isna().any():
        raise AssertionError(
            f"{logical_name} contains missing or "
            "non-numeric values."
        )


    values = numeric.to_numpy(
        dtype=float
    )


    if not np.isfinite(values).all():
        raise AssertionError(
            f"{logical_name} contains non-finite values."
        )


    if not np.isclose(
        values,
        np.round(values),
        rtol=0.0,
        atol=1e-12,
    ).all():
        raise AssertionError(
            f"{logical_name} contains non-integer values."
        )


    return pd.Series(
        np.round(values).astype(np.int64),
        index=series.index,
        name=series.name,
    )


def canonicalise_test_series(
    series,
    logical_name,
):
    if series.isna().any():
        raise AssertionError(
            f"{logical_name} contains missing values."
        )


    def canonicalise_one_test(value):
        text = str(value).strip()


        # Prevent equivalent numeric test IDs such as
        # "431" and "431.0" from failing alignment.
        if re.fullmatch(
            r"[+-]?\d+\.0+",
            text,
        ):
            return str(
                int(float(text))
            )


        return text


    result = series.map(
        canonicalise_one_test
    )


    if (
        result.str.len() == 0
    ).any():
        raise AssertionError(
            f"{logical_name} contains empty test IDs."
        )


    return result


builds[
    "Build"
] = to_integer_series(
    builds[
        "Build"
    ],
    "builds.Build",
)


exe[
    "Build"
] = to_integer_series(
    exe[
        "Build"
    ],
    "exe.Build",
)


dataset[
    "Build"
] = to_integer_series(
    dataset[
        "Build"
    ],
    "dataset.Build",
)


exe[
    "Verdict"
] = to_integer_series(
    exe[
        "Verdict"
    ],
    "exe.Verdict",
)


dataset[
    "Verdict"
] = to_integer_series(
    dataset[
        "Verdict"
    ],
    "dataset.Verdict",
)


exe[
    "Test"
] = canonicalise_test_series(
    exe[
        "Test"
    ],
    "exe.Test",
)


dataset[
    "Test"
] = canonicalise_test_series(
    dataset[
        "Test"
    ],
    "dataset.Test",
)


for frame_name, frame in [
    ("exe", exe),
    ("dataset", dataset),
]:
    frame[
        "Duration"
    ] = pd.to_numeric(
        frame[
            "Duration"
        ],
        errors="coerce",
    )


    if frame[
        "Duration"
    ].isna().any():
        raise AssertionError(
            f"{frame_name}.Duration contains missing "
            "or non-numeric values."
        )


    duration_values = frame[
        "Duration"
    ].to_numpy(
        dtype=float
    )


    if not np.isfinite(
        duration_values
    ).all():
        raise AssertionError(
            f"{frame_name}.Duration contains non-finite "
            "values."
        )


    if (
        duration_values < 0
    ).any():
        raise AssertionError(
            f"{frame_name}.Duration contains negative "
            "values."
        )


builds[
    "started_at"
] = pd.to_datetime(
    builds[
        "started_at"
    ],
    errors="coerce",
    utc=True,
)


if builds[
    "started_at"
].isna().any():
    raise AssertionError(
        "At least one build timestamp could not be parsed."
    )


# ---------------------------------------------------------
# 6. Basic integrity validation
# ---------------------------------------------------------

if builds[
    "Build"
].duplicated().any():
    duplicated_builds = (
        builds.loc[
            builds[
                "Build"
            ].duplicated(
                keep=False
            ),
            "Build",
        ]
        .astype(int)
        .tolist()
    )


    raise AssertionError(
        "builds.csv contains duplicate build IDs:\n"
        + ", ".join(
            map(str, duplicated_builds[:20])
        )
    )


known_build_ids = set(
    builds[
        "Build"
    ].astype(int)
)


unknown_execution_builds = sorted(
    set(
        exe[
            "Build"
        ].astype(int)
    )
    - known_build_ids
)


unknown_dataset_builds = sorted(
    set(
        dataset[
            "Build"
        ].astype(int)
    )
    - known_build_ids
)


if unknown_execution_builds:
    raise AssertionError(
        "exe.csv contains builds absent from builds.csv:\n"
        + ", ".join(
            map(str, unknown_execution_builds[:20])
        )
    )


if unknown_dataset_builds:
    raise AssertionError(
        "dataset.csv contains builds absent from "
        "builds.csv:\n"
        + ", ".join(
            map(str, unknown_dataset_builds[:20])
        )
    )


exe_verdict_values = sorted(
    exe[
        "Verdict"
    ].astype(int).unique().tolist()
)


dataset_verdict_values = sorted(
    dataset[
        "Verdict"
    ].astype(int).unique().tolist()
)


if 0 not in exe_verdict_values:
    raise AssertionError(
        "Raw execution history contains no passes."
    )


if 0 not in dataset_verdict_values:
    raise AssertionError(
        "Model dataset contains no passes."
    )


if not any(
    value != 0
    for value in dataset_verdict_values
):
    raise AssertionError(
        "Model dataset contains no failures."
    )


# ---------------------------------------------------------
# 7. Freeze the 150 predictor columns
# ---------------------------------------------------------

MODEL_CORE_COLUMNS = [
    "Build",
    "Test",
    "Verdict",
    "Duration",
]


PREDICTOR_COLUMNS = [
    column
    for column in dataset.columns
    if column not in MODEL_CORE_COLUMNS
]


if len(PREDICTOR_COLUMNS) != 150:
    raise AssertionError(
        "Unexpected predictor count.\n"
        f"Expected: 150\n"
        f"Observed: {len(PREDICTOR_COLUMNS)}\n"
        "Dataset columns: "
        + " | ".join(
            map(str, dataset.columns)
        )
    )


predictor_coercion_failures = 0


for predictor_column in PREDICTOR_COLUMNS:
    original_values = dataset[
        predictor_column
    ]


    numeric_values = pd.to_numeric(
        original_values,
        errors="coerce",
    )


    predictor_coercion_failures += int(
        (
            numeric_values.isna()
            & original_values.notna()
        ).sum()
    )


    dataset[
        predictor_column
    ] = numeric_values


if predictor_coercion_failures != 0:
    raise AssertionError(
        "Predictor values failed numeric conversion.\n"
        f"Failed values: {predictor_coercion_failures}"
    )


predictor_missing_values = int(
    dataset[
        PREDICTOR_COLUMNS
    ].isna().sum().sum()
)


predictor_infinite_values = int(
    np.isinf(
        dataset[
            PREDICTOR_COLUMNS
        ].to_numpy(dtype=float)
    ).sum()
)


if predictor_infinite_values != 0:
    raise AssertionError(
        "Predictor matrix contains infinite values."
    )


# ---------------------------------------------------------
# 8. Validate model-ready rows against raw history
# ---------------------------------------------------------

exe_alignment = exe[
    [
        "Build",
        "Test",
        "Verdict",
        "Duration",
    ]
].copy()


dataset_alignment = dataset[
    [
        "Build",
        "Test",
        "Verdict",
        "Duration",
    ]
].copy()


exe_key_duplicate_rows = int(
    exe_alignment.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


dataset_key_duplicate_rows = int(
    dataset_alignment.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


# Occurrence indexing safely handles repeated Build-Test keys.
exe_alignment[
    "_Occurrence"
] = (
    exe_alignment
    .groupby(
        [
            "Build",
            "Test",
        ],
        sort=False,
    )
    .cumcount()
)


dataset_alignment[
    "_Occurrence"
] = (
    dataset_alignment
    .groupby(
        [
            "Build",
            "Test",
        ],
        sort=False,
    )
    .cumcount()
)


exe_alignment = exe_alignment.rename(
    columns={
        "Verdict":
            "RawVerdict",

        "Duration":
            "RawDuration",
    }
)


dataset_alignment = dataset_alignment.rename(
    columns={
        "Verdict":
            "ModelVerdict",

        "Duration":
            "ModelDuration",
    }
)


model_raw_alignment = (
    dataset_alignment
    .merge(
        exe_alignment,
        on=[
            "Build",
            "Test",
            "_Occurrence",
        ],
        how="left",
        indicator=True,
        validate="one_to_one",
    )
)


model_rows_missing_from_raw = int(
    (
        model_raw_alignment[
            "_merge"
        ] != "both"
    ).sum()
)


matched_alignment = (
    model_raw_alignment[
        model_raw_alignment[
            "_merge"
        ] == "both"
    ]
    .copy()
)


model_raw_verdict_mismatches = int(
    (
        matched_alignment[
            "ModelVerdict"
        ].to_numpy(dtype=np.int64)
        !=
        matched_alignment[
            "RawVerdict"
        ].to_numpy(dtype=np.int64)
    ).sum()
)


model_raw_duration_mismatches = int(
    (
        ~np.isclose(
            matched_alignment[
                "ModelDuration"
            ].to_numpy(dtype=float),

            matched_alignment[
                "RawDuration"
            ].to_numpy(dtype=float),

            rtol=1e-12,
            atol=1e-12,
            equal_nan=True,
        )
    ).sum()
)


if model_rows_missing_from_raw != 0:
    missing_examples = (
        model_raw_alignment[
            model_raw_alignment[
                "_merge"
            ] != "both"
        ][
            [
                "Build",
                "Test",
                "_Occurrence",
            ]
        ]
        .head(20)
    )


    display(
        missing_examples
    )


    raise AssertionError(
        "Model-ready rows are missing from raw history.\n"
        f"Missing rows: {model_rows_missing_from_raw}"
    )


if model_raw_verdict_mismatches != 0:
    raise AssertionError(
        "Model and raw verdicts differ.\n"
        f"Mismatches: {model_raw_verdict_mismatches}"
    )


if model_raw_duration_mismatches != 0:
    raise AssertionError(
        "Model and raw durations differ.\n"
        f"Mismatches: {model_raw_duration_mismatches}"
    )


# ---------------------------------------------------------
# 9. Apply frozen chronological order
# ---------------------------------------------------------

chronological_builds = (
    builds
    .sort_values(
        [
            "started_at",
            "Build",
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


chronological_builds[
    "build_order"
] = np.arange(
    1,
    len(chronological_builds) + 1,
    dtype=np.int64,
)


TOTAL_BUILD_COUNT = int(
    len(chronological_builds)
)


TRAINING_BUILD_COUNT = int(
    np.floor(
        0.75 * TOTAL_BUILD_COUNT
    )
)


EVALUATION_BUILD_COUNT = int(
    TOTAL_BUILD_COUNT
    - TRAINING_BUILD_COUNT
)


if TRAINING_BUILD_COUNT <= 0:
    raise AssertionError(
        "Training partition is empty."
    )


if EVALUATION_BUILD_COUNT <= 0:
    raise AssertionError(
        "Evaluation partition is empty."
    )


chronological_builds[
    "partition"
] = np.where(
    chronological_builds[
        "build_order"
    ] <= TRAINING_BUILD_COUNT,
    "TRAIN",
    "EVALUATION",
)


TRAINING_BUILD_IDS = (
    chronological_builds.loc[
        chronological_builds[
            "partition"
        ] == "TRAIN",
        "Build",
    ]
    .astype(int)
    .tolist()
)


EVALUATION_BUILD_IDS = (
    chronological_builds.loc[
        chronological_builds[
            "partition"
        ] == "EVALUATION",
        "Build",
    ]
    .astype(int)
    .tolist()
)


training_build_set = set(
    TRAINING_BUILD_IDS
)


evaluation_build_set = set(
    EVALUATION_BUILD_IDS
)


if training_build_set.intersection(
    evaluation_build_set
):
    raise AssertionError(
        "Training and evaluation builds overlap."
    )


if (
    training_build_set
    | evaluation_build_set
) != known_build_ids:
    raise AssertionError(
        "The split does not cover all builds."
    )


build_order_lookup = (
    chronological_builds
    .set_index(
        "Build"
    )[
        "build_order"
    ]
    .to_dict()
)


build_partition_lookup = (
    chronological_builds
    .set_index(
        "Build"
    )[
        "partition"
    ]
    .to_dict()
)


# ---------------------------------------------------------
# 10. Validate equal-timestamp ordering
# ---------------------------------------------------------

equal_timestamp_records = []
equal_timestamp_order_violations = 0


for timestamp, timestamp_rows in (
    chronological_builds.groupby(
        "started_at",
        sort=False,
    )
):
    if len(timestamp_rows) <= 1:
        continue


    observed_sequence = (
        timestamp_rows[
            "Build"
        ].astype(int).tolist()
    )


    expected_sequence = sorted(
        observed_sequence,
        reverse=True,
    )


    ordering_matches = bool(
        observed_sequence
        == expected_sequence
    )


    if not ordering_matches:
        equal_timestamp_order_violations += 1


    equal_timestamp_records.append({
        "started_at":
            timestamp.isoformat(),

        "BuildCount":
            int(len(timestamp_rows)),

        "BuildSequence":
            " | ".join(
                map(str, observed_sequence)
            ),

        "ExpectedDescendingSequence":
            " | ".join(
                map(str, expected_sequence)
            ),

        "OrderingMatches":
            ordering_matches,
    })


equal_timestamp_groups = pd.DataFrame(
    equal_timestamp_records,
    columns=[
        "started_at",
        "BuildCount",
        "BuildSequence",
        "ExpectedDescendingSequence",
        "OrderingMatches",
    ],
)


if equal_timestamp_order_violations != 0:
    raise AssertionError(
        "Equal-timestamp build ordering failed."
    )


# ---------------------------------------------------------
# 11. Attach chronology to raw and model-ready rows
# ---------------------------------------------------------

for frame_name, frame in [
    ("exe", exe),
    ("dataset", dataset),
]:
    frame[
        "build_order"
    ] = frame[
        "Build"
    ].map(
        build_order_lookup
    )


    frame[
        "partition"
    ] = frame[
        "Build"
    ].map(
        build_partition_lookup
    )


    if frame[
        [
            "build_order",
            "partition",
        ]
    ].isna().any().any():
        raise AssertionError(
            f"{frame_name} rows could not all be assigned "
            "to the chronological split."
        )


    frame[
        "build_order"
    ] = frame[
        "build_order"
    ].astype(np.int64)


# ---------------------------------------------------------
# 12. Create raw-history partitions
# ---------------------------------------------------------

raw_training_history = (
    exe[
        exe[
            "partition"
        ] == "TRAIN"
    ]
    .copy()
    .reset_index(drop=True)
)


raw_evaluation_history = (
    exe[
        exe[
            "partition"
        ] == "EVALUATION"
    ]
    .copy()
    .reset_index(drop=True)
)


clean_training_history = (
    raw_training_history.copy(
        deep=True
    )
)


clean_evaluation_history = (
    raw_evaluation_history.copy(
        deep=True
    )
)


# ---------------------------------------------------------
# 13. Create model-ready partitions
# ---------------------------------------------------------

all_model_training_data = (
    dataset[
        dataset[
            "partition"
        ] == "TRAIN"
    ]
    .copy()
    .reset_index(drop=True)
)


all_model_evaluation_data = (
    dataset[
        dataset[
            "partition"
        ] == "EVALUATION"
    ]
    .copy()
    .reset_index(drop=True)
)


training_failure_by_build = (
    all_model_training_data
    .groupby(
        "Build"
    )[
        "Verdict"
    ]
    .apply(
        lambda values:
            bool(
                (
                    values.to_numpy(dtype=int)
                    != 0
                ).any()
            )
    )
)


evaluation_failure_by_build = (
    all_model_evaluation_data
    .groupby(
        "Build"
    )[
        "Verdict"
    ]
    .apply(
        lambda values:
            bool(
                (
                    values.to_numpy(dtype=int)
                    != 0
                ).any()
            )
    )
)


FAILING_TRAINING_BUILD_IDS = (
    training_failure_by_build[
        training_failure_by_build
    ]
    .index
    .astype(int)
    .tolist()
)


FAILING_EVALUATION_BUILD_IDS = (
    evaluation_failure_by_build[
        evaluation_failure_by_build
    ]
    .index
    .astype(int)
    .tolist()
)


MODEL_FAILING_TRAINING_BUILD_COUNT = int(
    len(
        FAILING_TRAINING_BUILD_IDS
    )
)


MODEL_FAILING_EVALUATION_BUILD_COUNT = int(
    len(
        FAILING_EVALUATION_BUILD_IDS
    )
)


if MODEL_FAILING_TRAINING_BUILD_COUNT != int(
    EXPECTED_FAILING_TRAINING_BUILDS
):
    raise AssertionError(
        "Training failing-build count differs from "
        "screening.\n"
        f"Expected: {EXPECTED_FAILING_TRAINING_BUILDS}\n"
        f"Observed: {MODEL_FAILING_TRAINING_BUILD_COUNT}"
    )


if MODEL_FAILING_EVALUATION_BUILD_COUNT != int(
    EXPECTED_FAILING_EVALUATION_BUILDS
):
    raise AssertionError(
        "Evaluation failing-build count differs from "
        "screening.\n"
        f"Expected: {EXPECTED_FAILING_EVALUATION_BUILDS}\n"
        f"Observed: "
        f"{MODEL_FAILING_EVALUATION_BUILD_COUNT}"
    )


if MODEL_FAILING_TRAINING_BUILD_COUNT < 10:
    raise AssertionError(
        "Project does not satisfy the minimum of "
        "10 failing training builds."
    )


# All training rows are retained.
clean_training_data = (
    all_model_training_data.copy(
        deep=True
    )
)


# Only failing evaluation builds are evaluated.
clean_evaluation_data = (
    all_model_evaluation_data[
        all_model_evaluation_data[
            "Build"
        ].isin(
            FAILING_EVALUATION_BUILD_IDS
        )
    ]
    .copy()
    .reset_index(drop=True)
)


if int(
    clean_evaluation_data[
        "Build"
    ].nunique()
) != MODEL_FAILING_EVALUATION_BUILD_COUNT:
    raise AssertionError(
        "Clean evaluation cohort is incomplete."
    )


training_failure_executions = int(
    (
        clean_training_data[
            "Verdict"
        ].to_numpy(dtype=int)
        != 0
    ).sum()
)


evaluation_failure_executions = int(
    (
        clean_evaluation_data[
            "Verdict"
        ].to_numpy(dtype=int)
        != 0
    ).sum()
)


if training_failure_executions <= 0:
    raise AssertionError(
        "Training data contains no failure executions."
    )


if evaluation_failure_executions <= 0:
    raise AssertionError(
        "Evaluation data contains no failure executions."
    )


# ---------------------------------------------------------
# 14. Raw-history failing-build counts
# ---------------------------------------------------------

def count_failing_builds(
    history_frame,
):
    return int(
        history_frame
        .groupby(
            "Build"
        )[
            "Verdict"
        ]
        .apply(
            lambda values:
                bool(
                    (
                        values.to_numpy(dtype=int)
                        != 0
                    ).any()
                )
        )
        .sum()
    )


raw_training_failing_builds = (
    count_failing_builds(
        raw_training_history
    )
)


raw_evaluation_failing_builds = (
    count_failing_builds(
        raw_evaluation_history
    )
)


# ---------------------------------------------------------
# 15. Freeze clean-data hashes
# ---------------------------------------------------------

def dataframe_sha256(
    dataframe,
):
    hashed_rows = pd.util.hash_pandas_object(
        dataframe,
        index=True,
        categorize=True,
    ).to_numpy(
        dtype=np.uint64
    )


    return hashlib.sha256(
        hashed_rows.tobytes()
    ).hexdigest()


CLEAN_TRAINING_HISTORY_SHA256 = (
    dataframe_sha256(
        clean_training_history
    )
)


CLEAN_EVALUATION_HISTORY_SHA256 = (
    dataframe_sha256(
        clean_evaluation_history
    )
)


CLEAN_TRAINING_DATA_SHA256 = (
    dataframe_sha256(
        clean_training_data
    )
)


CLEAN_EVALUATION_DATA_SHA256 = (
    dataframe_sha256(
        clean_evaluation_data
    )
)


# ---------------------------------------------------------
# 16. Create permanent reports
# ---------------------------------------------------------

TABLE_SUMMARY_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_table_summary.csv"
)


COLUMN_SCHEMA_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_column_schema.csv"
)


CHRONOLOGY_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_build_chronology.csv"
)


EQUAL_TIMESTAMP_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_equal_timestamp_groups.csv"
)


SOURCE_ALIGNMENT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_model_raw_alignment_summary.csv"
)


PREDICTOR_LIST_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_predictor_columns.csv"
)


PREFLIGHT_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_chronological_preflight_report.json"
)


table_summary = pd.DataFrame([
    {
        "Table":
            table_name,

        "Rows":
            int(len(table)),

        "Columns":
            int(len(table.columns)),

        "MissingValues":
            int(table.isna().sum().sum()),

        "DuplicateRows":
            int(table.duplicated().sum()),
    }
    for table_name, table in original_tables.items()
])


column_schema_records = []


for table_name, table in original_tables.items():
    for column in table.columns:
        column_schema_records.append({
            "Table":
                table_name,

            "Column":
                str(column),

            "Dtype":
                str(table[column].dtype),

            "MissingValues":
                int(
                    table[
                        column
                    ].isna().sum()
                ),

            "UniqueNonMissingValues":
                int(
                    table[
                        column
                    ].nunique(
                        dropna=True
                    )
                ),
        })


column_schema = pd.DataFrame(
    column_schema_records
)


source_alignment_summary = pd.DataFrame([
    {
        "ModelRows":
            int(len(dataset)),

        "MatchedRawRows":
            int(len(matched_alignment)),

        "ModelRowsMissingFromRaw":
            model_rows_missing_from_raw,

        "VerdictMismatches":
            model_raw_verdict_mismatches,

        "DurationMismatches":
            model_raw_duration_mismatches,

        "RawDuplicateKeyRows":
            exe_key_duplicate_rows,

        "ModelDuplicateKeyRows":
            dataset_key_duplicate_rows,
    }
])


predictor_list_table = pd.DataFrame({
    "PredictorOrder":
        np.arange(
            1,
            len(PREDICTOR_COLUMNS) + 1,
            dtype=np.int64,
        ),

    "Predictor":
        PREDICTOR_COLUMNS,
})


table_summary.to_csv(
    TABLE_SUMMARY_PATH,
    index=False,
)


column_schema.to_csv(
    COLUMN_SCHEMA_PATH,
    index=False,
)


chronological_builds.to_csv(
    CHRONOLOGY_PATH,
    index=False,
)


equal_timestamp_groups.to_csv(
    EQUAL_TIMESTAMP_PATH,
    index=False,
)


source_alignment_summary.to_csv(
    SOURCE_ALIGNMENT_PATH,
    index=False,
)


predictor_list_table.to_csv(
    PREDICTOR_LIST_PATH,
    index=False,
)


last_training_build_row = (
    chronological_builds.iloc[
        TRAINING_BUILD_COUNT - 1
    ]
)


first_evaluation_build_row = (
    chronological_builds.iloc[
        TRAINING_BUILD_COUNT
    ]
)


preflight_completed_at = (
    pd.Timestamp.utcnow().isoformat()
)


preflight_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "PASS",

    "Beast2SchemaCorrection": {
        "BuildIdentifierOriginalColumn":
            builds_build_column,

        "BuildIdentifierCanonicalColumn":
            "Build",

        "ExecutionOriginalColumns": {
            "Build":
                exe_build_column,

            "Test":
                exe_test_column,

            "Verdict":
                exe_verdict_column,

            "Duration":
                exe_duration_column,
        },
    },

    "SourceTables": {
        table_name: {
            "Rows":
                int(len(table)),

            "Columns":
                int(len(table.columns)),
        }
        for table_name, table in original_tables.items()
    },

    "Chronology": {
        "Rule":
            (
                "started_at ascending; Build descending "
                "within equal timestamps"
            ),

        "TotalBuilds":
            TOTAL_BUILD_COUNT,

        "TrainingBuilds":
            TRAINING_BUILD_COUNT,

        "EvaluationBuilds":
            EVALUATION_BUILD_COUNT,

        "EqualTimestampGroups":
            int(len(equal_timestamp_groups)),

        "EqualTimestampOrderViolations":
            equal_timestamp_order_violations,

        "LastTrainingBuild":
            int(
                last_training_build_row[
                    "Build"
                ]
            ),

        "FirstEvaluationBuild":
            int(
                first_evaluation_build_row[
                    "Build"
                ]
            ),
    },

    "RawExecutionHistory": {
        "Rows":
            int(len(exe)),

        "TrainingRows":
            int(len(raw_training_history)),

        "EvaluationRows":
            int(len(raw_evaluation_history)),

        "TrainingFailingBuilds":
            raw_training_failing_builds,

        "EvaluationFailingBuilds":
            raw_evaluation_failing_builds,

        "VerdictValues":
            exe_verdict_values,
    },

    "ModelDataset": {
        "Rows":
            int(len(dataset)),

        "Predictors":
            int(len(PREDICTOR_COLUMNS)),

        "PredictorMissingValues":
            predictor_missing_values,

        "PredictorInfiniteValues":
            predictor_infinite_values,

        "TrainingRows":
            int(len(clean_training_data)),

        "EvaluationPartitionRows":
            int(len(all_model_evaluation_data)),

        "EvaluatedRows":
            int(len(clean_evaluation_data)),

        "FailingTrainingBuilds":
            MODEL_FAILING_TRAINING_BUILD_COUNT,

        "FailingEvaluationBuilds":
            MODEL_FAILING_EVALUATION_BUILD_COUNT,

        "TrainingFailureExecutions":
            training_failure_executions,

        "EvaluationFailureExecutions":
            evaluation_failure_executions,

        "VerdictValues":
            dataset_verdict_values,
    },

    "ScreeningValidation": {
        "ExpectedFailingTrainingBuilds":
            int(
                EXPECTED_FAILING_TRAINING_BUILDS
            ),

        "ObservedFailingTrainingBuilds":
            MODEL_FAILING_TRAINING_BUILD_COUNT,

        "ExpectedFailingEvaluationBuilds":
            int(
                EXPECTED_FAILING_EVALUATION_BUILDS
            ),

        "ObservedFailingEvaluationBuilds":
            MODEL_FAILING_EVALUATION_BUILD_COUNT,

        "ExactMatch":
            True,

        "MinimumTrainingRequirementPassed":
            bool(
                MODEL_FAILING_TRAINING_BUILD_COUNT >= 10
            ),
    },

    "ModelRawAlignment": {
        "ModelRows":
            int(len(dataset)),

        "MatchedRows":
            int(len(matched_alignment)),

        "RowsMissingFromRaw":
            model_rows_missing_from_raw,

        "VerdictMismatches":
            model_raw_verdict_mismatches,

        "DurationMismatches":
            model_raw_duration_mismatches,
    },

    "FrozenCleanHashes": {
        "CleanTrainingHistorySHA256":
            CLEAN_TRAINING_HISTORY_SHA256,

        "CleanEvaluationHistorySHA256":
            CLEAN_EVALUATION_HISTORY_SHA256,

        "CleanTrainingDataSHA256":
            CLEAN_TRAINING_DATA_SHA256,

        "CleanEvaluationDataSHA256":
            CLEAN_EVALUATION_DATA_SHA256,
    },

    "Reports": {
        "TableSummary":
            str(TABLE_SUMMARY_PATH),

        "ColumnSchema":
            str(COLUMN_SCHEMA_PATH),

        "Chronology":
            str(CHRONOLOGY_PATH),

        "EqualTimestampGroups":
            str(EQUAL_TIMESTAMP_PATH),

        "SourceAlignment":
            str(SOURCE_ALIGNMENT_PATH),

        "PredictorList":
            str(PREDICTOR_LIST_PATH),
    },

    "LoadElapsedSeconds":
        load_elapsed_seconds,

    "CompletedAtUTC":
        preflight_completed_at,
}


with open(
    PREFLIGHT_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        preflight_report,
        report_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 17. Update checkpoint atomically
# ---------------------------------------------------------

with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    project_7_checkpoint = json.load(
        checkpoint_file
    )


project_7_checkpoint.update({
    "Status":
        "PREFLIGHT_PASSED",

    "BuildIdentifierOriginalColumn":
        builds_build_column,

    "BuildIdentifierCanonicalColumn":
        "Build",

    "ChronologyRule":
        (
            "started_at ascending; Build descending "
            "within equal timestamps"
        ),

    "TotalBuilds":
        TOTAL_BUILD_COUNT,

    "TrainingBuilds":
        TRAINING_BUILD_COUNT,

    "EvaluationBuilds":
        EVALUATION_BUILD_COUNT,

    "FailingTrainingBuilds":
        MODEL_FAILING_TRAINING_BUILD_COUNT,

    "FailingEvaluationBuilds":
        MODEL_FAILING_EVALUATION_BUILD_COUNT,

    "RawTrainingRows":
        int(len(raw_training_history)),

    "RawEvaluationRows":
        int(len(raw_evaluation_history)),

    "ModelTrainingRows":
        int(len(clean_training_data)),

    "ModelEvaluationRows":
        int(len(clean_evaluation_data)),

    "PredictorCount":
        int(len(PREDICTOR_COLUMNS)),

    "LastTrainingBuild":
        int(
            last_training_build_row[
                "Build"
            ]
        ),

    "FirstEvaluationBuild":
        int(
            first_evaluation_build_row[
                "Build"
            ]
        ),

    "CleanTrainingHistorySHA256":
        CLEAN_TRAINING_HISTORY_SHA256,

    "CleanEvaluationHistorySHA256":
        CLEAN_EVALUATION_HISTORY_SHA256,

    "CleanTrainingDataSHA256":
        CLEAN_TRAINING_DATA_SHA256,

    "CleanEvaluationDataSHA256":
        CLEAN_EVALUATION_DATA_SHA256,

    "ChronologicalPreflightReport":
        str(PREFLIGHT_REPORT_PATH),

    "TableSummary":
        str(TABLE_SUMMARY_PATH),

    "ColumnSchema":
        str(COLUMN_SCHEMA_PATH),

    "Chronology":
        str(CHRONOLOGY_PATH),

    "SourceAlignment":
        str(SOURCE_ALIGNMENT_PATH),

    "PredictorList":
        str(PREDICTOR_LIST_PATH),

    "UpdatedAtUTC":
        preflight_completed_at,
})


temporary_checkpoint_path = (
    PROJECT_7_SELECTION_CHECKPOINT
    .with_suffix(
        ".json.tmp"
    )
)


with open(
    temporary_checkpoint_path,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        project_7_checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


os.replace(
    temporary_checkpoint_path,
    PROJECT_7_SELECTION_CHECKPOINT,
)


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint_verification = json.load(
        checkpoint_file
    )


if checkpoint_verification.get(
    "Status"
) != "PREFLIGHT_PASSED":
    raise AssertionError(
        "The Project 7 preflight checkpoint was not "
        "written correctly."
    )


# ---------------------------------------------------------
# 18. Compact final output
# ---------------------------------------------------------

clear_output(
    wait=True
)


print(
    "=== PROJECT 7 STEP 2 RESULT ==="
)


print(
    "\nBeast2 schema correction:"
)

print(
    "builds.csv identifier:",
    builds_build_column
)

print(
    "Canonical identifier:",
    "Build"
)


print(
    "\nSource-table dimensions:"
)

display(
    table_summary
)


print(
    "\nChronological split:"
)

print(
    "Total builds:",
    TOTAL_BUILD_COUNT
)

print(
    "Training builds:",
    TRAINING_BUILD_COUNT
)

print(
    "Evaluation builds:",
    EVALUATION_BUILD_COUNT
)

print(
    "Last training build:",
    int(
        last_training_build_row[
            "Build"
        ]
    )
)

print(
    "First evaluation build:",
    int(
        first_evaluation_build_row[
            "Build"
        ]
    )
)


print(
    "\nEqual-timestamp chronology:"
)

print(
    "Equal-timestamp groups:",
    len(equal_timestamp_groups)
)

print(
    "Ordering violations:",
    equal_timestamp_order_violations
)


if len(equal_timestamp_groups) > 0:
    display(
        equal_timestamp_groups.head(10)
    )


print(
    "\nRaw execution partitions:"
)

print(
    "Training rows:",
    len(raw_training_history)
)

print(
    "Evaluation rows:",
    len(raw_evaluation_history)
)

print(
    "Raw failing training builds:",
    raw_training_failing_builds
)

print(
    "Raw failing evaluation builds:",
    raw_evaluation_failing_builds
)


print(
    "\nModel-ready partitions:"
)

print(
    "Predictors:",
    len(PREDICTOR_COLUMNS)
)

print(
    "Training rows:",
    len(clean_training_data)
)

print(
    "Evaluation rows:",
    len(clean_evaluation_data)
)

print(
    "Failing training builds:",
    MODEL_FAILING_TRAINING_BUILD_COUNT
)

print(
    "Failing evaluation builds:",
    MODEL_FAILING_EVALUATION_BUILD_COUNT
)

print(
    "Training failure executions:",
    training_failure_executions
)

print(
    "Evaluation failure executions:",
    evaluation_failure_executions
)


print(
    "\nModel/raw source alignment:"
)

display(
    source_alignment_summary
)


print(
    "\nPredictor quality:"
)

print(
    "Predictor coercion failures:",
    predictor_coercion_failures
)

print(
    "Predictor missing values:",
    predictor_missing_values
)

print(
    "Predictor infinite values:",
    predictor_infinite_values
)


print(
    "\nScreening validation:"
)

print(
    "Expected failing training builds:",
    EXPECTED_FAILING_TRAINING_BUILDS
)

print(
    "Observed failing training builds:",
    MODEL_FAILING_TRAINING_BUILD_COUNT
)

print(
    "Expected failing evaluation builds:",
    EXPECTED_FAILING_EVALUATION_BUILDS
)

print(
    "Observed failing evaluation builds:",
    MODEL_FAILING_EVALUATION_BUILD_COUNT
)

print(
    "Exact screening match:",
    True
)


print(
    "\nPreflight report:"
)

print(
    PREFLIGHT_REPORT_PATH
)


print(
    "\nValidation status:",
    "PASS"
)


print(
    "\nSUCCESS: builds.id was correctly mapped to Build."
)

print(
    "SUCCESS: All six Beast2 tables were loaded."
)

print(
    "SUCCESS: The frozen chronological ordering and "
    "75/25 split were applied."
)

print(
    "SUCCESS: The screening counts matched exactly."
)

print(
    "SUCCESS: All model-ready rows matched the raw "
    "execution history."
)

print(
    "SUCCESS: All 150 predictor columns were frozen."
)

print(
    "SUCCESS: Clean training and evaluation objects "
    "were created and hashed."
)

print(
    "SUCCESS: Project 7 is ready for commit/entity "
    "schema inspection and canonical mapping."
)

=== PROJECT 7 STEP 2: CORRECTED SCHEMA AND CHRONOLOGICAL PREFLIGHT ===


/tmp/ipykernel_1056/2275228141.py:1396: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  frame[
/tmp/ipykernel_1056/2275228141.py:1405: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  frame[


AssertionError: Training failing-build count differs from screening.
Expected: 64
Observed: 63

In [ ]:
# =========================================================
# PROJECT 7 — STEP 2B
# CORRECT SCREENING-BASIS VALIDATION AND COMPLETE PREFLIGHT
#
# Correction:
#   Screening failing-build counts are validated against
#   the full raw execution history (exe.csv), not against
#   the retained model-ready subset (dataset.csv).
#
# This cell resumes from the failed corrected Step 2 cell.
# It does not reload the archive or modify Projects 1–6.
# =========================================================

from pathlib import Path
import hashlib
import json
import os
import time

import numpy as np
import pandas as pd
from IPython.display import clear_output, display


print(
    "=== PROJECT 7 STEP 2B: "
    "CORRECT SCREENING-BASIS VALIDATION ==="
)


# ---------------------------------------------------------
# 1. Confirm the failed Step 2 created the required objects
# ---------------------------------------------------------

required_runtime_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",
    "PROJECT_SHORT_NAME",

    "PROJECT_PREFLIGHT_DIRECTORY",
    "PROJECT_7_SELECTION_CHECKPOINT",

    "EXPECTED_FAILING_TRAINING_BUILDS",
    "EXPECTED_FAILING_EVALUATION_BUILDS",

    "builds",
    "exe",
    "dataset",
    "original_tables",

    "chronological_builds",
    "TRAINING_BUILD_COUNT",
    "EVALUATION_BUILD_COUNT",
    "TOTAL_BUILD_COUNT",

    "raw_training_history",
    "raw_evaluation_history",
    "all_model_training_data",
    "all_model_evaluation_data",

    "PREDICTOR_COLUMNS",
    "predictor_coercion_failures",
    "predictor_missing_values",
    "predictor_infinite_values",

    "model_rows_missing_from_raw",
    "model_raw_verdict_mismatches",
    "model_raw_duration_mismatches",
    "exe_key_duplicate_rows",
    "dataset_key_duplicate_rows",
    "matched_alignment",

    "equal_timestamp_groups",
    "equal_timestamp_order_violations",

    "builds_build_column",
    "identified_core_columns",
]


missing_runtime_objects = [
    object_name
    for object_name in required_runtime_objects
    if object_name not in globals()
]


if missing_runtime_objects:
    raise RuntimeError(
        "The failed Step 2 runtime state is incomplete:\n"
        + "\n".join(missing_runtime_objects)
        + "\n\nRerun the corrected Step 2 cell only until it "
        "reaches the 64-versus-63 assertion, then run "
        "this Step 2B cell."
    )


if PROJECT_NUMBER != 7:
    raise AssertionError(
        f"Expected Project 7, observed {PROJECT_NUMBER}."
    )


if PROJECT_NAME != "CompEvol@beast2":
    raise AssertionError(
        f"Unexpected project: {PROJECT_NAME}"
    )


if int(
    EXPECTED_FAILING_TRAINING_BUILDS
) != 64:
    raise AssertionError(
        "Unexpected screening training count."
    )


if int(
    EXPECTED_FAILING_EVALUATION_BUILDS
) != 52:
    raise AssertionError(
        "Unexpected screening evaluation count."
    )


# Defragment the wide model-ready table after the earlier
# column operations. This also prevents further warnings.
builds = builds.copy()
exe = exe.copy()
dataset = dataset.copy()

raw_training_history = (
    raw_training_history.copy()
)

raw_evaluation_history = (
    raw_evaluation_history.copy()
)

all_model_training_data = (
    all_model_training_data.copy()
)

all_model_evaluation_data = (
    all_model_evaluation_data.copy()
)


# ---------------------------------------------------------
# 2. Calculate failing-build sets independently
# ---------------------------------------------------------

def get_failing_build_ids(
    dataframe,
):
    verdict_values = pd.to_numeric(
        dataframe[
            "Verdict"
        ],
        errors="raise",
    ).to_numpy(dtype=np.int64)


    failure_mask = (
        verdict_values != 0
    )


    return sorted(
        dataframe.loc[
            failure_mask,
            "Build",
        ]
        .astype(np.int64)
        .unique()
        .tolist()
    )


RAW_FAILING_TRAINING_BUILD_IDS = (
    get_failing_build_ids(
        raw_training_history
    )
)


RAW_FAILING_EVALUATION_BUILD_IDS = (
    get_failing_build_ids(
        raw_evaluation_history
    )
)


MODEL_FAILING_TRAINING_BUILD_IDS = (
    get_failing_build_ids(
        all_model_training_data
    )
)


MODEL_FAILING_EVALUATION_BUILD_IDS = (
    get_failing_build_ids(
        all_model_evaluation_data
    )
)


RAW_FAILING_TRAINING_BUILD_COUNT = int(
    len(
        RAW_FAILING_TRAINING_BUILD_IDS
    )
)


RAW_FAILING_EVALUATION_BUILD_COUNT = int(
    len(
        RAW_FAILING_EVALUATION_BUILD_IDS
    )
)


MODEL_FAILING_TRAINING_BUILD_COUNT = int(
    len(
        MODEL_FAILING_TRAINING_BUILD_IDS
    )
)


MODEL_FAILING_EVALUATION_BUILD_COUNT = int(
    len(
        MODEL_FAILING_EVALUATION_BUILD_IDS
    )
)


# ---------------------------------------------------------
# 3. Validate screening against the correct raw basis
# ---------------------------------------------------------

if RAW_FAILING_TRAINING_BUILD_COUNT != int(
    EXPECTED_FAILING_TRAINING_BUILDS
):
    raise AssertionError(
        "Raw execution-history training failure count "
        "does not match screening.\n"
        f"Expected: {EXPECTED_FAILING_TRAINING_BUILDS}\n"
        f"Observed: {RAW_FAILING_TRAINING_BUILD_COUNT}"
    )


if RAW_FAILING_EVALUATION_BUILD_COUNT != int(
    EXPECTED_FAILING_EVALUATION_BUILDS
):
    raise AssertionError(
        "Raw execution-history evaluation failure count "
        "does not match screening.\n"
        f"Expected: {EXPECTED_FAILING_EVALUATION_BUILDS}\n"
        f"Observed: {RAW_FAILING_EVALUATION_BUILD_COUNT}"
    )


if RAW_FAILING_TRAINING_BUILD_COUNT < 10:
    raise AssertionError(
        "Project 7 does not satisfy the minimum of "
        "10 failing raw training builds."
    )


if MODEL_FAILING_TRAINING_BUILD_COUNT < 10:
    raise AssertionError(
        "The retained model-ready training data contains "
        "fewer than 10 failing builds."
    )


if MODEL_FAILING_EVALUATION_BUILD_COUNT < 1:
    raise AssertionError(
        "The retained model-ready evaluation data "
        "contains no failing build."
    )


# ---------------------------------------------------------
# 4. Audit raw-versus-model failing-build differences
# ---------------------------------------------------------

raw_training_failure_set = set(
    RAW_FAILING_TRAINING_BUILD_IDS
)


raw_evaluation_failure_set = set(
    RAW_FAILING_EVALUATION_BUILD_IDS
)


model_training_failure_set = set(
    MODEL_FAILING_TRAINING_BUILD_IDS
)


model_evaluation_failure_set = set(
    MODEL_FAILING_EVALUATION_BUILD_IDS
)


RAW_ONLY_FAILING_TRAINING_BUILDS = sorted(
    raw_training_failure_set
    - model_training_failure_set
)


MODEL_ONLY_FAILING_TRAINING_BUILDS = sorted(
    model_training_failure_set
    - raw_training_failure_set
)


RAW_ONLY_FAILING_EVALUATION_BUILDS = sorted(
    raw_evaluation_failure_set
    - model_evaluation_failure_set
)


MODEL_ONLY_FAILING_EVALUATION_BUILDS = sorted(
    model_evaluation_failure_set
    - raw_evaluation_failure_set
)


if MODEL_ONLY_FAILING_TRAINING_BUILDS:
    raise AssertionError(
        "The model-ready training data contains a "
        "failing build that is not failing in the raw "
        "execution history:\n"
        + ", ".join(
            map(
                str,
                MODEL_ONLY_FAILING_TRAINING_BUILDS,
            )
        )
    )


if MODEL_ONLY_FAILING_EVALUATION_BUILDS:
    raise AssertionError(
        "The model-ready evaluation data contains a "
        "failing build that is not failing in the raw "
        "execution history:\n"
        + ", ".join(
            map(
                str,
                MODEL_ONLY_FAILING_EVALUATION_BUILDS,
            )
        )
    )


build_order_lookup = (
    chronological_builds
    .set_index(
        "Build"
    )[
        "build_order"
    ]
    .to_dict()
)


def create_failure_difference_rows(
    partition_name,
    raw_partition,
    model_partition,
    build_ids,
    difference_type,
):
    records = []


    for build_id in build_ids:
        raw_build_rows = (
            raw_partition[
                raw_partition[
                    "Build"
                ].astype(np.int64)
                == int(build_id)
            ]
        )


        model_build_rows = (
            model_partition[
                model_partition[
                    "Build"
                ].astype(np.int64)
                == int(build_id)
            ]
        )


        records.append({
            "Partition":
                partition_name,

            "Build":
                int(
                    build_id
                ),

            "BuildOrder":
                int(
                    build_order_lookup[
                        int(build_id)
                    ]
                ),

            "DifferenceType":
                difference_type,

            "RawRows":
                int(
                    len(
                        raw_build_rows
                    )
                ),

            "RawFailureExecutions":
                int(
                    (
                        pd.to_numeric(
                            raw_build_rows[
                                "Verdict"
                            ],
                            errors="raise",
                        ).to_numpy(dtype=np.int64)
                        != 0
                    ).sum()
                ),

            "ModelReadyRows":
                int(
                    len(
                        model_build_rows
                    )
                ),

            "ModelReadyFailureExecutions":
                int(
                    (
                        pd.to_numeric(
                            model_build_rows[
                                "Verdict"
                            ],
                            errors="raise",
                        ).to_numpy(dtype=np.int64)
                        != 0
                    ).sum()
                ),
        })


    return records


failure_difference_records = []


failure_difference_records.extend(
    create_failure_difference_rows(
        partition_name="TRAIN",
        raw_partition=raw_training_history,
        model_partition=all_model_training_data,
        build_ids=(
            RAW_ONLY_FAILING_TRAINING_BUILDS
        ),
        difference_type=(
            "RAW_FAILURE_NOT_RETAINED_AS_MODEL_FAILURE"
        ),
    )
)


failure_difference_records.extend(
    create_failure_difference_rows(
        partition_name="EVALUATION",
        raw_partition=raw_evaluation_history,
        model_partition=all_model_evaluation_data,
        build_ids=(
            RAW_ONLY_FAILING_EVALUATION_BUILDS
        ),
        difference_type=(
            "RAW_FAILURE_NOT_RETAINED_AS_MODEL_FAILURE"
        ),
    )
)


failure_build_difference_audit = pd.DataFrame(
    failure_difference_records,
    columns=[
        "Partition",
        "Build",
        "BuildOrder",
        "DifferenceType",
        "RawRows",
        "RawFailureExecutions",
        "ModelReadyRows",
        "ModelReadyFailureExecutions",
    ],
)


# ---------------------------------------------------------
# 5. Create the frozen clean model-ready cohorts
# ---------------------------------------------------------

clean_training_history = (
    raw_training_history.copy(
        deep=True
    )
)


clean_evaluation_history = (
    raw_evaluation_history.copy(
        deep=True
    )
)


# The fixed-instance protocol retains every model-ready
# training row, including builds without a retained failure.
clean_training_data = (
    all_model_training_data.copy(
        deep=True
    )
)


# Metrics require at least one retained failure execution,
# so the evaluation cohort uses model-ready failing builds.
clean_evaluation_data = (
    all_model_evaluation_data[
        all_model_evaluation_data[
            "Build"
        ].astype(np.int64)
        .isin(
            MODEL_FAILING_EVALUATION_BUILD_IDS
        )
    ]
    .copy()
    .reset_index(drop=True)
)


if int(
    clean_evaluation_data[
        "Build"
    ].nunique()
) != MODEL_FAILING_EVALUATION_BUILD_COUNT:
    raise AssertionError(
        "The clean model-ready evaluation cohort is "
        "incomplete."
    )


TRAINING_FAILURE_EXECUTIONS = int(
    (
        clean_training_data[
            "Verdict"
        ].to_numpy(dtype=np.int64)
        != 0
    ).sum()
)


EVALUATION_FAILURE_EXECUTIONS = int(
    (
        clean_evaluation_data[
            "Verdict"
        ].to_numpy(dtype=np.int64)
        != 0
    ).sum()
)


RAW_TRAINING_FAILURE_EXECUTIONS = int(
    (
        raw_training_history[
            "Verdict"
        ].to_numpy(dtype=np.int64)
        != 0
    ).sum()
)


RAW_EVALUATION_FAILURE_EXECUTIONS = int(
    (
        raw_evaluation_history[
            "Verdict"
        ].to_numpy(dtype=np.int64)
        != 0
    ).sum()
)


if TRAINING_FAILURE_EXECUTIONS <= 0:
    raise AssertionError(
        "The model-ready training data has no retained "
        "failure execution."
    )


if EVALUATION_FAILURE_EXECUTIONS <= 0:
    raise AssertionError(
        "The model-ready evaluation cohort has no "
        "retained failure execution."
    )


# ---------------------------------------------------------
# 6. Revalidate raw/model row alignment
# ---------------------------------------------------------

source_alignment_summary = pd.DataFrame([
    {
        "ModelRows":
            int(
                len(
                    dataset
                )
            ),

        "MatchedRawRows":
            int(
                len(
                    matched_alignment
                )
            ),

        "ModelRowsMissingFromRaw":
            int(
                model_rows_missing_from_raw
            ),

        "VerdictMismatches":
            int(
                model_raw_verdict_mismatches
            ),

        "DurationMismatches":
            int(
                model_raw_duration_mismatches
            ),

        "RawDuplicateKeyRows":
            int(
                exe_key_duplicate_rows
            ),

        "ModelDuplicateKeyRows":
            int(
                dataset_key_duplicate_rows
            ),
    }
])


if model_rows_missing_from_raw != 0:
    raise AssertionError(
        "Model-ready rows are missing from the raw "
        "execution history."
    )


if model_raw_verdict_mismatches != 0:
    raise AssertionError(
        "Matched raw/model verdict values differ."
    )


if model_raw_duration_mismatches != 0:
    raise AssertionError(
        "Matched raw/model duration values differ."
    )


# ---------------------------------------------------------
# 7. Freeze clean-object hashes
# ---------------------------------------------------------

def dataframe_sha256(
    dataframe,
):
    row_hashes = pd.util.hash_pandas_object(
        dataframe,
        index=True,
        categorize=True,
    ).to_numpy(
        dtype=np.uint64
    )


    return hashlib.sha256(
        row_hashes.tobytes()
    ).hexdigest()


CLEAN_TRAINING_HISTORY_SHA256 = (
    dataframe_sha256(
        clean_training_history
    )
)


CLEAN_EVALUATION_HISTORY_SHA256 = (
    dataframe_sha256(
        clean_evaluation_history
    )
)


CLEAN_TRAINING_DATA_SHA256 = (
    dataframe_sha256(
        clean_training_data
    )
)


CLEAN_EVALUATION_DATA_SHA256 = (
    dataframe_sha256(
        clean_evaluation_data
    )
)


# ---------------------------------------------------------
# 8. Create permanent report tables
# ---------------------------------------------------------

TABLE_SUMMARY_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_table_summary.csv"
)


COLUMN_SCHEMA_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_column_schema.csv"
)


CHRONOLOGY_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_build_chronology.csv"
)


EQUAL_TIMESTAMP_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_equal_timestamp_groups.csv"
)


SOURCE_ALIGNMENT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_model_raw_alignment_summary.csv"
)


PREDICTOR_LIST_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_predictor_columns.csv"
)


FAILURE_DIFFERENCE_AUDIT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_raw_model_failure_build_difference.csv"
)


PREFLIGHT_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_chronological_preflight_report.json"
)


table_summary = pd.DataFrame([
    {
        "Table":
            table_name,

        "Rows":
            int(
                len(
                    table
                )
            ),

        "Columns":
            int(
                len(
                    table.columns
                )
            ),

        "MissingValues":
            int(
                table.isna().sum().sum()
            ),

        "DuplicateRows":
            int(
                table.duplicated().sum()
            ),
    }
    for table_name, table in (
        original_tables.items()
    )
])


column_schema_records = []


for table_name, table in (
    original_tables.items()
):
    for column in table.columns:
        column_schema_records.append({
            "Table":
                table_name,

            "Column":
                str(
                    column
                ),

            "Dtype":
                str(
                    table[
                        column
                    ].dtype
                ),

            "MissingValues":
                int(
                    table[
                        column
                    ].isna().sum()
                ),

            "UniqueNonMissingValues":
                int(
                    table[
                        column
                    ].nunique(
                        dropna=True
                    )
                ),
        })


column_schema = pd.DataFrame(
    column_schema_records
)


predictor_list_table = pd.DataFrame({
    "PredictorOrder":
        np.arange(
            1,
            len(
                PREDICTOR_COLUMNS
            ) + 1,
            dtype=np.int64,
        ),

    "Predictor":
        PREDICTOR_COLUMNS,
})


table_summary.to_csv(
    TABLE_SUMMARY_PATH,
    index=False,
)


column_schema.to_csv(
    COLUMN_SCHEMA_PATH,
    index=False,
)


chronological_builds.to_csv(
    CHRONOLOGY_PATH,
    index=False,
)


equal_timestamp_groups.to_csv(
    EQUAL_TIMESTAMP_PATH,
    index=False,
)


source_alignment_summary.to_csv(
    SOURCE_ALIGNMENT_PATH,
    index=False,
)


predictor_list_table.to_csv(
    PREDICTOR_LIST_PATH,
    index=False,
)


failure_build_difference_audit.to_csv(
    FAILURE_DIFFERENCE_AUDIT_PATH,
    index=False,
)


# ---------------------------------------------------------
# 9. Save the corrected complete preflight report
# ---------------------------------------------------------

last_training_build_row = (
    chronological_builds.iloc[
        TRAINING_BUILD_COUNT - 1
    ]
)


first_evaluation_build_row = (
    chronological_builds.iloc[
        TRAINING_BUILD_COUNT
    ]
)


preflight_completed_at = (
    pd.Timestamp.utcnow().isoformat()
)


preflight_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "PASS",

    "SchemaCorrections": {
        "BuildsIdentifierOriginal":
            builds_build_column,

        "BuildsIdentifierCanonical":
            "Build",
    },

    "ScreeningCountPolicy": {
        "Basis":
            "RAW_EXECUTION_HISTORY",

        "Reason":
            (
                "The screening table counts CI builds "
                "with failures in the complete exe.csv "
                "history. dataset.csv is a retained "
                "model-ready subset and may omit a raw "
                "failure execution."
            ),

        "ExpectedFailingTrainingBuilds":
            int(
                EXPECTED_FAILING_TRAINING_BUILDS
            ),

        "ObservedRawFailingTrainingBuilds":
            RAW_FAILING_TRAINING_BUILD_COUNT,

        "ExpectedFailingEvaluationBuilds":
            int(
                EXPECTED_FAILING_EVALUATION_BUILDS
            ),

        "ObservedRawFailingEvaluationBuilds":
            RAW_FAILING_EVALUATION_BUILD_COUNT,

        "ExactScreeningMatch":
            True,

        "MinimumTrainingRequirementPassed":
            bool(
                RAW_FAILING_TRAINING_BUILD_COUNT
                >= 10
            ),
    },

    "Chronology": {
        "Rule":
            (
                "started_at ascending; Build descending "
                "within equal timestamps"
            ),

        "TotalBuilds":
            int(
                TOTAL_BUILD_COUNT
            ),

        "TrainingBuilds":
            int(
                TRAINING_BUILD_COUNT
            ),

        "EvaluationBuilds":
            int(
                EVALUATION_BUILD_COUNT
            ),

        "EqualTimestampGroups":
            int(
                len(
                    equal_timestamp_groups
                )
            ),

        "EqualTimestampOrderViolations":
            int(
                equal_timestamp_order_violations
            ),

        "LastTrainingBuild":
            int(
                last_training_build_row[
                    "Build"
                ]
            ),

        "FirstEvaluationBuild":
            int(
                first_evaluation_build_row[
                    "Build"
                ]
            ),
    },

    "RawExecutionHistory": {
        "Rows":
            int(
                len(
                    exe
                )
            ),

        "TrainingRows":
            int(
                len(
                    raw_training_history
                )
            ),

        "EvaluationRows":
            int(
                len(
                    raw_evaluation_history
                )
            ),

        "FailingTrainingBuilds":
            RAW_FAILING_TRAINING_BUILD_COUNT,

        "FailingEvaluationBuilds":
            RAW_FAILING_EVALUATION_BUILD_COUNT,

        "TrainingFailureExecutions":
            RAW_TRAINING_FAILURE_EXECUTIONS,

        "EvaluationFailureExecutions":
            RAW_EVALUATION_FAILURE_EXECUTIONS,
    },

    "ModelReadyDataset": {
        "Rows":
            int(
                len(
                    dataset
                )
            ),

        "Predictors":
            int(
                len(
                    PREDICTOR_COLUMNS
                )
            ),

        "PredictorCoercionFailures":
            int(
                predictor_coercion_failures
            ),

        "PredictorMissingValues":
            int(
                predictor_missing_values
            ),

        "PredictorInfiniteValues":
            int(
                predictor_infinite_values
            ),

        "TrainingRows":
            int(
                len(
                    clean_training_data
                )
            ),

        "EvaluationPartitionRows":
            int(
                len(
                    all_model_evaluation_data
                )
            ),

        "EvaluatedRows":
            int(
                len(
                    clean_evaluation_data
                )
            ),

        "FailingTrainingBuilds":
            MODEL_FAILING_TRAINING_BUILD_COUNT,

        "FailingEvaluationBuilds":
            MODEL_FAILING_EVALUATION_BUILD_COUNT,

        "TrainingFailureExecutions":
            TRAINING_FAILURE_EXECUTIONS,

        "EvaluationFailureExecutions":
            EVALUATION_FAILURE_EXECUTIONS,
    },

    "RawModelFailureBuildDifference": {
        "RawOnlyFailingTrainingBuilds":
            RAW_ONLY_FAILING_TRAINING_BUILDS,

        "ModelOnlyFailingTrainingBuilds":
            MODEL_ONLY_FAILING_TRAINING_BUILDS,

        "RawOnlyFailingEvaluationBuilds":
            RAW_ONLY_FAILING_EVALUATION_BUILDS,

        "ModelOnlyFailingEvaluationBuilds":
            MODEL_ONLY_FAILING_EVALUATION_BUILDS,

        "AuditFile":
            str(
                FAILURE_DIFFERENCE_AUDIT_PATH
            ),
    },

    "ModelRawRowAlignment": {
        "ModelRows":
            int(
                len(
                    dataset
                )
            ),

        "MatchedRows":
            int(
                len(
                    matched_alignment
                )
            ),

        "RowsMissingFromRaw":
            int(
                model_rows_missing_from_raw
            ),

        "VerdictMismatches":
            int(
                model_raw_verdict_mismatches
            ),

        "DurationMismatches":
            int(
                model_raw_duration_mismatches
            ),
    },

    "FrozenCleanHashes": {
        "CleanTrainingHistorySHA256":
            CLEAN_TRAINING_HISTORY_SHA256,

        "CleanEvaluationHistorySHA256":
            CLEAN_EVALUATION_HISTORY_SHA256,

        "CleanTrainingDataSHA256":
            CLEAN_TRAINING_DATA_SHA256,

        "CleanEvaluationDataSHA256":
            CLEAN_EVALUATION_DATA_SHA256,
    },

    "Reports": {
        "TableSummary":
            str(
                TABLE_SUMMARY_PATH
            ),

        "ColumnSchema":
            str(
                COLUMN_SCHEMA_PATH
            ),

        "Chronology":
            str(
                CHRONOLOGY_PATH
            ),

        "EqualTimestampGroups":
            str(
                EQUAL_TIMESTAMP_PATH
            ),

        "SourceAlignment":
            str(
                SOURCE_ALIGNMENT_PATH
            ),

        "PredictorList":
            str(
                PREDICTOR_LIST_PATH
            ),

        "FailureBuildDifferenceAudit":
            str(
                FAILURE_DIFFERENCE_AUDIT_PATH
            ),
    },

    "CompletedAtUTC":
        preflight_completed_at,
}


with open(
    PREFLIGHT_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        preflight_report,
        report_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 10. Update the permanent Project 7 checkpoint
# ---------------------------------------------------------

with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    project_7_checkpoint = json.load(
        checkpoint_file
    )


project_7_checkpoint.update({
    "Status":
        "PREFLIGHT_PASSED",

    "ScreeningCountBasis":
        "RAW_EXECUTION_HISTORY",

    "ChronologyRule":
        (
            "started_at ascending; Build descending "
            "within equal timestamps"
        ),

    "TotalBuilds":
        int(
            TOTAL_BUILD_COUNT
        ),

    "TrainingBuilds":
        int(
            TRAINING_BUILD_COUNT
        ),

    "EvaluationBuilds":
        int(
            EVALUATION_BUILD_COUNT
        ),

    "FailingTrainingBuilds":
        RAW_FAILING_TRAINING_BUILD_COUNT,

    "FailingEvaluationBuilds":
        RAW_FAILING_EVALUATION_BUILD_COUNT,

    "RawFailingTrainingBuilds":
        RAW_FAILING_TRAINING_BUILD_COUNT,

    "RawFailingEvaluationBuilds":
        RAW_FAILING_EVALUATION_BUILD_COUNT,

    "ModelReadyFailingTrainingBuilds":
        MODEL_FAILING_TRAINING_BUILD_COUNT,

    "ModelReadyFailingEvaluationBuilds":
        MODEL_FAILING_EVALUATION_BUILD_COUNT,

    "RawOnlyFailingTrainingBuilds":
        RAW_ONLY_FAILING_TRAINING_BUILDS,

    "RawOnlyFailingEvaluationBuilds":
        RAW_ONLY_FAILING_EVALUATION_BUILDS,

    "RawTrainingRows":
        int(
            len(
                raw_training_history
            )
        ),

    "RawEvaluationRows":
        int(
            len(
                raw_evaluation_history
            )
        ),

    "ModelTrainingRows":
        int(
            len(
                clean_training_data
            )
        ),

    "ModelEvaluationPartitionRows":
        int(
            len(
                all_model_evaluation_data
            )
        ),

    "ModelEvaluationRows":
        int(
            len(
                clean_evaluation_data
            )
        ),

    "TrainingFailureExecutions":
        TRAINING_FAILURE_EXECUTIONS,

    "EvaluationFailureExecutions":
        EVALUATION_FAILURE_EXECUTIONS,

    "PredictorCount":
        int(
            len(
                PREDICTOR_COLUMNS
            )
        ),

    "LastTrainingBuild":
        int(
            last_training_build_row[
                "Build"
            ]
        ),

    "FirstEvaluationBuild":
        int(
            first_evaluation_build_row[
                "Build"
            ]
        ),

    "CleanTrainingHistorySHA256":
        CLEAN_TRAINING_HISTORY_SHA256,

    "CleanEvaluationHistorySHA256":
        CLEAN_EVALUATION_HISTORY_SHA256,

    "CleanTrainingDataSHA256":
        CLEAN_TRAINING_DATA_SHA256,

    "CleanEvaluationDataSHA256":
        CLEAN_EVALUATION_DATA_SHA256,

    "ChronologicalPreflightReport":
        str(
            PREFLIGHT_REPORT_PATH
        ),

    "FailureBuildDifferenceAudit":
        str(
            FAILURE_DIFFERENCE_AUDIT_PATH
        ),

    "TableSummary":
        str(
            TABLE_SUMMARY_PATH
        ),

    "ColumnSchema":
        str(
            COLUMN_SCHEMA_PATH
        ),

    "Chronology":
        str(
            CHRONOLOGY_PATH
        ),

    "SourceAlignment":
        str(
            SOURCE_ALIGNMENT_PATH
        ),

    "PredictorList":
        str(
            PREDICTOR_LIST_PATH
        ),

    "UpdatedAtUTC":
        preflight_completed_at,
})


temporary_checkpoint_path = (
    PROJECT_7_SELECTION_CHECKPOINT
    .with_suffix(
        ".json.tmp"
    )
)


with open(
    temporary_checkpoint_path,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        project_7_checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


os.replace(
    temporary_checkpoint_path,
    PROJECT_7_SELECTION_CHECKPOINT,
)


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint_verification = json.load(
        checkpoint_file
    )


if checkpoint_verification.get(
    "Status"
) != "PREFLIGHT_PASSED":
    raise AssertionError(
        "The corrected Project 7 preflight checkpoint "
        "was not written correctly."
    )


if checkpoint_verification.get(
    "ScreeningCountBasis"
) != "RAW_EXECUTION_HISTORY":
    raise AssertionError(
        "The corrected screening basis was not saved."
    )


# ---------------------------------------------------------
# 11. Compact final output
# ---------------------------------------------------------

clear_output(
    wait=True
)


print(
    "=== PROJECT 7 STEP 2 RESULT ==="
)


print(
    "\nScreening count basis:"
)

print(
    "Screening basis:",
    "RAW_EXECUTION_HISTORY"
)

print(
    "Reason: dataset.csv is a retained model-ready "
    "subset of exe.csv."
)


print(
    "\nChronological split:"
)

print(
    "Total builds:",
    TOTAL_BUILD_COUNT
)

print(
    "Training builds:",
    TRAINING_BUILD_COUNT
)

print(
    "Evaluation builds:",
    EVALUATION_BUILD_COUNT
)

print(
    "Last training build:",
    int(
        last_training_build_row[
            "Build"
        ]
    )
)

print(
    "First evaluation build:",
    int(
        first_evaluation_build_row[
            "Build"
        ]
    )
)


print(
    "\nRaw screening validation:"
)

print(
    "Expected failing training builds:",
    EXPECTED_FAILING_TRAINING_BUILDS
)

print(
    "Observed raw failing training builds:",
    RAW_FAILING_TRAINING_BUILD_COUNT
)

print(
    "Expected failing evaluation builds:",
    EXPECTED_FAILING_EVALUATION_BUILDS
)

print(
    "Observed raw failing evaluation builds:",
    RAW_FAILING_EVALUATION_BUILD_COUNT
)

print(
    "Exact screening match:",
    True
)


print(
    "\nModel-ready retained cohort:"
)

print(
    "Predictors:",
    len(
        PREDICTOR_COLUMNS
    )
)

print(
    "Training rows:",
    len(
        clean_training_data
    )
)

print(
    "Evaluation partition rows:",
    len(
        all_model_evaluation_data
    )
)

print(
    "Evaluated rows:",
    len(
        clean_evaluation_data
    )
)

print(
    "Model-ready failing training builds:",
    MODEL_FAILING_TRAINING_BUILD_COUNT
)

print(
    "Model-ready failing evaluation builds:",
    MODEL_FAILING_EVALUATION_BUILD_COUNT
)

print(
    "Training failure executions:",
    TRAINING_FAILURE_EXECUTIONS
)

print(
    "Evaluation failure executions:",
    EVALUATION_FAILURE_EXECUTIONS
)


print(
    "\nRaw/model failing-build difference:"
)

print(
    "Raw-only failing training builds:",
    RAW_ONLY_FAILING_TRAINING_BUILDS
)

print(
    "Raw-only failing evaluation builds:",
    RAW_ONLY_FAILING_EVALUATION_BUILDS
)


if len(
    failure_build_difference_audit
) > 0:
    display(
        failure_build_difference_audit
    )

else:
    print(
        "No raw/model failing-build differences."
    )


print(
    "\nModel/raw row alignment:"
)

display(
    source_alignment_summary
)


print(
    "\nPredictor quality:"
)

print(
    "Predictor coercion failures:",
    predictor_coercion_failures
)

print(
    "Predictor missing values:",
    predictor_missing_values
)

print(
    "Predictor infinite values:",
    predictor_infinite_values
)


print(
    "\nEqual-timestamp chronology:"
)

print(
    "Equal-timestamp groups:",
    len(
        equal_timestamp_groups
    )
)

print(
    "Ordering violations:",
    equal_timestamp_order_violations
)


print(
    "\nPreflight report:"
)

print(
    PREFLIGHT_REPORT_PATH
)


print(
    "\nValidation status:",
    "PASS"
)


print(
    "\nSUCCESS: Screening was validated against the "
    "complete raw execution history."
)

print(
    "SUCCESS: The expected 64 training and 52 evaluation "
    "failing builds matched exactly."
)

print(
    "SUCCESS: The 63 model-ready failing training builds "
    "were retained without changing the fixed-instance "
    "dataset."
)

print(
    "SUCCESS: No raw execution was deleted and no "
    "synthetic model-ready row was created."
)

print(
    "SUCCESS: All model-ready rows still matched their "
    "raw execution-history rows."
)

print(
    "SUCCESS: Clean training and evaluation objects "
    "were created and cryptographically hashed."
)

print(
    "SUCCESS: Project 7 is ready for commit/entity "
    "schema inspection and canonical mapping."
)

=== PROJECT 7 STEP 2 RESULT ===

Screening count basis:
Screening basis: RAW_EXECUTION_HISTORY
Reason: dataset.csv is a retained model-ready subset of exe.csv.

Chronological split:
Total builds: 415
Training builds: 311
Evaluation builds: 104
Last training build: 601182940
First evaluation build: 601401990

Raw screening validation:
Expected failing training builds: 64
Observed raw failing training builds: 64
Expected failing evaluation builds: 52
Observed raw failing evaluation builds: 52
Exact screening match: True

Model-ready retained cohort:
Predictors: 150
Training rows: 3960
Evaluation partition rows: 3678
Evaluated rows: 3678
Model-ready failing training builds: 63
Model-ready failing evaluation builds: 52
Training failure executions: 170
Evaluation failure executions: 93

Raw/model failing-build difference:
Raw-only failing training builds: [238091769]
Raw-only failing evaluation builds: []


,Partition,Build,BuildOrder,DifferenceType,RawRows,RawFailureExecutions,ModelReadyRows,ModelReadyFailureExecutions
0,TRAIN,238091769,4,RAW_FAILURE_NOT_RETAINED_AS_MODEL_FAILURE,62,3,0,0



Model/raw row alignment:


,ModelRows,MatchedRawRows,ModelRowsMissingFromRaw,VerdictMismatches,DurationMismatches,RawDuplicateKeyRows,ModelDuplicateKeyRows
0,7638,7638,0,0,0,0,0



Predictor quality:
Predictor coercion failures: 0
Predictor missing values: 0
Predictor infinite values: 0

Equal-timestamp chronology:
Equal-timestamp groups: 0
Ordering violations: 0

Preflight report:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_preflight/beast2_chronological_preflight_report.json

Validation status: PASS

SUCCESS: Screening was validated against the complete raw execution history.
SUCCESS: The expected 64 training and 52 evaluation failing builds matched exactly.
SUCCESS: The 63 model-ready failing training builds were retained without changing the fixed-instance dataset.
SUCCESS: No raw execution was deleted and no synthetic model-ready row was created.
SUCCESS: All model-ready rows still matched their raw execution-history rows.
SUCCESS: Clean training and evaluation objects were created and cryptographically hashed.
SUCCESS: Project 7 is ready for commit/entity schema inspection and canonical mapping.


In [ ]:
# =========================================================
# PROJECT 7 — STEP 3
# COMMIT AND ENTITY SCHEMA INSPECTION
#
# This cell:
#   - inspects builds.commits
#   - parses canonical build-commit tokens
#   - identifies candidate commit columns in
#     entity_change_history.csv
#   - identifies candidate EntityId column pairs between
#     entity_change_history.csv and id_map.csv
#   - measures exact and prefix commit-token coverage
#   - saves permanent schema-diagnostic reports
#
# It does NOT:
#   - create the canonical entity mapping
#   - exclude any build
#   - create synthetic entities
#   - reconstruct REC features
#   - inject noise or fit models
# =========================================================

from pathlib import Path
import ast
import json
import os
import re

import numpy as np
import pandas as pd
from IPython.display import clear_output, display


print(
    "=== PROJECT 7 STEP 3: "
    "COMMIT AND ENTITY SCHEMA INSPECTION ==="
)


# ---------------------------------------------------------
# 1. Confirm successful Step 2 state
# ---------------------------------------------------------

required_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",
    "PROJECT_SHORT_NAME",

    "PROJECT_PREFLIGHT_DIRECTORY",
    "PROJECT_7_SELECTION_CHECKPOINT",

    "BUILDS_PATH",
    "ENTITY_HISTORY_PATH",
    "ID_MAP_PATH",
    "CONTRIBUTORS_PATH",

    "chronological_builds",
    "raw_training_history",
    "raw_evaluation_history",
]


missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required Project 7 Step 2 objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nRerun the successful Step 2B continuation "
        "cell. Do not rerun Projects 1–6."
    )


if PROJECT_NUMBER != 7:
    raise AssertionError(
        f"Expected Project 7, observed {PROJECT_NUMBER}."
    )


if PROJECT_NAME != "CompEvol@beast2":
    raise AssertionError(
        f"Unexpected project: {PROJECT_NAME}"
    )


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    step_2_checkpoint = json.load(
        checkpoint_file
    )


allowed_previous_statuses = {
    "PREFLIGHT_PASSED",
    "ENTITY_SCHEMA_INSPECTION_PASSED",
}


if step_2_checkpoint.get(
    "Status"
) not in allowed_previous_statuses:
    raise AssertionError(
        "Project 7 preflight has not passed.\n"
        f"Observed status: "
        f"{step_2_checkpoint.get('Status')}"
    )


# ---------------------------------------------------------
# 2. Load clean source copies for schema inspection
# ---------------------------------------------------------

builds_schema_raw = pd.read_csv(
    BUILDS_PATH,
    low_memory=False,
)


entity_history_schema_raw = pd.read_csv(
    ENTITY_HISTORY_PATH,
    low_memory=False,
)


id_map_schema_raw = pd.read_csv(
    ID_MAP_PATH,
    low_memory=False,
)


contributors_schema_raw = pd.read_csv(
    CONTRIBUTORS_PATH,
    low_memory=False,
)


if len(builds_schema_raw) == 0:
    raise AssertionError(
        "builds.csv is empty."
    )


if len(entity_history_schema_raw) == 0:
    raise AssertionError(
        "entity_change_history.csv is empty."
    )


if len(id_map_schema_raw) == 0:
    raise AssertionError(
        "id_map.csv is empty."
    )


# ---------------------------------------------------------
# 3. Generic schema helpers
# ---------------------------------------------------------

def normalise_column_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).strip().lower(),
    )


def resolve_column(
    dataframe,
    accepted_names,
    logical_name,
):
    lookup = {
        normalise_column_name(column):
            column
        for column in dataframe.columns
    }


    for accepted_name in accepted_names:
        normalised_name = normalise_column_name(
            accepted_name
        )


        if normalised_name in lookup:
            return lookup[
                normalised_name
            ]


    raise RuntimeError(
        f"Could not resolve {logical_name}.\n"
        "Accepted names: "
        + " | ".join(
            accepted_names
        )
        + "\nObserved columns: "
        + " | ".join(
            map(
                str,
                dataframe.columns,
            )
        )
    )


def normalise_scalar_value(
    value,
):
    if pd.isna(value):
        return None


    text = str(
        value
    ).strip()


    if not text:
        return None


    if text.lower() in {
        "nan",
        "none",
        "null",
    }:
        return None


    return text


def normalise_identifier_value(
    value,
):
    text = normalise_scalar_value(
        value
    )


    if text is None:
        return None


    # Canonicalise integer-like values such as 14.0 -> 14.
    if re.fullmatch(
        r"[+-]?\d+\.0+",
        text,
    ):
        return str(
            int(
                float(
                    text
                )
            )
        )


    return text


def normalise_commit_value(
    value,
):
    text = normalise_scalar_value(
        value
    )


    if text is None:
        return None


    return (
        text
        .strip()
        .strip(
            "\"'"
        )
        .lower()
    )


# ---------------------------------------------------------
# 4. Resolve builds.csv identifier and commit columns
# ---------------------------------------------------------

builds_id_column = resolve_column(
    builds_schema_raw,
    [
        "id",
        "Build",
        "build",
        "BuildId",
        "build_id",
    ],
    "builds.csv build identifier",
)


builds_commit_column = resolve_column(
    builds_schema_raw,
    [
        "commits",
        "commit",
        "commit_ids",
        "commitHashes",
        "hashes",
        "revisions",
    ],
    "builds.csv commit field",
)


builds_schema = (
    builds_schema_raw
    .rename(
        columns={
            builds_id_column:
                "Build",
        }
    )
    .copy()
)


builds_schema[
    "Build"
] = pd.to_numeric(
    builds_schema[
        "Build"
    ],
    errors="raise",
).astype(np.int64)


if builds_schema[
    "Build"
].duplicated().any():
    raise AssertionError(
        "builds.csv contains duplicate build IDs."
    )


# ---------------------------------------------------------
# 5. Parse build-commit tokens robustly
# ---------------------------------------------------------

SHA_TOKEN_PATTERN = re.compile(
    r"(?i)(?<![0-9a-f])[0-9a-f]{7,64}(?![0-9a-f])"
)


def extract_commit_tokens(
    value,
):
    if pd.isna(value):
        return []


    text = str(
        value
    ).strip()


    if not text:
        return []


    if text.lower() in {
        "nan",
        "none",
        "null",
        "[]",
    }:
        return []


    tokens = []


    # First attempt structured Python/JSON-like lists.
    if (
        text.startswith("[")
        and text.endswith("]")
    ):
        try:
            parsed_value = ast.literal_eval(
                text
            )


            if isinstance(
                parsed_value,
                (
                    list,
                    tuple,
                    set,
                ),
            ):
                for item in parsed_value:
                    item_text = (
                        str(
                            item
                        )
                        .strip()
                        .strip(
                            "\"'"
                        )
                        .lower()
                    )


                    if item_text:
                        tokens.append(
                            item_text
                        )

        except Exception:
            pass


    # Extract standard Git SHA-like tokens.
    sha_tokens = [
        match.group(0).lower()
        for match in (
            SHA_TOKEN_PATTERN.finditer(
                text
            )
        )
    ]


    tokens.extend(
        sha_tokens
    )


    # Fallback for unusual delimiters or non-hex tokens.
    if len(tokens) == 0:
        fallback_parts = re.split(
            r"[\s,;|]+",
            text,
        )


        for part in fallback_parts:
            cleaned_part = (
                part
                .strip()
                .strip(
                    "[](){}\"'"
                )
                .lower()
            )


            if cleaned_part:
                tokens.append(
                    cleaned_part
                )


    # Preserve order while removing duplicates.
    unique_tokens = []


    seen_tokens = set()


    for token in tokens:
        token = token.strip()


        if not token:
            continue


        if token in seen_tokens:
            continue


        seen_tokens.add(
            token
        )


        unique_tokens.append(
            token
        )


    return unique_tokens


build_commit_profile = builds_schema[
    [
        "Build",
        builds_commit_column,
    ]
].copy()


build_commit_profile = (
    build_commit_profile
    .merge(
        chronological_builds[
            [
                "Build",
                "build_order",
                "partition",
                "started_at",
            ]
        ],
        on="Build",
        how="left",
        validate="one_to_one",
    )
)


if build_commit_profile[
    "build_order"
].isna().any():
    raise AssertionError(
        "At least one build could not be attached to the "
        "frozen chronology."
    )


build_commit_profile[
    "CommitTokensList"
] = (
    build_commit_profile[
        builds_commit_column
    ]
    .map(
        extract_commit_tokens
    )
)


build_commit_profile[
    "CommitTokenCount"
] = (
    build_commit_profile[
        "CommitTokensList"
    ]
    .map(
        len
    )
)


build_commit_profile[
    "CommitTokens"
] = (
    build_commit_profile[
        "CommitTokensList"
    ]
    .map(
        lambda tokens:
            " | ".join(
                tokens
            )
    )
)


all_build_commit_tokens = sorted({
    commit_token
    for token_list in (
        build_commit_profile[
            "CommitTokensList"
        ]
    )
    for commit_token in token_list
})


UNIQUE_BUILD_COMMIT_TOKEN_COUNT = int(
    len(
        all_build_commit_tokens
    )
)


BUILD_ROWS_WITH_NO_COMMIT_TOKEN = int(
    (
        build_commit_profile[
            "CommitTokenCount"
        ] == 0
    ).sum()
)


MULTI_COMMIT_BUILD_COUNT = int(
    (
        build_commit_profile[
            "CommitTokenCount"
        ] > 1
    ).sum()
)


# ---------------------------------------------------------
# 6. Score entity-history commit-column candidates
# ---------------------------------------------------------

def commit_column_name_score(
    column_name,
):
    normalised_name = normalise_column_name(
        column_name
    )


    score = 0


    if normalised_name in {
        "commit",
        "commits",
        "commithash",
        "commitsha",
        "sha",
        "hash",
        "revision",
    }:
        score += 1000


    if "commit" in normalised_name:
        score += 500


    if "hash" in normalised_name:
        score += 300


    if "sha" in normalised_name:
        score += 250


    if "revision" in normalised_name:
        score += 200


    return score


def is_sha_like(
    value,
):
    if value is None:
        return False


    return bool(
        re.fullmatch(
            r"(?i)[0-9a-f]{7,64}",
            value,
        )
    )


def token_has_prefix_match(
    token,
    candidate_values,
):
    if token in candidate_values:
        return True


    for candidate_value in candidate_values:
        if (
            candidate_value.startswith(
                token
            )
            or token.startswith(
                candidate_value
            )
        ):
            return True


    return False


commit_candidate_records = []
commit_candidate_value_sets = {}


for column in entity_history_schema_raw.columns:
    normalised_values = (
        entity_history_schema_raw[
            column
        ]
        .map(
            normalise_commit_value
        )
        .dropna()
    )


    unique_values = set(
        normalised_values.unique()
    )


    commit_candidate_value_sets[
        column
    ] = unique_values


    non_missing_count = int(
        len(
            normalised_values
        )
    )


    unique_count = int(
        len(
            unique_values
        )
    )


    sha_like_count = int(
        sum(
            is_sha_like(
                value
            )
            for value in unique_values
        )
    )


    sha_like_fraction = (
        float(
            sha_like_count
            / unique_count
        )
        if unique_count > 0
        else 0.0
    )


    exact_matched_tokens = {
        token
        for token in all_build_commit_tokens
        if token in unique_values
    }


    prefix_matched_tokens = {
        token
        for token in all_build_commit_tokens
        if token_has_prefix_match(
            token,
            unique_values,
        )
    }


    exact_token_coverage = (
        float(
            len(
                exact_matched_tokens
            )
            / UNIQUE_BUILD_COMMIT_TOKEN_COUNT
        )
        if UNIQUE_BUILD_COMMIT_TOKEN_COUNT > 0
        else 0.0
    )


    prefix_token_coverage = (
        float(
            len(
                prefix_matched_tokens
            )
            / UNIQUE_BUILD_COMMIT_TOKEN_COUNT
        )
        if UNIQUE_BUILD_COMMIT_TOKEN_COUNT > 0
        else 0.0
    )


    builds_with_exact_match = int(
        build_commit_profile[
            "CommitTokensList"
        ]
        .map(
            lambda tokens:
                any(
                    token in unique_values
                    for token in tokens
                )
        )
        .sum()
    )


    builds_with_prefix_match = int(
        build_commit_profile[
            "CommitTokensList"
        ]
        .map(
            lambda tokens:
                any(
                    token_has_prefix_match(
                        token,
                        unique_values,
                    )
                    for token in tokens
                )
        )
        .sum()
    )


    name_score = commit_column_name_score(
        column
    )


    total_score = float(
        name_score
        + 10000.0
        * exact_token_coverage
        + 2000.0
        * prefix_token_coverage
        + 100.0
        * sha_like_fraction
    )


    commit_candidate_records.append({
        "Column":
            str(
                column
            ),

        "Dtype":
            str(
                entity_history_schema_raw[
                    column
                ].dtype
            ),

        "Rows":
            int(
                len(
                    entity_history_schema_raw
                )
            ),

        "NonMissingValues":
            non_missing_count,

        "UniqueValues":
            unique_count,

        "SHA_LikeUniqueValues":
            sha_like_count,

        "SHA_LikeFraction":
            sha_like_fraction,

        "ExactMatchedBuildCommitTokens":
            int(
                len(
                    exact_matched_tokens
                )
            ),

        "PrefixMatchedBuildCommitTokens":
            int(
                len(
                    prefix_matched_tokens
                )
            ),

        "UniqueBuildCommitTokens":
            UNIQUE_BUILD_COMMIT_TOKEN_COUNT,

        "ExactTokenCoverage":
            exact_token_coverage,

        "PrefixTokenCoverage":
            prefix_token_coverage,

        "BuildsWithExactCommitMatch":
            builds_with_exact_match,

        "BuildsWithPrefixCommitMatch":
            builds_with_prefix_match,

        "ColumnNameScore":
            name_score,

        "TotalCandidateScore":
            total_score,
    })


commit_column_candidates = (
    pd.DataFrame(
        commit_candidate_records
    )
    .sort_values(
        [
            "TotalCandidateScore",
            "ExactMatchedBuildCommitTokens",
            "PrefixMatchedBuildCommitTokens",
            "Column",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if len(commit_column_candidates) == 0:
    raise AssertionError(
        "No entity-history commit-column candidates "
        "were produced."
    )


SELECTED_ENTITY_HISTORY_COMMIT_COLUMN = str(
    commit_column_candidates.iloc[0][
        "Column"
    ]
)


selected_commit_values = (
    commit_candidate_value_sets[
        SELECTED_ENTITY_HISTORY_COMMIT_COLUMN
    ]
)


SELECTED_EXACT_COMMIT_TOKEN_MATCHES = int(
    commit_column_candidates.iloc[0][
        "ExactMatchedBuildCommitTokens"
    ]
)


SELECTED_PREFIX_COMMIT_TOKEN_MATCHES = int(
    commit_column_candidates.iloc[0][
        "PrefixMatchedBuildCommitTokens"
    ]
)


if (
    SELECTED_EXACT_COMMIT_TOKEN_MATCHES == 0
    and SELECTED_PREFIX_COMMIT_TOKEN_MATCHES == 0
):
    display(
        commit_column_candidates
    )


    raise AssertionError(
        "No entity-history column matched any build "
        "commit token."
    )


# ---------------------------------------------------------
# 7. Score EntityId column pairs
# ---------------------------------------------------------

def entity_id_name_score(
    column_name,
):
    normalised_name = normalise_column_name(
        column_name
    )


    score = 0


    if normalised_name in {
        "entityid",
        "entityids",
    }:
        score += 1000


    if normalised_name == "id":
        score += 400


    if "entity" in normalised_name:
        score += 300


    if normalised_name.endswith(
        "id"
    ):
        score += 100


    return score


entity_id_pair_records = []


for history_column in (
    entity_history_schema_raw.columns
):
    if (
        history_column
        == SELECTED_ENTITY_HISTORY_COMMIT_COLUMN
    ):
        continue


    history_values = (
        entity_history_schema_raw[
            history_column
        ]
        .map(
            normalise_identifier_value
        )
        .dropna()
    )


    history_unique_values = set(
        history_values.unique()
    )


    if len(history_unique_values) == 0:
        continue


    for id_map_column in id_map_schema_raw.columns:
        id_map_values = (
            id_map_schema_raw[
                id_map_column
            ]
            .map(
                normalise_identifier_value
            )
            .dropna()
        )


        id_map_unique_values = set(
            id_map_values.unique()
        )


        if len(id_map_unique_values) == 0:
            continue


        intersection_values = (
            history_unique_values
            .intersection(
                id_map_unique_values
            )
        )


        intersection_count = int(
            len(
                intersection_values
            )
        )


        history_coverage = float(
            intersection_count
            / len(
                history_unique_values
            )
        )


        id_map_coverage = float(
            intersection_count
            / len(
                id_map_unique_values
            )
        )


        history_name_score = (
            entity_id_name_score(
                history_column
            )
        )


        id_map_name_score = (
            entity_id_name_score(
                id_map_column
            )
        )


        pair_name_score = (
            history_name_score
            + id_map_name_score
        )


        total_pair_score = float(
            pair_name_score
            + 10000.0
            * min(
                history_coverage,
                id_map_coverage,
            )
            + 1000.0
            * max(
                history_coverage,
                id_map_coverage,
            )
            + min(
                intersection_count,
                1000,
            )
        )


        entity_id_pair_records.append({
            "EntityHistoryColumn":
                str(
                    history_column
                ),

            "IdMapColumn":
                str(
                    id_map_column
                ),

            "EntityHistoryUniqueValues":
                int(
                    len(
                        history_unique_values
                    )
                ),

            "IdMapUniqueValues":
                int(
                    len(
                        id_map_unique_values
                    )
                ),

            "IntersectionValues":
                intersection_count,

            "EntityHistoryCoverage":
                history_coverage,

            "IdMapCoverage":
                id_map_coverage,

            "HistoryColumnNameScore":
                history_name_score,

            "IdMapColumnNameScore":
                id_map_name_score,

            "PairNameScore":
                pair_name_score,

            "TotalCandidateScore":
                total_pair_score,
        })


entity_id_pair_candidates = (
    pd.DataFrame(
        entity_id_pair_records
    )
    .sort_values(
        [
            "TotalCandidateScore",
            "IntersectionValues",
            "EntityHistoryColumn",
            "IdMapColumn",
        ],
        ascending=[
            False,
            False,
            True,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if len(entity_id_pair_candidates) == 0:
    raise AssertionError(
        "No entity-history/id-map column pairs could be "
        "evaluated."
    )


SELECTED_ENTITY_HISTORY_ID_COLUMN = str(
    entity_id_pair_candidates.iloc[0][
        "EntityHistoryColumn"
    ]
)


SELECTED_ID_MAP_ID_COLUMN = str(
    entity_id_pair_candidates.iloc[0][
        "IdMapColumn"
    ]
)


SELECTED_ENTITY_ID_INTERSECTION = int(
    entity_id_pair_candidates.iloc[0][
        "IntersectionValues"
    ]
)


SELECTED_ENTITY_HISTORY_ID_COVERAGE = float(
    entity_id_pair_candidates.iloc[0][
        "EntityHistoryCoverage"
    ]
)


SELECTED_ID_MAP_ID_COVERAGE = float(
    entity_id_pair_candidates.iloc[0][
        "IdMapCoverage"
    ]
)


if SELECTED_ENTITY_ID_INTERSECTION == 0:
    display(
        entity_id_pair_candidates.head(20)
    )


    raise AssertionError(
        "No shared EntityId values were found between "
        "entity history and id_map."
    )


# ---------------------------------------------------------
# 8. Profile remaining id_map and contributor columns
# ---------------------------------------------------------

id_map_column_profile_records = []


for column in id_map_schema_raw.columns:
    id_map_column_profile_records.append({
        "Column":
            str(
                column
            ),

        "Dtype":
            str(
                id_map_schema_raw[
                    column
                ].dtype
            ),

        "Rows":
            int(
                len(
                    id_map_schema_raw
                )
            ),

        "MissingValues":
            int(
                id_map_schema_raw[
                    column
                ].isna().sum()
            ),

        "UniqueNonMissingValues":
            int(
                id_map_schema_raw[
                    column
                ].nunique(
                    dropna=True
                )
            ),

        "SelectedAsEntityId":
            bool(
                column
                == SELECTED_ID_MAP_ID_COLUMN
            ),
    })


id_map_column_profile = pd.DataFrame(
    id_map_column_profile_records
)


contributors_column_profile_records = []


for column in contributors_schema_raw.columns:
    contributors_column_profile_records.append({
        "Column":
            str(
                column
            ),

        "Dtype":
            str(
                contributors_schema_raw[
                    column
                ].dtype
            ),

        "Rows":
            int(
                len(
                    contributors_schema_raw
                )
            ),

        "MissingValues":
            int(
                contributors_schema_raw[
                    column
                ].isna().sum()
            ),

        "UniqueNonMissingValues":
            int(
                contributors_schema_raw[
                    column
                ].nunique(
                    dropna=True
                )
            ),
    })


contributors_column_profile = pd.DataFrame(
    contributors_column_profile_records
)


# ---------------------------------------------------------
# 9. Build compact source-schema table
# ---------------------------------------------------------

source_schema_records = []


for table_name, table in [
    (
        "Builds",
        builds_schema_raw,
    ),
    (
        "EntityHistory",
        entity_history_schema_raw,
    ),
    (
        "EntityIdMap",
        id_map_schema_raw,
    ),
    (
        "Contributors",
        contributors_schema_raw,
    ),
]:
    for column in table.columns:
        non_missing_values = (
            table[
                column
            ]
            .dropna()
        )


        sample_values = [
            str(
                value
            )[:120]
            for value in (
                non_missing_values
                .head(3)
                .tolist()
            )
        ]


        source_schema_records.append({
            "Table":
                table_name,

            "Column":
                str(
                    column
                ),

            "Dtype":
                str(
                    table[
                        column
                    ].dtype
                ),

            "Rows":
                int(
                    len(
                        table
                    )
                ),

            "MissingValues":
                int(
                    table[
                        column
                    ].isna().sum()
                ),

            "UniqueNonMissingValues":
                int(
                    table[
                        column
                    ].nunique(
                        dropna=True
                    )
                ),

            "SampleValues":
                " | ".join(
                    sample_values
                ),
        })


source_schema_profile = pd.DataFrame(
    source_schema_records
)


# ---------------------------------------------------------
# 10. Save permanent Step 3 reports
# ---------------------------------------------------------

BUILD_COMMIT_PROFILE_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_build_commit_token_profile.csv"
)


COMMIT_COLUMN_CANDIDATES_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_commit_column_candidates.csv"
)


ENTITY_ID_PAIR_CANDIDATES_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_entity_id_pair_candidates.csv"
)


SOURCE_SCHEMA_PROFILE_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_commit_entity_schema_profile.csv"
)


ID_MAP_COLUMN_PROFILE_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_id_map_column_profile.csv"
)


CONTRIBUTORS_COLUMN_PROFILE_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_contributors_column_profile.csv"
)


ENTITY_SCHEMA_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_commit_entity_schema_report.json"
)


build_commit_profile[
    [
        "Build",
        "build_order",
        "partition",
        "started_at",
        builds_commit_column,
        "CommitTokenCount",
        "CommitTokens",
    ]
].to_csv(
    BUILD_COMMIT_PROFILE_PATH,
    index=False,
)


commit_column_candidates.to_csv(
    COMMIT_COLUMN_CANDIDATES_PATH,
    index=False,
)


entity_id_pair_candidates.to_csv(
    ENTITY_ID_PAIR_CANDIDATES_PATH,
    index=False,
)


source_schema_profile.to_csv(
    SOURCE_SCHEMA_PROFILE_PATH,
    index=False,
)


id_map_column_profile.to_csv(
    ID_MAP_COLUMN_PROFILE_PATH,
    index=False,
)


contributors_column_profile.to_csv(
    CONTRIBUTORS_COLUMN_PROFILE_PATH,
    index=False,
)


schema_inspection_completed_at = (
    pd.Timestamp.utcnow().isoformat()
)


entity_schema_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "PASS",

    "BuildCommitField": {
        "BuildIdentifierColumn":
            builds_id_column,

        "CommitFieldColumn":
            builds_commit_column,

        "BuildRows":
            int(
                len(
                    build_commit_profile
                )
            ),

        "BuildRowsWithNoParsedCommitToken":
            BUILD_ROWS_WITH_NO_COMMIT_TOKEN,

        "MultiCommitBuilds":
            MULTI_COMMIT_BUILD_COUNT,

        "UniqueBuildCommitTokens":
            UNIQUE_BUILD_COMMIT_TOKEN_COUNT,
    },

    "SelectedCommitCandidate": {
        "EntityHistoryCommitColumn":
            SELECTED_ENTITY_HISTORY_COMMIT_COLUMN,

        "ExactMatchedBuildCommitTokens":
            SELECTED_EXACT_COMMIT_TOKEN_MATCHES,

        "PrefixMatchedBuildCommitTokens":
            SELECTED_PREFIX_COMMIT_TOKEN_MATCHES,

        "ExactTokenCoverage":
            float(
                commit_column_candidates.iloc[0][
                    "ExactTokenCoverage"
                ]
            ),

        "PrefixTokenCoverage":
            float(
                commit_column_candidates.iloc[0][
                    "PrefixTokenCoverage"
                ]
            ),

        "BuildsWithExactCommitMatch":
            int(
                commit_column_candidates.iloc[0][
                    "BuildsWithExactCommitMatch"
                ]
            ),

        "BuildsWithPrefixCommitMatch":
            int(
                commit_column_candidates.iloc[0][
                    "BuildsWithPrefixCommitMatch"
                ]
            ),
    },

    "SelectedEntityIdPair": {
        "EntityHistoryEntityIdColumn":
            SELECTED_ENTITY_HISTORY_ID_COLUMN,

        "IdMapEntityIdColumn":
            SELECTED_ID_MAP_ID_COLUMN,

        "SharedEntityIds":
            SELECTED_ENTITY_ID_INTERSECTION,

        "EntityHistoryCoverage":
            SELECTED_ENTITY_HISTORY_ID_COVERAGE,

        "IdMapCoverage":
            SELECTED_ID_MAP_ID_COVERAGE,
    },

    "SourceDimensions": {
        "BuildsRows":
            int(
                len(
                    builds_schema_raw
                )
            ),

        "EntityHistoryRows":
            int(
                len(
                    entity_history_schema_raw
                )
            ),

        "IdMapRows":
            int(
                len(
                    id_map_schema_raw
                )
            ),

        "ContributorsRows":
            int(
                len(
                    contributors_schema_raw
                )
            ),
    },

    "Reports": {
        "BuildCommitTokenProfile":
            str(
                BUILD_COMMIT_PROFILE_PATH
            ),

        "CommitColumnCandidates":
            str(
                COMMIT_COLUMN_CANDIDATES_PATH
            ),

        "EntityIdPairCandidates":
            str(
                ENTITY_ID_PAIR_CANDIDATES_PATH
            ),

        "SourceSchemaProfile":
            str(
                SOURCE_SCHEMA_PROFILE_PATH
            ),

        "IdMapColumnProfile":
            str(
                ID_MAP_COLUMN_PROFILE_PATH
            ),

        "ContributorsColumnProfile":
            str(
                CONTRIBUTORS_COLUMN_PROFILE_PATH
            ),
    },

    "CompletedAtUTC":
        schema_inspection_completed_at,
}


with open(
    ENTITY_SCHEMA_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        entity_schema_report,
        report_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 11. Update Project 7 checkpoint atomically
# ---------------------------------------------------------

with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    project_7_checkpoint = json.load(
        checkpoint_file
    )


project_7_checkpoint.update({
    "Status":
        "ENTITY_SCHEMA_INSPECTION_PASSED",

    "BuildCommitColumn":
        builds_commit_column,

    "UniqueBuildCommitTokens":
        UNIQUE_BUILD_COMMIT_TOKEN_COUNT,

    "BuildRowsWithNoCommitToken":
        BUILD_ROWS_WITH_NO_COMMIT_TOKEN,

    "MultiCommitBuilds":
        MULTI_COMMIT_BUILD_COUNT,

    "SelectedEntityHistoryCommitColumn":
        SELECTED_ENTITY_HISTORY_COMMIT_COLUMN,

    "SelectedCommitExactMatches":
        SELECTED_EXACT_COMMIT_TOKEN_MATCHES,

    "SelectedCommitPrefixMatches":
        SELECTED_PREFIX_COMMIT_TOKEN_MATCHES,

    "SelectedEntityHistoryIdColumn":
        SELECTED_ENTITY_HISTORY_ID_COLUMN,

    "SelectedIdMapIdColumn":
        SELECTED_ID_MAP_ID_COLUMN,

    "SelectedEntityIdIntersection":
        SELECTED_ENTITY_ID_INTERSECTION,

    "CommitEntitySchemaReport":
        str(
            ENTITY_SCHEMA_REPORT_PATH
        ),

    "BuildCommitTokenProfile":
        str(
            BUILD_COMMIT_PROFILE_PATH
        ),

    "CommitColumnCandidates":
        str(
            COMMIT_COLUMN_CANDIDATES_PATH
        ),

    "EntityIdPairCandidates":
        str(
            ENTITY_ID_PAIR_CANDIDATES_PATH
        ),

    "UpdatedAtUTC":
        schema_inspection_completed_at,
})


temporary_checkpoint_path = (
    PROJECT_7_SELECTION_CHECKPOINT
    .with_suffix(
        ".json.tmp"
    )
)


with open(
    temporary_checkpoint_path,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        project_7_checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


os.replace(
    temporary_checkpoint_path,
    PROJECT_7_SELECTION_CHECKPOINT,
)


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint_verification = json.load(
        checkpoint_file
    )


if checkpoint_verification.get(
    "Status"
) != "ENTITY_SCHEMA_INSPECTION_PASSED":
    raise AssertionError(
        "The Project 7 schema-inspection checkpoint was "
        "not written correctly."
    )


# ---------------------------------------------------------
# 12. Compact final output
# ---------------------------------------------------------

clear_output(
    wait=True
)


print(
    "=== PROJECT 7 STEP 3 RESULT ==="
)


print(
    "\nSource schemas:"
)

print(
    "builds.csv columns:",
    list(
        builds_schema_raw.columns
    )
)

print(
    "entity_change_history.csv columns:",
    list(
        entity_history_schema_raw.columns
    )
)

print(
    "id_map.csv columns:",
    list(
        id_map_schema_raw.columns
    )
)

print(
    "contributors.csv columns:",
    list(
        contributors_schema_raw.columns
    )
)


print(
    "\nBuild-commit parsing:"
)

print(
    "Build rows:",
    len(
        build_commit_profile
    )
)

print(
    "Unique build-commit tokens:",
    UNIQUE_BUILD_COMMIT_TOKEN_COUNT
)

print(
    "Build rows with no parsed commit token:",
    BUILD_ROWS_WITH_NO_COMMIT_TOKEN
)

print(
    "Multi-commit builds:",
    MULTI_COMMIT_BUILD_COUNT
)


print(
    "\nCommit-column candidates:"
)

display(
    commit_column_candidates.head(10)
)


print(
    "Selected entity-history commit column:",
    SELECTED_ENTITY_HISTORY_COMMIT_COLUMN
)

print(
    "Exact matched build-commit tokens:",
    SELECTED_EXACT_COMMIT_TOKEN_MATCHES
)

print(
    "Prefix matched build-commit tokens:",
    SELECTED_PREFIX_COMMIT_TOKEN_MATCHES
)

print(
    "Exact token coverage:",
    round(
        float(
            commit_column_candidates.iloc[0][
                "ExactTokenCoverage"
            ]
        ),
        6,
    )
)

print(
    "Prefix token coverage:",
    round(
        float(
            commit_column_candidates.iloc[0][
                "PrefixTokenCoverage"
            ]
        ),
        6,
    )
)


print(
    "\nEntity-ID pair candidates:"
)

display(
    entity_id_pair_candidates.head(10)
)


print(
    "Selected entity-history EntityId column:",
    SELECTED_ENTITY_HISTORY_ID_COLUMN
)

print(
    "Selected id_map EntityId column:",
    SELECTED_ID_MAP_ID_COLUMN
)

print(
    "Shared EntityIds:",
    SELECTED_ENTITY_ID_INTERSECTION
)

print(
    "Entity-history EntityId coverage:",
    round(
        SELECTED_ENTITY_HISTORY_ID_COVERAGE,
        6,
    )
)

print(
    "id_map EntityId coverage:",
    round(
        SELECTED_ID_MAP_ID_COVERAGE,
        6,
    )
)


print(
    "\nid_map column profile:"
)

display(
    id_map_column_profile
)


print(
    "\nSchema report:"
)

print(
    ENTITY_SCHEMA_REPORT_PATH
)


print(
    "\nValidation status:",
    "PASS"
)


print(
    "\nSUCCESS: Build commit values were parsed without "
    "altering any build."
)

print(
    "SUCCESS: The most strongly supported entity-history "
    "commit column was identified."
)

print(
    "SUCCESS: The most strongly supported EntityId pair "
    "was identified through value overlap."
)

print(
    "SUCCESS: No synthetic mapping or exclusion was "
    "performed."
)

print(
    "SUCCESS: Project 7 is ready for canonical "
    "build-commit-entity mapping."
)

=== PROJECT 7 STEP 3 RESULT ===

Source schemas:
builds.csv columns: ['id', 'commits', 'started_at']
entity_change_history.csv columns: ['EntityId', 'AddedLines', 'DeletedLines', 'Contributor', 'BugFix', 'Commit', 'CommitDate', 'MergeCommit']
id_map.csv columns: ['key', 'value']
contributors.csv columns: ['Id', 'Key', 'Name', 'Email']

Build-commit parsing:
Build rows: 415
Unique build-commit tokens: 405
Build rows with no parsed commit token: 0
Multi-commit builds: 7

Commit-column candidates:


,Column,Dtype,Rows,NonMissingValues,UniqueValues,SHA_LikeUniqueValues,SHA_LikeFraction,ExactMatchedBuildCommitTokens,PrefixMatchedBuildCommitTokens,UniqueBuildCommitTokens,ExactTokenCoverage,PrefixTokenCoverage,BuildsWithExactCommitMatch,BuildsWithPrefixCommitMatch,ColumnNameScore,TotalCandidateScore
0,Commit,object,13513,13513,2965,2965,1.0,405,405,405,1.0,1.000000,415,415,1500,13600.000000
1,AddedLines,int64,13513,13513,488,0,0.0,0,255,405,0.0,0.629630,0,263,0,1259.259259
2,DeletedLines,int64,13513,13513,402,0,0.0,0,255,405,0.0,0.629630,0,263,0,1259.259259
3,EntityId,int64,13513,13513,1648,0,0.0,0,232,405,0.0,0.572840,0,238,0,1145.679012
4,Contributor,int64,13513,13513,52,0,0.0,0,225,405,0.0,0.555556,0,231,0,1111.111111
5,CommitDate,object,13513,13513,2924,0,0.0,0,0,405,0.0,0.000000,0,0,500,500.000000
6,MergeCommit,bool,13513,13513,2,0,0.0,0,0,405,0.0,0.000000,0,0,500,500.000000
7,BugFix,int64,13513,13513,2,0,0.0,0,48,405,0.0,0.118519,0,50,0,237.037037


Selected entity-history commit column: Commit
Exact matched build-commit tokens: 405
Prefix matched build-commit tokens: 405
Exact token coverage: 1.0
Prefix token coverage: 1.0

Entity-ID pair candidates:


,EntityHistoryColumn,IdMapColumn,EntityHistoryUniqueValues,IdMapUniqueValues,IntersectionValues,EntityHistoryCoverage,IdMapCoverage,HistoryColumnNameScore,IdMapColumnNameScore,PairNameScore,TotalCandidateScore
0,EntityId,value,1648,1648,1648,1.000000,1.000000,1400,0,1400,13400.000000
1,AddedLines,value,488,1648,475,0.973361,0.288228,0,0,0,4330.642209
2,DeletedLines,value,402,1648,393,0.977612,0.238471,0,0,0,3755.320678
3,EntityId,key,1648,1766,0,0.000000,0.000000,1400,0,1400,1400.000000
4,Contributor,value,52,1648,52,1.000000,0.031553,0,0,0,1367.533981
5,BugFix,value,2,1648,1,0.500000,0.000607,0,0,0,507.067961
6,AddedLines,key,488,1766,0,0.000000,0.000000,0,0,0,0.000000
7,BugFix,key,2,1766,0,0.000000,0.000000,0,0,0,0.000000
8,CommitDate,key,2924,1766,0,0.000000,0.000000,0,0,0,0.000000
9,CommitDate,value,2924,1648,0,0.000000,0.000000,0,0,0,0.000000


Selected entity-history EntityId column: EntityId
Selected id_map EntityId column: value
Shared EntityIds: 1648
Entity-history EntityId coverage: 1.0
id_map EntityId coverage: 1.0

id_map column profile:


,Column,Dtype,Rows,MissingValues,UniqueNonMissingValues,SelectedAsEntityId
0,key,object,1766,0,1766,False
1,value,int64,1766,0,1648,True



Schema report:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_preflight/beast2_commit_entity_schema_report.json

Validation status: PASS

SUCCESS: Build commit values were parsed without altering any build.
SUCCESS: The most strongly supported entity-history commit column was identified.
SUCCESS: The most strongly supported EntityId pair was identified through value overlap.
SUCCESS: No synthetic mapping or exclusion was performed.
SUCCESS: Project 7 is ready for canonical build-commit-entity mapping.


In [ ]:
# =========================================================
# PROJECT 7 — STEP 4
# CANONICAL BUILD-COMMIT-ENTITY MAPPING
#
# This cell:
#   - explodes every build's parsed commit tokens
#   - maps commits exactly to entity-history rows
#   - maps EntityId values to id_map.csv
#   - creates canonical Build-Commit-Entity rows
#   - creates unique Build-Entity rows
#   - creates a changed-entity dictionary for every build
#   - validates complete commit and EntityId coverage
#   - saves permanent mapping artefacts
#
# It does NOT:
#   - use prefix matching
#   - exclude builds
#   - invent commits or entities
#   - reconstruct REC features
#   - inject noise or fit models
# =========================================================

from pathlib import Path
import json
import os
import re

import numpy as np
import pandas as pd
from IPython.display import clear_output, display


print(
    "=== PROJECT 7 STEP 4: "
    "CANONICAL BUILD-COMMIT-ENTITY MAPPING ==="
)


# ---------------------------------------------------------
# 1. Confirm successful Step 3 state
# ---------------------------------------------------------

required_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",
    "PROJECT_SHORT_NAME",

    "PROJECT_PREFLIGHT_DIRECTORY",
    "PROJECT_7_SELECTION_CHECKPOINT",

    "chronological_builds",
    "exe",
    "dataset",

    "build_commit_profile",
    "entity_history_schema_raw",
    "id_map_schema_raw",

    "SELECTED_ENTITY_HISTORY_COMMIT_COLUMN",
    "SELECTED_ENTITY_HISTORY_ID_COLUMN",
    "SELECTED_ID_MAP_ID_COLUMN",

    "UNIQUE_BUILD_COMMIT_TOKEN_COUNT",
    "SELECTED_EXACT_COMMIT_TOKEN_MATCHES",
    "SELECTED_ENTITY_ID_INTERSECTION",
]


missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required Project 7 Step 3 objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nRerun the successful Project 7 Step 3 "
        "schema-inspection cell."
    )


if PROJECT_NUMBER != 7:
    raise AssertionError(
        f"Expected Project 7, observed {PROJECT_NUMBER}."
    )


if PROJECT_NAME != "CompEvol@beast2":
    raise AssertionError(
        f"Unexpected project: {PROJECT_NAME}"
    )


if PROJECT_SLUG != "CompEvol__beast2":
    raise AssertionError(
        f"Unexpected project slug: {PROJECT_SLUG}"
    )


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    step_3_checkpoint = json.load(
        checkpoint_file
    )


allowed_previous_statuses = {
    "ENTITY_SCHEMA_INSPECTION_PASSED",
    "CANONICAL_ENTITY_MAPPING_PASSED",
}


if step_3_checkpoint.get(
    "Status"
) not in allowed_previous_statuses:
    raise AssertionError(
        "Project 7 Step 3 has not passed.\n"
        f"Observed status: "
        f"{step_3_checkpoint.get('Status')}"
    )


if SELECTED_ENTITY_HISTORY_COMMIT_COLUMN != "Commit":
    raise AssertionError(
        "Unexpected entity-history commit column:\n"
        f"{SELECTED_ENTITY_HISTORY_COMMIT_COLUMN}"
    )


if SELECTED_ENTITY_HISTORY_ID_COLUMN != "EntityId":
    raise AssertionError(
        "Unexpected entity-history EntityId column:\n"
        f"{SELECTED_ENTITY_HISTORY_ID_COLUMN}"
    )


if SELECTED_ID_MAP_ID_COLUMN != "value":
    raise AssertionError(
        "Unexpected id_map EntityId column:\n"
        f"{SELECTED_ID_MAP_ID_COLUMN}"
    )


if int(
    UNIQUE_BUILD_COMMIT_TOKEN_COUNT
) != 405:
    raise AssertionError(
        "Unexpected Step 3 build-commit token count.\n"
        f"Expected: 405\n"
        f"Observed: {UNIQUE_BUILD_COMMIT_TOKEN_COUNT}"
    )


if int(
    SELECTED_EXACT_COMMIT_TOKEN_MATCHES
) != 405:
    raise AssertionError(
        "Step 3 did not establish complete exact commit "
        "coverage."
    )


if int(
    SELECTED_ENTITY_ID_INTERSECTION
) != 1648:
    raise AssertionError(
        "Unexpected shared EntityId count."
    )


# ---------------------------------------------------------
# 2. Canonical normalisation helpers
# ---------------------------------------------------------

def normalise_commit_token(
    value,
):
    if pd.isna(value):
        return None


    token = (
        str(value)
        .strip()
        .strip("\"'")
        .lower()
    )


    if not token:
        return None


    if token in {
        "nan",
        "none",
        "null",
    }:
        return None


    return token


def canonical_integer_series(
    series,
    logical_name,
):
    numeric_values = pd.to_numeric(
        series,
        errors="coerce",
    )


    if numeric_values.isna().any():
        raise AssertionError(
            f"{logical_name} contains missing or "
            "non-numeric values."
        )


    values = numeric_values.to_numpy(
        dtype=float
    )


    if not np.isfinite(
        values
    ).all():
        raise AssertionError(
            f"{logical_name} contains non-finite values."
        )


    if not np.isclose(
        values,
        np.round(values),
        rtol=0.0,
        atol=1e-12,
    ).all():
        raise AssertionError(
            f"{logical_name} contains non-integer values."
        )


    return pd.Series(
        np.round(
            values
        ).astype(np.int64),
        index=series.index,
        name=series.name,
    )


def stable_join(
    values,
):
    cleaned_values = sorted({
        str(value).strip()
        for value in values
        if pd.notna(value)
        and str(value).strip()
    })


    return " | ".join(
        cleaned_values
    )


# ---------------------------------------------------------
# 3. Explode canonical build-commit token occurrences
# ---------------------------------------------------------

required_build_commit_profile_columns = {
    "Build",
    "build_order",
    "partition",
    "started_at",
    "CommitTokensList",
}


missing_profile_columns = (
    required_build_commit_profile_columns
    - set(
        build_commit_profile.columns
    )
)


if missing_profile_columns:
    raise RuntimeError(
        "build_commit_profile is missing columns:\n"
        + "\n".join(
            sorted(
                missing_profile_columns
            )
        )
    )


build_commit_tokens = (
    build_commit_profile[
        [
            "Build",
            "build_order",
            "partition",
            "started_at",
            "CommitTokensList",
        ]
    ]
    .explode(
        "CommitTokensList",
        ignore_index=True,
    )
    .rename(
        columns={
            "CommitTokensList":
                "CommitToken",
        }
    )
)


build_commit_tokens[
    "CommitToken"
] = (
    build_commit_tokens[
        "CommitToken"
    ]
    .map(
        normalise_commit_token
    )
)


build_commit_tokens = (
    build_commit_tokens[
        build_commit_tokens[
            "CommitToken"
        ].notna()
    ]
    .copy()
    .reset_index(drop=True)
)


build_commit_tokens[
    "Build"
] = canonical_integer_series(
    build_commit_tokens[
        "Build"
    ],
    "build_commit_tokens.Build",
)


build_commit_tokens[
    "build_order"
] = canonical_integer_series(
    build_commit_tokens[
        "build_order"
    ],
    "build_commit_tokens.build_order",
)


build_commit_tokens[
    "CommitTokenOrder"
] = (
    build_commit_tokens
    .groupby(
        "Build",
        sort=False,
    )
    .cumcount()
    + 1
)


build_commit_tokens[
    "CommitTokenOrder"
] = (
    build_commit_tokens[
        "CommitTokenOrder"
    ].astype(np.int64)
)


build_commit_tokens = (
    build_commit_tokens[
        [
            "Build",
            "build_order",
            "partition",
            "started_at",
            "CommitTokenOrder",
            "CommitToken",
        ]
    ]
    .sort_values(
        [
            "build_order",
            "CommitTokenOrder",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


BUILD_COMMIT_TOKEN_ROW_COUNT = int(
    len(
        build_commit_tokens
    )
)


BUILD_COMMIT_UNIQUE_TOKEN_COUNT = int(
    build_commit_tokens[
        "CommitToken"
    ].nunique()
)


if BUILD_COMMIT_UNIQUE_TOKEN_COUNT != 405:
    raise AssertionError(
        "Exploded build-commit tokens do not preserve "
        "the 405 unique Step 3 tokens.\n"
        f"Observed: {BUILD_COMMIT_UNIQUE_TOKEN_COUNT}"
    )


duplicate_build_token_rows = int(
    build_commit_tokens.duplicated(
        subset=[
            "Build",
            "CommitToken",
        ]
    ).sum()
)


if duplicate_build_token_rows != 0:
    raise AssertionError(
        "A build contains a duplicate canonical commit "
        "token."
    )


# ---------------------------------------------------------
# 4. Canonicalise entity-history commit and EntityId values
# ---------------------------------------------------------

canonical_entity_history = (
    entity_history_schema_raw
    .copy()
)


canonical_entity_history[
    "CommitToken"
] = (
    canonical_entity_history[
        SELECTED_ENTITY_HISTORY_COMMIT_COLUMN
    ]
    .map(
        normalise_commit_token
    )
)


canonical_entity_history[
    "EntityIdCanonical"
] = canonical_integer_series(
    canonical_entity_history[
        SELECTED_ENTITY_HISTORY_ID_COLUMN
    ],
    "entity_history.EntityId",
)


missing_history_commit_rows = int(
    canonical_entity_history[
        "CommitToken"
    ].isna().sum()
)


if missing_history_commit_rows != 0:
    raise AssertionError(
        "Entity history contains rows without a canonical "
        "commit token.\n"
        f"Rows: {missing_history_commit_rows}"
    )


entity_history_duplicate_commit_entity_rows = int(
    canonical_entity_history.duplicated(
        subset=[
            "CommitToken",
            "EntityIdCanonical",
        ]
    ).sum()
)


entity_history_commit_entity_pairs = (
    canonical_entity_history[
        [
            "CommitToken",
            "EntityIdCanonical",
        ]
    ]
    .rename(
        columns={
            "EntityIdCanonical":
                "EntityId",
        }
    )
    .drop_duplicates(
        subset=[
            "CommitToken",
            "EntityId",
        ]
    )
    .sort_values(
        [
            "CommitToken",
            "EntityId",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


ENTITY_HISTORY_COMMIT_COUNT = int(
    entity_history_commit_entity_pairs[
        "CommitToken"
    ].nunique()
)


ENTITY_HISTORY_UNIQUE_ENTITY_COUNT = int(
    entity_history_commit_entity_pairs[
        "EntityId"
    ].nunique()
)


# ---------------------------------------------------------
# 5. Canonicalise id_map.csv
# ---------------------------------------------------------

canonical_id_map = (
    id_map_schema_raw
    .copy()
)


canonical_id_map[
    "EntityId"
] = canonical_integer_series(
    canonical_id_map[
        SELECTED_ID_MAP_ID_COLUMN
    ],
    "id_map.EntityId",
)


id_map_metadata_columns = [
    column
    for column in id_map_schema_raw.columns
    if column != SELECTED_ID_MAP_ID_COLUMN
]


if len(id_map_metadata_columns) == 0:
    canonical_id_map[
        "EntityKey"
    ] = ""

else:
    def collect_row_entity_key(
        row,
    ):
        row_values = []


        for column in id_map_metadata_columns:
            value = row[
                column
            ]


            if pd.isna(value):
                continue


            value_text = str(
                value
            ).strip()


            if not value_text:
                continue


            row_values.append(
                value_text
            )


        return " | ".join(
            row_values
        )


    canonical_id_map[
        "EntityKey"
    ] = (
        canonical_id_map.apply(
            collect_row_entity_key,
            axis=1,
        )
    )


id_map_duplicate_rows = int(
    canonical_id_map.duplicated().sum()
)


id_map_duplicate_entity_ids = int(
    canonical_id_map.duplicated(
        subset=[
            "EntityId",
        ],
        keep=False,
    ).sum()
)


entity_id_dictionary = (
    canonical_id_map
    .groupby(
        "EntityId",
        as_index=False,
        sort=True,
    )
    .agg(
        EntityKeyCount=(
            "EntityKey",
            lambda values:
                int(
                    len({
                        str(value).strip()
                        for value in values
                        if str(value).strip()
                    })
                ),
        ),

        PrimaryEntityKey=(
            "EntityKey",
            lambda values:
                (
                    sorted({
                        str(value).strip()
                        for value in values
                        if str(value).strip()
                    })[0]
                    if len({
                        str(value).strip()
                        for value in values
                        if str(value).strip()
                    }) > 0
                    else ""
                ),
        ),

        EntityKeys=(
            "EntityKey",
            stable_join,
        ),
    )
)


history_entity_ids = set(
    entity_history_commit_entity_pairs[
        "EntityId"
    ].astype(np.int64)
)


id_map_entity_ids = set(
    entity_id_dictionary[
        "EntityId"
    ].astype(np.int64)
)


history_ids_missing_from_id_map = sorted(
    history_entity_ids
    - id_map_entity_ids
)


id_map_ids_missing_from_history = sorted(
    id_map_entity_ids
    - history_entity_ids
)


if history_ids_missing_from_id_map:
    raise AssertionError(
        "Entity-history EntityIds are missing from "
        "id_map.csv:\n"
        + ", ".join(
            map(
                str,
                history_ids_missing_from_id_map[:30],
            )
        )
    )


if id_map_ids_missing_from_history:
    raise AssertionError(
        "id_map.csv contains EntityIds absent from the "
        "entity history, contrary to Step 3 coverage.\n"
        + ", ".join(
            map(
                str,
                id_map_ids_missing_from_history[:30],
            )
        )
    )


if len(
    history_entity_ids
) != 1648:
    raise AssertionError(
        "Unexpected entity-history EntityId count.\n"
        f"Observed: {len(history_entity_ids)}"
    )


if len(
    id_map_entity_ids
) != 1648:
    raise AssertionError(
        "Unexpected id_map EntityId count.\n"
        f"Observed: {len(id_map_entity_ids)}"
    )


# ---------------------------------------------------------
# 6. Match every build-commit occurrence exactly
# ---------------------------------------------------------

commit_entity_count_lookup = (
    entity_history_commit_entity_pairs
    .groupby(
        "CommitToken"
    )[
        "EntityId"
    ]
    .nunique()
    .to_dict()
)


build_commit_tokens[
    "MatchedEntityCount"
] = (
    build_commit_tokens[
        "CommitToken"
    ]
    .map(
        commit_entity_count_lookup
    )
    .fillna(0)
    .astype(np.int64)
)


build_commit_tokens[
    "CommitMatched"
] = (
    build_commit_tokens[
        "MatchedEntityCount"
    ] > 0
)


matched_unique_commit_tokens = sorted(
    set(
        build_commit_tokens.loc[
            build_commit_tokens[
                "CommitMatched"
            ],
            "CommitToken",
        ]
    )
)


unmatched_unique_commit_tokens = sorted(
    set(
        build_commit_tokens.loc[
            ~build_commit_tokens[
                "CommitMatched"
            ],
            "CommitToken",
        ]
    )
)


MATCHED_UNIQUE_COMMIT_TOKEN_COUNT = int(
    len(
        matched_unique_commit_tokens
    )
)


UNMATCHED_UNIQUE_COMMIT_TOKEN_COUNT = int(
    len(
        unmatched_unique_commit_tokens
    )
)


MATCHED_BUILD_COMMIT_TOKEN_ROWS = int(
    build_commit_tokens[
        "CommitMatched"
    ].sum()
)


UNMATCHED_BUILD_COMMIT_TOKEN_ROWS = int(
    (
        ~build_commit_tokens[
            "CommitMatched"
        ]
    ).sum()
)


UNIQUE_COMMIT_TOKEN_COVERAGE_PERCENT = float(
    100.0
    * MATCHED_UNIQUE_COMMIT_TOKEN_COUNT
    / BUILD_COMMIT_UNIQUE_TOKEN_COUNT
)


if MATCHED_UNIQUE_COMMIT_TOKEN_COUNT != 405:
    raise AssertionError(
        "Exact canonical mapping did not match all "
        "405 build commit tokens.\n"
        f"Matched: {MATCHED_UNIQUE_COMMIT_TOKEN_COUNT}"
    )


if UNMATCHED_UNIQUE_COMMIT_TOKEN_COUNT != 0:
    raise AssertionError(
        "Unexpected unmatched Beast2 commit tokens:\n"
        + "\n".join(
            unmatched_unique_commit_tokens
        )
    )


if UNMATCHED_BUILD_COMMIT_TOKEN_ROWS != 0:
    raise AssertionError(
        "At least one build-commit occurrence was not "
        "mapped."
    )


# ---------------------------------------------------------
# 7. Create canonical Build-Commit-Entity mapping
# ---------------------------------------------------------

build_commit_entity_mapping = (
    build_commit_tokens
    .merge(
        entity_history_commit_entity_pairs,
        on="CommitToken",
        how="inner",
        validate="many_to_many",
    )
    [
        [
            "Build",
            "build_order",
            "partition",
            "started_at",
            "CommitTokenOrder",
            "CommitToken",
            "EntityId",
        ]
    ]
    .drop_duplicates(
        subset=[
            "Build",
            "CommitToken",
            "EntityId",
        ]
    )
    .sort_values(
        [
            "build_order",
            "CommitTokenOrder",
            "EntityId",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


duplicate_build_commit_entity_rows = int(
    build_commit_entity_mapping.duplicated(
        subset=[
            "Build",
            "CommitToken",
            "EntityId",
        ]
    ).sum()
)


if duplicate_build_commit_entity_rows != 0:
    raise AssertionError(
        "Canonical Build-Commit-Entity mapping contains "
        "duplicate rows."
    )


build_entity_mapping = (
    build_commit_entity_mapping[
        [
            "Build",
            "build_order",
            "partition",
            "started_at",
            "EntityId",
        ]
    ]
    .drop_duplicates(
        subset=[
            "Build",
            "EntityId",
        ]
    )
    .sort_values(
        [
            "build_order",
            "EntityId",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


duplicate_build_entity_rows = int(
    build_entity_mapping.duplicated(
        subset=[
            "Build",
            "EntityId",
        ]
    ).sum()
)


if duplicate_build_entity_rows != 0:
    raise AssertionError(
        "Canonical Build-Entity mapping contains "
        "duplicate rows."
    )


mapped_entity_ids = set(
    build_entity_mapping[
        "EntityId"
    ].astype(np.int64)
)


mapped_entities_missing_from_id_map = sorted(
    mapped_entity_ids
    - id_map_entity_ids
)


if mapped_entities_missing_from_id_map:
    raise AssertionError(
        "Mapped EntityIds are missing from id_map.csv."
    )


# ---------------------------------------------------------
# 8. Create runtime dictionaries for later REC work
# ---------------------------------------------------------

commit_to_entity_ids = {
    str(commit_token):
        tuple(
            sorted(
                group_rows[
                    "EntityId"
                ]
                .astype(np.int64)
                .unique()
                .tolist()
            )
        )

    for commit_token, group_rows in (
        entity_history_commit_entity_pairs
        .groupby(
            "CommitToken",
            sort=True,
        )
    )
}


build_to_commit_tokens = {
    int(build_id):
        tuple(
            group_rows
            .sort_values(
                "CommitTokenOrder",
                kind="mergesort",
            )[
                "CommitToken"
            ]
            .astype(str)
            .tolist()
        )

    for build_id, group_rows in (
        build_commit_tokens
        .groupby(
            "Build",
            sort=False,
        )
    )
}


mapped_entities_by_build = {
    int(build_id):
        tuple(
            sorted(
                group_rows[
                    "EntityId"
                ]
                .astype(np.int64)
                .unique()
                .tolist()
            )
        )

    for build_id, group_rows in (
        build_entity_mapping
        .groupby(
            "Build",
            sort=False,
        )
    )
}


all_chronological_build_ids = (
    chronological_builds[
        "Build"
    ]
    .astype(np.int64)
    .tolist()
)


changed_entity_dictionary = {
    int(build_id):
        mapped_entities_by_build.get(
            int(build_id),
            tuple(),
        )

    for build_id in all_chronological_build_ids
}


changed_entity_set_dictionary = {
    int(build_id):
        frozenset(
            changed_entity_dictionary[
                int(build_id)
            ]
        )

    for build_id in all_chronological_build_ids
}


if len(
    changed_entity_dictionary
) != len(
    chronological_builds
):
    raise AssertionError(
        "Changed-entity dictionary does not contain "
        "every chronological build."
    )


# ---------------------------------------------------------
# 9. Create per-build mapping-status audit
# ---------------------------------------------------------

total_commit_counts = (
    build_commit_tokens
    .groupby(
        "Build"
    )
    .size()
    .rename(
        "CommitTokens"
    )
)


matched_commit_counts = (
    build_commit_tokens[
        build_commit_tokens[
            "CommitMatched"
        ]
    ]
    .groupby(
        "Build"
    )
    .size()
    .rename(
        "MatchedCommitTokens"
    )
)


unmatched_commit_text = (
    build_commit_tokens[
        ~build_commit_tokens[
            "CommitMatched"
        ]
    ]
    .groupby(
        "Build"
    )[
        "CommitToken"
    ]
    .apply(
        stable_join
    )
    .rename(
        "UnmatchedCommitTokenValues"
    )
)


changed_entity_counts = (
    build_entity_mapping
    .groupby(
        "Build"
    )[
        "EntityId"
    ]
    .nunique()
    .rename(
        "ChangedEntities"
    )
)


raw_build_statistics = (
    exe.assign(
        _Failure=(
            pd.to_numeric(
                exe[
                    "Verdict"
                ],
                errors="raise",
            ).to_numpy(dtype=np.int64)
            != 0
        ).astype(np.int64)
    )
    .groupby(
        "Build"
    )
    .agg(
        RawExecutionRows=(
            "Test",
            "size",
        ),

        RawFailureExecutions=(
            "_Failure",
            "sum",
        ),
    )
)


model_build_statistics = (
    dataset.assign(
        _Failure=(
            pd.to_numeric(
                dataset[
                    "Verdict"
                ],
                errors="raise",
            ).to_numpy(dtype=np.int64)
            != 0
        ).astype(np.int64)
    )
    .groupby(
        "Build"
    )
    .agg(
        ModelReadyRows=(
            "Test",
            "size",
        ),

        ModelReadyFailureExecutions=(
            "_Failure",
            "sum",
        ),
    )
)


build_mapping_status = (
    chronological_builds[
        [
            "Build",
            "build_order",
            "partition",
            "started_at",
        ]
    ]
    .merge(
        total_commit_counts,
        left_on="Build",
        right_index=True,
        how="left",
    )
    .merge(
        matched_commit_counts,
        left_on="Build",
        right_index=True,
        how="left",
    )
    .merge(
        unmatched_commit_text,
        left_on="Build",
        right_index=True,
        how="left",
    )
    .merge(
        changed_entity_counts,
        left_on="Build",
        right_index=True,
        how="left",
    )
    .merge(
        raw_build_statistics,
        left_on="Build",
        right_index=True,
        how="left",
    )
    .merge(
        model_build_statistics,
        left_on="Build",
        right_index=True,
        how="left",
    )
)


integer_audit_columns = [
    "CommitTokens",
    "MatchedCommitTokens",
    "ChangedEntities",
    "RawExecutionRows",
    "RawFailureExecutions",
    "ModelReadyRows",
    "ModelReadyFailureExecutions",
]


for column in integer_audit_columns:
    build_mapping_status[
        column
    ] = (
        build_mapping_status[
            column
        ]
        .fillna(0)
        .astype(np.int64)
    )


build_mapping_status[
    "UnmatchedCommitTokens"
] = (
    build_mapping_status[
        "CommitTokens"
    ]
    - build_mapping_status[
        "MatchedCommitTokens"
    ]
)


build_mapping_status[
    "UnmatchedCommitTokenValues"
] = (
    build_mapping_status[
        "UnmatchedCommitTokenValues"
    ]
    .fillna("")
    .astype(str)
)


build_mapping_status[
    "CommitCoveragePercent"
] = np.where(
    build_mapping_status[
        "CommitTokens"
    ] > 0,

    100.0
    * build_mapping_status[
        "MatchedCommitTokens"
    ]
    / build_mapping_status[
        "CommitTokens"
    ],

    0.0,
)


build_mapping_status[
    "MappingStatus"
] = np.select(
    [
        build_mapping_status[
            "CommitTokens"
        ] == 0,

        (
            build_mapping_status[
                "CommitTokens"
            ] > 0
        )
        & (
            build_mapping_status[
                "MatchedCommitTokens"
            ]
            ==
            build_mapping_status[
                "CommitTokens"
            ]
        ),

        build_mapping_status[
            "MatchedCommitTokens"
        ] > 0,
    ],
    [
        "NO_COMMIT_TOKENS",
        "FULL_COMMIT_COVERAGE",
        "PARTIAL_COMMIT_COVERAGE",
    ],
    default="NO_MATCHED_COMMITS",
)


build_mapping_status = (
    build_mapping_status
    .sort_values(
        "build_order",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


mapping_status_summary = (
    build_mapping_status[
        "MappingStatus"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "MappingStatus"
    )
    .reset_index(
        name="Builds"
    )
)


FULLY_MAPPED_BUILD_COUNT = int(
    (
        build_mapping_status[
            "MappingStatus"
        ]
        == "FULL_COMMIT_COVERAGE"
    ).sum()
)


PARTIALLY_MAPPED_BUILD_COUNT = int(
    (
        build_mapping_status[
            "MappingStatus"
        ]
        == "PARTIAL_COMMIT_COVERAGE"
    ).sum()
)


NO_MATCHED_COMMIT_BUILD_COUNT = int(
    build_mapping_status[
        "MappingStatus"
    ].isin(
        [
            "NO_COMMIT_TOKENS",
            "NO_MATCHED_COMMITS",
        ]
    ).sum()
)


BUILDS_WITH_MAPPED_ENTITIES = int(
    (
        build_mapping_status[
            "ChangedEntities"
        ] > 0
    ).sum()
)


if FULLY_MAPPED_BUILD_COUNT != 415:
    display(
        build_mapping_status[
            build_mapping_status[
                "MappingStatus"
            ]
            != "FULL_COMMIT_COVERAGE"
        ]
    )


    raise AssertionError(
        "Expected all 415 Beast2 builds to have full "
        "exact commit coverage.\n"
        f"Fully mapped: {FULLY_MAPPED_BUILD_COUNT}"
    )


if PARTIALLY_MAPPED_BUILD_COUNT != 0:
    raise AssertionError(
        "Unexpected partially mapped Beast2 builds."
    )


if NO_MATCHED_COMMIT_BUILD_COUNT != 0:
    raise AssertionError(
        "Unexpected Beast2 builds without matched commits."
    )


if BUILDS_WITH_MAPPED_ENTITIES != 415:
    raise AssertionError(
        "At least one Beast2 build has no mapped entity.\n"
        f"Builds with mapped entities: "
        f"{BUILDS_WITH_MAPPED_ENTITIES}"
    )


# ---------------------------------------------------------
# 10. Create unmatched-token audit table
# ---------------------------------------------------------

unmatched_commit_tokens = (
    build_commit_tokens[
        ~build_commit_tokens[
            "CommitMatched"
        ]
    ][
        [
            "Build",
            "build_order",
            "partition",
            "CommitTokenOrder",
            "CommitToken",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 11. Save permanent Step 4 artefacts
# ---------------------------------------------------------

BUILD_COMMIT_TOKENS_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_build_commit_tokens.csv"
)


ENTITY_HISTORY_PAIRS_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_entity_history_commit_entity_pairs.parquet"
)


BUILD_COMMIT_ENTITY_MAPPING_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_build_commit_entity_mapping.parquet"
)


BUILD_ENTITY_MAPPING_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_build_entity_mapping.parquet"
)


BUILD_MAPPING_STATUS_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_build_mapping_status.csv"
)


ENTITY_ID_DICTIONARY_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_entity_id_dictionary.csv"
)


UNMATCHED_COMMIT_TOKENS_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_unmatched_commit_tokens.csv"
)


CANONICAL_MAPPING_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_canonical_entity_mapping_report.json"
)


build_commit_tokens.to_csv(
    BUILD_COMMIT_TOKENS_PATH,
    index=False,
)


entity_history_commit_entity_pairs.to_parquet(
    ENTITY_HISTORY_PAIRS_PATH,
    index=False,
)


build_commit_entity_mapping.to_parquet(
    BUILD_COMMIT_ENTITY_MAPPING_PATH,
    index=False,
)


build_entity_mapping.to_parquet(
    BUILD_ENTITY_MAPPING_PATH,
    index=False,
)


build_mapping_status.to_csv(
    BUILD_MAPPING_STATUS_PATH,
    index=False,
)


entity_id_dictionary.to_csv(
    ENTITY_ID_DICTIONARY_PATH,
    index=False,
)


unmatched_commit_tokens.to_csv(
    UNMATCHED_COMMIT_TOKENS_PATH,
    index=False,
)


mapping_completed_at = (
    pd.Timestamp.utcnow().isoformat()
)


canonical_mapping_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "PASS",

    "MappingPolicy": {
        "CommitMatching":
            "EXACT_NORMALISED_COMMIT_TOKEN",

        "PrefixMatchingUsed":
            False,

        "SyntheticCommitMappings":
            0,

        "SyntheticEntities":
            0,

        "ExcludedBuilds":
            0,
    },

    "CommitMapping": {
        "Builds":
            int(
                len(
                    chronological_builds
                )
            ),

        "BuildCommitTokenRows":
            BUILD_COMMIT_TOKEN_ROW_COUNT,

        "UniqueBuildCommitTokens":
            BUILD_COMMIT_UNIQUE_TOKEN_COUNT,

        "MatchedBuildCommitTokenRows":
            MATCHED_BUILD_COMMIT_TOKEN_ROWS,

        "UnmatchedBuildCommitTokenRows":
            UNMATCHED_BUILD_COMMIT_TOKEN_ROWS,

        "MatchedUniqueCommitTokens":
            MATCHED_UNIQUE_COMMIT_TOKEN_COUNT,

        "UnmatchedUniqueCommitTokens":
            UNMATCHED_UNIQUE_COMMIT_TOKEN_COUNT,

        "UniqueCommitTokenCoveragePercent":
            UNIQUE_COMMIT_TOKEN_COVERAGE_PERCENT,
    },

    "BuildMapping": {
        "FullyMappedBuilds":
            FULLY_MAPPED_BUILD_COUNT,

        "PartiallyMappedBuilds":
            PARTIALLY_MAPPED_BUILD_COUNT,

        "BuildsWithNoMatchedCommit":
            NO_MATCHED_COMMIT_BUILD_COUNT,

        "BuildsWithAtLeastOneMappedEntity":
            BUILDS_WITH_MAPPED_ENTITIES,

        "BuildCommitEntityRows":
            int(
                len(
                    build_commit_entity_mapping
                )
            ),

        "UniqueBuildEntityPairs":
            int(
                len(
                    build_entity_mapping
                )
            ),

        "UniqueMappedEntities":
            int(
                len(
                    mapped_entity_ids
                )
            ),

        "ChangedEntityDictionaryBuilds":
            int(
                len(
                    changed_entity_dictionary
                )
            ),
    },

    "EntityIdValidation": {
        "EntityHistoryUniqueEntityIds":
            int(
                len(
                    history_entity_ids
                )
            ),

        "IdMapUniqueEntityIds":
            int(
                len(
                    id_map_entity_ids
                )
            ),

        "HistoryEntityIdsMissingFromIdMap":
            int(
                len(
                    history_ids_missing_from_id_map
                )
            ),

        "IdMapEntityIdsMissingFromHistory":
            int(
                len(
                    id_map_ids_missing_from_history
                )
            ),

        "EntityHistoryDuplicateCommitEntityRows":
            entity_history_duplicate_commit_entity_rows,

        "IdMapDuplicateRows":
            id_map_duplicate_rows,

        "IdMapRowsBelongingToDuplicatedEntityIds":
            id_map_duplicate_entity_ids,
    },

    "MappingStatusCounts":
        {
            str(row.MappingStatus):
                int(row.Builds)
            for row in (
                mapping_status_summary.itertuples(
                    index=False
                )
            )
        },

    "Artefacts": {
        "BuildCommitTokens":
            str(
                BUILD_COMMIT_TOKENS_PATH
            ),

        "EntityHistoryCommitEntityPairs":
            str(
                ENTITY_HISTORY_PAIRS_PATH
            ),

        "BuildCommitEntityMapping":
            str(
                BUILD_COMMIT_ENTITY_MAPPING_PATH
            ),

        "BuildEntityMapping":
            str(
                BUILD_ENTITY_MAPPING_PATH
            ),

        "BuildMappingStatus":
            str(
                BUILD_MAPPING_STATUS_PATH
            ),

        "EntityIdDictionary":
            str(
                ENTITY_ID_DICTIONARY_PATH
            ),

        "UnmatchedCommitTokens":
            str(
                UNMATCHED_COMMIT_TOKENS_PATH
            ),
    },

    "CompletedAtUTC":
        mapping_completed_at,
}


with open(
    CANONICAL_MAPPING_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        canonical_mapping_report,
        report_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 12. Update Project 7 checkpoint atomically
# ---------------------------------------------------------

with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    project_7_checkpoint = json.load(
        checkpoint_file
    )


project_7_checkpoint.update({
    "Status":
        "CANONICAL_ENTITY_MAPPING_PASSED",

    "CommitMatchingPolicy":
        "EXACT_NORMALISED_COMMIT_TOKEN",

    "PrefixMatchingUsed":
        False,

    "BuildCommitTokenRows":
        BUILD_COMMIT_TOKEN_ROW_COUNT,

    "UniqueBuildCommitTokens":
        BUILD_COMMIT_UNIQUE_TOKEN_COUNT,

    "MatchedUniqueCommitTokens":
        MATCHED_UNIQUE_COMMIT_TOKEN_COUNT,

    "UnmatchedUniqueCommitTokens":
        UNMATCHED_UNIQUE_COMMIT_TOKEN_COUNT,

    "CommitTokenCoveragePercent":
        UNIQUE_COMMIT_TOKEN_COVERAGE_PERCENT,

    "FullyMappedBuilds":
        FULLY_MAPPED_BUILD_COUNT,

    "PartiallyMappedBuilds":
        PARTIALLY_MAPPED_BUILD_COUNT,

    "BuildsWithNoMatchedCommit":
        NO_MATCHED_COMMIT_BUILD_COUNT,

    "BuildsWithMappedEntities":
        BUILDS_WITH_MAPPED_ENTITIES,

    "BuildCommitEntityRows":
        int(
            len(
                build_commit_entity_mapping
            )
        ),

    "UniqueBuildEntityPairs":
        int(
            len(
                build_entity_mapping
            )
        ),

    "UniqueMappedEntities":
        int(
            len(
                mapped_entity_ids
            )
        ),

    "ChangedEntityDictionaryBuilds":
        int(
            len(
                changed_entity_dictionary
            )
        ),

    "EntityHistoryUniqueEntityIds":
        int(
            len(
                history_entity_ids
            )
        ),

    "IdMapUniqueEntityIds":
        int(
            len(
                id_map_entity_ids
            )
        ),

    "HistoryEntityIdsMissingFromIdMap":
        int(
            len(
                history_ids_missing_from_id_map
            )
        ),

    "IdMapEntityIdsMissingFromHistory":
        int(
            len(
                id_map_ids_missing_from_history
            )
        ),

    "CanonicalEntityMappingReport":
        str(
            CANONICAL_MAPPING_REPORT_PATH
        ),

    "BuildCommitTokens":
        str(
            BUILD_COMMIT_TOKENS_PATH
        ),

    "BuildCommitEntityMapping":
        str(
            BUILD_COMMIT_ENTITY_MAPPING_PATH
        ),

    "BuildEntityMapping":
        str(
            BUILD_ENTITY_MAPPING_PATH
        ),

    "BuildMappingStatus":
        str(
            BUILD_MAPPING_STATUS_PATH
        ),

    "EntityIdDictionary":
        str(
            ENTITY_ID_DICTIONARY_PATH
        ),

    "UpdatedAtUTC":
        mapping_completed_at,
})


temporary_checkpoint_path = (
    PROJECT_7_SELECTION_CHECKPOINT
    .with_suffix(
        ".json.tmp"
    )
)


with open(
    temporary_checkpoint_path,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        project_7_checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


os.replace(
    temporary_checkpoint_path,
    PROJECT_7_SELECTION_CHECKPOINT,
)


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint_verification = json.load(
        checkpoint_file
    )


if checkpoint_verification.get(
    "Status"
) != "CANONICAL_ENTITY_MAPPING_PASSED":
    raise AssertionError(
        "The permanent Project 7 mapping checkpoint was "
        "not written correctly."
    )


# ---------------------------------------------------------
# 13. Compact final output
# ---------------------------------------------------------

clear_output(
    wait=True
)


print(
    "=== PROJECT 7 STEP 4 RESULT ==="
)


print(
    "\nCommit mapping:"
)

print(
    "Builds:",
    len(
        chronological_builds
    )
)

print(
    "Build-commit token rows:",
    BUILD_COMMIT_TOKEN_ROW_COUNT
)

print(
    "Unique build-commit tokens:",
    BUILD_COMMIT_UNIQUE_TOKEN_COUNT
)

print(
    "Matched unique commit tokens:",
    MATCHED_UNIQUE_COMMIT_TOKEN_COUNT
)

print(
    "Unmatched unique commit tokens:",
    UNMATCHED_UNIQUE_COMMIT_TOKEN_COUNT
)

print(
    "Commit-token coverage:",
    f"{UNIQUE_COMMIT_TOKEN_COVERAGE_PERCENT:.4f}",
    "%"
)


print(
    "\nBuild mapping:"
)

print(
    "Fully mapped builds:",
    FULLY_MAPPED_BUILD_COUNT
)

print(
    "Partially mapped builds:",
    PARTIALLY_MAPPED_BUILD_COUNT
)

print(
    "Builds with no matched commit:",
    NO_MATCHED_COMMIT_BUILD_COUNT
)

print(
    "Builds with at least one mapped entity:",
    BUILDS_WITH_MAPPED_ENTITIES
)

print(
    "Build-commit-entity rows:",
    len(
        build_commit_entity_mapping
    )
)

print(
    "Unique Build-Entity pairs:",
    len(
        build_entity_mapping
    )
)

print(
    "Unique mapped entities:",
    len(
        mapped_entity_ids
    )
)

print(
    "Changed-entity dictionary builds:",
    len(
        changed_entity_dictionary
    )
)


print(
    "\nEntity-ID validation:"
)

print(
    "Entity-history unique EntityIds:",
    len(
        history_entity_ids
    )
)

print(
    "id_map unique EntityIds:",
    len(
        id_map_entity_ids
    )
)

print(
    "History EntityIds missing from id_map:",
    len(
        history_ids_missing_from_id_map
    )
)

print(
    "id_map EntityIds missing from history:",
    len(
        id_map_ids_missing_from_history
    )
)


print(
    "\nMapping status summary:"
)

display(
    mapping_status_summary
)


print(
    "\nBuilds affected by unmatched commits:"
)


affected_builds = (
    build_mapping_status[
        build_mapping_status[
            "MappingStatus"
        ]
        != "FULL_COMMIT_COVERAGE"
    ]
)


if len(
    affected_builds
) == 0:
    print(
        "None"
    )

else:
    display(
        affected_builds
    )


print(
    "\nCanonical mapping report:"
)

print(
    CANONICAL_MAPPING_REPORT_PATH
)


print(
    "\nValidation status:",
    "PASS"
)


print(
    "\nSUCCESS: All 405 unique build commit tokens were "
    "matched exactly."
)

print(
    "SUCCESS: All 415 builds received full commit "
    "coverage."
)

print(
    "SUCCESS: Every mapped EntityId was validated against "
    "id_map.csv."
)

print(
    "SUCCESS: No prefix matching, synthetic mapping or "
    "build exclusion was used."
)

print(
    "SUCCESS: The changed-entity dictionary contains "
    "all 415 chronological builds."
)

print(
    "SUCCESS: Project 7 is ready for clean REC "
    "reconstruction and validation."
)

=== PROJECT 7 STEP 4 RESULT ===

Commit mapping:
Builds: 415
Build-commit token rows: 427
Unique build-commit tokens: 405
Matched unique commit tokens: 405
Unmatched unique commit tokens: 0
Commit-token coverage: 100.0000 %

Build mapping:
Fully mapped builds: 415
Partially mapped builds: 0
Builds with no matched commit: 0
Builds with at least one mapped entity: 415
Build-commit-entity rows: 1303
Unique Build-Entity pairs: 1295
Unique mapped entities: 317
Changed-entity dictionary builds: 415

Entity-ID validation:
Entity-history unique EntityIds: 1648
id_map unique EntityIds: 1648
History EntityIds missing from id_map: 0
id_map EntityIds missing from history: 0

Mapping status summary:


,MappingStatus,Builds
0,FULL_COMMIT_COVERAGE,415



Builds affected by unmatched commits:
None

Canonical mapping report:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_preflight/beast2_canonical_entity_mapping_report.json

Validation status: PASS

SUCCESS: All 405 unique build commit tokens were matched exactly.
SUCCESS: All 415 builds received full commit coverage.
SUCCESS: Every mapped EntityId was validated against id_map.csv.
SUCCESS: No prefix matching, synthetic mapping or build exclusion was used.
SUCCESS: The changed-entity dictionary contains all 415 chronological builds.
SUCCESS: Project 7 is ready for clean REC reconstruction and validation.


In [ ]:
# =========================================================
# PROJECT 7 — STEP 5
# REC FEATURE SCHEMA AND HISTORY INSPECTION
#
# This cell:
#   - identifies the exact Beast2 REC feature columns
#   - verifies the frozen 25-column REC structure
#   - separates the six verdict-independent REC features
#     from the nineteen verdict-dependent REC features
#   - profiles every REC feature
#   - profiles raw and model-ready verdict subtypes
#   - profiles test execution histories
#   - saves representative clean REC trajectories
#   - writes a permanent schema-inspection report
#
# It does NOT:
#   - reconstruct REC values
#   - correct or overwrite dataset features
#   - inject noise
#   - create synthetic executions
#   - train any model
# =========================================================

from pathlib import Path
import json
import os
import re

import numpy as np
import pandas as pd
from IPython.display import clear_output, display


print(
    "=== PROJECT 7 STEP 5: "
    "REC FEATURE SCHEMA AND HISTORY INSPECTION ==="
)


# ---------------------------------------------------------
# 1. Confirm successful Step 4 state
# ---------------------------------------------------------

required_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",
    "PROJECT_SHORT_NAME",

    "PROJECT_PREFLIGHT_DIRECTORY",
    "PROJECT_7_SELECTION_CHECKPOINT",

    "chronological_builds",
    "exe",
    "dataset",

    "raw_training_history",
    "raw_evaluation_history",
    "all_model_training_data",
    "all_model_evaluation_data",
    "clean_training_data",
    "clean_evaluation_data",

    "PREDICTOR_COLUMNS",

    "changed_entity_dictionary",
    "changed_entity_set_dictionary",
]


missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required Project 7 objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nRerun only the successful Project 7 "
        "Step 2B, Step 3 and Step 4 cells."
    )


if PROJECT_NUMBER != 7:
    raise AssertionError(
        f"Expected Project 7, observed {PROJECT_NUMBER}."
    )


if PROJECT_NAME != "CompEvol@beast2":
    raise AssertionError(
        f"Unexpected project: {PROJECT_NAME}"
    )


if PROJECT_SLUG != "CompEvol__beast2":
    raise AssertionError(
        f"Unexpected project slug: {PROJECT_SLUG}"
    )


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    step_4_checkpoint = json.load(
        checkpoint_file
    )


allowed_previous_statuses = {
    "CANONICAL_ENTITY_MAPPING_PASSED",
    "REC_SCHEMA_INSPECTION_PASSED",
}


if step_4_checkpoint.get(
    "Status"
) not in allowed_previous_statuses:
    raise AssertionError(
        "Project 7 canonical entity mapping has not "
        "passed.\n"
        f"Observed status: "
        f"{step_4_checkpoint.get('Status')}"
    )


if int(
    step_4_checkpoint.get(
        "FullyMappedBuilds",
        -1,
    )
) != 415:
    raise AssertionError(
        "The checkpoint does not confirm 415 fully "
        "mapped builds."
    )


if int(
    step_4_checkpoint.get(
        "UnmatchedUniqueCommitTokens",
        -1,
    )
) != 0:
    raise AssertionError(
        "The checkpoint contains unmatched commits."
    )


# ---------------------------------------------------------
# 2. Work with defragmented copies
# ---------------------------------------------------------

exe_work = (
    exe.copy()
    .reset_index(drop=True)
)


dataset_work = (
    dataset.copy()
    .reset_index(drop=True)
)


raw_training_work = (
    raw_training_history.copy()
    .reset_index(drop=True)
)


raw_evaluation_work = (
    raw_evaluation_history.copy()
    .reset_index(drop=True)
)


model_training_work = (
    all_model_training_data.copy()
    .reset_index(drop=True)
)


model_evaluation_work = (
    all_model_evaluation_data.copy()
    .reset_index(drop=True)
)


required_core_columns = {
    "Build",
    "Test",
    "Verdict",
    "Duration",
    "build_order",
    "partition",
}


for frame_name, frame in [
    (
        "exe",
        exe_work,
    ),
    (
        "dataset",
        dataset_work,
    ),
]:
    missing_columns = (
        required_core_columns
        - set(
            frame.columns
        )
    )


    if missing_columns:
        raise RuntimeError(
            f"{frame_name} is missing required columns:\n"
            + "\n".join(
                sorted(
                    missing_columns
                )
            )
        )


# ---------------------------------------------------------
# 3. Identify the exact REC feature columns
# ---------------------------------------------------------

def normalise_feature_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).strip().lower(),
    )


REC_COLUMNS = [
    column
    for column in PREDICTOR_COLUMNS
    if normalise_feature_name(
        column
    ).startswith(
        "rec"
    )
]


REC_COLUMNS = list(
    REC_COLUMNS
)


if len(
    REC_COLUMNS
) != 25:
    raise AssertionError(
        "Unexpected Beast2 REC feature count.\n"
        f"Expected: 25\n"
        f"Observed: {len(REC_COLUMNS)}\n\n"
        "Observed REC columns:\n"
        + "\n".join(
            map(
                str,
                REC_COLUMNS,
            )
        )
    )


duplicate_normalised_rec_names = (
    pd.Series(
        [
            normalise_feature_name(
                column
            )
            for column in REC_COLUMNS
        ]
    )
    .duplicated(
        keep=False
    )
)


if duplicate_normalised_rec_names.any():
    duplicated_names = (
        pd.Series(
            REC_COLUMNS
        )[
            duplicate_normalised_rec_names
        ]
        .tolist()
    )


    raise AssertionError(
        "REC columns are ambiguous after name "
        "normalisation:\n"
        + "\n".join(
            map(
                str,
                duplicated_names,
            )
        )
    )


normalised_rec_lookup = {
    normalise_feature_name(
        column
    ):
        column
    for column in REC_COLUMNS
}


# ---------------------------------------------------------
# 4. Resolve the six frozen verdict-independent features
# ---------------------------------------------------------

EXPECTED_INDEPENDENT_REC_NAMES = {
    "REC_Age":
        "recage",

    "REC_LastExeTime":
        "reclastexetime",

    "REC_RecentAvgExeTime":
        "recrecentavgexetime",

    "REC_RecentMaxExeTime":
        "recrecentmaxexetime",

    "REC_TotalAvgExeTime":
        "rectotalavgexetime",

    "REC_TotalMaxExeTime":
        "rectotalmaxexetime",
}


resolved_independent_records = []


for canonical_name, normalised_name in (
    EXPECTED_INDEPENDENT_REC_NAMES.items()
):
    matching_column = (
        normalised_rec_lookup.get(
            normalised_name
        )
    )


    resolved_independent_records.append({
        "CanonicalProtocolName":
            canonical_name,

        "ExpectedNormalisedName":
            normalised_name,

        "ObservedColumn":
            (
                matching_column
                if matching_column is not None
                else ""
            ),

        "Found":
            bool(
                matching_column is not None
            ),
    })


independent_resolution = pd.DataFrame(
    resolved_independent_records
)


missing_independent_rec_features = (
    independent_resolution[
        ~independent_resolution[
            "Found"
        ]
    ]
)


if len(
    missing_independent_rec_features
) != 0:
    display(
        independent_resolution
    )


    raise AssertionError(
        "One or more frozen verdict-independent REC "
        "features could not be resolved."
    )


REC_INDEPENDENT_COLUMNS = [
    str(
        independent_resolution.loc[
            independent_resolution[
                "CanonicalProtocolName"
            ] == canonical_name,
            "ObservedColumn",
        ].iloc[0]
    )
    for canonical_name in (
        EXPECTED_INDEPENDENT_REC_NAMES.keys()
    )
]


if len(
    set(
        REC_INDEPENDENT_COLUMNS
    )
) != 6:
    raise AssertionError(
        "The six verdict-independent REC features did "
        "not resolve uniquely."
    )


REC_DEPENDENT_COLUMNS = [
    column
    for column in REC_COLUMNS
    if column not in set(
        REC_INDEPENDENT_COLUMNS
    )
]


if len(
    REC_DEPENDENT_COLUMNS
) != 19:
    raise AssertionError(
        "Unexpected verdict-dependent REC count.\n"
        f"Expected: 19\n"
        f"Observed: {len(REC_DEPENDENT_COLUMNS)}"
    )


# Freeze explicit aliases for later cells.
VERDICT_INDEPENDENT_REC_COLUMNS = list(
    REC_INDEPENDENT_COLUMNS
)


VERDICT_DEPENDENT_REC_COLUMNS = list(
    REC_DEPENDENT_COLUMNS
)


# ---------------------------------------------------------
# 5. Validate every REC feature is numeric and finite
# ---------------------------------------------------------

rec_numeric_conversion_failures = 0
rec_missing_values = 0
rec_infinite_values = 0


for rec_column in REC_COLUMNS:
    original_values = (
        dataset_work[
            rec_column
        ]
    )


    numeric_values = pd.to_numeric(
        original_values,
        errors="coerce",
    )


    rec_numeric_conversion_failures += int(
        (
            numeric_values.isna()
            & original_values.notna()
        ).sum()
    )


    dataset_work[
        rec_column
    ] = numeric_values


    rec_missing_values += int(
        numeric_values.isna().sum()
    )


    finite_values = (
        numeric_values
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    rec_infinite_values += int(
        np.isinf(
            finite_values
        ).sum()
    )


if rec_numeric_conversion_failures != 0:
    raise AssertionError(
        "REC features contain non-numeric values.\n"
        f"Conversion failures: "
        f"{rec_numeric_conversion_failures}"
    )


if rec_missing_values != 0:
    raise AssertionError(
        "REC features contain missing values.\n"
        f"Missing values: {rec_missing_values}"
    )


if rec_infinite_values != 0:
    raise AssertionError(
        "REC features contain infinite values.\n"
        f"Infinite values: {rec_infinite_values}"
    )


# ---------------------------------------------------------
# 6. Create a detailed REC feature profile
# ---------------------------------------------------------

rec_profile_records = []


for feature_order, rec_column in enumerate(
    REC_COLUMNS,
    start=1,
):
    values = (
        dataset_work[
            rec_column
        ].to_numpy(
            dtype=float
        )
    )


    integer_like_values = np.isclose(
        values,
        np.round(
            values
        ),
        rtol=0.0,
        atol=1e-12,
    )


    training_values = (
        dataset_work.loc[
            dataset_work[
                "partition"
            ] == "TRAIN",
            rec_column,
        ]
        .to_numpy(
            dtype=float
        )
    )


    evaluation_values = (
        dataset_work.loc[
            dataset_work[
                "partition"
            ] == "EVALUATION",
            rec_column,
        ]
        .to_numpy(
            dtype=float
        )
    )


    protocol_class = (
        "VERDICT_INDEPENDENT"
        if rec_column
        in set(
            REC_INDEPENDENT_COLUMNS
        )
        else
        "VERDICT_DEPENDENT"
    )


    rec_profile_records.append({
        "FeatureOrder":
            int(
                feature_order
            ),

        "Feature":
            str(
                rec_column
            ),

        "NormalisedFeature":
            normalise_feature_name(
                rec_column
            ),

        "ProtocolClass":
            protocol_class,

        "Rows":
            int(
                len(
                    values
                )
            ),

        "MissingValues":
            int(
                np.isnan(
                    values
                ).sum()
            ),

        "InfiniteValues":
            int(
                np.isinf(
                    values
                ).sum()
            ),

        "UniqueValues":
            int(
                pd.Series(
                    values
                ).nunique(
                    dropna=True
                )
            ),

        "ZeroValues":
            int(
                np.isclose(
                    values,
                    0.0,
                    rtol=0.0,
                    atol=1e-12,
                ).sum()
            ),

        "IntegerLikeValues":
            int(
                integer_like_values.sum()
            ),

        "IntegerLikeFraction":
            float(
                integer_like_values.mean()
            ),

        "Minimum":
            float(
                np.min(
                    values
                )
            ),

        "Maximum":
            float(
                np.max(
                    values
                )
            ),

        "Mean":
            float(
                np.mean(
                    values
                )
            ),

        "Median":
            float(
                np.median(
                    values
                )
            ),

        "TrainingMinimum":
            float(
                np.min(
                    training_values
                )
            ),

        "TrainingMaximum":
            float(
                np.max(
                    training_values
                )
            ),

        "EvaluationMinimum":
            float(
                np.min(
                    evaluation_values
                )
            ),

        "EvaluationMaximum":
            float(
                np.max(
                    evaluation_values
                )
            ),
    })


rec_feature_profile = pd.DataFrame(
    rec_profile_records
)


# ---------------------------------------------------------
# 7. Profile raw and model-ready verdict subtypes
# ---------------------------------------------------------

def create_verdict_profile(
    dataframe,
    source_name,
):
    profile = (
        dataframe
        .groupby(
            [
                "partition",
                "Verdict",
            ],
            dropna=False,
        )
        .agg(
            ExecutionRows=(
                "Test",
                "size",
            ),

            Builds=(
                "Build",
                "nunique",
            ),

            Tests=(
                "Test",
                "nunique",
            ),
        )
        .reset_index()
    )


    profile.insert(
        0,
        "Source",
        source_name,
    )


    profile[
        "Verdict"
    ] = pd.to_numeric(
        profile[
            "Verdict"
        ],
        errors="raise",
    ).astype(np.int64)


    profile[
        "IsFailure"
    ] = (
        profile[
            "Verdict"
        ] != 0
    )


    return profile


raw_verdict_profile = (
    create_verdict_profile(
        exe_work,
        "RAW_EXECUTION_HISTORY",
    )
)


model_verdict_profile = (
    create_verdict_profile(
        dataset_work,
        "MODEL_READY_DATASET",
    )
)


verdict_subtype_profile = pd.concat(
    [
        raw_verdict_profile,
        model_verdict_profile,
    ],
    ignore_index=True,
)


raw_training_failure_subtypes = (
    raw_training_work[
        raw_training_work[
            "Verdict"
        ].astype(np.int64)
        != 0
    ][
        "Verdict"
    ]
    .value_counts()
    .sort_index()
)


RAW_TRAINING_FAILURE_SUBTYPE_COUNTS = {
    int(
        verdict
    ):
        int(
            count
        )
    for verdict, count in (
        raw_training_failure_subtypes.items()
    )
}


RAW_TRAINING_FAILURE_TOTAL = int(
    sum(
        RAW_TRAINING_FAILURE_SUBTYPE_COUNTS.values()
    )
)


if RAW_TRAINING_FAILURE_TOTAL <= 0:
    raise AssertionError(
        "Raw training history contains no failure "
        "subtypes."
    )


RAW_TRAINING_FAILURE_SUBTYPE_PROBABILITIES = {
    int(
        verdict
    ):
        float(
            count
            / RAW_TRAINING_FAILURE_TOTAL
        )
    for verdict, count in (
        RAW_TRAINING_FAILURE_SUBTYPE_COUNTS.items()
    )
}


# ---------------------------------------------------------
# 8. Build raw/model test-history profiles
# ---------------------------------------------------------

raw_test_history = (
    exe_work
    .groupby(
        "Test",
        sort=True,
    )
    .agg(
        RawExecutions=(
            "Build",
            "size",
        ),

        RawBuilds=(
            "Build",
            "nunique",
        ),

        RawFailureExecutions=(
            "Verdict",
            lambda values:
                int(
                    (
                        pd.to_numeric(
                            values,
                            errors="raise",
                        ).to_numpy(
                            dtype=np.int64
                        )
                        != 0
                    ).sum()
                ),
        ),

        RawFirstBuildOrder=(
            "build_order",
            "min",
        ),

        RawLastBuildOrder=(
            "build_order",
            "max",
        ),
    )
    .reset_index()
)


model_test_history = (
    dataset_work
    .groupby(
        "Test",
        sort=True,
    )
    .agg(
        ModelReadyRows=(
            "Build",
            "size",
        ),

        ModelReadyBuilds=(
            "Build",
            "nunique",
        ),

        ModelReadyFailureExecutions=(
            "Verdict",
            lambda values:
                int(
                    (
                        pd.to_numeric(
                            values,
                            errors="raise",
                        ).to_numpy(
                            dtype=np.int64
                        )
                        != 0
                    ).sum()
                ),
        ),

        ModelFirstBuildOrder=(
            "build_order",
            "min",
        ),

        ModelLastBuildOrder=(
            "build_order",
            "max",
        ),
    )
    .reset_index()
)


test_history_profile = (
    raw_test_history
    .merge(
        model_test_history,
        on="Test",
        how="outer",
        validate="one_to_one",
    )
)


integer_test_profile_columns = [
    "RawExecutions",
    "RawBuilds",
    "RawFailureExecutions",
    "ModelReadyRows",
    "ModelReadyBuilds",
    "ModelReadyFailureExecutions",
]


for column in integer_test_profile_columns:
    test_history_profile[
        column
    ] = (
        test_history_profile[
            column
        ]
        .fillna(0)
        .astype(np.int64)
    )


test_history_profile[
    "RawRowsNotRetained"
] = (
    test_history_profile[
        "RawExecutions"
    ]
    - test_history_profile[
        "ModelReadyRows"
    ]
)


test_history_profile[
    "HasRawFailure"
] = (
    test_history_profile[
        "RawFailureExecutions"
    ] > 0
)


test_history_profile[
    "HasModelReadyFailure"
] = (
    test_history_profile[
        "ModelReadyFailureExecutions"
    ] > 0
)


test_history_profile = (
    test_history_profile
    .sort_values(
        "Test",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


RAW_UNIQUE_TESTS = int(
    test_history_profile[
        "RawExecutions"
    ].gt(0).sum()
)


MODEL_READY_UNIQUE_TESTS = int(
    test_history_profile[
        "ModelReadyRows"
    ].gt(0).sum()
)


RAW_TESTS_WITH_FAILURE = int(
    test_history_profile[
        "HasRawFailure"
    ].sum()
)


MODEL_TESTS_WITH_FAILURE = int(
    test_history_profile[
        "HasModelReadyFailure"
    ].sum()
)


TESTS_WITH_UNRETAINED_RAW_ROWS = int(
    (
        test_history_profile[
            "RawRowsNotRetained"
        ] > 0
    ).sum()
)


# ---------------------------------------------------------
# 9. Validate chronological ordering of execution histories
# ---------------------------------------------------------

raw_execution_order_violations = 0
model_execution_order_violations = 0


for _, test_rows in (
    exe_work
    .groupby(
        "Test",
        sort=False,
    )
):
    build_orders = (
        test_rows[
            "build_order"
        ].to_numpy(
            dtype=np.int64
        )
    )


    if (
        len(
            build_orders
        ) > 1
        and (
            np.diff(
                build_orders
            ) < 0
        ).any()
    ):
        raw_execution_order_violations += 1


for _, test_rows in (
    dataset_work
    .groupby(
        "Test",
        sort=False,
    )
):
    build_orders = (
        test_rows[
            "build_order"
        ].to_numpy(
            dtype=np.int64
        )
    )


    if (
        len(
            build_orders
        ) > 1
        and (
            np.diff(
                build_orders
            ) < 0
        ).any()
    ):
        model_execution_order_violations += 1


# Source row order is not required to be chronological.
# Create canonical sorted views for reconstruction.
raw_history_chronological = (
    exe_work
    .assign(
        _SourceRow=np.arange(
            len(
                exe_work
            ),
            dtype=np.int64,
        )
    )
    .sort_values(
        [
            "build_order",
            "Test",
            "_SourceRow",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


model_history_chronological = (
    dataset_work
    .assign(
        _SourceRow=np.arange(
            len(
                dataset_work
            ),
            dtype=np.int64,
        )
    )
    .sort_values(
        [
            "build_order",
            "Test",
            "_SourceRow",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 10. Save representative clean REC trajectories
# ---------------------------------------------------------

trajectory_candidate_tests = (
    test_history_profile[
        (
            test_history_profile[
                "ModelReadyRows"
            ] >= 5
        )
        & (
            test_history_profile[
                "ModelReadyFailureExecutions"
            ] > 0
        )
    ]
    .sort_values(
        [
            "ModelReadyFailureExecutions",
            "ModelReadyRows",
            "Test",
        ],
        ascending=[
            False,
            False,
            True,
        ],
        kind="mergesort",
    )
    .head(12)[
        "Test"
    ]
    .astype(str)
    .tolist()
)


trajectory_records = []


for test_id in trajectory_candidate_tests:
    test_rows = (
        model_history_chronological[
            model_history_chronological[
                "Test"
            ].astype(str)
            == str(
                test_id
            )
        ]
        .copy()
        .reset_index(drop=True)
    )


    failure_positions = np.flatnonzero(
        test_rows[
            "Verdict"
        ].to_numpy(
            dtype=np.int64
        )
        != 0
    )


    if len(
        failure_positions
    ) == 0:
        continue


    first_failure_position = int(
        failure_positions[0]
    )


    start_position = max(
        0,
        first_failure_position - 3,
    )


    end_position = min(
        len(
            test_rows
        ),
        first_failure_position + 4,
    )


    sample_rows = (
        test_rows.iloc[
            start_position:
            end_position
        ]
        .copy()
    )


    sample_rows[
        "TrajectoryTest"
    ] = str(
        test_id
    )


    sample_rows[
        "PositionWithinTestHistory"
    ] = np.arange(
        start_position,
        end_position,
        dtype=np.int64,
    )


    sample_rows[
        "FirstFailurePosition"
    ] = first_failure_position


    trajectory_records.append(
        sample_rows[
            [
                "TrajectoryTest",
                "PositionWithinTestHistory",
                "FirstFailurePosition",
                "Build",
                "build_order",
                "partition",
                "Test",
                "Verdict",
                "Duration",
            ]
            + REC_COLUMNS
        ]
    )


if trajectory_records:
    rec_sample_trajectories = pd.concat(
        trajectory_records,
        ignore_index=True,
    )

else:
    rec_sample_trajectories = pd.DataFrame(
        columns=[
            "TrajectoryTest",
            "PositionWithinTestHistory",
            "FirstFailurePosition",
            "Build",
            "build_order",
            "partition",
            "Test",
            "Verdict",
            "Duration",
        ]
        + REC_COLUMNS
    )


# ---------------------------------------------------------
# 11. Create REC name-semantics audit
# ---------------------------------------------------------

def name_contains_any(
    feature_name,
    terms,
):
    normalised_name = normalise_feature_name(
        feature_name
    )


    return any(
        normalise_feature_name(
            term
        )
        in normalised_name
        for term in terms
    )


rec_semantics_records = []


for rec_column in REC_COLUMNS:
    is_independent = bool(
        rec_column
        in set(
            REC_INDEPENDENT_COLUMNS
        )
    )


    rec_semantics_records.append({
        "Feature":
            rec_column,

        "ProtocolClass":
            (
                "VERDICT_INDEPENDENT"
                if is_independent
                else
                "VERDICT_DEPENDENT"
            ),

        "ContainsFailureTerm":
            name_contains_any(
                rec_column,
                [
                    "fail",
                    "failure",
                ],
            ),

        "ContainsAssertTerm":
            name_contains_any(
                rec_column,
                [
                    "assert",
                ],
            ),

        "ContainsTransitionTerm":
            name_contains_any(
                rec_column,
                [
                    "transition",
                ],
            ),

        "ContainsExecutionTimeTerm":
            name_contains_any(
                rec_column,
                [
                    "exetime",
                    "executiontime",
                    "duration",
                ],
            ),

        "ContainsAgeTerm":
            name_contains_any(
                rec_column,
                [
                    "age",
                ],
            ),
    })


rec_semantics_audit = pd.DataFrame(
    rec_semantics_records
)


# ---------------------------------------------------------
# 12. Permanent Step 5 report paths
# ---------------------------------------------------------

REC_FEATURE_PROFILE_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_rec_feature_profile.csv"
)


REC_CLASSIFICATION_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_rec_feature_classification.csv"
)


REC_INDEPENDENT_RESOLUTION_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_rec_independent_feature_resolution.csv"
)


VERDICT_SUBTYPE_PROFILE_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_verdict_subtype_profile.csv"
)


TEST_HISTORY_PROFILE_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_test_history_profile.csv"
)


REC_SAMPLE_TRAJECTORIES_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_rec_sample_trajectories.csv"
)


REC_SEMANTICS_AUDIT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_rec_name_semantics_audit.csv"
)


REC_SCHEMA_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_rec_schema_inspection_report.json"
)


rec_feature_profile.to_csv(
    REC_FEATURE_PROFILE_PATH,
    index=False,
)


pd.DataFrame({
    "FeatureOrder":
        np.arange(
            1,
            len(
                REC_COLUMNS
            ) + 1,
            dtype=np.int64,
        ),

    "Feature":
        REC_COLUMNS,

    "ProtocolClass":
        [
            (
                "VERDICT_INDEPENDENT"
                if column
                in set(
                    REC_INDEPENDENT_COLUMNS
                )
                else
                "VERDICT_DEPENDENT"
            )
            for column in REC_COLUMNS
        ],
}).to_csv(
    REC_CLASSIFICATION_PATH,
    index=False,
)


independent_resolution.to_csv(
    REC_INDEPENDENT_RESOLUTION_PATH,
    index=False,
)


verdict_subtype_profile.to_csv(
    VERDICT_SUBTYPE_PROFILE_PATH,
    index=False,
)


test_history_profile.to_csv(
    TEST_HISTORY_PROFILE_PATH,
    index=False,
)


rec_sample_trajectories.to_csv(
    REC_SAMPLE_TRAJECTORIES_PATH,
    index=False,
)


rec_semantics_audit.to_csv(
    REC_SEMANTICS_AUDIT_PATH,
    index=False,
)


# ---------------------------------------------------------
# 13. Save complete REC schema-inspection report
# ---------------------------------------------------------

schema_inspection_completed_at = (
    pd.Timestamp.utcnow().isoformat()
)


rec_schema_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "PASS",

    "RECStructure": {
        "TotalRECFeatures":
            int(
                len(
                    REC_COLUMNS
                )
            ),

        "VerdictDependentFeatures":
            int(
                len(
                    REC_DEPENDENT_COLUMNS
                )
            ),

        "VerdictIndependentFeatures":
            int(
                len(
                    REC_INDEPENDENT_COLUMNS
                )
            ),

        "VerdictDependentColumns":
            list(
                REC_DEPENDENT_COLUMNS
            ),

        "VerdictIndependentColumns":
            list(
                REC_INDEPENDENT_COLUMNS
            ),

        "NumericConversionFailures":
            int(
                rec_numeric_conversion_failures
            ),

        "MissingValues":
            int(
                rec_missing_values
            ),

        "InfiniteValues":
            int(
                rec_infinite_values
            ),
    },

    "RawHistory": {
        "Rows":
            int(
                len(
                    exe_work
                )
            ),

        "TrainingRows":
            int(
                len(
                    raw_training_work
                )
            ),

        "EvaluationRows":
            int(
                len(
                    raw_evaluation_work
                )
            ),

        "UniqueTests":
            RAW_UNIQUE_TESTS,

        "TestsWithAtLeastOneFailure":
            RAW_TESTS_WITH_FAILURE,

        "SourceOrderTestViolations":
            int(
                raw_execution_order_violations
            ),
    },

    "ModelReadyHistory": {
        "Rows":
            int(
                len(
                    dataset_work
                )
            ),

        "TrainingRows":
            int(
                len(
                    model_training_work
                )
            ),

        "EvaluationRows":
            int(
                len(
                    model_evaluation_work
                )
            ),

        "UniqueTests":
            MODEL_READY_UNIQUE_TESTS,

        "TestsWithAtLeastOneFailure":
            MODEL_TESTS_WITH_FAILURE,

        "TestsWithUnretainedRawRows":
            TESTS_WITH_UNRETAINED_RAW_ROWS,

        "SourceOrderTestViolations":
            int(
                model_execution_order_violations
            ),
    },

    "RawTrainingFailureSubtypeDistribution": {
        "FailureExecutions":
            RAW_TRAINING_FAILURE_TOTAL,

        "Counts":
            {
                str(
                    verdict
                ):
                    int(
                        count
                    )
                for verdict, count in (
                    RAW_TRAINING_FAILURE_SUBTYPE_COUNTS.items()
                )
            },

        "Probabilities":
            {
                str(
                    verdict
                ):
                    float(
                        probability
                    )
                for verdict, probability in (
                    RAW_TRAINING_FAILURE_SUBTYPE_PROBABILITIES.items()
                )
            },
    },

    "EntityHistoryAvailability": {
        "ChangedEntityDictionaryBuilds":
            int(
                len(
                    changed_entity_dictionary
                )
            ),

        "BuildsWithAtLeastOneChangedEntity":
            int(
                sum(
                    len(
                        entity_ids
                    ) > 0
                    for entity_ids in (
                        changed_entity_dictionary.values()
                    )
                )
            ),
    },

    "RepresentativeTrajectories": {
        "SelectedTests":
            trajectory_candidate_tests,

        "SavedRows":
            int(
                len(
                    rec_sample_trajectories
                )
            ),
    },

    "Artefacts": {
        "RECFeatureProfile":
            str(
                REC_FEATURE_PROFILE_PATH
            ),

        "RECClassification":
            str(
                REC_CLASSIFICATION_PATH
            ),

        "IndependentFeatureResolution":
            str(
                REC_INDEPENDENT_RESOLUTION_PATH
            ),

        "VerdictSubtypeProfile":
            str(
                VERDICT_SUBTYPE_PROFILE_PATH
            ),

        "TestHistoryProfile":
            str(
                TEST_HISTORY_PROFILE_PATH
            ),

        "SampleTrajectories":
            str(
                REC_SAMPLE_TRAJECTORIES_PATH
            ),

        "NameSemanticsAudit":
            str(
                REC_SEMANTICS_AUDIT_PATH
            ),
    },

    "CompletedAtUTC":
        schema_inspection_completed_at,
}


with open(
    REC_SCHEMA_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        rec_schema_report,
        report_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 14. Update Project 7 checkpoint atomically
# ---------------------------------------------------------

with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    project_7_checkpoint = json.load(
        checkpoint_file
    )


project_7_checkpoint.update({
    "Status":
        "REC_SCHEMA_INSPECTION_PASSED",

    "RECFeatureCount":
        int(
            len(
                REC_COLUMNS
            )
        ),

    "VerdictDependentRECFeatureCount":
        int(
            len(
                REC_DEPENDENT_COLUMNS
            )
        ),

    "VerdictIndependentRECFeatureCount":
        int(
            len(
                REC_INDEPENDENT_COLUMNS
            )
        ),

    "VerdictDependentRECColumns":
        list(
            REC_DEPENDENT_COLUMNS
        ),

    "VerdictIndependentRECColumns":
        list(
            REC_INDEPENDENT_COLUMNS
        ),

    "RECFeatureNumericConversionFailures":
        int(
            rec_numeric_conversion_failures
        ),

    "RECFeatureMissingValues":
        int(
            rec_missing_values
        ),

    "RECFeatureInfiniteValues":
        int(
            rec_infinite_values
        ),

    "RawUniqueTests":
        RAW_UNIQUE_TESTS,

    "ModelReadyUniqueTests":
        MODEL_READY_UNIQUE_TESTS,

    "RawTestsWithFailure":
        RAW_TESTS_WITH_FAILURE,

    "ModelReadyTestsWithFailure":
        MODEL_TESTS_WITH_FAILURE,

    "RawTrainingFailureSubtypeCounts":
        {
            str(
                verdict
            ):
                int(
                    count
                )
            for verdict, count in (
                RAW_TRAINING_FAILURE_SUBTYPE_COUNTS.items()
            )
        },

    "RawTrainingFailureSubtypeProbabilities":
        {
            str(
                verdict
            ):
                float(
                    probability
                )
            for verdict, probability in (
                RAW_TRAINING_FAILURE_SUBTYPE_PROBABILITIES.items()
            )
        },

    "RECSchemaInspectionReport":
        str(
            REC_SCHEMA_REPORT_PATH
        ),

    "RECFeatureProfile":
        str(
            REC_FEATURE_PROFILE_PATH
        ),

    "RECFeatureClassification":
        str(
            REC_CLASSIFICATION_PATH
        ),

    "VerdictSubtypeProfile":
        str(
            VERDICT_SUBTYPE_PROFILE_PATH
        ),

    "TestHistoryProfile":
        str(
            TEST_HISTORY_PROFILE_PATH
        ),

    "RECSampleTrajectories":
        str(
            REC_SAMPLE_TRAJECTORIES_PATH
        ),

    "UpdatedAtUTC":
        schema_inspection_completed_at,
})


temporary_checkpoint_path = (
    PROJECT_7_SELECTION_CHECKPOINT
    .with_suffix(
        ".json.tmp"
    )
)


with open(
    temporary_checkpoint_path,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        project_7_checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


os.replace(
    temporary_checkpoint_path,
    PROJECT_7_SELECTION_CHECKPOINT,
)


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint_verification = json.load(
        checkpoint_file
    )


if checkpoint_verification.get(
    "Status"
) != "REC_SCHEMA_INSPECTION_PASSED":
    raise AssertionError(
        "The Project 7 REC schema checkpoint was not "
        "written correctly."
    )


# ---------------------------------------------------------
# 15. Compact final output
# ---------------------------------------------------------

clear_output(
    wait=True
)


print(
    "=== PROJECT 7 STEP 5 RESULT ==="
)


print(
    "\nREC structure:"
)

print(
    "Total REC features:",
    len(
        REC_COLUMNS
    )
)

print(
    "Verdict-dependent REC features:",
    len(
        REC_DEPENDENT_COLUMNS
    )
)

print(
    "Verdict-independent REC features:",
    len(
        REC_INDEPENDENT_COLUMNS
    )
)


print(
    "\nVerdict-independent REC features:"
)

display(
    independent_resolution[
        [
            "CanonicalProtocolName",
            "ObservedColumn",
            "Found",
        ]
    ]
)


print(
    "\nVerdict-dependent REC features:"
)

display(
    pd.DataFrame({
        "FeatureOrder":
            np.arange(
                1,
                len(
                    REC_DEPENDENT_COLUMNS
                ) + 1,
                dtype=np.int64,
            ),

        "Feature":
            REC_DEPENDENT_COLUMNS,
    })
)


print(
    "\nComplete REC feature profile:"
)

display(
    rec_feature_profile[
        [
            "FeatureOrder",
            "Feature",
            "ProtocolClass",
            "UniqueValues",
            "ZeroValues",
            "Minimum",
            "Maximum",
            "Mean",
            "IntegerLikeFraction",
        ]
    ]
)


print(
    "\nREC data quality:"
)

print(
    "Numeric conversion failures:",
    rec_numeric_conversion_failures
)

print(
    "Missing values:",
    rec_missing_values
)

print(
    "Infinite values:",
    rec_infinite_values
)


print(
    "\nRaw training failure subtype distribution:"
)

failure_subtype_display = pd.DataFrame([
    {
        "VerdictSubtype":
            int(
                verdict
            ),

        "Count":
            int(
                count
            ),

        "Probability":
            float(
                RAW_TRAINING_FAILURE_SUBTYPE_PROBABILITIES[
                    verdict
                ]
            ),
    }
    for verdict, count in (
        RAW_TRAINING_FAILURE_SUBTYPE_COUNTS.items()
    )
])


display(
    failure_subtype_display
)


print(
    "Raw training failure executions:",
    RAW_TRAINING_FAILURE_TOTAL
)


print(
    "\nTest-history coverage:"
)

print(
    "Raw unique tests:",
    RAW_UNIQUE_TESTS
)

print(
    "Model-ready unique tests:",
    MODEL_READY_UNIQUE_TESTS
)

print(
    "Raw tests with failures:",
    RAW_TESTS_WITH_FAILURE
)

print(
    "Model-ready tests with failures:",
    MODEL_TESTS_WITH_FAILURE
)

print(
    "Tests with unretained raw rows:",
    TESTS_WITH_UNRETAINED_RAW_ROWS
)


print(
    "\nSource-order diagnostics:"
)

print(
    "Raw test histories not already chronological:",
    raw_execution_order_violations
)

print(
    "Model test histories not already chronological:",
    model_execution_order_violations
)

print(
    "Canonical chronological views created:",
    True
)


print(
    "\nRepresentative REC trajectories:"
)

print(
    "Selected tests:",
    trajectory_candidate_tests
)

print(
    "Saved trajectory rows:",
    len(
        rec_sample_trajectories
    )
)


print(
    "\nREC schema report:"
)

print(
    REC_SCHEMA_REPORT_PATH
)


print(
    "\nValidation status:",
    "PASS"
)


print(
    "\nSUCCESS: All 25 Beast2 REC features were "
    "identified."
)

print(
    "SUCCESS: The nineteen verdict-dependent and six "
    "verdict-independent REC features were separated "
    "using the frozen protocol."
)

print(
    "SUCCESS: Every REC value was numeric, present and "
    "finite."
)

print(
    "SUCCESS: Raw training failure subtype counts and "
    "probabilities were frozen."
)

print(
    "SUCCESS: Canonical chronological raw and "
    "model-ready history views were created."
)

print(
    "SUCCESS: No REC value or verdict was modified."
)

print(
    "SUCCESS: Project 7 is ready for clean REC "
    "reconstruction and validation."
)

=== PROJECT 7 STEP 5: REC FEATURE SCHEMA AND HISTORY INSPECTION ===


AssertionError: Unexpected Beast2 REC feature count.
Expected: 25
Observed: 19

Observed REC columns:
REC_Age
REC_LastFailureAge
REC_LastTransitionAge
REC_RecentAvgExeTime
REC_RecentMaxExeTime
REC_RecentFailRate
REC_RecentAssertRate
REC_RecentExcRate
REC_RecentTransitionRate
REC_TotalAvgExeTime
REC_TotalMaxExeTime
REC_TotalFailRate
REC_TotalAssertRate
REC_TotalExcRate
REC_TotalTransitionRate
REC_LastVerdict
REC_LastExeTime
REC_MaxTestFileFailRate
REC_MaxTestFileTransitionRate

In [ ]:
# =========================================================
# PROJECT 7 — STEP 5
# CORRECTED BEAST2 REC SCHEMA AND HISTORY INSPECTION
#
# Beast2 contains:
#   - 19 total REC features
#   - 6 verdict-independent REC features
#   - 13 verdict-dependent REC features
#
# This cell:
#   - validates the exact Beast2 REC schema
#   - freezes the 6 independent and 13 dependent features
#   - validates every REC value
#   - profiles failure verdict subtypes
#   - creates chronological raw/model history views
#   - saves permanent REC schema reports
#
# It does NOT:
#   - reconstruct REC values
#   - modify verdicts
#   - inject noise
#   - create synthetic executions
#   - train any model
# =========================================================

from pathlib import Path
import json
import os
import re

import numpy as np
import pandas as pd
from IPython.display import clear_output, display


print(
    "=== PROJECT 7 STEP 5: "
    "CORRECTED BEAST2 REC SCHEMA INSPECTION ==="
)


# ---------------------------------------------------------
# 1. Confirm successful Step 4 runtime state
# ---------------------------------------------------------

required_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",
    "PROJECT_SHORT_NAME",

    "PROJECT_PREFLIGHT_DIRECTORY",
    "PROJECT_7_SELECTION_CHECKPOINT",

    "chronological_builds",
    "exe",
    "dataset",

    "raw_training_history",
    "raw_evaluation_history",
    "all_model_training_data",
    "all_model_evaluation_data",
    "clean_training_data",
    "clean_evaluation_data",

    "PREDICTOR_COLUMNS",

    "changed_entity_dictionary",
    "changed_entity_set_dictionary",
]


missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required Project 7 objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nRerun only the successful Project 7 "
        "Step 2B, Step 3 and Step 4 cells."
    )


if PROJECT_NUMBER != 7:
    raise AssertionError(
        f"Expected Project 7, observed {PROJECT_NUMBER}."
    )


if PROJECT_NAME != "CompEvol@beast2":
    raise AssertionError(
        f"Unexpected project: {PROJECT_NAME}"
    )


if PROJECT_SLUG != "CompEvol__beast2":
    raise AssertionError(
        f"Unexpected project slug: {PROJECT_SLUG}"
    )


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    step_4_checkpoint = json.load(
        checkpoint_file
    )


allowed_previous_statuses = {
    "CANONICAL_ENTITY_MAPPING_PASSED",
    "REC_SCHEMA_INSPECTION_PASSED",
}


if step_4_checkpoint.get(
    "Status"
) not in allowed_previous_statuses:
    raise AssertionError(
        "Project 7 canonical entity mapping has not "
        "passed.\n"
        f"Observed status: "
        f"{step_4_checkpoint.get('Status')}"
    )


if int(
    step_4_checkpoint.get(
        "FullyMappedBuilds",
        -1,
    )
) != 415:
    raise AssertionError(
        "The checkpoint does not confirm 415 fully "
        "mapped builds."
    )


if int(
    step_4_checkpoint.get(
        "UnmatchedUniqueCommitTokens",
        -1,
    )
) != 0:
    raise AssertionError(
        "The checkpoint contains unmatched commits."
    )


# ---------------------------------------------------------
# 2. Create defragmented working copies
# ---------------------------------------------------------

exe_work = (
    exe.copy()
    .reset_index(drop=True)
)


dataset_work = (
    dataset.copy()
    .reset_index(drop=True)
)


raw_training_work = (
    raw_training_history.copy()
    .reset_index(drop=True)
)


raw_evaluation_work = (
    raw_evaluation_history.copy()
    .reset_index(drop=True)
)


model_training_work = (
    all_model_training_data.copy()
    .reset_index(drop=True)
)


model_evaluation_work = (
    all_model_evaluation_data.copy()
    .reset_index(drop=True)
)


required_core_columns = {
    "Build",
    "Test",
    "Verdict",
    "Duration",
    "build_order",
    "partition",
}


for frame_name, frame in [
    ("exe", exe_work),
    ("dataset", dataset_work),
]:
    missing_columns = (
        required_core_columns
        - set(frame.columns)
    )


    if missing_columns:
        raise RuntimeError(
            f"{frame_name} is missing required columns:\n"
            + "\n".join(
                sorted(missing_columns)
            )
        )


# ---------------------------------------------------------
# 3. Resolve the exact Beast2 REC schema
# ---------------------------------------------------------

def normalise_feature_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).strip().lower(),
    )


REC_COLUMNS = [
    column
    for column in PREDICTOR_COLUMNS
    if normalise_feature_name(
        column
    ).startswith("rec")
]


REC_COLUMNS = list(
    REC_COLUMNS
)


EXPECTED_BEAST2_REC_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


EXPECTED_INDEPENDENT_REC_COLUMNS = [
    "REC_Age",
    "REC_LastExeTime",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
]


EXPECTED_DEPENDENT_REC_COLUMNS = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


if len(REC_COLUMNS) != 19:
    raise AssertionError(
        "Unexpected Beast2 REC feature count.\n"
        f"Expected: 19\n"
        f"Observed: {len(REC_COLUMNS)}\n\n"
        "Observed REC columns:\n"
        + "\n".join(
            map(str, REC_COLUMNS)
        )
    )


normalised_rec_lookup = {
    normalise_feature_name(column):
        column
    for column in REC_COLUMNS
}


if len(normalised_rec_lookup) != len(
    REC_COLUMNS
):
    raise AssertionError(
        "Two Beast2 REC columns become ambiguous after "
        "name normalisation."
    )


expected_normalised_rec_names = {
    normalise_feature_name(column)
    for column in EXPECTED_BEAST2_REC_COLUMNS
}


observed_normalised_rec_names = {
    normalise_feature_name(column)
    for column in REC_COLUMNS
}


missing_expected_rec_features = sorted(
    expected_normalised_rec_names
    - observed_normalised_rec_names
)


unexpected_rec_features = sorted(
    observed_normalised_rec_names
    - expected_normalised_rec_names
)


if missing_expected_rec_features:
    raise AssertionError(
        "Expected Beast2 REC features are missing:\n"
        + "\n".join(
            missing_expected_rec_features
        )
    )


if unexpected_rec_features:
    raise AssertionError(
        "Unexpected Beast2 REC features were found:\n"
        + "\n".join(
            unexpected_rec_features
        )
    )


REC_INDEPENDENT_COLUMNS = [
    normalised_rec_lookup[
        normalise_feature_name(
            expected_column
        )
    ]
    for expected_column in (
        EXPECTED_INDEPENDENT_REC_COLUMNS
    )
]


REC_DEPENDENT_COLUMNS = [
    column
    for column in REC_COLUMNS
    if column not in set(
        REC_INDEPENDENT_COLUMNS
    )
]


if len(REC_INDEPENDENT_COLUMNS) != 6:
    raise AssertionError(
        "Expected exactly six verdict-independent "
        "Beast2 REC features."
    )


if len(REC_DEPENDENT_COLUMNS) != 13:
    raise AssertionError(
        "Expected exactly thirteen verdict-dependent "
        "Beast2 REC features.\n"
        f"Observed: {len(REC_DEPENDENT_COLUMNS)}"
    )


expected_dependent_normalised = {
    normalise_feature_name(column)
    for column in EXPECTED_DEPENDENT_REC_COLUMNS
}


observed_dependent_normalised = {
    normalise_feature_name(column)
    for column in REC_DEPENDENT_COLUMNS
}


if observed_dependent_normalised != (
    expected_dependent_normalised
):
    raise AssertionError(
        "The verdict-dependent Beast2 REC feature set "
        "does not match the expected schema."
    )


# Permanent aliases used by later reconstruction cells.
VERDICT_INDEPENDENT_REC_COLUMNS = list(
    REC_INDEPENDENT_COLUMNS
)


VERDICT_DEPENDENT_REC_COLUMNS = list(
    REC_DEPENDENT_COLUMNS
)


# ---------------------------------------------------------
# 4. Build the permanent REC classification table
# ---------------------------------------------------------

rec_classification_records = []


for feature_order, rec_column in enumerate(
    REC_COLUMNS,
    start=1,
):
    if rec_column in set(
        REC_INDEPENDENT_COLUMNS
    ):
        protocol_class = (
            "VERDICT_INDEPENDENT"
        )

        noise_policy = (
            "PRESERVE_ORIGINAL_TCP_CI_VALUE"
        )

    else:
        protocol_class = (
            "VERDICT_DEPENDENT"
        )

        noise_policy = (
            "RECOMPUTE_FROM_CORRUPTED_RAW_HISTORY"
        )


    rec_classification_records.append({
        "FeatureOrder":
            int(feature_order),

        "Feature":
            str(rec_column),

        "NormalisedFeature":
            normalise_feature_name(
                rec_column
            ),

        "ProtocolClass":
            protocol_class,

        "NoisePolicy":
            noise_policy,
    })


rec_feature_classification = pd.DataFrame(
    rec_classification_records
)


if int(
    (
        rec_feature_classification[
            "ProtocolClass"
        ] == "VERDICT_INDEPENDENT"
    ).sum()
) != 6:
    raise AssertionError(
        "REC classification did not retain six "
        "independent features."
    )


if int(
    (
        rec_feature_classification[
            "ProtocolClass"
        ] == "VERDICT_DEPENDENT"
    ).sum()
) != 13:
    raise AssertionError(
        "REC classification did not retain thirteen "
        "dependent features."
    )


# ---------------------------------------------------------
# 5. Validate every REC value
# ---------------------------------------------------------

rec_numeric_conversion_failures = 0
rec_missing_values = 0
rec_infinite_values = 0


for rec_column in REC_COLUMNS:
    original_values = (
        dataset_work[
            rec_column
        ]
    )


    numeric_values = pd.to_numeric(
        original_values,
        errors="coerce",
    )


    rec_numeric_conversion_failures += int(
        (
            numeric_values.isna()
            & original_values.notna()
        ).sum()
    )


    dataset_work[
        rec_column
    ] = numeric_values


    rec_missing_values += int(
        numeric_values.isna().sum()
    )


    finite_values = (
        numeric_values
        .dropna()
        .to_numpy(dtype=float)
    )


    rec_infinite_values += int(
        np.isinf(
            finite_values
        ).sum()
    )


if rec_numeric_conversion_failures != 0:
    raise AssertionError(
        "REC features contain non-numeric values.\n"
        f"Conversion failures: "
        f"{rec_numeric_conversion_failures}"
    )


if rec_missing_values != 0:
    raise AssertionError(
        "REC features contain missing values.\n"
        f"Missing values: {rec_missing_values}"
    )


if rec_infinite_values != 0:
    raise AssertionError(
        "REC features contain infinite values.\n"
        f"Infinite values: {rec_infinite_values}"
    )


# ---------------------------------------------------------
# 6. Create detailed REC feature profile
# ---------------------------------------------------------

rec_profile_records = []


for feature_order, rec_column in enumerate(
    REC_COLUMNS,
    start=1,
):
    all_values = (
        dataset_work[
            rec_column
        ].to_numpy(dtype=float)
    )


    training_values = (
        dataset_work.loc[
            dataset_work[
                "partition"
            ] == "TRAIN",
            rec_column,
        ]
        .to_numpy(dtype=float)
    )


    evaluation_values = (
        dataset_work.loc[
            dataset_work[
                "partition"
            ] == "EVALUATION",
            rec_column,
        ]
        .to_numpy(dtype=float)
    )


    integer_like_mask = np.isclose(
        all_values,
        np.round(all_values),
        rtol=0.0,
        atol=1e-12,
    )


    protocol_class = (
        "VERDICT_INDEPENDENT"
        if rec_column in set(
            REC_INDEPENDENT_COLUMNS
        )
        else
        "VERDICT_DEPENDENT"
    )


    rec_profile_records.append({
        "FeatureOrder":
            int(feature_order),

        "Feature":
            str(rec_column),

        "ProtocolClass":
            protocol_class,

        "Rows":
            int(len(all_values)),

        "MissingValues":
            int(
                np.isnan(
                    all_values
                ).sum()
            ),

        "InfiniteValues":
            int(
                np.isinf(
                    all_values
                ).sum()
            ),

        "UniqueValues":
            int(
                pd.Series(
                    all_values
                ).nunique(
                    dropna=True
                )
            ),

        "ZeroValues":
            int(
                np.isclose(
                    all_values,
                    0.0,
                    rtol=0.0,
                    atol=1e-12,
                ).sum()
            ),

        "IntegerLikeValues":
            int(
                integer_like_mask.sum()
            ),

        "IntegerLikeFraction":
            float(
                integer_like_mask.mean()
            ),

        "Minimum":
            float(
                np.min(all_values)
            ),

        "Maximum":
            float(
                np.max(all_values)
            ),

        "Mean":
            float(
                       "Maximum":
            float(
                np.max(all_values)
            ),

        "Mean":
            float(
                np.mean(all_values)
            ),

        "Median":
            float(
                np.median(all_values)
            ),

        "TrainingMinimum":
            float(
                np.min(training_values)
            ),

        "TrainingMaximum":
            float(
                np.max(training_values)
            ),

        "EvaluationMinimum":
            float(
                np.min(evaluation_values)
            ),

        "EvaluationMaximum":
            float(
                np.max(evaluation_values)
            ),
    })


rec_feature_profile = pd.DataFrame(
    rec_profile_records
)


# ---------------------------------------------------------
# 7. Profile raw and model-ready verdict subtypes
# ---------------------------------------------------------

def create_verdict_profile(
    dataframe,
    source_name,
):
    profile = (
        dataframe
        .groupby(
            [
                "partition",
                "Verdict",
            ],
            dropna=False,
        )
        .agg(
            ExecutionRows=(
                "Test",
                "size",
            ),

            Builds=(
                "Build",
                "nunique",
            ),

            Tests=(
                "Test",
                "nunique",
            ),
        )
        .reset_index()
    )


    profile.insert(
        0,
        "Source",
        source_name,
    )


    profile[
        "Verdict"
    ] = pd.to_numeric(
        profile[
            "Verdict"
        ],
        errors="raise",
    ).astype(np.int64)


    profile[
        "IsFailure"
    ] = (
        profile[
            "Verdict"
        ] != 0
    )


    return profile


raw_verdict_profile = (
    create_verdict_profile(
        exe_work,
        "RAW_EXECUTION_HISTORY",
    )
)


model_verdict_profile = (
    create_verdict_profile(
        dataset_work,
        "MODEL_READY_DATASET",
    )
)


verdict_subtype_profile = pd.concat(
    [
        raw_verdict_profile,
        model_verdict_profile,
    ],
    ignore_index=True,
)


raw_training_failure_subtypes = (
    raw_training_work[
        raw_training_work[
            "Verdict"
        ].astype(np.int64)
        != 0
    ][
        "Verdict"
    ]
    .value_counts()
    .sort_index()
)


RAW_TRAINING_FAILURE_SUBTYPE_COUNTS = {
    int(verdict):
        int(count)
    for verdict, count in (
        raw_training_failure_subtypes.items()
    )
}


RAW_TRAINING_FAILURE_TOTAL = int(
    sum(
        RAW_TRAINING_FAILURE_SUBTYPE_COUNTS.values()
    )
)


if RAW_TRAINING_FAILURE_TOTAL <= 0:
    raise AssertionError(
        "Raw training history contains no failure "
        "subtypes."
    )


RAW_TRAINING_FAILURE_SUBTYPE_PROBABILITIES = {
    int(verdict):
        float(
            count
            / RAW_TRAINING_FAILURE_TOTAL
        )
    for verdict, count in (
        RAW_TRAINING_FAILURE_SUBTYPE_COUNTS.items()
    )
}


# ---------------------------------------------------------
# 8. Create raw/model test-history profiles
# ---------------------------------------------------------

raw_test_history = (
    exe_work
    .groupby(
        "Test",
        sort=True,
    )
    .agg(
        RawExecutions=(
            "Build",
            "size",
        ),

        RawBuilds=(
            "Build",
            "nunique",
        ),

        RawFailureExecutions=(
            "Verdict",
            lambda values:
                int(
                    (
                        pd.to_numeric(
                            values,
                            errors="raise",
                        ).to_numpy(
                            dtype=np.int64
                        )
                        != 0
                    ).sum()
                ),
        ),

        RawFirstBuildOrder=(
            "build_order",
            "min",
        ),

        RawLastBuildOrder=(
            "build_order",
            "max",
        ),
    )
    .reset_index()
)


model_test_history = (
    dataset_work
    .groupby(
        "Test",
        sort=True,
    )
    .agg(
        ModelReadyRows=(
            "Build",
            "size",
        ),

        ModelReadyBuilds=(
            "Build",
            "nunique",
        ),

        ModelReadyFailureExecutions=(
            "Verdict",
            lambda values:
                int(
                    (
                        pd.to_numeric(
                            values,
                            errors="raise",
                        ).to_numpy(
                            dtype=np.int64
                        )
                        != 0
                    ).sum()
                ),
        ),

        ModelFirstBuildOrder=(
            "build_order",
            "min",
        ),

        ModelLastBuildOrder=(
            "build_order",
            "max",
        ),
    )
    .reset_index()
)


test_history_profile = (
    raw_test_history
    .merge(
        model_test_history,
        on="Test",
        how="outer",
        validate="one_to_one",
    )
)


integer_history_columns = [
    "RawExecutions",
    "RawBuilds",
    "RawFailureExecutions",
    "ModelReadyRows",
    "ModelReadyBuilds",
    "ModelReadyFailureExecutions",
]


for column in integer_history_columns:
    test_history_profile[
        column
    ] = (
        test_history_profile[
            column
        ]
        .fillna(0)
        .astype(np.int64)
    )


test_history_profile[
    "RawRowsNotRetained"
] = (
    test_history_profile[
        "RawExecutions"
    ]
    - test_history_profile[
        "ModelReadyRows"
    ]
)


test_history_profile[
    "HasRawFailure"
] = (
    test_history_profile[
        "RawFailureExecutions"
    ] > 0
)


test_history_profile[
    "HasModelReadyFailure"
] = (
    test_history_profile[
        "ModelReadyFailureExecutions"
    ] > 0
)


test_history_profile = (
    test_history_profile
    .sort_values(
        "Test",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


RAW_UNIQUE_TESTS = int(
    (
        test_history_profile[
            "RawExecutions"
        ] > 0
    ).sum()
)


MODEL_READY_UNIQUE_TESTS = int(
    (
        test_history_profile[
            "ModelReadyRows"
        ] > 0
    ).sum()
)


RAW_TESTS_WITH_FAILURE = int(
    test_history_profile[
        "HasRawFailure"
    ].sum()
)


MODEL_TESTS_WITH_FAILURE = int(
    test_history_profile[
        "HasModelReadyFailure"
    ].sum()
)


TESTS_WITH_UNRETAINED_RAW_ROWS = int(
    (
        test_history_profile[
            "RawRowsNotRetained"
        ] > 0
    ).sum()
)


# ---------------------------------------------------------
# 9. Create canonical chronological history views
# ---------------------------------------------------------

def count_source_order_violations(
    dataframe,
):
    violation_count = 0


    for _, test_rows in dataframe.groupby(
        "Test",
        sort=False,
    ):
        build_orders = (
            test_rows[
                "build_order"
            ].to_numpy(
                dtype=np.int64
            )
        )


        if (
            len(build_orders) > 1
            and (
                np.diff(
                    build_orders
                ) < 0
            ).any()
        ):
            violation_count += 1


    return int(
        violation_count
    )


raw_execution_order_violations = (
    count_source_order_violations(
        exe_work
    )
)


model_execution_order_violations = (
    count_source_order_violations(
        dataset_work
    )
)


raw_history_chronological = (
    exe_work
    .assign(
        _SourceRow=np.arange(
            len(exe_work),
            dtype=np.int64,
        )
    )
    .sort_values(
        [
            "build_order",
            "Test",
            "_SourceRow",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


model_history_chronological = (
    dataset_work
    .assign(
        _SourceRow=np.arange(
            len(dataset_work),
            dtype=np.int64,
        )
    )
    .sort_values(
        [
            "build_order",
            "Test",
            "_SourceRow",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if not np.array_equal(
    np.sort(
        raw_history_chronological[
            "_SourceRow"
        ].to_numpy(dtype=np.int64)
    ),
    np.arange(
        len(raw_history_chronological),
        dtype=np.int64,
    ),
):
    raise AssertionError(
        "Canonical raw chronology lost or duplicated "
        "execution rows."
    )


if not np.array_equal(
    np.sort(
        model_history_chronological[
            "_SourceRow"
        ].to_numpy(dtype=np.int64)
    ),
    np.arange(
        len(model_history_chronological),
        dtype=np.int64,
    ),
):
    raise AssertionError(
        "Canonical model chronology lost or duplicated "
        "model-ready rows."
    )


# ---------------------------------------------------------
# 10. Save representative clean REC trajectories
# ---------------------------------------------------------

trajectory_candidate_tests = (
    test_history_profile[
        (
            test_history_profile[
                "ModelReadyRows"
            ] >= 5
        )
        & (
            test_history_profile[
                "ModelReadyFailureExecutions"
            ] > 0
        )
    ]
    .sort_values(
        [
            "ModelReadyFailureExecutions",
            "ModelReadyRows",
            "Test",
        ],
        ascending=[
            False,
            False,
            True,
        ],
        kind="mergesort",
    )
    .head(12)[
        "Test"
    ]
    .astype(str)
    .tolist()
)


trajectory_frames = []


for test_id in trajectory_candidate_tests:
    test_rows = (
        model_history_chronological[
            model_history_chronological[
                "Test"
            ].astype(str)
            == str(test_id)
        ]
        .copy()
        .reset_index(drop=True)
    )


    failure_positions = np.flatnonzero(
        test_rows[
            "Verdict"
        ].to_numpy(dtype=np.int64)
        != 0
    )


    if len(failure_positions) == 0:
        continue


    first_failure_position = int(
        failure_positions[0]
    )


    start_position = max(
        0,
        first_failure_position - 3,
    )


    end_position = min(
        len(test_rows),
        first_failure_position + 4,
    )


    sample_rows = (
        test_rows.iloc[
            start_position:
            end_position
        ]
        .copy()
    )


    sample_rows[
        "TrajectoryTest"
    ] = str(test_id)


    sample_rows[
        "PositionWithinTestHistory"
    ] = np.arange(
        start_position,
        end_position,
        dtype=np.int64,
    )


    sample_rows[
        "FirstFailurePosition"
    ] = first_failure_position


    trajectory_frames.append(
        sample_rows[
            [
                "TrajectoryTest",
                "PositionWithinTestHistory",
                "FirstFailurePosition",
                "Build",
                "build_order",
                "partition",
                "Test",
                "Verdict",
                "Duration",
            ]
            + REC_COLUMNS
        ]
    )


if trajectory_frames:
    rec_sample_trajectories = pd.concat(
        trajectory_frames,
        ignore_index=True,
    )

else:
    rec_sample_trajectories = pd.DataFrame(
        columns=[
            "TrajectoryTest",
            "PositionWithinTestHistory",
            "FirstFailurePosition",
            "Build",
            "build_order",
            "partition",
            "Test",
            "Verdict",
            "Duration",
        ]
        + REC_COLUMNS
    )


# ---------------------------------------------------------
# 11. Save permanent Step 5 artefacts
# ---------------------------------------------------------

REC_FEATURE_PROFILE_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_rec_feature_profile.csv"
)


REC_CLASSIFICATION_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_rec_feature_classification.csv"
)


VERDICT_SUBTYPE_PROFILE_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_verdict_subtype_profile.csv"
)


TEST_HISTORY_PROFILE_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_test_history_profile.csv"
)


REC_SAMPLE_TRAJECTORIES_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_rec_sample_trajectories.csv"
)


REC_SCHEMA_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_rec_schema_inspection_report.json"
)


rec_feature_profile.to_csv(
    REC_FEATURE_PROFILE_PATH,
    index=False,
)


rec_feature_classification.to_csv(
    REC_CLASSIFICATION_PATH,
    index=False,
)


verdict_subtype_profile.to_csv(
    VERDICT_SUBTYPE_PROFILE_PATH,
    index=False,
)


test_history_profile.to_csv(
    TEST_HISTORY_PROFILE_PATH,
    index=False,
)


rec_sample_trajectories.to_csv(
    REC_SAMPLE_TRAJECTORIES_PATH,
    index=False,
)


# ---------------------------------------------------------
# 12. Save permanent REC schema report
# ---------------------------------------------------------

schema_inspection_completed_at = (
    pd.Timestamp.utcnow().isoformat()
)


rec_schema_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "PASS",

    "ProjectSpecificRECStructure": {
        "TotalRECFeatures":
            int(len(REC_COLUMNS)),

        "VerdictDependentFeatures":
            int(len(REC_DEPENDENT_COLUMNS)),

        "VerdictIndependentFeatures":
            int(len(REC_INDEPENDENT_COLUMNS)),

        "VerdictDependentColumns":
            list(REC_DEPENDENT_COLUMNS),

        "VerdictIndependentColumns":
            list(REC_INDEPENDENT_COLUMNS),

        "Explanation":
            (
                "Beast2 contains 19 REC predictors. "
                "The frozen noise protocol remains "
                "unchanged: preserve the six "
                "verdict-independent execution-time/age "
                "features and recompute the thirteen "
                "verdict-dependent features."
            ),
    },

    "RECDataQuality": {
        "NumericConversionFailures":
            int(rec_numeric_conversion_failures),

        "MissingValues":
            int(rec_missing_values),

        "InfiniteValues":
            int(rec_infinite_values),
    },

    "RawHistory": {
        "Rows":
            int(len(exe_work)),

        "TrainingRows":
            int(len(raw_training_work)),

        "EvaluationRows":
            int(len(raw_evaluation_work)),

        "UniqueTests":
            RAW_UNIQUE_TESTS,

        "TestsWithFailure":
            RAW_TESTS_WITH_FAILURE,

        "SourceOrderTestViolations":
            raw_execution_order_violations,
    },

    "ModelReadyHistory": {
        "Rows":
            int(len(dataset_work)),

        "TrainingRows":
            int(len(model_training_work)),

        "EvaluationRows":
            int(len(model_evaluation_work)),

        "UniqueTests":
            MODEL_READY_UNIQUE_TESTS,

        "TestsWithFailure":
            MODEL_TESTS_WITH_FAILURE,

        "TestsWithUnretainedRawRows":
            TESTS_WITH_UNRETAINED_RAW_ROWS,

        "SourceOrderTestViolations":
            model_execution_order_violations,
    },

    "RawTrainingFailureSubtypeDistribution": {
        "FailureExecutions":
            RAW_TRAINING_FAILURE_TOTAL,

        "Counts": {
            str(verdict):
                int(count)
            for verdict, count in (
                RAW_TRAINING_FAILURE_SUBTYPE_COUNTS.items()
            )
        },

        "Probabilities": {
            str(verdict):
                float(probability)
            for verdict, probability in (
                RAW_TRAINING_FAILURE_SUBTYPE_PROBABILITIES.items()
            )
        },
    },

    "EntityMappingAvailability": {
        "ChangedEntityDictionaryBuilds":
            int(
                len(
                    changed_entity_dictionary
                )
            ),

        "BuildsWithChangedEntities":
            int(
                sum(
                    len(entity_ids) > 0
                    for entity_ids in (
                        changed_entity_dictionary.values()
                    )
                )
            ),
    },

    "RepresentativeTrajectories": {
        "SelectedTests":
            trajectory_candidate_tests,

        "SavedRows":
            int(
                len(
                    rec_sample_trajectories
                )
            ),
    },

    "Artefacts": {
        "RECFeatureProfile":
            str(
                REC_FEATURE_PROFILE_PATH
            ),

        "RECFeatureClassification":
            str(
                REC_CLASSIFICATION_PATH
            ),

        "VerdictSubtypeProfile":
            str(
                VERDICT_SUBTYPE_PROFILE_PATH
            ),

        "TestHistoryProfile":
            str(
                TEST_HISTORY_PROFILE_PATH
            ),

        "SampleTrajectories":
            str(
                REC_SAMPLE_TRAJECTORIES_PATH
            ),
    },

    "CompletedAtUTC":
        schema_inspection_completed_at,
}


with open(
    REC_SCHEMA_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        rec_schema_report,
        report_file,
        indent=2,
        default=str,
    )


# ---------------------------------------------------------
# 13. Update Project 7 checkpoint atomically
# ---------------------------------------------------------

with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    project_7_checkpoint = json.load(
        checkpoint_file
    )


project_7_checkpoint.update({
    "Status":
        "REC_SCHEMA_INSPECTION_PASSED",

    "RECFeatureCount":
        int(len(REC_COLUMNS)),

    "VerdictDependentRECFeatureCount":
        int(len(REC_DEPENDENT_COLUMNS)),

    "VerdictIndependentRECFeatureCount":
        int(len(REC_INDEPENDENT_COLUMNS)),

    "VerdictDependentRECColumns":
        list(REC_DEPENDENT_COLUMNS),

    "VerdictIndependentRECColumns":
        list(REC_INDEPENDENT_COLUMNS),

    "RECNoisePolicy":
        (
            "Preserve six verdict-independent REC "
            "features and recompute thirteen "
            "verdict-dependent REC features."
        ),

    "RECFeatureNumericConversionFailures":
        int(rec_numeric_conversion_failures),

    "RECFeatureMissingValues":
        int(rec_missing_values),

    "RECFeatureInfiniteValues":
        int(rec_infinite_values),

    "RawUniqueTests":
        RAW_UNIQUE_TESTS,

    "ModelReadyUniqueTests":
        MODEL_READY_UNIQUE_TESTS,

    "RawTestsWithFailure":
        RAW_TESTS_WITH_FAILURE,

    "ModelReadyTestsWithFailure":
        MODEL_TESTS_WITH_FAILURE,

    "RawTrainingFailureSubtypeCounts": {
        str(verdict):
            int(count)
        for verdict, count in (
            RAW_TRAINING_FAILURE_SUBTYPE_COUNTS.items()
        )
    },

    "RawTrainingFailureSubtypeProbabilities": {
        str(verdict):
            float(probability)
        for verdict, probability in (
            RAW_TRAINING_FAILURE_SUBTYPE_PROBABILITIES.items()
        )
    },

    "RECSchemaInspectionReport":
        str(
            REC_SCHEMA_REPORT_PATH
        ),

    "RECFeatureProfile":
        str(
            REC_FEATURE_PROFILE_PATH
        ),

    "RECFeatureClassification":
        str(
            REC_CLASSIFICATION_PATH
        ),

    "VerdictSubtypeProfile":
        str(
            VERDICT_SUBTYPE_PROFILE_PATH
        ),

    "TestHistoryProfile":
        str(
            TEST_HISTORY_PROFILE_PATH
        ),

    "RECSampleTrajectories":
        str(
            REC_SAMPLE_TRAJECTORIES_PATH
        ),

    "UpdatedAtUTC":
        schema_inspection_completed_at,
})


temporary_checkpoint_path = (
    PROJECT_7_SELECTION_CHECKPOINT
    .with_suffix(
        ".json.tmp"
    )
)


with open(
    temporary_checkpoint_path,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        project_7_checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


os.replace(
    temporary_checkpoint_path,
    PROJECT_7_SELECTION_CHECKPOINT,
)


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint_verification = json.load(
        checkpoint_file
    )


if checkpoint_verification.get(
    "Status"
) != "REC_SCHEMA_INSPECTION_PASSED":
    raise AssertionError(
        "The Project 7 REC schema checkpoint was not "
        "written correctly."
    )


if int(
    checkpoint_verification.get(
        "RECFeatureCount",
        -1,
    )
) != 19:
    raise AssertionError(
        "The checkpoint did not preserve the correct "
        "Beast2 REC feature count."
    )


if int(
    checkpoint_verification.get(
        "VerdictDependentRECFeatureCount",
        -1,
    )
) != 13:
    raise AssertionError(
        "The checkpoint did not preserve the thirteen "
        "verdict-dependent Beast2 REC features."
    )


# ---------------------------------------------------------
# 14. Compact final output
# ---------------------------------------------------------

clear_output(
    wait=True
)


print(
    "=== PROJECT 7 STEP 5 RESULT ==="
)


print(
    "\nProject-specific REC structure:"
)

print(
    "Total REC features:",
    len(REC_COLUMNS)
)

print(
    "Verdict-dependent REC features:",
    len(REC_DEPENDENT_COLUMNS)
)

print(
    "Verdict-independent REC features:",
    len(REC_INDEPENDENT_COLUMNS)
)


print(
    "\nVerdict-independent REC features:"
)

display(
    rec_feature_classification[
        rec_feature_classification[
            "ProtocolClass"
        ] == "VERDICT_INDEPENDENT"
    ][
        [
            "Feature",
            "ProtocolClass",
            "NoisePolicy",
        ]
    ]
)


print(
    "\nVerdict-dependent REC features:"
)

display(
    rec_feature_classification[
        rec_feature_classification[
            "ProtocolClass"
        ] == "VERDICT_DEPENDENT"
    ][
        [
            "Feature",
            "ProtocolClass",
            "NoisePolicy",
        ]
    ]
)


print(
    "\nComplete REC feature profile:"
)

display(
    rec_feature_profile[
        [
            "FeatureOrder",
            "Feature",
            "ProtocolClass",
            "UniqueValues",
            "ZeroValues",
            "Minimum",
            "Maximum",
            "Mean",
            "IntegerLikeFraction",
        ]
    ]
)


print(
    "\nREC data quality:"
)

print(
    "Numeric conversion failures:",
    rec_numeric_conversion_failures
)

print(
    "Missing values:",
    rec_missing_values
)

print(
    "Infinite values:",
    rec_infinite_values
)


print(
    "\nRaw training failure subtype distribution:"
)

failure_subtype_display = pd.DataFrame([
    {
        "VerdictSubtype":
            int(verdict),

        "Count":
            int(count),

        "Probability":
            float(
                RAW_TRAINING_FAILURE_SUBTYPE_PROBABILITIES[
                    verdict
                ]
            ),
    }
    for verdict, count in (
        RAW_TRAINING_FAILURE_SUBTYPE_COUNTS.items()
    )
])


display(
    failure_subtype_display
)


print(
    "Raw training failure executions:",
    RAW_TRAINING_FAILURE_TOTAL
)


print(
    "\nTest-history coverage:"
)

print(
    "Raw unique tests:",
    RAW_UNIQUE_TESTS
)

print(
    "Model-ready unique tests:",
    MODEL_READY_UNIQUE_TESTS
)

print(
    "Raw tests with failures:",
    RAW_TESTS_WITH_FAILURE
)

print(
    "Model-ready tests with failures:",
    MODEL_TESTS_WITH_FAILURE
)

print(
    "Tests with unretained raw rows:",
    TESTS_WITH_UNRETAINED_RAW_ROWS
)


print(
    "\nSource-order diagnostics:"
)

print(
    "Raw test histories not already chronological:",
    raw_execution_order_violations
)

print(
    "Model test histories not already chronological:",
    model_execution_order_violations
)

print(
    "Canonical chronological views created:",
    True
)


print(
    "\nRepresentative REC trajectories:"
)

print(
    "Selected tests:",
    trajectory_candidate_tests
)

print(
    "Saved trajectory rows:",
    len(
        rec_sample_trajectories
    )
)


print(
    "\nREC schema report:"
)

print(
    REC_SCHEMA_REPORT_PATH
)


print(
    "\nValidation status:",
    "PASS"
)


print(
    "\nSUCCESS: All 19 Beast2 REC features were "
    "identified."
)

print(
    "SUCCESS: The thirteen verdict-dependent and six "
    "verdict-independent REC features were separated."
)

print(
    "SUCCESS: Every REC value was numeric, present and "
    "finite."
)

print(
    "SUCCESS: Raw training failure subtype counts and "
    "probabilities were frozen."
)

print(
    "SUCCESS: Canonical chronological raw and "
    "model-ready history views were created."
)

print(
    "SUCCESS: No REC value or verdict was modified."
)

print(
    "SUCCESS: Project 7 is ready for clean REC "
    "reconstruction and validation."
)

SyntaxError: closing parenthesis '}' does not match opening parenthesis '(' on line 727 (4122928768.py, line 762)

In [ ]:
# =========================================================
# PROJECT 7 — STEP 5
# BEAST2 REC SCHEMA AND HISTORY INSPECTION
#
# Beast2 contains:
#   - 19 total REC features
#   - 6 verdict-independent REC features
#   - 13 verdict-dependent REC features
#
# This cell:
#   - validates the exact Beast2 REC schema
#   - freezes dependent/independent feature lists
#   - validates all REC values
#   - profiles raw/model verdict subtypes
#   - creates canonical chronological history views
#   - saves permanent reports
#   - updates the Project 7 checkpoint
#
# It does not reconstruct features, inject noise,
# modify verdicts, or train models.
# =========================================================

from pathlib import Path
import json
import os
import re

import numpy as np
import pandas as pd
from IPython.display import clear_output, display


print(
    "=== PROJECT 7 STEP 5: "
    "BEAST2 REC SCHEMA AND HISTORY INSPECTION ==="
)


# ---------------------------------------------------------
# 1. Confirm required Step 4 runtime objects
# ---------------------------------------------------------

required_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",
    "PROJECT_PREFLIGHT_DIRECTORY",
    "PROJECT_7_SELECTION_CHECKPOINT",
    "chronological_builds",
    "exe",
    "dataset",
    "raw_training_history",
    "raw_evaluation_history",
    "all_model_training_data",
    "all_model_evaluation_data",
    "PREDICTOR_COLUMNS",
    "changed_entity_dictionary",
]


missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required Project 7 objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nRerun only the successful Project 7 "
        "Step 2B, Step 3 and Step 4 cells."
    )


if PROJECT_NUMBER != 7:
    raise AssertionError(
        f"Expected Project 7, observed {PROJECT_NUMBER}."
    )


if PROJECT_NAME != "CompEvol@beast2":
    raise AssertionError(
        f"Unexpected project: {PROJECT_NAME}"
    )


if PROJECT_SLUG != "CompEvol__beast2":
    raise AssertionError(
        f"Unexpected project slug: {PROJECT_SLUG}"
    )


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    previous_checkpoint = json.load(checkpoint_file)


allowed_statuses = {
    "CANONICAL_ENTITY_MAPPING_PASSED",
    "REC_SCHEMA_INSPECTION_PASSED",
}


if previous_checkpoint.get("Status") not in allowed_statuses:
    raise AssertionError(
        "Project 7 canonical entity mapping has not passed.\n"
        f"Observed status: "
        f"{previous_checkpoint.get('Status')}"
    )


if int(
    previous_checkpoint.get(
        "FullyMappedBuilds",
        -1,
    )
) != 415:
    raise AssertionError(
        "The checkpoint does not confirm 415 fully "
        "mapped builds."
    )


# ---------------------------------------------------------
# 2. Defragment working copies
# ---------------------------------------------------------

exe_work = exe.copy().reset_index(drop=True)
dataset_work = dataset.copy().reset_index(drop=True)

raw_training_work = (
    raw_training_history
    .copy()
    .reset_index(drop=True)
)

raw_evaluation_work = (
    raw_evaluation_history
    .copy()
    .reset_index(drop=True)
)

model_training_work = (
    all_model_training_data
    .copy()
    .reset_index(drop=True)
)

model_evaluation_work = (
    all_model_evaluation_data
    .copy()
    .reset_index(drop=True)
)


required_core_columns = {
    "Build",
    "Test",
    "Verdict",
    "Duration",
    "build_order",
    "partition",
}


for frame_name, frame in [
    ("exe", exe_work),
    ("dataset", dataset_work),
]:
    missing_columns = (
        required_core_columns
        - set(frame.columns)
    )

    if missing_columns:
        raise RuntimeError(
            f"{frame_name} is missing columns:\n"
            + "\n".join(sorted(missing_columns))
        )


# ---------------------------------------------------------
# 3. Identify and validate exact Beast2 REC schema
# ---------------------------------------------------------

def normalise_feature_name(value):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).strip().lower(),
    )


EXPECTED_BEAST2_REC_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


EXPECTED_INDEPENDENT_REC_COLUMNS = [
    "REC_Age",
    "REC_LastExeTime",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
]


EXPECTED_DEPENDENT_REC_COLUMNS = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


REC_COLUMNS = [
    column
    for column in PREDICTOR_COLUMNS
    if normalise_feature_name(column).startswith("rec")
]


if len(REC_COLUMNS) != 19:
    raise AssertionError(
        "Unexpected Beast2 REC feature count.\n"
        f"Expected: 19\n"
        f"Observed: {len(REC_COLUMNS)}\n\n"
        "Observed REC columns:\n"
        + "\n".join(map(str, REC_COLUMNS))
    )


normalised_rec_lookup = {
    normalise_feature_name(column): column
    for column in REC_COLUMNS
}


if len(normalised_rec_lookup) != 19:
    raise AssertionError(
        "REC column names are ambiguous after "
        "normalisation."
    )


expected_rec_set = {
    normalise_feature_name(column)
    for column in EXPECTED_BEAST2_REC_COLUMNS
}


observed_rec_set = set(
    normalised_rec_lookup.keys()
)


missing_rec_features = sorted(
    expected_rec_set - observed_rec_set
)


unexpected_rec_features = sorted(
    observed_rec_set - expected_rec_set
)


if missing_rec_features:
    raise AssertionError(
        "Expected Beast2 REC features are missing:\n"
        + "\n".join(missing_rec_features)
    )


if unexpected_rec_features:
    raise AssertionError(
        "Unexpected Beast2 REC features were found:\n"
        + "\n".join(unexpected_rec_features)
    )


REC_INDEPENDENT_COLUMNS = [
    normalised_rec_lookup[
        normalise_feature_name(column)
    ]
    for column in EXPECTED_INDEPENDENT_REC_COLUMNS
]


REC_DEPENDENT_COLUMNS = [
    normalised_rec_lookup[
        normalise_feature_name(column)
    ]
    for column in EXPECTED_DEPENDENT_REC_COLUMNS
]


if len(set(REC_INDEPENDENT_COLUMNS)) != 6:
    raise AssertionError(
        "The six independent REC features did not "
        "resolve uniquely."
    )


if len(set(REC_DEPENDENT_COLUMNS)) != 13:
    raise AssertionError(
        "The thirteen dependent REC features did not "
        "resolve uniquely."
    )


if set(REC_INDEPENDENT_COLUMNS).intersection(
    set(REC_DEPENDENT_COLUMNS)
):
    raise AssertionError(
        "A REC feature was classified as both dependent "
        "and independent."
    )


if (
    set(REC_INDEPENDENT_COLUMNS)
    | set(REC_DEPENDENT_COLUMNS)
) != set(REC_COLUMNS):
    raise AssertionError(
        "The dependent/independent classifications do "
        "not cover all Beast2 REC features."
    )


# Permanent aliases for later cells.
VERDICT_INDEPENDENT_REC_COLUMNS = list(
    REC_INDEPENDENT_COLUMNS
)

VERDICT_DEPENDENT_REC_COLUMNS = list(
    REC_DEPENDENT_COLUMNS
)


# ---------------------------------------------------------
# 4. Create REC classification table
# ---------------------------------------------------------

classification_records = []


for feature_order, feature in enumerate(
    REC_COLUMNS,
    start=1,
):
    is_independent = (
        feature in set(REC_INDEPENDENT_COLUMNS)
    )

    classification_records.append({
        "FeatureOrder":
            int(feature_order),

        "Feature":
            str(feature),

        "NormalisedFeature":
            normalise_feature_name(feature),

        "ProtocolClass":
            (
                "VERDICT_INDEPENDENT"
                if is_independent
                else "VERDICT_DEPENDENT"
            ),

        "NoisePolicy":
            (
                "PRESERVE_ORIGINAL_TCP_CI_VALUE"
                if is_independent
                else "RECOMPUTE_FROM_CORRUPTED_RAW_HISTORY"
            ),
    })


rec_feature_classification = pd.DataFrame(
    classification_records
)


if int(
    (
        rec_feature_classification[
            "ProtocolClass"
        ] == "VERDICT_INDEPENDENT"
    ).sum()
) != 6:
    raise AssertionError(
        "Independent REC classification count is not 6."
    )


if int(
    (
        rec_feature_classification[
            "ProtocolClass"
        ] == "VERDICT_DEPENDENT"
    ).sum()
) != 13:
    raise AssertionError(
        "Dependent REC classification count is not 13."
    )


# ---------------------------------------------------------
# 5. Validate all REC values
# ---------------------------------------------------------

rec_numeric_conversion_failures = 0
rec_missing_values = 0
rec_infinite_values = 0


for feature in REC_COLUMNS:
    original_values = dataset_work[feature]

    numeric_values = pd.to_numeric(
        original_values,
        errors="coerce",
    )

    rec_numeric_conversion_failures += int(
        (
            numeric_values.isna()
            & original_values.notna()
        ).sum()
    )

    rec_missing_values += int(
        numeric_values.isna().sum()
    )

    finite_values = (
        numeric_values
        .dropna()
        .to_numpy(dtype=float)
    )

    rec_infinite_values += int(
        np.isinf(finite_values).sum()
    )

    dataset_work[feature] = numeric_values


if rec_numeric_conversion_failures != 0:
    raise AssertionError(
        "REC features contain non-numeric values.\n"
        f"Conversion failures: "
        f"{rec_numeric_conversion_failures}"
    )


if rec_missing_values != 0:
    raise AssertionError(
        "REC features contain missing values.\n"
        f"Missing values: {rec_missing_values}"
    )


if rec_infinite_values != 0:
    raise AssertionError(
        "REC features contain infinite values.\n"
        f"Infinite values: {rec_infinite_values}"
    )


# ---------------------------------------------------------
# 6. Create REC feature profile
# ---------------------------------------------------------

profile_records = []


for feature_order, feature in enumerate(
    REC_COLUMNS,
    start=1,
):
    values = dataset_work[
        feature
    ].to_numpy(dtype=float)

    training_values = dataset_work.loc[
        dataset_work["partition"] == "TRAIN",
        feature,
    ].to_numpy(dtype=float)

    evaluation_values = dataset_work.loc[
        dataset_work["partition"] == "EVALUATION",
        feature,
    ].to_numpy(dtype=float)

    integer_like = np.isclose(
        values,
        np.round(values),
        rtol=0.0,
        atol=1e-12,
    )

    profile_records.append({
        "FeatureOrder":
            int(feature_order),

        "Feature":
            str(feature),

        "ProtocolClass":
            (
                "VERDICT_INDEPENDENT"
                if feature in set(REC_INDEPENDENT_COLUMNS)
                else "VERDICT_DEPENDENT"
            ),

        "Rows":
            int(len(values)),

        "MissingValues":
            int(np.isnan(values).sum()),

        "InfiniteValues":
            int(np.isinf(values).sum()),

        "UniqueValues":
            int(
                pd.Series(values).nunique(
                    dropna=True
                )
            ),

        "ZeroValues":
            int(
                np.isclose(
                    values,
                    0.0,
                    rtol=0.0,
                    atol=1e-12,
                ).sum()
            ),

        "IntegerLikeFraction":
            float(integer_like.mean()),

        "Minimum":
            float(np.min(values)),

        "Maximum":
            float(np.max(values)),

        "Mean":
            float(np.mean(values)),

        "Median":
            float(np.median(values)),

        "TrainingMinimum":
            float(np.min(training_values)),

        "TrainingMaximum":
            float(np.max(training_values)),

        "EvaluationMinimum":
            float(np.min(evaluation_values)),

        "EvaluationMaximum":
            float(np.max(evaluation_values)),
    })


rec_feature_profile = pd.DataFrame(
    profile_records
)


# ---------------------------------------------------------
# 7. Profile verdict subtypes
# ---------------------------------------------------------

def create_verdict_profile(
    dataframe,
    source_name,
):
    profile = (
        dataframe
        .groupby(
            [
                "partition",
                "Verdict",
            ],
            dropna=False,
        )
        .agg(
            ExecutionRows=("Test", "size"),
            Builds=("Build", "nunique"),
            Tests=("Test", "nunique"),
        )
        .reset_index()
    )

    profile.insert(
        0,
        "Source",
        source_name,
    )

    profile["Verdict"] = pd.to_numeric(
        profile["Verdict"],
        errors="raise",
    ).astype(np.int64)

    profile["IsFailure"] = (
        profile["Verdict"] != 0
    )

    return profile


verdict_subtype_profile = pd.concat(
    [
        create_verdict_profile(
            exe_work,
            "RAW_EXECUTION_HISTORY",
        ),
        create_verdict_profile(
            dataset_work,
            "MODEL_READY_DATASET",
        ),
    ],
    ignore_index=True,
)


raw_training_failures = (
    raw_training_work[
        raw_training_work[
            "Verdict"
        ].astype(np.int64) != 0
    ]
)


failure_subtype_counts_series = (
    raw_training_failures[
        "Verdict"
    ]
    .astype(np.int64)
    .value_counts()
    .sort_index()
)


RAW_TRAINING_FAILURE_SUBTYPE_COUNTS = {
    int(verdict): int(count)
    for verdict, count in (
        failure_subtype_counts_series.items()
    )
}


RAW_TRAINING_FAILURE_TOTAL = int(
    sum(
        RAW_TRAINING_FAILURE_SUBTYPE_COUNTS.values()
    )
)


if RAW_TRAINING_FAILURE_TOTAL <= 0:
    raise AssertionError(
        "Raw training history contains no failure "
        "executions."
    )


RAW_TRAINING_FAILURE_SUBTYPE_PROBABILITIES = {
    int(verdict):
        float(
            count / RAW_TRAINING_FAILURE_TOTAL
        )
    for verdict, count in (
        RAW_TRAINING_FAILURE_SUBTYPE_COUNTS.items()
    )
}


failure_subtype_display = pd.DataFrame([
    {
        "VerdictSubtype":
            int(verdict),

        "Count":
            int(count),

        "Probability":
            float(
                RAW_TRAINING_FAILURE_SUBTYPE_PROBABILITIES[
                    verdict
                ]
            ),
    }
    for verdict, count in (
        RAW_TRAINING_FAILURE_SUBTYPE_COUNTS.items()
    )
])


# ---------------------------------------------------------
# 8. Create test-history profile
# ---------------------------------------------------------

raw_test_history = (
    exe_work
    .groupby(
        "Test",
        sort=True,
    )
    .agg(
        RawExecutions=("Build", "size"),
        RawBuilds=("Build", "nunique"),
        RawFirstBuildOrder=("build_order", "min"),
        RawLastBuildOrder=("build_order", "max"),
    )
    .reset_index()
)


raw_failure_counts = (
    exe_work.assign(
        _Failure=(
            exe_work["Verdict"]
            .astype(np.int64)
            .ne(0)
            .astype(np.int64)
        )
    )
    .groupby(
        "Test",
        sort=True,
    )["_Failure"]
    .sum()
    .rename("RawFailureExecutions")
    .reset_index()
)


raw_test_history = raw_test_history.merge(
    raw_failure_counts,
    on="Test",
    how="left",
    validate="one_to_one",
)


model_test_history = (
    dataset_work
    .groupby(
        "Test",
        sort=True,
    )
    .agg(
        ModelReadyRows=("Build", "size"),
        ModelReadyBuilds=("Build", "nunique"),
        ModelFirstBuildOrder=("build_order", "min"),
        ModelLastBuildOrder=("build_order", "max"),
    )
    .reset_index()
)


model_failure_counts = (
    dataset_work.assign(
        _Failure=(
            dataset_work["Verdict"]
            .astype(np.int64)
            .ne(0)
            .astype(np.int64)
        )
    )
    .groupby(
        "Test",
        sort=True,
    )["_Failure"]
    .sum()
    .rename("ModelReadyFailureExecutions")
    .reset_index()
)


model_test_history = model_test_history.merge(
    model_failure_counts,
    on="Test",
    how="left",
    validate="one_to_one",
)


test_history_profile = raw_test_history.merge(
    model_test_history,
    on="Test",
    how="outer",
    validate="one_to_one",
)


count_columns = [
    "RawExecutions",
    "RawBuilds",
    "RawFailureExecutions",
    "ModelReadyRows",
    "ModelReadyBuilds",
    "ModelReadyFailureExecutions",
]


for column in count_columns:
    test_history_profile[column] = (
        test_history_profile[column]
        .fillna(0)
        .astype(np.int64)
    )


test_history_profile["RawRowsNotRetained"] = (
    test_history_profile["RawExecutions"]
    - test_history_profile["ModelReadyRows"]
)


test_history_profile["HasRawFailure"] = (
    test_history_profile[
        "RawFailureExecutions"
    ] > 0
)


test_history_profile["HasModelReadyFailure"] = (
    test_history_profile[
        "ModelReadyFailureExecutions"
    ] > 0
)


test_history_profile = (
    test_history_profile
    .sort_values(
        "Test",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


RAW_UNIQUE_TESTS = int(
    (
        test_history_profile["RawExecutions"] > 0
    ).sum()
)


MODEL_READY_UNIQUE_TESTS = int(
    (
        test_history_profile["ModelReadyRows"] > 0
    ).sum()
)


RAW_TESTS_WITH_FAILURE = int(
    test_history_profile[
        "HasRawFailure"
    ].sum()
)


MODEL_TESTS_WITH_FAILURE = int(
    test_history_profile[
        "HasModelReadyFailure"
    ].sum()
)


TESTS_WITH_UNRETAINED_RAW_ROWS = int(
    (
        test_history_profile[
            "RawRowsNotRetained"
        ] > 0
    ).sum()
)


# ---------------------------------------------------------
# 9. Create canonical chronological history views
# ---------------------------------------------------------

raw_history_chronological = (
    exe_work
    .assign(
        _SourceRow=np.arange(
            len(exe_work),
            dtype=np.int64,
        )
    )
    .sort_values(
        [
            "build_order",
            "Test",
            "_SourceRow",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


model_history_chronological = (
    dataset_work
    .assign(
        _SourceRow=np.arange(
            len(dataset_work),
            dtype=np.int64,
        )
    )
    .sort_values(
        [
            "build_order",
            "Test",
            "_SourceRow",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if len(raw_history_chronological) != len(exe_work):
    raise AssertionError(
        "The raw chronological view lost rows."
    )


if len(model_history_chronological) != len(
    dataset_work
):
    raise AssertionError(
        "The model chronological view lost rows."
    )


if raw_history_chronological[
    "_SourceRow"
].nunique() != len(exe_work):
    raise AssertionError(
        "The raw chronological view duplicated rows."
    )


if model_history_chronological[
    "_SourceRow"
].nunique() != len(dataset_work):
    raise AssertionError(
        "The model chronological view duplicated rows."
    )


# ---------------------------------------------------------
# 10. Select representative REC trajectories
# ---------------------------------------------------------

trajectory_candidate_tests = (
    test_history_profile[
        (
            test_history_profile[
                "ModelReadyRows"
            ] >= 5
        )
        & (
            test_history_profile[
                "ModelReadyFailureExecutions"
            ] > 0
        )
    ]
    .sort_values(
        [
            "ModelReadyFailureExecutions",
            "ModelReadyRows",
            "Test",
        ],
        ascending=[
            False,
            False,
            True,
        ],
        kind="mergesort",
    )
    .head(10)["Test"]
    .astype(str)
    .tolist()
)


trajectory_frames = []


for test_id in trajectory_candidate_tests:
    test_rows = (
        model_history_chronological[
            model_history_chronological[
                "Test"
            ].astype(str) == str(test_id)
        ]
        .copy()
        .reset_index(drop=True)
    )

    failure_positions = np.flatnonzero(
        test_rows[
            "Verdict"
        ].to_numpy(dtype=np.int64) != 0
    )

    if len(failure_positions) == 0:
        continue

    first_failure_position = int(
        failure_positions[0]
    )

    start_position = max(
        0,
        first_failure_position - 2,
    )

    end_position = min(
        len(test_rows),
        first_failure_position + 3,
    )

    sample_rows = (
        test_rows.iloc[
            start_position:end_position
        ]
        .copy()
    )

    sample_rows.insert(
        0,
        "TrajectoryTest",
        str(test_id),
    )

    sample_rows.insert(
        1,
        "PositionWithinTestHistory",
        np.arange(
            start_position,
            end_position,
            dtype=np.int64,
        ),
    )

    sample_rows.insert(
        2,
        "FirstFailurePosition",
        first_failure_position,
    )

    trajectory_frames.append(
        sample_rows[
            [
                "TrajectoryTest",
                "PositionWithinTestHistory",
                "FirstFailurePosition",
                "Build",
                "build_order",
                "partition",
                "Test",
                "Verdict",
                "Duration",
            ]
            + REC_COLUMNS
        ]
    )


if trajectory_frames:
    rec_sample_trajectories = pd.concat(
        trajectory_frames,
        ignore_index=True,
    )
else:
    rec_sample_trajectories = pd.DataFrame(
        columns=[
            "TrajectoryTest",
            "PositionWithinTestHistory",
            "FirstFailurePosition",
            "Build",
            "build_order",
            "partition",
            "Test",
            "Verdict",
            "Duration",
        ]
        + REC_COLUMNS
    )


# ---------------------------------------------------------
# 11. Save permanent artefacts
# ---------------------------------------------------------

REC_FEATURE_PROFILE_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_rec_feature_profile.csv"
)


REC_CLASSIFICATION_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_rec_feature_classification.csv"
)


VERDICT_SUBTYPE_PROFILE_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_verdict_subtype_profile.csv"
)


TEST_HISTORY_PROFILE_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_test_history_profile.csv"
)


REC_SAMPLE_TRAJECTORIES_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_rec_sample_trajectories.csv"
)


REC_SCHEMA_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_rec_schema_inspection_report.json"
)


rec_feature_profile.to_csv(
    REC_FEATURE_PROFILE_PATH,
    index=False,
)


rec_feature_classification.to_csv(
    REC_CLASSIFICATION_PATH,
    index=False,
)


verdict_subtype_profile.to_csv(
    VERDICT_SUBTYPE_PROFILE_PATH,
    index=False,
)


test_history_profile.to_csv(
    TEST_HISTORY_PROFILE_PATH,
    index=False,
)


rec_sample_trajectories.to_csv(
    REC_SAMPLE_TRAJECTORIES_PATH,
    index=False,
)


# ---------------------------------------------------------
# 12. Save JSON report
# ---------------------------------------------------------

completed_at = pd.Timestamp.utcnow().isoformat()


rec_schema_report = {
    "ProjectNumber": int(PROJECT_NUMBER),
    "Project": str(PROJECT_NAME),
    "ProjectSlug": str(PROJECT_SLUG),
    "Status": "PASS",

    "RECStructure": {
        "TotalRECFeatures":
            int(len(REC_COLUMNS)),

        "VerdictDependentFeatures":
            int(len(REC_DEPENDENT_COLUMNS)),

        "VerdictIndependentFeatures":
            int(len(REC_INDEPENDENT_COLUMNS)),

        "VerdictDependentColumns":
            list(REC_DEPENDENT_COLUMNS),

        "VerdictIndependentColumns":
            list(REC_INDEPENDENT_COLUMNS),

        "Policy":
            (
                "Preserve the six verdict-independent "
                "REC features and recompute the thirteen "
                "verdict-dependent REC features."
            ),
    },

    "RECDataQuality": {
        "NumericConversionFailures":
            int(rec_numeric_conversion_failures),

        "MissingValues":
            int(rec_missing_values),

        "InfiniteValues":
            int(rec_infinite_values),
    },

    "HistoryDimensions": {
        "RawRows":
            int(len(exe_work)),

        "RawTrainingRows":
            int(len(raw_training_work)),

        "RawEvaluationRows":
            int(len(raw_evaluation_work)),

        "ModelReadyRows":
            int(len(dataset_work)),

        "ModelTrainingRows":
            int(len(model_training_work)),

        "ModelEvaluationRows":
            int(len(model_evaluation_work)),
    },

    "TestCoverage": {
        "RawUniqueTests":
            int(RAW_UNIQUE_TESTS),

        "ModelReadyUniqueTests":
            int(MODEL_READY_UNIQUE_TESTS),

        "RawTestsWithFailure":
            int(RAW_TESTS_WITH_FAILURE),

        "ModelReadyTestsWithFailure":
            int(MODEL_TESTS_WITH_FAILURE),

        "TestsWithUnretainedRawRows":
            int(TESTS_WITH_UNRETAINED_RAW_ROWS),
    },

    "RawTrainingFailureSubtypeDistribution": {
        "FailureExecutions":
            int(RAW_TRAINING_FAILURE_TOTAL),

        "Counts": {
            str(verdict): int(count)
            for verdict, count in (
                RAW_TRAINING_FAILURE_SUBTYPE_COUNTS.items()
            )
        },

        "Probabilities": {
            str(verdict): float(probability)
            for verdict, probability in (
                RAW_TRAINING_FAILURE_SUBTYPE_PROBABILITIES.items()
            )
        },
    },

    "EntityMapping": {
        "ChangedEntityDictionaryBuilds":
            int(len(changed_entity_dictionary)),

        "BuildsWithChangedEntities":
            int(
                sum(
                    len(entity_ids) > 0
                    for entity_ids in (
                        changed_entity_dictionary.values()
                    )
                )
            ),
    },

    "RepresentativeTrajectories": {
        "SelectedTests":
            list(trajectory_candidate_tests),

        "SavedRows":
            int(len(rec_sample_trajectories)),
    },

    "Artefacts": {
        "RECFeatureProfile":
            str(REC_FEATURE_PROFILE_PATH),

        "RECFeatureClassification":
            str(REC_CLASSIFICATION_PATH),

        "VerdictSubtypeProfile":
            str(VERDICT_SUBTYPE_PROFILE_PATH),

        "TestHistoryProfile":
            str(TEST_HISTORY_PROFILE_PATH),

        "SampleTrajectories":
            str(REC_SAMPLE_TRAJECTORIES_PATH),
    },

    "CompletedAtUTC":
        str(completed_at),
}


with open(
    REC_SCHEMA_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        rec_schema_report,
        report_file,
        indent=2,
    )


# ---------------------------------------------------------
# 13. Update checkpoint atomically
# ---------------------------------------------------------

with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    project_7_checkpoint = json.load(
        checkpoint_file
    )


project_7_checkpoint.update({
    "Status":
        "REC_SCHEMA_INSPECTION_PASSED",

    "RECFeatureCount":
        int(len(REC_COLUMNS)),

    "VerdictDependentRECFeatureCount":
        int(len(REC_DEPENDENT_COLUMNS)),

    "VerdictIndependentRECFeatureCount":
        int(len(REC_INDEPENDENT_COLUMNS)),

    "VerdictDependentRECColumns":
        list(REC_DEPENDENT_COLUMNS),

    "VerdictIndependentRECColumns":
        list(REC_INDEPENDENT_COLUMNS),

    "RECNoisePolicy":
        (
            "Preserve six verdict-independent REC "
            "features and recompute thirteen "
            "verdict-dependent REC features."
        ),

    "RECFeatureNumericConversionFailures":
        int(rec_numeric_conversion_failures),

    "RECFeatureMissingValues":
        int(rec_missing_values),

    "RECFeatureInfiniteValues":
        int(rec_infinite_values),

    "RawUniqueTests":
        int(RAW_UNIQUE_TESTS),

    "ModelReadyUniqueTests":
        int(MODEL_READY_UNIQUE_TESTS),

    "RawTestsWithFailure":
        int(RAW_TESTS_WITH_FAILURE),

    "ModelReadyTestsWithFailure":
        int(MODEL_TESTS_WITH_FAILURE),

    "RawTrainingFailureSubtypeCounts": {
        str(verdict): int(count)
        for verdict, count in (
            RAW_TRAINING_FAILURE_SUBTYPE_COUNTS.items()
        )
    },

    "RawTrainingFailureSubtypeProbabilities": {
        str(verdict): float(probability)
        for verdict, probability in (
            RAW_TRAINING_FAILURE_SUBTYPE_PROBABILITIES.items()
        )
    },

    "RECSchemaInspectionReport":
        str(REC_SCHEMA_REPORT_PATH),

    "RECFeatureProfile":
        str(REC_FEATURE_PROFILE_PATH),

    "RECFeatureClassification":
        str(REC_CLASSIFICATION_PATH),

    "VerdictSubtypeProfile":
        str(VERDICT_SUBTYPE_PROFILE_PATH),

    "TestHistoryProfile":
        str(TEST_HISTORY_PROFILE_PATH),

    "RECSampleTrajectories":
        str(REC_SAMPLE_TRAJECTORIES_PATH),

    "UpdatedAtUTC":
        str(completed_at),
})


temporary_checkpoint_path = (
    PROJECT_7_SELECTION_CHECKPOINT
    .with_suffix(".json.tmp")
)


with open(
    temporary_checkpoint_path,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        project_7_checkpoint,
        checkpoint_file,
        indent=2,
    )


os.replace(
    temporary_checkpoint_path,
    PROJECT_7_SELECTION_CHECKPOINT,
)


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint_verification = json.load(
        checkpoint_file
    )


if checkpoint_verification.get(
    "Status"
) != "REC_SCHEMA_INSPECTION_PASSED":
    raise AssertionError(
        "The Project 7 REC schema checkpoint was not "
        "written correctly."
    )


if int(
    checkpoint_verification.get(
        "RECFeatureCount",
        -1,
    )
) != 19:
    raise AssertionError(
        "The checkpoint contains the wrong REC count."
    )


if int(
    checkpoint_verification.get(
        "VerdictDependentRECFeatureCount",
        -1,
    )
) != 13:
    raise AssertionError(
        "The checkpoint contains the wrong dependent "
        "REC count."
    )


if int(
    checkpoint_verification.get(
        "VerdictIndependentRECFeatureCount",
        -1,
    )
) != 6:
    raise AssertionError(
        "The checkpoint contains the wrong independent "
        "REC count."
    )


# ---------------------------------------------------------
# 14. Compact final output
# ---------------------------------------------------------

clear_output(wait=True)


print(
    "=== PROJECT 7 STEP 5 RESULT ==="
)


print(
    "\nProject-specific REC structure:"
)

print(
    "Total REC features:",
    len(REC_COLUMNS)
)

print(
    "Verdict-dependent REC features:",
    len(REC_DEPENDENT_COLUMNS)
)

print(
    "Verdict-independent REC features:",
    len(REC_INDEPENDENT_COLUMNS)
)


print(
    "\nVerdict-independent REC features:"
)

display(
    rec_feature_classification[
        rec_feature_classification[
            "ProtocolClass"
        ] == "VERDICT_INDEPENDENT"
    ][
        [
            "Feature",
            "ProtocolClass",
            "NoisePolicy",
        ]
    ]
)


print(
    "\nVerdict-dependent REC features:"
)

display(
    rec_feature_classification[
        rec_feature_classification[
            "ProtocolClass"
        ] == "VERDICT_DEPENDENT"
    ][
        [
            "Feature",
            "ProtocolClass",
            "NoisePolicy",
        ]
    ]
)


print(
    "\nREC data quality:"
)

print(
    "Numeric conversion failures:",
    rec_numeric_conversion_failures
)

print(
    "Missing values:",
    rec_missing_values
)

print(
    "Infinite values:",
    rec_infinite_values
)


print(
    "\nRaw training failure subtype distribution:"
)

display(
    failure_subtype_display
)

print(
    "Raw training failure executions:",
    RAW_TRAINING_FAILURE_TOTAL
)


print(
    "\nTest-history coverage:"
)

print(
    "Raw unique tests:",
    RAW_UNIQUE_TESTS
)

print(
    "Model-ready unique tests:",
    MODEL_READY_UNIQUE_TESTS
)

print(
    "Raw tests with failures:",
    RAW_TESTS_WITH_FAILURE
)

print(
    "Model-ready tests with failures:",
    MODEL_TESTS_WITH_FAILURE
)

print(
    "Tests with unretained raw rows:",
    TESTS_WITH_UNRETAINED_RAW_ROWS
)


print(
    "\nCanonical chronological views:"
)

print(
    "Raw chronological rows:",
    len(raw_history_chronological)
)

print(
    "Model chronological rows:",
    len(model_history_chronological)
)


print(
    "\nRepresentative REC trajectories:"
)

print(
    "Selected tests:",
    trajectory_candidate_tests
)

print(
    "Saved trajectory rows:",
    len(rec_sample_trajectories)
)


print(
    "\nREC schema report:"
)

print(
    REC_SCHEMA_REPORT_PATH
)


print(
    "\nValidation status:",
    "PASS"
)


print(
    "\nSUCCESS: All 19 Beast2 REC features were "
    "identified."
)

print(
    "SUCCESS: The thirteen verdict-dependent and six "
    "verdict-independent features were separated."
)

print(
    "SUCCESS: Every REC value was numeric, present and "
    "finite."
)

print(
    "SUCCESS: Raw training failure subtype counts and "
    "probabilities were frozen."
)

print(
    "SUCCESS: Canonical chronological raw and "
    "model-ready history views were created."
)

print(
    "SUCCESS: No REC feature or verdict was modified."
)

print(
    "SUCCESS: Project 7 is ready for clean REC "
    "reconstruction and validation."
)

=== PROJECT 7 STEP 5 RESULT ===

Project-specific REC structure:
Total REC features: 19
Verdict-dependent REC features: 13
Verdict-independent REC features: 6

Verdict-independent REC features:


,Feature,ProtocolClass,NoisePolicy
0,REC_Age,VERDICT_INDEPENDENT,PRESERVE_ORIGINAL_TCP_CI_VALUE
3,REC_RecentAvgExeTime,VERDICT_INDEPENDENT,PRESERVE_ORIGINAL_TCP_CI_VALUE
4,REC_RecentMaxExeTime,VERDICT_INDEPENDENT,PRESERVE_ORIGINAL_TCP_CI_VALUE
9,REC_TotalAvgExeTime,VERDICT_INDEPENDENT,PRESERVE_ORIGINAL_TCP_CI_VALUE
10,REC_TotalMaxExeTime,VERDICT_INDEPENDENT,PRESERVE_ORIGINAL_TCP_CI_VALUE
16,REC_LastExeTime,VERDICT_INDEPENDENT,PRESERVE_ORIGINAL_TCP_CI_VALUE



Verdict-dependent REC features:


,Feature,ProtocolClass,NoisePolicy
1,REC_LastFailureAge,VERDICT_DEPENDENT,RECOMPUTE_FROM_CORRUPTED_RAW_HISTORY
2,REC_LastTransitionAge,VERDICT_DEPENDENT,RECOMPUTE_FROM_CORRUPTED_RAW_HISTORY
5,REC_RecentFailRate,VERDICT_DEPENDENT,RECOMPUTE_FROM_CORRUPTED_RAW_HISTORY
6,REC_RecentAssertRate,VERDICT_DEPENDENT,RECOMPUTE_FROM_CORRUPTED_RAW_HISTORY
7,REC_RecentExcRate,VERDICT_DEPENDENT,RECOMPUTE_FROM_CORRUPTED_RAW_HISTORY
8,REC_RecentTransitionRate,VERDICT_DEPENDENT,RECOMPUTE_FROM_CORRUPTED_RAW_HISTORY
11,REC_TotalFailRate,VERDICT_DEPENDENT,RECOMPUTE_FROM_CORRUPTED_RAW_HISTORY
12,REC_TotalAssertRate,VERDICT_DEPENDENT,RECOMPUTE_FROM_CORRUPTED_RAW_HISTORY
13,REC_TotalExcRate,VERDICT_DEPENDENT,RECOMPUTE_FROM_CORRUPTED_RAW_HISTORY
14,REC_TotalTransitionRate,VERDICT_DEPENDENT,RECOMPUTE_FROM_CORRUPTED_RAW_HISTORY



REC data quality:
Numeric conversion failures: 0
Missing values: 0
Infinite values: 0

Raw training failure subtype distribution:


,VerdictSubtype,Count,Probability
0,1,36,0.208092
1,2,137,0.791908


Raw training failure executions: 173

Test-history coverage:
Raw unique tests: 71
Model-ready unique tests: 71
Raw tests with failures: 22
Model-ready tests with failures: 22
Tests with unretained raw rows: 71

Canonical chronological views:
Raw chronological rows: 27083
Model chronological rows: 7638

Representative REC trajectories:
Selected tests: ['1135', '1120', '1125', '1121', '1253', '378', '1123', '379', '1184', '1614']
Saved trajectory rows: 40

REC schema report:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_preflight/beast2_rec_schema_inspection_report.json

Validation status: PASS

SUCCESS: All 19 Beast2 REC features were identified.
SUCCESS: The thirteen verdict-dependent and six verdict-independent features were separated.
SUCCESS: Every REC value was numeric, present and finite.
SUCCESS: Raw training failure subtype counts and probabilities were frozen.
SUCCESS: Canonical chronological raw and model-ready history views were created.


In [ ]:
# =========================================================
# PROJECT 7 — STEP 6A
# CLEAN REC RECONSTRUCTION AUDIT
# =========================================================

import json
import os
import numpy as np
import pandas as pd
from IPython.display import clear_output, display

print(
    "=== PROJECT 7 STEP 6A: "
    "CLEAN REC RECONSTRUCTION AUDIT ==="
)

required = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",
    "PROJECT_PREFLIGHT_DIRECTORY",
    "PROJECT_7_SELECTION_CHECKPOINT",
    "chronological_builds",
    "exe_work",
    "dataset_work",
    "REC_COLUMNS",
    "REC_DEPENDENT_COLUMNS",
    "REC_INDEPENDENT_COLUMNS",
    "changed_entity_dictionary",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing Step 5 objects:\n"
        + "\n".join(missing)
        + "\n\nRerun only the successful "
        "Project 7 Step 5 cell."
    )


if (
    PROJECT_NUMBER != 7
    or PROJECT_NAME != "CompEvol@beast2"
):
    raise AssertionError(
        "Unexpected Project 7 identity."
    )


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint = json.load(
        checkpoint_file
    )


if checkpoint.get(
    "Status"
) not in {
    "REC_SCHEMA_INSPECTION_PASSED",
    "CLEAN_REC_VALIDATION_PASSED",
}:
    raise AssertionError(
        "Step 5 has not passed. Observed status: "
        + str(
            checkpoint.get(
                "Status"
            )
        )
    )


expected_dependent = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]


expected_independent = {
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
}


if list(
    REC_DEPENDENT_COLUMNS
) != expected_dependent:
    raise AssertionError(
        "Unexpected dependent REC feature order."
    )


if set(
    REC_INDEPENDENT_COLUMNS
) != expected_independent:
    raise AssertionError(
        "Unexpected independent REC feature set."
    )


history_clean = (
    exe_work
    .copy()
    .reset_index(drop=True)
)


model_clean = (
    dataset_work
    .copy()
    .reset_index(drop=True)
)


for frame_name, frame in [
    (
        "history",
        history_clean,
    ),
    (
        "model",
        model_clean,
    ),
]:
    frame[
        "Build"
    ] = pd.to_numeric(
        frame[
            "Build"
        ],
        errors="raise",
    ).astype(np.int64)


    frame[
        "build_order"
    ] = pd.to_numeric(
        frame[
            "build_order"
        ],
        errors="raise",
    ).astype(np.int64)


    frame[
        "Verdict"
    ] = pd.to_numeric(
        frame[
            "Verdict"
        ],
        errors="raise",
    ).astype(np.int64)


    frame[
        "Duration"
    ] = pd.to_numeric(
        frame[
            "Duration"
        ],
        errors="raise",
    ).astype(float)


    frame[
        "Test"
    ] = frame[
        "Test"
    ].astype(str)


    if frame[
        [
            "Build",
            "Test",
        ]
    ].duplicated().any():
        raise AssertionError(
            f"{frame_name} has duplicate "
            "Build-Test keys."
        )


model_key_to_index = {
    (
        int(
            row.Build
        ),
        str(
            row.Test
        ),
    ):
        int(
            index
        )

    for index, row in (
        model_clean[
            [
                "Build",
                "Test",
            ]
        ].iterrows()
    )
}


chronology = (
    chronological_builds[
        [
            "Build",
            "build_order",
        ]
    ]
    .copy()
    .sort_values(
        "build_order",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


chronology[
    "Build"
] = chronology[
    "Build"
].astype(np.int64)


chronology[
    "build_order"
] = chronology[
    "build_order"
].astype(np.int64)


def round6(
    value,
):
    return float(
        np.round(
            float(
                value
            ),
            6,
        )
    )


def reconstruct_project_7_raw_rec(
    history_frame,
):
    history = (
        history_frame
        .copy()
        .reset_index(drop=True)
    )


    history[
        "Build"
    ] = pd.to_numeric(
        history[
            "Build"
        ],
        errors="raise",
    ).astype(np.int64)


    history[
        "build_order"
    ] = pd.to_numeric(
        history[
            "build_order"
        ],
        errors="raise",
    ).astype(np.int64)


    history[
        "Verdict"
    ] = pd.to_numeric(
        history[
            "Verdict"
        ],
        errors="raise",
    ).astype(np.int64)


    history[
        "Duration"
    ] = pd.to_numeric(
        history[
            "Duration"
        ],
        errors="raise",
    ).astype(float)


    history[
        "Test"
    ] = history[
        "Test"
    ].astype(str)


    if history[
        [
            "Build",
            "Test",
        ]
    ].duplicated().any():
        raise AssertionError(
            "History has duplicate Build-Test keys."
        )


    rows_by_build = {
        int(
            build
        ):
            (
                group
                .sort_values(
                    "Test",
                    kind="mergesort",
                )
                .copy()
            )

        for build, group in (
            history.groupby(
                "Build",
                sort=False,
            )
        )
    }


    result = {
        feature:
            np.full(
                len(
                    model_clean
                ),
                np.nan,
                dtype=float,
            )

        for feature in REC_COLUMNS
    }


    state_by_test = {}

    file_failure_counts = {}

    file_transition_counts = {}


    for build_row in chronology.itertuples(
        index=False
    ):
        build = int(
            build_row.Build
        )

        order = int(
            build_row.build_order
        )

        build_rows = rows_by_build.get(
            build
        )


        if build_rows is None:
            continue


        changed_entities = tuple(
            int(
                entity
            )

            for entity in (
                changed_entity_dictionary.get(
                    build,
                    tuple(),
                )
            )
        )


        # Calculate features using history strictly
        # before the current build.
        for row in build_rows.itertuples(
            index=False
        ):
            test = str(
                row.Test
            )


            output_index = (
                model_key_to_index.get(
                    (
                        build,
                        test,
                    )
                )
            )


            if output_index is None:
                continue


            state = state_by_test.get(
                test
            )


            if state is None:
                records = []

                first_order = order

                last_failure_order = None

                last_transition_order = None

                last_verdict = 0

                last_duration = 0.0

                failures = 0

                assertions = 0

                exceptions = 0

                transitions = 0

            else:
                records = state[
                    "records"
                ]

                first_order = state[
                    "first_order"
                ]

                last_failure_order = state[
                    "last_failure_order"
                ]

                last_transition_order = state[
                    "last_transition_order"
                ]

                last_verdict = state[
                    "last_verdict"
                ]

                last_duration = state[
                    "last_duration"
                ]

                failures = state[
                    "failures"
                ]

                assertions = state[
                    "assertions"
                ]

                exceptions = state[
                    "exceptions"
                ]

                transitions = state[
                    "transitions"
                ]


            # The previous six chronological build
            # positions are order-6 through order-1.
            recent = [
                record
                for record in records
                if record[0] >= order - 6
            ]


            total_count = len(
                records
            )

            recent_count = len(
                recent
            )


            total_durations = [
                record[2]
                for record in records
            ]


            recent_durations = [
                record[2]
                for record in recent
            ]


            recent_verdicts = [
                record[1]
                for record in recent
            ]


            recent_transitions = [
                record[3]
                for record in recent
            ]


            result[
                "REC_Age"
            ][
                output_index
            ] = (
                order
                - first_order
            )


            result[
                "REC_LastFailureAge"
            ][
                output_index
            ] = (
                -1
                if last_failure_order is None
                else (
                    order
                    - int(
                        last_failure_order
                    )
                    - 1
                )
            )


            result[
                "REC_LastTransitionAge"
            ][
                output_index
            ] = (
                -1
                if last_transition_order is None
                else (
                    order
                    - int(
                        last_transition_order
                    )
                    - 1
                )
            )


            result[
                "REC_RecentAvgExeTime"
            ][
                output_index
            ] = (
                0.0
                if recent_count == 0
                else round6(
                    np.mean(
                        recent_durations
                    )
                )
            )


            result[
                "REC_RecentMaxExeTime"
            ][
                output_index
            ] = (
                0.0
                if recent_count == 0
                else float(
                    np.max(
                        recent_durations
                    )
                )
            )


            result[
                "REC_RecentFailRate"
            ][
                output_index
            ] = (
                0.0
                if recent_count == 0
                else round6(
                    sum(
                        verdict != 0
                        for verdict
                        in recent_verdicts
                    )
                    / recent_count
                )
            )


            result[
                "REC_RecentAssertRate"
            ][
                output_index
            ] = (
                0.0
                if recent_count == 0
                else round6(
                    sum(
                        verdict == 2
                        for verdict
                        in recent_verdicts
                    )
                    / recent_count
                )
            )


            result[
                "REC_RecentExcRate"
            ][
                output_index
            ] = (
                0.0
                if recent_count == 0
                else round6(
                    sum(
                        verdict == 1
                        for verdict
                        in recent_verdicts
                    )
                    / recent_count
                )
            )


            result[
                "REC_RecentTransitionRate"
            ][
                output_index
            ] = (
                0.0
                if recent_count == 0
                else round6(
                    sum(
                        recent_transitions
                    )
                    / recent_count
                )
            )


            result[
                "REC_TotalAvgExeTime"
            ][
                output_index
            ] = (
                0.0
                if total_count == 0
                else round6(
                    np.mean(
                        total_durations
                    )
                )
            )


            result[
                "REC_TotalMaxExeTime"
            ][
                output_index
            ] = (
                0.0
                if total_count == 0
                else float(
                    np.max(
                        total_durations
                    )
                )
            )


            result[
                "REC_TotalFailRate"
            ][
                output_index
            ] = (
                0.0
                if total_count == 0
                else round6(
                    failures
                    / total_count
                )
            )


            result[
                "REC_TotalAssertRate"
            ][
                output_index
            ] = (
                0.0
                if total_count == 0
                else round6(
                    assertions
                    / total_count
                )
            )


            result[
                "REC_TotalExcRate"
            ][
                output_index
            ] = (
                0.0
                if total_count == 0
                else round6(
                    exceptions
                    / total_count
                )
            )


            result[
                "REC_TotalTransitionRate"
            ][
                output_index
            ] = (
                0.0
                if total_count == 0
                else round6(
                    transitions
                    / total_count
                )
            )


            result[
                "REC_LastVerdict"
            ][
                output_index
            ] = last_verdict


            result[
                "REC_LastExeTime"
            ][
                output_index
            ] = last_duration


            if failures == 0:
                result[
                    "REC_MaxTestFileFailRate"
                ][
                    output_index
                ] = -1.0

            else:
                numerator = max(
                    [
                        file_failure_counts.get(
                            (
                                test,
                                entity,
                            ),
                            0,
                        )

                        for entity in (
                            changed_entities
                        )
                    ]
                    or [
                        0
                    ]
                )


                result[
                    "REC_MaxTestFileFailRate"
                ][
                    output_index
                ] = round6(
                    numerator
                    / failures
                )


            if transitions == 0:
                result[
                    "REC_MaxTestFileTransitionRate"
                ][
                    output_index
                ] = -1.0

            else:
                numerator = max(
                    [
                        file_transition_counts.get(
                            (
                                test,
                                entity,
                            ),
                            0,
                        )

                        for entity in (
                            changed_entities
                        )
                    ]
                    or [
                        0
                    ]
                )


                result[
                    "REC_MaxTestFileTransitionRate"
                ][
                    output_index
                ] = round6(
                    numerator
                    / transitions
                )


        # Update history after every feature for this
        # build has been calculated.
        for row in build_rows.itertuples(
            index=False
        ):
            test = str(
                row.Test
            )

            verdict = int(
                row.Verdict
            )

            duration = float(
                row.Duration
            )

            failed = (
                verdict != 0
            )

            state = state_by_test.get(
                test
            )


            if state is None:
                transitioned = False

                state = {
                    "records":
                        [],

                    "first_order":
                        order,

                    "last_failure_order":
                        None,

                    "last_transition_order":
                        None,

                    "last_verdict":
                        0,

                    "last_failed":
                        False,

                    "last_duration":
                        0.0,

                    "failures":
                        0,

                    "assertions":
                        0,

                    "exceptions":
                        0,

                    "transitions":
                        0,
                }

                state_by_test[
                    test
                ] = state

            else:
                transitioned = bool(
                    failed
                    != state[
                        "last_failed"
                    ]
                )


            if failed:
                state[
                    "failures"
                ] += 1

                state[
                    "last_failure_order"
                ] = order


            if verdict == 2:
                state[
                    "assertions"
                ] += 1


            if verdict == 1:
                state[
                    "exceptions"
                ] += 1


            if transitioned:
                state[
                    "transitions"
                ] += 1

                state[
                    "last_transition_order"
                ] = order


            state[
                "records"
            ].append(
                (
                    order,
                    verdict,
                    duration,
                    int(
                        transitioned
                    ),
                )
            )


            state[
                "last_verdict"
            ] = verdict


            state[
                "last_failed"
            ] = failed


            state[
                "last_duration"
            ] = duration


            if failed:
                for entity in changed_entities:
                    key = (
                        test,
                        entity,
                    )

                    file_failure_counts[
                        key
                    ] = (
                        file_failure_counts.get(
                            key,
                            0,
                        )
                        + 1
                    )


            if transitioned:
                for entity in changed_entities:
                    key = (
                        test,
                        entity,
                    )

                    file_transition_counts[
                        key
                    ] = (
                        file_transition_counts.get(
                            key,
                            0,
                        )
                        + 1
                    )


    reconstruction = pd.DataFrame(
        result,
        index=model_clean.index,
    )


    if reconstruction.isna().any().any():
        raise AssertionError(
            "REC reconstruction has missing values: "
            + str(
                int(
                    reconstruction
                    .isna()
                    .sum()
                    .sum()
                )
            )
        )


    return reconstruction


def mismatch_mask(
    source,
    reconstructed,
    tolerance,
):
    return ~np.isclose(
        np.asarray(
            source,
            dtype=float,
        ),
        np.asarray(
            reconstructed,
            dtype=float,
        ),
        rtol=0.0,
        atol=float(
            tolerance
        ),
        equal_nan=True,
    )


raw_clean_rec_reconstruction = (
    reconstruct_project_7_raw_rec(
        history_clean
    )
)


integer_features = {
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_LastVerdict",
}


summary_rows = []

detail_frames = []


for feature in REC_COLUMNS:
    tolerance = (
        1e-12
        if feature in integer_features
        else 5e-7
    )


    authoritative = model_clean[
        feature
    ].to_numpy(
        dtype=float
    )


    reconstructed = (
        raw_clean_rec_reconstruction[
            feature
        ].to_numpy(
            dtype=float
        )
    )


    mask = mismatch_mask(
        authoritative,
        reconstructed,
        tolerance,
    )


    difference = (
        authoritative
        - reconstructed
    )


    summary_rows.append({
        "Feature":
            feature,

        "ProtocolClass":
            (
                "VERDICT_DEPENDENT"
                if feature in set(
                    REC_DEPENDENT_COLUMNS
                )
                else "VERDICT_INDEPENDENT"
            ),

        "RowsChecked":
            int(
                len(
                    model_clean
                )
            ),

        "Mismatches":
            int(
                mask.sum()
            ),

        "MismatchPercent":
            float(
                100.0
                * mask.mean()
            ),

        "MaximumAbsoluteDifference":
            float(
                np.max(
                    np.abs(
                        difference
                    )
                )
            ),

        "MeanAbsoluteDifference":
            float(
                np.mean(
                    np.abs(
                        difference
                    )
                )
            ),

        "Tolerance":
            float(
                tolerance
            ),
    })


    if mask.any():
        detail = model_clean.loc[
            mask,
            [
                "Build",
                "Test",
                "build_order",
                "partition",
                "Verdict",
            ],
        ].copy()


        detail[
            "Feature"
        ] = feature


        detail[
            "AuthoritativeValue"
        ] = authoritative[
            mask
        ]


        detail[
            "RawReconstructedValue"
        ] = reconstructed[
            mask
        ]


        detail[
            "CleanOffset"
        ] = difference[
            mask
        ]


        detail_frames.append(
            detail
        )


raw_audit_summary = pd.DataFrame(
    summary_rows
)


raw_mismatch_details = (
    pd.concat(
        detail_frames,
        ignore_index=True,
    )
    if detail_frames
    else pd.DataFrame()
)


dependent_summary = raw_audit_summary[
    raw_audit_summary[
        "ProtocolClass"
    ] == "VERDICT_DEPENDENT"
]


independent_summary = raw_audit_summary[
    raw_audit_summary[
        "ProtocolClass"
    ] == "VERDICT_INDEPENDENT"
]


dependent_values_checked = int(
    len(
        model_clean
    )
    * len(
        REC_DEPENDENT_COLUMNS
    )
)


dependent_raw_mismatches = int(
    dependent_summary[
        "Mismatches"
    ].sum()
)


dependent_raw_mismatch_percent = float(
    100.0
    * dependent_raw_mismatches
    / dependent_values_checked
)


independent_values_checked = int(
    len(
        model_clean
    )
    * len(
        REC_INDEPENDENT_COLUMNS
    )
)


independent_raw_mismatches = int(
    independent_summary[
        "Mismatches"
    ].sum()
)


if dependent_raw_mismatch_percent > 5.0:
    display(
        raw_audit_summary
    )


    if len(
        raw_mismatch_details
    ):
        display(
            raw_mismatch_details.head(
                100
            )
        )


    raise AssertionError(
        "Dependent REC raw mismatch rate exceeds 5%. "
        "The clean anchor was not applied."
    )


# Save the audit only. The Project 7 checkpoint is not
# changed by Step 6A.
audit_summary_path = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_clean_rec_raw_audit_summary.csv"
)


audit_details_path = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_clean_rec_raw_mismatches.csv"
)


raw_reconstruction_path = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_clean_raw_rec_reconstruction.parquet"
)


raw_audit_summary.to_csv(
    audit_summary_path,
    index=False,
)


raw_mismatch_details.to_csv(
    audit_details_path,
    index=False,
)


keys = (
    model_clean[
        [
            "Build",
            "Test",
            "build_order",
            "partition",
        ]
    ]
    .reset_index(drop=True)
)


pd.concat(
    [
        keys,
        raw_clean_rec_reconstruction
        .reset_index(drop=True),
    ],
    axis=1,
).to_parquet(
    raw_reconstruction_path,
    index=False,
)


clear_output(
    wait=True
)


print(
    "=== PROJECT 7 STEP 6A RESULT ==="
)


print(
    "\nRaw clean REC reconstruction audit:"
)


display(
    raw_audit_summary
)


print(
    "Dependent REC values checked:",
    dependent_values_checked
)


print(
    "Dependent raw mismatches:",
    dependent_raw_mismatches
)


print(
    "Dependent raw mismatch percent:",
    f"{dependent_raw_mismatch_percent:.6f}%"
)


print(
    "Independent REC values checked:",
    independent_values_checked
)


print(
    "Independent raw audit mismatches:",
    independent_raw_mismatches
)


if len(
    raw_mismatch_details
):
    print(
        "\nFirst reconstruction mismatches:"
    )

    display(
        raw_mismatch_details.head(
            100
        )
    )

else:
    print(
        "\nReconstruction mismatches: None"
    )


print(
    "\nSaved audit summary:"
)


print(
    audit_summary_path
)


print(
    "Saved raw reconstruction:"
)


print(
    raw_reconstruction_path
)


print(
    "\nAudit status:",
    "COMPLETE"
)


print(
    "\nSUCCESS: Beast2 clean REC values were "
    "independently reconstructed from raw history."
)


print(
    "SUCCESS: No source value, verdict, or execution "
    "row was modified."
)


print(
    "SUCCESS: The audit is ready for clean-anchor "
    "policy selection."
)

=== PROJECT 7 STEP 6A RESULT ===

Raw clean REC reconstruction audit:


,Feature,ProtocolClass,RowsChecked,Mismatches,MismatchPercent,MaximumAbsoluteDifference,MeanAbsoluteDifference,Tolerance
0,REC_Age,VERDICT_INDEPENDENT,7638,0,0.000000,0.000000e+00,0.000000e+00,1.000000e-12
1,REC_LastFailureAge,VERDICT_DEPENDENT,7638,5,0.065462,1.000000e+00,6.546216e-04,1.000000e-12
2,REC_LastTransitionAge,VERDICT_DEPENDENT,7638,7,0.091647,2.000000e+00,1.047395e-03,1.000000e-12
3,REC_RecentAvgExeTime,VERDICT_INDEPENDENT,7638,10,0.130924,2.187500e+02,6.377127e-02,5.000000e-07
4,REC_RecentMaxExeTime,VERDICT_INDEPENDENT,7638,6,0.078555,8.860000e+02,2.395915e-01,5.000000e-07
5,REC_RecentFailRate,VERDICT_DEPENDENT,7638,3,0.039277,1.000000e+00,3.927823e-04,5.000000e-07
6,REC_RecentAssertRate,VERDICT_DEPENDENT,7638,3,0.039277,1.000000e+00,3.927794e-04,5.000000e-07
7,REC_RecentExcRate,VERDICT_DEPENDENT,7638,3,0.039277,1.000000e+00,3.927778e-04,5.000000e-07
8,REC_RecentTransitionRate,VERDICT_DEPENDENT,7638,16,0.209479,1.000000e+00,7.419144e-04,5.000000e-07
9,REC_TotalAvgExeTime,VERDICT_INDEPENDENT,7638,13,0.170202,1.000000e+00,3.929996e-04,5.000000e-07


Dependent REC values checked: 99294
Dependent raw mismatches: 232
Dependent raw mismatch percent: 0.233650%
Independent REC values checked: 45828
Independent raw audit mismatches: 35

First reconstruction mismatches:


,Build,Test,build_order,partition,Verdict,Feature,AuthoritativeValue,RawReconstructedValue,CleanOffset
0,328867077,1125,57,TRAIN,0,REC_LastFailureAge,48.000000,49.000000,-1.000000
1,328867077,1121,57,TRAIN,2,REC_LastFailureAge,20.000000,21.000000,-1.000000
2,328868486,1125,58,TRAIN,0,REC_LastFailureAge,49.000000,50.000000,-1.000000
3,344145961,1125,74,TRAIN,1,REC_LastFailureAge,65.000000,66.000000,-1.000000
4,542557309,1614,245,TRAIN,2,REC_LastFailureAge,0.000000,1.000000,-1.000000
...,...,...,...,...,...,...,...,...,...
95,543947354,1253,254,TRAIN,2,REC_TotalTransitionRate,0.019763,0.015810,0.003953
96,544036927,1253,256,TRAIN,1,REC_TotalTransitionRate,0.027451,0.023529,0.003922
97,544950109,1253,261,TRAIN,0,REC_TotalTransitionRate,0.034615,0.030769,0.003846
98,546487254,1253,264,TRAIN,0,REC_TotalTransitionRate,0.034221,0.030418,0.003803



Saved audit summary:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_preflight/beast2_clean_rec_raw_audit_summary.csv
Saved raw reconstruction:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_preflight/beast2_clean_raw_rec_reconstruction.parquet

Audit status: COMPLETE

SUCCESS: Beast2 clean REC values were independently reconstructed from raw history.
SUCCESS: No source value, verdict, or execution row was modified.
SUCCESS: The audit is ready for clean-anchor policy selection.


In [ ]:
# =========================================================
# PROJECT 7 — STEP 6B
# FREEZE CLEAN REC ANCHOR AND VALIDATE THE 0% CONDITION
# =========================================================

from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
from IPython.display import clear_output, display


print(
    "=== PROJECT 7 STEP 6B: "
    "FREEZE CLEAN REC ANCHOR ==="
)


# ---------------------------------------------------------
# 1. Confirm the successful Step 6A runtime state
# ---------------------------------------------------------

required_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",
    "PROJECT_PREFLIGHT_DIRECTORY",
    "PROJECT_7_SELECTION_CHECKPOINT",

    "model_clean",
    "clean_training_data",
    "clean_evaluation_data",
    "raw_clean_rec_reconstruction",

    "REC_COLUMNS",
    "REC_DEPENDENT_COLUMNS",
    "REC_INDEPENDENT_COLUMNS",

    "dependent_raw_mismatches",
    "dependent_raw_mismatch_percent",
    "independent_raw_mismatches",
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required Step 6A objects are missing:\n"
        + "\n".join(
            missing_objects
        )
        + "\n\nRerun only the successful Project 7 "
        "Step 6A cell."
    )


if PROJECT_NUMBER != 7:
    raise AssertionError(
        f"Expected Project 7, observed "
        f"{PROJECT_NUMBER}."
    )


if PROJECT_NAME != "CompEvol@beast2":
    raise AssertionError(
        f"Unexpected project: {PROJECT_NAME}"
    )


if PROJECT_SLUG != "CompEvol__beast2":
    raise AssertionError(
        f"Unexpected project slug: {PROJECT_SLUG}"
    )


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    previous_checkpoint = json.load(
        checkpoint_file
    )


if previous_checkpoint.get(
    "Status"
) not in {
    "REC_SCHEMA_INSPECTION_PASSED",
    "CLEAN_REC_VALIDATION_PASSED",
}:
    raise AssertionError(
        "Project 7 Step 5 has not passed. "
        "Observed status: "
        + str(
            previous_checkpoint.get(
                "Status"
            )
        )
    )


if float(
    dependent_raw_mismatch_percent
) > 5.0:
    raise AssertionError(
        "Dependent REC raw mismatch rate exceeds "
        "the accepted 5% audit threshold."
    )


# ---------------------------------------------------------
# 2. Validate and align the clean tables
# ---------------------------------------------------------

model_reference = (
    model_clean
    .copy()
    .reset_index(drop=True)
)


raw_reference = (
    raw_clean_rec_reconstruction
    .copy()
    .reset_index(drop=True)
)


training_reference = (
    clean_training_data
    .copy()
    .reset_index(drop=True)
)


evaluation_reference = (
    clean_evaluation_data
    .copy()
    .reset_index(drop=True)
)


KEY_COLUMNS = [
    "Build",
    "Test",
    "build_order",
    "partition",
]


missing_model_columns = [
    column
    for column in [
        *KEY_COLUMNS,
        *REC_COLUMNS,
    ]
    if column not in model_reference.columns
]


missing_raw_columns = [
    column
    for column in REC_COLUMNS
    if column not in raw_reference.columns
]


if missing_model_columns:
    raise RuntimeError(
        "model_clean is missing columns:\n"
        + "\n".join(
            missing_model_columns
        )
    )


if missing_raw_columns:
    raise RuntimeError(
        "Step 6A reconstruction is missing columns:\n"
        + "\n".join(
            missing_raw_columns
        )
    )


if len(
    model_reference
) != len(
    raw_reference
):
    raise AssertionError(
        "Authoritative and reconstructed clean REC "
        "tables have different row counts."
    )


model_reference[
    "Build"
] = pd.to_numeric(
    model_reference[
        "Build"
    ],
    errors="raise",
).astype(np.int64)


model_reference[
    "Test"
] = (
    model_reference[
        "Test"
    ].astype(str)
)


if model_reference[
    [
        "Build",
        "Test",
    ]
].duplicated().any():
    raise AssertionError(
        "model_clean contains duplicate Build-Test keys."
    )


for feature in REC_COLUMNS:
    model_reference[
        feature
    ] = pd.to_numeric(
        model_reference[
            feature
        ],
        errors="raise",
    ).astype(float)


    raw_reference[
        feature
    ] = pd.to_numeric(
        raw_reference[
            feature
        ],
        errors="raise",
    ).astype(float)


if not np.isfinite(
    model_reference[
        REC_COLUMNS
    ].to_numpy(dtype=float)
).all():
    raise AssertionError(
        "Authoritative REC values are not all finite."
    )


if not np.isfinite(
    raw_reference[
        REC_COLUMNS
    ].to_numpy(dtype=float)
).all():
    raise AssertionError(
        "Reconstructed clean REC values are not all "
        "finite."
    )


# ---------------------------------------------------------
# 3. Freeze the clean anchor
# ---------------------------------------------------------

# For each dependent feature:
#
# anchored noisy =
#     authoritative clean
#     + (raw noisy - raw clean)
#
# The six independent features remain exactly as supplied
# by the TCP-CI dataset.

REC_CLEAN_ANCHOR_POLICY = (
    "AUTHORITATIVE_CLEAN_PLUS_"
    "NOISE_INDUCED_RAW_DELTA"
)


REC_CLEAN_ANCHOR_FORMULA = (
    "anchored_noisy = authoritative_clean + "
    "(raw_noisy_reconstruction - "
    "raw_clean_reconstruction)"
)


authoritative_dependent = (
    model_reference[
        REC_DEPENDENT_COLUMNS
    ]
    .copy()
    .add_prefix(
        "AuthoritativeClean__"
    )
)


raw_clean_dependent = (
    raw_reference[
        REC_DEPENDENT_COLUMNS
    ]
    .copy()
    .add_prefix(
        "RawClean__"
    )
)


offset_dependent = (
    model_reference[
        REC_DEPENDENT_COLUMNS
    ].to_numpy(dtype=float)
    -
    raw_reference[
        REC_DEPENDENT_COLUMNS
    ].to_numpy(dtype=float)
)


offset_dependent = pd.DataFrame(
    offset_dependent,
    columns=[
        f"CleanOffset__{feature}"
        for feature in (
            REC_DEPENDENT_COLUMNS
        )
    ],
)


REC_CLEAN_ANCHOR_REFERENCE = pd.concat(
    [
        model_reference[
            KEY_COLUMNS
        ].reset_index(drop=True),

        authoritative_dependent
        .reset_index(drop=True),

        raw_clean_dependent
        .reset_index(drop=True),

        offset_dependent
        .reset_index(drop=True),
    ],
    axis=1,
)


# ---------------------------------------------------------
# 4. Reusable clean-anchor application helper
# ---------------------------------------------------------

def apply_project_7_clean_rec_anchor(
    requested_model_rows,
    raw_noisy_reconstruction,
):
    """
    Preserve the fixed model-ready rows and all original
    non-dependent columns.

    Apply only the noise-induced raw-history delta to the
    thirteen verdict-dependent REC features.
    """

    requested = (
        requested_model_rows
        .copy()
        .reset_index(drop=True)
    )


    noisy = (
        raw_noisy_reconstruction
        .copy()
        .reset_index(drop=True)
    )


    required_requested = {
        "Build",
        "Test",
        *REC_COLUMNS,
    }


    missing_requested = (
        required_requested
        - set(
            requested.columns
        )
    )


    if missing_requested:
        raise RuntimeError(
            "Requested rows are missing columns:\n"
            + "\n".join(
                sorted(
                    missing_requested
                )
            )
        )


    missing_noisy = [
        feature
        for feature in (
            REC_DEPENDENT_COLUMNS
        )
        if feature not in noisy.columns
    ]


    if missing_noisy:
        raise RuntimeError(
            "Noisy reconstruction is missing dependent "
            "REC columns:\n"
            + "\n".join(
                missing_noisy
            )
        )


    requested[
        "Build"
    ] = pd.to_numeric(
        requested[
            "Build"
        ],
        errors="raise",
    ).astype(np.int64)


    requested[
        "Test"
    ] = requested[
        "Test"
    ].astype(str)


    if requested[
        [
            "Build",
            "Test",
        ]
    ].duplicated().any():
        raise AssertionError(
            "Requested rows contain duplicate "
            "Build-Test keys."
        )


    requested_keys = requested[
        [
            "Build",
            "Test",
        ]
    ].copy()


    requested_keys[
        "_row_order"
    ] = np.arange(
        len(
            requested_keys
        ),
        dtype=np.int64,
    )


    # Align the noisy reconstruction by Build-Test keys
    # whenever those keys are present.
    if {
        "Build",
        "Test",
    }.issubset(
        noisy.columns
    ):
        noisy[
            "Build"
        ] = pd.to_numeric(
            noisy[
                "Build"
            ],
            errors="raise",
        ).astype(np.int64)


        noisy[
            "Test"
        ] = noisy[
            "Test"
        ].astype(str)


        if noisy[
            [
                "Build",
                "Test",
            ]
        ].duplicated().any():
            raise AssertionError(
                "Noisy reconstruction contains duplicate "
                "Build-Test keys."
            )


        noisy_aligned = (
            requested_keys
            .merge(
                noisy[
                    [
                        "Build",
                        "Test",
                        *REC_DEPENDENT_COLUMNS,
                    ]
                ],
                on=[
                    "Build",
                    "Test",
                ],
                how="left",
                validate="one_to_one",
            )
            .sort_values(
                "_row_order",
                kind="mergesort",
            )
            .reset_index(drop=True)
        )


    else:
        if len(
            noisy
        ) != len(
            requested
        ):
            raise AssertionError(
                "Noisy reconstruction row count does "
                "not match requested rows."
            )


        noisy_aligned = pd.concat(
            [
                requested_keys,

                noisy[
                    REC_DEPENDENT_COLUMNS
                ].reset_index(drop=True),
            ],
            axis=1,
        )


    if noisy_aligned[
        REC_DEPENDENT_COLUMNS
    ].isna().any().any():
        raise AssertionError(
            "At least one requested row has no noisy "
            "reconstructed REC value."
        )


    reference_columns = [
        "Build",
        "Test",
    ]


    for feature in REC_DEPENDENT_COLUMNS:
        reference_columns.extend(
            [
                f"AuthoritativeClean__{feature}",
                f"RawClean__{feature}",
            ]
        )


    aligned_reference = (
        requested_keys
        .merge(
            REC_CLEAN_ANCHOR_REFERENCE[
                reference_columns
            ],
            on=[
                "Build",
                "Test",
            ],
            how="left",
            validate="one_to_one",
        )
        .sort_values(
            "_row_order",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    value_columns = [
        column
        for column in (
            aligned_reference.columns
        )
        if column.startswith(
            "AuthoritativeClean__"
        )
        or column.startswith(
            "RawClean__"
        )
    ]


    if aligned_reference[
        value_columns
    ].isna().any().any():
        raise AssertionError(
            "At least one requested key is missing from "
            "the clean anchor."
        )


    anchored = requested.copy()


    for feature in REC_DEPENDENT_COLUMNS:
        authoritative_clean = (
            aligned_reference[
                f"AuthoritativeClean__{feature}"
            ].to_numpy(dtype=float)
        )


        raw_clean = (
            aligned_reference[
                f"RawClean__{feature}"
            ].to_numpy(dtype=float)
        )


        raw_noisy = pd.to_numeric(
            noisy_aligned[
                feature
            ],
            errors="raise",
        ).to_numpy(dtype=float)


        anchored_values = (
            authoritative_clean
            + (
                raw_noisy
                - raw_clean
            )
        )


        if not np.isfinite(
            anchored_values
        ).all():
            raise AssertionError(
                f"Anchored values for {feature} are not "
                "all finite."
            )


        anchored[
            feature
        ] = anchored_values


    # Explicitly verify that independent features stayed
    # unchanged.
    for feature in REC_INDEPENDENT_COLUMNS:
        if not np.array_equal(
            anchored[
                feature
            ].to_numpy(),

            requested[
                feature
            ].to_numpy(),

            equal_nan=True,
        ):
            raise AssertionError(
                f"Independent feature {feature} changed "
                "during anchoring."
            )


    return anchored


# Compatibility alias for later cells.
apply_clean_rec_anchor = (
    apply_project_7_clean_rec_anchor
)


# ---------------------------------------------------------
# 5. Select clean raw reconstruction for any row subset
# ---------------------------------------------------------

def select_clean_raw_reconstruction(
    requested_rows,
):
    requested = (
        requested_rows[
            [
                "Build",
                "Test",
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )


    requested[
        "Build"
    ] = pd.to_numeric(
        requested[
            "Build"
        ],
        errors="raise",
    ).astype(np.int64)


    requested[
        "Test"
    ] = requested[
        "Test"
    ].astype(str)


    requested[
        "_row_order"
    ] = np.arange(
        len(
            requested
        ),
        dtype=np.int64,
    )


    raw_columns = [
        "Build",
        "Test",
    ] + [
        f"RawClean__{feature}"
        for feature in (
            REC_DEPENDENT_COLUMNS
        )
    ]


    rename_map = {
        f"RawClean__{feature}":
            feature
        for feature in (
            REC_DEPENDENT_COLUMNS
        )
    }


    selected = (
        requested
        .merge(
            REC_CLEAN_ANCHOR_REFERENCE[
                raw_columns
            ],
            on=[
                "Build",
                "Test",
            ],
            how="left",
            validate="one_to_one",
        )
        .sort_values(
            "_row_order",
            kind="mergesort",
        )
        .reset_index(drop=True)
        .rename(
            columns=rename_map
        )
    )


    if selected[
        REC_DEPENDENT_COLUMNS
    ].isna().any().any():
        raise AssertionError(
            "Selected clean raw reconstruction has "
            "missing values."
        )


    return selected[
        [
            "Build",
            "Test",
            *REC_DEPENDENT_COLUMNS,
        ]
    ].copy()


# ---------------------------------------------------------
# 6. Exact 0% validation on all retained rows
# ---------------------------------------------------------

full_zero_raw = (
    select_clean_raw_reconstruction(
        model_reference
    )
)


full_zero_anchored = (
    apply_project_7_clean_rec_anchor(
        model_reference,
        full_zero_raw,
    )
)


full_zero_dependent_mismatches = int(
    sum(
        (
            full_zero_anchored[
                feature
            ].to_numpy(dtype=float)
            !=
            model_reference[
                feature
            ].to_numpy(dtype=float)
        ).sum()

        for feature in (
            REC_DEPENDENT_COLUMNS
        )
    )
)


full_zero_independent_mismatches = int(
    sum(
        (
            full_zero_anchored[
                feature
            ].to_numpy(dtype=float)
            !=
            model_reference[
                feature
            ].to_numpy(dtype=float)
        ).sum()

        for feature in (
            REC_INDEPENDENT_COLUMNS
        )
    )
)


if full_zero_dependent_mismatches != 0:
    raise AssertionError(
        "The clean anchor failed exact 0% reproduction "
        "for dependent REC features."
    )


if full_zero_independent_mismatches != 0:
    raise AssertionError(
        "Independent REC features changed during full "
        "0% validation."
    )


# ---------------------------------------------------------
# 7. Exact fixed-instance 0% training validation
# ---------------------------------------------------------

training_zero_raw = (
    select_clean_raw_reconstruction(
        training_reference
    )
)


zero_noise_model_training_data = (
    apply_project_7_clean_rec_anchor(
        training_reference,
        training_zero_raw,
    )
)


training_zero_dependent_mismatches = int(
    sum(
        (
            zero_noise_model_training_data[
                feature
            ].to_numpy(dtype=float)
            !=
            training_reference[
                feature
            ].to_numpy(dtype=float)
        ).sum()

        for feature in (
            REC_DEPENDENT_COLUMNS
        )
    )
)


training_zero_independent_mismatches = int(
    sum(
        (
            zero_noise_model_training_data[
                feature
            ].to_numpy(dtype=float)
            !=
            training_reference[
                feature
            ].to_numpy(dtype=float)
        ).sum()

        for feature in (
            REC_INDEPENDENT_COLUMNS
        )
    )
)


training_zero_label_mismatches = int(
    (
        zero_noise_model_training_data[
            "Verdict"
        ].to_numpy()
        !=
        training_reference[
            "Verdict"
        ].to_numpy()
    ).sum()
)


non_rec_columns = [
    column
    for column in (
        training_reference.columns
    )
    if column not in REC_COLUMNS
]


training_zero_non_rec_columns_changed = int(
    sum(
        not zero_noise_model_training_data[
            column
        ].equals(
            training_reference[
                column
            ]
        )

        for column in non_rec_columns
    )
)


training_rows_preserved = bool(
    len(
        zero_noise_model_training_data
    )
    ==
    len(
        training_reference
    )
)


if training_zero_dependent_mismatches != 0:
    raise AssertionError(
        "0% validation changed dependent training "
        "REC values."
    )


if training_zero_independent_mismatches != 0:
    raise AssertionError(
        "0% validation changed independent training "
        "REC values."
    )


if training_zero_label_mismatches != 0:
    raise AssertionError(
        "0% validation changed training labels."
    )


if training_zero_non_rec_columns_changed != 0:
    raise AssertionError(
        "0% validation changed non-REC training columns."
    )


if not training_rows_preserved:
    raise AssertionError(
        "0% validation changed the fixed training-row "
        "count."
    )


# Verify clean evaluation remains untouched.
def dataframe_hash(
    dataframe,
):
    import hashlib


    hashed_values = (
        pd.util.hash_pandas_object(
            dataframe,
            index=True,
            categorize=True,
        )
        .to_numpy(dtype=np.uint64)
    )


    return hashlib.sha256(
        hashed_values.tobytes()
    ).hexdigest()


evaluation_hash_before = dataframe_hash(
    evaluation_reference
)


evaluation_hash_after = dataframe_hash(
    evaluation_reference.copy(
        deep=True
    )
)


evaluation_unchanged = bool(
    evaluation_hash_before
    == evaluation_hash_after
)


if not evaluation_unchanged:
    raise AssertionError(
        "Clean evaluation data changed unexpectedly."
    )


# ---------------------------------------------------------
# 8. Summarise offsets
# ---------------------------------------------------------

offset_summary_records = []


integer_features = {
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_LastVerdict",
}


for feature in REC_DEPENDENT_COLUMNS:
    offsets = (
        REC_CLEAN_ANCHOR_REFERENCE[
            f"CleanOffset__{feature}"
        ].to_numpy(dtype=float)
    )


    tolerance = (
        1e-12
        if feature in integer_features
        else 5e-7
    )


    nonzero = ~np.isclose(
        offsets,
        0.0,
        rtol=0.0,
        atol=tolerance,
    )


    offset_summary_records.append(
        {
            "Feature":
                feature,

            "Rows":
                int(
                    len(
                        offsets
                    )
                ),

            "NonZeroCleanOffsets":
                int(
                    nonzero.sum()
                ),

            "NonZeroOffsetPercent":
                float(
                    100.0
                    * nonzero.mean()
                ),

            "MinimumOffset":
                float(
                    np.min(
                        offsets
                    )
                ),

            "MaximumOffset":
                float(
                    np.max(
                        offsets
                    )
                ),

            "MeanAbsoluteOffset":
                float(
                    np.mean(
                        np.abs(
                            offsets
                        )
                    )
                ),

            "MaximumAbsoluteOffset":
                float(
                    np.max(
                        np.abs(
                            offsets
                        )
                    )
                ),
        }
    )


clean_anchor_offset_summary = pd.DataFrame(
    offset_summary_records
)


TOTAL_DEPENDENT_ANCHOR_VALUES = int(
    len(
        model_reference
    )
    * len(
        REC_DEPENDENT_COLUMNS
    )
)


TOTAL_NONZERO_DEPENDENT_OFFSETS = int(
    clean_anchor_offset_summary[
        "NonZeroCleanOffsets"
    ].sum()
)


zero_validation_summary = pd.DataFrame(
    [
        {
            "ValidationScope":
                "ALL_MODEL_READY_ROWS",

            "Rows":
                int(
                    len(
                        model_reference
                    )
                ),

            "DependentRECMismatches":
                full_zero_dependent_mismatches,

            "IndependentRECMismatches":
                full_zero_independent_mismatches,

            "LabelMismatches":
                0,

            "NonRECColumnsChanged":
                0,

            "RowsPreserved":
                True,
        },

        {
            "ValidationScope":
                "TRAINING_MODEL_READY_ROWS",

            "Rows":
                int(
                    len(
                        training_reference
                    )
                ),

            "DependentRECMismatches":
                training_zero_dependent_mismatches,

            "IndependentRECMismatches":
                training_zero_independent_mismatches,

            "LabelMismatches":
                training_zero_label_mismatches,

            "NonRECColumnsChanged":
                training_zero_non_rec_columns_changed,

            "RowsPreserved":
                training_rows_preserved,
        },
    ]
)


# ---------------------------------------------------------
# 9. Save permanent Step 6B artefacts
# ---------------------------------------------------------

CLEAN_REC_ANCHOR_REFERENCE_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_clean_rec_anchor_reference.parquet"
)


CLEAN_REC_ANCHOR_OFFSET_SUMMARY_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_clean_rec_anchor_offset_summary.csv"
)


CLEAN_REC_ZERO_VALIDATION_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_clean_rec_zero_percent_validation.csv"
)


CLEAN_REC_ANCHOR_POLICY_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_clean_rec_anchor_policy.json"
)


REC_CLEAN_ANCHOR_REFERENCE.to_parquet(
    CLEAN_REC_ANCHOR_REFERENCE_PATH,
    index=False,
)


clean_anchor_offset_summary.to_csv(
    CLEAN_REC_ANCHOR_OFFSET_SUMMARY_PATH,
    index=False,
)


zero_validation_summary.to_csv(
    CLEAN_REC_ZERO_VALIDATION_PATH,
    index=False,
)


completed_at = (
    pd.Timestamp.utcnow().isoformat()
)


clean_anchor_policy = {
    "ProjectNumber":
        int(
            PROJECT_NUMBER
        ),

    "Project":
        str(
            PROJECT_NAME
        ),

    "ProjectSlug":
        str(
            PROJECT_SLUG
        ),

    "Status":
        "PASS",

    "PolicyName":
        REC_CLEAN_ANCHOR_POLICY,

    "Formula":
        REC_CLEAN_ANCHOR_FORMULA,

    "VerdictDependentRECFeatures":
        list(
            REC_DEPENDENT_COLUMNS
        ),

    "VerdictIndependentRECFeatures":
        list(
            REC_INDEPENDENT_COLUMNS
        ),

    "DependentPolicy":
        (
            "Apply only the noise-induced raw "
            "reconstruction delta to the authoritative "
            "TCP-CI clean value."
        ),

    "IndependentPolicy":
        (
            "Preserve original TCP-CI values exactly."
        ),

    "EvaluationPolicy":
        (
            "Keep evaluation data clean and untouched."
        ),

    "Step6AAudit": {
        "DependentRawMismatches":
            int(
                dependent_raw_mismatches
            ),

        "DependentRawMismatchPercent":
            float(
                dependent_raw_mismatch_percent
            ),

        "IndependentRawAuditMismatches":
            int(
                independent_raw_mismatches
            ),
    },

    "AnchorOffsets": {
        "TotalDependentAnchorValues":
            TOTAL_DEPENDENT_ANCHOR_VALUES,

        "NonZeroDependentOffsets":
            TOTAL_NONZERO_DEPENDENT_OFFSETS,
    },

    "ZeroPercentValidation": {
        "AllModelReadyDependentRECMismatches":
            full_zero_dependent_mismatches,

        "AllModelReadyIndependentRECMismatches":
            full_zero_independent_mismatches,

        "TrainingDependentRECMismatches":
            training_zero_dependent_mismatches,

        "TrainingIndependentRECMismatches":
            training_zero_independent_mismatches,

        "TrainingLabelMismatches":
            training_zero_label_mismatches,

        "TrainingNonRECColumnsChanged":
            training_zero_non_rec_columns_changed,

        "TrainingRowsPreserved":
            training_rows_preserved,

        "EvaluationUnchanged":
            evaluation_unchanged,

        "ExactReproductionPassed":
            True,
    },

    "Artefacts": {
        "AnchorReference":
            str(
                CLEAN_REC_ANCHOR_REFERENCE_PATH
            ),

        "OffsetSummary":
            str(
                CLEAN_REC_ANCHOR_OFFSET_SUMMARY_PATH
            ),

        "ZeroPercentValidation":
            str(
                CLEAN_REC_ZERO_VALIDATION_PATH
            ),
    },

    "CompletedAtUTC":
        completed_at,
}


with open(
    CLEAN_REC_ANCHOR_POLICY_PATH,
    "w",
    encoding="utf-8",
) as policy_file:
    json.dump(
        clean_anchor_policy,
        policy_file,
        indent=2,
    )


# ---------------------------------------------------------
# 10. Update the Project 7 checkpoint atomically
# ---------------------------------------------------------

with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    project_7_checkpoint = json.load(
        checkpoint_file
    )


project_7_checkpoint.update(
    {
        "Status":
            "CLEAN_REC_VALIDATION_PASSED",

        "CleanRECAnchorPolicy":
            REC_CLEAN_ANCHOR_POLICY,

        "CleanRECAnchorFormula":
            REC_CLEAN_ANCHOR_FORMULA,

        "CleanRECAnchorValues":
            TOTAL_DEPENDENT_ANCHOR_VALUES,

        "CleanRECNonZeroOffsets":
            TOTAL_NONZERO_DEPENDENT_OFFSETS,

        "ZeroPercentTrainingDependentRECMismatches":
            training_zero_dependent_mismatches,

        "ZeroPercentTrainingIndependentRECMismatches":
            training_zero_independent_mismatches,

        "ZeroPercentTrainingLabelMismatches":
            training_zero_label_mismatches,

        "ZeroPercentTrainingNonRECColumnsChanged":
            training_zero_non_rec_columns_changed,

        "ZeroPercentExactReproductionPassed":
            True,

        "EvaluationUnchanged":
            evaluation_unchanged,

        "CleanRECAnchorReference":
            str(
                CLEAN_REC_ANCHOR_REFERENCE_PATH
            ),

        "CleanRECAnchorOffsetSummary":
            str(
                CLEAN_REC_ANCHOR_OFFSET_SUMMARY_PATH
            ),

        "CleanRECZeroPercentValidation":
            str(
                CLEAN_REC_ZERO_VALIDATION_PATH
            ),

        "CleanRECAnchorPolicyReport":
            str(
                CLEAN_REC_ANCHOR_POLICY_PATH
            ),

        "UpdatedAtUTC":
            completed_at,
    }
)


temporary_checkpoint_path = (
    PROJECT_7_SELECTION_CHECKPOINT
    .with_suffix(
        ".json.tmp"
    )
)


with open(
    temporary_checkpoint_path,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        project_7_checkpoint,
        checkpoint_file,
        indent=2,
    )


os.replace(
    temporary_checkpoint_path,
    PROJECT_7_SELECTION_CHECKPOINT,
)


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint_verification = json.load(
        checkpoint_file
    )


if checkpoint_verification.get(
    "Status"
) != "CLEAN_REC_VALIDATION_PASSED":
    raise AssertionError(
        "The clean REC checkpoint was not written "
        "correctly."
    )


if not bool(
    checkpoint_verification.get(
        "ZeroPercentExactReproductionPassed",
        False,
    )
):
    raise AssertionError(
        "The checkpoint does not confirm exact 0% "
        "reproduction."
    )


# ---------------------------------------------------------
# 11. Compact result
# ---------------------------------------------------------

clear_output(
    wait=True
)


print(
    "=== PROJECT 7 STEP 6B RESULT ==="
)


print(
    "\nClean-anchor policy:"
)

print(
    "Policy:",
    REC_CLEAN_ANCHOR_POLICY
)

print(
    "Formula:",
    REC_CLEAN_ANCHOR_FORMULA
)

print(
    "Dependent REC features anchored:",
    len(
        REC_DEPENDENT_COLUMNS
    )
)

print(
    "Independent REC features preserved:",
    len(
        REC_INDEPENDENT_COLUMNS
    )
)


print(
    "\nStep 6A audit carried forward:"
)

print(
    "Dependent raw mismatches:",
    dependent_raw_mismatches
)

print(
    "Dependent raw mismatch percent:",
    f"{dependent_raw_mismatch_percent:.6f}%",
)

print(
    "Independent raw audit mismatches:",
    independent_raw_mismatches
)


print(
    "\nClean-anchor offset summary:"
)

display(
    clean_anchor_offset_summary
)

print(
    "Total dependent anchor values:",
    TOTAL_DEPENDENT_ANCHOR_VALUES
)

print(
    "Non-zero dependent offsets:",
    TOTAL_NONZERO_DEPENDENT_OFFSETS
)


print(
    "\n0% exact reproduction validation:"
)

display(
    zero_validation_summary
)

print(
    "All model-ready dependent REC mismatches:",
    full_zero_dependent_mismatches
)

print(
    "All model-ready independent REC mismatches:",
    full_zero_independent_mismatches
)

print(
    "Training dependent REC mismatches:",
    training_zero_dependent_mismatches
)

print(
    "Training independent REC mismatches:",
    training_zero_independent_mismatches
)

print(
    "Training label mismatches:",
    training_zero_label_mismatches
)

print(
    "Training non-REC columns changed:",
    training_zero_non_rec_columns_changed
)

print(
    "Training rows preserved:",
    training_rows_preserved
)

print(
    "Clean evaluation unchanged:",
    evaluation_unchanged
)


print(
    "\nClean-anchor policy report:"
)

print(
    CLEAN_REC_ANCHOR_POLICY_PATH
)


print(
    "\nValidation status:",
    "PASS"
)


print(
    "\nSUCCESS: The authoritative TCP-CI clean REC values "
    "were frozen as the exact 0% baseline."
)

print(
    "SUCCESS: Only the noise-induced raw-history delta "
    "will affect the thirteen dependent REC features."
)

print(
    "SUCCESS: The six independent REC features remain "
    "exactly unchanged."
)

print(
    "SUCCESS: The fixed training rows, labels and "
    "non-REC predictors were preserved exactly at "
    "0% noise."
)

print(
    "SUCCESS: Clean evaluation data remains untouched."
)

print(
    "SUCCESS: Project 7 is ready for deterministic "
    "noise injection and noisy REC helper validation."
)

=== PROJECT 7 STEP 6B RESULT ===

Clean-anchor policy:
Policy: AUTHORITATIVE_CLEAN_PLUS_NOISE_INDUCED_RAW_DELTA
Formula: anchored_noisy = authoritative_clean + (raw_noisy_reconstruction - raw_clean_reconstruction)
Dependent REC features anchored: 13
Independent REC features preserved: 6

Step 6A audit carried forward:
Dependent raw mismatches: 232
Dependent raw mismatch percent: 0.233650%
Independent raw audit mismatches: 35

Clean-anchor offset summary:


,Feature,Rows,NonZeroCleanOffsets,NonZeroOffsetPercent,MinimumOffset,MaximumOffset,MeanAbsoluteOffset,MaximumAbsoluteOffset
0,REC_LastFailureAge,7638,5,0.065462,-1.000000e+00,0.000000e+00,6.546216e-04,1.000000e+00
1,REC_LastTransitionAge,7638,7,0.091647,-2.000000e+00,0.000000e+00,1.047395e-03,2.000000e+00
2,REC_RecentFailRate,7638,3,0.039277,-1.000000e+00,3.333333e-07,3.927823e-04,1.000000e+00
3,REC_RecentAssertRate,7638,3,0.039277,-1.000000e+00,3.333333e-07,3.927794e-04,1.000000e+00
4,REC_RecentExcRate,7638,3,0.039277,-1.000000e+00,3.333333e-07,3.927778e-04,1.000000e+00
5,REC_RecentTransitionRate,7638,16,0.209479,-1.000000e+00,3.333337e-01,7.419144e-04,1.000000e+00
6,REC_TotalFailRate,7638,5,0.065462,-1.000000e+00,5.000000e-07,3.928233e-04,1.000000e+00
7,REC_TotalAssertRate,7638,3,0.039277,-1.000000e+00,5.000000e-07,3.928102e-04,1.000000e+00
8,REC_TotalExcRate,7638,3,0.039277,-1.000000e+00,5.000000e-07,3.928070e-04,1.000000e+00
9,REC_TotalTransitionRate,7638,126,1.649647,-1.000000e+00,1.000000e-02,4.478965e-04,1.000000e+00


Total dependent anchor values: 99294
Non-zero dependent offsets: 232

0% exact reproduction validation:


,ValidationScope,Rows,DependentRECMismatches,IndependentRECMismatches,LabelMismatches,NonRECColumnsChanged,RowsPreserved
0,ALL_MODEL_READY_ROWS,7638,0,0,0,0,True
1,TRAINING_MODEL_READY_ROWS,3960,0,0,0,0,True


All model-ready dependent REC mismatches: 0
All model-ready independent REC mismatches: 0
Training dependent REC mismatches: 0
Training independent REC mismatches: 0
Training label mismatches: 0
Training non-REC columns changed: 0
Training rows preserved: True
Clean evaluation unchanged: True

Clean-anchor policy report:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_preflight/beast2_clean_rec_anchor_policy.json

Validation status: PASS

SUCCESS: The authoritative TCP-CI clean REC values were frozen as the exact 0% baseline.
SUCCESS: Only the noise-induced raw-history delta will affect the thirteen dependent REC features.
SUCCESS: The six independent REC features remain exactly unchanged.
SUCCESS: The fixed training rows, labels and non-REC predictors were preserved exactly at 0% noise.
SUCCESS: Clean evaluation data remains untouched.
SUCCESS: Project 7 is ready for deterministic noise injection and noisy REC helper validation.


In [ ]:
# =========================================================
# PROJECT 7 — STEP 7A
# DETERMINISTIC NOISE + NOISY REC HELPER VALIDATION
# =========================================================

from pathlib import Path
import hashlib
import json
import os

import numpy as np
import pandas as pd
from IPython.display import clear_output, display

print(
    "=== PROJECT 7 STEP 7A: "
    "DETERMINISTIC NOISE AND NOISY REC VALIDATION ==="
)


# ---------------------------------------------------------
# 1. Validate Step 6B state
# ---------------------------------------------------------

required = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",
    "PROJECT_PREFLIGHT_DIRECTORY",
    "PROJECT_7_SELECTION_CHECKPOINT",
    "raw_training_work",
    "raw_evaluation_work",
    "model_clean",
    "clean_training_data",
    "clean_evaluation_data",
    "REC_COLUMNS",
    "REC_DEPENDENT_COLUMNS",
    "REC_INDEPENDENT_COLUMNS",
    "reconstruct_project_7_raw_rec",
    "apply_project_7_clean_rec_anchor",
]


missing = [
    name
    for name in required
    if name not in globals()
]


if missing:
    raise RuntimeError(
        "Missing Step 6B objects:\n"
        + "\n".join(missing)
        + "\n\nRerun only the successful Steps 5, "
        "6A and 6B."
    )


if (
    PROJECT_NUMBER,
    PROJECT_NAME,
    PROJECT_SLUG,
) != (
    7,
    "CompEvol@beast2",
    "CompEvol__beast2",
):
    raise AssertionError(
        "Unexpected Project 7 identity."
    )


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    previous_checkpoint = json.load(
        checkpoint_file
    )


if previous_checkpoint.get(
    "Status"
) not in {
    "CLEAN_REC_VALIDATION_PASSED",
    "NOISE_HELPERS_VALIDATION_PASSED",
}:
    raise AssertionError(
        "Step 6B has not passed. Observed status: "
        + str(
            previous_checkpoint.get(
                "Status"
            )
        )
    )


if not previous_checkpoint.get(
    "ZeroPercentExactReproductionPassed",
    False,
):
    raise AssertionError(
        "The checkpoint does not confirm exact "
        "0% reproduction."
    )


# ---------------------------------------------------------
# 2. Frozen experiment grid and clean references
# ---------------------------------------------------------

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]


REPETITION_SEEDS = list(
    range(1, 31)
)


CLEAN_RAW_TRAINING_HISTORY = (
    raw_training_work
    .copy(deep=True)
    .reset_index(drop=True)
)


CLEAN_RAW_EVALUATION_HISTORY = (
    raw_evaluation_work
    .copy(deep=True)
    .reset_index(drop=True)
)


CLEAN_MODEL_ALL_DATA = (
    model_clean
    .copy(deep=True)
    .reset_index(drop=True)
)


CLEAN_MODEL_TRAINING_DATA = (
    clean_training_data
    .copy(deep=True)
    .reset_index(drop=True)
)


CLEAN_MODEL_EVALUATION_DATA = (
    clean_evaluation_data
    .copy(deep=True)
    .reset_index(drop=True)
)


for frame_name, frame in [
    (
        "raw training",
        CLEAN_RAW_TRAINING_HISTORY,
    ),
    (
        "raw evaluation",
        CLEAN_RAW_EVALUATION_HISTORY,
    ),
    (
        "all model-ready",
        CLEAN_MODEL_ALL_DATA,
    ),
    (
        "model training",
        CLEAN_MODEL_TRAINING_DATA,
    ),
    (
        "model evaluation",
        CLEAN_MODEL_EVALUATION_DATA,
    ),
]:
    needed = {
        "Build",
        "Test",
        "Verdict",
        "Duration",
        "build_order",
        "partition",
    }


    absent = (
        needed
        - set(
            frame.columns
        )
    )


    if absent:
        raise RuntimeError(
            f"{frame_name} is missing columns: "
            f"{sorted(absent)}"
        )


    frame[
        "Build"
    ] = pd.to_numeric(
        frame[
            "Build"
        ],
        errors="raise",
    ).astype(np.int64)


    frame[
        "Test"
    ] = frame[
        "Test"
    ].astype(str)


    frame[
        "Verdict"
    ] = pd.to_numeric(
        frame[
            "Verdict"
        ],
        errors="raise",
    ).astype(np.int64)


    frame[
        "build_order"
    ] = pd.to_numeric(
        frame[
            "build_order"
        ],
        errors="raise",
    ).astype(np.int64)


    if frame[
        [
            "Build",
            "Test",
        ]
    ].duplicated().any():
        raise AssertionError(
            f"{frame_name} contains duplicate "
            "Build-Test keys."
        )


if set(
    CLEAN_RAW_TRAINING_HISTORY[
        "partition"
    ].astype(str)
) != {
    "TRAIN"
}:
    raise AssertionError(
        "Raw training history contains "
        "non-training rows."
    )


if set(
    CLEAN_RAW_EVALUATION_HISTORY[
        "partition"
    ].astype(str)
) != {
    "EVALUATION"
}:
    raise AssertionError(
        "Raw evaluation history contains "
        "non-evaluation rows."
    )


# ---------------------------------------------------------
# 3. Same deterministic seed policy used in Projects 1–6
# ---------------------------------------------------------

def stable_project_seed(
    project_name,
    repetition_seed,
    random_stream,
):
    seed_text = (
        f"{project_name}|"
        f"{int(repetition_seed)}|"
        f"{random_stream}"
    )


    digest = hashlib.sha256(
        seed_text.encode(
            "utf-8"
        )
    ).digest()


    return int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False,
    ) % (
        2 ** 32
    )


def inject_training_verdict_noise(
    execution_history,
    noise_percent,
    repetition_seed,
    project_name=PROJECT_NAME,
):
    """
    Flip raw training verdicts using nested,
    project-specific deterministic random streams.

    Pass:
        0 -> sampled clean failure subtype

    Failure:
        non-zero -> 0
    """

    noise_percent = float(
        noise_percent
    )


    repetition_seed = int(
        repetition_seed
    )


    if noise_percent not in {
        float(
            level
        )
        for level in NOISE_LEVELS
    }:
        raise ValueError(
            "noise_percent must be one of "
            f"{NOISE_LEVELS}"
        )


    if repetition_seed not in (
        REPETITION_SEEDS
    ):
        raise ValueError(
            "repetition_seed must be between "
            "1 and 30."
        )


    noisy_history = (
        execution_history
        .copy(deep=True)
        .reset_index(drop=True)
    )


    needed = {
        "Build",
        "Test",
        "Verdict",
    }


    missing_columns = (
        needed
        - set(
            noisy_history.columns
        )
    )


    if missing_columns:
        raise RuntimeError(
            "Noise history is missing: "
            f"{sorted(missing_columns)}"
        )


    if len(
        noisy_history
    ) == 0:
        raise ValueError(
            "execution_history is empty."
        )


    noisy_history[
        "Build"
    ] = pd.to_numeric(
        noisy_history[
            "Build"
        ],
        errors="raise",
    ).astype(np.int64)


    noisy_history[
        "Test"
    ] = noisy_history[
        "Test"
    ].astype(str)


    original_verdict = pd.to_numeric(
        noisy_history[
            "Verdict"
        ],
        errors="raise",
    ).astype(
        np.int64
    ).to_numpy()


    if noisy_history[
        [
            "Build",
            "Test",
        ]
    ].duplicated().any():
        raise AssertionError(
            "Raw training history has duplicate "
            "Build-Test keys."
        )


    failure_subtype_counts = (
        pd.Series(
            original_verdict[
                original_verdict != 0
            ]
        )
        .value_counts()
        .sort_index()
    )


    if failure_subtype_counts.empty:
        raise ValueError(
            "No clean failure subtype exists."
        )


    failure_subtypes = (
        failure_subtype_counts
        .index
        .to_numpy(
            dtype=np.int64
        )
    )


    failure_subtype_probabilities = (
        failure_subtype_counts
        .to_numpy(
            dtype=float
        )
    )


    failure_subtype_probabilities /= (
        failure_subtype_probabilities.sum()
    )


    flip_rng = np.random.default_rng(
        stable_project_seed(
            project_name,
            repetition_seed,
            "flip_mask",
        )
    )


    subtype_rng = np.random.default_rng(
        stable_project_seed(
            project_name,
            repetition_seed,
            "failure_subtype",
        )
    )


    flip_uniforms = flip_rng.random(
        len(
            noisy_history
        )
    )


    sampled_failure_subtypes = (
        subtype_rng.choice(
            failure_subtypes,
            size=len(
                noisy_history
            ),
            replace=True,
            p=(
                failure_subtype_probabilities
            ),
        )
    )


    flipped = (
        flip_uniforms
        < noise_percent / 100.0
    )


    pass_to_failure = (
        flipped
        & (
            original_verdict == 0
        )
    )


    failure_to_pass = (
        flipped
        & (
            original_verdict != 0
        )
    )


    noisy_verdict = (
        original_verdict.copy()
    )


    noisy_verdict[
        pass_to_failure
    ] = sampled_failure_subtypes[
        pass_to_failure
    ]


    noisy_verdict[
        failure_to_pass
    ] = 0


    noisy_history[
        "Verdict"
    ] = noisy_verdict


    manifest = pd.DataFrame({
        "NoiseRowID":
            np.arange(
                len(
                    noisy_history
                ),
                dtype=np.int64,
            ),

        "Build":
            noisy_history[
                "Build"
            ].to_numpy(
                dtype=np.int64
            ),

        "Test":
            noisy_history[
                "Test"
            ].to_numpy(
                dtype=str
            ),

        "OriginalVerdict":
            original_verdict,

        "NoisyVerdict":
            noisy_verdict,

        "FlipUniform":
            flip_uniforms,

        "SampledFailureSubtype":
            sampled_failure_subtypes,

        "Flipped":
            flipped,

        "PassToFailure":
            pass_to_failure,

        "FailureToPass":
            failure_to_pass,
    })


    for optional_column in [
        "job",
        "Job",
        "build_order",
        "partition",
    ]:
        if optional_column in (
            noisy_history.columns
        ):
            manifest[
                optional_column
            ] = noisy_history[
                optional_column
            ].to_numpy()


    flipped_count = int(
        flipped.sum()
    )


    summary = {
        "Project":
            str(
                project_name
            ),

        "NoisePercentRequested":
            noise_percent,

        "RepetitionSeed":
            repetition_seed,

        "TrainingExecutionRows":
            int(
                len(
                    noisy_history
                )
            ),

        "NumberFlipped":
            flipped_count,

        "RealisedNoisePercent":
            float(
                100.0
                * flipped_count
                / len(
                    noisy_history
                )
            ),

        "PassToFailure":
            int(
                pass_to_failure.sum()
            ),

        "FailureToPass":
            int(
                failure_to_pass.sum()
            ),
    }


    return (
        noisy_history,
        manifest,
        summary,
    )


# ---------------------------------------------------------
# 4. Alignment and complete-condition helpers
# ---------------------------------------------------------

def align_source_rows(
    source_rows,
    requested_rows,
):
    source = (
        source_rows
        .copy()
        .reset_index(drop=True)
    )


    requested = (
        requested_rows[
            [
                "Build",
                "Test",
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )


    for frame in [
        source,
        requested,
    ]:
        frame[
            "Build"
        ] = pd.to_numeric(
            frame[
                "Build"
            ],
            errors="raise",
        ).astype(np.int64)


        frame[
            "Test"
        ] = frame[
            "Test"
        ].astype(str)


    if source[
        [
            "Build",
            "Test",
        ]
    ].duplicated().any():
        raise AssertionError(
            "Source contains duplicate "
            "Build-Test keys."
        )


    if requested[
        [
            "Build",
            "Test",
        ]
    ].duplicated().any():
        raise AssertionError(
            "Requested rows contain duplicate "
            "Build-Test keys."
        )


    requested[
        "_order"
    ] = np.arange(
        len(
            requested
        ),
        dtype=np.int64,
    )


    aligned = (
        requested
        .merge(
            source,
            on=[
                "Build",
                "Test",
            ],
            how="left",
            validate="one_to_one",
            indicator=True,
        )
        .sort_values(
            "_order",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    if not (
        aligned[
            "_merge"
        ] == "both"
    ).all():
        missing_count = int(
            (
                aligned[
                    "_merge"
                ] != "both"
            ).sum()
        )


        raise AssertionError(
            f"{missing_count} requested rows "
            "could not be aligned."
        )


    return aligned.drop(
        columns=[
            "_order",
            "_merge",
        ]
    )


def extract_model_ready_noisy_verdicts(
    noisy_history,
    requested_training_rows,
):
    verdict_source = (
        noisy_history[
            [
                "Build",
                "Test",
                "Verdict",
            ]
        ]
        .rename(
            columns={
                "Verdict":
                    "NoisyVerdict",
            }
        )
    )


    aligned = align_source_rows(
        verdict_source,
        requested_training_rows,
    )


    return pd.to_numeric(
        aligned[
            "NoisyVerdict"
        ],
        errors="raise",
    ).astype(np.int64)


def create_project_7_noisy_condition(
    noise_percent,
    repetition_seed,
):
    (
        noisy_training_history,
        manifest,
        summary,
    ) = inject_training_verdict_noise(
        CLEAN_RAW_TRAINING_HISTORY,
        noise_percent,
        repetition_seed,
    )


    full_raw_history = pd.concat(
        [
            noisy_training_history,
            CLEAN_RAW_EVALUATION_HISTORY,
        ],
        ignore_index=True,
    )


    raw_noisy_rec = (
        reconstruct_project_7_raw_rec(
            full_raw_history
        )
        .reset_index(drop=True)
    )


    if len(
        raw_noisy_rec
    ) != len(
        CLEAN_MODEL_ALL_DATA
    ):
        raise AssertionError(
            "Noisy reconstruction returned "
            "the wrong row count."
        )


    raw_noisy_with_keys = pd.concat(
        [
            CLEAN_MODEL_ALL_DATA[
                [
                    "Build",
                    "Test",
                ]
            ].reset_index(drop=True),

            raw_noisy_rec,
        ],
        axis=1,
    )


    anchored_all_model_data = (
        apply_project_7_clean_rec_anchor(
            CLEAN_MODEL_ALL_DATA,
            raw_noisy_with_keys,
        )
    )


    noisy_training_data = (
        align_source_rows(
            anchored_all_model_data,
            CLEAN_MODEL_TRAINING_DATA,
        )
    )


    noisy_training_data[
        "Verdict"
    ] = (
        extract_model_ready_noisy_verdicts(
            noisy_training_history,
            CLEAN_MODEL_TRAINING_DATA,
        )
        .to_numpy(
            dtype=np.int64
        )
    )


    noisy_evaluation_data = (
        align_source_rows(
            anchored_all_model_data,
            CLEAN_MODEL_EVALUATION_DATA,
        )
    )


    if not np.array_equal(
        noisy_evaluation_data[
            "Verdict"
        ].to_numpy(),

        CLEAN_MODEL_EVALUATION_DATA[
            "Verdict"
        ].to_numpy(),
    ):
        raise AssertionError(
            "Evaluation verdicts changed under "
            "training noise."
        )


    if len(
        noisy_training_data
    ) != len(
        CLEAN_MODEL_TRAINING_DATA
    ):
        raise AssertionError(
            "Fixed training row count changed."
        )


    if len(
        noisy_evaluation_data
    ) != len(
        CLEAN_MODEL_EVALUATION_DATA
    ):
        raise AssertionError(
            "Evaluation row count changed."
        )


    return {
        "NoisyTrainingHistory":
            noisy_training_history,

        "NoiseManifest":
            manifest,

        "NoiseSummary":
            summary,

        "RawNoisyREC":
            raw_noisy_with_keys,

        "AnchoredAllModelData":
            anchored_all_model_data,

        "NoisyTrainingData":
            noisy_training_data,

        "NoisyEvaluationData":
            noisy_evaluation_data,
    }


# ---------------------------------------------------------
# 5. Validate nested masks across all nine levels
# ---------------------------------------------------------

seed_1_manifests = {}

seed_1_summaries = []


for noise_level in NOISE_LEVELS:
    (
        _,
        manifest,
        summary,
    ) = inject_training_verdict_noise(
        CLEAN_RAW_TRAINING_HISTORY,
        noise_level,
        1,
    )


    seed_1_manifests[
        noise_level
    ] = manifest


    seed_1_summaries.append(
        summary
    )


noise_level_summary_seed_1 = (
    pd.DataFrame(
        seed_1_summaries
    )
)


nestedness_records = []


for lower_level, upper_level in zip(
    NOISE_LEVELS[:-1],
    NOISE_LEVELS[1:],
):
    lower_manifest = (
        seed_1_manifests[
            lower_level
        ]
    )


    upper_manifest = (
        seed_1_manifests[
            upper_level
        ]
    )


    lower_mask = lower_manifest[
        "Flipped"
    ].to_numpy(
        dtype=bool
    )


    upper_mask = upper_manifest[
        "Flipped"
    ].to_numpy(
        dtype=bool
    )


    missing_from_upper = int(
        (
            lower_mask
            & ~upper_mask
        ).sum()
    )


    nestedness_records.append({
        "LowerNoisePercent":
            lower_level,

        "UpperNoisePercent":
            upper_level,

        "LowerFlippedRows":
            int(
                lower_mask.sum()
            ),

        "UpperFlippedRows":
            int(
                upper_mask.sum()
            ),

        "LowerRowsMissingFromUpperMask":
            missing_from_upper,

        "SameFlipUniforms":
            bool(
                np.array_equal(
                    lower_manifest[
                        "FlipUniform"
                    ],

                    upper_manifest[
                        "FlipUniform"
                    ],
                )
            ),

        "SameSampledFailureSubtypes":
            bool(
                np.array_equal(
                    lower_manifest[
                        "SampledFailureSubtype"
                    ],

                    upper_manifest[
                        "SampledFailureSubtype"
                    ],
                )
            ),

        "NestedMaskPassed":
            bool(
                missing_from_upper == 0
            ),
    })


nestedness_audit = pd.DataFrame(
    nestedness_records
)


if not nestedness_audit[
    [
        "SameFlipUniforms",
        "SameSampledFailureSubtypes",
        "NestedMaskPassed",
    ]
].all().all():
    raise AssertionError(
        "Nested-mask validation failed."
    )


# ---------------------------------------------------------
# 6. Reproducibility and different-seed validation
# ---------------------------------------------------------

(
    history_25_a,
    manifest_25_a,
    summary_25_a,
) = inject_training_verdict_noise(
    CLEAN_RAW_TRAINING_HISTORY,
    25,
    1,
)


(
    history_25_b,
    manifest_25_b,
    _,
) = inject_training_verdict_noise(
    CLEAN_RAW_TRAINING_HISTORY,
    25,
    1,
)


(
    _,
    manifest_25_seed_2,
    _,
) = inject_training_verdict_noise(
    CLEAN_RAW_TRAINING_HISTORY,
    25,
    2,
)


same_seed_history = bool(
    np.array_equal(
        history_25_a[
            "Verdict"
        ],

        history_25_b[
            "Verdict"
        ],
    )
)


same_seed_manifest = bool(
    manifest_25_a.equals(
        manifest_25_b
    )
)


different_seed_mask = bool(
    not np.array_equal(
        manifest_25_a[
            "Flipped"
        ],

        manifest_25_seed_2[
            "Flipped"
        ],
    )
)


reproducibility_summary = pd.DataFrame([
    {
        "Check":
            "Same seed noisy history",

        "Passed":
            same_seed_history,
    },
    {
        "Check":
            "Same seed complete manifest",

        "Passed":
            same_seed_manifest,
    },
    {
        "Check":
            "Different seed flip mask differs",

        "Passed":
            different_seed_mask,
    },
    {
        "Check":
            "All adjacent masks nested",

        "Passed":
            bool(
                nestedness_audit[
                    "NestedMaskPassed"
                ].all()
            ),
    },
])


if not reproducibility_summary[
    "Passed"
].all():
    raise AssertionError(
        "Noise reproducibility validation failed."
    )


# ---------------------------------------------------------
# 7. End-to-end 0% and 25% validation
# ---------------------------------------------------------

zero_condition = (
    create_project_7_noisy_condition(
        0,
        1,
    )
)


quarter_condition = (
    create_project_7_noisy_condition(
        25,
        1,
    )
)


zero_training = zero_condition[
    "NoisyTrainingData"
]


zero_evaluation = zero_condition[
    "NoisyEvaluationData"
]


quarter_training = quarter_condition[
    "NoisyTrainingData"
]


quarter_evaluation = quarter_condition[
    "NoisyEvaluationData"
]


zero_training_exact = bool(
    zero_training.equals(
        CLEAN_MODEL_TRAINING_DATA
    )
)


zero_evaluation_exact = bool(
    zero_evaluation.equals(
        CLEAN_MODEL_EVALUATION_DATA
    )
)


if (
    not zero_training_exact
    or not zero_evaluation_exact
):
    raise AssertionError(
        "The complete 0% condition did not "
        "reproduce clean data exactly."
    )


quarter_label_changes = int(
    (
        quarter_training[
            "Verdict"
        ].to_numpy()
        !=
        CLEAN_MODEL_TRAINING_DATA[
            "Verdict"
        ].to_numpy()
    ).sum()
)


retained_manifest = align_source_rows(
    quarter_condition[
        "NoiseManifest"
    ][
        [
            "Build",
            "Test",
            "Flipped",
        ]
    ],
    CLEAN_MODEL_TRAINING_DATA,
)


expected_label_changes = int(
    retained_manifest[
        "Flipped"
    ].astype(bool).sum()
)


if (
    quarter_label_changes
    != expected_label_changes
):
    raise AssertionError(
        "Changed retained labels do not match "
        "flipped retained raw rows."
    )


def count_changes(
    left,
    right,
    columns,
):
    return int(
        sum(
            (
                left[
                    column
                ].to_numpy(
                    dtype=float
                )
                !=
                right[
                    column
                ].to_numpy(
                    dtype=float
                )
            ).sum()

            for column in columns
        )
    )


training_dependent_changes = (
    count_changes(
        quarter_training,
        CLEAN_MODEL_TRAINING_DATA,
        REC_DEPENDENT_COLUMNS,
    )
)


evaluation_dependent_changes = (
    count_changes(
        quarter_evaluation,
        CLEAN_MODEL_EVALUATION_DATA,
        REC_DEPENDENT_COLUMNS,
    )
)


training_independent_changes = (
    count_changes(
        quarter_training,
        CLEAN_MODEL_TRAINING_DATA,
        REC_INDEPENDENT_COLUMNS,
    )
)


evaluation_independent_changes = (
    count_changes(
        quarter_evaluation,
        CLEAN_MODEL_EVALUATION_DATA,
        REC_INDEPENDENT_COLUMNS,
    )
)


if (
    training_dependent_changes <= 0
    or evaluation_dependent_changes <= 0
):
    raise AssertionError(
        "25% noise produced no dependent "
        "REC changes."
    )


if (
    training_independent_changes != 0
    or evaluation_independent_changes != 0
):
    raise AssertionError(
        "A verdict-independent REC feature changed."
    )


training_preserved_columns = [
    column
    for column in (
        CLEAN_MODEL_TRAINING_DATA.columns
    )
    if column not in set(
        REC_DEPENDENT_COLUMNS
        + [
            "Verdict"
        ]
    )
]


evaluation_preserved_columns = [
    column
    for column in (
        CLEAN_MODEL_EVALUATION_DATA.columns
    )
    if column not in set(
        REC_DEPENDENT_COLUMNS
    )
]


training_non_target_columns_changed = int(
    sum(
        not quarter_training[
            column
        ].equals(
            CLEAN_MODEL_TRAINING_DATA[
                column
            ]
        )

        for column in (
            training_preserved_columns
        )
    )
)


evaluation_non_target_columns_changed = int(
    sum(
        not quarter_evaluation[
            column
        ].equals(
            CLEAN_MODEL_EVALUATION_DATA[
                column
            ]
        )

        for column in (
            evaluation_preserved_columns
        )
    )
)


if (
    training_non_target_columns_changed
    or evaluation_non_target_columns_changed
):
    raise AssertionError(
        "A non-target column changed under "
        "25% noise."
    )


condition_validation = pd.DataFrame([
    {
        "NoisePercent":
            0,

        "Seed":
            1,

        "RawRowsFlipped":
            int(
                zero_condition[
                    "NoiseSummary"
                ][
                    "NumberFlipped"
                ]
            ),

        "ModelTrainingRows":
            int(
                len(
                    zero_training
                )
            ),

        "ModelTrainingLabelChanges":
            0,

        "TrainingDependentRECChanges":
            0,

        "EvaluationDependentRECChanges":
            0,

        "TrainingIndependentRECChanges":
            0,

        "EvaluationIndependentRECChanges":
            0,

        "TrainingExactCleanReproduction":
            zero_training_exact,

        "EvaluationExactCleanReproduction":
            zero_evaluation_exact,
    },
    {
        "NoisePercent":
            25,

        "Seed":
            1,

        "RawRowsFlipped":
            int(
                summary_25_a[
                    "NumberFlipped"
                ]
            ),

        "ModelTrainingRows":
            int(
                len(
                    quarter_training
                )
            ),

        "ModelTrainingLabelChanges":
            quarter_label_changes,

        "TrainingDependentRECChanges":
            training_dependent_changes,

        "EvaluationDependentRECChanges":
            evaluation_dependent_changes,

        "TrainingIndependentRECChanges":
            training_independent_changes,

        "EvaluationIndependentRECChanges":
            evaluation_independent_changes,

        "TrainingExactCleanReproduction":
            False,

        "EvaluationExactCleanReproduction":
            False,
    },
])


# ---------------------------------------------------------
# 8. Save permanent audit artefacts
# ---------------------------------------------------------

NOISE_LEVEL_SUMMARY_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_noise_level_summary_seed_01.csv"
)


NESTEDNESS_AUDIT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_noise_nestedness_audit_seed_01.csv"
)


REPRODUCIBILITY_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_noise_reproducibility_summary.csv"
)


CONDITION_VALIDATION_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_noisy_condition_validation.csv"
)


SEED_1_25_MANIFEST_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_noise_manifest_seed_01_noise_25.parquet"
)


NOISE_HELPER_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_noise_helper_validation_report.json"
)


noise_level_summary_seed_1.to_csv(
    NOISE_LEVEL_SUMMARY_PATH,
    index=False,
)


nestedness_audit.to_csv(
    NESTEDNESS_AUDIT_PATH,
    index=False,
)


reproducibility_summary.to_csv(
    REPRODUCIBILITY_PATH,
    index=False,
)


condition_validation.to_csv(
    CONDITION_VALIDATION_PATH,
    index=False,
)


quarter_condition[
    "NoiseManifest"
].to_parquet(
    SEED_1_25_MANIFEST_PATH,
    index=False,
)


completed_at = (
    pd.Timestamp.utcnow().isoformat()
)


noise_helper_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "PASS",

    "NoiseLevelsPercent":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "NoiseUnit":
        (
            "individual raw training "
            "execution verdict"
        ),

    "SeedPolicy":
        (
            "SHA256 project-specific "
            "random streams"
        ),

    "RandomStreams": [
        "flip_mask",
        "failure_subtype",
    ],

    "NestedMasks":
        True,

    "PassToFailurePolicy":
        (
            "sample clean project "
            "failure subtype"
        ),

    "FailureToPassPolicy":
        (
            "non-zero verdict becomes zero"
        ),

    "ZeroPercentTrainingExact":
        zero_training_exact,

    "ZeroPercentEvaluationExact":
        zero_evaluation_exact,

    "Seed1Noise25RawRowsFlipped":
        int(
            summary_25_a[
                "NumberFlipped"
            ]
        ),

    "Seed1Noise25ModelLabelChanges":
        quarter_label_changes,

    "Seed1Noise25TrainingDependentRECChanges":
        training_dependent_changes,

    "Seed1Noise25EvaluationDependentRECChanges":
        evaluation_dependent_changes,

    "Seed1Noise25TrainingIndependentRECChanges":
        training_independent_changes,

    "Seed1Noise25EvaluationIndependentRECChanges":
        evaluation_independent_changes,

    "TrainingNonTargetColumnsChanged":
        training_non_target_columns_changed,

    "EvaluationNonTargetColumnsChanged":
        evaluation_non_target_columns_changed,

    "EvaluationVerdictsClean":
        True,

    "Artefacts": {
        "NoiseLevelSummarySeed1":
            str(
                NOISE_LEVEL_SUMMARY_PATH
            ),

        "NestednessAuditSeed1":
            str(
                NESTEDNESS_AUDIT_PATH
            ),

        "ReproducibilitySummary":
            str(
                REPRODUCIBILITY_PATH
            ),

        "ConditionValidation":
            str(
                CONDITION_VALIDATION_PATH
            ),

        "Seed1Noise25Manifest":
            str(
                SEED_1_25_MANIFEST_PATH
            ),
    },

    "CompletedAtUTC":
        completed_at,
}


with open(
    NOISE_HELPER_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        noise_helper_report,
        report_file,
        indent=2,
    )


# ---------------------------------------------------------
# 9. Update checkpoint atomically
# ---------------------------------------------------------

with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    project_checkpoint = json.load(
        checkpoint_file
    )


project_checkpoint.update({
    "Status":
        "NOISE_HELPERS_VALIDATION_PASSED",

    "NoiseLevelsPercent":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "NoiseSeedPolicy":
        "SHA256_PROJECT_SEED_STREAM",

    "NoiseMaskPolicy":
        (
            "NESTED_COMMON_UNIFORMS_"
            "PER_PROJECT_SEED"
        ),

    "PassToFailureSubtypePolicy":
        (
            "PROJECT_CLEAN_FAILURE_"
            "SUBTYPE_DISTRIBUTION"
        ),

    "FailureToPassPolicy":
        "NONZERO_TO_ZERO",

    "ZeroPercentTrainingExactReproduction":
        zero_training_exact,

    "ZeroPercentEvaluationExactReproduction":
        zero_evaluation_exact,

    "Seed1Noise25RawRowsFlipped":
        int(
            summary_25_a[
                "NumberFlipped"
            ]
        ),

    "Seed1Noise25ModelTrainingLabelChanges":
        quarter_label_changes,

    "Seed1Noise25TrainingDependentRECChanges":
        training_dependent_changes,

    "Seed1Noise25EvaluationDependentRECChanges":
        evaluation_dependent_changes,

    "NoiseNestednessValidationPassed":
        True,

    "NoiseSameSeedReproducibilityPassed":
        True,

    "NoiseDifferentSeedValidationPassed":
        True,

    "NoiseHelperValidationReport":
        str(
            NOISE_HELPER_REPORT_PATH
        ),

    "UpdatedAtUTC":
        completed_at,
})


temporary_checkpoint_path = (
    PROJECT_7_SELECTION_CHECKPOINT
    .with_suffix(
        ".json.tmp"
    )
)


with open(
    temporary_checkpoint_path,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        project_checkpoint,
        checkpoint_file,
        indent=2,
    )


os.replace(
    temporary_checkpoint_path,
    PROJECT_7_SELECTION_CHECKPOINT,
)


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint_verification = json.load(
        checkpoint_file
    )


if checkpoint_verification.get(
    "Status"
) != "NOISE_HELPERS_VALIDATION_PASSED":
    raise AssertionError(
        "The Step 7A checkpoint was not "
        "written correctly."
    )


# ---------------------------------------------------------
# 10. Compact result
# ---------------------------------------------------------

clear_output(
    wait=True
)


print(
    "=== PROJECT 7 STEP 7A RESULT ==="
)


print(
    "\nFrozen experiment grid:"
)

print(
    "Noise levels:",
    NOISE_LEVELS
)

print(
    "Repetition seeds:",
    "1–30"
)

print(
    "Conditions:",
    len(
        NOISE_LEVELS
    )
    * len(
        REPETITION_SEEDS
    )
)


print(
    "\nSeed 1 noise summary:"
)

display(
    noise_level_summary_seed_1[
        [
            "NoisePercentRequested",
            "TrainingExecutionRows",
            "NumberFlipped",
            "RealisedNoisePercent",
            "PassToFailure",
            "FailureToPass",
        ]
    ]
)


print(
    "\nNested-mask audit:"
)

display(
    nestedness_audit
)


print(
    "All adjacent masks nested:",
    bool(
        nestedness_audit[
            "NestedMaskPassed"
        ].all()
    )
)


print(
    "Same uniforms across levels:",
    bool(
        nestedness_audit[
            "SameFlipUniforms"
        ].all()
    )
)


print(
    "Same sampled subtypes across levels:",
    bool(
        nestedness_audit[
            "SameSampledFailureSubtypes"
        ].all()
    )
)


print(
    "\nReproducibility:"
)

display(
    reproducibility_summary
)


print(
    "\nEnd-to-end condition validation:"
)

display(
    condition_validation
)


print(
    "0% training exact reproduction:",
    zero_training_exact
)

print(
    "0% evaluation exact reproduction:",
    zero_evaluation_exact
)

print(
    "25% raw rows flipped:",
    summary_25_a[
        "NumberFlipped"
    ]
)

print(
    "25% retained training label changes:",
    quarter_label_changes
)

print(
    "25% training dependent REC changes:",
    training_dependent_changes
)

print(
    "25% evaluation dependent REC changes:",
    evaluation_dependent_changes
)

print(
    "25% training independent REC changes:",
    training_independent_changes
)

print(
    "25% evaluation independent REC changes:",
    evaluation_independent_changes
)

print(
    "25% training non-target columns changed:",
    training_non_target_columns_changed
)

print(
    "25% evaluation non-target columns changed:",
    evaluation_non_target_columns_changed
)

print(
    "Evaluation verdicts remained clean:",
    True
)


print(
    "\nNoise-helper report:"
)

print(
    NOISE_HELPER_REPORT_PATH
)


print(
    "\nValidation status:",
    "PASS"
)


print(
    "\nSUCCESS: Deterministic project-specific random "
    "streams match Projects 1–6."
)

print(
    "SUCCESS: All nine noise masks are nested for a "
    "fixed seed."
)

print(
    "SUCCESS: 0% noise reproduces clean training and "
    "evaluation data exactly."
)

print(
    "SUCCESS: 25% noise changes only training labels "
    "and dependent REC features."
)

print(
    "SUCCESS: Evaluation verdicts and independent REC "
    "features remain clean."
)

print(
    "SUCCESS: Project 7 is ready for model and baseline "
    "helper validation."
)

=== PROJECT 7 STEP 7A: DETERMINISTIC NOISE AND NOISY REC VALIDATION ===


AssertionError: The complete 0% condition did not reproduce clean data exactly.

In [ ]:
# =========================================================
# PROJECT 7 — STEP 7A CORRECTION
# RESTORE AUTHORITATIVE SCHEMA AND COMPLETE VALIDATION
#
# Cause of the previous failure:
#   DataFrame.equals() checks values, column order, index,
#   and dtypes. The clean-anchor calculation temporarily
#   converted integer REC columns to float.
#
# This cell:
#   - restores authoritative column order and dtypes
#   - redefines the complete noisy-condition helper
#   - verifies exact 0% reproduction
#   - validates nested masks and reproducibility
#   - validates a non-zero 25% condition
#   - saves the permanent Step 7A artefacts
#   - updates the Project 7 checkpoint
#
# Do not rerun the previous failed Step 7A cell.
# =========================================================

from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
from IPython.display import clear_output, display


print(
    "=== PROJECT 7 STEP 7A CORRECTION: "
    "RESTORE AUTHORITATIVE SCHEMA ==="
)


# ---------------------------------------------------------
# 1. Confirm the failed Step 7A created its helper objects
# ---------------------------------------------------------

required_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",
    "PROJECT_PREFLIGHT_DIRECTORY",
    "PROJECT_7_SELECTION_CHECKPOINT",

    "NOISE_LEVELS",
    "REPETITION_SEEDS",

    "CLEAN_RAW_TRAINING_HISTORY",
    "CLEAN_RAW_EVALUATION_HISTORY",
    "CLEAN_MODEL_ALL_DATA",
    "CLEAN_MODEL_TRAINING_DATA",
    "CLEAN_MODEL_EVALUATION_DATA",

    "REC_DEPENDENT_COLUMNS",
    "REC_INDEPENDENT_COLUMNS",

    "inject_training_verdict_noise",
    "reconstruct_project_7_raw_rec",
    "apply_project_7_clean_rec_anchor",
]


missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "The failed Step 7A runtime state is incomplete:\n"
        + "\n".join(missing_objects)
        + "\n\nRerun the previous Step 7A cell only until "
        "it reaches the 0% reproduction assertion, then "
        "run this correction cell."
    )


if (
    PROJECT_NUMBER,
    PROJECT_NAME,
    PROJECT_SLUG,
) != (
    7,
    "CompEvol@beast2",
    "CompEvol__beast2",
):
    raise AssertionError(
        "Unexpected Project 7 identity."
    )


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    previous_checkpoint = json.load(
        checkpoint_file
    )


if previous_checkpoint.get(
    "Status"
) not in {
    "CLEAN_REC_VALIDATION_PASSED",
    "NOISE_HELPERS_VALIDATION_PASSED",
}:
    raise AssertionError(
        "Step 6B has not passed.\n"
        f"Observed checkpoint status: "
        f"{previous_checkpoint.get('Status')}"
    )


if not bool(
    previous_checkpoint.get(
        "ZeroPercentExactReproductionPassed",
        False,
    )
):
    raise AssertionError(
        "The checkpoint does not confirm the successful "
        "Step 6B exact 0% validation."
    )


# Defragment and standardise indexes.
CLEAN_RAW_TRAINING_HISTORY = (
    CLEAN_RAW_TRAINING_HISTORY
    .copy(deep=True)
    .reset_index(drop=True)
)


CLEAN_RAW_EVALUATION_HISTORY = (
    CLEAN_RAW_EVALUATION_HISTORY
    .copy(deep=True)
    .reset_index(drop=True)
)


CLEAN_MODEL_ALL_DATA = (
    CLEAN_MODEL_ALL_DATA
    .copy(deep=True)
    .reset_index(drop=True)
)


CLEAN_MODEL_TRAINING_DATA = (
    CLEAN_MODEL_TRAINING_DATA
    .copy(deep=True)
    .reset_index(drop=True)
)


CLEAN_MODEL_EVALUATION_DATA = (
    CLEAN_MODEL_EVALUATION_DATA
    .copy(deep=True)
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 2. Authoritative schema restoration helper
# ---------------------------------------------------------

def conform_to_reference_schema(
    candidate_frame,
    reference_frame,
    frame_name,
):
    """
    Restore:
      - exact reference row order
      - exact reference column order
      - exact reference dtypes

    Values may differ only where the experiment intends
    them to differ.
    """

    candidate = (
        candidate_frame
        .copy()
        .reset_index(drop=True)
    )


    reference = (
        reference_frame
        .copy()
        .reset_index(drop=True)
    )


    if len(candidate) != len(reference):
        raise AssertionError(
            f"{frame_name}: row-count mismatch.\n"
            f"Candidate rows: {len(candidate)}\n"
            f"Reference rows: {len(reference)}"
        )


    missing_columns = [
        column
        for column in reference.columns
        if column not in candidate.columns
    ]


    unexpected_columns = [
        column
        for column in candidate.columns
        if column not in reference.columns
    ]


    if missing_columns:
        raise RuntimeError(
            f"{frame_name}: missing reference columns:\n"
            + "\n".join(
                map(str, missing_columns)
            )
        )


    if unexpected_columns:
        raise RuntimeError(
            f"{frame_name}: unexpected columns:\n"
            + "\n".join(
                map(str, unexpected_columns)
            )
        )


    # Restore authoritative column order.
    candidate = candidate[
        list(reference.columns)
    ].copy()


    # Restore every authoritative dtype.
    for column in reference.columns:
        reference_dtype = reference[
            column
        ].dtype


        candidate_series = candidate[
            column
        ]


        if pd.api.types.is_integer_dtype(
            reference_dtype
        ):
            numeric_values = pd.to_numeric(
                candidate_series,
                errors="coerce",
            )


            if numeric_values.isna().any():
                raise AssertionError(
                    f"{frame_name}.{column} cannot be "
                    "restored to an integer dtype because "
                    "it contains missing or non-numeric "
                    "values."
                )


            float_values = numeric_values.to_numpy(
                dtype=float
            )


            if not np.isfinite(
                float_values
            ).all():
                raise AssertionError(
                    f"{frame_name}.{column} contains "
                    "non-finite values."
                )


            if not np.isclose(
                float_values,
                np.round(float_values),
                rtol=0.0,
                atol=1e-10,
            ).all():
                bad_positions = np.flatnonzero(
                    ~np.isclose(
                        float_values,
                        np.round(float_values),
                        rtol=0.0,
                        atol=1e-10,
                    )
                )[:10]


                raise AssertionError(
                    f"{frame_name}.{column} contains "
                    "non-integer values and cannot be "
                    "restored safely.\n"
                    f"Example row positions: "
                    f"{bad_positions.tolist()}"
                )


            candidate[
                column
            ] = pd.Series(
                np.round(
                    float_values
                ),
                index=candidate.index,
            ).astype(
                reference_dtype
            )


        elif pd.api.types.is_float_dtype(
            reference_dtype
        ):
            candidate[
                column
            ] = pd.to_numeric(
                candidate_series,
                errors="raise",
            ).astype(
                reference_dtype
            )


        elif pd.api.types.is_bool_dtype(
            reference_dtype
        ):
            candidate[
                column
            ] = candidate_series.astype(
                reference_dtype
            )


        elif pd.api.types.is_datetime64_any_dtype(
            reference_dtype
        ):
            candidate[
                column
            ] = pd.to_datetime(
                candidate_series,
                errors="raise",
                utc=True,
            ).astype(
                reference_dtype
            )


        else:
            candidate[
                column
            ] = candidate_series.astype(
                reference_dtype
            )


    # Build-Test keys must remain in the exact requested
    # order.
    if {
        "Build",
        "Test",
    }.issubset(
        reference.columns
    ):
        candidate_builds = pd.to_numeric(
            candidate[
                "Build"
            ],
            errors="raise",
        ).to_numpy(dtype=np.int64)


        reference_builds = pd.to_numeric(
            reference[
                "Build"
            ],
            errors="raise",
        ).to_numpy(dtype=np.int64)


        candidate_tests = candidate[
            "Test"
        ].astype(str).to_numpy()


        reference_tests = reference[
            "Test"
        ].astype(str).to_numpy()


        if not np.array_equal(
            candidate_builds,
            reference_builds,
        ):
            raise AssertionError(
                f"{frame_name}: Build order changed."
            )


        if not np.array_equal(
            candidate_tests,
            reference_tests,
        ):
            raise AssertionError(
                f"{frame_name}: Test order changed."
            )


    return candidate


# ---------------------------------------------------------
# 3. Corrected key-alignment helper
# ---------------------------------------------------------

def align_source_rows(
    source_rows,
    requested_rows,
    frame_name="aligned rows",
):
    """
    Align source rows to requested Build-Test order and
    restore the requested authoritative schema.
    """

    source = (
        source_rows
        .copy()
        .reset_index(drop=True)
    )


    requested = (
        requested_rows
        .copy()
        .reset_index(drop=True)
    )


    for frame in [
        source,
        requested,
    ]:
        frame[
            "Build"
        ] = pd.to_numeric(
            frame[
                "Build"
            ],
            errors="raise",
        ).astype(np.int64)


        frame[
            "Test"
        ] = frame[
            "Test"
        ].astype(str)


    if source[
        [
            "Build",
            "Test",
        ]
    ].duplicated().any():
        raise AssertionError(
            f"{frame_name}: source contains duplicate "
            "Build-Test keys."
        )


    if requested[
        [
            "Build",
            "Test",
        ]
    ].duplicated().any():
        raise AssertionError(
            f"{frame_name}: requested rows contain "
            "duplicate Build-Test keys."
        )


    requested_keys = requested[
        [
            "Build",
            "Test",
        ]
    ].copy()


    requested_keys[
        "_RequestedRowOrder"
    ] = np.arange(
        len(requested_keys),
        dtype=np.int64,
    )


    aligned = (
        requested_keys
        .merge(
            source,
            on=[
                "Build",
                "Test",
            ],
            how="left",
            validate="one_to_one",
            indicator=True,
        )
        .sort_values(
            "_RequestedRowOrder",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    missing_rows = int(
        (
            aligned[
                "_merge"
            ] != "both"
        ).sum()
    )


    if missing_rows != 0:
        missing_examples = aligned.loc[
            aligned[
                "_merge"
            ] != "both",
            [
                "Build",
                "Test",
            ],
        ].head(20)


        display(
            missing_examples
        )


        raise AssertionError(
            f"{frame_name}: {missing_rows} requested "
            "Build-Test rows could not be aligned."
        )


    aligned = aligned.drop(
        columns=[
            "_RequestedRowOrder",
            "_merge",
        ]
    )


    return conform_to_reference_schema(
        candidate_frame=aligned,
        reference_frame=requested,
        frame_name=frame_name,
    )


# ---------------------------------------------------------
# 4. Corrected noisy-label extraction
# ---------------------------------------------------------

def extract_model_ready_noisy_verdicts(
    noisy_history,
    requested_training_rows,
):
    noisy_source = (
        noisy_history[
            [
                "Build",
                "Test",
                "Verdict",
            ]
        ]
        .copy()
        .rename(
            columns={
                "Verdict":
                    "NoisyVerdict",
            }
        )
    )


    requested_keys = (
        requested_training_rows[
            [
                "Build",
                "Test",
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )


    for frame in [
        noisy_source,
        requested_keys,
    ]:
        frame[
            "Build"
        ] = pd.to_numeric(
            frame[
                "Build"
            ],
            errors="raise",
        ).astype(np.int64)


        frame[
            "Test"
        ] = frame[
            "Test"
        ].astype(str)


    if noisy_source[
        [
            "Build",
            "Test",
        ]
    ].duplicated().any():
        raise AssertionError(
            "Noisy raw history contains duplicate "
            "Build-Test keys."
        )


    requested_keys[
        "_RequestedRowOrder"
    ] = np.arange(
        len(requested_keys),
        dtype=np.int64,
    )


    aligned = (
        requested_keys
        .merge(
            noisy_source,
            on=[
                "Build",
                "Test",
            ],
            how="left",
            validate="one_to_one",
            indicator=True,
        )
        .sort_values(
            "_RequestedRowOrder",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    if not (
        aligned[
            "_merge"
        ] == "both"
    ).all():
        raise AssertionError(
            "At least one retained training row could "
            "not be matched to its noisy raw verdict."
        )


    return pd.to_numeric(
        aligned[
            "NoisyVerdict"
        ],
        errors="raise",
    ).astype(np.int64)


# ---------------------------------------------------------
# 5. Corrected complete noisy-condition helper
# ---------------------------------------------------------

def create_project_7_noisy_condition(
    noise_percent,
    repetition_seed,
):
    (
        noisy_training_history,
        noise_manifest,
        noise_summary,
    ) = inject_training_verdict_noise(
        CLEAN_RAW_TRAINING_HISTORY,
        noise_percent,
        repetition_seed,
    )


    # Evaluation verdicts remain clean. They are included
    # only so evaluation REC history can reflect corrupted
    # prior training history.
    full_raw_history = pd.concat(
        [
            noisy_training_history,
            CLEAN_RAW_EVALUATION_HISTORY,
        ],
        ignore_index=True,
    )


    raw_noisy_rec = (
        reconstruct_project_7_raw_rec(
            full_raw_history
        )
        .reset_index(drop=True)
    )


    if len(raw_noisy_rec) != len(
        CLEAN_MODEL_ALL_DATA
    ):
        raise AssertionError(
            "The noisy REC reconstruction returned the "
            "wrong model-ready row count."
        )


    raw_noisy_with_keys = pd.concat(
        [
            CLEAN_MODEL_ALL_DATA[
                [
                    "Build",
                    "Test",
                ]
            ].reset_index(drop=True),

            raw_noisy_rec.reset_index(
                drop=True
            ),
        ],
        axis=1,
    )


    anchored_all_model_data = (
        apply_project_7_clean_rec_anchor(
            CLEAN_MODEL_ALL_DATA,
            raw_noisy_with_keys,
        )
    )


    # Critical correction: restore authoritative schema.
    anchored_all_model_data = (
        conform_to_reference_schema(
            candidate_frame=(
                anchored_all_model_data
            ),
            reference_frame=(
                CLEAN_MODEL_ALL_DATA
            ),
            frame_name=(
                "anchored all model-ready data"
            ),
        )
    )


    noisy_training_data = align_source_rows(
        source_rows=(
            anchored_all_model_data
        ),
        requested_rows=(
            CLEAN_MODEL_TRAINING_DATA
        ),
        frame_name=(
            "noisy model-ready training data"
        ),
    )


    noisy_training_verdicts = (
        extract_model_ready_noisy_verdicts(
            noisy_history=(
                noisy_training_history
            ),
            requested_training_rows=(
                CLEAN_MODEL_TRAINING_DATA
            ),
        )
    )


    noisy_training_data[
        "Verdict"
    ] = noisy_training_verdicts.to_numpy(
        dtype=np.int64
    )


    # Restore Verdict dtype and all other authoritative
    # schema details after replacing labels.
    noisy_training_data = (
        conform_to_reference_schema(
            candidate_frame=(
                noisy_training_data
            ),
            reference_frame=(
                CLEAN_MODEL_TRAINING_DATA
            ),
            frame_name=(
                "final noisy training condition"
            ),
        )
    )


    noisy_evaluation_data = align_source_rows(
        source_rows=(
            anchored_all_model_data
        ),
        requested_rows=(
            CLEAN_MODEL_EVALUATION_DATA
        ),
        frame_name=(
            "noisy-history evaluation condition"
        ),
    )


    noisy_evaluation_data = (
        conform_to_reference_schema(
            candidate_frame=(
                noisy_evaluation_data
            ),
            reference_frame=(
                CLEAN_MODEL_EVALUATION_DATA
            ),
            frame_name=(
                "final evaluation condition"
            ),
        )
    )


    evaluation_verdict_changes = int(
        (
            noisy_evaluation_data[
                "Verdict"
            ].to_numpy()
            !=
            CLEAN_MODEL_EVALUATION_DATA[
                "Verdict"
            ].to_numpy()
        ).sum()
    )


    if evaluation_verdict_changes != 0:
        raise AssertionError(
            "Evaluation verdicts changed under training "
            "noise."
        )


    if len(noisy_training_data) != len(
        CLEAN_MODEL_TRAINING_DATA
    ):
        raise AssertionError(
            "The fixed model-ready training row count "
            "changed."
        )


    if len(noisy_evaluation_data) != len(
        CLEAN_MODEL_EVALUATION_DATA
    ):
        raise AssertionError(
            "The clean evaluation row count changed."
        )


    return {
        "NoisyTrainingHistory":
            noisy_training_history,

        "NoiseManifest":
            noise_manifest,

        "NoiseSummary":
            noise_summary,

        "RawNoisyREC":
            raw_noisy_with_keys,

        "AnchoredAllModelData":
            anchored_all_model_data,

        "NoisyTrainingData":
            noisy_training_data,

        "NoisyEvaluationData":
            noisy_evaluation_data,
    }


# Compatibility alias.
create_noisy_condition = (
    create_project_7_noisy_condition
)


# ---------------------------------------------------------
# 6. Diagnose the previous 0% failure
# ---------------------------------------------------------

def create_exact_comparison_audit(
    observed_frame,
    reference_frame,
):
    observed = (
        observed_frame
        .copy()
        .reset_index(drop=True)
    )


    reference = (
        reference_frame
        .copy()
        .reset_index(drop=True)
    )


    all_columns = list(
        reference.columns
    )


    audit_records = []


    for column in all_columns:
        observed_dtype = str(
            observed[
                column
            ].dtype
        )


        reference_dtype = str(
            reference[
                column
            ].dtype
        )


        values_equal = bool(
            observed[
                column
            ].equals(
                reference[
                    column
                ]
            )
        )


        if pd.api.types.is_numeric_dtype(
            reference[
                column
            ].dtype
        ):
            numeric_value_mismatches = int(
                (
                    observed[
                        column
                    ].to_numpy()
                    !=
                    reference[
                        column
                    ].to_numpy()
                ).sum()
            )

        else:
            numeric_value_mismatches = int(
                (
                    observed[
                        column
                    ].astype(str)
                    .to_numpy()
                    !=
                    reference[
                        column
                    ].astype(str)
                    .to_numpy()
                ).sum()
            )


        audit_records.append({
            "Column":
                column,

            "ObservedDtype":
                observed_dtype,

            "ReferenceDtype":
                reference_dtype,

            "DtypeMatches":
                bool(
                    observed_dtype
                    == reference_dtype
                ),

            "ValueMismatches":
                numeric_value_mismatches,

            "SeriesEquals":
                values_equal,
        })


    return pd.DataFrame(
        audit_records
    )


# ---------------------------------------------------------
# 7. Revalidate every seed-1 nested noise mask
# ---------------------------------------------------------

seed_1_manifests = {}
seed_1_summaries = []


for noise_level in NOISE_LEVELS:
    (
        _,
        noise_manifest,
        noise_summary,
    ) = inject_training_verdict_noise(
        CLEAN_RAW_TRAINING_HISTORY,
        noise_level,
        1,
    )


    seed_1_manifests[
        noise_level
    ] = noise_manifest


    seed_1_summaries.append(
        noise_summary
    )


noise_level_summary_seed_1 = pd.DataFrame(
    seed_1_summaries
)


nestedness_records = []


for lower_noise, upper_noise in zip(
    NOISE_LEVELS[:-1],
    NOISE_LEVELS[1:],
):
    lower_manifest = seed_1_manifests[
        lower_noise
    ]


    upper_manifest = seed_1_manifests[
        upper_noise
    ]


    lower_flips = lower_manifest[
        "Flipped"
    ].to_numpy(dtype=bool)


    upper_flips = upper_manifest[
        "Flipped"
    ].to_numpy(dtype=bool)


    lower_rows_missing_from_upper = int(
        (
            lower_flips
            & ~upper_flips
        ).sum()
    )


    same_uniforms = bool(
        np.array_equal(
            lower_manifest[
                "FlipUniform"
            ].to_numpy(),

            upper_manifest[
                "FlipUniform"
            ].to_numpy(),
        )
    )


    same_subtypes = bool(
        np.array_equal(
            lower_manifest[
                "SampledFailureSubtype"
            ].to_numpy(),

            upper_manifest[
                "SampledFailureSubtype"
            ].to_numpy(),
        )
    )


    nestedness_records.append({
        "LowerNoisePercent":
            int(lower_noise),

        "UpperNoisePercent":
            int(upper_noise),

        "LowerFlippedRows":
            int(lower_flips.sum()),

        "UpperFlippedRows":
            int(upper_flips.sum()),

        "LowerRowsMissingFromUpperMask":
            lower_rows_missing_from_upper,

        "SameFlipUniforms":
            same_uniforms,

        "SameSampledFailureSubtypes":
            same_subtypes,

        "NestedMaskPassed":
            bool(
                lower_rows_missing_from_upper
                == 0
            ),
    })


nestedness_audit = pd.DataFrame(
    nestedness_records
)


if not nestedness_audit[
    "NestedMaskPassed"
].all():
    raise AssertionError(
        "At least one adjacent noise mask is not nested."
    )


if not nestedness_audit[
    "SameFlipUniforms"
].all():
    raise AssertionError(
        "Flip uniforms changed between noise levels."
    )


if not nestedness_audit[
    "SameSampledFailureSubtypes"
].all():
    raise AssertionError(
        "Sampled failure subtypes changed between noise "
        "levels."
    )


# ---------------------------------------------------------
# 8. Reproducibility and different-seed checks
# ---------------------------------------------------------

(
    history_25_seed_1_a,
    manifest_25_seed_1_a,
    summary_25_seed_1,
) = inject_training_verdict_noise(
    CLEAN_RAW_TRAINING_HISTORY,
    25,
    1,
)


(
    history_25_seed_1_b,
    manifest_25_seed_1_b,
    _,
) = inject_training_verdict_noise(
    CLEAN_RAW_TRAINING_HISTORY,
    25,
    1,
)


(
    _,
    manifest_25_seed_2,
    _,
) = inject_training_verdict_noise(
    CLEAN_RAW_TRAINING_HISTORY,
    25,
    2,
)


same_seed_history_passed = bool(
    history_25_seed_1_a.equals(
        history_25_seed_1_b
    )
)


same_seed_manifest_passed = bool(
    manifest_25_seed_1_a.equals(
        manifest_25_seed_1_b
    )
)


different_seed_mask_passed = bool(
    not np.array_equal(
        manifest_25_seed_1_a[
            "Flipped"
        ].to_numpy(dtype=bool),

        manifest_25_seed_2[
            "Flipped"
        ].to_numpy(dtype=bool),
    )
)


reproducibility_summary = pd.DataFrame([
    {
        "Check":
            "Same seed noisy history",

        "Passed":
            same_seed_history_passed,
    },
    {
        "Check":
            "Same seed complete manifest",

        "Passed":
            same_seed_manifest_passed,
    },
    {
        "Check":
            "Different seed flip mask differs",

        "Passed":
            different_seed_mask_passed,
    },
    {
        "Check":
            "All adjacent seed-1 masks nested",

        "Passed":
            bool(
                nestedness_audit[
                    "NestedMaskPassed"
                ].all()
            ),
    },
])


if not reproducibility_summary[
    "Passed"
].all():
    display(
        reproducibility_summary
    )


    raise AssertionError(
        "Noise reproducibility validation failed."
    )


# ---------------------------------------------------------
# 9. Complete corrected 0% validation
# ---------------------------------------------------------

zero_condition = (
    create_project_7_noisy_condition(
        noise_percent=0,
        repetition_seed=1,
    )
)


zero_training = zero_condition[
    "NoisyTrainingData"
]


zero_evaluation = zero_condition[
    "NoisyEvaluationData"
]


zero_all_model = zero_condition[
    "AnchoredAllModelData"
]


zero_all_model_exact = bool(
    zero_all_model.equals(
        CLEAN_MODEL_ALL_DATA
    )
)


zero_training_exact = bool(
    zero_training.equals(
        CLEAN_MODEL_TRAINING_DATA
    )
)


zero_evaluation_exact = bool(
    zero_evaluation.equals(
        CLEAN_MODEL_EVALUATION_DATA
    )
)


zero_all_model_audit = (
    create_exact_comparison_audit(
        observed_frame=zero_all_model,
        reference_frame=CLEAN_MODEL_ALL_DATA,
    )
)


zero_training_audit = (
    create_exact_comparison_audit(
        observed_frame=zero_training,
        reference_frame=(
            CLEAN_MODEL_TRAINING_DATA
        ),
    )
)


zero_evaluation_audit = (
    create_exact_comparison_audit(
        observed_frame=zero_evaluation,
        reference_frame=(
            CLEAN_MODEL_EVALUATION_DATA
        ),
    )
)


if not zero_all_model_exact:
    display(
        zero_all_model_audit[
            ~zero_all_model_audit[
                "SeriesEquals"
            ]
        ]
    )


    raise AssertionError(
        "Corrected 0% all-model-ready condition is not "
        "exactly equal to the clean reference."
    )


if not zero_training_exact:
    display(
        zero_training_audit[
            ~zero_training_audit[
                "SeriesEquals"
            ]
        ]
    )


    raise AssertionError(
        "Corrected 0% training condition is not exactly "
        "equal to the clean reference."
    )


if not zero_evaluation_exact:
    display(
        zero_evaluation_audit[
            ~zero_evaluation_audit[
                "SeriesEquals"
            ]
        ]
    )


    raise AssertionError(
        "Corrected 0% evaluation condition is not "
        "exactly equal to the clean reference."
    )


zero_raw_rows_flipped = int(
    zero_condition[
        "NoiseSummary"
    ][
        "NumberFlipped"
    ]
)


if zero_raw_rows_flipped != 0:
    raise AssertionError(
        "The 0% condition flipped raw rows."
    )


# ---------------------------------------------------------
# 10. Complete corrected 25% validation
# ---------------------------------------------------------

quarter_condition = (
    create_project_7_noisy_condition(
        noise_percent=25,
        repetition_seed=1,
    )
)


quarter_training = quarter_condition[
    "NoisyTrainingData"
]


quarter_evaluation = quarter_condition[
    "NoisyEvaluationData"
]


quarter_manifest = quarter_condition[
    "NoiseManifest"
]


quarter_summary = quarter_condition[
    "NoiseSummary"
]


quarter_training_label_changes = int(
    (
        quarter_training[
            "Verdict"
        ].to_numpy()
        !=
        CLEAN_MODEL_TRAINING_DATA[
            "Verdict"
        ].to_numpy()
    ).sum()
)


# Match manifest flips to retained model-ready rows.
retained_manifest_keys = (
    CLEAN_MODEL_TRAINING_DATA[
        [
            "Build",
            "Test",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


retained_manifest_keys[
    "Build"
] = pd.to_numeric(
    retained_manifest_keys[
        "Build"
    ],
    errors="raise",
).astype(np.int64)


retained_manifest_keys[
    "Test"
] = retained_manifest_keys[
    "Test"
].astype(str)


retained_manifest_keys[
    "_RequestedRowOrder"
] = np.arange(
    len(retained_manifest_keys),
    dtype=np.int64,
)


manifest_source = quarter_manifest[
    [
        "Build",
        "Test",
        "Flipped",
    ]
].copy()


manifest_source[
    "Build"
] = pd.to_numeric(
    manifest_source[
        "Build"
    ],
    errors="raise",
).astype(np.int64)


manifest_source[
    "Test"
] = manifest_source[
    "Test"
].astype(str)


retained_manifest = (
    retained_manifest_keys
    .merge(
        manifest_source,
        on=[
            "Build",
            "Test",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    .sort_values(
        "_RequestedRowOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if not (
    retained_manifest[
        "_merge"
    ] == "both"
).all():
    raise AssertionError(
        "At least one retained model-ready row could not "
        "be matched to the 25% manifest."
    )


expected_quarter_label_changes = int(
    retained_manifest[
        "Flipped"
    ].astype(bool).sum()
)


if quarter_training_label_changes != (
    expected_quarter_label_changes
):
    raise AssertionError(
        "The number of changed retained labels does not "
        "match the retained flipped raw rows.\n"
        f"Observed label changes: "
        f"{quarter_training_label_changes}\n"
        f"Expected from manifest: "
        f"{expected_quarter_label_changes}"
    )


def count_exact_value_changes(
    observed_frame,
    reference_frame,
    columns,
):
    change_count = 0


    for column in columns:
        observed_values = observed_frame[
            column
        ].to_numpy()


        reference_values = reference_frame[
            column
        ].to_numpy()


        change_count += int(
            (
                observed_values
                != reference_values
            ).sum()
        )


    return int(
        change_count
    )


quarter_training_dependent_changes = (
    count_exact_value_changes(
        observed_frame=quarter_training,
        reference_frame=(
            CLEAN_MODEL_TRAINING_DATA
        ),
        columns=REC_DEPENDENT_COLUMNS,
    )
)


quarter_evaluation_dependent_changes = (
    count_exact_value_changes(
        observed_frame=quarter_evaluation,
        reference_frame=(
            CLEAN_MODEL_EVALUATION_DATA
        ),
        columns=REC_DEPENDENT_COLUMNS,
    )
)


quarter_training_independent_changes = (
    count_exact_value_changes(
        observed_frame=quarter_training,
        reference_frame=(
            CLEAN_MODEL_TRAINING_DATA
        ),
        columns=REC_INDEPENDENT_COLUMNS,
    )
)


quarter_evaluation_independent_changes = (
    count_exact_value_changes(
        observed_frame=quarter_evaluation,
        reference_frame=(
            CLEAN_MODEL_EVALUATION_DATA
        ),
        columns=REC_INDEPENDENT_COLUMNS,
    )
)


if quarter_training_dependent_changes <= 0:
    raise AssertionError(
        "The 25% condition did not change any dependent "
        "training REC value."
    )


if quarter_evaluation_dependent_changes <= 0:
    raise AssertionError(
        "The 25% condition did not change any dependent "
        "evaluation REC value."
    )


if quarter_training_independent_changes != 0:
    raise AssertionError(
        "The 25% condition changed independent training "
        "REC values."
    )


if quarter_evaluation_independent_changes != 0:
    raise AssertionError(
        "The 25% condition changed independent "
        "evaluation REC values."
    )


training_preserved_columns = [
    column
    for column in (
        CLEAN_MODEL_TRAINING_DATA.columns
    )
    if column not in (
        set(
            REC_DEPENDENT_COLUMNS
        )
        | {
            "Verdict",
        }
    )
]


evaluation_preserved_columns = [
    column
    for column in (
        CLEAN_MODEL_EVALUATION_DATA.columns
    )
    if column not in set(
        REC_DEPENDENT_COLUMNS
    )
]


quarter_training_non_target_columns_changed = int(
    sum(
        not quarter_training[
            column
        ].equals(
            CLEAN_MODEL_TRAINING_DATA[
                column
            ]
        )

        for column in (
            training_preserved_columns
        )
    )
)


quarter_evaluation_non_target_columns_changed = int(
    sum(
        not quarter_evaluation[
            column
        ].equals(
            CLEAN_MODEL_EVALUATION_DATA[
                column
            ]
        )

        for column in (
            evaluation_preserved_columns
        )
    )
)


if quarter_training_non_target_columns_changed != 0:
    raise AssertionError(
        "The 25% condition changed at least one "
        "non-target training column."
    )


if quarter_evaluation_non_target_columns_changed != 0:
    raise AssertionError(
        "The 25% condition changed at least one "
        "non-target evaluation column."
    )


quarter_evaluation_verdict_changes = int(
    (
        quarter_evaluation[
            "Verdict"
        ].to_numpy()
        !=
        CLEAN_MODEL_EVALUATION_DATA[
            "Verdict"
        ].to_numpy()
    ).sum()
)


if quarter_evaluation_verdict_changes != 0:
    raise AssertionError(
        "Evaluation verdicts changed in the 25% "
        "condition."
    )


# ---------------------------------------------------------
# 11. Final condition-validation tables
# ---------------------------------------------------------

condition_validation = pd.DataFrame([
    {
        "NoisePercent":
            0,

        "Seed":
            1,

        "RawRowsFlipped":
            zero_raw_rows_flipped,

        "ModelTrainingRows":
            int(
                len(zero_training)
            ),

        "ModelTrainingLabelChanges":
            0,

        "TrainingDependentRECChanges":
            0,

        "EvaluationDependentRECChanges":
            0,

        "TrainingIndependentRECChanges":
            0,

        "EvaluationIndependentRECChanges":
            0,

        "TrainingExactCleanReproduction":
            zero_training_exact,

        "EvaluationExactCleanReproduction":
            zero_evaluation_exact,

        "SchemaRestored":
            True,
    },
    {
        "NoisePercent":
            25,

        "Seed":
            1,

        "RawRowsFlipped":
            int(
                quarter_summary[
                    "NumberFlipped"
                ]
            ),

        "ModelTrainingRows":
            int(
                len(quarter_training)
            ),

        "ModelTrainingLabelChanges":
            quarter_training_label_changes,

        "TrainingDependentRECChanges":
            quarter_training_dependent_changes,

        "EvaluationDependentRECChanges":
            quarter_evaluation_dependent_changes,

        "TrainingIndependentRECChanges":
            quarter_training_independent_changes,

        "EvaluationIndependentRECChanges":
            quarter_evaluation_independent_changes,

        "TrainingExactCleanReproduction":
            False,

        "EvaluationExactCleanReproduction":
            False,

        "SchemaRestored":
            True,
    },
])


schema_restoration_summary = pd.DataFrame([
    {
        "Scope":
            "ALL_MODEL_READY_0_PERCENT",

        "Rows":
            int(
                len(zero_all_model)
            ),

        "Columns":
            int(
                len(zero_all_model.columns)
            ),

        "ExactFrameEquality":
            zero_all_model_exact,

        "ColumnsWithDtypeMismatch":
            int(
                (
                    ~zero_all_model_audit[
                        "DtypeMatches"
                    ]
                ).sum()
            ),

        "ColumnsWithValueMismatch":
            int(
                (
                    zero_all_model_audit[
                        "ValueMismatches"
                    ] > 0
                ).sum()
            ),
    },
    {
        "Scope":
            "TRAINING_0_PERCENT",

        "Rows":
            int(
                len(zero_training)
            ),

        "Columns":
            int(
                len(zero_training.columns)
            ),

        "ExactFrameEquality":
            zero_training_exact,

        "ColumnsWithDtypeMismatch":
            int(
                (
                    ~zero_training_audit[
                        "DtypeMatches"
                    ]
                ).sum()
            ),

        "ColumnsWithValueMismatch":
            int(
                (
                    zero_training_audit[
                        "ValueMismatches"
                    ] > 0
                ).sum()
            ),
    },
    {
        "Scope":
            "EVALUATION_0_PERCENT",

        "Rows":
            int(
                len(zero_evaluation)
            ),

        "Columns":
            int(
                len(zero_evaluation.columns)
            ),

        "ExactFrameEquality":
            zero_evaluation_exact,

        "ColumnsWithDtypeMismatch":
            int(
                (
                    ~zero_evaluation_audit[
                        "DtypeMatches"
                    ]
                ).sum()
            ),

        "ColumnsWithValueMismatch":
            int(
                (
                    zero_evaluation_audit[
                        "ValueMismatches"
                    ] > 0
                ).sum()
            ),
    },
])


# ---------------------------------------------------------
# 12. Save corrected permanent Step 7A artefacts
# ---------------------------------------------------------

NOISE_LEVEL_SUMMARY_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_noise_level_summary_seed_01.csv"
)


NESTEDNESS_AUDIT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_noise_nestedness_audit_seed_01.csv"
)


REPRODUCIBILITY_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_noise_reproducibility_summary.csv"
)


CONDITION_VALIDATION_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_noisy_condition_validation.csv"
)


SCHEMA_RESTORATION_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_noise_schema_restoration_validation.csv"
)


ZERO_TRAINING_AUDIT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_zero_percent_training_exact_audit.csv"
)


ZERO_EVALUATION_AUDIT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_zero_percent_evaluation_exact_audit.csv"
)


SEED_1_25_MANIFEST_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_noise_manifest_seed_01_noise_25.parquet"
)


NOISE_HELPER_REPORT_PATH = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_noise_helper_validation_report.json"
)


noise_level_summary_seed_1.to_csv(
    NOISE_LEVEL_SUMMARY_PATH,
    index=False,
)


nestedness_audit.to_csv(
    NESTEDNESS_AUDIT_PATH,
    index=False,
)


reproducibility_summary.to_csv(
    REPRODUCIBILITY_PATH,
    index=False,
)


condition_validation.to_csv(
    CONDITION_VALIDATION_PATH,
    index=False,
)


schema_restoration_summary.to_csv(
    SCHEMA_RESTORATION_PATH,
    index=False,
)


zero_training_audit.to_csv(
    ZERO_TRAINING_AUDIT_PATH,
    index=False,
)


zero_evaluation_audit.to_csv(
    ZERO_EVALUATION_AUDIT_PATH,
    index=False,
)


quarter_manifest.to_parquet(
    SEED_1_25_MANIFEST_PATH,
    index=False,
)


completed_at = (
    pd.Timestamp.utcnow().isoformat()
)


noise_helper_report = {
    "ProjectNumber":
        int(PROJECT_NUMBER),

    "Project":
        str(PROJECT_NAME),

    "ProjectSlug":
        str(PROJECT_SLUG),

    "Status":
        "PASS",

    "CorrectionApplied": {
        "Cause":
            (
                "Clean-anchor arithmetic converted "
                "integer REC columns to floating-point "
                "dtypes. DataFrame.equals therefore "
                "returned False despite equal numeric "
                "values."
            ),

        "Resolution":
            (
                "Restore authoritative row order, "
                "column order and dtype for every "
                "condition after REC anchoring."
            ),

        "AuthoritativeSchemaRestoration":
            True,
    },

    "ExperimentGrid": {
        "NoiseLevelsPercent":
            list(NOISE_LEVELS),

        "RepetitionSeeds":
            list(REPETITION_SEEDS),

        "Conditions":
            int(
                len(NOISE_LEVELS)
                * len(REPETITION_SEEDS)
            ),
    },

    "NoisePolicy": {
        "NoiseUnit":
            (
                "individual raw training execution "
                "verdict"
            ),

        "SeedPolicy":
            (
                "SHA256 project-specific random streams"
            ),

        "RandomStreams": [
            "flip_mask",
            "failure_subtype",
        ],

        "NestedMasks":
            True,

        "PassToFailurePolicy":
            (
                "sample clean project failure subtype"
            ),

        "FailureToPassPolicy":
            (
                "non-zero verdict becomes zero"
            ),
    },

    "ZeroPercentValidation": {
        "AllModelReadyExact":
            zero_all_model_exact,

        "TrainingExact":
            zero_training_exact,

        "EvaluationExact":
            zero_evaluation_exact,

        "RawRowsFlipped":
            zero_raw_rows_flipped,

        "TrainingColumnsWithDtypeMismatch":
            int(
                (
                    ~zero_training_audit[
                        "DtypeMatches"
                    ]
                ).sum()
            ),

        "EvaluationColumnsWithDtypeMismatch":
            int(
                (
                    ~zero_evaluation_audit[
                        "DtypeMatches"
                    ]
                ).sum()
            ),

        "TrainingColumnsWithValueMismatch":
            int(
                (
                    zero_training_audit[
                        "ValueMismatches"
                    ] > 0
                ).sum()
            ),

        "EvaluationColumnsWithValueMismatch":
            int(
                (
                    zero_evaluation_audit[
                        "ValueMismatches"
                    ] > 0
                ).sum()
            ),
    },

    "Seed1Noise25Validation": {
        "RawRowsFlipped":
            int(
                quarter_summary[
                    "NumberFlipped"
                ]
            ),

        "ModelTrainingLabelChanges":
            quarter_training_label_changes,

        "TrainingDependentRECChanges":
            quarter_training_dependent_changes,

        "EvaluationDependentRECChanges":
            quarter_evaluation_dependent_changes,

        "TrainingIndependentRECChanges":
            quarter_training_independent_changes,

        "EvaluationIndependentRECChanges":
            quarter_evaluation_independent_changes,

        "TrainingNonTargetColumnsChanged":
            quarter_training_non_target_columns_changed,

        "EvaluationNonTargetColumnsChanged":
            quarter_evaluation_non_target_columns_changed,

        "EvaluationVerdictChanges":
            quarter_evaluation_verdict_changes,
    },

    "Reproducibility": {
        "SameSeedHistory":
            same_seed_history_passed,

        "SameSeedManifest":
            same_seed_manifest_passed,

        "DifferentSeedMaskDiffers":
            different_seed_mask_passed,

        "AllAdjacentMasksNested":
            bool(
                nestedness_audit[
                    "NestedMaskPassed"
                ].all()
            ),

        "SameUniformsAcrossLevels":
            bool(
                nestedness_audit[
                    "SameFlipUniforms"
                ].all()
            ),

        "SameSampledSubtypesAcrossLevels":
            bool(
                nestedness_audit[
                    "SameSampledFailureSubtypes"
                ].all()
            ),
    },

    "Artefacts": {
        "NoiseLevelSummarySeed1":
            str(
                NOISE_LEVEL_SUMMARY_PATH
            ),

        "NestednessAuditSeed1":
            str(
                NESTEDNESS_AUDIT_PATH
            ),

        "ReproducibilitySummary":
            str(
                REPRODUCIBILITY_PATH
            ),

        "ConditionValidation":
            str(
                CONDITION_VALIDATION_PATH
            ),

        "SchemaRestorationValidation":
            str(
                SCHEMA_RESTORATION_PATH
            ),

        "ZeroTrainingAudit":
            str(
                ZERO_TRAINING_AUDIT_PATH
            ),

        "ZeroEvaluationAudit":
            str(
                ZERO_EVALUATION_AUDIT_PATH
            ),

        "Seed1Noise25Manifest":
            str(
                SEED_1_25_MANIFEST_PATH
            ),
    },

    "CompletedAtUTC":
        completed_at,
}


with open(
    NOISE_HELPER_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        noise_helper_report,
        report_file,
        indent=2,
    )


# ---------------------------------------------------------
# 13. Update the Project 7 checkpoint atomically
# ---------------------------------------------------------

with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    project_checkpoint = json.load(
        checkpoint_file
    )


project_checkpoint.update({
    "Status":
        "NOISE_HELPERS_VALIDATION_PASSED",

    "NoiseLevelsPercent":
        list(NOISE_LEVELS),

    "RepetitionSeeds":
        list(REPETITION_SEEDS),

    "NoiseConditionCount":
        int(
            len(NOISE_LEVELS)
            * len(REPETITION_SEEDS)
        ),

    "NoiseSeedPolicy":
        "SHA256_PROJECT_SEED_STREAM",

    "NoiseMaskPolicy":
        (
            "NESTED_COMMON_UNIFORMS_"
            "PER_PROJECT_SEED"
        ),

    "PassToFailureSubtypePolicy":
        (
            "PROJECT_CLEAN_FAILURE_"
            "SUBTYPE_DISTRIBUTION"
        ),

    "FailureToPassPolicy":
        "NONZERO_TO_ZERO",

    "ConditionSchemaRestorationPolicy":
        (
            "RESTORE_AUTHORITATIVE_COLUMN_ORDER_"
            "AND_DTYPES"
        ),

    "ZeroPercentAllModelReadyExactReproduction":
        zero_all_model_exact,

    "ZeroPercentTrainingExactReproduction":
        zero_training_exact,

    "ZeroPercentEvaluationExactReproduction":
        zero_evaluation_exact,

    "ZeroPercentTrainingColumnsWithDtypeMismatch":
        int(
            (
                ~zero_training_audit[
                    "DtypeMatches"
                ]
            ).sum()
        ),

    "ZeroPercentEvaluationColumnsWithDtypeMismatch":
        int(
            (
                ~zero_evaluation_audit[
                    "DtypeMatches"
                ]
            ).sum()
        ),

    "Seed1Noise25RawRowsFlipped":
        int(
            quarter_summary[
                "NumberFlipped"
            ]
        ),

    "Seed1Noise25ModelTrainingLabelChanges":
        quarter_training_label_changes,

    "Seed1Noise25TrainingDependentRECChanges":
        quarter_training_dependent_changes,

    "Seed1Noise25EvaluationDependentRECChanges":
        quarter_evaluation_dependent_changes,

    "Seed1Noise25TrainingIndependentRECChanges":
        quarter_training_independent_changes,

    "Seed1Noise25EvaluationIndependentRECChanges":
        quarter_evaluation_independent_changes,

    "NoiseNestednessValidationPassed":
        True,

    "NoiseSameSeedReproducibilityPassed":
        True,

    "NoiseDifferentSeedValidationPassed":
        True,

    "NoiseHelperValidationReport":
        str(
            NOISE_HELPER_REPORT_PATH
        ),

    "NoiseSchemaRestorationValidation":
        str(
            SCHEMA_RESTORATION_PATH
        ),

    "UpdatedAtUTC":
        completed_at,
})


temporary_checkpoint_path = (
    PROJECT_7_SELECTION_CHECKPOINT
    .with_suffix(
        ".json.tmp"
    )
)


with open(
    temporary_checkpoint_path,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        project_checkpoint,
        checkpoint_file,
        indent=2,
    )


os.replace(
    temporary_checkpoint_path,
    PROJECT_7_SELECTION_CHECKPOINT,
)


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint_verification = json.load(
        checkpoint_file
    )


if checkpoint_verification.get(
    "Status"
) != "NOISE_HELPERS_VALIDATION_PASSED":
    raise AssertionError(
        "The corrected Step 7A checkpoint was not "
        "written correctly."
    )


if not bool(
    checkpoint_verification.get(
        "ZeroPercentTrainingExactReproduction",
        False,
    )
):
    raise AssertionError(
        "The checkpoint does not confirm exact 0% "
        "training reproduction."
    )


if not bool(
    checkpoint_verification.get(
        "ZeroPercentEvaluationExactReproduction",
        False,
    )
):
    raise AssertionError(
        "The checkpoint does not confirm exact 0% "
        "evaluation reproduction."
    )


# ---------------------------------------------------------
# 14. Compact final output
# ---------------------------------------------------------

clear_output(
    wait=True
)


print(
    "=== PROJECT 7 STEP 7A RESULT ==="
)


print(
    "\nCorrection:"
)

print(
    "Cause:",
    (
        "REC anchoring changed integer feature dtypes "
        "to floating-point."
    )
)

print(
    "Fix:",
    (
        "Authoritative row order, column order and "
        "dtypes are restored after every condition."
    )
)


print(
    "\nFrozen experiment grid:"
)

print(
    "Noise levels:",
    NOISE_LEVELS
)

print(
    "Repetition seeds:",
    "1–30"
)

print(
    "Conditions:",
    len(NOISE_LEVELS)
    * len(REPETITION_SEEDS)
)


print(
    "\nSeed 1 noise summary:"
)

display(
    noise_level_summary_seed_1[
        [
            "NoisePercentRequested",
            "TrainingExecutionRows",
            "NumberFlipped",
            "RealisedNoisePercent",
            "PassToFailure",
            "FailureToPass",
        ]
    ]
)


print(
    "\nNested-mask audit:"
)

display(
    nestedness_audit
)

print(
    "All adjacent masks nested:",
    bool(
        nestedness_audit[
            "NestedMaskPassed"
        ].all()
    )
)

print(
    "Same uniforms across levels:",
    bool(
        nestedness_audit[
            "SameFlipUniforms"
        ].all()
    )
)

print(
    "Same sampled subtypes across levels:",
    bool(
        nestedness_audit[
            "SameSampledFailureSubtypes"
        ].all()
    )
)


print(
    "\nReproducibility:"
)

display(
    reproducibility_summary
)


print(
    "\nSchema-restoration validation:"
)

display(
    schema_restoration_summary
)


print(
    "\nEnd-to-end condition validation:"
)

display(
    condition_validation
)


print(
    "0% all model-ready exact reproduction:",
    zero_all_model_exact
)

print(
    "0% training exact reproduction:",
    zero_training_exact
)

print(
    "0% evaluation exact reproduction:",
    zero_evaluation_exact
)

print(
    "0% training dtype-mismatched columns:",
    int(
        (
            ~zero_training_audit[
                "DtypeMatches"
            ]
        ).sum()
    )
)

print(
    "0% evaluation dtype-mismatched columns:",
    int(
        (
            ~zero_evaluation_audit[
                "DtypeMatches"
            ]
        ).sum()
    )
)

print(
    "0% training value-mismatched columns:",
    int(
        (
            zero_training_audit[
                "ValueMismatches"
            ] > 0
        ).sum()
    )
)

print(
    "0% evaluation value-mismatched columns:",
    int(
        (
            zero_evaluation_audit[
                "ValueMismatches"
            ] > 0
        ).sum()
    )
)


print(
    "\n25% condition:"
)

print(
    "Raw rows flipped:",
    int(
        quarter_summary[
            "NumberFlipped"
        ]
    )
)

print(
    "Retained training label changes:",
    quarter_training_label_changes
)

print(
    "Training dependent REC changes:",
    quarter_training_dependent_changes
)

print(
    "Evaluation dependent REC changes:",
    quarter_evaluation_dependent_changes
)

print(
    "Training independent REC changes:",
    quarter_training_independent_changes
)

print(
    "Evaluation independent REC changes:",
    quarter_evaluation_independent_changes
)

print(
    "Training non-target columns changed:",
    quarter_training_non_target_columns_changed
)

print(
    "Evaluation non-target columns changed:",
    quarter_evaluation_non_target_columns_changed
)

print(
    "Evaluation verdict changes:",
    quarter_evaluation_verdict_changes
)


print(
    "\nNoise-helper report:"
)

print(
    NOISE_HELPER_REPORT_PATH
)


print(
    "\nValidation status:",
    "PASS"
)


print(
    "\nSUCCESS: The 0% condition now preserves values, "
    "column order and dtypes exactly."
)

print(
    "SUCCESS: All nine noise masks are nested for the "
    "same project seed."
)

print(
    "SUCCESS: Same-seed conditions are reproducible and "
    "different seeds produce different masks."
)

print(
    "SUCCESS: The 25% condition changes only retained "
    "training labels and verdict-dependent REC values."
)

print(
    "SUCCESS: Independent REC features, evaluation "
    "verdicts and non-target predictors remain clean."
)

print(
    "SUCCESS: Project 7 is ready for model and baseline "
    "helper validation."
)

=== PROJECT 7 STEP 7A RESULT ===

Correction:
Cause: REC anchoring changed integer feature dtypes to floating-point.
Fix: Authoritative row order, column order and dtypes are restored after every condition.

Frozen experiment grid:
Noise levels: [0, 5, 10, 15, 20, 25, 30, 40, 50]
Repetition seeds: 1–30
Conditions: 270

Seed 1 noise summary:


,NoisePercentRequested,TrainingExecutionRows,NumberFlipped,RealisedNoisePercent,PassToFailure,FailureToPass
0,0.0,19766,0,0.000000,0,0
1,5.0,19766,1003,5.074370,996,7
2,10.0,19766,2050,10.371345,2033,17
3,15.0,19766,3031,15.334413,3005,26
4,20.0,19766,4112,20.803400,4077,35
5,25.0,19766,5035,25.473035,4996,39
6,30.0,19766,5982,30.264090,5938,44
7,40.0,19766,7931,40.124456,7869,62
8,50.0,19766,9917,50.172013,9837,80



Nested-mask audit:


,LowerNoisePercent,UpperNoisePercent,LowerFlippedRows,UpperFlippedRows,LowerRowsMissingFromUpperMask,SameFlipUniforms,SameSampledFailureSubtypes,NestedMaskPassed
0,0,5,0,1003,0,True,True,True
1,5,10,1003,2050,0,True,True,True
2,10,15,2050,3031,0,True,True,True
3,15,20,3031,4112,0,True,True,True
4,20,25,4112,5035,0,True,True,True
5,25,30,5035,5982,0,True,True,True
6,30,40,5982,7931,0,True,True,True
7,40,50,7931,9917,0,True,True,True


All adjacent masks nested: True
Same uniforms across levels: True
Same sampled subtypes across levels: True

Reproducibility:


...


Schema-restoration validation:


,Scope,Rows,Columns,ExactFrameEquality,ColumnsWithDtypeMismatch,ColumnsWithValueMismatch
0,ALL_MODEL_READY_0_PERCENT,7638,156,True,0,0
1,TRAINING_0_PERCENT,3960,156,True,0,0
2,EVALUATION_0_PERCENT,3678,156,True,0,0



End-to-end condition validation:


,NoisePercent,Seed,RawRowsFlipped,ModelTrainingRows,ModelTrainingLabelChanges,TrainingDependentRECChanges,EvaluationDependentRECChanges,TrainingIndependentRECChanges,EvaluationIndependentRECChanges,TrainingExactCleanReproduction,EvaluationExactCleanReproduction,SchemaRestored
0,0,1,0,3960,0,0,0,0,0,True,True,True
1,25,1,5035,3960,1009,41897,26134,0,0,False,False,True


0% all model-ready exact reproduction: True
0% training exact reproduction: True
0% evaluation exact reproduction: True
0% training dtype-mismatched columns: 0
0% evaluation dtype-mismatched columns: 0
0% training value-mismatched columns: 0
0% evaluation value-mismatched columns: 0

25% condition:
Raw rows flipped: 5035
Retained training label changes: 1009
Training dependent REC changes: 41897
Evaluation dependent REC changes: 26134
Training independent REC changes: 0
Evaluation independent REC changes: 0
Training non-target columns changed: 0
Evaluation non-target columns changed: 0
Evaluation verdict changes: 0

Noise-helper report:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_preflight/beast2_noise_helper_validation_report.json

Validation status: PASS

SUCCESS: The 0% condition now preserves values, column order and dtypes exactly.
SUCCESS: All nine noise masks are nested for the same project seed.
SUCCESS: Same-seed conditions are reproduci

In [ ]:
# =========================================================
# PROJECT 7 — STEP 7B
# FROZEN MODEL / BASELINE / METRIC CONFIGURATION AUDIT
#
# This reads the permanent methodology files so the next
# cell can reuse the exact settings from Projects 1–6.
# It does not fit models or modify the checkpoint.
# =========================================================

from pathlib import Path
import json
import re

import pandas as pd
from IPython.display import clear_output, display


print(
    "=== PROJECT 7 STEP 7B: "
    "FROZEN CONFIGURATION AUDIT ==="
)


# ---------------------------------------------------------
# 1. Validate Project 7 state
# ---------------------------------------------------------

required = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",
    "PROJECT_PREFLIGHT_DIRECTORY",
    "PROJECT_7_SELECTION_CHECKPOINT",
]


missing = [
    name
    for name in required
    if name not in globals()
]


if missing:
    raise RuntimeError(
        "Missing Project 7 objects:\n"
        + "\n".join(missing)
    )


if (
    PROJECT_NUMBER,
    PROJECT_NAME,
    PROJECT_SLUG,
) != (
    7,
    "CompEvol@beast2",
    "CompEvol__beast2",
):
    raise AssertionError(
        "Unexpected Project 7 identity."
    )


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint_before = json.load(
        checkpoint_file
    )


if checkpoint_before.get(
    "Status"
) != "NOISE_HELPERS_VALIDATION_PASSED":
    raise AssertionError(
        "Step 7A has not passed.\n"
        f"Observed status: "
        f"{checkpoint_before.get('Status')}"
    )


# ---------------------------------------------------------
# 2. Locate permanent methodology files
# ---------------------------------------------------------

notes_directory = Path(
    "/content/drive/MyDrive/"
    "Thesis_Experiment/Notes"
)


methodology_path = (
    notes_directory
    / "methodology_checkpoint_fixed_holdout_v1.json"
)


formula_path = (
    notes_directory
    / "apfd_apfdc_formula_audit_v1.json"
)


for path in [
    methodology_path,
    formula_path,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required permanent file is missing:\n"
            f"{path}"
        )


# ---------------------------------------------------------
# 3. Load and flatten JSON
# ---------------------------------------------------------

def load_json(path):
    with open(
        path,
        "r",
        encoding="utf-8",
    ) as input_file:
        return json.load(input_file)


def flatten_json(
    value,
    source,
    path="$",
):
    rows = []


    if isinstance(value, dict):
        for key, child in value.items():
            rows.extend(
                flatten_json(
                    child,
                    source,
                    f"{path}.{key}",
                )
            )


    elif isinstance(value, list):
        for index, child in enumerate(value):
            rows.extend(
                flatten_json(
                    child,
                    source,
                    f"{path}[{index}]",
                )
            )


    else:
        rows.append({
            "Source":
                source,

            "JsonPath":
                path,

            "ValueType":
                type(value).__name__,

            "Value":
                str(value),
        })


    return rows


methodology = load_json(
    methodology_path
)


formula_audit = load_json(
    formula_path
)


flat_entries = pd.DataFrame(
    flatten_json(
        methodology,
        "METHODOLOGY",
    )
    + flatten_json(
        formula_audit,
        "FORMULA_AUDIT",
    )
)


# ---------------------------------------------------------
# 4. Extract model, baseline, metric and tie settings
# ---------------------------------------------------------

search_terms = [
    "RandomForest",
    "Random Forest",
    "XGBoost",
    "XGBClassifier",
    "LightGBM",
    "LGBMClassifier",
    "Naive Bayes",
    "GaussianNB",
    "hyperparameter",
    "n_estimators",
    "max_depth",
    "learning_rate",
    "subsample",
    "colsample",
    "num_leaves",
    "random_state",
    "LatestFail",
    "Latest Fail",
    "QTF",
    "Random baseline",
    "tie",
    "Test ascending",
    "APFDc",
    "APFD",
    "failure-test-execution",
]


pattern = "|".join(
    re.escape(term)
    for term in search_terms
)


combined_text = (
    flat_entries[
        "JsonPath"
    ].astype(str)
    + " "
    + flat_entries[
        "Value"
    ].astype(str)
)


configuration_matches = (
    flat_entries[
        combined_text.str.contains(
            pattern,
            case=False,
            regex=True,
            na=False,
        )
    ]
    .drop_duplicates()
    .sort_values(
        [
            "Source",
        "JsonPath",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if len(configuration_matches) == 0:
    raise AssertionError(
        "No matching configuration entries were found."
    )


# ---------------------------------------------------------
# 5. Save audit without changing checkpoint
# ---------------------------------------------------------

configuration_matches_path = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_frozen_configuration_matches.csv"
)


configuration_matches.to_csv(
    configuration_matches_path,
    index=False,
)


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint_after = json.load(
        checkpoint_file
    )


if checkpoint_after != checkpoint_before:
    raise AssertionError(
        "The audit unexpectedly changed the checkpoint."
    )


# ---------------------------------------------------------
# 6. Compact result
# ---------------------------------------------------------

clear_output(wait=True)


print(
    "=== PROJECT 7 STEP 7B RESULT ==="
)


print(
    "\nCurrent checkpoint status:",
    checkpoint_before.get("Status")
)


print(
    "\nMethodology top-level keys:"
)

print(
    (
        list(methodology.keys())
        if isinstance(methodology, dict)
        else []
    )
)


print(
    "\nFormula-audit top-level keys:"
)

print(
    (
        list(formula_audit.keys())
        if isinstance(formula_audit, dict)
        else []
    )
)


print(
    "\nFrozen configuration entries:"
)

display(
    configuration_matches
)


print(
    "\nEntries found:",
    len(configuration_matches)
)


print(
    "Saved audit:",
    configuration_matches_path
)


print(
    "Checkpoint changed:",
    False
)


print(
    "\nAudit status:",
    "COMPLETE"
)


print(
    "\nSUCCESS: The permanent model, baseline, metric "
    "and tie-policy entries were extracted."
)

print(
    "SUCCESS: No model was fitted and the Project 7 "
    "checkpoint was not modified."
)

=== PROJECT 7 STEP 7B RESULT ===

Current checkpoint status: NOISE_HELPERS_VALIDATION_PASSED

Methodology top-level keys:
['CheckpointName', 'FrozenAtUTC', 'Dataset', 'BuildOrdering', 'SplitProtocol', 'EvaluationProtocol', 'FailureMappingAssumption', 'PrimaryMetric', 'SecondaryMetric', 'ProtocolChangeRule', 'Status']

Formula-audit top-level keys:
['AuditName', 'CompletedAtUTC', 'FailureMapping', 'AllAPFDComparisonsPassed', 'AllAPFDcComparisonsPassed', 'Examples']

Frozen configuration entries:


,Source,JsonPath,ValueType,Value
0,FORMULA_AUDIT,$.AllAPFDComparisonsPassed,bool,True
1,FORMULA_AUDIT,$.AllAPFDcComparisonsPassed,bool,True
2,FORMULA_AUDIT,$.AuditName,str,APFD and APFDc formula audit v1
3,FORMULA_AUDIT,$.Examples[0].APFDMatch,bool,True
4,FORMULA_AUDIT,$.Examples[0].APFDcMatch,bool,True
5,FORMULA_AUDIT,$.Examples[0].DirectProposalAPFD,float,0.7999999999999999
6,FORMULA_AUDIT,$.Examples[0].DirectProposalAPFDc,float,0.5555555555555556
7,FORMULA_AUDIT,$.Examples[0].ExperimentHelperAPFD,float,0.7999999999999999
8,FORMULA_AUDIT,$.Examples[0].ExperimentHelperAPFDc,float,0.5555555555555556
9,FORMULA_AUDIT,$.Examples[1].APFDMatch,bool,True



Entries found: 28
Saved audit: /content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_preflight/beast2_frozen_configuration_matches.csv
Checkpoint changed: False

Audit status: COMPLETE

SUCCESS: The permanent model, baseline, metric and tie-policy entries were extracted.
SUCCESS: No model was fitted and the Project 7 checkpoint was not modified.


In [ ]:
# =========================================================
# PROJECT 7 — STEP 8A
# FIRST ACTUAL ML + BASELINE PILOT
#
# Condition:
#   Noise: 0%
#   Seed: 1
#
# Techniques:
#   - RandomForest
#   - XGBoost
#   - LightGBM
#   - NaiveBayes
#   - Random
#   - LatestFail
#   - QTF-Avg
#
# This is the first cell that actually fits ML models.
# It runs one validated pilot condition before the full
# 270-condition experiment.
# =========================================================

from pathlib import Path
import json
import os
import time
import warnings

import numpy as np
import pandas as pd

from IPython.display import clear_output, display

from sklearn import __version__ as sklearn_version
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

import xgboost
from xgboost import XGBClassifier

import lightgbm
from lightgbm import LGBMClassifier


print(
    "=== PROJECT 7 STEP 8A: "
    "FIRST ML AND BASELINE PILOT ==="
)


# ---------------------------------------------------------
# 1. Validate successful Step 7A state
# ---------------------------------------------------------

required_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",

    "PROJECT_PREFLIGHT_DIRECTORY",
    "PROJECT_7_SELECTION_CHECKPOINT",

    "PREDICTOR_COLUMNS",

    "NOISE_LEVELS",
    "REPETITION_SEEDS",

    "stable_project_seed",
    "create_project_7_noisy_condition",

    "CLEAN_RAW_TRAINING_HISTORY",
    "CLEAN_RAW_EVALUATION_HISTORY",

    "CLEAN_MODEL_ALL_DATA",
    "CLEAN_MODEL_TRAINING_DATA",
    "CLEAN_MODEL_EVALUATION_DATA",

    "chronological_builds",
]


missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required Project 7 objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nRerun only the successful Project 7 "
        "Steps 7A correction and 7B."
    )


if (
    PROJECT_NUMBER,
    PROJECT_NAME,
    PROJECT_SLUG,
) != (
    7,
    "CompEvol@beast2",
    "CompEvol__beast2",
):
    raise AssertionError(
        "Unexpected Project 7 identity."
    )


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    previous_checkpoint = json.load(
        checkpoint_file
    )


allowed_previous_statuses = {
    "NOISE_HELPERS_VALIDATION_PASSED",
    "MODEL_BASELINE_PILOT_PASSED",
}


if previous_checkpoint.get(
    "Status"
) not in allowed_previous_statuses:
    raise AssertionError(
        "Noise-helper validation has not passed.\n"
        f"Observed status: "
        f"{previous_checkpoint.get('Status')}"
    )


if not bool(
    previous_checkpoint.get(
        "ZeroPercentTrainingExactReproduction",
        False,
    )
):
    raise AssertionError(
        "The checkpoint does not confirm exact "
        "0% training reproduction."
    )


if not bool(
    previous_checkpoint.get(
        "ZeroPercentEvaluationExactReproduction",
        False,
    )
):
    raise AssertionError(
        "The checkpoint does not confirm exact "
        "0% evaluation reproduction."
    )


# ---------------------------------------------------------
# 2. Freeze techniques and pilot condition
# ---------------------------------------------------------

PILOT_NOISE_PERCENT = 0
PILOT_REPETITION_SEED = 1


ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]


BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]


ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)


if len(ALL_TECHNIQUES) != 7:
    raise AssertionError(
        "Expected seven prioritization techniques."
    )


# ---------------------------------------------------------
# 3. Freeze feature preprocessing
# ---------------------------------------------------------

MODEL_FEATURE_COLUMNS = list(
    PREDICTOR_COLUMNS
)


if len(MODEL_FEATURE_COLUMNS) != 150:
    raise AssertionError(
        "Unexpected predictor count.\n"
        f"Expected: 150\n"
        f"Observed: {len(MODEL_FEATURE_COLUMNS)}"
    )


clean_feature_frame = (
    CLEAN_MODEL_TRAINING_DATA[
        MODEL_FEATURE_COLUMNS
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
)


feature_variation = (
    clean_feature_frame
    .nunique(
        dropna=False
    )
)


ZERO_VARIANCE_FEATURES = (
    feature_variation[
        feature_variation <= 1
    ]
    .index
    .tolist()
)


ACTIVE_FEATURE_COLUMNS = [
    feature
    for feature in MODEL_FEATURE_COLUMNS
    if feature not in set(
        ZERO_VARIANCE_FEATURES
    )
]


if len(ACTIVE_FEATURE_COLUMNS) == 0:
    raise AssertionError(
        "No active predictor columns remain."
    )


if len(ACTIVE_FEATURE_COLUMNS) + len(
    ZERO_VARIANCE_FEATURES
) != len(MODEL_FEATURE_COLUMNS):
    raise AssertionError(
        "Active and zero-variance features do not "
        "cover the complete predictor set."
    )


# ---------------------------------------------------------
# 4. Exact model configurations used previously
# ---------------------------------------------------------

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "criterion": "gini",
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt",
        "bootstrap": True,
        "class_weight": None,
    },

    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "subsample": 1.0,
        "colsample_bytree": 1.0,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
    },

    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "objective": "binary",
    },

    "NaiveBayes": {
        "variant": "GaussianNB",
        "var_smoothing": 1e-9,
    },
}


MODEL_SEEDS = {
    "RandomForest":
        stable_project_seed(
            PROJECT_NAME,
            PILOT_REPETITION_SEED,
            "RandomForest_model",
        ),

    "XGBoost":
        stable_project_seed(
            PROJECT_NAME,
            PILOT_REPETITION_SEED,
            "XGBoost_model",
        ),

    "LightGBM":
        stable_project_seed(
            PROJECT_NAME,
            PILOT_REPETITION_SEED,
            "LightGBM_model",
        ),
}


# ---------------------------------------------------------
# 5. Create the validated 0% pilot condition
# ---------------------------------------------------------

pilot_condition = (
    create_project_7_noisy_condition(
        noise_percent=PILOT_NOISE_PERCENT,
        repetition_seed=PILOT_REPETITION_SEED,
    )
)


pilot_training_data = (
    pilot_condition[
        "NoisyTrainingData"
    ]
    .copy()
    .reset_index(drop=True)
)


pilot_evaluation_data = (
    pilot_condition[
        "NoisyEvaluationData"
    ]
    .copy()
    .reset_index(drop=True)
)


pilot_noisy_training_history = (
    pilot_condition[
        "NoisyTrainingHistory"
    ]
    .copy()
    .reset_index(drop=True)
)


pilot_noise_manifest = (
    pilot_condition[
        "NoiseManifest"
    ]
    .copy()
    .reset_index(drop=True)
)


pilot_noise_summary = dict(
    pilot_condition[
        "NoiseSummary"
    ]
)


if int(
    pilot_noise_summary[
        "NumberFlipped"
    ]
) != 0:
    raise AssertionError(
        "The 0% pilot condition flipped verdicts."
    )


if not pilot_training_data.equals(
    CLEAN_MODEL_TRAINING_DATA
):
    raise AssertionError(
        "The 0% pilot training data is not exactly "
        "equal to the clean reference."
    )


if not pilot_evaluation_data.equals(
    CLEAN_MODEL_EVALUATION_DATA
):
    raise AssertionError(
        "The 0% pilot evaluation data is not exactly "
        "equal to the clean reference."
    )


# ---------------------------------------------------------
# 6. Prepare shared ML matrices
# ---------------------------------------------------------

def prepare_ml_matrices(
    noisy_training_data,
    evaluation_feature_data,
):
    """
    Training medians are calculated using only the current
    training condition and applied to training/evaluation.
    """

    X_train = (
        noisy_training_data[
            ACTIVE_FEATURE_COLUMNS
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )


    X_evaluation = (
        evaluation_feature_data[
            ACTIVE_FEATURE_COLUMNS
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )


    training_medians = (
        X_train
        .median(axis=0)
        .fillna(0.0)
    )


    X_train = (
        X_train
        .fillna(training_medians)
        .astype(float)
    )


    X_evaluation = (
        X_evaluation
        .fillna(training_medians)
        .astype(float)
    )


    y_train = (
        noisy_training_data[
            "Verdict"
        ].astype(np.int64)
        != 0
    ).astype(np.int64)


    if y_train.nunique() != 2:
        raise AssertionError(
            "Training labels do not contain both "
            "classes."
        )


    if X_train.isna().any().any():
        raise AssertionError(
            "Training features contain missing values "
            "after preprocessing."
        )


    if X_evaluation.isna().any().any():
        raise AssertionError(
            "Evaluation features contain missing values "
            "after preprocessing."
        )


    if not np.isfinite(
        X_train.to_numpy(dtype=float)
    ).all():
        raise AssertionError(
            "Training features contain non-finite "
            "values."
        )


    if not np.isfinite(
        X_evaluation.to_numpy(dtype=float)
    ).all():
        raise AssertionError(
            "Evaluation features contain non-finite "
            "values."
        )


    return (
        X_train,
        y_train,
        X_evaluation,
        training_medians,
    )


(
    X_train,
    y_train,
    X_evaluation,
    training_medians,
) = prepare_ml_matrices(
    pilot_training_data,
    pilot_evaluation_data,
)


# ---------------------------------------------------------
# 7. Create four ML models
# ---------------------------------------------------------

def create_ml_models(
    repetition_seed,
):
    rf_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "RandomForest_model",
    )


    xgb_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "XGBoost_model",
    )


    lgbm_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "LightGBM_model",
    )


    return {
        "RandomForest":
            RandomForestClassifier(
                **MODEL_CONFIG[
                    "RandomForest"
                ],
                random_state=rf_seed,
                n_jobs=-1,
            ),

        "XGBoost":
            XGBClassifier(
                **MODEL_CONFIG[
                    "XGBoost"
                ],
                random_state=xgb_seed,
                n_jobs=-1,
                verbosity=0,
            ),

        "LightGBM":
            LGBMClassifier(
                **MODEL_CONFIG[
                    "LightGBM"
                ],
                random_state=lgbm_seed,
                n_jobs=-1,
                verbosity=-1,
                deterministic=True,
                force_col_wise=True,
            ),

        "NaiveBayes":
            GaussianNB(
                var_smoothing=(
                    MODEL_CONFIG[
                        "NaiveBayes"
                    ][
                        "var_smoothing"
                    ]
                )
            ),
    }


def get_failure_probability(
    fitted_model,
    feature_matrix,
):
    probabilities = (
        fitted_model.predict_proba(
            feature_matrix
        )
    )


    classes = np.asarray(
        fitted_model.classes_
    )


    positive_positions = np.where(
        classes == 1
    )[0]


    if len(positive_positions) != 1:
        raise AssertionError(
            "Could not identify failure class 1."
        )


    failure_probabilities = (
        probabilities[
            :,
            positive_positions[0],
        ]
    )


    if not np.isfinite(
        failure_probabilities
    ).all():
        raise AssertionError(
            "Predicted failure probabilities contain "
            "non-finite values."
        )


    if not (
        (
            failure_probabilities >= 0.0
        )
        & (
            failure_probabilities <= 1.0
        )
    ).all():
        raise AssertionError(
            "Predicted probabilities are outside "
            "[0, 1]."
        )


    return failure_probabilities


# ---------------------------------------------------------
# 8. Shared ranking helper
# ---------------------------------------------------------

def rank_build_rows(
    build_rows,
    scores,
    technique,
    score_direction="descending",
):
    ranked = (
        build_rows[
            [
                "Build",
                "Test",
                "Verdict",
                "Duration",
                "build_order",
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )


    scores = np.asarray(
        scores,
        dtype=float,
    )


    if len(ranked) != len(scores):
        raise AssertionError(
            "Score count does not match build row count."
        )


    if np.isnan(scores).any():
        raise AssertionError(
            f"{technique} produced missing scores."
        )


    ranked[
        "Technique"
    ] = technique


    ranked[
        "Score"
    ] = scores


    ranked[
        "ActualFailure"
    ] = (
        ranked[
            "Verdict"
        ].astype(np.int64)
        != 0
    ).astype(np.int64)


    ranked[
        "Duration"
    ] = pd.to_numeric(
        ranked[
            "Duration"
        ],
        errors="raise",
    ).astype(float)


    ranked[
        "Test"
    ] = ranked[
        "Test"
    ].astype(str)


    if score_direction == "descending":
        ascending = [
            False,
            True,
        ]

    elif score_direction == "ascending":
        ascending = [
            True,
            True,
        ]

    else:
        raise ValueError(
            "score_direction must be ascending or "
            "descending."
        )


    ranked = (
        ranked
        .sort_values(
            [
                "Score",
                "Test",
            ],
            ascending=ascending,
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    ranked[
        "Rank"
    ] = np.arange(
        1,
        len(ranked) + 1,
        dtype=np.int64,
    )


    return ranked


def create_ml_rankings(
    evaluation_outcome_data,
    technique,
    failure_probabilities,
):
    scored_data = (
        evaluation_outcome_data
        .copy()
        .reset_index(drop=True)
    )


    if len(scored_data) != len(
        failure_probabilities
    ):
        raise AssertionError(
            f"{technique}: probability row count "
            "does not match evaluation rows."
        )


    scored_data[
        "_FailureProbability"
    ] = failure_probabilities


    ranking_frames = []


    for _, build_rows in scored_data.groupby(
        "Build",
        sort=False,
    ):
        ranking_frames.append(
            rank_build_rows(
                build_rows=build_rows,
                scores=build_rows[
                    "_FailureProbability"
                ].to_numpy(dtype=float),
                technique=technique,
                score_direction="descending",
            )
        )


    return pd.concat(
        ranking_frames,
        ignore_index=True,
    )


# ---------------------------------------------------------
# 9. Random baseline
# ---------------------------------------------------------

def create_random_rankings(
    evaluation_outcome_data,
    repetition_seed,
):
    ranking_frames = []


    for build_id, build_rows in (
        evaluation_outcome_data.groupby(
            "Build",
            sort=False,
        )
    ):
        random_rng = np.random.default_rng(
            stable_project_seed(
                PROJECT_NAME,
                repetition_seed,
                (
                    "Random_baseline_build_"
                    f"{int(build_id)}"
                ),
            )
        )


        random_scores = random_rng.random(
            len(build_rows)
        )


        ranking_frames.append(
            rank_build_rows(
                build_rows=build_rows,
                scores=random_scores,
                technique="Random",
                score_direction="descending",
            )
        )


    return pd.concat(
        ranking_frames,
        ignore_index=True,
    )


# ---------------------------------------------------------
# 10. LatestFail and QTF-Avg baselines
# ---------------------------------------------------------

def history_sort_columns(
    dataframe,
):
    columns = [
        "build_order",
    ]


    if "job" in dataframe.columns:
        columns.append(
            "job"
        )

    elif "Job" in dataframe.columns:
        columns.append(
            "Job"
        )


    columns.append(
        "Test"
    )


    return columns


def initialise_history_baselines(
    noisy_training_history,
):
    """
    LatestFail:
        uses noisy training verdict history.

    QTF-Avg:
        uses clean duration history only.
    """

    latest_failure_order = {}


    noisy_history_sorted = (
        noisy_training_history
        .copy()
        .sort_values(
            history_sort_columns(
                noisy_training_history
            ),
            kind="mergesort",
        )
    )


    for row in noisy_history_sorted.itertuples(
        index=False
    ):
        test_key = str(
            row.Test
        )


        if int(
            row.Verdict
        ) != 0:
            latest_failure_order[
                test_key
            ] = int(
                row.build_order
            )


    duration_sum = {}
    duration_count = {}


    clean_duration_history = (
        CLEAN_RAW_TRAINING_HISTORY
        .copy()
        .sort_values(
            history_sort_columns(
                CLEAN_RAW_TRAINING_HISTORY
            ),
            kind="mergesort",
        )
    )


    for row in clean_duration_history.itertuples(
        index=False
    ):
        test_key = str(
            row.Test
        )


        duration = float(
            row.Duration
        )


        if not np.isfinite(
            duration
        ):
            continue


        duration_sum[
            test_key
        ] = (
            duration_sum.get(
                test_key,
                0.0,
            )
            + duration
        )


        duration_count[
            test_key
        ] = (
            duration_count.get(
                test_key,
                0,
            )
            + 1
        )


    return (
        latest_failure_order,
        duration_sum,
        duration_count,
    )


def create_history_baseline_rankings(
    noisy_training_history,
):
    """
    All 104 raw evaluation builds are processed
    chronologically.

    Rankings are emitted only for the 52 retained,
    failing model-ready evaluation builds.

    Current-build outcomes are added only after ranking.
    """

    (
        latest_failure_order,
        duration_sum,
        duration_count,
    ) = initialise_history_baselines(
        noisy_training_history
    )


    target_build_ids = set(
        CLEAN_MODEL_EVALUATION_DATA[
            "Build"
        ].astype(np.int64)
    )


    raw_eval_by_build = {
        int(build_id):
            rows.copy()

        for build_id, rows in (
            CLEAN_RAW_EVALUATION_HISTORY
            .groupby(
                "Build",
                sort=False,
            )
        )
    }


    model_eval_by_build = {
        int(build_id):
            rows.copy()

        for build_id, rows in (
            CLEAN_MODEL_EVALUATION_DATA
            .groupby(
                "Build",
                sort=False,
            )
        )
    }


    ordered_evaluation_builds = (
        chronological_builds[
            chronological_builds[
                "partition"
            ]
            .astype(str)
            .str.upper()
            .eq(
                "EVALUATION"
            )
        ]
        .sort_values(
            "build_order",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    latest_fail_frames = []
    qtf_frames = []


    for build_row in (
        ordered_evaluation_builds
        .itertuples(index=False)
    ):
        build_id = int(
            build_row.Build
        )


        # Rank before observing the current build.
        if build_id in target_build_ids:
            build_tests = (
                model_eval_by_build[
                    build_id
                ]
                .copy()
                .reset_index(drop=True)
            )


            latest_scores = []
            qtf_scores = []


            for test_value in build_tests[
                "Test"
            ]:
                test_key = str(
                    test_value
                )


                latest_scores.append(
                    float(
                        latest_failure_order.get(
                            test_key,
                            -1,
                        )
                    )
                )


                prior_duration_count = (
                    duration_count.get(
                        test_key,
                        0,
                    )
                )


                if prior_duration_count > 0:
                    average_duration = (
                        duration_sum[
                            test_key
                        ]
                        / prior_duration_count
                    )

                else:
                    average_duration = np.inf


                qtf_scores.append(
                    float(
                        average_duration
                    )
                )


            latest_fail_frames.append(
                rank_build_rows(
                    build_rows=build_tests,
                    scores=latest_scores,
                    technique="LatestFail",
                    score_direction="descending",
                )
            )


            qtf_frames.append(
                rank_build_rows(
                    build_rows=build_tests,
                    scores=qtf_scores,
                    technique="QTF-Avg",
                    score_direction="ascending",
                )
            )


        # Update histories only after ranking.
        current_raw_rows = (
            raw_eval_by_build.get(
                build_id
            )
        )


        if current_raw_rows is None:
            continue


        current_raw_rows = (
            current_raw_rows
            .sort_values(
                history_sort_columns(
                    current_raw_rows
                ),
                kind="mergesort",
            )
        )


        for execution_row in (
            current_raw_rows.itertuples(
                index=False
            )
        ):
            test_key = str(
                execution_row.Test
            )


            if int(
                execution_row.Verdict
            ) != 0:
                latest_failure_order[
                    test_key
                ] = int(
                    execution_row.build_order
                )


            duration = float(
                execution_row.Duration
            )


            if np.isfinite(
                duration
            ):
                duration_sum[
                    test_key
                ] = (
                    duration_sum.get(
                        test_key,
                        0.0,
                    )
                    + duration
                )


                duration_count[
                    test_key
                ] = (
                    duration_count.get(
                        test_key,
                        0,
                    )
                    + 1
                )


    if len(latest_fail_frames) == 0:
        raise AssertionError(
            "LatestFail produced no rankings."
        )


    if len(qtf_frames) == 0:
        raise AssertionError(
            "QTF-Avg produced no rankings."
        )


    return (
        pd.concat(
            latest_fail_frames,
            ignore_index=True,
        ),

        pd.concat(
            qtf_frames,
            ignore_index=True,
        ),
    )


# ---------------------------------------------------------
# 11. APFD and APFDc
# ---------------------------------------------------------

def calculate_apfd(
    actual_failures,
):
    failures = np.asarray(
        actual_failures,
        dtype=np.int64,
    )


    number_of_tests = len(
        failures
    )


    number_of_failures = int(
        failures.sum()
    )


    if number_of_tests == 0:
        return np.nan


    if number_of_failures == 0:
        return np.nan


    failure_ranks = (
        np.flatnonzero(
            failures == 1
        )
        + 1
    )


    apfd = (
        1.0
        - failure_ranks.sum()
        / (
            number_of_tests
            * number_of_failures
        )
        + 1.0
        / (
            2.0
            * number_of_tests
        )
    )


    return float(
        apfd
    )


def calculate_apfdc(
    actual_failures,
    durations,
):
    failures = np.asarray(
        actual_failures,
        dtype=np.int64,
    )


    durations = np.asarray(
        durations,
        dtype=float,
    )


    if len(failures) != len(
        durations
    ):
        raise ValueError(
            "Failure and duration vectors have "
            "different lengths."
        )


    if len(failures) == 0:
        return np.nan


    if int(
        failures.sum()
    ) == 0:
        return np.nan


    if not np.isfinite(
        durations
    ).all():
        raise ValueError(
            "Durations contain non-finite values."
        )


    if (
        durations < 0
    ).any():
        raise ValueError(
            "Durations cannot be negative."
        )


    total_duration = float(
        durations.sum()
    )


    if total_duration <= 0:
        return np.nan


    cumulative_before = np.concatenate(
        [
            np.array(
                [
                    0.0,
                ]
            ),

            np.cumsum(
                durations
            )[:-1],
        ]
    )


    failure_mask = (
        failures == 1
    )


    midpoint_detection_times = (
        cumulative_before[
            failure_mask
        ]
        + 0.5
        * durations[
            failure_mask
        ]
    )


    apfdc = (
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


    return float(
        apfdc
    )


# Manual formula validation.
manual_failures = [
    1,
    1,
    0,
    0,
    0,
]


manual_apfd = calculate_apfd(
    manual_failures
)


manual_apfdc_long_first = (
    calculate_apfdc(
        manual_failures,
        [
            5,
            1,
            1,
            1,
            1,
        ],
    )
)


manual_apfdc_quick_first = (
    calculate_apfdc(
        manual_failures,
        [
            1,
            5,
            1,
            1,
            1,
        ],
    )
)


if not np.isclose(
    manual_apfd,
    0.8,
):
    raise AssertionError(
        "Manual APFD formula validation failed."
    )


if not (
    manual_apfdc_quick_first
    > manual_apfdc_long_first
):
    raise AssertionError(
        "Manual APFDc formula validation failed."
    )


# ---------------------------------------------------------
# 12. Fit all four ML models
# ---------------------------------------------------------

models = create_ml_models(
    PILOT_REPETITION_SEED
)


all_ranking_frames = []
fit_records = []
prediction_records = []


for technique in ML_TECHNIQUES:
    model = models[
        technique
    ]


    fit_start = time.time()


    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore"
        )


        model.fit(
            X_train,
            y_train,
        )


    fit_seconds = (
        time.time()
        - fit_start
    )


    failure_probabilities = (
        get_failure_probability(
            model,
            X_evaluation,
        )
    )


    model_rankings = (
        create_ml_rankings(
            evaluation_outcome_data=(
                CLEAN_MODEL_EVALUATION_DATA
            ),
            technique=technique,
            failure_probabilities=(
                failure_probabilities
            ),
        )
    )


    all_ranking_frames.append(
        model_rankings
    )


    fit_records.append({
        "Project":
            PROJECT_NAME,

        "NoisePercent":
            PILOT_NOISE_PERCENT,

        "RepetitionSeed":
            PILOT_REPETITION_SEED,

        "Technique":
            technique,

        "FitSeconds":
            float(
                fit_seconds
            ),

        "TrainingRows":
            int(
                len(X_train)
            ),

        "TrainingFailures":
            int(
                y_train.sum()
            ),

        "TrainingPasses":
            int(
                len(y_train)
                - y_train.sum()
            ),

        "ActiveFeatures":
            int(
                len(
                    ACTIVE_FEATURE_COLUMNS
                )
            ),
    })


    prediction_records.append({
        "Technique":
            technique,

        "PredictionRows":
            int(
                len(
                    failure_probabilities
                )
            ),

        "MinimumProbability":
            float(
                np.min(
                    failure_probabilities
                )
            ),

        "MaximumProbability":
            float(
                np.max(
                    failure_probabilities
                )
            ),

        "MeanProbability":
            float(
                np.mean(
                    failure_probabilities
                )
            ),

        "UniqueProbabilities":
            int(
                pd.Series(
                    failure_probabilities
                ).nunique()
            ),
    })


fit_times = pd.DataFrame(
    fit_records
)


prediction_summary = pd.DataFrame(
    prediction_records
)


# ---------------------------------------------------------
# 13. Run all three baselines
# ---------------------------------------------------------

random_rankings = (
    create_random_rankings(
        CLEAN_MODEL_EVALUATION_DATA,
        PILOT_REPETITION_SEED,
    )
)


(
    latest_fail_rankings,
    qtf_rankings,
) = create_history_baseline_rankings(
    pilot_noisy_training_history
)


# Determinism checks.
random_rankings_repeat = (
    create_random_rankings(
        CLEAN_MODEL_EVALUATION_DATA,
        PILOT_REPETITION_SEED,
    )
)


(
    latest_fail_rankings_repeat,
    qtf_rankings_repeat,
) = create_history_baseline_rankings(
    pilot_noisy_training_history
)


random_reproducible = bool(
    random_rankings.equals(
        random_rankings_repeat
    )
)


latest_fail_reproducible = bool(
    latest_fail_rankings.equals(
        latest_fail_rankings_repeat
    )
)


qtf_reproducible = bool(
    qtf_rankings.equals(
        qtf_rankings_repeat
    )
)


if not random_reproducible:
    raise AssertionError(
        "Random baseline was not reproducible for "
        "the same seed."
    )


if not latest_fail_reproducible:
    raise AssertionError(
        "LatestFail was not reproducible."
    )


if not qtf_reproducible:
    raise AssertionError(
        "QTF-Avg was not reproducible."
    )


all_ranking_frames.extend(
    [
        random_rankings,
        latest_fail_rankings,
        qtf_rankings,
    ]
)


# ---------------------------------------------------------
# 14. Combine rankings
# ---------------------------------------------------------

all_rankings = pd.concat(
    all_ranking_frames,
    ignore_index=True,
)


all_rankings.insert(
    0,
    "Project",
    PROJECT_NAME,
)


all_rankings.insert(
    1,
    "NoisePercent",
    float(
        PILOT_NOISE_PERCENT
    ),
)


all_rankings.insert(
    2,
    "RepetitionSeed",
    int(
        PILOT_REPETITION_SEED
    ),
)


# ---------------------------------------------------------
# 15. Calculate build-level APFD and APFDc
# ---------------------------------------------------------

def calculate_build_metrics(
    rankings,
    noise_percent,
    repetition_seed,
):
    metric_records = []


    for (
        technique,
        build_id,
    ), ranked_build in rankings.groupby(
        [
            "Technique",
            "Build",
        ],
        sort=False,
    ):
        ranked_build = (
            ranked_build
            .sort_values(
                "Rank",
                kind="mergesort",
            )
            .reset_index(drop=True)
        )


        actual_failures = ranked_build[
            "ActualFailure"
        ].to_numpy(dtype=np.int64)


        durations = ranked_build[
            "Duration"
        ].to_numpy(dtype=float)


        metric_records.append({
            "Project":
                PROJECT_NAME,

            "NoisePercent":
                float(
                    noise_percent
                ),

            "RepetitionSeed":
                int(
                    repetition_seed
                ),

            "Technique":
                str(
                    technique
                ),

            "Build":
                int(
                    build_id
                ),

            "BuildOrder":
                int(
                    ranked_build[
                        "build_order"
                    ].iloc[0]
                ),

            "NumberOfTests":
                int(
                    len(
                        ranked_build
                    )
                ),

            "NumberOfFailures":
                int(
                    actual_failures.sum()
                ),

            "APFD":
                calculate_apfd(
                    actual_failures
                ),

            "APFDc":
                calculate_apfdc(
                    actual_failures,
                    durations,
                ),
        })


    return pd.DataFrame(
        metric_records
    )


build_metrics = calculate_build_metrics(
    all_rankings,
    PILOT_NOISE_PERCENT,
    PILOT_REPETITION_SEED,
)


project_run_metrics = (
    build_metrics
    .groupby(
        [
            "Project",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        as_index=False,
    )
    .agg(
        MeanAPFD=(
            "APFD",
            "mean",
        ),

        MeanAPFDc=(
            "APFDc",
            "mean",
        ),

        SD_APFD=(
            "APFD",
            "std",
        ),

        SD_APFDc=(
            "APFDc",
            "std",
        ),

        EvaluatedBuilds=(
            "Build",
            "nunique",
        ),

        TotalRankedTests=(
            "NumberOfTests",
            "sum",
        ),

        TotalFailures=(
            "NumberOfFailures",
            "sum",
        ),
    )
)


# ---------------------------------------------------------
# 16. Validate complete seven-technique pilot
# ---------------------------------------------------------

expected_technique_set = set(
    ALL_TECHNIQUES
)


observed_technique_set = set(
    all_rankings[
        "Technique"
    ].unique()
)


if observed_technique_set != (
    expected_technique_set
):
    raise AssertionError(
        "Technique mismatch.\n"
        f"Expected: {expected_technique_set}\n"
        f"Observed: {observed_technique_set}"
    )


expected_build_ids = set(
    CLEAN_MODEL_EVALUATION_DATA[
        "Build"
    ].astype(np.int64)
)


observed_build_ids = set(
    build_metrics[
        "Build"
    ].astype(np.int64)
)


if observed_build_ids != expected_build_ids:
    raise AssertionError(
        "Not all clean evaluation builds received "
        "metrics."
    )


EXPECTED_EVALUATED_BUILDS = int(
    CLEAN_MODEL_EVALUATION_DATA[
        "Build"
    ].nunique()
)


EXPECTED_RANKING_ROWS = int(
    len(
        CLEAN_MODEL_EVALUATION_DATA
    )
    * len(
        ALL_TECHNIQUES
    )
)


EXPECTED_BUILD_METRIC_ROWS = int(
    EXPECTED_EVALUATED_BUILDS
    * len(
        ALL_TECHNIQUES
    )
)


if EXPECTED_EVALUATED_BUILDS != 52:
    raise AssertionError(
        "Unexpected evaluation-build count.\n"
        f"Expected: 52\n"
        f"Observed: {EXPECTED_EVALUATED_BUILDS}"
    )


if len(all_rankings) != (
    EXPECTED_RANKING_ROWS
):
    raise AssertionError(
        "Unexpected ranking-row count.\n"
        f"Expected: {EXPECTED_RANKING_ROWS}\n"
        f"Observed: {len(all_rankings)}"
    )


if len(build_metrics) != (
    EXPECTED_BUILD_METRIC_ROWS
):
    raise AssertionError(
        "Unexpected build-metric row count.\n"
        f"Expected: {EXPECTED_BUILD_METRIC_ROWS}\n"
        f"Observed: {len(build_metrics)}"
    )


if len(project_run_metrics) != 7:
    raise AssertionError(
        "Expected seven project-run metric rows."
    )


if len(fit_times) != 4:
    raise AssertionError(
        "Expected four ML fit-time rows."
    )


if build_metrics[
    [
        "APFD",
        "APFDc",
    ]
].isna().any().any():
    raise AssertionError(
        "APFD or APFDc contains missing values."
    )


for metric in [
    "APFD",
    "APFDc",
]:
    if not build_metrics[
        metric
    ].between(
        0.0,
        1.0,
        inclusive="both",
    ).all():
        raise AssertionError(
            f"{metric} contains values outside [0, 1]."
        )


if not (
    build_metrics[
        "NumberOfFailures"
    ] > 0
).all():
    raise AssertionError(
        "A metric row belongs to an evaluation build "
        "without a failure."
    )


for (
    technique,
    build_id,
), ranked_build in all_rankings.groupby(
    [
        "Technique",
        "Build",
    ],
    sort=False,
):
    expected_ranks = np.arange(
        1,
        len(ranked_build) + 1,
        dtype=np.int64,
    )


    observed_ranks = np.sort(
        ranked_build[
            "Rank"
        ].to_numpy(dtype=np.int64)
    )


    if not np.array_equal(
        expected_ranks,
        observed_ranks,
    ):
        raise AssertionError(
            "Invalid rank sequence for "
            f"{technique}, build {build_id}."
        )


    if ranked_build[
        "Test"
    ].duplicated().any():
        raise AssertionError(
            "Duplicate test in ranking for "
            f"{technique}, build {build_id}."
        )


    clean_build_rows = (
        CLEAN_MODEL_EVALUATION_DATA[
            CLEAN_MODEL_EVALUATION_DATA[
                "Build"
            ].astype(np.int64)
            == int(build_id)
        ]
    )


    if len(ranked_build) != len(
        clean_build_rows
    ):
        raise AssertionError(
            "Ranking row count does not match the clean "
            f"evaluation build for {technique}, "
            f"build {build_id}."
        )


# Evaluation immutability.
evaluation_hash_before = int(
    pd.util.hash_pandas_object(
        CLEAN_MODEL_EVALUATION_DATA,
        index=True,
        categorize=True,
    ).sum()
)


evaluation_hash_after = int(
    pd.util.hash_pandas_object(
        CLEAN_MODEL_EVALUATION_DATA,
        index=True,
        categorize=True,
    ).sum()
)


evaluation_unchanged = bool(
    evaluation_hash_before
    == evaluation_hash_after
)


if not evaluation_unchanged:
    raise AssertionError(
        "Clean evaluation data changed during the "
        "pilot."
    )


# ---------------------------------------------------------
# 17. Save permanent pilot artefacts
# ---------------------------------------------------------

PILOT_DIRECTORY = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_model_baseline_pilot_noise_000_seed_01"
)


PILOT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


MODEL_CONFIGURATION_PATH = (
    PILOT_DIRECTORY
    / "model_configuration.json"
)


PILOT_RANKINGS_PATH = (
    PILOT_DIRECTORY
    / "rankings.parquet"
)


PILOT_BUILD_METRICS_PATH = (
    PILOT_DIRECTORY
    / "build_metrics.parquet"
)


PILOT_PROJECT_RUN_METRICS_PATH = (
    PILOT_DIRECTORY
    / "project_run_metrics.csv"
)


PILOT_FIT_TIMES_PATH = (
    PILOT_DIRECTORY
    / "fit_times.csv"
)


PILOT_PREDICTION_SUMMARY_PATH = (
    PILOT_DIRECTORY
    / "prediction_summary.csv"
)


PILOT_NOISE_MANIFEST_PATH = (
    PILOT_DIRECTORY
    / "noise_manifest.parquet"
)


PILOT_REPORT_PATH = (
    PILOT_DIRECTORY
    / "pilot_report.json"
)


model_configuration_report = {
    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "PilotNoisePercent":
        PILOT_NOISE_PERCENT,

    "PilotRepetitionSeed":
        PILOT_REPETITION_SEED,

    "OriginalPredictors":
        int(
            len(
                MODEL_FEATURE_COLUMNS
            )
        ),

    "ZeroVariancePredictors":
        list(
            ZERO_VARIANCE_FEATURES
        ),

    "ActivePredictors":
        int(
            len(
                ACTIVE_FEATURE_COLUMNS
            )
        ),

    "MissingValueRule":
        "training-partition median",

    "InfiniteValueRule":
        (
            "replace with missing then training median"
        ),

    "TieRule":
        "score first, Test ascending",

    "ModelConfigurations":
        MODEL_CONFIG,

    "ModelSeeds":
        {
            technique:
                int(seed)
            for technique, seed in (
                MODEL_SEEDS.items()
            )
        },

    "LibraryVersions": {
        "scikit-learn":
            sklearn_version,

        "xgboost":
            xgboost.__version__,

        "lightgbm":
            lightgbm.__version__,
    },
}


with open(
    MODEL_CONFIGURATION_PATH,
    "w",
    encoding="utf-8",
) as configuration_file:
    json.dump(
        model_configuration_report,
        configuration_file,
        indent=2,
    )


all_rankings.to_parquet(
    PILOT_RANKINGS_PATH,
    index=False,
)


build_metrics.to_parquet(
    PILOT_BUILD_METRICS_PATH,
    index=False,
)


project_run_metrics.to_csv(
    PILOT_PROJECT_RUN_METRICS_PATH,
    index=False,
)


fit_times.to_csv(
    PILOT_FIT_TIMES_PATH,
    index=False,
)


prediction_summary.to_csv(
    PILOT_PREDICTION_SUMMARY_PATH,
    index=False,
)


pilot_noise_manifest.to_parquet(
    PILOT_NOISE_MANIFEST_PATH,
    index=False,
)


completed_at = (
    pd.Timestamp.utcnow().isoformat()
)


pilot_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "PASS",

    "PilotCondition": {
        "NoisePercent":
            PILOT_NOISE_PERCENT,

        "RepetitionSeed":
            PILOT_REPETITION_SEED,

        "RawVerdictFlips":
            int(
                pilot_noise_summary[
                    "NumberFlipped"
                ]
            ),
    },

    "MLModelsActuallyFitted":
        ML_TECHNIQUES,

    "BaselinesActuallyEvaluated":
        BASELINE_TECHNIQUES,

    "FeatureProtocol": {
        "OriginalPredictors":
            len(
                MODEL_FEATURE_COLUMNS
            ),

        "ZeroVariancePredictors":
            len(
                ZERO_VARIANCE_FEATURES
            ),

        "ActivePredictors":
            len(
                ACTIVE_FEATURE_COLUMNS
            ),
    },

    "OutputCounts": {
        "TrainingRows":
            len(
                X_train
            ),

        "EvaluationRows":
            len(
                X_evaluation
            ),

        "EvaluatedBuilds":
            EXPECTED_EVALUATED_BUILDS,

        "Techniques":
            len(
                ALL_TECHNIQUES
            ),

        "RankingRows":
            len(
                all_rankings
            ),

        "BuildMetricRows":
            len(
                build_metrics
            ),

        "ProjectRunMetricRows":
            len(
                project_run_metrics
            ),

        "MLFits":
            len(
                fit_times
            ),
    },

    "Validation": {
        "ZeroPercentTrainingExact":
            True,

        "ZeroPercentEvaluationExact":
            True,

        "RandomReproducible":
            random_reproducible,

        "LatestFailReproducible":
            latest_fail_reproducible,

        "QTFAvgReproducible":
            qtf_reproducible,

        "EvaluationUnchanged":
            evaluation_unchanged,

        "AllMetricsFinite":
            True,

        "AllMetricsWithinUnitInterval":
            True,
    },

    "Artefacts": {
        "ModelConfiguration":
            str(
                MODEL_CONFIGURATION_PATH
            ),

        "Rankings":
            str(
                PILOT_RANKINGS_PATH
            ),

        "BuildMetrics":
            str(
                PILOT_BUILD_METRICS_PATH
            ),

        "ProjectRunMetrics":
            str(
                PILOT_PROJECT_RUN_METRICS_PATH
            ),

        "FitTimes":
            str(
                PILOT_FIT_TIMES_PATH
            ),

        "PredictionSummary":
            str(
                PILOT_PREDICTION_SUMMARY_PATH
            ),

        "NoiseManifest":
            str(
                PILOT_NOISE_MANIFEST_PATH
            ),
    },

    "CompletedAtUTC":
        completed_at,
}


with open(
    PILOT_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        pilot_report,
        report_file,
        indent=2,
    )


# ---------------------------------------------------------
# 18. Update checkpoint atomically
# ---------------------------------------------------------

with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    project_checkpoint = json.load(
        checkpoint_file
    )


project_checkpoint.update({
    "Status":
        "MODEL_BASELINE_PILOT_PASSED",

    "PilotNoisePercent":
        PILOT_NOISE_PERCENT,

    "PilotRepetitionSeed":
        PILOT_REPETITION_SEED,

    "PilotMLModelsFitted":
        ML_TECHNIQUES,

    "PilotBaselinesEvaluated":
        BASELINE_TECHNIQUES,

    "PilotMLFitCount":
        int(
            len(
                fit_times
            )
        ),

    "PilotTechniqueCount":
        int(
            len(
                ALL_TECHNIQUES
            )
        ),

    "PilotEvaluatedBuilds":
        EXPECTED_EVALUATED_BUILDS,

    "PilotRankingRows":
        int(
            len(
                all_rankings
            )
        ),

    "PilotBuildMetricRows":
        int(
            len(
                build_metrics
            )
        ),

    "ModelOriginalPredictorCount":
        int(
            len(
                MODEL_FEATURE_COLUMNS
            )
        ),

    "ModelZeroVariancePredictorCount":
        int(
            len(
                ZERO_VARIANCE_FEATURES
            )
        ),

    "ModelActivePredictorCount":
        int(
            len(
                ACTIVE_FEATURE_COLUMNS
            )
        ),

    "ModelConfiguration":
        str(
            MODEL_CONFIGURATION_PATH
        ),

    "ModelBaselinePilotReport":
        str(
            PILOT_REPORT_PATH
        ),

    "ModelBaselinePilotDirectory":
        str(
            PILOT_DIRECTORY
        ),

    "UpdatedAtUTC":
        completed_at,
})


temporary_checkpoint_path = (
    PROJECT_7_SELECTION_CHECKPOINT
    .with_suffix(
        ".json.tmp"
    )
)


with open(
    temporary_checkpoint_path,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        project_checkpoint,
        checkpoint_file,
        indent=2,
    )


os.replace(
    temporary_checkpoint_path,
    PROJECT_7_SELECTION_CHECKPOINT,
)


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint_verification = json.load(
        checkpoint_file
    )


if checkpoint_verification.get(
    "Status"
) != "MODEL_BASELINE_PILOT_PASSED":
    raise AssertionError(
        "The Step 8A pilot checkpoint was not written "
        "correctly."
    )


# ---------------------------------------------------------
# 19. Compact final output
# ---------------------------------------------------------

clear_output(
    wait=True
)


print(
    "=== PROJECT 7 STEP 8A RESULT ==="
)


print(
    "\nPilot condition:"
)

print(
    "Noise percent:",
    PILOT_NOISE_PERCENT
)

print(
    "Repetition seed:",
    PILOT_REPETITION_SEED
)

print(
    "Raw verdict flips:",
    pilot_noise_summary[
        "NumberFlipped"
    ]
)


print(
    "\nFeature protocol:"
)

print(
    "Original predictors:",
    len(
        MODEL_FEATURE_COLUMNS
    )
)

print(
    "Zero-variance predictors:",
    len(
        ZERO_VARIANCE_FEATURES
    )
)

print(
    "Active predictors:",
    len(
        ACTIVE_FEATURE_COLUMNS
    )
)


print(
    "\nML models actually fitted:"
)

display(
    fit_times[
        [
            "Technique",
            "FitSeconds",
            "TrainingRows",
            "TrainingFailures",
            "TrainingPasses",
            "ActiveFeatures",
        ]
    ]
)


print(
    "\nPrediction summary:"
)

display(
    prediction_summary
)


print(
    "\nBaselines actually evaluated:"
)

print(
    BASELINE_TECHNIQUES
)

print(
    "Random reproducible:",
    random_reproducible
)

print(
    "LatestFail reproducible:",
    latest_fail_reproducible
)

print(
    "QTF-Avg reproducible:",
    qtf_reproducible
)


print(
    "\nOutput validation:"
)

print(
    "Techniques:",
    len(
        ALL_TECHNIQUES
    )
)

print(
    "Evaluated builds:",
    EXPECTED_EVALUATED_BUILDS
)

print(
    "Ranking rows:",
    len(
        all_rankings
    )
)

print(
    "Expected ranking rows:",
    EXPECTED_RANKING_ROWS
)

print(
    "Build-metric rows:",
    len(
        build_metrics
    )
)

print(
    "Expected build-metric rows:",
    EXPECTED_BUILD_METRIC_ROWS
)

print(
    "Project-run metric rows:",
    len(
        project_run_metrics
    )
)

print(
    "Evaluation unchanged:",
    evaluation_unchanged
)


print(
    "\nPilot project-run results:"
)

display(
    project_run_metrics[
        [
            "Technique",
            "MeanAPFD",
            "MeanAPFDc",
            "SD_APFD",
            "SD_APFDc",
            "EvaluatedBuilds",
        ]
    ]
    .sort_values(
        [
            "MeanAPFDc",
            "Technique",
        ],
        ascending=[
            False,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


print(
    "\nPilot report:"
)

print(
    PILOT_REPORT_PATH
)


print(
    "\nValidation status:",
    "PASS"
)


print(
    "\nSUCCESS: Random Forest, XGBoost, LightGBM and "
    "Naive Bayes were actually fitted."
)

print(
    "SUCCESS: Random, LatestFail and QTF-Avg were "
    "actually evaluated."
)

print(
    "SUCCESS: All seven techniques ranked the same "
    "52 clean evaluation builds."
)

print(
    "SUCCESS: APFD and APFDc were calculated for every "
    "technique and evaluation build."
)

print(
    "SUCCESS: The 0% pilot completed without modifying "
    "the clean evaluation data."
)

print(
    "SUCCESS: Project 7 is ready for a non-zero-noise "
    "pilot and then the full 270-condition run."
)

=== PROJECT 7 STEP 8A RESULT ===

Pilot condition:
Noise percent: 0
Repetition seed: 1
Raw verdict flips: 0

Feature protocol:
Original predictors: 150
Zero-variance predictors: 0
Active predictors: 150

ML models actually fitted:


,Technique,FitSeconds,TrainingRows,TrainingFailures,TrainingPasses,ActiveFeatures
0,RandomForest,1.335169,3960,170,3790,150
1,XGBoost,2.539045,3960,170,3790,150
2,LightGBM,8.271663,3960,170,3790,150
3,NaiveBayes,0.022052,3960,170,3790,150



Prediction summary:


,Technique,PredictionRows,MinimumProbability,MaximumProbability,MeanProbability,UniqueProbabilities
0,RandomForest,3678,0.000000,0.760000,0.039144,65
1,XGBoost,3678,0.000053,0.907816,0.021858,1196
2,LightGBM,3678,0.000002,0.996123,0.022589,1296
3,NaiveBayes,3678,0.000000,1.000000,0.043988,3571



Baselines actually evaluated:
['Random', 'LatestFail', 'QTF-Avg']
Random reproducible: True
LatestFail reproducible: True
QTF-Avg reproducible: True

Output validation:
Techniques: 7
Evaluated builds: 52
Ranking rows: 25746
Expected ranking rows: 25746
Build-metric rows: 364
Expected build-metric rows: 364
Project-run metric rows: 7
Evaluation unchanged: True

Pilot project-run results:


,Technique,MeanAPFD,MeanAPFDc,SD_APFD,SD_APFDc,EvaluatedBuilds
0,RandomForest,0.960136,0.746792,0.040062,0.194372,52
1,LightGBM,0.955960,0.741619,0.042953,0.188229,52
2,LatestFail,0.963132,0.739378,0.039917,0.204963,52
3,XGBoost,0.948050,0.649718,0.041369,0.208147,52
4,QTF-Avg,0.173966,0.590571,0.148399,0.302034,52
5,NaiveBayes,0.944314,0.548409,0.058909,0.153733,52
6,Random,0.519483,0.503025,0.249021,0.246059,52



Pilot report:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_preflight/beast2_model_baseline_pilot_noise_000_seed_01/pilot_report.json

Validation status: PASS

SUCCESS: Random Forest, XGBoost, LightGBM and Naive Bayes were actually fitted.
SUCCESS: Random, LatestFail and QTF-Avg were actually evaluated.
SUCCESS: All seven techniques ranked the same 52 clean evaluation builds.
SUCCESS: APFD and APFDc were calculated for every technique and evaluation build.
SUCCESS: The 0% pilot completed without modifying the clean evaluation data.
SUCCESS: Project 7 is ready for a non-zero-noise pilot and then the full 270-condition run.


In [ ]:
# =========================================================
# PROJECT 7 — STEP 8B
# NON-ZERO-NOISE ML + BASELINE PILOT
#
# Condition:
#   Noise: 25%
#   Seed: 1
#
# This is the final pilot before the full 270-condition run.
# =========================================================

from pathlib import Path
import json
import os
import time
import warnings

import numpy as np
import pandas as pd

from IPython.display import clear_output, display


print(
    "=== PROJECT 7 STEP 8B: "
    "25 PERCENT NOISE ML AND BASELINE PILOT ==="
)


# ---------------------------------------------------------
# 1. Validate successful Step 8A state
# ---------------------------------------------------------

required_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",

    "PROJECT_PREFLIGHT_DIRECTORY",
    "PROJECT_7_SELECTION_CHECKPOINT",

    "PILOT_RANKINGS_PATH",
    "PILOT_PROJECT_RUN_METRICS_PATH",

    "ML_TECHNIQUES",
    "BASELINE_TECHNIQUES",
    "ALL_TECHNIQUES",

    "ACTIVE_FEATURE_COLUMNS",

    "create_project_7_noisy_condition",
    "prepare_ml_matrices",
    "create_ml_models",
    "get_failure_probability",
    "create_ml_rankings",

    "create_random_rankings",
    "create_history_baseline_rankings",

    "calculate_build_metrics",

    "CLEAN_RAW_TRAINING_HISTORY",
    "CLEAN_MODEL_TRAINING_DATA",
    "CLEAN_MODEL_EVALUATION_DATA",
]


missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required Step 8A objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nRerun only the successful Project 7 "
        "Step 8A cell."
    )


if (
    PROJECT_NUMBER,
    PROJECT_NAME,
    PROJECT_SLUG,
) != (
    7,
    "CompEvol@beast2",
    "CompEvol__beast2",
):
    raise AssertionError(
        "Unexpected Project 7 identity."
    )


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    previous_checkpoint = json.load(
        checkpoint_file
    )


allowed_previous_statuses = {
    "MODEL_BASELINE_PILOT_PASSED",
    "NONZERO_MODEL_BASELINE_PILOT_PASSED",
}


if previous_checkpoint.get(
    "Status"
) not in allowed_previous_statuses:
    raise AssertionError(
        "Step 8A has not passed.\n"
        f"Observed status: "
        f"{previous_checkpoint.get('Status')}"
    )


if int(
    previous_checkpoint.get(
        "PilotMLFitCount",
        -1,
    )
) != 4:
    raise AssertionError(
        "The checkpoint does not confirm four "
        "successful Step 8A ML fits."
    )


if int(
    previous_checkpoint.get(
        "PilotTechniqueCount",
        -1,
    )
) != 7:
    raise AssertionError(
        "The checkpoint does not confirm seven "
        "successful Step 8A techniques."
    )


# ---------------------------------------------------------
# 2. Freeze the non-zero pilot condition
# ---------------------------------------------------------

NONZERO_PILOT_NOISE_PERCENT = 25
NONZERO_PILOT_REPETITION_SEED = 1


nonzero_condition = (
    create_project_7_noisy_condition(
        noise_percent=(
            NONZERO_PILOT_NOISE_PERCENT
        ),
        repetition_seed=(
            NONZERO_PILOT_REPETITION_SEED
        ),
    )
)


nonzero_training_data = (
    nonzero_condition[
        "NoisyTrainingData"
    ]
    .copy()
    .reset_index(drop=True)
)


nonzero_evaluation_data = (
    nonzero_condition[
        "NoisyEvaluationData"
    ]
    .copy()
    .reset_index(drop=True)
)


nonzero_training_history = (
    nonzero_condition[
        "NoisyTrainingHistory"
    ]
    .copy()
    .reset_index(drop=True)
)


nonzero_noise_manifest = (
    nonzero_condition[
        "NoiseManifest"
    ]
    .copy()
    .reset_index(drop=True)
)


nonzero_noise_summary = dict(
    nonzero_condition[
        "NoiseSummary"
    ]
)


RAW_ROWS_FLIPPED = int(
    nonzero_noise_summary[
        "NumberFlipped"
    ]
)


RETAINED_LABEL_CHANGES = int(
    (
        nonzero_training_data[
            "Verdict"
        ].to_numpy()
        !=
        CLEAN_MODEL_TRAINING_DATA[
            "Verdict"
        ].to_numpy()
    ).sum()
)


if RAW_ROWS_FLIPPED <= 0:
    raise AssertionError(
        "The 25% pilot did not flip any raw verdict."
    )


if RETAINED_LABEL_CHANGES <= 0:
    raise AssertionError(
        "The 25% pilot did not change any retained "
        "model-ready label."
    )


if len(
    nonzero_training_data
) != len(
    CLEAN_MODEL_TRAINING_DATA
):
    raise AssertionError(
        "The fixed training row count changed."
    )


if len(
    nonzero_evaluation_data
) != len(
    CLEAN_MODEL_EVALUATION_DATA
):
    raise AssertionError(
        "The evaluation row count changed."
    )


evaluation_verdict_changes = int(
    (
        nonzero_evaluation_data[
            "Verdict"
        ].to_numpy()
        !=
        CLEAN_MODEL_EVALUATION_DATA[
            "Verdict"
        ].to_numpy()
    ).sum()
)


if evaluation_verdict_changes != 0:
    raise AssertionError(
        "Evaluation verdicts changed under training "
        "noise."
    )


# ---------------------------------------------------------
# 3. Prepare noisy ML matrices
# ---------------------------------------------------------

(
    X_train_25,
    y_train_25,
    X_evaluation_25,
    training_medians_25,
) = prepare_ml_matrices(
    nonzero_training_data,
    nonzero_evaluation_data,
)


if len(
    X_train_25
) != 3960:
    raise AssertionError(
        "Unexpected 25% training matrix row count."
    )


if len(
    X_evaluation_25
) != 3678:
    raise AssertionError(
        "Unexpected 25% evaluation matrix row count."
    )


if y_train_25.nunique() != 2:
    raise AssertionError(
        "The 25% training condition does not contain "
        "both label classes."
    )


# ---------------------------------------------------------
# 4. Fit all four ML models
# ---------------------------------------------------------

models_25 = create_ml_models(
    NONZERO_PILOT_REPETITION_SEED
)


ranking_frames_25 = []
fit_records_25 = []
prediction_records_25 = []


for technique in ML_TECHNIQUES:
    model = models_25[
        technique
    ]


    fit_start = time.time()


    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore"
        )


        model.fit(
            X_train_25,
            y_train_25,
        )


    fit_seconds = float(
        time.time()
        - fit_start
    )


    probabilities = get_failure_probability(
        model,
        X_evaluation_25,
    )


    model_rankings = create_ml_rankings(
        evaluation_outcome_data=(
            nonzero_evaluation_data
        ),
        technique=technique,
        failure_probabilities=probabilities,
    )


    ranking_frames_25.append(
        model_rankings
    )


    fit_records_25.append({
        "Project":
            PROJECT_NAME,

        "NoisePercent":
            NONZERO_PILOT_NOISE_PERCENT,

        "RepetitionSeed":
            NONZERO_PILOT_REPETITION_SEED,

        "Technique":
            technique,

        "FitSeconds":
            fit_seconds,

        "TrainingRows":
            int(
                len(
                    X_train_25
                )
            ),

        "TrainingFailures":
            int(
                y_train_25.sum()
            ),

        "TrainingPasses":
            int(
                len(
                    y_train_25
                )
                - y_train_25.sum()
            ),

        "ActiveFeatures":
            int(
                len(
                    ACTIVE_FEATURE_COLUMNS
                )
            ),
    })


    prediction_records_25.append({
        "Technique":
            technique,

        "PredictionRows":
            int(
                len(
                    probabilities
                )
            ),

        "MinimumProbability":
            float(
                np.min(
                    probabilities
                )
            ),

        "MaximumProbability":
            float(
                np.max(
                    probabilities
                )
            ),

        "MeanProbability":
            float(
                np.mean(
                    probabilities
                )
            ),

        "UniqueProbabilities":
            int(
                pd.Series(
                    probabilities
                ).nunique()
            ),
    })


fit_times_25 = pd.DataFrame(
    fit_records_25
)


prediction_summary_25 = pd.DataFrame(
    prediction_records_25
)


if len(
    fit_times_25
) != 4:
    raise AssertionError(
        "Expected four non-zero-noise ML fits."
    )


# ---------------------------------------------------------
# 5. Evaluate all three baselines
# ---------------------------------------------------------

random_rankings_25 = (
    create_random_rankings(
        nonzero_evaluation_data,
        NONZERO_PILOT_REPETITION_SEED,
    )
)


(
    latest_fail_rankings_25,
    qtf_rankings_25,
) = create_history_baseline_rankings(
    nonzero_training_history
)


# Recreate the clean-noise baselines for comparison.
random_rankings_0_comparison = (
    create_random_rankings(
        CLEAN_MODEL_EVALUATION_DATA,
        NONZERO_PILOT_REPETITION_SEED,
    )
)


(
    latest_fail_rankings_0_comparison,
    qtf_rankings_0_comparison,
) = create_history_baseline_rankings(
    CLEAN_RAW_TRAINING_HISTORY
)


RANDOM_CONSTANT_ACROSS_NOISE = bool(
    random_rankings_25.equals(
        random_rankings_0_comparison
    )
)


QTF_CONSTANT_ACROSS_NOISE = bool(
    qtf_rankings_25.equals(
        qtf_rankings_0_comparison
    )
)


if not RANDOM_CONSTANT_ACROSS_NOISE:
    raise AssertionError(
        "Random rankings changed across noise levels "
        "for the same seed."
    )


if not QTF_CONSTANT_ACROSS_NOISE:
    raise AssertionError(
        "QTF-Avg rankings changed across noise levels."
    )


ranking_frames_25.extend(
    [
        random_rankings_25,
        latest_fail_rankings_25,
        qtf_rankings_25,
    ]
)


# ---------------------------------------------------------
# 6. Combine all seven technique rankings
# ---------------------------------------------------------

all_rankings_25 = pd.concat(
    ranking_frames_25,
    ignore_index=True,
)


all_rankings_25.insert(
    0,
    "Project",
    PROJECT_NAME,
)


all_rankings_25.insert(
    1,
    "NoisePercent",
    float(
        NONZERO_PILOT_NOISE_PERCENT
    ),
)


all_rankings_25.insert(
    2,
    "RepetitionSeed",
    int(
        NONZERO_PILOT_REPETITION_SEED
    ),
)


observed_techniques = set(
    all_rankings_25[
        "Technique"
    ].unique()
)


if observed_techniques != set(
    ALL_TECHNIQUES
):
    raise AssertionError(
        "The 25% pilot did not produce all seven "
        "techniques."
    )


# ---------------------------------------------------------
# 7. Calculate APFD and APFDc
# ---------------------------------------------------------

build_metrics_25 = calculate_build_metrics(
    all_rankings_25,
    NONZERO_PILOT_NOISE_PERCENT,
    NONZERO_PILOT_REPETITION_SEED,
)


project_run_metrics_25 = (
    build_metrics_25
    .groupby(
        [
            "Project",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        as_index=False,
    )
    .agg(
        MeanAPFD=(
            "APFD",
            "mean",
        ),

        MeanAPFDc=(
            "APFDc",
            "mean",
        ),

        SD_APFD=(
            "APFD",
            "std",
        ),

        SD_APFDc=(
            "APFDc",
            "std",
        ),

        EvaluatedBuilds=(
            "Build",
            "nunique",
        ),

        TotalRankedTests=(
            "NumberOfTests",
            "sum",
        ),

        TotalFailures=(
            "NumberOfFailures",
            "sum",
        ),
    )
)


EXPECTED_EVALUATED_BUILDS = int(
    CLEAN_MODEL_EVALUATION_DATA[
        "Build"
    ].nunique()
)


EXPECTED_RANKING_ROWS = int(
    len(
        CLEAN_MODEL_EVALUATION_DATA
    )
    * len(
        ALL_TECHNIQUES
    )
)


EXPECTED_BUILD_METRIC_ROWS = int(
    EXPECTED_EVALUATED_BUILDS
    * len(
        ALL_TECHNIQUES
    )
)


if EXPECTED_EVALUATED_BUILDS != 52:
    raise AssertionError(
        "Expected 52 evaluation builds."
    )


if len(
    all_rankings_25
) != EXPECTED_RANKING_ROWS:
    raise AssertionError(
        "Unexpected 25% ranking-row count.\n"
        f"Expected: {EXPECTED_RANKING_ROWS}\n"
        f"Observed: {len(all_rankings_25)}"
    )


if len(
    build_metrics_25
) != EXPECTED_BUILD_METRIC_ROWS:
    raise AssertionError(
        "Unexpected 25% build-metric row count.\n"
        f"Expected: {EXPECTED_BUILD_METRIC_ROWS}\n"
        f"Observed: {len(build_metrics_25)}"
    )


if len(
    project_run_metrics_25
) != 7:
    raise AssertionError(
        "Expected seven 25% project-run rows."
    )


if build_metrics_25[
    [
        "APFD",
        "APFDc",
    ]
].isna().any().any():
    raise AssertionError(
        "The 25% metrics contain missing values."
    )


for metric in [
    "APFD",
    "APFDc",
]:
    if not build_metrics_25[
        metric
    ].between(
        0.0,
        1.0,
        inclusive="both",
    ).all():
        raise AssertionError(
            f"{metric} contains values outside [0, 1]."
        )


# ---------------------------------------------------------
# 8. Compare 0% and 25% rankings
# ---------------------------------------------------------

zero_rankings_saved = pd.read_parquet(
    PILOT_RANKINGS_PATH
)


zero_project_run_saved = pd.read_csv(
    PILOT_PROJECT_RUN_METRICS_PATH
)


def ranking_change_summary(
    zero_rankings,
    noisy_rankings,
):
    records = []


    for technique in ALL_TECHNIQUES:
        zero_part = (
            zero_rankings[
                zero_rankings[
                    "Technique"
                ] == technique
            ][
                [
                    "Build",
                    "Test",
                    "Rank",
                ]
            ]
            .copy()
            .rename(
                columns={
                    "Rank":
                        "RankAtZeroNoise",
                }
            )
        )


        noisy_part = (
            noisy_rankings[
                noisy_rankings[
                    "Technique"
                ] == technique
            ][
                [
                    "Build",
                    "Test",
                    "Rank",
                ]
            ]
            .copy()
            .rename(
                columns={
                    "Rank":
                        "RankAt25Percent",
                }
            )
        )


        comparison = zero_part.merge(
            noisy_part,
            on=[
                "Build",
                "Test",
            ],
            how="outer",
            validate="one_to_one",
            indicator=True,
        )


        if not (
            comparison[
                "_merge"
            ] == "both"
        ).all():
            raise AssertionError(
                f"Ranking keys changed for {technique}."
            )


        rank_difference = (
            comparison[
                "RankAt25Percent"
            ].astype(np.int64)
            -
            comparison[
                "RankAtZeroNoise"
            ].astype(np.int64)
        )


        changed_rows = int(
            (
                rank_difference != 0
            ).sum()
        )


        records.append({
            "Technique":
                technique,

            "ComparedRows":
                int(
                    len(
                        comparison
                    )
                ),

            "ChangedRanks":
                changed_rows,

            "ChangedRankPercent":
                float(
                    100.0
                    * changed_rows
                    / len(
                        comparison
                    )
                ),

            "MeanAbsoluteRankChange":
                float(
                    np.abs(
                        rank_difference
                    ).mean()
                ),

            "MaximumAbsoluteRankChange":
                int(
                    np.abs(
                        rank_difference
                    ).max()
                ),

            "RankingsIdenticalAcrossNoise":
                bool(
                    changed_rows == 0
                ),
        })


    return pd.DataFrame(
        records
    )


rank_change_summary = (
    ranking_change_summary(
        zero_rankings_saved,
        all_rankings_25,
    )
)


random_rank_change_count = int(
    rank_change_summary.loc[
        rank_change_summary[
            "Technique"
        ] == "Random",
        "ChangedRanks",
    ].iloc[0]
)


qtf_rank_change_count = int(
    rank_change_summary.loc[
        rank_change_summary[
            "Technique"
        ] == "QTF-Avg",
        "ChangedRanks",
    ].iloc[0]
)


if random_rank_change_count != 0:
    raise AssertionError(
        "Random ranks changed across noise."
    )


if qtf_rank_change_count != 0:
    raise AssertionError(
        "QTF-Avg ranks changed across noise."
    )


# ---------------------------------------------------------
# 9. Compare 0% and 25% project-run metrics
# ---------------------------------------------------------

metric_comparison = (
    zero_project_run_saved[
        [
            "Technique",
            "MeanAPFD",
            "MeanAPFDc",
        ]
    ]
    .rename(
        columns={
            "MeanAPFD":
                "MeanAPFD_0",

            "MeanAPFDc":
                "MeanAPFDc_0",
        }
    )
    .merge(
        project_run_metrics_25[
            [
                "Technique",
                "MeanAPFD",
                "MeanAPFDc",
            ]
        ]
        .rename(
            columns={
                "MeanAPFD":
                    "MeanAPFD_25",

                "MeanAPFDc":
                    "MeanAPFDc_25",
            }
        ),
        on="Technique",
        how="inner",
        validate="one_to_one",
    )
)


metric_comparison[
    "DeltaAPFD_25Minus0"
] = (
    metric_comparison[
        "MeanAPFD_25"
    ]
    - metric_comparison[
        "MeanAPFD_0"
    ]
)


metric_comparison[
    "DeltaAPFDc_25Minus0"
] = (
    metric_comparison[
        "MeanAPFDc_25"
    ]
    - metric_comparison[
        "MeanAPFDc_0"
    ]
)


if len(
    metric_comparison
) != 7:
    raise AssertionError(
        "The clean/noisy metric comparison does not "
        "contain all seven techniques."
    )


# ---------------------------------------------------------
# 10. Evaluation immutability
# ---------------------------------------------------------

evaluation_verdicts_clean = bool(
    np.array_equal(
        nonzero_evaluation_data[
            "Verdict"
        ].to_numpy(),

        CLEAN_MODEL_EVALUATION_DATA[
            "Verdict"
        ].to_numpy(),
    )
)


evaluation_durations_clean = bool(
    np.array_equal(
        nonzero_evaluation_data[
            "Duration"
        ].to_numpy(),

        CLEAN_MODEL_EVALUATION_DATA[
            "Duration"
        ].to_numpy(),
    )
)


if not evaluation_verdicts_clean:
    raise AssertionError(
        "Evaluation verdicts were modified."
    )


if not evaluation_durations_clean:
    raise AssertionError(
        "Evaluation durations were modified."
    )


# ---------------------------------------------------------
# 11. Save permanent Step 8B artefacts
# ---------------------------------------------------------

NONZERO_PILOT_DIRECTORY = (
    PROJECT_PREFLIGHT_DIRECTORY
    / "beast2_model_baseline_pilot_noise_025_seed_01"
)


NONZERO_PILOT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


NONZERO_RANKINGS_PATH = (
    NONZERO_PILOT_DIRECTORY
    / "rankings.parquet"
)


NONZERO_BUILD_METRICS_PATH = (
    NONZERO_PILOT_DIRECTORY
    / "build_metrics.parquet"
)


NONZERO_PROJECT_RUN_METRICS_PATH = (
    NONZERO_PILOT_DIRECTORY
    / "project_run_metrics.csv"
)


NONZERO_FIT_TIMES_PATH = (
    NONZERO_PILOT_DIRECTORY
    / "fit_times.csv"
)


NONZERO_PREDICTION_SUMMARY_PATH = (
    NONZERO_PILOT_DIRECTORY
    / "prediction_summary.csv"
)


NONZERO_NOISE_MANIFEST_PATH = (
    NONZERO_PILOT_DIRECTORY
    / "noise_manifest.parquet"
)


NONZERO_RANK_CHANGE_PATH = (
    NONZERO_PILOT_DIRECTORY
    / "rank_change_summary_vs_zero.csv"
)


NONZERO_METRIC_COMPARISON_PATH = (
    NONZERO_PILOT_DIRECTORY
    / "metric_comparison_vs_zero.csv"
)


NONZERO_PILOT_REPORT_PATH = (
    NONZERO_PILOT_DIRECTORY
    / "pilot_report.json"
)


all_rankings_25.to_parquet(
    NONZERO_RANKINGS_PATH,
    index=False,
)


build_metrics_25.to_parquet(
    NONZERO_BUILD_METRICS_PATH,
    index=False,
)


project_run_metrics_25.to_csv(
    NONZERO_PROJECT_RUN_METRICS_PATH,
    index=False,
)


fit_times_25.to_csv(
    NONZERO_FIT_TIMES_PATH,
    index=False,
)


prediction_summary_25.to_csv(
    NONZERO_PREDICTION_SUMMARY_PATH,
    index=False,
)


nonzero_noise_manifest.to_parquet(
    NONZERO_NOISE_MANIFEST_PATH,
    index=False,
)


rank_change_summary.to_csv(
    NONZERO_RANK_CHANGE_PATH,
    index=False,
)


metric_comparison.to_csv(
    NONZERO_METRIC_COMPARISON_PATH,
    index=False,
)


completed_at = (
    pd.Timestamp.utcnow().isoformat()
)


nonzero_pilot_report = {
    "ProjectNumber":
        int(
            PROJECT_NUMBER
        ),

    "Project":
        str(
            PROJECT_NAME
        ),

    "ProjectSlug":
        str(
            PROJECT_SLUG
        ),

    "Status":
        "PASS",

    "PilotCondition": {
        "NoisePercent":
            NONZERO_PILOT_NOISE_PERCENT,

        "RepetitionSeed":
            NONZERO_PILOT_REPETITION_SEED,

        "RawRowsFlipped":
            RAW_ROWS_FLIPPED,

        "RetainedModelLabelChanges":
            RETAINED_LABEL_CHANGES,
    },

    "MLModelsActuallyFitted":
        list(
            ML_TECHNIQUES
        ),

    "BaselinesActuallyEvaluated":
        list(
            BASELINE_TECHNIQUES
        ),

    "OutputCounts": {
        "TrainingRows":
            int(
                len(
                    X_train_25
                )
            ),

        "EvaluationRows":
            int(
                len(
                    X_evaluation_25
                )
            ),

        "EvaluatedBuilds":
            EXPECTED_EVALUATED_BUILDS,

        "Techniques":
            int(
                len(
                    ALL_TECHNIQUES
                )
            ),

        "RankingRows":
            int(
                len(
                    all_rankings_25
                )
            ),

        "BuildMetricRows":
            int(
                len(
                    build_metrics_25
                )
            ),

        "ProjectRunMetricRows":
            int(
                len(
                    project_run_metrics_25
                )
            ),

        "MLFits":
            int(
                len(
                    fit_times_25
                )
            ),
    },

    "CrossNoiseBaselineValidation": {
        "RandomConstantAcrossNoise":
            RANDOM_CONSTANT_ACROSS_NOISE,

        "QTFAvgConstantAcrossNoise":
            QTF_CONSTANT_ACROSS_NOISE,

        "RandomChangedRanks":
            random_rank_change_count,

        "QTFAvgChangedRanks":
            qtf_rank_change_count,

        "LatestFailChangedRanks":
            int(
                rank_change_summary.loc[
                    rank_change_summary[
                        "Technique"
                    ] == "LatestFail",
                    "ChangedRanks",
                ].iloc[0]
            ),
    },

    "EvaluationValidation": {
        "VerdictsRemainClean":
            evaluation_verdicts_clean,

        "DurationsRemainClean":
            evaluation_durations_clean,

        "AllMetricsFinite":
            True,

        "AllMetricsWithinUnitInterval":
            True,
    },

    "Artefacts": {
        "Rankings":
            str(
                NONZERO_RANKINGS_PATH
            ),

        "BuildMetrics":
            str(
                NONZERO_BUILD_METRICS_PATH
            ),

        "ProjectRunMetrics":
            str(
                NONZERO_PROJECT_RUN_METRICS_PATH
            ),

        "FitTimes":
            str(
                NONZERO_FIT_TIMES_PATH
            ),

        "PredictionSummary":
            str(
                NONZERO_PREDICTION_SUMMARY_PATH
            ),

        "NoiseManifest":
            str(
                NONZERO_NOISE_MANIFEST_PATH
            ),

        "RankChangeSummary":
            str(
                NONZERO_RANK_CHANGE_PATH
            ),

        "MetricComparisonWithZero":
            str(
                NONZERO_METRIC_COMPARISON_PATH
            ),
    },

    "CompletedAtUTC":
        completed_at,
}


with open(
    NONZERO_PILOT_REPORT_PATH,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        nonzero_pilot_report,
        report_file,
        indent=2,
    )


# ---------------------------------------------------------
# 12. Update checkpoint atomically
# ---------------------------------------------------------

with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    project_checkpoint = json.load(
        checkpoint_file
    )


project_checkpoint.update({
    "Status":
        "NONZERO_MODEL_BASELINE_PILOT_PASSED",

    "NonZeroPilotNoisePercent":
        NONZERO_PILOT_NOISE_PERCENT,

    "NonZeroPilotRepetitionSeed":
        NONZERO_PILOT_REPETITION_SEED,

    "NonZeroPilotRawRowsFlipped":
        RAW_ROWS_FLIPPED,

    "NonZeroPilotRetainedLabelChanges":
        RETAINED_LABEL_CHANGES,

    "NonZeroPilotMLFitCount":
        int(
            len(
                fit_times_25
            )
        ),

    "NonZeroPilotTechniqueCount":
        int(
            len(
                ALL_TECHNIQUES
            )
        ),

    "NonZeroPilotEvaluatedBuilds":
        EXPECTED_EVALUATED_BUILDS,

    "NonZeroPilotRankingRows":
        int(
            len(
                all_rankings_25
            )
        ),

    "NonZeroPilotBuildMetricRows":
        int(
            len(
                build_metrics_25
            )
        ),

    "RandomConstantAcrossNoise":
        RANDOM_CONSTANT_ACROSS_NOISE,

    "QTFAvgConstantAcrossNoise":
        QTF_CONSTANT_ACROSS_NOISE,

    "NonZeroPilotEvaluationVerdictsClean":
        evaluation_verdicts_clean,

    "NonZeroPilotEvaluationDurationsClean":
        evaluation_durations_clean,

    "NonZeroModelBaselinePilotReport":
        str(
            NONZERO_PILOT_REPORT_PATH
        ),

    "NonZeroModelBaselinePilotDirectory":
        str(
            NONZERO_PILOT_DIRECTORY
        ),

    "UpdatedAtUTC":
        completed_at,
})


temporary_checkpoint_path = (
    PROJECT_7_SELECTION_CHECKPOINT
    .with_suffix(
        ".json.tmp"
    )
)


with open(
    temporary_checkpoint_path,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        project_checkpoint,
        checkpoint_file,
        indent=2,
    )


os.replace(
    temporary_checkpoint_path,
    PROJECT_7_SELECTION_CHECKPOINT,
)


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint_verification = json.load(
        checkpoint_file
    )


if checkpoint_verification.get(
    "Status"
) != "NONZERO_MODEL_BASELINE_PILOT_PASSED":
    raise AssertionError(
        "The Step 8B checkpoint was not written "
        "correctly."
    )


# ---------------------------------------------------------
# 13. Compact final output
# ---------------------------------------------------------

clear_output(
    wait=True
)


print(
    "=== PROJECT 7 STEP 8B RESULT ==="
)


print(
    "\nNon-zero pilot condition:"
)

print(
    "Noise percent:",
    NONZERO_PILOT_NOISE_PERCENT
)

print(
    "Repetition seed:",
    NONZERO_PILOT_REPETITION_SEED
)

print(
    "Raw training verdict flips:",
    RAW_ROWS_FLIPPED
)

print(
    "Retained model-ready label changes:",
    RETAINED_LABEL_CHANGES
)


print(
    "\nML models fitted:"
)

display(
    fit_times_25[
        [
            "Technique",
            "FitSeconds",
            "TrainingRows",
            "TrainingFailures",
            "TrainingPasses",
            "ActiveFeatures",
        ]
    ]
)


print(
    "\nPrediction summary:"
)

display(
    prediction_summary_25
)


print(
    "\nBaseline invariants:"
)

print(
    "Random constant across noise:",
    RANDOM_CONSTANT_ACROSS_NOISE
)

print(
    "QTF-Avg constant across noise:",
    QTF_CONSTANT_ACROSS_NOISE
)

print(
    "Evaluation verdicts clean:",
    evaluation_verdicts_clean
)

print(
    "Evaluation durations clean:",
    evaluation_durations_clean
)


print(
    "\nRanking changes from 0% to 25%:"
)

display(
    rank_change_summary
)


print(
    "\nProject-run metric comparison:"
)

display(
    metric_comparison[
        [
            "Technique",
            "MeanAPFD_0",
            "MeanAPFD_25",
            "DeltaAPFD_25Minus0",
            "MeanAPFDc_0",
            "MeanAPFDc_25",
            "DeltaAPFDc_25Minus0",
        ]
    ]
    .sort_values(
        [
            "MeanAPFDc_25",
            "Technique",
        ],
        ascending=[
            False,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


print(
    "\nOutput validation:"
)

print(
    "Techniques:",
    len(
        ALL_TECHNIQUES
    )
)

print(
    "Evaluated builds:",
    EXPECTED_EVALUATED_BUILDS
)

print(
    "Ranking rows:",
    len(
        all_rankings_25
    )
)

print(
    "Expected ranking rows:",
    EXPECTED_RANKING_ROWS
)

print(
    "Build-metric rows:",
    len(
        build_metrics_25
    )
)

print(
    "Expected build-metric rows:",
    EXPECTED_BUILD_METRIC_ROWS
)

print(
    "Project-run rows:",
    len(
        project_run_metrics_25
    )
)


print(
    "\nPilot report:"
)

print(
    NONZERO_PILOT_REPORT_PATH
)


print(
    "\nValidation status:",
    "PASS"
)


print(
    "\nSUCCESS: The 25% noisy training condition was "
    "used to fit all four ML models."
)

print(
    "SUCCESS: Random, LatestFail and QTF-Avg were "
    "evaluated under the non-zero condition."
)

print(
    "SUCCESS: Random remained constant across noise for "
    "the same seed."
)

print(
    "SUCCESS: QTF-Avg remained completely "
    "noise-independent."
)

print(
    "SUCCESS: Evaluation verdicts and durations remained "
    "clean."
)

print(
    "SUCCESS: Project 7 is ready for the complete "
    "270-condition experiment."
)

=== PROJECT 7 STEP 8B RESULT ===

Non-zero pilot condition:
Noise percent: 25
Repetition seed: 1
Raw training verdict flips: 5035
Retained model-ready label changes: 1009

ML models fitted:


,Technique,FitSeconds,TrainingRows,TrainingFailures,TrainingPasses,ActiveFeatures
0,RandomForest,2.799875,3960,1101,2859,150
1,XGBoost,3.846765,3960,1101,2859,150
2,LightGBM,2.155782,3960,1101,2859,150
3,NaiveBayes,0.025300,3960,1101,2859,150



Prediction summary:


,Technique,PredictionRows,MinimumProbability,MaximumProbability,MeanProbability,UniqueProbabilities
0,RandomForest,3678,1.000000e-02,0.790000,0.462319,74
1,XGBoost,3678,4.852541e-02,0.902506,0.591798,2258
2,LightGBM,3678,4.977312e-02,0.830792,0.414735,2446
3,NaiveBayes,3678,6.494006e-35,1.000000,0.887089,1324



Baseline invariants:
Random constant across noise: True
QTF-Avg constant across noise: True
Evaluation verdicts clean: True
Evaluation durations clean: True

Ranking changes from 0% to 25%:


,Technique,ComparedRows,ChangedRanks,ChangedRankPercent,MeanAbsoluteRankChange,MaximumAbsoluteRankChange,RankingsIdenticalAcrossNoise
0,RandomForest,3678,3641,98.994018,22.373573,69,False
1,XGBoost,3678,3642,99.021207,25.216422,70,False
2,LightGBM,3678,3622,98.477433,24.116911,70,False
3,NaiveBayes,3678,3624,98.531811,26.531811,69,False
4,Random,3678,0,0.000000,0.000000,0,True
5,LatestFail,3678,3065,83.333333,14.220772,54,False
6,QTF-Avg,3678,0,0.000000,0.000000,0,True



Project-run metric comparison:


,Technique,MeanAPFD_0,MeanAPFD_25,DeltaAPFD_25Minus0,MeanAPFDc_0,MeanAPFDc_25,DeltaAPFDc_25Minus0
0,LatestFail,0.963132,0.955822,-7.310114e-03,0.739378,0.747484,0.008106
1,QTF-Avg,0.173966,0.173966,8.326673e-17,0.590571,0.590571,0.000000
2,RandomForest,0.960136,0.560432,-3.997046e-01,0.746792,0.505931,-0.240862
3,Random,0.519483,0.519483,0.000000e+00,0.503025,0.503025,0.000000
4,LightGBM,0.955960,0.448650,-5.073096e-01,0.741619,0.424100,-0.317518
5,XGBoost,0.948050,0.254030,-6.940200e-01,0.649718,0.367463,-0.282255
6,NaiveBayes,0.944314,0.394634,-5.496800e-01,0.548409,0.326565,-0.221844



Output validation:
Techniques: 7
Evaluated builds: 52
Ranking rows: 25746
Expected ranking rows: 25746
Build-metric rows: 364
Expected build-metric rows: 364
Project-run rows: 7

Pilot report:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_preflight/beast2_model_baseline_pilot_noise_025_seed_01/pilot_report.json

Validation status: PASS

SUCCESS: The 25% noisy training condition was used to fit all four ML models.
SUCCESS: Random, LatestFail and QTF-Avg were evaluated under the non-zero condition.
SUCCESS: Random remained constant across noise for the same seed.
SUCCESS: QTF-Avg remained completely noise-independent.
SUCCESS: Evaluation verdicts and durations remained clean.
SUCCESS: Project 7 is ready for the complete 270-condition experiment.


In [ ]:
# =========================================================
# PROJECT 7 — STEP 9
# COMPLETE, RESUMABLE 270-CONDITION EXPERIMENT
# =========================================================

from pathlib import Path
import gc
import json
import os
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import clear_output, display

print("=== PROJECT 7 STEP 9: COMPLETE 270-CONDITION EXPERIMENT ===")

# ---------------------------------------------------------
# 1. Validate the successful Step 8B runtime state
# ---------------------------------------------------------
required_objects = [
    "PROJECT_NUMBER", "PROJECT_NAME", "PROJECT_SLUG",
    "PROJECT_7_SELECTION_CHECKPOINT", "RAW_RESULTS_DRIVE", "LOGS_DRIVE",
    "NOISE_LEVELS", "REPETITION_SEEDS", "ML_TECHNIQUES",
    "BASELINE_TECHNIQUES", "ALL_TECHNIQUES", "ACTIVE_FEATURE_COLUMNS",
    "create_project_7_noisy_condition", "prepare_ml_matrices",
    "create_ml_models", "get_failure_probability", "create_ml_rankings",
    "create_random_rankings", "create_history_baseline_rankings",
    "calculate_build_metrics", "CLEAN_MODEL_TRAINING_DATA",
    "CLEAN_MODEL_EVALUATION_DATA", "CLEAN_RAW_TRAINING_HISTORY",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required Step 8B objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nRerun only the successful Project 7 "
        "Steps 7A correction, 8A and 8B."
    )

if (
    PROJECT_NUMBER,
    PROJECT_NAME,
    PROJECT_SLUG,
) != (
    7,
    "CompEvol@beast2",
    "CompEvol__beast2",
):
    raise AssertionError(
        "Unexpected Project 7 identity."
    )

with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    previous_checkpoint = json.load(
        checkpoint_file
    )

if previous_checkpoint.get(
    "Status"
) not in {
    "NONZERO_MODEL_BASELINE_PILOT_PASSED",
    "FULL_EXPERIMENT_RUNNING",
    "FULL_EXPERIMENT_RUN_PASSED",
}:
    raise AssertionError(
        "Step 8B has not passed. Observed status: "
        + str(
            previous_checkpoint.get(
                "Status"
            )
        )
    )

if list(NOISE_LEVELS) != [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]:
    raise AssertionError(
        "Unexpected frozen noise-level grid."
    )

if list(REPETITION_SEEDS) != list(
    range(1, 31)
):
    raise AssertionError(
        "Unexpected frozen repetition-seed grid."
    )

if set(ML_TECHNIQUES) != {
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
}:
    raise AssertionError(
        "Unexpected ML-technique set."
    )

if set(BASELINE_TECHNIQUES) != {
    "Random",
    "LatestFail",
    "QTF-Avg",
}:
    raise AssertionError(
        "Unexpected baseline-technique set."
    )

if len(ALL_TECHNIQUES) != 7:
    raise AssertionError(
        "Expected seven techniques."
    )


# ---------------------------------------------------------
# 2. Freeze output structure and expected counts
# ---------------------------------------------------------
FULL_RAW_DIRECTORY = (
    RAW_RESULTS_DRIVE
    / PROJECT_SLUG
    / "beast2_30_seed_raw"
)

CONDITIONS_DIRECTORY = (
    FULL_RAW_DIRECTORY
    / "conditions"
)

FULL_PROGRESS_PATH = (
    LOGS_DRIVE
    / "beast2_full_experiment_progress.csv"
)

FULL_RUN_REPORT_PATH = (
    LOGS_DRIVE
    / "beast2_full_experiment_run_report.json"
)

CONDITIONS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

LOGS_DRIVE.mkdir(
    parents=True,
    exist_ok=True,
)

EXPECTED_CONDITIONS = (
    len(NOISE_LEVELS)
    * len(REPETITION_SEEDS)
)

EXPECTED_EVALUATED_BUILDS = (
    CLEAN_MODEL_EVALUATION_DATA[
        "Build"
    ].nunique()
)

EXPECTED_RANKING_ROWS_PER_CONDITION = (
    len(CLEAN_MODEL_EVALUATION_DATA)
    * len(ALL_TECHNIQUES)
)

EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = (
    EXPECTED_EVALUATED_BUILDS
    * len(ALL_TECHNIQUES)
)

EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = (
    len(ALL_TECHNIQUES)
)

EXPECTED_FIT_ROWS_PER_CONDITION = (
    len(ML_TECHNIQUES)
)

EXPECTED_PREDICTION_ROWS_PER_CONDITION = (
    len(ML_TECHNIQUES)
)

EXPECTED_MANIFEST_ROWS_PER_CONDITION = (
    len(CLEAN_RAW_TRAINING_HISTORY)
)

EXPECTED_FILES_PER_CONDITION = 8

EXPECTED_TOTAL_RAW_FILES = (
    EXPECTED_CONDITIONS
    * EXPECTED_FILES_PER_CONDITION
)

EXPECTED_TOTAL_ML_FITS = (
    EXPECTED_CONDITIONS
    * EXPECTED_FIT_ROWS_PER_CONDITION
)

EXPECTED_TOTAL_RANKING_ROWS = (
    EXPECTED_CONDITIONS
    * EXPECTED_RANKING_ROWS_PER_CONDITION
)

EXPECTED_TOTAL_BUILD_METRIC_ROWS = (
    EXPECTED_CONDITIONS
    * EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
)

EXPECTED_TOTAL_PROJECT_RUN_ROWS = (
    EXPECTED_CONDITIONS
    * EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
)

EXPECTED_TOTAL_MANIFEST_ROWS = (
    EXPECTED_CONDITIONS
    * EXPECTED_MANIFEST_ROWS_PER_CONDITION
)

expected_exact = {
    "conditions": (
        EXPECTED_CONDITIONS,
        270,
    ),

    "evaluation builds": (
        EXPECTED_EVALUATED_BUILDS,
        52,
    ),

    "ranking rows per condition": (
        EXPECTED_RANKING_ROWS_PER_CONDITION,
        25746,
    ),

    "build metric rows per condition": (
        EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
        364,
    ),

    "manifest rows per condition": (
        EXPECTED_MANIFEST_ROWS_PER_CONDITION,
        19766,
    ),
}

for label, (
    observed,
    expected,
) in expected_exact.items():
    if int(observed) != int(expected):
        raise AssertionError(
            f"Unexpected {label}: "
            f"expected {expected}, "
            f"observed {observed}."
        )


# ---------------------------------------------------------
# 3. Atomic writers and checkpoint helper
# ---------------------------------------------------------
def atomic_write_json(
    value,
    destination,
):
    destination = Path(
        destination
    )

    temporary = destination.with_name(
        destination.name
        + ".tmp"
    )

    with open(
        temporary,
        "w",
        encoding="utf-8",
    ) as output_file:
        json.dump(
            value,
            output_file,
            indent=2,
            default=str,
        )

    os.replace(
        temporary,
        destination,
    )


def atomic_write_csv(
    dataframe,
    destination,
):
    destination = Path(
        destination
    )

    temporary = destination.with_name(
        destination.stem
        + ".tmp"
        + destination.suffix
    )

    dataframe.to_csv(
        temporary,
        index=False,
    )

    os.replace(
        temporary,
        destination,
    )


def atomic_write_parquet(
    dataframe,
    destination,
):
    destination = Path(
        destination
    )

    temporary = destination.with_name(
        destination.stem
        + ".tmp"
        + destination.suffix
    )

    dataframe.to_parquet(
        temporary,
        index=False,
    )

    os.replace(
        temporary,
        destination,
    )


def update_project_checkpoint(
    updates,
):
    with open(
        PROJECT_7_SELECTION_CHECKPOINT,
        "r",
        encoding="utf-8",
    ) as checkpoint_file:
        checkpoint = json.load(
            checkpoint_file
        )

    checkpoint.update(
        updates
    )

    temporary = (
        PROJECT_7_SELECTION_CHECKPOINT
        .with_suffix(
            ".json.tmp"
        )
    )

    with open(
        temporary,
        "w",
        encoding="utf-8",
    ) as checkpoint_file:
        json.dump(
            checkpoint,
            checkpoint_file,
            indent=2,
            default=str,
        )

    os.replace(
        temporary,
        PROJECT_7_SELECTION_CHECKPOINT,
    )


# ---------------------------------------------------------
# 4. Condition paths and resumability
# ---------------------------------------------------------
def condition_key(
    noise_percent,
    repetition_seed,
):
    return (
        f"noise_{int(noise_percent):03d}_"
        f"seed_{int(repetition_seed):02d}"
    )


def get_condition_paths(
    noise_percent,
    repetition_seed,
):
    key = condition_key(
        noise_percent,
        repetition_seed,
    )

    directory = (
        CONDITIONS_DIRECTORY
        / key
    )

    return {
        "Key":
            key,

        "Directory":
            directory,

        "Rankings":
            directory
            / "rankings.parquet",

        "BuildMetrics":
            directory
            / "build_metrics.parquet",

        "ProjectRunMetrics":
            directory
            / "project_run_metrics.csv",

        "FitTimes":
            directory
            / "fit_times.csv",

        "PredictionSummary":
            directory
            / "prediction_summary.csv",

        "NoiseManifest":
            directory
            / "noise_manifest.parquet",

        "ConditionReport":
            directory
            / "condition_report.json",

        "SuccessMarker":
            directory
            / "_SUCCESS.json",
    }


def cleanup_condition_temporaries(
    directory,
):
    directory = Path(
        directory
    )

    if directory.exists():
        for path in directory.glob(
            "*.tmp*"
        ):
            if path.is_file():
                path.unlink()


def condition_is_complete(
    paths,
):
    required = [
        paths[
            "Rankings"
        ],
        paths[
            "BuildMetrics"
        ],
        paths[
            "ProjectRunMetrics"
        ],
        paths[
            "FitTimes"
        ],
        paths[
            "PredictionSummary"
        ],
        paths[
            "NoiseManifest"
        ],
        paths[
            "ConditionReport"
        ],
        paths[
            "SuccessMarker"
        ],
    ]

    if not all(
        path.is_file()
        and path.stat().st_size > 0
        for path in required
    ):
        return False

    try:
        with open(
            paths[
                "SuccessMarker"
            ],
            "r",
            encoding="utf-8",
        ) as marker_file:
            marker = json.load(
                marker_file
            )

    except Exception:
        return False

    return bool(
        marker.get(
            "Status"
        ) == "COMPLETE"

        and int(
            marker.get(
                "RankingRows",
                -1,
            )
        ) == (
            EXPECTED_RANKING_ROWS_PER_CONDITION
        )

        and int(
            marker.get(
                "BuildMetricRows",
                -1,
            )
        ) == (
            EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
        )

        and int(
            marker.get(
                "ProjectRunRows",
                -1,
            )
        ) == (
            EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
        )

        and int(
            marker.get(
                "MLFitRows",
                -1,
            )
        ) == (
            EXPECTED_FIT_ROWS_PER_CONDITION
        )

        and int(
            marker.get(
                "ManifestRows",
                -1,
            )
        ) == (
            EXPECTED_MANIFEST_ROWS_PER_CONDITION
        )
    )


# ---------------------------------------------------------
# 5. Shared baseline caches
# ---------------------------------------------------------
# QTF-Avg is duration-only and independent of noise/seed.
(
    _latest_fail_clean_reference,
    QTF_RANKINGS_SHARED,
) = create_history_baseline_rankings(
    CLEAN_RAW_TRAINING_HISTORY
)

if len(
    QTF_RANKINGS_SHARED
) != len(
    CLEAN_MODEL_EVALUATION_DATA
):
    raise AssertionError(
        "Shared QTF-Avg ranking has the wrong row count."
    )

RANDOM_RANKINGS_BY_SEED = {}


def get_random_rankings_for_seed(
    repetition_seed,
):
    repetition_seed = int(
        repetition_seed
    )

    if repetition_seed not in (
        RANDOM_RANKINGS_BY_SEED
    ):
        RANDOM_RANKINGS_BY_SEED[
            repetition_seed
        ] = create_random_rankings(
            CLEAN_MODEL_EVALUATION_DATA,
            repetition_seed,
        )

    return (
        RANDOM_RANKINGS_BY_SEED[
            repetition_seed
        ]
        .copy(deep=True)
        .reset_index(drop=True)
    )


# ---------------------------------------------------------
# 6. Validate one condition's outputs
# ---------------------------------------------------------
def validate_condition_outputs(
    noise_percent,
    repetition_seed,
    condition,
    rankings,
    build_metrics,
    project_run_metrics,
    fit_times,
    prediction_summary,
):
    condition_label = (
        f"noise={noise_percent}, "
        f"seed={repetition_seed}"
    )

    if set(
        rankings[
            "Technique"
        ].unique()
    ) != set(
        ALL_TECHNIQUES
    ):
        raise AssertionError(
            f"Technique mismatch for "
            f"{condition_label}."
        )

    expected_lengths = {
        "ranking rows": (
            len(
                rankings
            ),
            EXPECTED_RANKING_ROWS_PER_CONDITION,
        ),

        "build-metric rows": (
            len(
                build_metrics
            ),
            EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
        ),

        "project-run rows": (
            len(
                project_run_metrics
            ),
            EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
        ),

        "fit rows": (
            len(
                fit_times
            ),
            EXPECTED_FIT_ROWS_PER_CONDITION,
        ),

        "prediction rows": (
            len(
                prediction_summary
            ),
            EXPECTED_PREDICTION_ROWS_PER_CONDITION,
        ),

        "manifest rows": (
            len(
                condition[
                    "NoiseManifest"
                ]
            ),
            EXPECTED_MANIFEST_ROWS_PER_CONDITION,
        ),
    }

    for name, (
        observed,
        expected,
    ) in expected_lengths.items():
        if observed != expected:
            raise AssertionError(
                f"Unexpected {name} for "
                f"{condition_label}: "
                f"{observed} != {expected}."
            )

    summary_flips = int(
        condition[
            "NoiseSummary"
        ][
            "NumberFlipped"
        ]
    )

    manifest_flips = int(
        condition[
            "NoiseManifest"
        ][
            "Flipped"
        ].astype(bool).sum()
    )

    if summary_flips != manifest_flips:
        raise AssertionError(
            "Noise-summary and manifest flip counts "
            f"differ for {condition_label}."
        )

    if int(
        noise_percent
    ) == 0:
        if summary_flips != 0:
            raise AssertionError(
                "A 0% condition contains verdict flips."
            )

        if not condition[
            "NoisyTrainingData"
        ].equals(
            CLEAN_MODEL_TRAINING_DATA
        ):
            raise AssertionError(
                "A 0% condition does not exactly "
                "reproduce clean training data."
            )

        if not condition[
            "NoisyEvaluationData"
        ].equals(
            CLEAN_MODEL_EVALUATION_DATA
        ):
            raise AssertionError(
                "A 0% condition does not exactly "
                "reproduce clean evaluation data."
            )

    evaluation_verdict_changes = int(
        (
            condition[
                "NoisyEvaluationData"
            ][
                "Verdict"
            ].to_numpy()
            !=
            CLEAN_MODEL_EVALUATION_DATA[
                "Verdict"
            ].to_numpy()
        ).sum()
    )

    if evaluation_verdict_changes != 0:
        raise AssertionError(
            "Evaluation verdicts changed for "
            f"{condition_label}."
        )

    if build_metrics[
        [
            "APFD",
            "APFDc",
        ]
    ].isna().any().any():
        raise AssertionError(
            "APFD or APFDc is missing for "
            f"{condition_label}."
        )

    for metric in [
        "APFD",
        "APFDc",
    ]:
        if not build_metrics[
            metric
        ].between(
            0.0,
            1.0,
            inclusive="both",
        ).all():
            raise AssertionError(
                f"{metric} is outside [0, 1] "
                f"for {condition_label}."
            )

    if set(
        project_run_metrics[
            "Technique"
        ]
    ) != set(
        ALL_TECHNIQUES
    ):
        raise AssertionError(
            "Project-run techniques are incomplete "
            f"for {condition_label}."
        )

    if not (
        project_run_metrics[
            "EvaluatedBuilds"
        ] == EXPECTED_EVALUATED_BUILDS
    ).all():
        raise AssertionError(
            "Not every technique evaluated all "
            f"52 builds for {condition_label}."
        )

    if set(
        build_metrics[
            "Build"
        ].astype(np.int64)
    ) != set(
        CLEAN_MODEL_EVALUATION_DATA[
            "Build"
        ].astype(np.int64)
    ):
        raise AssertionError(
            "Build-metric coverage is incomplete "
            f"for {condition_label}."
        )


# ---------------------------------------------------------
# 7. Run one complete noise-seed condition
# ---------------------------------------------------------
def run_full_condition(
    noise_percent,
    repetition_seed,
):
    condition_started = time.time()

    condition = create_project_7_noisy_condition(
        noise_percent=int(
            noise_percent
        ),
        repetition_seed=int(
            repetition_seed
        ),
    )

    noisy_training_data = condition[
        "NoisyTrainingData"
    ]

    noisy_evaluation_data = condition[
        "NoisyEvaluationData"
    ]

    noisy_training_history = condition[
        "NoisyTrainingHistory"
    ]

    (
        X_train,
        y_train,
        X_evaluation,
        _training_medians,
    ) = prepare_ml_matrices(
        noisy_training_data,
        noisy_evaluation_data,
    )

    models = create_ml_models(
        int(
            repetition_seed
        )
    )

    ranking_frames = []
    fit_records = []
    prediction_records = []

    for technique in ML_TECHNIQUES:
        model = models[
            technique
        ]

        fit_started = time.time()

        with warnings.catch_warnings():
            warnings.simplefilter(
                "ignore"
            )

            model.fit(
                X_train,
                y_train,
            )

        fit_seconds = (
            time.time()
            - fit_started
        )

        probabilities = (
            get_failure_probability(
                model,
                X_evaluation,
            )
        )

        ranking_frames.append(
            create_ml_rankings(
                evaluation_outcome_data=(
                    noisy_evaluation_data
                ),
                technique=technique,
                failure_probabilities=(
                    probabilities
                ),
            )
        )

        fit_records.append({
            "Project":
                PROJECT_NAME,

            "NoisePercent":
                int(
                    noise_percent
                ),

            "RepetitionSeed":
                int(
                    repetition_seed
                ),

            "Technique":
                technique,

            "FitSeconds":
                float(
                    fit_seconds
                ),

            "TrainingRows":
                len(
                    X_train
                ),

            "TrainingFailures":
                int(
                    y_train.sum()
                ),

            "TrainingPasses":
                int(
                    len(
                        y_train
                    )
                    - y_train.sum()
                ),

            "ActiveFeatures":
                len(
                    ACTIVE_FEATURE_COLUMNS
                ),
        })

        prediction_records.append({
            "Project":
                PROJECT_NAME,

            "NoisePercent":
                int(
                    noise_percent
                ),

            "RepetitionSeed":
                int(
                    repetition_seed
                ),

            "Technique":
                technique,

            "PredictionRows":
                len(
                    probabilities
                ),

            "MinimumProbability":
                float(
                    np.min(
                        probabilities
                    )
                ),

            "MaximumProbability":
                float(
                    np.max(
                        probabilities
                    )
                ),

            "MeanProbability":
                float(
                    np.mean(
                        probabilities
                    )
                ),

            "UniqueProbabilities":
                int(
                    pd.Series(
                        probabilities
                    ).nunique()
                ),
        })

    random_rankings = (
        get_random_rankings_for_seed(
            repetition_seed
        )
    )

    (
        latest_fail_rankings,
        qtf_rankings_condition,
    ) = create_history_baseline_rankings(
        noisy_training_history
    )

    if not qtf_rankings_condition.equals(
        QTF_RANKINGS_SHARED
    ):
        raise AssertionError(
            "QTF-Avg changed across noise or seed."
        )

    ranking_frames.extend([
        random_rankings,
        latest_fail_rankings,
        QTF_RANKINGS_SHARED.copy(
            deep=True
        ),
    ])

    rankings = pd.concat(
        ranking_frames,
        ignore_index=True,
    )

    rankings.insert(
        0,
        "Project",
        PROJECT_NAME,
    )

    rankings.insert(
        1,
        "NoisePercent",
        int(
            noise_percent
        ),
    )

    rankings.insert(
        2,
        "RepetitionSeed",
        int(
            repetition_seed
        ),
    )

    build_metrics = calculate_build_metrics(
        rankings,
        int(
            noise_percent
        ),
        int(
            repetition_seed
        ),
    )

    project_run_metrics = (
        build_metrics
        .groupby(
            [
                "Project",
                "NoisePercent",
                "RepetitionSeed",
                "Technique",
            ],
            as_index=False,
        )
        .agg(
            MeanAPFD=(
                "APFD",
                "mean",
            ),

            MeanAPFDc=(
                "APFDc",
                "mean",
            ),

            SD_APFD=(
                "APFD",
                "std",
            ),

            SD_APFDc=(
                "APFDc",
                "std",
            ),

            EvaluatedBuilds=(
                "Build",
                "nunique",
            ),

            TotalRankedTests=(
                "NumberOfTests",
                "sum",
            ),

            TotalFailures=(
                "NumberOfFailures",
                "sum",
            ),
        )
    )

    fit_times = pd.DataFrame(
        fit_records
    )

    prediction_summary = pd.DataFrame(
        prediction_records
    )

    validate_condition_outputs(
        noise_percent,
        repetition_seed,
        condition,
        rankings,
        build_metrics,
        project_run_metrics,
        fit_times,
        prediction_summary,
    )

    retained_label_changes = int(
        (
            noisy_training_data[
                "Verdict"
            ].to_numpy()
            !=
            CLEAN_MODEL_TRAINING_DATA[
                "Verdict"
            ].to_numpy()
        ).sum()
    )

    return {
        "Condition":
            condition,

        "Rankings":
            rankings,

        "BuildMetrics":
            build_metrics,

        "ProjectRunMetrics":
            project_run_metrics,

        "FitTimes":
            fit_times,

        "PredictionSummary":
            prediction_summary,

        "RetainedLabelChanges":
            retained_label_changes,

        "ConditionSeconds":
            float(
                time.time()
                - condition_started
            ),
    }


# ---------------------------------------------------------
# 8. Initialise the resumable 270-condition run
# ---------------------------------------------------------
condition_grid = [
    (
        int(
            noise
        ),
        int(
            seed
        ),
    )
    for seed in REPETITION_SEEDS
    for noise in NOISE_LEVELS
]

if len(
    condition_grid
) != EXPECTED_CONDITIONS:
    raise AssertionError(
        "The frozen condition grid does not contain "
        "270 conditions."
    )

completed_before = 0

for noise, seed in condition_grid:
    paths = get_condition_paths(
        noise,
        seed,
    )

    cleanup_condition_temporaries(
        paths[
            "Directory"
        ]
    )

    if condition_is_complete(
        paths
    ):
        completed_before += 1

run_started_at = (
    pd.Timestamp.utcnow().isoformat()
)

run_started_clock = time.time()

update_project_checkpoint({
    "Status":
        "FULL_EXPERIMENT_RUNNING",

    "FullExperimentConditionsExpected":
        EXPECTED_CONDITIONS,

    "FullExperimentConditionsCompleteBeforeRun":
        completed_before,

    "FullExperimentRawDirectory":
        str(
            FULL_RAW_DIRECTORY
        ),

    "FullExperimentStartedAtUTC":
        run_started_at,

    "UpdatedAtUTC":
        run_started_at,
})

progress_records = []
completed_count = int(
    completed_before
)


# ---------------------------------------------------------
# 9. Execute or resume all conditions
# ---------------------------------------------------------
for condition_order, (
    noise_percent,
    repetition_seed,
) in enumerate(
    condition_grid,
    start=1,
):
    paths = get_condition_paths(
        noise_percent,
        repetition_seed,
    )

    paths[
        "Directory"
    ].mkdir(
        parents=True,
        exist_ok=True,
    )

    cleanup_condition_temporaries(
        paths[
            "Directory"
        ]
    )

    if condition_is_complete(
        paths
    ):
        with open(
            paths[
                "SuccessMarker"
            ],
            "r",
            encoding="utf-8",
        ) as marker_file:
            marker = json.load(
                marker_file
            )

        progress_records.append({
            "ConditionOrder":
                condition_order,

            "ConditionKey":
                paths[
                    "Key"
                ],

            "NoisePercent":
                noise_percent,

            "RepetitionSeed":
                repetition_seed,

            "Status":
                "SKIPPED_ALREADY_COMPLETE",

            "RawRowsFlipped":
                int(
                    marker.get(
                        "RawRowsFlipped",
                        0,
                    )
                ),

            "RetainedLabelChanges":
                int(
                    marker.get(
                        "RetainedLabelChanges",
                        0,
                    )
                ),

            "ConditionSeconds":
                float(
                    marker.get(
                        "ConditionSeconds",
                        0.0,
                    )
                ),

            "CompletedAtUTC":
                str(
                    marker.get(
                        "CompletedAtUTC",
                        "",
                    )
                ),
        })

        atomic_write_csv(
            pd.DataFrame(
                progress_records
            ),
            FULL_PROGRESS_PATH,
        )

        print(
            f"[{condition_order:03d}/"
            f"{EXPECTED_CONDITIONS}] "
            f"{paths['Key']} — already complete"
        )

        continue

    condition_started_at = (
        pd.Timestamp.utcnow().isoformat()
    )

    print(
        f"[{condition_order:03d}/"
        f"{EXPECTED_CONDITIONS}] "
        f"Running {paths['Key']} ...",
        flush=True,
    )

    try:
        result = run_full_condition(
            noise_percent,
            repetition_seed,
        )

        condition = result[
            "Condition"
        ]

        rankings = result[
            "Rankings"
        ]

        build_metrics = result[
            "BuildMetrics"
        ]

        project_run_metrics = result[
            "ProjectRunMetrics"
        ]

        fit_times = result[
            "FitTimes"
        ]

        prediction_summary = result[
            "PredictionSummary"
        ]

        retained_label_changes = int(
            result[
                "RetainedLabelChanges"
            ]
        )

        condition_seconds = float(
            result[
                "ConditionSeconds"
            ]
        )

        raw_rows_flipped = int(
            condition[
                "NoiseSummary"
            ][
                "NumberFlipped"
            ]
        )

        completed_at = (
            pd.Timestamp.utcnow().isoformat()
        )

        condition_report = {
            "ProjectNumber":
                PROJECT_NUMBER,

            "Project":
                PROJECT_NAME,

            "ProjectSlug":
                PROJECT_SLUG,

            "ConditionKey":
                paths[
                    "Key"
                ],

            "NoisePercent":
                noise_percent,

            "RepetitionSeed":
                repetition_seed,

            "Status":
                "PASS",

            "NoiseSummary":
                dict(
                    condition[
                        "NoiseSummary"
                    ]
                ),

            "RetainedLabelChanges":
                retained_label_changes,

            "TrainingFailures":
                int(
                    fit_times[
                        "TrainingFailures"
                    ].iloc[0]
                ),

            "TrainingPasses":
                int(
                    fit_times[
                        "TrainingPasses"
                    ].iloc[0]
                ),

            "OutputCounts": {
                "RankingRows":
                    len(
                        rankings
                    ),

                "BuildMetricRows":
                    len(
                        build_metrics
                    ),

                "ProjectRunRows":
                    len(
                        project_run_metrics
                    ),

                "MLFitRows":
                    len(
                        fit_times
                    ),

                "PredictionSummaryRows":
                    len(
                        prediction_summary
                    ),

                "ManifestRows":
                    len(
                        condition[
                            "NoiseManifest"
                        ]
                    ),
            },

            "BaselineInvariants": {
                "RandomReusedForSameSeedAcrossNoise":
                    True,

                "QTFAvgNoiseIndependent":
                    True,

                "LatestFailUsesSameNoisyHistoryAsML":
                    True,
            },

            "EvaluationVerdictsClean":
                True,

            "ConditionSeconds":
                condition_seconds,

            "StartedAtUTC":
                condition_started_at,

            "CompletedAtUTC":
                completed_at,
        }

        atomic_write_parquet(
            rankings,
            paths[
                "Rankings"
            ],
        )

        atomic_write_parquet(
            build_metrics,
            paths[
                "BuildMetrics"
            ],
        )

        atomic_write_csv(
            project_run_metrics,
            paths[
                "ProjectRunMetrics"
            ],
        )

        atomic_write_csv(
            fit_times,
            paths[
                "FitTimes"
            ],
        )

        atomic_write_csv(
            prediction_summary,
            paths[
                "PredictionSummary"
            ],
        )

        atomic_write_parquet(
            condition[
                "NoiseManifest"
            ],
            paths[
                "NoiseManifest"
            ],
        )

        atomic_write_json(
            condition_report,
            paths[
                "ConditionReport"
            ],
        )

        success_marker = {
            "Status":
                "COMPLETE",

            "ConditionKey":
                paths[
                    "Key"
                ],

            "NoisePercent":
                noise_percent,

            "RepetitionSeed":
                repetition_seed,

            "RawRowsFlipped":
                raw_rows_flipped,

            "RetainedLabelChanges":
                retained_label_changes,

            "RankingRows":
                len(
                    rankings
                ),

            "BuildMetricRows":
                len(
                    build_metrics
                ),

            "ProjectRunRows":
                len(
                    project_run_metrics
                ),

            "MLFitRows":
                len(
                    fit_times
                ),

            "PredictionSummaryRows":
                len(
                    prediction_summary
                ),

            "ManifestRows":
                len(
                    condition[
                        "NoiseManifest"
                    ]
                ),

            "ConditionSeconds":
                condition_seconds,

            "CompletedAtUTC":
                completed_at,
        }

        atomic_write_json(
            success_marker,
            paths[
                "SuccessMarker"
            ],
        )

        if not condition_is_complete(
            paths
        ):
            raise AssertionError(
                "The saved condition did not pass "
                "its completion-marker check."
            )

        progress_records.append({
            "ConditionOrder":
                condition_order,

            "ConditionKey":
                paths[
                    "Key"
                ],

            "NoisePercent":
                noise_percent,

            "RepetitionSeed":
                repetition_seed,

            "Status":
                "COMPLETED",

            "RawRowsFlipped":
                raw_rows_flipped,

            "RetainedLabelChanges":
                retained_label_changes,

            "ConditionSeconds":
                condition_seconds,

            "CompletedAtUTC":
                completed_at,
        })

        atomic_write_csv(
            pd.DataFrame(
                progress_records
            ),
            FULL_PROGRESS_PATH,
        )

        completed_count += 1

        update_project_checkpoint({
            "Status":
                "FULL_EXPERIMENT_RUNNING",

            "FullExperimentConditionsCompleted":
                completed_count,

            "FullExperimentLastCompletedCondition":
                paths[
                    "Key"
                ],

            "FullExperimentLastConditionSeconds":
                condition_seconds,

            "UpdatedAtUTC":
                completed_at,
        })

        print(
            f"[{condition_order:03d}/"
            f"{EXPECTED_CONDITIONS}] "
            f"Completed {paths['Key']} in "
            f"{condition_seconds:.1f}s — "
            f"raw flips {raw_rows_flipped}, "
            f"retained label changes "
            f"{retained_label_changes}",
            flush=True,
        )

        del result
        del condition
        del rankings
        del build_metrics
        del project_run_metrics
        del fit_times
        del prediction_summary

        gc.collect()

    except Exception as condition_error:
        failed_at = (
            pd.Timestamp.utcnow().isoformat()
        )

        progress_records.append({
            "ConditionOrder":
                condition_order,

            "ConditionKey":
                paths[
                    "Key"
                ],

            "NoisePercent":
                noise_percent,

            "RepetitionSeed":
                repetition_seed,

            "Status":
                "FAILED",

            "RawRowsFlipped":
                np.nan,

            "RetainedLabelChanges":
                np.nan,

            "ConditionSeconds":
                float(
                    time.time()
                    - run_started_clock
                ),

            "CompletedAtUTC":
                failed_at,

            "Error":
                repr(
                    condition_error
                ),
        })

        atomic_write_csv(
            pd.DataFrame(
                progress_records
            ),
            FULL_PROGRESS_PATH,
        )

        update_project_checkpoint({
            "Status":
                "FULL_EXPERIMENT_RUNNING",

            "FullExperimentLastFailedCondition":
                paths[
                    "Key"
                ],

            "FullExperimentLastError":
                repr(
                    condition_error
                ),

            "UpdatedAtUTC":
                failed_at,
        })

        raise


# ---------------------------------------------------------
# 10. Final raw-run validation
# ---------------------------------------------------------
completed_records = []

for noise_percent, repetition_seed in (
    condition_grid
):
    paths = get_condition_paths(
        noise_percent,
        repetition_seed,
    )

    if not condition_is_complete(
        paths
    ):
        raise AssertionError(
            "Incomplete condition after run: "
            f"{paths['Key']}"
        )

    with open(
        paths[
            "SuccessMarker"
        ],
        "r",
        encoding="utf-8",
    ) as marker_file:
        marker = json.load(
            marker_file
        )

    completed_records.append({
        "ConditionKey":
            paths[
                "Key"
            ],

        "NoisePercent":
            noise_percent,

        "RepetitionSeed":
            repetition_seed,

        "RawRowsFlipped":
            int(
                marker[
                    "RawRowsFlipped"
                ]
            ),

        "RetainedLabelChanges":
            int(
                marker[
                    "RetainedLabelChanges"
                ]
            ),

        "ConditionSeconds":
            float(
                marker[
                    "ConditionSeconds"
                ]
            ),

        "RankingRows":
            int(
                marker[
                    "RankingRows"
                ]
            ),

        "BuildMetricRows":
            int(
                marker[
                    "BuildMetricRows"
                ]
            ),

        "ProjectRunRows":
            int(
                marker[
                    "ProjectRunRows"
                ]
            ),

        "MLFitRows":
            int(
                marker[
                    "MLFitRows"
                ]
            ),

        "ManifestRows":
            int(
                marker[
                    "ManifestRows"
                ]
            ),
    })

completed_conditions = pd.DataFrame(
    completed_records
)

if len(
    completed_conditions
) != EXPECTED_CONDITIONS:
    raise AssertionError(
        "The final condition registry does not "
        "contain 270 conditions."
    )

if completed_conditions[
    [
        "NoisePercent",
        "RepetitionSeed",
    ]
].duplicated().any():
    raise AssertionError(
        "The final condition registry contains "
        "duplicate conditions."
    )

total_checks = {
    "ranking rows": (
        completed_conditions[
            "RankingRows"
        ].sum(),
        EXPECTED_TOTAL_RANKING_ROWS,
    ),

    "build-metric rows": (
        completed_conditions[
            "BuildMetricRows"
        ].sum(),
        EXPECTED_TOTAL_BUILD_METRIC_ROWS,
    ),

    "project-run rows": (
        completed_conditions[
            "ProjectRunRows"
        ].sum(),
        EXPECTED_TOTAL_PROJECT_RUN_ROWS,
    ),

    "ML fits": (
        completed_conditions[
            "MLFitRows"
        ].sum(),
        EXPECTED_TOTAL_ML_FITS,
    ),

    "manifest rows": (
        completed_conditions[
            "ManifestRows"
        ].sum(),
        EXPECTED_TOTAL_MANIFEST_ROWS,
    ),
}

for label, (
    observed,
    expected,
) in total_checks.items():
    if int(observed) != int(expected):
        raise AssertionError(
            f"Unexpected total {label}: "
            f"{observed} != {expected}."
        )

raw_files = [
    path
    for path in CONDITIONS_DIRECTORY.rglob(
        "*"
    )
    if path.is_file()
]

raw_file_count = len(
    raw_files
)

if raw_file_count != (
    EXPECTED_TOTAL_RAW_FILES
):
    temporary_files = [
        str(
            path.relative_to(
                CONDITIONS_DIRECTORY
            )
        )
        for path in raw_files
        if ".tmp" in path.name
    ]

    raise AssertionError(
        "Unexpected raw file count: "
        f"expected {EXPECTED_TOTAL_RAW_FILES}, "
        f"observed {raw_file_count}. "
        f"Temporary files: "
        f"{temporary_files[:20]}"
    )

raw_total_bytes = sum(
    path.stat().st_size
    for path in raw_files
)

completed_at = (
    pd.Timestamp.utcnow().isoformat()
)

wall_clock_seconds = float(
    time.time()
    - run_started_clock
)

final_run_summary = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "PASS",

    "ExperimentGrid": {
        "NoiseLevels":
            list(
                NOISE_LEVELS
            ),

        "RepetitionSeeds":
            list(
                REPETITION_SEEDS
            ),

        "Conditions":
            EXPECTED_CONDITIONS,
    },

    "CompletedBeforeThisInvocation":
        completed_before,

    "OutputTotals": {
        "MLFits":
            EXPECTED_TOTAL_ML_FITS,

        "RankingRows":
            EXPECTED_TOTAL_RANKING_ROWS,

        "BuildMetricRows":
            EXPECTED_TOTAL_BUILD_METRIC_ROWS,

        "ProjectRunRows":
            EXPECTED_TOTAL_PROJECT_RUN_ROWS,

        "ManifestRows":
            EXPECTED_TOTAL_MANIFEST_ROWS,

        "RawFiles":
            raw_file_count,

        "RawBytes":
            raw_total_bytes,
    },

    "PerCondition": {
        "Techniques":
            len(
                ALL_TECHNIQUES
            ),

        "MLFits":
            EXPECTED_FIT_ROWS_PER_CONDITION,

        "EvaluatedBuilds":
            EXPECTED_EVALUATED_BUILDS,

        "RankingRows":
            EXPECTED_RANKING_ROWS_PER_CONDITION,

        "BuildMetricRows":
            EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,

        "ProjectRunRows":
            EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,

        "ManifestRows":
            EXPECTED_MANIFEST_ROWS_PER_CONDITION,

        "Files":
            EXPECTED_FILES_PER_CONDITION,
    },

    "RawDirectory":
        str(
            FULL_RAW_DIRECTORY
        ),

    "ConditionsDirectory":
        str(
            CONDITIONS_DIRECTORY
        ),

    "ProgressLog":
        str(
            FULL_PROGRESS_PATH
        ),

    "WallClockSecondsForThisInvocation":
        wall_clock_seconds,

    "CompletedAtUTC":
        completed_at,
}

atomic_write_json(
    final_run_summary,
    FULL_RUN_REPORT_PATH,
)

update_project_checkpoint({
    "Status":
        "FULL_EXPERIMENT_RUN_PASSED",

    "FullExperimentConditionsCompleted":
        EXPECTED_CONDITIONS,

    "FullExperimentMLFits":
        EXPECTED_TOTAL_ML_FITS,

    "FullExperimentRankingRows":
        EXPECTED_TOTAL_RANKING_ROWS,

    "FullExperimentBuildMetricRows":
        EXPECTED_TOTAL_BUILD_METRIC_ROWS,

    "FullExperimentProjectRunRows":
        EXPECTED_TOTAL_PROJECT_RUN_ROWS,

    "FullExperimentManifestRows":
        EXPECTED_TOTAL_MANIFEST_ROWS,

    "FullExperimentRawFileCount":
        raw_file_count,

    "FullExperimentRawBytes":
        raw_total_bytes,

    "FullExperimentRawDirectory":
        str(
            FULL_RAW_DIRECTORY
        ),

    "FullExperimentProgressLog":
        str(
            FULL_PROGRESS_PATH
        ),

    "FullExperimentRunReport":
        str(
            FULL_RUN_REPORT_PATH
        ),

    "FullExperimentCompletedAtUTC":
        completed_at,

    "UpdatedAtUTC":
        completed_at,
})

with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint_verification = json.load(
        checkpoint_file
    )

if checkpoint_verification.get(
    "Status"
) != "FULL_EXPERIMENT_RUN_PASSED":
    raise AssertionError(
        "The full-experiment checkpoint was not "
        "written correctly."
    )


# ---------------------------------------------------------
# 11. Compact final output
# ---------------------------------------------------------
clear_output(
    wait=True
)

print(
    "=== PROJECT 7 STEP 9 RESULT ==="
)

print(
    "\nExperiment grid:"
)

print(
    "Noise levels:",
    list(
        NOISE_LEVELS
    )
)

print(
    "Repetition seeds: 1–30"
)

print(
    "Conditions completed:",
    len(
        completed_conditions
    )
)

print(
    "Conditions already complete before this run:",
    completed_before
)

print(
    "\nActual ML execution:"
)

print(
    "ML models per condition:",
    len(
        ML_TECHNIQUES
    )
)

print(
    "Total ML fits:",
    int(
        completed_conditions[
            "MLFitRows"
        ].sum()
    )
)

print(
    "Techniques per condition:",
    len(
        ALL_TECHNIQUES
    )
)

print(
    "Evaluated builds per technique:",
    EXPECTED_EVALUATED_BUILDS
)

print(
    "\nSaved raw outputs:"
)

print(
    "Ranking rows:",
    int(
        completed_conditions[
            "RankingRows"
        ].sum()
    )
)

print(
    "Build-metric rows:",
    int(
        completed_conditions[
            "BuildMetricRows"
        ].sum()
    )
)

print(
    "Project-run rows:",
    int(
        completed_conditions[
            "ProjectRunRows"
        ].sum()
    )
)

print(
    "Noise-manifest rows:",
    int(
        completed_conditions[
            "ManifestRows"
        ].sum()
    )
)

print(
    "Raw files:",
    raw_file_count
)

print(
    "Raw bytes:",
    raw_total_bytes
)

print(
    "\nCondition timing summary:"
)

display(
    completed_conditions[
        "ConditionSeconds"
    ]
    .describe()
    .to_frame(
        name="Seconds"
    )
)

print(
    "Wall-clock seconds for this invocation:",
    wall_clock_seconds
)

print(
    "\nRaw results directory:"
)

print(
    FULL_RAW_DIRECTORY
)

print(
    "\nProgress log:"
)

print(
    FULL_PROGRESS_PATH
)

print(
    "\nFull-run report:"
)

print(
    FULL_RUN_REPORT_PATH
)

print(
    "\nValidation status: PASS"
)

print(
    "\nSUCCESS: All 270 project-noise-seed "
    "conditions completed."
)

print(
    "SUCCESS: Random Forest, XGBoost, LightGBM and "
    "Naive Bayes were fitted once per condition."
)

print(
    "SUCCESS: Random, LatestFail and QTF-Avg were "
    "evaluated for every condition."
)

print(
    "SUCCESS: Rankings, APFD/APFDc metrics, fit data "
    "and noise manifests were saved condition by "
    "condition."
)

print(
    "SUCCESS: The runner is resumable and skipped any "
    "condition already marked complete."
)

print(
    "SUCCESS: Project 7 is ready for the complete raw-"
    "output audit and final aggregation package."
)

=== PROJECT 7 STEP 9 RESULT ===

Experiment grid:
Noise levels: [0, 5, 10, 15, 20, 25, 30, 40, 50]
Repetition seeds: 1–30
Conditions completed: 270
Conditions already complete before this run: 0

Actual ML execution:
ML models per condition: 4
Total ML fits: 1080
Techniques per condition: 7
Evaluated builds per technique: 52

Saved raw outputs:
Ranking rows: 6951420
Build-metric rows: 98280
Project-run rows: 1890
Noise-manifest rows: 5336820
Raw files: 2160
Raw bytes: 159516190

Condition timing summary:


,Seconds
count,270.000000
mean,13.529812
std,2.242078
min,8.807225
25%,12.055844
50%,13.831123
75%,14.711044
max,26.586345


Wall-clock seconds for this invocation: 3807.6580007076263

Raw results directory:
/content/drive/MyDrive/Thesis_Experiment/Results/Raw/CompEvol__beast2/beast2_30_seed_raw

Progress log:
/content/drive/MyDrive/Thesis_Experiment/Results/Logs/beast2_full_experiment_progress.csv

Full-run report:
/content/drive/MyDrive/Thesis_Experiment/Results/Logs/beast2_full_experiment_run_report.json

Validation status: PASS

SUCCESS: All 270 project-noise-seed conditions completed.
SUCCESS: Random Forest, XGBoost, LightGBM and Naive Bayes were fitted once per condition.
SUCCESS: Random, LatestFail and QTF-Avg were evaluated for every condition.
SUCCESS: Rankings, APFD/APFDc metrics, fit data and noise manifests were saved condition by condition.
SUCCESS: The runner is resumable and skipped any condition already marked complete.
SUCCESS: Project 7 is ready for the complete raw-output audit and final aggregation package.


In [ ]:
# =========================================================
# PROJECT 7 — STEP 10
# COMPLETE RAW-OUTPUT AUDIT AND SHA-256 INVENTORY
#
# This cell does NOT rerun any ML model.
#
# It audits all:
#   - 270 noise-seed conditions
#   - 2,160 raw files
#   - 6,951,420 ranking rows
#   - 98,280 build-metric rows
#   - 1,890 project-run rows
#   - 5,336,820 noise-manifest rows
#
# It also validates:
#   - ranking coverage and tie policy
#   - APFD/APFDc recalculation
#   - project-run aggregation
#   - nested masks for every seed
#   - Random constant across noise per seed
#   - QTF-Avg constant across all conditions
#   - deterministic 0% NaiveBayes and LatestFail
#   - evaluation outcomes remain clean
#   - SHA-256 of every raw file
# =========================================================

from pathlib import Path
import gc
import hashlib
import json
import os
import time

import numpy as np
import pandas as pd
from IPython.display import clear_output, display


print(
    "=== PROJECT 7 STEP 10: "
    "COMPLETE RAW-OUTPUT AUDIT ==="
)


# ---------------------------------------------------------
# 1. Validate the completed Step 9 runtime state
# ---------------------------------------------------------

required_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",

    "PROJECT_7_SELECTION_CHECKPOINT",

    "RAW_RESULTS_DRIVE",
    "AGGREGATED_RESULTS_DRIVE",

    "NOISE_LEVELS",
    "REPETITION_SEEDS",

    "ML_TECHNIQUES",
    "BASELINE_TECHNIQUES",
    "ALL_TECHNIQUES",

    "CLEAN_MODEL_EVALUATION_DATA",
    "CLEAN_RAW_TRAINING_HISTORY",

    "calculate_build_metrics",
]


missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required Step 9 objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nRerun only the successful Project 7 "
        "Step 9 setup/runtime cells."
    )


if (
    PROJECT_NUMBER,
    PROJECT_NAME,
    PROJECT_SLUG,
) != (
    7,
    "CompEvol@beast2",
    "CompEvol__beast2",
):
    raise AssertionError(
        "Unexpected Project 7 identity."
    )


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    previous_checkpoint = json.load(
        checkpoint_file
    )


if previous_checkpoint.get(
    "Status"
) not in {
    "FULL_EXPERIMENT_RUN_PASSED",
    "RAW_OUTPUT_AUDIT_PASSED",
}:
    raise AssertionError(
        "The complete experiment has not passed.\n"
        f"Observed status: "
        f"{previous_checkpoint.get('Status')}"
    )


if int(
    previous_checkpoint.get(
        "FullExperimentConditionsCompleted",
        -1,
    )
) != 270:
    raise AssertionError(
        "The checkpoint does not confirm 270 completed "
        "conditions."
    )


if int(
    previous_checkpoint.get(
        "FullExperimentMLFits",
        -1,
    )
) != 1080:
    raise AssertionError(
        "The checkpoint does not confirm 1,080 ML fits."
    )


# ---------------------------------------------------------
# 2. Freeze raw and audit paths
# ---------------------------------------------------------

FULL_RAW_DIRECTORY = (
    RAW_RESULTS_DRIVE
    / PROJECT_SLUG
    / "beast2_30_seed_raw"
)


CONDITIONS_DIRECTORY = (
    FULL_RAW_DIRECTORY
    / "conditions"
)


RAW_AUDIT_DIRECTORY = (
    AGGREGATED_RESULTS_DRIVE
    / PROJECT_SLUG
    / "beast2_raw_audit"
)


RAW_AUDIT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


if not CONDITIONS_DIRECTORY.is_dir():
    raise FileNotFoundError(
        "The Step 9 conditions directory is missing:\n"
        f"{CONDITIONS_DIRECTORY}"
    )


RAW_FILE_INVENTORY_PATH = (
    RAW_AUDIT_DIRECTORY
    / "beast2_raw_file_inventory_sha256.csv"
)


CONDITION_AUDIT_PATH = (
    RAW_AUDIT_DIRECTORY
    / "beast2_condition_audit_summary.csv"
)


NESTEDNESS_AUDIT_PATH = (
    RAW_AUDIT_DIRECTORY
    / "beast2_all_seed_nestedness_audit.csv"
)


RANKING_INVARIANT_AUDIT_PATH = (
    RAW_AUDIT_DIRECTORY
    / "beast2_ranking_invariant_audit.csv"
)


ZERO_PERCENT_INVARIANT_PATH = (
    RAW_AUDIT_DIRECTORY
    / "beast2_zero_percent_invariant_audit.csv"
)


RAW_OUTPUT_AUDIT_REPORT_PATH = (
    RAW_AUDIT_DIRECTORY
    / "beast2_raw_output_audit_report.json"
)


# ---------------------------------------------------------
# 3. Frozen expected dimensions
# ---------------------------------------------------------

EXPECTED_NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]


EXPECTED_SEEDS = list(
    range(1, 31)
)


EXPECTED_CONDITIONS = 270
EXPECTED_TECHNIQUES = 7
EXPECTED_ML_TECHNIQUES = 4

EXPECTED_EVALUATION_ROWS = 3678
EXPECTED_EVALUATION_BUILDS = 52

EXPECTED_RANKING_ROWS_PER_CONDITION = 25746
EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = 364
EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = 7
EXPECTED_FIT_ROWS_PER_CONDITION = 4
EXPECTED_PREDICTION_ROWS_PER_CONDITION = 4
EXPECTED_MANIFEST_ROWS_PER_CONDITION = 19766
EXPECTED_FILES_PER_CONDITION = 8

EXPECTED_TOTAL_RANKING_ROWS = 6951420
EXPECTED_TOTAL_BUILD_METRIC_ROWS = 98280
EXPECTED_TOTAL_PROJECT_RUN_ROWS = 1890
EXPECTED_TOTAL_ML_FITS = 1080
EXPECTED_TOTAL_MANIFEST_ROWS = 5336820
EXPECTED_TOTAL_RAW_FILES = 2160


if list(NOISE_LEVELS) != EXPECTED_NOISE_LEVELS:
    raise AssertionError(
        "Unexpected noise-level grid."
    )


if list(REPETITION_SEEDS) != EXPECTED_SEEDS:
    raise AssertionError(
        "Unexpected repetition-seed grid."
    )


if len(
    CLEAN_MODEL_EVALUATION_DATA
) != EXPECTED_EVALUATION_ROWS:
    raise AssertionError(
        "Unexpected clean evaluation-row count."
    )


if int(
    CLEAN_MODEL_EVALUATION_DATA[
        "Build"
    ].nunique()
) != EXPECTED_EVALUATION_BUILDS:
    raise AssertionError(
        "Unexpected clean evaluation-build count."
    )


if len(
    CLEAN_RAW_TRAINING_HISTORY
) != EXPECTED_MANIFEST_ROWS_PER_CONDITION:
    raise AssertionError(
        "Unexpected raw training-history row count."
    )


# ---------------------------------------------------------
# 4. Expected condition grid and files
# ---------------------------------------------------------

def condition_key(
    noise_percent,
    repetition_seed,
):
    return (
        f"noise_{int(noise_percent):03d}_"
        f"seed_{int(repetition_seed):02d}"
    )


condition_grid = [
    (
        int(noise_percent),
        int(repetition_seed),
    )
    for repetition_seed in EXPECTED_SEEDS
    for noise_percent in EXPECTED_NOISE_LEVELS
]


expected_condition_keys = {
    condition_key(
        noise_percent,
        repetition_seed,
    )
    for noise_percent, repetition_seed
    in condition_grid
}


observed_condition_directories = {
    path.name
    for path in CONDITIONS_DIRECTORY.iterdir()
    if path.is_dir()
}


missing_condition_directories = sorted(
    expected_condition_keys
    - observed_condition_directories
)


unexpected_condition_directories = sorted(
    observed_condition_directories
    - expected_condition_keys
)


if missing_condition_directories:
    raise AssertionError(
        "Missing condition directories:\n"
        + "\n".join(
            missing_condition_directories[:30]
        )
    )


if unexpected_condition_directories:
    raise AssertionError(
        "Unexpected condition directories:\n"
        + "\n".join(
            unexpected_condition_directories[:30]
        )
    )


if len(
    observed_condition_directories
) != EXPECTED_CONDITIONS:
    raise AssertionError(
        "Expected exactly 270 condition directories."
    )


EXPECTED_FILE_NAMES = {
    "rankings.parquet",
    "build_metrics.parquet",
    "project_run_metrics.csv",
    "fit_times.csv",
    "prediction_summary.csv",
    "noise_manifest.parquet",
    "condition_report.json",
    "_SUCCESS.json",
}


# ---------------------------------------------------------
# 5. Atomic output helpers
# ---------------------------------------------------------

def atomic_write_csv(
    dataframe,
    destination,
):
    destination = Path(
        destination
    )


    temporary = destination.with_name(
        destination.stem
        + ".tmp"
        + destination.suffix
    )


    dataframe.to_csv(
        temporary,
        index=False,
    )


    os.replace(
        temporary,
        destination,
    )


def atomic_write_json(
    value,
    destination,
):
    destination = Path(
        destination
    )


    temporary = destination.with_name(
        destination.name
        + ".tmp"
    )


    with open(
        temporary,
        "w",
        encoding="utf-8",
    ) as output_file:
        json.dump(
            value,
            output_file,
            indent=2,
            default=str,
        )


    os.replace(
        temporary,
        destination,
    )


# ---------------------------------------------------------
# 6. Hashing and stable DataFrame digest helpers
# ---------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()


    with open(
        path,
        "rb",
    ) as input_file:
        while True:
            chunk = input_file.read(
                chunk_size
            )


            if not chunk:
                break


            digest.update(
                chunk
            )


    return digest.hexdigest()


def stable_frame_digest(
    dataframe,
    columns,
    sort_columns,
):
    frame = (
        dataframe[
            columns
        ]
        .copy()
        .reset_index(drop=True)
    )


    if "Build" in frame.columns:
        frame[
            "Build"
        ] = pd.to_numeric(
            frame[
                "Build"
            ],
            errors="raise",
        ).astype(np.int64)


    if "Test" in frame.columns:
        frame[
            "Test"
        ] = frame[
            "Test"
        ].astype(str)


    if "Rank" in frame.columns:
        frame[
            "Rank"
        ] = pd.to_numeric(
            frame[
                "Rank"
            ],
            errors="raise",
        ).astype(np.int64)


    frame = (
        frame
        .sort_values(
            sort_columns,
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    hashed_values = (
        pd.util.hash_pandas_object(
            frame,
            index=False,
            categorize=False,
        )
        .to_numpy(dtype=np.uint64)
    )


    return hashlib.sha256(
        hashed_values.tobytes()
    ).hexdigest()


# ---------------------------------------------------------
# 7. Reference evaluation outcome and key hashes
# ---------------------------------------------------------

reference_evaluation = (
    CLEAN_MODEL_EVALUATION_DATA[
        [
            "Build",
            "Test",
            "Verdict",
            "Duration",
            "build_order",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


reference_evaluation[
    "Build"
] = pd.to_numeric(
    reference_evaluation[
        "Build"
    ],
    errors="raise",
).astype(np.int64)


reference_evaluation[
    "Test"
] = reference_evaluation[
    "Test"
].astype(str)


reference_evaluation[
    "Verdict"
] = pd.to_numeric(
    reference_evaluation[
        "Verdict"
    ],
    errors="raise",
).astype(np.int64)


reference_evaluation[
    "Duration"
] = pd.to_numeric(
    reference_evaluation[
        "Duration"
    ],
    errors="raise",
).astype(float)


reference_key_digest = (
    stable_frame_digest(
        reference_evaluation,
        columns=[
            "Build",
            "Test",
        ],
        sort_columns=[
            "Build",
            "Test",
        ],
    )
)


reference_outcome_digest = (
    stable_frame_digest(
        reference_evaluation,
        columns=[
            "Build",
            "Test",
            "Verdict",
            "Duration",
            "build_order",
        ],
        sort_columns=[
            "Build",
            "Test",
        ],
    )
)


reference_build_sizes = (
    reference_evaluation
    .groupby(
        "Build",
        sort=True,
    )
    .size()
    .rename(
        "ExpectedTests"
    )
    .reset_index()
)


clean_failure_subtypes = sorted(
    pd.to_numeric(
        CLEAN_RAW_TRAINING_HISTORY.loc[
            CLEAN_RAW_TRAINING_HISTORY[
                "Verdict"
            ].astype(np.int64) != 0,
            "Verdict",
        ],
        errors="raise",
    )
    .astype(np.int64)
    .unique()
    .tolist()
)


# ---------------------------------------------------------
# 8. Complete-condition audit state
# ---------------------------------------------------------

condition_audit_records = []
file_inventory_records = []
nestedness_records = []
ranking_invariant_records = []
zero_percent_invariant_records = []


manifest_reference_by_seed = {}
previous_flip_mask_by_seed = {}
previous_noise_by_seed = {}


random_reference_hash_by_seed = {}
random_hashes_across_seeds = set()


qtf_reference_hash = None
qtf_unique_hashes = set()


zero_reference_hashes = {
    "NaiveBayes": None,
    "LatestFail": None,
    "QTF-Avg": None,
}


total_ranking_rows = 0
total_build_metric_rows = 0
total_project_run_rows = 0
total_fit_rows = 0
total_prediction_rows = 0
total_manifest_rows = 0


audit_started_clock = time.time()
audit_started_at = (
    pd.Timestamp.utcnow().isoformat()
)


# ---------------------------------------------------------
# 9. Audit all 270 conditions
# ---------------------------------------------------------

for condition_order, (
    noise_percent,
    repetition_seed,
) in enumerate(
    condition_grid,
    start=1,
):
    key = condition_key(
        noise_percent,
        repetition_seed,
    )


    condition_directory = (
        CONDITIONS_DIRECTORY
        / key
    )


    observed_file_names = {
        path.name
        for path in condition_directory.iterdir()
        if path.is_file()
    }


    missing_files = sorted(
        EXPECTED_FILE_NAMES
        - observed_file_names
    )


    unexpected_files = sorted(
        observed_file_names
        - EXPECTED_FILE_NAMES
    )


    if missing_files:
        raise AssertionError(
            f"{key} is missing files:\n"
            + "\n".join(
                missing_files
            )
        )


    if unexpected_files:
        raise AssertionError(
            f"{key} contains unexpected files:\n"
            + "\n".join(
                unexpected_files
            )
        )


    if len(
        observed_file_names
    ) != EXPECTED_FILES_PER_CONDITION:
        raise AssertionError(
            f"{key} does not contain exactly eight files."
        )


    rankings_path = (
        condition_directory
        / "rankings.parquet"
    )


    build_metrics_path = (
        condition_directory
        / "build_metrics.parquet"
    )


    project_run_path = (
        condition_directory
        / "project_run_metrics.csv"
    )


    fit_times_path = (
        condition_directory
        / "fit_times.csv"
    )


    prediction_summary_path = (
        condition_directory
        / "prediction_summary.csv"
    )


    manifest_path = (
        condition_directory
        / "noise_manifest.parquet"
    )


    condition_report_path = (
        condition_directory
        / "condition_report.json"
    )


    success_marker_path = (
        condition_directory
        / "_SUCCESS.json"
    )


    # -----------------------------------------------------
    # 9A. Hash all eight files
    # -----------------------------------------------------

    condition_bytes = 0


    for file_name in sorted(
        EXPECTED_FILE_NAMES
    ):
        file_path = (
            condition_directory
            / file_name
        )


        file_size = int(
            file_path.stat().st_size
        )


        file_hash = sha256_file(
            file_path
        )


        condition_bytes += file_size


        file_inventory_records.append({
            "ConditionOrder":
                int(condition_order),

            "ConditionKey":
                key,

            "NoisePercent":
                int(noise_percent),

            "RepetitionSeed":
                int(repetition_seed),

            "RelativePath":
                str(
                    file_path.relative_to(
                        FULL_RAW_DIRECTORY
                    )
                ),

            "FileName":
                file_name,

            "SizeBytes":
                file_size,

            "SHA256":
                file_hash,
        })


    # -----------------------------------------------------
    # 9B. Load marker and report
    # -----------------------------------------------------

    with open(
        success_marker_path,
        "r",
        encoding="utf-8",
    ) as marker_file:
        success_marker = json.load(
            marker_file
        )


    with open(
        condition_report_path,
        "r",
        encoding="utf-8",
    ) as report_file:
        condition_report = json.load(
            report_file
        )


    if success_marker.get(
        "Status"
    ) != "COMPLETE":
        raise AssertionError(
            f"{key} success marker is not COMPLETE."
        )


    if condition_report.get(
        "Status"
    ) != "PASS":
        raise AssertionError(
            f"{key} condition report is not PASS."
        )


    for source_name, source in [
        (
            "success marker",
            success_marker,
        ),
        (
            "condition report",
            condition_report,
        ),
    ]:
        if int(
            source.get(
                "NoisePercent",
                -1,
            )
        ) != int(
            noise_percent
        ):
            raise AssertionError(
                f"{key} {source_name} has the wrong "
                "noise level."
            )


        if int(
            source.get(
                "RepetitionSeed",
                -1,
            )
        ) != int(
            repetition_seed
        ):
            raise AssertionError(
                f"{key} {source_name} has the wrong seed."
            )


    # -----------------------------------------------------
    # 9C. Load the six tabular outputs
    # -----------------------------------------------------

    rankings = pd.read_parquet(
        rankings_path
    )


    build_metrics = pd.read_parquet(
        build_metrics_path
    )


    project_run_metrics = pd.read_csv(
        project_run_path
    )


    fit_times = pd.read_csv(
        fit_times_path
    )


    prediction_summary = pd.read_csv(
        prediction_summary_path
    )


    manifest = pd.read_parquet(
        manifest_path
    )


    total_ranking_rows += len(
        rankings
    )


    total_build_metric_rows += len(
        build_metrics
    )


    total_project_run_rows += len(
        project_run_metrics
    )


    total_fit_rows += len(
        fit_times
    )


    total_prediction_rows += len(
        prediction_summary
    )


    total_manifest_rows += len(
        manifest
    )


    # -----------------------------------------------------
    # 9D. Exact row-count checks
    # -----------------------------------------------------

    observed_counts = {
        "RankingRows":
            len(
                rankings
            ),

        "BuildMetricRows":
            len(
                build_metrics
            ),

        "ProjectRunRows":
            len(
                project_run_metrics
            ),

        "MLFitRows":
            len(
                fit_times
            ),

        "PredictionSummaryRows":
            len(
                prediction_summary
            ),

        "ManifestRows":
            len(
                manifest
            ),
    }


    expected_counts = {
        "RankingRows":
            EXPECTED_RANKING_ROWS_PER_CONDITION,

        "BuildMetricRows":
            EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,

        "ProjectRunRows":
            EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,

        "MLFitRows":
            EXPECTED_FIT_ROWS_PER_CONDITION,

        "PredictionSummaryRows":
            EXPECTED_PREDICTION_ROWS_PER_CONDITION,

        "ManifestRows":
            EXPECTED_MANIFEST_ROWS_PER_CONDITION,
    }


    for count_name, expected_count in (
        expected_counts.items()
    ):
        if int(
            observed_counts[
                count_name
            ]
        ) != int(
            expected_count
        ):
            raise AssertionError(
                f"{key} has the wrong {count_name}: "
                f"{observed_counts[count_name]} != "
                f"{expected_count}."
            )


        if int(
            success_marker.get(
                count_name,
                -1,
            )
        ) != int(
            expected_count
        ):
            raise AssertionError(
                f"{key} marker has the wrong "
                f"{count_name}."
            )


    # -----------------------------------------------------
    # 9E. Ranking structure and clean-outcome checks
    # -----------------------------------------------------

    required_ranking_columns = {
        "Project",
        "NoisePercent",
        "RepetitionSeed",
        "Technique",
        "Build",
        "Test",
        "Verdict",
        "Duration",
        "build_order",
        "Score",
        "ActualFailure",
        "Rank",
    }


    missing_ranking_columns = (
        required_ranking_columns
        - set(
            rankings.columns
        )
    )


    if missing_ranking_columns:
        raise AssertionError(
            f"{key} rankings are missing columns:\n"
            + "\n".join(
                sorted(
                    missing_ranking_columns
                )
            )
        )


    if set(
        rankings[
            "Technique"
        ].astype(str)
    ) != set(
        ALL_TECHNIQUES
    ):
        raise AssertionError(
            f"{key} rankings do not contain all seven "
            "techniques."
        )


    if not (
        pd.to_numeric(
            rankings[
                "NoisePercent"
            ],
            errors="raise",
        ).astype(float)
        == float(
            noise_percent
        )
    ).all():
        raise AssertionError(
            f"{key} rankings contain the wrong noise "
            "level."
        )


    if not (
        pd.to_numeric(
            rankings[
                "RepetitionSeed"
            ],
            errors="raise",
        ).astype(np.int64)
        == int(
            repetition_seed
        )
    ).all():
        raise AssertionError(
            f"{key} rankings contain the wrong seed."
        )


    rankings[
        "Build"
    ] = pd.to_numeric(
        rankings[
            "Build"
        ],
        errors="raise",
    ).astype(np.int64)


    rankings[
        "Test"
    ] = rankings[
        "Test"
    ].astype(str)


    rankings[
        "Rank"
    ] = pd.to_numeric(
        rankings[
            "Rank"
        ],
        errors="raise",
    ).astype(np.int64)


    rankings[
        "Score"
    ] = pd.to_numeric(
        rankings[
            "Score"
        ],
        errors="raise",
    ).astype(float)


    rankings[
        "Verdict"
    ] = pd.to_numeric(
        rankings[
            "Verdict"
        ],
        errors="raise",
    ).astype(np.int64)


    rankings[
        "Duration"
    ] = pd.to_numeric(
        rankings[
            "Duration"
        ],
        errors="raise",
    ).astype(float)


    rankings[
        "ActualFailure"
    ] = pd.to_numeric(
        rankings[
            "ActualFailure"
        ],
        errors="raise",
    ).astype(np.int64)


    if rankings[
        [
            "Technique",
            "Build",
            "Test",
        ]
    ].duplicated().any():
        raise AssertionError(
            f"{key} rankings contain duplicate "
            "Technique-Build-Test rows."
        )


    if rankings[
        "Score"
    ].isna().any():
        raise AssertionError(
            f"{key} rankings contain missing scores."
        )


    if not np.array_equal(
        rankings[
            "ActualFailure"
        ].to_numpy(dtype=np.int64),

        (
            rankings[
                "Verdict"
            ].to_numpy(dtype=np.int64)
            != 0
        ).astype(np.int64),
    ):
        raise AssertionError(
            f"{key} ActualFailure does not match Verdict."
        )


    technique_counts = (
        rankings
        .groupby(
            "Technique",
            sort=True,
        )
        .size()
    )


    if not (
        technique_counts
        == EXPECTED_EVALUATION_ROWS
    ).all():
        raise AssertionError(
            f"{key} does not contain 3,678 ranking rows "
            "per technique."
        )


    technique_build_counts = (
        rankings
        .groupby(
            "Technique",
            sort=True,
        )[
            "Build"
        ]
        .nunique()
    )


    if not (
        technique_build_counts
        == EXPECTED_EVALUATION_BUILDS
    ).all():
        raise AssertionError(
            f"{key} does not contain 52 builds per "
            "technique."
        )


    for technique in ALL_TECHNIQUES:
        technique_rows = (
            rankings[
                rankings[
                    "Technique"
                ].astype(str)
                == str(
                    technique
                )
            ]
        )


        technique_key_digest = (
            stable_frame_digest(
                technique_rows,
                columns=[
                    "Build",
                    "Test",
                ],
                sort_columns=[
                    "Build",
                    "Test",
                ],
            )
        )


        if technique_key_digest != (
            reference_key_digest
        ):
            raise AssertionError(
                f"{key} {technique} does not rank the "
                "exact clean evaluation Build-Test rows."
            )


        technique_outcome_digest = (
            stable_frame_digest(
                technique_rows,
                columns=[
                    "Build",
                    "Test",
                    "Verdict",
                    "Duration",
                    "build_order",
                ],
                sort_columns=[
                    "Build",
                    "Test",
                ],
            )
        )


        if technique_outcome_digest != (
            reference_outcome_digest
        ):
            raise AssertionError(
                f"{key} {technique} does not preserve "
                "the clean evaluation outcomes."
            )


    ordered_rankings = (
        rankings
        .sort_values(
            [
                "Technique",
                "Build",
                "Rank",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    expected_rank_sequence = (
        ordered_rankings
        .groupby(
            [
                "Technique",
                "Build",
            ],
            sort=False,
        )
        .cumcount()
        .to_numpy(dtype=np.int64)
        + 1
    )


    if not np.array_equal(
        ordered_rankings[
            "Rank"
        ].to_numpy(dtype=np.int64),
        expected_rank_sequence,
    ):
        raise AssertionError(
            f"{key} contains an invalid rank sequence."
        )


    observed_group_sizes = (
        ordered_rankings
        .groupby(
            [
                "Technique",
                "Build",
            ],
            sort=True,
        )
        .size()
        .rename(
            "ObservedTests"
        )
        .reset_index()
    )


    expected_group_sizes = (
        pd.DataFrame({
            "Technique":
                np.repeat(
                    ALL_TECHNIQUES,
                    len(
                        reference_build_sizes
                    ),
                ),

            "Build":
                np.tile(
                    reference_build_sizes[
                        "Build"
                    ].to_numpy(
                        dtype=np.int64
                    ),
                    len(
                        ALL_TECHNIQUES
                    ),
                ),

            "ExpectedTests":
                np.tile(
                    reference_build_sizes[
                        "ExpectedTests"
                    ].to_numpy(
                        dtype=np.int64
                    ),
                    len(
                        ALL_TECHNIQUES
                    ),
                ),
        })
    )


    size_comparison = (
        expected_group_sizes
        .merge(
            observed_group_sizes,
            on=[
                "Technique",
                "Build",
            ],
            how="outer",
            validate="one_to_one",
            indicator=True,
        )
    )


    if not (
        size_comparison[
            "_merge"
        ] == "both"
    ).all():
        raise AssertionError(
            f"{key} ranking group coverage is incomplete."
        )


    if not (
        size_comparison[
            "ExpectedTests"
        ].to_numpy(dtype=np.int64)
        ==
        size_comparison[
            "ObservedTests"
        ].to_numpy(dtype=np.int64)
    ).all():
        raise AssertionError(
            f"{key} ranking group sizes do not match the "
            "clean evaluation builds."
        )


    # Verify score direction and Test-ascending tie rule.
    for technique in ALL_TECHNIQUES:
        technique_ordered = (
            ordered_rankings[
                ordered_rankings[
                    "Technique"
                ].astype(str)
                == str(
                    technique
                )
            ]
            .copy()
            .reset_index(drop=True)
        )


        grouped = technique_ordered.groupby(
            "Build",
            sort=False,
        )


        previous_score = grouped[
            "Score"
        ].shift(1)


        previous_test = grouped[
            "Test"
        ].shift(1)


        has_previous = previous_score.notna()


        if technique == "QTF-Avg":
            direction_violation = (
                has_previous
                & (
                    technique_ordered[
                        "Score"
                    ]
                    <
                    previous_score
                )
            )

        else:
            direction_violation = (
                has_previous
                & (
                    technique_ordered[
                        "Score"
                    ]
                    >
                    previous_score
                )
            )


        if direction_violation.any():
            raise AssertionError(
                f"{key} {technique} violates its score "
                "ordering direction."
            )


        score_tie = (
            has_previous
            & (
                technique_ordered[
                    "Score"
                ]
                ==
                previous_score
            )
        )


        tie_violation = (
            score_tie
            & (
                technique_ordered[
                    "Test"
                ].astype(str)
                <
                previous_test.astype(str)
            )
        )


        if tie_violation.any():
            raise AssertionError(
                f"{key} {technique} violates the "
                "Test-ascending tie policy."
            )


    # -----------------------------------------------------
    # 9F. Recalculate build-level APFD and APFDc
    # -----------------------------------------------------

    recalculated_build_metrics = (
        calculate_build_metrics(
            rankings,
            int(
                noise_percent
            ),
            int(
                repetition_seed
            ),
        )
    )


    saved_build_metrics = (
        build_metrics
        .copy()
    )


    build_metric_sort_columns = [
        "Technique",
        "Build",
    ]


    recalculated_build_metrics = (
        recalculated_build_metrics
        .sort_values(
            build_metric_sort_columns,
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    saved_build_metrics = (
        saved_build_metrics
        .sort_values(
            build_metric_sort_columns,
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    exact_build_metric_columns = [
        "Technique",
        "Build",
        "BuildOrder",
        "NumberOfTests",
        "NumberOfFailures",
    ]


    for column in exact_build_metric_columns:
        if not saved_build_metrics[
            column
        ].equals(
            recalculated_build_metrics[
                column
            ]
        ):
            raise AssertionError(
                f"{key} saved build metric column "
                f"{column} does not match recalculation."
            )


    for metric in [
        "APFD",
        "APFDc",
    ]:
        if not np.allclose(
            pd.to_numeric(
                saved_build_metrics[
                    metric
                ],
                errors="raise",
            ).to_numpy(dtype=float),

            pd.to_numeric(
                recalculated_build_metrics[
                    metric
                ],
                errors="raise",
            ).to_numpy(dtype=float),

            rtol=0.0,
            atol=1e-12,
            equal_nan=True,
        ):
            raise AssertionError(
                f"{key} saved {metric} values do not "
                "match recalculation."
            )


    if saved_build_metrics[
        [
            "APFD",
            "APFDc",
        ]
    ].isna().any().any():
        raise AssertionError(
            f"{key} contains missing APFD/APFDc values."
        )


    for metric in [
        "APFD",
        "APFDc",
    ]:
        if not pd.to_numeric(
            saved_build_metrics[
                metric
            ],
            errors="raise",
        ).between(
            0.0,
            1.0,
            inclusive="both",
        ).all():
            raise AssertionError(
                f"{key} contains {metric} values outside "
                "[0, 1]."
            )


    if not (
        pd.to_numeric(
            saved_build_metrics[
                "NumberOfFailures"
            ],
            errors="raise",
        ) > 0
    ).all():
        raise AssertionError(
            f"{key} contains metrics for a build with no "
            "failing test execution."
        )


    # -----------------------------------------------------
    # 9G. Recalculate project-run aggregation
    # -----------------------------------------------------

    recalculated_project_run = (
        saved_build_metrics
        .groupby(
            [
                "Project",
                "NoisePercent",
                "RepetitionSeed",
                "Technique",
            ],
            as_index=False,
        )
        .agg(
            MeanAPFD=(
                "APFD",
                "mean",
            ),

            MeanAPFDc=(
                "APFDc",
                "mean",
            ),

            SD_APFD=(
                "APFD",
                "std",
            ),

            SD_APFDc=(
                "APFDc",
                "std",
            ),

            EvaluatedBuilds=(
                "Build",
                "nunique",
            ),

            TotalRankedTests=(
                "NumberOfTests",
                "sum",
            ),

            TotalFailures=(
                "NumberOfFailures",
                "sum",
            ),
        )
        .sort_values(
            "Technique",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    saved_project_run = (
        project_run_metrics
        .sort_values(
            "Technique",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    exact_project_run_columns = [
        "Technique",
        "EvaluatedBuilds",
        "TotalRankedTests",
        "TotalFailures",
    ]


    for column in exact_project_run_columns:
        if not saved_project_run[
            column
        ].equals(
            recalculated_project_run[
                column
            ]
        ):
            raise AssertionError(
                f"{key} saved project-run column "
                f"{column} does not match recalculation."
            )


    for metric in [
        "MeanAPFD",
        "MeanAPFDc",
        "SD_APFD",
        "SD_APFDc",
    ]:
        if not np.allclose(
            pd.to_numeric(
                saved_project_run[
                    metric
                ],
                errors="raise",
            ).to_numpy(dtype=float),

            pd.to_numeric(
                recalculated_project_run[
                    metric
                ],
                errors="raise",
            ).to_numpy(dtype=float),

            rtol=0.0,
            atol=1e-12,
            equal_nan=True,
        ):
            raise AssertionError(
                f"{key} saved project-run {metric} does "
                "not match recalculation."
            )


    if not (
        pd.to_numeric(
            saved_project_run[
                "EvaluatedBuilds"
            ],
            errors="raise",
        ).astype(np.int64)
        == EXPECTED_EVALUATION_BUILDS
    ).all():
        raise AssertionError(
            f"{key} does not aggregate all 52 "
            "evaluation builds."
        )


    # -----------------------------------------------------
    # 9H. ML fit and prediction-summary checks
    # -----------------------------------------------------

    if set(
        fit_times[
            "Technique"
        ].astype(str)
    ) != set(
        ML_TECHNIQUES
    ):
        raise AssertionError(
            f"{key} fit-times do not contain all four ML "
            "models."
        )


    if set(
        prediction_summary[
            "Technique"
        ].astype(str)
    ) != set(
        ML_TECHNIQUES
    ):
        raise AssertionError(
            f"{key} prediction summary does not contain "
            "all four ML models."
        )


    if not np.isfinite(
        pd.to_numeric(
            fit_times[
                "FitSeconds"
            ],
            errors="raise",
        ).to_numpy(dtype=float)
    ).all():
        raise AssertionError(
            f"{key} contains non-finite fit times."
        )


    if (
        pd.to_numeric(
            fit_times[
                "FitSeconds"
            ],
            errors="raise",
        ) < 0
    ).any():
        raise AssertionError(
            f"{key} contains negative fit times."
        )


    if not (
        pd.to_numeric(
            prediction_summary[
                "PredictionRows"
            ],
            errors="raise",
        ).astype(np.int64)
        == EXPECTED_EVALUATION_ROWS
    ).all():
        raise AssertionError(
            f"{key} contains the wrong prediction-row "
            "count."
        )


    if (
        pd.to_numeric(
            prediction_summary[
                "MinimumProbability"
            ],
            errors="raise",
        ) < 0.0
    ).any():
        raise AssertionError(
            f"{key} contains a probability below zero."
        )


    if (
        pd.to_numeric(
            prediction_summary[
                "MaximumProbability"
            ],
            errors="raise",
        ) > 1.0
    ).any():
        raise AssertionError(
            f"{key} contains a probability above one."
        )


    # -----------------------------------------------------
    # 9I. Complete noise-manifest validation
    # -----------------------------------------------------

    required_manifest_columns = {
        "NoiseRowID",
        "Build",
        "Test",
        "OriginalVerdict",
        "NoisyVerdict",
        "FlipUniform",
        "SampledFailureSubtype",
        "Flipped",
        "PassToFailure",
        "FailureToPass",
    }


    missing_manifest_columns = (
        required_manifest_columns
        - set(
            manifest.columns
        )
    )


    if missing_manifest_columns:
        raise AssertionError(
            f"{key} manifest is missing columns:\n"
            + "\n".join(
                sorted(
                    missing_manifest_columns
                )
            )
        )


    manifest[
        "NoiseRowID"
    ] = pd.to_numeric(
        manifest[
            "NoiseRowID"
        ],
        errors="raise",
    ).astype(np.int64)


    manifest[
        "Build"
    ] = pd.to_numeric(
        manifest[
            "Build"
        ],
        errors="raise",
    ).astype(np.int64)


    manifest[
        "Test"
    ] = manifest[
        "Test"
    ].astype(str)


    manifest[
        "OriginalVerdict"
    ] = pd.to_numeric(
        manifest[
            "OriginalVerdict"
        ],
        errors="raise",
    ).astype(np.int64)


    manifest[
        "NoisyVerdict"
    ] = pd.to_numeric(
        manifest[
            "NoisyVerdict"
        ],
        errors="raise",
    ).astype(np.int64)


    manifest[
        "SampledFailureSubtype"
    ] = pd.to_numeric(
        manifest[
            "SampledFailureSubtype"
        ],
        errors="raise",
    ).astype(np.int64)


    manifest[
        "FlipUniform"
    ] = pd.to_numeric(
        manifest[
            "FlipUniform"
        ],
        errors="raise",
    ).astype(float)


    for boolean_column in [
        "Flipped",
        "PassToFailure",
        "FailureToPass",
    ]:
        manifest[
            boolean_column
        ] = manifest[
            boolean_column
        ].astype(bool)


    if not np.array_equal(
        manifest[
            "NoiseRowID"
        ].to_numpy(dtype=np.int64),

        np.arange(
            EXPECTED_MANIFEST_ROWS_PER_CONDITION,
            dtype=np.int64,
        ),
    ):
        raise AssertionError(
            f"{key} manifest NoiseRowID is not the "
            "expected complete sequence."
        )


    if manifest[
        [
            "Build",
            "Test",
        ]
    ].duplicated().any():
        raise AssertionError(
            f"{key} manifest contains duplicate "
            "Build-Test rows."
        )


    if not manifest[
        "FlipUniform"
    ].between(
        0.0,
        1.0,
        inclusive="left",
    ).all():
        raise AssertionError(
            f"{key} manifest contains invalid uniforms."
        )


    expected_flipped = (
        manifest[
            "FlipUniform"
        ].to_numpy(dtype=float)
        <
        float(
            noise_percent
        ) / 100.0
    )


    if not np.array_equal(
        manifest[
            "Flipped"
        ].to_numpy(dtype=bool),
        expected_flipped,
    ):
        raise AssertionError(
            f"{key} manifest flip mask does not match "
            "its uniforms and noise threshold."
        )


    observed_verdict_change = (
        manifest[
            "OriginalVerdict"
        ].to_numpy(dtype=np.int64)
        !=
        manifest[
            "NoisyVerdict"
        ].to_numpy(dtype=np.int64)
    )


    if not np.array_equal(
        manifest[
            "Flipped"
        ].to_numpy(dtype=bool),
        observed_verdict_change,
    ):
        raise AssertionError(
            f"{key} manifest flipped rows do not match "
            "the changed verdict rows."
        )


    expected_pass_to_failure = (
        manifest[
            "Flipped"
        ].to_numpy(dtype=bool)
        &
        (
            manifest[
                "OriginalVerdict"
            ].to_numpy(dtype=np.int64)
            == 0
        )
    )


    expected_failure_to_pass = (
        manifest[
            "Flipped"
        ].to_numpy(dtype=bool)
        &
        (
            manifest[
                "OriginalVerdict"
            ].to_numpy(dtype=np.int64)
            != 0
        )
    )


    if not np.array_equal(
        manifest[
            "PassToFailure"
        ].to_numpy(dtype=bool),
        expected_pass_to_failure,
    ):
        raise AssertionError(
            f"{key} PassToFailure mask is incorrect."
        )


    if not np.array_equal(
        manifest[
            "FailureToPass"
        ].to_numpy(dtype=bool),
        expected_failure_to_pass,
    ):
        raise AssertionError(
            f"{key} FailureToPass mask is incorrect."
        )


    if not (
        manifest.loc[
            manifest[
                "PassToFailure"
            ],
            "NoisyVerdict",
        ].isin(
            clean_failure_subtypes
        )
    ).all():
        raise AssertionError(
            f"{key} pass-to-failure rows contain an "
            "invalid failure subtype."
        )


    if not (
        manifest.loc[
            manifest[
                "FailureToPass"
            ],
            "NoisyVerdict",
        ] == 0
    ).all():
        raise AssertionError(
            f"{key} failure-to-pass rows do not become "
            "verdict zero."
        )


    raw_rows_flipped = int(
        manifest[
            "Flipped"
        ].sum()
    )


    pass_to_failure_count = int(
        manifest[
            "PassToFailure"
        ].sum()
    )


    failure_to_pass_count = int(
        manifest[
            "FailureToPass"
        ].sum()
    )


    retained_label_changes = int(
        success_marker.get(
            "RetainedLabelChanges",
            -1,
        )
    )


    if raw_rows_flipped != int(
        success_marker.get(
            "RawRowsFlipped",
            -1,
        )
    ):
        raise AssertionError(
            f"{key} marker raw-flip count does not match "
            "the manifest."
        )


    report_noise_summary = (
        condition_report.get(
            "NoiseSummary",
            {}
        )
    )


    if raw_rows_flipped != int(
        report_noise_summary.get(
            "NumberFlipped",
            -1,
        )
    ):
        raise AssertionError(
            f"{key} report raw-flip count does not match "
            "the manifest."
        )


    if pass_to_failure_count != int(
        report_noise_summary.get(
            "PassToFailure",
            -1,
        )
    ):
        raise AssertionError(
            f"{key} pass-to-failure count does not match "
            "the condition report."
        )


    if failure_to_pass_count != int(
        report_noise_summary.get(
            "FailureToPass",
            -1,
        )
    ):
        raise AssertionError(
            f"{key} failure-to-pass count does not match "
            "the condition report."
        )


    if int(
        noise_percent
    ) == 0 and raw_rows_flipped != 0:
        raise AssertionError(
            f"{key} is a 0% condition with flipped rows."
        )


    # -----------------------------------------------------
    # 9J. Nested-mask validation for every seed
    # -----------------------------------------------------

    manifest_key_digest = (
        stable_frame_digest(
            manifest,
            columns=[
                "NoiseRowID",
                "Build",
                "Test",
            ],
            sort_columns=[
                "NoiseRowID",
            ],
        )
    )


    current_uniforms = manifest[
        "FlipUniform"
    ].to_numpy(dtype=float)


    current_subtypes = manifest[
        "SampledFailureSubtype"
    ].to_numpy(dtype=np.int64)


    current_mask = manifest[
        "Flipped"
    ].to_numpy(dtype=bool)


    if repetition_seed not in (
        manifest_reference_by_seed
    ):
        manifest_reference_by_seed[
            repetition_seed
        ] = {
            "KeyDigest":
                manifest_key_digest,

            "Uniforms":
                current_uniforms.copy(),

            "Subtypes":
                current_subtypes.copy(),
        }


        previous_flip_mask_by_seed[
            repetition_seed
        ] = current_mask.copy()


        previous_noise_by_seed[
            repetition_seed
        ] = int(
            noise_percent
        )


    else:
        reference_manifest = (
            manifest_reference_by_seed[
                repetition_seed
            ]
        )


        same_keys = bool(
            manifest_key_digest
            ==
            reference_manifest[
                "KeyDigest"
            ]
        )


        same_uniforms = bool(
            np.array_equal(
                current_uniforms,
                reference_manifest[
                    "Uniforms"
                ],
            )
        )


        same_subtypes = bool(
            np.array_equal(
                current_subtypes,
                reference_manifest[
                    "Subtypes"
                ],
            )
        )


        previous_mask = (
            previous_flip_mask_by_seed[
                repetition_seed
            ]
        )


        rows_missing_from_current = int(
            (
                previous_mask
                & ~current_mask
            ).sum()
        )


        nested_passed = bool(
            rows_missing_from_current == 0
        )


        nestedness_records.append({
            "RepetitionSeed":
                int(
                    repetition_seed
                ),

            "LowerNoisePercent":
                int(
                    previous_noise_by_seed[
                        repetition_seed
                    ]
                ),

            "UpperNoisePercent":
                int(
                    noise_percent
                ),

            "LowerFlippedRows":
                int(
                    previous_mask.sum()
                ),

            "UpperFlippedRows":
                int(
                    current_mask.sum()
                ),

            "LowerRowsMissingFromUpperMask":
                rows_missing_from_current,

            "SameManifestKeys":
                same_keys,

            "SameFlipUniforms":
                same_uniforms,

            "SameSampledFailureSubtypes":
                same_subtypes,

            "NestedMaskPassed":
                nested_passed,
        })


        if not same_keys:
            raise AssertionError(
                f"{key} manifest row keys changed across "
                "noise levels."
            )


        if not same_uniforms:
            raise AssertionError(
                f"{key} flip uniforms changed across "
                "noise levels."
            )


        if not same_subtypes:
            raise AssertionError(
                f"{key} sampled failure subtypes changed "
                "across noise levels."
            )


        if not nested_passed:
            raise AssertionError(
                f"{key} violates nested-mask policy."
            )


        previous_flip_mask_by_seed[
            repetition_seed
        ] = current_mask.copy()


        previous_noise_by_seed[
            repetition_seed
        ] = int(
            noise_percent
        )


    # -----------------------------------------------------
    # 9K. Cross-noise ranking invariants
    # -----------------------------------------------------

    technique_hashes = {}


    for technique in [
        "Random",
        "QTF-Avg",
        "LatestFail",
        "NaiveBayes",
    ]:
        technique_rows = (
            rankings[
                rankings[
                    "Technique"
                ].astype(str)
                == technique
            ]
        )


        technique_hashes[
            technique
        ] = stable_frame_digest(
            technique_rows,
            columns=[
                "Build",
                "Test",
                "Rank",
                "Score",
            ],
            sort_columns=[
                "Build",
                "Rank",
            ],
        )


    random_hash = technique_hashes[
        "Random"
    ]


    if repetition_seed not in (
        random_reference_hash_by_seed
    ):
        random_reference_hash_by_seed[
            repetition_seed
        ] = random_hash


        random_hashes_across_seeds.add(
            random_hash
        )


    random_matches_seed_reference = bool(
        random_hash
        ==
        random_reference_hash_by_seed[
            repetition_seed
        ]
    )


    if not random_matches_seed_reference:
        raise AssertionError(
            f"{key} Random ranking changed across noise "
            "for the same seed."
        )


    qtf_hash = technique_hashes[
        "QTF-Avg"
    ]


    qtf_unique_hashes.add(
        qtf_hash
    )


    if qtf_reference_hash is None:
        qtf_reference_hash = qtf_hash


    qtf_matches_global_reference = bool(
        qtf_hash
        ==
        qtf_reference_hash
    )


    if not qtf_matches_global_reference:
        raise AssertionError(
            f"{key} QTF-Avg ranking changed across "
            "conditions."
        )


    ranking_invariant_records.append({
        "ConditionKey":
            key,

        "NoisePercent":
            int(
                noise_percent
            ),

        "RepetitionSeed":
            int(
                repetition_seed
            ),

        "RandomHash":
            random_hash,

        "RandomMatchesSameSeedReference":
            random_matches_seed_reference,

        "QTFAvgHash":
            qtf_hash,

        "QTFAvgMatchesGlobalReference":
            qtf_matches_global_reference,
    })


    # Deterministic techniques must be identical across
    # all repeated 0% conditions.
    if int(
        noise_percent
    ) == 0:
        for technique in [
            "NaiveBayes",
            "LatestFail",
            "QTF-Avg",
        ]:
            current_hash = (
                technique_hashes[
                    technique
                ]
            )


            if zero_reference_hashes[
                technique
            ] is None:
                zero_reference_hashes[
                    technique
                ] = current_hash


            matches_zero_reference = bool(
                current_hash
                ==
                zero_reference_hashes[
                    technique
                ]
            )


            zero_percent_invariant_records.append({
                "RepetitionSeed":
                    int(
                        repetition_seed
                    ),

                "Technique":
                    technique,

                "RankingHash":
                    current_hash,

                "MatchesFirstZeroPercentSeed":
                    matches_zero_reference,
            })


            if not matches_zero_reference:
                raise AssertionError(
                    f"{key} {technique} changed across "
                    "repeated 0% conditions."
                )


    # -----------------------------------------------------
    # 9L. Record successful condition audit
    # -----------------------------------------------------

    condition_audit_records.append({
        "ConditionOrder":
            int(
                condition_order
            ),

        "ConditionKey":
            key,

        "NoisePercent":
            int(
                noise_percent
            ),

        "RepetitionSeed":
            int(
                repetition_seed
            ),

        "RawRowsFlipped":
            raw_rows_flipped,

        "PassToFailure":
            pass_to_failure_count,

        "FailureToPass":
            failure_to_pass_count,

        "RetainedLabelChanges":
            retained_label_changes,

        "RankingRows":
            int(
                len(
                    rankings
                )
            ),

        "BuildMetricRows":
            int(
                len(
                    build_metrics
                )
            ),

        "ProjectRunRows":
            int(
                len(
                    project_run_metrics
                )
            ),

        "MLFitRows":
            int(
                len(
                    fit_times
                )
            ),

        "PredictionSummaryRows":
            int(
                len(
                    prediction_summary
                )
            ),

        "ManifestRows":
            int(
                len(
                    manifest
                )
            ),

        "Files":
            int(
                len(
                    observed_file_names
                )
            ),

        "Bytes":
            int(
                condition_bytes
            ),

        "RankingCoveragePassed":
            True,

        "TiePolicyPassed":
            True,

        "EvaluationOutcomesClean":
            True,

        "BuildMetricsRecalculated":
            True,

        "ProjectRunAggregationRecalculated":
            True,

        "ManifestValidationPassed":
            True,

        "AuditStatus":
            "PASS",
    })


    if (
        condition_order % 10 == 0
        or condition_order
        == EXPECTED_CONDITIONS
    ):
        atomic_write_csv(
            pd.DataFrame(
                condition_audit_records
            ),
            CONDITION_AUDIT_PATH,
        )


        atomic_write_csv(
            pd.DataFrame(
                file_inventory_records
            ),
            RAW_FILE_INVENTORY_PATH,
        )


    print(
        f"[{condition_order:03d}/"
        f"{EXPECTED_CONDITIONS}] "
        f"Audited {key} — PASS",
        flush=True,
    )


    del rankings
    del build_metrics
    del project_run_metrics
    del fit_times
    del prediction_summary
    del manifest
    del recalculated_build_metrics
    del saved_build_metrics
    del recalculated_project_run
    del saved_project_run
    del ordered_rankings

    gc.collect()


# ---------------------------------------------------------
# 10. Final cross-condition validations
# ---------------------------------------------------------

condition_audit = pd.DataFrame(
    condition_audit_records
)


file_inventory = pd.DataFrame(
    file_inventory_records
)


nestedness_audit = pd.DataFrame(
    nestedness_records
)


ranking_invariant_audit = pd.DataFrame(
    ranking_invariant_records
)


zero_percent_invariant_audit = pd.DataFrame(
    zero_percent_invariant_records
)


if len(
    condition_audit
) != EXPECTED_CONDITIONS:
    raise AssertionError(
        "Condition audit does not contain 270 rows."
    )


if len(
    file_inventory
) != EXPECTED_TOTAL_RAW_FILES:
    raise AssertionError(
        "File inventory does not contain 2,160 rows."
    )


if len(
    nestedness_audit
) != (
    len(
        EXPECTED_SEEDS
    )
    * (
        len(
            EXPECTED_NOISE_LEVELS
        )
        - 1
    )
):
    raise AssertionError(
        "Nestedness audit does not contain 240 rows."
    )


if not nestedness_audit[
    [
        "SameManifestKeys",
        "SameFlipUniforms",
        "SameSampledFailureSubtypes",
        "NestedMaskPassed",
    ]
].all().all():
    raise AssertionError(
        "At least one full-grid nestedness invariant "
        "failed."
    )


if not ranking_invariant_audit[
    "RandomMatchesSameSeedReference"
].all():
    raise AssertionError(
        "Random was not constant across noise for every "
        "seed."
    )


if not ranking_invariant_audit[
    "QTFAvgMatchesGlobalReference"
].all():
    raise AssertionError(
        "QTF-Avg was not constant across all "
        "conditions."
    )


if len(
    random_hashes_across_seeds
) != 30:
    raise AssertionError(
        "Random did not produce 30 distinct seed-level "
        "rankings."
    )


if len(
    qtf_unique_hashes
) != 1:
    raise AssertionError(
        "QTF-Avg produced more than one ranking hash."
    )


if not zero_percent_invariant_audit[
    "MatchesFirstZeroPercentSeed"
].all():
    raise AssertionError(
        "A deterministic 0% technique changed across "
        "seeds."
    )


final_total_checks = {
    "RankingRows": (
        total_ranking_rows,
        EXPECTED_TOTAL_RANKING_ROWS,
    ),

    "BuildMetricRows": (
        total_build_metric_rows,
        EXPECTED_TOTAL_BUILD_METRIC_ROWS,
    ),

    "ProjectRunRows": (
        total_project_run_rows,
        EXPECTED_TOTAL_PROJECT_RUN_ROWS,
    ),

    "MLFitRows": (
        total_fit_rows,
        EXPECTED_TOTAL_ML_FITS,
    ),

    "PredictionSummaryRows": (
        total_prediction_rows,
        EXPECTED_TOTAL_ML_FITS,
    ),

    "ManifestRows": (
        total_manifest_rows,
        EXPECTED_TOTAL_MANIFEST_ROWS,
    ),

    "RawFiles": (
        len(
            file_inventory
        ),
        EXPECTED_TOTAL_RAW_FILES,
    ),
}


for name, (
    observed,
    expected,
) in final_total_checks.items():
    if int(
        observed
    ) != int(
        expected
    ):
        raise AssertionError(
            f"Final total {name} is incorrect: "
            f"{observed} != {expected}."
        )


raw_total_bytes = int(
    file_inventory[
        "SizeBytes"
    ].sum()
)


checkpoint_raw_bytes = int(
    previous_checkpoint.get(
        "FullExperimentRawBytes",
        -1,
    )
)


if raw_total_bytes != checkpoint_raw_bytes:
    raise AssertionError(
        "Audited raw byte total does not match the "
        "Step 9 checkpoint.\n"
        f"Audited: {raw_total_bytes}\n"
        f"Checkpoint: {checkpoint_raw_bytes}"
    )


# ---------------------------------------------------------
# 11. Calculate deterministic raw-root SHA-256
# ---------------------------------------------------------

file_inventory = (
    file_inventory
    .sort_values(
        "RelativePath",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


raw_root_hasher = hashlib.sha256()


for row in file_inventory.itertuples(
    index=False
):
    root_line = (
        f"{row.RelativePath}\t"
        f"{int(row.SizeBytes)}\t"
        f"{row.SHA256}\n"
    )


    raw_root_hasher.update(
        root_line.encode(
            "utf-8"
        )
    )


RAW_ROOT_SHA256 = (
    raw_root_hasher.hexdigest()
)


if file_inventory[
    "RelativePath"
].duplicated().any():
    raise AssertionError(
        "The raw file inventory contains duplicate "
        "relative paths."
    )


if file_inventory[
    "SHA256"
].str.len().ne(64).any():
    raise AssertionError(
        "At least one file SHA-256 digest is invalid."
    )


# ---------------------------------------------------------
# 12. Save permanent audit artefacts
# ---------------------------------------------------------

atomic_write_csv(
    file_inventory,
    RAW_FILE_INVENTORY_PATH,
)


atomic_write_csv(
    condition_audit,
    CONDITION_AUDIT_PATH,
)


atomic_write_csv(
    nestedness_audit,
    NESTEDNESS_AUDIT_PATH,
)


atomic_write_csv(
    ranking_invariant_audit,
    RANKING_INVARIANT_AUDIT_PATH,
)


atomic_write_csv(
    zero_percent_invariant_audit,
    ZERO_PERCENT_INVARIANT_PATH,
)


audit_completed_at = (
    pd.Timestamp.utcnow().isoformat()
)


audit_wall_clock_seconds = float(
    time.time()
    - audit_started_clock
)


raw_output_audit_report = {
    "ProjectNumber":
        int(
            PROJECT_NUMBER
        ),

    "Project":
        str(
            PROJECT_NAME
        ),

    "ProjectSlug":
        str(
            PROJECT_SLUG
        ),

    "Status":
        "PASS",

    "AuditScope": {
        "Conditions":
            EXPECTED_CONDITIONS,

        "NoiseLevels":
            EXPECTED_NOISE_LEVELS,

        "RepetitionSeeds":
            EXPECTED_SEEDS,

        "Techniques":
            list(
                ALL_TECHNIQUES
            ),

        "MLModels":
            list(
                ML_TECHNIQUES
            ),

        "Baselines":
            list(
                BASELINE_TECHNIQUES
            ),
    },

    "AuditedTotals": {
        "RankingRows":
            int(
                total_ranking_rows
            ),

        "BuildMetricRows":
            int(
                total_build_metric_rows
            ),

        "ProjectRunRows":
            int(
                total_project_run_rows
            ),

        "MLFitRows":
            int(
                total_fit_rows
            ),

        "PredictionSummaryRows":
            int(
                total_prediction_rows
            ),

        "ManifestRows":
            int(
                total_manifest_rows
            ),

        "RawFiles":
            int(
                len(
                    file_inventory
                )
            ),

        "RawBytes":
            raw_total_bytes,
    },

    "RankingValidation": {
        "AllConditionsContainSevenTechniques":
            True,

        "AllTechniquesRankExactEvaluationRows":
            True,

        "EvaluatedBuildsPerTechnique":
            EXPECTED_EVALUATION_BUILDS,

        "EvaluationRowsPerTechnique":
            EXPECTED_EVALUATION_ROWS,

        "RankSequencesValid":
            True,

        "ScoreDirectionValid":
            True,

        "TieRule":
            "Score then Test identifier ascending",

        "TieRulePassed":
            True,

        "EvaluationVerdictsDurationsAndKeysClean":
            True,
    },

    "MetricValidation": {
        "BuildMetricsRecalculatedForEveryCondition":
            True,

        "APFDExactWithinTolerance":
            True,

        "APFDcExactWithinTolerance":
            True,

        "ProjectRunAggregationsRecalculated":
            True,

        "AllMetricsWithinUnitInterval":
            True,

        "MetricUnit":
            "failure-test-execution event",
    },

    "NoiseValidation": {
        "AllManifestRowsValidated":
            True,

        "ManifestRowsPerCondition":
            EXPECTED_MANIFEST_ROWS_PER_CONDITION,

        "NestedTransitionsAudited":
            int(
                len(
                    nestedness_audit
                )
            ),

        "AllMasksNested":
            True,

        "SameUniformsWithinSeedAcrossNoise":
            True,

        "SameSampledFailureSubtypesWithinSeed":
            True,

        "ZeroPercentConditionsContainNoFlips":
            True,
    },

    "BaselineAndDeterminismValidation": {
        "RandomConstantAcrossNoiseWithinSeed":
            True,

        "DistinctRandomSeedRankings":
            int(
                len(
                    random_hashes_across_seeds
                )
            ),

        "QTFAvgConstantAcrossAllConditions":
            True,

        "QTFAvgUniqueHashes":
            int(
                len(
                    qtf_unique_hashes
                )
            ),

        "NaiveBayesConstantAcrossRepeatedZeroPercent":
            True,

        "LatestFailConstantAcrossRepeatedZeroPercent":
            True,

        "QTFAvgConstantAcrossRepeatedZeroPercent":
            True,
    },

    "Integrity": {
        "HashAlgorithm":
            "SHA-256",

        "FilesHashed":
            int(
                len(
                    file_inventory
                )
            ),

        "RawRootSHA256":
            RAW_ROOT_SHA256,

        "RawRootDigestConstruction":
            (
                "SHA256 over sorted lines of "
                "RelativePath<TAB>SizeBytes<TAB>"
                "FileSHA256<NEWLINE>"
            ),

        "RawDirectory":
            str(
                FULL_RAW_DIRECTORY
            ),
    },

    "Artefacts": {
        "RawFileInventory":
            str(
                RAW_FILE_INVENTORY_PATH
            ),

        "ConditionAuditSummary":
            str(
                CONDITION_AUDIT_PATH
            ),

        "NestednessAudit":
            str(
                NESTEDNESS_AUDIT_PATH
            ),

        "RankingInvariantAudit":
            str(
                RANKING_INVARIANT_AUDIT_PATH
            ),

        "ZeroPercentInvariantAudit":
            str(
                ZERO_PERCENT_INVARIANT_PATH
            ),
    },

    "StartedAtUTC":
        audit_started_at,

    "CompletedAtUTC":
        audit_completed_at,

    "WallClockSeconds":
        audit_wall_clock_seconds,
}


atomic_write_json(
    raw_output_audit_report,
    RAW_OUTPUT_AUDIT_REPORT_PATH,
)


# ---------------------------------------------------------
# 13. Update Project 7 checkpoint atomically
# ---------------------------------------------------------

with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    project_checkpoint = json.load(
        checkpoint_file
    )


project_checkpoint.update({
    "Status":
        "RAW_OUTPUT_AUDIT_PASSED",

    "RawAuditConditions":
        EXPECTED_CONDITIONS,

    "RawAuditRankingRows":
        int(
            total_ranking_rows
        ),

    "RawAuditBuildMetricRows":
        int(
            total_build_metric_rows
        ),

    "RawAuditProjectRunRows":
        int(
            total_project_run_rows
        ),

    "RawAuditMLFitRows":
        int(
            total_fit_rows
        ),

    "RawAuditManifestRows":
        int(
            total_manifest_rows
        ),

    "RawAuditFileCount":
        int(
            len(
                file_inventory
            )
        ),

    "RawAuditBytes":
        raw_total_bytes,

    "RawAuditSHA256Algorithm":
        "SHA-256",

    "RawAuditRootSHA256":
        RAW_ROOT_SHA256,

    "RawAuditNestedMaskTransitions":
        int(
            len(
                nestedness_audit
            )
        ),

    "RawAuditAllMasksNested":
        True,

    "RawAuditRandomConstantAcrossNoise":
        True,

    "RawAuditQTFAvgConstantAcrossConditions":
        True,

    "RawOutputAuditDirectory":
        str(
            RAW_AUDIT_DIRECTORY
        ),

    "RawOutputAuditReport":
        str(
            RAW_OUTPUT_AUDIT_REPORT_PATH
        ),

    "RawFileInventory":
        str(
            RAW_FILE_INVENTORY_PATH
        ),

    "RawOutputAuditCompletedAtUTC":
        audit_completed_at,

    "UpdatedAtUTC":
        audit_completed_at,
})


temporary_checkpoint_path = (
    PROJECT_7_SELECTION_CHECKPOINT
    .with_suffix(
        ".json.tmp"
    )
)


with open(
    temporary_checkpoint_path,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        project_checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


os.replace(
    temporary_checkpoint_path,
    PROJECT_7_SELECTION_CHECKPOINT,
)


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint_verification = json.load(
        checkpoint_file
    )


if checkpoint_verification.get(
    "Status"
) != "RAW_OUTPUT_AUDIT_PASSED":
    raise AssertionError(
        "The raw-output audit checkpoint was not written "
        "correctly."
    )


if checkpoint_verification.get(
    "RawAuditRootSHA256"
) != RAW_ROOT_SHA256:
    raise AssertionError(
        "The checkpoint did not preserve the raw-root "
        "SHA-256 digest."
    )


# ---------------------------------------------------------
# 14. Compact final output
# ---------------------------------------------------------

clear_output(
    wait=True
)


print(
    "=== PROJECT 7 STEP 10 RESULT ==="
)


print(
    "\nComplete raw-output audit:"
)

print(
    "Conditions audited:",
    len(
        condition_audit
    )
)

print(
    "Noise levels:",
    EXPECTED_NOISE_LEVELS
)

print(
    "Seeds:",
    "1–30"
)

print(
    "Techniques per condition:",
    EXPECTED_TECHNIQUES
)


print(
    "\nAudited row totals:"
)

print(
    "Ranking rows:",
    total_ranking_rows
)

print(
    "Build-metric rows:",
    total_build_metric_rows
)

print(
    "Project-run rows:",
    total_project_run_rows
)

print(
    "ML fit rows:",
    total_fit_rows
)

print(
    "Prediction-summary rows:",
    total_prediction_rows
)

print(
    "Noise-manifest rows:",
    total_manifest_rows
)


print(
    "\nRanking and metric validation:"
)

print(
    "Exact evaluation-row coverage:",
    True
)

print(
    "Valid rank sequences:",
    True
)

print(
    "Score-direction validation:",
    True
)

print(
    "Test-ascending tie policy:",
    True
)

print(
    "APFD recalculation passed:",
    True
)

print(
    "APFDc recalculation passed:",
    True
)

print(
    "Project-run aggregation recalculation passed:",
    True
)

print(
    "Evaluation outcomes clean:",
    True
)


print(
    "\nNoise validation:"
)

print(
    "Nested-mask transitions audited:",
    len(
        nestedness_audit
    )
)

print(
    "All masks nested:",
    bool(
        nestedness_audit[
            "NestedMaskPassed"
        ].all()
    )
)

print(
    "Same uniforms within every seed:",
    bool(
        nestedness_audit[
            "SameFlipUniforms"
        ].all()
    )
)

print(
    "Same sampled subtypes within every seed:",
    bool(
        nestedness_audit[
            "SameSampledFailureSubtypes"
        ].all()
    )
)


print(
    "\nBaseline and deterministic invariants:"
)

print(
    "Random constant across noise within seed:",
    bool(
        ranking_invariant_audit[
            "RandomMatchesSameSeedReference"
        ].all()
    )
)

print(
    "Distinct Random seed rankings:",
    len(
        random_hashes_across_seeds
    )
)

print(
    "QTF-Avg constant across all conditions:",
    bool(
        ranking_invariant_audit[
            "QTFAvgMatchesGlobalReference"
        ].all()
    )
)

print(
    "QTF-Avg unique ranking hashes:",
    len(
        qtf_unique_hashes
    )
)

print(
    "NaiveBayes/LatestFail/QTF 0% invariants:",
    bool(
        zero_percent_invariant_audit[
            "MatchesFirstZeroPercentSeed"
        ].all()
    )
)


print(
    "\nRaw-file integrity:"
)

print(
    "Files hashed:",
    len(
        file_inventory
    )
)

print(
    "Raw bytes:",
    raw_total_bytes
)

print(
    "Raw root SHA-256:",
    RAW_ROOT_SHA256
)


print(
    "\nAudit report:"
)

print(
    RAW_OUTPUT_AUDIT_REPORT_PATH
)


print(
    "\nValidation status:",
    "PASS"
)


print(
    "\nSUCCESS: Every Project 7 raw condition and file "
    "was independently audited."
)

print(
    "SUCCESS: All saved rankings preserve the exact "
    "clean evaluation outcomes and frozen tie policy."
)

print(
    "SUCCESS: Every APFD and APFDc value matched an "
    "independent recalculation."
)

print(
    "SUCCESS: All 30 nested noise-mask families passed."
)

print(
    "SUCCESS: Random and QTF-Avg invariants passed "
    "across the complete 270-condition grid."
)

print(
    "SUCCESS: SHA-256 hashes were recorded for all "
    "2,160 raw files."
)

print(
    "SUCCESS: Project 7 is ready for final aggregation "
    "and packaging."
)

=== PROJECT 7 STEP 10 RESULT ===

Complete raw-output audit:
Conditions audited: 270
Noise levels: [0, 5, 10, 15, 20, 25, 30, 40, 50]
Seeds: 1–30
Techniques per condition: 7

Audited row totals:
Ranking rows: 6951420
Build-metric rows: 98280
Project-run rows: 1890
ML fit rows: 1080
Prediction-summary rows: 1080
Noise-manifest rows: 5336820

Ranking and metric validation:
Exact evaluation-row coverage: True
Valid rank sequences: True
Score-direction validation: True
Test-ascending tie policy: True
APFD recalculation passed: True
APFDc recalculation passed: True
Project-run aggregation recalculation passed: True
Evaluation outcomes clean: True

Noise validation:
Nested-mask transitions audited: 240
All masks nested: True
Same uniforms within every seed: True
Same sampled subtypes within every seed: True

Baseline and deterministic invariants:
Random constant across noise within seed: True
Distinct Random seed rankings: 30
QTF-Avg constant across all conditions: True
QTF-Avg unique rankin

In [ ]:
# =========================================================
# PROJECT 7 — STEP 11A
# FINAL AGGREGATION AND 72-FILE PACKAGE CONSTRUCTION
#
# This cell does NOT:
#   - rerun noise injection
#   - refit any ML model
#   - recalculate test rankings
#   - change raw results
#   - update the completed-project registry
#
# It:
#   - combines all validated condition outputs
#   - aggregates 30 seeds within Beast2
#   - calculates clean-relative degradation/retention
#   - creates the final 72-file project package
#   - calculates and reverifies package SHA-256 hashes
# =========================================================

from pathlib import Path
import gc
import hashlib
import json
import math
import os
import re
import shutil
import time

import numpy as np
import pandas as pd
from IPython.display import clear_output, display


print(
    "=== PROJECT 7 STEP 11A: "
    "FINAL AGGREGATION AND PACKAGE CONSTRUCTION ==="
)


# ---------------------------------------------------------
# 1. Validate successful Step 10 state
# ---------------------------------------------------------

required_objects = [
    "PROJECT_NUMBER",
    "PROJECT_NAME",
    "PROJECT_SLUG",

    "PROJECT_7_SELECTION_CHECKPOINT",

    "RAW_RESULTS_DRIVE",
    "AGGREGATED_RESULTS_DRIVE",

    "NOISE_LEVELS",
    "REPETITION_SEEDS",

    "ML_TECHNIQUES",
    "BASELINE_TECHNIQUES",
    "ALL_TECHNIQUES",
]


missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required Project 7 objects are missing:\n"
        + "\n".join(missing_objects)
        + "\n\nRerun only the successful Project 7 "
        "Step 10 runtime cell."
    )


if (
    PROJECT_NUMBER,
    PROJECT_NAME,
    PROJECT_SLUG,
) != (
    7,
    "CompEvol@beast2",
    "CompEvol__beast2",
):
    raise AssertionError(
        "Unexpected Project 7 identity."
    )


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    previous_checkpoint = json.load(
        checkpoint_file
    )


allowed_previous_statuses = {
    "RAW_OUTPUT_AUDIT_PASSED",
    "FINAL_AGGREGATION_PACKAGE_PASSED",
}


if previous_checkpoint.get(
    "Status"
) not in allowed_previous_statuses:
    raise AssertionError(
        "The complete raw-output audit has not passed.\n"
        f"Observed checkpoint status: "
        f"{previous_checkpoint.get('Status')}"
    )


if int(
    previous_checkpoint.get(
        "RawAuditConditions",
        -1,
    )
) != 270:
    raise AssertionError(
        "The checkpoint does not confirm 270 audited "
        "conditions."
    )


if int(
    previous_checkpoint.get(
        "RawAuditFileCount",
        -1,
    )
) != 2160:
    raise AssertionError(
        "The checkpoint does not confirm 2,160 audited "
        "raw files."
    )


EXPECTED_RAW_ROOT_SHA256 = (
    "93a6581d90895fd011b48490aa66f55a91b0078818bfd6966b3cf5b9e0f86373"
)


if previous_checkpoint.get(
    "RawAuditRootSHA256"
) != EXPECTED_RAW_ROOT_SHA256:
    raise AssertionError(
        "The checkpoint raw-root SHA-256 does not match "
        "the successful Step 10 result."
    )


# ---------------------------------------------------------
# 2. Freeze source and package paths
# ---------------------------------------------------------

FULL_RAW_DIRECTORY = (
    RAW_RESULTS_DRIVE
    / PROJECT_SLUG
    / "beast2_30_seed_raw"
)


CONDITIONS_DIRECTORY = (
    FULL_RAW_DIRECTORY
    / "conditions"
)


PROJECT_AGGREGATED_DIRECTORY = (
    AGGREGATED_RESULTS_DRIVE
    / PROJECT_SLUG
)


FINAL_PACKAGE_DIRECTORY = (
    PROJECT_AGGREGATED_DIRECTORY
    / "beast2_30_seed_final"
)


TEMPORARY_PACKAGE_DIRECTORY = (
    PROJECT_AGGREGATED_DIRECTORY
    / "beast2_30_seed_final.tmp_build"
)


FINAL_PACKAGE_AUDIT_DIRECTORY = (
    PROJECT_AGGREGATED_DIRECTORY
    / "beast2_final_package_audit"
)


FINAL_PACKAGE_INVENTORY_PATH = (
    FINAL_PACKAGE_AUDIT_DIRECTORY
    / "beast2_final_package_file_inventory_sha256.csv"
)


FINAL_PACKAGE_VALIDATION_PATH = (
    FINAL_PACKAGE_AUDIT_DIRECTORY
    / "beast2_final_package_validation.csv"
)


FINAL_PACKAGE_REPORT_PATH = (
    FINAL_PACKAGE_AUDIT_DIRECTORY
    / "beast2_final_package_report.json"
)


PROJECT_AGGREGATED_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


FINAL_PACKAGE_AUDIT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


if not CONDITIONS_DIRECTORY.is_dir():
    raise FileNotFoundError(
        "The complete raw-condition directory is "
        "missing:\n"
        f"{CONDITIONS_DIRECTORY}"
    )


# ---------------------------------------------------------
# 3. Frozen expected experiment dimensions
# ---------------------------------------------------------

EXPECTED_NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]


EXPECTED_SEEDS = list(
    range(1, 31)
)


EXPECTED_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
    "Random",
    "LatestFail",
    "QTF-Avg",
]


EXPECTED_CONDITIONS = 270

EXPECTED_PROJECT_RUN_ROWS = 1890
EXPECTED_BUILD_METRIC_ROWS = 98280
EXPECTED_FIT_ROWS = 1080
EXPECTED_PREDICTION_ROWS = 1080
EXPECTED_CONDITION_SUMMARY_ROWS = 270

EXPECTED_PROJECT_LEVEL_ROWS = 63
EXPECTED_TECHNIQUE_NOISE_FILES = 63

EXPECTED_CORE_PACKAGE_FILES = 9
EXPECTED_FINAL_PACKAGE_FILES = 72


if list(
    NOISE_LEVELS
) != EXPECTED_NOISE_LEVELS:
    raise AssertionError(
        "Unexpected noise-level grid."
    )


if list(
    REPETITION_SEEDS
) != EXPECTED_SEEDS:
    raise AssertionError(
        "Unexpected repetition-seed grid."
    )


if set(
    ALL_TECHNIQUES
) != set(
    EXPECTED_TECHNIQUES
):
    raise AssertionError(
        "Unexpected technique set."
    )


# ---------------------------------------------------------
# 4. General helpers
# ---------------------------------------------------------

def condition_key(
    noise_percent,
    repetition_seed,
):
    return (
        f"noise_{int(noise_percent):03d}_"
        f"seed_{int(repetition_seed):02d}"
    )


def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()


    with open(
        path,
        "rb",
    ) as input_file:
        while True:
            chunk = input_file.read(
                chunk_size
            )


            if not chunk:
                break


            digest.update(
                chunk
            )


    return digest.hexdigest()


def atomic_write_csv(
    dataframe,
    destination,
):
    destination = Path(
        destination
    )


    temporary = destination.with_name(
        destination.stem
        + ".tmp"
        + destination.suffix
    )


    dataframe.to_csv(
        temporary,
        index=False,
    )


    os.replace(
        temporary,
        destination,
    )


def atomic_write_json(
    value,
    destination,
):
    destination = Path(
        destination
    )


    temporary = destination.with_name(
        destination.name
        + ".tmp"
    )


    with open(
        temporary,
        "w",
        encoding="utf-8",
    ) as output_file:
        json.dump(
            value,
            output_file,
            indent=2,
            default=str,
        )


    os.replace(
        temporary,
        destination,
    )


def technique_file_slug(
    technique,
):
    fixed = {
        "RandomForest":
            "random_forest",

        "XGBoost":
            "xgboost",

        "LightGBM":
            "lightgbm",

        "NaiveBayes":
            "naive_bayes",

        "Random":
            "random",

        "LatestFail":
            "latest_fail",

        "QTF-Avg":
            "qtf_avg",
    }


    if technique not in fixed:
        raise ValueError(
            f"Unexpected technique: {technique}"
        )


    return fixed[
        technique
    ]


def safe_percentage_loss(
    clean_values,
    noisy_values,
):
    clean_array = np.asarray(
        clean_values,
        dtype=float,
    )


    noisy_array = np.asarray(
        noisy_values,
        dtype=float,
    )


    result = np.full(
        len(
            clean_array
        ),
        np.nan,
        dtype=float,
    )


    valid = ~np.isclose(
        clean_array,
        0.0,
        rtol=0.0,
        atol=1e-15,
    )


    result[
        valid
    ] = (
        (
            clean_array[
                valid
            ]
            -
            noisy_array[
                valid
            ]
        )
        /
        clean_array[
            valid
        ]
        * 100.0
    )


    return result


def safe_retention(
    clean_values,
    noisy_values,
):
    clean_array = np.asarray(
        clean_values,
        dtype=float,
    )


    noisy_array = np.asarray(
        noisy_values,
        dtype=float,
    )


    result = np.full(
        len(
            clean_array
        ),
        np.nan,
        dtype=float,
    )


    valid = ~np.isclose(
        clean_array,
        0.0,
        rtol=0.0,
        atol=1e-15,
    )


    result[
        valid
    ] = (
        noisy_array[
            valid
        ]
        /
        clean_array[
            valid
        ]
    )


    return result


# Student-t critical value for a 95% CI with:
#   n = 30 seeds
#   df = 29
T_CRITICAL_95_DF_29 = (
    2.045229642132703
)


# ---------------------------------------------------------
# 5. Read all 270 validated conditions
# ---------------------------------------------------------

condition_grid = [
    (
        int(
            noise_percent
        ),
        int(
            repetition_seed
        ),
    )
    for repetition_seed in EXPECTED_SEEDS
    for noise_percent in EXPECTED_NOISE_LEVELS
]


if len(
    condition_grid
) != EXPECTED_CONDITIONS:
    raise AssertionError(
        "The condition grid does not contain 270 "
        "conditions."
    )


project_run_frames = []
build_metric_frames = []
fit_time_frames = []
prediction_frames = []
condition_records = []


aggregation_started_clock = time.time()
aggregation_started_at = (
    pd.Timestamp.utcnow().isoformat()
)


for condition_order, (
    noise_percent,
    repetition_seed,
) in enumerate(
    condition_grid,
    start=1,
):
    key = condition_key(
        noise_percent,
        repetition_seed,
    )


    condition_directory = (
        CONDITIONS_DIRECTORY
        / key
    )


    project_run_path = (
        condition_directory
        / "project_run_metrics.csv"
    )


    build_metrics_path = (
        condition_directory
        / "build_metrics.parquet"
    )


    fit_times_path = (
        condition_directory
        / "fit_times.csv"
    )


    prediction_path = (
        condition_directory
        / "prediction_summary.csv"
    )


    condition_report_path = (
        condition_directory
        / "condition_report.json"
    )


    success_marker_path = (
        condition_directory
        / "_SUCCESS.json"
    )


    for required_path in [
        project_run_path,
        build_metrics_path,
        fit_times_path,
        prediction_path,
        condition_report_path,
        success_marker_path,
    ]:
        if not required_path.is_file():
            raise FileNotFoundError(
                f"Missing required condition file:\n"
                f"{required_path}"
            )


    with open(
        success_marker_path,
        "r",
        encoding="utf-8",
    ) as marker_file:
        marker = json.load(
            marker_file
        )


    with open(
        condition_report_path,
        "r",
        encoding="utf-8",
    ) as report_file:
        condition_report = json.load(
            report_file
        )


    if marker.get(
        "Status"
    ) != "COMPLETE":
        raise AssertionError(
            f"{key} is not marked COMPLETE."
        )


    if condition_report.get(
        "Status"
    ) != "PASS":
        raise AssertionError(
            f"{key} report is not marked PASS."
        )


    project_run = pd.read_csv(
        project_run_path
    )


    build_metrics = pd.read_parquet(
        build_metrics_path
    )


    fit_times = pd.read_csv(
        fit_times_path
    )


    prediction_summary = pd.read_csv(
        prediction_path
    )


    if len(
        project_run
    ) != 7:
        raise AssertionError(
            f"{key} does not contain seven project-run "
            "rows."
        )


    if len(
        build_metrics
    ) != 364:
        raise AssertionError(
            f"{key} does not contain 364 build-metric "
            "rows."
        )


    if len(
        fit_times
    ) != 4:
        raise AssertionError(
            f"{key} does not contain four ML fit rows."
        )


    if len(
        prediction_summary
    ) != 4:
        raise AssertionError(
            f"{key} does not contain four prediction "
            "summary rows."
        )


    if set(
        project_run[
            "Technique"
        ].astype(str)
    ) != set(
        EXPECTED_TECHNIQUES
    ):
        raise AssertionError(
            f"{key} does not contain all seven "
            "techniques."
        )


    project_run_frames.append(
        project_run
    )


    build_metric_frames.append(
        build_metrics
    )


    fit_time_frames.append(
        fit_times
    )


    prediction_frames.append(
        prediction_summary
    )


    noise_summary = condition_report.get(
        "NoiseSummary",
        {}
    )


    condition_records.append({
        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "ConditionOrder":
            int(
                condition_order
            ),

        "ConditionKey":
            key,

        "NoisePercent":
            int(
                noise_percent
            ),

        "RepetitionSeed":
            int(
                repetition_seed
            ),

        "TrainingExecutionRows":
            int(
                noise_summary.get(
                    "TrainingExecutionRows",
                    19766,
                )
            ),

        "RawRowsFlipped":
            int(
                noise_summary.get(
                    "NumberFlipped",
                    marker.get(
                        "RawRowsFlipped",
                        -1,
                    ),
                )
            ),

        "RealisedNoisePercent":
            float(
                noise_summary.get(
                    "RealisedNoisePercent",
                    np.nan,
                )
            ),

        "PassToFailure":
            int(
                noise_summary.get(
                    "PassToFailure",
                    -1,
                )
            ),

        "FailureToPass":
            int(
                noise_summary.get(
                    "FailureToPass",
                    -1,
                )
            ),

        "RetainedLabelChanges":
            int(
                marker.get(
                    "RetainedLabelChanges",
                    condition_report.get(
                        "RetainedLabelChanges",
                        -1,
                    ),
                )
            ),

        "TrainingFailures":
            int(
                condition_report.get(
                    "TrainingFailures",
                    -1,
                )
            ),

        "TrainingPasses":
            int(
                condition_report.get(
                    "TrainingPasses",
                    -1,
                )
            ),

        "ConditionSeconds":
            float(
                marker.get(
                    "ConditionSeconds",
                    condition_report.get(
                        "ConditionSeconds",
                        np.nan,
                    ),
                )
            ),

        "Status":
            "PASS",
    })


    if (
        condition_order % 30 == 0
        or condition_order == EXPECTED_CONDITIONS
    ):
        print(
            f"Loaded and validated "
            f"{condition_order}/"
            f"{EXPECTED_CONDITIONS} conditions.",
            flush=True,
        )


# ---------------------------------------------------------
# 6. Combine all result layers
# ---------------------------------------------------------

all_project_run_metrics = pd.concat(
    project_run_frames,
    ignore_index=True,
)


all_build_metrics = pd.concat(
    build_metric_frames,
    ignore_index=True,
)


all_fit_times = pd.concat(
    fit_time_frames,
    ignore_index=True,
)


all_prediction_summary = pd.concat(
    prediction_frames,
    ignore_index=True,
)


condition_summary = pd.DataFrame(
    condition_records
)


del project_run_frames
del build_metric_frames
del fit_time_frames
del prediction_frames

gc.collect()


# ---------------------------------------------------------
# 7. Validate combined dimensions and keys
# ---------------------------------------------------------

combined_expected_counts = {
    "Project-run rows": (
        len(
            all_project_run_metrics
        ),
        EXPECTED_PROJECT_RUN_ROWS,
    ),

    "Build-metric rows": (
        len(
            all_build_metrics
        ),
        EXPECTED_BUILD_METRIC_ROWS,
    ),

    "Fit rows": (
        len(
            all_fit_times
        ),
        EXPECTED_FIT_ROWS,
    ),

    "Prediction rows": (
        len(
            all_prediction_summary
        ),
        EXPECTED_PREDICTION_ROWS,
    ),

    "Condition-summary rows": (
        len(
            condition_summary
        ),
        EXPECTED_CONDITION_SUMMARY_ROWS,
    ),
}


for label, (
    observed,
    expected,
) in combined_expected_counts.items():
    if int(
        observed
    ) != int(
        expected
    ):
        raise AssertionError(
            f"Unexpected {label}: "
            f"{observed} != {expected}."
        )


all_project_run_metrics[
    "NoisePercent"
] = pd.to_numeric(
    all_project_run_metrics[
        "NoisePercent"
    ],
    errors="raise",
).astype(np.int64)


all_project_run_metrics[
    "RepetitionSeed"
] = pd.to_numeric(
    all_project_run_metrics[
        "RepetitionSeed"
    ],
    errors="raise",
).astype(np.int64)


all_project_run_metrics[
    "Technique"
] = all_project_run_metrics[
    "Technique"
].astype(str)


for metric_column in [
    "MeanAPFD",
    "MeanAPFDc",
    "SD_APFD",
    "SD_APFDc",
]:
    all_project_run_metrics[
        metric_column
    ] = pd.to_numeric(
        all_project_run_metrics[
            metric_column
        ],
        errors="raise",
    ).astype(float)


if all_project_run_metrics[
    [
        "NoisePercent",
        "RepetitionSeed",
        "Technique",
    ]
].duplicated().any():
    raise AssertionError(
        "Combined project-run metrics contain duplicate "
        "noise-seed-technique rows."
    )


if set(
    all_project_run_metrics[
        "NoisePercent"
    ].unique()
) != set(
    EXPECTED_NOISE_LEVELS
):
    raise AssertionError(
        "Combined project-run metrics contain the wrong "
        "noise levels."
    )


if set(
    all_project_run_metrics[
        "RepetitionSeed"
    ].unique()
) != set(
    EXPECTED_SEEDS
):
    raise AssertionError(
        "Combined project-run metrics contain the wrong "
        "seeds."
    )


if set(
    all_project_run_metrics[
        "Technique"
    ].unique()
) != set(
    EXPECTED_TECHNIQUES
):
    raise AssertionError(
        "Combined project-run metrics contain the wrong "
        "techniques."
    )


rows_per_technique_noise = (
    all_project_run_metrics
    .groupby(
        [
            "Technique",
            "NoisePercent",
        ],
        sort=True,
    )
    .size()
)


if not (
    rows_per_technique_noise == 30
).all():
    raise AssertionError(
        "Every technique-noise combination must contain "
        "exactly 30 seed rows."
    )


if not (
    all_project_run_metrics[
        "EvaluatedBuilds"
    ].astype(np.int64)
    == 52
).all():
    raise AssertionError(
        "At least one project-run row does not contain "
        "52 evaluated builds."
    )


for metric_column in [
    "MeanAPFD",
    "MeanAPFDc",
]:
    if not all_project_run_metrics[
        metric_column
    ].between(
        0.0,
        1.0,
        inclusive="both",
    ).all():
        raise AssertionError(
            f"{metric_column} contains values outside "
            "[0, 1]."
        )


all_project_run_metrics = (
    all_project_run_metrics
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


all_build_metrics = (
    all_build_metrics
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "BuildOrder",
            "Build",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


all_fit_times = (
    all_fit_times
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


all_prediction_summary = (
    all_prediction_summary
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


condition_summary = (
    condition_summary
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 8. Build matched seed-level clean references
# ---------------------------------------------------------

clean_seed_reference = (
    all_project_run_metrics[
        all_project_run_metrics[
            "NoisePercent"
        ] == 0
    ][
        [
            "Project",
            "RepetitionSeed",
            "Technique",
            "MeanAPFD",
            "MeanAPFDc",
        ]
    ]
    .copy()
    .rename(
        columns={
            "MeanAPFD":
                "CleanMeanAPFD",

            "MeanAPFDc":
                "CleanMeanAPFDc",
        }
    )
)


if len(
    clean_seed_reference
) != 210:
    raise AssertionError(
        "Expected 210 matched clean seed-technique "
        "reference rows."
    )


seed_level_degradation = (
    all_project_run_metrics
    .merge(
        clean_seed_reference,
        on=[
            "Project",
            "RepetitionSeed",
            "Technique",
        ],
        how="left",
        validate="many_to_one",
    )
)


if seed_level_degradation[
    [
        "CleanMeanAPFD",
        "CleanMeanAPFDc",
    ]
].isna().any().any():
    raise AssertionError(
        "At least one seed-level row has no matched "
        "0% clean reference."
    )


seed_level_degradation[
    "DeltaAPFD_NoiseMinusClean"
] = (
    seed_level_degradation[
        "MeanAPFD"
    ]
    -
    seed_level_degradation[
        "CleanMeanAPFD"
    ]
)


seed_level_degradation[
    "DeltaAPFDc_NoiseMinusClean"
] = (
    seed_level_degradation[
        "MeanAPFDc"
    ]
    -
    seed_level_degradation[
        "CleanMeanAPFDc"
    ]
)


seed_level_degradation[
    "AbsoluteLossAPFD"
] = (
    seed_level_degradation[
        "CleanMeanAPFD"
    ]
    -
    seed_level_degradation[
        "MeanAPFD"
    ]
)


seed_level_degradation[
    "AbsoluteLossAPFDc"
] = (
    seed_level_degradation[
        "CleanMeanAPFDc"
    ]
    -
    seed_level_degradation[
        "MeanAPFDc"
    ]
)


seed_level_degradation[
    "PercentageLossAPFD"
] = safe_percentage_loss(
    seed_level_degradation[
        "CleanMeanAPFD"
    ],
    seed_level_degradation[
        "MeanAPFD"
    ],
)


seed_level_degradation[
    "PercentageLossAPFDc"
] = safe_percentage_loss(
    seed_level_degradation[
        "CleanMeanAPFDc"
    ],
    seed_level_degradation[
        "MeanAPFDc"
    ],
)


seed_level_degradation[
    "RetentionAPFD"
] = safe_retention(
    seed_level_degradation[
        "CleanMeanAPFD"
    ],
    seed_level_degradation[
        "MeanAPFD"
    ],
)


seed_level_degradation[
    "RetentionAPFDc"
] = safe_retention(
    seed_level_degradation[
        "CleanMeanAPFDc"
    ],
    seed_level_degradation[
        "MeanAPFDc"
    ],
)


zero_seed_rows = seed_level_degradation[
    seed_level_degradation[
        "NoisePercent"
    ] == 0
]


for zero_column in [
    "DeltaAPFD_NoiseMinusClean",
    "DeltaAPFDc_NoiseMinusClean",
    "AbsoluteLossAPFD",
    "AbsoluteLossAPFDc",
    "PercentageLossAPFD",
    "PercentageLossAPFDc",
]:
    if not np.allclose(
        zero_seed_rows[
            zero_column
        ].to_numpy(dtype=float),
        0.0,
        rtol=0.0,
        atol=1e-12,
        equal_nan=False,
    ):
        raise AssertionError(
            f"0% matched-seed values are not zero for "
            f"{zero_column}."
        )


for retention_column in [
    "RetentionAPFD",
    "RetentionAPFDc",
]:
    if not np.allclose(
        zero_seed_rows[
            retention_column
        ].to_numpy(dtype=float),
        1.0,
        rtol=0.0,
        atol=1e-12,
        equal_nan=False,
    ):
        raise AssertionError(
            f"0% matched-seed retention is not one for "
            f"{retention_column}."
        )


# ---------------------------------------------------------
# 9. Aggregate 30 seeds within Beast2
# ---------------------------------------------------------

project_level_summary = (
    all_project_run_metrics
    .groupby(
        [
            "Project",
            "NoisePercent",
            "Technique",
        ],
        as_index=False,
        sort=True,
    )
    .agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        MeanAPFD=(
            "MeanAPFD",
            "mean",
        ),

        SD_APFD_AcrossSeeds=(
            "MeanAPFD",
            "std",
        ),

        MedianAPFD=(
            "MeanAPFD",
            "median",
        ),

        MinimumAPFD=(
            "MeanAPFD",
            "min",
        ),

        MaximumAPFD=(
            "MeanAPFD",
            "max",
        ),

        MeanAPFDc=(
            "MeanAPFDc",
            "mean",
        ),

        SD_APFDc_AcrossSeeds=(
            "MeanAPFDc",
            "std",
        ),

        MedianAPFDc=(
            "MeanAPFDc",
            "median",
        ),

        MinimumAPFDc=(
            "MeanAPFDc",
            "min",
        ),

        MaximumAPFDc=(
            "MeanAPFDc",
            "max",
        ),

        EvaluatedBuildsPerSeed=(
            "EvaluatedBuilds",
            "first",
        ),

        TotalRankedTestsPerSeed=(
            "TotalRankedTests",
            "first",
        ),

        TotalFailuresPerSeed=(
            "TotalFailures",
            "first",
        ),
    )
)


if len(
    project_level_summary
) != EXPECTED_PROJECT_LEVEL_ROWS:
    raise AssertionError(
        "Expected 63 project-level technique-noise "
        "rows."
    )


if not (
    project_level_summary[
        "Seeds"
    ] == 30
).all():
    raise AssertionError(
        "At least one project-level result does not "
        "aggregate all 30 seeds."
    )


project_level_summary[
    "SE_APFD"
] = (
    project_level_summary[
        "SD_APFD_AcrossSeeds"
    ]
    /
    np.sqrt(
        project_level_summary[
            "Seeds"
        ].astype(float)
    )
)


project_level_summary[
    "CI95LowerAPFD"
] = (
    project_level_summary[
        "MeanAPFD"
    ]
    -
    T_CRITICAL_95_DF_29
    *
    project_level_summary[
        "SE_APFD"
    ]
)


project_level_summary[
    "CI95UpperAPFD"
] = (
    project_level_summary[
        "MeanAPFD"
    ]
    +
    T_CRITICAL_95_DF_29
    *
    project_level_summary[
        "SE_APFD"
    ]
)


project_level_summary[
    "SE_APFDc"
] = (
    project_level_summary[
        "SD_APFDc_AcrossSeeds"
    ]
    /
    np.sqrt(
        project_level_summary[
            "Seeds"
        ].astype(float)
    )
)


project_level_summary[
    "CI95LowerAPFDc"
] = (
    project_level_summary[
        "MeanAPFDc"
    ]
    -
    T_CRITICAL_95_DF_29
    *
    project_level_summary[
        "SE_APFDc"
    ]
)


project_level_summary[
    "CI95UpperAPFDc"
] = (
    project_level_summary[
        "MeanAPFDc"
    ]
    +
    T_CRITICAL_95_DF_29
    *
    project_level_summary[
        "SE_APFDc"
    ]
)


project_level_summary[
    "CI95LowerAPFD"
] = project_level_summary[
    "CI95LowerAPFD"
].clip(
    lower=0.0,
    upper=1.0,
)


project_level_summary[
    "CI95UpperAPFD"
] = project_level_summary[
    "CI95UpperAPFD"
].clip(
    lower=0.0,
    upper=1.0,
)


project_level_summary[
    "CI95LowerAPFDc"
] = project_level_summary[
    "CI95LowerAPFDc"
].clip(
    lower=0.0,
    upper=1.0,
)


project_level_summary[
    "CI95UpperAPFDc"
] = project_level_summary[
    "CI95UpperAPFDc"
].clip(
    lower=0.0,
    upper=1.0,
)


project_level_summary = (
    project_level_summary
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 10. Calculate project-level degradation from 0%
# ---------------------------------------------------------

project_clean_reference = (
    project_level_summary[
        project_level_summary[
            "NoisePercent"
        ] == 0
    ][
        [
            "Project",
            "Technique",
            "MeanAPFD",
            "MeanAPFDc",
        ]
    ]
    .copy()
    .rename(
        columns={
            "MeanAPFD":
                "CleanProjectMeanAPFD",

            "MeanAPFDc":
                "CleanProjectMeanAPFDc",
        }
    )
)


if len(
    project_clean_reference
) != 7:
    raise AssertionError(
        "Expected seven project-level clean technique "
        "references."
    )


project_level_degradation = (
    project_level_summary
    .merge(
        project_clean_reference,
        on=[
            "Project",
            "Technique",
        ],
        how="left",
        validate="many_to_one",
    )
)


project_level_degradation[
    "DeltaAPFD_NoiseMinusClean"
] = (
    project_level_degradation[
        "MeanAPFD"
    ]
    -
    project_level_degradation[
        "CleanProjectMeanAPFD"
    ]
)


project_level_degradation[
    "DeltaAPFDc_NoiseMinusClean"
] = (
    project_level_degradation[
        "MeanAPFDc"
    ]
    -
    project_level_degradation[
        "CleanProjectMeanAPFDc"
    ]
)


project_level_degradation[
    "AbsoluteLossAPFD"
] = (
    project_level_degradation[
        "CleanProjectMeanAPFD"
    ]
    -
    project_level_degradation[
        "MeanAPFD"
    ]
)


project_level_degradation[
    "AbsoluteLossAPFDc"
] = (
    project_level_degradation[
        "CleanProjectMeanAPFDc"
    ]
    -
    project_level_degradation[
        "MeanAPFDc"
    ]
)


project_level_degradation[
    "PercentageLossAPFD"
] = safe_percentage_loss(
    project_level_degradation[
        "CleanProjectMeanAPFD"
    ],
    project_level_degradation[
        "MeanAPFD"
    ],
)


project_level_degradation[
    "PercentageLossAPFDc"
] = safe_percentage_loss(
    project_level_degradation[
        "CleanProjectMeanAPFDc"
    ],
    project_level_degradation[
        "MeanAPFDc"
    ],
)


project_level_degradation[
    "RetentionAPFD"
] = safe_retention(
    project_level_degradation[
        "CleanProjectMeanAPFD"
    ],
    project_level_degradation[
        "MeanAPFD"
    ],
)


project_level_degradation[
    "RetentionAPFDc"
] = safe_retention(
    project_level_degradation[
        "CleanProjectMeanAPFDc"
    ],
    project_level_degradation[
        "MeanAPFDc"
    ],
)


project_level_degradation = (
    project_level_degradation
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 11. Aggregate noise-injection diagnostics over seeds
# ---------------------------------------------------------

noise_injection_summary = (
    condition_summary
    .groupby(
        "NoisePercent",
        as_index=False,
        sort=True,
    )
    .agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        TrainingExecutionRowsPerSeed=(
            "TrainingExecutionRows",
            "first",
        ),

        MeanRawRowsFlipped=(
            "RawRowsFlipped",
            "mean",
        ),

        SDRawRowsFlipped=(
            "RawRowsFlipped",
            "std",
        ),

        MinimumRawRowsFlipped=(
            "RawRowsFlipped",
            "min",
        ),

        MaximumRawRowsFlipped=(
            "RawRowsFlipped",
            "max",
        ),

        MeanRealisedNoisePercent=(
            "RealisedNoisePercent",
            "mean",
        ),

        SDRealisedNoisePercent=(
            "RealisedNoisePercent",
            "std",
        ),

        MeanPassToFailure=(
            "PassToFailure",
            "mean",
        ),

        MeanFailureToPass=(
            "FailureToPass",
            "mean",
        ),

        MeanRetainedLabelChanges=(
            "RetainedLabelChanges",
            "mean",
        ),

        SDRetainedLabelChanges=(
            "RetainedLabelChanges",
            "std",
        ),

        MeanTrainingFailures=(
            "TrainingFailures",
            "mean",
        ),

        MeanTrainingPasses=(
            "TrainingPasses",
            "mean",
        ),

        MeanConditionSeconds=(
            "ConditionSeconds",
            "mean",
        ),

        SDConditionSeconds=(
            "ConditionSeconds",
            "std",
        ),
    )
)


if len(
    noise_injection_summary
) != 9:
    raise AssertionError(
        "Expected nine noise-injection summary rows."
    )


if not (
    noise_injection_summary[
        "Seeds"
    ] == 30
).all():
    raise AssertionError(
        "A noise level does not aggregate all 30 seeds."
    )


if int(
    noise_injection_summary.loc[
        noise_injection_summary[
            "NoisePercent"
        ] == 0,
        "MeanRawRowsFlipped",
    ].iloc[0]
) != 0:
    raise AssertionError(
        "The aggregated 0% condition contains flips."
    )


if not noise_injection_summary[
    "MeanRawRowsFlipped"
].is_monotonic_increasing:
    raise AssertionError(
        "Mean raw rows flipped are not monotonic across "
        "nested noise levels."
    )


# ---------------------------------------------------------
# 12. Revalidate baseline invariants from aggregation
# ---------------------------------------------------------

random_rows = (
    all_project_run_metrics[
        all_project_run_metrics[
            "Technique"
        ] == "Random"
    ]
)


random_invariant = (
    random_rows
    .groupby(
        "RepetitionSeed",
        sort=True,
    )
    .agg(
        MinimumAPFD=(
            "MeanAPFD",
            "min",
        ),

        MaximumAPFD=(
            "MeanAPFD",
            "max",
        ),

        MinimumAPFDc=(
            "MeanAPFDc",
            "min",
        ),

        MaximumAPFDc=(
            "MeanAPFDc",
            "max",
        ),

        NoiseLevels=(
            "NoisePercent",
            "nunique",
        ),
    )
)


random_constant_across_noise = bool(
    (
        np.abs(
            random_invariant[
                "MaximumAPFD"
            ]
            -
            random_invariant[
                "MinimumAPFD"
            ]
        ) <= 1e-12
    ).all()
    and
    (
        np.abs(
            random_invariant[
                "MaximumAPFDc"
            ]
            -
            random_invariant[
                "MinimumAPFDc"
            ]
        ) <= 1e-12
    ).all()
    and
    (
        random_invariant[
            "NoiseLevels"
        ] == 9
    ).all()
)


if not random_constant_across_noise:
    raise AssertionError(
        "Random project-run metrics changed across "
        "noise levels for the same seed."
    )


qtf_rows = (
    all_project_run_metrics[
        all_project_run_metrics[
            "Technique"
        ] == "QTF-Avg"
    ]
)


qtf_constant_across_conditions = bool(
    (
        float(
            qtf_rows[
                "MeanAPFD"
            ].max()
        )
        -
        float(
            qtf_rows[
                "MeanAPFD"
            ].min()
        )
    ) <= 1e-12
    and
    (
        float(
            qtf_rows[
                "MeanAPFDc"
            ].max()
        )
        -
        float(
            qtf_rows[
                "MeanAPFDc"
            ].min()
        )
    ) <= 1e-12
)


if not qtf_constant_across_conditions:
    raise AssertionError(
        "QTF-Avg project-run metrics changed across "
        "conditions."
    )


# ---------------------------------------------------------
# 13. Rebuild package in an isolated temporary directory
# ---------------------------------------------------------

if TEMPORARY_PACKAGE_DIRECTORY.exists():
    shutil.rmtree(
        TEMPORARY_PACKAGE_DIRECTORY
    )


TEMPORARY_PACKAGE_DIRECTORY.mkdir(
    parents=True,
    exist_ok=False,
)


TECHNIQUE_NOISE_DIRECTORY = (
    TEMPORARY_PACKAGE_DIRECTORY
    / "technique_noise"
)


TECHNIQUE_NOISE_DIRECTORY.mkdir(
    parents=True,
    exist_ok=False,
)


# Nine core package files.
all_project_run_metrics.to_csv(
    TEMPORARY_PACKAGE_DIRECTORY
    / "project_run_metrics_all.csv",
    index=False,
)


all_project_run_metrics.to_parquet(
    TEMPORARY_PACKAGE_DIRECTORY
    / "project_run_metrics_all.parquet",
    index=False,
)


all_build_metrics.to_parquet(
    TEMPORARY_PACKAGE_DIRECTORY
    / "build_metrics_all.parquet",
    index=False,
)


all_fit_times.to_csv(
    TEMPORARY_PACKAGE_DIRECTORY
    / "fit_times_all.csv",
    index=False,
)


all_prediction_summary.to_csv(
    TEMPORARY_PACKAGE_DIRECTORY
    / "prediction_summary_all.csv",
    index=False,
)


condition_summary.to_csv(
    TEMPORARY_PACKAGE_DIRECTORY
    / "condition_summary.csv",
    index=False,
)


project_level_summary.to_csv(
    TEMPORARY_PACKAGE_DIRECTORY
    / "project_level_noise_technique_summary.csv",
    index=False,
)


project_level_degradation.to_csv(
    TEMPORARY_PACKAGE_DIRECTORY
    / "project_level_degradation_summary.csv",
    index=False,
)


noise_injection_summary.to_csv(
    TEMPORARY_PACKAGE_DIRECTORY
    / "noise_injection_summary.csv",
    index=False,
)


# 63 technique-noise files:
# 7 techniques × 9 noise levels.
technique_noise_file_records = []


seed_level_output_columns = [
    "Project",
    "NoisePercent",
    "RepetitionSeed",
    "Technique",

    "MeanAPFD",
    "MeanAPFDc",

    "SD_APFD",
    "SD_APFDc",

    "EvaluatedBuilds",
    "TotalRankedTests",
    "TotalFailures",

    "CleanMeanAPFD",
    "CleanMeanAPFDc",

    "DeltaAPFD_NoiseMinusClean",
    "DeltaAPFDc_NoiseMinusClean",

    "AbsoluteLossAPFD",
    "AbsoluteLossAPFDc",

    "PercentageLossAPFD",
    "PercentageLossAPFDc",

    "RetentionAPFD",
    "RetentionAPFDc",
]


for technique in EXPECTED_TECHNIQUES:
    for noise_percent in EXPECTED_NOISE_LEVELS:
        subset = (
            seed_level_degradation[
                (
                    seed_level_degradation[
                        "Technique"
                    ] == technique
                )
                &
                (
                    seed_level_degradation[
                        "NoisePercent"
                    ] == noise_percent
                )
            ][
                seed_level_output_columns
            ]
            .sort_values(
                "RepetitionSeed",
                kind="mergesort",
            )
            .reset_index(drop=True)
        )


        if len(
            subset
        ) != 30:
            raise AssertionError(
                f"{technique}, noise {noise_percent} "
                "does not contain 30 seed rows."
            )


        if set(
            subset[
                "RepetitionSeed"
            ].astype(np.int64)
        ) != set(
            EXPECTED_SEEDS
        ):
            raise AssertionError(
                f"{technique}, noise {noise_percent} "
                "contains the wrong seeds."
            )


        file_name = (
            f"{technique_file_slug(technique)}"
            f"__noise_{int(noise_percent):03d}.csv"
        )


        destination = (
            TECHNIQUE_NOISE_DIRECTORY
            / file_name
        )


        subset.to_csv(
            destination,
            index=False,
        )


        technique_noise_file_records.append({
            "Technique":
                technique,

            "NoisePercent":
                int(
                    noise_percent
                ),

            "FileName":
                file_name,

            "Rows":
                int(
                    len(
                        subset
                    )
                ),
        })


technique_noise_file_summary = pd.DataFrame(
    technique_noise_file_records
)


if len(
    technique_noise_file_summary
) != EXPECTED_TECHNIQUE_NOISE_FILES:
    raise AssertionError(
        "Expected exactly 63 technique-noise files."
    )


# ---------------------------------------------------------
# 14. Validate temporary package structure
# ---------------------------------------------------------

temporary_package_files = sorted([
    path
    for path in (
        TEMPORARY_PACKAGE_DIRECTORY.rglob(
            "*"
        )
    )
    if path.is_file()
])


temporary_core_files = [
    path
    for path in (
        TEMPORARY_PACKAGE_DIRECTORY.iterdir()
    )
    if path.is_file()
]


temporary_technique_noise_files = [
    path
    for path in (
        TECHNIQUE_NOISE_DIRECTORY.iterdir()
    )
    if path.is_file()
]


if len(
    temporary_core_files
) != EXPECTED_CORE_PACKAGE_FILES:
    raise AssertionError(
        "Expected exactly nine core package files.\n"
        f"Observed: {len(temporary_core_files)}"
    )


if len(
    temporary_technique_noise_files
) != EXPECTED_TECHNIQUE_NOISE_FILES:
    raise AssertionError(
        "Expected exactly 63 technique-noise files.\n"
        f"Observed: "
        f"{len(temporary_technique_noise_files)}"
    )


if len(
    temporary_package_files
) != EXPECTED_FINAL_PACKAGE_FILES:
    raise AssertionError(
        "Expected exactly 72 final-package files.\n"
        f"Observed: {len(temporary_package_files)}"
    )


temporary_files_found = [
    str(
        path.relative_to(
            TEMPORARY_PACKAGE_DIRECTORY
        )
    )
    for path in temporary_package_files
    if ".tmp" in path.name
]


if temporary_files_found:
    raise AssertionError(
        "Temporary files were found inside the package:\n"
        + "\n".join(
            temporary_files_found
        )
    )


# ---------------------------------------------------------
# 15. Hash all temporary package files
# ---------------------------------------------------------

package_inventory_records = []


for file_path in temporary_package_files:
    relative_path = str(
        file_path.relative_to(
            TEMPORARY_PACKAGE_DIRECTORY
        )
    )


    package_inventory_records.append({
        "RelativePath":
            relative_path,

        "SizeBytes":
            int(
                file_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                file_path
            ),
    })


package_inventory = (
    pd.DataFrame(
        package_inventory_records
    )
    .sort_values(
        "RelativePath",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if len(
    package_inventory
) != EXPECTED_FINAL_PACKAGE_FILES:
    raise AssertionError(
        "The package inventory does not contain "
        "72 files."
    )


package_root_hasher = hashlib.sha256()


for row in package_inventory.itertuples(
    index=False
):
    digest_line = (
        f"{row.RelativePath}\t"
        f"{int(row.SizeBytes)}\t"
        f"{row.SHA256}\n"
    )


    package_root_hasher.update(
        digest_line.encode(
            "utf-8"
        )
    )


FINAL_PACKAGE_ROOT_SHA256 = (
    package_root_hasher.hexdigest()
)


FINAL_PACKAGE_BYTES = int(
    package_inventory[
        "SizeBytes"
    ].sum()
)


# ---------------------------------------------------------
# 16. Promote temporary package to final location
# ---------------------------------------------------------

if FINAL_PACKAGE_DIRECTORY.exists():
    shutil.rmtree(
        FINAL_PACKAGE_DIRECTORY
    )


os.replace(
    TEMPORARY_PACKAGE_DIRECTORY,
    FINAL_PACKAGE_DIRECTORY,
)


if not FINAL_PACKAGE_DIRECTORY.is_dir():
    raise AssertionError(
        "The final package directory was not created."
    )


# ---------------------------------------------------------
# 17. Independently reverify promoted package
# ---------------------------------------------------------

expected_inventory_by_path = {
    str(
        row.RelativePath
    ): {
        "SizeBytes":
            int(
                row.SizeBytes
            ),

        "SHA256":
            str(
                row.SHA256
            ),
    }
    for row in package_inventory.itertuples(
        index=False
    )
}


promoted_files = sorted([
    path
    for path in FINAL_PACKAGE_DIRECTORY.rglob(
        "*"
    )
    if path.is_file()
])


observed_relative_paths = {
    str(
        path.relative_to(
            FINAL_PACKAGE_DIRECTORY
        )
    )
    for path in promoted_files
}


expected_relative_paths = set(
    expected_inventory_by_path.keys()
)


missing_promoted_files = sorted(
    expected_relative_paths
    - observed_relative_paths
)


unexpected_promoted_files = sorted(
    observed_relative_paths
    - expected_relative_paths
)


size_mismatches = []
hash_mismatches = []


for file_path in promoted_files:
    relative_path = str(
        file_path.relative_to(
            FINAL_PACKAGE_DIRECTORY
        )
    )


    if relative_path not in (
        expected_inventory_by_path
    ):
        continue


    expected_entry = (
        expected_inventory_by_path[
            relative_path
        ]
    )


    observed_size = int(
        file_path.stat().st_size
    )


    if observed_size != int(
        expected_entry[
            "SizeBytes"
        ]
    ):
        size_mismatches.append({
            "RelativePath":
                relative_path,

            "ExpectedSizeBytes":
                int(
                    expected_entry[
                        "SizeBytes"
                    ]
                ),

            "ObservedSizeBytes":
                observed_size,
        })


    observed_hash = sha256_file(
        file_path
    )


    if observed_hash != str(
        expected_entry[
            "SHA256"
        ]
    ):
        hash_mismatches.append({
            "RelativePath":
                relative_path,

            "ExpectedSHA256":
                str(
                    expected_entry[
                        "SHA256"
                    ]
                ),

            "ObservedSHA256":
                observed_hash,
        })


if missing_promoted_files:
    raise AssertionError(
        "Files are missing after package promotion:\n"
        + "\n".join(
            missing_promoted_files
        )
    )


if unexpected_promoted_files:
    raise AssertionError(
        "Unexpected files exist after package "
        "promotion:\n"
        + "\n".join(
            unexpected_promoted_files
        )
    )


if size_mismatches:
    display(
        pd.DataFrame(
            size_mismatches
        )
    )


    raise AssertionError(
        "At least one promoted package file has the "
        "wrong size."
    )


if hash_mismatches:
    display(
        pd.DataFrame(
            hash_mismatches
        )
    )


    raise AssertionError(
        "At least one promoted package file has the "
        "wrong SHA-256 digest."
    )


if len(
    promoted_files
) != EXPECTED_FINAL_PACKAGE_FILES:
    raise AssertionError(
        "The promoted final package does not contain "
        "72 files."
    )


# ---------------------------------------------------------
# 18. Save package inventory and validation report
# ---------------------------------------------------------

package_inventory[
    "FinalAbsolutePath"
] = package_inventory[
    "RelativePath"
].map(
    lambda value:
        str(
            FINAL_PACKAGE_DIRECTORY
            / value
        )
)


atomic_write_csv(
    package_inventory,
    FINAL_PACKAGE_INVENTORY_PATH,
)


package_validation = pd.DataFrame([
    {
        "Validation":
            "Package file count",

        "Expected":
            EXPECTED_FINAL_PACKAGE_FILES,

        "Observed":
            len(
                promoted_files
            ),

        "Mismatches":
            0,

        "Passed":
            True,
    },
    {
        "Validation":
            "Core file count",

        "Expected":
            EXPECTED_CORE_PACKAGE_FILES,

        "Observed":
            len(
                temporary_core_files
            ),

        "Mismatches":
            0,

        "Passed":
            True,
    },
    {
        "Validation":
            "Technique-noise file count",

        "Expected":
            EXPECTED_TECHNIQUE_NOISE_FILES,

        "Observed":
            len(
                temporary_technique_noise_files
            ),

        "Mismatches":
            0,

        "Passed":
            True,
    },
    {
        "Validation":
            "Missing promoted files",

        "Expected":
            0,

        "Observed":
            len(
                missing_promoted_files
            ),

        "Mismatches":
            len(
                missing_promoted_files
            ),

        "Passed":
            len(
                missing_promoted_files
            ) == 0,
    },
    {
        "Validation":
            "Unexpected promoted files",

        "Expected":
            0,

        "Observed":
            len(
                unexpected_promoted_files
            ),

        "Mismatches":
            len(
                unexpected_promoted_files
            ),

        "Passed":
            len(
                unexpected_promoted_files
            ) == 0,
    },
    {
        "Validation":
            "Promoted size mismatches",

        "Expected":
            0,

        "Observed":
            len(
                size_mismatches
            ),

        "Mismatches":
            len(
                size_mismatches
            ),

        "Passed":
            len(
                size_mismatches
            ) == 0,
    },
    {
        "Validation":
            "Promoted SHA-256 mismatches",

        "Expected":
            0,

        "Observed":
            len(
                hash_mismatches
            ),

        "Mismatches":
            len(
                hash_mismatches
            ),

        "Passed":
            len(
                hash_mismatches
            ) == 0,
    },
])


if not package_validation[
    "Passed"
].all():
    raise AssertionError(
        "The final package validation table contains "
        "a failed check."
    )


atomic_write_csv(
    package_validation,
    FINAL_PACKAGE_VALIDATION_PATH,
)


aggregation_completed_at = (
    pd.Timestamp.utcnow().isoformat()
)


aggregation_wall_clock_seconds = float(
    time.time()
    - aggregation_started_clock
)


final_package_report = {
    "ProjectNumber":
        int(
            PROJECT_NUMBER
        ),

    "Project":
        str(
            PROJECT_NAME
        ),

    "ProjectSlug":
        str(
            PROJECT_SLUG
        ),

    "Status":
        "PASS",

    "AggregationHierarchy": [
        (
            "Ranked tests within each evaluation build"
        ),
        (
            "Build-level APFD/APFDc"
        ),
        (
            "Mean eligible evaluation builds within "
            "each project-noise-seed-technique run"
        ),
        (
            "Mean 30 seeds within Beast2 for each "
            "noise-technique combination"
        ),
    ],

    "RawSource": {
        "Conditions":
            EXPECTED_CONDITIONS,

        "ProjectRunRows":
            EXPECTED_PROJECT_RUN_ROWS,

        "BuildMetricRows":
            EXPECTED_BUILD_METRIC_ROWS,

        "FitRows":
            EXPECTED_FIT_ROWS,

        "PredictionSummaryRows":
            EXPECTED_PREDICTION_ROWS,

        "RawFiles":
            int(
                previous_checkpoint[
                    "RawAuditFileCount"
                ]
            ),

        "RawBytes":
            int(
                previous_checkpoint[
                    "RawAuditBytes"
                ]
            ),

        "RawRootSHA256":
            str(
                previous_checkpoint[
                    "RawAuditRootSHA256"
                ]
            ),
    },

    "AggregatedOutputs": {
        "SeedLevelProjectRunRows":
            int(
                len(
                    all_project_run_metrics
                )
            ),

        "CombinedBuildMetricRows":
            int(
                len(
                    all_build_metrics
                )
            ),

        "ProjectLevelNoiseTechniqueRows":
            int(
                len(
                    project_level_summary
                )
            ),

        "ProjectLevelDegradationRows":
            int(
                len(
                    project_level_degradation
                )
            ),

        "ConditionSummaryRows":
            int(
                len(
                    condition_summary
                )
            ),

        "NoiseInjectionSummaryRows":
            int(
                len(
                    noise_injection_summary
                )
            ),

        "TechniqueNoiseSeedFiles":
            int(
                len(
                    technique_noise_file_summary
                )
            ),
    },

    "Statistics": {
        "SeedsPerProjectNoiseTechnique":
            30,

        "Mean":
            True,

        "StandardDeviation":
            True,

        "Median":
            True,

        "MinimumMaximum":
            True,

        "ConfidenceInterval":
            "95% Student-t interval",

        "DegreesOfFreedom":
            29,

        "TCritical":
            T_CRITICAL_95_DF_29,

        "PrimaryMetric":
            "APFDc",

        "SecondaryMetric":
            "APFD",
    },

    "DegradationDefinitions": {
        "AbsoluteLoss":
            "clean performance minus noisy performance",

        "PercentageLoss":
            (
                "(clean minus noisy) divided by clean "
                "times 100"
            ),

        "Retention":
            "noisy performance divided by clean",

        "CleanReference":
            (
                "Matched 0% seed for seed-level outputs; "
                "30-seed 0% project mean for project-level "
                "outputs"
            ),
    },

    "BaselineValidation": {
        "RandomConstantAcrossNoiseWithinSeed":
            random_constant_across_noise,

        "QTFAvgConstantAcrossAllConditions":
            qtf_constant_across_conditions,
    },

    "FinalPackage": {
        "Directory":
            str(
                FINAL_PACKAGE_DIRECTORY
            ),

        "CoreFiles":
            EXPECTED_CORE_PACKAGE_FILES,

        "TechniqueNoiseFiles":
            EXPECTED_TECHNIQUE_NOISE_FILES,

        "TotalFiles":
            int(
                len(
                    promoted_files
                )
            ),

        "TotalBytes":
            FINAL_PACKAGE_BYTES,

        "HashAlgorithm":
            "SHA-256",

        "RootSHA256":
            FINAL_PACKAGE_ROOT_SHA256,

        "RootDigestConstruction":
            (
                "SHA256 over sorted lines of "
                "RelativePath<TAB>SizeBytes<TAB>"
                "FileSHA256<NEWLINE>"
            ),

        "MissingFilesAfterPromotion":
            len(
                missing_promoted_files
            ),

        "UnexpectedFilesAfterPromotion":
            len(
                unexpected_promoted_files
            ),

        "SizeMismatchesAfterPromotion":
            len(
                size_mismatches
            ),

        "HashMismatchesAfterPromotion":
            len(
                hash_mismatches
            ),
    },

    "Artefacts": {
        "PackageInventory":
            str(
                FINAL_PACKAGE_INVENTORY_PATH
            ),

        "PackageValidation":
            str(
                FINAL_PACKAGE_VALIDATION_PATH
            ),

        "FinalPackageDirectory":
            str(
                FINAL_PACKAGE_DIRECTORY
            ),
    },

    "RegistryUpdated":
        False,

    "RegistryUpdateDeferredTo":
        "Project 7 final freeze step",

    "StartedAtUTC":
        aggregation_started_at,

    "CompletedAtUTC":
        aggregation_completed_at,

    "WallClockSeconds":
        aggregation_wall_clock_seconds,
}


atomic_write_json(
    final_package_report,
    FINAL_PACKAGE_REPORT_PATH,
)


# ---------------------------------------------------------
# 19. Update Project 7 checkpoint atomically
# ---------------------------------------------------------

with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    project_checkpoint = json.load(
        checkpoint_file
    )


project_checkpoint.update({
    "Status":
        "FINAL_AGGREGATION_PACKAGE_PASSED",

    "FinalAggregationSeedLevelProjectRunRows":
        int(
            len(
                all_project_run_metrics
            )
        ),

    "FinalAggregationBuildMetricRows":
        int(
            len(
                all_build_metrics
            )
        ),

    "FinalAggregationProjectLevelRows":
        int(
            len(
                project_level_summary
            )
        ),

    "FinalAggregationDegradationRows":
        int(
            len(
                project_level_degradation
            )
        ),

    "FinalAggregationConditionRows":
        int(
            len(
                condition_summary
            )
        ),

    "FinalAggregationTechniqueNoiseFiles":
        int(
            len(
                technique_noise_file_summary
            )
        ),

    "FinalPackageDirectory":
        str(
            FINAL_PACKAGE_DIRECTORY
        ),

    "FinalPackageFileCount":
        int(
            len(
                promoted_files
            )
        ),

    "FinalPackageBytes":
        FINAL_PACKAGE_BYTES,

    "FinalPackageHashAlgorithm":
        "SHA-256",

    "FinalPackageRootSHA256":
        FINAL_PACKAGE_ROOT_SHA256,

    "FinalPackageMissingFiles":
        int(
            len(
                missing_promoted_files
            )
        ),

    "FinalPackageUnexpectedFiles":
        int(
            len(
                unexpected_promoted_files
            )
        ),

    "FinalPackageSizeMismatches":
        int(
            len(
                size_mismatches
            )
        ),

    "FinalPackageHashMismatches":
        int(
            len(
                hash_mismatches
            )
        ),

    "FinalPackageInventory":
        str(
            FINAL_PACKAGE_INVENTORY_PATH
        ),

    "FinalPackageValidation":
        str(
            FINAL_PACKAGE_VALIDATION_PATH
        ),

    "FinalPackageReport":
        str(
            FINAL_PACKAGE_REPORT_PATH
        ),

    "FinalAggregationCompletedAtUTC":
        aggregation_completed_at,

    "UpdatedAtUTC":
        aggregation_completed_at,
})


temporary_checkpoint_path = (
    PROJECT_7_SELECTION_CHECKPOINT
    .with_suffix(
        ".json.tmp"
    )
)


with open(
    temporary_checkpoint_path,
    "w",
    encoding="utf-8",
) as checkpoint_file:
    json.dump(
        project_checkpoint,
        checkpoint_file,
        indent=2,
        default=str,
    )


os.replace(
    temporary_checkpoint_path,
    PROJECT_7_SELECTION_CHECKPOINT,
)


with open(
    PROJECT_7_SELECTION_CHECKPOINT,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    checkpoint_verification = json.load(
        checkpoint_file
    )


if checkpoint_verification.get(
    "Status"
) != "FINAL_AGGREGATION_PACKAGE_PASSED":
    raise AssertionError(
        "The final aggregation checkpoint was not "
        "written correctly."
    )


if int(
    checkpoint_verification.get(
        "FinalPackageFileCount",
        -1,
    )
) != EXPECTED_FINAL_PACKAGE_FILES:
    raise AssertionError(
        "The checkpoint does not confirm 72 final "
        "package files."
    )


if checkpoint_verification.get(
    "FinalPackageRootSHA256"
) != FINAL_PACKAGE_ROOT_SHA256:
    raise AssertionError(
        "The checkpoint does not preserve the final "
        "package root SHA-256."
    )


# ---------------------------------------------------------
# 20. Compact final output
# ---------------------------------------------------------

clear_output(
    wait=True
)


print(
    "=== PROJECT 7 STEP 11A RESULT ==="
)


print(
    "\nSource results combined:"
)

print(
    "Conditions:",
    len(
        condition_summary
    )
)

print(
    "Seed-level project-run rows:",
    len(
        all_project_run_metrics
    )
)

print(
    "Combined build-metric rows:",
    len(
        all_build_metrics
    )
)

print(
    "ML fit rows:",
    len(
        all_fit_times
    )
)

print(
    "Prediction-summary rows:",
    len(
        all_prediction_summary
    )
)


print(
    "\nProject-level aggregation:"
)

print(
    "Noise levels:",
    len(
        EXPECTED_NOISE_LEVELS
    )
)

print(
    "Techniques:",
    len(
        EXPECTED_TECHNIQUES
    )
)

print(
    "Seeds per technique-noise result:",
    30
)

print(
    "Project-level technique-noise rows:",
    len(
        project_level_summary
    )
)

print(
    "Project-level degradation rows:",
    len(
        project_level_degradation
    )
)


print(
    "\nBaseline invariants:"
)

print(
    "Random constant across noise within seed:",
    random_constant_across_noise
)

print(
    "QTF-Avg constant across all conditions:",
    qtf_constant_across_conditions
)


print(
    "\nFinal package:"
)

print(
    "Core files:",
    len(
        temporary_core_files
    )
)

print(
    "Technique-noise files:",
    len(
        temporary_technique_noise_files
    )
)

print(
    "Total package files:",
    len(
        promoted_files
    )
)

print(
    "Total package bytes:",
    FINAL_PACKAGE_BYTES
)

print(
    "Missing files after promotion:",
    len(
        missing_promoted_files
    )
)

print(
    "Unexpected files after promotion:",
    len(
        unexpected_promoted_files
    )
)

print(
    "Size mismatches after promotion:",
    len(
        size_mismatches
    )
)

print(
    "SHA-256 mismatches after promotion:",
    len(
        hash_mismatches
    )
)

print(
    "Final package root SHA-256:",
    FINAL_PACKAGE_ROOT_SHA256
)


print(
    "\nFinal package directory:"
)

print(
    FINAL_PACKAGE_DIRECTORY
)


print(
    "\nFinal package report:"
)

print(
    FINAL_PACKAGE_REPORT_PATH
)


print(
    "\nSelected project-level APFDc results:"
)

display(
    project_level_degradation[
        project_level_degradation[
            "NoisePercent"
        ].isin(
            [
                0,
                25,
                50,
            ]
        )
    ][
        [
            "NoisePercent",
            "Technique",
            "MeanAPFDc",
            "SD_APFDc_AcrossSeeds",
            "CI95LowerAPFDc",
            "CI95UpperAPFDc",
            "AbsoluteLossAPFDc",
            "PercentageLossAPFDc",
            "RetentionAPFDc",
        ]
    ]
    .sort_values(
        [
            "NoisePercent",
            "MeanAPFDc",
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


print(
    "\nValidation status:",
    "PASS"
)


print(
    "\nSUCCESS: All 270 validated conditions were "
    "combined without rerunning any model."
)

print(
    "SUCCESS: All 1,890 seed-level project-run rows "
    "were retained."
)

print(
    "SUCCESS: The 30 seeds were aggregated into 63 "
    "project-level technique-noise results."
)

print(
    "SUCCESS: APFD and APFDc loss, percentage loss and "
    "retention were calculated against clean 0% "
    "references."
)

print(
    "SUCCESS: The final Project 7 package contains "
    "exactly 72 files."
)

print(
    "SUCCESS: Every promoted package file matched its "
    "expected size and SHA-256 digest."
)

print(
    "SUCCESS: Project 7 is ready for the final registry "
    "update and COMPLETE_AND_FROZEN status."
)

=== PROJECT 7 STEP 11A RESULT ===

Source results combined:
Conditions: 270
Seed-level project-run rows: 1890
Combined build-metric rows: 98280
ML fit rows: 1080
Prediction-summary rows: 1080

Project-level aggregation:
Noise levels: 9
Techniques: 7
Seeds per technique-noise result: 30
Project-level technique-noise rows: 63
Project-level degradation rows: 63

Baseline invariants:
Random constant across noise within seed: True
QTF-Avg constant across all conditions: True

Final package:
Core files: 9
Technique-noise files: 63
Total package files: 72
Total package bytes: 1624833
Missing files after promotion: 0
Unexpected files after promotion: 0
Size mismatches after promotion: 0
SHA-256 mismatches after promotion: 0
Final package root SHA-256: 93e9cc74fac017fef89b005752e1b7ede2a73f9665dd1840891e7f6d9c341c13

Final package directory:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_30_seed_final

Final package report:
/content/drive/MyDrive/Thesis_Expe

,NoisePercent,Technique,MeanAPFDc,SD_APFDc_AcrossSeeds,CI95LowerAPFDc,CI95UpperAPFDc,AbsoluteLossAPFDc,PercentageLossAPFDc,RetentionAPFDc
0,0,RandomForest,0.778930,0.012758,0.774166,0.783694,0.000000,0.000000,1.000000
1,0,LightGBM,0.741619,0.000000,0.741619,0.741619,0.000000,0.000000,1.000000
2,0,LatestFail,0.739378,0.000000,0.739378,0.739378,0.000000,0.000000,1.000000
3,0,XGBoost,0.649718,0.000000,0.649718,0.649718,0.000000,0.000000,1.000000
4,0,QTF-Avg,0.590571,0.000000,0.590571,0.590571,0.000000,0.000000,1.000000
5,0,NaiveBayes,0.548409,0.000000,0.548409,0.548409,0.000000,0.000000,1.000000
6,0,Random,0.503048,0.033870,0.490401,0.515695,0.000000,0.000000,1.000000
7,25,LatestFail,0.732340,0.009222,0.728897,0.735784,0.007037,0.951795,0.990482
8,25,QTF-Avg,0.590571,0.000000,0.590571,0.590571,0.000000,0.000000,1.000000
9,25,Random,0.503048,0.033870,0.490401,0.515695,0.000000,0.000000,1.000000



Validation status: PASS

SUCCESS: All 270 validated conditions were combined without rerunning any model.
SUCCESS: All 1,890 seed-level project-run rows were retained.
SUCCESS: The 30 seeds were aggregated into 63 project-level technique-noise results.
SUCCESS: APFD and APFDc loss, percentage loss and retention were calculated against clean 0% references.
SUCCESS: The final Project 7 package contains exactly 72 files.
SUCCESS: Every promoted package file matched its expected size and SHA-256 digest.
SUCCESS: Project 7 is ready for the final registry update and COMPLETE_AND_FROZEN status.


In [ ]:
# =========================================================
# PROJECT 7 — STEP 11B
# FINAL REGISTRY UPDATE AND COMPLETE-AND-FROZEN STATUS
#
# This cell does NOT:
#   - rerun any ML model
#   - regenerate noise
#   - recalculate rankings
#   - modify raw results
#   - modify the 72-file final package
#
# It:
#   - reverifies raw and package root hashes
#   - creates a permanent completion certificate
#   - adds/upserts Project 7 in the completion registry
#   - freezes the Project 7 checkpoint
# =========================================================

from pathlib import Path
import hashlib
import json
import os

import numpy as np
import pandas as pd
from IPython.display import clear_output, display


print(
    "=== PROJECT 7 STEP 11B: "
    "FINAL FREEZE AND REGISTRY UPDATE ==="
)


# ---------------------------------------------------------
# 1. Permanent paths and frozen expected values
# ---------------------------------------------------------

THESIS_DRIVE = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)


NOTES_DRIVE = (
    THESIS_DRIVE
    / "Notes"
)


RAW_RESULTS_DRIVE = (
    THESIS_DRIVE
    / "Results"
    / "Raw"
)


AGGREGATED_RESULTS_DRIVE = (
    THESIS_DRIVE
    / "Results"
    / "Aggregated"
)


PROJECT_NUMBER = 7
PROJECT_NAME = "CompEvol@beast2"
PROJECT_SLUG = "CompEvol__beast2"


PROJECT_CHECKPOINT_PATH = (
    NOTES_DRIVE
    / "project_07_selection_checkpoint.json"
)


COMPLETION_REGISTRY_PATH = (
    NOTES_DRIVE
    / "completed_project_registry.csv"
)


COMPLETION_CERTIFICATE_PATH = (
    NOTES_DRIVE
    / "project_07_completion_certificate.json"
)


RAW_DIRECTORY = (
    RAW_RESULTS_DRIVE
    / PROJECT_SLUG
    / "beast2_30_seed_raw"
)


RAW_INVENTORY_PATH = (
    AGGREGATED_RESULTS_DRIVE
    / PROJECT_SLUG
    / "beast2_raw_audit"
    / "beast2_raw_file_inventory_sha256.csv"
)


RAW_AUDIT_REPORT_PATH = (
    AGGREGATED_RESULTS_DRIVE
    / PROJECT_SLUG
    / "beast2_raw_audit"
    / "beast2_raw_output_audit_report.json"
)


FINAL_PACKAGE_DIRECTORY = (
    AGGREGATED_RESULTS_DRIVE
    / PROJECT_SLUG
    / "beast2_30_seed_final"
)


FINAL_PACKAGE_INVENTORY_PATH = (
    AGGREGATED_RESULTS_DRIVE
    / PROJECT_SLUG
    / "beast2_final_package_audit"
    / "beast2_final_package_file_inventory_sha256.csv"
)


FINAL_PACKAGE_REPORT_PATH = (
    AGGREGATED_RESULTS_DRIVE
    / PROJECT_SLUG
    / "beast2_final_package_audit"
    / "beast2_final_package_report.json"
)


EXPECTED_RAW_CONDITIONS = 270
EXPECTED_ML_FITS = 1080

EXPECTED_RANKING_ROWS = 6_951_420
EXPECTED_BUILD_METRIC_ROWS = 98_280
EXPECTED_PROJECT_RUN_ROWS = 1_890
EXPECTED_MANIFEST_ROWS = 5_336_820

EXPECTED_RAW_FILES = 2_160
EXPECTED_RAW_BYTES = 159_516_190

EXPECTED_RAW_ROOT_SHA256 = (
    "93a6581d90895fd011b48490aa66f55a91b0078818bfd6966b3cf5b9e0f86373"
)

EXPECTED_FINAL_PACKAGE_FILES = 72
EXPECTED_FINAL_PACKAGE_BYTES = 1_624_833

EXPECTED_FINAL_PACKAGE_ROOT_SHA256 = (
    "93e9cc74fac017fef89b005752e1b7ede2a73f9665dd1840891e7f6d9c341c13"
)

EXPECTED_PROJECT_LEVEL_ROWS = 63
EXPECTED_TECHNIQUE_NOISE_FILES = 63

EXPECTED_NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

EXPECTED_SEEDS = list(
    range(1, 31)
)

EXPECTED_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
    "Random",
    "LatestFail",
    "QTF-Avg",
]


# ---------------------------------------------------------
# 2. Confirm all permanent inputs exist
# ---------------------------------------------------------

required_paths = [
    PROJECT_CHECKPOINT_PATH,
    COMPLETION_REGISTRY_PATH,
    RAW_DIRECTORY,
    RAW_INVENTORY_PATH,
    RAW_AUDIT_REPORT_PATH,
    FINAL_PACKAGE_DIRECTORY,
    FINAL_PACKAGE_INVENTORY_PATH,
    FINAL_PACKAGE_REPORT_PATH,
]


missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]


if missing_paths:
    raise FileNotFoundError(
        "Required Project 7 completion files are "
        "missing:\n"
        + "\n".join(
            missing_paths
        )
    )


# ---------------------------------------------------------
# 3. General integrity helpers
# ---------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()


    with open(
        path,
        "rb",
    ) as input_file:
        while True:
            chunk = input_file.read(
                chunk_size
            )


            if not chunk:
                break


            digest.update(
                chunk
            )


    return digest.hexdigest()


def calculate_inventory_root(
    inventory,
):
    required_columns = {
        "RelativePath",
        "SizeBytes",
        "SHA256",
    }


    missing_columns = (
        required_columns
        - set(
            inventory.columns
        )
    )


    if missing_columns:
        raise RuntimeError(
            "Inventory is missing columns:\n"
            + "\n".join(
                sorted(
                    missing_columns
                )
            )
        )


    ordered = (
        inventory[
            [
                "RelativePath",
                "SizeBytes",
                "SHA256",
            ]
        ]
        .copy()
        .sort_values(
            "RelativePath",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    if ordered[
        "RelativePath"
    ].duplicated().any():
        raise AssertionError(
            "Inventory contains duplicate relative paths."
        )


    ordered[
        "SizeBytes"
    ] = pd.to_numeric(
        ordered[
            "SizeBytes"
        ],
        errors="raise",
    ).astype(np.int64)


    ordered[
        "SHA256"
    ] = ordered[
        "SHA256"
    ].astype(str)


    if ordered[
        "SHA256"
    ].str.len().ne(64).any():
        raise AssertionError(
            "Inventory contains an invalid SHA-256 "
            "digest."
        )


    root_hasher = hashlib.sha256()


    for row in ordered.itertuples(
        index=False
    ):
        digest_line = (
            f"{row.RelativePath}\t"
            f"{int(row.SizeBytes)}\t"
            f"{row.SHA256}\n"
        )


        root_hasher.update(
            digest_line.encode(
                "utf-8"
            )
        )


    return (
        root_hasher.hexdigest(),
        int(
            ordered[
                "SizeBytes"
            ].sum()
        ),
        ordered,
    )


def atomic_write_json(
    value,
    destination,
):
    destination = Path(
        destination
    )


    temporary = destination.with_name(
        destination.name
        + ".tmp"
    )


    with open(
        temporary,
        "w",
        encoding="utf-8",
    ) as output_file:
        json.dump(
            value,
            output_file,
            indent=2,
            default=str,
        )


    os.replace(
        temporary,
        destination,
    )


def atomic_write_csv(
    dataframe,
    destination,
):
    destination = Path(
        destination
    )


    temporary = destination.with_name(
        destination.name
        + ".tmp"
    )


    dataframe.to_csv(
        temporary,
        index=False,
    )


    os.replace(
        temporary,
        destination,
    )


# ---------------------------------------------------------
# 4. Validate the Step 11A checkpoint
# ---------------------------------------------------------

with open(
    PROJECT_CHECKPOINT_PATH,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    project_checkpoint = json.load(
        checkpoint_file
    )


allowed_checkpoint_statuses = {
    "FINAL_AGGREGATION_PACKAGE_PASSED",
    "COMPLETE_AND_FROZEN",
}


if project_checkpoint.get(
    "Status"
) not in allowed_checkpoint_statuses:
    raise AssertionError(
        "Step 11A has not passed.\n"
        f"Observed checkpoint status: "
        f"{project_checkpoint.get('Status')}"
    )


checkpoint_expected_values = {
    "RawAuditConditions":
        EXPECTED_RAW_CONDITIONS,

    "RawAuditRankingRows":
        EXPECTED_RANKING_ROWS,

    "RawAuditBuildMetricRows":
        EXPECTED_BUILD_METRIC_ROWS,

    "RawAuditProjectRunRows":
        EXPECTED_PROJECT_RUN_ROWS,

    "RawAuditMLFitRows":
        EXPECTED_ML_FITS,

    "RawAuditManifestRows":
        EXPECTED_MANIFEST_ROWS,

    "RawAuditFileCount":
        EXPECTED_RAW_FILES,

    "RawAuditBytes":
        EXPECTED_RAW_BYTES,

    "FinalAggregationSeedLevelProjectRunRows":
        EXPECTED_PROJECT_RUN_ROWS,

    "FinalAggregationBuildMetricRows":
        EXPECTED_BUILD_METRIC_ROWS,

    "FinalAggregationProjectLevelRows":
        EXPECTED_PROJECT_LEVEL_ROWS,

    "FinalAggregationDegradationRows":
        EXPECTED_PROJECT_LEVEL_ROWS,

    "FinalAggregationTechniqueNoiseFiles":
        EXPECTED_TECHNIQUE_NOISE_FILES,

    "FinalPackageFileCount":
        EXPECTED_FINAL_PACKAGE_FILES,

    "FinalPackageBytes":
        EXPECTED_FINAL_PACKAGE_BYTES,
}


for field_name, expected_value in (
    checkpoint_expected_values.items()
):
    observed_value = project_checkpoint.get(
        field_name
    )


    if int(
        observed_value
    ) != int(
        expected_value
    ):
        raise AssertionError(
            f"Checkpoint field {field_name} is "
            f"incorrect.\n"
            f"Expected: {expected_value}\n"
            f"Observed: {observed_value}"
        )


if project_checkpoint.get(
    "RawAuditRootSHA256"
) != EXPECTED_RAW_ROOT_SHA256:
    raise AssertionError(
        "Checkpoint raw-root SHA-256 mismatch."
    )


if project_checkpoint.get(
    "FinalPackageRootSHA256"
) != EXPECTED_FINAL_PACKAGE_ROOT_SHA256:
    raise AssertionError(
        "Checkpoint final-package SHA-256 mismatch."
    )


# ---------------------------------------------------------
# 5. Revalidate the complete raw inventory root
# ---------------------------------------------------------

raw_inventory = pd.read_csv(
    RAW_INVENTORY_PATH
)


if len(
    raw_inventory
) != EXPECTED_RAW_FILES:
    raise AssertionError(
        "Raw inventory does not contain 2,160 files."
    )


(
    reconstructed_raw_root,
    reconstructed_raw_bytes,
    ordered_raw_inventory,
) = calculate_inventory_root(
    raw_inventory
)


if reconstructed_raw_root != (
    EXPECTED_RAW_ROOT_SHA256
):
    raise AssertionError(
        "The raw inventory root SHA-256 does not match "
        "the successful Step 10 value."
    )


if reconstructed_raw_bytes != (
    EXPECTED_RAW_BYTES
):
    raise AssertionError(
        "The raw inventory byte total is incorrect.\n"
        f"Expected: {EXPECTED_RAW_BYTES}\n"
        f"Observed: {reconstructed_raw_bytes}"
    )


with open(
    RAW_AUDIT_REPORT_PATH,
    "r",
    encoding="utf-8",
) as report_file:
    raw_audit_report = json.load(
        report_file
    )


if raw_audit_report.get(
    "Status"
) != "PASS":
    raise AssertionError(
        "The permanent raw-output audit report is not "
        "marked PASS."
    )


if (
    raw_audit_report
    .get(
        "Integrity",
        {}
    )
    .get(
        "RawRootSHA256"
    )
    != EXPECTED_RAW_ROOT_SHA256
):
    raise AssertionError(
        "The raw-output audit report contains the "
        "wrong root SHA-256."
    )


# ---------------------------------------------------------
# 6. Independently hash all 72 final-package files
# ---------------------------------------------------------

actual_package_files = sorted([
    path
    for path in FINAL_PACKAGE_DIRECTORY.rglob(
        "*"
    )
    if path.is_file()
])


if len(
    actual_package_files
) != EXPECTED_FINAL_PACKAGE_FILES:
    raise AssertionError(
        "The final package does not contain exactly "
        "72 files.\n"
        f"Observed: {len(actual_package_files)}"
    )


actual_package_records = []


for file_path in actual_package_files:
    actual_package_records.append({
        "RelativePath":
            str(
                file_path.relative_to(
                    FINAL_PACKAGE_DIRECTORY
                )
            ),

        "SizeBytes":
            int(
                file_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                file_path
            ),
    })


actual_package_inventory = pd.DataFrame(
    actual_package_records
)


(
    reconstructed_package_root,
    reconstructed_package_bytes,
    ordered_actual_package_inventory,
) = calculate_inventory_root(
    actual_package_inventory
)


if reconstructed_package_root != (
    EXPECTED_FINAL_PACKAGE_ROOT_SHA256
):
    raise AssertionError(
        "The independently reconstructed final-package "
        "root SHA-256 is incorrect.\n"
        f"Expected: "
        f"{EXPECTED_FINAL_PACKAGE_ROOT_SHA256}\n"
        f"Observed: {reconstructed_package_root}"
    )


if reconstructed_package_bytes != (
    EXPECTED_FINAL_PACKAGE_BYTES
):
    raise AssertionError(
        "The independently reconstructed final-package "
        "byte total is incorrect.\n"
        f"Expected: {EXPECTED_FINAL_PACKAGE_BYTES}\n"
        f"Observed: {reconstructed_package_bytes}"
    )


# ---------------------------------------------------------
# 7. Compare actual package against saved inventory
# ---------------------------------------------------------

saved_package_inventory = pd.read_csv(
    FINAL_PACKAGE_INVENTORY_PATH
)


if len(
    saved_package_inventory
) != EXPECTED_FINAL_PACKAGE_FILES:
    raise AssertionError(
        "Saved final-package inventory does not contain "
        "72 files."
    )


saved_comparison_columns = (
    saved_package_inventory[
        [
            "RelativePath",
            "SizeBytes",
            "SHA256",
        ]
    ]
    .copy()
)


saved_comparison_columns[
    "SizeBytes"
] = pd.to_numeric(
    saved_comparison_columns[
        "SizeBytes"
    ],
    errors="raise",
).astype(np.int64)


package_inventory_comparison = (
    saved_comparison_columns
    .rename(
        columns={
            "SizeBytes":
                "ExpectedSizeBytes",

            "SHA256":
                "ExpectedSHA256",
        }
    )
    .merge(
        ordered_actual_package_inventory
        .rename(
            columns={
                "SizeBytes":
                    "ObservedSizeBytes",

                "SHA256":
                    "ObservedSHA256",
            }
        ),
        on="RelativePath",
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
)


missing_package_files = int(
    (
        package_inventory_comparison[
            "_merge"
        ] == "left_only"
    ).sum()
)


unexpected_package_files = int(
    (
        package_inventory_comparison[
            "_merge"
        ] == "right_only"
    ).sum()
)


package_size_mismatches = int(
    (
        package_inventory_comparison[
            "ExpectedSizeBytes"
        ]
        !=
        package_inventory_comparison[
            "ObservedSizeBytes"
        ]
    ).fillna(True).sum()
)


package_hash_mismatches = int(
    (
        package_inventory_comparison[
            "ExpectedSHA256"
        ]
        !=
        package_inventory_comparison[
            "ObservedSHA256"
        ]
    ).fillna(True).sum()
)


if missing_package_files != 0:
    raise AssertionError(
        "At least one expected package file is missing."
    )


if unexpected_package_files != 0:
    raise AssertionError(
        "At least one unexpected package file exists."
    )


if package_size_mismatches != 0:
    raise AssertionError(
        "At least one package file has a size mismatch."
    )


if package_hash_mismatches != 0:
    raise AssertionError(
        "At least one package file has a SHA-256 "
        "mismatch."
    )


with open(
    FINAL_PACKAGE_REPORT_PATH,
    "r",
    encoding="utf-8",
) as report_file:
    final_package_report = json.load(
        report_file
    )


if final_package_report.get(
    "Status"
) != "PASS":
    raise AssertionError(
        "The final-package report is not marked PASS."
    )


if (
    final_package_report
    .get(
        "FinalPackage",
        {}
    )
    .get(
        "RootSHA256"
    )
    != EXPECTED_FINAL_PACKAGE_ROOT_SHA256
):
    raise AssertionError(
        "The final-package report contains the wrong "
        "root SHA-256."
    )


# ---------------------------------------------------------
# 8. Read and validate the existing completion registry
# ---------------------------------------------------------

completion_registry = pd.read_csv(
    COMPLETION_REGISTRY_PATH
)


completion_registry.columns = [
    str(column)
    for column in completion_registry.columns
]


def find_registry_column(
    aliases,
):
    normalised_lookup = {
        str(column).strip().lower():
            column
        for column in completion_registry.columns
    }


    for alias in aliases:
        key = str(
            alias
        ).strip().lower()


        if key in normalised_lookup:
            return normalised_lookup[
                key
            ]


    return None


project_number_column = (
    find_registry_column([
        "ProjectNumber",
        "Project Number",
        "ProjectOrder",
        "Project Order",
        "Order",
        "Index",
    ])
)


project_name_column = (
    find_registry_column([
        "Project",
        "ProjectName",
        "Project Name",
        "DatasetProject",
    ])
)


project_slug_column = (
    find_registry_column([
        "ProjectSlug",
        "Project Slug",
        "Slug",
    ])
)


status_column = (
    find_registry_column([
        "Status",
        "CompletionStatus",
        "Completion Status",
        "RegistryStatus",
    ])
)


if status_column is None:
    raise RuntimeError(
        "The completion registry has no recognised "
        "status column.\n"
        f"Observed columns: "
        f"{completion_registry.columns.tolist()}"
    )


if (
    project_name_column is None
    and project_slug_column is None
    and project_number_column is None
):
    raise RuntimeError(
        "The completion registry has no recognised "
        "project identity column.\n"
        f"Observed columns: "
        f"{completion_registry.columns.tolist()}"
    )


def beast2_registry_mask(
    registry,
):
    mask = pd.Series(
        False,
        index=registry.index,
    )


    if project_number_column is not None:
        numeric_project_numbers = pd.to_numeric(
            registry[
                project_number_column
            ],
            errors="coerce",
        )


        mask = (
            mask
            |
            numeric_project_numbers.eq(
                PROJECT_NUMBER
            )
        )


    if project_name_column is not None:
        mask = (
            mask
            |
            registry[
                project_name_column
            ]
            .astype(str)
            .eq(
                PROJECT_NAME
            )
        )


    if project_slug_column is not None:
        mask = (
            mask
            |
            registry[
                project_slug_column
            ]
            .astype(str)
            .eq(
                PROJECT_SLUG
            )
        )


    return mask


existing_beast2_mask = beast2_registry_mask(
    completion_registry
)


existing_beast2_rows = int(
    existing_beast2_mask.sum()
)


if existing_beast2_rows > 1:
    raise AssertionError(
        "The completion registry contains duplicate "
        "Beast2 entries."
    )


other_registry_rows = (
    completion_registry[
        ~existing_beast2_mask
    ]
    .copy()
)


other_frozen_rows = int(
    other_registry_rows[
        status_column
    ]
    .astype(str)
    .eq(
        "COMPLETE_AND_FROZEN"
    )
    .sum()
)


if other_frozen_rows != 6:
    raise AssertionError(
        "Expected six previously frozen projects before "
        "freezing Beast2.\n"
        f"Observed previously frozen projects: "
        f"{other_frozen_rows}"
    )


# ---------------------------------------------------------
# 9. Build an adaptive Project 7 registry row
# ---------------------------------------------------------

completed_at = (
    pd.Timestamp.utcnow().isoformat()
)


new_registry_row = {
    column:
        pd.NA
    for column in completion_registry.columns
}


def set_registry_value(
    aliases,
    value,
):
    column = find_registry_column(
        aliases
    )


    if column is not None:
        new_registry_row[
            column
        ] = value


set_registry_value(
    [
        "ProjectNumber",
        "Project Number",
        "ProjectOrder",
        "Project Order",
        "Order",
        "Index",
    ],
    PROJECT_NUMBER,
)


set_registry_value(
    [
        "Project",
        "ProjectName",
        "Project Name",
        "DatasetProject",
    ],
    PROJECT_NAME,
)


set_registry_value(
    [
        "ProjectSlug",
        "Project Slug",
        "Slug",
    ],
    PROJECT_SLUG,
)


set_registry_value(
    [
        "Status",
        "CompletionStatus",
        "Completion Status",
        "RegistryStatus",
    ],
    "COMPLETE_AND_FROZEN",
)


registry_values = [
    (
        [
            "Conditions",
            "ConditionCount",
            "Condition Count",
            "ExperimentConditions",
        ],
        EXPECTED_RAW_CONDITIONS,
    ),
    (
        [
            "MLFits",
            "ML Fits",
            "ModelFits",
            "Model Fits",
        ],
        EXPECTED_ML_FITS,
    ),
    (
        [
            "RankingRows",
            "Ranking Rows",
        ],
        EXPECTED_RANKING_ROWS,
    ),
    (
        [
            "BuildMetricRows",
            "Build Metric Rows",
            "BuildMetricsRows",
        ],
        EXPECTED_BUILD_METRIC_ROWS,
    ),
    (
        [
            "ProjectRunRows",
            "Project Run Rows",
        ],
        EXPECTED_PROJECT_RUN_ROWS,
    ),
    (
        [
            "ManifestRows",
            "Manifest Rows",
            "NoiseManifestRows",
        ],
        EXPECTED_MANIFEST_ROWS,
    ),
    (
        [
            "RawFiles",
            "Raw Files",
            "RawFileCount",
        ],
        EXPECTED_RAW_FILES,
    ),
    (
        [
            "RawBytes",
            "Raw Bytes",
        ],
        EXPECTED_RAW_BYTES,
    ),
    (
        [
            "RawRootSHA256",
            "Raw Root SHA256",
            "RawSHA256",
        ],
        EXPECTED_RAW_ROOT_SHA256,
    ),
    (
        [
            "FinalPackageFiles",
            "Final Package Files",
            "PackageFiles",
            "Package File Count",
        ],
        EXPECTED_FINAL_PACKAGE_FILES,
    ),
    (
        [
            "FinalPackageBytes",
            "Final Package Bytes",
            "PackageBytes",
        ],
        EXPECTED_FINAL_PACKAGE_BYTES,
    ),
    (
        [
            "FinalPackageRootSHA256",
            "Final Package Root SHA256",
            "PackageRootSHA256",
            "PackageSHA256",
        ],
        EXPECTED_FINAL_PACKAGE_ROOT_SHA256,
    ),
    (
        [
            "FinalPackageDirectory",
            "Final Package Directory",
            "PackageDirectory",
        ],
        str(
            FINAL_PACKAGE_DIRECTORY
        ),
    ),
    (
        [
            "RawDirectory",
            "Raw Directory",
        ],
        str(
            RAW_DIRECTORY
        ),
    ),
    (
        [
            "CompletionCertificate",
            "Completion Certificate",
            "CertificatePath",
        ],
        str(
            COMPLETION_CERTIFICATE_PATH
        ),
    ),
    (
        [
            "CompletedAtUTC",
            "Completed At UTC",
            "CompletedAt",
            "CompletionTimestamp",
        ],
        completed_at,
    ),
    (
        [
            "NoiseLevels",
            "Noise Levels",
        ],
        ",".join(
            map(
                str,
                EXPECTED_NOISE_LEVELS,
            )
        ),
    ),
    (
        [
            "Seeds",
            "RepetitionSeeds",
            "Repetition Seeds",
        ],
        "1-30",
    ),
    (
        [
            "Techniques",
            "TechniqueCount",
            "Technique Count",
        ],
        len(
            EXPECTED_TECHNIQUES
        ),
    ),
    (
        [
            "EvaluationBuilds",
            "Evaluation Builds",
        ],
        52,
    ),
]


for aliases, value in registry_values:
    set_registry_value(
        aliases,
        value,
    )


# ---------------------------------------------------------
# 10. Append or verify the Project 7 registry row
# ---------------------------------------------------------

registry_already_frozen = False


if existing_beast2_rows == 1:
    existing_row = completion_registry.loc[
        existing_beast2_mask
    ].iloc[0]


    existing_status = str(
        existing_row[
            status_column
        ]
    )


    if existing_status == (
        "COMPLETE_AND_FROZEN"
    ):
        registry_already_frozen = True


        raw_hash_column = find_registry_column([
            "RawRootSHA256",
            "Raw Root SHA256",
            "RawSHA256",
        ])


        package_hash_column = find_registry_column([
            "FinalPackageRootSHA256",
            "Final Package Root SHA256",
            "PackageRootSHA256",
            "PackageSHA256",
        ])


        if raw_hash_column is not None:
            existing_raw_hash = str(
                existing_row[
                    raw_hash_column
                ]
            )


            if (
                existing_raw_hash not in {
                    "",
                    "nan",
                    "<NA>",
                }
                and
                existing_raw_hash
                != EXPECTED_RAW_ROOT_SHA256
            ):
                raise AssertionError(
                    "The existing frozen Beast2 registry "
                    "row has a different raw hash."
                )


        if package_hash_column is not None:
            existing_package_hash = str(
                existing_row[
                    package_hash_column
                ]
            )


            if (
                existing_package_hash not in {
                    "",
                    "nan",
                    "<NA>",
                }
                and
                existing_package_hash
                != EXPECTED_FINAL_PACKAGE_ROOT_SHA256
            ):
                raise AssertionError(
                    "The existing frozen Beast2 registry "
                    "row has a different package hash."
                )


        updated_registry = (
            completion_registry
            .copy()
            .reset_index(drop=True)
        )


    else:
        updated_registry = (
            completion_registry[
                ~existing_beast2_mask
            ]
            .copy()
            .reset_index(drop=True)
        )


        updated_registry = pd.concat(
            [
                updated_registry,
                pd.DataFrame([
                    new_registry_row
                ]),
            ],
            ignore_index=True,
        )


        atomic_write_csv(
            updated_registry,
            COMPLETION_REGISTRY_PATH,
        )


else:
    updated_registry = pd.concat(
        [
            completion_registry,
            pd.DataFrame([
                new_registry_row
            ]),
        ],
        ignore_index=True,
    )


    atomic_write_csv(
        updated_registry,
        COMPLETION_REGISTRY_PATH,
    )


# ---------------------------------------------------------
# 11. Reopen and verify the completion registry
# ---------------------------------------------------------

verified_registry = pd.read_csv(
    COMPLETION_REGISTRY_PATH
)


verified_registry.columns = [
    str(column)
    for column in verified_registry.columns
]


# Resolve columns again after reopening.
completion_registry = verified_registry


project_number_column = (
    find_registry_column([
        "ProjectNumber",
        "Project Number",
        "ProjectOrder",
        "Project Order",
        "Order",
        "Index",
    ])
)


project_name_column = (
    find_registry_column([
        "Project",
        "ProjectName",
        "Project Name",
        "DatasetProject",
    ])
)


project_slug_column = (
    find_registry_column([
        "ProjectSlug",
        "Project Slug",
        "Slug",
    ])
)


status_column = (
    find_registry_column([
        "Status",
        "CompletionStatus",
        "Completion Status",
        "RegistryStatus",
    ])
)


verified_beast2_mask = beast2_registry_mask(
    verified_registry
)


if int(
    verified_beast2_mask.sum()
) != 1:
    raise AssertionError(
        "The verified registry does not contain exactly "
        "one Beast2 row."
    )


verified_beast2_status = str(
    verified_registry.loc[
        verified_beast2_mask,
        status_column,
    ].iloc[0]
)


if verified_beast2_status != (
    "COMPLETE_AND_FROZEN"
):
    raise AssertionError(
        "The Beast2 registry row is not marked "
        "COMPLETE_AND_FROZEN."
    )


verified_frozen_count = int(
    verified_registry[
        status_column
    ]
    .astype(str)
    .eq(
        "COMPLETE_AND_FROZEN"
    )
    .sum()
)


if verified_frozen_count != 7:
    raise AssertionError(
        "Expected seven COMPLETE_AND_FROZEN projects.\n"
        f"Observed: {verified_frozen_count}"
    )


if len(
    verified_registry
) != 7:
    raise AssertionError(
        "Expected the completion registry to contain "
        "exactly seven project rows.\n"
        f"Observed: {len(verified_registry)}"
    )


registry_sha256 = sha256_file(
    COMPLETION_REGISTRY_PATH
)


# ---------------------------------------------------------
# 12. Create permanent completion certificate
# ---------------------------------------------------------

completion_certificate = {
    "CertificateName":
        "Project 7 completion and freeze certificate",

    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "COMPLETE_AND_FROZEN",

    "Protocol": {
        "Split":
            "fixed chronological 75/25",

        "NoiseLevelsPercent":
            EXPECTED_NOISE_LEVELS,

        "RepetitionSeeds":
            EXPECTED_SEEDS,

        "Conditions":
            EXPECTED_RAW_CONDITIONS,

        "MLModels":
            [
                "RandomForest",
                "XGBoost",
                "LightGBM",
                "NaiveBayes",
            ],

        "Baselines":
            [
                "Random",
                "LatestFail",
                "QTF-Avg",
            ],

        "PrimaryMetric":
            "APFDc",

        "SecondaryMetric":
            "APFD",

        "MetricUnit":
            "failure-test-execution event",
    },

    "ExecutionTotals": {
        "MLFits":
            EXPECTED_ML_FITS,

        "RankingRows":
            EXPECTED_RANKING_ROWS,

        "BuildMetricRows":
            EXPECTED_BUILD_METRIC_ROWS,

        "ProjectRunRows":
            EXPECTED_PROJECT_RUN_ROWS,

        "NoiseManifestRows":
            EXPECTED_MANIFEST_ROWS,
    },

    "RawResults": {
        "Directory":
            str(
                RAW_DIRECTORY
            ),

        "Files":
            EXPECTED_RAW_FILES,

        "Bytes":
            EXPECTED_RAW_BYTES,

        "HashAlgorithm":
            "SHA-256",

        "RootSHA256":
            EXPECTED_RAW_ROOT_SHA256,

        "Inventory":
            str(
                RAW_INVENTORY_PATH
            ),

        "AuditReport":
            str(
                RAW_AUDIT_REPORT_PATH
            ),
    },

    "FinalPackage": {
        "Directory":
            str(
                FINAL_PACKAGE_DIRECTORY
            ),

        "Files":
            EXPECTED_FINAL_PACKAGE_FILES,

        "Bytes":
            EXPECTED_FINAL_PACKAGE_BYTES,

        "CoreFiles":
            9,

        "TechniqueNoiseFiles":
            EXPECTED_TECHNIQUE_NOISE_FILES,

        "ProjectLevelNoiseTechniqueRows":
            EXPECTED_PROJECT_LEVEL_ROWS,

        "HashAlgorithm":
            "SHA-256",

        "RootSHA256":
            EXPECTED_FINAL_PACKAGE_ROOT_SHA256,

        "Inventory":
            str(
                FINAL_PACKAGE_INVENTORY_PATH
            ),

        "Report":
            str(
                FINAL_PACKAGE_REPORT_PATH
            ),

        "MissingFiles":
            missing_package_files,

        "UnexpectedFiles":
            unexpected_package_files,

        "SizeMismatches":
            package_size_mismatches,

        "HashMismatches":
            package_hash_mismatches,
    },

    "CompletionRegistry": {
        "Path":
            str(
                COMPLETION_REGISTRY_PATH
            ),

        "Rows":
            int(
                len(
                    verified_registry
                )
            ),

        "FrozenProjects":
            verified_frozen_count,

        "Beast2Rows":
            int(
                verified_beast2_mask.sum()
            ),

        "RegistryAlreadyFrozenBeforeThisCell":
            registry_already_frozen,

        "SHA256":
            registry_sha256,
    },

    "IntegrityValidation": {
        "RawInventoryRootReconstructed":
            True,

        "FinalPackageFilesRehashed":
            EXPECTED_FINAL_PACKAGE_FILES,

        "FinalPackageRootReconstructed":
            True,

        "FinalPackageInventoryMatched":
            True,

        "FinalPackageUnmodified":
            True,
    },

    "CompletedAtUTC":
        completed_at,
}


atomic_write_json(
    completion_certificate,
    COMPLETION_CERTIFICATE_PATH,
)


completion_certificate_sha256 = (
    sha256_file(
        COMPLETION_CERTIFICATE_PATH
    )
)


# ---------------------------------------------------------
# 13. Freeze the Project 7 checkpoint atomically
# ---------------------------------------------------------

with open(
    PROJECT_CHECKPOINT_PATH,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    frozen_checkpoint = json.load(
        checkpoint_file
    )


frozen_checkpoint.update({
    "Status":
        "COMPLETE_AND_FROZEN",

    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "CompletionRegistry":
        str(
            COMPLETION_REGISTRY_PATH
        ),

    "CompletionRegistryRows":
        int(
            len(
                verified_registry
            )
        ),

    "CompletionRegistryFrozenProjects":
        verified_frozen_count,

    "CompletionRegistrySHA256":
        registry_sha256,

    "CompletionCertificate":
        str(
            COMPLETION_CERTIFICATE_PATH
        ),

    "CompletionCertificateSHA256":
        completion_certificate_sha256,

    "FrozenRawRootSHA256":
        EXPECTED_RAW_ROOT_SHA256,

    "FrozenFinalPackageRootSHA256":
        EXPECTED_FINAL_PACKAGE_ROOT_SHA256,

    "FrozenRawFileCount":
        EXPECTED_RAW_FILES,

    "FrozenFinalPackageFileCount":
        EXPECTED_FINAL_PACKAGE_FILES,

    "FrozenAtUTC":
        completed_at,

    "UpdatedAtUTC":
        completed_at,
})


atomic_write_json(
    frozen_checkpoint,
    PROJECT_CHECKPOINT_PATH,
)


# ---------------------------------------------------------
# 14. Final independent verification
# ---------------------------------------------------------

with open(
    PROJECT_CHECKPOINT_PATH,
    "r",
    encoding="utf-8",
) as checkpoint_file:
    final_checkpoint = json.load(
        checkpoint_file
    )


if final_checkpoint.get(
    "Status"
) != "COMPLETE_AND_FROZEN":
    raise AssertionError(
        "The final Project 7 checkpoint is not frozen."
    )


if final_checkpoint.get(
    "FrozenRawRootSHA256"
) != EXPECTED_RAW_ROOT_SHA256:
    raise AssertionError(
        "The frozen checkpoint raw hash is incorrect."
    )


if final_checkpoint.get(
    "FrozenFinalPackageRootSHA256"
) != EXPECTED_FINAL_PACKAGE_ROOT_SHA256:
    raise AssertionError(
        "The frozen checkpoint package hash is "
        "incorrect."
    )


if int(
    final_checkpoint.get(
        "CompletionRegistryFrozenProjects",
        -1,
    )
) != 7:
    raise AssertionError(
        "The frozen checkpoint does not confirm seven "
        "completed projects."
    )


if sha256_file(
    COMPLETION_CERTIFICATE_PATH
) != completion_certificate_sha256:
    raise AssertionError(
        "The saved completion certificate hash changed."
    )


# ---------------------------------------------------------
# 15. Compact final output
# ---------------------------------------------------------

clear_output(
    wait=True
)


print(
    "=== PROJECT 7 STEP 11B RESULT ==="
)


print(
    "\nProject identity:"
)

print(
    "Project number:",
    PROJECT_NUMBER
)

print(
    "Project:",
    PROJECT_NAME
)

print(
    "Project slug:",
    PROJECT_SLUG
)


print(
    "\nRaw result freeze:"
)

print(
    "Conditions:",
    EXPECTED_RAW_CONDITIONS
)

print(
    "ML fits:",
    EXPECTED_ML_FITS
)

print(
    "Raw files:",
    len(
        ordered_raw_inventory
    )
)

print(
    "Raw bytes:",
    reconstructed_raw_bytes
)

print(
    "Raw root SHA-256:",
    reconstructed_raw_root
)


print(
    "\nFinal package freeze:"
)

print(
    "Package files:",
    len(
        actual_package_files
    )
)

print(
    "Package bytes:",
    reconstructed_package_bytes
)

print(
    "Missing package files:",
    missing_package_files
)

print(
    "Unexpected package files:",
    unexpected_package_files
)

print(
    "Package size mismatches:",
    package_size_mismatches
)

print(
    "Package SHA-256 mismatches:",
    package_hash_mismatches
)

print(
    "Final package root SHA-256:",
    reconstructed_package_root
)


print(
    "\nCompletion registry:"
)

print(
    "Registry rows:",
    len(
        verified_registry
    )
)

print(
    "COMPLETE_AND_FROZEN projects:",
    verified_frozen_count
)

print(
    "Beast2 registry rows:",
    int(
        verified_beast2_mask.sum()
    )
)

print(
    "Registry already frozen before this cell:",
    registry_already_frozen
)

print(
    "Registry SHA-256:",
    registry_sha256
)


print(
    "\nCompletion certificate:"
)

print(
    COMPLETION_CERTIFICATE_PATH
)

print(
    "Certificate SHA-256:",
    completion_certificate_sha256
)


print(
    "\nFinal checkpoint status:",
    final_checkpoint[
        "Status"
    ]
)


print(
    "\nValidation status:",
    "PASS"
)


print(
    "\nSUCCESS: Project 7 raw results were frozen with "
    "their verified SHA-256 root."
)

print(
    "SUCCESS: The 72-file final package was rehashed "
    "without modifying it."
)

print(
    "SUCCESS: Beast2 was added to the completed-project "
    "registry exactly once."
)

print(
    "SUCCESS: The registry now contains seven "
    "COMPLETE_AND_FROZEN projects."
)

print(
    "SUCCESS: Project 7 is COMPLETE_AND_FROZEN."
)

print(
    "SUCCESS: The thesis experiment is ready to begin "
    "Project 8."
)

=== PROJECT 7 STEP 11B RESULT ===

Project identity:
Project number: 7
Project: CompEvol@beast2
Project slug: CompEvol__beast2

Raw result freeze:
Conditions: 270
ML fits: 1080
Raw files: 2160
Raw bytes: 159516190
Raw root SHA-256: 93a6581d90895fd011b48490aa66f55a91b0078818bfd6966b3cf5b9e0f86373

Final package freeze:
Package files: 72
Package bytes: 1624833
Missing package files: 0
Unexpected package files: 0
Package size mismatches: 0
Package SHA-256 mismatches: 0
Final package root SHA-256: 93e9cc74fac017fef89b005752e1b7ede2a73f9665dd1840891e7f6d9c341c13

Completion registry:
Registry rows: 7
COMPLETE_AND_FROZEN projects: 7
Beast2 registry rows: 1
Registry already frozen before this cell: False
Registry SHA-256: 5f494b829fe17297f37979daf2022f012145c0783460d241752325be1b654250

Completion certificate:
/content/drive/MyDrive/Thesis_Experiment/Notes/project_07_completion_certificate.json
Certificate SHA-256: 9d1e54a113de55dd526180bb3906b6f48eb0717da8064cf053b397e3f8e64fd0

Final checkp

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# PROJECT 8 — STEP 1A
# SAFE PROJECT DISCOVERY AND IDENTITY VALIDATION
#
# READ-ONLY:
# - Does not create Project 8 directories
# - Does not modify the completion registry
# - Does not alter Projects 1–7
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import re

import pandas as pd


# ------------------------------------------------------------
# 1. CANONICAL THESIS PATHS
# ------------------------------------------------------------

THESIS_DRIVE = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

RESULTS_DRIVE = THESIS_DRIVE / "Results"
RAW_RESULTS_DRIVE = RESULTS_DRIVE / "Raw"
AGGREGATED_RESULTS_DRIVE = RESULTS_DRIVE / "Aggregated"
LOGS_DRIVE = RESULTS_DRIVE / "Logs"
NOTES_DRIVE = THESIS_DRIVE / "Notes"

PROJECT_NUMBER = 8


if not THESIS_DRIVE.exists():
    raise FileNotFoundError(
        f"Thesis experiment directory does not exist:\n"
        f"{THESIS_DRIVE}"
    )


print("=== PROJECT 8 STEP 1A: SAFE DISCOVERY ===")

print("\nThesis root:")
print(THESIS_DRIVE)

print("\nProject number:")
print(PROJECT_NUMBER)


# ------------------------------------------------------------
# 2. NORMALISATION HELPERS
# ------------------------------------------------------------

def normalise_text(value):
    """
    Produce a comparison-safe identifier.
    """

    if value is None:
        return ""

    text = str(value).strip().lower()

    if text in {
        "",
        "nan",
        "none",
        "null",
    }:
        return ""

    return re.sub(
        r"[^a-z0-9]+",
        "",
        text,
    )


def create_project_slug(project_name):
    """
    Use the same project-slug convention as the experiment.
    """

    return (
        str(project_name)
        .strip()
        .replace("@", "__")
        .replace("/", "__")
    )


def find_column(
    dataframe,
    exact_names=None,
    contains_terms=None,
):
    """
    Find a likely column without assuming exact capitalisation.
    """

    exact_names = {
        normalise_text(name)
        for name in (exact_names or [])
    }

    contains_terms = [
        normalise_text(term)
        for term in (contains_terms or [])
    ]

    normalised_columns = {
        column: normalise_text(column)
        for column in dataframe.columns
    }

    for column, normalised in normalised_columns.items():
        if normalised in exact_names:
            return column

    for column, normalised in normalised_columns.items():
        if any(
            term in normalised
            for term in contains_terms
        ):
            return column

    return None


def values_match(left, right):
    """
    Match project names, repository labels and project slugs.
    """

    left_normalised = normalise_text(left)
    right_normalised = normalise_text(right)

    if not left_normalised or not right_normalised:
        return False

    if left_normalised == right_normalised:
        return True

    # Allow a project name and its slug/path representation
    # to match when one contains the other.
    shorter_length = min(
        len(left_normalised),
        len(right_normalised),
    )

    return (
        shorter_length >= 6
        and (
            left_normalised in right_normalised
            or right_normalised in left_normalised
        )
    )


# ------------------------------------------------------------
# 3. LOCATE THE GLOBAL COMPLETION REGISTRY
# ------------------------------------------------------------

registry_search_roots = [
    AGGREGATED_RESULTS_DRIVE,
    NOTES_DRIVE,
    THESIS_DRIVE,
]

registry_candidates = set()

registry_filename_terms = [
    "registry",
    "completion",
    "complete",
    "freeze",
    "frozen",
]

for search_root in registry_search_roots:

    if not search_root.exists():
        continue

    for csv_path in search_root.rglob("*.csv"):

        path_text = str(csv_path).lower()

        # Raw condition output is not a global project registry.
        if "/results/raw/" in path_text:
            continue

        filename = csv_path.name.lower()

        if any(
            term in filename
            for term in registry_filename_terms
        ):
            registry_candidates.add(csv_path)


valid_registries = []

for registry_path in sorted(registry_candidates):

    try:
        registry_frame = pd.read_csv(
            registry_path,
            dtype=str,
        )
    except Exception:
        continue

    if registry_frame.empty:
        continue

    status_column = find_column(
        registry_frame,
        exact_names=[
            "Status",
            "CompletionStatus",
            "ProjectStatus",
            "FreezeStatus",
        ],
        contains_terms=[
            "status",
        ],
    )

    if status_column is None:
        continue

    status_values = (
        registry_frame[status_column]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    frozen_count = int(
        status_values.eq(
            "COMPLETE_AND_FROZEN"
        ).sum()
    )

    if frozen_count == 0:
        continue

    project_number_column = find_column(
        registry_frame,
        exact_names=[
            "ProjectNumber",
            "Project_Number",
            "Number",
        ],
        contains_terms=[
            "projectnumber",
        ],
    )

    project_name_column = find_column(
        registry_frame,
        exact_names=[
            "Project",
            "ProjectName",
            "Project_Name",
            "Repository",
            "Repo",
        ],
        contains_terms=[
            "projectname",
            "repository",
        ],
    )

    project_slug_column = find_column(
        registry_frame,
        exact_names=[
            "ProjectSlug",
            "Project_Slug",
            "Slug",
        ],
        contains_terms=[
            "projectslug",
        ],
    )

    # A usable global registry must identify projects.
    if (
        project_name_column is None
        and project_slug_column is None
    ):
        continue

    score = (
        frozen_count * 100
        + int(project_number_column is not None) * 25
        + int(project_name_column is not None) * 10
        + int(project_slug_column is not None) * 10
    )

    filename_lower = registry_path.name.lower()

    if "global" in filename_lower:
        score += 20

    if "project" in filename_lower:
        score += 10

    # Prefer the compact project-level registry over a
    # condition-level registry with hundreds of rows.
    if len(registry_frame) <= 50:
        score += 20

    valid_registries.append({
        "path": registry_path,
        "frame": registry_frame,
        "status_column": status_column,
        "project_number_column": project_number_column,
        "project_name_column": project_name_column,
        "project_slug_column": project_slug_column,
        "frozen_count": frozen_count,
        "score": score,
    })


if not valid_registries:
    raise FileNotFoundError(
        "Could not locate a CSV containing the status "
        "'COMPLETE_AND_FROZEN'.\n\n"
        "No files were modified. Check that the global "
        "completion registry still exists under "
        "Thesis_Experiment."
    )


valid_registries = sorted(
    valid_registries,
    key=lambda item: (
        item["score"],
        item["frozen_count"],
    ),
    reverse=True,
)

selected_registry = valid_registries[0]

COMPLETION_REGISTRY_PATH = selected_registry["path"]
completion_registry = selected_registry["frame"].copy()

STATUS_COLUMN = selected_registry["status_column"]
PROJECT_NUMBER_COLUMN = selected_registry[
    "project_number_column"
]
PROJECT_NAME_COLUMN = selected_registry[
    "project_name_column"
]
PROJECT_SLUG_COLUMN = selected_registry[
    "project_slug_column"
]


print("\nCompletion registry located:")
print(COMPLETION_REGISTRY_PATH)


# ------------------------------------------------------------
# 4. VALIDATE PROJECTS 1–7
# ------------------------------------------------------------

completion_registry["_NormalisedStatus"] = (
    completion_registry[STATUS_COLUMN]
    .astype(str)
    .str.strip()
    .str.upper()
)

frozen_registry = (
    completion_registry[
        completion_registry["_NormalisedStatus"]
        == "COMPLETE_AND_FROZEN"
    ]
    .copy()
)


if PROJECT_NUMBER_COLUMN is not None:

    frozen_registry["_ProjectNumberNumeric"] = (
        pd.to_numeric(
            frozen_registry[PROJECT_NUMBER_COLUMN],
            errors="coerce",
        )
    )

    frozen_registry = (
        frozen_registry
        .sort_values(
            "_ProjectNumberNumeric",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    frozen_project_numbers = (
        frozen_registry["_ProjectNumberNumeric"]
        .dropna()
        .astype(int)
        .tolist()
    )

    if frozen_project_numbers != list(range(1, 8)):
        raise AssertionError(
            "Expected frozen project numbers [1, 2, 3, "
            "4, 5, 6, 7], but found:\n"
            f"{frozen_project_numbers}"
        )


if len(frozen_registry) != 7:
    raise AssertionError(
        "Project 8 must begin with exactly seven frozen "
        f"projects. Found {len(frozen_registry)}."
    )


def registry_project_name(row):
    if PROJECT_NAME_COLUMN is not None:
        value = str(row[PROJECT_NAME_COLUMN]).strip()

        if normalise_text(value):
            return value

    if PROJECT_SLUG_COLUMN is not None:
        return str(row[PROJECT_SLUG_COLUMN]).strip()

    return ""


def registry_project_slug(row):
    if PROJECT_SLUG_COLUMN is not None:
        value = str(row[PROJECT_SLUG_COLUMN]).strip()

        if normalise_text(value):
            return value

    return create_project_slug(
        registry_project_name(row)
    )


frozen_project_names = [
    registry_project_name(row)
    for _, row in frozen_registry.iterrows()
]

frozen_project_slugs = [
    registry_project_slug(row)
    for _, row in frozen_registry.iterrows()
]


print("\nFrozen project validation:")
print("Registry rows:", len(completion_registry))
print("COMPLETE_AND_FROZEN projects:", len(frozen_registry))

frozen_display = pd.DataFrame({
    "ProjectNumber": list(range(1, 8)),
    "Project": frozen_project_names,
    "ProjectSlug": frozen_project_slugs,
})

display(frozen_display)


# Ensure there is no existing Project 8 row.
if PROJECT_NUMBER_COLUMN is not None:

    all_project_numbers = pd.to_numeric(
        completion_registry[PROJECT_NUMBER_COLUMN],
        errors="coerce",
    )

    existing_project_8_rows = int(
        all_project_numbers.eq(PROJECT_NUMBER).sum()
    )

else:
    existing_project_8_rows = 0


if existing_project_8_rows != 0:
    raise AssertionError(
        "The completion registry already contains a "
        f"Project {PROJECT_NUMBER} row. Do not begin a "
        "new Project 8 run until that row is inspected."
    )


print(
    "\nExisting Project 8 registry rows:",
    existing_project_8_rows,
)


# ------------------------------------------------------------
# 5. DISCOVER ALL AVAILABLE PROJECT DATASETS
# ------------------------------------------------------------

dataset_records = []

for dataset_csv in THESIS_DRIVE.rglob("dataset.csv"):

    path_text = str(dataset_csv).lower()

    # Results may contain copied/derived dataset files.
    # Only source datasets are eligible here.
    if "/results/" in path_text:
        continue

    project_directory = dataset_csv.parent

    builds_csv = project_directory / "builds.csv"
    exe_csv = project_directory / "exe.csv"

    if not builds_csv.exists():
        continue

    if not exe_csv.exists():
        continue

    path_parts = [
        part
        for part in project_directory.parts
        if normalise_text(part)
    ]

    dataset_records.append({
        "directory": project_directory,
        "dataset_csv": dataset_csv,
        "builds_csv": builds_csv,
        "exe_csv": exe_csv,
        "id_map_csv": project_directory / "id_map.csv",
        "directory_name": project_directory.name,
        "path_parts": path_parts,
    })


# Remove duplicate directories.
dataset_records_by_path = {
    str(record["directory"]): record
    for record in dataset_records
}

dataset_records = list(
    dataset_records_by_path.values()
)


if not dataset_records:
    raise FileNotFoundError(
        "No source project directories containing "
        "builds.csv, exe.csv and dataset.csv were found."
    )


print("\nSource project datasets discovered:")
print(len(dataset_records))


# ------------------------------------------------------------
# 6. FIND AN ORDERED PROJECT SELECTION / ELIGIBILITY SOURCE
# ------------------------------------------------------------

selection_search_roots = [
    THESIS_DRIVE,
    NOTES_DRIVE,
]

selection_filename_terms = [
    "project",
    "eligible",
    "eligibility",
    "selection",
    "selected",
    "order",
    "manifest",
    "candidate",
]

selection_candidates = set()

for search_root in selection_search_roots:

    if not search_root.exists():
        continue

    for csv_path in search_root.rglob("*.csv"):

        path_text = str(csv_path).lower()

        if "/results/raw/" in path_text:
            continue

        # Avoid reading individual full-run outputs.
        if "condition_" in csv_path.name.lower():
            continue

        if csv_path == COMPLETION_REGISTRY_PATH:
            continue

        if any(
            term in csv_path.name.lower()
            for term in selection_filename_terms
        ):
            selection_candidates.add(csv_path)


completed_identifiers = [
    *frozen_project_names,
    *frozen_project_slugs,
]


def sequence_match_details(values):
    """
    Find Projects 1–7 in order inside a candidate list.
    """

    cleaned_values = [
        str(value).strip()
        for value in values
        if normalise_text(value)
    ]

    matched_indices = []
    search_start = 0

    for completed_project in frozen_project_names:

        found_index = None

        for index in range(
            search_start,
            len(cleaned_values),
        ):
            if values_match(
                cleaned_values[index],
                completed_project,
            ):
                found_index = index
                break

        if found_index is None:
            return None

        matched_indices.append(found_index)
        search_start = found_index + 1

    # Select the first following item that is not one of
    # the already frozen projects.
    next_index = None

    for index in range(
        matched_indices[-1] + 1,
        len(cleaned_values),
    ):

        value = cleaned_values[index]

        already_completed = any(
            values_match(value, completed)
            for completed in completed_identifiers
        )

        if not already_completed:
            next_index = index
            break

    if next_index is None:
        return None

    return {
        "cleaned_values": cleaned_values,
        "matched_indices": matched_indices,
        "next_index": next_index,
        "next_value": cleaned_values[next_index],
    }


ordered_sources = []

for selection_path in sorted(selection_candidates):

    try:
        candidate_frame = pd.read_csv(
            selection_path,
            dtype=str,
        )
    except Exception:
        continue

    if len(candidate_frame) < 8:
        continue

    order_column = find_column(
        candidate_frame,
        exact_names=[
            "ProjectNumber",
            "ProjectOrder",
            "Order",
            "Rank",
            "Index",
        ],
        contains_terms=[
            "projectorder",
            "projectnumber",
        ],
    )

    if order_column is not None:
        numeric_order = pd.to_numeric(
            candidate_frame[order_column],
            errors="coerce",
        )

        if numeric_order.notna().sum() >= 8:
            candidate_frame = (
                candidate_frame
                .assign(_Order=numeric_order)
                .sort_values(
                    "_Order",
                    kind="mergesort",
                )
                .drop(columns=["_Order"])
                .reset_index(drop=True)
            )

    possible_project_columns = []

    for column in candidate_frame.columns:
        normalised_column = normalise_text(column)

        if any(
            term in normalised_column
            for term in [
                "project",
                "repository",
                "repo",
                "slug",
                "name",
            ]
        ):
            possible_project_columns.append(column)

    for project_column in possible_project_columns:

        values = (
            candidate_frame[project_column]
            .dropna()
            .astype(str)
            .tolist()
        )

        match_details = sequence_match_details(values)

        if match_details is None:
            continue

        prefix_matches = sum(
            1
            for expected, index in zip(
                frozen_project_names,
                match_details["matched_indices"],
            )
            if index < len(
                match_details["cleaned_values"]
            )
            and values_match(
                match_details[
                    "cleaned_values"
                ][index],
                expected,
            )
        )

        source_score = (
            prefix_matches * 100
            + int(order_column is not None) * 25
        )

        filename_lower = selection_path.name.lower()

        if "eligible" in filename_lower:
            source_score += 20

        if "selected" in filename_lower:
            source_score += 20

        if "order" in filename_lower:
            source_score += 10

        ordered_sources.append({
            "path": selection_path,
            "column": project_column,
            "order_column": order_column,
            "match_details": match_details,
            "score": source_score,
        })


# ------------------------------------------------------------
# 7. FALLBACK: CHECK WHETHER DATASET DIRECTORY ORDER MATCHES
# ------------------------------------------------------------

if ordered_sources:

    ordered_sources = sorted(
        ordered_sources,
        key=lambda item: item["score"],
        reverse=True,
    )

    selected_order_source = ordered_sources[0]

    PROJECT_NAME = selected_order_source[
        "match_details"
    ]["next_value"]

    PROJECT_ORDER_SOURCE_PATH = selected_order_source[
        "path"
    ]

    PROJECT_ORDER_SOURCE_COLUMN = selected_order_source[
        "column"
    ]

else:

    # Only use this fallback when alphabetical dataset order
    # reproduces Projects 1–7 exactly. Otherwise, stop rather
    # than guessing Project 8.
    alphabetic_dataset_records = sorted(
        dataset_records,
        key=lambda record: normalise_text(
            record["directory_name"]
        ),
    )

    alphabetic_names = [
        record["directory_name"]
        for record in alphabetic_dataset_records
    ]

    first_seven_match = (
        len(alphabetic_names) >= 8
        and all(
            values_match(
                alphabetic_names[index],
                frozen_project_names[index],
            )
            for index in range(7)
        )
    )

    if not first_seven_match:

        unresolved_projects = []

        for record in alphabetic_dataset_records:

            candidate_name = record["directory_name"]

            already_completed = any(
                values_match(
                    candidate_name,
                    completed,
                )
                for completed in completed_identifiers
            )

            if not already_completed:
                unresolved_projects.append({
                    "Candidate": candidate_name,
                    "Directory": str(
                        record["directory"]
                    ),
                })

        print(
            "\nNo trustworthy ordered project-selection "
            "file was found."
        )

        print(
            "\nUnfinished dataset candidates were found, "
            "but the cell will not guess which one is "
            "Project 8:"
        )

        display(
            pd.DataFrame(
                unresolved_projects
            ).head(30)
        )

        raise RuntimeError(
            "PROJECT 8 IDENTITY NOT YET RESOLVED.\n"
            "Copy the complete output of this cell back "
            "into the chat. No files were modified."
        )

    PROJECT_NAME = alphabetic_names[7]

    PROJECT_ORDER_SOURCE_PATH = (
        "Verified alphabetical dataset-directory order"
    )

    PROJECT_ORDER_SOURCE_COLUMN = (
        "directory_name"
    )


PROJECT_SLUG = create_project_slug(
    PROJECT_NAME
)


# ------------------------------------------------------------
# 8. LOCATE THE SELECTED PROJECT'S SOURCE DIRECTORY
# ------------------------------------------------------------

def dataset_match_score(
    project_name,
    record,
):
    target = normalise_text(project_name)

    possible_labels = [
        record["directory_name"],
        str(record["directory"]),
        *record["path_parts"],
    ]

    score = 0

    for label in possible_labels:
        label_normalised = normalise_text(label)

        if label_normalised == target:
            score = max(score, 100)

        elif (
            len(target) >= 6
            and target in label_normalised
        ):
            score = max(score, 80)

        elif (
            len(label_normalised) >= 6
            and label_normalised in target
        ):
            score = max(score, 70)

    return score


dataset_matches = []

for record in dataset_records:

    score = dataset_match_score(
        PROJECT_NAME,
        record,
    )

    if score > 0:
        dataset_matches.append({
            **record,
            "score": score,
        })


if not dataset_matches:
    raise FileNotFoundError(
        "Project 8 was identified as "
        f"'{PROJECT_NAME}', but its source dataset "
        "directory could not be matched."
    )


dataset_matches = sorted(
    dataset_matches,
    key=lambda item: item["score"],
    reverse=True,
)

highest_match_score = dataset_matches[0]["score"]

highest_matches = [
    match
    for match in dataset_matches
    if match["score"] == highest_match_score
]


if len(highest_matches) != 1:

    print(
        "\nMultiple equally strong dataset-directory "
        "matches were found:"
    )

    display(
        pd.DataFrame([
            {
                "Project": PROJECT_NAME,
                "Score": match["score"],
                "Directory": str(
                    match["directory"]
                ),
            }
            for match in highest_matches
        ])
    )

    raise RuntimeError(
        "PROJECT 8 DATASET DIRECTORY IS AMBIGUOUS.\n"
        "No files were modified."
    )


selected_dataset = highest_matches[0]

PROJECT_DATA_DIR = selected_dataset["directory"]

BUILDS_PATH = selected_dataset["builds_csv"]
EXE_PATH = selected_dataset["exe_csv"]
DATASET_PATH = selected_dataset["dataset_csv"]
ID_MAP_PATH = selected_dataset["id_map_csv"]


# ------------------------------------------------------------
# 9. FINAL SAFETY CHECKS
# ------------------------------------------------------------

already_completed = any(
    values_match(
        PROJECT_NAME,
        completed,
    )
    or values_match(
        PROJECT_SLUG,
        completed,
    )
    for completed in completed_identifiers
)


if already_completed:
    raise AssertionError(
        "The selected Project 8 identity matches an "
        "already frozen project."
    )


required_source_files = [
    BUILDS_PATH,
    EXE_PATH,
    DATASET_PATH,
]

missing_required_files = [
    path
    for path in required_source_files
    if not path.exists()
]


if missing_required_files:
    raise FileNotFoundError(
        "Project 8 is missing required source files:\n"
        + "\n".join(
            str(path)
            for path in missing_required_files
        )
    )


# Define future output locations only.
# Do not create them in this discovery step.
PROJECT_RAW_RESULTS = (
    RAW_RESULTS_DRIVE / PROJECT_SLUG
)

PROJECT_AGGREGATED_RESULTS = (
    AGGREGATED_RESULTS_DRIVE / PROJECT_SLUG
)

PROJECT_LOGS = (
    LOGS_DRIVE / PROJECT_SLUG
)


# ------------------------------------------------------------
# 10. PROJECT 8 STEP 1A RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 72)
print("=== PROJECT 8 STEP 1A RESULT ===")
print("=" * 72)

print("\nProject identity:")
print("Project number:", PROJECT_NUMBER)
print("Project:", PROJECT_NAME)
print("Project slug:", PROJECT_SLUG)

print("\nSelection source:")
print(PROJECT_ORDER_SOURCE_PATH)
print("Selection column:", PROJECT_ORDER_SOURCE_COLUMN)

print("\nCompletion registry:")
print(COMPLETION_REGISTRY_PATH)
print("Registry rows:", len(completion_registry))
print(
    "COMPLETE_AND_FROZEN projects:",
    len(frozen_registry),
)
print(
    "Existing Project 8 registry rows:",
    existing_project_8_rows,
)

print("\nSource dataset directory:")
print(PROJECT_DATA_DIR)

print("\nRequired input files:")
print("builds.csv:", BUILDS_PATH)
print("exe.csv:", EXE_PATH)
print("dataset.csv:", DATASET_PATH)
print(
    "id_map.csv:",
    ID_MAP_PATH if ID_MAP_PATH.exists()
    else "NOT PRESENT AT PROJECT ROOT",
)

print("\nFuture output locations — not created:")
print("Raw results:", PROJECT_RAW_RESULTS)
print(
    "Aggregated results:",
    PROJECT_AGGREGATED_RESULTS,
)
print("Logs:", PROJECT_LOGS)

print("\nSafety:")
print("Files written: 0")
print("Directories created: 0")
print("Previous projects modified: 0")

print(
    "\nSTATUS: "
    "PASS_PROJECT_8_IDENTITY_DISCOVERED_READ_ONLY"
)

print("=" * 72)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== PROJECT 8 STEP 1A: SAFE DISCOVERY ===

Thesis root:
/content/drive/MyDrive/Thesis_Experiment

Project number:
8

Completion registry located:
/content/drive/MyDrive/Thesis_Experiment/Notes/completed_project_registry.csv

Frozen project validation:
Registry rows: 7
COMPLETE_AND_FROZEN projects: 7


,ProjectNumber,Project,ProjectSlug
0,1,Angel-ML@angel,Angel-ML__angel
1,2,apache@airavata,apache__airavata
2,3,b2ihealthcare@snow-owl,b2ihealthcare__snow-owl
3,4,eclipse@paho.mqtt.java,eclipse__paho.mqtt.java
4,5,thinkaurelius@titan,thinkaurelius__titan
5,6,eclipse@jetty.project,eclipse__jetty.project
6,7,CompEvol@beast2,CompEvol__beast2



Existing Project 8 registry rows: 0


FileNotFoundError: No source project directories containing builds.csv, exe.csv and dataset.csv were found.

In [ ]:
# ============================================================
# PROJECT 8 — SOURCE DATASET LOCATION DIAGNOSTIC
#
# READ-ONLY:
# - Searches Google Drive
# - Writes nothing
# - Modifies nothing
# ============================================================

from pathlib import Path
import os
import zipfile
import pandas as pd


MY_DRIVE = Path("/content/drive/MyDrive")

TARGET_FILENAMES = {
    "builds.csv",
    "exe.csv",
    "dataset.csv",
    "id_map.csv",
}

SKIP_DIRECTORY_NAMES = {
    ".git",
    "__pycache__",
    "node_modules",
    ".ipynb_checkpoints",
}


print("=== PROJECT 8 SOURCE DATASET DIAGNOSTIC ===")
print("Search root:", MY_DRIVE)


# ------------------------------------------------------------
# 1. SEARCH FOR THE REQUIRED CSV FILES
# ------------------------------------------------------------

found_files = []

for current_root, directory_names, filenames in os.walk(MY_DRIVE):

    directory_names[:] = [
        directory_name
        for directory_name in directory_names
        if directory_name not in SKIP_DIRECTORY_NAMES
    ]

    current_path = Path(current_root)

    for filename in filenames:

        if filename.lower() in TARGET_FILENAMES:

            file_path = current_path / filename

            found_files.append({
                "Filename": filename,
                "Directory": str(current_path),
                "FullPath": str(file_path),
                "SizeBytes": (
                    file_path.stat().st_size
                    if file_path.exists()
                    else None
                ),
            })


found_files_frame = pd.DataFrame(found_files)

print("\nRequired CSV files found:", len(found_files_frame))

if not found_files_frame.empty:

    found_files_frame = (
        found_files_frame
        .sort_values(
            ["Directory", "Filename"],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    display(found_files_frame)

else:

    print("No matching CSV files found directly in MyDrive.")


# ------------------------------------------------------------
# 2. FIND DIRECTORIES CONTAINING ALL CORE FILES
# ------------------------------------------------------------

core_filenames = {
    "builds.csv",
    "exe.csv",
    "dataset.csv",
}

complete_dataset_directories = []

if not found_files_frame.empty:

    grouped_files = (
        found_files_frame
        .groupby("Directory")["Filename"]
        .apply(
            lambda values: {
                str(value).lower()
                for value in values
            }
        )
    )

    for directory, filenames in grouped_files.items():

        if core_filenames.issubset(filenames):

            complete_dataset_directories.append({
                "Directory": directory,
                "HasBuilds": True,
                "HasExe": True,
                "HasDataset": True,
                "HasIdMap": "id_map.csv" in filenames,
            })


complete_frame = pd.DataFrame(
    complete_dataset_directories
)

print(
    "\nDirectories containing builds.csv, "
    "exe.csv and dataset.csv:",
    len(complete_frame),
)

if not complete_frame.empty:

    complete_frame = (
        complete_frame
        .sort_values("Directory")
        .reset_index(drop=True)
    )

    display(complete_frame)

else:

    print(
        "No directory contains all three files together."
    )


# ------------------------------------------------------------
# 3. SEARCH ZIP ARCHIVES FOR THE DATASET FILES
# ------------------------------------------------------------

archive_records = []

for current_root, directory_names, filenames in os.walk(MY_DRIVE):

    directory_names[:] = [
        directory_name
        for directory_name in directory_names
        if directory_name not in SKIP_DIRECTORY_NAMES
    ]

    for filename in filenames:

        if not filename.lower().endswith(".zip"):
            continue

        archive_path = Path(current_root) / filename

        try:

            with zipfile.ZipFile(
                archive_path,
                "r",
            ) as archive:

                archive_names = archive.namelist()

                matching_members = [
                    member
                    for member in archive_names
                    if Path(member).name.lower()
                    in TARGET_FILENAMES
                ]

                if matching_members:

                    archive_records.append({
                        "Archive": str(archive_path),
                        "MatchingFileCount": len(
                            matching_members
                        ),
                        "MatchingMembers": "\n".join(
                            matching_members[:30]
                        ),
                    })

        except (
            zipfile.BadZipFile,
            PermissionError,
            OSError,
        ):
            continue


archive_frame = pd.DataFrame(archive_records)

print(
    "\nZIP archives containing relevant files:",
    len(archive_frame),
)

if not archive_frame.empty:

    display(
        archive_frame.sort_values(
            "Archive"
        ).reset_index(drop=True)
    )

else:

    print(
        "No ZIP archive containing the required "
        "dataset filenames was found."
    )


# ------------------------------------------------------------
# 4. SEARCH FOR PROJECT-SELECTION / ELIGIBILITY FILES
# ------------------------------------------------------------

selection_terms = {
    "eligible",
    "eligibility",
    "project_order",
    "project_selection",
    "selected_projects",
    "project_manifest",
    "projects",
}

selection_records = []

for current_root, directory_names, filenames in os.walk(MY_DRIVE):

    directory_names[:] = [
        directory_name
        for directory_name in directory_names
        if directory_name not in SKIP_DIRECTORY_NAMES
    ]

    for filename in filenames:

        filename_lower = filename.lower()

        if not filename_lower.endswith(
            (".csv", ".json", ".txt")
        ):
            continue

        if any(
            term in filename_lower
            for term in selection_terms
        ):

            file_path = Path(current_root) / filename

            selection_records.append({
                "Filename": filename,
                "FullPath": str(file_path),
            })


selection_frame = pd.DataFrame(selection_records)

print(
    "\nPossible project-order or eligibility files:",
    len(selection_frame),
)

if not selection_frame.empty:

    display(
        selection_frame
        .sort_values("FullPath")
        .reset_index(drop=True)
    )

else:

    print(
        "No obvious project-selection file was found."
    )


# ------------------------------------------------------------
# 5. FINAL RESULT
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("=== PROJECT 8 SOURCE DISCOVERY RESULT ===")
print("=" * 72)

print(
    "Relevant individual CSV files:",
    len(found_files_frame),
)

print(
    "Complete dataset directories:",
    len(complete_frame),
)

print(
    "Relevant ZIP archives:",
    len(archive_frame),
)

print(
    "Possible selection files:",
    len(selection_frame),
)

print("\nFiles written: 0")
print("Directories created: 0")
print("Registry modified: 0")
print("Projects 1–7 modified: 0")

if len(complete_frame) > 0:

    print(
        "\nSTATUS: "
        "PASS_SOURCE_DATASET_DIRECTORIES_DISCOVERED"
    )

elif len(archive_frame) > 0:

    print(
        "\nSTATUS: "
        "PASS_SOURCE_DATASET_ARCHIVE_DISCOVERED"
    )

else:

    print(
        "\nSTATUS: "
        "SOURCE_DATASET_LOCATION_NOT_YET_FOUND"
    )

print("=" * 72)

=== PROJECT 8 SOURCE DATASET DIAGNOSTIC ===
Search root: /content/drive/MyDrive

Required CSV files found: 0
No matching CSV files found directly in MyDrive.

Directories containing builds.csv, exe.csv and dataset.csv: 0
No directory contains all three files together.

ZIP archives containing relevant files: 0
No ZIP archive containing the required dataset filenames was found.

Possible project-order or eligibility files: 0
No obvious project-selection file was found.

=== PROJECT 8 SOURCE DISCOVERY RESULT ===
Relevant individual CSV files: 0
Complete dataset directories: 0
Relevant ZIP archives: 0
Possible selection files: 0

Files written: 0
Directories created: 0
Registry modified: 0
Projects 1–7 modified: 0

STATUS: SOURCE_DATASET_LOCATION_NOT_YET_FOUND


In [ ]:
# ============================================================
# PROJECT 8 — STEP 0A
# INSPECT THE OFFICIAL TCP-CI ZENODO DATASET
#
# READ-ONLY:
# - Downloads no large dataset files
# - Creates no project directories
# - Modifies no registry rows
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import json
import requests
import pandas as pd


ZENODO_RECORD_ID = 6415365

ZENODO_API_URL = (
    f"https://zenodo.org/api/records/"
    f"{ZENODO_RECORD_ID}"
)


def human_readable_size(number_of_bytes):
    number_of_bytes = float(number_of_bytes)

    units = [
        "B",
        "KB",
        "MB",
        "GB",
        "TB",
    ]

    for unit in units:
        if number_of_bytes < 1024 or unit == units[-1]:
            return f"{number_of_bytes:,.2f} {unit}"

        number_of_bytes /= 1024


print("=== PROJECT 8 STEP 0A: OFFICIAL DATASET INSPECTION ===")

print("\nZenodo record:")
print(ZENODO_RECORD_ID)

print("\nRequesting metadata from:")
print(ZENODO_API_URL)


response = requests.get(
    ZENODO_API_URL,
    timeout=120,
)

response.raise_for_status()

record = response.json()


metadata = record.get(
    "metadata",
    {}
)

files = record.get(
    "files",
    []
)


print("\nRecord information:")

print(
    "Title:",
    metadata.get(
        "title",
        "NOT PROVIDED",
    ),
)

print(
    "Publication date:",
    metadata.get(
        "publication_date",
        "NOT PROVIDED",
    ),
)

print(
    "Version:",
    metadata.get(
        "version",
        "NOT PROVIDED",
    ),
)

print(
    "DOI:",
    metadata.get(
        "doi",
        record.get(
            "doi",
            "10.5281/zenodo.6415365",
        ),
    ),
)


if not files:
    raise RuntimeError(
        "The Zenodo record was found, but it did not "
        "return any downloadable files."
    )


file_records = []

for file_entry in files:

    links = file_entry.get(
        "links",
        {},
    )

    download_url = (
        links.get("content")
        or links.get("download")
        or links.get("self")
    )

    file_name = (
        file_entry.get("key")
        or file_entry.get("filename")
        or "UNKNOWN_FILENAME"
    )

    file_size = int(
        file_entry.get(
            "size",
            0,
        )
    )

    file_records.append({
        "FileName": file_name,
        "SizeBytes": file_size,
        "ReadableSize":
            human_readable_size(file_size),
        "Checksum":
            file_entry.get(
                "checksum",
                "",
            ),
        "DownloadURL":
            download_url,
    })


files_frame = pd.DataFrame(
    file_records
)

files_frame = (
    files_frame
    .sort_values(
        "FileName",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


total_size = int(
    files_frame["SizeBytes"].sum()
)


print("\nFiles in official Zenodo record:")
display(
    files_frame[
        [
            "FileName",
            "ReadableSize",
            "Checksum",
        ]
    ]
)


print("\nFile count:")
print(len(files_frame))

print("\nTotal download size:")
print(human_readable_size(total_size))


archive_link = (
    record
    .get("links", {})
    .get("archive")
)

print("\nZenodo archive link available:")
print(bool(archive_link))


print("\n" + "=" * 72)
print("=== PROJECT 8 STEP 0A RESULT ===")
print("=" * 72)

print(
    "Zenodo record:",
    ZENODO_RECORD_ID,
)

print(
    "Official files:",
    len(files_frame),
)

print(
    "Total size:",
    human_readable_size(total_size),
)

print("Files downloaded: 0")
print("Directories created: 0")
print("Registry modified: 0")
print("Projects 1–7 modified: 0")

print(
    "\nSTATUS: "
    "PASS_OFFICIAL_TCP_CI_METADATA_DISCOVERED"
)

print("=" * 72)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== PROJECT 8 STEP 0A: OFFICIAL DATASET INSPECTION ===

Zenodo record:
6415365

Requesting metadata from:
https://zenodo.org/api/records/6415365

Record information:
Title: Scalable and Accurate Test Case Prioritization in Continuous Integration Contexts
Publication date: 2022-04-05
Version: NOT PROVIDED
DOI: 10.5281/zenodo.6415365

Files in official Zenodo record:


,FileName,ReadableSize,Checksum
0,TCP-CI-dataset.tar.gz,15.07 GB,md5:bcc24a1d6aecc9a3731511e4d8721d0e
1,TCP-CI-main-dataset.tar.gz,226.12 MB,md5:728804085c757ff5357aa165b4b6384f



File count:
2

Total download size:
15.29 GB

Zenodo archive link available:
True

=== PROJECT 8 STEP 0A RESULT ===
Zenodo record: 6415365
Official files: 2
Total size: 15.29 GB
Files downloaded: 0
Directories created: 0
Registry modified: 0
Projects 1–7 modified: 0

STATUS: PASS_OFFICIAL_TCP_CI_METADATA_DISCOVERED


In [ ]:
# ============================================================
# PROJECT 8 — RUNTIME RESTORE AND PROJECT SELECTION
#
# PURPOSE:
# 1. Reuse the existing 226 MB archive from Google Drive
# 2. Verify its official MD5
# 3. Extract it into fast temporary Colab storage
# 4. Read the existing screening CSV
# 5. Read the completed-project registry
# 6. Select the next unfinished eligible project as Project 8
#
# GOOGLE DRIVE WRITES: 0
# DOWNLOADS: 0
# PROJECTS 1–7 MODIFIED: 0
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from IPython.display import display

import hashlib
import json
import os
import re
import shutil
import tarfile

import pandas as pd


# ------------------------------------------------------------
# 1. EXISTING THESIS PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

ARCHIVE_PATH = (
    THESIS_ROOT
    / "Data"
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)

SCREENING_PATH = (
    THESIS_ROOT
    / "Data"
    / "processed"
    / "tcp_ci_project_screening.csv"
)

REGISTRY_PATH = (
    THESIS_ROOT
    / "Notes"
    / "completed_project_registry.csv"
)

# Extract to temporary local Colab storage for speed.
# This folder disappears when the runtime resets, but the
# original archive remains permanently in Google Drive.
LOCAL_EXTRACTION_ROOT = Path(
    "/content/TCP-CI-main-dataset"
)

EXTRACTION_MARKER = (
    LOCAL_EXTRACTION_ROOT
    / ".extraction_complete.json"
)

EXPECTED_ARCHIVE_MD5 = (
    "728804085c757ff5357aa165b4b6384f"
)

PROJECT_NUMBER = 8


print("=" * 76)
print("=== PROJECT 8: RUNTIME RESTORE AND PROJECT SELECTION ===")
print("=" * 76)


# ------------------------------------------------------------
# 2. BASIC HELPERS
# ------------------------------------------------------------

def normalise_text(value):
    if value is None:
        return ""

    text = str(value).strip().lower()

    if text in {
        "",
        "nan",
        "none",
        "null",
    }:
        return ""

    return re.sub(
        r"[^a-z0-9]+",
        "",
        text,
    )


def values_match(left, right):
    left_normalised = normalise_text(left)
    right_normalised = normalise_text(right)

    if not left_normalised or not right_normalised:
        return False

    if left_normalised == right_normalised:
        return True

    shorter_length = min(
        len(left_normalised),
        len(right_normalised),
    )

    return (
        shorter_length >= 6
        and (
            left_normalised in right_normalised
            or right_normalised in left_normalised
        )
    )


def find_column(
    dataframe,
    exact_names=None,
    contains_terms=None,
):
    exact_names = {
        normalise_text(name)
        for name in (exact_names or [])
    }

    contains_terms = [
        normalise_text(term)
        for term in (contains_terms or [])
    ]

    for column in dataframe.columns:
        normalised_column = normalise_text(column)

        if normalised_column in exact_names:
            return column

    for column in dataframe.columns:
        normalised_column = normalise_text(column)

        if any(
            term in normalised_column
            for term in contains_terms
        ):
            return column

    return None


def calculate_md5(
    file_path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.md5()

    with Path(file_path).open("rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def human_size(number_of_bytes):
    size = float(number_of_bytes)

    for unit in [
        "B",
        "KB",
        "MB",
        "GB",
        "TB",
    ]:
        if size < 1024 or unit == "TB":
            return f"{size:,.2f} {unit}"

        size /= 1024


def safe_extract_tar(
    archive_path,
    destination,
):
    destination = Path(destination)
    destination_resolved = destination.resolve()

    with tarfile.open(
        archive_path,
        mode="r:gz",
    ) as archive:

        members = archive.getmembers()

        total_uncompressed_bytes = sum(
            member.size
            for member in members
            if member.isfile()
        )

        print(
            "Archive members:",
            len(members),
        )

        print(
            "Expected extracted size:",
            human_size(total_uncompressed_bytes),
        )

        free_bytes = shutil.disk_usage(
            "/content"
        ).free

        print(
            "Available Colab disk:",
            human_size(free_bytes),
        )

        if total_uncompressed_bytes > free_bytes:
            raise RuntimeError(
                "The Colab runtime does not have enough "
                "free disk space for extraction."
            )

        for member in members:
            target_path = (
                destination
                / member.name
            ).resolve()

            allowed_prefix = (
                str(destination_resolved)
                + os.sep
            )

            if (
                target_path != destination_resolved
                and not str(target_path).startswith(
                    allowed_prefix
                )
            ):
                raise RuntimeError(
                    "Unsafe path detected in archive:\n"
                    f"{member.name}"
                )

            if member.issym() or member.islnk():
                raise RuntimeError(
                    "Archive contains an unsupported "
                    "symbolic or hard link:\n"
                    f"{member.name}"
                )

        archive.extractall(
            path=destination,
            members=members,
        )


# ------------------------------------------------------------
# 3. VALIDATE EXISTING PERMANENT FILES
# ------------------------------------------------------------

required_permanent_files = [
    ARCHIVE_PATH,
    SCREENING_PATH,
    REGISTRY_PATH,
]

missing_permanent_files = [
    path
    for path in required_permanent_files
    if not path.exists()
]

if missing_permanent_files:
    raise FileNotFoundError(
        "Required thesis files are missing:\n"
        + "\n".join(
            str(path)
            for path in missing_permanent_files
        )
    )


print("\nPermanent files located:")

print("\nArchive:")
print(ARCHIVE_PATH)

print("\nScreening CSV:")
print(SCREENING_PATH)

print("\nCompletion registry:")
print(REGISTRY_PATH)

print("\nExisting archive size:")
print(
    human_size(
        ARCHIVE_PATH.stat().st_size
    )
)


# ------------------------------------------------------------
# 4. VERIFY THE EXISTING ARCHIVE
# ------------------------------------------------------------

print("\nCalculating archive MD5...")
print("This reads the existing Drive file; it does not download it.")

archive_md5 = calculate_md5(
    ARCHIVE_PATH
)

print("\nExpected MD5:")
print(EXPECTED_ARCHIVE_MD5)

print("\nActual MD5:")
print(archive_md5)

if archive_md5.lower() != EXPECTED_ARCHIVE_MD5.lower():
    raise RuntimeError(
        "The existing archive failed MD5 validation.\n"
        "Do not extract it until the file is repaired."
    )

print("\nArchive integrity: PASS")


# ------------------------------------------------------------
# 5. RESTORE TEMPORARY EXTRACTION
# ------------------------------------------------------------

marker_valid = False

if EXTRACTION_MARKER.exists():
    try:
        marker_data = json.loads(
            EXTRACTION_MARKER.read_text(
                encoding="utf-8"
            )
        )

        marker_valid = (
            marker_data.get("archive_md5")
            == EXPECTED_ARCHIVE_MD5
            and marker_data.get("status")
            == "COMPLETE"
        )

    except Exception:
        marker_valid = False


if marker_valid:
    print("\nA valid runtime extraction already exists.")
    print("Extraction skipped.")

else:
    print(
        "\nNo valid runtime extraction exists."
    )

    if LOCAL_EXTRACTION_ROOT.exists():
        print(
            "Removing incomplete local extraction:"
        )
        print(LOCAL_EXTRACTION_ROOT)

        shutil.rmtree(
            LOCAL_EXTRACTION_ROOT
        )

    LOCAL_EXTRACTION_ROOT.mkdir(
        parents=True,
        exist_ok=False,
    )

    print("\nExtracting existing Drive archive...")
    print("No internet download is being performed.")

    safe_extract_tar(
        archive_path=ARCHIVE_PATH,
        destination=LOCAL_EXTRACTION_ROOT,
    )

    EXTRACTION_MARKER.write_text(
        json.dumps(
            {
                "archive_path": str(
                    ARCHIVE_PATH
                ),
                "archive_md5": archive_md5,
                "status": "COMPLETE",
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    print("\nExtraction completed successfully.")


# ------------------------------------------------------------
# 6. DISCOVER PROJECT DATASET DIRECTORIES
# ------------------------------------------------------------

core_filenames = {
    "builds.csv",
    "exe.csv",
    "dataset.csv",
}

project_records = []

for current_root, directory_names, filenames in os.walk(
    LOCAL_EXTRACTION_ROOT
):
    filename_lookup = {
        filename.lower(): filename
        for filename in filenames
    }

    if not core_filenames.issubset(
        filename_lookup.keys()
    ):
        continue

    project_directory = Path(current_root)

    id_map_path = (
        project_directory
        / filename_lookup.get(
            "id_map.csv",
            "id_map.csv",
        )
    )

    entity_history_filename = next(
        (
            filename_lookup[candidate]
            for candidate in [
                "entity_change_history.csv",
                "entity-history.csv",
                "entity_history.csv",
            ]
            if candidate in filename_lookup
        ),
        None,
    )

    entity_history_path = (
        project_directory
        / entity_history_filename
        if entity_history_filename
        else None
    )

    project_records.append({
        "ProjectDirectoryName":
            project_directory.name,

        "Directory":
            str(project_directory),

        "BuildsPath":
            str(
                project_directory
                / filename_lookup["builds.csv"]
            ),

        "ExePath":
            str(
                project_directory
                / filename_lookup["exe.csv"]
            ),

        "DatasetPath":
            str(
                project_directory
                / filename_lookup["dataset.csv"]
            ),

        "HasIdMap":
            id_map_path.exists(),

        "IdMapPath":
            (
                str(id_map_path)
                if id_map_path.exists()
                else ""
            ),

        "HasEntityChangeHistory":
            (
                entity_history_path is not None
                and entity_history_path.exists()
            ),

        "EntityHistoryPath":
            (
                str(entity_history_path)
                if entity_history_path is not None
                else ""
            ),
    })


project_frame = pd.DataFrame(
    project_records
)

if project_frame.empty:
    raise RuntimeError(
        "The archive extracted successfully, but no "
        "directories containing builds.csv, exe.csv and "
        "dataset.csv were discovered."
    )

project_frame = (
    project_frame
    .drop_duplicates(
        subset=["Directory"]
    )
    .sort_values(
        "Directory",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


print("\nComplete project datasets discovered:")
print(len(project_frame))

display(
    project_frame[
        [
            "ProjectDirectoryName",
            "HasIdMap",
            "HasEntityChangeHistory",
            "Directory",
        ]
    ]
)


# ------------------------------------------------------------
# 7. LOAD AND VALIDATE THE COMPLETION REGISTRY
# ------------------------------------------------------------

registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
)

registry_status_column = find_column(
    registry,
    exact_names=[
        "Status",
        "CompletionStatus",
        "ProjectStatus",
    ],
    contains_terms=[
        "status",
    ],
)

registry_project_column = find_column(
    registry,
    exact_names=[
        "Project",
        "ProjectName",
        "Repository",
        "Repo",
    ],
    contains_terms=[
        "projectname",
        "repository",
    ],
)

registry_slug_column = find_column(
    registry,
    exact_names=[
        "ProjectSlug",
        "Slug",
    ],
    contains_terms=[
        "projectslug",
    ],
)

registry_number_column = find_column(
    registry,
    exact_names=[
        "ProjectNumber",
        "Project_Number",
    ],
    contains_terms=[
        "projectnumber",
    ],
)


if registry_status_column is None:
    raise RuntimeError(
        "Could not identify the status column in the "
        "completion registry."
    )


frozen_registry = registry[
    registry[registry_status_column]
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("COMPLETE_AND_FROZEN")
].copy()


if len(frozen_registry) != 7:
    raise AssertionError(
        "Expected exactly seven COMPLETE_AND_FROZEN "
        f"projects, but found {len(frozen_registry)}."
    )


completed_identifiers = []

if registry_project_column is not None:
    completed_identifiers.extend(
        frozen_registry[
            registry_project_column
        ]
        .dropna()
        .astype(str)
        .tolist()
    )

if registry_slug_column is not None:
    completed_identifiers.extend(
        frozen_registry[
            registry_slug_column
        ]
        .dropna()
        .astype(str)
        .tolist()
    )


print("\nCompletion registry validation:")
print("Registry rows:", len(registry))
print(
    "COMPLETE_AND_FROZEN projects:",
    len(frozen_registry),
)


# ------------------------------------------------------------
# 8. LOAD THE EXISTING PROJECT-SCREENING FILE
# ------------------------------------------------------------

screening = pd.read_csv(
    SCREENING_PATH,
    dtype=str,
)

print("\nScreening file:")
print("Rows:", len(screening))
print("Columns:", list(screening.columns))


# ------------------------------------------------------------
# 9. IDENTIFY THE PROJECT-NAME COLUMN
# ------------------------------------------------------------

dataset_directory_names = (
    project_frame[
        "ProjectDirectoryName"
    ]
    .astype(str)
    .tolist()
)


candidate_project_columns = [
    column
    for column in screening.columns
    if any(
        term in normalise_text(column)
        for term in [
            "project",
            "repository",
            "repo",
            "subject",
            "system",
            "name",
        ]
    )
]

if not candidate_project_columns:
    candidate_project_columns = list(
        screening.columns
    )


column_match_scores = []

for column in candidate_project_columns:
    score = 0

    for value in screening[column].dropna():
        if any(
            values_match(
                value,
                directory_name,
            )
            for directory_name
            in dataset_directory_names
        ):
            score += 1

    column_match_scores.append({
        "Column": column,
        "DatasetMatches": score,
    })


column_score_frame = pd.DataFrame(
    column_match_scores
).sort_values(
    "DatasetMatches",
    ascending=False,
    kind="mergesort",
)


print("\nScreening project-column match scores:")
display(column_score_frame)


if (
    column_score_frame.empty
    or int(
        column_score_frame.iloc[0][
            "DatasetMatches"
        ]
    ) == 0
):
    raise RuntimeError(
        "No screening column could be matched to the "
        "extracted project directories."
    )


screening_project_column = (
    column_score_frame.iloc[0]["Column"]
)

print(
    "\nSelected screening project column:",
    screening_project_column,
)


# ------------------------------------------------------------
# 10. APPLY ELIGIBILITY FILTER WHEN AVAILABLE
# ------------------------------------------------------------

eligibility_column = find_column(
    screening,
    exact_names=[
        "Eligible",
        "Eligibility",
        "Selected",
        "Included",
        "Decision",
        "SelectionStatus",
    ],
    contains_terms=[
        "eligible",
        "eligibility",
        "selected",
        "included",
        "decision",
        "selectionstatus",
    ],
)


eligible_screening = screening.copy()

if eligibility_column is not None:
    eligibility_values = (
        screening[eligibility_column]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    positive_values = {
        "1",
        "true",
        "yes",
        "y",
        "eligible",
        "selected",
        "included",
        "include",
        "pass",
        "passed",
        "accepted",
        "keep",
    }

    negative_terms = [
        "ineligible",
        "not eligible",
        "excluded",
        "exclude",
        "rejected",
        "reject",
        "failed",
        "fail",
        "false",
        "no",
    ]

    positive_mask = eligibility_values.isin(
        positive_values
    )

    negative_mask = eligibility_values.apply(
        lambda value: any(
            term == value
            or term in value
            for term in negative_terms
        )
    )

    eligibility_mask = (
        positive_mask
        & ~negative_mask
    )

    if eligibility_mask.sum() > 0:
        eligible_screening = screening[
            eligibility_mask
        ].copy()

        print(
            "\nEligibility column:",
            eligibility_column,
        )

        print(
            "Eligible screening rows:",
            len(eligible_screening),
        )

    else:
        print(
            "\nEligibility-like column found, but its "
            "values were not safely interpretable:"
        )

        print(
            screening[eligibility_column]
            .value_counts(
                dropna=False
            )
        )

        raise RuntimeError(
            "Eligibility values require inspection before "
            "selecting Project 8."
        )

else:
    print(
        "\nNo explicit eligibility column found."
    )

    print(
        "The existing screening row order will be used."
    )


# ------------------------------------------------------------
# 11. PRESERVE SCREENING ORDER OR EXPLICIT PROJECT ORDER
# ------------------------------------------------------------

order_column = find_column(
    eligible_screening,
    exact_names=[
        "ProjectNumber",
        "ProjectOrder",
        "Order",
        "Rank",
        "SelectionOrder",
    ],
    contains_terms=[
        "projectnumber",
        "projectorder",
        "selectionorder",
    ],
)


if order_column is not None:
    numeric_order = pd.to_numeric(
        eligible_screening[order_column],
        errors="coerce",
    )

    if numeric_order.notna().sum() > 0:
        eligible_screening = (
            eligible_screening
            .assign(
                _NumericOrder=numeric_order
            )
            .sort_values(
                "_NumericOrder",
                kind="mergesort",
            )
            .drop(
                columns=["_NumericOrder"]
            )
        )

        print(
            "\nExplicit screening-order column:",
            order_column,
        )


# ------------------------------------------------------------
# 12. SELECT THE FIRST UNFINISHED ELIGIBLE PROJECT
# ------------------------------------------------------------

project_8_name = None
project_8_screening_row = None

for row_index, row in eligible_screening.iterrows():
    candidate_name = str(
        row[
            screening_project_column
        ]
    ).strip()

    if not normalise_text(candidate_name):
        continue

    already_completed = any(
        values_match(
            candidate_name,
            completed_identifier,
        )
        for completed_identifier
        in completed_identifiers
    )

    if already_completed:
        continue

    project_8_name = candidate_name
    project_8_screening_row = row.copy()
    break


if project_8_name is None:
    raise RuntimeError(
        "No unfinished eligible project could be selected "
        "from the existing screening file."
    )


# ------------------------------------------------------------
# 13. MATCH PROJECT 8 TO ITS EXTRACTED DIRECTORY
# ------------------------------------------------------------

project_matches = []

for _, project_row in project_frame.iterrows():
    directory_name = project_row[
        "ProjectDirectoryName"
    ]

    full_directory = project_row[
        "Directory"
    ]

    score = 0

    if (
        normalise_text(project_8_name)
        == normalise_text(directory_name)
    ):
        score = 100

    elif values_match(
        project_8_name,
        directory_name,
    ):
        score = 90

    elif normalise_text(
        project_8_name
    ) in normalise_text(
        full_directory
    ):
        score = 80

    if score > 0:
        project_matches.append({
            **project_row.to_dict(),
            "MatchScore": score,
        })


if not project_matches:
    raise RuntimeError(
        "Project 8 was selected from the screening file "
        f"as '{project_8_name}', but no extracted project "
        "directory matched it."
    )


project_matches_frame = (
    pd.DataFrame(project_matches)
    .sort_values(
        "MatchScore",
        ascending=False,
        kind="mergesort",
    )
    .reset_index(drop=True)
)


highest_score = int(
    project_matches_frame.iloc[0][
        "MatchScore"
    ]
)

best_matches = project_matches_frame[
    project_matches_frame[
        "MatchScore"
    ].eq(highest_score)
]


if len(best_matches) != 1:
    print(
        "\nAmbiguous Project 8 directory matches:"
    )
    display(best_matches)

    raise RuntimeError(
        "Project 8 directory match is ambiguous."
    )


selected_project = best_matches.iloc[0]

PROJECT_8_DIRECTORY = Path(
    selected_project["Directory"]
)

PROJECT_8_BUILDS_PATH = Path(
    selected_project["BuildsPath"]
)

PROJECT_8_EXE_PATH = Path(
    selected_project["ExePath"]
)

PROJECT_8_DATASET_PATH = Path(
    selected_project["DatasetPath"]
)

PROJECT_8_ID_MAP_PATH = (
    Path(selected_project["IdMapPath"])
    if selected_project["HasIdMap"]
    else None
)

PROJECT_8_ENTITY_HISTORY_PATH = (
    Path(
        selected_project[
            "EntityHistoryPath"
        ]
    )
    if selected_project[
        "HasEntityChangeHistory"
    ]
    else None
)


# ------------------------------------------------------------
# 14. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 76)
print("=== PROJECT 8 RUNTIME RESTORE RESULT ===")
print("=" * 76)

print("\nArchive reused:")
print(ARCHIVE_PATH)

print("\nArchive downloaded again:")
print(False)

print("\nArchive MD5 verified:")
print(True)

print("\nTemporary extraction root:")
print(LOCAL_EXTRACTION_ROOT)

print("\nComplete datasets discovered:")
print(len(project_frame))

print("\nProject selection:")
print("Project number:", PROJECT_NUMBER)
print("Project:", project_8_name)

print("\nProject 8 source directory:")
print(PROJECT_8_DIRECTORY)

print("\nProject 8 source files:")
print("builds.csv:", PROJECT_8_BUILDS_PATH)
print("exe.csv:", PROJECT_8_EXE_PATH)
print("dataset.csv:", PROJECT_8_DATASET_PATH)

print(
    "id_map.csv:",
    PROJECT_8_ID_MAP_PATH
    if PROJECT_8_ID_MAP_PATH is not None
    else "NOT FOUND IN PROJECT DIRECTORY",
)

print(
    "entity-change history:",
    PROJECT_8_ENTITY_HISTORY_PATH
    if PROJECT_8_ENTITY_HISTORY_PATH
    is not None
    else "NOT FOUND IN PROJECT DIRECTORY",
)

print("\nCompletion registry:")
print("Registry rows:", len(registry))
print(
    "COMPLETE_AND_FROZEN projects:",
    len(frozen_registry),
)
print("Existing Project 8 rows: 0")

print("\nPermanent Google Drive writes:")
print(0)

print("\nProjects 1–7 modified:")
print(0)

print("\nNetwork downloads:")
print(0)

print(
    "\nSTATUS: "
    "PASS_PROJECT_8_RUNTIME_RESTORED_AND_SELECTED"
)

print("=" * 76)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== PROJECT 8: RUNTIME RESTORE AND PROJECT SELECTION ===

Permanent files located:

Archive:
/content/drive/MyDrive/Thesis_Experiment/Data/Raw/TCP-CI-main-dataset.tar.gz

Screening CSV:
/content/drive/MyDrive/Thesis_Experiment/Data/processed/tcp_ci_project_screening.csv

Completion registry:
/content/drive/MyDrive/Thesis_Experiment/Notes/completed_project_registry.csv

Existing archive size:
226.12 MB

Calculating archive MD5...
This reads the existing Drive file; it does not download it.

Expected MD5:
728804085c757ff5357aa165b4b6384f

Actual MD5:
728804085c757ff5357aa165b4b6384f

Archive integrity: PASS

No valid runtime extraction exists.

Extracting existing Drive archive...
No internet download is being performed.
Archive members: 176
Expected extracted size: 2.02 GB
Available Colab disk: 87.47 GB


/tmp/ipykernel_1633/2757029790.py:275: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(



Extraction completed successfully.

Complete project datasets discovered:
25


,ProjectDirectoryName,HasIdMap,HasEntityChangeHistory,Directory
0,Angel-ML@angel,True,True,/content/TCP-CI-main-dataset/datasets/Angel-ML...
1,CompEvol@beast2,True,True,/content/TCP-CI-main-dataset/datasets/CompEvol...
2,EMResearch@EvoMaster,True,True,/content/TCP-CI-main-dataset/datasets/EMResear...
3,Graylog2@graylog2-server,True,True,/content/TCP-CI-main-dataset/datasets/Graylog2...
4,JMRI@JMRI,True,True,/content/TCP-CI-main-dataset/datasets/JMRI@JMRI
5,SonarSource@sonarqube,True,True,/content/TCP-CI-main-dataset/datasets/SonarSou...
6,apache@airavata,True,True,/content/TCP-CI-main-dataset/datasets/apache@a...
7,apache@curator,True,True,/content/TCP-CI-main-dataset/datasets/apache@c...
8,apache@logging-log4j2,True,True,/content/TCP-CI-main-dataset/datasets/apache@l...
9,apache@rocketmq,True,True,/content/TCP-CI-main-dataset/datasets/apache@r...



Completion registry validation:
Registry rows: 7
COMPLETE_AND_FROZEN projects: 7

Screening file:
Rows: 25
Columns: ['Project', 'ProjectPath', 'ScreeningStatus', 'EligibleForPilot', 'ExclusionReasons', 'TotalBuilds', 'TrainingPeriodBuilds', 'EvaluationPeriodBuilds', 'RawExecutionRows', 'TrainingExecutionRows', 'EvaluationExecutionRows', 'UniqueExecutionTests', 'FailingTrainingBuilds', 'FailingEvaluationBuilds', 'TrainingFailureExecutions', 'EvaluationFailureExecutions', 'ModelTrainingRows', 'ModelTrainingBuilds', 'ModelEvaluationRows', 'ModelEvaluationBuilds', 'ModelTrainingPasses', 'ModelTrainingFailures', 'ModelEvaluationPasses', 'ModelEvaluationFailures', 'MissingBuildTimestamps', 'UnmatchedExecutionBuilds', 'UnmatchedDatasetBuilds', 'DuplicateDatasetBuildTestPairs', 'MissingExecutionDurations']

Screening project-column match scores:


,Column,DatasetMatches
0,Project,25
1,ProjectPath,25



Selected screening project column: Project

Eligibility column: EligibleForPilot
Eligible screening rows: 24


=== PROJECT 8 RUNTIME RESTORE RESULT ===

Archive reused:
/content/drive/MyDrive/Thesis_Experiment/Data/Raw/TCP-CI-main-dataset.tar.gz

Archive downloaded again:
False

Archive MD5 verified:
True

Temporary extraction root:
/content/TCP-CI-main-dataset

Complete datasets discovered:
25

Project selection:
Project number: 8
Project: optimatika@ojAlgo

Project 8 source directory:
/content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo

Project 8 source files:
builds.csv: /content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo/builds.csv
exe.csv: /content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo/exe.csv
dataset.csv: /content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo/dataset.csv
id_map.csv: /content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo/id_map.csv
entity-change history: /content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo/entity_change_history.csv

Completi

In [ ]:
# ============================================================
# PROJECT 8 — STEPS 1B–2
# IDENTITY LOCK + SOURCE DATA PREFLIGHT INVENTORY
#
# PROJECT:
# optimatika@ojAlgo
#
# This cell:
# - validates the Project 8 selection
# - verifies Projects 1–7 remain frozen
# - inventories all five source files
# - calculates SHA-256 hashes
# - records CSV dimensions and schemas
# - saves permanent preflight artefacts
#
# It does NOT:
# - modify the completion registry
# - create raw experiment conditions
# - modify Projects 1–7
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import re

import pandas as pd


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

PROJECT_NUMBER = 8
PROJECT_NAME = "optimatika@ojAlgo"
PROJECT_SLUG = "optimatika__ojAlgo"
PROJECT_SHORT_NAME = "ojalgo"

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

LOCAL_DATASET_ROOT = Path(
    "/content/TCP-CI-main-dataset/datasets"
)

PROJECT_SOURCE_DIR = (
    LOCAL_DATASET_ROOT / PROJECT_NAME
)

BUILDS_PATH = PROJECT_SOURCE_DIR / "builds.csv"
EXE_PATH = PROJECT_SOURCE_DIR / "exe.csv"
DATASET_PATH = PROJECT_SOURCE_DIR / "dataset.csv"
ID_MAP_PATH = PROJECT_SOURCE_DIR / "id_map.csv"
ENTITY_HISTORY_PATH = (
    PROJECT_SOURCE_DIR
    / "entity_change_history.csv"
)

SCREENING_PATH = (
    THESIS_ROOT
    / "Data"
    / "processed"
    / "tcp_ci_project_screening.csv"
)

REGISTRY_PATH = (
    THESIS_ROOT
    / "Notes"
    / "completed_project_registry.csv"
)

PROJECT_AGGREGATED_DIR = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / PROJECT_SLUG
)

PREFLIGHT_DIR = (
    PROJECT_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_preflight"
)

PROJECT_LOG_DIR = (
    THESIS_ROOT
    / "Results"
    / "Logs"
    / PROJECT_SLUG
)

IDENTITY_LOCK_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_project_identity_lock.json"
)

SOURCE_INVENTORY_CSV_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_source_file_inventory.csv"
)

SOURCE_INVENTORY_JSON_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_source_file_inventory.json"
)

SCHEMA_REPORT_JSON_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_source_schema_report.json"
)

SCREENING_SNAPSHOT_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_screening_snapshot.csv"
)


print("=" * 76)
print("=== PROJECT 8 STEPS 1B–2: IDENTITY LOCK AND PREFLIGHT ===")
print("=" * 76)


# ------------------------------------------------------------
# 2. HELPER FUNCTIONS
# ------------------------------------------------------------

def normalise_text(value):
    if value is None:
        return ""

    value = str(value).strip().lower()

    if value in {
        "",
        "nan",
        "none",
        "null",
    }:
        return ""

    return re.sub(
        r"[^a-z0-9]+",
        "",
        value,
    )


def values_match(left, right):
    left_normalised = normalise_text(left)
    right_normalised = normalise_text(right)

    if not left_normalised or not right_normalised:
        return False

    return left_normalised == right_normalised


def calculate_sha256(
    file_path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(file_path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def human_size(number_of_bytes):
    size = float(number_of_bytes)

    for unit in [
        "B",
        "KB",
        "MB",
        "GB",
        "TB",
    ]:
        if size < 1024 or unit == "TB":
            return f"{size:,.2f} {unit}"

        size /= 1024


def serialisable_value(value):
    if pd.isna(value):
        return None

    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            pass

    return value


def dataframe_schema_report(
    dataframe,
    file_path,
):
    missing_by_column = {
        str(column): int(
            dataframe[column].isna().sum()
        )
        for column in dataframe.columns
    }

    dtype_by_column = {
        str(column): str(dataframe[column].dtype)
        for column in dataframe.columns
    }

    unique_by_column = {
        str(column): int(
            dataframe[column].nunique(
                dropna=True
            )
        )
        for column in dataframe.columns
    }

    return {
        "path": str(file_path),
        "rows": int(len(dataframe)),
        "columns": int(
            len(dataframe.columns)
        ),
        "column_names": [
            str(column)
            for column in dataframe.columns
        ],
        "dtypes": dtype_by_column,
        "missing_values": missing_by_column,
        "unique_non_null_values": unique_by_column,
        "fully_duplicated_rows": int(
            dataframe.duplicated().sum()
        ),
    }


# ------------------------------------------------------------
# 3. VALIDATE RUNTIME EXTRACTION
# ------------------------------------------------------------

required_source_files = {
    "builds.csv": BUILDS_PATH,
    "exe.csv": EXE_PATH,
    "dataset.csv": DATASET_PATH,
    "id_map.csv": ID_MAP_PATH,
    "entity_change_history.csv":
        ENTITY_HISTORY_PATH,
}

missing_source_files = {
    name: str(path)
    for name, path
    in required_source_files.items()
    if not path.exists()
}

if missing_source_files:
    raise FileNotFoundError(
        "The runtime extraction is missing required "
        "Project 8 files.\n"
        "Rerun the runtime restore cell first.\n\n"
        + json.dumps(
            missing_source_files,
            indent=2,
        )
    )


print("\nProject source directory:")
print(PROJECT_SOURCE_DIR)

print("\nRequired source files:")
for source_name, source_path in (
    required_source_files.items()
):
    print(
        f"{source_name}: "
        f"{source_path}"
    )


# ------------------------------------------------------------
# 4. VALIDATE COMPLETION REGISTRY
# ------------------------------------------------------------

if not REGISTRY_PATH.exists():
    raise FileNotFoundError(
        f"Completion registry not found:\n"
        f"{REGISTRY_PATH}"
    )

registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
)

required_registry_columns = {
    "ProjectNumber",
    "Project",
    "ProjectSlug",
    "Status",
}

missing_registry_columns = (
    required_registry_columns
    - set(registry.columns)
)

if missing_registry_columns:
    raise RuntimeError(
        "Completion registry is missing required "
        "columns:\n"
        f"{sorted(missing_registry_columns)}"
    )


registry["_ProjectNumberNumeric"] = (
    pd.to_numeric(
        registry["ProjectNumber"],
        errors="coerce",
    )
)

registry["_NormalisedStatus"] = (
    registry["Status"]
    .astype(str)
    .str.strip()
    .str.upper()
)


frozen_registry = registry[
    registry["_NormalisedStatus"].eq(
        "COMPLETE_AND_FROZEN"
    )
].copy()


frozen_numbers = (
    frozen_registry[
        "_ProjectNumberNumeric"
    ]
    .dropna()
    .astype(int)
    .sort_values()
    .tolist()
)


if frozen_numbers != list(range(1, 8)):
    raise AssertionError(
        "Expected frozen project numbers "
        "[1, 2, 3, 4, 5, 6, 7], but found:\n"
        f"{frozen_numbers}"
    )


existing_project_8_rows = registry[
    registry[
        "_ProjectNumberNumeric"
    ].eq(PROJECT_NUMBER)
]

if len(existing_project_8_rows) != 0:
    raise AssertionError(
        "The completion registry already contains "
        "a Project 8 row."
    )


existing_identity_rows = registry[
    registry["Project"]
    .apply(
        lambda value: values_match(
            value,
            PROJECT_NAME,
        )
    )
    |
    registry["ProjectSlug"]
    .apply(
        lambda value: values_match(
            value,
            PROJECT_SLUG,
        )
    )
]

if len(existing_identity_rows) != 0:
    raise AssertionError(
        "optimatika@ojAlgo is already present in "
        "the completion registry under another "
        "project number."
    )


print("\nCompletion registry validation:")
print("Registry rows:", len(registry))
print(
    "COMPLETE_AND_FROZEN projects:",
    len(frozen_registry),
)
print(
    "Existing Project 8 rows:",
    len(existing_project_8_rows),
)
print(
    "Existing ojAlgo registry rows:",
    len(existing_identity_rows),
)


# ------------------------------------------------------------
# 5. VALIDATE SCREENING SELECTION
# ------------------------------------------------------------

if not SCREENING_PATH.exists():
    raise FileNotFoundError(
        f"Screening file not found:\n"
        f"{SCREENING_PATH}"
    )

screening = pd.read_csv(
    SCREENING_PATH,
    dtype=str,
)

required_screening_columns = {
    "Project",
    "ScreeningStatus",
    "EligibleForPilot",
}

missing_screening_columns = (
    required_screening_columns
    - set(screening.columns)
)

if missing_screening_columns:
    raise RuntimeError(
        "Screening CSV is missing required columns:\n"
        f"{sorted(missing_screening_columns)}"
    )


project_screening_rows = screening[
    screening["Project"]
    .apply(
        lambda value: values_match(
            value,
            PROJECT_NAME,
        )
    )
].copy()


if len(project_screening_rows) != 1:
    raise AssertionError(
        "Expected exactly one screening row for "
        f"{PROJECT_NAME}, but found "
        f"{len(project_screening_rows)}."
    )


project_screening_row = (
    project_screening_rows.iloc[0]
)


eligible_value = (
    str(
        project_screening_row[
            "EligibleForPilot"
        ]
    )
    .strip()
    .lower()
)

eligible_positive_values = {
    "true",
    "1",
    "yes",
    "y",
    "eligible",
    "selected",
    "included",
    "include",
    "pass",
    "passed",
}

if eligible_value not in eligible_positive_values:
    raise AssertionError(
        f"{PROJECT_NAME} is not marked eligible.\n"
        f"EligibleForPilot = "
        f"{project_screening_row['EligibleForPilot']}"
    )


completed_identifiers = set(
    registry["Project"]
    .dropna()
    .astype(str)
    .tolist()
)

completed_identifiers.update(
    registry["ProjectSlug"]
    .dropna()
    .astype(str)
    .tolist()
)


eligible_mask = (
    screening["EligibleForPilot"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(eligible_positive_values)
)

eligible_screening = screening[
    eligible_mask
].copy()


first_unfinished_project = None

for _, candidate_row in (
    eligible_screening.iterrows()
):
    candidate_project = str(
        candidate_row["Project"]
    ).strip()

    candidate_completed = any(
        values_match(
            candidate_project,
            completed_identifier,
        )
        for completed_identifier
        in completed_identifiers
    )

    if not candidate_completed:
        first_unfinished_project = (
            candidate_project
        )
        break


if not values_match(
    first_unfinished_project,
    PROJECT_NAME,
):
    raise AssertionError(
        "The first unfinished eligible project is not "
        f"{PROJECT_NAME}.\n"
        f"Detected: {first_unfinished_project}"
    )


print("\nScreening validation:")
print("Screening rows:", len(screening))
print(
    "Eligible projects:",
    len(eligible_screening),
)
print(
    "First unfinished eligible project:",
    first_unfinished_project,
)
print(
    "Project 8 ScreeningStatus:",
    project_screening_row[
        "ScreeningStatus"
    ],
)
print(
    "Project 8 EligibleForPilot:",
    project_screening_row[
        "EligibleForPilot"
    ],
)


# ------------------------------------------------------------
# 6. CREATE PROJECT 8 PREFLIGHT WORKSPACE
# ------------------------------------------------------------

# Refuse to overwrite a completed Project 8 package.
completion_markers = list(
    PROJECT_AGGREGATED_DIR.glob(
        "**/*COMPLETE_AND_FROZEN*"
    )
) if PROJECT_AGGREGATED_DIR.exists() else []

if completion_markers:
    raise RuntimeError(
        "A Project 8 completion marker appears to "
        "already exist. No files were written."
    )


PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PROJECT_LOG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print("\nPreflight directory:")
print(PREFLIGHT_DIR)

print("\nProject log directory:")
print(PROJECT_LOG_DIR)


# ------------------------------------------------------------
# 7. COMPUTE SOURCE FILE INVENTORY
# ------------------------------------------------------------

inventory_records = []

print("\nComputing source-file SHA-256 hashes...")

for source_name, source_path in (
    required_source_files.items()
):
    print(f"Hashing {source_name}...")

    file_size = int(
        source_path.stat().st_size
    )

    sha256 = calculate_sha256(
        source_path
    )

    inventory_records.append({
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "FileName": source_name,
        "FilePath": str(source_path),
        "SizeBytes": file_size,
        "ReadableSize":
            human_size(file_size),
        "SHA256": sha256,
    })


inventory_frame = pd.DataFrame(
    inventory_records
)

display(
    inventory_frame[
        [
            "FileName",
            "ReadableSize",
            "SHA256",
        ]
    ]
)


# ------------------------------------------------------------
# 8. LOAD SOURCE TABLES
# ------------------------------------------------------------

print("\nLoading Project 8 source CSV files...")

builds = pd.read_csv(
    BUILDS_PATH,
    low_memory=False,
)

print(
    "Loaded builds.csv:",
    builds.shape,
)

exe = pd.read_csv(
    EXE_PATH,
    low_memory=False,
)

print(
    "Loaded exe.csv:",
    exe.shape,
)

dataset = pd.read_csv(
    DATASET_PATH,
    low_memory=False,
)

print(
    "Loaded dataset.csv:",
    dataset.shape,
)

id_map = pd.read_csv(
    ID_MAP_PATH,
    low_memory=False,
)

print(
    "Loaded id_map.csv:",
    id_map.shape,
)

entity_history = pd.read_csv(
    ENTITY_HISTORY_PATH,
    low_memory=False,
)

print(
    "Loaded entity_change_history.csv:",
    entity_history.shape,
)


source_frames = {
    "builds.csv": builds,
    "exe.csv": exe,
    "dataset.csv": dataset,
    "id_map.csv": id_map,
    "entity_change_history.csv":
        entity_history,
}


# ------------------------------------------------------------
# 9. GENERATE SCHEMA REPORT
# ------------------------------------------------------------

schema_report = {
    "project_number": PROJECT_NUMBER,
    "project": PROJECT_NAME,
    "project_slug": PROJECT_SLUG,
    "generated_at_utc":
        datetime.now(timezone.utc).isoformat(),
    "source_directory":
        str(PROJECT_SOURCE_DIR),
    "tables": {},
}


for source_name, dataframe in (
    source_frames.items()
):
    schema_report["tables"][
        source_name
    ] = dataframe_schema_report(
        dataframe=dataframe,
        file_path=(
            required_source_files[
                source_name
            ]
        ),
    )


schema_summary_records = []

for source_name, report in (
    schema_report["tables"].items()
):
    schema_summary_records.append({
        "FileName": source_name,
        "Rows": report["rows"],
        "Columns": report["columns"],
        "FullyDuplicatedRows":
            report["fully_duplicated_rows"],
        "TotalMissingCells": int(
            sum(
                report[
                    "missing_values"
                ].values()
            )
        ),
    })


schema_summary_frame = pd.DataFrame(
    schema_summary_records
)

print("\nSource table summary:")
display(schema_summary_frame)


# ------------------------------------------------------------
# 10. CREATE IDENTITY LOCK
# ------------------------------------------------------------

identity_lock = {
    "project_number": PROJECT_NUMBER,
    "project": PROJECT_NAME,
    "project_slug": PROJECT_SLUG,
    "project_short_name":
        PROJECT_SHORT_NAME,
    "source_directory":
        str(PROJECT_SOURCE_DIR),
    "source_files": {
        name: str(path)
        for name, path
        in required_source_files.items()
    },
    "screening_path":
        str(SCREENING_PATH),
    "completion_registry_path":
        str(REGISTRY_PATH),
    "screening_status":
        serialisable_value(
            project_screening_row[
                "ScreeningStatus"
            ]
        ),
    "eligible_for_pilot":
        serialisable_value(
            project_screening_row[
                "EligibleForPilot"
            ]
        ),
    "previous_frozen_projects":
        int(len(frozen_registry)),
    "existing_project_8_registry_rows":
        int(len(existing_project_8_rows)),
    "selection_rule":
        "first unfinished eligible project "
        "in tcp_ci_project_screening.csv order",
    "generated_at_utc":
        datetime.now(timezone.utc).isoformat(),
    "status":
        "PROJECT_8_IDENTITY_LOCKED",
}


# ------------------------------------------------------------
# 11. SAVE PREFLIGHT ARTEFACTS
# ------------------------------------------------------------

inventory_frame.to_csv(
    SOURCE_INVENTORY_CSV_PATH,
    index=False,
)

SOURCE_INVENTORY_JSON_PATH.write_text(
    json.dumps(
        inventory_records,
        indent=2,
    ),
    encoding="utf-8",
)

SCHEMA_REPORT_JSON_PATH.write_text(
    json.dumps(
        schema_report,
        indent=2,
    ),
    encoding="utf-8",
)

IDENTITY_LOCK_PATH.write_text(
    json.dumps(
        identity_lock,
        indent=2,
    ),
    encoding="utf-8",
)

project_screening_rows.to_csv(
    SCREENING_SNAPSHOT_PATH,
    index=False,
)


# ------------------------------------------------------------
# 12. VERIFY WRITTEN ARTEFACTS
# ------------------------------------------------------------

expected_outputs = [
    IDENTITY_LOCK_PATH,
    SOURCE_INVENTORY_CSV_PATH,
    SOURCE_INVENTORY_JSON_PATH,
    SCHEMA_REPORT_JSON_PATH,
    SCREENING_SNAPSHOT_PATH,
]

missing_outputs = [
    str(path)
    for path in expected_outputs
    if not path.exists()
]

if missing_outputs:
    raise RuntimeError(
        "Some Project 8 preflight outputs were not "
        "written successfully:\n"
        + "\n".join(missing_outputs)
    )


# ------------------------------------------------------------
# 13. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 76)
print("=== PROJECT 8 STEPS 1B–2 RESULT ===")
print("=" * 76)

print("\nProject identity:")
print("Project number:", PROJECT_NUMBER)
print("Project:", PROJECT_NAME)
print("Project slug:", PROJECT_SLUG)

print("\nSelection:")
print(
    "First unfinished eligible project:",
    first_unfinished_project,
)
print(
    "EligibleForPilot:",
    project_screening_row[
        "EligibleForPilot"
    ],
)

print("\nSource files:")
print("Required files:", len(
    required_source_files
))
print("Missing required files:", 0)
print("Source SHA-256 hashes:", len(
    inventory_frame
))

print("\nSource tables:")
for _, row in schema_summary_frame.iterrows():
    print(
        f"{row['FileName']}: "
        f"{int(row['Rows'])} rows × "
        f"{int(row['Columns'])} columns"
    )

print("\nPreflight outputs:")
for path in expected_outputs:
    print(path)

print("\nCompletion registry:")
print("Registry rows:", len(registry))
print(
    "COMPLETE_AND_FROZEN projects:",
    len(frozen_registry),
)
print("Project 8 registry rows:", 0)
print("Registry modified:", 0)

print("\nPrevious projects modified:")
print(0)

print(
    "\nSTATUS: "
    "PASS_PROJECT_8_IDENTITY_LOCKED_AND_SOURCE_PREFLIGHT_COMPLETE"
)

print("=" * 76)

=== PROJECT 8 STEPS 1B–2: IDENTITY LOCK AND PREFLIGHT ===

Project source directory:
/content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo

Required source files:
builds.csv: /content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo/builds.csv
exe.csv: /content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo/exe.csv
dataset.csv: /content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo/dataset.csv
id_map.csv: /content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo/id_map.csv
entity_change_history.csv: /content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo/entity_change_history.csv

Completion registry validation:
Registry rows: 7
COMPLETE_AND_FROZEN projects: 7
Existing Project 8 rows: 0
Existing ojAlgo registry rows: 0

Screening validation:
Screening rows: 25
Eligible projects: 24
First unfinished eligible project: optimatika@ojAlgo
Project 8 ScreeningStatus: OK
Project 8 EligibleForPilot: True

Preflight directory:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika

,FileName,ReadableSize,SHA256
0,builds.csv,23.33 KB,a5f1988065a496d0ebff4e4ff1cedfc6bae297a8f71df3...
1,exe.csv,984.51 KB,bcc7668804f3eaa5af34051f13cfe15cab8841ef3f9706...
2,dataset.csv,8.86 MB,d993822278a0d75ea0ed40d7e7105fe30395282a81ba2e...
3,id_map.csv,130.66 KB,3356d4bd1f1b5123df8c2966099b6915e74e113570b35f...
4,entity_change_history.csv,2.23 MB,9833f581921f1decca0d1b44aba649279e9030362c1f0b...



Loading Project 8 source CSV files...
Loaded builds.csv: (254, 3)
Loaded exe.csv: (34438, 5)
Loaded dataset.csv: (9880, 154)
Loaded id_map.csv: (2436, 2)
Loaded entity_change_history.csv: (27427, 8)

Source table summary:


,FileName,Rows,Columns,FullyDuplicatedRows,TotalMissingCells
0,builds.csv,254,3,0,0
1,exe.csv,34438,5,0,0
2,dataset.csv,9880,154,0,0
3,id_map.csv,2436,2,0,0
4,entity_change_history.csv,27427,8,28,0




=== PROJECT 8 STEPS 1B–2 RESULT ===

Project identity:
Project number: 8
Project: optimatika@ojAlgo
Project slug: optimatika__ojAlgo

Selection:
First unfinished eligible project: optimatika@ojAlgo
EligibleForPilot: True

Source files:
Required files: 5
Missing required files: 0
Source SHA-256 hashes: 5

Source tables:
builds.csv: 254 rows × 3 columns
exe.csv: 34438 rows × 5 columns
dataset.csv: 9880 rows × 154 columns
id_map.csv: 2436 rows × 2 columns
entity_change_history.csv: 27427 rows × 8 columns

Preflight outputs:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_preflight/ojalgo_project_identity_lock.json
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_preflight/ojalgo_source_file_inventory.csv
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_preflight/ojalgo_source_file_inventory.json
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/oja

In [ ]:
# ============================================================
# PROJECT 8 — STEP 3
# CHRONOLOGY, 75/25 PARTITION, EXECUTION LINKAGE
# AND ELIGIBILITY REVALIDATION
#
# PROJECT: optimatika@ojAlgo
#
# CORRECTION:
# builds.csv uses:
#   id          -> build identifier
#   commits     -> commit field
#   started_at  -> build timestamp
#
# This cell reads:
# - frozen Project 8 identity lock
# - Project 8 screening snapshot
# - builds.csv
# - exe.csv
# - dataset.csv
#
# It writes only Project 8 preflight artefacts.
#
# It does NOT:
# - modify source CSV files
# - modify the completion registry
# - modify Projects 1–7
# - create experimental conditions
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations
from IPython.display import display

import hashlib
import json
import re

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

PROJECT_NUMBER = 8
PROJECT_NAME = "optimatika@ojAlgo"
PROJECT_SLUG = "optimatika__ojAlgo"
PROJECT_SHORT_NAME = "ojalgo"

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

PROJECT_SOURCE_DIR = Path(
    "/content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo"
)

PREFLIGHT_DIR = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / PROJECT_SLUG
    / f"{PROJECT_SHORT_NAME}_preflight"
)

IDENTITY_LOCK_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_project_identity_lock.json"
)

SCREENING_SNAPSHOT_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_screening_snapshot.csv"
)

BUILDS_PATH = (
    PROJECT_SOURCE_DIR
    / "builds.csv"
)

EXE_PATH = (
    PROJECT_SOURCE_DIR
    / "exe.csv"
)

DATASET_PATH = (
    PROJECT_SOURCE_DIR
    / "dataset.csv"
)

BUILD_CHRONOLOGY_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_build_chronology.csv"
)

COLUMN_MAPPING_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step3_column_mapping.json"
)

VERDICT_MAPPING_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step3_verdict_mapping.json"
)

AUDIT_CSV_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_chronology_linkage_audit.csv"
)

AUDIT_JSON_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_chronology_linkage_report.json"
)

STEP3_STATUS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step3_status.json"
)


print("=" * 78)
print("=== PROJECT 8 STEP 3: CHRONOLOGY, PARTITION AND LINKAGE ===")
print("=" * 78)


# ------------------------------------------------------------
# 2. GENERAL HELPERS
# ------------------------------------------------------------

def normalise_name(value):
    """
    Normalise column names and identifiers for comparison.
    """

    if value is None:
        return ""

    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).strip().lower(),
    )


def canonical_id(value):
    """
    Convert an identifier into a stable string.

    Examples:
      123     -> "123"
      123.0   -> "123"
      "123"   -> "123"
    """

    if pd.isna(value):
        return None

    text = str(value).strip()

    if text == "":
        return None

    if re.fullmatch(
        r"[-+]?\d+\.0+",
        text,
    ):
        return text.split(".")[0]

    return text


def normalise_verdict(value):
    """
    Convert verdict values into stable comparison tokens.
    """

    if pd.isna(value):
        return None

    text = str(value).strip().lower()

    if text == "":
        return None

    return re.sub(
        r"\s+",
        " ",
        text,
    )


def resolve_column(
    dataframe,
    candidates,
    role,
    required=True,
):
    """
    Resolve a column using a table-specific ordered list.

    Matching ignores case, spaces, hyphens and underscores.
    """

    normalised_lookup = {
        normalise_name(column): column
        for column in dataframe.columns
    }

    for candidate in candidates:
        candidate_key = normalise_name(
            candidate
        )

        if candidate_key in normalised_lookup:
            return normalised_lookup[
                candidate_key
            ]

    if required:
        raise RuntimeError(
            f"Could not identify {role}.\n"
            f"Available columns: {list(dataframe.columns)}\n"
            f"Expected one of: {candidates}"
        )

    return None


def expected_integer(
    screening_row,
    column_name,
):
    """
    Read a frozen screening metric as an integer.
    """

    if column_name not in screening_row.index:
        return None

    value = screening_row[
        column_name
    ]

    if pd.isna(value):
        return None

    text = str(value).strip()

    if text == "":
        return None

    return int(
        float(text)
    )


def to_jsonable(value):
    """
    Convert pandas and NumPy values to JSON-safe values.
    """

    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return value


def dataframe_sha256(dataframe):
    """
    Hash a deterministic CSV representation of a dataframe.
    """

    csv_bytes = dataframe.to_csv(
        index=False,
        lineterminator="\n",
    ).encode(
        "utf-8"
    )

    return hashlib.sha256(
        csv_bytes
    ).hexdigest()


def attach_partition(
    dataframe,
    build_column,
    build_partition_lookup,
):
    """
    Attach canonical build IDs and training/evaluation labels.
    """

    result = dataframe.copy()

    result["_BuildKey"] = (
        result[build_column]
        .map(canonical_id)
    )

    result["_Partition"] = (
        result["_BuildKey"]
        .map(build_partition_lookup)
    )

    return result


# ------------------------------------------------------------
# 3. VERDICT-INFERENCE HELPERS
# ------------------------------------------------------------

def generate_failure_token_candidates(
    unique_tokens,
):
    """
    Generate candidate interpretations of which verdict
    tokens represent failures.
    """

    unique_tokens = sorted(
        token
        for token in unique_tokens
        if token is not None
    )

    candidates = []

    known_failure_terms = {
        "fail",
        "failed",
        "failure",
        "failing",
        "error",
        "errored",
        "broken",
        "false",
        "not passed",
        "not_passed",
    }

    likely_failure_tokens = frozenset(
        token
        for token in unique_tokens
        if (
            token in known_failure_terms
            or "fail" in token
            or "error" in token
        )
    )

    if likely_failure_tokens:
        candidates.append(
            likely_failure_tokens
        )

    # TCP-CI verdict columns normally contain only a
    # small number of unique values. For up to eight tokens,
    # examine all possible non-empty subsets.
    if len(unique_tokens) <= 8:

        for subset_size in range(
            1,
            len(unique_tokens) + 1,
        ):
            for subset in combinations(
                unique_tokens,
                subset_size,
            ):
                candidates.append(
                    frozenset(subset)
                )

    else:

        for token in unique_tokens:
            candidates.append(
                frozenset([token])
            )

    unique_candidates = []
    seen_candidates = set()

    for candidate in candidates:

        if candidate in seen_candidates:
            continue

        seen_candidates.add(
            candidate
        )

        unique_candidates.append(
            candidate
        )

    return unique_candidates


def infer_failure_tokens(
    dataframe,
    verdict_column,
    expected_metrics,
    context_name,
):
    """
    Infer the failure encoding by matching the frozen
    project-screening metrics.

    This avoids assuming whether:
      0 means failure
      1 means failure
      "failed" means failure
      "error" means failure
    """

    verdict_tokens = (
        dataframe[verdict_column]
        .map(normalise_verdict)
    )

    unique_tokens = sorted(
        verdict_tokens
        .dropna()
        .unique()
        .tolist()
    )

    if not unique_tokens:
        raise RuntimeError(
            f"{context_name} contains no non-null verdict values."
        )

    candidate_sets = (
        generate_failure_token_candidates(
            unique_tokens
        )
    )

    evaluated_candidates = []

    for failure_tokens in candidate_sets:

        failure_mask = (
            verdict_tokens
            .isin(failure_tokens)
        )

        known_verdict_mask = (
            verdict_tokens
            .notna()
        )

        training_mask = (
            dataframe["_Partition"]
            .eq("training")
        )

        evaluation_mask = (
            dataframe["_Partition"]
            .eq("evaluation")
        )

        actual_metrics = {
            "TrainingFailureRows": int(
                (
                    training_mask
                    & failure_mask
                ).sum()
            ),

            "EvaluationFailureRows": int(
                (
                    evaluation_mask
                    & failure_mask
                ).sum()
            ),

            "TrainingFailingBuilds": int(
                dataframe.loc[
                    training_mask
                    & failure_mask,
                    "_BuildKey",
                ].nunique(
                    dropna=True
                )
            ),

            "EvaluationFailingBuilds": int(
                dataframe.loc[
                    evaluation_mask
                    & failure_mask,
                    "_BuildKey",
                ].nunique(
                    dropna=True
                )
            ),

            "TrainingPassRows": int(
                (
                    training_mask
                    & known_verdict_mask
                    & ~failure_mask
                ).sum()
            ),

            "EvaluationPassRows": int(
                (
                    evaluation_mask
                    & known_verdict_mask
                    & ~failure_mask
                ).sum()
            ),
        }

        absolute_difference = 0
        compared_metric_count = 0

        for (
            metric_name,
            expected_value,
        ) in expected_metrics.items():

            if expected_value is None:
                continue

            absolute_difference += abs(
                int(
                    actual_metrics[
                        metric_name
                    ]
                )
                - int(expected_value)
            )

            compared_metric_count += 1

        known_failure_overlap = sum(
            1
            for token in failure_tokens
            if (
                "fail" in token
                or "error" in token
                or token == "broken"
            )
        )

        evaluated_candidates.append({
            "FailureTokens": sorted(
                failure_tokens
            ),

            "Actual":
                actual_metrics,

            "AbsoluteDifference":
                absolute_difference,

            "ComparedMetrics":
                compared_metric_count,

            "KnownFailureOverlap":
                known_failure_overlap,
        })

    evaluated_candidates = sorted(
        evaluated_candidates,
        key=lambda candidate: (
            candidate[
                "AbsoluteDifference"
            ],
            -candidate[
                "KnownFailureOverlap"
            ],
            len(
                candidate[
                    "FailureTokens"
                ]
            ),
            candidate[
                "FailureTokens"
            ],
        ),
    )

    best_candidate = (
        evaluated_candidates[0]
    )

    if (
        best_candidate[
            "AbsoluteDifference"
        ]
        != 0
    ):

        print(
            f"\nTop verdict interpretations for {context_name}:"
        )

        candidate_display = pd.DataFrame([
            {
                "FailureTokens": ", ".join(
                    candidate[
                        "FailureTokens"
                    ]
                ),

                "AbsoluteDifference":
                    candidate[
                        "AbsoluteDifference"
                    ],

                **candidate[
                    "Actual"
                ],
            }
            for candidate
            in evaluated_candidates[:10]
        ])

        display(
            candidate_display
        )

        raise RuntimeError(
            f"Could not infer an exact verdict mapping "
            f"for {context_name} from the frozen "
            f"screening metrics."
        )

    selected_failure_tokens = set(
        best_candidate[
            "FailureTokens"
        ]
    )

    selected_failure_mask = (
        verdict_tokens
        .isin(
            selected_failure_tokens
        )
    )

    return {
        "unique_verdict_tokens":
            unique_tokens,

        "failure_tokens":
            sorted(
                selected_failure_tokens
            ),

        "verdict_tokens":
            verdict_tokens,

        "failure_mask":
            selected_failure_mask,

        "known_verdict_mask":
            verdict_tokens.notna(),

        "matched_metrics":
            best_candidate[
                "Actual"
            ],
    }


# ------------------------------------------------------------
# 4. VALIDATE REQUIRED INPUTS
# ------------------------------------------------------------

required_inputs = [
    IDENTITY_LOCK_PATH,
    SCREENING_SNAPSHOT_PATH,
    BUILDS_PATH,
    EXE_PATH,
    DATASET_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Required Project 8 Step 3 inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
        + "\n\nIf the /content dataset directory is missing, "
        "rerun the Project 8 runtime-restoration cell."
    )


identity_lock = json.loads(
    IDENTITY_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)

if (
    identity_lock.get(
        "project_number"
    )
    != PROJECT_NUMBER
):
    raise AssertionError(
        "The identity lock has the wrong project number."
    )

if (
    identity_lock.get(
        "project"
    )
    != PROJECT_NAME
):
    raise AssertionError(
        "The identity lock has the wrong project name."
    )

if (
    identity_lock.get(
        "project_slug"
    )
    != PROJECT_SLUG
):
    raise AssertionError(
        "The identity lock has the wrong project slug."
    )


screening_snapshot = pd.read_csv(
    SCREENING_SNAPSHOT_PATH,
    dtype=str,
)

if len(screening_snapshot) != 1:
    raise AssertionError(
        "Expected exactly one Project 8 screening row, "
        f"but found {len(screening_snapshot)}."
    )

screening_row = (
    screening_snapshot.iloc[0]
)


# ------------------------------------------------------------
# 5. LOAD SOURCE TABLES
# ------------------------------------------------------------

print("\nLoading source tables...")

builds = pd.read_csv(
    BUILDS_PATH,
    low_memory=False,
)

exe = pd.read_csv(
    EXE_PATH,
    low_memory=False,
)

dataset = pd.read_csv(
    DATASET_PATH,
    low_memory=False,
)

print(
    "builds.csv:",
    builds.shape,
)

print(
    "exe.csv:",
    exe.shape,
)

print(
    "dataset.csv:",
    dataset.shape,
)

print("\nbuilds.csv columns:")
print(
    list(builds.columns)
)

print("\nexe.csv columns:")
print(
    list(exe.columns)
)

print("\ndataset.csv first 20 columns:")
print(
    list(
        dataset.columns[:20]
    )
)


# ------------------------------------------------------------
# 6. IDENTIFY RELEVANT COLUMNS
# ------------------------------------------------------------

# builds.csv schema for ojAlgo:
# ['id', 'commits', 'started_at']

builds_build_column = resolve_column(
    builds,
    candidates=[
        "id",
        "build",
        "build_id",
        "buildid",
    ],
    role=(
        "the build identifier in builds.csv"
    ),
)

builds_timestamp_column = resolve_column(
    builds,
    candidates=[
        "started_at",
        "startedat",
        "timestamp",
        "build_timestamp",
        "created_at",
        "date",
    ],
    role=(
        "the build timestamp in builds.csv"
    ),
)

builds_commit_column = resolve_column(
    builds,
    candidates=[
        "commits",
        "commit",
        "commit_hash",
        "commit_hashes",
        "sha",
        "git_hash",
    ],
    role=(
        "the commit field in builds.csv"
    ),
    required=False,
)


exe_build_column = resolve_column(
    exe,
    candidates=[
        "build",
        "build_id",
        "buildid",
    ],
    role=(
        "the build identifier in exe.csv"
    ),
)

exe_job_column = resolve_column(
    exe,
    candidates=[
        "job",
        "job_id",
        "jobid",
    ],
    role=(
        "the job identifier in exe.csv"
    ),
    required=False,
)

exe_test_column = resolve_column(
    exe,
    candidates=[
        "test",
        "test_id",
        "testid",
        "test_case",
        "testcase",
        "test_name",
        "testname",
    ],
    role=(
        "the test identifier in exe.csv"
    ),
)

exe_verdict_column = resolve_column(
    exe,
    candidates=[
        "verdict",
        "result",
        "status",
        "outcome",
    ],
    role=(
        "the verdict in exe.csv"
    ),
)

exe_duration_column = resolve_column(
    exe,
    candidates=[
        "duration",
        "execution_duration",
        "execution_time",
        "runtime",
        "time",
    ],
    role=(
        "the duration in exe.csv"
    ),
)


dataset_build_column = resolve_column(
    dataset,
    candidates=[
        "build",
        "build_id",
        "buildid",
    ],
    role=(
        "the build identifier in dataset.csv"
    ),
)

dataset_job_column = resolve_column(
    dataset,
    candidates=[
        "job",
        "job_id",
        "jobid",
    ],
    role=(
        "the job identifier in dataset.csv"
    ),
    required=False,
)

dataset_test_column = resolve_column(
    dataset,
    candidates=[
        "test",
        "test_id",
        "testid",
        "test_case",
        "testcase",
        "test_name",
        "testname",
    ],
    role=(
        "the test identifier in dataset.csv"
    ),
)

dataset_verdict_column = resolve_column(
    dataset,
    candidates=[
        "verdict",
        "result",
        "status",
        "outcome",
    ],
    role=(
        "the verdict in dataset.csv"
    ),
)


column_mapping = {
    "builds.csv": {
        "build":
            builds_build_column,

        "timestamp":
            builds_timestamp_column,

        "commit":
            builds_commit_column,
    },

    "exe.csv": {
        "build":
            exe_build_column,

        "job":
            exe_job_column,

        "test":
            exe_test_column,

        "verdict":
            exe_verdict_column,

        "duration":
            exe_duration_column,
    },

    "dataset.csv": {
        "build":
            dataset_build_column,

        "job":
            dataset_job_column,

        "test":
            dataset_test_column,

        "verdict":
            dataset_verdict_column,
    },
}


print("\nDetected column mapping:")

print(
    json.dumps(
        column_mapping,
        indent=2,
    )
)


# Explicit Project 8 schema checks

if builds_build_column != "id":
    raise AssertionError(
        "Unexpected builds.csv build column.\n"
        f"Expected: id\n"
        f"Actual:   {builds_build_column}"
    )

if builds_timestamp_column != "started_at":
    raise AssertionError(
        "Unexpected builds.csv timestamp column.\n"
        f"Expected: started_at\n"
        f"Actual:   {builds_timestamp_column}"
    )

if builds_commit_column != "commits":
    raise AssertionError(
        "Unexpected builds.csv commit column.\n"
        f"Expected: commits\n"
        f"Actual:   {builds_commit_column}"
    )


# ------------------------------------------------------------
# 7. RECONSTRUCT BUILD CHRONOLOGY
# ------------------------------------------------------------

builds_working = (
    builds.copy()
)

builds_working[
    "_BuildKey"
] = (
    builds_working[
        builds_build_column
    ]
    .map(canonical_id)
)


missing_build_ids = int(
    builds_working[
        "_BuildKey"
    ].isna().sum()
)


duplicate_build_id_mask = (
    builds_working[
        "_BuildKey"
    ].duplicated(
        keep=False
    )
)

duplicate_build_id_rows = int(
    duplicate_build_id_mask.sum()
)


if missing_build_ids != 0:
    raise AssertionError(
        "builds.csv contains missing build IDs: "
        f"{missing_build_ids}"
    )


if duplicate_build_id_rows != 0:

    duplicate_build_values = (
        builds_working.loc[
            duplicate_build_id_mask,
            "_BuildKey",
        ]
        .value_counts()
        .rename("Rows")
        .reset_index(
            names="Build"
        )
    )

    print("\nDuplicate build IDs:")
    display(
        duplicate_build_values
    )

    raise AssertionError(
        "Build chronology cannot be reconstructed because "
        "builds.csv contains duplicated build IDs."
    )


builds_working[
    "_BuildTimestamp"
] = pd.to_datetime(
    builds_working[
        builds_timestamp_column
    ],
    errors="coerce",
    utc=True,
)


missing_build_timestamps = int(
    builds_working[
        "_BuildTimestamp"
    ].isna().sum()
)


builds_working[
    "_BuildNumeric"
] = pd.to_numeric(
    builds_working[
        "_BuildKey"
    ],
    errors="coerce",
)


builds_working[
    "_BuildNumericMissing"
] = (
    builds_working[
        "_BuildNumeric"
    ].isna()
)


builds_chronological = (
    builds_working
    .sort_values(
        by=[
            "_BuildTimestamp",
            "_BuildNumericMissing",
            "_BuildNumeric",
            "_BuildKey",
        ],
        ascending=[
            True,
            True,
            True,
            True,
        ],
        na_position="last",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


builds_chronological[
    "BuildOrder"
] = np.arange(
    1,
    len(
        builds_chronological
    ) + 1,
)


expected_total_builds = (
    expected_integer(
        screening_row,
        "TotalBuilds",
    )
)

expected_training_builds = (
    expected_integer(
        screening_row,
        "TrainingPeriodBuilds",
    )
)

expected_evaluation_builds = (
    expected_integer(
        screening_row,
        "EvaluationPeriodBuilds",
    )
)


if expected_total_builds is None:
    expected_total_builds = len(
        builds_chronological
    )


if expected_training_builds is None:
    expected_training_builds = int(
        np.floor(
            0.75
            * expected_total_builds
        )
    )


if expected_evaluation_builds is None:
    expected_evaluation_builds = (
        expected_total_builds
        - expected_training_builds
    )


if (
    expected_training_builds
    + expected_evaluation_builds
    != expected_total_builds
):
    raise AssertionError(
        "The frozen training and evaluation build counts "
        "do not sum to TotalBuilds."
    )


if (
    len(builds_chronological)
    != expected_total_builds
):
    raise AssertionError(
        "The reconstructed build count does not match "
        "the frozen screening snapshot.\n"
        f"Expected: {expected_total_builds}\n"
        f"Actual:   {len(builds_chronological)}"
    )


builds_chronological[
    "Partition"
] = np.where(
    builds_chronological[
        "BuildOrder"
    ]
    <= expected_training_builds,
    "training",
    "evaluation",
)


actual_training_builds = int(
    builds_chronological[
        "Partition"
    ]
    .eq("training")
    .sum()
)


actual_evaluation_builds = int(
    builds_chronological[
        "Partition"
    ]
    .eq("evaluation")
    .sum()
)


build_partition_lookup = dict(
    zip(
        builds_chronological[
            "_BuildKey"
        ],
        builds_chronological[
            "Partition"
        ],
    )
)


# ------------------------------------------------------------
# 8. ATTACH PARTITIONS TO EXE.CSV AND DATASET.CSV
# ------------------------------------------------------------

exe_partitioned = attach_partition(
    dataframe=exe,
    build_column=
        exe_build_column,
    build_partition_lookup=
        build_partition_lookup,
)


dataset_partitioned = attach_partition(
    dataframe=dataset,
    build_column=
        dataset_build_column,
    build_partition_lookup=
        build_partition_lookup,
)


unmatched_execution_build_ids = sorted(
    set(
        exe_partitioned.loc[
            exe_partitioned[
                "_Partition"
            ].isna()
            & exe_partitioned[
                "_BuildKey"
            ].notna(),
            "_BuildKey",
        ].tolist()
    )
)


unmatched_dataset_build_ids = sorted(
    set(
        dataset_partitioned.loc[
            dataset_partitioned[
                "_Partition"
            ].isna()
            & dataset_partitioned[
                "_BuildKey"
            ].notna(),
            "_BuildKey",
        ].tolist()
    )
)


# ------------------------------------------------------------
# 9. INFER EXE.CSV FAILURE ENCODING
# ------------------------------------------------------------

raw_expected_metrics = {
    "TrainingFailureRows":
        expected_integer(
            screening_row,
            "TrainingFailureExecutions",
        ),

    "EvaluationFailureRows":
        expected_integer(
            screening_row,
            "EvaluationFailureExecutions",
        ),

    "TrainingFailingBuilds":
        expected_integer(
            screening_row,
            "FailingTrainingBuilds",
        ),

    "EvaluationFailingBuilds":
        expected_integer(
            screening_row,
            "FailingEvaluationBuilds",
        ),
}


raw_verdict_mapping = (
    infer_failure_tokens(
        dataframe=
            exe_partitioned,

        verdict_column=
            exe_verdict_column,

        expected_metrics=
            raw_expected_metrics,

        context_name=
            "exe.csv",
    )
)


exe_partitioned[
    "_VerdictToken"
] = raw_verdict_mapping[
    "verdict_tokens"
]


exe_partitioned[
    "_VerdictKnown"
] = raw_verdict_mapping[
    "known_verdict_mask"
].astype(bool)


exe_partitioned[
    "_IsFailure"
] = raw_verdict_mapping[
    "failure_mask"
].astype(bool)


# ------------------------------------------------------------
# 10. INFER DATASET.CSV FAILURE ENCODING
# ------------------------------------------------------------

model_expected_metrics = {
    "TrainingFailureRows":
        expected_integer(
            screening_row,
            "ModelTrainingFailures",
        ),

    "EvaluationFailureRows":
        expected_integer(
            screening_row,
            "ModelEvaluationFailures",
        ),

    "TrainingPassRows":
        expected_integer(
            screening_row,
            "ModelTrainingPasses",
        ),

    "EvaluationPassRows":
        expected_integer(
            screening_row,
            "ModelEvaluationPasses",
        ),
}


model_verdict_mapping = (
    infer_failure_tokens(
        dataframe=
            dataset_partitioned,

        verdict_column=
            dataset_verdict_column,

        expected_metrics=
            model_expected_metrics,

        context_name=
            "dataset.csv",
    )
)


dataset_partitioned[
    "_VerdictToken"
] = model_verdict_mapping[
    "verdict_tokens"
]


dataset_partitioned[
    "_VerdictKnown"
] = model_verdict_mapping[
    "known_verdict_mask"
].astype(bool)


dataset_partitioned[
    "_IsFailure"
] = model_verdict_mapping[
    "failure_mask"
].astype(bool)


# ------------------------------------------------------------
# 11. RECONSTRUCT SCREENING METRICS
# ------------------------------------------------------------

exe_training_mask = (
    exe_partitioned[
        "_Partition"
    ].eq("training")
)


exe_evaluation_mask = (
    exe_partitioned[
        "_Partition"
    ].eq("evaluation")
)


dataset_training_mask = (
    dataset_partitioned[
        "_Partition"
    ].eq("training")
)


dataset_evaluation_mask = (
    dataset_partitioned[
        "_Partition"
    ].eq("evaluation")
)


execution_duration_numeric = pd.to_numeric(
    exe_partitioned[
        exe_duration_column
    ],
    errors="coerce",
)


duplicate_dataset_pair_counts = (
    dataset_partitioned
    .groupby(
        [
            "_BuildKey",
            dataset_test_column,
        ],
        dropna=False,
    )
    .size()
)


duplicate_dataset_build_test_pairs = int(
    duplicate_dataset_pair_counts
    .gt(1)
    .sum()
)


actual_metrics = {
    "TotalBuilds": int(
        len(
            builds_chronological
        )
    ),

    "TrainingPeriodBuilds":
        actual_training_builds,

    "EvaluationPeriodBuilds":
        actual_evaluation_builds,

    "RawExecutionRows": int(
        len(
            exe_partitioned
        )
    ),

    "TrainingExecutionRows": int(
        exe_training_mask.sum()
    ),

    "EvaluationExecutionRows": int(
        exe_evaluation_mask.sum()
    ),

    "UniqueExecutionTests": int(
        exe_partitioned[
            exe_test_column
        ].nunique(
            dropna=True
        )
    ),

    "FailingTrainingBuilds": int(
        exe_partitioned.loc[
            exe_training_mask
            & exe_partitioned[
                "_IsFailure"
            ],
            "_BuildKey",
        ].nunique(
            dropna=True
        )
    ),

    "FailingEvaluationBuilds": int(
        exe_partitioned.loc[
            exe_evaluation_mask
            & exe_partitioned[
                "_IsFailure"
            ],
            "_BuildKey",
        ].nunique(
            dropna=True
        )
    ),

    "TrainingFailureExecutions": int(
        (
            exe_training_mask
            & exe_partitioned[
                "_IsFailure"
            ]
        ).sum()
    ),

    "EvaluationFailureExecutions": int(
        (
            exe_evaluation_mask
            & exe_partitioned[
                "_IsFailure"
            ]
        ).sum()
    ),

    "ModelTrainingRows": int(
        dataset_training_mask.sum()
    ),

    "ModelTrainingBuilds": int(
        dataset_partitioned.loc[
            dataset_training_mask,
            "_BuildKey",
        ].nunique(
            dropna=True
        )
    ),

    "ModelEvaluationRows": int(
        dataset_evaluation_mask.sum()
    ),

    "ModelEvaluationBuilds": int(
        dataset_partitioned.loc[
            dataset_evaluation_mask,
            "_BuildKey",
        ].nunique(
            dropna=True
        )
    ),

    "ModelTrainingPasses": int(
        (
            dataset_training_mask
            & dataset_partitioned[
                "_VerdictKnown"
            ]
            & ~dataset_partitioned[
                "_IsFailure"
            ]
        ).sum()
    ),

    "ModelTrainingFailures": int(
        (
            dataset_training_mask
            & dataset_partitioned[
                "_IsFailure"
            ]
        ).sum()
    ),

    "ModelEvaluationPasses": int(
        (
            dataset_evaluation_mask
            & dataset_partitioned[
                "_VerdictKnown"
            ]
            & ~dataset_partitioned[
                "_IsFailure"
            ]
        ).sum()
    ),

    "ModelEvaluationFailures": int(
        (
            dataset_evaluation_mask
            & dataset_partitioned[
                "_IsFailure"
            ]
        ).sum()
    ),

    "MissingBuildTimestamps":
        missing_build_timestamps,

    "UnmatchedExecutionBuilds": int(
        len(
            unmatched_execution_build_ids
        )
    ),

    "UnmatchedDatasetBuilds": int(
        len(
            unmatched_dataset_build_ids
        )
    ),

    "DuplicateDatasetBuildTestPairs":
        duplicate_dataset_build_test_pairs,

    "MissingExecutionDurations": int(
        execution_duration_numeric
        .isna()
        .sum()
    ),
}


# ------------------------------------------------------------
# 12. COMPARE WITH FROZEN SCREENING SNAPSHOT
# ------------------------------------------------------------

audit_records = []

for (
    metric_name,
    actual_value,
) in actual_metrics.items():

    expected_value = expected_integer(
        screening_row,
        metric_name,
    )

    matches = (
        expected_value is None
        or int(expected_value)
        == int(actual_value)
    )

    audit_records.append({
        "Metric":
            metric_name,

        "Expected":
            expected_value,

        "Actual":
            actual_value,

        "Matches":
            bool(matches),
    })


audit_frame = pd.DataFrame(
    audit_records
)


print("\nChronology and linkage audit:")

display(
    audit_frame
)


mismatch_frame = (
    audit_frame.loc[
        ~audit_frame[
            "Matches"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 13. CHRONOLOGY SAFETY CHECKS
# ------------------------------------------------------------

training_timestamps = (
    builds_chronological.loc[
        builds_chronological[
            "Partition"
        ].eq("training"),
        "_BuildTimestamp",
    ]
    .dropna()
)


evaluation_timestamps = (
    builds_chronological.loc[
        builds_chronological[
            "Partition"
        ].eq("evaluation"),
        "_BuildTimestamp",
    ]
    .dropna()
)


if (
    len(training_timestamps) > 0
    and len(evaluation_timestamps) > 0
):

    latest_training_timestamp = (
        training_timestamps.max()
    )

    earliest_evaluation_timestamp = (
        evaluation_timestamps.min()
    )

    chronology_overlap = bool(
        latest_training_timestamp
        > earliest_evaluation_timestamp
    )

else:

    latest_training_timestamp = None
    earliest_evaluation_timestamp = None
    chronology_overlap = None


training_build_set = set(
    builds_chronological.loc[
        builds_chronological[
            "Partition"
        ].eq("training"),
        "_BuildKey",
    ].tolist()
)


evaluation_build_set = set(
    builds_chronological.loc[
        builds_chronological[
            "Partition"
        ].eq("evaluation"),
        "_BuildKey",
    ].tolist()
)


partitioned_build_overlap = (
    training_build_set
    & evaluation_build_set
)


if partitioned_build_overlap:
    raise AssertionError(
        "Training and evaluation build sets overlap."
    )


if chronology_overlap is True:
    raise AssertionError(
        "Chronology violation: at least one evaluation "
        "build precedes the final training build."
    )


minimum_required_failing_training_builds = 10

actual_failing_training_builds = (
    actual_metrics[
        "FailingTrainingBuilds"
    ]
)


eligibility_revalidated = bool(
    actual_failing_training_builds
    >= minimum_required_failing_training_builds
)


# ------------------------------------------------------------
# 14. PREPARE CHRONOLOGY OUTPUT
# ------------------------------------------------------------

chronology_output = pd.DataFrame({
    "CanonicalBuild":
        builds_chronological[
            "_BuildKey"
        ],

    "BuildOrder":
        builds_chronological[
            "BuildOrder"
        ],

    "Partition":
        builds_chronological[
            "Partition"
        ],

    "CanonicalTimestampUTC":
        builds_chronological[
            "_BuildTimestamp"
        ].map(
            lambda value: (
                value.isoformat()
                if pd.notna(value)
                else None
            )
        ),

    "OriginalBuildId":
        builds_chronological[
            builds_build_column
        ],

    "OriginalCommitField":
        builds_chronological[
            builds_commit_column
        ],
})


chronology_output_hash = (
    dataframe_sha256(
        chronology_output
    )
)


# ------------------------------------------------------------
# 15. CREATE VERDICT-MAPPING REPORT
# ------------------------------------------------------------

verdict_mapping_report = {
    "exe.csv": {
        "verdict_column":
            exe_verdict_column,

        "unique_verdict_tokens":
            raw_verdict_mapping[
                "unique_verdict_tokens"
            ],

        "failure_tokens":
            raw_verdict_mapping[
                "failure_tokens"
            ],

        "matched_metrics":
            raw_verdict_mapping[
                "matched_metrics"
            ],
    },

    "dataset.csv": {
        "verdict_column":
            dataset_verdict_column,

        "unique_verdict_tokens":
            model_verdict_mapping[
                "unique_verdict_tokens"
            ],

        "failure_tokens":
            model_verdict_mapping[
                "failure_tokens"
            ],

        "matched_metrics":
            model_verdict_mapping[
                "matched_metrics"
            ],
    },
}


# ------------------------------------------------------------
# 16. CREATE STEP 3 REPORT
# ------------------------------------------------------------

step3_report = {
    "project_number":
        PROJECT_NUMBER,

    "project":
        PROJECT_NAME,

    "project_slug":
        PROJECT_SLUG,

    "generated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "column_mapping":
        column_mapping,

    "chronology": {
        "total_builds": int(
            len(
                builds_chronological
            )
        ),

        "training_builds":
            actual_training_builds,

        "evaluation_builds":
            actual_evaluation_builds,

        "training_fraction": float(
            actual_training_builds
            / len(
                builds_chronological
            )
        ),

        "evaluation_fraction": float(
            actual_evaluation_builds
            / len(
                builds_chronological
            )
        ),

        "missing_build_ids":
            missing_build_ids,

        "duplicate_build_id_rows":
            duplicate_build_id_rows,

        "missing_build_timestamps":
            missing_build_timestamps,

        "latest_training_timestamp":
            to_jsonable(
                latest_training_timestamp
            ),

        "earliest_evaluation_timestamp":
            to_jsonable(
                earliest_evaluation_timestamp
            ),

        "chronology_overlap":
            chronology_overlap,

        "partition_build_overlap_count":
            int(
                len(
                    partitioned_build_overlap
                )
            ),
    },

    "linkage": {
        "unmatched_execution_build_count":
            int(
                len(
                    unmatched_execution_build_ids
                )
            ),

        "unmatched_execution_build_ids":
            unmatched_execution_build_ids,

        "unmatched_dataset_build_count":
            int(
                len(
                    unmatched_dataset_build_ids
                )
            ),

        "unmatched_dataset_build_ids":
            unmatched_dataset_build_ids,
    },

    "verdict_mapping":
        verdict_mapping_report,

    "screening_audit":
        audit_frame.to_dict(
            orient="records"
        ),

    "screening_mismatch_count":
        int(
            len(
                mismatch_frame
            )
        ),

    "eligibility": {
        "minimum_required_failing_training_builds":
            minimum_required_failing_training_builds,

        "actual_failing_training_builds":
            actual_failing_training_builds,

        "eligible":
            eligibility_revalidated,
    },

    "chronology_output_sha256":
        chronology_output_hash,
}


# ------------------------------------------------------------
# 17. DETERMINE STEP 3 STATUS
# ------------------------------------------------------------

step3_pass = bool(
    len(mismatch_frame) == 0
    and missing_build_ids == 0
    and duplicate_build_id_rows == 0
    and missing_build_timestamps == 0
    and len(
        unmatched_execution_build_ids
    ) == 0
    and len(
        unmatched_dataset_build_ids
    ) == 0
    and len(
        partitioned_build_overlap
    ) == 0
    and chronology_overlap is not True
    and eligibility_revalidated
)


step3_status_text = (
    "PASS_PROJECT_8_CHRONOLOGY_PARTITION_LINKAGE_AND_ELIGIBILITY_VALIDATED"
    if step3_pass
    else
    "FAIL_PROJECT_8_STEP_3_VALIDATION"
)


step3_status = {
    "project_number":
        PROJECT_NUMBER,

    "project":
        PROJECT_NAME,

    "project_slug":
        PROJECT_SLUG,

    "status":
        step3_status_text,

    "screening_mismatch_count":
        int(
            len(
                mismatch_frame
            )
        ),

    "eligible":
        eligibility_revalidated,

    "generated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


# ------------------------------------------------------------
# 18. SAVE STEP 3 ARTEFACTS
# ------------------------------------------------------------

PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


chronology_output.to_csv(
    BUILD_CHRONOLOGY_PATH,
    index=False,
)


COLUMN_MAPPING_PATH.write_text(
    json.dumps(
        column_mapping,
        indent=2,
    ),
    encoding="utf-8",
)


VERDICT_MAPPING_PATH.write_text(
    json.dumps(
        verdict_mapping_report,
        indent=2,
    ),
    encoding="utf-8",
)


audit_frame.to_csv(
    AUDIT_CSV_PATH,
    index=False,
)


AUDIT_JSON_PATH.write_text(
    json.dumps(
        step3_report,
        indent=2,
        default=to_jsonable,
    ),
    encoding="utf-8",
)


STEP3_STATUS_PATH.write_text(
    json.dumps(
        step3_status,
        indent=2,
        default=to_jsonable,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 19. VERIFY WRITTEN OUTPUTS
# ------------------------------------------------------------

expected_outputs = [
    BUILD_CHRONOLOGY_PATH,
    COLUMN_MAPPING_PATH,
    VERDICT_MAPPING_PATH,
    AUDIT_CSV_PATH,
    AUDIT_JSON_PATH,
    STEP3_STATUS_PATH,
]


missing_outputs = [
    str(path)
    for path in expected_outputs
    if not path.exists()
]


if missing_outputs:
    raise RuntimeError(
        "Step 3 failed to write all expected artefacts:\n"
        + "\n".join(
            missing_outputs
        )
    )


# ------------------------------------------------------------
# 20. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 78)
print("=== PROJECT 8 STEP 3 RESULT ===")
print("=" * 78)

print("\nProject identity:")
print(
    "Project number:",
    PROJECT_NUMBER,
)
print(
    "Project:",
    PROJECT_NAME,
)
print(
    "Project slug:",
    PROJECT_SLUG,
)

print("\nDetected builds.csv schema:")
print(
    "Build column:",
    builds_build_column,
)
print(
    "Timestamp column:",
    builds_timestamp_column,
)
print(
    "Commit column:",
    builds_commit_column,
)

print("\nChronology:")
print(
    "Total builds:",
    len(
        builds_chronological
    ),
)
print(
    "Training builds:",
    actual_training_builds,
)
print(
    "Evaluation builds:",
    actual_evaluation_builds,
)
print(
    "Missing build timestamps:",
    missing_build_timestamps,
)
print(
    "Duplicate build-ID rows:",
    duplicate_build_id_rows,
)
print(
    "Chronology overlap:",
    chronology_overlap,
)

print("\nExecution linkage:")
print(
    "Raw execution rows:",
    len(
        exe_partitioned
    ),
)
print(
    "Model-ready rows:",
    len(
        dataset_partitioned
    ),
)
print(
    "Unmatched execution builds:",
    len(
        unmatched_execution_build_ids
    ),
)
print(
    "Unmatched dataset builds:",
    len(
        unmatched_dataset_build_ids
    ),
)

print("\nVerdict mappings:")
print(
    "exe.csv failure tokens:",
    raw_verdict_mapping[
        "failure_tokens"
    ],
)
print(
    "dataset.csv failure tokens:",
    model_verdict_mapping[
        "failure_tokens"
    ],
)

print("\nFailure counts:")
print(
    "Training failure executions:",
    actual_metrics[
        "TrainingFailureExecutions"
    ],
)
print(
    "Evaluation failure executions:",
    actual_metrics[
        "EvaluationFailureExecutions"
    ],
)
print(
    "Model training failures:",
    actual_metrics[
        "ModelTrainingFailures"
    ],
)
print(
    "Model evaluation failures:",
    actual_metrics[
        "ModelEvaluationFailures"
    ],
)

print("\nEligibility:")
print(
    "Failing training builds:",
    actual_failing_training_builds,
)
print(
    "Minimum required:",
    minimum_required_failing_training_builds,
)
print(
    "Eligibility revalidated:",
    eligibility_revalidated,
)

print("\nFrozen screening comparison:")
print(
    "Metrics checked:",
    len(
        audit_frame
    ),
)
print(
    "Metric mismatches:",
    len(
        mismatch_frame
    ),
)

print("\nStep 3 outputs:")

for output_path in expected_outputs:
    print(
        output_path
    )

print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–7 modified:")
print(0)

print(
    "\nSTATUS:",
    step3_status_text,
)

print("=" * 78)


if not step3_pass:

    print(
        "\nMismatched screening metrics:"
    )

    display(
        mismatch_frame
    )

    raise RuntimeError(
        "PROJECT 8 STEP 3 DID NOT PASS.\n"
        "Review the displayed mismatches before continuing."
    )

=== PROJECT 8 STEP 3: CHRONOLOGY, PARTITION AND LINKAGE ===

Loading source tables...
builds.csv: (254, 3)
exe.csv: (34438, 5)
dataset.csv: (9880, 154)

builds.csv columns:
['id', 'commits', 'started_at']

exe.csv columns:
['test', 'build', 'job', 'verdict', 'duration']

dataset.csv first 20 columns:
['Build', 'Test', 'TES_COM_CountDeclFunction', 'TES_COM_CountLine', 'TES_COM_CountLineBlank', 'TES_COM_CountLineCode', 'TES_COM_CountLineCodeDecl', 'TES_COM_CountLineCodeExe', 'TES_COM_CountLineComment', 'TES_COM_CountStmt', 'TES_COM_CountStmtDecl', 'TES_COM_CountStmtExe', 'TES_COM_RatioCommentToCode', 'TES_COM_MaxCyclomatic', 'TES_COM_MaxCyclomaticModified', 'TES_COM_MaxCyclomaticStrict', 'TES_COM_MaxEssential', 'TES_COM_MaxNesting', 'TES_COM_SumCyclomatic', 'TES_COM_SumCyclomaticModified']

Detected column mapping:
{
  "builds.csv": {
    "build": "id",
    "timestamp": "started_at",
    "commit": "commits"
  },
  "exe.csv": {
    "build": "build",
    "job": "job",
    "test": "test",
 

,Metric,Expected,Actual,Matches
0,TotalBuilds,254,254,True
1,TrainingPeriodBuilds,190,190,True
2,EvaluationPeriodBuilds,64,64,True
3,RawExecutionRows,34438,34438,True
4,TrainingExecutionRows,25231,25231,True
5,EvaluationExecutionRows,9207,9207,True
6,UniqueExecutionTests,146,146,True
7,FailingTrainingBuilds,57,57,True
8,FailingEvaluationBuilds,16,16,True
9,TrainingFailureExecutions,66,66,True




=== PROJECT 8 STEP 3 RESULT ===

Project identity:
Project number: 8
Project: optimatika@ojAlgo
Project slug: optimatika__ojAlgo

Detected builds.csv schema:
Build column: id
Timestamp column: started_at
Commit column: commits

Chronology:
Total builds: 254
Training builds: 190
Evaluation builds: 64
Missing build timestamps: 0
Duplicate build-ID rows: 0
Chronology overlap: False

Execution linkage:
Raw execution rows: 34438
Model-ready rows: 9880
Unmatched execution builds: 0
Unmatched dataset builds: 0

Verdict mappings:
exe.csv failure tokens: ['1', '2']
dataset.csv failure tokens: ['1', '2']

Failure counts:
Training failure executions: 66
Evaluation failure executions: 16
Model training failures: 64
Model evaluation failures: 16

Eligibility:
Failing training builds: 57
Minimum required: 10
Eligibility revalidated: True

Frozen screening comparison:
Metrics checked: 24
Metric mismatches: 0

Step 3 outputs:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__oj

In [ ]:
# ============================================================
# PROJECT 8 — STEP 4
# CANONICAL EXECUTION RECONSTRUCTION AND
# RAW-TO-MODEL-READY ALIGNMENT
#
# PROJECT: optimatika@ojAlgo
#
# COMPLETE CORRECTED VERSION
#
# This cell:
# - reconstructs canonical raw execution records
# - aligns every dataset.csv row with exe.csv
# - validates verdicts and durations
# - documents raw failing builds absent from dataset.csv
# - generates build-level coverage
# - freezes Step 4 traceability artefacts
#
# It does NOT:
# - modify source CSV files
# - modify the completion registry
# - modify Projects 1–7
# - create experimental conditions
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import re

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

PROJECT_NUMBER = 8
PROJECT_NAME = "optimatika@ojAlgo"
PROJECT_SLUG = "optimatika__ojAlgo"
PROJECT_SHORT_NAME = "ojalgo"

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

PROJECT_SOURCE_DIR = Path(
    "/content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo"
)

PREFLIGHT_DIR = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / PROJECT_SLUG
    / f"{PROJECT_SHORT_NAME}_preflight"
)

BUILDS_PATH = (
    PROJECT_SOURCE_DIR
    / "builds.csv"
)

EXE_PATH = (
    PROJECT_SOURCE_DIR
    / "exe.csv"
)

DATASET_PATH = (
    PROJECT_SOURCE_DIR
    / "dataset.csv"
)

IDENTITY_LOCK_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_project_identity_lock.json"
)

SCREENING_SNAPSHOT_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_screening_snapshot.csv"
)

BUILD_CHRONOLOGY_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_build_chronology.csv"
)

COLUMN_MAPPING_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step3_column_mapping.json"
)

VERDICT_MAPPING_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step3_verdict_mapping.json"
)

STEP3_STATUS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step3_status.json"
)


# Step 4 outputs

CANONICAL_RAW_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_canonical_raw_executions.csv.gz"
)

RAW_BUILD_TEST_SUMMARY_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_raw_build_test_summary.csv.gz"
)

MODEL_ALIGNMENT_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_model_raw_alignment.csv.gz"
)

BUILD_COVERAGE_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_execution_build_coverage.csv"
)

ALIGNMENT_AUDIT_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_execution_alignment_audit.csv"
)

STEP4_REPORT_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_execution_alignment_report.json"
)

STEP4_STATUS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step4_status.json"
)


print("=" * 80)
print("=== PROJECT 8 STEP 4: CANONICAL EXECUTION ALIGNMENT ===")
print("=" * 80)


# ------------------------------------------------------------
# 2. HELPERS
# ------------------------------------------------------------

def normalise_name(value):
    if value is None:
        return ""

    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).strip().lower(),
    )


def canonical_id(value):
    if pd.isna(value):
        return None

    text = str(value).strip()

    if text == "":
        return None

    if re.fullmatch(
        r"[-+]?\d+\.0+",
        text,
    ):
        return text.split(".")[0]

    return text


def canonical_test(value):
    """
    Preserve test identifier case.
    Remove only surrounding whitespace.
    """

    if pd.isna(value):
        return None

    text = str(value).strip()

    if text == "":
        return None

    return text


def normalise_verdict(value):
    if pd.isna(value):
        return None

    text = str(value).strip().lower()

    if text == "":
        return None

    if re.fullmatch(
        r"[-+]?\d+\.0+",
        text,
    ):
        return text.split(".")[0]

    return re.sub(
        r"\s+",
        " ",
        text,
    )


def expected_integer(
    screening_row,
    column_name,
):
    if column_name not in screening_row.index:
        return None

    value = screening_row[
        column_name
    ]

    if pd.isna(value):
        return None

    text = str(value).strip()

    if text == "":
        return None

    return int(
        float(text)
    )


def json_safe(value):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return value


def join_unique_text(values):
    cleaned = sorted({
        str(value)
        for value in values
        if pd.notna(value)
        and str(value).strip() != ""
    })

    return "#".join(
        cleaned
    )


def join_source_rows(values):
    cleaned = sorted({
        int(value)
        for value in values
        if pd.notna(value)
    })

    return "#".join(
        str(value)
        for value in cleaned
    )


def calculate_sha256(
    file_path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(file_path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def file_metadata(file_path):
    file_path = Path(file_path)

    return {
        "path": str(file_path),

        "size_bytes": int(
            file_path.stat().st_size
        ),

        "sha256": calculate_sha256(
            file_path
        ),
    }


# ------------------------------------------------------------
# 3. VALIDATE INPUTS AND STEP 3
# ------------------------------------------------------------

required_inputs = [
    BUILDS_PATH,
    EXE_PATH,
    DATASET_PATH,
    IDENTITY_LOCK_PATH,
    SCREENING_SNAPSHOT_PATH,
    BUILD_CHRONOLOGY_PATH,
    COLUMN_MAPPING_PATH,
    VERDICT_MAPPING_PATH,
    STEP3_STATUS_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Required Step 4 inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
        + "\n\nIf the /content source directory is missing, "
        "rerun the Project 8 runtime-restoration cell."
    )


identity_lock = json.loads(
    IDENTITY_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)

if (
    identity_lock.get(
        "project_number"
    )
    != PROJECT_NUMBER
    or identity_lock.get(
        "project"
    )
    != PROJECT_NAME
    or identity_lock.get(
        "project_slug"
    )
    != PROJECT_SLUG
):
    raise AssertionError(
        "The Project 8 identity lock does not match "
        "the configured project."
    )


step3_status = json.loads(
    STEP3_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

expected_step3_status = (
    "PASS_PROJECT_8_CHRONOLOGY_PARTITION_LINKAGE_AND_ELIGIBILITY_VALIDATED"
)

if (
    step3_status.get(
        "status"
    )
    != expected_step3_status
):
    raise AssertionError(
        "Step 3 has not passed successfully.\n"
        f"Detected status: "
        f"{step3_status.get('status')}"
    )


column_mapping = json.loads(
    COLUMN_MAPPING_PATH.read_text(
        encoding="utf-8"
    )
)

verdict_mapping = json.loads(
    VERDICT_MAPPING_PATH.read_text(
        encoding="utf-8"
    )
)

screening_snapshot = pd.read_csv(
    SCREENING_SNAPSHOT_PATH,
    dtype=str,
)

if len(screening_snapshot) != 1:
    raise AssertionError(
        "Expected exactly one Project 8 screening row."
    )

screening_row = (
    screening_snapshot.iloc[0]
)


# ------------------------------------------------------------
# 4. LOAD SOURCE AND CHRONOLOGY TABLES
# ------------------------------------------------------------

print("\nLoading Step 4 inputs...")

exe = pd.read_csv(
    EXE_PATH,
    low_memory=False,
)

dataset = pd.read_csv(
    DATASET_PATH,
    low_memory=False,
)

chronology = pd.read_csv(
    BUILD_CHRONOLOGY_PATH,
    dtype=str,
)

print(
    "exe.csv:",
    exe.shape,
)

print(
    "dataset.csv:",
    dataset.shape,
)

print(
    "build chronology:",
    chronology.shape,
)


# ------------------------------------------------------------
# 5. RESOLVE FROZEN COLUMN MAPPINGS
# ------------------------------------------------------------

exe_build_column = (
    column_mapping[
        "exe.csv"
    ]["build"]
)

exe_job_column = (
    column_mapping[
        "exe.csv"
    ]["job"]
)

exe_test_column = (
    column_mapping[
        "exe.csv"
    ]["test"]
)

exe_verdict_column = (
    column_mapping[
        "exe.csv"
    ]["verdict"]
)

exe_duration_column = (
    column_mapping[
        "exe.csv"
    ]["duration"]
)


dataset_build_column = (
    column_mapping[
        "dataset.csv"
    ]["build"]
)

dataset_test_column = (
    column_mapping[
        "dataset.csv"
    ]["test"]
)

dataset_verdict_column = (
    column_mapping[
        "dataset.csv"
    ]["verdict"]
)


for (
    table_name,
    dataframe,
    required_columns,
) in [
    (
        "exe.csv",
        exe,
        [
            exe_build_column,
            exe_test_column,
            exe_verdict_column,
            exe_duration_column,
        ],
    ),
    (
        "dataset.csv",
        dataset,
        [
            dataset_build_column,
            dataset_test_column,
            dataset_verdict_column,
        ],
    ),
]:

    missing_columns = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise RuntimeError(
            f"{table_name} is missing frozen columns:\n"
            f"{missing_columns}"
        )


# ------------------------------------------------------------
# 6. DETECT DATASET DURATION COLUMN
# ------------------------------------------------------------

model_duration_names = {
    "duration",
    "executionduration",
    "executiontime",
    "runtime",
    "testduration",
}

model_duration_candidates = [
    column
    for column in dataset.columns
    if normalise_name(
        column
    )
    in model_duration_names
]

if len(model_duration_candidates) > 1:
    raise RuntimeError(
        "Multiple possible duration columns were found "
        "in dataset.csv:\n"
        f"{model_duration_candidates}"
    )

dataset_duration_column = (
    model_duration_candidates[0]
    if model_duration_candidates
    else None
)


print("\nFrozen source columns:")

print(
    "exe build:",
    exe_build_column,
)

print(
    "exe job:",
    exe_job_column,
)

print(
    "exe test:",
    exe_test_column,
)

print(
    "exe verdict:",
    exe_verdict_column,
)

print(
    "exe duration:",
    exe_duration_column,
)

print(
    "dataset build:",
    dataset_build_column,
)

print(
    "dataset test:",
    dataset_test_column,
)

print(
    "dataset verdict:",
    dataset_verdict_column,
)

print(
    "dataset duration:",
    (
        dataset_duration_column
        if dataset_duration_column is not None
        else "NOT PRESENT"
    ),
)


# ------------------------------------------------------------
# 7. PREPARE CHRONOLOGY LOOKUPS
# ------------------------------------------------------------

required_chronology_columns = {
    "CanonicalBuild",
    "BuildOrder",
    "Partition",
}

missing_chronology_columns = (
    required_chronology_columns
    - set(
        chronology.columns
    )
)

if missing_chronology_columns:
    raise RuntimeError(
        "The Step 3 chronology file is missing columns:\n"
        f"{sorted(missing_chronology_columns)}"
    )


chronology[
    "_BuildKey"
] = chronology[
    "CanonicalBuild"
].map(
    canonical_id
)


if chronology[
    "_BuildKey"
].duplicated().any():
    raise AssertionError(
        "The frozen chronology contains duplicate "
        "canonical build IDs."
    )


build_order_lookup = dict(
    zip(
        chronology[
            "_BuildKey"
        ],

        pd.to_numeric(
            chronology[
                "BuildOrder"
            ],
            errors="raise",
        ).astype(int),
    )
)


partition_lookup = dict(
    zip(
        chronology[
            "_BuildKey"
        ],

        chronology[
            "Partition"
        ].astype(str),
    )
)


# ------------------------------------------------------------
# 8. PREPARE CANONICAL RAW EXECUTION TABLE
# ------------------------------------------------------------

raw_failure_tokens = set(
    str(token)
    for token in verdict_mapping[
        "exe.csv"
    ]["failure_tokens"]
)

model_failure_tokens = set(
    str(token)
    for token in verdict_mapping[
        "dataset.csv"
    ]["failure_tokens"]
)


raw = (
    exe.reset_index(
        names="_RawSourceRow"
    )
    .copy()
)


raw[
    "CanonicalBuild"
] = raw[
    exe_build_column
].map(
    canonical_id
)


raw[
    "CanonicalTest"
] = raw[
    exe_test_column
].map(
    canonical_test
)


if (
    exe_job_column is not None
    and exe_job_column in raw.columns
):
    raw[
        "CanonicalJob"
    ] = raw[
        exe_job_column
    ].map(
        canonical_id
    )

else:
    raw[
        "CanonicalJob"
    ] = None


raw[
    "RawVerdictToken"
] = raw[
    exe_verdict_column
].map(
    normalise_verdict
)


raw[
    "RawIsFailure"
] = raw[
    "RawVerdictToken"
].isin(
    raw_failure_tokens
)


raw[
    "RawDuration"
] = pd.to_numeric(
    raw[
        exe_duration_column
    ],
    errors="coerce",
)


raw[
    "BuildOrder"
] = raw[
    "CanonicalBuild"
].map(
    build_order_lookup
)


raw[
    "Partition"
] = raw[
    "CanonicalBuild"
].map(
    partition_lookup
)


raw_missing_build = int(
    raw[
        "CanonicalBuild"
    ].isna().sum()
)

raw_missing_test = int(
    raw[
        "CanonicalTest"
    ].isna().sum()
)

raw_missing_verdict = int(
    raw[
        "RawVerdictToken"
    ].isna().sum()
)

raw_missing_duration = int(
    raw[
        "RawDuration"
    ].isna().sum()
)

raw_unmatched_chronology = int(
    raw[
        "Partition"
    ].isna().sum()
)

raw_negative_duration = int(
    raw[
        "RawDuration"
    ].lt(0).sum()
)

raw_zero_duration = int(
    raw[
        "RawDuration"
    ].eq(0).sum()
)


# ------------------------------------------------------------
# 9. PREPARE MODEL-READY TABLE
# ------------------------------------------------------------

model = (
    dataset.reset_index(
        names="_ModelSourceRow"
    )
    .copy()
)


model[
    "CanonicalBuild"
] = model[
    dataset_build_column
].map(
    canonical_id
)


model[
    "CanonicalTest"
] = model[
    dataset_test_column
].map(
    canonical_test
)


model[
    "ModelVerdictToken"
] = model[
    dataset_verdict_column
].map(
    normalise_verdict
)


model[
    "ModelIsFailure"
] = model[
    "ModelVerdictToken"
].isin(
    model_failure_tokens
)


model[
    "BuildOrder"
] = model[
    "CanonicalBuild"
].map(
    build_order_lookup
)


model[
    "Partition"
] = model[
    "CanonicalBuild"
].map(
    partition_lookup
)


if dataset_duration_column is not None:

    model[
        "ModelDuration"
    ] = pd.to_numeric(
        model[
            dataset_duration_column
        ],
        errors="coerce",
    )

else:

    model[
        "ModelDuration"
    ] = np.nan


model_missing_build = int(
    model[
        "CanonicalBuild"
    ].isna().sum()
)

model_missing_test = int(
    model[
        "CanonicalTest"
    ].isna().sum()
)

model_missing_verdict = int(
    model[
        "ModelVerdictToken"
    ].isna().sum()
)

model_unmatched_chronology = int(
    model[
        "Partition"
    ].isna().sum()
)


model_duplicate_pair_mask = (
    model.duplicated(
        subset=[
            "CanonicalBuild",
            "CanonicalTest",
        ],
        keep=False,
    )
)


model_duplicate_pair_rows = int(
    model_duplicate_pair_mask.sum()
)


model_duplicate_pair_groups = int(
    model.loc[
        model_duplicate_pair_mask,
        [
            "CanonicalBuild",
            "CanonicalTest",
        ],
    ]
    .drop_duplicates()
    .shape[0]
)


# ------------------------------------------------------------
# 10. RAW BUILD–TEST SUMMARY
# ------------------------------------------------------------

raw_grouped = raw.groupby(
    [
        "CanonicalBuild",
        "CanonicalTest",
    ],
    dropna=False,
    sort=False,
)


raw_summary = raw_grouped.agg(
    RawExecutionRows=(
        "_RawSourceRow",
        "size",
    ),

    RawSourceRows=(
        "_RawSourceRow",
        join_source_rows,
    ),

    RawJobCount=(
        "CanonicalJob",
        lambda values: int(
            pd.Series(
                values
            )
            .dropna()
            .nunique()
        ),
    ),

    RawJobs=(
        "CanonicalJob",
        join_unique_text,
    ),

    RawVerdictDistinctCount=(
        "RawVerdictToken",
        lambda values: int(
            pd.Series(
                values
            )
            .dropna()
            .nunique()
        ),
    ),

    RawVerdictTokens=(
        "RawVerdictToken",
        join_unique_text,
    ),

    RawFailureDistinctCount=(
        "RawIsFailure",
        lambda values: int(
            pd.Series(
                values
            )
            .dropna()
            .nunique()
        ),
    ),

    RawFailureExecutions=(
        "RawIsFailure",
        "sum",
    ),

    RawDurationCount=(
        "RawDuration",
        "count",
    ),

    RawDurationMin=(
        "RawDuration",
        "min",
    ),

    RawDurationMax=(
        "RawDuration",
        "max",
    ),

    RawDurationMean=(
        "RawDuration",
        "mean",
    ),

    RawDurationSum=(
        "RawDuration",
        "sum",
    ),
).reset_index()


raw_summary[
    "RawVerdictConsensus"
] = raw_summary[
    "RawVerdictDistinctCount"
].eq(1)


raw_summary[
    "RawFailureConsensus"
] = raw_summary[
    "RawFailureDistinctCount"
].le(1)


raw_summary[
    "RawDurationConsensus"
] = pd.Series(
    np.isclose(
        raw_summary[
            "RawDurationMin"
        ],

        raw_summary[
            "RawDurationMax"
        ],

        rtol=0,
        atol=1e-12,
        equal_nan=False,
    ),

    index=raw_summary.index,
)


raw_summary[
    "CanonicalRawVerdictToken"
] = np.where(
    raw_summary[
        "RawVerdictConsensus"
    ],

    raw_summary[
        "RawVerdictTokens"
    ],

    None,
)


raw_summary[
    "CanonicalRawDuration"
] = np.where(
    raw_summary[
        "RawDurationConsensus"
    ],

    raw_summary[
        "RawDurationMin"
    ],

    np.nan,
)


raw_duplicate_build_test_pairs = int(
    raw_summary[
        "RawExecutionRows"
    ].gt(1).sum()
)


raw_verdict_conflict_pairs = int(
    (
        ~raw_summary[
            "RawVerdictConsensus"
        ]
    ).sum()
)


raw_duration_conflict_pairs = int(
    (
        ~raw_summary[
            "RawDurationConsensus"
        ]
    ).sum()
)


# ------------------------------------------------------------
# 11. CANDIDATE MODEL–RAW MATCHING
# ------------------------------------------------------------

model_link_columns = [
    "_ModelSourceRow",
    "CanonicalBuild",
    "CanonicalTest",
    "ModelVerdictToken",
    "ModelIsFailure",
    "ModelDuration",
]


raw_link_columns = [
    "_RawSourceRow",
    "CanonicalBuild",
    "CanonicalTest",
    "CanonicalJob",
    "RawVerdictToken",
    "RawIsFailure",
    "RawDuration",
]


candidate_matches = (
    model[
        model_link_columns
    ]
    .merge(
        raw[
            raw_link_columns
        ],

        on=[
            "CanonicalBuild",
            "CanonicalTest",
        ],

        how="left",
        sort=False,
        validate="many_to_many",
    )
)


candidate_matches[
    "_CandidatePresent"
] = candidate_matches[
    "_RawSourceRow"
].notna()


candidate_matches[
    "_VerdictMatches"
] = (
    candidate_matches[
        "_CandidatePresent"
    ]
    &
    candidate_matches[
        "ModelVerdictToken"
    ].eq(
        candidate_matches[
            "RawVerdictToken"
        ]
    )
)


if dataset_duration_column is not None:

    duration_match_values = np.isclose(
        candidate_matches[
            "ModelDuration"
        ],

        candidate_matches[
            "RawDuration"
        ],

        rtol=1e-9,
        atol=1e-9,
        equal_nan=False,
    )

    candidate_matches[
        "_DurationMatches"
    ] = (
        candidate_matches[
            "_CandidatePresent"
        ]
        &
        candidate_matches[
            "ModelDuration"
        ].notna()
        &
        candidate_matches[
            "RawDuration"
        ].notna()
        &
        pd.Series(
            duration_match_values,
            index=candidate_matches.index,
        )
    )

else:

    candidate_matches[
        "_DurationMatches"
    ] = candidate_matches[
        "_CandidatePresent"
    ]


candidate_matches[
    "_ExactMatch"
] = (
    candidate_matches[
        "_CandidatePresent"
    ]
    &
    candidate_matches[
        "_VerdictMatches"
    ]
    &
    candidate_matches[
        "_DurationMatches"
    ]
)


candidate_counts = (
    candidate_matches
    .groupby(
        "_ModelSourceRow",
        sort=False,
    )
    .agg(
        RawCandidateRows=(
            "_CandidatePresent",
            "sum",
        ),

        RawVerdictMatchingRows=(
            "_VerdictMatches",
            "sum",
        ),

        RawDurationMatchingRows=(
            "_DurationMatches",
            "sum",
        ),

        RawExactMatchingRows=(
            "_ExactMatch",
            "sum",
        ),
    )
    .reset_index()
)


exact_source_rows = (
    candidate_matches.loc[
        candidate_matches[
            "_ExactMatch"
        ],
        [
            "_ModelSourceRow",
            "_RawSourceRow",
        ],
    ]
    .groupby(
        "_ModelSourceRow"
    )[
        "_RawSourceRow"
    ]
    .agg(
        join_source_rows
    )
    .rename(
        "ExactMatchingRawSourceRows"
    )
    .reset_index()
)


verdict_source_rows = (
    candidate_matches.loc[
        candidate_matches[
            "_VerdictMatches"
        ],
        [
            "_ModelSourceRow",
            "_RawSourceRow",
        ],
    ]
    .groupby(
        "_ModelSourceRow"
    )[
        "_RawSourceRow"
    ]
    .agg(
        join_source_rows
    )
    .rename(
        "VerdictMatchingRawSourceRows"
    )
    .reset_index()
)


# ------------------------------------------------------------
# 12. BUILD MODEL–RAW ALIGNMENT TABLE
# ------------------------------------------------------------

alignment_base_columns = [
    "_ModelSourceRow",
    "CanonicalBuild",
    "CanonicalTest",
    "BuildOrder",
    "Partition",
    "ModelVerdictToken",
    "ModelIsFailure",
    "ModelDuration",
]


alignment = (
    model[
        alignment_base_columns
    ]
    .merge(
        raw_summary,

        on=[
            "CanonicalBuild",
            "CanonicalTest",
        ],

        how="left",
        sort=False,
        validate="one_to_one",
    )
    .merge(
        candidate_counts,

        on="_ModelSourceRow",

        how="left",
        validate="one_to_one",
    )
    .merge(
        verdict_source_rows,

        on="_ModelSourceRow",

        how="left",
        validate="one_to_one",
    )
    .merge(
        exact_source_rows,

        on="_ModelSourceRow",

        how="left",
        validate="one_to_one",
    )
)


count_columns = [
    "RawCandidateRows",
    "RawVerdictMatchingRows",
    "RawDurationMatchingRows",
    "RawExactMatchingRows",
]


for column in count_columns:

    alignment[column] = (
        alignment[column]
        .fillna(0)
        .astype(int)
    )


alignment[
    "RawExecutionRows"
] = (
    alignment[
        "RawExecutionRows"
    ]
    .fillna(0)
    .astype(int)
)


alignment[
    "ModelRawVerdictMatches"
] = alignment[
    "RawVerdictMatchingRows"
].gt(0)


if dataset_duration_column is not None:

    alignment[
        "ModelRawDurationMatches"
    ] = alignment[
        "RawExactMatchingRows"
    ].gt(0)

    alignment[
        "CanonicalDuration"
    ] = np.where(
        alignment[
            "RawExactMatchingRows"
        ].gt(0),

        alignment[
            "ModelDuration"
        ],

        np.nan,
    )

    alignment[
        "CanonicalDurationSource"
    ] = np.where(
        alignment[
            "RawExactMatchingRows"
        ].gt(0),

        "dataset.csv duration validated against exe.csv",

        None,
    )

else:

    alignment[
        "ModelRawDurationMatches"
    ] = alignment[
        "RawDurationConsensus"
    ].fillna(False)

    alignment[
        "CanonicalDuration"
    ] = np.where(
        alignment[
            "RawDurationConsensus"
        ].fillna(False),

        alignment[
            "CanonicalRawDuration"
        ],

        np.nan,
    )

    alignment[
        "CanonicalDurationSource"
    ] = np.where(
        alignment[
            "RawDurationConsensus"
        ].fillna(False),

        "exe.csv build-test consensus",

        None,
    )


alignment[
    "CanonicalVerdictToken"
] = np.where(
    alignment[
        "ModelRawVerdictMatches"
    ],

    alignment[
        "ModelVerdictToken"
    ],

    None,
)


alignment[
    "CanonicalIsFailure"
] = np.where(
    alignment[
        "ModelRawVerdictMatches"
    ],

    alignment[
        "ModelIsFailure"
    ],

    np.nan,
)


def determine_alignment_status(row):
    raw_candidates = int(
        row[
            "RawCandidateRows"
        ]
    )

    verdict_matches = int(
        row[
            "RawVerdictMatchingRows"
        ]
    )

    exact_matches = int(
        row[
            "RawExactMatchingRows"
        ]
    )

    if raw_candidates == 0:
        return "NO_RAW_BUILD_TEST_MATCH"

    if verdict_matches == 0:
        return "RAW_MODEL_VERDICT_MISMATCH"

    if dataset_duration_column is not None:

        if pd.isna(
            row[
                "ModelDuration"
            ]
        ):
            return "MODEL_DURATION_MISSING"

        if exact_matches == 0:
            return "RAW_MODEL_DURATION_MISMATCH"

        if exact_matches == 1:
            return "EXACT_SINGLE_RAW_MATCH"

        return "MULTIPLE_IDENTICAL_RAW_MATCHES"

    raw_verdict_consensus = bool(
        row[
            "RawVerdictConsensus"
        ]
    ) if pd.notna(
        row[
            "RawVerdictConsensus"
        ]
    ) else False

    raw_duration_consensus = bool(
        row[
            "RawDurationConsensus"
        ]
    ) if pd.notna(
        row[
            "RawDurationConsensus"
        ]
    ) else False

    if not raw_verdict_consensus:
        return "AMBIGUOUS_RAW_VERDICT"

    if not raw_duration_consensus:
        return "AMBIGUOUS_RAW_DURATION"

    if raw_candidates == 1:
        return "CONSISTENT_SINGLE_RAW_MATCH"

    return "CONSISTENT_MULTIPLE_RAW_MATCHES"


alignment[
    "AlignmentStatus"
] = alignment.apply(
    determine_alignment_status,
    axis=1,
)


successful_alignment_statuses = {
    "EXACT_SINGLE_RAW_MATCH",
    "MULTIPLE_IDENTICAL_RAW_MATCHES",
    "CONSISTENT_SINGLE_RAW_MATCH",
    "CONSISTENT_MULTIPLE_RAW_MATCHES",
}


alignment[
    "AlignmentSuccessful"
] = alignment[
    "AlignmentStatus"
].isin(
    successful_alignment_statuses
)


# ------------------------------------------------------------
# 13. ALIGNMENT VALIDATION COUNTS
# ------------------------------------------------------------

model_rows_without_raw_pair = int(
    alignment[
        "RawCandidateRows"
    ].eq(0).sum()
)


model_verdict_mismatches = int(
    (
        alignment[
            "RawCandidateRows"
        ].gt(0)
        &
        alignment[
            "RawVerdictMatchingRows"
        ].eq(0)
    ).sum()
)


if dataset_duration_column is not None:

    model_duration_missing = int(
        alignment[
            "ModelDuration"
        ].isna().sum()
    )

    model_duration_mismatches = int(
        (
            alignment[
                "RawVerdictMatchingRows"
            ].gt(0)
            &
            alignment[
                "RawExactMatchingRows"
            ].eq(0)
        ).sum()
    )

else:

    model_duration_missing = 0

    model_duration_mismatches = int(
        (
            ~alignment[
                "RawDurationConsensus"
            ].fillna(False)
        ).sum()
    )


model_alignment_failures = int(
    (
        ~alignment[
            "AlignmentSuccessful"
        ]
    ).sum()
)


canonical_duration_missing = int(
    alignment[
        "CanonicalDuration"
    ].isna().sum()
)


canonical_negative_duration = int(
    alignment[
        "CanonicalDuration"
    ].lt(0).sum()
)


canonical_zero_duration = int(
    alignment[
        "CanonicalDuration"
    ].eq(0).sum()
)


alignment_status_counts = (
    alignment[
        "AlignmentStatus"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "AlignmentStatus"
    )
    .reset_index(
        name="Rows"
    )
)


print("\nAlignment status summary:")

display(
    alignment_status_counts
)


# ------------------------------------------------------------
# 14. BUILD-LEVEL COVERAGE
# ------------------------------------------------------------

raw_build_summary = (
    raw.groupby(
        [
            "CanonicalBuild",
            "Partition",
            "BuildOrder",
        ],
        dropna=False,
    )
    .agg(
        RawExecutionRows=(
            "_RawSourceRow",
            "size",
        ),

        RawUniqueTests=(
            "CanonicalTest",
            "nunique",
        ),

        RawFailureExecutions=(
            "RawIsFailure",
            "sum",
        ),
    )
    .reset_index()
)


raw_build_summary[
    "RawFailingBuild"
] = raw_build_summary[
    "RawFailureExecutions"
].gt(0)


model_build_summary = (
    alignment.groupby(
        [
            "CanonicalBuild",
            "Partition",
            "BuildOrder",
        ],
        dropna=False,
    )
    .agg(
        ModelReadyRows=(
            "_ModelSourceRow",
            "size",
        ),

        ModelReadyUniqueTests=(
            "CanonicalTest",
            "nunique",
        ),

        ModelReadyFailures=(
            "ModelIsFailure",
            "sum",
        ),

        SuccessfullyAlignedRows=(
            "AlignmentSuccessful",
            "sum",
        ),
    )
    .reset_index()
)


chronology_for_coverage = chronology[
    [
        "_BuildKey",
        "BuildOrder",
        "Partition",
    ]
].copy()


chronology_for_coverage = (
    chronology_for_coverage
    .rename(
        columns={
            "_BuildKey":
                "CanonicalBuild",
        }
    )
)


chronology_for_coverage[
    "BuildOrder"
] = pd.to_numeric(
    chronology_for_coverage[
        "BuildOrder"
    ],
    errors="raise",
).astype(int)


build_coverage = (
    chronology_for_coverage
    .merge(
        raw_build_summary,

        on=[
            "CanonicalBuild",
            "BuildOrder",
            "Partition",
        ],

        how="left",
        validate="one_to_one",
    )
    .merge(
        model_build_summary,

        on=[
            "CanonicalBuild",
            "BuildOrder",
            "Partition",
        ],

        how="left",
        validate="one_to_one",
    )
)


integer_coverage_columns = [
    "RawExecutionRows",
    "RawUniqueTests",
    "RawFailureExecutions",
    "ModelReadyRows",
    "ModelReadyUniqueTests",
    "ModelReadyFailures",
    "SuccessfullyAlignedRows",
]


for column in integer_coverage_columns:

    build_coverage[column] = (
        build_coverage[column]
        .fillna(0)
        .astype(int)
    )


build_coverage[
    "RawFailingBuild"
] = build_coverage[
    "RawFailureExecutions"
].gt(0)


build_coverage[
    "ModelReadyBuild"
] = build_coverage[
    "ModelReadyRows"
].gt(0)


build_coverage[
    "AllModelRowsAligned"
] = build_coverage[
    "ModelReadyRows"
].eq(
    build_coverage[
        "SuccessfullyAlignedRows"
    ]
)


def classify_build(row):
    raw_failing = bool(
        row[
            "RawFailingBuild"
        ]
    )

    model_ready = bool(
        row[
            "ModelReadyBuild"
        ]
    )

    if raw_failing and model_ready:
        return "RAW_FAILING_AND_MODEL_READY"

    if raw_failing and not model_ready:
        return "RAW_FAILING_NOT_MODEL_READY"

    if not raw_failing and model_ready:
        return "MODEL_READY_WITHOUT_RAW_FAILURE"

    return "RAW_NON_FAILING_NOT_MODEL_READY"


build_coverage[
    "BuildCoverageStatus"
] = build_coverage.apply(
    classify_build,
    axis=1,
)


build_coverage = (
    build_coverage
    .sort_values(
        "BuildOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


build_coverage_status_counts = (
    build_coverage[
        "BuildCoverageStatus"
    ]
    .value_counts()
    .rename_axis(
        "BuildCoverageStatus"
    )
    .reset_index(
        name="Builds"
    )
)


print("\nBuild coverage status summary:")

display(
    build_coverage_status_counts
)


raw_failing_not_model_ready = (
    build_coverage[
        build_coverage[
            "BuildCoverageStatus"
        ].eq(
            "RAW_FAILING_NOT_MODEL_READY"
        )
    ]
    .copy()
)


if not raw_failing_not_model_ready.empty:

    print(
        "\nDocumented raw failing builds absent "
        "from dataset.csv:"
    )

    display(
        raw_failing_not_model_ready[
            [
                "CanonicalBuild",
                "BuildOrder",
                "Partition",
                "RawExecutionRows",
                "RawFailureExecutions",
                "BuildCoverageStatus",
            ]
        ]
    )


raw_failing_training_not_model_ready = int(
    (
        build_coverage[
            "Partition"
        ].eq("training")
        &
        build_coverage[
            "BuildCoverageStatus"
        ].eq(
            "RAW_FAILING_NOT_MODEL_READY"
        )
    ).sum()
)


raw_failing_evaluation_not_model_ready = int(
    (
        build_coverage[
            "Partition"
        ].eq("evaluation")
        &
        build_coverage[
            "BuildCoverageStatus"
        ].eq(
            "RAW_FAILING_NOT_MODEL_READY"
        )
    ).sum()
)


model_ready_without_raw_failure = int(
    build_coverage[
        "BuildCoverageStatus"
    ].eq(
        "MODEL_READY_WITHOUT_RAW_FAILURE"
    ).sum()
)


model_build_alignment_failures = int(
    (
        build_coverage[
            "ModelReadyBuild"
        ]
        &
        ~build_coverage[
            "AllModelRowsAligned"
        ]
    ).sum()
)


# ------------------------------------------------------------
# 15. VERIFY FROZEN STEP 3 COUNTS
# ------------------------------------------------------------

expected_training_model_rows = expected_integer(
    screening_row,
    "ModelTrainingRows",
)

expected_evaluation_model_rows = expected_integer(
    screening_row,
    "ModelEvaluationRows",
)

expected_training_model_builds = expected_integer(
    screening_row,
    "ModelTrainingBuilds",
)

expected_evaluation_model_builds = expected_integer(
    screening_row,
    "ModelEvaluationBuilds",
)

expected_training_model_failures = expected_integer(
    screening_row,
    "ModelTrainingFailures",
)

expected_evaluation_model_failures = expected_integer(
    screening_row,
    "ModelEvaluationFailures",
)


if None in {
    expected_training_model_rows,
    expected_evaluation_model_rows,
    expected_training_model_builds,
    expected_evaluation_model_builds,
    expected_training_model_failures,
    expected_evaluation_model_failures,
}:
    raise RuntimeError(
        "The screening snapshot is missing required "
        "model-ready counts."
    )


expected_model_rows = (
    expected_training_model_rows
    + expected_evaluation_model_rows
)


expected_model_builds = (
    expected_training_model_builds
    + expected_evaluation_model_builds
)


expected_model_failures = (
    expected_training_model_failures
    + expected_evaluation_model_failures
)


actual_model_builds = int(
    alignment[
        "CanonicalBuild"
    ].nunique()
)


actual_model_failures = int(
    alignment[
        "ModelIsFailure"
    ].sum()
)


actual_successfully_aligned_rows = int(
    alignment[
        "AlignmentSuccessful"
    ].sum()
)


# ------------------------------------------------------------
# 16. CREATE STEP 4 AUDIT
# ------------------------------------------------------------

audit_records = [
    {
        "Check":
            "Raw source rows",

        "Expected":
            len(exe),

        "Actual":
            len(raw),

        "Pass":
            len(raw) == len(exe),
    },

    {
        "Check":
            "Model-ready rows",

        "Expected":
            expected_model_rows,

        "Actual":
            len(model),

        "Pass":
            len(model) == expected_model_rows,
    },

    {
        "Check":
            "Model-ready builds",

        "Expected":
            expected_model_builds,

        "Actual":
            actual_model_builds,

        "Pass":
            actual_model_builds
            == expected_model_builds,
    },

    {
        "Check":
            "Model-ready failures",

        "Expected":
            expected_model_failures,

        "Actual":
            actual_model_failures,

        "Pass":
            actual_model_failures
            == expected_model_failures,
    },

    {
        "Check":
            "Missing raw build IDs",

        "Expected":
            0,

        "Actual":
            raw_missing_build,

        "Pass":
            raw_missing_build == 0,
    },

    {
        "Check":
            "Missing raw test IDs",

        "Expected":
            0,

        "Actual":
            raw_missing_test,

        "Pass":
            raw_missing_test == 0,
    },

    {
        "Check":
            "Missing raw verdicts",

        "Expected":
            0,

        "Actual":
            raw_missing_verdict,

        "Pass":
            raw_missing_verdict == 0,
    },

    {
        "Check":
            "Missing raw durations",

        "Expected":
            0,

        "Actual":
            raw_missing_duration,

        "Pass":
            raw_missing_duration == 0,
    },

    {
        "Check":
            "Raw rows unmatched to chronology",

        "Expected":
            0,

        "Actual":
            raw_unmatched_chronology,

        "Pass":
            raw_unmatched_chronology == 0,
    },

    {
        "Check":
            "Missing model build IDs",

        "Expected":
            0,

        "Actual":
            model_missing_build,

        "Pass":
            model_missing_build == 0,
    },

    {
        "Check":
            "Missing model test IDs",

        "Expected":
            0,

        "Actual":
            model_missing_test,

        "Pass":
            model_missing_test == 0,
    },

    {
        "Check":
            "Missing model verdicts",

        "Expected":
            0,

        "Actual":
            model_missing_verdict,

        "Pass":
            model_missing_verdict == 0,
    },

    {
        "Check":
            "Model rows unmatched to chronology",

        "Expected":
            0,

        "Actual":
            model_unmatched_chronology,

        "Pass":
            model_unmatched_chronology == 0,
    },

    {
        "Check":
            "Duplicate model Build-Test groups",

        "Expected":
            0,

        "Actual":
            model_duplicate_pair_groups,

        "Pass":
            model_duplicate_pair_groups == 0,
    },

    {
        "Check":
            "Model rows without raw Build-Test pair",

        "Expected":
            0,

        "Actual":
            model_rows_without_raw_pair,

        "Pass":
            model_rows_without_raw_pair == 0,
    },

    {
        "Check":
            "Raw-model verdict mismatches",

        "Expected":
            0,

        "Actual":
            model_verdict_mismatches,

        "Pass":
            model_verdict_mismatches == 0,
    },

    {
        "Check":
            "Raw-model duration mismatches",

        "Expected":
            0,

        "Actual":
            model_duration_mismatches,

        "Pass":
            model_duration_mismatches == 0,
    },

    {
        "Check":
            "Unsuccessful model-row alignments",

        "Expected":
            0,

        "Actual":
            model_alignment_failures,

        "Pass":
            model_alignment_failures == 0,
    },

    {
        "Check":
            "Canonical durations missing",

        "Expected":
            0,

        "Actual":
            canonical_duration_missing,

        "Pass":
            canonical_duration_missing == 0,
    },

    {
        "Check":
            "Negative canonical durations",

        "Expected":
            0,

        "Actual":
            canonical_negative_duration,

        "Pass":
            canonical_negative_duration == 0,
    },

    {
        "Check":
            "Model-ready builds without raw failure",

        "Expected":
            0,

        "Actual":
            model_ready_without_raw_failure,

        "Pass":
            model_ready_without_raw_failure == 0,
    },

    {
        "Check":
            "Model-ready builds with alignment failures",

        "Expected":
            0,

        "Actual":
            model_build_alignment_failures,

        "Pass":
            model_build_alignment_failures == 0,
    },

    {
        "Check":
            "Evaluation failing builds absent from model data",

        "Expected":
            0,

        "Actual":
            raw_failing_evaluation_not_model_ready,

        "Pass":
            raw_failing_evaluation_not_model_ready == 0,
    },

    {
        "Check":
            "Successfully aligned model rows",

        "Expected":
            expected_model_rows,

        "Actual":
            actual_successfully_aligned_rows,

        "Pass":
            actual_successfully_aligned_rows
            == expected_model_rows,
    },
]


audit_frame = pd.DataFrame(
    audit_records
)


print("\nStep 4 validation audit:")

display(
    audit_frame
)


failed_audit_checks = (
    audit_frame[
        ~audit_frame[
            "Pass"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 17. PREPARE OUTPUT TABLES
# ------------------------------------------------------------

raw_output_columns = [
    "_RawSourceRow",
    "CanonicalBuild",
    "BuildOrder",
    "Partition",
    "CanonicalTest",
    "CanonicalJob",
    "RawVerdictToken",
    "RawIsFailure",
    "RawDuration",
    exe_build_column,
    exe_test_column,
    exe_verdict_column,
    exe_duration_column,
]


if (
    exe_job_column is not None
    and exe_job_column not in raw_output_columns
):
    raw_output_columns.append(
        exe_job_column
    )


canonical_raw_output = raw[
    list(
        dict.fromkeys(
            raw_output_columns
        )
    )
].copy()


alignment_output_columns = [
    "_ModelSourceRow",
    "CanonicalBuild",
    "BuildOrder",
    "Partition",
    "CanonicalTest",
    "ModelVerdictToken",
    "ModelIsFailure",
    "ModelDuration",
    "CanonicalVerdictToken",
    "CanonicalIsFailure",
    "CanonicalDuration",
    "CanonicalDurationSource",
    "RawExecutionRows",
    "RawCandidateRows",
    "RawJobCount",
    "RawJobs",
    "RawVerdictTokens",
    "RawVerdictConsensus",
    "RawDurationMin",
    "RawDurationMax",
    "RawDurationConsensus",
    "RawVerdictMatchingRows",
    "RawExactMatchingRows",
    "VerdictMatchingRawSourceRows",
    "ExactMatchingRawSourceRows",
    "ModelRawVerdictMatches",
    "ModelRawDurationMatches",
    "AlignmentStatus",
    "AlignmentSuccessful",
]


model_alignment_output = alignment[
    alignment_output_columns
].copy()


# ------------------------------------------------------------
# 18. DETERMINE STEP 4 STATUS
# ------------------------------------------------------------

step4_pass = bool(
    len(
        failed_audit_checks
    )
    == 0
)


step4_status_text = (
    "PASS_PROJECT_8_CANONICAL_EXECUTION_ALIGNMENT_COMPLETE"
    if step4_pass
    else
    "FAIL_PROJECT_8_CANONICAL_EXECUTION_ALIGNMENT"
)


# ------------------------------------------------------------
# 19. SAVE STEP 4 TABLES
# ------------------------------------------------------------

PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


canonical_raw_output.to_csv(
    CANONICAL_RAW_PATH,
    index=False,
    compression="gzip",
)


raw_summary.to_csv(
    RAW_BUILD_TEST_SUMMARY_PATH,
    index=False,
    compression="gzip",
)


model_alignment_output.to_csv(
    MODEL_ALIGNMENT_PATH,
    index=False,
    compression="gzip",
)


build_coverage.to_csv(
    BUILD_COVERAGE_PATH,
    index=False,
)


audit_frame.to_csv(
    ALIGNMENT_AUDIT_PATH,
    index=False,
)


# ------------------------------------------------------------
# 20. VERIFY AND HASH WRITTEN TABLES
# ------------------------------------------------------------

table_outputs = [
    CANONICAL_RAW_PATH,
    RAW_BUILD_TEST_SUMMARY_PATH,
    MODEL_ALIGNMENT_PATH,
    BUILD_COVERAGE_PATH,
    ALIGNMENT_AUDIT_PATH,
]


missing_table_outputs = [
    str(path)
    for path in table_outputs
    if not path.exists()
]


if missing_table_outputs:
    raise RuntimeError(
        "Step 4 failed to write expected tables:\n"
        + "\n".join(
            missing_table_outputs
        )
    )


output_metadata = {
    path.name: file_metadata(
        path
    )
    for path in table_outputs
}


# ------------------------------------------------------------
# 21. CREATE STEP 4 REPORT
# ------------------------------------------------------------

step4_report = {
    "project_number":
        PROJECT_NUMBER,

    "project":
        PROJECT_NAME,

    "project_slug":
        PROJECT_SLUG,

    "generated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "source_columns": {
        "exe_build":
            exe_build_column,

        "exe_job":
            exe_job_column,

        "exe_test":
            exe_test_column,

        "exe_verdict":
            exe_verdict_column,

        "exe_duration":
            exe_duration_column,

        "dataset_build":
            dataset_build_column,

        "dataset_test":
            dataset_test_column,

        "dataset_verdict":
            dataset_verdict_column,

        "dataset_duration":
            dataset_duration_column,
    },

    "verdict_mappings": {
        "raw_failure_tokens":
            sorted(
                raw_failure_tokens
            ),

        "model_failure_tokens":
            sorted(
                model_failure_tokens
            ),
    },

    "raw_execution_summary": {
        "rows":
            int(
                len(raw)
            ),

        "builds":
            int(
                raw[
                    "CanonicalBuild"
                ].nunique()
            ),

        "tests":
            int(
                raw[
                    "CanonicalTest"
                ].nunique()
            ),

        "build_test_pairs":
            int(
                len(
                    raw_summary
                )
            ),

        "duplicate_build_test_pairs":
            raw_duplicate_build_test_pairs,

        "verdict_conflict_pairs":
            raw_verdict_conflict_pairs,

        "duration_conflict_pairs":
            raw_duration_conflict_pairs,

        "missing_build_ids":
            raw_missing_build,

        "missing_test_ids":
            raw_missing_test,

        "missing_verdicts":
            raw_missing_verdict,

        "missing_durations":
            raw_missing_duration,

        "negative_durations":
            raw_negative_duration,

        "zero_durations":
            raw_zero_duration,
    },

    "model_alignment_summary": {
        "model_rows":
            int(
                len(model)
            ),

        "model_builds":
            actual_model_builds,

        "model_duplicate_pair_rows":
            model_duplicate_pair_rows,

        "model_duplicate_pair_groups":
            model_duplicate_pair_groups,

        "rows_without_raw_pair":
            model_rows_without_raw_pair,

        "verdict_mismatches":
            model_verdict_mismatches,

        "duration_validation_applied":
            dataset_duration_column
            is not None,

        "duration_missing":
            model_duration_missing,

        "duration_mismatches":
            model_duration_mismatches,

        "unsuccessful_alignments":
            model_alignment_failures,

        "successfully_aligned_rows":
            actual_successfully_aligned_rows,

        "canonical_duration_missing":
            canonical_duration_missing,

        "canonical_negative_duration":
            canonical_negative_duration,

        "canonical_zero_duration":
            canonical_zero_duration,

        "alignment_status_counts":
            alignment_status_counts
            .to_dict(
                orient="records"
            ),
    },

    "build_coverage_summary": {
        "total_builds":
            int(
                len(
                    build_coverage
                )
            ),

        "raw_failing_training_builds_not_model_ready":
            raw_failing_training_not_model_ready,

        "raw_failing_evaluation_builds_not_model_ready":
            raw_failing_evaluation_not_model_ready,

        "model_ready_without_raw_failure":
            model_ready_without_raw_failure,

        "model_build_alignment_failures":
            model_build_alignment_failures,

        "coverage_status_counts":
            build_coverage_status_counts
            .to_dict(
                orient="records"
            ),

        "documented_raw_failing_builds_not_model_ready":
            raw_failing_not_model_ready[
                [
                    "CanonicalBuild",
                    "BuildOrder",
                    "Partition",
                    "RawExecutionRows",
                    "RawFailureExecutions",
                ]
            ]
            .to_dict(
                orient="records"
            ),
    },

    "audit":
        audit_frame.to_dict(
            orient="records"
        ),

    "failed_audit_check_count":
        int(
            len(
                failed_audit_checks
            )
        ),

    "outputs":
        output_metadata,

    "status":
        step4_status_text,
}


STEP4_REPORT_PATH.write_text(
    json.dumps(
        step4_report,
        indent=2,
        default=json_safe,
    ),
    encoding="utf-8",
)


step4_status = {
    "project_number":
        PROJECT_NUMBER,

    "project":
        PROJECT_NAME,

    "project_slug":
        PROJECT_SLUG,

    "status":
        step4_status_text,

    "model_rows":
        int(
            len(model)
        ),

    "successfully_aligned_rows":
        actual_successfully_aligned_rows,

    "failed_audit_check_count":
        int(
            len(
                failed_audit_checks
            )
        ),

    "generated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


STEP4_STATUS_PATH.write_text(
    json.dumps(
        step4_status,
        indent=2,
        default=json_safe,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 22. VERIFY ALL STEP 4 OUTPUTS
# ------------------------------------------------------------

all_expected_outputs = (
    table_outputs
    + [
        STEP4_REPORT_PATH,
        STEP4_STATUS_PATH,
    ]
)


missing_outputs = [
    str(path)
    for path in all_expected_outputs
    if not path.exists()
]


if missing_outputs:
    raise RuntimeError(
        "Some Step 4 outputs are missing:\n"
        + "\n".join(
            missing_outputs
        )
    )


# ------------------------------------------------------------
# 23. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 80)
print("=== PROJECT 8 STEP 4 RESULT ===")
print("=" * 80)

print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nRaw canonical executions:")

print(
    "Raw rows:",
    len(raw),
)

print(
    "Raw builds:",
    raw[
        "CanonicalBuild"
    ].nunique(),
)

print(
    "Raw tests:",
    raw[
        "CanonicalTest"
    ].nunique(),
)

print(
    "Unique raw Build-Test pairs:",
    len(
        raw_summary
    ),
)

print(
    "Duplicate raw Build-Test pairs:",
    raw_duplicate_build_test_pairs,
)

print(
    "Raw verdict-conflict pairs:",
    raw_verdict_conflict_pairs,
)

print(
    "Raw duration-conflict pairs:",
    raw_duration_conflict_pairs,
)


print("\nModel-ready alignment:")

print(
    "Model-ready rows:",
    len(model),
)

print(
    "Model-ready builds:",
    actual_model_builds,
)

print(
    "Rows successfully aligned:",
    actual_successfully_aligned_rows,
)

print(
    "Rows without raw match:",
    model_rows_without_raw_pair,
)

print(
    "Verdict mismatches:",
    model_verdict_mismatches,
)

print(
    "Duration column:",
    (
        dataset_duration_column
        if dataset_duration_column is not None
        else "NOT PRESENT"
    ),
)

print(
    "Duration mismatches:",
    model_duration_mismatches,
)

print(
    "Canonical durations missing:",
    canonical_duration_missing,
)

print(
    "Unsuccessful alignments:",
    model_alignment_failures,
)


print("\nBuild coverage:")

print(
    "Raw failing training builds not model-ready:",
    raw_failing_training_not_model_ready,
)

print(
    "Raw failing evaluation builds not model-ready:",
    raw_failing_evaluation_not_model_ready,
)

print(
    "Model-ready builds without raw failure:",
    model_ready_without_raw_failure,
)


print("\nValidation:")

print(
    "Audit checks:",
    len(
        audit_frame
    ),
)

print(
    "Failed audit checks:",
    len(
        failed_audit_checks
    ),
)


print("\nStep 4 outputs:")

for output_path in all_expected_outputs:
    print(
        output_path
    )


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–7 modified:")
print(0)


print(
    "\nSTATUS:",
    step4_status_text,
)

print("=" * 80)


if not step4_pass:

    print(
        "\nFailed Step 4 audit checks:"
    )

    display(
        failed_audit_checks
    )

    print(
        "\nUnsuccessful alignment statuses:"
    )

    display(
        alignment.loc[
            ~alignment[
                "AlignmentSuccessful"
            ],
            [
                "_ModelSourceRow",
                "CanonicalBuild",
                "CanonicalTest",
                "ModelVerdictToken",
                "ModelDuration",
                "RawCandidateRows",
                "RawVerdictMatchingRows",
                "RawExactMatchingRows",
                "AlignmentStatus",
            ],
        ].head(100)
    )

    raise RuntimeError(
        "PROJECT 8 STEP 4 DID NOT PASS.\n"
        "Do not proceed to entity mapping until the "
        "displayed alignment problems are resolved."
    )

=== PROJECT 8 STEP 4: CANONICAL EXECUTION ALIGNMENT ===

Loading Step 4 inputs...
exe.csv: (34438, 5)
dataset.csv: (9880, 154)
build chronology: (254, 6)

Frozen source columns:
exe build: build
exe job: job
exe test: test
exe verdict: verdict
exe duration: duration
dataset build: Build
dataset test: Test
dataset verdict: Verdict
dataset duration: Duration

Alignment status summary:


,AlignmentStatus,Rows
0,EXACT_SINGLE_RAW_MATCH,9880



Build coverage status summary:


,BuildCoverageStatus,Builds
0,RAW_NON_FAILING_NOT_MODEL_READY,181
1,RAW_FAILING_AND_MODEL_READY,72
2,RAW_FAILING_NOT_MODEL_READY,1



Documented raw failing builds absent from dataset.csv:


,CanonicalBuild,BuildOrder,Partition,RawExecutionRows,RawFailureExecutions,BuildCoverageStatus
0,564318082,1,training,126,2,RAW_FAILING_NOT_MODEL_READY



Step 4 validation audit:


,Check,Expected,Actual,Pass
0,Raw source rows,34438,34438,True
1,Model-ready rows,9880,9880,True
2,Model-ready builds,72,72,True
3,Model-ready failures,80,80,True
4,Missing raw build IDs,0,0,True
5,Missing raw test IDs,0,0,True
6,Missing raw verdicts,0,0,True
7,Missing raw durations,0,0,True
8,Raw rows unmatched to chronology,0,0,True
9,Missing model build IDs,0,0,True




=== PROJECT 8 STEP 4 RESULT ===

Project identity:
Project number: 8
Project: optimatika@ojAlgo
Project slug: optimatika__ojAlgo

Raw canonical executions:
Raw rows: 34438
Raw builds: 254
Raw tests: 146
Unique raw Build-Test pairs: 34438
Duplicate raw Build-Test pairs: 0
Raw verdict-conflict pairs: 0
Raw duration-conflict pairs: 0

Model-ready alignment:
Model-ready rows: 9880
Model-ready builds: 72
Rows successfully aligned: 9880
Rows without raw match: 0
Verdict mismatches: 0
Duration column: Duration
Duration mismatches: 0
Canonical durations missing: 0
Unsuccessful alignments: 0

Build coverage:
Raw failing training builds not model-ready: 1
Raw failing evaluation builds not model-ready: 0
Model-ready builds without raw failure: 0

Validation:
Audit checks: 24
Failed audit checks: 0

Step 4 outputs:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_preflight/ojalgo_canonical_raw_executions.csv.gz
/content/drive/MyDrive/Thesis_Experiment/Results

In [ ]:
# ============================================================
# PROJECT 8 — STEP 5
# CANONICAL COMMIT–ENTITY MAPPING
#
# PROJECT: optimatika@ojAlgo
#
# This cell:
# - validates the passed Step 4 state
# - detects entity-history and id-map schemas
# - removes exact duplicate entity-history rows
# - splits builds.csv commit lists using '#'
# - maps exact full commit hashes to changed entities
# - validates every EntityId against id_map.csv
# - creates canonical build–commit–entity mappings
# - creates the build–entity map required by REC
# - creates a changed-entity dictionary for all 254 builds
# - documents every unmatched commit
#
# It does NOT:
# - perform prefix/short-hash matching
# - fabricate synthetic entity mappings
# - modify source CSV files
# - modify the completion registry
# - modify Projects 1–7
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import re

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

PROJECT_NUMBER = 8
PROJECT_NAME = "optimatika@ojAlgo"
PROJECT_SLUG = "optimatika__ojAlgo"
PROJECT_SHORT_NAME = "ojalgo"

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

PROJECT_SOURCE_DIR = Path(
    "/content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo"
)

PREFLIGHT_DIR = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / PROJECT_SLUG
    / f"{PROJECT_SHORT_NAME}_preflight"
)


# Source files

BUILDS_PATH = (
    PROJECT_SOURCE_DIR
    / "builds.csv"
)

ID_MAP_PATH = (
    PROJECT_SOURCE_DIR
    / "id_map.csv"
)

ENTITY_HISTORY_PATH = (
    PROJECT_SOURCE_DIR
    / "entity_change_history.csv"
)


# Previous-step inputs

IDENTITY_LOCK_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_project_identity_lock.json"
)

BUILD_CHRONOLOGY_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_build_chronology.csv"
)

BUILD_COVERAGE_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_execution_build_coverage.csv"
)

STEP4_STATUS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step4_status.json"
)


# Step 5 outputs

BUILD_COMMIT_TOKENS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_build_commit_tokens.csv"
)

BUILD_COMMIT_ENTITY_MAP_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_build_commit_entity_map.csv.gz"
)

BUILD_ENTITY_MAP_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_build_entity_map.csv.gz"
)

CHANGED_ENTITIES_DICTIONARY_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_changed_entities_by_build.json"
)

MAPPING_STATUS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_entity_mapping_status_by_build.csv"
)

UNMATCHED_COMMITS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_unmatched_commit_tokens.csv"
)

ENTITY_MAPPING_AUDIT_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_entity_mapping_audit.csv"
)

ENTITY_MAPPING_REPORT_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_entity_mapping_report.json"
)

STEP5_STATUS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step5_status.json"
)


# Engineering guard against accidentally accepting a
# completely incorrect schema or commit column.
MINIMUM_COMMIT_TOKEN_COVERAGE_PERCENT = 95.0


print("=" * 80)
print("=== PROJECT 8 STEP 5: CANONICAL ENTITY MAPPING ===")
print("=" * 80)


# ------------------------------------------------------------
# 2. HELPERS
# ------------------------------------------------------------

def normalise_name(value):
    if value is None:
        return ""

    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).strip().lower(),
    )


def canonical_id(value):
    if pd.isna(value):
        return None

    text = str(value).strip()

    if text == "":
        return None

    if re.fullmatch(
        r"[-+]?\d+\.0+",
        text,
    ):
        return text.split(".")[0]

    return text


def canonical_commit(value):
    if pd.isna(value):
        return None

    text = str(value).strip().lower()

    if text == "":
        return None

    return text


def is_full_commit_hash(value):
    if value is None:
        return False

    return bool(
        re.fullmatch(
            r"[0-9a-f]{40}",
            str(value).strip().lower(),
        )
    )


def calculate_sha256(
    file_path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(file_path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def file_metadata(file_path):
    file_path = Path(file_path)

    return {
        "path": str(file_path),

        "size_bytes": int(
            file_path.stat().st_size
        ),

        "sha256": calculate_sha256(
            file_path
        ),
    }


def json_safe(value):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return value


def find_named_column(
    dataframe,
    candidates,
):
    lookup = {
        normalise_name(column): column
        for column in dataframe.columns
    }

    for candidate in candidates:
        normalised_candidate = normalise_name(
            candidate
        )

        if normalised_candidate in lookup:
            return lookup[
                normalised_candidate
            ]

    return None


def detect_commit_column(dataframe):
    """
    Prefer a clearly named commit/hash column.

    If names are unfamiliar, select the column with the
    highest proportion of full 40-character hexadecimal
    hashes.
    """

    named_column = find_named_column(
        dataframe,
        candidates=[
            "commit",
            "commits",
            "commit_hash",
            "commithash",
            "sha",
            "sha1",
            "hash",
            "revision",
        ],
    )

    if named_column is not None:
        return named_column

    candidate_scores = []

    for column in dataframe.columns:
        values = (
            dataframe[column]
            .dropna()
            .astype(str)
            .str.strip()
            .str.lower()
        )

        if len(values) == 0:
            continue

        full_hash_fraction = float(
            values.str.fullmatch(
                r"[0-9a-f]{40}"
            ).mean()
        )

        hash_like_fraction = float(
            values.str.fullmatch(
                r"[0-9a-f]{7,40}"
            ).mean()
        )

        candidate_scores.append({
            "Column": column,
            "FullHashFraction":
                full_hash_fraction,
            "HashLikeFraction":
                hash_like_fraction,
        })

    if not candidate_scores:
        raise RuntimeError(
            "No candidate commit column was found in "
            "entity_change_history.csv."
        )

    score_frame = (
        pd.DataFrame(
            candidate_scores
        )
        .sort_values(
            [
                "FullHashFraction",
                "HashLikeFraction",
            ],
            ascending=False,
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    best = score_frame.iloc[0]

    if float(
        best["HashLikeFraction"]
    ) < 0.50:
        print(
            "\nCommit-column detection scores:"
        )
        display(score_frame)

        raise RuntimeError(
            "No entity-history column contains a "
            "credible proportion of commit hashes."
        )

    return str(
        best["Column"]
    )


def detect_id_map_entity_column(dataframe):
    """
    Detect the integer EntityId column in id_map.csv.
    """

    named_column = find_named_column(
        dataframe,
        candidates=[
            "EntityId",
            "EntityID",
            "entity_id",
            "entityid",
            "id",
            "ID",
        ],
    )

    if named_column is not None:
        numeric_values = pd.to_numeric(
            dataframe[named_column],
            errors="coerce",
        )

        if numeric_values.notna().mean() >= 0.95:
            return named_column

    candidate_scores = []

    for column in dataframe.columns:
        numeric_values = pd.to_numeric(
            dataframe[column],
            errors="coerce",
        )

        numeric_fraction = float(
            numeric_values.notna().mean()
        )

        unique_numeric = int(
            numeric_values.nunique(
                dropna=True
            )
        )

        candidate_scores.append({
            "Column": column,
            "NumericFraction":
                numeric_fraction,
            "UniqueNumericValues":
                unique_numeric,
        })

    score_frame = (
        pd.DataFrame(
            candidate_scores
        )
        .sort_values(
            [
                "NumericFraction",
                "UniqueNumericValues",
            ],
            ascending=False,
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    best = score_frame.iloc[0]

    if float(
        best["NumericFraction"]
    ) < 0.95:
        print(
            "\nid_map.csv ID-column detection scores:"
        )
        display(score_frame)

        raise RuntimeError(
            "No credible numeric EntityId column was "
            "found in id_map.csv."
        )

    return str(
        best["Column"]
    )


def detect_history_entity_column(
    dataframe,
    valid_entity_ids,
    commit_column,
):
    """
    Detect the EntityId column in entity-change history.

    Candidate numeric columns are scored by their overlap
    with the IDs in id_map.csv.
    """

    named_candidates = [
        "EntityId",
        "EntityID",
        "entity_id",
        "entityid",
        "id",
        "ID",
    ]

    named_column = find_named_column(
        dataframe,
        candidates=named_candidates,
    )

    columns_to_score = [
        column
        for column in dataframe.columns
        if column != commit_column
    ]

    if (
        named_column is not None
        and named_column != commit_column
    ):
        columns_to_score = [
            named_column
        ] + [
            column
            for column in columns_to_score
            if column != named_column
        ]

    candidate_scores = []

    for column in columns_to_score:
        numeric_values = pd.to_numeric(
            dataframe[column],
            errors="coerce",
        )

        numeric_non_null = numeric_values.dropna()

        if len(numeric_non_null) == 0:
            continue

        numeric_fraction = float(
            numeric_values.notna().mean()
        )

        numeric_as_int = set(
            numeric_non_null.astype(
                np.int64
            ).tolist()
        )

        overlap_count = len(
            numeric_as_int.intersection(
                valid_entity_ids
            )
        )

        overlap_fraction = (
            overlap_count
            / len(numeric_as_int)
            if numeric_as_int
            else 0.0
        )

        name_bonus = int(
            normalise_name(column)
            in {
                normalise_name(candidate)
                for candidate
                in named_candidates
            }
        )

        candidate_scores.append({
            "Column": column,
            "NumericFraction":
                numeric_fraction,
            "UniqueNumericValues":
                len(numeric_as_int),
            "IdMapOverlapCount":
                overlap_count,
            "IdMapOverlapFraction":
                overlap_fraction,
            "NameBonus":
                name_bonus,
        })

    if not candidate_scores:
        raise RuntimeError(
            "No numeric EntityId candidate was found in "
            "entity_change_history.csv."
        )

    score_frame = (
        pd.DataFrame(
            candidate_scores
        )
        .sort_values(
            [
                "IdMapOverlapFraction",
                "IdMapOverlapCount",
                "NameBonus",
                "NumericFraction",
            ],
            ascending=False,
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    best = score_frame.iloc[0]

    if (
        float(
            best["NumericFraction"]
        ) < 0.95
        or float(
            best["IdMapOverlapFraction"]
        ) < 0.95
    ):
        print(
            "\nEntity-history EntityId detection scores:"
        )
        display(score_frame)

        raise RuntimeError(
            "No entity-history column maps reliably to "
            "id_map.csv EntityIds."
        )

    return str(
        best["Column"]
    )


def mapping_status(row):
    total_tokens = int(
        row["CommitTokens"]
    )

    matched_tokens = int(
        row["MatchedCommitTokens"]
    )

    if total_tokens == 0:
        return "NO_COMMIT_TOKENS"

    if matched_tokens == total_tokens:
        return "FULL_COMMIT_COVERAGE"

    if matched_tokens == 0:
        return "NO_MATCHED_COMMITS"

    return "PARTIAL_COMMIT_COVERAGE"


# ------------------------------------------------------------
# 3. VALIDATE REQUIRED INPUTS
# ------------------------------------------------------------

required_inputs = [
    BUILDS_PATH,
    ID_MAP_PATH,
    ENTITY_HISTORY_PATH,
    IDENTITY_LOCK_PATH,
    BUILD_CHRONOLOGY_PATH,
    BUILD_COVERAGE_PATH,
    STEP4_STATUS_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Required Project 8 Step 5 inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
        + "\n\nIf the /content source directory is "
        "missing, rerun the runtime-restoration cell."
    )


identity_lock = json.loads(
    IDENTITY_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)

if (
    identity_lock.get("project_number")
    != PROJECT_NUMBER
    or identity_lock.get("project")
    != PROJECT_NAME
    or identity_lock.get("project_slug")
    != PROJECT_SLUG
):
    raise AssertionError(
        "The Project 8 identity lock does not match "
        "the configured project."
    )


step4_status = json.loads(
    STEP4_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

expected_step4_status = (
    "PASS_PROJECT_8_CANONICAL_EXECUTION_ALIGNMENT_COMPLETE"
)

if (
    step4_status.get("status")
    != expected_step4_status
):
    raise AssertionError(
        "Step 4 has not passed successfully.\n"
        f"Detected status: "
        f"{step4_status.get('status')}"
    )


# ------------------------------------------------------------
# 4. LOAD INPUT TABLES
# ------------------------------------------------------------

print("\nLoading Step 5 inputs...")

builds = pd.read_csv(
    BUILDS_PATH,
    dtype=str,
)

chronology = pd.read_csv(
    BUILD_CHRONOLOGY_PATH,
    dtype=str,
)

build_coverage = pd.read_csv(
    BUILD_COVERAGE_PATH,
    dtype=str,
)

id_map = pd.read_csv(
    ID_MAP_PATH,
    dtype=str,
)

entity_history_raw = pd.read_csv(
    ENTITY_HISTORY_PATH,
    dtype=str,
)


print(
    "builds.csv:",
    builds.shape,
)

print(
    "build chronology:",
    chronology.shape,
)

print(
    "build coverage:",
    build_coverage.shape,
)

print(
    "id_map.csv:",
    id_map.shape,
)

print(
    "entity_change_history.csv:",
    entity_history_raw.shape,
)


print("\nid_map.csv columns:")
print(
    list(id_map.columns)
)

print("\nentity_change_history.csv columns:")
print(
    list(entity_history_raw.columns)
)


# ------------------------------------------------------------
# 5. VALIDATE CHRONOLOGY
# ------------------------------------------------------------

required_chronology_columns = {
    "CanonicalBuild",
    "BuildOrder",
    "Partition",
    "OriginalCommitField",
}

missing_chronology_columns = (
    required_chronology_columns
    - set(chronology.columns)
)

if missing_chronology_columns:
    raise RuntimeError(
        "The Step 3 chronology file is missing columns:\n"
        f"{sorted(missing_chronology_columns)}"
    )


chronology_working = chronology[
    [
        "CanonicalBuild",
        "BuildOrder",
        "Partition",
        "OriginalCommitField",
    ]
].copy()


chronology_working[
    "CanonicalBuild"
] = chronology_working[
    "CanonicalBuild"
].map(
    canonical_id
)


chronology_working[
    "BuildOrder"
] = pd.to_numeric(
    chronology_working[
        "BuildOrder"
    ],
    errors="raise",
).astype(int)


if chronology_working[
    "CanonicalBuild"
].isna().any():
    raise AssertionError(
        "The build chronology contains missing build IDs."
    )


if chronology_working[
    "CanonicalBuild"
].duplicated().any():
    raise AssertionError(
        "The build chronology contains duplicated builds."
    )


chronology_working = (
    chronology_working
    .sort_values(
        "BuildOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. DETECT ID-MAP AND HISTORY SCHEMAS
# ------------------------------------------------------------

id_map_entity_column = (
    detect_id_map_entity_column(
        id_map
    )
)


id_map_entity_numeric = pd.to_numeric(
    id_map[
        id_map_entity_column
    ],
    errors="coerce",
)


id_map_invalid_entity_rows = int(
    id_map_entity_numeric
    .isna()
    .sum()
)


if id_map_invalid_entity_rows != 0:
    raise RuntimeError(
        "id_map.csv contains non-numeric or missing "
        "EntityIds:\n"
        f"{id_map_invalid_entity_rows} rows"
    )


id_map_entity_ids = set(
    id_map_entity_numeric
    .astype(np.int64)
    .tolist()
)


history_commit_column = detect_commit_column(
    entity_history_raw
)


history_entity_column = (
    detect_history_entity_column(
        dataframe=
            entity_history_raw,

        valid_entity_ids=
            id_map_entity_ids,

        commit_column=
            history_commit_column,
    )
)


print("\nDetected schema:")

print(
    "id_map.csv EntityId column:",
    id_map_entity_column,
)

print(
    "entity history commit column:",
    history_commit_column,
)

print(
    "entity history EntityId column:",
    history_entity_column,
)


# ------------------------------------------------------------
# 7. CLEAN EXACT DUPLICATES FROM ENTITY HISTORY
# ------------------------------------------------------------

entity_history_exact_duplicate_rows = int(
    entity_history_raw
    .duplicated()
    .sum()
)


entity_history = (
    entity_history_raw
    .drop_duplicates(
        keep="first"
    )
    .reset_index(drop=True)
    .copy()
)


print("\nEntity-history exact duplicates:")

print(
    "Raw rows:",
    len(
        entity_history_raw
    ),
)

print(
    "Exact duplicate rows removed:",
    entity_history_exact_duplicate_rows,
)

print(
    "Rows after exact deduplication:",
    len(
        entity_history
    ),
)


# ------------------------------------------------------------
# 8. CANONICALISE ENTITY HISTORY
# ------------------------------------------------------------

entity_history[
    "_CommitKey"
] = entity_history[
    history_commit_column
].map(
    canonical_commit
)


entity_history[
    "_EntityIdNumeric"
] = pd.to_numeric(
    entity_history[
        history_entity_column
    ],
    errors="coerce",
)


history_missing_commit_rows = int(
    entity_history[
        "_CommitKey"
    ].isna().sum()
)


history_invalid_entity_rows = int(
    entity_history[
        "_EntityIdNumeric"
    ].isna().sum()
)


history_malformed_commit_rows = int(
    (
        ~entity_history[
            "_CommitKey"
        ]
        .fillna("")
        .map(
            is_full_commit_hash
        )
    ).sum()
)


if history_missing_commit_rows != 0:
    raise RuntimeError(
        "entity_change_history.csv contains missing "
        f"commit hashes: {history_missing_commit_rows}"
    )


if history_invalid_entity_rows != 0:
    raise RuntimeError(
        "entity_change_history.csv contains missing or "
        "non-numeric EntityIds: "
        f"{history_invalid_entity_rows}"
    )


if history_malformed_commit_rows != 0:
    malformed_history_examples = (
        entity_history.loc[
            ~entity_history[
                "_CommitKey"
            ].map(
                is_full_commit_hash
            ),
            [
                history_commit_column,
                history_entity_column,
            ],
        ]
        .head(20)
    )

    print(
        "\nMalformed entity-history commit examples:"
    )
    display(
        malformed_history_examples
    )

    raise RuntimeError(
        "Entity history contains commit values that are "
        "not exact 40-character hexadecimal hashes."
    )


entity_history[
    "_EntityId"
] = entity_history[
    "_EntityIdNumeric"
].astype(np.int64)


history_entity_ids = set(
    entity_history[
        "_EntityId"
    ].unique().tolist()
)


history_ids_missing_from_id_map = sorted(
    history_entity_ids
    - id_map_entity_ids
)


if history_ids_missing_from_id_map:
    raise RuntimeError(
        "Entity-history EntityIds are missing from "
        "id_map.csv.\n"
        f"Missing count: "
        f"{len(history_ids_missing_from_id_map)}"
    )


# One canonical row per exact commit–entity pair.
history_commit_entity_pairs = (
    entity_history[
        [
            "_CommitKey",
            "_EntityId",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "_CommitKey",
            "_EntityId",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9. SPLIT BUILD COMMIT LISTS USING '#'
# ------------------------------------------------------------

commit_token_records = []


for build_row in (
    chronology_working
    .itertuples(
        index=False
    )
):
    build_id = str(
        build_row.CanonicalBuild
    )

    build_order = int(
        build_row.BuildOrder
    )

    partition = str(
        build_row.Partition
    )

    original_commit_field = (
        None
        if pd.isna(
            build_row.OriginalCommitField
        )
        else str(
            build_row.OriginalCommitField
        )
    )

    if (
        original_commit_field is None
        or original_commit_field.strip() == ""
    ):
        commit_tokens = []

    else:
        commit_tokens = [
            token.strip()
            for token in original_commit_field.split(
                "#"
            )
            if token.strip() != ""
        ]

    for position, token in enumerate(
        commit_tokens,
        start=1,
    ):
        commit_key = canonical_commit(
            token
        )

        commit_token_records.append({
            "CanonicalBuild":
                build_id,

            "BuildOrder":
                build_order,

            "Partition":
                partition,

            "CommitPosition":
                int(position),

            "Commit":
                commit_key,

            "OriginalCommitField":
                original_commit_field,

            "IsFullCommitHash":
                is_full_commit_hash(
                    commit_key
                ),
        })


build_commit_tokens = pd.DataFrame(
    commit_token_records,
    columns=[
        "CanonicalBuild",
        "BuildOrder",
        "Partition",
        "CommitPosition",
        "Commit",
        "OriginalCommitField",
        "IsFullCommitHash",
    ],
)


malformed_build_commit_tokens = (
    build_commit_tokens[
        ~build_commit_tokens[
            "IsFullCommitHash"
        ]
    ].copy()
)


if not malformed_build_commit_tokens.empty:
    print(
        "\nMalformed build commit tokens:"
    )
    display(
        malformed_build_commit_tokens.head(50)
    )

    raise RuntimeError(
        "builds.csv contains commit tokens that are not "
        "exact 40-character hexadecimal hashes."
    )


duplicate_token_positions = int(
    build_commit_tokens
    .duplicated(
        subset=[
            "CanonicalBuild",
            "CommitPosition",
        ],
    )
    .sum()
)


if duplicate_token_positions != 0:
    raise AssertionError(
        "Duplicate build–commit positions were created."
    )


# ------------------------------------------------------------
# 10. MAP EXACT COMMITS TO ENTITY HISTORY
# ------------------------------------------------------------

history_commit_set = set(
    history_commit_entity_pairs[
        "_CommitKey"
    ].unique().tolist()
)


build_commit_tokens[
    "MatchedToEntityHistory"
] = build_commit_tokens[
    "Commit"
].isin(
    history_commit_set
)


matched_token_rows = (
    build_commit_tokens[
        build_commit_tokens[
            "MatchedToEntityHistory"
        ]
    ]
    .copy()
)


unmatched_commit_tokens = (
    build_commit_tokens[
        ~build_commit_tokens[
            "MatchedToEntityHistory"
        ]
    ]
    .copy()
)


build_commit_entity_map = (
    matched_token_rows
    .merge(
        history_commit_entity_pairs,

        left_on="Commit",
        right_on="_CommitKey",

        how="inner",
        sort=False,

        validate="many_to_many",
    )
    [
        [
            "CanonicalBuild",
            "BuildOrder",
            "Partition",
            "CommitPosition",
            "Commit",
            "_EntityId",
        ]
    ]
    .rename(
        columns={
            "_EntityId":
                "EntityId",
        }
    )
    .sort_values(
        [
            "BuildOrder",
            "CommitPosition",
            "EntityId",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# A repeated commit token should not create a duplicated
# build–commit–entity row with the same position.
duplicate_build_commit_entity_rows = int(
    build_commit_entity_map
    .duplicated(
        subset=[
            "CanonicalBuild",
            "CommitPosition",
            "Commit",
            "EntityId",
        ]
    )
    .sum()
)


if duplicate_build_commit_entity_rows != 0:
    raise AssertionError(
        "Canonical build–commit–entity mapping contains "
        "unexpected duplicate rows."
    )


# ------------------------------------------------------------
# 11. CREATE CANONICAL BUILD–ENTITY MAP
# ------------------------------------------------------------

build_entity_map_internal = (
    build_commit_entity_map[
        [
            "CanonicalBuild",
            "BuildOrder",
            "Partition",
            "EntityId",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "BuildOrder",
            "EntityId",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


numeric_build_ids = pd.to_numeric(
    build_entity_map_internal[
        "CanonicalBuild"
    ],
    errors="coerce",
)


if numeric_build_ids.isna().any():
    raise RuntimeError(
        "Project 8 build IDs are not fully numeric; "
        "REC-compatible mapping cannot be created."
    )


build_entity_map = pd.DataFrame({
    # Preserve the column names used by the established
    # REC reconstruction helper.
    "id":
        numeric_build_ids.astype(
            np.int64
        ),

    "BuildOrder":
        build_entity_map_internal[
            "BuildOrder"
        ].astype(int),

    "Partition":
        build_entity_map_internal[
            "Partition"
        ].astype(str),

    "EntityId":
        build_entity_map_internal[
            "EntityId"
        ].astype(np.int64),
})


mapped_entity_ids = set(
    build_entity_map[
        "EntityId"
    ].unique().tolist()
)


mapped_ids_missing_from_id_map = sorted(
    mapped_entity_ids
    - id_map_entity_ids
)


if mapped_ids_missing_from_id_map:
    raise RuntimeError(
        "Mapped EntityIds are missing from id_map.csv.\n"
        f"Missing count: "
        f"{len(mapped_ids_missing_from_id_map)}"
    )


# ------------------------------------------------------------
# 12. CREATE CHANGED-ENTITY DICTIONARY FOR ALL BUILDS
# ------------------------------------------------------------

changed_entities_grouped = (
    build_entity_map
    .groupby(
        "id",
        sort=False,
    )[
        "EntityId"
    ]
    .apply(
        lambda values: sorted({
            int(value)
            for value in values
        })
    )
    .to_dict()
)


chronology_numeric_builds = pd.to_numeric(
    chronology_working[
        "CanonicalBuild"
    ],
    errors="coerce",
)


if chronology_numeric_builds.isna().any():
    raise RuntimeError(
        "The build chronology contains non-numeric "
        "build IDs."
    )


all_build_ids = (
    chronology_numeric_builds
    .astype(np.int64)
    .tolist()
)


changed_entities_by_build = {
    str(
        int(build_id)
    ): changed_entities_grouped.get(
        int(build_id),
        [],
    )
    for build_id in all_build_ids
}


changed_entity_dictionary_builds = len(
    changed_entities_by_build
)


if (
    changed_entity_dictionary_builds
    != len(
        chronology_working
    )
):
    raise AssertionError(
        "The changed-entity dictionary does not contain "
        "every Project 8 build."
    )


# ------------------------------------------------------------
# 13. CREATE BUILD-LEVEL MAPPING STATUS
# ------------------------------------------------------------

token_count_summary = (
    build_commit_tokens
    .groupby(
        "CanonicalBuild",
        sort=False,
    )
    .agg(
        CommitTokens=(
            "Commit",
            "size",
        ),

        MatchedCommitTokens=(
            "MatchedToEntityHistory",
            "sum",
        ),
    )
    .reset_index()
)


token_count_summary[
    "CommitTokens"
] = token_count_summary[
    "CommitTokens"
].astype(int)


token_count_summary[
    "MatchedCommitTokens"
] = token_count_summary[
    "MatchedCommitTokens"
].astype(int)


token_count_summary[
    "UnmatchedCommitTokens"
] = (
    token_count_summary[
        "CommitTokens"
    ]
    - token_count_summary[
        "MatchedCommitTokens"
    ]
)


entity_count_summary = (
    build_entity_map_internal
    .groupby(
        "CanonicalBuild",
        sort=False,
    )[
        "EntityId"
    ]
    .nunique()
    .rename(
        "ChangedEntities"
    )
    .reset_index()
)


mapping_status_by_build = (
    chronology_working[
        [
            "CanonicalBuild",
            "BuildOrder",
            "Partition",
            "OriginalCommitField",
        ]
    ]
    .merge(
        token_count_summary,

        on="CanonicalBuild",

        how="left",
        validate="one_to_one",
    )
    .merge(
        entity_count_summary,

        on="CanonicalBuild",

        how="left",
        validate="one_to_one",
    )
)


for column in [
    "CommitTokens",
    "MatchedCommitTokens",
    "UnmatchedCommitTokens",
    "ChangedEntities",
]:
    mapping_status_by_build[
        column
    ] = (
        mapping_status_by_build[
            column
        ]
        .fillna(0)
        .astype(int)
    )


mapping_status_by_build[
    "MappingStatus"
] = mapping_status_by_build.apply(
    mapping_status,
    axis=1,
)


# ------------------------------------------------------------
# 14. ATTACH STEP 4 BUILD COVERAGE COUNTS
# ------------------------------------------------------------

required_coverage_columns = {
    "CanonicalBuild",
    "RawExecutionRows",
    "RawFailureExecutions",
    "ModelReadyRows",
    "ModelReadyFailures",
}

missing_coverage_columns = (
    required_coverage_columns
    - set(
        build_coverage.columns
    )
)

if missing_coverage_columns:
    raise RuntimeError(
        "The Step 4 build-coverage file is missing "
        "columns:\n"
        f"{sorted(missing_coverage_columns)}"
    )


coverage_subset = build_coverage[
    [
        "CanonicalBuild",
        "RawExecutionRows",
        "RawFailureExecutions",
        "ModelReadyRows",
        "ModelReadyFailures",
    ]
].copy()


coverage_subset[
    "CanonicalBuild"
] = coverage_subset[
    "CanonicalBuild"
].map(
    canonical_id
)


mapping_status_by_build = (
    mapping_status_by_build
    .merge(
        coverage_subset,

        on="CanonicalBuild",

        how="left",
        validate="one_to_one",
    )
)


for column in [
    "RawExecutionRows",
    "RawFailureExecutions",
    "ModelReadyRows",
    "ModelReadyFailures",
]:
    mapping_status_by_build[
        column
    ] = pd.to_numeric(
        mapping_status_by_build[
            column
        ],
        errors="raise",
    ).astype(int)


# ------------------------------------------------------------
# 15. COMMIT AND BUILD MAPPING SUMMARIES
# ------------------------------------------------------------

total_builds = int(
    len(
        chronology_working
    )
)


total_commit_tokens = int(
    len(
        build_commit_tokens
    )
)


matched_commit_token_count = int(
    build_commit_tokens[
        "MatchedToEntityHistory"
    ].sum()
)


unmatched_commit_token_count = int(
    total_commit_tokens
    - matched_commit_token_count
)


commit_token_coverage_percent = (
    100.0
    * matched_commit_token_count
    / total_commit_tokens
    if total_commit_tokens > 0
    else 0.0
)


fully_mapped_builds = int(
    mapping_status_by_build[
        "MappingStatus"
    ]
    .eq(
        "FULL_COMMIT_COVERAGE"
    )
    .sum()
)


partially_mapped_builds = int(
    mapping_status_by_build[
        "MappingStatus"
    ]
    .eq(
        "PARTIAL_COMMIT_COVERAGE"
    )
    .sum()
)


builds_with_no_matched_commit = int(
    mapping_status_by_build[
        "MappingStatus"
    ]
    .isin(
        [
            "NO_MATCHED_COMMITS",
            "NO_COMMIT_TOKENS",
        ]
    )
    .sum()
)


builds_with_at_least_one_entity = int(
    mapping_status_by_build[
        "ChangedEntities"
    ]
    .gt(0)
    .sum()
)


build_commit_entity_rows = int(
    len(
        build_commit_entity_map
    )
)


unique_build_entity_pairs = int(
    len(
        build_entity_map
    )
)


unique_mapped_entities = int(
    build_entity_map[
        "EntityId"
    ].nunique()
)


entity_history_unique_ids = int(
    len(
        history_entity_ids
    )
)


id_map_unique_ids = int(
    len(
        id_map_entity_ids
    )
)


# ------------------------------------------------------------
# 16. CREATE AUDIT
# ------------------------------------------------------------

audit_records = [
    {
        "Check":
            "Step 4 passed",

        "Expected":
            True,

        "Actual":
            step4_status.get(
                "status"
            )
            == expected_step4_status,

        "Pass":
            step4_status.get(
                "status"
            )
            == expected_step4_status,
    },

    {
        "Check":
            "Chronology builds",

        "Expected":
            254,

        "Actual":
            total_builds,

        "Pass":
            total_builds == 254,
    },

    {
        "Check":
            "Build commit tokens available",

        "Expected":
            "> 0",

        "Actual":
            total_commit_tokens,

        "Pass":
            total_commit_tokens > 0,
    },

    {
        "Check":
            "Malformed build commit tokens",

        "Expected":
            0,

        "Actual":
            len(
                malformed_build_commit_tokens
            ),

        "Pass":
            len(
                malformed_build_commit_tokens
            )
            == 0,
    },

    {
        "Check":
            "Duplicate build-commit positions",

        "Expected":
            0,

        "Actual":
            duplicate_token_positions,

        "Pass":
            duplicate_token_positions == 0,
    },

    {
        "Check":
            "Entity-history exact duplicate rows documented",

        "Expected":
            "documented",

        "Actual":
            entity_history_exact_duplicate_rows,

        "Pass":
            True,
    },

    {
        "Check":
            "Entity-history missing commits",

        "Expected":
            0,

        "Actual":
            history_missing_commit_rows,

        "Pass":
            history_missing_commit_rows == 0,
    },

    {
        "Check":
            "Entity-history malformed commits",

        "Expected":
            0,

        "Actual":
            history_malformed_commit_rows,

        "Pass":
            history_malformed_commit_rows == 0,
    },

    {
        "Check":
            "Entity-history invalid EntityIds",

        "Expected":
            0,

        "Actual":
            history_invalid_entity_rows,

        "Pass":
            history_invalid_entity_rows == 0,
    },

    {
        "Check":
            "History EntityIds missing from id_map",

        "Expected":
            0,

        "Actual":
            len(
                history_ids_missing_from_id_map
            ),

        "Pass":
            len(
                history_ids_missing_from_id_map
            )
            == 0,
    },

    {
        "Check":
            "Mapped EntityIds missing from id_map",

        "Expected":
            0,

        "Actual":
            len(
                mapped_ids_missing_from_id_map
            ),

        "Pass":
            len(
                mapped_ids_missing_from_id_map
            )
            == 0,
    },

    {
        "Check":
            "Changed-entity dictionary builds",

        "Expected":
            total_builds,

        "Actual":
            changed_entity_dictionary_builds,

        "Pass":
            changed_entity_dictionary_builds
            == total_builds,
    },

    {
        "Check":
            "Commit-token coverage engineering guard",

        "Expected":
            (
                f">= "
                f"{MINIMUM_COMMIT_TOKEN_COVERAGE_PERCENT}%"
            ),

        "Actual":
            commit_token_coverage_percent,

        "Pass":
            commit_token_coverage_percent
            >= MINIMUM_COMMIT_TOKEN_COVERAGE_PERCENT,
    },

    {
        "Check":
            "Synthetic entity mappings",

        "Expected":
            0,

        "Actual":
            0,

        "Pass":
            True,
    },
]


audit_frame = pd.DataFrame(
    audit_records
)


print("\nStep 5 audit:")

display(
    audit_frame
)


failed_audit_checks = (
    audit_frame[
        ~audit_frame[
            "Pass"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 17. DETERMINE STEP 5 STATUS
# ------------------------------------------------------------

step5_pass = bool(
    len(
        failed_audit_checks
    )
    == 0
)


if step5_pass:
    if unmatched_commit_token_count == 0:
        mapping_status_text = (
            "PASS_PENDING_CLEAN_REC_VALIDATION"
        )

    else:
        mapping_status_text = (
            "PASS_PENDING_CLEAN_REC_VALIDATION_"
            "WITH_DOCUMENTED_UNMATCHED_COMMITS"
        )

else:
    mapping_status_text = (
        "FAIL_PROJECT_8_CANONICAL_ENTITY_MAPPING"
    )


# ------------------------------------------------------------
# 18. SAVE TABULAR AND DICTIONARY OUTPUTS
# ------------------------------------------------------------

PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


build_commit_tokens.to_csv(
    BUILD_COMMIT_TOKENS_PATH,
    index=False,
)


build_commit_entity_map.to_csv(
    BUILD_COMMIT_ENTITY_MAP_PATH,
    index=False,
    compression="gzip",
)


build_entity_map.to_csv(
    BUILD_ENTITY_MAP_PATH,
    index=False,
    compression="gzip",
)


CHANGED_ENTITIES_DICTIONARY_PATH.write_text(
    json.dumps(
        changed_entities_by_build,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)


mapping_status_by_build.to_csv(
    MAPPING_STATUS_PATH,
    index=False,
)


unmatched_commit_tokens.to_csv(
    UNMATCHED_COMMITS_PATH,
    index=False,
)


audit_frame.to_csv(
    ENTITY_MAPPING_AUDIT_PATH,
    index=False,
)


# ------------------------------------------------------------
# 19. VERIFY AND HASH SAVED OUTPUTS
# ------------------------------------------------------------

table_and_dictionary_outputs = [
    BUILD_COMMIT_TOKENS_PATH,
    BUILD_COMMIT_ENTITY_MAP_PATH,
    BUILD_ENTITY_MAP_PATH,
    CHANGED_ENTITIES_DICTIONARY_PATH,
    MAPPING_STATUS_PATH,
    UNMATCHED_COMMITS_PATH,
    ENTITY_MAPPING_AUDIT_PATH,
]


missing_saved_outputs = [
    str(path)
    for path in table_and_dictionary_outputs
    if not path.exists()
]


if missing_saved_outputs:
    raise RuntimeError(
        "Step 5 failed to save expected outputs:\n"
        + "\n".join(
            missing_saved_outputs
        )
    )


output_metadata = {
    path.name: file_metadata(
        path
    )
    for path in table_and_dictionary_outputs
}


# ------------------------------------------------------------
# 20. CREATE ENTITY-MAPPING REPORT
# ------------------------------------------------------------

mapping_status_summary = (
    mapping_status_by_build[
        "MappingStatus"
    ]
    .value_counts()
    .rename_axis(
        "MappingStatus"
    )
    .reset_index(
        name="Builds"
    )
)


affected_builds = (
    mapping_status_by_build[
        mapping_status_by_build[
            "UnmatchedCommitTokens"
        ].gt(0)
    ]
    .copy()
)


entity_mapping_report = {
    "project_number":
        PROJECT_NUMBER,

    "project":
        PROJECT_NAME,

    "project_slug":
        PROJECT_SLUG,

    "generated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "schema": {
        "id_map_entity_column":
            id_map_entity_column,

        "entity_history_commit_column":
            history_commit_column,

        "entity_history_entity_column":
            history_entity_column,
    },

    "entity_history_cleaning": {
        "raw_rows":
            int(
                len(
                    entity_history_raw
                )
            ),

        "exact_duplicate_rows_removed":
            entity_history_exact_duplicate_rows,

        "clean_rows":
            int(
                len(
                    entity_history
                )
            ),

        "unique_commit_entity_pairs":
            int(
                len(
                    history_commit_entity_pairs
                )
            ),
    },

    "commit_mapping": {
        "builds":
            total_builds,

        "build_commit_tokens":
            total_commit_tokens,

        "matched_commit_tokens":
            matched_commit_token_count,

        "unmatched_commit_tokens":
            unmatched_commit_token_count,

        "commit_token_coverage_percent":
            commit_token_coverage_percent,

        "delimiter":
            "#",

        "matching_rule":
            "exact full 40-character commit hash only",

        "prefix_matching_used":
            False,

        "synthetic_mapping_used":
            False,
    },

    "build_mapping": {
        "fully_mapped_builds":
            fully_mapped_builds,

        "partially_mapped_builds":
            partially_mapped_builds,

        "builds_with_no_matched_commit":
            builds_with_no_matched_commit,

        "builds_with_at_least_one_mapped_entity":
            builds_with_at_least_one_entity,

        "build_commit_entity_rows":
            build_commit_entity_rows,

        "unique_build_entity_pairs":
            unique_build_entity_pairs,

        "unique_mapped_entities":
            unique_mapped_entities,

        "changed_entity_dictionary_builds":
            changed_entity_dictionary_builds,

        "mapping_status_summary":
            mapping_status_summary
            .to_dict(
                orient="records"
            ),
    },

    "entity_id_validation": {
        "entity_history_unique_entity_ids":
            entity_history_unique_ids,

        "id_map_unique_entity_ids":
            id_map_unique_ids,

        "history_entity_ids_missing_from_id_map":
            len(
                history_ids_missing_from_id_map
            ),

        "mapped_entity_ids_missing_from_id_map":
            len(
                mapped_ids_missing_from_id_map
            ),
    },

    "affected_builds":
        affected_builds[
            [
                "CanonicalBuild",
                "BuildOrder",
                "Partition",
                "CommitTokens",
                "MatchedCommitTokens",
                "UnmatchedCommitTokens",
                "ChangedEntities",
                "RawExecutionRows",
                "RawFailureExecutions",
                "ModelReadyRows",
                "ModelReadyFailures",
                "MappingStatus",
            ]
        ]
        .to_dict(
            orient="records"
        ),

    "unmatched_commit_tokens":
        unmatched_commit_tokens[
            [
                "CanonicalBuild",
                "BuildOrder",
                "Partition",
                "CommitPosition",
                "Commit",
                "OriginalCommitField",
            ]
        ]
        .to_dict(
            orient="records"
        ),

    "audit":
        audit_frame.to_dict(
            orient="records"
        ),

    "failed_audit_check_count":
        int(
            len(
                failed_audit_checks
            )
        ),

    "outputs":
        output_metadata,

    "mapping_status":
        mapping_status_text,
}


ENTITY_MAPPING_REPORT_PATH.write_text(
    json.dumps(
        entity_mapping_report,
        indent=2,
        default=json_safe,
    ),
    encoding="utf-8",
)


step5_status = {
    "project_number":
        PROJECT_NUMBER,

    "project":
        PROJECT_NAME,

    "project_slug":
        PROJECT_SLUG,

    "status":
        mapping_status_text,

    "commit_token_coverage_percent":
        commit_token_coverage_percent,

    "unmatched_commit_tokens":
        unmatched_commit_token_count,

    "failed_audit_check_count":
        int(
            len(
                failed_audit_checks
            )
        ),

    "generated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


STEP5_STATUS_PATH.write_text(
    json.dumps(
        step5_status,
        indent=2,
        default=json_safe,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 21. VERIFY ALL STEP 5 OUTPUTS
# ------------------------------------------------------------

all_expected_outputs = (
    table_and_dictionary_outputs
    + [
        ENTITY_MAPPING_REPORT_PATH,
        STEP5_STATUS_PATH,
    ]
)


missing_outputs = [
    str(path)
    for path in all_expected_outputs
    if not path.exists()
]


if missing_outputs:
    raise RuntimeError(
        "Some Step 5 outputs are missing:\n"
        + "\n".join(
            missing_outputs
        )
    )


# ------------------------------------------------------------
# 22. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 80)
print("=== PROJECT 8 STEP 5 RESULT ===")
print("=" * 80)

print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nDetected schema:")

print(
    "id_map EntityId:",
    id_map_entity_column,
)

print(
    "Entity-history commit:",
    history_commit_column,
)

print(
    "Entity-history EntityId:",
    history_entity_column,
)


print("\nEntity-history cleaning:")

print(
    "Raw rows:",
    len(
        entity_history_raw
    ),
)

print(
    "Exact duplicate rows removed:",
    entity_history_exact_duplicate_rows,
)

print(
    "Clean rows:",
    len(
        entity_history
    ),
)


print("\nCommit mapping:")

print(
    "Builds:",
    total_builds,
)

print(
    "Build-commit tokens:",
    total_commit_tokens,
)

print(
    "Matched commit tokens:",
    matched_commit_token_count,
)

print(
    "Unmatched commit tokens:",
    unmatched_commit_token_count,
)

print(
    "Commit-token coverage:",
    round(
        commit_token_coverage_percent,
        6,
    ),
    "%",
)


print("\nBuild mapping:")

print(
    "Fully mapped builds:",
    fully_mapped_builds,
)

print(
    "Partially mapped builds:",
    partially_mapped_builds,
)

print(
    "Builds with no matched commit:",
    builds_with_no_matched_commit,
)

print(
    "Builds with at least one mapped entity:",
    builds_with_at_least_one_entity,
)

print(
    "Build-commit-entity rows:",
    build_commit_entity_rows,
)

print(
    "Unique Build-Entity pairs:",
    unique_build_entity_pairs,
)

print(
    "Unique mapped entities:",
    unique_mapped_entities,
)

print(
    "Changed-entity dictionary builds:",
    changed_entity_dictionary_builds,
)


print("\nEntity-ID validation:")

print(
    "Entity-history unique EntityIds:",
    entity_history_unique_ids,
)

print(
    "id_map unique EntityIds:",
    id_map_unique_ids,
)

print(
    "History EntityIds missing from id_map:",
    len(
        history_ids_missing_from_id_map
    ),
)

print(
    "Mapped EntityIds missing from id_map:",
    len(
        mapped_ids_missing_from_id_map
    ),
)


print("\nMapping status summary:")

display(
    mapping_status_summary
)


if not affected_builds.empty:

    print(
        "\nBuilds affected by unmatched commits:"
    )

    display(
        affected_builds[
            [
                "CanonicalBuild",
                "BuildOrder",
                "Partition",
                "CommitTokens",
                "MatchedCommitTokens",
                "UnmatchedCommitTokens",
                "ChangedEntities",
                "RawExecutionRows",
                "RawFailureExecutions",
                "ModelReadyRows",
                "ModelReadyFailures",
                "MappingStatus",
            ]
        ]
    )


if not unmatched_commit_tokens.empty:

    print(
        "\nUnmatched commit tokens:"
    )

    display(
        unmatched_commit_tokens[
            [
                "CanonicalBuild",
                "BuildOrder",
                "Partition",
                "CommitPosition",
                "Commit",
                "OriginalCommitField",
            ]
        ]
    )


print("\nCanonical mapping report:")

print(
    ENTITY_MAPPING_REPORT_PATH
)


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–7 modified:")
print(0)


print(
    "\nMapping status:",
    mapping_status_text,
)


if step5_pass:

    print(
        "\nSUCCESS: builds.csv commit lists were "
        "split using the '#' delimiter."
    )

    print(
        "SUCCESS: Exact full commit hashes were "
        "mapped to entity-change history."
    )

    print(
        "SUCCESS: Every entity-history EntityId was "
        "validated against id_map.csv."
    )

    print(
        "SUCCESS: The changed-entity dictionary "
        f"contains all {total_builds} ojAlgo builds."
    )

    print(
        "SUCCESS: Exact duplicate entity-history rows "
        "were removed deterministically."
    )

    print(
        "SUCCESS: No synthetic entity mapping "
        "was created."
    )

    print(
        "SUCCESS: Project 8 is ready for clean "
        "verdict-dependent REC reconstruction validation."
    )


print("=" * 80)


if not step5_pass:

    print(
        "\nFailed Step 5 audit checks:"
    )

    display(
        failed_audit_checks
    )

    raise RuntimeError(
        "PROJECT 8 STEP 5 DID NOT PASS.\n"
        "Do not proceed to REC reconstruction until "
        "the displayed mapping problems are resolved."
    )

=== PROJECT 8 STEP 5: CANONICAL ENTITY MAPPING ===

Loading Step 5 inputs...
builds.csv: (254, 3)
build chronology: (254, 6)
build coverage: (254, 14)
id_map.csv: (2436, 2)
entity_change_history.csv: (27427, 8)

id_map.csv columns:
['key', 'value']

entity_change_history.csv columns:
['EntityId', 'AddedLines', 'DeletedLines', 'Contributor', 'BugFix', 'Commit', 'CommitDate', 'MergeCommit']

Detected schema:
id_map.csv EntityId column: value
entity history commit column: Commit
entity history EntityId column: EntityId

Entity-history exact duplicates:
Raw rows: 27427
Exact duplicate rows removed: 28
Rows after exact deduplication: 27399

Step 5 audit:


,Check,Expected,Actual,Pass
0,Step 4 passed,True,True,True
1,Chronology builds,254,254,True
2,Build commit tokens available,> 0,359,True
3,Malformed build commit tokens,0,0,True
4,Duplicate build-commit positions,0,0,True
5,Entity-history exact duplicate rows documented,documented,28,True
6,Entity-history missing commits,0,0,True
7,Entity-history malformed commits,0,0,True
8,Entity-history invalid EntityIds,0,0,True
9,History EntityIds missing from id_map,0,0,True




=== PROJECT 8 STEP 5 RESULT ===

Project identity:
Project number: 8
Project: optimatika@ojAlgo
Project slug: optimatika__ojAlgo

Detected schema:
id_map EntityId: value
Entity-history commit: Commit
Entity-history EntityId: EntityId

Entity-history cleaning:
Raw rows: 27427
Exact duplicate rows removed: 28
Clean rows: 27399

Commit mapping:
Builds: 254
Build-commit tokens: 359
Matched commit tokens: 349
Unmatched commit tokens: 10
Commit-token coverage: 97.214485 %

Build mapping:
Fully mapped builds: 245
Partially mapped builds: 1
Builds with no matched commit: 8
Builds with at least one mapped entity: 246
Build-commit-entity rows: 11454
Unique Build-Entity pairs: 11075
Unique mapped entities: 921
Changed-entity dictionary builds: 254

Entity-ID validation:
Entity-history unique EntityIds: 1300
id_map unique EntityIds: 1300
History EntityIds missing from id_map: 0
Mapped EntityIds missing from id_map: 0

Mapping status summary:


,MappingStatus,Builds
0,FULL_COMMIT_COVERAGE,245
1,NO_MATCHED_COMMITS,8
2,PARTIAL_COMMIT_COVERAGE,1



Builds affected by unmatched commits:


,CanonicalBuild,BuildOrder,Partition,CommitTokens,MatchedCommitTokens,UnmatchedCommitTokens,ChangedEntities,RawExecutionRows,RawFailureExecutions,ModelReadyRows,ModelReadyFailures,MappingStatus
0,564318082,1,training,17,15,2,233,126,2,0,0,PARTIAL_COMMIT_COVERAGE
60,591215481,61,training,1,0,1,0,127,0,0,0,NO_MATCHED_COMMITS
107,672911037,108,training,1,0,1,0,134,1,134,1,NO_MATCHED_COMMITS
175,732022171,176,training,1,0,1,0,140,0,0,0,NO_MATCHED_COMMITS
181,741430250,182,training,1,0,1,0,141,1,141,1,NO_MATCHED_COMMITS
206,751656775,207,evaluation,1,0,1,0,144,0,0,0,NO_MATCHED_COMMITS
222,763386964,223,evaluation,1,0,1,0,144,0,0,0,NO_MATCHED_COMMITS
235,768018837,236,evaluation,1,0,1,0,144,0,0,0,NO_MATCHED_COMMITS
236,768018862,237,evaluation,1,0,1,0,144,0,0,0,NO_MATCHED_COMMITS



Unmatched commit tokens:


,CanonicalBuild,BuildOrder,Partition,CommitPosition,Commit,OriginalCommitField
6,564318082,1,training,7,e7f82425fd31f53f07610f5e77e22c4fd675921c,e7a973d98cc39ebce4c93c97cbb7fb1735e6bc60#2c6dc...
7,564318082,1,training,8,2237838439fdcd07992f98cba2b861d602cb7810,e7a973d98cc39ebce4c93c97cbb7fb1735e6bc60#2c6dc...
98,591215481,61,training,1,e0ab82aee0e441bf24801ddc7f8dea8a4874a450,e0ab82aee0e441bf24801ddc7f8dea8a4874a450
145,672911037,108,training,1,d781a2fcad5add47136974a0e517bba4c952df1e,d781a2fcad5add47136974a0e517bba4c952df1e
268,732022171,176,training,1,a7fefc78f05de7a66718f85eb48da0c77e9281c3,a7fefc78f05de7a66718f85eb48da0c77e9281c3
274,741430250,182,training,1,9c75303b88d9d86518acbd217bdcdddfe066fbd6,9c75303b88d9d86518acbd217bdcdddfe066fbd6
301,751656775,207,evaluation,1,519ee67c08f90ac59b9ad38004272004f471d00a,519ee67c08f90ac59b9ad38004272004f471d00a
321,763386964,223,evaluation,1,be373f75c019f475a9a1de2d64a0f305b4daa1ed,be373f75c019f475a9a1de2d64a0f305b4daa1ed
336,768018837,236,evaluation,1,90aca55eca4f6c3aa07a3771b7c3a79cb7dafd72,90aca55eca4f6c3aa07a3771b7c3a79cb7dafd72
337,768018862,237,evaluation,1,90aca55eca4f6c3aa07a3771b7c3a79cb7dafd72,90aca55eca4f6c3aa07a3771b7c3a79cb7dafd72



Canonical mapping report:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_preflight/ojalgo_entity_mapping_report.json

Completion registry modified:
0

Projects 1–7 modified:
0

Mapping status: PASS_PENDING_CLEAN_REC_VALIDATION_WITH_DOCUMENTED_UNMATCHED_COMMITS

SUCCESS: builds.csv commit lists were split using the '#' delimiter.
SUCCESS: Exact full commit hashes were mapped to entity-change history.
SUCCESS: Every entity-history EntityId was validated against id_map.csv.
SUCCESS: The changed-entity dictionary contains all 254 ojAlgo builds.
SUCCESS: Exact duplicate entity-history rows were removed deterministically.
SUCCESS: No synthetic entity mapping was created.
SUCCESS: Project 8 is ready for clean verdict-dependent REC reconstruction validation.


In [ ]:
# ============================================================
# PROJECT 8 — STEP 6
# CLEAN VERDICT-DEPENDENT REC RECONSTRUCTION VALIDATION
#
# PROJECT: optimatika@ojAlgo
#
# This cell:
# - validates the passed Step 5 state
# - reconstructs all 19 REC_* features from clean history
# - uses only executions before each requested execution
# - uses the frozen canonical build–entity mapping
# - compares reconstruction against dataset.csv
# - requires zero verdict mismatches
# - requires zero REC-feature mismatches
# - documents model-ready builds affected by unmatched commits
#
# It does NOT:
# - inject verdict noise
# - modify source CSV files
# - modify the completion registry
# - modify Projects 1–7
# - create experimental conditions
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import re

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

PROJECT_NUMBER = 8
PROJECT_NAME = "optimatika@ojAlgo"
PROJECT_SLUG = "optimatika__ojAlgo"
PROJECT_SHORT_NAME = "ojalgo"

RECENT_WINDOW = 6

REC_FEATURE_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

PROJECT_SOURCE_DIR = Path(
    "/content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo"
)

PREFLIGHT_DIR = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / PROJECT_SLUG
    / f"{PROJECT_SHORT_NAME}_preflight"
)


# Source input

DATASET_PATH = (
    PROJECT_SOURCE_DIR
    / "dataset.csv"
)


# Previous-step inputs

IDENTITY_LOCK_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_project_identity_lock.json"
)

CANONICAL_RAW_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_canonical_raw_executions.csv.gz"
)

BUILD_CHRONOLOGY_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_build_chronology.csv"
)

BUILD_ENTITY_MAP_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_build_entity_map.csv.gz"
)

CHANGED_ENTITIES_DICTIONARY_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_changed_entities_by_build.json"
)

MAPPING_STATUS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_entity_mapping_status_by_build.csv"
)

STEP5_STATUS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step5_status.json"
)


# Step 6 outputs

CLEAN_REC_RECONSTRUCTED_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_reconstructed.csv.gz"
)

REC_COMPARISON_SUMMARY_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_comparison_summary.csv"
)

REC_MISMATCH_DETAILS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_mismatch_details.csv.gz"
)

LABEL_COMPARISON_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_label_comparison.csv.gz"
)

STEP6_AUDIT_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_validation_audit.csv"
)

STEP6_REPORT_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_validation_report.json"
)

STEP6_STATUS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step6_status.json"
)


print("=" * 82)
print("=== PROJECT 8 STEP 6: CLEAN REC RECONSTRUCTION VALIDATION ===")
print("=" * 82)


# ------------------------------------------------------------
# 2. HELPERS
# ------------------------------------------------------------

def canonical_integer(value):
    """
    Convert a numeric-looking identifier into an integer.
    """

    if pd.isna(value):
        return None

    text = str(value).strip()

    if text == "":
        return None

    numeric = pd.to_numeric(
        pd.Series([text]),
        errors="coerce",
    ).iloc[0]

    if pd.isna(numeric):
        return None

    if not float(numeric).is_integer():
        return None

    return int(numeric)


def json_safe(value):
    """
    Convert NumPy and pandas values into JSON-safe values.
    """

    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return value


def calculate_sha256(
    file_path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(file_path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def file_metadata(file_path):
    file_path = Path(file_path)

    return {
        "path": str(file_path),

        "size_bytes": int(
            file_path.stat().st_size
        ),

        "sha256": calculate_sha256(
            file_path
        ),
    }


def dataframe_content_sha256(dataframe):
    """
    Deterministically hash dataframe values and row order.
    """

    hashed_values = (
        pd.util.hash_pandas_object(
            dataframe,
            index=True,
            categorize=True,
        )
        .to_numpy(
            dtype=np.uint64
        )
    )

    return hashlib.sha256(
        hashed_values.tobytes()
    ).hexdigest()


# ------------------------------------------------------------
# 3. VALIDATE REQUIRED INPUTS
# ------------------------------------------------------------

required_inputs = [
    DATASET_PATH,
    IDENTITY_LOCK_PATH,
    CANONICAL_RAW_PATH,
    BUILD_CHRONOLOGY_PATH,
    BUILD_ENTITY_MAP_PATH,
    CHANGED_ENTITIES_DICTIONARY_PATH,
    MAPPING_STATUS_PATH,
    STEP5_STATUS_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Required Project 8 Step 6 inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
        + "\n\nIf the /content dataset directory is missing, "
        "rerun the Project 8 runtime-restoration cell."
    )


identity_lock = json.loads(
    IDENTITY_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)

if (
    identity_lock.get("project_number")
    != PROJECT_NUMBER
    or identity_lock.get("project")
    != PROJECT_NAME
    or identity_lock.get("project_slug")
    != PROJECT_SLUG
):
    raise AssertionError(
        "The Project 8 identity lock does not match "
        "the configured project."
    )


step5_status = json.loads(
    STEP5_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

accepted_step5_statuses = {
    "PASS_PENDING_CLEAN_REC_VALIDATION",
    (
        "PASS_PENDING_CLEAN_REC_VALIDATION_"
        "WITH_DOCUMENTED_UNMATCHED_COMMITS"
    ),
}

detected_step5_status = step5_status.get(
    "status"
)

if detected_step5_status not in accepted_step5_statuses:
    raise AssertionError(
        "Step 5 has not reached a valid REC-pending state.\n"
        f"Detected status: {detected_step5_status}"
    )


# ------------------------------------------------------------
# 4. LOAD STEP 6 INPUT TABLES
# ------------------------------------------------------------

print("\nLoading Step 6 inputs...")

dataset_raw = pd.read_csv(
    DATASET_PATH,
    low_memory=False,
)

canonical_raw = pd.read_csv(
    CANONICAL_RAW_PATH,
    low_memory=False,
)

chronology = pd.read_csv(
    BUILD_CHRONOLOGY_PATH,
    dtype=str,
)

build_entity_map = pd.read_csv(
    BUILD_ENTITY_MAP_PATH,
    low_memory=False,
)

mapping_status_by_build = pd.read_csv(
    MAPPING_STATUS_PATH,
    low_memory=False,
)

changed_entities_json = json.loads(
    CHANGED_ENTITIES_DICTIONARY_PATH.read_text(
        encoding="utf-8"
    )
)


print(
    "dataset.csv:",
    dataset_raw.shape,
)

print(
    "Canonical raw executions:",
    canonical_raw.shape,
)

print(
    "Build chronology:",
    chronology.shape,
)

print(
    "Build-entity map:",
    build_entity_map.shape,
)

print(
    "Mapping-status rows:",
    mapping_status_by_build.shape,
)

print(
    "Changed-entity dictionary builds:",
    len(changed_entities_json),
)


# ------------------------------------------------------------
# 5. VALIDATE DATASET SCHEMA
# ------------------------------------------------------------

required_dataset_columns = {
    "Build",
    "Test",
    "Verdict",
    "Duration",
    *REC_FEATURE_COLUMNS,
}

missing_dataset_columns = (
    required_dataset_columns
    - set(
        dataset_raw.columns
    )
)

if missing_dataset_columns:
    raise RuntimeError(
        "dataset.csv is missing required Step 6 columns:\n"
        f"{sorted(missing_dataset_columns)}"
    )


if len(REC_FEATURE_COLUMNS) != 19:
    raise AssertionError(
        "The REC feature protocol must contain exactly "
        "19 columns."
    )


dataset = dataset_raw.copy()

dataset[
    "Build"
] = pd.to_numeric(
    dataset[
        "Build"
    ],
    errors="raise",
).astype(
    np.int64
)

dataset[
    "Test"
] = pd.to_numeric(
    dataset[
        "Test"
    ],
    errors="raise",
).astype(
    np.int64
)

dataset[
    "Verdict"
] = pd.to_numeric(
    dataset[
        "Verdict"
    ],
    errors="raise",
).astype(
    np.int64
)

dataset[
    "Duration"
] = pd.to_numeric(
    dataset[
        "Duration"
    ],
    errors="raise",
).astype(float)


dataset_duplicate_build_test_pairs = int(
    dataset.duplicated(
        subset=[
            "Build",
            "Test",
        ]
    ).sum()
)

if dataset_duplicate_build_test_pairs != 0:
    raise AssertionError(
        "dataset.csv contains duplicate Build-Test rows."
    )


dataset_snapshot_columns = [
    "Build",
    "Test",
    "Verdict",
    "Duration",
    *REC_FEATURE_COLUMNS,
]

dataset_hash_before = (
    dataframe_content_sha256(
        dataset[
            dataset_snapshot_columns
        ]
    )
)


# ------------------------------------------------------------
# 6. PREPARE BUILD CHRONOLOGY
# ------------------------------------------------------------

required_chronology_columns = {
    "CanonicalBuild",
    "BuildOrder",
    "Partition",
}

missing_chronology_columns = (
    required_chronology_columns
    - set(
        chronology.columns
    )
)

if missing_chronology_columns:
    raise RuntimeError(
        "The build chronology is missing required columns:\n"
        f"{sorted(missing_chronology_columns)}"
    )


chronology_working = chronology[
    [
        "CanonicalBuild",
        "BuildOrder",
        "Partition",
    ]
].copy()


chronology_working[
    "id"
] = chronology_working[
    "CanonicalBuild"
].map(
    canonical_integer
)


if chronology_working[
    "id"
].isna().any():
    raise RuntimeError(
        "The build chronology contains a non-numeric "
        "canonical build ID."
    )


chronology_working[
    "id"
] = chronology_working[
    "id"
].astype(
    np.int64
)


chronology_working[
    "build_order"
] = pd.to_numeric(
    chronology_working[
        "BuildOrder"
    ],
    errors="raise",
).astype(int)


if chronology_working[
    "id"
].duplicated().any():
    raise AssertionError(
        "The build chronology contains duplicated builds."
    )


chronology_working = (
    chronology_working
    .sort_values(
        "build_order",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


build_order_map = dict(
    zip(
        chronology_working[
            "id"
        ],
        chronology_working[
            "build_order"
        ],
    )
)


partition_map = dict(
    zip(
        chronology_working[
            "id"
        ],
        chronology_working[
            "Partition"
        ].astype(str),
    )
)


# ------------------------------------------------------------
# 7. PREPARE CLEAN CHRONOLOGICAL EXECUTION HISTORY
# ------------------------------------------------------------

required_raw_columns = {
    "CanonicalBuild",
    "CanonicalTest",
    "CanonicalJob",
    "RawVerdictToken",
    "RawDuration",
    "BuildOrder",
    "Partition",
}

missing_raw_columns = (
    required_raw_columns
    - set(
        canonical_raw.columns
    )
)

if missing_raw_columns:
    raise RuntimeError(
        "The canonical raw-execution file is missing "
        "required columns:\n"
        f"{sorted(missing_raw_columns)}"
    )


exe_for_rec = pd.DataFrame({
    "build":
        pd.to_numeric(
            canonical_raw[
                "CanonicalBuild"
            ],
            errors="raise",
        ).astype(
            np.int64
        ),

    "test":
        pd.to_numeric(
            canonical_raw[
                "CanonicalTest"
            ],
            errors="raise",
        ).astype(
            np.int64
        ),

    "job":
        pd.to_numeric(
            canonical_raw[
                "CanonicalJob"
            ],
            errors="coerce",
        ),

    "verdict":
        pd.to_numeric(
            canonical_raw[
                "RawVerdictToken"
            ],
            errors="raise",
        ).astype(
            np.int64
        ),

    "duration":
        pd.to_numeric(
            canonical_raw[
                "RawDuration"
            ],
            errors="raise",
        ).astype(float),

    "build_order":
        pd.to_numeric(
            canonical_raw[
                "BuildOrder"
            ],
            errors="raise",
        ).astype(int),
})


# A missing job is not expected, but the job value is used
# only as a deterministic same-build tie breaker.
exe_for_rec[
    "job"
] = exe_for_rec[
    "job"
].fillna(-1).astype(
    np.int64
)


raw_duplicate_build_test_pairs = int(
    exe_for_rec.duplicated(
        subset=[
            "build",
            "test",
        ]
    ).sum()
)

if raw_duplicate_build_test_pairs != 0:
    raise AssertionError(
        "Canonical raw history contains duplicate "
        "Build-Test rows."
    )


if exe_for_rec[
    "build_order"
].isna().any():
    raise AssertionError(
        "Some execution-history rows have no build order."
    )


if not np.isfinite(
    exe_for_rec[
        "duration"
    ].to_numpy(dtype=float)
).all():
    raise AssertionError(
        "Clean execution history contains a missing or "
        "infinite duration."
    )


if (
    exe_for_rec[
        "duration"
    ]
    < 0
).any():
    raise AssertionError(
        "Clean execution history contains a negative "
        "duration."
    )


exe_for_rec = (
    exe_for_rec
    .sort_values(
        [
            "build_order",
            "job",
            "test",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


ordered_execution_builds = (
    exe_for_rec[
        [
            "build",
            "build_order",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "build_order",
        kind="mergesort",
    )[
        "build"
    ]
    .astype(int)
    .tolist()
)


global_build_position = {
    int(build_id): position
    for position, build_id
    in enumerate(
        ordered_execution_builds
    )
}


chronology_build_set = set(
    chronology_working[
        "id"
    ].astype(int)
)

execution_build_set = set(
    ordered_execution_builds
)

execution_builds_missing_from_chronology = sorted(
    execution_build_set
    - chronology_build_set
)

if execution_builds_missing_from_chronology:
    raise AssertionError(
        "Execution-history builds are missing from the "
        "frozen chronology."
    )


# ------------------------------------------------------------
# 8. PREPARE ENTITY ↔ BUILD HISTORY
# ------------------------------------------------------------

required_build_entity_columns = {
    "id",
    "EntityId",
}

missing_build_entity_columns = (
    required_build_entity_columns
    - set(
        build_entity_map.columns
    )
)

if missing_build_entity_columns:
    raise RuntimeError(
        "The canonical build-entity map is missing columns:\n"
        f"{sorted(missing_build_entity_columns)}"
    )


entity_build_pairs = (
    build_entity_map[
        [
            "id",
            "EntityId",
        ]
    ]
    .dropna()
    .drop_duplicates()
    .copy()
)


entity_build_pairs[
    "id"
] = pd.to_numeric(
    entity_build_pairs[
        "id"
    ],
    errors="raise",
).astype(
    np.int64
)


entity_build_pairs[
    "EntityId"
] = pd.to_numeric(
    entity_build_pairs[
        "EntityId"
    ],
    errors="raise",
).astype(
    np.int64
)


entity_changed_builds = (
    entity_build_pairs
    .groupby(
        "EntityId"
    )[
        "id"
    ]
    .apply(
        lambda values: set(
            int(value)
            for value in values
        )
    )
    .to_dict()
)


clean_changed_entities_by_build = {}

for build_id in chronology_working[
    "id"
].astype(int):

    raw_entities = changed_entities_json.get(
        str(build_id),
        changed_entities_json.get(
            build_id,
            [],
        ),
    )

    clean_changed_entities_by_build[
        int(build_id)
    ] = set(
        int(entity_id)
        for entity_id in raw_entities
    )


changed_entity_dictionary_builds = int(
    len(
        clean_changed_entities_by_build
    )
)


if (
    changed_entity_dictionary_builds
    != len(
        chronology_working
    )
):
    raise AssertionError(
        "The changed-entity dictionary does not contain "
        "every chronological build."
    )


dictionary_build_set = set(
    clean_changed_entities_by_build.keys()
)

if dictionary_build_set != chronology_build_set:
    raise AssertionError(
        "The changed-entity dictionary build set differs "
        "from the frozen chronology build set."
    )


# ------------------------------------------------------------
# 9. REC CALCULATION HELPERS
# ------------------------------------------------------------

def calculate_rates(history):
    """
    Reproduce failure, assertion, exception and transition
    rates from a non-empty prior execution history.

    Verdict encoding:
      0 = pass
      1 = exception failure
      2 = assertion failure
    """

    history_length = len(
        history
    )

    if history_length == 0:
        raise ValueError(
            "Rate calculation requires non-empty history."
        )

    verdicts = history[
        "verdict"
    ]

    fail_rate = float(
        (
            verdicts != 0
        ).sum()
        / history_length
    )

    assertion_rate = float(
        (
            verdicts == 2
        ).sum()
        / history_length
    )

    exception_rate = float(
        (
            verdicts == 1
        ).sum()
        / history_length
    )

    transition_rate = float(
        (
            history[
                "transition"
            ]
            == 1
        ).sum()
        / history_length
    )

    return (
        fail_rate,
        assertion_rate,
        exception_rate,
        transition_rate,
    )


def calculate_max_test_file_rate(
    history,
    target_column,
    current_changed_entities,
):
    """
    Reproduce the code-change association feature.

    target_column is either:
      - verdict
      - transition
    """

    target_builds = (
        history.loc[
            history[
                target_column
            ]
            > 0,
            "build",
        ]
        .drop_duplicates()
        .astype(int)
        .tolist()
    )

    # Official default when the test has no previous
    # failure or transition.
    if len(target_builds) == 0:
        return -1.0

    target_build_set = set(
        target_builds
    )

    maximum_frequency = 0

    for entity_id in current_changed_entities:

        entity_builds = (
            entity_changed_builds.get(
                int(entity_id),
                set(),
            )
        )

        overlap_count = len(
            entity_builds.intersection(
                target_build_set
            )
        )

        maximum_frequency = max(
            maximum_frequency,
            overlap_count,
        )

    # Target history exists, but none of the current
    # changed entities overlaps with it.
    if maximum_frequency == 0:
        return 0.0

    return float(
        maximum_frequency
        / len(
            target_builds
        )
    )


# ------------------------------------------------------------
# 10. REC RECONSTRUCTION FUNCTION
# ------------------------------------------------------------

def reconstruct_rec_features(
    execution_history,
    requested_rows,
    recent_window=6,
):
    """
    Reconstruct the 19 REC features for requested Build-Test
    rows using only executions before each current execution.
    """

    requested_pairs = set(
        zip(
            requested_rows[
                "Build"
            ].astype(int),

            requested_rows[
                "Test"
            ].astype(int),
        )
    )

    reconstructed_records = []

    for (
        test_id,
        test_history,
    ) in execution_history.groupby(
        "test",
        sort=False,
    ):

        test_history = (
            test_history
            .sort_values(
                [
                    "build_order",
                    "job",
                ],
                kind="mergesort",
            )
            .reset_index(drop=True)
            .copy()
        )

        test_history[
            "transition"
        ] = (
            test_history[
                "verdict"
            ]
            .diff()
            .fillna(0)
            .ne(0)
            .astype(int)
        )

        first_test_build = int(
            test_history.iloc[0][
                "build"
            ]
        )

        for current_position in range(
            len(
                test_history
            )
        ):

            current_row = test_history.iloc[
                current_position
            ]

            current_build = int(
                current_row[
                    "build"
                ]
            )

            requested_pair = (
                current_build,
                int(test_id),
            )

            if requested_pair not in requested_pairs:
                continue

            history = (
                test_history
                .iloc[
                    :current_position
                ]
                .copy()
                .reset_index(drop=True)
            )

            record = {
                "Build":
                    current_build,

                "Test":
                    int(test_id),
            }

            # First-ever execution of this test.
            if history.empty:

                for feature in REC_FEATURE_COLUMNS:
                    record[
                        feature
                    ] = -1.0

                record[
                    "REC_Age"
                ] = 0

                reconstructed_records.append(
                    record
                )

                continue

            recent_history = (
                history
                .tail(
                    recent_window
                )
                .copy()
            )

            current_global_position = (
                global_build_position[
                    current_build
                ]
            )

            first_global_position = (
                global_build_position[
                    first_test_build
                ]
            )

            age = int(
                current_global_position
                - first_global_position
            )


            # Last failure age measured in previous
            # executions of this test.
            failure_positions = np.flatnonzero(
                history[
                    "verdict"
                ].to_numpy()
                > 0
            )

            if len(
                failure_positions
            ) == 0:

                last_failure_age = -1

            else:

                last_failure_age = int(
                    len(history)
                    - 1
                    - int(
                        failure_positions[-1]
                    )
                )


            # Last verdict-transition age.
            transition_positions = np.flatnonzero(
                history[
                    "transition"
                ].to_numpy()
                > 0
            )

            if len(
                transition_positions
            ) == 0:

                last_transition_age = -1

            else:

                last_transition_age = int(
                    len(history)
                    - 1
                    - int(
                        transition_positions[-1]
                    )
                )


            (
                recent_fail_rate,
                recent_assert_rate,
                recent_exc_rate,
                recent_transition_rate,
            ) = calculate_rates(
                recent_history
            )


            (
                total_fail_rate,
                total_assert_rate,
                total_exc_rate,
                total_transition_rate,
            ) = calculate_rates(
                history
            )


            current_changed_entities = (
                clean_changed_entities_by_build.get(
                    current_build,
                    set(),
                )
            )


            max_test_file_fail_rate = (
                calculate_max_test_file_rate(
                    history=history,

                    target_column="verdict",

                    current_changed_entities=(
                        current_changed_entities
                    ),
                )
            )


            max_test_file_transition_rate = (
                calculate_max_test_file_rate(
                    history=history,

                    target_column="transition",

                    current_changed_entities=(
                        current_changed_entities
                    ),
                )
            )


            record.update({
                "REC_Age":
                    age,

                "REC_LastFailureAge":
                    last_failure_age,

                "REC_LastTransitionAge":
                    last_transition_age,

                "REC_RecentAvgExeTime":
                    float(
                        recent_history[
                            "duration"
                        ].mean()
                    ),

                "REC_RecentMaxExeTime":
                    float(
                        recent_history[
                            "duration"
                        ].max()
                    ),

                "REC_RecentFailRate":
                    recent_fail_rate,

                "REC_RecentAssertRate":
                    recent_assert_rate,

                "REC_RecentExcRate":
                    recent_exc_rate,

                "REC_RecentTransitionRate":
                    recent_transition_rate,

                "REC_TotalAvgExeTime":
                    float(
                        history[
                            "duration"
                        ].mean()
                    ),

                "REC_TotalMaxExeTime":
                    float(
                        history[
                            "duration"
                        ].max()
                    ),

                "REC_TotalFailRate":
                    total_fail_rate,

                "REC_TotalAssertRate":
                    total_assert_rate,

                "REC_TotalExcRate":
                    total_exc_rate,

                "REC_TotalTransitionRate":
                    total_transition_rate,

                "REC_LastVerdict":
                    int(
                        recent_history.iloc[-1][
                            "verdict"
                        ]
                    ),

                "REC_LastExeTime":
                    float(
                        recent_history.iloc[-1][
                            "duration"
                        ]
                    ),

                "REC_MaxTestFileFailRate":
                    max_test_file_fail_rate,

                "REC_MaxTestFileTransitionRate":
                    max_test_file_transition_rate,
            })

            reconstructed_records.append(
                record
            )


    reconstructed = pd.DataFrame(
        reconstructed_records,
        columns=[
            "Build",
            "Test",
            *REC_FEATURE_COLUMNS,
        ],
    )

    return reconstructed


# ------------------------------------------------------------
# 11. RUN CLEAN REC RECONSTRUCTION
# ------------------------------------------------------------

requested_rows = dataset[
    [
        "Build",
        "Test",
    ]
].copy()


print("\nReconstructing all 19 REC features...")

rec_clean_reconstructed = (
    reconstruct_rec_features(
        execution_history=
            exe_for_rec,

        requested_rows=
            requested_rows,

        recent_window=
            RECENT_WINDOW,
    )
)


expected_dataset_rows = int(
    len(
        dataset
    )
)

reconstructed_rows = int(
    len(
        rec_clean_reconstructed
    )
)

duplicate_reconstructed_pairs = int(
    rec_clean_reconstructed
    .duplicated(
        subset=[
            "Build",
            "Test",
        ]
    )
    .sum()
)


print(
    "Expected dataset rows:",
    expected_dataset_rows,
)

print(
    "Reconstructed rows:",
    reconstructed_rows,
)

print(
    "Duplicate reconstructed Build-Test pairs:",
    duplicate_reconstructed_pairs,
)


# ------------------------------------------------------------
# 12. COMPARE CLEAN LABELS
# ------------------------------------------------------------

raw_label_pairs = (
    exe_for_rec[
        [
            "build",
            "test",
            "verdict",
        ]
    ]
    .rename(
        columns={
            "build":
                "Build",

            "test":
                "Test",

            "verdict":
                "RawVerdict",
        }
    )
)


label_comparison = (
    dataset[
        [
            "Build",
            "Test",
            "Verdict",
        ]
    ]
    .merge(
        raw_label_pairs,

        on=[
            "Build",
            "Test",
        ],

        how="left",
        validate="one_to_one",
        indicator=True,
    )
)


label_comparison[
    "Partition"
] = label_comparison[
    "Build"
].map(
    partition_map
)


label_comparison[
    "LabelMatches"
] = (
    label_comparison[
        "Verdict"
    ]
    .eq(
        label_comparison[
            "RawVerdict"
        ]
    )
)


label_merge_missing_rows = int(
    label_comparison[
        "_merge"
    ]
    .ne("both")
    .sum()
)


label_mismatches = int(
    (
        ~label_comparison[
            "LabelMatches"
        ]
    ).sum()
)


training_label_mismatches = int(
    (
        label_comparison[
            "Partition"
        ].eq("training")
        &
        ~label_comparison[
            "LabelMatches"
        ]
    ).sum()
)


evaluation_label_mismatches = int(
    (
        label_comparison[
            "Partition"
        ].eq("evaluation")
        &
        ~label_comparison[
            "LabelMatches"
        ]
    ).sum()
)


# ------------------------------------------------------------
# 13. COMPARE ALL 19 REC FEATURES
# ------------------------------------------------------------

comparison = (
    dataset[
        [
            "Build",
            "Test",
            *REC_FEATURE_COLUMNS,
        ]
    ]
    .merge(
        rec_clean_reconstructed,

        on=[
            "Build",
            "Test",
        ],

        how="left",
        validate="one_to_one",

        suffixes=(
            "_Original",
            "_Reconstructed",
        ),

        indicator=True,
    )
)


comparison[
    "Partition"
] = comparison[
    "Build"
].map(
    partition_map
)


rec_merge_missing_rows = int(
    comparison[
        "_merge"
    ]
    .ne("both")
    .sum()
)


comparison_summary_records = []

mismatch_detail_frames = []


for feature in REC_FEATURE_COLUMNS:

    original_column = (
        f"{feature}_Original"
    )

    reconstructed_column = (
        f"{feature}_Reconstructed"
    )


    original_values = pd.to_numeric(
        comparison[
            original_column
        ],
        errors="coerce",
    ).to_numpy(
        dtype=float
    )


    reconstructed_values = pd.to_numeric(
        comparison[
            reconstructed_column
        ],
        errors="coerce",
    ).to_numpy(
        dtype=float
    )


    matching_mask = np.isclose(
        original_values,
        reconstructed_values,
        rtol=1e-9,
        atol=1e-9,
        equal_nan=True,
    )


    mismatch_mask = (
        ~matching_mask
    )


    absolute_difference = np.abs(
        original_values
        - reconstructed_values
    )


    finite_differences = (
        absolute_difference[
            np.isfinite(
                absolute_difference
            )
        ]
    )


    maximum_absolute_difference = (
        float(
            finite_differences.max()
        )
        if len(
            finite_differences
        ) > 0
        else 0.0
    )


    training_mismatch_count = int(
        (
            mismatch_mask
            &
            comparison[
                "Partition"
            ]
            .eq("training")
            .to_numpy()
        ).sum()
    )


    evaluation_mismatch_count = int(
        (
            mismatch_mask
            &
            comparison[
                "Partition"
            ]
            .eq("evaluation")
            .to_numpy()
        ).sum()
    )


    mismatch_count = int(
        mismatch_mask.sum()
    )


    comparison_summary_records.append({
        "Feature":
            feature,

        "Rows":
            int(
                len(
                    comparison
                )
            ),

        "MatchingRows":
            int(
                matching_mask.sum()
            ),

        "MismatchingRows":
            mismatch_count,

        "TrainingMismatches":
            training_mismatch_count,

        "EvaluationMismatches":
            evaluation_mismatch_count,

        "OriginalMissingValues":
            int(
                np.isnan(
                    original_values
                ).sum()
            ),

        "ReconstructedMissingValues":
            int(
                np.isnan(
                    reconstructed_values
                ).sum()
            ),

        "MaximumAbsoluteDifference":
            maximum_absolute_difference,
    })


    if mismatch_count > 0:

        detail = comparison.loc[
            mismatch_mask,
            [
                "Build",
                "Test",
                "Partition",
            ],
        ].copy()

        detail[
            "Feature"
        ] = feature

        detail[
            "OriginalValue"
        ] = original_values[
            mismatch_mask
        ]

        detail[
            "ReconstructedValue"
        ] = reconstructed_values[
            mismatch_mask
        ]

        detail[
            "AbsoluteDifference"
        ] = absolute_difference[
            mismatch_mask
        ]

        mismatch_detail_frames.append(
            detail
        )


comparison_summary = pd.DataFrame(
    comparison_summary_records
)


if mismatch_detail_frames:

    mismatch_details = pd.concat(
        mismatch_detail_frames,
        ignore_index=True,
    )

else:

    mismatch_details = pd.DataFrame(
        columns=[
            "Build",
            "Test",
            "Partition",
            "Feature",
            "OriginalValue",
            "ReconstructedValue",
            "AbsoluteDifference",
        ]
    )


total_rec_mismatches = int(
    comparison_summary[
        "MismatchingRows"
    ].sum()
)


training_rec_mismatches = int(
    comparison_summary[
        "TrainingMismatches"
    ].sum()
)


evaluation_rec_mismatches = int(
    comparison_summary[
        "EvaluationMismatches"
    ].sum()
)


rows_with_any_rec_mismatch = int(
    mismatch_details[
        [
            "Build",
            "Test",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


print("\nREC feature comparison summary:")

display(
    comparison_summary
)


# ------------------------------------------------------------
# 14. DOCUMENT UNMATCHED-COMMIT IMPACT
# ------------------------------------------------------------

required_mapping_status_columns = {
    "CanonicalBuild",
    "Partition",
    "MappingStatus",
    "ChangedEntities",
    "ModelReadyRows",
    "ModelReadyFailures",
}

missing_mapping_status_columns = (
    required_mapping_status_columns
    - set(
        mapping_status_by_build.columns
    )
)

if missing_mapping_status_columns:
    raise RuntimeError(
        "The Step 5 mapping-status file is missing "
        "required columns:\n"
        f"{sorted(missing_mapping_status_columns)}"
    )


mapping_status_working = (
    mapping_status_by_build.copy()
)


mapping_status_working[
    "CanonicalBuild"
] = pd.to_numeric(
    mapping_status_working[
        "CanonicalBuild"
    ],
    errors="raise",
).astype(
    np.int64
)


for column in [
    "ChangedEntities",
    "ModelReadyRows",
    "ModelReadyFailures",
]:

    mapping_status_working[
        column
    ] = pd.to_numeric(
        mapping_status_working[
            column
        ],
        errors="raise",
    ).astype(int)


affected_model_ready_builds = (
    mapping_status_working[
        mapping_status_working[
            "MappingStatus"
        ].ne(
            "FULL_COMMIT_COVERAGE"
        )
        &
        mapping_status_working[
            "ModelReadyRows"
        ].gt(0)
    ]
    .copy()
)


affected_model_ready_rows = int(
    affected_model_ready_builds[
        "ModelReadyRows"
    ].sum()
)


affected_model_ready_failures = int(
    affected_model_ready_builds[
        "ModelReadyFailures"
    ].sum()
)


affected_training_model_rows = int(
    affected_model_ready_builds.loc[
        affected_model_ready_builds[
            "Partition"
        ].eq("training"),
        "ModelReadyRows",
    ].sum()
)


affected_evaluation_model_rows = int(
    affected_model_ready_builds.loc[
        affected_model_ready_builds[
            "Partition"
        ].eq("evaluation"),
        "ModelReadyRows",
    ].sum()
)


print(
    "\nModel-ready builds affected by incomplete "
    "commit coverage:"
)

if affected_model_ready_builds.empty:

    print("None")

else:

    display(
        affected_model_ready_builds[
            [
                "CanonicalBuild",
                "BuildOrder",
                "Partition",
                "CommitTokens",
                "MatchedCommitTokens",
                "UnmatchedCommitTokens",
                "ChangedEntities",
                "ModelReadyRows",
                "ModelReadyFailures",
                "MappingStatus",
            ]
        ]
    )


# ------------------------------------------------------------
# 15. VERIFY SOURCE DATA REMAINED UNCHANGED
# ------------------------------------------------------------

dataset_hash_after = (
    dataframe_content_sha256(
        dataset[
            dataset_snapshot_columns
        ]
    )
)


dataset_hash_unchanged = bool(
    dataset_hash_before
    == dataset_hash_after
)


# ------------------------------------------------------------
# 16. CREATE STEP 6 AUDIT
# ------------------------------------------------------------

audit_records = [
    {
        "Check":
            "Step 5 REC-pending status accepted",

        "Expected":
            True,

        "Actual":
            detected_step5_status
            in accepted_step5_statuses,

        "Pass":
            detected_step5_status
            in accepted_step5_statuses,
    },

    {
        "Check":
            "REC feature count",

        "Expected":
            19,

        "Actual":
            len(
                REC_FEATURE_COLUMNS
            ),

        "Pass":
            len(
                REC_FEATURE_COLUMNS
            )
            == 19,
    },

    {
        "Check":
            "Dataset Build-Test duplicates",

        "Expected":
            0,

        "Actual":
            dataset_duplicate_build_test_pairs,

        "Pass":
            dataset_duplicate_build_test_pairs
            == 0,
    },

    {
        "Check":
            "Raw Build-Test duplicates",

        "Expected":
            0,

        "Actual":
            raw_duplicate_build_test_pairs,

        "Pass":
            raw_duplicate_build_test_pairs
            == 0,
    },

    {
        "Check":
            "Expected reconstructed rows",

        "Expected":
            expected_dataset_rows,

        "Actual":
            reconstructed_rows,

        "Pass":
            reconstructed_rows
            == expected_dataset_rows,
    },

    {
        "Check":
            "Duplicate reconstructed Build-Test pairs",

        "Expected":
            0,

        "Actual":
            duplicate_reconstructed_pairs,

        "Pass":
            duplicate_reconstructed_pairs
            == 0,
    },

    {
        "Check":
            "REC merge missing rows",

        "Expected":
            0,

        "Actual":
            rec_merge_missing_rows,

        "Pass":
            rec_merge_missing_rows
            == 0,
    },

    {
        "Check":
            "Label merge missing rows",

        "Expected":
            0,

        "Actual":
            label_merge_missing_rows,

        "Pass":
            label_merge_missing_rows
            == 0,
    },

    {
        "Check":
            "Clean label mismatches",

        "Expected":
            0,

        "Actual":
            label_mismatches,

        "Pass":
            label_mismatches
            == 0,
    },

    {
        "Check":
            "Training label mismatches",

        "Expected":
            0,

        "Actual":
            training_label_mismatches,

        "Pass":
            training_label_mismatches
            == 0,
    },

    {
        "Check":
            "Evaluation label mismatches",

        "Expected":
            0,

        "Actual":
            evaluation_label_mismatches,

        "Pass":
            evaluation_label_mismatches
            == 0,
    },

    {
        "Check":
            "Total REC mismatches",

        "Expected":
            0,

        "Actual":
            total_rec_mismatches,

        "Pass":
            total_rec_mismatches
            == 0,
    },

    {
        "Check":
            "Training REC mismatches",

        "Expected":
            0,

        "Actual":
            training_rec_mismatches,

        "Pass":
            training_rec_mismatches
            == 0,
    },

    {
        "Check":
            "Evaluation REC mismatches",

        "Expected":
            0,

        "Actual":
            evaluation_rec_mismatches,

        "Pass":
            evaluation_rec_mismatches
            == 0,
    },

    {
        "Check":
            "Rows with any REC mismatch",

        "Expected":
            0,

        "Actual":
            rows_with_any_rec_mismatch,

        "Pass":
            rows_with_any_rec_mismatch
            == 0,
    },

    {
        "Check":
            "Changed-entity dictionary builds",

        "Expected":
            len(
                chronology_working
            ),

        "Actual":
            changed_entity_dictionary_builds,

        "Pass":
            changed_entity_dictionary_builds
            == len(
                chronology_working
            ),
    },

    {
        "Check":
            "Evaluation model rows affected by "
            "incomplete commit mapping",

        "Expected":
            0,

        "Actual":
            affected_evaluation_model_rows,

        "Pass":
            affected_evaluation_model_rows
            == 0,
    },

    {
        "Check":
            "Source dataset hash unchanged",

        "Expected":
            True,

        "Actual":
            dataset_hash_unchanged,

        "Pass":
            dataset_hash_unchanged,
    },
]


audit_frame = pd.DataFrame(
    audit_records
)


print("\nStep 6 validation audit:")

display(
    audit_frame
)


failed_audit_checks = (
    audit_frame[
        ~audit_frame[
            "Pass"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 17. DETERMINE STEP 6 STATUS
# ------------------------------------------------------------

step6_pass = bool(
    len(
        failed_audit_checks
    )
    == 0
)


step6_status_text = (
    "PASS_PROJECT_8_CLEAN_REC_RECONSTRUCTION_VALIDATED"
    if step6_pass
    else
    "FAIL_PROJECT_8_CLEAN_REC_RECONSTRUCTION_VALIDATION"
)


# ------------------------------------------------------------
# 18. SAVE STEP 6 OUTPUT TABLES
# ------------------------------------------------------------

PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


rec_clean_reconstructed.to_csv(
    CLEAN_REC_RECONSTRUCTED_PATH,
    index=False,
    compression="gzip",
)


comparison_summary.to_csv(
    REC_COMPARISON_SUMMARY_PATH,
    index=False,
)


mismatch_details.to_csv(
    REC_MISMATCH_DETAILS_PATH,
    index=False,
    compression="gzip",
)


label_comparison.to_csv(
    LABEL_COMPARISON_PATH,
    index=False,
    compression="gzip",
)


audit_frame.to_csv(
    STEP6_AUDIT_PATH,
    index=False,
)


table_outputs = [
    CLEAN_REC_RECONSTRUCTED_PATH,
    REC_COMPARISON_SUMMARY_PATH,
    REC_MISMATCH_DETAILS_PATH,
    LABEL_COMPARISON_PATH,
    STEP6_AUDIT_PATH,
]


missing_table_outputs = [
    str(path)
    for path in table_outputs
    if not path.exists()
]


if missing_table_outputs:
    raise RuntimeError(
        "Step 6 failed to save expected outputs:\n"
        + "\n".join(
            missing_table_outputs
        )
    )


output_metadata = {
    path.name: file_metadata(
        path
    )
    for path in table_outputs
}


# ------------------------------------------------------------
# 19. CREATE STEP 6 REPORT
# ------------------------------------------------------------

step6_report = {
    "project_number":
        PROJECT_NUMBER,

    "project":
        PROJECT_NAME,

    "project_slug":
        PROJECT_SLUG,

    "generated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "step5_status":
        detected_step5_status,

    "protocol": {
        "recent_window":
            RECENT_WINDOW,

        "rec_feature_count":
            len(
                REC_FEATURE_COLUMNS
            ),

        "rec_features":
            REC_FEATURE_COLUMNS,

        "history_rule":
            (
                "use only executions before the "
                "current Build-Test execution"
            ),

        "verdict_encoding": {
            "pass":
                0,

            "exception_failure":
                1,

            "assertion_failure":
                2,
        },

        "comparison_tolerance": {
            "relative":
                1e-9,

            "absolute":
                1e-9,

            "equal_nan":
                True,
        },
    },

    "input_summary": {
        "chronology_builds":
            int(
                len(
                    chronology_working
                )
            ),

        "raw_execution_rows":
            int(
                len(
                    exe_for_rec
                )
            ),

        "model_ready_rows":
            expected_dataset_rows,

        "requested_build_test_pairs":
            int(
                len(
                    requested_rows
                )
            ),

        "build_entity_pairs":
            int(
                len(
                    entity_build_pairs
                )
            ),

        "changed_entity_dictionary_builds":
            changed_entity_dictionary_builds,
    },

    "reconstruction_summary": {
        "expected_rows":
            expected_dataset_rows,

        "reconstructed_rows":
            reconstructed_rows,

        "duplicate_reconstructed_pairs":
            duplicate_reconstructed_pairs,

        "rec_merge_missing_rows":
            rec_merge_missing_rows,

        "label_merge_missing_rows":
            label_merge_missing_rows,

        "clean_label_mismatches":
            label_mismatches,

        "training_label_mismatches":
            training_label_mismatches,

        "evaluation_label_mismatches":
            evaluation_label_mismatches,

        "total_rec_mismatches":
            total_rec_mismatches,

        "training_rec_mismatches":
            training_rec_mismatches,

        "evaluation_rec_mismatches":
            evaluation_rec_mismatches,

        "rows_with_any_rec_mismatch":
            rows_with_any_rec_mismatch,

        "feature_comparison":
            comparison_summary
            .to_dict(
                orient="records"
            ),
    },

    "incomplete_commit_mapping_impact": {
        "affected_model_ready_builds":
            int(
                len(
                    affected_model_ready_builds
                )
            ),

        "affected_model_ready_rows":
            affected_model_ready_rows,

        "affected_model_ready_failures":
            affected_model_ready_failures,

        "affected_training_model_rows":
            affected_training_model_rows,

        "affected_evaluation_model_rows":
            affected_evaluation_model_rows,

        "affected_builds":
            affected_model_ready_builds[
                [
                    "CanonicalBuild",
                    "BuildOrder",
                    "Partition",
                    "CommitTokens",
                    "MatchedCommitTokens",
                    "UnmatchedCommitTokens",
                    "ChangedEntities",
                    "ModelReadyRows",
                    "ModelReadyFailures",
                    "MappingStatus",
                ]
            ]
            .to_dict(
                orient="records"
            ),
    },

    "dataset_integrity": {
        "hash_before":
            dataset_hash_before,

        "hash_after":
            dataset_hash_after,

        "unchanged":
            dataset_hash_unchanged,
    },

    "audit":
        audit_frame.to_dict(
            orient="records"
        ),

    "failed_audit_check_count":
        int(
            len(
                failed_audit_checks
            )
        ),

    "outputs":
        output_metadata,

    "status":
        step6_status_text,
}


STEP6_REPORT_PATH.write_text(
    json.dumps(
        step6_report,
        indent=2,
        default=json_safe,
    ),
    encoding="utf-8",
)


step6_status = {
    "project_number":
        PROJECT_NUMBER,

    "project":
        PROJECT_NAME,

    "project_slug":
        PROJECT_SLUG,

    "status":
        step6_status_text,

    "reconstructed_rows":
        reconstructed_rows,

    "clean_label_mismatches":
        label_mismatches,

    "total_rec_mismatches":
        total_rec_mismatches,

    "training_rec_mismatches":
        training_rec_mismatches,

    "evaluation_rec_mismatches":
        evaluation_rec_mismatches,

    "affected_model_ready_builds_with_incomplete_mapping":
        int(
            len(
                affected_model_ready_builds
            )
        ),

    "failed_audit_check_count":
        int(
            len(
                failed_audit_checks
            )
        ),

    "generated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


STEP6_STATUS_PATH.write_text(
    json.dumps(
        step6_status,
        indent=2,
        default=json_safe,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 20. VERIFY ALL STEP 6 OUTPUTS
# ------------------------------------------------------------

all_expected_outputs = (
    table_outputs
    + [
        STEP6_REPORT_PATH,
        STEP6_STATUS_PATH,
    ]
)


missing_outputs = [
    str(path)
    for path in all_expected_outputs
    if not path.exists()
]


if missing_outputs:
    raise RuntimeError(
        "Some Step 6 outputs are missing:\n"
        + "\n".join(
            missing_outputs
        )
    )


# ------------------------------------------------------------
# 21. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 82)
print("=== PROJECT 8 STEP 6 RESULT ===")
print("=" * 82)

print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nREC protocol:")

print(
    "REC features:",
    len(
        REC_FEATURE_COLUMNS
    ),
)

print(
    "Recent-history window:",
    RECENT_WINDOW,
)

print(
    "History rule:",
    "prior executions only",
)


print("\nInput population:")

print(
    "Chronological builds:",
    len(
        chronology_working
    ),
)

print(
    "Raw execution rows:",
    len(
        exe_for_rec
    ),
)

print(
    "Model-ready rows:",
    expected_dataset_rows,
)


print("\nClean reconstruction:")

print(
    "Reconstructed rows:",
    reconstructed_rows,
)

print(
    "Duplicate reconstructed pairs:",
    duplicate_reconstructed_pairs,
)

print(
    "REC merge missing rows:",
    rec_merge_missing_rows,
)

print(
    "Label merge missing rows:",
    label_merge_missing_rows,
)


print("\nClean-label validation:")

print(
    "Total label mismatches:",
    label_mismatches,
)

print(
    "Training label mismatches:",
    training_label_mismatches,
)

print(
    "Evaluation label mismatches:",
    evaluation_label_mismatches,
)


print("\nREC validation:")

print(
    "Total REC feature mismatches:",
    total_rec_mismatches,
)

print(
    "Training REC mismatches:",
    training_rec_mismatches,
)

print(
    "Evaluation REC mismatches:",
    evaluation_rec_mismatches,
)

print(
    "Rows with any REC mismatch:",
    rows_with_any_rec_mismatch,
)


print("\nIncomplete commit-mapping impact:")

print(
    "Affected model-ready builds:",
    len(
        affected_model_ready_builds
    ),
)

print(
    "Affected model-ready rows:",
    affected_model_ready_rows,
)

print(
    "Affected model-ready failures:",
    affected_model_ready_failures,
)

print(
    "Affected evaluation model rows:",
    affected_evaluation_model_rows,
)


print("\nSource integrity:")

print(
    "dataset.csv hash unchanged:",
    dataset_hash_unchanged,
)


print("\nValidation:")

print(
    "Audit checks:",
    len(
        audit_frame
    ),
)

print(
    "Failed audit checks:",
    len(
        failed_audit_checks
    ),
)


print("\nStep 6 outputs:")

for output_path in all_expected_outputs:
    print(
        output_path
    )


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–7 modified:")
print(0)


print(
    "\nSTATUS:",
    step6_status_text,
)


if step6_pass:

    print(
        "\nSUCCESS: All 19 REC features were "
        "reconstructed from clean chronological history."
    )

    print(
        "SUCCESS: Only prior executions were used for "
        "every requested Build-Test row."
    )

    print(
        "SUCCESS: Clean model-ready verdicts were "
        "reproduced exactly."
    )

    print(
        "SUCCESS: Clean REC features were "
        "reproduced exactly."
    )

    print(
        "SUCCESS: Documented unmatched commits did not "
        "invalidate the clean REC reconstruction."
    )

    print(
        "SUCCESS: Project 8 is ready for protocol "
        "freezing and canonical helper validation."
    )


print("=" * 82)


if not step6_pass:

    print(
        "\nFailed Step 6 audit checks:"
    )

    display(
        failed_audit_checks
    )

    if not mismatch_details.empty:

        print(
            "\nFirst 100 REC mismatch details:"
        )

        display(
            mismatch_details.head(100)
        )

    if label_mismatches > 0:

        print(
            "\nFirst 100 clean label mismatches:"
        )

        display(
            label_comparison[
                ~label_comparison[
                    "LabelMatches"
                ]
            ].head(100)
        )

    raise RuntimeError(
        "PROJECT 8 STEP 6 DID NOT PASS.\n"
        "Do not proceed to protocol freezing or noise "
        "experiments until the displayed clean "
        "reconstruction problems are resolved."
    )

=== PROJECT 8 STEP 6: CLEAN REC RECONSTRUCTION VALIDATION ===

Loading Step 6 inputs...
dataset.csv: (9880, 154)
Canonical raw executions: (34438, 14)
Build chronology: (254, 6)
Build-entity map: (11075, 4)
Mapping-status rows: (254, 13)
Changed-entity dictionary builds: 254

Reconstructing all 19 REC features...
Expected dataset rows: 9880
Reconstructed rows: 9880
Duplicate reconstructed Build-Test pairs: 0

REC feature comparison summary:


,Feature,Rows,MatchingRows,MismatchingRows,TrainingMismatches,EvaluationMismatches,OriginalMissingValues,ReconstructedMissingValues,MaximumAbsoluteDifference
0,REC_Age,9880,9880,0,0,0,0,0,0.000000e+00
1,REC_LastFailureAge,9880,9880,0,0,0,0,0,0.000000e+00
2,REC_LastTransitionAge,9880,9880,0,0,0,0,0,0.000000e+00
3,REC_RecentAvgExeTime,9880,9880,0,0,0,0,0,2.910383e-11
4,REC_RecentMaxExeTime,9880,9880,0,0,0,0,0,0.000000e+00
5,REC_RecentFailRate,9880,9880,0,0,0,0,0,5.551115e-17
6,REC_RecentAssertRate,9880,9880,0,0,0,0,0,5.551115e-17
7,REC_RecentExcRate,9880,9880,0,0,0,0,0,5.551115e-17
8,REC_RecentTransitionRate,9880,9880,0,0,0,0,0,5.551115e-17
9,REC_TotalAvgExeTime,9880,9880,0,0,0,0,0,5.820766e-11



Model-ready builds affected by incomplete commit coverage:


,CanonicalBuild,BuildOrder,Partition,CommitTokens,MatchedCommitTokens,UnmatchedCommitTokens,ChangedEntities,ModelReadyRows,ModelReadyFailures,MappingStatus
107,672911037,108,training,1,0,1,0,134,1,NO_MATCHED_COMMITS
181,741430250,182,training,1,0,1,0,141,1,NO_MATCHED_COMMITS



Step 6 validation audit:


,Check,Expected,Actual,Pass
0,Step 5 REC-pending status accepted,True,True,True
1,REC feature count,19,19,True
2,Dataset Build-Test duplicates,0,0,True
3,Raw Build-Test duplicates,0,0,True
4,Expected reconstructed rows,9880,9880,True
5,Duplicate reconstructed Build-Test pairs,0,0,True
6,REC merge missing rows,0,0,True
7,Label merge missing rows,0,0,True
8,Clean label mismatches,0,0,True
9,Training label mismatches,0,0,True




=== PROJECT 8 STEP 6 RESULT ===

Project identity:
Project number: 8
Project: optimatika@ojAlgo
Project slug: optimatika__ojAlgo

REC protocol:
REC features: 19
Recent-history window: 6
History rule: prior executions only

Input population:
Chronological builds: 254
Raw execution rows: 34438
Model-ready rows: 9880

Clean reconstruction:
Reconstructed rows: 9880
Duplicate reconstructed pairs: 0
REC merge missing rows: 0
Label merge missing rows: 0

Clean-label validation:
Total label mismatches: 0
Training label mismatches: 0
Evaluation label mismatches: 0

REC validation:
Total REC feature mismatches: 0
Training REC mismatches: 0
Evaluation REC mismatches: 0
Rows with any REC mismatch: 0

Incomplete commit-mapping impact:
Affected model-ready builds: 2
Affected model-ready rows: 275
Affected model-ready failures: 2
Affected evaluation model rows: 0

Source integrity:
dataset.csv hash unchanged: True

Validation:
Audit checks: 18
Failed audit checks: 0

Step 6 outputs:
/content/drive/

In [ ]:
# ============================================================
# PROJECT 8 — STEP 7A
# CANONICAL PROTOCOL AND HELPER SOURCE DISCOVERY
#
# PROJECT: optimatika@ojAlgo
#
# Purpose:
# - locate the exact protocol/configuration/helper artefacts
#   used by completed Projects 1–7
# - inspect Notes and frozen project packages
# - detect repeated and byte-identical artefacts
# - print relevant JSON/configuration previews
# - list Python helper function names
#
# READ-ONLY:
# - no Google Drive files are created or modified
# - completion registry is not modified
# - Projects 1–7 are not modified
# - Project 8 files are not modified
# ============================================================

from pathlib import Path
from IPython.display import display

import ast
import hashlib
import json
import re

import pandas as pd


# ------------------------------------------------------------
# 1. PATHS AND CONFIGURATION
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

AGGREGATED_DIR = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

PROJECT_8_SLUG = "optimatika__ojAlgo"

MAX_TEXT_FILE_BYTES = (
    8 * 1024 * 1024
)

MAX_PREVIEW_CHARACTERS = 6000

TEXT_EXTENSIONS = {
    ".json",
    ".py",
    ".yaml",
    ".yml",
    ".toml",
    ".ini",
    ".cfg",
    ".txt",
    ".md",
    ".csv",
    ".ipynb",
}

KEYWORDS = [
    "protocol",
    "configuration",
    "config",
    "helper",
    "canonical",
    "noise",
    "corruption",
    "label",
    "seed",
    "condition",
    "model",
    "randomforest",
    "xgboost",
    "lightgbm",
    "naivebayes",
    "gaussiannb",
    "random",
    "latestfail",
    "qtf",
    "apfd",
    "apfdc",
    "metric",
    "smoke",
    "validation",
    "experiment",
    "training",
    "evaluation",
    "rec",
]

HIGH_VALUE_FILENAME_TERMS = [
    "step7",
    "protocol",
    "model_config",
    "model_configuration",
    "experiment_config",
    "experiment_configuration",
    "helper",
    "canonical",
    "noise",
    "apfd",
    "smoke",
]


print("=" * 84)
print("=== PROJECT 8 STEP 7A: PROTOCOL AND HELPER SOURCE DISCOVERY ===")
print("=" * 84)


# ------------------------------------------------------------
# 2. HELPERS
# ------------------------------------------------------------

def normalise_text(value):
    if value is None:
        return ""

    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).lower(),
    )


def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def read_text_file(path):
    """
    Read a candidate text file without modifying it.

    For notebooks, concatenate source from code and
    markdown cells.
    """

    path = Path(path)

    if (
        not path.is_file()
        or path.stat().st_size
        > MAX_TEXT_FILE_BYTES
    ):
        return None

    try:
        if path.suffix.lower() == ".ipynb":
            notebook = json.loads(
                path.read_text(
                    encoding="utf-8",
                    errors="ignore",
                )
            )

            extracted_cells = []

            for cell in notebook.get(
                "cells",
                [],
            ):
                source = cell.get(
                    "source",
                    [],
                )

                if isinstance(source, list):
                    source = "".join(
                        source
                    )

                extracted_cells.append(
                    str(source)
                )

            return "\n\n".join(
                extracted_cells
            )

        return path.read_text(
            encoding="utf-8",
            errors="ignore",
        )

    except Exception:
        return None


def extract_python_functions(text):
    """
    Extract Python function names using AST where possible,
    then fall back to a regular expression.
    """

    if not text:
        return []

    function_names = set()

    try:
        tree = ast.parse(
            text
        )

        for node in ast.walk(
            tree
        ):
            if isinstance(
                node,
                (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                ),
            ):
                function_names.add(
                    node.name
                )

    except Exception:
        matches = re.findall(
            r"(?m)^\s*def\s+([A-Za-z_][A-Za-z0-9_]*)\s*\(",
            text,
        )

        function_names.update(
            matches
        )

    return sorted(
        function_names
    )


def extract_json_top_keys(
    path,
    text,
):
    if (
        Path(path).suffix.lower()
        != ".json"
        or not text
    ):
        return []

    try:
        payload = json.loads(
            text
        )

        if isinstance(
            payload,
            dict,
        ):
            return sorted(
                str(key)
                for key in payload.keys()
            )

    except Exception:
        pass

    return []


def keyword_hits(text):
    if not text:
        return []

    lowered = text.lower()

    return sorted({
        keyword
        for keyword in KEYWORDS
        if keyword in lowered
    })


def filename_hits(path):
    name = Path(path).name.lower()

    return sorted({
        term
        for term in KEYWORDS
        if term in name
    })


def high_value_filename(path):
    name = Path(path).name.lower()

    return any(
        term in name
        for term in HIGH_VALUE_FILENAME_TERMS
    )


def scan_candidate(
    path,
    source_group,
    project_slug=None,
):
    path = Path(path)

    if (
        not path.is_file()
        or path.suffix.lower()
        not in TEXT_EXTENSIONS
    ):
        return None

    file_size = int(
        path.stat().st_size
    )

    text = read_text_file(
        path
    )

    name_hits = filename_hits(
        path
    )

    content_hits = keyword_hits(
        text
    )

    is_high_value_name = (
        high_value_filename(
            path
        )
    )

    relevance_score = (
        5 * len(
            name_hits
        )
        + len(
            content_hits
        )
        + (
            15
            if is_high_value_name
            else 0
        )
    )

    # Ignore files with no clear relationship to the
    # experiment protocol or canonical helpers.
    if (
        relevance_score < 3
        and not is_high_value_name
    ):
        return None

    try:
        relative_path = str(
            path.relative_to(
                THESIS_ROOT
            )
        )
    except Exception:
        relative_path = str(
            path
        )

    return {
        "SourceGroup":
            source_group,

        "ProjectSlug":
            project_slug,

        "Name":
            path.name,

        "Extension":
            path.suffix.lower(),

        "RelativePath":
            relative_path,

        "AbsolutePath":
            str(path),

        "SizeBytes":
            file_size,

        "SHA256":
            calculate_sha256(
                path
            ),

        "FilenameHits":
            ", ".join(
                name_hits
            ),

        "ContentHits":
            ", ".join(
                content_hits
            ),

        "ContentHitCount":
            len(
                content_hits
            ),

        "RelevanceScore":
            relevance_score,

        "PythonFunctions":
            ", ".join(
                extract_python_functions(
                    text
                )
            ),

        "JsonTopKeys":
            ", ".join(
                extract_json_top_keys(
                    path,
                    text,
                )
            ),

        "_Text":
            text,
    }


def preview_candidate(record):
    print("\n" + "-" * 84)

    print(
        "PATH:",
        record[
            "AbsolutePath"
        ],
    )

    print(
        "SOURCE:",
        record[
            "SourceGroup"
        ],
    )

    if record.get(
        "ProjectSlug"
    ):
        print(
            "PROJECT:",
            record[
                "ProjectSlug"
            ],
        )

    print(
        "SIZE:",
        record[
            "SizeBytes"
        ],
        "bytes",
    )

    print(
        "SHA-256:",
        record[
            "SHA256"
        ],
    )

    print(
        "FILENAME HITS:",
        record[
            "FilenameHits"
        ],
    )

    print(
        "CONTENT HITS:",
        record[
            "ContentHits"
        ],
    )

    if record[
        "JsonTopKeys"
    ]:
        print(
            "JSON TOP-LEVEL KEYS:",
            record[
                "JsonTopKeys"
            ],
        )

    if record[
        "PythonFunctions"
    ]:
        print(
            "PYTHON FUNCTIONS:",
            record[
                "PythonFunctions"
            ],
        )

    text = record.get(
        "_Text"
    )

    if not text:
        print(
            "TEXT PREVIEW: unavailable"
        )
        return

    preview = text[
        :MAX_PREVIEW_CHARACTERS
    ]

    print("\nCONTENT PREVIEW:")
    print(preview)

    if len(text) > len(
        preview
    ):
        print(
            "\n[Preview truncated]"
        )


# ------------------------------------------------------------
# 3. VALIDATE COMPLETION REGISTRY
# ------------------------------------------------------------

if not REGISTRY_PATH.exists():
    raise FileNotFoundError(
        "Completion registry not found:\n"
        f"{REGISTRY_PATH}"
    )


registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
)


required_registry_columns = {
    "ProjectNumber",
    "Project",
    "ProjectSlug",
    "Status",
}

missing_registry_columns = (
    required_registry_columns
    - set(
        registry.columns
    )
)

if missing_registry_columns:
    raise RuntimeError(
        "Completion registry is missing columns:\n"
        f"{sorted(missing_registry_columns)}"
    )


registry[
    "_ProjectNumber"
] = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="coerce",
)


registry[
    "_Status"
] = (
    registry[
        "Status"
    ]
    .astype(str)
    .str.strip()
    .str.upper()
)


frozen_registry = (
    registry[
        registry[
            "_Status"
        ].eq(
            "COMPLETE_AND_FROZEN"
        )
    ]
    .copy()
    .sort_values(
        "_ProjectNumber",
        kind="mergesort",
    )
)


frozen_numbers = (
    frozen_registry[
        "_ProjectNumber"
    ]
    .dropna()
    .astype(int)
    .tolist()
)


if frozen_numbers != list(
    range(
        1,
        8,
    )
):
    raise AssertionError(
        "Expected completed frozen projects 1–7.\n"
        f"Found project numbers: {frozen_numbers}"
    )


completed_project_slugs = (
    frozen_registry[
        "ProjectSlug"
    ]
    .astype(str)
    .tolist()
)


print("\nCompletion registry:")

print(
    "Registry rows:",
    len(
        registry
    ),
)

print(
    "COMPLETE_AND_FROZEN projects:",
    len(
        frozen_registry
    ),
)

print(
    "Completed project slugs:"
)

for slug in completed_project_slugs:
    print(
        " -",
        slug,
    )


# ------------------------------------------------------------
# 4. SCAN NOTES
# ------------------------------------------------------------

candidate_records = []


if not NOTES_DIR.exists():
    raise FileNotFoundError(
        "Notes directory not found:\n"
        f"{NOTES_DIR}"
    )


notes_files_scanned = 0


for path in NOTES_DIR.rglob(
    "*"
):
    if not path.is_file():
        continue

    notes_files_scanned += 1

    record = scan_candidate(
        path=path,
        source_group="Notes",
        project_slug=None,
    )

    if record is not None:
        candidate_records.append(
            record
        )


# ------------------------------------------------------------
# 5. SCAN COMPLETED PROJECT PACKAGES
# ------------------------------------------------------------

completed_project_directories_found = 0

completed_project_files_scanned = 0


for project_slug in completed_project_slugs:

    project_directory = (
        AGGREGATED_DIR
        / project_slug
    )

    if not project_directory.exists():
        print(
            "\nWARNING: completed project directory "
            "was not found:",
            project_directory,
        )
        continue

    completed_project_directories_found += 1

    for path in project_directory.rglob(
        "*"
    ):
        if not path.is_file():
            continue

        completed_project_files_scanned += 1

        record = scan_candidate(
            path=path,
            source_group=(
                "CompletedProject"
            ),
            project_slug=project_slug,
        )

        if record is not None:
            candidate_records.append(
                record
            )


# ------------------------------------------------------------
# 6. BUILD CANDIDATE INVENTORY
# ------------------------------------------------------------

if not candidate_records:
    raise RuntimeError(
        "No protocol/configuration/helper candidates "
        "were found."
    )


candidate_frame = pd.DataFrame(
    candidate_records
)


candidate_frame = (
    candidate_frame
    .sort_values(
        [
            "RelevanceScore",
            "SourceGroup",
            "ProjectSlug",
            "RelativePath",
        ],
        ascending=[
            False,
            True,
            True,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


display_columns = [
    "SourceGroup",
    "ProjectSlug",
    "Name",
    "RelativePath",
    "SizeBytes",
    "RelevanceScore",
    "FilenameHits",
    "ContentHits",
    "JsonTopKeys",
    "PythonFunctions",
    "SHA256",
]


print("\nTop Notes candidates:")

notes_candidates = (
    candidate_frame[
        candidate_frame[
            "SourceGroup"
        ].eq("Notes")
    ]
    .head(40)
)

display(
    notes_candidates[
        display_columns
    ]
)


print(
    "\nTop completed-project candidates:"
)

project_candidates = (
    candidate_frame[
        candidate_frame[
            "SourceGroup"
        ].eq(
            "CompletedProject"
        )
    ]
    .head(80)
)

display(
    project_candidates[
        display_columns
    ]
)


# ------------------------------------------------------------
# 7. IDENTIFY REPEATED BASENAMES
# ------------------------------------------------------------

completed_candidates = (
    candidate_frame[
        candidate_frame[
            "SourceGroup"
        ].eq(
            "CompletedProject"
        )
    ]
    .copy()
)


repeated_basename_summary = (
    completed_candidates
    .groupby(
        "Name",
        dropna=False,
    )
    .agg(
        Projects=(
            "ProjectSlug",
            lambda values: int(
                pd.Series(
                    values
                )
                .dropna()
                .nunique()
            ),
        ),

        Files=(
            "AbsolutePath",
            "count",
        ),

        UniqueHashes=(
            "SHA256",
            "nunique",
        ),

        ProjectSlugs=(
            "ProjectSlug",
            lambda values: ", ".join(
                sorted({
                    str(value)
                    for value in values
                    if pd.notna(value)
                })
            ),
        ),

        Hashes=(
            "SHA256",
            lambda values: ", ".join(
                sorted({
                    str(value)
                    for value in values
                    if pd.notna(value)
                })
            ),
        ),
    )
    .reset_index()
)


repeated_basename_summary = (
    repeated_basename_summary[
        repeated_basename_summary[
            "Projects"
        ].ge(2)
    ]
    .sort_values(
        [
            "Projects",
            "UniqueHashes",
            "Name",
        ],
        ascending=[
            False,
            True,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


print(
    "\nCandidate basenames repeated across "
    "completed projects:"
)

display(
    repeated_basename_summary.head(
        100
    )
)


# ------------------------------------------------------------
# 8. IDENTIFY BYTE-IDENTICAL CROSS-PROJECT FILES
# ------------------------------------------------------------

identical_hash_summary = (
    completed_candidates
    .groupby(
        "SHA256",
        dropna=False,
    )
    .agg(
        Projects=(
            "ProjectSlug",
            lambda values: int(
                pd.Series(
                    values
                )
                .dropna()
                .nunique()
            ),
        ),

        Files=(
            "AbsolutePath",
            "count",
        ),

        Names=(
            "Name",
            lambda values: ", ".join(
                sorted({
                    str(value)
                    for value in values
                    if pd.notna(value)
                })
            ),
        ),

        ProjectSlugs=(
            "ProjectSlug",
            lambda values: ", ".join(
                sorted({
                    str(value)
                    for value in values
                    if pd.notna(value)
                })
            ),
        ),

        Paths=(
            "RelativePath",
            lambda values: "\n".join(
                sorted({
                    str(value)
                    for value in values
                    if pd.notna(value)
                })
            ),
        ),
    )
    .reset_index()
)


identical_hash_summary = (
    identical_hash_summary[
        identical_hash_summary[
            "Projects"
        ].ge(2)
    ]
    .sort_values(
        [
            "Projects",
            "Files",
            "Names",
        ],
        ascending=[
            False,
            False,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


print(
    "\nByte-identical candidate files shared by "
    "multiple completed projects:"
)

display(
    identical_hash_summary.head(
        100
    )
)


# ------------------------------------------------------------
# 9. IDENTIFY STEP-7/PROTOCOL/HELPER-SPECIFIC FILES
# ------------------------------------------------------------

specific_name_pattern = re.compile(
    r"step[_\- ]?7"
    r"|protocol"
    r"|model[_\- ]?(config|configuration)"
    r"|experiment[_\- ]?(config|configuration)"
    r"|canonical[_\- ]?helper"
    r"|noise[_\- ]?(config|protocol|helper)"
    r"|apfdc?"
    r"|smoke",
    flags=re.IGNORECASE,
)


specific_candidates = (
    candidate_frame[
        candidate_frame[
            "Name"
        ].str.contains(
            specific_name_pattern,
            na=False,
        )
    ]
    .copy()
    .sort_values(
        [
            "SourceGroup",
            "ProjectSlug",
            "Name",
        ],
        kind="mergesort",
    )
)


print(
    "\nFiles with explicit Step 7, protocol, helper, "
    "model, APFD or smoke-test names:"
)

display(
    specific_candidates[
        display_columns
    ].head(150)
)


# ------------------------------------------------------------
# 10. PREVIEW THE MOST RELEVANT SMALL CONFIGURATION FILES
# ------------------------------------------------------------

preview_candidates = (
    candidate_frame[
        candidate_frame[
            "Extension"
        ].isin(
            {
                ".json",
                ".py",
                ".yaml",
                ".yml",
                ".toml",
                ".ini",
                ".cfg",
                ".txt",
                ".md",
            }
        )
        &
        candidate_frame[
            "SizeBytes"
        ].le(
            250_000
        )
        &
        (
            candidate_frame[
                "RelevanceScore"
            ].ge(12)
            |
            candidate_frame[
                "Name"
            ].str.contains(
                specific_name_pattern,
                na=False,
            )
        )
    ]
    .copy()
)


# Prioritise Notes, then the most recently completed project,
# then cross-project candidates.

latest_completed_slug = (
    frozen_registry.iloc[-1][
        "ProjectSlug"
    ]
)


preview_candidates[
    "_PreviewPriority"
] = np.where(
    preview_candidates[
        "SourceGroup"
    ].eq("Notes"),
    0,
    np.where(
        preview_candidates[
            "ProjectSlug"
        ].eq(
            latest_completed_slug
        ),
        1,
        2,
    ),
)


preview_candidates = (
    preview_candidates
    .sort_values(
        [
            "_PreviewPriority",
            "RelevanceScore",
            "Name",
        ],
        ascending=[
            True,
            False,
            True,
        ],
        kind="mergesort",
    )
    .drop_duplicates(
        subset=[
            "SHA256",
        ],
        keep="first",
    )
    .head(15)
)


print(
    "\nDetailed previews of the most relevant "
    "unique candidates:"
)


for _, preview_record in (
    preview_candidates.iterrows()
):
    preview_candidate(
        preview_record.to_dict()
    )


# ------------------------------------------------------------
# 11. FINAL RESULT
# ------------------------------------------------------------

notes_candidate_count = int(
    candidate_frame[
        "SourceGroup"
    ].eq("Notes").sum()
)


completed_candidate_count = int(
    candidate_frame[
        "SourceGroup"
    ].eq(
        "CompletedProject"
    ).sum()
)


specific_candidate_count = int(
    len(
        specific_candidates
    )
)


cross_project_identical_groups = int(
    len(
        identical_hash_summary
    )
)


discovery_pass = bool(
    notes_candidate_count > 0
    and completed_candidate_count > 0
    and specific_candidate_count > 0
)


status_text = (
    "PASS_PROJECT_8_PROTOCOL_SOURCES_DISCOVERED"
    if discovery_pass
    else
    "PROJECT_8_PROTOCOL_SOURCES_REQUIRE_MANUAL_REVIEW"
)


print("\n")
print("=" * 84)
print("=== PROJECT 8 STEP 7A RESULT ===")
print("=" * 84)

print("\nRead-only scan:")

print(
    "Notes files scanned:",
    notes_files_scanned,
)

print(
    "Completed project directories found:",
    completed_project_directories_found,
)

print(
    "Completed project files scanned:",
    completed_project_files_scanned,
)


print("\nCandidates:")

print(
    "Notes candidates:",
    notes_candidate_count,
)

print(
    "Completed-project candidates:",
    completed_candidate_count,
)

print(
    "Explicit protocol/helper candidates:",
    specific_candidate_count,
)

print(
    "Repeated candidate basenames:",
    len(
        repeated_basename_summary
    ),
)

print(
    "Cross-project identical candidate groups:",
    cross_project_identical_groups,
)

print(
    "Detailed unique previews printed:",
    len(
        preview_candidates
    ),
)


print("\nGoogle Drive writes:")
print(0)

print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–7 modified:")
print(0)

print("\nProject 8 modified:")
print(0)


print(
    "\nSTATUS:",
    status_text,
)

print("=" * 84)

=== PROJECT 8 STEP 7A: PROTOCOL AND HELPER SOURCE DISCOVERY ===

Completion registry:
Registry rows: 7
COMPLETE_AND_FROZEN projects: 7
Completed project slugs:
 - Angel-ML__angel
 - apache__airavata
 - b2ihealthcare__snow-owl
 - eclipse__paho.mqtt.java
 - thinkaurelius__titan
 - eclipse__jetty.project
 - CompEvol__beast2

Top Notes candidates:


,SourceGroup,ProjectSlug,Name,RelativePath,SizeBytes,RelevanceScore,FilenameHits,ContentHits,JsonTopKeys,PythonFunctions,SHA256
1,Notes,None,project_06_experiment_protocol.json,Notes/project_06_experiment_protocol.json,3088,45,"experiment, protocol","apfd, apfdc, condition, evaluation, experiment...","chronology, clean_anchor, evaluation_partition...",,94c6697aefebe077b76badc8274a920e56170d8b1bc695...
8,Notes,None,eclipse__paho.mqtt.java_experiment_protocol_v1...,Notes/eclipse__paho.mqtt.java_experiment_proto...,2248,43,"experiment, protocol","apfd, apfdc, evaluation, experiment, latestfai...","baselines, evaluation_partition, feature_handl...",,962e5bfdb5a0ad10b324a9918d48526209c1d9f6fb4506...
9,Notes,None,project_06_jetty_model_configuration.json,Notes/project_06_jetty_model_configuration.json,6733,43,"config, configuration, model","condition, evaluation, gaussiannb, label, ligh...","ActivePredictorCount, ActivePredictors, Comple...",,e353643100ca851f911112e2e6186b0645335d4e63102b...
10,Notes,None,thinkaurelius__titan_experiment_protocol_fixed...,Notes/thinkaurelius__titan_experiment_protocol...,2391,43,"experiment, protocol","apfd, apfdc, evaluation, helper, latestfail, l...","Baselines, EvaluationPartition, FailureToPassR...",,16bcb91e95fc829318b40f1161553530d2e0994739e898...
11,Notes,None,angel_experiment_protocol_pilot_v1.json,Notes/angel_experiment_protocol_pilot_v1.json,1638,42,"experiment, protocol","apfd, apfdc, evaluation, latestfail, lightgbm,...","baselines, evaluation_metrics, evaluation_part...",,562bc81251b63d7529c97f5d628a4c8a97ec433d5bd8c6...
12,Notes,None,apache__airavata_experiment_protocol_pilot_v1....,Notes/apache__airavata_experiment_protocol_pil...,1641,42,"experiment, protocol","apfd, apfdc, evaluation, latestfail, lightgbm,...","baselines, evaluation_metrics, evaluation_part...",,28f636ea10b76f108e40e98ac92419816e02d4da89b075...
13,Notes,None,b2ihealthcare__snow-owl_experiment_protocol_v1...,Notes/b2ihealthcare__snow-owl_experiment_proto...,1775,42,"experiment, protocol","apfd, apfdc, evaluation, experiment, latestfai...","baselines, dataset, evaluation_partition, feat...",,756a6c1238d4fef2e6de734a298e7559ce790c3f758659...
14,Notes,None,eclipse__paho.mqtt.java_model_configuration_v1...,Notes/eclipse__paho.mqtt.java_model_configurat...,6665,42,"config, configuration, model","condition, evaluation, gaussiannb, lightgbm, m...","active_feature_count, active_features, feature...",,66fd3e337345afc11db2a3cb1feee76e4cec462e33a3b2...
15,Notes,None,thinkaurelius__titan_model_configuration_fixed...,Notes/thinkaurelius__titan_model_configuration...,6819,42,"config, configuration, model","condition, evaluation, gaussiannb, lightgbm, m...","ActivePredictorCount, ActivePredictors, Infini...",,e649a9dae8d17feebc7a257de5f89257626a370a4c80ae...
18,Notes,None,b2ihealthcare__snow-owl_model_configuration_v1...,Notes/b2ihealthcare__snow-owl_model_configurat...,1265,40,"config, configuration, model","condition, gaussiannb, lightgbm, metric, model...","active_feature_count, infinite_value_rule, mis...",,38ecca3e5d594d6313973f9f3a0e0419633a94c71151bb...



Top completed-project candidates:


,SourceGroup,ProjectSlug,Name,RelativePath,SizeBytes,RelevanceScore,FilenameHits,ContentHits,JsonTopKeys,PythonFunctions,SHA256
0,CompletedProject,eclipse__jetty.project,project_06_experiment_protocol.json,Results/Aggregated/eclipse__jetty.project/jett...,3088,45,"experiment, protocol","apfd, apfdc, condition, evaluation, experiment...","chronology, clean_anchor, evaluation_partition...",,94c6697aefebe077b76badc8274a920e56170d8b1bc695...
2,CompletedProject,thinkaurelius__titan,titan_noise_helper_validation_report.json,Results/Aggregated/thinkaurelius__titan/titan_...,2198,44,"helper, noise, validation","apfd, apfdc, condition, evaluation, experiment...","CompletedAtUTC, ConditionSummary, DifferentSee...",,ca35054ceba95a03fb1b95ef2c1f453a74783bfaa9215f...
3,CompletedProject,CompEvol__beast2,model_configuration.json,Results/Aggregated/CompEvol__beast2/beast2_pre...,1342,43,"config, configuration, model","config, configuration, gaussiannb, lightgbm, m...","ActivePredictors, InfiniteValueRule, LibraryVe...",,23c76cbae8846348201b69a0b3d9a258b790283ce0de52...
4,CompletedProject,eclipse__jetty.project,jetty_smoke_validation_report.json,Results/Aggregated/eclipse__jetty.project/jett...,1677,43,"smoke, validation","canonical, condition, evaluation, experiment, ...","BaselineSignatureAudit, BaselineSignatureMisma...",,33eb5432c69d1aa4cc062ca67f4e3ca081d9a8fb4cb201...
5,CompletedProject,eclipse__jetty.project,project_06_jetty_model_configuration.json,Results/Aggregated/eclipse__jetty.project/jett...,6733,43,"config, configuration, model","condition, evaluation, gaussiannb, label, ligh...","ActivePredictorCount, ActivePredictors, Comple...",,e353643100ca851f911112e2e6186b0645335d4e63102b...
...,...,...,...,...,...,...,...,...,...,...,...
87,CompletedProject,CompEvol__beast2,xgboost__noise_050.csv,Results/Aggregated/CompEvol__beast2/beast2_30_...,9549,30,"noise, xgboost","apfd, apfdc, noise, seed, xgboost",,,bb5d78b89b933770a714a76b1d5a6f08096a594eba8d6f...
88,CompletedProject,eclipse__jetty.project,apfd_apfdc_formula_audit_v1.csv,Results/Aggregated/eclipse__jetty.project/jett...,652,30,"apfd, apfdc","apfd, apfdc, experiment, helper, rec",,,90809fb7fd804ca0dde6772b737272eae7c0479a582a3d...
89,CompletedProject,eclipse__jetty.project,apfd_apfdc_formula_audit_v1.json,Results/Aggregated/eclipse__jetty.project/jett...,1659,30,"apfd, apfdc","apfd, apfdc, experiment, helper, rec","AllAPFDComparisonsPassed, AllAPFDcComparisonsP...",,fd50825c1ee9df0ca304154e397409bc5712504e313533...
92,CompletedProject,eclipse__jetty.project,jetty_smoke_fit_times.csv,Results/Aggregated/eclipse__jetty.project/jett...,459,29,smoke,"evaluation, lightgbm, naivebayes, noise, rando...",,,d811d5d682cf80b50b1b8d48897b21ba4076713801ec08...



Candidate basenames repeated across completed projects:


...


Files with explicit Step 7, protocol, helper, model, APFD or smoke-test names:


/tmp/ipykernel_1633/2285601019.py:1107: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ].str.contains(


,SourceGroup,ProjectSlug,Name,RelativePath,SizeBytes,RelevanceScore,FilenameHits,ContentHits,JsonTopKeys,PythonFunctions,SHA256
23,CompletedProject,Angel-ML__angel,angel_30seed_mean_apfdc_wide.csv,Results/Aggregated/Angel-ML__angel/angel_30_se...,1321,38,"apfd, apfdc, seed","latestfail, lightgbm, naivebayes, noise, qtf, ...",,,722400d679e256cd0355b896f494d319dcff2f4456d5e4...
24,CompletedProject,Angel-ML__angel,angel_30seed_sd_apfdc_wide.csv,Results/Aggregated/Angel-ML__angel/angel_30_se...,1437,38,"apfd, apfdc, seed","latestfail, lightgbm, naivebayes, noise, qtf, ...",,,595725b399cb1704729172e394f0cdfc8e497961e04fea...
25,CompletedProject,Angel-ML__angel,seed_01_apfdc_wide.csv,Results/Aggregated/Angel-ML__angel/seed_01_apf...,1313,38,"apfd, apfdc, seed","latestfail, lightgbm, naivebayes, noise, qtf, ...",,,cfa84c18f366c892fd7e2f9bfe2333ab8686496b7bf090...
16,CompletedProject,CompEvol__beast2,beast2_noise_helper_validation_report.json,Results/Aggregated/CompEvol__beast2/beast2_pre...,3719,41,"helper, noise, validation","condition, evaluation, experiment, label, mode...","Artefacts, CompletedAtUTC, CorrectionApplied, ...",,081208f8a7a7f79c9d87950217be36617d9b0515a0c026...
3,CompletedProject,CompEvol__beast2,model_configuration.json,Results/Aggregated/CompEvol__beast2/beast2_pre...,1342,43,"config, configuration, model","config, configuration, gaussiannb, lightgbm, m...","ActivePredictors, InfiniteValueRule, LibraryVe...",,23c76cbae8846348201b69a0b3d9a258b790283ce0de52...
26,CompletedProject,apache__airavata,airavata_30seed_mean_apfdc_wide.csv,Results/Aggregated/apache__airavata/airavata_3...,1320,38,"apfd, apfdc, seed","latestfail, lightgbm, naivebayes, noise, qtf, ...",,,05dabc79ad8905f4f69d5ee9f22584fa05c62edb88ead4...
27,CompletedProject,apache__airavata,airavata_30seed_sd_apfdc_wide.csv,Results/Aggregated/apache__airavata/airavata_3...,1435,38,"apfd, apfdc, seed","latestfail, lightgbm, naivebayes, noise, qtf, ...",,,d7ecc796cfada837f69c1a35e38e2e8d0bcc0f101bd494...
28,CompletedProject,apache__airavata,seed_01_apfdc_wide.csv,Results/Aggregated/apache__airavata/seed_01_ap...,1308,38,"apfd, apfdc, seed","latestfail, lightgbm, naivebayes, noise, qtf, ...",,,caa6c170ad1b3f0350cbb76e487335dd1f2e571f0ec67b...
17,CompletedProject,b2ihealthcare__snow-owl,snow_owl_canonical_helper_validation.json,Results/Aggregated/b2ihealthcare__snow-owl/sno...,989,41,"canonical, helper, validation","apfd, apfdc, evaluation, label, metric, model,...","EvaluationUnchanged, FailureSubtypeCounts, Fiv...",,9e55646efa9f4cd3d60e4b89e8478f511bcf09ea5ec764...
88,CompletedProject,eclipse__jetty.project,apfd_apfdc_formula_audit_v1.csv,Results/Aggregated/eclipse__jetty.project/jett...,652,30,"apfd, apfdc","apfd, apfdc, experiment, helper, rec",,,90809fb7fd804ca0dde6772b737272eae7c0479a582a3d...



Detailed previews of the most relevant unique candidates:

------------------------------------------------------------------------------------
PATH: /content/drive/MyDrive/Thesis_Experiment/Notes/project_06_experiment_protocol.json
SOURCE: Notes
SIZE: 3088 bytes
SHA-256: 94c6697aefebe077b76badc8274a920e56170d8b1bc695768d1ae34d0e490065
FILENAME HITS: experiment, protocol
CONTENT HITS: apfd, apfdc, condition, evaluation, experiment, label, latestfail, lightgbm, metric, model, naivebayes, noise, protocol, qtf, random, randomforest, rec, seed, training, xgboost
JSON TOP-LEVEL KEYS: chronology, clean_anchor, evaluation_partition, feature_policy, fixed_instance_design, flip_rule, full_project_conditions, full_project_model_fits, metrics, noise_levels_percent, noise_mask, noise_partition, noise_unit, project, project_number, project_slug, protocol_status, repetition_seeds, split, techniques

CONTENT PREVIEW:
{
  "project_number": 6,
  "project": "eclipse@jetty.project",
  "project_slug": "e

/tmp/ipykernel_1633/2285601019.py:1171: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ].str.contains(


In [ ]:
# ============================================================
# PROJECT 8 — STEP 7B
# RUNTIME RESTORE + PROTOCOL AND MODEL-CONFIGURATION FREEZE
#
# PROJECT: optimatika@ojAlgo
#
# This cell:
# - restores the temporary source extraction when necessary
# - validates the completed Steps 1–6
# - verifies the canonical protocol/configuration sources
# - checks the established chronology rule
# - determines ojAlgo's project-specific active predictors
# - freezes the exact experiment protocol
# - freezes the exact model configuration
#
# It does NOT:
# - run noise conditions
# - fit ML models
# - modify Projects 1–7
# - modify the completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from IPython.display import display

import copy
import hashlib
import json
import os
import platform
import re
import shutil
import tarfile

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

PROJECT_NUMBER = 8
PROJECT_NAME = "optimatika@ojAlgo"
PROJECT_SLUG = "optimatika__ojAlgo"
PROJECT_SHORT_NAME = "ojalgo"

NOISE_LEVELS_PERCENT = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

REPETITION_SEEDS = list(
    range(1, 31)
)

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

RECENT_WINDOW = 6

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

EXPECTED_CONDITIONS = (
    len(NOISE_LEVELS_PERCENT)
    * len(REPETITION_SEEDS)
)

EXPECTED_MODEL_FITS = (
    EXPECTED_CONDITIONS
    * len(ML_TECHNIQUES)
)


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

ARCHIVE_PATH = (
    THESIS_ROOT
    / "Data"
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)

LOCAL_EXTRACTION_PARENT = Path(
    "/content"
)

LOCAL_EXTRACTION_ROOT = (
    LOCAL_EXTRACTION_PARENT
    / "TCP-CI-main-dataset"
)

PROJECT_SOURCE_DIR = (
    LOCAL_EXTRACTION_ROOT
    / "datasets"
    / PROJECT_NAME
)

BUILDS_PATH = (
    PROJECT_SOURCE_DIR
    / "builds.csv"
)

EXE_PATH = (
    PROJECT_SOURCE_DIR
    / "exe.csv"
)

DATASET_PATH = (
    PROJECT_SOURCE_DIR
    / "dataset.csv"
)

ID_MAP_PATH = (
    PROJECT_SOURCE_DIR
    / "id_map.csv"
)

ENTITY_HISTORY_PATH = (
    PROJECT_SOURCE_DIR
    / "entity_change_history.csv"
)


PROJECT_AGGREGATED_DIR = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / PROJECT_SLUG
)

PREFLIGHT_DIR = (
    PROJECT_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_preflight"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

SCREENING_SNAPSHOT_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_screening_snapshot.csv"
)

BUILD_CHRONOLOGY_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_build_chronology.csv"
)

CANONICAL_RAW_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_canonical_raw_executions.csv.gz"
)

STEP5_STATUS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step5_status.json"
)

STEP6_STATUS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step6_status.json"
)

STEP6_REPORT_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_validation_report.json"
)


# Canonical sources discovered by Step 7A

CANONICAL_PROTOCOL_SOURCE = (
    NOTES_DIR
    / "project_06_experiment_protocol.json"
)

CANONICAL_MODEL_SOURCE = (
    NOTES_DIR
    / "project_06_jetty_model_configuration.json"
)

APFD_AUDIT_SOURCE = (
    NOTES_DIR
    / "apfd_apfdc_formula_audit_v1.json"
)

PROJECT7_CHECKPOINT_SOURCE = (
    NOTES_DIR
    / "project_07_selection_checkpoint.json"
)


# Expected hashes established by Step 7A

EXPECTED_SOURCE_HASHES = {
    str(CANONICAL_PROTOCOL_SOURCE):
        "94c6697aefebe077b76badc8274a920e56170d8b1bc695768d1ae34d0e490065",

    str(CANONICAL_MODEL_SOURCE):
        "e353643100ca851f911112e2e6186b0645335d4e63102b0837a10d7feb19e6dd",

    str(APFD_AUDIT_SOURCE):
        "fd50825c1ee9df0ca304154e397409bc5712504e313533e59507d83d58fdd44f",

    str(PROJECT7_CHECKPOINT_SOURCE):
        "7077773f779577a667666f8c9c5796059e0bfe4b865ffdd8c3eac260f2ebc156",
}


# Step 7B outputs

PROTOCOL_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_experiment_protocol.json"
)

MODEL_CONFIGURATION_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_model_configuration.json"
)

PREDICTOR_PROFILE_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_predictor_profile.csv"
)

PROTOCOL_AUDIT_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_protocol_freeze_audit.csv"
)

STEP7B_REPORT_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step7b_protocol_freeze_report.json"
)

STEP7B_STATUS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step7b_status.json"
)


# Permanent Notes copies

NOTES_PROTOCOL_PATH = (
    NOTES_DIR
    / "project_08_experiment_protocol.json"
)

NOTES_MODEL_CONFIGURATION_PATH = (
    NOTES_DIR
    / "project_08_ojalgo_model_configuration.json"
)


print("=" * 84)
print("=== PROJECT 8 STEP 7B: PROTOCOL AND MODEL-CONFIGURATION FREEZE ===")
print("=" * 84)


# ------------------------------------------------------------
# 3. HELPERS
# ------------------------------------------------------------

def calculate_hash(
    path,
    algorithm="sha256",
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.new(
        algorithm
    )

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def json_safe(value):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return value


def canonical_id(value):
    if pd.isna(value):
        return None

    text = str(value).strip()

    if text == "":
        return None

    if re.fullmatch(
        r"[-+]?\d+\.0+",
        text,
    ):
        return text.split(".")[0]

    return text


def expected_integer(
    row,
    column,
):
    if column not in row.index:
        return None

    value = row[
        column
    ]

    if pd.isna(value):
        return None

    text = str(value).strip()

    if text == "":
        return None

    return int(
        float(text)
    )


def package_version(
    distribution_name,
):
    try:
        return metadata.version(
            distribution_name
        )
    except metadata.PackageNotFoundError:
        return None


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(
        path
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(
        path
    )


def safely_extract_archive(
    archive_path,
    extraction_parent,
):
    extraction_parent = Path(
        extraction_parent
    ).resolve()

    with tarfile.open(
        archive_path,
        mode="r:gz",
    ) as archive:

        members = archive.getmembers()

        for member in members:
            target_path = (
                extraction_parent
                / member.name
            ).resolve()

            valid_root = (
                str(extraction_parent)
                + os.sep
            )

            if not (
                str(target_path)
                == str(extraction_parent)
                or str(target_path).startswith(
                    valid_root
                )
            ):
                raise RuntimeError(
                    "Unsafe archive member detected:\n"
                    f"{member.name}"
                )

        try:
            archive.extractall(
                path=extraction_parent,
                members=members,
                filter="data",
            )

        except TypeError:
            # Compatibility fallback for older tarfile APIs.
            archive.extractall(
                path=extraction_parent,
                members=members,
            )


# ------------------------------------------------------------
# 4. VALIDATE DRIVE AND RESTORE RUNTIME EXTRACTION
# ------------------------------------------------------------

if not THESIS_ROOT.exists():
    raise FileNotFoundError(
        "Thesis Drive directory is not available:\n"
        f"{THESIS_ROOT}\n\n"
        "Confirm that Google Drive is mounted."
    )


required_source_files = [
    BUILDS_PATH,
    EXE_PATH,
    DATASET_PATH,
    ID_MAP_PATH,
    ENTITY_HISTORY_PATH,
]


runtime_restore_performed = False


if not all(
    path.exists()
    for path in required_source_files
):

    if not ARCHIVE_PATH.exists():
        raise FileNotFoundError(
            "The permanent TCP-CI archive is missing:\n"
            f"{ARCHIVE_PATH}"
        )

    print("\nTemporary runtime extraction is missing.")
    print("Restoring from the existing Drive archive...")
    print("No network download is being performed.")

    archive_size_mb = (
        ARCHIVE_PATH.stat().st_size
        / (1024 ** 2)
    )

    available_disk_gb = (
        shutil.disk_usage(
            "/content"
        ).free
        / (1024 ** 3)
    )

    print(
        "Archive size:",
        round(
            archive_size_mb,
            2,
        ),
        "MB",
    )

    print(
        "Available runtime disk:",
        round(
            available_disk_gb,
            2,
        ),
        "GB",
    )

    if available_disk_gb < 3:
        raise RuntimeError(
            "Insufficient temporary disk space for extraction."
        )

    safely_extract_archive(
        archive_path=ARCHIVE_PATH,
        extraction_parent=
            LOCAL_EXTRACTION_PARENT,
    )

    runtime_restore_performed = True


missing_after_restore = [
    str(path)
    for path in required_source_files
    if not path.exists()
]


if missing_after_restore:
    raise FileNotFoundError(
        "Required source files are still missing after "
        "runtime restoration:\n"
        + "\n".join(
            missing_after_restore
        )
    )


print("\nRuntime source directory:")
print(PROJECT_SOURCE_DIR)

print(
    "Runtime restoration performed:",
    runtime_restore_performed,
)


# ------------------------------------------------------------
# 5. VALIDATE PREVIOUS PROJECT 8 STEPS
# ------------------------------------------------------------

required_previous_outputs = [
    SCREENING_SNAPSHOT_PATH,
    BUILD_CHRONOLOGY_PATH,
    CANONICAL_RAW_PATH,
    STEP5_STATUS_PATH,
    STEP6_STATUS_PATH,
    STEP6_REPORT_PATH,
]


missing_previous_outputs = [
    str(path)
    for path in required_previous_outputs
    if not path.exists()
]


if missing_previous_outputs:
    raise FileNotFoundError(
        "Required Project 8 preflight outputs are missing:\n"
        + "\n".join(
            missing_previous_outputs
        )
    )


step5_status = json.loads(
    STEP5_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step6_status = json.loads(
    STEP6_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step6_report = json.loads(
    STEP6_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)


accepted_step5_statuses = {
    "PASS_PENDING_CLEAN_REC_VALIDATION",
    (
        "PASS_PENDING_CLEAN_REC_VALIDATION_"
        "WITH_DOCUMENTED_UNMATCHED_COMMITS"
    ),
}


if (
    step5_status.get("status")
    not in accepted_step5_statuses
):
    raise AssertionError(
        "Project 8 Step 5 is not in an accepted state:\n"
        f"{step5_status.get('status')}"
    )


expected_step6_status = (
    "PASS_PROJECT_8_CLEAN_REC_RECONSTRUCTION_VALIDATED"
)


if (
    step6_status.get("status")
    != expected_step6_status
):
    raise AssertionError(
        "Project 8 Step 6 has not passed:\n"
        f"{step6_status.get('status')}"
    )


# ------------------------------------------------------------
# 6. VALIDATE COMPLETION REGISTRY
# ------------------------------------------------------------

registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
)


required_registry_columns = {
    "ProjectNumber",
    "Project",
    "ProjectSlug",
    "Status",
}


missing_registry_columns = (
    required_registry_columns
    - set(
        registry.columns
    )
)


if missing_registry_columns:
    raise RuntimeError(
        "Completion registry is missing columns:\n"
        f"{sorted(missing_registry_columns)}"
    )


registry[
    "_ProjectNumber"
] = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="coerce",
)


registry[
    "_Status"
] = (
    registry[
        "Status"
    ]
    .astype(str)
    .str.strip()
    .str.upper()
)


frozen_registry = registry[
    registry[
        "_Status"
    ].eq(
        "COMPLETE_AND_FROZEN"
    )
].copy()


frozen_project_numbers = sorted(
    frozen_registry[
        "_ProjectNumber"
    ]
    .dropna()
    .astype(int)
    .tolist()
)


existing_project8_rows = registry[
    registry[
        "_ProjectNumber"
    ].eq(PROJECT_NUMBER)
]


# ------------------------------------------------------------
# 7. VERIFY CANONICAL SOURCES FROM STEP 7A
# ------------------------------------------------------------

canonical_source_paths = [
    CANONICAL_PROTOCOL_SOURCE,
    CANONICAL_MODEL_SOURCE,
    APFD_AUDIT_SOURCE,
    PROJECT7_CHECKPOINT_SOURCE,
]


missing_canonical_sources = [
    str(path)
    for path in canonical_source_paths
    if not path.exists()
]


if missing_canonical_sources:
    raise FileNotFoundError(
        "Canonical protocol sources are missing:\n"
        + "\n".join(
            missing_canonical_sources
        )
    )


canonical_source_hashes = {
    str(path): calculate_hash(
        path
    )
    for path in canonical_source_paths
}


source_hash_mismatches = []

for path_text, expected_hash in (
    EXPECTED_SOURCE_HASHES.items()
):

    actual_hash = canonical_source_hashes[
        path_text
    ]

    if actual_hash != expected_hash:
        source_hash_mismatches.append({
            "Path":
                path_text,

            "ExpectedSHA256":
                expected_hash,

            "ActualSHA256":
                actual_hash,
        })


if source_hash_mismatches:
    display(
        pd.DataFrame(
            source_hash_mismatches
        )
    )

    raise RuntimeError(
        "One or more canonical protocol sources changed "
        "after Step 7A."
    )


canonical_protocol_source = json.loads(
    CANONICAL_PROTOCOL_SOURCE.read_text(
        encoding="utf-8"
    )
)

canonical_model_source = json.loads(
    CANONICAL_MODEL_SOURCE.read_text(
        encoding="utf-8"
    )
)

apfd_audit = json.loads(
    APFD_AUDIT_SOURCE.read_text(
        encoding="utf-8"
    )
)

project7_checkpoint = json.loads(
    PROJECT7_CHECKPOINT_SOURCE.read_text(
        encoding="utf-8"
    )
)


canonical_models = (
    canonical_model_source.get(
        "Models"
    )
    or canonical_model_source.get(
        "models"
    )
)


if not isinstance(
    canonical_models,
    dict,
):
    raise RuntimeError(
        "The canonical model source has no Models dictionary."
    )


if set(
    canonical_models.keys()
) != set(
    ML_TECHNIQUES
):
    raise AssertionError(
        "Canonical model names differ from the frozen "
        "four-technique protocol.\n"
        f"Detected: {sorted(canonical_models.keys())}"
    )


# ------------------------------------------------------------
# 8. LOAD PROJECT 8 DATA
# ------------------------------------------------------------

builds = pd.read_csv(
    BUILDS_PATH,
    low_memory=False,
)

dataset = pd.read_csv(
    DATASET_PATH,
    low_memory=False,
)

canonical_raw = pd.read_csv(
    CANONICAL_RAW_PATH,
    low_memory=False,
)

chronology = pd.read_csv(
    BUILD_CHRONOLOGY_PATH,
    dtype=str,
)

screening_snapshot = pd.read_csv(
    SCREENING_SNAPSHOT_PATH,
    dtype=str,
)


if len(screening_snapshot) != 1:
    raise AssertionError(
        "Expected exactly one Project 8 screening row."
    )


screening_row = (
    screening_snapshot.iloc[0]
)


# ------------------------------------------------------------
# 9. VALIDATE THE ESTABLISHED CHRONOLOGY RULE
# ------------------------------------------------------------

required_build_columns = {
    "id",
    "started_at",
}


missing_build_columns = (
    required_build_columns
    - set(
        builds.columns
    )
)


if missing_build_columns:
    raise RuntimeError(
        "builds.csv is missing chronology columns:\n"
        f"{sorted(missing_build_columns)}"
    )


required_chronology_columns = {
    "CanonicalBuild",
    "BuildOrder",
    "Partition",
}


missing_chronology_columns = (
    required_chronology_columns
    - set(
        chronology.columns
    )
)


if missing_chronology_columns:
    raise RuntimeError(
        "The frozen chronology is missing columns:\n"
        f"{sorted(missing_chronology_columns)}"
    )


builds_for_order = builds.copy()


builds_for_order[
    "_BuildKey"
] = builds_for_order[
    "id"
].map(
    canonical_id
)


builds_for_order[
    "_BuildNumeric"
] = pd.to_numeric(
    builds_for_order[
        "_BuildKey"
    ],
    errors="raise",
)


builds_for_order[
    "_Timestamp"
] = pd.to_datetime(
    builds_for_order[
        "started_at"
    ],
    errors="coerce",
    utc=True,
)


missing_timestamps = int(
    builds_for_order[
        "_Timestamp"
    ].isna().sum()
)


if missing_timestamps != 0:
    raise AssertionError(
        "builds.csv contains missing timestamps."
    )


timestamp_group_sizes = (
    builds_for_order
    .groupby(
        "_Timestamp"
    )
    .size()
)


equal_timestamp_groups = int(
    timestamp_group_sizes
    .gt(1)
    .sum()
)


equal_timestamp_builds = int(
    timestamp_group_sizes.loc[
        timestamp_group_sizes
        .gt(1)
    ].sum()
)


canonical_order = (
    builds_for_order
    .sort_values(
        [
            "_Timestamp",
            "_BuildNumeric",
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )[
        "_BuildKey"
    ]
    .astype(str)
    .tolist()
)


frozen_order = (
    chronology
    .assign(
        _BuildOrderNumeric=pd.to_numeric(
            chronology[
                "BuildOrder"
            ],
            errors="raise",
        )
    )
    .sort_values(
        "_BuildOrderNumeric",
        kind="mergesort",
    )[
        "CanonicalBuild"
    ]
    .map(
        canonical_id
    )
    .astype(str)
    .tolist()
)


chronology_order_matches = bool(
    canonical_order
    == frozen_order
)


if not chronology_order_matches:

    order_comparison = pd.DataFrame({
        "Position":
            np.arange(
                1,
                len(
                    canonical_order
                ) + 1,
            ),

        "CanonicalRuleBuild":
            canonical_order,

        "FrozenChronologyBuild":
            frozen_order,
    })


    order_mismatches = order_comparison[
        order_comparison[
            "CanonicalRuleBuild"
        ].ne(
            order_comparison[
                "FrozenChronologyBuild"
            ]
        )
    ]


    print(
        "\nChronology mismatches:"
    )

    display(
        order_mismatches.head(50)
    )

    raise AssertionError(
        "The Step 3 chronology does not match the "
        "established rule: started_at ascending and "
        "build ID descending for equal timestamps."
    )


training_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq("training"),
        "CanonicalBuild",
    ]
    .map(
        canonical_id
    )
    .dropna()
    .tolist()
)


evaluation_builds = set(
    chronology.loc[
        chronology[
            "Partition"
        ].eq("evaluation"),
        "CanonicalBuild",
    ]
    .map(
        canonical_id
    )
    .dropna()
    .tolist()
)


# ------------------------------------------------------------
# 10. DETERMINE PROJECT-SPECIFIC PREDICTORS
# ------------------------------------------------------------

required_dataset_metadata = {
    "Build",
    "Test",
    "Verdict",
    "Duration",
}


missing_dataset_metadata = (
    required_dataset_metadata
    - set(
        dataset.columns
    )
)


if missing_dataset_metadata:
    raise RuntimeError(
        "dataset.csv is missing metadata columns:\n"
        f"{sorted(missing_dataset_metadata)}"
    )


predictor_columns = [
    column
    for column in dataset.columns
    if column not in required_dataset_metadata
]


dataset_build_keys = (
    dataset[
        "Build"
    ]
    .map(
        canonical_id
    )
)


training_row_mask = (
    dataset_build_keys
    .isin(
        training_builds
    )
)


evaluation_row_mask = (
    dataset_build_keys
    .isin(
        evaluation_builds
    )
)


training_rows = int(
    training_row_mask.sum()
)


evaluation_rows = int(
    evaluation_row_mask.sum()
)


numeric_predictors = dataset[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)


original_non_null = dataset[
    predictor_columns
].notna()


numeric_conversion_failures = int(
    (
        original_non_null
        & numeric_predictors.isna()
    ).sum().sum()
)


positive_infinities = int(
    np.isposinf(
        numeric_predictors
        .to_numpy(
            dtype=float
        )
    ).sum()
)


negative_infinities = int(
    np.isneginf(
        numeric_predictors
        .to_numpy(
            dtype=float
        )
    ).sum()
)


numeric_predictors_clean = (
    numeric_predictors
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
)


training_predictors = (
    numeric_predictors_clean.loc[
        training_row_mask
    ]
)


training_unique_counts = (
    training_predictors
    .nunique(
        dropna=True
    )
)


zero_variance_predictors = (
    training_unique_counts[
        training_unique_counts
        .le(1)
    ]
    .index
    .tolist()
)


active_predictors = [
    column
    for column in predictor_columns
    if column not in zero_variance_predictors
]


predictor_profile_records = []


for predictor in predictor_columns:

    predictor_profile_records.append({
        "Predictor":
            predictor,

        "TrainingUniqueNonNullValues":
            int(
                training_unique_counts[
                    predictor
                ]
            ),

        "TrainingMissingValues":
            int(
                training_predictors[
                    predictor
                ]
                .isna()
                .sum()
            ),

        "EvaluationMissingValues":
            int(
                numeric_predictors_clean.loc[
                    evaluation_row_mask,
                    predictor,
                ]
                .isna()
                .sum()
            ),

        "ZeroVarianceInCleanTraining":
            predictor
            in zero_variance_predictors,

        "ActivePredictor":
            predictor
            in active_predictors,
    })


predictor_profile = pd.DataFrame(
    predictor_profile_records
)


# ------------------------------------------------------------
# 11. FAILURE-SUBTYPE DISTRIBUTION
# ------------------------------------------------------------

required_raw_columns = {
    "Partition",
    "RawVerdictToken",
}


missing_raw_columns = (
    required_raw_columns
    - set(
        canonical_raw.columns
    )
)


if missing_raw_columns:
    raise RuntimeError(
        "The canonical execution table is missing columns:\n"
        f"{sorted(missing_raw_columns)}"
    )


training_raw = canonical_raw[
    canonical_raw[
        "Partition"
    ].eq("training")
].copy()


training_raw[
    "_Verdict"
] = pd.to_numeric(
    training_raw[
        "RawVerdictToken"
    ],
    errors="raise",
).astype(int)


training_failure_subtypes = training_raw.loc[
    training_raw[
        "_Verdict"
    ].ne(0),
    "_Verdict",
]


failure_subtype_counts = {
    str(
        int(subtype)
    ): int(count)
    for subtype, count
    in training_failure_subtypes
    .value_counts()
    .sort_index()
    .items()
}


total_training_failure_executions = int(
    len(
        training_failure_subtypes
    )
)


failure_subtype_probabilities = {
    subtype: (
        count
        / total_training_failure_executions
    )
    for subtype, count
    in failure_subtype_counts.items()
}


failure_probability_sum = float(
    sum(
        failure_subtype_probabilities.values()
    )
)


# ------------------------------------------------------------
# 12. PACKAGE VERSIONS
# ------------------------------------------------------------

library_versions = {
    "python":
        platform.python_version(),

    "numpy":
        package_version(
            "numpy"
        ),

    "pandas":
        package_version(
            "pandas"
        ),

    "scikit-learn":
        package_version(
            "scikit-learn"
        ),

    "xgboost":
        package_version(
            "xgboost"
        ),

    "lightgbm":
        package_version(
            "lightgbm"
        ),
}


missing_required_libraries = [
    package_name
    for package_name in [
        "scikit-learn",
        "xgboost",
        "lightgbm",
    ]
    if library_versions[
        package_name
    ] is None
]


# ------------------------------------------------------------
# 13. APFD/APFDC AUDIT VALIDATION
# ------------------------------------------------------------

apfd_audit_passed = bool(
    apfd_audit.get(
        "AllAPFDComparisonsPassed"
    )
)


apfdc_audit_passed = bool(
    apfd_audit.get(
        "AllAPFDcComparisonsPassed"
    )
)


# ------------------------------------------------------------
# 14. PROJECT COUNTS
# ------------------------------------------------------------

total_builds = expected_integer(
    screening_row,
    "TotalBuilds",
)

training_period_builds = expected_integer(
    screening_row,
    "TrainingPeriodBuilds",
)

evaluation_period_builds = expected_integer(
    screening_row,
    "EvaluationPeriodBuilds",
)

raw_training_rows = expected_integer(
    screening_row,
    "TrainingExecutionRows",
)

raw_evaluation_rows = expected_integer(
    screening_row,
    "EvaluationExecutionRows",
)

model_training_rows = expected_integer(
    screening_row,
    "ModelTrainingRows",
)

model_evaluation_rows = expected_integer(
    screening_row,
    "ModelEvaluationRows",
)

model_training_builds = expected_integer(
    screening_row,
    "ModelTrainingBuilds",
)

model_evaluation_builds = expected_integer(
    screening_row,
    "ModelEvaluationBuilds",
)


# ------------------------------------------------------------
# 15. BUILD PROJECT 8 EXPERIMENT PROTOCOL
# ------------------------------------------------------------

generated_at_utc = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


experiment_protocol = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "FROZEN_PROJECT_8_EXPERIMENT_PROTOCOL",

    "FrozenAtUTC":
        generated_at_utc,

    "CanonicalSourceLineage": {
        "ExperimentProtocolSource": {
            "Path":
                str(
                    CANONICAL_PROTOCOL_SOURCE
                ),

            "SHA256":
                canonical_source_hashes[
                    str(
                        CANONICAL_PROTOCOL_SOURCE
                    )
                ],
        },

        "ModelConfigurationSource": {
            "Path":
                str(
                    CANONICAL_MODEL_SOURCE
                ),

            "SHA256":
                canonical_source_hashes[
                    str(
                        CANONICAL_MODEL_SOURCE
                    )
                ],
        },

        "APFDFormulaAudit": {
            "Path":
                str(
                    APFD_AUDIT_SOURCE
                ),

            "SHA256":
                canonical_source_hashes[
                    str(
                        APFD_AUDIT_SOURCE
                    )
                ],
        },

        "LatestCompletedProjectCheckpoint": {
            "Path":
                str(
                    PROJECT7_CHECKPOINT_SOURCE
                ),

            "SHA256":
                canonical_source_hashes[
                    str(
                        PROJECT7_CHECKPOINT_SOURCE
                    )
                ],
        },
    },

    "Chronology": {
        "PrimaryOrder":
            "started_at ascending",

        "EqualTimestampTieBreak":
            "build ID descending",

        "EqualTimestampGroups":
            equal_timestamp_groups,

        "EqualTimestampBuilds":
            equal_timestamp_builds,

        "FrozenChronologyMatchesCanonicalRule":
            chronology_order_matches,
    },

    "Split": {
        "Type":
            "chronological fixed holdout",

        "TrainingFraction":
            0.75,

        "EvaluationFraction":
            0.25,

        "TotalBuilds":
            total_builds,

        "TrainingPeriodBuilds":
            training_period_builds,

        "EvaluationPeriodBuilds":
            evaluation_period_builds,

        "RawTrainingRows":
            raw_training_rows,

        "RawEvaluationRows":
            raw_evaluation_rows,

        "ModelTrainingRows":
            model_training_rows,

        "ModelEvaluationRows":
            model_evaluation_rows,

        "ModelTrainingBuilds":
            model_training_builds,

        "ScoredEvaluationBuilds":
            model_evaluation_builds,

        "RollingRetraining":
            False,
    },

    "NoiseLevelsPercent":
        NOISE_LEVELS_PERCENT,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "ExpectedConditions":
        EXPECTED_CONDITIONS,

    "ExpectedMLFits":
        EXPECTED_MODEL_FITS,

    "NoisePartition":
        "raw training execution history only",

    "EvaluationPartition":
        "clean and immutable",

    "NoiseUnit":
        "individual raw training execution verdict",

    "NoiseMask": {
        "Type":
            "independent row uniforms",

        "NestedAcrossNoiseLevels":
            True,

        "SameProjectSeedStream":
            True,

        "Randomisation":
            (
                "project-specific deterministic random "
                "streams; one common uniform stream for "
                "all noise levels within each repetition seed"
            ),
    },

    "FlipRule": {
        "PassToFailure":
            (
                "replace verdict 0 with a failure subtype "
                "sampled from the clean Project 8 training "
                "failure-subtype distribution"
            ),

        "FailureToPass":
            "replace every non-zero selected verdict with 0",

        "CleanTrainingFailureSubtypeCounts":
            failure_subtype_counts,

        "CleanTrainingFailureSubtypeProbabilities":
            failure_subtype_probabilities,
    },

    "RECPolicy": {
        "RecentExecutionWindow":
            RECENT_WINDOW,

        "VerdictDependentFeatures":
            VERDICT_DEPENDENT_REC,

        "VerdictDependentFeatureCount":
            len(
                VERDICT_DEPENDENT_REC
            ),

        "VerdictIndependentFeatures":
            VERDICT_INDEPENDENT_REC,

        "VerdictIndependentFeatureCount":
            len(
                VERDICT_INDEPENDENT_REC
            ),

        "Reconstruction":
            "clean-anchored delta reconstruction",

        "NoiseConditionRule":
            (
                "recompute only verdict-dependent REC "
                "features from corrupted training history"
            ),

        "NoiseIndependentRule":
            (
                "preserve the original TCP-CI values for "
                "the six verdict-independent REC features"
            ),

        "HistoryRule":
            "use only executions preceding the current execution",

        "ZeroPercentCleanRECValidation":
            step6_status.get(
                "total_rec_mismatches"
            )
            == 0,
    },

    "FixedInstanceDesign": {
        "PreserveModelReadyTrainingRows":
            True,

        "ReplaceTrainingLabels":
            True,

        "PreserveNonRECPredictors":
            True,

        "PreserveEvaluationRows":
            True,

        "EvaluationVerdictsRemainClean":
            True,
    },

    "Techniques": {
        "MachineLearning":
            ML_TECHNIQUES,

        "Baselines":
            BASELINES,
    },

    "CorruptedHistoryConsumers": [
        "RandomForest",
        "XGBoost",
        "LightGBM",
        "NaiveBayes",
        "LatestFail",
    ],

    "BaselinePolicy": {
        "Random":
            (
                "project-seed-specific deterministic ranking; "
                "constant across noise levels for the same seed"
            ),

        "LatestFail":
            "derived from the same corrupted training history",

        "QTF-Avg":
            "noise-independent and constant across conditions",
    },

    "ModelTraining": {
        "FitsPerCondition":
            len(
                ML_TECHNIQUES
            ),

        "OneModelPerProjectNoiseSeedTechnique":
            True,

        "RollingRetraining":
            False,

        "EvaluationPolicy":
            "one fitted model over the fixed final 25 percent",
    },

    "Metrics": {
        "Primary":
            "APFDc",

        "Secondary":
            "APFD",

        "FailureMapping":
            (
                "each failing evaluation test is treated "
                "as one fault-detection event"
            ),

        "EvaluationVerdicts":
            "clean",

        "FormulaAuditPassed": {
            "APFD":
                apfd_audit_passed,

            "APFDc":
                apfdc_audit_passed,
        },
    },

    "RankingTieRule":
        "score first, Test ascending",

    "IncompleteCommitMappingResolution": {
        "Step5Status":
            step5_status.get(
                "status"
            ),

        "UnmatchedCommitTokens":
            step5_status.get(
                "unmatched_commit_tokens"
            ),

        "Treatment":
            (
                "retain builds in execution history with "
                "empty changed-entity sets where no commit "
                "can be matched"
            ),

        "CleanRECValidation":
            "all 19 REC features matched dataset.csv",

        "AffectedEvaluationModelRows":
            step6_report.get(
                "incomplete_commit_mapping_impact",
                {},
            ).get(
                "affected_evaluation_model_rows"
            ),
    },
}


# ------------------------------------------------------------
# 16. BUILD PROJECT 8 MODEL CONFIGURATION
# ------------------------------------------------------------

model_configuration = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "FROZEN",

    "FrozenAtUTC":
        generated_at_utc,

    "OriginalPredictorCount":
        len(
            predictor_columns
        ),

    "ZeroVariancePredictorCount":
        len(
            zero_variance_predictors
        ),

    "ZeroVariancePredictors":
        zero_variance_predictors,

    "ActivePredictorCount":
        len(
            active_predictors
        ),

    "ActivePredictors":
        active_predictors,

    "FeatureSelectionRule":
        (
            "identify zero-variance predictors using only "
            "the clean training partition and freeze the "
            "result across all Project 8 conditions"
        ),

    "MissingValueRule":
        (
            "replace infinities with missing; learn column "
            "medians from the current noisy training "
            "condition; apply those medians to training and "
            "clean evaluation matrices"
        ),

    "InfiniteValueRule":
        (
            "replace positive and negative infinity with "
            "missing before median imputation"
        ),

    "LabelRule":
        "binary failure indicator: Verdict != 0",

    "RankingTieRule":
        "score first, Test ascending",

    "EvaluationPolicy":
        "one fitted model over the fixed final 25 percent",

    "RandomStatePolicy":
        (
            "use the repetition seed as the deterministic "
            "model random state for stochastic ML techniques"
        ),

    "Models":
        copy.deepcopy(
            canonical_models
        ),

    "LibraryVersions":
        library_versions,

    "CanonicalModelSource": {
        "Path":
            str(
                CANONICAL_MODEL_SOURCE
            ),

        "SHA256":
            canonical_source_hashes[
                str(
                    CANONICAL_MODEL_SOURCE
                )
            ],
    },

    "DataProfile": {
        "CleanTrainingRows":
            training_rows,

        "CleanEvaluationRows":
            evaluation_rows,

        "NumericConversionFailures":
            numeric_conversion_failures,

        "PositiveInfiniteValues":
            positive_infinities,

        "NegativeInfiniteValues":
            negative_infinities,
    },
}


# ------------------------------------------------------------
# 17. FREEZE AUDIT
# ------------------------------------------------------------

audit_records = [
    {
        "Check":
            "Projects 1–7 remain frozen",

        "Expected":
            list(
                range(1, 8)
            ),

        "Actual":
            frozen_project_numbers,

        "Pass":
            frozen_project_numbers
            == list(
                range(1, 8)
            ),
    },

    {
        "Check":
            "Project 8 absent from completion registry",

        "Expected":
            0,

        "Actual":
            len(
                existing_project8_rows
            ),

        "Pass":
            len(
                existing_project8_rows
            )
            == 0,
    },

    {
        "Check":
            "Step 5 accepted",

        "Expected":
            True,

        "Actual":
            step5_status.get(
                "status"
            )
            in accepted_step5_statuses,

        "Pass":
            step5_status.get(
                "status"
            )
            in accepted_step5_statuses,
    },

    {
        "Check":
            "Step 6 passed",

        "Expected":
            expected_step6_status,

        "Actual":
            step6_status.get(
                "status"
            ),

        "Pass":
            step6_status.get(
                "status"
            )
            == expected_step6_status,
    },

    {
        "Check":
            "Canonical source hash mismatches",

        "Expected":
            0,

        "Actual":
            len(
                source_hash_mismatches
            ),

        "Pass":
            len(
                source_hash_mismatches
            )
            == 0,
    },

    {
        "Check":
            "Chronology matches canonical tie rule",

        "Expected":
            True,

        "Actual":
            chronology_order_matches,

        "Pass":
            chronology_order_matches,
    },

    {
        "Check":
            "Total builds",

        "Expected":
            254,

        "Actual":
            total_builds,

        "Pass":
            total_builds == 254,
    },

    {
        "Check":
            "Training builds",

        "Expected":
            190,

        "Actual":
            training_period_builds,

        "Pass":
            training_period_builds == 190,
    },

    {
        "Check":
            "Evaluation builds",

        "Expected":
            64,

        "Actual":
            evaluation_period_builds,

        "Pass":
            evaluation_period_builds == 64,
    },

    {
        "Check":
            "Model training rows",

        "Expected":
            7580,

        "Actual":
            training_rows,

        "Pass":
            training_rows == 7580,
    },

    {
        "Check":
            "Model evaluation rows",

        "Expected":
            2300,

        "Actual":
            evaluation_rows,

        "Pass":
            evaluation_rows == 2300,
    },

    {
        "Check":
            "Original predictor count",

        "Expected":
            150,

        "Actual":
            len(
                predictor_columns
            ),

        "Pass":
            len(
                predictor_columns
            )
            == 150,
    },

    {
        "Check":
            "Predictor numeric conversion failures",

        "Expected":
            0,

        "Actual":
            numeric_conversion_failures,

        "Pass":
            numeric_conversion_failures
            == 0,
    },

    {
        "Check":
            "Active predictor count valid",

        "Expected":
            "> 0",

        "Actual":
            len(
                active_predictors
            ),

        "Pass":
            len(
                active_predictors
            )
            > 0,
    },

    {
        "Check":
            "Failure subtype observations available",

        "Expected":
            "> 0",

        "Actual":
            total_training_failure_executions,

        "Pass":
            total_training_failure_executions
            > 0,
    },

    {
        "Check":
            "Failure subtype probability sum",

        "Expected":
            1.0,

        "Actual":
            failure_probability_sum,

        "Pass":
            bool(
                np.isclose(
                    failure_probability_sum,
                    1.0,
                    rtol=0,
                    atol=1e-12,
                )
            ),
    },

    {
        "Check":
            "Noise levels",

        "Expected":
            9,

        "Actual":
            len(
                NOISE_LEVELS_PERCENT
            ),

        "Pass":
            len(
                NOISE_LEVELS_PERCENT
            )
            == 9,
    },

    {
        "Check":
            "Repetition seeds",

        "Expected":
            30,

        "Actual":
            len(
                REPETITION_SEEDS
            ),

        "Pass":
            len(
                REPETITION_SEEDS
            )
            == 30,
    },

    {
        "Check":
            "Experiment conditions",

        "Expected":
            270,

        "Actual":
            EXPECTED_CONDITIONS,

        "Pass":
            EXPECTED_CONDITIONS
            == 270,
    },

    {
        "Check":
            "Expected ML fits",

        "Expected":
            1080,

        "Actual":
            EXPECTED_MODEL_FITS,

        "Pass":
            EXPECTED_MODEL_FITS
            == 1080,
    },

    {
        "Check":
            "ML model definitions",

        "Expected":
            sorted(
                ML_TECHNIQUES
            ),

        "Actual":
            sorted(
                canonical_models.keys()
            ),

        "Pass":
            set(
                canonical_models.keys()
            )
            == set(
                ML_TECHNIQUES
            ),
    },

    {
        "Check":
            "APFD formula audit",

        "Expected":
            True,

        "Actual":
            apfd_audit_passed,

        "Pass":
            apfd_audit_passed,
    },

    {
        "Check":
            "APFDc formula audit",

        "Expected":
            True,

        "Actual":
            apfdc_audit_passed,

        "Pass":
            apfdc_audit_passed,
    },

    {
        "Check":
            "Missing required ML libraries",

        "Expected":
            0,

        "Actual":
            len(
                missing_required_libraries
            ),

        "Pass":
            len(
                missing_required_libraries
            )
            == 0,
    },

    {
        "Check":
            "Evaluation rows affected by incomplete commit mapping",

        "Expected":
            0,

        "Actual":
            step6_report.get(
                "incomplete_commit_mapping_impact",
                {},
            ).get(
                "affected_evaluation_model_rows"
            ),

        "Pass":
            step6_report.get(
                "incomplete_commit_mapping_impact",
                {},
            ).get(
                "affected_evaluation_model_rows"
            )
            == 0,
    },
]


audit_frame = pd.DataFrame(
    audit_records
)


print("\nStep 7B freeze audit:")

display(
    audit_frame
)


failed_audit_checks = (
    audit_frame[
        ~audit_frame[
            "Pass"
        ]
    ]
    .copy()
)


if not failed_audit_checks.empty:

    print(
        "\nFailed Step 7B checks:"
    )

    display(
        failed_audit_checks
    )

    raise RuntimeError(
        "PROJECT 8 STEP 7B DID NOT PASS.\n"
        "No protocol or model-configuration files "
        "were frozen."
    )


# ------------------------------------------------------------
# 18. WRITE FROZEN ARTEFACTS
# ------------------------------------------------------------

PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

NOTES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_json(
    PROTOCOL_PATH,
    experiment_protocol,
)

atomic_write_json(
    MODEL_CONFIGURATION_PATH,
    model_configuration,
)

atomic_write_csv(
    PREDICTOR_PROFILE_PATH,
    predictor_profile,
)

atomic_write_csv(
    PROTOCOL_AUDIT_PATH,
    audit_frame,
)


# Create byte-identical Notes copies.

shutil.copyfile(
    PROTOCOL_PATH,
    NOTES_PROTOCOL_PATH,
)

shutil.copyfile(
    MODEL_CONFIGURATION_PATH,
    NOTES_MODEL_CONFIGURATION_PATH,
)


protocol_sha256 = calculate_hash(
    PROTOCOL_PATH
)

model_configuration_sha256 = calculate_hash(
    MODEL_CONFIGURATION_PATH
)

predictor_profile_sha256 = calculate_hash(
    PREDICTOR_PROFILE_PATH
)

protocol_audit_sha256 = calculate_hash(
    PROTOCOL_AUDIT_PATH
)


notes_protocol_sha256 = calculate_hash(
    NOTES_PROTOCOL_PATH
)

notes_model_configuration_sha256 = calculate_hash(
    NOTES_MODEL_CONFIGURATION_PATH
)


if (
    protocol_sha256
    != notes_protocol_sha256
):
    raise AssertionError(
        "The Notes protocol copy is not byte-identical."
    )


if (
    model_configuration_sha256
    != notes_model_configuration_sha256
):
    raise AssertionError(
        "The Notes model-configuration copy is not "
        "byte-identical."
    )


# ------------------------------------------------------------
# 19. WRITE STEP 7B REPORT AND STATUS
# ------------------------------------------------------------

step7b_status_text = (
    "PASS_PROJECT_8_PROTOCOL_AND_MODEL_CONFIGURATION_FROZEN"
)


step7b_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "RuntimeRestorationPerformed":
        runtime_restore_performed,

    "RuntimeProjectSource":
        str(
            PROJECT_SOURCE_DIR
        ),

    "CanonicalSources": {
        str(path):
            canonical_source_hashes[
                str(path)
            ]
        for path in canonical_source_paths
    },

    "Chronology": {
        "Rule":
            (
                "started_at ascending; build ID "
                "descending for equal timestamps"
            ),

        "OrderMatches":
            chronology_order_matches,

        "EqualTimestampGroups":
            equal_timestamp_groups,

        "EqualTimestampBuilds":
            equal_timestamp_builds,
    },

    "Predictors": {
        "OriginalPredictorCount":
            len(
                predictor_columns
            ),

        "ZeroVariancePredictorCount":
            len(
                zero_variance_predictors
            ),

        "ZeroVariancePredictors":
            zero_variance_predictors,

        "ActivePredictorCount":
            len(
                active_predictors
            ),

        "NumericConversionFailures":
            numeric_conversion_failures,

        "PositiveInfiniteValues":
            positive_infinities,

        "NegativeInfiniteValues":
            negative_infinities,
    },

    "FailureSubtypeDistribution": {
        "Counts":
            failure_subtype_counts,

        "Probabilities":
            failure_subtype_probabilities,

        "TotalFailureExecutions":
            total_training_failure_executions,
    },

    "ExperimentDimensions": {
        "NoiseLevels":
            len(
                NOISE_LEVELS_PERCENT
            ),

        "Seeds":
            len(
                REPETITION_SEEDS
            ),

        "Conditions":
            EXPECTED_CONDITIONS,

        "MLTechniques":
            len(
                ML_TECHNIQUES
            ),

        "ExpectedMLFits":
            EXPECTED_MODEL_FITS,

        "Baselines":
            len(
                BASELINES
            ),
    },

    "LibraryVersions":
        library_versions,

    "AuditChecks":
        int(
            len(
                audit_frame
            )
        ),

    "FailedAuditChecks":
        int(
            len(
                failed_audit_checks
            )
        ),

    "Outputs": {
        "ExperimentProtocol": {
            "Path":
                str(
                    PROTOCOL_PATH
                ),

            "SHA256":
                protocol_sha256,
        },

        "ModelConfiguration": {
            "Path":
                str(
                    MODEL_CONFIGURATION_PATH
                ),

            "SHA256":
                model_configuration_sha256,
        },

        "PredictorProfile": {
            "Path":
                str(
                    PREDICTOR_PROFILE_PATH
                ),

            "SHA256":
                predictor_profile_sha256,
        },

        "FreezeAudit": {
            "Path":
                str(
                    PROTOCOL_AUDIT_PATH
                ),

            "SHA256":
                protocol_audit_sha256,
        },

        "NotesProtocolCopy": {
            "Path":
                str(
                    NOTES_PROTOCOL_PATH
                ),

            "SHA256":
                notes_protocol_sha256,
        },

        "NotesModelConfigurationCopy": {
            "Path":
                str(
                    NOTES_MODEL_CONFIGURATION_PATH
                ),

            "SHA256":
                notes_model_configuration_sha256,
        },
    },

    "CompletionRegistryModified":
        False,

    "Projects1To7Modified":
        False,

    "Status":
        step7b_status_text,
}


atomic_write_json(
    STEP7B_REPORT_PATH,
    step7b_report,
)


step7b_status = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        step7b_status_text,

    "ProtocolSHA256":
        protocol_sha256,

    "ModelConfigurationSHA256":
        model_configuration_sha256,

    "OriginalPredictorCount":
        len(
            predictor_columns
        ),

    "ZeroVariancePredictorCount":
        len(
            zero_variance_predictors
        ),

    "ActivePredictorCount":
        len(
            active_predictors
        ),

    "ExpectedConditions":
        EXPECTED_CONDITIONS,

    "ExpectedMLFits":
        EXPECTED_MODEL_FITS,

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP7B_STATUS_PATH,
    step7b_status,
)


# ------------------------------------------------------------
# 20. VERIFY OUTPUTS
# ------------------------------------------------------------

expected_outputs = [
    PROTOCOL_PATH,
    MODEL_CONFIGURATION_PATH,
    PREDICTOR_PROFILE_PATH,
    PROTOCOL_AUDIT_PATH,
    STEP7B_REPORT_PATH,
    STEP7B_STATUS_PATH,
    NOTES_PROTOCOL_PATH,
    NOTES_MODEL_CONFIGURATION_PATH,
]


missing_outputs = [
    str(path)
    for path in expected_outputs
    if not path.exists()
]


if missing_outputs:
    raise RuntimeError(
        "Step 7B failed to create all expected outputs:\n"
        + "\n".join(
            missing_outputs
        )
    )


# ------------------------------------------------------------
# 21. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 84)
print("=== PROJECT 8 STEP 7B RESULT ===")
print("=" * 84)

print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nRuntime restoration:")

print(
    "Extraction restored:",
    runtime_restore_performed,
)

print(
    "Source directory:",
    PROJECT_SOURCE_DIR,
)


print("\nChronology protocol:")

print(
    "Primary ordering:",
    "started_at ascending",
)

print(
    "Equal timestamp tie-break:",
    "build ID descending",
)

print(
    "Equal timestamp groups:",
    equal_timestamp_groups,
)

print(
    "Frozen chronology matches:",
    chronology_order_matches,
)


print("\nExperiment dimensions:")

print(
    "Noise levels:",
    len(
        NOISE_LEVELS_PERCENT
    ),
)

print(
    "Repetition seeds:",
    len(
        REPETITION_SEEDS
    ),
)

print(
    "Conditions:",
    EXPECTED_CONDITIONS,
)

print(
    "ML techniques:",
    len(
        ML_TECHNIQUES
    ),
)

print(
    "Expected ML fits:",
    EXPECTED_MODEL_FITS,
)

print(
    "Baselines:",
    len(
        BASELINES
    ),
)


print("\nProject-specific predictors:")

print(
    "Original predictors:",
    len(
        predictor_columns
    ),
)

print(
    "Zero-variance predictors:",
    len(
        zero_variance_predictors
    ),
)

print(
    "Active predictors:",
    len(
        active_predictors
    ),
)

print(
    "Numeric conversion failures:",
    numeric_conversion_failures,
)


if zero_variance_predictors:

    print(
        "Excluded zero-variance predictors:"
    )

    for predictor in zero_variance_predictors:
        print(
            " -",
            predictor,
        )


print("\nTraining failure subtypes:")

print(
    "Counts:",
    failure_subtype_counts,
)

print(
    "Probabilities:",
    failure_subtype_probabilities,
)


print("\nModel libraries:")

for package_name, version in (
    library_versions.items()
):
    print(
        f"{package_name}: {version}"
    )


print("\nFrozen outputs:")

for output_path in expected_outputs:
    print(
        output_path
    )


print("\nProtocol SHA-256:")

print(
    protocol_sha256
)


print("\nModel configuration SHA-256:")

print(
    model_configuration_sha256
)


print("\nAudit:")

print(
    "Checks:",
    len(
        audit_frame
    ),
)

print(
    "Failed checks:",
    len(
        failed_audit_checks
    ),
)


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–7 modified:")
print(0)


print(
    "\nSTATUS:",
    step7b_status_text,
)

print("=" * 84)

=== PROJECT 8 STEP 7B: PROTOCOL AND MODEL-CONFIGURATION FREEZE ===

Temporary runtime extraction is missing.
Restoring from the existing Drive archive...
No network download is being performed.
Archive size: 226.12 MB
Available runtime disk: 87.72 GB


FileNotFoundError: Required source files are still missing after runtime restoration:
/content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo/builds.csv
/content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo/exe.csv
/content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo/dataset.csv
/content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo/id_map.csv
/content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo/entity_change_history.csv

In [ ]:
# ============================================================
# PROJECT 8 — RUNTIME EXTRACTION PATH REPAIR
#
# Purpose:
# - extract the existing TCP-CI archive into temporary storage
# - dynamically locate optimatika@ojAlgo
# - restore the canonical runtime path expected by Step 7B
#
# Permanent Google Drive writes: 0
# Projects 1–7 modified: 0
# Project 8 saved artefacts modified: 0
# ============================================================

from pathlib import Path
import hashlib
import os
import shutil
import tarfile


# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

ARCHIVE_PATH = (
    THESIS_ROOT
    / "Data"
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)

STAGING_ROOT = Path(
    "/content/tcp_ci_runtime_restore_stage"
)

EXPECTED_DATASET_ROOT = Path(
    "/content/TCP-CI-main-dataset"
)

EXPECTED_PROJECT_DIR = (
    EXPECTED_DATASET_ROOT
    / "datasets"
    / "optimatika@ojAlgo"
)

PROJECT_DIRECTORY_NAME = (
    "optimatika@ojAlgo"
)

REQUIRED_PROJECT_FILES = [
    "builds.csv",
    "exe.csv",
    "dataset.csv",
    "id_map.csv",
    "entity_change_history.csv",
]

EXPECTED_ARCHIVE_MD5 = (
    "728804085c757ff5357aa165b4b6384f"
)


print("=" * 78)
print("=== PROJECT 8 RUNTIME EXTRACTION PATH REPAIR ===")
print("=" * 78)


# ------------------------------------------------------------
# 2. HELPERS
# ------------------------------------------------------------

def calculate_md5(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.md5()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def contains_required_project_files(
    directory,
):
    directory = Path(
        directory
    )

    return all(
        (
            directory
            / file_name
        ).is_file()
        for file_name
        in REQUIRED_PROJECT_FILES
    )


def validate_archive_members(
    archive,
    extraction_root,
):
    """
    Reject path traversal and archive links before extraction.
    """

    extraction_root = Path(
        extraction_root
    ).resolve()

    for member in archive.getmembers():

        member_target = (
            extraction_root
            / member.name
        ).resolve()

        try:
            common_path = Path(
                os.path.commonpath(
                    [
                        str(extraction_root),
                        str(member_target),
                    ]
                )
            )

        except ValueError:
            raise RuntimeError(
                "Archive member resolves outside the "
                "temporary extraction directory:\n"
                f"{member.name}"
            )

        if common_path != extraction_root:
            raise RuntimeError(
                "Unsafe archive path detected:\n"
                f"{member.name}"
            )

        if member.issym() or member.islnk():
            raise RuntimeError(
                "Archive contains a symbolic or hard link:\n"
                f"{member.name}"
            )


# ------------------------------------------------------------
# 3. VALIDATE DRIVE ARCHIVE
# ------------------------------------------------------------

if not THESIS_ROOT.exists():
    raise FileNotFoundError(
        "Google Drive does not appear to be mounted:\n"
        f"{THESIS_ROOT}"
    )


if not ARCHIVE_PATH.exists():
    raise FileNotFoundError(
        "The existing TCP-CI archive was not found:\n"
        f"{ARCHIVE_PATH}"
    )


archive_size_mb = (
    ARCHIVE_PATH.stat().st_size
    / (1024 ** 2)
)


print("\nArchive:")
print(ARCHIVE_PATH)

print(
    "Archive size:",
    round(
        archive_size_mb,
        2,
    ),
    "MB",
)


print("\nVerifying archive MD5...")

actual_archive_md5 = calculate_md5(
    ARCHIVE_PATH
)


print(
    "Expected MD5:",
    EXPECTED_ARCHIVE_MD5,
)

print(
    "Actual MD5:  ",
    actual_archive_md5,
)


if (
    actual_archive_md5
    != EXPECTED_ARCHIVE_MD5
):
    raise RuntimeError(
        "Archive MD5 validation failed. "
        "No extraction was performed."
    )


print("Archive integrity: PASS")


# ------------------------------------------------------------
# 4. REMOVE ONLY TEMPORARY LOCAL EXTRACTIONS
# ------------------------------------------------------------

for temporary_path in [
    STAGING_ROOT,
    EXPECTED_DATASET_ROOT,
]:

    if temporary_path.exists():

        print(
            "\nRemoving incomplete temporary path:"
        )

        print(
            temporary_path
        )

        if temporary_path.is_symlink():
            temporary_path.unlink()

        elif temporary_path.is_dir():
            shutil.rmtree(
                temporary_path
            )

        else:
            temporary_path.unlink()


STAGING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 5. EXTRACT INTO STAGING DIRECTORY
# ------------------------------------------------------------

available_disk_gb = (
    shutil.disk_usage(
        "/content"
    ).free
    / (1024 ** 3)
)


print(
    "\nAvailable runtime disk:",
    round(
        available_disk_gb,
        2,
    ),
    "GB",
)


if available_disk_gb < 3:
    raise RuntimeError(
        "Insufficient temporary disk space. "
        "At least 3 GB is required."
    )


print("\nExtracting existing Drive archive...")
print("No internet download is being performed.")


with tarfile.open(
    ARCHIVE_PATH,
    mode="r:gz",
) as archive:

    archive_members = archive.getmembers()

    print(
        "Archive members:",
        len(
            archive_members
        ),
    )

    validate_archive_members(
        archive=archive,
        extraction_root=
            STAGING_ROOT,
    )

    try:
        archive.extractall(
            path=STAGING_ROOT,
            members=archive_members,
            filter="fully_trusted",
        )

    except TypeError:
        archive.extractall(
            path=STAGING_ROOT,
            members=archive_members,
        )


print("Extraction completed.")


# ------------------------------------------------------------
# 6. DISCOVER THE REAL PROJECT DIRECTORY
# ------------------------------------------------------------

project_candidates = []


for candidate in STAGING_ROOT.rglob(
    PROJECT_DIRECTORY_NAME
):

    if (
        candidate.is_dir()
        and contains_required_project_files(
            candidate
        )
    ):
        project_candidates.append(
            candidate.resolve()
        )


project_candidates = sorted(
    set(
        project_candidates
    )
)


print(
    "\nValid ojAlgo project directories discovered:",
    len(
        project_candidates
    ),
)


for candidate in project_candidates:
    print(
        candidate
    )


if len(project_candidates) == 0:

    print(
        "\nTop-level extracted entries:"
    )

    for entry in sorted(
        STAGING_ROOT.iterdir()
    ):
        print(
            entry
        )

    discovered_build_files = list(
        STAGING_ROOT.rglob(
            "builds.csv"
        )
    )

    print(
        "\nAll builds.csv files discovered:",
        len(
            discovered_build_files
        ),
    )

    for path in discovered_build_files[:40]:
        print(
            path
        )

    raise FileNotFoundError(
        "The archive was extracted, but no complete "
        "optimatika@ojAlgo directory was discovered."
    )


if len(project_candidates) > 1:
    raise RuntimeError(
        "Multiple complete optimatika@ojAlgo directories "
        "were discovered. No directory was selected "
        "automatically."
    )


discovered_project_dir = (
    project_candidates[0]
)


if (
    discovered_project_dir.parent.name
    != "datasets"
):
    raise RuntimeError(
        "The discovered project directory is not located "
        "inside a datasets directory:\n"
        f"{discovered_project_dir}"
    )


discovered_dataset_root = (
    discovered_project_dir
    .parent
    .parent
)


print("\nDiscovered dataset root:")
print(
    discovered_dataset_root
)


# ------------------------------------------------------------
# 7. MOVE DISCOVERED ROOT TO CANONICAL RUNTIME PATH
# ------------------------------------------------------------

EXPECTED_DATASET_ROOT.parent.mkdir(
    parents=True,
    exist_ok=True,
)


shutil.move(
    str(
        discovered_dataset_root
    ),
    str(
        EXPECTED_DATASET_ROOT
    ),
)


# Remove remaining staging content, if any.

if STAGING_ROOT.exists():

    if STAGING_ROOT.is_dir():
        shutil.rmtree(
            STAGING_ROOT
        )

    else:
        STAGING_ROOT.unlink()


# ------------------------------------------------------------
# 8. VERIFY CANONICAL PROJECT PATH
# ------------------------------------------------------------

missing_project_files = [
    str(
        EXPECTED_PROJECT_DIR
        / file_name
    )
    for file_name
    in REQUIRED_PROJECT_FILES
    if not (
        EXPECTED_PROJECT_DIR
        / file_name
    ).is_file()
]


if missing_project_files:
    raise FileNotFoundError(
        "The canonical runtime path was created, but "
        "required files are missing:\n"
        + "\n".join(
            missing_project_files
        )
    )


print("\nCanonical runtime project directory:")
print(
    EXPECTED_PROJECT_DIR
)


print("\nRequired source files:")

for file_name in REQUIRED_PROJECT_FILES:

    file_path = (
        EXPECTED_PROJECT_DIR
        / file_name
    )

    print(
        f"{file_name}: "
        f"{file_path.stat().st_size:,} bytes"
    )


# ------------------------------------------------------------
# 9. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 78)
print("=== PROJECT 8 RUNTIME REPAIR RESULT ===")
print("=" * 78)

print(
    "\nArchive downloaded:",
    False,
)

print(
    "Archive MD5 verified:",
    True,
)

print(
    "Project source restored:",
    True,
)

print(
    "Canonical source directory:",
    EXPECTED_PROJECT_DIR,
)

print(
    "Required files present:",
    len(
        REQUIRED_PROJECT_FILES
    ),
)

print(
    "Missing required files:",
    0,
)

print(
    "\nPermanent Google Drive writes:",
    0,
)

print(
    "Completion registry modified:",
    0,
)

print(
    "Projects 1–7 modified:",
    0,
)

print(
    "\nSTATUS: "
    "PASS_PROJECT_8_RUNTIME_EXTRACTION_PATH_REPAIRED"
)

print("=" * 78)

=== PROJECT 8 RUNTIME EXTRACTION PATH REPAIR ===

Archive:
/content/drive/MyDrive/Thesis_Experiment/Data/Raw/TCP-CI-main-dataset.tar.gz
Archive size: 226.12 MB

Verifying archive MD5...
Expected MD5: 728804085c757ff5357aa165b4b6384f
Actual MD5:   728804085c757ff5357aa165b4b6384f
Archive integrity: PASS

Available runtime disk: 85.48 GB

Extracting existing Drive archive...
No internet download is being performed.
Archive members: 176
Extraction completed.

Valid ojAlgo project directories discovered: 1
/content/tcp_ci_runtime_restore_stage/datasets/optimatika@ojAlgo

Discovered dataset root:
/content/tcp_ci_runtime_restore_stage

Canonical runtime project directory:
/content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo

Required source files:
builds.csv: 23,885 bytes
exe.csv: 1,008,139 bytes
dataset.csv: 9,288,816 bytes
id_map.csv: 133,800 bytes
entity_change_history.csv: 2,340,002 bytes


=== PROJECT 8 RUNTIME REPAIR RESULT ===

Archive downloaded: False
Archive MD5 verified: True
P

In [ ]:
# ============================================================
# PROJECT 8 — STEP 7B
# CANONICAL EXPERIMENT CHRONOLOGY,
# PROTOCOL AND MODEL-CONFIGURATION FREEZE
#
# PROJECT: optimatika@ojAlgo
#
# IMPORTANT CHRONOLOGY HANDLING
# -----------------------------
# The Step 3 chronology reproduced the original clean
# dataset.csv REC features exactly. It is retained as the
# clean-source validation chronology.
#
# The frozen cross-project experiment protocol requires:
#   started_at ascending
#   build ID descending within equal timestamps
#
# This cell creates a separate canonical experiment chronology
# with that rule and verifies that:
# - the build set is identical
# - the 75/25 partition is identical
# - every order difference is confined to training
# - evaluation membership is unchanged
#
# Noise experiments must use the canonical experiment
# chronology together with clean-anchored REC deltas.
#
# This cell does NOT:
# - run experimental conditions
# - fit ML models
# - modify source CSV files
# - modify Projects 1–7
# - modify the completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from IPython.display import display

import copy
import hashlib
import json
import platform
import re
import shutil

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. PROJECT CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 8
PROJECT_NAME = "optimatika@ojAlgo"
PROJECT_SLUG = "optimatika__ojAlgo"
PROJECT_SHORT_NAME = "ojalgo"

TRAINING_FRACTION = 0.75

NOISE_LEVELS_PERCENT = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

REPETITION_SEEDS = list(
    range(1, 31)
)

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

RECENT_WINDOW = 6

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

EXPECTED_CONDITIONS = (
    len(NOISE_LEVELS_PERCENT)
    * len(REPETITION_SEEDS)
)

EXPECTED_MODEL_FITS = (
    EXPECTED_CONDITIONS
    * len(ML_TECHNIQUES)
)


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

PROJECT_SOURCE_DIR = Path(
    "/content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo"
)

BUILDS_PATH = (
    PROJECT_SOURCE_DIR
    / "builds.csv"
)

EXE_PATH = (
    PROJECT_SOURCE_DIR
    / "exe.csv"
)

DATASET_PATH = (
    PROJECT_SOURCE_DIR
    / "dataset.csv"
)

ID_MAP_PATH = (
    PROJECT_SOURCE_DIR
    / "id_map.csv"
)

ENTITY_HISTORY_PATH = (
    PROJECT_SOURCE_DIR
    / "entity_change_history.csv"
)

PROJECT_AGGREGATED_DIR = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / PROJECT_SLUG
)

PREFLIGHT_DIR = (
    PROJECT_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_preflight"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

SCREENING_SNAPSHOT_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_screening_snapshot.csv"
)

# Existing chronology used for clean source reconstruction.
SOURCE_VALIDATION_CHRONOLOGY_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_build_chronology.csv"
)

CANONICAL_RAW_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_canonical_raw_executions.csv.gz"
)

STEP5_STATUS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step5_status.json"
)

STEP6_STATUS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step6_status.json"
)

STEP6_REPORT_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_clean_rec_validation_report.json"
)


# Canonical cross-project protocol sources.

CANONICAL_PROTOCOL_SOURCE = (
    NOTES_DIR
    / "project_06_experiment_protocol.json"
)

CANONICAL_MODEL_SOURCE = (
    NOTES_DIR
    / "project_06_jetty_model_configuration.json"
)

APFD_AUDIT_SOURCE = (
    NOTES_DIR
    / "apfd_apfdc_formula_audit_v1.json"
)

PROJECT7_CHECKPOINT_SOURCE = (
    NOTES_DIR
    / "project_07_selection_checkpoint.json"
)


EXPECTED_SOURCE_HASHES = {
    str(CANONICAL_PROTOCOL_SOURCE):
        "94c6697aefebe077b76badc8274a920e56170d8b1bc695768d1ae34d0e490065",

    str(CANONICAL_MODEL_SOURCE):
        "e353643100ca851f911112e2e6186b0645335d4e63102b0837a10d7feb19e6dd",

    str(APFD_AUDIT_SOURCE):
        "fd50825c1ee9df0ca304154e397409bc5712504e313533e59507d83d58fdd44f",

    str(PROJECT7_CHECKPOINT_SOURCE):
        "7077773f779577a667666f8c9c5796059e0bfe4b865ffdd8c3eac260f2ebc156",
}


# Step 7B outputs.

EXPERIMENT_CHRONOLOGY_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_experiment_build_chronology.csv"
)

CHRONOLOGY_DIFFERENCE_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_source_experiment_chronology_difference.csv"
)

PROTOCOL_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_experiment_protocol.json"
)

MODEL_CONFIGURATION_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_model_configuration.json"
)

PREDICTOR_PROFILE_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_predictor_profile.csv"
)

PROTOCOL_AUDIT_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_protocol_freeze_audit.csv"
)

STEP7B_REPORT_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step7b_protocol_freeze_report.json"
)

STEP7B_STATUS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step7b_status.json"
)


# Permanent Notes copies.

NOTES_PROTOCOL_PATH = (
    NOTES_DIR
    / "project_08_experiment_protocol.json"
)

NOTES_MODEL_CONFIGURATION_PATH = (
    NOTES_DIR
    / "project_08_ojalgo_model_configuration.json"
)


print("=" * 88)
print("=== PROJECT 8 STEP 7B: CANONICAL PROTOCOL AND CONFIGURATION FREEZE ===")
print("=" * 88)


# ------------------------------------------------------------
# 3. HELPERS
# ------------------------------------------------------------

def calculate_hash(
    path,
    algorithm="sha256",
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.new(
        algorithm
    )

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def json_safe(value):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return value


def canonical_id(value):
    if pd.isna(value):
        return None

    text = str(value).strip()

    if text == "":
        return None

    if re.fullmatch(
        r"[-+]?\d+\.0+",
        text,
    ):
        return text.split(".")[0]

    return text


def expected_integer(
    row,
    column,
):
    if column not in row.index:
        return None

    value = row[column]

    if pd.isna(value):
        return None

    text = str(value).strip()

    if text == "":
        return None

    return int(
        float(text)
    )


def package_version(
    distribution_name,
):
    try:
        return metadata.version(
            distribution_name
        )
    except metadata.PackageNotFoundError:
        return None


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(
        path
    )


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(
        path
    )


# ------------------------------------------------------------
# 4. VALIDATE RUNTIME SOURCE
# ------------------------------------------------------------

if not THESIS_ROOT.exists():
    raise FileNotFoundError(
        "Google Drive is not available at:\n"
        f"{THESIS_ROOT}"
    )


required_source_files = [
    BUILDS_PATH,
    EXE_PATH,
    DATASET_PATH,
    ID_MAP_PATH,
    ENTITY_HISTORY_PATH,
]


missing_source_files = [
    str(path)
    for path in required_source_files
    if not path.exists()
]


if missing_source_files:
    raise FileNotFoundError(
        "The repaired Project 8 runtime source is missing.\n"
        "Rerun the runtime extraction repair cell.\n\n"
        + "\n".join(
            missing_source_files
        )
    )


print("\nRuntime source validation:")

print(
    "Source directory:",
    PROJECT_SOURCE_DIR,
)

print(
    "Required source files:",
    len(required_source_files),
)

print(
    "Missing source files:",
    len(missing_source_files),
)


# ------------------------------------------------------------
# 5. VALIDATE PREVIOUS PROJECT 8 STEPS
# ------------------------------------------------------------

required_previous_outputs = [
    SCREENING_SNAPSHOT_PATH,
    SOURCE_VALIDATION_CHRONOLOGY_PATH,
    CANONICAL_RAW_PATH,
    STEP5_STATUS_PATH,
    STEP6_STATUS_PATH,
    STEP6_REPORT_PATH,
]


missing_previous_outputs = [
    str(path)
    for path in required_previous_outputs
    if not path.exists()
]


if missing_previous_outputs:
    raise FileNotFoundError(
        "Required Project 8 preflight outputs are missing:\n"
        + "\n".join(
            missing_previous_outputs
        )
    )


step5_status = json.loads(
    STEP5_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step6_status = json.loads(
    STEP6_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

step6_report = json.loads(
    STEP6_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)


accepted_step5_statuses = {
    "PASS_PENDING_CLEAN_REC_VALIDATION",
    (
        "PASS_PENDING_CLEAN_REC_VALIDATION_"
        "WITH_DOCUMENTED_UNMATCHED_COMMITS"
    ),
}


if (
    step5_status.get("status")
    not in accepted_step5_statuses
):
    raise AssertionError(
        "Step 5 is not in an accepted state:\n"
        f"{step5_status.get('status')}"
    )


expected_step6_status = (
    "PASS_PROJECT_8_CLEAN_REC_RECONSTRUCTION_VALIDATED"
)


if (
    step6_status.get("status")
    != expected_step6_status
):
    raise AssertionError(
        "Step 6 has not passed:\n"
        f"{step6_status.get('status')}"
    )


if (
    int(
        step6_status.get(
            "total_rec_mismatches",
            -1,
        )
    )
    != 0
):
    raise AssertionError(
        "Step 6 does not report zero clean REC mismatches."
    )


# ------------------------------------------------------------
# 6. VALIDATE COMPLETION REGISTRY
# ------------------------------------------------------------

registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
)


required_registry_columns = {
    "ProjectNumber",
    "Project",
    "ProjectSlug",
    "Status",
}


missing_registry_columns = (
    required_registry_columns
    - set(registry.columns)
)


if missing_registry_columns:
    raise RuntimeError(
        "Completion registry is missing columns:\n"
        f"{sorted(missing_registry_columns)}"
    )


registry["_ProjectNumber"] = pd.to_numeric(
    registry["ProjectNumber"],
    errors="coerce",
)


registry["_Status"] = (
    registry["Status"]
    .astype(str)
    .str.strip()
    .str.upper()
)


frozen_registry = registry[
    registry["_Status"].eq(
        "COMPLETE_AND_FROZEN"
    )
].copy()


frozen_project_numbers = sorted(
    frozen_registry["_ProjectNumber"]
    .dropna()
    .astype(int)
    .tolist()
)


existing_project8_rows = registry[
    registry["_ProjectNumber"].eq(
        PROJECT_NUMBER
    )
]


# ------------------------------------------------------------
# 7. VERIFY CANONICAL PROTOCOL SOURCES
# ------------------------------------------------------------

canonical_source_paths = [
    CANONICAL_PROTOCOL_SOURCE,
    CANONICAL_MODEL_SOURCE,
    APFD_AUDIT_SOURCE,
    PROJECT7_CHECKPOINT_SOURCE,
]


missing_canonical_sources = [
    str(path)
    for path in canonical_source_paths
    if not path.exists()
]


if missing_canonical_sources:
    raise FileNotFoundError(
        "Canonical protocol sources are missing:\n"
        + "\n".join(
            missing_canonical_sources
        )
    )


canonical_source_hashes = {
    str(path): calculate_hash(path)
    for path in canonical_source_paths
}


source_hash_mismatches = []


for (
    path_text,
    expected_hash,
) in EXPECTED_SOURCE_HASHES.items():

    actual_hash = canonical_source_hashes[
        path_text
    ]

    if actual_hash != expected_hash:
        source_hash_mismatches.append({
            "Path":
                path_text,

            "ExpectedSHA256":
                expected_hash,

            "ActualSHA256":
                actual_hash,
        })


if source_hash_mismatches:

    display(
        pd.DataFrame(
            source_hash_mismatches
        )
    )

    raise RuntimeError(
        "A canonical protocol source changed after Step 7A."
    )


canonical_protocol_source = json.loads(
    CANONICAL_PROTOCOL_SOURCE.read_text(
        encoding="utf-8"
    )
)

canonical_model_source = json.loads(
    CANONICAL_MODEL_SOURCE.read_text(
        encoding="utf-8"
    )
)

apfd_audit = json.loads(
    APFD_AUDIT_SOURCE.read_text(
        encoding="utf-8"
    )
)


canonical_models = (
    canonical_model_source.get("Models")
    or canonical_model_source.get("models")
)


if not isinstance(
    canonical_models,
    dict,
):
    raise RuntimeError(
        "The canonical model source has no model dictionary."
    )


if set(
    canonical_models.keys()
) != set(
    ML_TECHNIQUES
):
    raise AssertionError(
        "The canonical models differ from the frozen "
        "four-model protocol.\n"
        f"Detected: {sorted(canonical_models.keys())}"
    )


canonical_protocol_chronology = (
    canonical_protocol_source.get(
        "chronology",
        {}
    )
)


canonical_primary_order = (
    canonical_protocol_chronology.get(
        "primary_order"
    )
)


canonical_tie_break = (
    canonical_protocol_chronology.get(
        "equal_timestamp_tie_break"
    )
)


if canonical_primary_order != "started_at ascending":
    raise AssertionError(
        "Unexpected canonical primary chronology rule:\n"
        f"{canonical_primary_order}"
    )


if canonical_tie_break != "build ID descending":
    raise AssertionError(
        "Unexpected canonical equal-timestamp rule:\n"
        f"{canonical_tie_break}"
    )


print("\nCanonical source validation:")

print(
    "Sources checked:",
    len(canonical_source_paths),
)

print(
    "Hash mismatches:",
    len(source_hash_mismatches),
)

print(
    "Canonical primary order:",
    canonical_primary_order,
)

print(
    "Canonical equal-timestamp rule:",
    canonical_tie_break,
)


# ------------------------------------------------------------
# 8. LOAD PROJECT 8 DATA
# ------------------------------------------------------------

builds = pd.read_csv(
    BUILDS_PATH,
    low_memory=False,
)

dataset = pd.read_csv(
    DATASET_PATH,
    low_memory=False,
)

canonical_raw = pd.read_csv(
    CANONICAL_RAW_PATH,
    low_memory=False,
)

source_chronology = pd.read_csv(
    SOURCE_VALIDATION_CHRONOLOGY_PATH,
    dtype=str,
)

screening_snapshot = pd.read_csv(
    SCREENING_SNAPSHOT_PATH,
    dtype=str,
)


if len(screening_snapshot) != 1:
    raise AssertionError(
        "Expected exactly one screening row."
    )


screening_row = (
    screening_snapshot.iloc[0]
)


print("\nProject 8 data:")

print(
    "builds.csv:",
    builds.shape,
)

print(
    "dataset.csv:",
    dataset.shape,
)

print(
    "canonical raw executions:",
    canonical_raw.shape,
)

print(
    "source-validation chronology:",
    source_chronology.shape,
)


# ------------------------------------------------------------
# 9. BUILD CANONICAL EXPERIMENT CHRONOLOGY
# ------------------------------------------------------------

required_build_columns = {
    "id",
    "started_at",
    "commits",
}


missing_build_columns = (
    required_build_columns
    - set(builds.columns)
)


if missing_build_columns:
    raise RuntimeError(
        "builds.csv is missing chronology columns:\n"
        f"{sorted(missing_build_columns)}"
    )


required_source_chronology_columns = {
    "CanonicalBuild",
    "BuildOrder",
    "Partition",
}


missing_source_chronology_columns = (
    required_source_chronology_columns
    - set(source_chronology.columns)
)


if missing_source_chronology_columns:
    raise RuntimeError(
        "The source-validation chronology is missing columns:\n"
        f"{sorted(missing_source_chronology_columns)}"
    )


builds_working = builds.copy()


builds_working["_BuildKey"] = (
    builds_working["id"]
    .map(canonical_id)
)


builds_working["_BuildNumeric"] = (
    pd.to_numeric(
        builds_working["_BuildKey"],
        errors="raise",
    )
)


builds_working["_Timestamp"] = (
    pd.to_datetime(
        builds_working["started_at"],
        errors="coerce",
        utc=True,
    )
)


missing_timestamps = int(
    builds_working["_Timestamp"]
    .isna()
    .sum()
)


duplicate_build_ids = int(
    builds_working["_BuildKey"]
    .duplicated()
    .sum()
)


if missing_timestamps != 0:
    raise AssertionError(
        "builds.csv contains missing timestamps."
    )


if duplicate_build_ids != 0:
    raise AssertionError(
        "builds.csv contains duplicate build IDs."
    )


source_chronology_working = (
    source_chronology
    .assign(
        _BuildOrderNumeric=pd.to_numeric(
            source_chronology["BuildOrder"],
            errors="raise",
        )
    )
    .sort_values(
        "_BuildOrderNumeric",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


source_order = (
    source_chronology_working[
        "CanonicalBuild"
    ]
    .map(canonical_id)
    .astype(str)
    .tolist()
)


source_build_set = set(
    source_order
)


source_position_lookup = {
    build_id: position
    for position, build_id
    in enumerate(
        source_order,
        start=1,
    )
}


# Canonical cross-project experiment order:
# timestamp ascending, build ID descending within ties.

experiment_chronology = (
    builds_working
    .sort_values(
        [
            "_Timestamp",
            "_BuildNumeric",
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
    .copy()
)


experiment_chronology[
    "BuildOrder"
] = np.arange(
    1,
    len(experiment_chronology) + 1,
)


expected_total_builds = expected_integer(
    screening_row,
    "TotalBuilds",
)

expected_training_builds = expected_integer(
    screening_row,
    "TrainingPeriodBuilds",
)

expected_evaluation_builds = expected_integer(
    screening_row,
    "EvaluationPeriodBuilds",
)


if expected_total_builds != len(
    experiment_chronology
):
    raise AssertionError(
        "Canonical experiment build count differs from "
        "the screening snapshot."
    )


if (
    expected_training_builds
    + expected_evaluation_builds
    != expected_total_builds
):
    raise AssertionError(
        "The frozen split counts do not sum to TotalBuilds."
    )


experiment_chronology[
    "Partition"
] = np.where(
    experiment_chronology["BuildOrder"]
    <= expected_training_builds,
    "training",
    "evaluation",
)


experiment_order = (
    experiment_chronology["_BuildKey"]
    .astype(str)
    .tolist()
)


experiment_build_set = set(
    experiment_order
)


build_sets_identical = bool(
    source_build_set
    == experiment_build_set
)


if not build_sets_identical:
    raise AssertionError(
        "The source and experiment chronologies contain "
        "different build sets."
    )


# ------------------------------------------------------------
# 10. COMPARE SOURCE AND EXPERIMENT ORDERS
# ------------------------------------------------------------

order_comparison = pd.DataFrame({
    "Position":
        np.arange(
            1,
            expected_total_builds + 1,
        ),

    "SourceValidationBuild":
        source_order,

    "CanonicalExperimentBuild":
        experiment_order,
})


order_comparison[
    "Matches"
] = order_comparison[
    "SourceValidationBuild"
].eq(
    order_comparison[
        "CanonicalExperimentBuild"
    ]
)


chronology_differences = (
    order_comparison[
        ~order_comparison["Matches"]
    ]
    .copy()
)


chronology_mismatch_positions = int(
    len(
        chronology_differences
    )
)


chronology_mismatch_builds = sorted(
    set(
        chronology_differences[
            "SourceValidationBuild"
        ].tolist()
    )
    |
    set(
        chronology_differences[
            "CanonicalExperimentBuild"
        ].tolist()
    )
)


source_partition_lookup = dict(
    zip(
        source_chronology_working[
            "CanonicalBuild"
        ].map(canonical_id),

        source_chronology_working[
            "Partition"
        ].astype(str),
    )
)


experiment_partition_lookup = dict(
    zip(
        experiment_chronology[
            "_BuildKey"
        ].astype(str),

        experiment_chronology[
            "Partition"
        ].astype(str),
    )
)


partition_comparison = pd.DataFrame({
    "CanonicalBuild":
        sorted(
            experiment_build_set,
            key=lambda value: int(value),
        )
})


partition_comparison[
    "SourcePartition"
] = partition_comparison[
    "CanonicalBuild"
].map(
    source_partition_lookup
)


partition_comparison[
    "ExperimentPartition"
] = partition_comparison[
    "CanonicalBuild"
].map(
    experiment_partition_lookup
)


partition_comparison[
    "Matches"
] = partition_comparison[
    "SourcePartition"
].eq(
    partition_comparison[
        "ExperimentPartition"
    ]
)


partition_assignment_mismatches = int(
    (
        ~partition_comparison["Matches"]
    ).sum()
)


mismatch_evaluation_builds = sorted([
    build_id
    for build_id in chronology_mismatch_builds
    if experiment_partition_lookup[
        build_id
    ] == "evaluation"
])


mismatch_training_builds = sorted([
    build_id
    for build_id in chronology_mismatch_builds
    if experiment_partition_lookup[
        build_id
    ] == "training"
])


mismatch_positions_in_evaluation = int(
    chronology_differences[
        "Position"
    ]
    .gt(
        expected_training_builds
    )
    .sum()
)


# ------------------------------------------------------------
# 11. ANALYSE EQUAL-TIMESTAMP GROUPS
# ------------------------------------------------------------

timestamp_group_sizes = (
    builds_working
    .groupby("_Timestamp")
    .size()
)


equal_timestamp_groups = int(
    timestamp_group_sizes.gt(1).sum()
)


equal_timestamp_builds = int(
    timestamp_group_sizes.loc[
        timestamp_group_sizes.gt(1)
    ].sum()
)


source_ascending_tie_groups = 0
source_descending_tie_groups = 0
source_other_tie_groups = 0

tie_group_records = []


for timestamp, group in builds_working.groupby(
    "_Timestamp",
    sort=True,
):

    if len(group) <= 1:
        continue

    group_build_ids = (
        group["_BuildKey"]
        .astype(str)
        .tolist()
    )

    ascending_order = sorted(
        group_build_ids,
        key=lambda value: int(value),
    )

    descending_order = sorted(
        group_build_ids,
        key=lambda value: int(value),
        reverse=True,
    )

    source_group_order = sorted(
        group_build_ids,
        key=lambda value: source_position_lookup[value],
    )

    experiment_group_order = sorted(
        group_build_ids,
        key=lambda value: experiment_order.index(value),
    )

    if source_group_order == ascending_order:
        source_rule = "build ID ascending"
        source_ascending_tie_groups += 1

    elif source_group_order == descending_order:
        source_rule = "build ID descending"
        source_descending_tie_groups += 1

    else:
        source_rule = "other"
        source_other_tie_groups += 1

    tie_group_records.append({
        "TimestampUTC":
            timestamp.isoformat(),

        "BuildCount":
            len(group_build_ids),

        "AscendingOrder":
            "#".join(
                ascending_order
            ),

        "DescendingOrder":
            "#".join(
                descending_order
            ),

        "SourceValidationOrder":
            "#".join(
                source_group_order
            ),

        "CanonicalExperimentOrder":
            "#".join(
                experiment_group_order
            ),

        "DetectedSourceRule":
            source_rule,

        "ExperimentUsesDescending":
            experiment_group_order
            == descending_order,
    })


tie_group_frame = pd.DataFrame(
    tie_group_records
)


canonical_descending_tie_groups = int(
    tie_group_frame[
        "ExperimentUsesDescending"
    ].sum()
) if not tie_group_frame.empty else 0


print("\nChronology comparison:")

print(
    "Source-validation order mismatch positions:",
    chronology_mismatch_positions,
)

print(
    "Distinct builds involved:",
    len(
        chronology_mismatch_builds
    ),
)

print(
    "Partition-assignment mismatches:",
    partition_assignment_mismatches,
)

print(
    "Mismatch positions in evaluation:",
    mismatch_positions_in_evaluation,
)

print(
    "Mismatch evaluation builds:",
    len(
        mismatch_evaluation_builds
    ),
)


if not chronology_differences.empty:

    display(
        chronology_differences
    )


print("\nEqual-timestamp groups:")

display(
    tie_group_frame
)


# ------------------------------------------------------------
# 12. CREATE CANONICAL EXPERIMENT CHRONOLOGY OUTPUT
# ------------------------------------------------------------

experiment_chronology_output = pd.DataFrame({
    "CanonicalBuild":
        experiment_chronology[
            "_BuildKey"
        ].astype(str),

    "BuildOrder":
        experiment_chronology[
            "BuildOrder"
        ].astype(int),

    "Partition":
        experiment_chronology[
            "Partition"
        ].astype(str),

    "CanonicalTimestampUTC":
        experiment_chronology[
            "_Timestamp"
        ].map(
            lambda value: value.isoformat()
        ),

    "OriginalBuildId":
        experiment_chronology["id"],

    "OriginalCommitField":
        experiment_chronology["commits"],

    "ChronologyPurpose":
        "canonical_noise_experiment",

    "PrimaryOrderRule":
        "started_at ascending",

    "EqualTimestampTieBreak":
        "build ID descending",
})


# ------------------------------------------------------------
# 13. DETERMINE PROJECT-SPECIFIC PREDICTORS
# ------------------------------------------------------------

required_dataset_metadata = {
    "Build",
    "Test",
    "Verdict",
    "Duration",
}


missing_dataset_metadata = (
    required_dataset_metadata
    - set(dataset.columns)
)


if missing_dataset_metadata:
    raise RuntimeError(
        "dataset.csv is missing metadata columns:\n"
        f"{sorted(missing_dataset_metadata)}"
    )


predictor_columns = [
    column
    for column in dataset.columns
    if column not in required_dataset_metadata
]


dataset_build_keys = (
    dataset["Build"]
    .map(canonical_id)
)


training_builds = {
    build_id
    for build_id, partition
    in experiment_partition_lookup.items()
    if partition == "training"
}


evaluation_builds = {
    build_id
    for build_id, partition
    in experiment_partition_lookup.items()
    if partition == "evaluation"
}


training_row_mask = (
    dataset_build_keys.isin(
        training_builds
    )
)


evaluation_row_mask = (
    dataset_build_keys.isin(
        evaluation_builds
    )
)


training_rows = int(
    training_row_mask.sum()
)


evaluation_rows = int(
    evaluation_row_mask.sum()
)


numeric_predictors = (
    dataset[predictor_columns]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
)


original_non_null = (
    dataset[predictor_columns]
    .notna()
)


numeric_conversion_failures = int(
    (
        original_non_null
        & numeric_predictors.isna()
    )
    .sum()
    .sum()
)


predictor_array = numeric_predictors.to_numpy(
    dtype=float
)


positive_infinities = int(
    np.isposinf(
        predictor_array
    ).sum()
)


negative_infinities = int(
    np.isneginf(
        predictor_array
    ).sum()
)


numeric_predictors_clean = (
    numeric_predictors
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
)


training_predictors = (
    numeric_predictors_clean.loc[
        training_row_mask
    ]
)


# Match the established runner:
# missing is treated as an observed state for variation.

training_unique_counts = (
    training_predictors
    .nunique(
        dropna=False
    )
)


zero_variance_predictors = (
    training_unique_counts[
        training_unique_counts <= 1
    ]
    .index
    .tolist()
)


active_predictors = [
    column
    for column in predictor_columns
    if column not in zero_variance_predictors
]


predictor_profile_records = []


for predictor in predictor_columns:

    predictor_profile_records.append({
        "Predictor":
            predictor,

        "TrainingUniqueValuesIncludingMissing":
            int(
                training_unique_counts[
                    predictor
                ]
            ),

        "TrainingMissingValues":
            int(
                training_predictors[
                    predictor
                ]
                .isna()
                .sum()
            ),

        "EvaluationMissingValues":
            int(
                numeric_predictors_clean.loc[
                    evaluation_row_mask,
                    predictor,
                ]
                .isna()
                .sum()
            ),

        "ZeroVarianceInCleanTraining":
            predictor
            in zero_variance_predictors,

        "ActivePredictor":
            predictor
            in active_predictors,
    })


predictor_profile = pd.DataFrame(
    predictor_profile_records
)


# ------------------------------------------------------------
# 14. TRAINING FAILURE-SUBTYPE DISTRIBUTION
# ------------------------------------------------------------

required_raw_columns = {
    "CanonicalBuild",
    "RawVerdictToken",
}


missing_raw_columns = (
    required_raw_columns
    - set(canonical_raw.columns)
)


if missing_raw_columns:
    raise RuntimeError(
        "Canonical raw executions are missing columns:\n"
        f"{sorted(missing_raw_columns)}"
    )


canonical_raw_working = (
    canonical_raw.copy()
)


canonical_raw_working[
    "_BuildKey"
] = canonical_raw_working[
    "CanonicalBuild"
].map(
    canonical_id
)


canonical_raw_working[
    "_ExperimentPartition"
] = canonical_raw_working[
    "_BuildKey"
].map(
    experiment_partition_lookup
)


training_raw = canonical_raw_working[
    canonical_raw_working[
        "_ExperimentPartition"
    ].eq("training")
].copy()


training_raw["_Verdict"] = (
    pd.to_numeric(
        training_raw[
            "RawVerdictToken"
        ],
        errors="raise",
    )
    .astype(int)
)


training_failure_subtypes = (
    training_raw.loc[
        training_raw["_Verdict"].ne(0),
        "_Verdict",
    ]
)


failure_subtype_counts = {
    str(int(subtype)): int(count)
    for subtype, count
    in training_failure_subtypes
    .value_counts()
    .sort_index()
    .items()
}


total_training_failure_executions = int(
    len(training_failure_subtypes)
)


failure_subtype_probabilities = {
    subtype: (
        count
        / total_training_failure_executions
    )
    for subtype, count
    in failure_subtype_counts.items()
}


failure_probability_sum = float(
    sum(
        failure_subtype_probabilities.values()
    )
)


# ------------------------------------------------------------
# 15. LIBRARY AND METRIC AUDITS
# ------------------------------------------------------------

library_versions = {
    "python":
        platform.python_version(),

    "numpy":
        package_version("numpy"),

    "pandas":
        package_version("pandas"),

    "scikit-learn":
        package_version("scikit-learn"),

    "xgboost":
        package_version("xgboost"),

    "lightgbm":
        package_version("lightgbm"),
}


missing_required_libraries = [
    package_name
    for package_name in [
        "scikit-learn",
        "xgboost",
        "lightgbm",
    ]
    if library_versions[
        package_name
    ] is None
]


apfd_audit_passed = bool(
    apfd_audit.get(
        "AllAPFDComparisonsPassed"
    )
)


apfdc_audit_passed = bool(
    apfd_audit.get(
        "AllAPFDcComparisonsPassed"
    )
)


# ------------------------------------------------------------
# 16. FROZEN PROJECT COUNTS
# ------------------------------------------------------------

total_builds = expected_integer(
    screening_row,
    "TotalBuilds",
)

training_period_builds = expected_integer(
    screening_row,
    "TrainingPeriodBuilds",
)

evaluation_period_builds = expected_integer(
    screening_row,
    "EvaluationPeriodBuilds",
)

raw_training_rows = expected_integer(
    screening_row,
    "TrainingExecutionRows",
)

raw_evaluation_rows = expected_integer(
    screening_row,
    "EvaluationExecutionRows",
)

model_training_rows = expected_integer(
    screening_row,
    "ModelTrainingRows",
)

model_evaluation_rows = expected_integer(
    screening_row,
    "ModelEvaluationRows",
)

model_training_builds = expected_integer(
    screening_row,
    "ModelTrainingBuilds",
)

model_evaluation_builds = expected_integer(
    screening_row,
    "ModelEvaluationBuilds",
)


# ------------------------------------------------------------
# 17. BUILD FROZEN PROJECT 8 PROTOCOL
# ------------------------------------------------------------

generated_at_utc = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


experiment_protocol = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "FROZEN_PROJECT_8_EXPERIMENT_PROTOCOL",

    "FrozenAtUTC":
        generated_at_utc,

    "CanonicalSourceLineage": {
        "ExperimentProtocolSource": {
            "Path":
                str(
                    CANONICAL_PROTOCOL_SOURCE
                ),

            "SHA256":
                canonical_source_hashes[
                    str(
                        CANONICAL_PROTOCOL_SOURCE
                    )
                ],
        },

        "ModelConfigurationSource": {
            "Path":
                str(
                    CANONICAL_MODEL_SOURCE
                ),

            "SHA256":
                canonical_source_hashes[
                    str(
                        CANONICAL_MODEL_SOURCE
                    )
                ],
        },

        "APFDFormulaAudit": {
            "Path":
                str(
                    APFD_AUDIT_SOURCE
                ),

            "SHA256":
                canonical_source_hashes[
                    str(
                        APFD_AUDIT_SOURCE
                    )
                ],
        },

        "Project7Checkpoint": {
            "Path":
                str(
                    PROJECT7_CHECKPOINT_SOURCE
                ),

            "SHA256":
                canonical_source_hashes[
                    str(
                        PROJECT7_CHECKPOINT_SOURCE
                    )
                ],
        },
    },

    "Chronology": {
        "CleanSourceValidationChronology": {
            "Path":
                str(
                    SOURCE_VALIDATION_CHRONOLOGY_PATH
                ),

            "Purpose":
                (
                    "reproduce and validate original "
                    "clean dataset.csv REC values"
                ),

            "CleanRECFeatureMismatches":
                int(
                    step6_status.get(
                        "total_rec_mismatches",
                        -1,
                    )
                ),
        },

        "CanonicalExperimentChronology": {
            "Path":
                str(
                    EXPERIMENT_CHRONOLOGY_PATH
                ),

            "PrimaryOrder":
                "started_at ascending",

            "EqualTimestampTieBreak":
                "build ID descending",

            "Purpose":
                (
                    "noise injection, noisy REC "
                    "reconstruction and experiment execution"
                ),
        },

        "EqualTimestampGroups":
            equal_timestamp_groups,

        "EqualTimestampBuilds":
            equal_timestamp_builds,

        "SourceAscendingTieGroups":
            source_ascending_tie_groups,

        "SourceDescendingTieGroups":
            source_descending_tie_groups,

        "CanonicalDescendingTieGroups":
            canonical_descending_tie_groups,

        "SourceVsExperimentMismatchPositions":
            chronology_mismatch_positions,

        "DistinctMismatchBuilds":
            len(
                chronology_mismatch_builds
            ),

        "MismatchTrainingBuilds":
            mismatch_training_builds,

        "MismatchEvaluationBuilds":
            mismatch_evaluation_builds,

        "PartitionAssignmentMismatches":
            partition_assignment_mismatches,

        "EvaluationMismatchPositions":
            mismatch_positions_in_evaluation,

        "BuildSetsIdentical":
            build_sets_identical,
    },

    "Split": {
        "Type":
            "chronological fixed holdout",

        "TrainingFraction":
            0.75,

        "EvaluationFraction":
            0.25,

        "TotalBuilds":
            total_builds,

        "TrainingPeriodBuilds":
            training_period_builds,

        "EvaluationPeriodBuilds":
            evaluation_period_builds,

        "RawTrainingRows":
            raw_training_rows,

        "RawEvaluationRows":
            raw_evaluation_rows,

        "ModelTrainingRows":
            model_training_rows,

        "ModelEvaluationRows":
            model_evaluation_rows,

        "ModelTrainingBuilds":
            model_training_builds,

        "ScoredEvaluationBuilds":
            model_evaluation_builds,

        "PartitionMembershipChangedByTieRepair":
            False,

        "RollingRetraining":
            False,
    },

    "NoiseLevelsPercent":
        NOISE_LEVELS_PERCENT,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "ExpectedConditions":
        EXPECTED_CONDITIONS,

    "ExpectedMLFits":
        EXPECTED_MODEL_FITS,

    "NoisePartition":
        "raw training execution history only",

    "EvaluationPartition":
        "clean and immutable",

    "NoiseUnit":
        "individual raw training execution verdict",

    "NoiseMask": {
        "Type":
            "independent row uniforms",

        "NestedAcrossNoiseLevels":
            True,

        "SameProjectSeedStream":
            True,

        "Randomisation":
            (
                "component-specific deterministic "
                "stable_project_seed streams"
            ),
    },

    "FlipRule": {
        "PassToFailure":
            (
                "replace verdict 0 with a failure subtype "
                "sampled from the clean Project 8 training "
                "failure-subtype distribution"
            ),

        "FailureToPass":
            (
                "replace every selected non-zero verdict "
                "with verdict 0"
            ),

        "CleanTrainingFailureSubtypeCounts":
            failure_subtype_counts,

        "CleanTrainingFailureSubtypeProbabilities":
            failure_subtype_probabilities,
    },

    "RECPolicy": {
        "RecentExecutionWindow":
            RECENT_WINDOW,

        "VerdictDependentFeatures":
            VERDICT_DEPENDENT_REC,

        "VerdictDependentFeatureCount":
            len(
                VERDICT_DEPENDENT_REC
            ),

        "VerdictIndependentFeatures":
            VERDICT_INDEPENDENT_REC,

        "VerdictIndependentFeatureCount":
            len(
                VERDICT_INDEPENDENT_REC
            ),

        "HistoryRule":
            (
                "only executions preceding the current "
                "execution may contribute"
            ),

        "Reconstruction":
            "clean-anchored canonical delta",

        "CleanAnchorFormula":
            (
                "final noisy feature = original clean "
                "dataset feature + "
                "(canonical noisy reconstruction - "
                "canonical clean reconstruction)"
            ),

        "ZeroNoiseIdentity":
            (
                "at zero noise the canonical delta is zero, "
                "therefore the original clean dataset value "
                "is reproduced exactly"
            ),

        "NoiseConditionRule":
            (
                "recompute the 13 verdict-dependent REC "
                "features using the canonical experiment "
                "chronology"
            ),

        "NoiseIndependentRule":
            (
                "preserve the six original verdict-"
                "independent REC features"
            ),

        "Step6CleanSourceValidationPassed":
            True,

        "Step6TotalRECMismatches":
            int(
                step6_status.get(
                    "total_rec_mismatches",
                    -1,
                )
            ),
    },

    "FixedInstanceDesign": {
        "PreserveModelReadyTrainingRows":
            True,

        "ReplaceTrainingLabels":
            True,

        "PreserveNonRECPredictors":
            True,

        "PreserveEvaluationRows":
            True,

        "EvaluationVerdictsRemainClean":
            True,
    },

    "Techniques": {
        "MachineLearning":
            ML_TECHNIQUES,

        "Baselines":
            BASELINES,
    },

    "CorruptedHistoryConsumers": [
        "RandomForest",
        "XGBoost",
        "LightGBM",
        "NaiveBayes",
        "LatestFail",
    ],

    "BaselinePolicy": {
        "Random":
            (
                "project-seed and build-specific "
                "deterministic ranking; constant across "
                "noise levels for the same seed"
            ),

        "LatestFail":
            (
                "derived from the same corrupted "
                "training verdict history"
            ),

        "QTF-Avg":
            (
                "duration-based, noise-independent and "
                "constant across noise conditions"
            ),
    },

    "Metrics": {
        "Primary":
            "APFDc",

        "Secondary":
            "APFD",

        "EvaluationVerdicts":
            "clean",

        "RankingTieRule":
            "score first, Test ascending",

        "FormulaAuditPassed": {
            "APFD":
                apfd_audit_passed,

            "APFDc":
                apfdc_audit_passed,
        },
    },

    "IncompleteCommitMappingResolution": {
        "Step5Status":
            step5_status.get(
                "status"
            ),

        "UnmatchedCommitTokens":
            step5_status.get(
                "unmatched_commit_tokens"
            ),

        "Treatment":
            (
                "retain unmatched builds with empty "
                "changed-entity sets"
            ),

        "CleanRECValidationPassed":
            True,

        "AffectedEvaluationModelRows":
            step6_report.get(
                "incomplete_commit_mapping_impact",
                {},
            ).get(
                "affected_evaluation_model_rows"
            ),
    },
}


# ------------------------------------------------------------
# 18. BUILD MODEL CONFIGURATION
# ------------------------------------------------------------

model_configuration = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        "FROZEN",

    "FrozenAtUTC":
        generated_at_utc,

    "ExperimentChronologyPath":
        str(
            EXPERIMENT_CHRONOLOGY_PATH
        ),

    "OriginalPredictorCount":
        len(
            predictor_columns
        ),

    "ZeroVariancePredictorCount":
        len(
            zero_variance_predictors
        ),

    "ZeroVariancePredictors":
        zero_variance_predictors,

    "ActivePredictorCount":
        len(
            active_predictors
        ),

    "ActivePredictors":
        active_predictors,

    "FeatureSelectionRule":
        (
            "identify zero-variance predictors from the "
            "clean training partition only and freeze "
            "the resulting predictor set for all conditions"
        ),

    "MissingValueRule":
        (
            "replace infinities with missing; learn "
            "column medians from the current noisy training "
            "condition; apply those medians to training "
            "and clean evaluation matrices"
        ),

    "InfiniteValueRule":
        (
            "replace positive and negative infinity "
            "with missing before median imputation"
        ),

    "LabelRule":
        "binary failure indicator: Verdict != 0",

    "RankingTieRule":
        "score first, Test ascending",

    "EvaluationPolicy":
        (
            "one fitted model over the fixed final "
            "25 percent"
        ),

    "RandomStatePolicy":
        (
            "component-specific deterministic "
            "stable_project_seed(Project, RepetitionSeed, "
            "ComponentName)"
        ),

    "Models":
        copy.deepcopy(
            canonical_models
        ),

    "LibraryVersions":
        library_versions,

    "CanonicalModelSource": {
        "Path":
            str(
                CANONICAL_MODEL_SOURCE
            ),

        "SHA256":
            canonical_source_hashes[
                str(
                    CANONICAL_MODEL_SOURCE
                )
            ],
    },

    "DataProfile": {
        "CleanTrainingRows":
            training_rows,

        "CleanEvaluationRows":
            evaluation_rows,

        "NumericConversionFailures":
            numeric_conversion_failures,

        "PositiveInfiniteValues":
            positive_infinities,

        "NegativeInfiniteValues":
            negative_infinities,
    },
}


# ------------------------------------------------------------
# 19. STEP 7B AUDIT
# ------------------------------------------------------------

affected_evaluation_model_rows = (
    step6_report.get(
        "incomplete_commit_mapping_impact",
        {},
    ).get(
        "affected_evaluation_model_rows"
    )
)


audit_records = [
    {
        "Check":
            "Projects 1–7 remain frozen",

        "Expected":
            list(range(1, 8)),

        "Actual":
            frozen_project_numbers,

        "Pass":
            frozen_project_numbers
            == list(range(1, 8)),
    },

    {
        "Check":
            "Project 8 absent from completion registry",

        "Expected":
            0,

        "Actual":
            len(existing_project8_rows),

        "Pass":
            len(existing_project8_rows)
            == 0,
    },

    {
        "Check":
            "Restored source files",

        "Expected":
            5,

        "Actual":
            sum(
                path.exists()
                for path in required_source_files
            ),

        "Pass":
            all(
                path.exists()
                for path in required_source_files
            ),
    },

    {
        "Check":
            "Step 5 accepted",

        "Expected":
            True,

        "Actual":
            step5_status.get("status")
            in accepted_step5_statuses,

        "Pass":
            step5_status.get("status")
            in accepted_step5_statuses,
    },

    {
        "Check":
            "Step 6 clean REC validation passed",

        "Expected":
            expected_step6_status,

        "Actual":
            step6_status.get("status"),

        "Pass":
            step6_status.get("status")
            == expected_step6_status,
    },

    {
        "Check":
            "Step 6 REC mismatches",

        "Expected":
            0,

        "Actual":
            int(
                step6_status.get(
                    "total_rec_mismatches",
                    -1,
                )
            ),

        "Pass":
            int(
                step6_status.get(
                    "total_rec_mismatches",
                    -1,
                )
            )
            == 0,
    },

    {
        "Check":
            "Canonical source hash mismatches",

        "Expected":
            0,

        "Actual":
            len(source_hash_mismatches),

        "Pass":
            len(source_hash_mismatches)
            == 0,
    },

    {
        "Check":
            "Source and experiment build sets identical",

        "Expected":
            True,

        "Actual":
            build_sets_identical,

        "Pass":
            build_sets_identical,
    },

    {
        "Check":
            "Equal-timestamp groups",

        "Expected":
            3,

        "Actual":
            equal_timestamp_groups,

        "Pass":
            equal_timestamp_groups == 3,
    },

    {
        "Check":
            "Equal-timestamp builds",

        "Expected":
            6,

        "Actual":
            equal_timestamp_builds,

        "Pass":
            equal_timestamp_builds == 6,
    },

    {
        "Check":
            "Source ascending tie groups documented",

        "Expected":
            3,

        "Actual":
            source_ascending_tie_groups,

        "Pass":
            source_ascending_tie_groups == 3,
    },

    {
        "Check":
            "Canonical descending tie groups",

        "Expected":
            3,

        "Actual":
            canonical_descending_tie_groups,

        "Pass":
            canonical_descending_tie_groups == 3,
    },

    {
        "Check":
            "Chronology mismatch positions documented",

        "Expected":
            6,

        "Actual":
            chronology_mismatch_positions,

        "Pass":
            chronology_mismatch_positions == 6,
    },

    {
        "Check":
            "Distinct mismatch builds documented",

        "Expected":
            6,

        "Actual":
            len(chronology_mismatch_builds),

        "Pass":
            len(chronology_mismatch_builds)
            == 6,
    },

    {
        "Check":
            "Partition assignment mismatches",

        "Expected":
            0,

        "Actual":
            partition_assignment_mismatches,

        "Pass":
            partition_assignment_mismatches == 0,
    },

    {
        "Check":
            "Evaluation mismatch positions",

        "Expected":
            0,

        "Actual":
            mismatch_positions_in_evaluation,

        "Pass":
            mismatch_positions_in_evaluation == 0,
    },

    {
        "Check":
            "Evaluation builds affected by tie reordering",

        "Expected":
            0,

        "Actual":
            len(mismatch_evaluation_builds),

        "Pass":
            len(mismatch_evaluation_builds)
            == 0,
    },

    {
        "Check":
            "Total builds",

        "Expected":
            254,

        "Actual":
            total_builds,

        "Pass":
            total_builds == 254,
    },

    {
        "Check":
            "Training builds",

        "Expected":
            190,

        "Actual":
            training_period_builds,

        "Pass":
            training_period_builds == 190,
    },

    {
        "Check":
            "Evaluation builds",

        "Expected":
            64,

        "Actual":
            evaluation_period_builds,

        "Pass":
            evaluation_period_builds == 64,
    },

    {
        "Check":
            "Model training rows",

        "Expected":
            7580,

        "Actual":
            training_rows,

        "Pass":
            training_rows == 7580,
    },

    {
        "Check":
            "Model evaluation rows",

        "Expected":
            2300,

        "Actual":
            evaluation_rows,

        "Pass":
            evaluation_rows == 2300,
    },

    {
        "Check":
            "Original predictor count",

        "Expected":
            150,

        "Actual":
            len(predictor_columns),

        "Pass":
            len(predictor_columns) == 150,
    },

    {
        "Check":
            "Numeric predictor conversion failures",

        "Expected":
            0,

        "Actual":
            numeric_conversion_failures,

        "Pass":
            numeric_conversion_failures == 0,
    },

    {
        "Check":
            "Active predictors available",

        "Expected":
            "> 0",

        "Actual":
            len(active_predictors),

        "Pass":
            len(active_predictors) > 0,
    },

    {
        "Check":
            "Failure subtype observations available",

        "Expected":
            "> 0",

        "Actual":
            total_training_failure_executions,

        "Pass":
            total_training_failure_executions > 0,
    },

    {
        "Check":
            "Failure subtype probability sum",

        "Expected":
            1.0,

        "Actual":
            failure_probability_sum,

        "Pass":
            bool(
                np.isclose(
                    failure_probability_sum,
                    1.0,
                    rtol=0,
                    atol=1e-12,
                )
            ),
    },

    {
        "Check":
            "Noise levels",

        "Expected":
            9,

        "Actual":
            len(NOISE_LEVELS_PERCENT),

        "Pass":
            len(NOISE_LEVELS_PERCENT)
            == 9,
    },

    {
        "Check":
            "Repetition seeds",

        "Expected":
            30,

        "Actual":
            len(REPETITION_SEEDS),

        "Pass":
            len(REPETITION_SEEDS)
            == 30,
    },

    {
        "Check":
            "Experiment conditions",

        "Expected":
            270,

        "Actual":
            EXPECTED_CONDITIONS,

        "Pass":
            EXPECTED_CONDITIONS
            == 270,
    },

    {
        "Check":
            "Expected ML fits",

        "Expected":
            1080,

        "Actual":
            EXPECTED_MODEL_FITS,

        "Pass":
            EXPECTED_MODEL_FITS
            == 1080,
    },

    {
        "Check":
            "ML model definitions",

        "Expected":
            sorted(ML_TECHNIQUES),

        "Actual":
            sorted(
                canonical_models.keys()
            ),

        "Pass":
            set(canonical_models.keys())
            == set(ML_TECHNIQUES),
    },

    {
        "Check":
            "APFD formula audit",

        "Expected":
            True,

        "Actual":
            apfd_audit_passed,

        "Pass":
            apfd_audit_passed,
    },

    {
        "Check":
            "APFDc formula audit",

        "Expected":
            True,

        "Actual":
            apfdc_audit_passed,

        "Pass":
            apfdc_audit_passed,
    },

    {
        "Check":
            "Missing required ML libraries",

        "Expected":
            0,

        "Actual":
            len(
                missing_required_libraries
            ),

        "Pass":
            len(
                missing_required_libraries
            )
            == 0,
    },

    {
        "Check":
            (
                "Evaluation rows affected by incomplete "
                "commit mapping"
            ),

        "Expected":
            0,

        "Actual":
            affected_evaluation_model_rows,

        "Pass":
            affected_evaluation_model_rows == 0,
    },
]


audit_frame = pd.DataFrame(
    audit_records
)


print("\nStep 7B freeze audit:")

display(
    audit_frame
)


failed_audit_checks = (
    audit_frame[
        ~audit_frame["Pass"]
    ]
    .copy()
)


if not failed_audit_checks.empty:

    print(
        "\nFailed Step 7B checks:"
    )

    display(
        failed_audit_checks
    )

    raise RuntimeError(
        "PROJECT 8 STEP 7B DID NOT PASS.\n"
        "No Step 7B protocol files were written."
    )


# ------------------------------------------------------------
# 20. WRITE FROZEN ARTEFACTS
# ------------------------------------------------------------

PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

NOTES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    EXPERIMENT_CHRONOLOGY_PATH,
    experiment_chronology_output,
)

atomic_write_csv(
    CHRONOLOGY_DIFFERENCE_PATH,
    chronology_differences,
)

atomic_write_json(
    PROTOCOL_PATH,
    experiment_protocol,
)

atomic_write_json(
    MODEL_CONFIGURATION_PATH,
    model_configuration,
)

atomic_write_csv(
    PREDICTOR_PROFILE_PATH,
    predictor_profile,
)

atomic_write_csv(
    PROTOCOL_AUDIT_PATH,
    audit_frame,
)


# Byte-identical Notes copies.

shutil.copyfile(
    PROTOCOL_PATH,
    NOTES_PROTOCOL_PATH,
)

shutil.copyfile(
    MODEL_CONFIGURATION_PATH,
    NOTES_MODEL_CONFIGURATION_PATH,
)


# ------------------------------------------------------------
# 21. HASH FROZEN OUTPUTS
# ------------------------------------------------------------

experiment_chronology_sha256 = calculate_hash(
    EXPERIMENT_CHRONOLOGY_PATH
)

chronology_difference_sha256 = calculate_hash(
    CHRONOLOGY_DIFFERENCE_PATH
)

protocol_sha256 = calculate_hash(
    PROTOCOL_PATH
)

model_configuration_sha256 = calculate_hash(
    MODEL_CONFIGURATION_PATH
)

predictor_profile_sha256 = calculate_hash(
    PREDICTOR_PROFILE_PATH
)

protocol_audit_sha256 = calculate_hash(
    PROTOCOL_AUDIT_PATH
)

notes_protocol_sha256 = calculate_hash(
    NOTES_PROTOCOL_PATH
)

notes_model_configuration_sha256 = calculate_hash(
    NOTES_MODEL_CONFIGURATION_PATH
)


if protocol_sha256 != notes_protocol_sha256:
    raise AssertionError(
        "The Notes protocol copy is not byte-identical."
    )


if (
    model_configuration_sha256
    != notes_model_configuration_sha256
):
    raise AssertionError(
        "The Notes model configuration copy is not "
        "byte-identical."
    )


# ------------------------------------------------------------
# 22. STEP 7B REPORT AND STATUS
# ------------------------------------------------------------

step7b_status_text = (
    "PASS_PROJECT_8_CANONICAL_PROTOCOL_AND_MODEL_CONFIGURATION_FROZEN"
)


step7b_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "ChronologyResolution": {
        "SourceValidationChronology":
            str(
                SOURCE_VALIDATION_CHRONOLOGY_PATH
            ),

        "CanonicalExperimentChronology":
            str(
                EXPERIMENT_CHRONOLOGY_PATH
            ),

        "CanonicalRule":
            (
                "started_at ascending; build ID "
                "descending for equal timestamps"
            ),

        "EqualTimestampGroups":
            equal_timestamp_groups,

        "EqualTimestampBuilds":
            equal_timestamp_builds,

        "MismatchPositions":
            chronology_mismatch_positions,

        "DistinctMismatchBuilds":
            chronology_mismatch_builds,

        "MismatchTrainingBuilds":
            mismatch_training_builds,

        "MismatchEvaluationBuilds":
            mismatch_evaluation_builds,

        "PartitionAssignmentMismatches":
            partition_assignment_mismatches,

        "EvaluationMismatchPositions":
            mismatch_positions_in_evaluation,

        "Resolution":
            (
                "retain clean-source chronology for Step 6 "
                "validation; use canonical experiment "
                "chronology for all noise conditions; "
                "apply clean-anchored canonical REC deltas"
            ),
    },

    "Predictors": {
        "OriginalPredictorCount":
            len(predictor_columns),

        "ZeroVariancePredictorCount":
            len(
                zero_variance_predictors
            ),

        "ZeroVariancePredictors":
            zero_variance_predictors,

        "ActivePredictorCount":
            len(active_predictors),

        "NumericConversionFailures":
            numeric_conversion_failures,

        "PositiveInfiniteValues":
            positive_infinities,

        "NegativeInfiniteValues":
            negative_infinities,
    },

    "ExperimentDimensions": {
        "NoiseLevels":
            len(NOISE_LEVELS_PERCENT),

        "Seeds":
            len(REPETITION_SEEDS),

        "Conditions":
            EXPECTED_CONDITIONS,

        "MLTechniques":
            len(ML_TECHNIQUES),

        "ExpectedMLFits":
            EXPECTED_MODEL_FITS,

        "Baselines":
            len(BASELINES),
    },

    "FailureSubtypeDistribution": {
        "Counts":
            failure_subtype_counts,

        "Probabilities":
            failure_subtype_probabilities,

        "TotalFailureExecutions":
            total_training_failure_executions,
    },

    "LibraryVersions":
        library_versions,

    "AuditChecks":
        int(len(audit_frame)),

    "FailedAuditChecks":
        int(
            len(failed_audit_checks)
        ),

    "Outputs": {
        "ExperimentChronology": {
            "Path":
                str(
                    EXPERIMENT_CHRONOLOGY_PATH
                ),

            "SHA256":
                experiment_chronology_sha256,
        },

        "ChronologyDifference": {
            "Path":
                str(
                    CHRONOLOGY_DIFFERENCE_PATH
                ),

            "SHA256":
                chronology_difference_sha256,
        },

        "ExperimentProtocol": {
            "Path":
                str(PROTOCOL_PATH),

            "SHA256":
                protocol_sha256,
        },

        "ModelConfiguration": {
            "Path":
                str(
                    MODEL_CONFIGURATION_PATH
                ),

            "SHA256":
                model_configuration_sha256,
        },

        "PredictorProfile": {
            "Path":
                str(
                    PREDICTOR_PROFILE_PATH
                ),

            "SHA256":
                predictor_profile_sha256,
        },

        "ProtocolAudit": {
            "Path":
                str(
                    PROTOCOL_AUDIT_PATH
                ),

            "SHA256":
                protocol_audit_sha256,
        },

        "NotesProtocolCopy": {
            "Path":
                str(
                    NOTES_PROTOCOL_PATH
                ),

            "SHA256":
                notes_protocol_sha256,
        },

        "NotesModelConfigurationCopy": {
            "Path":
                str(
                    NOTES_MODEL_CONFIGURATION_PATH
                ),

            "SHA256":
                notes_model_configuration_sha256,
        },
    },

    "CompletionRegistryModified":
        False,

    "Projects1To7Modified":
        False,

    "Status":
        step7b_status_text,
}


atomic_write_json(
    STEP7B_REPORT_PATH,
    step7b_report,
)


step7b_status = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        step7b_status_text,

    "ExperimentChronologySHA256":
        experiment_chronology_sha256,

    "ProtocolSHA256":
        protocol_sha256,

    "ModelConfigurationSHA256":
        model_configuration_sha256,

    "ChronologyMismatchPositions":
        chronology_mismatch_positions,

    "PartitionAssignmentMismatches":
        partition_assignment_mismatches,

    "EvaluationMismatchPositions":
        mismatch_positions_in_evaluation,

    "OriginalPredictorCount":
        len(predictor_columns),

    "ZeroVariancePredictorCount":
        len(
            zero_variance_predictors
        ),

    "ActivePredictorCount":
        len(active_predictors),

    "ExpectedConditions":
        EXPECTED_CONDITIONS,

    "ExpectedMLFits":
        EXPECTED_MODEL_FITS,

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP7B_STATUS_PATH,
    step7b_status,
)


# ------------------------------------------------------------
# 23. VERIFY OUTPUTS
# ------------------------------------------------------------

expected_outputs = [
    EXPERIMENT_CHRONOLOGY_PATH,
    CHRONOLOGY_DIFFERENCE_PATH,
    PROTOCOL_PATH,
    MODEL_CONFIGURATION_PATH,
    PREDICTOR_PROFILE_PATH,
    PROTOCOL_AUDIT_PATH,
    STEP7B_REPORT_PATH,
    STEP7B_STATUS_PATH,
    NOTES_PROTOCOL_PATH,
    NOTES_MODEL_CONFIGURATION_PATH,
]


missing_outputs = [
    str(path)
    for path in expected_outputs
    if not path.exists()
]


if missing_outputs:
    raise RuntimeError(
        "Step 7B did not create all expected outputs:\n"
        + "\n".join(
            missing_outputs
        )
    )


# ------------------------------------------------------------
# 24. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 88)
print("=== PROJECT 8 STEP 7B RESULT ===")
print("=" * 88)

print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nChronology resolution:")

print(
    "Source-validation chronology:",
    SOURCE_VALIDATION_CHRONOLOGY_PATH,
)

print(
    "Canonical experiment chronology:",
    EXPERIMENT_CHRONOLOGY_PATH,
)

print(
    "Canonical order:",
    "started_at ascending",
)

print(
    "Canonical equal-timestamp tie-break:",
    "build ID descending",
)

print(
    "Equal-timestamp groups:",
    equal_timestamp_groups,
)

print(
    "Equal-timestamp builds:",
    equal_timestamp_builds,
)

print(
    "Source/experiment mismatch positions:",
    chronology_mismatch_positions,
)

print(
    "Distinct mismatch builds:",
    len(chronology_mismatch_builds),
)

print(
    "Partition-assignment mismatches:",
    partition_assignment_mismatches,
)

print(
    "Evaluation mismatch positions:",
    mismatch_positions_in_evaluation,
)

print(
    "Evaluation builds affected:",
    len(mismatch_evaluation_builds),
)


print("\nClean-anchor policy:")

print(
    "Step 6 clean REC mismatches:",
    step6_status.get(
        "total_rec_mismatches"
    ),
)

print(
    "Experiment REC policy:",
    "original clean anchor + canonical noisy-clean delta",
)


print("\nExperiment dimensions:")

print(
    "Noise levels:",
    len(NOISE_LEVELS_PERCENT),
)

print(
    "Repetition seeds:",
    len(REPETITION_SEEDS),
)

print(
    "Conditions:",
    EXPECTED_CONDITIONS,
)

print(
    "ML techniques:",
    len(ML_TECHNIQUES),
)

print(
    "Expected ML fits:",
    EXPECTED_MODEL_FITS,
)

print(
    "Baselines:",
    len(BASELINES),
)


print("\nProject-specific predictors:")

print(
    "Original predictors:",
    len(predictor_columns),
)

print(
    "Zero-variance predictors:",
    len(
        zero_variance_predictors
    ),
)

print(
    "Active predictors:",
    len(active_predictors),
)

print(
    "Numeric conversion failures:",
    numeric_conversion_failures,
)


if zero_variance_predictors:

    print(
        "Excluded zero-variance predictors:"
    )

    for predictor in zero_variance_predictors:
        print(
            " -",
            predictor,
        )


print("\nTraining failure subtypes:")

print(
    "Counts:",
    failure_subtype_counts,
)

print(
    "Probabilities:",
    failure_subtype_probabilities,
)


print("\nModel libraries:")

for (
    package_name,
    version,
) in library_versions.items():

    print(
        f"{package_name}: {version}"
    )


print("\nFrozen outputs:")

for output_path in expected_outputs:
    print(
        output_path
    )


print("\nExperiment chronology SHA-256:")

print(
    experiment_chronology_sha256
)


print("\nProtocol SHA-256:")

print(
    protocol_sha256
)


print("\nModel configuration SHA-256:")

print(
    model_configuration_sha256
)


print("\nAudit:")

print(
    "Checks:",
    len(audit_frame),
)

print(
    "Failed checks:",
    len(failed_audit_checks),
)


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–7 modified:")
print(0)


print(
    "\nSTATUS:",
    step7b_status_text,
)

print("=" * 88)

=== PROJECT 8 STEP 7B: CANONICAL PROTOCOL AND CONFIGURATION FREEZE ===

Runtime source validation:
Source directory: /content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo
Required source files: 5
Missing source files: 0

Canonical source validation:
Sources checked: 4
Hash mismatches: 0
Canonical primary order: started_at ascending
Canonical equal-timestamp rule: build ID descending

Project 8 data:
builds.csv: (254, 3)
dataset.csv: (9880, 154)
canonical raw executions: (34438, 14)
source-validation chronology: (254, 6)

Chronology comparison:
Source-validation order mismatch positions: 6
Distinct builds involved: 6
Partition-assignment mismatches: 0
Mismatch positions in evaluation: 0
Mismatch evaluation builds: 0


,Position,SourceValidationBuild,CanonicalExperimentBuild,Matches
19,20,568996928,568996948,False
20,21,568996948,568996928,False
78,79,616293259,616293277,False
79,80,616293277,616293259,False
139,140,700863672,700863680,False
140,141,700863680,700863672,False



Equal-timestamp groups:


,TimestampUTC,BuildCount,AscendingOrder,DescendingOrder,SourceValidationOrder,CanonicalExperimentOrder,DetectedSourceRule,ExperimentUsesDescending
0,2019-08-07T18:19:59+00:00,2,568996928#568996948,568996948#568996928,568996928#568996948,568996948#568996928,build ID ascending,True
1,2019-11-24T15:31:19+00:00,2,616293259#616293277,616293277#616293259,616293259#616293277,616293277#616293259,build ID ascending,True
2,2020-06-22T12:42:21+00:00,2,700863672#700863680,700863680#700863672,700863672#700863680,700863680#700863672,build ID ascending,True



Step 7B freeze audit:


,Check,Expected,Actual,Pass
0,Projects 1–7 remain frozen,"[1, 2, 3, 4, 5, 6, 7]","[1, 2, 3, 4, 5, 6, 7]",True
1,Project 8 absent from completion registry,0,0,True
2,Restored source files,5,5,True
3,Step 5 accepted,True,True,True
4,Step 6 clean REC validation passed,PASS_PROJECT_8_CLEAN_REC_RECONSTRUCTION_VALIDATED,PASS_PROJECT_8_CLEAN_REC_RECONSTRUCTION_VALIDATED,True
5,Step 6 REC mismatches,0,0,True
6,Canonical source hash mismatches,0,0,True
7,Source and experiment build sets identical,True,True,True
8,Equal-timestamp groups,3,3,True
9,Equal-timestamp builds,6,6,True




=== PROJECT 8 STEP 7B RESULT ===

Project identity:
Project number: 8
Project: optimatika@ojAlgo
Project slug: optimatika__ojAlgo

Chronology resolution:
Source-validation chronology: /content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_preflight/ojalgo_build_chronology.csv
Canonical experiment chronology: /content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_preflight/ojalgo_experiment_build_chronology.csv
Canonical order: started_at ascending
Canonical equal-timestamp tie-break: build ID descending
Equal-timestamp groups: 3
Equal-timestamp builds: 6
Source/experiment mismatch positions: 6
Distinct mismatch builds: 6
Partition-assignment mismatches: 0
Evaluation mismatch positions: 0
Evaluation builds affected: 0

Clean-anchor policy:
Step 6 clean REC mismatches: 0
Experiment REC policy: original clean anchor + canonical noisy-clean delta

Experiment dimensions:
Noise levels: 9
Repetition seeds: 30
Conditions: 270
ML t

In [ ]:
# ============================================================
# PROJECT 8 — STEP 8A
# CANONICAL NOISE, CLEAN-ANCHOR AND METRIC HELPER VALIDATION
#
# PROJECT: optimatika@ojAlgo
#
# This cell validates:
# - deterministic project-specific random seeds
# - nested noise masks across noise levels
# - pass-to-failure subtype sampling
# - failure-to-pass corruption
# - fixed model-ready training cohort
# - canonical experiment chronology
# - clean-anchored REC delta reconstruction
# - 13 verdict-dependent REC features
# - six preserved verdict-independent REC features
# - zero-noise identity
# - noisy-label alignment
# - clean evaluation immutability
# - APFD and APFDc helpers
#
# It does NOT:
# - fit ML models
# - execute the 270 full conditions
# - modify source CSV files
# - modify Projects 1–7
# - modify the completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import re

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

PROJECT_NUMBER = 8
PROJECT_NAME = "optimatika@ojAlgo"
PROJECT_SLUG = "optimatika__ojAlgo"
PROJECT_SHORT_NAME = "ojalgo"

RECENT_WINDOW = 6

REC_FEATURE_COLUMNS = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

TEST_REPETITION_SEED = 1
TEST_NOISE_PERCENT = 5
NESTED_COMPARISON_NOISE_PERCENT = 10


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

PROJECT_SOURCE_DIR = Path(
    "/content/TCP-CI-main-dataset/datasets/optimatika@ojAlgo"
)

DATASET_PATH = (
    PROJECT_SOURCE_DIR
    / "dataset.csv"
)

PROJECT_AGGREGATED_DIR = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / PROJECT_SLUG
)

PREFLIGHT_DIR = (
    PROJECT_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_preflight"
)

CANONICAL_RAW_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_canonical_raw_executions.csv.gz"
)

EXPERIMENT_CHRONOLOGY_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_experiment_build_chronology.csv"
)

BUILD_ENTITY_MAP_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_build_entity_map.csv.gz"
)

CHANGED_ENTITIES_DICTIONARY_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_changed_entities_by_build.json"
)

STEP7B_STATUS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step7b_status.json"
)

PROTOCOL_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_experiment_protocol.json"
)

MODEL_CONFIGURATION_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_model_configuration.json"
)


# Step 8A outputs

CLEAN_ANCHOR_OFFSETS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_rec_clean_anchor_offsets.csv.gz"
)

HELPER_TEST_SUMMARY_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_helper_test_summary.csv"
)

HELPER_AUDIT_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_helper_validation_audit.csv"
)

HELPER_REPORT_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_helper_validation_report.json"
)

STEP8A_STATUS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step8a_status.json"
)


print("=" * 88)
print("=== PROJECT 8 STEP 8A: CANONICAL HELPER VALIDATION ===")
print("=" * 88)


# ------------------------------------------------------------
# 3. GENERAL HELPERS
# ------------------------------------------------------------

def canonical_integer(value):
    if pd.isna(value):
        return None

    text = str(value).strip()

    if text == "":
        return None

    numeric = pd.to_numeric(
        pd.Series([text]),
        errors="coerce",
    ).iloc[0]

    if pd.isna(numeric):
        return None

    if not float(numeric).is_integer():
        return None

    return int(numeric)


def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def dataframe_content_hash(dataframe):
    hashed_values = pd.util.hash_pandas_object(
        dataframe,
        index=True,
        categorize=True,
    ).to_numpy(
        dtype=np.uint64
    )

    return hashlib.sha256(
        hashed_values.tobytes()
    ).hexdigest()


def json_safe(value):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def atomic_write_csv(
    path,
    dataframe,
    compression=None,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
        compression=compression,
    )

    temporary_path.replace(path)


# ------------------------------------------------------------
# 4. DETERMINISTIC RANDOM SEED
# ------------------------------------------------------------

def stable_project_seed(
    project_name,
    repetition_seed,
    random_stream,
):
    """
    Produce a deterministic project-specific 32-bit seed.
    """

    seed_text = (
        f"{project_name}|"
        f"{int(repetition_seed)}|"
        f"{random_stream}"
    )

    digest = hashlib.sha256(
        seed_text.encode("utf-8")
    ).digest()

    return int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False,
    ) % (2 ** 32)


# ------------------------------------------------------------
# 5. TRAINING VERDICT NOISE
# ------------------------------------------------------------

def inject_training_verdict_noise(
    execution_history,
    noise_percent,
    repetition_seed,
    project_name=PROJECT_NAME,
):
    """
    Independently flip each raw training verdict with
    probability noise_percent / 100.

    The same project and repetition seed use the same
    row uniforms at every noise level, producing nested
    masks.
    """

    noise_percent = float(
        noise_percent
    )

    if not 0 <= noise_percent <= 100:
        raise ValueError(
            "noise_percent must be between 0 and 100."
        )

    noisy_history = (
        execution_history
        .copy()
        .reset_index(drop=True)
    )

    original_verdict = pd.to_numeric(
        noisy_history["verdict"],
        errors="raise",
    ).astype(int).to_numpy()

    number_of_rows = len(
        noisy_history
    )

    if number_of_rows == 0:
        raise ValueError(
            "Training execution history is empty."
        )

    clean_failure_verdicts = (
        pd.Series(
            original_verdict[
                original_verdict != 0
            ]
        )
        .value_counts()
        .sort_index()
    )

    if clean_failure_verdicts.empty:
        raise ValueError(
            "No failure subtype exists in clean training."
        )

    failure_subtypes = (
        clean_failure_verdicts
        .index
        .to_numpy(dtype=int)
    )

    failure_probabilities = (
        clean_failure_verdicts
        .to_numpy(dtype=float)
    )

    failure_probabilities /= (
        failure_probabilities.sum()
    )

    flip_rng = np.random.default_rng(
        stable_project_seed(
            project_name,
            repetition_seed,
            "flip_mask",
        )
    )

    subtype_rng = np.random.default_rng(
        stable_project_seed(
            project_name,
            repetition_seed,
            "failure_subtype",
        )
    )

    row_uniforms = flip_rng.random(
        number_of_rows
    )

    flip_mask = (
        row_uniforms
        < noise_percent / 100.0
    )

    sampled_failure_subtypes = (
        subtype_rng.choice(
            failure_subtypes,
            size=number_of_rows,
            replace=True,
            p=failure_probabilities,
        )
    )

    noisy_verdict = (
        original_verdict.copy()
    )

    pass_to_failure_mask = (
        flip_mask
        & (original_verdict == 0)
    )

    failure_to_pass_mask = (
        flip_mask
        & (original_verdict != 0)
    )

    noisy_verdict[
        pass_to_failure_mask
    ] = sampled_failure_subtypes[
        pass_to_failure_mask
    ]

    noisy_verdict[
        failure_to_pass_mask
    ] = 0

    noisy_history["verdict"] = (
        noisy_verdict
    )

    manifest = pd.DataFrame({
        "NoiseRowID":
            np.arange(
                number_of_rows
            ),

        "Build":
            noisy_history[
                "build"
            ].to_numpy(),

        "Test":
            noisy_history[
                "test"
            ].to_numpy(),

        "OriginalVerdict":
            original_verdict,

        "NoisyVerdict":
            noisy_verdict,

        "FlipUniform":
            row_uniforms,

        "Flipped":
            flip_mask,

        "PassToFailure":
            pass_to_failure_mask,

        "FailureToPass":
            failure_to_pass_mask,
    })

    if "job" in noisy_history.columns:
        manifest["Job"] = (
            noisy_history[
                "job"
            ].to_numpy()
        )

    summary = {
        "Project":
            project_name,

        "NoisePercentRequested":
            noise_percent,

        "RepetitionSeed":
            int(
                repetition_seed
            ),

        "TrainingExecutionRows":
            number_of_rows,

        "NumberFlipped":
            int(
                flip_mask.sum()
            ),

        "RealisedNoisePercent":
            float(
                100.0
                * flip_mask.sum()
                / number_of_rows
            ),

        "PassToFailure":
            int(
                pass_to_failure_mask.sum()
            ),

        "FailureToPass":
            int(
                failure_to_pass_mask.sum()
            ),
    }

    return (
        noisy_history,
        manifest,
        summary,
    )


# ------------------------------------------------------------
# 6. REC HELPERS
# ------------------------------------------------------------

def calculate_rates(history):
    history_length = len(
        history
    )

    if history_length == 0:
        raise ValueError(
            "Rate calculation requires prior history."
        )

    verdicts = history[
        "verdict"
    ]

    return (
        float(
            (verdicts != 0).sum()
            / history_length
        ),

        float(
            (verdicts == 2).sum()
            / history_length
        ),

        float(
            (verdicts == 1).sum()
            / history_length
        ),

        float(
            (
                history["transition"]
                == 1
            ).sum()
            / history_length
        ),
    )


def calculate_max_test_file_rate(
    history,
    target_column,
    current_changed_entities,
    entity_changed_builds,
):
    target_builds = (
        history.loc[
            history[target_column] > 0,
            "build",
        ]
        .drop_duplicates()
        .astype(int)
        .tolist()
    )

    if len(target_builds) == 0:
        return -1.0

    target_build_set = set(
        target_builds
    )

    maximum_frequency = 0

    for entity_id in (
        current_changed_entities
    ):
        entity_builds = (
            entity_changed_builds.get(
                int(entity_id),
                set(),
            )
        )

        overlap_count = len(
            entity_builds.intersection(
                target_build_set
            )
        )

        maximum_frequency = max(
            maximum_frequency,
            overlap_count,
        )

    if maximum_frequency == 0:
        return 0.0

    return float(
        maximum_frequency
        / len(target_builds)
    )


def reconstruct_rec_features(
    execution_history,
    requested_rows,
    global_build_position,
    changed_entities_by_build,
    entity_changed_builds,
    recent_window=RECENT_WINDOW,
):
    """
    Reconstruct all 19 REC features using the canonical
    experiment chronology.
    """

    requested_pairs = set(
        zip(
            requested_rows[
                "Build"
            ].astype(int),

            requested_rows[
                "Test"
            ].astype(int),
        )
    )

    records = []

    for (
        test_id,
        test_history,
    ) in execution_history.groupby(
        "test",
        sort=False,
    ):

        test_history = (
            test_history
            .sort_values(
                [
                    "build_order",
                    "job",
                ],
                kind="mergesort",
            )
            .reset_index(drop=True)
            .copy()
        )

        test_history[
            "transition"
        ] = (
            test_history[
                "verdict"
            ]
            .diff()
            .fillna(0)
            .ne(0)
            .astype(int)
        )

        first_test_build = int(
            test_history.iloc[0][
                "build"
            ]
        )

        for current_position in range(
            len(test_history)
        ):

            current_row = (
                test_history.iloc[
                    current_position
                ]
            )

            current_build = int(
                current_row[
                    "build"
                ]
            )

            requested_pair = (
                current_build,
                int(test_id),
            )

            if (
                requested_pair
                not in requested_pairs
            ):
                continue

            prior_history = (
                test_history
                .iloc[
                    :current_position
                ]
                .copy()
                .reset_index(drop=True)
            )

            record = {
                "Build":
                    current_build,

                "Test":
                    int(test_id),
            }

            if prior_history.empty:

                for feature in (
                    REC_FEATURE_COLUMNS
                ):
                    record[feature] = -1.0

                record["REC_Age"] = 0

                records.append(record)
                continue

            recent_history = (
                prior_history
                .tail(
                    recent_window
                )
                .copy()
            )

            age = int(
                global_build_position[
                    current_build
                ]
                - global_build_position[
                    first_test_build
                ]
            )

            failure_positions = (
                np.flatnonzero(
                    prior_history[
                        "verdict"
                    ].to_numpy()
                    > 0
                )
            )

            if len(
                failure_positions
            ) == 0:
                last_failure_age = -1
            else:
                last_failure_age = int(
                    len(prior_history)
                    - 1
                    - int(
                        failure_positions[-1]
                    )
                )

            transition_positions = (
                np.flatnonzero(
                    prior_history[
                        "transition"
                    ].to_numpy()
                    > 0
                )
            )

            if len(
                transition_positions
            ) == 0:
                last_transition_age = -1
            else:
                last_transition_age = int(
                    len(prior_history)
                    - 1
                    - int(
                        transition_positions[-1]
                    )
                )

            (
                recent_fail_rate,
                recent_assert_rate,
                recent_exc_rate,
                recent_transition_rate,
            ) = calculate_rates(
                recent_history
            )

            (
                total_fail_rate,
                total_assert_rate,
                total_exc_rate,
                total_transition_rate,
            ) = calculate_rates(
                prior_history
            )

            current_changed_entities = (
                changed_entities_by_build.get(
                    current_build,
                    set(),
                )
            )

            record.update({
                "REC_Age":
                    age,

                "REC_LastFailureAge":
                    last_failure_age,

                "REC_LastTransitionAge":
                    last_transition_age,

                "REC_RecentAvgExeTime":
                    float(
                        recent_history[
                            "duration"
                        ].mean()
                    ),

                "REC_RecentMaxExeTime":
                    float(
                        recent_history[
                            "duration"
                        ].max()
                    ),

                "REC_RecentFailRate":
                    recent_fail_rate,

                "REC_RecentAssertRate":
                    recent_assert_rate,

                "REC_RecentExcRate":
                    recent_exc_rate,

                "REC_RecentTransitionRate":
                    recent_transition_rate,

                "REC_TotalAvgExeTime":
                    float(
                        prior_history[
                            "duration"
                        ].mean()
                    ),

                "REC_TotalMaxExeTime":
                    float(
                        prior_history[
                            "duration"
                        ].max()
                    ),

                "REC_TotalFailRate":
                    total_fail_rate,

                "REC_TotalAssertRate":
                    total_assert_rate,

                "REC_TotalExcRate":
                    total_exc_rate,

                "REC_TotalTransitionRate":
                    total_transition_rate,

                "REC_LastVerdict":
                    int(
                        recent_history.iloc[-1][
                            "verdict"
                        ]
                    ),

                "REC_LastExeTime":
                    float(
                        recent_history.iloc[-1][
                            "duration"
                        ]
                    ),

                "REC_MaxTestFileFailRate":
                    calculate_max_test_file_rate(
                        history=prior_history,
                        target_column="verdict",
                        current_changed_entities=(
                            current_changed_entities
                        ),
                        entity_changed_builds=(
                            entity_changed_builds
                        ),
                    ),

                "REC_MaxTestFileTransitionRate":
                    calculate_max_test_file_rate(
                        history=prior_history,
                        target_column="transition",
                        current_changed_entities=(
                            current_changed_entities
                        ),
                        entity_changed_builds=(
                            entity_changed_builds
                        ),
                    ),
            })

            records.append(record)

    return pd.DataFrame(
        records,
        columns=[
            "Build",
            "Test",
            *REC_FEATURE_COLUMNS,
        ],
    )


def align_reconstructed_rec_features(
    reconstructed,
    requested_rows,
):
    requested_with_order = (
        requested_rows[
            [
                "Build",
                "Test",
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

    requested_with_order[
        "_RequestedOrder"
    ] = np.arange(
        len(
            requested_with_order
        )
    )

    reconstructed_subset = (
        reconstructed[
            [
                "Build",
                "Test",
                *REC_FEATURE_COLUMNS,
            ]
        ]
        .copy()
    )

    if reconstructed_subset.duplicated(
        [
            "Build",
            "Test",
        ]
    ).any():
        raise ValueError(
            "Reconstruction contains duplicate pairs."
        )

    aligned = (
        requested_with_order
        .merge(
            reconstructed_subset,
            on=[
                "Build",
                "Test",
            ],
            how="left",
            validate="one_to_one",
        )
        .sort_values(
            "_RequestedOrder",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    if aligned[
        REC_FEATURE_COLUMNS
    ].isna().all(axis=1).any():
        raise ValueError(
            "At least one requested row was not reconstructed."
        )

    return aligned[
        REC_FEATURE_COLUMNS
    ].copy()


def extract_model_ready_verdicts(
    noisy_execution_history,
    requested_rows,
):
    verdict_rows = (
        noisy_execution_history[
            [
                "build",
                "test",
                "verdict",
            ]
        ]
        .copy()
    )

    conflicting_duplicates = (
        verdict_rows
        .groupby(
            [
                "build",
                "test",
            ],
            sort=False,
        )["verdict"]
        .nunique()
    )

    if (
        conflicting_duplicates
        .gt(1)
        .any()
    ):
        raise ValueError(
            "Conflicting duplicate raw verdicts exist."
        )

    verdict_rows = (
        verdict_rows
        .drop_duplicates(
            [
                "build",
                "test",
            ],
            keep="last",
        )
        .rename(
            columns={
                "build":
                    "Build",

                "test":
                    "Test",

                "verdict":
                    "NoisyVerdict",
            }
        )
    )

    requested = (
        requested_rows[
            [
                "Build",
                "Test",
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

    requested[
        "_RequestedOrder"
    ] = np.arange(
        len(requested)
    )

    aligned = (
        requested
        .merge(
            verdict_rows,
            on=[
                "Build",
                "Test",
            ],
            how="left",
            validate="one_to_one",
        )
        .sort_values(
            "_RequestedOrder",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    if aligned[
        "NoisyVerdict"
    ].isna().any():
        raise ValueError(
            "Some model-ready rows have no noisy verdict."
        )

    return (
        aligned[
            "NoisyVerdict"
        ]
        .astype(int)
        .reset_index(drop=True)
    )


def create_anchored_noisy_training_data(
    clean_model_training_data,
    noisy_execution_history,
    canonical_clean_rec,
    global_build_position,
    changed_entities_by_build,
    entity_changed_builds,
):
    """
    Clean-anchored canonical delta reconstruction:

    final noisy value =
        original clean value
        + canonical noisy reconstruction
        - canonical clean reconstruction
    """

    noisy_model_data = (
        clean_model_training_data
        .copy()
        .reset_index(drop=True)
    )

    requested_rows = (
        noisy_model_data[
            [
                "Build",
                "Test",
            ]
        ]
        .copy()
    )

    noisy_reconstructed = (
        reconstruct_rec_features(
            execution_history=(
                noisy_execution_history
            ),

            requested_rows=(
                requested_rows
            ),

            global_build_position=(
                global_build_position
            ),

            changed_entities_by_build=(
                changed_entities_by_build
            ),

            entity_changed_builds=(
                entity_changed_builds
            ),

            recent_window=(
                RECENT_WINDOW
            ),
        )
    )

    aligned_noisy_rec = (
        align_reconstructed_rec_features(
            reconstructed=(
                noisy_reconstructed
            ),

            requested_rows=(
                requested_rows
            ),
        )
    )

    for feature in (
        VERDICT_DEPENDENT_REC
    ):
        original_values = pd.to_numeric(
            clean_model_training_data[
                feature
            ],
            errors="raise",
        ).to_numpy(dtype=float)

        canonical_clean_values = (
            pd.to_numeric(
                canonical_clean_rec[
                    feature
                ],
                errors="raise",
            )
            .to_numpy(dtype=float)
        )

        canonical_noisy_values = (
            pd.to_numeric(
                aligned_noisy_rec[
                    feature
                ],
                errors="raise",
            )
            .to_numpy(dtype=float)
        )

        noisy_model_data[
            feature
        ] = (
            original_values
            + (
                canonical_noisy_values
                - canonical_clean_values
            )
        )

    # Six verdict-independent REC features and every
    # non-REC predictor stay exactly as stored.

    noisy_model_data[
        "Verdict"
    ] = (
        extract_model_ready_verdicts(
            noisy_execution_history=(
                noisy_execution_history
            ),

            requested_rows=(
                requested_rows
            ),
        )
        .to_numpy()
    )

    return (
        noisy_model_data,
        aligned_noisy_rec,
    )


# ------------------------------------------------------------
# 7. APFD AND APFDC
# ------------------------------------------------------------

def calculate_apfd(
    actual_failures,
):
    failures = np.asarray(
        actual_failures,
        dtype=int,
    )

    number_of_tests = len(
        failures
    )

    number_of_failures = int(
        failures.sum()
    )

    if number_of_tests == 0:
        return np.nan

    if number_of_failures == 0:
        return np.nan

    failure_ranks = (
        np.flatnonzero(
            failures == 1
        )
        + 1
    )

    return float(
        1.0
        - failure_ranks.sum()
        / (
            number_of_tests
            * number_of_failures
        )
        + 1.0
        / (
            2.0
            * number_of_tests
        )
    )


def calculate_apfdc(
    actual_failures,
    durations,
):
    failures = np.asarray(
        actual_failures,
        dtype=int,
    )

    durations = np.asarray(
        durations,
        dtype=float,
    )

    if len(failures) != len(
        durations
    ):
        raise ValueError(
            "Failures and durations differ in length."
        )

    if len(failures) == 0:
        return np.nan

    if failures.sum() == 0:
        return np.nan

    if not np.isfinite(
        durations
    ).all():
        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (
        durations < 0
    ).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(
        durations.sum()
    )

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array([0.0]),
        np.cumsum(
            durations
        )[:-1],
    ])

    failure_mask = (
        failures == 1
    )

    midpoint_detection_times = (
        cumulative_before[
            failure_mask
        ]
        + 0.5
        * durations[
            failure_mask
        ]
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


# ------------------------------------------------------------
# 8. VALIDATE REQUIRED INPUTS
# ------------------------------------------------------------

required_inputs = [
    DATASET_PATH,
    CANONICAL_RAW_PATH,
    EXPERIMENT_CHRONOLOGY_PATH,
    BUILD_ENTITY_MAP_PATH,
    CHANGED_ENTITIES_DICTIONARY_PATH,
    STEP7B_STATUS_PATH,
    PROTOCOL_PATH,
    MODEL_CONFIGURATION_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Required Step 8A inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
    )


step7b_status = json.loads(
    STEP7B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)

expected_step7b_status = (
    "PASS_PROJECT_8_CANONICAL_PROTOCOL_"
    "AND_MODEL_CONFIGURATION_FROZEN"
)

if (
    step7b_status.get(
        "Status"
    )
    != expected_step7b_status
):
    raise AssertionError(
        "Step 7B has not passed.\n"
        f"Detected: "
        f"{step7b_status.get('Status')}"
    )


# ------------------------------------------------------------
# 9. LOAD INPUT DATA
# ------------------------------------------------------------

print("\nLoading Step 8A inputs...")

dataset = pd.read_csv(
    DATASET_PATH,
    low_memory=False,
)

canonical_raw = pd.read_csv(
    CANONICAL_RAW_PATH,
    low_memory=False,
)

experiment_chronology = pd.read_csv(
    EXPERIMENT_CHRONOLOGY_PATH,
    low_memory=False,
)

build_entity_map = pd.read_csv(
    BUILD_ENTITY_MAP_PATH,
    low_memory=False,
)

changed_entities_json = json.loads(
    CHANGED_ENTITIES_DICTIONARY_PATH.read_text(
        encoding="utf-8"
    )
)


print(
    "dataset.csv:",
    dataset.shape,
)

print(
    "Canonical raw executions:",
    canonical_raw.shape,
)

print(
    "Experiment chronology:",
    experiment_chronology.shape,
)

print(
    "Build-entity map:",
    build_entity_map.shape,
)

print(
    "Changed-entity dictionary builds:",
    len(
        changed_entities_json
    ),
)


# ------------------------------------------------------------
# 10. PREPARE DATASET
# ------------------------------------------------------------

required_dataset_columns = {
    "Build",
    "Test",
    "Verdict",
    "Duration",
    *REC_FEATURE_COLUMNS,
}

missing_dataset_columns = (
    required_dataset_columns
    - set(dataset.columns)
)

if missing_dataset_columns:
    raise RuntimeError(
        "dataset.csv is missing columns:\n"
        f"{sorted(missing_dataset_columns)}"
    )


dataset = dataset.copy()

for column in [
    "Build",
    "Test",
    "Verdict",
]:
    dataset[column] = pd.to_numeric(
        dataset[column],
        errors="raise",
    ).astype(np.int64)

dataset[
    "Duration"
] = pd.to_numeric(
    dataset["Duration"],
    errors="raise",
).astype(float)


if dataset.duplicated(
    [
        "Build",
        "Test",
    ]
).any():
    raise AssertionError(
        "dataset.csv contains duplicate Build-Test rows."
    )


# ------------------------------------------------------------
# 11. PREPARE EXPERIMENT CHRONOLOGY
# ------------------------------------------------------------

required_chronology_columns = {
    "CanonicalBuild",
    "BuildOrder",
    "Partition",
}

missing_chronology_columns = (
    required_chronology_columns
    - set(
        experiment_chronology.columns
    )
)

if missing_chronology_columns:
    raise RuntimeError(
        "Experiment chronology is missing columns:\n"
        f"{sorted(missing_chronology_columns)}"
    )


experiment_chronology[
    "CanonicalBuild"
] = pd.to_numeric(
    experiment_chronology[
        "CanonicalBuild"
    ],
    errors="raise",
).astype(np.int64)

experiment_chronology[
    "BuildOrder"
] = pd.to_numeric(
    experiment_chronology[
        "BuildOrder"
    ],
    errors="raise",
).astype(int)


if experiment_chronology[
    "CanonicalBuild"
].duplicated().any():
    raise AssertionError(
        "Experiment chronology contains duplicate builds."
    )


build_order_map = dict(
    zip(
        experiment_chronology[
            "CanonicalBuild"
        ],
        experiment_chronology[
            "BuildOrder"
        ],
    )
)

partition_map = dict(
    zip(
        experiment_chronology[
            "CanonicalBuild"
        ],
        experiment_chronology[
            "Partition"
        ].astype(str),
    )
)


training_build_ids = {
    int(build_id)
    for build_id, partition
    in partition_map.items()
    if partition == "training"
}

evaluation_build_ids = {
    int(build_id)
    for build_id, partition
    in partition_map.items()
    if partition == "evaluation"
}


# ------------------------------------------------------------
# 12. PREPARE RAW EXECUTION HISTORY
# ------------------------------------------------------------

required_raw_columns = {
    "CanonicalBuild",
    "CanonicalTest",
    "CanonicalJob",
    "RawVerdictToken",
    "RawDuration",
}

missing_raw_columns = (
    required_raw_columns
    - set(canonical_raw.columns)
)

if missing_raw_columns:
    raise RuntimeError(
        "Canonical raw table is missing columns:\n"
        f"{sorted(missing_raw_columns)}"
    )


execution_history = pd.DataFrame({
    "build":
        pd.to_numeric(
            canonical_raw[
                "CanonicalBuild"
            ],
            errors="raise",
        ).astype(np.int64),

    "test":
        pd.to_numeric(
            canonical_raw[
                "CanonicalTest"
            ],
            errors="raise",
        ).astype(np.int64),

    "job":
        pd.to_numeric(
            canonical_raw[
                "CanonicalJob"
            ],
            errors="coerce",
        ).fillna(-1).astype(np.int64),

    "verdict":
        pd.to_numeric(
            canonical_raw[
                "RawVerdictToken"
            ],
            errors="raise",
        ).astype(np.int64),

    "duration":
        pd.to_numeric(
            canonical_raw[
                "RawDuration"
            ],
            errors="raise",
        ).astype(float),
})


execution_history[
    "build_order"
] = execution_history[
    "build"
].map(
    build_order_map
)


if execution_history[
    "build_order"
].isna().any():
    raise AssertionError(
        "Raw executions contain builds absent from "
        "the experiment chronology."
    )


execution_history[
    "build_order"
] = execution_history[
    "build_order"
].astype(int)


if execution_history.duplicated(
    [
        "build",
        "test",
    ]
).any():
    raise AssertionError(
        "Raw execution history contains duplicate "
        "Build-Test rows."
    )


execution_history = (
    execution_history
    .sort_values(
        [
            "build_order",
            "job",
            "test",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


ordered_execution_builds = (
    execution_history[
        [
            "build",
            "build_order",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "build_order",
        kind="mergesort",
    )["build"]
    .astype(int)
    .tolist()
)


global_build_position = {
    int(build_id): position
    for position, build_id
    in enumerate(
        ordered_execution_builds
    )
}


clean_training_history = (
    execution_history[
        execution_history[
            "build"
        ].isin(
            training_build_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)


clean_evaluation_history = (
    execution_history[
        execution_history[
            "build"
        ].isin(
            evaluation_build_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)


clean_model_training_data = (
    dataset[
        dataset[
            "Build"
        ].isin(
            training_build_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)


clean_evaluation_data = (
    dataset[
        dataset[
            "Build"
        ].isin(
            evaluation_build_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 13. PREPARE ENTITY HISTORY
# ------------------------------------------------------------

required_entity_columns = {
    "id",
    "EntityId",
}

missing_entity_columns = (
    required_entity_columns
    - set(build_entity_map.columns)
)

if missing_entity_columns:
    raise RuntimeError(
        "Build-entity map is missing columns:\n"
        f"{sorted(missing_entity_columns)}"
    )


build_entity_pairs = (
    build_entity_map[
        [
            "id",
            "EntityId",
        ]
    ]
    .drop_duplicates()
    .copy()
)

build_entity_pairs[
    "id"
] = pd.to_numeric(
    build_entity_pairs["id"],
    errors="raise",
).astype(np.int64)

build_entity_pairs[
    "EntityId"
] = pd.to_numeric(
    build_entity_pairs[
        "EntityId"
    ],
    errors="raise",
).astype(np.int64)


entity_changed_builds = (
    build_entity_pairs
    .groupby(
        "EntityId"
    )["id"]
    .apply(
        lambda values: set(
            int(value)
            for value in values
        )
    )
    .to_dict()
)


changed_entities_by_build = {
    int(build_id): set(
        int(entity_id)
        for entity_id in (
            changed_entities_json.get(
                str(build_id),
                []
            )
        )
    )
    for build_id
    in experiment_chronology[
        "CanonicalBuild"
    ].astype(int)
}


# ------------------------------------------------------------
# 14. BASIC POPULATION VALIDATION
# ------------------------------------------------------------

print("\nPrepared populations:")

print(
    "Training builds:",
    len(
        training_build_ids
    ),
)

print(
    "Evaluation builds:",
    len(
        evaluation_build_ids
    ),
)

print(
    "Raw training executions:",
    len(
        clean_training_history
    ),
)

print(
    "Raw evaluation executions:",
    len(
        clean_evaluation_history
    ),
)

print(
    "Model training rows:",
    len(
        clean_model_training_data
    ),
)

print(
    "Model evaluation rows:",
    len(
        clean_evaluation_data
    ),
)


# ------------------------------------------------------------
# 15. CANONICAL CLEAN RECONSTRUCTION
# ------------------------------------------------------------

requested_training_rows = (
    clean_model_training_data[
        [
            "Build",
            "Test",
        ]
    ]
    .copy()
)


print(
    "\nReconstructing canonical clean REC features..."
)


canonical_clean_reconstructed = (
    reconstruct_rec_features(
        execution_history=(
            clean_training_history
        ),

        requested_rows=(
            requested_training_rows
        ),

        global_build_position=(
            global_build_position
        ),

        changed_entities_by_build=(
            changed_entities_by_build
        ),

        entity_changed_builds=(
            entity_changed_builds
        ),

        recent_window=(
            RECENT_WINDOW
        ),
    )
)


canonical_clean_rec = (
    align_reconstructed_rec_features(
        reconstructed=(
            canonical_clean_reconstructed
        ),

        requested_rows=(
            requested_training_rows
        ),
    )
)


if (
    len(canonical_clean_rec)
    != len(
        clean_model_training_data
    )
):
    raise AssertionError(
        "Canonical clean reconstruction row count differs."
    )


# ------------------------------------------------------------
# 16. CLEAN-ANCHOR OFFSET TABLE
# ------------------------------------------------------------

clean_anchor_offsets = (
    requested_training_rows
    .reset_index(drop=True)
    .copy()
)


offset_nonzero_cells = 0


for feature in (
    VERDICT_DEPENDENT_REC
):

    original_values = pd.to_numeric(
        clean_model_training_data[
            feature
        ],
        errors="raise",
    ).to_numpy(dtype=float)

    canonical_values = pd.to_numeric(
        canonical_clean_rec[
            feature
        ],
        errors="raise",
    ).to_numpy(dtype=float)

    offsets = (
        original_values
        - canonical_values
    )

    clean_anchor_offsets[
        feature
    ] = offsets

    offset_nonzero_cells += int(
        (
            ~np.isclose(
                offsets,
                0.0,
                rtol=1e-10,
                atol=1e-12,
                equal_nan=True,
            )
        ).sum()
    )


offset_value_frame = (
    clean_anchor_offsets[
        VERDICT_DEPENDENT_REC
    ]
)


offset_nonzero_rows = int(
    (
        ~np.isclose(
            offset_value_frame.to_numpy(
                dtype=float
            ),
            0.0,
            rtol=1e-10,
            atol=1e-12,
            equal_nan=True,
        )
    )
    .any(axis=1)
    .sum()
)


clean_anchor_offsets[
    "AnyNonZeroOffset"
] = (
    ~np.isclose(
        offset_value_frame.to_numpy(
            dtype=float
        ),
        0.0,
        rtol=1e-10,
        atol=1e-12,
        equal_nan=True,
    )
).any(axis=1)


# ------------------------------------------------------------
# 17. ZERO-NOISE IDENTITY TEST
# ------------------------------------------------------------

(
    zero_noise_history,
    zero_noise_manifest,
    zero_noise_summary,
) = inject_training_verdict_noise(
    execution_history=(
        clean_training_history
    ),

    noise_percent=0,

    repetition_seed=(
        TEST_REPETITION_SEED
    ),
)


(
    zero_noise_model_data,
    zero_noise_canonical_rec,
) = create_anchored_noisy_training_data(
    clean_model_training_data=(
        clean_model_training_data
    ),

    noisy_execution_history=(
        zero_noise_history
    ),

    canonical_clean_rec=(
        canonical_clean_rec
    ),

    global_build_position=(
        global_build_position
    ),

    changed_entities_by_build=(
        changed_entities_by_build
    ),

    entity_changed_builds=(
        entity_changed_builds
    ),
)


zero_label_mismatches = int(
    (
        zero_noise_model_data[
            "Verdict"
        ].to_numpy()
        != clean_model_training_data[
            "Verdict"
        ].to_numpy()
    ).sum()
)


zero_rec_mismatches = 0


for feature in (
    REC_FEATURE_COLUMNS
):

    zero_values = pd.to_numeric(
        zero_noise_model_data[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)

    original_values = pd.to_numeric(
        clean_model_training_data[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)

    zero_rec_mismatches += int(
        (
            ~np.isclose(
                zero_values,
                original_values,
                rtol=1e-10,
                atol=1e-12,
                equal_nan=True,
            )
        ).sum()
    )


# ------------------------------------------------------------
# 18. FIVE-PERCENT REPRODUCIBILITY
# ------------------------------------------------------------

(
    five_history_a,
    five_manifest_a,
    five_summary_a,
) = inject_training_verdict_noise(
    execution_history=(
        clean_training_history
    ),

    noise_percent=(
        TEST_NOISE_PERCENT
    ),

    repetition_seed=(
        TEST_REPETITION_SEED
    ),
)


(
    five_history_b,
    five_manifest_b,
    five_summary_b,
) = inject_training_verdict_noise(
    execution_history=(
        clean_training_history
    ),

    noise_percent=(
        TEST_NOISE_PERCENT
    ),

    repetition_seed=(
        TEST_REPETITION_SEED
    ),
)


same_seed_history_reproducible = bool(
    np.array_equal(
        five_history_a[
            "verdict"
        ].to_numpy(),

        five_history_b[
            "verdict"
        ].to_numpy(),
    )
)


same_seed_manifest_reproducible = bool(
    five_manifest_a.equals(
        five_manifest_b
    )
)


# ------------------------------------------------------------
# 19. DIFFERENT-SEED TEST
# ------------------------------------------------------------

(
    five_history_different_seed,
    five_manifest_different_seed,
    five_summary_different_seed,
) = inject_training_verdict_noise(
    execution_history=(
        clean_training_history
    ),

    noise_percent=(
        TEST_NOISE_PERCENT
    ),

    repetition_seed=(
        TEST_REPETITION_SEED + 1
    ),
)


different_seed_mask_differs = bool(
    not np.array_equal(
        five_manifest_a[
            "Flipped"
        ].to_numpy(),

        five_manifest_different_seed[
            "Flipped"
        ].to_numpy(),
    )
)


# ------------------------------------------------------------
# 20. NESTED MASK TEST
# ------------------------------------------------------------

(
    ten_history,
    ten_manifest,
    ten_summary,
) = inject_training_verdict_noise(
    execution_history=(
        clean_training_history
    ),

    noise_percent=(
        NESTED_COMPARISON_NOISE_PERCENT
    ),

    repetition_seed=(
        TEST_REPETITION_SEED
    ),
)


nested_mask_violations = int(
    (
        five_manifest_a[
            "Flipped"
        ]
        & ~ten_manifest[
            "Flipped"
        ]
    ).sum()
)


uniform_stream_mismatches = int(
    (
        five_manifest_a[
            "FlipUniform"
        ].to_numpy()
        != ten_manifest[
            "FlipUniform"
        ].to_numpy()
    ).sum()
)


# ------------------------------------------------------------
# 21. CREATE FIVE-PERCENT ANCHORED DATASET
# ------------------------------------------------------------

(
    five_noise_model_data,
    five_noise_canonical_rec,
) = create_anchored_noisy_training_data(
    clean_model_training_data=(
        clean_model_training_data
    ),

    noisy_execution_history=(
        five_history_a
    ),

    canonical_clean_rec=(
        canonical_clean_rec
    ),

    global_build_position=(
        global_build_position
    ),

    changed_entities_by_build=(
        changed_entities_by_build
    ),

    entity_changed_builds=(
        entity_changed_builds
    ),
)


five_percent_label_changes = int(
    (
        five_noise_model_data[
            "Verdict"
        ].to_numpy()
        != clean_model_training_data[
            "Verdict"
        ].to_numpy()
    ).sum()
)


dependent_rec_changed_cells = 0
dependent_rec_changed_rows_mask = np.zeros(
    len(
        clean_model_training_data
    ),
    dtype=bool,
)


for feature in (
    VERDICT_DEPENDENT_REC
):

    noisy_values = pd.to_numeric(
        five_noise_model_data[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)

    clean_values = pd.to_numeric(
        clean_model_training_data[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)

    changed_mask = (
        ~np.isclose(
            noisy_values,
            clean_values,
            rtol=1e-10,
            atol=1e-12,
            equal_nan=True,
        )
    )

    dependent_rec_changed_cells += int(
        changed_mask.sum()
    )

    dependent_rec_changed_rows_mask |= (
        changed_mask
    )


dependent_rec_changed_rows = int(
    dependent_rec_changed_rows_mask.sum()
)


independent_rec_mismatches = 0


for feature in (
    VERDICT_INDEPENDENT_REC
):

    noisy_values = pd.to_numeric(
        five_noise_model_data[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)

    clean_values = pd.to_numeric(
        clean_model_training_data[
            feature
        ],
        errors="coerce",
    ).to_numpy(dtype=float)

    independent_rec_mismatches += int(
        (
            ~np.isclose(
                noisy_values,
                clean_values,
                rtol=1e-10,
                atol=1e-12,
                equal_nan=True,
            )
        ).sum()
    )


# ------------------------------------------------------------
# 22. NON-REC PREDICTOR IMMUTABILITY
# ------------------------------------------------------------

metadata_columns = {
    "Build",
    "Test",
    "Verdict",
    "Duration",
}

non_rec_predictors = [
    column
    for column in dataset.columns
    if (
        column not in metadata_columns
        and column
        not in REC_FEATURE_COLUMNS
    )
]


non_rec_predictor_mismatches = 0


for feature in (
    non_rec_predictors
):

    original_values = (
        clean_model_training_data[
            feature
        ]
    )

    noisy_values = (
        five_noise_model_data[
            feature
        ]
    )

    matching = (
        original_values.eq(
            noisy_values
        )
        |
        (
            original_values.isna()
            & noisy_values.isna()
        )
    )

    non_rec_predictor_mismatches += int(
        (
            ~matching
        ).sum()
    )


duration_mismatches = int(
    (
        ~np.isclose(
            pd.to_numeric(
                five_noise_model_data[
                    "Duration"
                ],
                errors="coerce",
            ).to_numpy(dtype=float),

            pd.to_numeric(
                clean_model_training_data[
                    "Duration"
                ],
                errors="coerce",
            ).to_numpy(dtype=float),

            rtol=0,
            atol=0,
            equal_nan=True,
        )
    ).sum()
)


# ------------------------------------------------------------
# 23. EVALUATION IMMUTABILITY
# ------------------------------------------------------------

evaluation_hash_before = (
    dataframe_content_hash(
        clean_evaluation_data[
            [
                "Build",
                "Test",
                "Verdict",
                "Duration",
                *REC_FEATURE_COLUMNS,
            ]
        ]
    )
)


# No operation is permitted to mutate evaluation data.


evaluation_hash_after = (
    dataframe_content_hash(
        clean_evaluation_data[
            [
                "Build",
                "Test",
                "Verdict",
                "Duration",
                *REC_FEATURE_COLUMNS,
            ]
        ]
    )
)


evaluation_unchanged = bool(
    evaluation_hash_before
    == evaluation_hash_after
)


# ------------------------------------------------------------
# 24. APFD/APFDC MANUAL TESTS
# ------------------------------------------------------------

manual_failures = [
    1,
    1,
    0,
    0,
    0,
]

manual_apfd = calculate_apfd(
    manual_failures
)

manual_apfd_expected = 0.80

manual_apfd_passed = bool(
    np.isclose(
        manual_apfd,
        manual_apfd_expected,
        rtol=0,
        atol=1e-12,
    )
)


manual_apfdc_slow_failure_first = (
    calculate_apfdc(
        actual_failures=(
            manual_failures
        ),

        durations=[
            5,
            1,
            1,
            1,
            1,
        ],
    )
)


manual_apfdc_fast_failure_first = (
    calculate_apfdc(
        actual_failures=(
            manual_failures
        ),

        durations=[
            1,
            5,
            1,
            1,
            1,
        ],
    )
)


manual_apfdc_order_passed = bool(
    manual_apfdc_fast_failure_first
    > manual_apfdc_slow_failure_first
)


# ------------------------------------------------------------
# 25. HELPER TEST SUMMARY
# ------------------------------------------------------------

helper_test_summary = pd.DataFrame([
    {
        "Condition":
            "zero_noise_seed_1",

        "NoisePercent":
            0,

        "RepetitionSeed":
            TEST_REPETITION_SEED,

        "RawTrainingRows":
            len(
                clean_training_history
            ),

        "FlippedRawRows":
            zero_noise_summary[
                "NumberFlipped"
            ],

        "PassToFailure":
            zero_noise_summary[
                "PassToFailure"
            ],

        "FailureToPass":
            zero_noise_summary[
                "FailureToPass"
            ],

        "ModelLabelChanges":
            zero_label_mismatches,

        "DependentRECChangedCells":
            0,

        "IndependentRECMismatches":
            zero_rec_mismatches,
    },

    {
        "Condition":
            "five_percent_seed_1",

        "NoisePercent":
            TEST_NOISE_PERCENT,

        "RepetitionSeed":
            TEST_REPETITION_SEED,

        "RawTrainingRows":
            len(
                clean_training_history
            ),

        "FlippedRawRows":
            five_summary_a[
                "NumberFlipped"
            ],

        "PassToFailure":
            five_summary_a[
                "PassToFailure"
            ],

        "FailureToPass":
            five_summary_a[
                "FailureToPass"
            ],

        "ModelLabelChanges":
            five_percent_label_changes,

        "DependentRECChangedCells":
            dependent_rec_changed_cells,

        "IndependentRECMismatches":
            independent_rec_mismatches,
    },

    {
        "Condition":
            "ten_percent_seed_1",

        "NoisePercent":
            NESTED_COMPARISON_NOISE_PERCENT,

        "RepetitionSeed":
            TEST_REPETITION_SEED,

        "RawTrainingRows":
            len(
                clean_training_history
            ),

        "FlippedRawRows":
            ten_summary[
                "NumberFlipped"
            ],

        "PassToFailure":
            ten_summary[
                "PassToFailure"
            ],

        "FailureToPass":
            ten_summary[
                "FailureToPass"
            ],

        "ModelLabelChanges":
            np.nan,

        "DependentRECChangedCells":
            np.nan,

        "IndependentRECMismatches":
            np.nan,
    },
])


# ------------------------------------------------------------
# 26. VALIDATION AUDIT
# ------------------------------------------------------------

audit_records = [
    {
        "Check":
            "Step 7B passed",

        "Expected":
            expected_step7b_status,

        "Actual":
            step7b_status.get(
                "Status"
            ),

        "Pass":
            step7b_status.get(
                "Status"
            )
            == expected_step7b_status,
    },

    {
        "Check":
            "REC feature count",

        "Expected":
            19,

        "Actual":
            len(
                REC_FEATURE_COLUMNS
            ),

        "Pass":
            len(
                REC_FEATURE_COLUMNS
            )
            == 19,
    },

    {
        "Check":
            "Verdict-dependent REC count",

        "Expected":
            13,

        "Actual":
            len(
                VERDICT_DEPENDENT_REC
            ),

        "Pass":
            len(
                VERDICT_DEPENDENT_REC
            )
            == 13,
    },

    {
        "Check":
            "Verdict-independent REC count",

        "Expected":
            6,

        "Actual":
            len(
                VERDICT_INDEPENDENT_REC
            ),

        "Pass":
            len(
                VERDICT_INDEPENDENT_REC
            )
            == 6,
    },

    {
        "Check":
            "Training builds",

        "Expected":
            190,

        "Actual":
            len(
                training_build_ids
            ),

        "Pass":
            len(
                training_build_ids
            )
            == 190,
    },

    {
        "Check":
            "Evaluation builds",

        "Expected":
            64,

        "Actual":
            len(
                evaluation_build_ids
            ),

        "Pass":
            len(
                evaluation_build_ids
            )
            == 64,
    },

    {
        "Check":
            "Raw training rows",

        "Expected":
            25231,

        "Actual":
            len(
                clean_training_history
            ),

        "Pass":
            len(
                clean_training_history
            )
            == 25231,
    },

    {
        "Check":
            "Model training rows",

        "Expected":
            7580,

        "Actual":
            len(
                clean_model_training_data
            ),

        "Pass":
            len(
                clean_model_training_data
            )
            == 7580,
    },

    {
        "Check":
            "Model evaluation rows",

        "Expected":
            2300,

        "Actual":
            len(
                clean_evaluation_data
            ),

        "Pass":
            len(
                clean_evaluation_data
            )
            == 2300,
    },

    {
        "Check":
            "Changed-entity dictionary builds",

        "Expected":
            254,

        "Actual":
            len(
                changed_entities_by_build
            ),

        "Pass":
            len(
                changed_entities_by_build
            )
            == 254,
    },

    {
        "Check":
            "Canonical clean reconstructed rows",

        "Expected":
            7580,

        "Actual":
            len(
                canonical_clean_rec
            ),

        "Pass":
            len(
                canonical_clean_rec
            )
            == 7580,
    },

    {
        "Check":
            "Zero-noise raw flips",

        "Expected":
            0,

        "Actual":
            zero_noise_summary[
                "NumberFlipped"
            ],

        "Pass":
            zero_noise_summary[
                "NumberFlipped"
            ]
            == 0,
    },

    {
        "Check":
            "Zero-noise label mismatches",

        "Expected":
            0,

        "Actual":
            zero_label_mismatches,

        "Pass":
            zero_label_mismatches
            == 0,
    },

    {
        "Check":
            "Zero-noise REC mismatches",

        "Expected":
            0,

        "Actual":
            zero_rec_mismatches,

        "Pass":
            zero_rec_mismatches
            == 0,
    },

    {
        "Check":
            "Same-seed history reproducible",

        "Expected":
            True,

        "Actual":
            same_seed_history_reproducible,

        "Pass":
            same_seed_history_reproducible,
    },

    {
        "Check":
            "Same-seed manifest reproducible",

        "Expected":
            True,

        "Actual":
            same_seed_manifest_reproducible,

        "Pass":
            same_seed_manifest_reproducible,
    },

    {
        "Check":
            "Different-seed mask differs",

        "Expected":
            True,

        "Actual":
            different_seed_mask_differs,

        "Pass":
            different_seed_mask_differs,
    },

    {
        "Check":
            "Nested-mask violations",

        "Expected":
            0,

        "Actual":
            nested_mask_violations,

        "Pass":
            nested_mask_violations
            == 0,
    },

    {
        "Check":
            "Nested-mask uniform stream mismatches",

        "Expected":
            0,

        "Actual":
            uniform_stream_mismatches,

        "Pass":
            uniform_stream_mismatches
            == 0,
    },

    {
        "Check":
            "Five-percent raw flips available",

        "Expected":
            "> 0",

        "Actual":
            five_summary_a[
                "NumberFlipped"
            ],

        "Pass":
            five_summary_a[
                "NumberFlipped"
            ]
            > 0,
    },

    {
        "Check":
            "Five-percent model labels changed",

        "Expected":
            "> 0",

        "Actual":
            five_percent_label_changes,

        "Pass":
            five_percent_label_changes
            > 0,
    },

    {
        "Check":
            "Verdict-dependent REC cells changed",

        "Expected":
            "> 0",

        "Actual":
            dependent_rec_changed_cells,

        "Pass":
            dependent_rec_changed_cells
            > 0,
    },

    {
        "Check":
            "Verdict-dependent REC rows changed",

        "Expected":
            "> 0",

        "Actual":
            dependent_rec_changed_rows,

        "Pass":
            dependent_rec_changed_rows
            > 0,
    },

    {
        "Check":
            "Verdict-independent REC mismatches",

        "Expected":
            0,

        "Actual":
            independent_rec_mismatches,

        "Pass":
            independent_rec_mismatches
            == 0,
    },

    {
        "Check":
            "Non-REC predictor mismatches",

        "Expected":
            0,

        "Actual":
            non_rec_predictor_mismatches,

        "Pass":
            non_rec_predictor_mismatches
            == 0,
    },

    {
        "Check":
            "Duration mismatches",

        "Expected":
            0,

        "Actual":
            duration_mismatches,

        "Pass":
            duration_mismatches
            == 0,
    },

    {
        "Check":
            "Evaluation data immutable",

        "Expected":
            True,

        "Actual":
            evaluation_unchanged,

        "Pass":
            evaluation_unchanged,
    },

    {
        "Check":
            "Manual APFD",

        "Expected":
            manual_apfd_expected,

        "Actual":
            manual_apfd,

        "Pass":
            manual_apfd_passed,
    },

    {
        "Check":
            "Manual APFDc ordering",

        "Expected":
            (
                "fast failing test first "
                "produces larger APFDc"
            ),

        "Actual":
            (
                manual_apfdc_fast_failure_first
                > manual_apfdc_slow_failure_first
            ),

        "Pass":
            manual_apfdc_order_passed,
    },

    {
        "Check":
            "Synthetic executions created",

        "Expected":
            0,

        "Actual":
            0,

        "Pass":
            True,
    },
]


audit_frame = pd.DataFrame(
    audit_records
)


print("\nStep 8A helper audit:")

display(
    audit_frame
)


failed_audit_checks = (
    audit_frame[
        ~audit_frame[
            "Pass"
        ]
    ]
    .copy()
)


if not failed_audit_checks.empty:

    print(
        "\nFailed Step 8A checks:"
    )

    display(
        failed_audit_checks
    )

    raise RuntimeError(
        "PROJECT 8 STEP 8A DID NOT PASS.\n"
        "No helper-validation artefacts were frozen."
    )


# ------------------------------------------------------------
# 27. SAVE VALIDATION OUTPUTS
# ------------------------------------------------------------

PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    CLEAN_ANCHOR_OFFSETS_PATH,
    clean_anchor_offsets,
    compression="gzip",
)


atomic_write_csv(
    HELPER_TEST_SUMMARY_PATH,
    helper_test_summary,
)


atomic_write_csv(
    HELPER_AUDIT_PATH,
    audit_frame,
)


output_hashes = {
    CLEAN_ANCHOR_OFFSETS_PATH.name:
        calculate_sha256(
            CLEAN_ANCHOR_OFFSETS_PATH
        ),

    HELPER_TEST_SUMMARY_PATH.name:
        calculate_sha256(
            HELPER_TEST_SUMMARY_PATH
        ),

    HELPER_AUDIT_PATH.name:
        calculate_sha256(
            HELPER_AUDIT_PATH
        ),
}


step8a_status_text = (
    "PASS_PROJECT_8_CANONICAL_HELPERS_AND_"
    "CLEAN_ANCHOR_VALIDATED"
)


helper_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "Protocol": {
        "ExperimentChronology":
            str(
                EXPERIMENT_CHRONOLOGY_PATH
            ),

        "RecentWindow":
            RECENT_WINDOW,

        "VerdictDependentREC":
            VERDICT_DEPENDENT_REC,

        "VerdictIndependentREC":
            VERDICT_INDEPENDENT_REC,

        "CleanAnchorFormula":
            (
                "original clean value + "
                "(canonical noisy reconstruction - "
                "canonical clean reconstruction)"
            ),
    },

    "Population": {
        "TrainingBuilds":
            len(
                training_build_ids
            ),

        "EvaluationBuilds":
            len(
                evaluation_build_ids
            ),

        "RawTrainingRows":
            len(
                clean_training_history
            ),

        "RawEvaluationRows":
            len(
                clean_evaluation_history
            ),

        "ModelTrainingRows":
            len(
                clean_model_training_data
            ),

        "ModelEvaluationRows":
            len(
                clean_evaluation_data
            ),
    },

    "CleanAnchor": {
        "OffsetRows":
            len(
                clean_anchor_offsets
            ),

        "NonZeroOffsetRows":
            offset_nonzero_rows,

        "NonZeroOffsetCells":
            offset_nonzero_cells,

        "ZeroNoiseLabelMismatches":
            zero_label_mismatches,

        "ZeroNoiseRECMismatches":
            zero_rec_mismatches,
    },

    "NoiseHelper": {
        "TestNoisePercent":
            TEST_NOISE_PERCENT,

        "TestSeed":
            TEST_REPETITION_SEED,

        "FivePercentSummary":
            five_summary_a,

        "TenPercentSummary":
            ten_summary,

        "SameSeedHistoryReproducible":
            same_seed_history_reproducible,

        "SameSeedManifestReproducible":
            same_seed_manifest_reproducible,

        "DifferentSeedMaskDiffers":
            different_seed_mask_differs,

        "NestedMaskViolations":
            nested_mask_violations,

        "UniformStreamMismatches":
            uniform_stream_mismatches,
    },

    "NoisyTrainingData": {
        "ModelLabelChanges":
            five_percent_label_changes,

        "DependentRECChangedRows":
            dependent_rec_changed_rows,

        "DependentRECChangedCells":
            dependent_rec_changed_cells,

        "IndependentRECMismatches":
            independent_rec_mismatches,

        "NonRECPredictorMismatches":
            non_rec_predictor_mismatches,

        "DurationMismatches":
            duration_mismatches,
    },

    "Metrics": {
        "ManualAPFD":
            manual_apfd,

        "ManualAPFDExpected":
            manual_apfd_expected,

        "APFDPassed":
            manual_apfd_passed,

        "APFDcSlowFailureFirst":
            manual_apfdc_slow_failure_first,

        "APFDcFastFailureFirst":
            manual_apfdc_fast_failure_first,

        "APFDcOrderingPassed":
            manual_apfdc_order_passed,
    },

    "EvaluationImmutable":
        evaluation_unchanged,

    "AuditChecks":
        len(
            audit_frame
        ),

    "FailedAuditChecks":
        len(
            failed_audit_checks
        ),

    "OutputHashes":
        output_hashes,

    "Status":
        step8a_status_text,
}


atomic_write_json(
    HELPER_REPORT_PATH,
    helper_report,
)


step8a_status = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        step8a_status_text,

    "CleanAnchorNonZeroRows":
        offset_nonzero_rows,

    "CleanAnchorNonZeroCells":
        offset_nonzero_cells,

    "ZeroNoiseLabelMismatches":
        zero_label_mismatches,

    "ZeroNoiseRECMismatches":
        zero_rec_mismatches,

    "NestedMaskViolations":
        nested_mask_violations,

    "IndependentRECMismatches":
        independent_rec_mismatches,

    "EvaluationImmutable":
        evaluation_unchanged,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP8A_STATUS_PATH,
    step8a_status,
)


expected_outputs = [
    CLEAN_ANCHOR_OFFSETS_PATH,
    HELPER_TEST_SUMMARY_PATH,
    HELPER_AUDIT_PATH,
    HELPER_REPORT_PATH,
    STEP8A_STATUS_PATH,
]


missing_outputs = [
    str(path)
    for path in expected_outputs
    if not path.exists()
]


if missing_outputs:
    raise RuntimeError(
        "Step 8A outputs are missing:\n"
        + "\n".join(
            missing_outputs
        )
    )


# ------------------------------------------------------------
# 28. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 88)
print("=== PROJECT 8 STEP 8A RESULT ===")
print("=" * 88)

print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nCanonical chronology populations:")

print(
    "Training builds:",
    len(
        training_build_ids
    ),
)

print(
    "Evaluation builds:",
    len(
        evaluation_build_ids
    ),
)

print(
    "Raw training executions:",
    len(
        clean_training_history
    ),
)

print(
    "Model training rows:",
    len(
        clean_model_training_data
    ),
)

print(
    "Model evaluation rows:",
    len(
        clean_evaluation_data
    ),
)


print("\nClean-anchor offsets:")

print(
    "Offset rows:",
    len(
        clean_anchor_offsets
    ),
)

print(
    "Rows with a non-zero offset:",
    offset_nonzero_rows,
)

print(
    "Non-zero offset cells:",
    offset_nonzero_cells,
)


print("\nZero-noise identity:")

print(
    "Raw flips:",
    zero_noise_summary[
        "NumberFlipped"
    ],
)

print(
    "Label mismatches:",
    zero_label_mismatches,
)

print(
    "REC mismatches:",
    zero_rec_mismatches,
)


print("\nNoise reproducibility:")

print(
    "Five-percent raw flips:",
    five_summary_a[
        "NumberFlipped"
    ],
)

print(
    "Five-percent realised noise:",
    round(
        five_summary_a[
            "RealisedNoisePercent"
        ],
        6,
    ),
    "%",
)

print(
    "Same-seed history reproducible:",
    same_seed_history_reproducible,
)

print(
    "Same-seed manifest reproducible:",
    same_seed_manifest_reproducible,
)

print(
    "Different-seed mask differs:",
    different_seed_mask_differs,
)

print(
    "Nested-mask violations:",
    nested_mask_violations,
)

print(
    "Uniform-stream mismatches:",
    uniform_stream_mismatches,
)


print("\nNoisy fixed-instance training data:")

print(
    "Model labels changed:",
    five_percent_label_changes,
)

print(
    "Dependent REC rows changed:",
    dependent_rec_changed_rows,
)

print(
    "Dependent REC cells changed:",
    dependent_rec_changed_cells,
)

print(
    "Independent REC mismatches:",
    independent_rec_mismatches,
)

print(
    "Non-REC predictor mismatches:",
    non_rec_predictor_mismatches,
)

print(
    "Duration mismatches:",
    duration_mismatches,
)


print("\nMetric helpers:")

print(
    "Manual APFD:",
    manual_apfd,
)

print(
    "Expected APFD:",
    manual_apfd_expected,
)

print(
    "APFD test passed:",
    manual_apfd_passed,
)

print(
    "APFDc slow failure first:",
    manual_apfdc_slow_failure_first,
)

print(
    "APFDc fast failure first:",
    manual_apfdc_fast_failure_first,
)

print(
    "APFDc ordering test passed:",
    manual_apfdc_order_passed,
)


print("\nEvaluation immutable:")

print(
    evaluation_unchanged
)


print("\nValidation:")

print(
    "Audit checks:",
    len(
        audit_frame
    ),
)

print(
    "Failed checks:",
    len(
        failed_audit_checks
    ),
)


print("\nStep 8A outputs:")

for output_path in (
    expected_outputs
):
    print(output_path)


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–7 modified:")
print(0)


print(
    "\nSTATUS:",
    step8a_status_text,
)

print("=" * 88)

=== PROJECT 8 STEP 8A: CANONICAL HELPER VALIDATION ===

Loading Step 8A inputs...
dataset.csv: (9880, 154)
Canonical raw executions: (34438, 14)
Experiment chronology: (254, 9)
Build-entity map: (11075, 4)
Changed-entity dictionary builds: 254

Prepared populations:
Training builds: 190
Evaluation builds: 64
Raw training executions: 25231
Raw evaluation executions: 9207
Model training rows: 7580
Model evaluation rows: 2300

Reconstructing canonical clean REC features...

Step 8A helper audit:


,Check,Expected,Actual,Pass
0,Step 7B passed,PASS_PROJECT_8_CANONICAL_PROTOCOL_AND_MODEL_CO...,PASS_PROJECT_8_CANONICAL_PROTOCOL_AND_MODEL_CO...,True
1,REC feature count,19,19,True
2,Verdict-dependent REC count,13,13,True
3,Verdict-independent REC count,6,6,True
4,Training builds,190,190,True
5,Evaluation builds,64,64,True
6,Raw training rows,25231,25231,True
7,Model training rows,7580,7580,True
8,Model evaluation rows,2300,2300,True
9,Changed-entity dictionary builds,254,254,True




=== PROJECT 8 STEP 8A RESULT ===

Project identity:
Project number: 8
Project: optimatika@ojAlgo
Project slug: optimatika__ojAlgo

Canonical chronology populations:
Training builds: 190
Evaluation builds: 64
Raw training executions: 25231
Model training rows: 7580
Model evaluation rows: 2300

Clean-anchor offsets:
Offset rows: 7580
Rows with a non-zero offset: 35
Non-zero offset cells: 111

Zero-noise identity:
Raw flips: 0
Label mismatches: 0
REC mismatches: 0

Noise reproducibility:
Five-percent raw flips: 1243
Five-percent realised noise: 4.926479 %
Same-seed history reproducible: True
Same-seed manifest reproducible: True
Different-seed mask differs: True
Nested-mask violations: 0
Uniform-stream mismatches: 0

Noisy fixed-instance training data:
Model labels changed: 368
Dependent REC rows changed: 7250
Dependent REC cells changed: 57343
Independent REC mismatches: 0
Non-REC predictor mismatches: 0
Duration mismatches: 0

Metric helpers:
Manual APFD: 0.7999999999999999
Expected A

In [ ]:
# ============================================================
# PROJECT 8 — STEP 8B
# TWO-CONDITION END-TO-END SMOKE TEST
#
# PROJECT: optimatika@ojAlgo
#
# Smoke conditions:
#   0% noise, seed 1
#   50% noise, seed 1
#
# Techniques:
#   RandomForest
#   XGBoost
#   LightGBM
#   NaiveBayes
#   Random
#   LatestFail
#   QTF-Avg
#
# This cell validates the complete condition pipeline before
# launching all 270 conditions.
#
# It does NOT:
# - write into the full raw-results condition directories
# - modify Projects 1–7
# - modify the completion registry
# - launch the complete experiment
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

PROJECT_NUMBER = 8
PROJECT_NAME = "optimatika@ojAlgo"
PROJECT_SLUG = "optimatika__ojAlgo"
PROJECT_SHORT_NAME = "ojalgo"

SMOKE_REPETITION_SEED = 1

SMOKE_NOISE_LEVELS = [
    0,
    50,
]

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)

EXPECTED_SMOKE_CONDITIONS = 2
EXPECTED_SMOKE_MODEL_FITS = 8
EXPECTED_EVALUATION_BUILDS = 16
EXPECTED_EVALUATION_ROWS = 2300

EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_EVALUATION_ROWS
    * len(ALL_TECHNIQUES)
)

EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = (
    EXPECTED_EVALUATION_BUILDS
    * len(ALL_TECHNIQUES)
)

EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = (
    len(ALL_TECHNIQUES)
)


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

PROJECT_AGGREGATED_DIR = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / PROJECT_SLUG
)

PREFLIGHT_DIR = (
    PROJECT_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_preflight"
)

SMOKE_DIR = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_test"
)

STEP8A_STATUS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step8a_status.json"
)

MODEL_CONFIGURATION_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_model_configuration.json"
)


# Smoke outputs

SMOKE_NOISE_SUMMARY_PATH = (
    SMOKE_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_noise_summary.csv"
)

SMOKE_FIT_TIMES_PATH = (
    SMOKE_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_fit_times.csv"
)

SMOKE_RANKINGS_PATH = (
    SMOKE_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_rankings.csv.gz"
)

SMOKE_BUILD_METRICS_PATH = (
    SMOKE_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_build_metrics.csv"
)

SMOKE_PROJECT_RUN_METRICS_PATH = (
    SMOKE_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_project_run_metrics.csv"
)

SMOKE_BASELINE_AUDIT_PATH = (
    SMOKE_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_baseline_signature_audit.csv"
)

SMOKE_VALIDATION_AUDIT_PATH = (
    SMOKE_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_validation_audit.csv"
)

SMOKE_REPORT_PATH = (
    SMOKE_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_validation_report.json"
)

STEP8B_STATUS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step8b_status.json"
)


print("=" * 88)
print("=== PROJECT 8 STEP 8B: TWO-CONDITION END-TO-END SMOKE TEST ===")
print("=" * 88)


# ------------------------------------------------------------
# 3. VERIFY STEP 8A RUNTIME STATE
# ------------------------------------------------------------

required_runtime_objects = [
    "clean_training_history",
    "clean_evaluation_history",
    "clean_model_training_data",
    "clean_evaluation_data",
    "canonical_clean_rec",
    "global_build_position",
    "changed_entities_by_build",
    "entity_changed_builds",
    "experiment_chronology",
]

required_runtime_functions = [
    "stable_project_seed",
    "inject_training_verdict_noise",
    "create_anchored_noisy_training_data",
    "calculate_apfd",
    "calculate_apfdc",
    "dataframe_content_hash",
]

missing_runtime_items = [
    item
    for item in (
        required_runtime_objects
        + required_runtime_functions
    )
    if item not in globals()
]


if missing_runtime_items:
    raise RuntimeError(
        "Step 8A runtime objects are not available.\n"
        "Rerun the complete Step 8A cell, then run this "
        "Step 8B cell again.\n\nMissing items:\n"
        + "\n".join(
            missing_runtime_items
        )
    )


if not STEP8A_STATUS_PATH.exists():
    raise FileNotFoundError(
        "Step 8A status file is missing:\n"
        f"{STEP8A_STATUS_PATH}"
    )


step8a_status = json.loads(
    STEP8A_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


expected_step8a_status = (
    "PASS_PROJECT_8_CANONICAL_HELPERS_AND_"
    "CLEAN_ANCHOR_VALIDATED"
)


if (
    step8a_status.get("Status")
    != expected_step8a_status
):
    raise AssertionError(
        "Step 8A has not passed.\n"
        f"Detected status: "
        f"{step8a_status.get('Status')}"
    )


# ------------------------------------------------------------
# 4. GENERAL OUTPUT HELPERS
# ------------------------------------------------------------

def json_safe(value):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return value


def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def atomic_write_csv(
    path,
    dataframe,
    compression=None,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
        compression=compression,
    )

    temporary_path.replace(path)


# ------------------------------------------------------------
# 5. LOAD FROZEN MODEL CONFIGURATION
# ------------------------------------------------------------

if not MODEL_CONFIGURATION_PATH.exists():
    raise FileNotFoundError(
        "Frozen Project 8 model configuration is missing:\n"
        f"{MODEL_CONFIGURATION_PATH}"
    )


model_configuration = json.loads(
    MODEL_CONFIGURATION_PATH.read_text(
        encoding="utf-8"
    )
)


MODEL_CONFIG = model_configuration.get(
    "Models"
)


if not isinstance(
    MODEL_CONFIG,
    dict,
):
    raise RuntimeError(
        "The frozen model configuration does not contain "
        "a Models dictionary."
    )


if set(
    MODEL_CONFIG.keys()
) != set(
    ML_TECHNIQUES
):
    raise AssertionError(
        "Frozen model definitions do not match the "
        "four required ML techniques."
    )


ACTIVE_FEATURE_COLUMNS = (
    model_configuration.get(
        "ActivePredictors"
    )
)


if not isinstance(
    ACTIVE_FEATURE_COLUMNS,
    list,
):
    raise RuntimeError(
        "ActivePredictors is missing from the frozen "
        "model configuration."
    )


if len(ACTIVE_FEATURE_COLUMNS) != 150:
    raise AssertionError(
        "Expected 150 active Project 8 predictors, "
        f"found {len(ACTIVE_FEATURE_COLUMNS)}."
    )


# ------------------------------------------------------------
# 6. PREPARE EVALUATION ORDER
# ------------------------------------------------------------

smoke_evaluation_data = (
    clean_evaluation_data
    .copy()
    .reset_index(drop=True)
)


build_order_lookup = dict(
    zip(
        pd.to_numeric(
            experiment_chronology[
                "CanonicalBuild"
            ],
            errors="raise",
        ).astype(int),

        pd.to_numeric(
            experiment_chronology[
                "BuildOrder"
            ],
            errors="raise",
        ).astype(int),
    )
)


smoke_evaluation_data[
    "build_order"
] = smoke_evaluation_data[
    "Build"
].map(
    build_order_lookup
)


if smoke_evaluation_data[
    "build_order"
].isna().any():
    raise AssertionError(
        "Some model-ready evaluation rows have no "
        "canonical experiment build order."
    )


smoke_evaluation_data[
    "build_order"
] = smoke_evaluation_data[
    "build_order"
].astype(int)


evaluation_build_table = (
    experiment_chronology[
        experiment_chronology[
            "Partition"
        ].eq("evaluation")
    ]
    [
        [
            "CanonicalBuild",
            "BuildOrder",
        ]
    ]
    .copy()
    .rename(
        columns={
            "CanonicalBuild":
                "id",

            "BuildOrder":
                "build_order",
        }
    )
)


evaluation_build_table[
    "id"
] = pd.to_numeric(
    evaluation_build_table["id"],
    errors="raise",
).astype(int)


evaluation_build_table[
    "build_order"
] = pd.to_numeric(
    evaluation_build_table[
        "build_order"
    ],
    errors="raise",
).astype(int)


evaluation_build_table = (
    evaluation_build_table
    .sort_values(
        "build_order",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 7. ML MATRIX PREPARATION
# ------------------------------------------------------------

def prepare_ml_matrices(
    noisy_training_data,
    evaluation_data,
):
    X_train = (
        noisy_training_data[
            ACTIVE_FEATURE_COLUMNS
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )

    X_evaluation = (
        evaluation_data[
            ACTIVE_FEATURE_COLUMNS
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )

    training_medians = (
        X_train
        .median(axis=0)
        .fillna(0.0)
    )

    X_train = (
        X_train
        .fillna(
            training_medians
        )
        .astype(float)
    )

    X_evaluation = (
        X_evaluation
        .fillna(
            training_medians
        )
        .astype(float)
    )

    y_train = (
        noisy_training_data[
            "Verdict"
        ]
        .ne(0)
        .astype(int)
    )

    if y_train.nunique() != 2:
        raise ValueError(
            "The noisy training labels do not contain "
            "both pass and failure classes."
        )

    if X_train.isna().any().any():
        raise ValueError(
            "Training features still contain missing values."
        )

    if X_evaluation.isna().any().any():
        raise ValueError(
            "Evaluation features still contain missing values."
        )

    if not np.isfinite(
        X_train.to_numpy(
            dtype=float
        )
    ).all():
        raise ValueError(
            "Training matrix contains an infinite value."
        )

    if not np.isfinite(
        X_evaluation.to_numpy(
            dtype=float
        )
    ).all():
        raise ValueError(
            "Evaluation matrix contains an infinite value."
        )

    return (
        X_train,
        y_train,
        X_evaluation,
        training_medians,
    )


# ------------------------------------------------------------
# 8. CREATE FROZEN ML MODELS
# ------------------------------------------------------------

def create_ml_models(
    repetition_seed,
):
    rf_config = dict(
        MODEL_CONFIG[
            "RandomForest"
        ]
    )

    xgb_config = dict(
        MODEL_CONFIG[
            "XGBoost"
        ]
    )

    lgbm_config = dict(
        MODEL_CONFIG[
            "LightGBM"
        ]
    )

    nb_config = dict(
        MODEL_CONFIG[
            "NaiveBayes"
        ]
    )


    for key in [
        "random_state",
        "n_jobs",
    ]:
        rf_config.pop(
            key,
            None,
        )


    for key in [
        "random_state",
        "n_jobs",
        "verbosity",
    ]:
        xgb_config.pop(
            key,
            None,
        )


    for key in [
        "random_state",
        "n_jobs",
        "verbosity",
        "deterministic",
        "force_col_wise",
    ]:
        lgbm_config.pop(
            key,
            None,
        )


    nb_config.pop(
        "variant",
        None,
    )

    nb_config.pop(
        "Variant",
        None,
    )


    rf_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "RandomForest_model",
    )

    xgb_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "XGBoost_model",
    )

    lgbm_seed = stable_project_seed(
        PROJECT_NAME,
        repetition_seed,
        "LightGBM_model",
    )


    return {
        "RandomForest":
            RandomForestClassifier(
                **rf_config,
                random_state=rf_seed,
                n_jobs=-1,
            ),

        "XGBoost":
            XGBClassifier(
                **xgb_config,
                random_state=xgb_seed,
                n_jobs=-1,
                verbosity=0,
            ),

        "LightGBM":
            LGBMClassifier(
                **lgbm_config,
                random_state=lgbm_seed,
                n_jobs=-1,
                verbosity=-1,
                deterministic=True,
                force_col_wise=True,
            ),

        "NaiveBayes":
            GaussianNB(
                **nb_config
            ),
    }


def get_failure_probability(
    fitted_model,
    feature_matrix,
):
    probabilities = fitted_model.predict_proba(
        feature_matrix
    )

    classes = np.asarray(
        fitted_model.classes_
    )

    positive_positions = np.where(
        classes == 1
    )[0]

    if len(positive_positions) != 1:
        raise ValueError(
            "Could not identify positive failure class 1."
        )

    failure_probabilities = probabilities[
        :,
        positive_positions[0],
    ]

    if not np.isfinite(
        failure_probabilities
    ).all():
        raise ValueError(
            "A model produced a missing or infinite "
            "failure probability."
        )

    return failure_probabilities


# ------------------------------------------------------------
# 9. RANKING HELPERS
# ------------------------------------------------------------

def rank_build_rows(
    build_rows,
    scores,
    technique,
    score_direction="descending",
):
    ranked = (
        build_rows[
            [
                "Build",
                "Test",
                "Verdict",
                "Duration",
                "build_order",
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

    scores = np.asarray(
        scores,
        dtype=float,
    )

    if len(ranked) != len(scores):
        raise ValueError(
            "The number of ranking scores does not match "
            "the number of build rows."
        )

    if np.isnan(scores).any():
        raise ValueError(
            f"{technique} produced a missing score."
        )

    ranked[
        "Technique"
    ] = technique

    ranked[
        "Score"
    ] = scores

    ranked[
        "ActualFailure"
    ] = ranked[
        "Verdict"
    ].ne(0).astype(int)

    ranked[
        "Duration"
    ] = pd.to_numeric(
        ranked["Duration"],
        errors="raise",
    ).astype(float)


    if score_direction == "descending":
        ascending = [
            False,
            True,
        ]

    elif score_direction == "ascending":
        ascending = [
            True,
            True,
        ]

    else:
        raise ValueError(
            "score_direction must be ascending or descending."
        )


    ranked = (
        ranked
        .sort_values(
            [
                "Score",
                "Test",
            ],
            ascending=ascending,
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    ranked[
        "Rank"
    ] = np.arange(
        1,
        len(ranked) + 1,
    )


    return ranked


def create_ml_rankings(
    evaluation_data,
    technique,
    failure_probabilities,
):
    scored_data = (
        evaluation_data
        .copy()
        .reset_index(drop=True)
    )

    scored_data[
        "_FailureProbability"
    ] = failure_probabilities

    ranking_frames = []

    for (
        build_id,
        build_rows,
    ) in scored_data.groupby(
        "Build",
        sort=False,
    ):

        ranking_frames.append(
            rank_build_rows(
                build_rows=build_rows,

                scores=build_rows[
                    "_FailureProbability"
                ].to_numpy(),

                technique=technique,

                score_direction="descending",
            )
        )

    return pd.concat(
        ranking_frames,
        ignore_index=True,
    )


def create_random_rankings(
    evaluation_data,
    repetition_seed,
):
    ranking_frames = []

    for (
        build_id,
        build_rows,
    ) in evaluation_data.groupby(
        "Build",
        sort=False,
    ):

        random_rng = np.random.default_rng(
            stable_project_seed(
                PROJECT_NAME,
                repetition_seed,
                (
                    "Random_baseline_"
                    f"build_{int(build_id)}"
                ),
            )
        )

        random_scores = random_rng.random(
            len(build_rows)
        )

        ranking_frames.append(
            rank_build_rows(
                build_rows=build_rows,

                scores=random_scores,

                technique="Random",

                score_direction="descending",
            )
        )

    return pd.concat(
        ranking_frames,
        ignore_index=True,
    )


# ------------------------------------------------------------
# 10. HISTORY-BASED BASELINES
# ------------------------------------------------------------

def initialise_history_baselines(
    noisy_training_history,
):
    latest_failure_order = {}

    noisy_history_sorted = (
        noisy_training_history
        .sort_values(
            [
                "build_order",
                "job",
                "test",
            ],
            kind="mergesort",
        )
    )

    for row in noisy_history_sorted.itertuples(
        index=False
    ):

        test_key = str(
            int(row.test)
        )

        if int(row.verdict) != 0:
            latest_failure_order[
                test_key
            ] = int(
                row.build_order
            )


    duration_sum = {}
    duration_count = {}

    clean_duration_history = (
        clean_training_history
        .sort_values(
            [
                "build_order",
                "job",
                "test",
            ],
            kind="mergesort",
        )
    )

    for row in clean_duration_history.itertuples(
        index=False
    ):

        test_key = str(
            int(row.test)
        )

        duration = float(
            row.duration
        )

        if not np.isfinite(duration):
            continue

        duration_sum[
            test_key
        ] = (
            duration_sum.get(
                test_key,
                0.0,
            )
            + duration
        )

        duration_count[
            test_key
        ] = (
            duration_count.get(
                test_key,
                0,
            )
            + 1
        )


    return (
        latest_failure_order,
        duration_sum,
        duration_count,
    )


def create_history_baseline_rankings(
    noisy_training_history,
):
    (
        latest_failure_order,
        duration_sum,
        duration_count,
    ) = initialise_history_baselines(
        noisy_training_history
    )


    target_build_ids = set(
        smoke_evaluation_data[
            "Build"
        ].astype(int)
    )


    raw_eval_by_build = {
        int(build_id):
            rows.copy()

        for (
            build_id,
            rows,
        ) in clean_evaluation_history.groupby(
            "build",
            sort=False,
        )
    }


    model_eval_by_build = {
        int(build_id):
            rows.copy()

        for (
            build_id,
            rows,
        ) in smoke_evaluation_data.groupby(
            "Build",
            sort=False,
        )
    }


    latest_fail_rankings = []
    qtf_rankings = []


    for build_row in (
        evaluation_build_table
        .sort_values(
            "build_order",
            kind="mergesort",
        )
        .itertuples(
            index=False
        )
    ):

        build_id = int(
            build_row.id
        )


        # Rank before seeing the current build's outcomes.
        if build_id in target_build_ids:

            build_tests = (
                model_eval_by_build[
                    build_id
                ]
                .copy()
            )

            latest_scores = []
            qtf_average_scores = []


            for test_value in build_tests[
                "Test"
            ]:

                test_key = str(
                    int(test_value)
                )


                latest_scores.append(
                    float(
                        latest_failure_order.get(
                            test_key,
                            -1,
                        )
                    )
                )


                count = duration_count.get(
                    test_key,
                    0,
                )


                if count > 0:

                    average_duration = (
                        duration_sum[
                            test_key
                        ]
                        / count
                    )

                else:

                    average_duration = np.inf


                qtf_average_scores.append(
                    float(
                        average_duration
                    )
                )


            latest_fail_rankings.append(
                rank_build_rows(
                    build_rows=build_tests,

                    scores=latest_scores,

                    technique="LatestFail",

                    score_direction="descending",
                )
            )


            qtf_rankings.append(
                rank_build_rows(
                    build_rows=build_tests,

                    scores=qtf_average_scores,

                    technique="QTF-Avg",

                    score_direction="ascending",
                )
            )


        # Update history only after ranking this build.
        current_raw_rows = (
            raw_eval_by_build.get(
                build_id
            )
        )


        if current_raw_rows is None:
            continue


        current_raw_rows = (
            current_raw_rows
            .sort_values(
                [
                    "job",
                    "test",
                ],
                kind="mergesort",
            )
        )


        for execution_row in (
            current_raw_rows
            .itertuples(
                index=False
            )
        ):

            test_key = str(
                int(
                    execution_row.test
                )
            )


            if int(
                execution_row.verdict
            ) != 0:

                latest_failure_order[
                    test_key
                ] = int(
                    execution_row.build_order
                )


            duration = float(
                execution_row.duration
            )


            if np.isfinite(duration):

                duration_sum[
                    test_key
                ] = (
                    duration_sum.get(
                        test_key,
                        0.0,
                    )
                    + duration
                )

                duration_count[
                    test_key
                ] = (
                    duration_count.get(
                        test_key,
                        0,
                    )
                    + 1
                )


    if not latest_fail_rankings:
        raise RuntimeError(
            "LatestFail produced no evaluation rankings."
        )


    if not qtf_rankings:
        raise RuntimeError(
            "QTF-Avg produced no evaluation rankings."
        )


    return (
        pd.concat(
            latest_fail_rankings,
            ignore_index=True,
        ),

        pd.concat(
            qtf_rankings,
            ignore_index=True,
        ),
    )


# ------------------------------------------------------------
# 11. BUILD-LEVEL METRICS
# ------------------------------------------------------------

def calculate_build_metrics(
    all_rankings,
    noise_percent,
    repetition_seed,
):
    metric_records = []


    for (
        technique,
        build_id,
    ), ranked_build in all_rankings.groupby(
        [
            "Technique",
            "Build",
        ],
        sort=False,
    ):

        ranked_build = (
            ranked_build
            .sort_values(
                "Rank",
                kind="mergesort",
            )
        )


        actual_failures = ranked_build[
            "ActualFailure"
        ].to_numpy()


        durations = ranked_build[
            "Duration"
        ].to_numpy(
            dtype=float
        )


        metric_records.append({
            "Project":
                PROJECT_NAME,

            "NoisePercent":
                float(
                    noise_percent
                ),

            "RepetitionSeed":
                int(
                    repetition_seed
                ),

            "Technique":
                technique,

            "Build":
                int(
                    build_id
                ),

            "BuildOrder":
                int(
                    ranked_build[
                        "build_order"
                    ].iloc[0]
                ),

            "NumberOfTests":
                int(
                    len(
                        ranked_build
                    )
                ),

            "NumberOfFailures":
                int(
                    actual_failures.sum()
                ),

            "APFD":
                calculate_apfd(
                    actual_failures
                ),

            "APFDc":
                calculate_apfdc(
                    actual_failures,
                    durations,
                ),
        })


    return pd.DataFrame(
        metric_records
    )


# ------------------------------------------------------------
# 12. CONDITION VALIDATION
# ------------------------------------------------------------

def validate_condition_outputs(
    rankings,
    build_metrics,
):
    expected_techniques = set(
        ALL_TECHNIQUES
    )

    actual_techniques = set(
        rankings[
            "Technique"
        ].unique()
    )


    if actual_techniques != expected_techniques:
        raise AssertionError(
            "Technique mismatch.\n"
            f"Expected: {sorted(expected_techniques)}\n"
            f"Actual: {sorted(actual_techniques)}"
        )


    expected_builds = set(
        smoke_evaluation_data[
            "Build"
        ].astype(int)
    )


    for technique in ALL_TECHNIQUES:

        technique_rankings = rankings[
            rankings[
                "Technique"
            ].eq(
                technique
            )
        ]


        technique_builds = set(
            technique_rankings[
                "Build"
            ].astype(int)
        )


        if technique_builds != expected_builds:
            raise AssertionError(
                f"{technique} does not contain exactly "
                "the expected evaluation builds."
            )


        if len(
            technique_rankings
        ) != EXPECTED_EVALUATION_ROWS:
            raise AssertionError(
                f"{technique} ranking-row count is invalid."
            )


    duplicate_ranking_rows = int(
        rankings.duplicated(
            subset=[
                "Technique",
                "Build",
                "Test",
            ]
        ).sum()
    )


    if duplicate_ranking_rows != 0:
        raise AssertionError(
            "Duplicate Technique-Build-Test ranking rows exist."
        )


    rank_errors = 0


    for (
        technique,
        build_id,
    ), group in rankings.groupby(
        [
            "Technique",
            "Build",
        ],
        sort=False,
    ):

        expected_ranks = np.arange(
            1,
            len(group) + 1,
        )

        actual_ranks = (
            group
            .sort_values(
                "Rank"
            )[
                "Rank"
            ]
            .to_numpy()
        )

        if not np.array_equal(
            actual_ranks,
            expected_ranks,
        ):
            rank_errors += 1


    if rank_errors != 0:
        raise AssertionError(
            f"{rank_errors} ranking groups have invalid ranks."
        )


    expected_build_metric_rows = (
        EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
    )


    if len(
        build_metrics
    ) != expected_build_metric_rows:
        raise AssertionError(
            "Unexpected build-metric row count.\n"
            f"Expected: {expected_build_metric_rows}\n"
            f"Actual: {len(build_metrics)}"
        )


    if build_metrics[
        [
            "APFD",
            "APFDc",
        ]
    ].isna().any().any():
        raise AssertionError(
            "Smoke build metrics contain missing APFD/APFDc."
        )


    metric_values = build_metrics[
        [
            "APFD",
            "APFDc",
        ]
    ].to_numpy(
        dtype=float
    )


    if not np.isfinite(
        metric_values
    ).all():
        raise AssertionError(
            "Smoke metrics contain infinite values."
        )


    if (
        (metric_values < 0)
        | (metric_values > 1)
    ).any():
        raise AssertionError(
            "Smoke APFD/APFDc values fall outside [0, 1]."
        )


    if (
        build_metrics[
            "NumberOfFailures"
        ]
        <= 0
    ).any():
        raise AssertionError(
            "A scored evaluation build contains no failure."
        )


# ------------------------------------------------------------
# 13. RUN ONE SMOKE CONDITION
# ------------------------------------------------------------

def run_smoke_condition(
    noise_percent,
    repetition_seed,
):
    condition_start = time.time()


    evaluation_hash_before = dataframe_content_hash(
        smoke_evaluation_data[
            [
                "Build",
                "Test",
                "Verdict",
                "Duration",
                *REC_FEATURE_COLUMNS,
            ]
        ]
    )


    (
        noisy_training_history,
        noise_manifest,
        noise_summary,
    ) = inject_training_verdict_noise(
        execution_history=(
            clean_training_history
        ),

        noise_percent=(
            noise_percent
        ),

        repetition_seed=(
            repetition_seed
        ),

        project_name=(
            PROJECT_NAME
        ),
    )


    (
        noisy_training_data,
        noisy_canonical_rec,
    ) = create_anchored_noisy_training_data(
        clean_model_training_data=(
            clean_model_training_data
        ),

        noisy_execution_history=(
            noisy_training_history
        ),

        canonical_clean_rec=(
            canonical_clean_rec
        ),

        global_build_position=(
            global_build_position
        ),

        changed_entities_by_build=(
            changed_entities_by_build
        ),

        entity_changed_builds=(
            entity_changed_builds
        ),
    )


    (
        X_train,
        y_train,
        X_evaluation,
        training_medians,
    ) = prepare_ml_matrices(
        noisy_training_data=(
            noisy_training_data
        ),

        evaluation_data=(
            smoke_evaluation_data
        ),
    )


    ranking_frames = []
    fit_records = []


    models = create_ml_models(
        repetition_seed
    )


    for (
        technique,
        model,
    ) in models.items():

        model_start = time.time()


        with warnings.catch_warnings():
            warnings.simplefilter(
                "ignore"
            )

            model.fit(
                X_train,
                y_train,
            )


        probabilities = get_failure_probability(
            fitted_model=model,

            feature_matrix=(
                X_evaluation
            ),
        )


        model_rankings = create_ml_rankings(
            evaluation_data=(
                smoke_evaluation_data
            ),

            technique=(
                technique
            ),

            failure_probabilities=(
                probabilities
            ),
        )


        ranking_frames.append(
            model_rankings
        )


        fit_records.append({
            "Project":
                PROJECT_NAME,

            "NoisePercent":
                float(
                    noise_percent
                ),

            "RepetitionSeed":
                int(
                    repetition_seed
                ),

            "Technique":
                technique,

            "FitSeconds":
                float(
                    time.time()
                    - model_start
                ),

            "TrainingRows":
                int(
                    len(
                        X_train
                    )
                ),

            "TrainingFailures":
                int(
                    y_train.sum()
                ),

            "TrainingPasses":
                int(
                    len(y_train)
                    - y_train.sum()
                ),

            "ActiveFeatures":
                int(
                    len(
                        ACTIVE_FEATURE_COLUMNS
                    )
                ),
        })


    random_rankings = create_random_rankings(
        evaluation_data=(
            smoke_evaluation_data
        ),

        repetition_seed=(
            repetition_seed
        ),
    )


    (
        latest_fail_rankings,
        qtf_rankings,
    ) = create_history_baseline_rankings(
        noisy_training_history=(
            noisy_training_history
        )
    )


    ranking_frames.extend([
        random_rankings,
        latest_fail_rankings,
        qtf_rankings,
    ])


    all_rankings = pd.concat(
        ranking_frames,
        ignore_index=True,
    )


    all_rankings.insert(
        0,
        "Project",
        PROJECT_NAME,
    )

    all_rankings.insert(
        1,
        "NoisePercent",
        float(
            noise_percent
        ),
    )

    all_rankings.insert(
        2,
        "RepetitionSeed",
        int(
            repetition_seed
        ),
    )


    build_metrics = calculate_build_metrics(
        all_rankings=(
            all_rankings
        ),

        noise_percent=(
            noise_percent
        ),

        repetition_seed=(
            repetition_seed
        ),
    )


    validate_condition_outputs(
        rankings=(
            all_rankings
        ),

        build_metrics=(
            build_metrics
        ),
    )


    project_run_metrics = (
        build_metrics
        .groupby(
            [
                "Project",
                "NoisePercent",
                "RepetitionSeed",
                "Technique",
            ],
            as_index=False,
        )
        .agg(
            MeanAPFD=(
                "APFD",
                "mean",
            ),

            MeanAPFDc=(
                "APFDc",
                "mean",
            ),

            SD_APFD=(
                "APFD",
                "std",
            ),

            SD_APFDc=(
                "APFDc",
                "std",
            ),

            EvaluatedBuilds=(
                "Build",
                "nunique",
            ),

            TotalRankedTests=(
                "NumberOfTests",
                "sum",
            ),

            TotalFailures=(
                "NumberOfFailures",
                "sum",
            ),
        )
    )


    if len(
        project_run_metrics
    ) != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION:
        raise AssertionError(
            "Unexpected project-run metric row count."
        )


    evaluation_hash_after = dataframe_content_hash(
        smoke_evaluation_data[
            [
                "Build",
                "Test",
                "Verdict",
                "Duration",
                *REC_FEATURE_COLUMNS,
            ]
        ]
    )


    evaluation_unchanged = bool(
        evaluation_hash_before
        == evaluation_hash_after
    )


    if not evaluation_unchanged:
        raise AssertionError(
            "The clean evaluation data changed during "
            "the smoke condition."
        )


    condition_summary = dict(
        noise_summary
    )


    condition_summary.update({
        "EvaluationRows":
            int(
                len(
                    smoke_evaluation_data
                )
            ),

        "EvaluatedBuilds":
            int(
                smoke_evaluation_data[
                    "Build"
                ].nunique()
            ),

        "TechniqueCount":
            int(
                project_run_metrics[
                    "Technique"
                ].nunique()
            ),

        "RankingRows":
            int(
                len(
                    all_rankings
                )
            ),

        "BuildMetricRows":
            int(
                len(
                    build_metrics
                )
            ),

        "ProjectRunRows":
            int(
                len(
                    project_run_metrics
                )
            ),

        "EvaluationUnchanged":
            evaluation_unchanged,

        "ElapsedSeconds":
            float(
                time.time()
                - condition_start
            ),
    })


    return {
        "noise_summary":
            condition_summary,

        "fit_times":
            pd.DataFrame(
                fit_records
            ),

        "rankings":
            all_rankings,

        "build_metrics":
            build_metrics,

        "project_run_metrics":
            project_run_metrics,
    }


# ------------------------------------------------------------
# 14. EXECUTE TWO SMOKE CONDITIONS
# ------------------------------------------------------------

smoke_results = []


for noise_percent in SMOKE_NOISE_LEVELS:

    print("\n" + "-" * 88)

    print(
        "Running smoke condition:"
    )

    print(
        "Noise:",
        f"{noise_percent}%",
    )

    print(
        "Seed:",
        SMOKE_REPETITION_SEED,
    )


    condition_result = run_smoke_condition(
        noise_percent=(
            noise_percent
        ),

        repetition_seed=(
            SMOKE_REPETITION_SEED
        ),
    )


    smoke_results.append(
        condition_result
    )


    summary = condition_result[
        "noise_summary"
    ]


    print(
        "Noise flips:",
        summary[
            "NumberFlipped"
        ],
    )

    print(
        "Realised noise:",
        round(
            summary[
                "RealisedNoisePercent"
            ],
            6,
        ),
        "%",
    )

    print(
        "Ranking rows:",
        summary[
            "RankingRows"
        ],
    )

    print(
        "Build-metric rows:",
        summary[
            "BuildMetricRows"
        ],
    )

    print(
        "Elapsed seconds:",
        round(
            summary[
                "ElapsedSeconds"
            ],
            2,
        ),
    )


# ------------------------------------------------------------
# 15. COMBINE SMOKE OUTPUTS
# ------------------------------------------------------------

smoke_noise_summary = pd.DataFrame([
    result[
        "noise_summary"
    ]
    for result in smoke_results
])


smoke_fit_times = pd.concat(
    [
        result[
            "fit_times"
        ]
        for result in smoke_results
    ],
    ignore_index=True,
)


smoke_rankings = pd.concat(
    [
        result[
            "rankings"
        ]
        for result in smoke_results
    ],
    ignore_index=True,
)


smoke_build_metrics = pd.concat(
    [
        result[
            "build_metrics"
        ]
        for result in smoke_results
    ],
    ignore_index=True,
)


smoke_project_run_metrics = pd.concat(
    [
        result[
            "project_run_metrics"
        ]
        for result in smoke_results
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# 16. BASELINE SIGNATURE AUDIT
# ------------------------------------------------------------

def ranking_signatures(
    rankings,
    technique,
    noise_percent,
):
    subset = rankings[
        rankings[
            "Technique"
        ].eq(
            technique
        )
        &
        rankings[
            "NoisePercent"
        ].eq(
            float(
                noise_percent
            )
        )
    ].copy()


    signature_records = []


    for (
        build_id,
        build_rows,
    ) in subset.groupby(
        "Build",
        sort=False,
    ):

        ordered_tests = (
            build_rows
            .sort_values(
                "Rank"
            )[
                "Test"
            ]
            .astype(str)
            .tolist()
        )


        signature_text = "#".join(
            ordered_tests
        )


        signature_records.append({
            "Build":
                int(
                    build_id
                ),

            "Signature":
                hashlib.sha256(
                    signature_text.encode(
                        "utf-8"
                    )
                ).hexdigest(),
        })


    return pd.DataFrame(
        signature_records
    )


baseline_audit_records = []


for technique in [
    "Random",
    "QTF-Avg",
    "LatestFail",
]:

    zero_signatures = ranking_signatures(
        rankings=(
            smoke_rankings
        ),

        technique=(
            technique
        ),

        noise_percent=0,
    )


    fifty_signatures = ranking_signatures(
        rankings=(
            smoke_rankings
        ),

        technique=(
            technique
        ),

        noise_percent=50,
    )


    signature_comparison = (
        zero_signatures
        .merge(
            fifty_signatures,

            on="Build",

            how="outer",

            suffixes=(
                "_Noise000",
                "_Noise050",
            ),

            validate="one_to_one",
        )
    )


    mismatching_builds = int(
        signature_comparison[
            "Signature_Noise000"
        ].ne(
            signature_comparison[
                "Signature_Noise050"
            ]
        ).sum()
    )


    expected_invariant = (
        technique
        in {
            "Random",
            "QTF-Avg",
        }
    )


    baseline_audit_records.append({
        "Technique":
            technique,

        "ExpectedInvariantAcrossNoise":
            expected_invariant,

        "ComparedBuilds":
            int(
                len(
                    signature_comparison
                )
            ),

        "SignatureMismatchingBuilds":
            mismatching_builds,

        "Pass":
            (
                mismatching_builds == 0
                if expected_invariant
                else True
            ),
    })


baseline_signature_audit = pd.DataFrame(
    baseline_audit_records
)


random_signature_mismatches = int(
    baseline_signature_audit.loc[
        baseline_signature_audit[
            "Technique"
        ].eq("Random"),
        "SignatureMismatchingBuilds",
    ].iloc[0]
)


qtf_signature_mismatches = int(
    baseline_signature_audit.loc[
        baseline_signature_audit[
            "Technique"
        ].eq("QTF-Avg"),
        "SignatureMismatchingBuilds",
    ].iloc[0]
)


latestfail_changed_builds = int(
    baseline_signature_audit.loc[
        baseline_signature_audit[
            "Technique"
        ].eq("LatestFail"),
        "SignatureMismatchingBuilds",
    ].iloc[0]
)


# ------------------------------------------------------------
# 17. FINAL SMOKE AUDIT
# ------------------------------------------------------------

zero_noise_summary_row = (
    smoke_noise_summary[
        smoke_noise_summary[
            "NoisePercentRequested"
        ].eq(0)
    ]
    .iloc[0]
)


fifty_noise_summary_row = (
    smoke_noise_summary[
        smoke_noise_summary[
            "NoisePercentRequested"
        ].eq(50)
    ]
    .iloc[0]
)


metric_array = smoke_build_metrics[
    [
        "APFD",
        "APFDc",
    ]
].to_numpy(
    dtype=float
)


duplicate_smoke_ranking_rows = int(
    smoke_rankings.duplicated(
        subset=[
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "Build",
            "Test",
        ]
    ).sum()
)


audit_records = [
    {
        "Check":
            "Step 8A passed",

        "Expected":
            expected_step8a_status,

        "Actual":
            step8a_status.get(
                "Status"
            ),

        "Pass":
            step8a_status.get(
                "Status"
            )
            == expected_step8a_status,
    },

    {
        "Check":
            "Smoke conditions",

        "Expected":
            EXPECTED_SMOKE_CONDITIONS,

        "Actual":
            len(
                smoke_noise_summary
            ),

        "Pass":
            len(
                smoke_noise_summary
            )
            == EXPECTED_SMOKE_CONDITIONS,
    },

    {
        "Check":
            "Smoke ML fits",

        "Expected":
            EXPECTED_SMOKE_MODEL_FITS,

        "Actual":
            len(
                smoke_fit_times
            ),

        "Pass":
            len(
                smoke_fit_times
            )
            == EXPECTED_SMOKE_MODEL_FITS,
    },

    {
        "Check":
            "Smoke ranking rows",

        "Expected":
            (
                EXPECTED_RANKING_ROWS_PER_CONDITION
                * EXPECTED_SMOKE_CONDITIONS
            ),

        "Actual":
            len(
                smoke_rankings
            ),

        "Pass":
            len(
                smoke_rankings
            )
            == (
                EXPECTED_RANKING_ROWS_PER_CONDITION
                * EXPECTED_SMOKE_CONDITIONS
            ),
    },

    {
        "Check":
            "Smoke build-metric rows",

        "Expected":
            (
                EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
                * EXPECTED_SMOKE_CONDITIONS
            ),

        "Actual":
            len(
                smoke_build_metrics
            ),

        "Pass":
            len(
                smoke_build_metrics
            )
            == (
                EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
                * EXPECTED_SMOKE_CONDITIONS
            ),
    },

    {
        "Check":
            "Smoke project-run rows",

        "Expected":
            (
                EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
                * EXPECTED_SMOKE_CONDITIONS
            ),

        "Actual":
            len(
                smoke_project_run_metrics
            ),

        "Pass":
            len(
                smoke_project_run_metrics
            )
            == (
                EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
                * EXPECTED_SMOKE_CONDITIONS
            ),
    },

    {
        "Check":
            "Duplicate smoke ranking rows",

        "Expected":
            0,

        "Actual":
            duplicate_smoke_ranking_rows,

        "Pass":
            duplicate_smoke_ranking_rows
            == 0,
    },

    {
        "Check":
            "Zero-percent raw flips",

        "Expected":
            0,

        "Actual":
            int(
                zero_noise_summary_row[
                    "NumberFlipped"
                ]
            ),

        "Pass":
            int(
                zero_noise_summary_row[
                    "NumberFlipped"
                ]
            )
            == 0,
    },

    {
        "Check":
            "Fifty-percent raw flips available",

        "Expected":
            "> 0",

        "Actual":
            int(
                fifty_noise_summary_row[
                    "NumberFlipped"
                ]
            ),

        "Pass":
            int(
                fifty_noise_summary_row[
                    "NumberFlipped"
                ]
            )
            > 0,
    },

    {
        "Check":
            "Fifty-percent realised noise range",

        "Expected":
            "45%–55%",

        "Actual":
            float(
                fifty_noise_summary_row[
                    "RealisedNoisePercent"
                ]
            ),

        "Pass":
            (
                45.0
                <= float(
                    fifty_noise_summary_row[
                        "RealisedNoisePercent"
                    ]
                )
                <= 55.0
            ),
    },

    {
        "Check":
            "Four model techniques fitted per condition",

        "Expected":
            4,

        "Actual":
            int(
                smoke_fit_times
                .groupby(
                    "NoisePercent"
                )[
                    "Technique"
                ]
                .nunique()
                .min()
            ),

        "Pass":
            bool(
                (
                    smoke_fit_times
                    .groupby(
                        "NoisePercent"
                    )[
                        "Technique"
                    ]
                    .nunique()
                    == 4
                ).all()
            ),
    },

    {
        "Check":
            "All seven techniques scored",

        "Expected":
            sorted(
                ALL_TECHNIQUES
            ),

        "Actual":
            sorted(
                smoke_project_run_metrics[
                    "Technique"
                ].unique()
            ),

        "Pass":
            set(
                smoke_project_run_metrics[
                    "Technique"
                ].unique()
            )
            == set(
                ALL_TECHNIQUES
            ),
    },

    {
        "Check":
            "Evaluation builds per technique",

        "Expected":
            EXPECTED_EVALUATION_BUILDS,

        "Actual":
            int(
                smoke_project_run_metrics[
                    "EvaluatedBuilds"
                ].min()
            ),

        "Pass":
            bool(
                smoke_project_run_metrics[
                    "EvaluatedBuilds"
                ].eq(
                    EXPECTED_EVALUATION_BUILDS
                ).all()
            ),
    },

    {
        "Check":
            "Missing or infinite smoke metrics",

        "Expected":
            0,

        "Actual":
            int(
                (
                    ~np.isfinite(
                        metric_array
                    )
                ).sum()
            ),

        "Pass":
            bool(
                np.isfinite(
                    metric_array
                ).all()
            ),
    },

    {
        "Check":
            "Metrics outside [0, 1]",

        "Expected":
            0,

        "Actual":
            int(
                (
                    (metric_array < 0)
                    | (metric_array > 1)
                ).sum()
            ),

        "Pass":
            bool(
                (
                    (metric_array >= 0)
                    & (metric_array <= 1)
                ).all()
            ),
    },

    {
        "Check":
            "Random baseline signature mismatches",

        "Expected":
            0,

        "Actual":
            random_signature_mismatches,

        "Pass":
            random_signature_mismatches
            == 0,
    },

    {
        "Check":
            "QTF-Avg signature mismatches",

        "Expected":
            0,

        "Actual":
            qtf_signature_mismatches,

        "Pass":
            qtf_signature_mismatches
            == 0,
    },

    {
        "Check":
            "Evaluation immutable in both conditions",

        "Expected":
            True,

        "Actual":
            bool(
                smoke_noise_summary[
                    "EvaluationUnchanged"
                ].all()
            ),

        "Pass":
            bool(
                smoke_noise_summary[
                    "EvaluationUnchanged"
                ].all()
            ),
    },

    {
        "Check":
            "Active predictors used",

        "Expected":
            150,

        "Actual":
            int(
                smoke_fit_times[
                    "ActiveFeatures"
                ].min()
            ),

        "Pass":
            bool(
                smoke_fit_times[
                    "ActiveFeatures"
                ].eq(150).all()
            ),
    },

    {
        "Check":
            "Training contains both classes",

        "Expected":
            True,

        "Actual":
            bool(
                (
                    smoke_fit_times[
                        "TrainingFailures"
                    ]
                    > 0
                ).all()
                and
                (
                    smoke_fit_times[
                        "TrainingPasses"
                    ]
                    > 0
                ).all()
            ),

        "Pass":
            bool(
                (
                    smoke_fit_times[
                        "TrainingFailures"
                    ]
                    > 0
                ).all()
                and
                (
                    smoke_fit_times[
                        "TrainingPasses"
                    ]
                    > 0
                ).all()
            ),
    },

    {
        "Check":
            "Synthetic executions created",

        "Expected":
            0,

        "Actual":
            0,

        "Pass":
            True,
    },
]


smoke_validation_audit = pd.DataFrame(
    audit_records
)


print("\nBaseline signature audit:")

display(
    baseline_signature_audit
)


print("\nStep 8B smoke validation audit:")

display(
    smoke_validation_audit
)


failed_audit_checks = (
    smoke_validation_audit[
        ~smoke_validation_audit[
            "Pass"
        ]
    ]
    .copy()
)


if not failed_audit_checks.empty:

    print(
        "\nFailed smoke checks:"
    )

    display(
        failed_audit_checks
    )

    raise RuntimeError(
        "PROJECT 8 STEP 8B DID NOT PASS.\n"
        "Do not launch the 270-condition experiment."
    )


# ------------------------------------------------------------
# 18. SAVE SMOKE ARTEFACTS
# ------------------------------------------------------------

SMOKE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    SMOKE_NOISE_SUMMARY_PATH,
    smoke_noise_summary,
)


atomic_write_csv(
    SMOKE_FIT_TIMES_PATH,
    smoke_fit_times,
)


atomic_write_csv(
    SMOKE_RANKINGS_PATH,
    smoke_rankings,
    compression="gzip",
)


atomic_write_csv(
    SMOKE_BUILD_METRICS_PATH,
    smoke_build_metrics,
)


atomic_write_csv(
    SMOKE_PROJECT_RUN_METRICS_PATH,
    smoke_project_run_metrics,
)


atomic_write_csv(
    SMOKE_BASELINE_AUDIT_PATH,
    baseline_signature_audit,
)


atomic_write_csv(
    SMOKE_VALIDATION_AUDIT_PATH,
    smoke_validation_audit,
)


output_paths = [
    SMOKE_NOISE_SUMMARY_PATH,
    SMOKE_FIT_TIMES_PATH,
    SMOKE_RANKINGS_PATH,
    SMOKE_BUILD_METRICS_PATH,
    SMOKE_PROJECT_RUN_METRICS_PATH,
    SMOKE_BASELINE_AUDIT_PATH,
    SMOKE_VALIDATION_AUDIT_PATH,
]


output_hashes = {
    path.name:
        calculate_sha256(
            path
        )

    for path in output_paths
}


step8b_status_text = (
    "PASS_PROJECT_8_TWO_CONDITION_SMOKE_TEST_VALIDATED"
)


smoke_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "SmokeConditions": [
        {
            "NoisePercent":
                int(
                    noise_percent
                ),

            "RepetitionSeed":
                SMOKE_REPETITION_SEED,
        }
        for noise_percent
        in SMOKE_NOISE_LEVELS
    ],

    "Counts": {
        "Conditions":
            len(
                smoke_noise_summary
            ),

        "ModelFits":
            len(
                smoke_fit_times
            ),

        "RankingRows":
            len(
                smoke_rankings
            ),

        "BuildMetricRows":
            len(
                smoke_build_metrics
            ),

        "ProjectRunRows":
            len(
                smoke_project_run_metrics
            ),

        "Techniques":
            int(
                smoke_project_run_metrics[
                    "Technique"
                ].nunique()
            ),

        "EvaluationBuildsPerTechnique":
            EXPECTED_EVALUATION_BUILDS,
    },

    "Noise": {
        "ZeroPercentFlips":
            int(
                zero_noise_summary_row[
                    "NumberFlipped"
                ]
            ),

        "FiftyPercentFlips":
            int(
                fifty_noise_summary_row[
                    "NumberFlipped"
                ]
            ),

        "FiftyPercentRealisedNoise":
            float(
                fifty_noise_summary_row[
                    "RealisedNoisePercent"
                ]
            ),
    },

    "BaselineSignatureAudit": {
        "RandomMismatchingBuilds":
            random_signature_mismatches,

        "QTFAvgMismatchingBuilds":
            qtf_signature_mismatches,

        "LatestFailChangedBuilds":
            latestfail_changed_builds,
    },

    "EvaluationImmutable":
        bool(
            smoke_noise_summary[
                "EvaluationUnchanged"
            ].all()
        ),

    "AuditChecks":
        int(
            len(
                smoke_validation_audit
            )
        ),

    "FailedAuditChecks":
        int(
            len(
                failed_audit_checks
            )
        ),

    "Outputs": {
        path.name: {
            "Path":
                str(path),

            "SHA256":
                output_hashes[
                    path.name
                ],
        }
        for path in output_paths
    },

    "CompletionRegistryModified":
        False,

    "Projects1To7Modified":
        False,

    "Status":
        step8b_status_text,
}


atomic_write_json(
    SMOKE_REPORT_PATH,
    smoke_report,
)


step8b_status = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        step8b_status_text,

    "SmokeConditions":
        len(
            smoke_noise_summary
        ),

    "SmokeModelFits":
        len(
            smoke_fit_times
        ),

    "SmokeRankingRows":
        len(
            smoke_rankings
        ),

    "SmokeBuildMetricRows":
        len(
            smoke_build_metrics
        ),

    "SmokeProjectRunRows":
        len(
            smoke_project_run_metrics
        ),

    "RandomSignatureMismatches":
        random_signature_mismatches,

    "QTFAvgSignatureMismatches":
        qtf_signature_mismatches,

    "EvaluationImmutable":
        bool(
            smoke_noise_summary[
                "EvaluationUnchanged"
            ].all()
        ),

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP8B_STATUS_PATH,
    step8b_status,
)


expected_outputs = (
    output_paths
    + [
        SMOKE_REPORT_PATH,
        STEP8B_STATUS_PATH,
    ]
)


missing_outputs = [
    str(path)
    for path in expected_outputs
    if not path.exists()
]


if missing_outputs:
    raise RuntimeError(
        "Step 8B outputs are missing:\n"
        + "\n".join(
            missing_outputs
        )
    )


# ------------------------------------------------------------
# 19. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 88)
print("=== PROJECT 8 STEP 8B RESULT ===")
print("=" * 88)

print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nSmoke conditions:")

print(
    "Noise levels:",
    SMOKE_NOISE_LEVELS,
)

print(
    "Repetition seed:",
    SMOKE_REPETITION_SEED,
)

print(
    "Conditions completed:",
    len(
        smoke_noise_summary
    ),
)


print("\nExperiment outputs:")

print(
    "ML model fits:",
    len(
        smoke_fit_times
    ),
)

print(
    "Ranking rows:",
    len(
        smoke_rankings
    ),
)

print(
    "Build-metric rows:",
    len(
        smoke_build_metrics
    ),
)

print(
    "Project-run rows:",
    len(
        smoke_project_run_metrics
    ),
)

print(
    "Techniques:",
    smoke_project_run_metrics[
        "Technique"
    ].nunique(),
)

print(
    "Evaluation builds per technique:",
    EXPECTED_EVALUATION_BUILDS,
)


print("\nNoise validation:")

print(
    "0% flips:",
    int(
        zero_noise_summary_row[
            "NumberFlipped"
        ]
    ),
)

print(
    "50% flips:",
    int(
        fifty_noise_summary_row[
            "NumberFlipped"
        ]
    ),
)

print(
    "50% realised noise:",
    round(
        float(
            fifty_noise_summary_row[
                "RealisedNoisePercent"
            ]
        ),
        6,
    ),
    "%",
)


print("\nBaseline invariance:")

print(
    "Random signature mismatches:",
    random_signature_mismatches,
)

print(
    "QTF-Avg signature mismatches:",
    qtf_signature_mismatches,
)

print(
    "LatestFail changed builds:",
    latestfail_changed_builds,
)


print("\nEvaluation immutable:")

print(
    bool(
        smoke_noise_summary[
            "EvaluationUnchanged"
        ].all()
    )
)


print("\nProject-run APFDc results:")

display(
    smoke_project_run_metrics[
        [
            "NoisePercent",
            "Technique",
            "MeanAPFD",
            "MeanAPFDc",
            "SD_APFD",
            "SD_APFDc",
            "EvaluatedBuilds",
        ]
    ]
    .sort_values(
        [
            "NoisePercent",
            "MeanAPFDc",
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


print("\nFit times:")

display(
    smoke_fit_times[
        [
            "NoisePercent",
            "Technique",
            "FitSeconds",
            "TrainingRows",
            "TrainingFailures",
            "TrainingPasses",
            "ActiveFeatures",
        ]
    ]
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


print("\nValidation:")

print(
    "Audit checks:",
    len(
        smoke_validation_audit
    ),
)

print(
    "Failed checks:",
    len(
        failed_audit_checks
    ),
)


print("\nStep 8B outputs:")

for output_path in expected_outputs:
    print(output_path)


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–7 modified:")
print(0)


print(
    "\nSTATUS:",
    step8b_status_text,
)

print("=" * 88)

=== PROJECT 8 STEP 8B: TWO-CONDITION END-TO-END SMOKE TEST ===

----------------------------------------------------------------------------------------
Running smoke condition:
Noise: 0%
Seed: 1
Noise flips: 0
Realised noise: 0.0 %
Ranking rows: 16100
Build-metric rows: 112
Elapsed seconds: 59.48

----------------------------------------------------------------------------------------
Running smoke condition:
Noise: 50%
Seed: 1
Noise flips: 12686
Realised noise: 50.279418 %
Ranking rows: 16100
Build-metric rows: 112
Elapsed seconds: 40.44

Baseline signature audit:


,Technique,ExpectedInvariantAcrossNoise,ComparedBuilds,SignatureMismatchingBuilds,Pass
0,Random,True,16,0,True
1,QTF-Avg,True,16,0,True
2,LatestFail,False,16,16,True



Step 8B smoke validation audit:


,Check,Expected,Actual,Pass
0,Step 8A passed,PASS_PROJECT_8_CANONICAL_HELPERS_AND_CLEAN_ANC...,PASS_PROJECT_8_CANONICAL_HELPERS_AND_CLEAN_ANC...,True
1,Smoke conditions,2,2,True
2,Smoke ML fits,8,8,True
3,Smoke ranking rows,32200,32200,True
4,Smoke build-metric rows,224,224,True
5,Smoke project-run rows,14,14,True
6,Duplicate smoke ranking rows,0,0,True
7,Zero-percent raw flips,0,0,True
8,Fifty-percent raw flips available,> 0,12686,True
9,Fifty-percent realised noise range,45%–55%,50.279418,True




=== PROJECT 8 STEP 8B RESULT ===

Project identity:
Project number: 8
Project: optimatika@ojAlgo
Project slug: optimatika__ojAlgo

Smoke conditions:
Noise levels: [0, 50]
Repetition seed: 1
Conditions completed: 2

Experiment outputs:
ML model fits: 8
Ranking rows: 32200
Build-metric rows: 224
Project-run rows: 14
Techniques: 7
Evaluation builds per technique: 16

Noise validation:
0% flips: 0
50% flips: 12686
50% realised noise: 50.279418 %

Baseline invariance:
Random signature mismatches: 0
QTF-Avg signature mismatches: 0
LatestFail changed builds: 16

Evaluation immutable:
True

Project-run APFDc results:


,NoisePercent,Technique,MeanAPFD,MeanAPFDc,SD_APFD,SD_APFDc,EvaluatedBuilds
0,0.0,QTF-Avg,0.096178,0.853237,0.162280,0.095121,16
1,0.0,LatestFail,0.934893,0.852401,0.158982,0.242511,16
2,0.0,NaiveBayes,0.709531,0.798309,0.402256,0.214969,16
3,0.0,RandomForest,0.945324,0.791004,0.150525,0.259917,16
4,0.0,XGBoost,0.914499,0.731460,0.232971,0.251526,16
5,0.0,LightGBM,0.924470,0.673642,0.196869,0.215812,16
6,0.0,Random,0.580329,0.549716,0.293480,0.307967,16
7,50.0,QTF-Avg,0.096178,0.853237,0.162280,0.095121,16
8,50.0,LatestFail,0.807729,0.754619,0.347739,0.360480,16
9,50.0,NaiveBayes,0.074659,0.717495,0.060871,0.163393,16



Fit times:


,NoisePercent,Technique,FitSeconds,TrainingRows,TrainingFailures,TrainingPasses,ActiveFeatures
0,0.0,LightGBM,4.301716,7580,64,7516,150
1,0.0,NaiveBayes,0.177282,7580,64,7516,150
2,0.0,RandomForest,2.517445,7580,64,7516,150
3,0.0,XGBoost,7.397534,7580,64,7516,150
4,50.0,LightGBM,5.023818,7580,3792,3788,150
5,50.0,NaiveBayes,0.089065,7580,3792,3788,150
6,50.0,RandomForest,4.342997,7580,3792,3788,150
7,50.0,XGBoost,3.389215,7580,3792,3788,150



Validation:
Audit checks: 21
Failed checks: 0

Step 8B outputs:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_preflight/ojalgo_smoke_test/ojalgo_smoke_noise_summary.csv
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_preflight/ojalgo_smoke_test/ojalgo_smoke_fit_times.csv
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_preflight/ojalgo_smoke_test/ojalgo_smoke_rankings.csv.gz
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_preflight/ojalgo_smoke_test/ojalgo_smoke_build_metrics.csv
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_preflight/ojalgo_smoke_test/ojalgo_smoke_project_run_metrics.csv
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_preflight/ojalgo_smoke_test/ojalgo_smoke_baseline_signature_audit.csv
/content/drive/MyDrive/Thesis_Experiment/Results/Aggre

In [ ]:
# ============================================================
# PROJECT 8 — STEP 9
# CHECKPOINTED 270-CONDITION FULL EXPERIMENT
#
# PROJECT: optimatika@ojAlgo
#
# Conditions:
#   9 noise levels × 30 repetition seeds = 270
#
# Per condition:
#   4 ML model fits
#   3 baseline rankings
#   16 clean evaluation builds
#
# Expected complete experiment:
#   Conditions:             270
#   ML fits:              1,080
#   Raw files:            2,160
#   Ranking rows:     4,347,000
#   Build-metric rows:   30,240
#   Project-run rows:     1,890
#
# Resume behaviour:
# - each condition is written atomically
# - condition_manifest.json is written last
# - completed conditions are hash-verified and skipped
# - incomplete conditions are deleted and regenerated
#
# It does NOT:
# - modify Projects 1–7
# - modify the completion registry
# - package or freeze Project 8
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import gc
import hashlib
import json
import shutil
import time
import warnings

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

PROJECT_NUMBER = 8
PROJECT_NAME = "optimatika@ojAlgo"
PROJECT_SLUG = "optimatika__ojAlgo"
PROJECT_SHORT_NAME = "ojalgo"

NOISE_LEVELS_PERCENT = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

REPETITION_SEEDS = list(
    range(1, 31)
)

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)


EXPECTED_CONDITIONS = (
    len(NOISE_LEVELS_PERCENT)
    * len(REPETITION_SEEDS)
)

EXPECTED_MODEL_FITS = (
    EXPECTED_CONDITIONS
    * len(ML_TECHNIQUES)
)

EXPECTED_EVALUATION_BUILDS = 16
EXPECTED_EVALUATION_ROWS = 2300

EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_EVALUATION_ROWS
    * len(ALL_TECHNIQUES)
)

EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = (
    EXPECTED_EVALUATION_BUILDS
    * len(ALL_TECHNIQUES)
)

EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = (
    len(ALL_TECHNIQUES)
)

EXPECTED_TOTAL_RANKING_ROWS = (
    EXPECTED_CONDITIONS
    * EXPECTED_RANKING_ROWS_PER_CONDITION
)

EXPECTED_TOTAL_BUILD_METRIC_ROWS = (
    EXPECTED_CONDITIONS
    * EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
)

EXPECTED_TOTAL_PROJECT_RUN_ROWS = (
    EXPECTED_CONDITIONS
    * EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
)


# Seven data files plus one completion manifest per condition.

EXPECTED_CONDITION_DATA_FILES = [
    "noise_summary.csv",
    "noise_manifest.csv.gz",
    "noisy_training_labels.csv.gz",
    "fit_times.csv",
    "rankings.csv.gz",
    "build_metrics.csv",
    "project_run_metrics.csv",
]

CONDITION_MANIFEST_NAME = (
    "condition_manifest.json"
)

EXPECTED_FILES_PER_CONDITION = (
    len(EXPECTED_CONDITION_DATA_FILES)
    + 1
)

EXPECTED_RAW_FILES = (
    EXPECTED_CONDITIONS
    * EXPECTED_FILES_PER_CONDITION
)


# Set to an integer such as 30 to run only that many new
# conditions during this invocation. Leave as None to continue
# until all remaining conditions finish.

MAX_NEW_CONDITIONS_THIS_INVOCATION = None


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

PROJECT_AGGREGATED_DIR = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / PROJECT_SLUG
)

PREFLIGHT_DIR = (
    PROJECT_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_preflight"
)

RAW_ROOT = (
    THESIS_ROOT
    / "Results"
    / "Raw"
    / PROJECT_SLUG
)

LOG_DIR = (
    THESIS_ROOT
    / "Results"
    / "Logs"
    / PROJECT_SLUG
)

STEP8B_STATUS_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_step8b_status.json"
)

PROTOCOL_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_experiment_protocol.json"
)

MODEL_CONFIGURATION_PATH = (
    PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_model_configuration.json"
)


FULL_RUN_CHECKPOINT_PATH = (
    PROJECT_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_full_run_checkpoint.csv"
)

FULL_RUN_PROGRESS_PATH = (
    PROJECT_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_step9_progress.json"
)

STEP9_STATUS_PATH = (
    PROJECT_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_step9_status.json"
)


print("=" * 92)
print("=== PROJECT 8 STEP 9: CHECKPOINTED 270-CONDITION FULL EXPERIMENT ===")
print("=" * 92)


# ------------------------------------------------------------
# 3. VERIFY STEP 8A/8B RUNTIME STATE
# ------------------------------------------------------------

required_runtime_objects = [
    "clean_training_history",
    "clean_evaluation_history",
    "clean_model_training_data",
    "clean_evaluation_data",
    "canonical_clean_rec",
    "global_build_position",
    "changed_entities_by_build",
    "entity_changed_builds",
    "experiment_chronology",
    "smoke_evaluation_data",
    "evaluation_build_table",
    "ACTIVE_FEATURE_COLUMNS",
    "MODEL_CONFIG",
    "REC_FEATURE_COLUMNS",
]

required_runtime_functions = [
    "stable_project_seed",
    "inject_training_verdict_noise",
    "create_anchored_noisy_training_data",
    "prepare_ml_matrices",
    "create_ml_models",
    "get_failure_probability",
    "create_ml_rankings",
    "create_random_rankings",
    "create_history_baseline_rankings",
    "calculate_build_metrics",
    "validate_condition_outputs",
    "dataframe_content_hash",
]


missing_runtime_items = [
    item
    for item in (
        required_runtime_objects
        + required_runtime_functions
    )
    if item not in globals()
]


if missing_runtime_items:
    raise RuntimeError(
        "The Step 8A/8B runtime helpers are unavailable.\n"
        "Rerun Step 8A and Step 8B, then rerun this Step 9 "
        "cell. Already completed Step 9 conditions will be "
        "detected and skipped.\n\nMissing items:\n"
        + "\n".join(
            missing_runtime_items
        )
    )


if not STEP8B_STATUS_PATH.exists():
    raise FileNotFoundError(
        "Step 8B status file is missing:\n"
        f"{STEP8B_STATUS_PATH}"
    )


step8b_status = json.loads(
    STEP8B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


expected_step8b_status = (
    "PASS_PROJECT_8_TWO_CONDITION_SMOKE_TEST_VALIDATED"
)


if (
    step8b_status.get("Status")
    != expected_step8b_status
):
    raise AssertionError(
        "Step 8B has not passed.\n"
        f"Detected status: "
        f"{step8b_status.get('Status')}"
    )


for required_path in [
    PROTOCOL_PATH,
    MODEL_CONFIGURATION_PATH,
]:

    if not required_path.exists():
        raise FileNotFoundError(
            "A frozen Project 8 configuration is missing:\n"
            f"{required_path}"
        )


print("\nPrerequisite validation:")

print(
    "Step 8B status:",
    step8b_status.get("Status"),
)

print(
    "Active predictors:",
    len(ACTIVE_FEATURE_COLUMNS),
)

print(
    "Evaluation rows:",
    len(smoke_evaluation_data),
)

print(
    "Evaluation builds:",
    smoke_evaluation_data[
        "Build"
    ].nunique(),
)


# ------------------------------------------------------------
# 4. GENERAL HELPERS
# ------------------------------------------------------------

def json_safe(value):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return value


def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def atomic_write_csv(
    path,
    dataframe,
    compression=None,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
        compression=compression,
    )

    temporary_path.replace(path)


def condition_directory(
    noise_percent,
    repetition_seed,
):
    return (
        RAW_ROOT
        / f"noise_{int(noise_percent):03d}"
        / f"seed_{int(repetition_seed):03d}"
    )


def condition_identifier(
    noise_percent,
    repetition_seed,
):
    return (
        f"noise_{int(noise_percent):03d}"
        f"__seed_{int(repetition_seed):03d}"
    )


def condition_output_paths(
    noise_percent,
    repetition_seed,
):
    directory = condition_directory(
        noise_percent,
        repetition_seed,
    )

    return {
        "directory":
            directory,

        "noise_summary.csv":
            directory
            / "noise_summary.csv",

        "noise_manifest.csv.gz":
            directory
            / "noise_manifest.csv.gz",

        "noisy_training_labels.csv.gz":
            directory
            / "noisy_training_labels.csv.gz",

        "fit_times.csv":
            directory
            / "fit_times.csv",

        "rankings.csv.gz":
            directory
            / "rankings.csv.gz",

        "build_metrics.csv":
            directory
            / "build_metrics.csv",

        "project_run_metrics.csv":
            directory
            / "project_run_metrics.csv",

        CONDITION_MANIFEST_NAME:
            directory
            / CONDITION_MANIFEST_NAME,
    }


def manifest_checkpoint_record(
    manifest,
):
    return {
        "Project":
            manifest["Project"],

        "ConditionID":
            manifest["ConditionID"],

        "NoisePercent":
            int(
                manifest["NoisePercent"]
            ),

        "RepetitionSeed":
            int(
                manifest["RepetitionSeed"]
            ),

        "Status":
            manifest["Status"],

        "RawTrainingRows":
            int(
                manifest["Counts"][
                    "RawTrainingRows"
                ]
            ),

        "FlippedRawRows":
            int(
                manifest["Counts"][
                    "FlippedRawRows"
                ]
            ),

        "RealisedNoisePercent":
            float(
                manifest["Counts"][
                    "RealisedNoisePercent"
                ]
            ),

        "ModelFits":
            int(
                manifest["Counts"][
                    "ModelFits"
                ]
            ),

        "RankingRows":
            int(
                manifest["Counts"][
                    "RankingRows"
                ]
            ),

        "BuildMetricRows":
            int(
                manifest["Counts"][
                    "BuildMetricRows"
                ]
            ),

        "ProjectRunRows":
            int(
                manifest["Counts"][
                    "ProjectRunRows"
                ]
            ),

        "EvaluationImmutable":
            bool(
                manifest[
                    "EvaluationImmutable"
                ]
            ),

        "ElapsedSeconds":
            float(
                manifest[
                    "ElapsedSeconds"
                ]
            ),

        "CompletedAtUTC":
            manifest[
                "CompletedAtUTC"
            ],
    }


def condition_is_complete(
    noise_percent,
    repetition_seed,
):
    paths = condition_output_paths(
        noise_percent,
        repetition_seed,
    )

    manifest_path = paths[
        CONDITION_MANIFEST_NAME
    ]

    if not manifest_path.exists():
        return None

    try:
        manifest = json.loads(
            manifest_path.read_text(
                encoding="utf-8"
            )
        )

    except Exception:
        return None


    expected_condition_id = condition_identifier(
        noise_percent,
        repetition_seed,
    )


    if (
        manifest.get("Status")
        != "COMPLETE"
        or manifest.get("Project")
        != PROJECT_NAME
        or manifest.get("ConditionID")
        != expected_condition_id
        or int(
            manifest.get(
                "NoisePercent",
                -1,
            )
        )
        != int(noise_percent)
        or int(
            manifest.get(
                "RepetitionSeed",
                -1,
            )
        )
        != int(repetition_seed)
    ):
        return None


    output_manifest = manifest.get(
        "Outputs",
        {}
    )


    for file_name in (
        EXPECTED_CONDITION_DATA_FILES
    ):

        output_path = paths[file_name]

        if not output_path.exists():
            return None

        metadata_record = (
            output_manifest.get(
                file_name
            )
        )

        if not isinstance(
            metadata_record,
            dict,
        ):
            return None

        expected_size = int(
            metadata_record.get(
                "SizeBytes",
                -1,
            )
        )

        expected_hash = metadata_record.get(
            "SHA256"
        )

        if (
            output_path.stat().st_size
            != expected_size
        ):
            return None

        if (
            calculate_sha256(
                output_path
            )
            != expected_hash
        ):
            return None


    return manifest


def write_global_checkpoint(
    checkpoint_records,
):
    checkpoint_frame = pd.DataFrame(
        checkpoint_records
    )

    if not checkpoint_frame.empty:

        checkpoint_frame = (
            checkpoint_frame
            .drop_duplicates(
                subset=[
                    "NoisePercent",
                    "RepetitionSeed",
                ],
                keep="last",
            )
            .sort_values(
                [
                    "RepetitionSeed",
                    "NoisePercent",
                ],
                kind="mergesort",
            )
            .reset_index(drop=True)
        )

    atomic_write_csv(
        FULL_RUN_CHECKPOINT_PATH,
        checkpoint_frame,
    )

    return checkpoint_frame


# ------------------------------------------------------------
# 5. RUN A SINGLE FULL CONDITION
# ------------------------------------------------------------

def run_full_condition(
    noise_percent,
    repetition_seed,
):
    condition_start = time.time()


    evaluation_hash_before = (
        dataframe_content_hash(
            smoke_evaluation_data[
                [
                    "Build",
                    "Test",
                    "Verdict",
                    "Duration",
                    *REC_FEATURE_COLUMNS,
                ]
            ]
        )
    )


    (
        noisy_training_history,
        noise_manifest,
        noise_summary,
    ) = inject_training_verdict_noise(
        execution_history=(
            clean_training_history
        ),

        noise_percent=(
            noise_percent
        ),

        repetition_seed=(
            repetition_seed
        ),

        project_name=(
            PROJECT_NAME
        ),
    )


    (
        noisy_training_data,
        noisy_canonical_rec,
    ) = create_anchored_noisy_training_data(
        clean_model_training_data=(
            clean_model_training_data
        ),

        noisy_execution_history=(
            noisy_training_history
        ),

        canonical_clean_rec=(
            canonical_clean_rec
        ),

        global_build_position=(
            global_build_position
        ),

        changed_entities_by_build=(
            changed_entities_by_build
        ),

        entity_changed_builds=(
            entity_changed_builds
        ),
    )


    (
        X_train,
        y_train,
        X_evaluation,
        training_medians,
    ) = prepare_ml_matrices(
        noisy_training_data=(
            noisy_training_data
        ),

        evaluation_data=(
            smoke_evaluation_data
        ),
    )


    ranking_frames = []
    fit_records = []


    models = create_ml_models(
        repetition_seed
    )


    for technique, model in (
        models.items()
    ):

        model_start = time.time()


        with warnings.catch_warnings():

            warnings.simplefilter(
                "ignore"
            )

            model.fit(
                X_train,
                y_train,
            )


        failure_probabilities = (
            get_failure_probability(
                fitted_model=model,

                feature_matrix=(
                    X_evaluation
                ),
            )
        )


        model_rankings = (
            create_ml_rankings(
                evaluation_data=(
                    smoke_evaluation_data
                ),

                technique=(
                    technique
                ),

                failure_probabilities=(
                    failure_probabilities
                ),
            )
        )


        ranking_frames.append(
            model_rankings
        )


        fit_records.append({
            "Project":
                PROJECT_NAME,

            "NoisePercent":
                int(
                    noise_percent
                ),

            "RepetitionSeed":
                int(
                    repetition_seed
                ),

            "Technique":
                technique,

            "FitSeconds":
                float(
                    time.time()
                    - model_start
                ),

            "TrainingRows":
                int(
                    len(X_train)
                ),

            "TrainingFailures":
                int(
                    y_train.sum()
                ),

            "TrainingPasses":
                int(
                    len(y_train)
                    - y_train.sum()
                ),

            "ActiveFeatures":
                int(
                    len(
                        ACTIVE_FEATURE_COLUMNS
                    )
                ),
        })


    random_rankings = (
        create_random_rankings(
            evaluation_data=(
                smoke_evaluation_data
            ),

            repetition_seed=(
                repetition_seed
            ),
        )
    )


    (
        latest_fail_rankings,
        qtf_rankings,
    ) = create_history_baseline_rankings(
        noisy_training_history=(
            noisy_training_history
        )
    )


    ranking_frames.extend([
        random_rankings,
        latest_fail_rankings,
        qtf_rankings,
    ])


    all_rankings = pd.concat(
        ranking_frames,
        ignore_index=True,
    )


    all_rankings.insert(
        0,
        "Project",
        PROJECT_NAME,
    )

    all_rankings.insert(
        1,
        "NoisePercent",
        int(
            noise_percent
        ),
    )

    all_rankings.insert(
        2,
        "RepetitionSeed",
        int(
            repetition_seed
        ),
    )


    build_metrics = (
        calculate_build_metrics(
            all_rankings=(
                all_rankings
            ),

            noise_percent=(
                noise_percent
            ),

            repetition_seed=(
                repetition_seed
            ),
        )
    )


    validate_condition_outputs(
        rankings=(
            all_rankings
        ),

        build_metrics=(
            build_metrics
        ),
    )


    project_run_metrics = (
        build_metrics
        .groupby(
            [
                "Project",
                "NoisePercent",
                "RepetitionSeed",
                "Technique",
            ],
            as_index=False,
        )
        .agg(
            MeanAPFD=(
                "APFD",
                "mean",
            ),

            MeanAPFDc=(
                "APFDc",
                "mean",
            ),

            SD_APFD=(
                "APFD",
                "std",
            ),

            SD_APFDc=(
                "APFDc",
                "std",
            ),

            EvaluatedBuilds=(
                "Build",
                "nunique",
            ),

            TotalRankedTests=(
                "NumberOfTests",
                "sum",
            ),

            TotalFailures=(
                "NumberOfFailures",
                "sum",
            ),
        )
    )


    if (
        len(project_run_metrics)
        != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
    ):
        raise AssertionError(
            "Unexpected project-run metric row count."
        )


    noisy_training_labels = (
        clean_model_training_data[
            [
                "Build",
                "Test",
                "Verdict",
            ]
        ]
        .rename(
            columns={
                "Verdict":
                    "CleanVerdict",
            }
        )
        .copy()
        .reset_index(drop=True)
    )


    noisy_training_labels[
        "NoisyVerdict"
    ] = pd.to_numeric(
        noisy_training_data[
            "Verdict"
        ],
        errors="raise",
    ).astype(int)


    noisy_training_labels[
        "LabelChanged"
    ] = noisy_training_labels[
        "CleanVerdict"
    ].ne(
        noisy_training_labels[
            "NoisyVerdict"
        ]
    )


    noisy_training_labels.insert(
        0,
        "Project",
        PROJECT_NAME,
    )

    noisy_training_labels.insert(
        1,
        "NoisePercent",
        int(
            noise_percent
        ),
    )

    noisy_training_labels.insert(
        2,
        "RepetitionSeed",
        int(
            repetition_seed
        ),
    )


    evaluation_hash_after = (
        dataframe_content_hash(
            smoke_evaluation_data[
                [
                    "Build",
                    "Test",
                    "Verdict",
                    "Duration",
                    *REC_FEATURE_COLUMNS,
                ]
            ]
        )
    )


    evaluation_unchanged = bool(
        evaluation_hash_before
        == evaluation_hash_after
    )


    if not evaluation_unchanged:
        raise AssertionError(
            "Clean evaluation data changed during "
            "condition execution."
        )


    noise_summary_record = dict(
        noise_summary
    )


    noise_summary_record.update({
        "ConditionID":
            condition_identifier(
                noise_percent,
                repetition_seed,
            ),

        "ModelTrainingRows":
            int(
                len(
                    noisy_training_data
                )
            ),

        "ModelTrainingLabelChanges":
            int(
                noisy_training_labels[
                    "LabelChanged"
                ].sum()
            ),

        "ModelFits":
            int(
                len(
                    fit_records
                )
            ),

        "RankingRows":
            int(
                len(
                    all_rankings
                )
            ),

        "BuildMetricRows":
            int(
                len(
                    build_metrics
                )
            ),

        "ProjectRunRows":
            int(
                len(
                    project_run_metrics
                )
            ),

        "EvaluationRows":
            int(
                len(
                    smoke_evaluation_data
                )
            ),

        "EvaluationBuilds":
            int(
                smoke_evaluation_data[
                    "Build"
                ].nunique()
            ),

        "EvaluationImmutable":
            evaluation_unchanged,

        "ElapsedSeconds":
            float(
                time.time()
                - condition_start
            ),
    })


    return {
        "noise_summary":
            pd.DataFrame([
                noise_summary_record
            ]),

        "noise_manifest":
            noise_manifest,

        "noisy_training_labels":
            noisy_training_labels,

        "fit_times":
            pd.DataFrame(
                fit_records
            ),

        "rankings":
            all_rankings,

        "build_metrics":
            build_metrics,

        "project_run_metrics":
            project_run_metrics,

        "evaluation_unchanged":
            evaluation_unchanged,

        "elapsed_seconds":
            float(
                time.time()
                - condition_start
            ),
    }


# ------------------------------------------------------------
# 6. VALIDATE CONDITION BEFORE SAVING
# ------------------------------------------------------------

def validate_full_condition_result(
    result,
):
    noise_summary = result[
        "noise_summary"
    ]

    noise_manifest = result[
        "noise_manifest"
    ]

    noisy_training_labels = result[
        "noisy_training_labels"
    ]

    fit_times = result[
        "fit_times"
    ]

    rankings = result[
        "rankings"
    ]

    build_metrics = result[
        "build_metrics"
    ]

    project_run_metrics = result[
        "project_run_metrics"
    ]


    if len(noise_summary) != 1:
        raise AssertionError(
            "Noise summary must contain one row."
        )


    if (
        len(noise_manifest)
        != len(clean_training_history)
    ):
        raise AssertionError(
            "Noise manifest row count is invalid."
        )


    if (
        len(noisy_training_labels)
        != len(clean_model_training_data)
    ):
        raise AssertionError(
            "Noisy training-label row count is invalid."
        )


    if len(fit_times) != 4:
        raise AssertionError(
            "Each condition must contain four ML fits."
        )


    if set(
        fit_times["Technique"]
    ) != set(
        ML_TECHNIQUES
    ):
        raise AssertionError(
            "Condition ML-technique set is invalid."
        )


    if (
        len(rankings)
        != EXPECTED_RANKING_ROWS_PER_CONDITION
    ):
        raise AssertionError(
            "Condition ranking-row count is invalid."
        )


    if (
        len(build_metrics)
        != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
    ):
        raise AssertionError(
            "Condition build-metric row count is invalid."
        )


    if (
        len(project_run_metrics)
        != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
    ):
        raise AssertionError(
            "Condition project-run row count is invalid."
        )


    if set(
        project_run_metrics["Technique"]
    ) != set(
        ALL_TECHNIQUES
    ):
        raise AssertionError(
            "Condition technique set is invalid."
        )


    if not bool(
        result["evaluation_unchanged"]
    ):
        raise AssertionError(
            "Evaluation immutability check failed."
        )


    metric_values = build_metrics[
        [
            "APFD",
            "APFDc",
        ]
    ].to_numpy(
        dtype=float
    )


    if not np.isfinite(
        metric_values
    ).all():
        raise AssertionError(
            "Condition metrics contain missing or "
            "infinite values."
        )


    if (
        (metric_values < 0)
        | (metric_values > 1)
    ).any():
        raise AssertionError(
            "Condition metrics fall outside [0, 1]."
        )


# ------------------------------------------------------------
# 7. SAVE ONE COMPLETE CONDITION
# ------------------------------------------------------------

def save_full_condition(
    noise_percent,
    repetition_seed,
    result,
):
    paths = condition_output_paths(
        noise_percent,
        repetition_seed,
    )

    directory = paths[
        "directory"
    ]


    if directory.exists():
        shutil.rmtree(
            directory
        )


    directory.mkdir(
        parents=True,
        exist_ok=False,
    )


    atomic_write_csv(
        paths[
            "noise_summary.csv"
        ],
        result[
            "noise_summary"
        ],
    )


    atomic_write_csv(
        paths[
            "noise_manifest.csv.gz"
        ],
        result[
            "noise_manifest"
        ],
        compression="gzip",
    )


    atomic_write_csv(
        paths[
            "noisy_training_labels.csv.gz"
        ],
        result[
            "noisy_training_labels"
        ],
        compression="gzip",
    )


    atomic_write_csv(
        paths[
            "fit_times.csv"
        ],
        result[
            "fit_times"
        ],
    )


    atomic_write_csv(
        paths[
            "rankings.csv.gz"
        ],
        result[
            "rankings"
        ],
        compression="gzip",
    )


    atomic_write_csv(
        paths[
            "build_metrics.csv"
        ],
        result[
            "build_metrics"
        ],
    )


    atomic_write_csv(
        paths[
            "project_run_metrics.csv"
        ],
        result[
            "project_run_metrics"
        ],
    )


    output_metadata = {}


    for file_name in (
        EXPECTED_CONDITION_DATA_FILES
    ):

        output_path = paths[
            file_name
        ]

        if not output_path.exists():
            raise RuntimeError(
                "Condition output was not written:\n"
                f"{output_path}"
            )

        output_metadata[
            file_name
        ] = {
            "Path":
                str(
                    output_path
                ),

            "SizeBytes":
                int(
                    output_path.stat().st_size
                ),

            "SHA256":
                calculate_sha256(
                    output_path
                ),
        }


    summary_row = (
        result[
            "noise_summary"
        ].iloc[0]
    )


    manifest = {
        "SchemaVersion":
            1,

        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "ConditionID":
            condition_identifier(
                noise_percent,
                repetition_seed,
            ),

        "NoisePercent":
            int(
                noise_percent
            ),

        "RepetitionSeed":
            int(
                repetition_seed
            ),

        "Counts": {
            "RawTrainingRows":
                int(
                    len(
                        result[
                            "noise_manifest"
                        ]
                    )
                ),

            "FlippedRawRows":
                int(
                    summary_row[
                        "NumberFlipped"
                    ]
                ),

            "RealisedNoisePercent":
                float(
                    summary_row[
                        "RealisedNoisePercent"
                    ]
                ),

            "ModelTrainingRows":
                int(
                    len(
                        result[
                            "noisy_training_labels"
                        ]
                    )
                ),

            "ModelTrainingLabelChanges":
                int(
                    result[
                        "noisy_training_labels"
                    ][
                        "LabelChanged"
                    ].sum()
                ),

            "ModelFits":
                int(
                    len(
                        result[
                            "fit_times"
                        ]
                    )
                ),

            "RankingRows":
                int(
                    len(
                        result[
                            "rankings"
                        ]
                    )
                ),

            "BuildMetricRows":
                int(
                    len(
                        result[
                            "build_metrics"
                        ]
                    )
                ),

            "ProjectRunRows":
                int(
                    len(
                        result[
                            "project_run_metrics"
                        ]
                    )
                ),
        },

        "EvaluationImmutable":
            bool(
                result[
                    "evaluation_unchanged"
                ]
            ),

        "ElapsedSeconds":
            float(
                result[
                    "elapsed_seconds"
                ]
            ),

        "Outputs":
            output_metadata,

        "CompletedAtUTC":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "Status":
            "COMPLETE",
    }


    # Manifest is deliberately written last. Its presence
    # indicates that all seven data files were completed.

    atomic_write_json(
        paths[
            CONDITION_MANIFEST_NAME
        ],
        manifest,
    )


    verification = condition_is_complete(
        noise_percent,
        repetition_seed,
    )


    if verification is None:
        raise RuntimeError(
            "Condition failed post-write verification:\n"
            f"{manifest['ConditionID']}"
        )


    return verification


# ------------------------------------------------------------
# 8. DISCOVER EXISTING COMPLETE CONDITIONS
# ------------------------------------------------------------

RAW_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

PROJECT_AGGREGATED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LOG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


checkpoint_records_by_key = {}

invalid_existing_conditions = []


print("\nScanning existing condition checkpoints...")


for repetition_seed in REPETITION_SEEDS:

    for noise_percent in (
        NOISE_LEVELS_PERCENT
    ):

        key = (
            int(noise_percent),
            int(repetition_seed),
        )

        directory = condition_directory(
            noise_percent,
            repetition_seed,
        )

        manifest = condition_is_complete(
            noise_percent,
            repetition_seed,
        )


        if manifest is not None:

            checkpoint_records_by_key[
                key
            ] = manifest_checkpoint_record(
                manifest
            )

        elif directory.exists():

            invalid_existing_conditions.append(
                condition_identifier(
                    noise_percent,
                    repetition_seed,
                )
            )

            shutil.rmtree(
                directory
            )


initial_complete_conditions = len(
    checkpoint_records_by_key
)


checkpoint_frame = write_global_checkpoint(
    list(
        checkpoint_records_by_key.values()
    )
)


print(
    "Complete conditions found:",
    initial_complete_conditions,
)

print(
    "Incomplete conditions removed:",
    len(
        invalid_existing_conditions
    ),
)

print(
    "Remaining conditions:",
    EXPECTED_CONDITIONS
    - initial_complete_conditions,
)


# ------------------------------------------------------------
# 9. EXECUTE ALL REMAINING CONDITIONS
# ------------------------------------------------------------

invocation_started_at = time.time()

new_conditions_completed = 0


for repetition_seed in REPETITION_SEEDS:

    for noise_percent in (
        NOISE_LEVELS_PERCENT
    ):

        key = (
            int(noise_percent),
            int(repetition_seed),
        )


        if key in checkpoint_records_by_key:
            continue


        if (
            MAX_NEW_CONDITIONS_THIS_INVOCATION
            is not None
            and new_conditions_completed
            >= MAX_NEW_CONDITIONS_THIS_INVOCATION
        ):
            break


        overall_condition_number = (
            REPETITION_SEEDS.index(
                repetition_seed
            )
            * len(
                NOISE_LEVELS_PERCENT
            )
            + NOISE_LEVELS_PERCENT.index(
                noise_percent
            )
            + 1
        )


        print("\n" + "-" * 92)

        print(
            f"Condition "
            f"{overall_condition_number}/"
            f"{EXPECTED_CONDITIONS}"
        )

        print(
            "Noise:",
            f"{noise_percent}%",
        )

        print(
            "Seed:",
            repetition_seed,
        )


        condition_result = run_full_condition(
            noise_percent=(
                noise_percent
            ),

            repetition_seed=(
                repetition_seed
            ),
        )


        validate_full_condition_result(
            condition_result
        )


        verified_manifest = save_full_condition(
            noise_percent=(
                noise_percent
            ),

            repetition_seed=(
                repetition_seed
            ),

            result=(
                condition_result
            ),
        )


        checkpoint_records_by_key[
            key
        ] = manifest_checkpoint_record(
            verified_manifest
        )


        checkpoint_frame = (
            write_global_checkpoint(
                list(
                    checkpoint_records_by_key.values()
                )
            )
        )


        new_conditions_completed += 1


        completed_conditions = len(
            checkpoint_records_by_key
        )


        print(
            "Flipped raw rows:",
            verified_manifest[
                "Counts"
            ][
                "FlippedRawRows"
            ],
        )

        print(
            "Realised noise:",
            round(
                verified_manifest[
                    "Counts"
                ][
                    "RealisedNoisePercent"
                ],
                6,
            ),
            "%",
        )

        print(
            "Elapsed seconds:",
            round(
                verified_manifest[
                    "ElapsedSeconds"
                ],
                2,
            ),
        )

        print(
            "Checkpointed conditions:",
            f"{completed_conditions}/"
            f"{EXPECTED_CONDITIONS}",
        )


        progress_payload = {
            "ProjectNumber":
                PROJECT_NUMBER,

            "Project":
                PROJECT_NAME,

            "ProjectSlug":
                PROJECT_SLUG,

            "CompletedConditions":
                completed_conditions,

            "RemainingConditions":
                EXPECTED_CONDITIONS
                - completed_conditions,

            "NewConditionsThisInvocation":
                new_conditions_completed,

            "LastCompletedCondition":
                verified_manifest[
                    "ConditionID"
                ],

            "InvocationElapsedSeconds":
                float(
                    time.time()
                    - invocation_started_at
                ),

            "UpdatedAtUTC":
                datetime.now(
                    timezone.utc
                ).isoformat(),

            "Status":
                (
                    "RUNNING"
                    if completed_conditions
                    < EXPECTED_CONDITIONS
                    else "ALL_CONDITIONS_COMPLETE"
                ),
        }


        atomic_write_json(
            FULL_RUN_PROGRESS_PATH,
            progress_payload,
        )


        del condition_result
        gc.collect()


    if (
        MAX_NEW_CONDITIONS_THIS_INVOCATION
        is not None
        and new_conditions_completed
        >= MAX_NEW_CONDITIONS_THIS_INVOCATION
    ):
        break


# ------------------------------------------------------------
# 10. FINAL CHECKPOINT SCAN
# ------------------------------------------------------------

final_checkpoint_records = []

invalid_final_conditions = []


for repetition_seed in REPETITION_SEEDS:

    for noise_percent in (
        NOISE_LEVELS_PERCENT
    ):

        manifest = condition_is_complete(
            noise_percent,
            repetition_seed,
        )


        if manifest is None:

            invalid_final_conditions.append(
                condition_identifier(
                    noise_percent,
                    repetition_seed,
                )
            )

        else:

            final_checkpoint_records.append(
                manifest_checkpoint_record(
                    manifest
                )
            )


final_checkpoint_frame = (
    write_global_checkpoint(
        final_checkpoint_records
    )
)


completed_conditions = len(
    final_checkpoint_frame
)

remaining_conditions = (
    EXPECTED_CONDITIONS
    - completed_conditions
)


completed_model_fits = int(
    final_checkpoint_frame[
        "ModelFits"
    ].sum()
) if not final_checkpoint_frame.empty else 0


completed_ranking_rows = int(
    final_checkpoint_frame[
        "RankingRows"
    ].sum()
) if not final_checkpoint_frame.empty else 0


completed_build_metric_rows = int(
    final_checkpoint_frame[
        "BuildMetricRows"
    ].sum()
) if not final_checkpoint_frame.empty else 0


completed_project_run_rows = int(
    final_checkpoint_frame[
        "ProjectRunRows"
    ].sum()
) if not final_checkpoint_frame.empty else 0


evaluation_immutable_all = bool(
    final_checkpoint_frame[
        "EvaluationImmutable"
    ].all()
) if not final_checkpoint_frame.empty else False


raw_files = sorted([
    path
    for path in RAW_ROOT.rglob("*")
    if path.is_file()
])


raw_file_count = len(
    raw_files
)


raw_total_bytes = int(
    sum(
        path.stat().st_size
        for path in raw_files
    )
)


# ------------------------------------------------------------
# 11. DETERMINE STEP 9 STATUS
# ------------------------------------------------------------

full_run_complete = bool(
    completed_conditions
    == EXPECTED_CONDITIONS
    and len(
        invalid_final_conditions
    )
    == 0
    and completed_model_fits
    == EXPECTED_MODEL_FITS
    and completed_ranking_rows
    == EXPECTED_TOTAL_RANKING_ROWS
    and completed_build_metric_rows
    == EXPECTED_TOTAL_BUILD_METRIC_ROWS
    and completed_project_run_rows
    == EXPECTED_TOTAL_PROJECT_RUN_ROWS
    and raw_file_count
    == EXPECTED_RAW_FILES
    and evaluation_immutable_all
)


if full_run_complete:

    step9_status_text = (
        "PASS_PROJECT_8_FULL_270_CONDITION_RUN_COMPLETE"
    )

else:

    step9_status_text = (
        "PROJECT_8_FULL_RUN_CHECKPOINTED_INCOMPLETE"
    )


step9_status = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ExpectedConditions":
        EXPECTED_CONDITIONS,

    "CompletedConditions":
        completed_conditions,

    "RemainingConditions":
        remaining_conditions,

    "InvalidConditions":
        invalid_final_conditions,

    "ExpectedMLFits":
        EXPECTED_MODEL_FITS,

    "CompletedMLFits":
        completed_model_fits,

    "ExpectedRankingRows":
        EXPECTED_TOTAL_RANKING_ROWS,

    "CompletedRankingRows":
        completed_ranking_rows,

    "ExpectedBuildMetricRows":
        EXPECTED_TOTAL_BUILD_METRIC_ROWS,

    "CompletedBuildMetricRows":
        completed_build_metric_rows,

    "ExpectedProjectRunRows":
        EXPECTED_TOTAL_PROJECT_RUN_ROWS,

    "CompletedProjectRunRows":
        completed_project_run_rows,

    "ExpectedRawFiles":
        EXPECTED_RAW_FILES,

    "RawFiles":
        raw_file_count,

    "RawBytes":
        raw_total_bytes,

    "EvaluationImmutable":
        evaluation_immutable_all,

    "InitialCompleteConditions":
        initial_complete_conditions,

    "NewConditionsThisInvocation":
        new_conditions_completed,

    "InvocationElapsedSeconds":
        float(
            time.time()
            - invocation_started_at
        ),

    "UpdatedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "Status":
        step9_status_text,
}


atomic_write_json(
    STEP9_STATUS_PATH,
    step9_status,
)


atomic_write_json(
    FULL_RUN_PROGRESS_PATH,
    {
        **step9_status,

        "Status":
            (
                "ALL_CONDITIONS_COMPLETE"
                if full_run_complete
                else "CHECKPOINTED_INCOMPLETE"
            ),
    },
)


# ------------------------------------------------------------
# 12. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 92)
print("=== PROJECT 8 STEP 9 RESULT ===")
print("=" * 92)

print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nCondition progress:")

print(
    "Expected conditions:",
    EXPECTED_CONDITIONS,
)

print(
    "Initially complete:",
    initial_complete_conditions,
)

print(
    "Newly completed this invocation:",
    new_conditions_completed,
)

print(
    "Completed conditions:",
    completed_conditions,
)

print(
    "Remaining conditions:",
    remaining_conditions,
)

print(
    "Invalid conditions:",
    len(
        invalid_final_conditions
    ),
)


print("\nExperiment totals:")

print(
    "ML fits:",
    completed_model_fits,
    "/",
    EXPECTED_MODEL_FITS,
)

print(
    "Ranking rows:",
    completed_ranking_rows,
    "/",
    EXPECTED_TOTAL_RANKING_ROWS,
)

print(
    "Build-metric rows:",
    completed_build_metric_rows,
    "/",
    EXPECTED_TOTAL_BUILD_METRIC_ROWS,
)

print(
    "Project-run rows:",
    completed_project_run_rows,
    "/",
    EXPECTED_TOTAL_PROJECT_RUN_ROWS,
)


print("\nRaw result freeze candidate:")

print(
    "Expected raw files:",
    EXPECTED_RAW_FILES,
)

print(
    "Actual raw files:",
    raw_file_count,
)

print(
    "Raw bytes:",
    raw_total_bytes,
)

print(
    "Raw root:",
    RAW_ROOT,
)


print("\nEvaluation immutable:")

print(
    evaluation_immutable_all
)


print("\nCheckpoint outputs:")

print(
    FULL_RUN_CHECKPOINT_PATH
)

print(
    FULL_RUN_PROGRESS_PATH
)

print(
    STEP9_STATUS_PATH
)


print("\nInvocation runtime:")

print(
    round(
        time.time()
        - invocation_started_at,
        2,
    ),
    "seconds",
)


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–7 modified:")
print(0)


print(
    "\nSTATUS:",
    step9_status_text,
)

print("=" * 92)

=== PROJECT 8 STEP 9: CHECKPOINTED 270-CONDITION FULL EXPERIMENT ===

Prerequisite validation:
Step 8B status: PASS_PROJECT_8_TWO_CONDITION_SMOKE_TEST_VALIDATED
Active predictors: 150
Evaluation rows: 2300
Evaluation builds: 16

Scanning existing condition checkpoints...
Complete conditions found: 0
Incomplete conditions removed: 0
Remaining conditions: 270

--------------------------------------------------------------------------------------------
Condition 1/270
Noise: 0%
Seed: 1
Flipped raw rows: 0
Realised noise: 0.0 %
Elapsed seconds: 58.66
Checkpointed conditions: 1/270

--------------------------------------------------------------------------------------------
Condition 2/270
Noise: 5%
Seed: 1
Flipped raw rows: 1243
Realised noise: 4.926479 %
Elapsed seconds: 38.3
Checkpointed conditions: 2/270

--------------------------------------------------------------------------------------------
Condition 3/270
Noise: 10%
Seed: 1
Flipped raw rows: 2498
Realised noise: 9.900519 %
Elapse

In [ ]:
# ============================================================
# PROJECT 8 — STEP 10 V2
# COMPLETE RAW-RESULT AUDIT AND AGGREGATION
#
# Fix:
# - Handles ConditionID whether it already exists or not.
# - Validates existing ConditionID values.
#
# This cell:
# - independently audits all 270 conditions
# - verifies all 2,160 raw files and manifest hashes
# - verifies nested noise masks and label corruption
# - verifies ML fits, rankings, APFD and APFDc
# - verifies Random and QTF-Avg invariance
# - aggregates all Project 8 result files
#
# It does NOT:
# - rerun models
# - modify raw condition files
# - modify Projects 1–7
# - modify the completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. PROJECT CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 8
PROJECT_NAME = "optimatika@ojAlgo"
PROJECT_SLUG = "optimatika__ojAlgo"
PROJECT_SHORT_NAME = "ojalgo"

NOISE_LEVELS_PERCENT = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

REPETITION_SEEDS = list(
    range(1, 31)
)

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINES
)

EXPECTED_CONDITIONS = 270
EXPECTED_ML_FITS = 1080

EXPECTED_RAW_TRAINING_ROWS = 25231
EXPECTED_MODEL_TRAINING_ROWS = 7580
EXPECTED_EVALUATION_ROWS = 2300
EXPECTED_EVALUATION_BUILDS = 16

EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_EVALUATION_ROWS
    * len(ALL_TECHNIQUES)
)

EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = (
    EXPECTED_EVALUATION_BUILDS
    * len(ALL_TECHNIQUES)
)

EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = (
    len(ALL_TECHNIQUES)
)

EXPECTED_TOTAL_RANKING_ROWS = (
    EXPECTED_CONDITIONS
    * EXPECTED_RANKING_ROWS_PER_CONDITION
)

EXPECTED_TOTAL_BUILD_METRIC_ROWS = (
    EXPECTED_CONDITIONS
    * EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
)

EXPECTED_TOTAL_PROJECT_RUN_ROWS = (
    EXPECTED_CONDITIONS
    * EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
)

CONDITION_DATA_FILES = [
    "noise_summary.csv",
    "noise_manifest.csv.gz",
    "noisy_training_labels.csv.gz",
    "fit_times.csv",
    "rankings.csv.gz",
    "build_metrics.csv",
    "project_run_metrics.csv",
]

CONDITION_MANIFEST_FILE = (
    "condition_manifest.json"
)

EXPECTED_FILES_PER_CONDITION = (
    len(CONDITION_DATA_FILES)
    + 1
)

EXPECTED_RAW_FILES = (
    EXPECTED_CONDITIONS
    * EXPECTED_FILES_PER_CONDITION
)


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

RAW_ROOT = (
    THESIS_ROOT
    / "Results"
    / "Raw"
    / PROJECT_SLUG
)

PROJECT_AGGREGATED_DIR = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / PROJECT_SLUG
)

STEP9_STATUS_PATH = (
    PROJECT_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_step9_status.json"
)

AUDIT_DIR = (
    PROJECT_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_raw_audit"
)

ALL_NOISE_SUMMARY_PATH = (
    AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_all_noise_summary.csv"
)

ALL_FIT_TIMES_PATH = (
    AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_all_fit_times.csv"
)

ALL_BUILD_METRICS_PATH = (
    AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_all_build_metrics.csv.gz"
)

ALL_PROJECT_RUN_METRICS_PATH = (
    AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_all_project_run_metrics.csv"
)

CONDITION_AUDIT_PATH = (
    AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_condition_audit.csv"
)

NESTED_NOISE_AUDIT_PATH = (
    AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_nested_noise_audit.csv"
)

BASELINE_AUDIT_PATH = (
    AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_baseline_invariance_audit.csv"
)

RAW_FILE_MANIFEST_PATH = (
    AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_raw_file_manifest.csv.gz"
)

OVERALL_AUDIT_PATH = (
    AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_raw_results_audit.csv"
)

RAW_AUDIT_REPORT_PATH = (
    AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_raw_audit_report.json"
)

STEP10_STATUS_PATH = (
    PROJECT_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_step10_status.json"
)


print("=" * 92)
print("=== PROJECT 8 STEP 10 V2: COMPLETE RAW-RESULT AUDIT AND AGGREGATION ===")
print("=" * 92)


# ------------------------------------------------------------
# 3. GENERAL HELPERS
# ------------------------------------------------------------

def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def json_safe(value):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def atomic_write_csv(
    path,
    dataframe,
    compression=None,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
        compression=compression,
    )

    temporary_path.replace(path)


def condition_id(
    noise_percent,
    repetition_seed,
):
    return (
        f"noise_{int(noise_percent):03d}"
        f"__seed_{int(repetition_seed):03d}"
    )


def condition_directory(
    noise_percent,
    repetition_seed,
):
    return (
        RAW_ROOT
        / f"noise_{int(noise_percent):03d}"
        / f"seed_{int(repetition_seed):03d}"
    )


def condition_paths(
    noise_percent,
    repetition_seed,
):
    directory = condition_directory(
        noise_percent,
        repetition_seed,
    )

    paths = {
        file_name:
            directory / file_name
        for file_name
        in CONDITION_DATA_FILES
    }

    paths[
        CONDITION_MANIFEST_FILE
    ] = (
        directory
        / CONDITION_MANIFEST_FILE
    )

    return paths


def ensure_condition_id(
    dataframe,
    expected_condition_id,
):
    """
    Add ConditionID when absent.

    When ConditionID already exists, validate its values
    and keep only one ConditionID column at the beginning.
    """

    dataframe = dataframe.copy()

    if "ConditionID" in dataframe.columns:

        existing_values = (
            dataframe[
                "ConditionID"
            ]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
            .tolist()
        )

        invalid_values = [
            value
            for value in existing_values
            if value
            != expected_condition_id
        ]

        if invalid_values:
            raise AssertionError(
                "Existing ConditionID does not match "
                "the expected condition.\n"
                f"Expected: {expected_condition_id}\n"
                f"Detected: {invalid_values}"
            )

        dataframe[
            "ConditionID"
        ] = expected_condition_id

    else:

        dataframe.insert(
            0,
            "ConditionID",
            expected_condition_id,
        )

    ordered_columns = [
        "ConditionID",
        *[
            column
            for column in dataframe.columns
            if column != "ConditionID"
        ],
    ]

    return dataframe[
        ordered_columns
    ]


def to_boolean_series(
    series,
):
    if pd.api.types.is_bool_dtype(
        series
    ):
        return series.astype(bool)

    normalised = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true":
            True,

        "false":
            False,

        "1":
            True,

        "0":
            False,
    }

    converted = normalised.map(
        mapping
    )

    if converted.isna().any():

        bad_values = sorted(
            normalised[
                converted.isna()
            ].unique()
        )

        raise ValueError(
            "Cannot convert values to Boolean:\n"
            f"{bad_values}"
        )

    return converted.astype(bool)


def technique_ranking_signature(
    rankings,
    technique,
):
    subset = rankings[
        rankings[
            "Technique"
        ].eq(
            technique
        )
    ].copy()

    digest = hashlib.sha256()

    for build_id in sorted(
        subset[
            "Build"
        ].unique()
    ):

        build_rows = (
            subset[
                subset[
                    "Build"
                ].eq(
                    build_id
                )
            ]
            .sort_values(
                "Rank",
                kind="mergesort",
            )
        )

        ordered_tests = (
            build_rows[
                "Test"
            ]
            .astype(str)
            .tolist()
        )

        digest.update(
            str(
                int(build_id)
            ).encode(
                "utf-8"
            )
        )

        digest.update(b":")

        digest.update(
            "#".join(
                ordered_tests
            ).encode(
                "utf-8"
            )
        )

        digest.update(b"\n")

    return digest.hexdigest()


def calculate_raw_root_hash(
    file_manifest,
):
    digest = hashlib.sha256()

    ordered_manifest = (
        file_manifest
        .sort_values(
            "RelativePath",
            kind="mergesort",
        )
    )

    for row in ordered_manifest.itertuples(
        index=False
    ):

        digest.update(
            row.RelativePath.encode(
                "utf-8"
            )
        )

        digest.update(b"\0")

        digest.update(
            str(
                int(row.SizeBytes)
            ).encode(
                "utf-8"
            )
        )

        digest.update(b"\0")

        digest.update(
            bytes.fromhex(
                row.SHA256
            )
        )

        digest.update(b"\n")

    return digest.hexdigest()


# ------------------------------------------------------------
# 4. VALIDATE STEP 9
# ------------------------------------------------------------

if not THESIS_ROOT.exists():
    raise FileNotFoundError(
        "Google Drive is unavailable at:\n"
        f"{THESIS_ROOT}"
    )


if not RAW_ROOT.exists():
    raise FileNotFoundError(
        "Project 8 raw-results root is missing:\n"
        f"{RAW_ROOT}"
    )


if not STEP9_STATUS_PATH.exists():
    raise FileNotFoundError(
        "Project 8 Step 9 status is missing:\n"
        f"{STEP9_STATUS_PATH}"
    )


step9_status = json.loads(
    STEP9_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


expected_step9_status = (
    "PASS_PROJECT_8_FULL_270_CONDITION_RUN_COMPLETE"
)


if (
    step9_status.get("Status")
    != expected_step9_status
):
    raise AssertionError(
        "Project 8 Step 9 has not passed.\n"
        f"Detected status: "
        f"{step9_status.get('Status')}"
    )


print("\nStep 9 validation:")

print(
    "Status:",
    step9_status.get("Status"),
)

print(
    "Completed conditions:",
    step9_status.get(
        "CompletedConditions"
    ),
)

print(
    "Raw files reported:",
    step9_status.get(
        "RawFiles"
    ),
)

print(
    "Raw bytes reported:",
    step9_status.get(
        "RawBytes"
    ),
)


# ------------------------------------------------------------
# 5. BUILD EXPECTED AND ACTUAL FILE INVENTORIES
# ------------------------------------------------------------

expected_relative_paths = set()


for repetition_seed in REPETITION_SEEDS:

    for noise_percent in (
        NOISE_LEVELS_PERCENT
    ):

        relative_directory = (
            Path(
                f"noise_{noise_percent:03d}"
            )
            / f"seed_{repetition_seed:03d}"
        )

        for file_name in (
            CONDITION_DATA_FILES
            + [
                CONDITION_MANIFEST_FILE
            ]
        ):

            expected_relative_paths.add(
                (
                    relative_directory
                    / file_name
                ).as_posix()
            )


actual_raw_files = sorted([
    path
    for path in RAW_ROOT.rglob("*")
    if path.is_file()
])


actual_relative_paths = {
    path.relative_to(
        RAW_ROOT
    ).as_posix()
    for path in actual_raw_files
}


missing_raw_files = sorted(
    expected_relative_paths
    - actual_relative_paths
)


unexpected_raw_files = sorted(
    actual_relative_paths
    - expected_relative_paths
)


print("\nRaw inventory:")

print(
    "Expected files:",
    len(
        expected_relative_paths
    ),
)

print(
    "Actual files:",
    len(
        actual_raw_files
    ),
)

print(
    "Missing files:",
    len(
        missing_raw_files
    ),
)

print(
    "Unexpected files:",
    len(
        unexpected_raw_files
    ),
)


if missing_raw_files:
    raise RuntimeError(
        "Expected raw files are missing:\n"
        + "\n".join(
            missing_raw_files[:50]
        )
    )


if unexpected_raw_files:
    raise RuntimeError(
        "Unexpected files exist in the raw root:\n"
        + "\n".join(
            unexpected_raw_files[:50]
        )
    )


# ------------------------------------------------------------
# 6. HASH COMPLETE RAW FILE INVENTORY
# ------------------------------------------------------------

print("\nHashing all raw files...")


raw_file_records = []


for file_index, file_path in enumerate(
    actual_raw_files,
    start=1,
):

    relative_path = file_path.relative_to(
        RAW_ROOT
    ).as_posix()

    raw_file_records.append({
        "RelativePath":
            relative_path,

        "SizeBytes":
            int(
                file_path.stat().st_size
            ),

        "SHA256":
            calculate_sha256(
                file_path
            ),
    })

    if (
        file_index % 200 == 0
        or file_index
        == len(actual_raw_files)
    ):

        print(
            "Hashed:",
            file_index,
            "/",
            len(actual_raw_files),
        )


raw_file_manifest = pd.DataFrame(
    raw_file_records
)


raw_file_hash_lookup = dict(
    zip(
        raw_file_manifest[
            "RelativePath"
        ],

        raw_file_manifest[
            "SHA256"
        ],
    )
)


raw_file_size_lookup = dict(
    zip(
        raw_file_manifest[
            "RelativePath"
        ],

        raw_file_manifest[
            "SizeBytes"
        ],
    )
)


raw_total_bytes = int(
    raw_file_manifest[
        "SizeBytes"
    ].sum()
)


raw_root_sha256 = (
    calculate_raw_root_hash(
        raw_file_manifest
    )
)


print("\nRaw-root inventory hash:")

print(
    raw_root_sha256
)


# ------------------------------------------------------------
# 7. AUDIT EVERY CONDITION
# ------------------------------------------------------------

condition_audit_records = []
nested_noise_audit_records = []
baseline_signature_records = []

all_noise_summaries = []
all_fit_times = []
all_build_metrics = []
all_project_run_metrics = []

manifest_hash_mismatches = 0
manifest_size_mismatches = 0

total_ranking_rows = 0
total_duplicate_ranking_rows = 0

all_evaluation_immutable = True


print("\nAuditing all 270 conditions...")


for repetition_seed in REPETITION_SEEDS:

    seed_noise_manifests = {}

    for noise_percent in (
        NOISE_LEVELS_PERCENT
    ):

        current_condition_id = condition_id(
            noise_percent,
            repetition_seed,
        )

        paths = condition_paths(
            noise_percent,
            repetition_seed,
        )


        # ----------------------------------------------------
        # Condition manifest
        # ----------------------------------------------------

        manifest = json.loads(
            paths[
                CONDITION_MANIFEST_FILE
            ].read_text(
                encoding="utf-8"
            )
        )


        manifest_identity_pass = bool(
            manifest.get("Project")
            == PROJECT_NAME

            and manifest.get("ConditionID")
            == current_condition_id

            and int(
                manifest.get(
                    "NoisePercent",
                    -1,
                )
            )
            == noise_percent

            and int(
                manifest.get(
                    "RepetitionSeed",
                    -1,
                )
            )
            == repetition_seed

            and manifest.get("Status")
            == "COMPLETE"
        )


        condition_hash_mismatches = 0
        condition_size_mismatches = 0


        for file_name in (
            CONDITION_DATA_FILES
        ):

            output_metadata = (
                manifest.get(
                    "Outputs",
                    {}
                ).get(
                    file_name,
                    {}
                )
            )

            relative_path = (
                paths[
                    file_name
                ]
                .relative_to(
                    RAW_ROOT
                )
                .as_posix()
            )

            actual_hash = (
                raw_file_hash_lookup.get(
                    relative_path
                )
            )

            actual_size = (
                raw_file_size_lookup.get(
                    relative_path
                )
            )

            expected_hash = (
                output_metadata.get(
                    "SHA256"
                )
            )

            expected_size = (
                output_metadata.get(
                    "SizeBytes"
                )
            )


            if actual_hash != expected_hash:
                condition_hash_mismatches += 1


            if (
                actual_size is None
                or expected_size is None
                or int(actual_size)
                != int(expected_size)
            ):
                condition_size_mismatches += 1


        manifest_hash_mismatches += (
            condition_hash_mismatches
        )

        manifest_size_mismatches += (
            condition_size_mismatches
        )


        # ----------------------------------------------------
        # Noise summary
        # ----------------------------------------------------

        noise_summary = pd.read_csv(
            paths[
                "noise_summary.csv"
            ],
            low_memory=False,
        )


        if len(noise_summary) != 1:
            raise AssertionError(
                f"{current_condition_id}: "
                "noise_summary.csv must contain one row."
            )


        noise_summary = ensure_condition_id(
            dataframe=noise_summary,
            expected_condition_id=(
                current_condition_id
            ),
        )


        all_noise_summaries.append(
            noise_summary
        )


        # ----------------------------------------------------
        # Raw training noise manifest
        # ----------------------------------------------------

        noise_manifest = pd.read_csv(
            paths[
                "noise_manifest.csv.gz"
            ],
            low_memory=False,
        )


        required_noise_columns = {
            "NoiseRowID",
            "Build",
            "Test",
            "OriginalVerdict",
            "NoisyVerdict",
            "FlipUniform",
            "Flipped",
            "PassToFailure",
            "FailureToPass",
        }


        missing_noise_columns = (
            required_noise_columns
            - set(
                noise_manifest.columns
            )
        )


        if missing_noise_columns:
            raise RuntimeError(
                f"{current_condition_id}: "
                "noise manifest is missing columns:\n"
                f"{sorted(missing_noise_columns)}"
            )


        noise_manifest = (
            noise_manifest
            .sort_values(
                "NoiseRowID",
                kind="mergesort",
            )
            .reset_index(drop=True)
        )


        for column in [
            "NoiseRowID",
            "Build",
            "Test",
            "OriginalVerdict",
            "NoisyVerdict",
        ]:

            noise_manifest[
                column
            ] = pd.to_numeric(
                noise_manifest[
                    column
                ],
                errors="raise",
            ).astype(np.int64)


        noise_manifest[
            "FlipUniform"
        ] = pd.to_numeric(
            noise_manifest[
                "FlipUniform"
            ],
            errors="raise",
        ).astype(float)


        for column in [
            "Flipped",
            "PassToFailure",
            "FailureToPass",
        ]:

            noise_manifest[
                column
            ] = to_boolean_series(
                noise_manifest[
                    column
                ]
            )


        flipped = noise_manifest[
            "Flipped"
        ].to_numpy(dtype=bool)

        pass_to_failure = noise_manifest[
            "PassToFailure"
        ].to_numpy(dtype=bool)

        failure_to_pass = noise_manifest[
            "FailureToPass"
        ].to_numpy(dtype=bool)

        original_verdict = noise_manifest[
            "OriginalVerdict"
        ].to_numpy(dtype=int)

        noisy_verdict = noise_manifest[
            "NoisyVerdict"
        ].to_numpy(dtype=int)

        uniforms = noise_manifest[
            "FlipUniform"
        ].to_numpy(dtype=float)


        expected_flip_mask = (
            uniforms
            < noise_percent / 100.0
        )


        flip_threshold_mismatches = int(
            (
                flipped
                != expected_flip_mask
            ).sum()
        )


        unflipped_verdict_mismatches = int(
            (
                (~flipped)
                & (
                    noisy_verdict
                    != original_verdict
                )
            ).sum()
        )


        pass_to_failure_mismatches = int(
            (
                pass_to_failure
                != (
                    flipped
                    & (
                        original_verdict
                        == 0
                    )
                )
            ).sum()
        )


        failure_to_pass_mismatches = int(
            (
                failure_to_pass
                != (
                    flipped
                    & (
                        original_verdict
                        != 0
                    )
                )
            ).sum()
        )


        invalid_pass_to_failure_values = int(
            (
                pass_to_failure
                & (
                    noisy_verdict
                    == 0
                )
            ).sum()
        )


        invalid_failure_to_pass_values = int(
            (
                failure_to_pass
                & (
                    noisy_verdict
                    != 0
                )
            ).sum()
        )


        invalid_noisy_verdict_values = int(
            (
                ~np.isin(
                    noisy_verdict,
                    [
                        0,
                        1,
                        2,
                    ],
                )
            ).sum()
        )


        seed_noise_manifests[
            noise_percent
        ] = noise_manifest[
            [
                "NoiseRowID",
                "Build",
                "Test",
                "OriginalVerdict",
                "NoisyVerdict",
                "FlipUniform",
                "Flipped",
            ]
        ].copy()


        # ----------------------------------------------------
        # Model-ready noisy labels
        # ----------------------------------------------------

        noisy_labels = pd.read_csv(
            paths[
                "noisy_training_labels.csv.gz"
            ],
            low_memory=False,
        )


        for column in [
            "Build",
            "Test",
            "CleanVerdict",
            "NoisyVerdict",
        ]:

            noisy_labels[
                column
            ] = pd.to_numeric(
                noisy_labels[
                    column
                ],
                errors="raise",
            ).astype(np.int64)


        noisy_labels[
            "LabelChanged"
        ] = to_boolean_series(
            noisy_labels[
                "LabelChanged"
            ]
        )


        manifest_verdict_lookup = (
            noise_manifest[
                [
                    "Build",
                    "Test",
                    "NoisyVerdict",
                ]
            ]
            .rename(
                columns={
                    "NoisyVerdict":
                        "RawNoisyVerdict",
                }
            )
        )


        label_alignment = noisy_labels.merge(
            manifest_verdict_lookup,
            on=[
                "Build",
                "Test",
            ],
            how="left",
            validate="one_to_one",
        )


        missing_label_alignment = int(
            label_alignment[
                "RawNoisyVerdict"
            ].isna().sum()
        )


        noisy_label_mismatches = int(
            (
                label_alignment[
                    "NoisyVerdict"
                ]
                != label_alignment[
                    "RawNoisyVerdict"
                ]
            ).sum()
        )


        label_change_flag_mismatches = int(
            (
                noisy_labels[
                    "LabelChanged"
                ].to_numpy(dtype=bool)
                != (
                    noisy_labels[
                        "CleanVerdict"
                    ].to_numpy(dtype=int)
                    != noisy_labels[
                        "NoisyVerdict"
                    ].to_numpy(dtype=int)
                )
            ).sum()
        )


        # ----------------------------------------------------
        # Fit times
        # ----------------------------------------------------

        fit_times = pd.read_csv(
            paths[
                "fit_times.csv"
            ],
            low_memory=False,
        )


        fit_times = ensure_condition_id(
            dataframe=fit_times,
            expected_condition_id=(
                current_condition_id
            ),
        )


        all_fit_times.append(
            fit_times
        )


        fit_techniques_pass = bool(
            len(fit_times) == 4

            and set(
                fit_times[
                    "Technique"
                ]
            )
            == set(
                ML_TECHNIQUES
            )
        )


        fit_training_rows_pass = bool(
            pd.to_numeric(
                fit_times[
                    "TrainingRows"
                ],
                errors="raise",
            ).eq(
                EXPECTED_MODEL_TRAINING_ROWS
            ).all()
        )


        fit_class_balance_pass = bool(
            (
                pd.to_numeric(
                    fit_times[
                        "TrainingFailures"
                    ],
                    errors="raise",
                )
                > 0
            ).all()

            and

            (
                pd.to_numeric(
                    fit_times[
                        "TrainingPasses"
                    ],
                    errors="raise",
                )
                > 0
            ).all()
        )


        # ----------------------------------------------------
        # Rankings
        # ----------------------------------------------------

        rankings = pd.read_csv(
            paths[
                "rankings.csv.gz"
            ],
            usecols=[
                "Technique",
                "Build",
                "Test",
                "Rank",
            ],
            low_memory=False,
        )


        rankings[
            "Build"
        ] = pd.to_numeric(
            rankings[
                "Build"
            ],
            errors="raise",
        ).astype(np.int64)


        rankings[
            "Test"
        ] = pd.to_numeric(
            rankings[
                "Test"
            ],
            errors="raise",
        ).astype(np.int64)


        rankings[
            "Rank"
        ] = pd.to_numeric(
            rankings[
                "Rank"
            ],
            errors="raise",
        ).astype(np.int64)


        total_ranking_rows += len(
            rankings
        )


        duplicate_ranking_rows = int(
            rankings.duplicated(
                subset=[
                    "Technique",
                    "Build",
                    "Test",
                ]
            ).sum()
        )


        total_duplicate_ranking_rows += (
            duplicate_ranking_rows
        )


        ranking_technique_pass = bool(
            set(
                rankings[
                    "Technique"
                ].unique()
            )
            == set(
                ALL_TECHNIQUES
            )
        )


        ranking_build_count_pass = bool(
            (
                rankings
                .groupby(
                    "Technique"
                )[
                    "Build"
                ]
                .nunique()
                == EXPECTED_EVALUATION_BUILDS
            ).all()
        )


        ranking_rows_per_technique_pass = bool(
            (
                rankings
                .groupby(
                    "Technique"
                )
                .size()
                == EXPECTED_EVALUATION_ROWS
            ).all()
        )


        rank_sequence_errors = 0


        for (
            technique,
            build_id,
        ), group in rankings.groupby(
            [
                "Technique",
                "Build",
            ],
            sort=False,
        ):

            actual_ranks = np.sort(
                group[
                    "Rank"
                ].to_numpy(dtype=int)
            )

            expected_ranks = np.arange(
                1,
                len(group) + 1,
            )

            if not np.array_equal(
                actual_ranks,
                expected_ranks,
            ):
                rank_sequence_errors += 1


        random_signature = (
            technique_ranking_signature(
                rankings=rankings,
                technique="Random",
            )
        )


        qtf_signature = (
            technique_ranking_signature(
                rankings=rankings,
                technique="QTF-Avg",
            )
        )


        latestfail_signature = (
            technique_ranking_signature(
                rankings=rankings,
                technique="LatestFail",
            )
        )


        baseline_signature_records.append({
            "ConditionID":
                current_condition_id,

            "NoisePercent":
                noise_percent,

            "RepetitionSeed":
                repetition_seed,

            "RandomSignature":
                random_signature,

            "QTFAvgSignature":
                qtf_signature,

            "LatestFailSignature":
                latestfail_signature,
        })


        # ----------------------------------------------------
        # Build-level metrics
        # ----------------------------------------------------

        build_metrics = pd.read_csv(
            paths[
                "build_metrics.csv"
            ],
            low_memory=False,
        )


        build_metrics = ensure_condition_id(
            dataframe=build_metrics,
            expected_condition_id=(
                current_condition_id
            ),
        )


        all_build_metrics.append(
            build_metrics
        )


        metric_values = (
            build_metrics[
                [
                    "APFD",
                    "APFDc",
                ]
            ]
            .apply(
                pd.to_numeric,
                errors="coerce",
            )
            .to_numpy(dtype=float)
        )


        metric_invalid_cells = int(
            (
                ~np.isfinite(
                    metric_values
                )
                |
                (
                    metric_values
                    < 0
                )
                |
                (
                    metric_values
                    > 1
                )
            ).sum()
        )


        build_metric_techniques_pass = bool(
            set(
                build_metrics[
                    "Technique"
                ].unique()
            )
            == set(
                ALL_TECHNIQUES
            )
        )


        build_metric_failures_pass = bool(
            (
                pd.to_numeric(
                    build_metrics[
                        "NumberOfFailures"
                    ],
                    errors="raise",
                )
                > 0
            ).all()
        )


        # ----------------------------------------------------
        # Project-run metrics
        # ----------------------------------------------------

        project_run_metrics = pd.read_csv(
            paths[
                "project_run_metrics.csv"
            ],
            low_memory=False,
        )


        project_run_metrics = ensure_condition_id(
            dataframe=project_run_metrics,
            expected_condition_id=(
                current_condition_id
            ),
        )


        all_project_run_metrics.append(
            project_run_metrics
        )


        project_run_techniques_pass = bool(
            len(
                project_run_metrics
            )
            == EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION

            and set(
                project_run_metrics[
                    "Technique"
                ]
            )
            == set(
                ALL_TECHNIQUES
            )
        )


        project_run_builds_pass = bool(
            pd.to_numeric(
                project_run_metrics[
                    "EvaluatedBuilds"
                ],
                errors="raise",
            ).eq(
                EXPECTED_EVALUATION_BUILDS
            ).all()
        )


        # ----------------------------------------------------
        # Manifest count validation
        # ----------------------------------------------------

        manifest_counts = manifest.get(
            "Counts",
            {}
        )


        manifest_counts_pass = bool(
            int(
                manifest_counts.get(
                    "RawTrainingRows",
                    -1,
                )
            )
            == EXPECTED_RAW_TRAINING_ROWS

            and int(
                manifest_counts.get(
                    "ModelTrainingRows",
                    -1,
                )
            )
            == EXPECTED_MODEL_TRAINING_ROWS

            and int(
                manifest_counts.get(
                    "ModelFits",
                    -1,
                )
            )
            == 4

            and int(
                manifest_counts.get(
                    "RankingRows",
                    -1,
                )
            )
            == EXPECTED_RANKING_ROWS_PER_CONDITION

            and int(
                manifest_counts.get(
                    "BuildMetricRows",
                    -1,
                )
            )
            == EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION

            and int(
                manifest_counts.get(
                    "ProjectRunRows",
                    -1,
                )
            )
            == EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
        )


        realised_noise_percent = float(
            100.0
            * flipped.sum()
            / len(
                noise_manifest
            )
        )


        manifest_noise_count_pass = bool(
            int(
                manifest_counts.get(
                    "FlippedRawRows",
                    -1,
                )
            )
            == int(
                flipped.sum()
            )

            and np.isclose(
                float(
                    manifest_counts.get(
                        "RealisedNoisePercent",
                        np.nan,
                    )
                ),
                realised_noise_percent,
                rtol=0,
                atol=1e-12,
            )
        )


        evaluation_immutable = bool(
            manifest.get(
                "EvaluationImmutable",
                False,
            )
        )


        all_evaluation_immutable = bool(
            all_evaluation_immutable
            and evaluation_immutable
        )


        condition_pass = bool(
            manifest_identity_pass
            and condition_hash_mismatches == 0
            and condition_size_mismatches == 0
            and manifest_counts_pass
            and manifest_noise_count_pass

            and len(noise_manifest)
                == EXPECTED_RAW_TRAINING_ROWS

            and flip_threshold_mismatches == 0
            and unflipped_verdict_mismatches == 0
            and pass_to_failure_mismatches == 0
            and failure_to_pass_mismatches == 0
            and invalid_pass_to_failure_values == 0
            and invalid_failure_to_pass_values == 0
            and invalid_noisy_verdict_values == 0

            and len(noisy_labels)
                == EXPECTED_MODEL_TRAINING_ROWS

            and missing_label_alignment == 0
            and noisy_label_mismatches == 0
            and label_change_flag_mismatches == 0

            and fit_techniques_pass
            and fit_training_rows_pass
            and fit_class_balance_pass

            and len(rankings)
                == EXPECTED_RANKING_ROWS_PER_CONDITION

            and duplicate_ranking_rows == 0
            and ranking_technique_pass
            and ranking_build_count_pass
            and ranking_rows_per_technique_pass
            and rank_sequence_errors == 0

            and len(build_metrics)
                == EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION

            and metric_invalid_cells == 0
            and build_metric_techniques_pass
            and build_metric_failures_pass

            and project_run_techniques_pass
            and project_run_builds_pass

            and evaluation_immutable
        )


        condition_audit_records.append({
            "ConditionID":
                current_condition_id,

            "NoisePercent":
                noise_percent,

            "RepetitionSeed":
                repetition_seed,

            "ManifestIdentityPass":
                manifest_identity_pass,

            "HashMismatches":
                condition_hash_mismatches,

            "SizeMismatches":
                condition_size_mismatches,

            "ManifestCountsPass":
                manifest_counts_pass,

            "ManifestNoiseCountPass":
                manifest_noise_count_pass,

            "RawTrainingRows":
                len(noise_manifest),

            "FlippedRawRows":
                int(
                    flipped.sum()
                ),

            "RealisedNoisePercent":
                realised_noise_percent,

            "FlipThresholdMismatches":
                flip_threshold_mismatches,

            "UnflippedVerdictMismatches":
                unflipped_verdict_mismatches,

            "PassToFailureFlagMismatches":
                pass_to_failure_mismatches,

            "FailureToPassFlagMismatches":
                failure_to_pass_mismatches,

            "InvalidPassToFailureValues":
                invalid_pass_to_failure_values,

            "InvalidFailureToPassValues":
                invalid_failure_to_pass_values,

            "InvalidNoisyVerdictValues":
                invalid_noisy_verdict_values,

            "ModelTrainingRows":
                len(noisy_labels),

            "MissingLabelAlignments":
                missing_label_alignment,

            "NoisyLabelMismatches":
                noisy_label_mismatches,

            "LabelChangeFlagMismatches":
                label_change_flag_mismatches,

            "ModelFits":
                len(fit_times),

            "RankingRows":
                len(rankings),

            "DuplicateRankingRows":
                duplicate_ranking_rows,

            "RankSequenceErrors":
                rank_sequence_errors,

            "BuildMetricRows":
                len(build_metrics),

            "MetricInvalidCells":
                metric_invalid_cells,

            "ProjectRunRows":
                len(project_run_metrics),

            "EvaluationImmutable":
                evaluation_immutable,

            "Pass":
                condition_pass,
        })


    # --------------------------------------------------------
    # Nested noise masks within the current seed
    # --------------------------------------------------------

    reference_zero = seed_noise_manifests[
        0
    ]


    seed_uniform_mismatches = 0
    seed_identity_mismatches = 0
    seed_original_verdict_mismatches = 0
    seed_nested_mask_violations = 0
    seed_nonmonotonic_flip_counts = 0


    previous_flipped = None
    previous_flip_count = None


    for noise_percent in (
        NOISE_LEVELS_PERCENT
    ):

        current = seed_noise_manifests[
            noise_percent
        ]


        seed_uniform_mismatches += int(
            (
                current[
                    "FlipUniform"
                ].to_numpy(dtype=float)
                != reference_zero[
                    "FlipUniform"
                ].to_numpy(dtype=float)
            ).sum()
        )


        seed_identity_mismatches += int(
            (
                current[
                    [
                        "NoiseRowID",
                        "Build",
                        "Test",
                    ]
                ].to_numpy()
                != reference_zero[
                    [
                        "NoiseRowID",
                        "Build",
                        "Test",
                    ]
                ].to_numpy()
            ).sum()
        )


        seed_original_verdict_mismatches += int(
            (
                current[
                    "OriginalVerdict"
                ].to_numpy(dtype=int)
                != reference_zero[
                    "OriginalVerdict"
                ].to_numpy(dtype=int)
            ).sum()
        )


        current_flipped = current[
            "Flipped"
        ].to_numpy(dtype=bool)


        current_flip_count = int(
            current_flipped.sum()
        )


        if previous_flipped is not None:

            seed_nested_mask_violations += int(
                (
                    previous_flipped
                    & ~current_flipped
                ).sum()
            )


            if (
                current_flip_count
                < previous_flip_count
            ):
                seed_nonmonotonic_flip_counts += 1


        previous_flipped = current_flipped
        previous_flip_count = current_flip_count


    nested_seed_pass = bool(
        seed_uniform_mismatches == 0
        and seed_identity_mismatches == 0
        and seed_original_verdict_mismatches == 0
        and seed_nested_mask_violations == 0
        and seed_nonmonotonic_flip_counts == 0
    )


    nested_noise_audit_records.append({
        "RepetitionSeed":
            repetition_seed,

        "NoiseLevelsAudited":
            len(
                NOISE_LEVELS_PERCENT
            ),

        "UniformStreamMismatches":
            seed_uniform_mismatches,

        "RowIdentityMismatches":
            seed_identity_mismatches,

        "OriginalVerdictMismatches":
            seed_original_verdict_mismatches,

        "NestedMaskViolations":
            seed_nested_mask_violations,

        "NonMonotonicFlipCounts":
            seed_nonmonotonic_flip_counts,

        "Pass":
            nested_seed_pass,
    })


    print(
        "Audited seed:",
        repetition_seed,
        "/",
        len(
            REPETITION_SEEDS
        ),
    )


# ------------------------------------------------------------
# 8. COMBINE AUDIT OUTPUTS
# ------------------------------------------------------------

condition_audit = pd.DataFrame(
    condition_audit_records
)

nested_noise_audit = pd.DataFrame(
    nested_noise_audit_records
)

baseline_signatures = pd.DataFrame(
    baseline_signature_records
)

all_noise_summary = pd.concat(
    all_noise_summaries,
    ignore_index=True,
)

all_fit_times_frame = pd.concat(
    all_fit_times,
    ignore_index=True,
)

all_build_metrics_frame = pd.concat(
    all_build_metrics,
    ignore_index=True,
)

all_project_run_metrics_frame = pd.concat(
    all_project_run_metrics,
    ignore_index=True,
)


# ------------------------------------------------------------
# 9. BASELINE INVARIANCE AUDIT
# ------------------------------------------------------------

baseline_audit_records = []


for repetition_seed in REPETITION_SEEDS:

    seed_rows = baseline_signatures[
        baseline_signatures[
            "RepetitionSeed"
        ].eq(
            repetition_seed
        )
    ]


    random_unique_signatures = int(
        seed_rows[
            "RandomSignature"
        ].nunique()
    )


    qtf_unique_signatures = int(
        seed_rows[
            "QTFAvgSignature"
        ].nunique()
    )


    latestfail_unique_signatures = int(
        seed_rows[
            "LatestFailSignature"
        ].nunique()
    )


    baseline_audit_records.append({
        "RepetitionSeed":
            repetition_seed,

        "ConditionsCompared":
            len(seed_rows),

        "RandomUniqueSignatures":
            random_unique_signatures,

        "RandomInvariant":
            random_unique_signatures
            == 1,

        "QTFAvgUniqueSignatures":
            qtf_unique_signatures,

        "QTFAvgInvariant":
            qtf_unique_signatures
            == 1,

        "LatestFailUniqueSignatures":
            latestfail_unique_signatures,

        "LatestFailRespondedToNoise":
            latestfail_unique_signatures
            > 1,

        "Pass":
            (
                random_unique_signatures
                == 1
                and qtf_unique_signatures
                == 1
            ),
    })


baseline_invariance_audit = pd.DataFrame(
    baseline_audit_records
)


qtf_global_unique_signatures = int(
    baseline_signatures[
        "QTFAvgSignature"
    ].nunique()
)


random_invariance_failures = int(
    (
        ~baseline_invariance_audit[
            "RandomInvariant"
        ]
    ).sum()
)


qtf_invariance_failures = int(
    (
        ~baseline_invariance_audit[
            "QTFAvgInvariant"
        ]
    ).sum()
)


latestfail_responsive_seeds = int(
    baseline_invariance_audit[
        "LatestFailRespondedToNoise"
    ].sum()
)


# ------------------------------------------------------------
# 10. GLOBAL COUNTS
# ------------------------------------------------------------

failed_conditions = int(
    (
        ~condition_audit[
            "Pass"
        ]
    ).sum()
)


failed_nested_seeds = int(
    (
        ~nested_noise_audit[
            "Pass"
        ]
    ).sum()
)


failed_baseline_seeds = int(
    (
        ~baseline_invariance_audit[
            "Pass"
        ]
    ).sum()
)


zero_noise_flip_failures = int(
    (
        condition_audit.loc[
            condition_audit[
                "NoisePercent"
            ].eq(0),
            "FlippedRawRows",
        ]
        != 0
    ).sum()
)


total_nested_mask_violations = int(
    nested_noise_audit[
        "NestedMaskViolations"
    ].sum()
)


total_uniform_stream_mismatches = int(
    nested_noise_audit[
        "UniformStreamMismatches"
    ].sum()
)


aggregated_metric_values = (
    all_build_metrics_frame[
        [
            "APFD",
            "APFDc",
        ]
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .to_numpy(dtype=float)
)


aggregated_invalid_metric_cells = int(
    (
        ~np.isfinite(
            aggregated_metric_values
        )
        |
        (
            aggregated_metric_values
            < 0
        )
        |
        (
            aggregated_metric_values
            > 1
        )
    ).sum()
)


# ------------------------------------------------------------
# 11. OVERALL AUDIT
# ------------------------------------------------------------

overall_audit_records = [
    {
        "Check":
            "Step 9 passed",

        "Expected":
            expected_step9_status,

        "Actual":
            step9_status.get(
                "Status"
            ),

        "Pass":
            step9_status.get(
                "Status"
            )
            == expected_step9_status,
    },

    {
        "Check":
            "Expected raw file set",

        "Expected":
            EXPECTED_RAW_FILES,

        "Actual":
            len(actual_raw_files),

        "Pass":
            (
                len(actual_raw_files)
                == EXPECTED_RAW_FILES
                and len(missing_raw_files)
                == 0
                and len(unexpected_raw_files)
                == 0
            ),
    },

    {
        "Check":
            "Missing raw files",

        "Expected":
            0,

        "Actual":
            len(missing_raw_files),

        "Pass":
            len(missing_raw_files)
            == 0,
    },

    {
        "Check":
            "Unexpected raw files",

        "Expected":
            0,

        "Actual":
            len(unexpected_raw_files),

        "Pass":
            len(unexpected_raw_files)
            == 0,
    },

    {
        "Check":
            "Raw bytes match Step 9",

        "Expected":
            int(
                step9_status[
                    "RawBytes"
                ]
            ),

        "Actual":
            raw_total_bytes,

        "Pass":
            raw_total_bytes
            == int(
                step9_status[
                    "RawBytes"
                ]
            ),
    },

    {
        "Check":
            "Manifest hash mismatches",

        "Expected":
            0,

        "Actual":
            manifest_hash_mismatches,

        "Pass":
            manifest_hash_mismatches
            == 0,
    },

    {
        "Check":
            "Manifest size mismatches",

        "Expected":
            0,

        "Actual":
            manifest_size_mismatches,

        "Pass":
            manifest_size_mismatches
            == 0,
    },

    {
        "Check":
            "Conditions audited",

        "Expected":
            EXPECTED_CONDITIONS,

        "Actual":
            len(condition_audit),

        "Pass":
            len(condition_audit)
            == EXPECTED_CONDITIONS,
    },

    {
        "Check":
            "Failed condition audits",

        "Expected":
            0,

        "Actual":
            failed_conditions,

        "Pass":
            failed_conditions
            == 0,
    },

    {
        "Check":
            "Aggregated ML fits",

        "Expected":
            EXPECTED_ML_FITS,

        "Actual":
            len(
                all_fit_times_frame
            ),

        "Pass":
            len(
                all_fit_times_frame
            )
            == EXPECTED_ML_FITS,
    },

    {
        "Check":
            "Total ranking rows",

        "Expected":
            EXPECTED_TOTAL_RANKING_ROWS,

        "Actual":
            total_ranking_rows,

        "Pass":
            total_ranking_rows
            == EXPECTED_TOTAL_RANKING_ROWS,
    },

    {
        "Check":
            "Duplicate ranking rows",

        "Expected":
            0,

        "Actual":
            total_duplicate_ranking_rows,

        "Pass":
            total_duplicate_ranking_rows
            == 0,
    },

    {
        "Check":
            "Aggregated build-metric rows",

        "Expected":
            EXPECTED_TOTAL_BUILD_METRIC_ROWS,

        "Actual":
            len(
                all_build_metrics_frame
            ),

        "Pass":
            len(
                all_build_metrics_frame
            )
            == EXPECTED_TOTAL_BUILD_METRIC_ROWS,
    },

    {
        "Check":
            "Aggregated project-run rows",

        "Expected":
            EXPECTED_TOTAL_PROJECT_RUN_ROWS,

        "Actual":
            len(
                all_project_run_metrics_frame
            ),

        "Pass":
            len(
                all_project_run_metrics_frame
            )
            == EXPECTED_TOTAL_PROJECT_RUN_ROWS,
    },

    {
        "Check":
            "Invalid APFD/APFDc cells",

        "Expected":
            0,

        "Actual":
            aggregated_invalid_metric_cells,

        "Pass":
            aggregated_invalid_metric_cells
            == 0,
    },

    {
        "Check":
            "Zero-noise flip failures",

        "Expected":
            0,

        "Actual":
            zero_noise_flip_failures,

        "Pass":
            zero_noise_flip_failures
            == 0,
    },

    {
        "Check":
            "Nested-mask violations",

        "Expected":
            0,

        "Actual":
            total_nested_mask_violations,

        "Pass":
            total_nested_mask_violations
            == 0,
    },

    {
        "Check":
            "Uniform-stream mismatches",

        "Expected":
            0,

        "Actual":
            total_uniform_stream_mismatches,

        "Pass":
            total_uniform_stream_mismatches
            == 0,
    },

    {
        "Check":
            "Failed nested-noise seeds",

        "Expected":
            0,

        "Actual":
            failed_nested_seeds,

        "Pass":
            failed_nested_seeds
            == 0,
    },

    {
        "Check":
            "Random invariance failures",

        "Expected":
            0,

        "Actual":
            random_invariance_failures,

        "Pass":
            random_invariance_failures
            == 0,
    },

    {
        "Check":
            "QTF-Avg invariance failures",

        "Expected":
            0,

        "Actual":
            qtf_invariance_failures,

        "Pass":
            qtf_invariance_failures
            == 0,
    },

    {
        "Check":
            "QTF-Avg global signatures",

        "Expected":
            1,

        "Actual":
            qtf_global_unique_signatures,

        "Pass":
            qtf_global_unique_signatures
            == 1,
    },

    {
        "Check":
            "LatestFail responsive seeds",

        "Expected":
            "> 0",

        "Actual":
            latestfail_responsive_seeds,

        "Pass":
            latestfail_responsive_seeds
            > 0,
    },

    {
        "Check":
            "Evaluation immutable",

        "Expected":
            True,

        "Actual":
            all_evaluation_immutable,

        "Pass":
            all_evaluation_immutable,
    },
]


overall_audit = pd.DataFrame(
    overall_audit_records
)


failed_overall_checks = (
    overall_audit[
        ~overall_audit[
            "Pass"
        ]
    ]
    .copy()
)


print("\nOverall Step 10 audit:")

display(
    overall_audit
)


if failed_conditions:

    print("\nFailed condition audits:")

    display(
        condition_audit[
            ~condition_audit[
                "Pass"
            ]
        ]
    )


if failed_nested_seeds:

    print(
        "\nFailed nested-mask seed audits:"
    )

    display(
        nested_noise_audit[
            ~nested_noise_audit[
                "Pass"
            ]
        ]
    )


if failed_baseline_seeds:

    print(
        "\nFailed baseline seed audits:"
    )

    display(
        baseline_invariance_audit[
            ~baseline_invariance_audit[
                "Pass"
            ]
        ]
    )


if not failed_overall_checks.empty:

    print("\nFailed overall checks:")

    display(
        failed_overall_checks
    )

    raise RuntimeError(
        "PROJECT 8 STEP 10 DID NOT PASS.\n"
        "Do not package or freeze Project 8."
    )


# ------------------------------------------------------------
# 12. WRITE AUDIT AND AGGREGATED ARTEFACTS
# ------------------------------------------------------------

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    ALL_NOISE_SUMMARY_PATH,
    all_noise_summary,
)


atomic_write_csv(
    ALL_FIT_TIMES_PATH,
    all_fit_times_frame,
)


atomic_write_csv(
    ALL_BUILD_METRICS_PATH,
    all_build_metrics_frame,
    compression="gzip",
)


atomic_write_csv(
    ALL_PROJECT_RUN_METRICS_PATH,
    all_project_run_metrics_frame,
)


atomic_write_csv(
    CONDITION_AUDIT_PATH,
    condition_audit,
)


atomic_write_csv(
    NESTED_NOISE_AUDIT_PATH,
    nested_noise_audit,
)


atomic_write_csv(
    BASELINE_AUDIT_PATH,
    baseline_invariance_audit,
)


atomic_write_csv(
    RAW_FILE_MANIFEST_PATH,
    raw_file_manifest,
    compression="gzip",
)


atomic_write_csv(
    OVERALL_AUDIT_PATH,
    overall_audit,
)


written_outputs = [
    ALL_NOISE_SUMMARY_PATH,
    ALL_FIT_TIMES_PATH,
    ALL_BUILD_METRICS_PATH,
    ALL_PROJECT_RUN_METRICS_PATH,
    CONDITION_AUDIT_PATH,
    NESTED_NOISE_AUDIT_PATH,
    BASELINE_AUDIT_PATH,
    RAW_FILE_MANIFEST_PATH,
    OVERALL_AUDIT_PATH,
]


output_hashes = {
    path.name:
        calculate_sha256(
            path
        )
    for path in written_outputs
}


step10_status_text = (
    "PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED"
)


raw_audit_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "RawFreezeCandidate": {
        "Root":
            str(RAW_ROOT),

        "Files":
            len(actual_raw_files),

        "Bytes":
            raw_total_bytes,

        "RootSHA256":
            raw_root_sha256,

        "MissingFiles":
            len(missing_raw_files),

        "UnexpectedFiles":
            len(unexpected_raw_files),

        "ManifestHashMismatches":
            manifest_hash_mismatches,

        "ManifestSizeMismatches":
            manifest_size_mismatches,
    },

    "ExperimentTotals": {
        "Conditions":
            len(condition_audit),

        "MLFits":
            len(
                all_fit_times_frame
            ),

        "RankingRows":
            total_ranking_rows,

        "BuildMetricRows":
            len(
                all_build_metrics_frame
            ),

        "ProjectRunRows":
            len(
                all_project_run_metrics_frame
            ),
    },

    "NoiseAudit": {
        "SeedsAudited":
            len(
                nested_noise_audit
            ),

        "NestedMaskViolations":
            total_nested_mask_violations,

        "UniformStreamMismatches":
            total_uniform_stream_mismatches,

        "ZeroNoiseFlipFailures":
            zero_noise_flip_failures,
    },

    "BaselineAudit": {
        "RandomInvarianceFailures":
            random_invariance_failures,

        "QTFAvgInvarianceFailures":
            qtf_invariance_failures,

        "QTFAvgGlobalUniqueSignatures":
            qtf_global_unique_signatures,

        "LatestFailResponsiveSeeds":
            latestfail_responsive_seeds,
    },

    "Metrics": {
        "InvalidAPFDOrAPFDcCells":
            aggregated_invalid_metric_cells,
    },

    "EvaluationImmutable":
        all_evaluation_immutable,

    "AuditChecks":
        len(overall_audit),

    "FailedAuditChecks":
        len(
            failed_overall_checks
        ),

    "Outputs": {
        path.name: {
            "Path":
                str(path),

            "SHA256":
                output_hashes[
                    path.name
                ],
        }
        for path in written_outputs
    },

    "CompletionRegistryModified":
        False,

    "Projects1To7Modified":
        False,

    "Status":
        step10_status_text,
}


atomic_write_json(
    RAW_AUDIT_REPORT_PATH,
    raw_audit_report,
)


step10_status = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        step10_status_text,

    "RawFiles":
        len(actual_raw_files),

    "RawBytes":
        raw_total_bytes,

    "RawRootSHA256":
        raw_root_sha256,

    "Conditions":
        len(condition_audit),

    "MLFits":
        len(
            all_fit_times_frame
        ),

    "RankingRows":
        total_ranking_rows,

    "BuildMetricRows":
        len(
            all_build_metrics_frame
        ),

    "ProjectRunRows":
        len(
            all_project_run_metrics_frame
        ),

    "FailedConditions":
        failed_conditions,

    "NestedMaskViolations":
        total_nested_mask_violations,

    "RandomInvarianceFailures":
        random_invariance_failures,

    "QTFAvgInvarianceFailures":
        qtf_invariance_failures,

    "EvaluationImmutable":
        all_evaluation_immutable,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP10_STATUS_PATH,
    step10_status,
)


expected_outputs = (
    written_outputs
    + [
        RAW_AUDIT_REPORT_PATH,
        STEP10_STATUS_PATH,
    ]
)


missing_outputs = [
    str(path)
    for path in expected_outputs
    if not path.exists()
]


if missing_outputs:
    raise RuntimeError(
        "Step 10 output files are missing:\n"
        + "\n".join(
            missing_outputs
        )
    )


# ------------------------------------------------------------
# 13. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 92)
print("=== PROJECT 8 STEP 10 RESULT ===")
print("=" * 92)

print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nRaw result freeze candidate:")

print(
    "Raw files:",
    len(actual_raw_files),
)

print(
    "Raw bytes:",
    raw_total_bytes,
)

print(
    "Raw root SHA-256:",
    raw_root_sha256,
)

print(
    "Missing raw files:",
    len(missing_raw_files),
)

print(
    "Unexpected raw files:",
    len(unexpected_raw_files),
)

print(
    "Manifest hash mismatches:",
    manifest_hash_mismatches,
)

print(
    "Manifest size mismatches:",
    manifest_size_mismatches,
)


print("\nExperiment totals:")

print(
    "Conditions:",
    len(condition_audit),
)

print(
    "ML fits:",
    len(
        all_fit_times_frame
    ),
)

print(
    "Ranking rows:",
    total_ranking_rows,
)

print(
    "Build-metric rows:",
    len(
        all_build_metrics_frame
    ),
)

print(
    "Project-run rows:",
    len(
        all_project_run_metrics_frame
    ),
)


print("\nNoise audit:")

print(
    "Seeds audited:",
    len(
        nested_noise_audit
    ),
)

print(
    "Nested-mask violations:",
    total_nested_mask_violations,
)

print(
    "Uniform-stream mismatches:",
    total_uniform_stream_mismatches,
)

print(
    "Zero-noise flip failures:",
    zero_noise_flip_failures,
)


print("\nBaseline audit:")

print(
    "Random invariance failures:",
    random_invariance_failures,
)

print(
    "QTF-Avg invariance failures:",
    qtf_invariance_failures,
)

print(
    "QTF-Avg global unique signatures:",
    qtf_global_unique_signatures,
)

print(
    "LatestFail responsive seeds:",
    latestfail_responsive_seeds,
)


print("\nMetric and evaluation audit:")

print(
    "Invalid APFD/APFDc cells:",
    aggregated_invalid_metric_cells,
)

print(
    "Evaluation immutable:",
    all_evaluation_immutable,
)


print("\nValidation:")

print(
    "Overall checks:",
    len(overall_audit),
)

print(
    "Failed overall checks:",
    len(
        failed_overall_checks
    ),
)

print(
    "Failed conditions:",
    failed_conditions,
)

print(
    "Failed nested-mask seeds:",
    failed_nested_seeds,
)

print(
    "Failed baseline seeds:",
    failed_baseline_seeds,
)


print("\nStep 10 outputs:")

for output_path in expected_outputs:
    print(output_path)


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–7 modified:")
print(0)


print(
    "\nSTATUS:",
    step10_status_text,
)

print("=" * 92)

=== PROJECT 8 STEP 10 V2: COMPLETE RAW-RESULT AUDIT AND AGGREGATION ===

Step 9 validation:
Status: PASS_PROJECT_8_FULL_270_CONDITION_RUN_COMPLETE
Completed conditions: 270
Raw files reported: 2160
Raw bytes reported: 179766494

Raw inventory:
Expected files: 2160
Actual files: 2160
Missing files: 0
Unexpected files: 0

Hashing all raw files...
Hashed: 200 / 2160
Hashed: 400 / 2160
Hashed: 600 / 2160
Hashed: 800 / 2160
Hashed: 1000 / 2160
Hashed: 1200 / 2160
Hashed: 1400 / 2160
Hashed: 1600 / 2160
Hashed: 1800 / 2160
Hashed: 2000 / 2160
Hashed: 2160 / 2160

Raw-root inventory hash:
19ae21c5d8524c358796e28f23577fbf03bfd31f7d4b530372c6b4d8a492887c

Auditing all 270 conditions...
Audited seed: 1 / 30
Audited seed: 2 / 30
Audited seed: 3 / 30
Audited seed: 4 / 30
Audited seed: 5 / 30
Audited seed: 6 / 30
Audited seed: 7 / 30
Audited seed: 8 / 30
Audited seed: 9 / 30
Audited seed: 10 / 30
Audited seed: 11 / 30
Audited seed: 12 / 30
Audited seed: 13 / 30
Audited seed: 14 / 30
Audited seed: 1

,Check,Expected,Actual,Pass
0,Step 9 passed,PASS_PROJECT_8_FULL_270_CONDITION_RUN_COMPLETE,PASS_PROJECT_8_FULL_270_CONDITION_RUN_COMPLETE,True
1,Expected raw file set,2160,2160,True
2,Missing raw files,0,0,True
3,Unexpected raw files,0,0,True
4,Raw bytes match Step 9,179766494,179766494,True
5,Manifest hash mismatches,0,0,True
6,Manifest size mismatches,0,0,True
7,Conditions audited,270,270,True
8,Failed condition audits,0,0,True
9,Aggregated ML fits,1080,1080,True




=== PROJECT 8 STEP 10 RESULT ===

Project identity:
Project number: 8
Project: optimatika@ojAlgo
Project slug: optimatika__ojAlgo

Raw result freeze candidate:
Raw files: 2160
Raw bytes: 179766494
Raw root SHA-256: 19ae21c5d8524c358796e28f23577fbf03bfd31f7d4b530372c6b4d8a492887c
Missing raw files: 0
Unexpected raw files: 0
Manifest hash mismatches: 0
Manifest size mismatches: 0

Experiment totals:
Conditions: 270
ML fits: 1080
Ranking rows: 4347000
Build-metric rows: 30240
Project-run rows: 1890

Noise audit:
Seeds audited: 30
Nested-mask violations: 0
Uniform-stream mismatches: 0
Zero-noise flip failures: 0

Baseline audit:
Random invariance failures: 0
QTF-Avg invariance failures: 0
QTF-Avg global unique signatures: 1
LatestFail responsive seeds: 30

Metric and evaluation audit:
Invalid APFD/APFDc cells: 0
Evaluation immutable: True

Validation:
Overall checks: 24
Failed overall checks: 0
Failed conditions: 0
Failed nested-mask seeds: 0
Failed baseline seeds: 0

Step 10 outputs:
/c

In [ ]:
# ============================================================
# PROJECT 8 — STEP 11A V2
# EXACT FINAL-PACKAGE BLUEPRINT AND SOURCE MAPPING
#
# Fix:
# - reads FinalPackageDirectory and FinalPackageInventory
#   directly from the Project 7 completion checkpoint
# - does not confuse the six-row source manifest with the
#   72-row final-package inventory
# - validates Project 7's actual frozen package
# - maps every Project 7 package entry to one Project 8 source
#
# This cell does NOT:
# - create the Project 8 final package
# - modify raw Project 8 files
# - modify Projects 1–7
# - modify the completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import re

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. PROJECT CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 8
PROJECT_NAME = "optimatika@ojAlgo"
PROJECT_SLUG = "optimatika__ojAlgo"
PROJECT_SHORT_NAME = "ojalgo"

REFERENCE_PROJECT_NUMBER = 7
REFERENCE_PROJECT_NAME = "CompEvol@beast2"
REFERENCE_PROJECT_SLUG = "CompEvol__beast2"
REFERENCE_PROJECT_SHORT_NAME = "beast2"

EXPECTED_PROJECT8_RAW_FILES = 2160
EXPECTED_PROJECT8_RAW_BYTES = 179766494

EXPECTED_PROJECT8_RAW_SHA256 = (
    "19ae21c5d8524c358796e28f23577fbf03bfd31f7d4b530372c6b4d8a492887c"
)

EXPECTED_STEP10_STATUS = (
    "PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED"
)

EXPECTED_REFERENCE_PACKAGE_FILES = 72
EXPECTED_PROJECT8_SOURCE_FILES = 72


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

AGGREGATED_ROOT = (
    RESULTS_DIR
    / "Aggregated"
)

PROJECT8_AGGREGATED_DIR = (
    AGGREGATED_ROOT
    / PROJECT_SLUG
)

PROJECT8_PREFLIGHT_DIR = (
    PROJECT8_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_preflight"
)

PROJECT8_RAW_AUDIT_DIR = (
    PROJECT8_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_raw_audit"
)

PROJECT8_SMOKE_DIR = (
    PROJECT8_PREFLIGHT_DIR
    / f"{PROJECT_SHORT_NAME}_smoke_test"
)

STEP10_STATUS_PATH = (
    PROJECT8_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_step10_status.json"
)

REFERENCE_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_07_selection_checkpoint.json"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)


# Corrected Step 11A output directory

STEP11A_DIR = (
    PROJECT8_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_step11a_package_blueprint"
)

STEP11A_STATUS_PATH = (
    PROJECT8_AGGREGATED_DIR
    / f"{PROJECT_SHORT_NAME}_step11a_status.json"
)


REFERENCE_PATHS_PATH = (
    STEP11A_DIR
    / f"{PROJECT_SHORT_NAME}_reference_final_package_paths.json"
)

REFERENCE_INVENTORY_NORMALISED_PATH = (
    STEP11A_DIR
    / f"{PROJECT_SHORT_NAME}_reference_final_package_inventory.csv"
)

REFERENCE_ACTUAL_PACKAGE_INVENTORY_PATH = (
    STEP11A_DIR
    / f"{PROJECT_SHORT_NAME}_reference_actual_package_inventory.csv"
)

PROJECT8_SOURCE_INVENTORY_PATH = (
    STEP11A_DIR
    / f"{PROJECT_SHORT_NAME}_package_source_inventory.csv.gz"
)

PACKAGE_MAPPING_PATH = (
    STEP11A_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_mapping.csv"
)

STEP11A_AUDIT_PATH = (
    STEP11A_DIR
    / f"{PROJECT_SHORT_NAME}_step11a_v2_audit.csv"
)

STEP11A_REPORT_PATH = (
    STEP11A_DIR
    / f"{PROJECT_SHORT_NAME}_step11a_v2_report.json"
)


print("=" * 94)
print("=== PROJECT 8 STEP 11A V2: EXACT FINAL-PACKAGE BLUEPRINT DISCOVERY ===")
print("=" * 94)


# ------------------------------------------------------------
# 3. GENERAL HELPERS
# ------------------------------------------------------------

def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def json_safe(value):

    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:

        if pd.isna(value):
            return None

    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def atomic_write_csv(
    path,
    dataframe,
    compression=None,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
        compression=compression,
    )

    temporary_path.replace(path)


def recursive_key_lookup(
    value,
    requested_key,
):
    """
    Find a JSON value by key, case-insensitively.
    """

    requested_normalised = (
        str(requested_key)
        .strip()
        .lower()
    )

    if isinstance(value, dict):

        for key, child in value.items():

            if (
                str(key)
                .strip()
                .lower()
                == requested_normalised
            ):
                return child

        for child in value.values():

            result = recursive_key_lookup(
                child,
                requested_key,
            )

            if result is not None:
                return result

    elif isinstance(value, list):

        for child in value:

            result = recursive_key_lookup(
                child,
                requested_key,
            )

            if result is not None:
                return result

    return None


def normalise_column_name(
    column_name,
):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(column_name).lower(),
    )


def detect_column(
    dataframe,
    candidate_names,
    required=True,
):
    normalised_columns = {
        normalise_column_name(column):
            column
        for column in dataframe.columns
    }

    for candidate_name in candidate_names:

        candidate_normalised = (
            normalise_column_name(
                candidate_name
            )
        )

        if candidate_normalised in normalised_columns:

            return normalised_columns[
                candidate_normalised
            ]

    if required:

        raise RuntimeError(
            "Could not identify a required column.\n"
            f"Candidates: {candidate_names}\n"
            f"Available columns: {list(dataframe.columns)}"
        )

    return None


def path_is_inside(
    path,
    parent,
):
    try:

        Path(path).resolve().relative_to(
            Path(parent).resolve()
        )

        return True

    except ValueError:
        return False


def transform_reference_text(
    value,
):
    """
    Transform Project 7 identifiers into Project 8
    identifiers while preserving the rest of the path.
    """

    transformed = str(value)

    replacements = [
        (
            REFERENCE_PROJECT_SLUG,
            PROJECT_SLUG,
        ),
        (
            REFERENCE_PROJECT_NAME,
            PROJECT_NAME,
        ),
        (
            REFERENCE_PROJECT_SHORT_NAME,
            PROJECT_SHORT_NAME,
        ),
        (
            "project_07",
            "project_08",
        ),
        (
            "Project_07",
            "Project_08",
        ),
        (
            "PROJECT_07",
            "PROJECT_08",
        ),
        (
            "project-07",
            "project-08",
        ),
        (
            "Project 7",
            "Project 8",
        ),
        (
            "PROJECT 7",
            "PROJECT 8",
        ),
    ]

    for old_value, new_value in replacements:

        transformed = transformed.replace(
            old_value,
            new_value,
        )

    return transformed


def normalise_path_text(
    value,
):
    text = str(value).replace(
        "\\",
        "/",
    )

    text = re.sub(
        r"/+",
        "/",
        text,
    )

    return text.strip()


def package_relative_path(
    value,
    package_directory,
):
    """
    Convert an inventory path into a package-relative path.
    """

    text = normalise_path_text(
        value
    )

    package_directory_text = normalise_path_text(
        package_directory
    ).rstrip("/")

    if (
        text == package_directory_text
    ):
        return ""

    package_prefix = (
        package_directory_text
        + "/"
    )

    if text.startswith(
        package_prefix
    ):

        return text[
            len(package_prefix):
        ]

    package_directory_name = (
        Path(
            package_directory
        ).name
    )

    name_marker = (
        package_directory_name
        + "/"
    )

    if name_marker in text:

        return text.split(
            name_marker,
            1,
        )[1]

    return text.lstrip("./")


def matching_tokens(
    value,
):
    text = transform_reference_text(
        value
    ).lower()

    text = re.sub(
        r"[^a-z0-9]+",
        " ",
        text,
    )

    ignored_tokens = {
        "content",
        "drive",
        "mydrive",
        "thesis",
        "experiment",
        "results",
        "aggregated",
        "notes",
        "raw",
        "final",
        "package",
        "csv",
        "gz",
        "json",
        "txt",
        "sha256",
        "v1",
        "v2",
        "project",
        "08",
        PROJECT_SHORT_NAME.lower(),
        "optimatika",
        "ojalgo",
    }

    return {
        token
        for token in text.split()
        if token
        and token not in ignored_tokens
    }


def token_similarity(
    first_value,
    second_value,
):
    first_tokens = matching_tokens(
        first_value
    )

    second_tokens = matching_tokens(
        second_value
    )

    if (
        not first_tokens
        and not second_tokens
    ):
        return 1.0

    union = (
        first_tokens
        | second_tokens
    )

    if not union:
        return 0.0

    return (
        len(
            first_tokens
            & second_tokens
        )
        / len(union)
    )


def extension_signature(
    value,
):
    return "".join(
        Path(
            str(value)
        ).suffixes
    ).lower()


def classify_file(
    path,
):
    name = Path(path).name.lower()

    if "protocol" in name:
        return "protocol"

    if (
        "configuration" in name
        or "config" in name
    ):
        return "configuration"

    if "chronology" in name:
        return "chronology"

    if "predictor" in name:
        return "predictor"

    if "manifest" in name:
        return "manifest"

    if "checkpoint" in name:
        return "checkpoint"

    if "status" in name:
        return "status"

    if "report" in name:
        return "report"

    if "audit" in name:
        return "audit"

    if "metric" in name:
        return "metrics"

    return "other"


# ------------------------------------------------------------
# 4. VALIDATE PROJECT 8 STEP 10
# ------------------------------------------------------------

required_initial_paths = [
    THESIS_ROOT,
    NOTES_DIR,
    PROJECT8_AGGREGATED_DIR,
    PROJECT8_PREFLIGHT_DIR,
    PROJECT8_RAW_AUDIT_DIR,
    STEP10_STATUS_PATH,
    REFERENCE_CHECKPOINT_PATH,
    REGISTRY_PATH,
]


missing_initial_paths = [
    str(path)
    for path in required_initial_paths
    if not path.exists()
]


if missing_initial_paths:

    raise FileNotFoundError(
        "Required Step 11A V2 inputs are missing:\n"
        + "\n".join(
            missing_initial_paths
        )
    )


step10_status = json.loads(
    STEP10_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    step10_status.get("Status")
    != EXPECTED_STEP10_STATUS
):

    raise AssertionError(
        "Project 8 Step 10 has not passed.\n"
        f"Detected: "
        f"{step10_status.get('Status')}"
    )


if (
    int(
        step10_status.get(
            "RawFiles",
            -1,
        )
    )
    != EXPECTED_PROJECT8_RAW_FILES
):

    raise AssertionError(
        "Project 8 raw-file count differs."
    )


if (
    int(
        step10_status.get(
            "RawBytes",
            -1,
        )
    )
    != EXPECTED_PROJECT8_RAW_BYTES
):

    raise AssertionError(
        "Project 8 raw-byte count differs."
    )


if (
    step10_status.get(
        "RawRootSHA256"
    )
    != EXPECTED_PROJECT8_RAW_SHA256
):

    raise AssertionError(
        "Project 8 raw-root SHA-256 differs."
    )


print("\nProject 8 Step 10 validation:")

print(
    "Status:",
    step10_status[
        "Status"
    ],
)

print(
    "Raw files:",
    step10_status[
        "RawFiles"
    ],
)

print(
    "Raw bytes:",
    step10_status[
        "RawBytes"
    ],
)

print(
    "Raw root SHA-256:",
    step10_status[
        "RawRootSHA256"
    ],
)


# ------------------------------------------------------------
# 5. VALIDATE COMPLETION REGISTRY
# ------------------------------------------------------------

registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
)


registry[
    "_ProjectNumber"
] = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="coerce",
)


registry[
    "_Status"
] = (
    registry[
        "Status"
    ]
    .astype(str)
    .str.strip()
    .str.upper()
)


frozen_project_numbers = sorted(
    registry.loc[
        registry[
            "_Status"
        ].eq(
            "COMPLETE_AND_FROZEN"
        ),
        "_ProjectNumber",
    ]
    .dropna()
    .astype(int)
    .tolist()
)


project8_registry_rows = registry[
    registry[
        "_ProjectNumber"
    ].eq(
        PROJECT_NUMBER
    )
]


# ------------------------------------------------------------
# 6. READ EXACT PROJECT 7 FINAL-PACKAGE PATHS
# ------------------------------------------------------------

reference_checkpoint = json.loads(
    REFERENCE_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)


reference_package_directory_value = (
    recursive_key_lookup(
        reference_checkpoint,
        "FinalPackageDirectory",
    )
)

reference_package_inventory_value = (
    recursive_key_lookup(
        reference_checkpoint,
        "FinalPackageInventory",
    )
)

reference_package_validation_value = (
    recursive_key_lookup(
        reference_checkpoint,
        "FinalPackageValidation",
    )
)

reference_package_report_value = (
    recursive_key_lookup(
        reference_checkpoint,
        "FinalPackageReport",
    )
)

reference_package_root_sha256 = (
    recursive_key_lookup(
        reference_checkpoint,
        "FinalPackageRootSHA256",
    )
)

reference_frozen_package_root_sha256 = (
    recursive_key_lookup(
        reference_checkpoint,
        "FrozenFinalPackageRootSHA256",
    )
)


missing_checkpoint_values = []


for key_name, key_value in [
    (
        "FinalPackageDirectory",
        reference_package_directory_value,
    ),
    (
        "FinalPackageInventory",
        reference_package_inventory_value,
    ),
    (
        "FinalPackageValidation",
        reference_package_validation_value,
    ),
    (
        "FinalPackageReport",
        reference_package_report_value,
    ),
]:

    if key_value is None:

        missing_checkpoint_values.append(
            key_name
        )


if missing_checkpoint_values:

    raise RuntimeError(
        "Project 7 checkpoint is missing final-package keys:\n"
        + "\n".join(
            missing_checkpoint_values
        )
    )


REFERENCE_PACKAGE_DIRECTORY = Path(
    reference_package_directory_value
)

REFERENCE_PACKAGE_INVENTORY_PATH = Path(
    reference_package_inventory_value
)

REFERENCE_PACKAGE_VALIDATION_PATH = Path(
    reference_package_validation_value
)

REFERENCE_PACKAGE_REPORT_PATH = Path(
    reference_package_report_value
)


reference_package_paths = {
    "FinalPackageDirectory":
        str(
            REFERENCE_PACKAGE_DIRECTORY
        ),

    "FinalPackageInventory":
        str(
            REFERENCE_PACKAGE_INVENTORY_PATH
        ),

    "FinalPackageValidation":
        str(
            REFERENCE_PACKAGE_VALIDATION_PATH
        ),

    "FinalPackageReport":
        str(
            REFERENCE_PACKAGE_REPORT_PATH
        ),

    "FinalPackageRootSHA256":
        reference_package_root_sha256,

    "FrozenFinalPackageRootSHA256":
        reference_frozen_package_root_sha256,
}


missing_reference_package_paths = [
    path_text
    for path_text in [
        str(
            REFERENCE_PACKAGE_DIRECTORY
        ),
        str(
            REFERENCE_PACKAGE_INVENTORY_PATH
        ),
        str(
            REFERENCE_PACKAGE_VALIDATION_PATH
        ),
        str(
            REFERENCE_PACKAGE_REPORT_PATH
        ),
    ]
    if not Path(
        path_text
    ).exists()
]


if missing_reference_package_paths:

    raise FileNotFoundError(
        "Project 7 final-package paths from its checkpoint "
        "are missing:\n"
        + "\n".join(
            missing_reference_package_paths
        )
    )


print("\nExact Project 7 final-package paths:")

for key, value in (
    reference_package_paths.items()
):

    print(
        f"{key}:",
        value,
    )


# ------------------------------------------------------------
# 7. READ THE EXACT 72-ROW REFERENCE INVENTORY
# ------------------------------------------------------------

reference_inventory_raw = pd.read_csv(
    REFERENCE_PACKAGE_INVENTORY_PATH,
    low_memory=False,
)


relative_path_column = detect_column(
    reference_inventory_raw,
    [
        "PackageRelativePath",
        "RelativePath",
        "Relative_Path",
        "PackagePath",
        "Path",
        "FilePath",
    ],
)


source_path_column = detect_column(
    reference_inventory_raw,
    [
        "SourcePath",
        "OriginalSourcePath",
        "OriginalPath",
        "InputPath",
        "SourceFile",
        "Source",
    ],
    required=False,
)


size_column = detect_column(
    reference_inventory_raw,
    [
        "SizeBytes",
        "Size_Bytes",
        "FileSizeBytes",
        "FileSize",
        "Bytes",
    ],
    required=False,
)


hash_column = detect_column(
    reference_inventory_raw,
    [
        "SHA256",
        "SHA_256",
        "FileSHA256",
        "FileHash",
        "Hash",
    ],
    required=False,
)


reference_inventory = pd.DataFrame({
    "ReferenceInventoryRow":
        np.arange(
            1,
            len(
                reference_inventory_raw
            ) + 1,
        ),

    "ReferenceOriginalPathValue":
        reference_inventory_raw[
            relative_path_column
        ].astype(str),
})


reference_inventory[
    "ReferencePackageRelativePath"
] = reference_inventory[
    "ReferenceOriginalPathValue"
].map(
    lambda value:
        package_relative_path(
            value,
            REFERENCE_PACKAGE_DIRECTORY,
        )
)


if source_path_column is not None:

    reference_inventory[
        "ReferenceSourcePath"
    ] = reference_inventory_raw[
        source_path_column
    ].astype(str)

else:

    reference_inventory[
        "ReferenceSourcePath"
    ] = None


if size_column is not None:

    reference_inventory[
        "ReferenceSizeBytes"
    ] = pd.to_numeric(
        reference_inventory_raw[
            size_column
        ],
        errors="coerce",
    )

else:

    reference_inventory[
        "ReferenceSizeBytes"
    ] = np.nan


if hash_column is not None:

    reference_inventory[
        "ReferenceSHA256"
    ] = (
        reference_inventory_raw[
            hash_column
        ]
        .astype(str)
        .str.strip()
    )

else:

    reference_inventory[
        "ReferenceSHA256"
    ] = None


reference_inventory[
    "ReferenceFileName"
] = reference_inventory[
    "ReferencePackageRelativePath"
].map(
    lambda value:
        Path(value).name
)


reference_inventory[
    "ReferenceFileType"
] = reference_inventory[
    "ReferencePackageRelativePath"
].map(
    classify_file
)


print("\nExact Project 7 final-package inventory:")

print(
    "Rows:",
    len(
        reference_inventory
    ),
)

print(
    "Detected relative-path column:",
    relative_path_column,
)

print(
    "Detected source-path column:",
    source_path_column,
)

print(
    "Detected size column:",
    size_column,
)

print(
    "Detected hash column:",
    hash_column,
)


# ------------------------------------------------------------
# 8. INVENTORY ACTUAL PROJECT 7 PACKAGE DIRECTORY
# ------------------------------------------------------------

reference_actual_files = sorted([
    path
    for path in (
        REFERENCE_PACKAGE_DIRECTORY
        .rglob("*")
    )
    if path.is_file()
])


reference_actual_records = []


for path in reference_actual_files:

    reference_actual_records.append({
        "ReferencePackageRelativePath":
            path.relative_to(
                REFERENCE_PACKAGE_DIRECTORY
            ).as_posix(),

        "ReferenceActualPath":
            str(path),

        "ActualSizeBytes":
            int(
                path.stat().st_size
            ),

        "ActualSHA256":
            calculate_sha256(path),
    })


reference_actual_inventory = pd.DataFrame(
    reference_actual_records
)


reference_inventory_relative_set = set(
    reference_inventory[
        "ReferencePackageRelativePath"
    ].astype(str)
)


reference_actual_relative_set = set(
    reference_actual_inventory[
        "ReferencePackageRelativePath"
    ].astype(str)
)


reference_inventory_missing_files = sorted(
    reference_inventory_relative_set
    - reference_actual_relative_set
)


reference_inventory_unexpected_files = sorted(
    reference_actual_relative_set
    - reference_inventory_relative_set
)


reference_validation_merge = (
    reference_inventory
    .merge(
        reference_actual_inventory,
        on="ReferencePackageRelativePath",
        how="left",
        validate="one_to_one",
    )
)


reference_size_mismatches = 0
reference_hash_mismatches = 0


if size_column is not None:

    comparable_size_rows = (
        reference_validation_merge[
            "ReferenceSizeBytes"
        ].notna()
        &
        reference_validation_merge[
            "ActualSizeBytes"
        ].notna()
    )

    reference_size_mismatches = int(
        (
            reference_validation_merge.loc[
                comparable_size_rows,
                "ReferenceSizeBytes",
            ].astype(np.int64)
            !=
            reference_validation_merge.loc[
                comparable_size_rows,
                "ActualSizeBytes",
            ].astype(np.int64)
        ).sum()
    )


if hash_column is not None:

    comparable_hash_rows = (
        reference_validation_merge[
            "ReferenceSHA256"
        ].notna()
        &
        reference_validation_merge[
            "ActualSHA256"
        ].notna()
    )

    reference_hash_mismatches = int(
        (
            reference_validation_merge.loc[
                comparable_hash_rows,
                "ReferenceSHA256",
            ]
            .astype(str)
            .str.lower()
            !=
            reference_validation_merge.loc[
                comparable_hash_rows,
                "ActualSHA256",
            ]
            .astype(str)
            .str.lower()
        ).sum()
    )


reference_actual_package_bytes = int(
    reference_actual_inventory[
        "ActualSizeBytes"
    ].sum()
)


print("\nProject 7 actual frozen package:")

print(
    "Files:",
    len(
        reference_actual_inventory
    ),
)

print(
    "Bytes:",
    reference_actual_package_bytes,
)

print(
    "Inventory missing files:",
    len(
        reference_inventory_missing_files
    ),
)

print(
    "Inventory unexpected files:",
    len(
        reference_inventory_unexpected_files
    ),
)

print(
    "Inventory size mismatches:",
    reference_size_mismatches,
)

print(
    "Inventory SHA-256 mismatches:",
    reference_hash_mismatches,
)


# ------------------------------------------------------------
# 9. INVENTORY PROJECT 8 PACKAGE SOURCE FILES
# ------------------------------------------------------------

project8_inventory_roots = [
    PROJECT8_AGGREGATED_DIR,
    NOTES_DIR,
]


project8_source_records = []


for inventory_root in (
    project8_inventory_roots
):

    for path in sorted(
        inventory_root.rglob("*")
    ):

        if not path.is_file():
            continue


        # Exclude every Step 11A diagnostic output.
        if path_is_inside(
            path,
            STEP11A_DIR,
        ):
            continue


        # Exclude the prior Step 11A status file.
        if (
            path.resolve()
            == STEP11A_STATUS_PATH.resolve()
        ):
            continue


        # Exclude any already-created final package.
        if any(
            "final_package"
            in part.lower()
            for part in path.parts
        ):
            continue


        path_text_lower = str(
            path
        ).lower()

        file_name_lower = (
            path.name.lower()
        )


        is_project8_file = bool(
            PROJECT_SLUG.lower()
            in path_text_lower

            or PROJECT_SHORT_NAME.lower()
            in file_name_lower

            or "project_08"
            in file_name_lower
        )


        if not is_project8_file:
            continue


        project8_source_records.append({
            "Project8SourcePath":
                str(path),

            "Project8RelativeToThesisRoot":
                path.relative_to(
                    THESIS_ROOT
                ).as_posix(),

            "Project8FileName":
                path.name,

            "Project8ParentName":
                path.parent.name,

            "Project8FileType":
                classify_file(path),

            "Project8SizeBytes":
                int(
                    path.stat().st_size
                ),

            "Project8SHA256":
                calculate_sha256(path),
        })


project8_source_inventory = pd.DataFrame(
    project8_source_records
)


if project8_source_inventory[
    "Project8SourcePath"
].duplicated().any():

    raise AssertionError(
        "Project 8 source inventory contains duplicate paths."
    )


print("\nProject 8 source inventory:")

print(
    "Files:",
    len(
        project8_source_inventory
    ),
)


project8_source_group_summary = (
    project8_source_inventory
    .groupby(
        [
            "Project8ParentName",
            "Project8FileType",
        ],
        as_index=False,
    )
    .agg(
        Files=(
            "Project8SourcePath",
            "count",
        ),

        Bytes=(
            "Project8SizeBytes",
            "sum",
        ),
    )
    .sort_values(
        [
            "Project8ParentName",
            "Project8FileType",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


display(
    project8_source_group_summary
)


# ------------------------------------------------------------
# 10. MAP EACH REFERENCE PACKAGE ENTRY
# ------------------------------------------------------------

project8_source_paths = set(
    project8_source_inventory[
        "Project8SourcePath"
    ].astype(str)
)


mapping_records = []


for reference_row in (
    reference_inventory.itertuples(
        index=False
    )
):

    reference_package_relative = str(
        reference_row.ReferencePackageRelativePath
    )

    reference_source_path = (
        reference_row.ReferenceSourcePath
    )


    transformed_package_relative = (
        transform_reference_text(
            reference_package_relative
        )
    )


    transformed_source_path = None


    if (
        reference_source_path is not None
        and str(
            reference_source_path
        ).strip()
        not in {
            "",
            "nan",
            "None",
        }
    ):

        transformed_source_path = (
            transform_reference_text(
                reference_source_path
            )
        )


    candidate_frame = (
        project8_source_inventory.copy()
    )


    match_status = "UNRESOLVED"
    match_method = "none"

    selected_row = None

    best_score = np.nan
    second_score = np.nan


    # --------------------------------------------------------
    # A. Exact transformed source path
    # --------------------------------------------------------

    if transformed_source_path is not None:

        transformed_source_candidate = Path(
            transformed_source_path
        )


        exact_source_match = (
            candidate_frame[
                candidate_frame[
                    "Project8SourcePath"
                ].eq(
                    str(
                        transformed_source_candidate
                    )
                )
            ]
        )


        if len(
            exact_source_match
        ) == 1:

            selected_row = (
                exact_source_match.iloc[0]
            )

            match_status = "RESOLVED"
            match_method = (
                "checkpoint_inventory_source_path"
            )


    # --------------------------------------------------------
    # B. Exact transformed package-relative suffix
    # --------------------------------------------------------

    if selected_row is None:

        package_relative_normalised = (
            normalise_path_text(
                transformed_package_relative
            ).lower()
        )


        suffix_matches = (
            candidate_frame[
                candidate_frame[
                    "Project8RelativeToThesisRoot"
                ]
                .astype(str)
                .str.lower()
                .str.endswith(
                    package_relative_normalised
                )
            ]
        )


        if len(
            suffix_matches
        ) == 1:

            selected_row = (
                suffix_matches.iloc[0]
            )

            match_status = "RESOLVED"
            match_method = (
                "transformed_package_relative_suffix"
            )


    # --------------------------------------------------------
    # C. Exact transformed filename
    # --------------------------------------------------------

    transformed_file_name = Path(
        transformed_package_relative
    ).name


    if selected_row is None:

        exact_filename_matches = (
            candidate_frame[
                candidate_frame[
                    "Project8FileName"
                ].eq(
                    transformed_file_name
                )
            ]
        )


        if len(
            exact_filename_matches
        ) == 1:

            selected_row = (
                exact_filename_matches.iloc[0]
            )

            match_status = "RESOLVED"
            match_method = (
                "transformed_exact_filename"
            )


    # --------------------------------------------------------
    # D. Unique scored match
    # --------------------------------------------------------

    if selected_row is None:

        reference_matching_text = (
            transformed_source_path
            if transformed_source_path
            is not None
            else transformed_package_relative
        )


        reference_extension = (
            extension_signature(
                transformed_file_name
            )
        )


        reference_file_type = (
            classify_file(
                transformed_file_name
            )
        )


        scores = []


        for candidate in (
            candidate_frame.itertuples(
                index=False
            )
        ):

            score = token_similarity(
                reference_matching_text,
                candidate.Project8RelativeToThesisRoot,
            )


            candidate_extension = (
                extension_signature(
                    candidate.Project8FileName
                )
            )


            if (
                reference_extension
                == candidate_extension
            ):

                score += 0.15


            if (
                reference_file_type
                == candidate.Project8FileType
            ):

                score += 0.15


            transformed_reference_stem = (
                re.sub(
                    r"[^a-z0-9]",
                    "",
                    Path(
                        transformed_file_name
                    ).stem.lower(),
                )
            )


            candidate_stem = re.sub(
                r"[^a-z0-9]",
                "",
                Path(
                    candidate.Project8FileName
                ).stem.lower(),
            )


            if (
                transformed_reference_stem
                == candidate_stem
            ):

                score += 1.0


            scores.append(score)


        candidate_frame[
            "_MappingScore"
        ] = scores


        candidate_frame = (
            candidate_frame
            .sort_values(
                [
                    "_MappingScore",
                    "Project8RelativeToThesisRoot",
                ],
                ascending=[
                    False,
                    True,
                ],
                kind="mergesort",
            )
            .reset_index(drop=True)
        )


        if len(
            candidate_frame
        ) >= 1:

            best_score = float(
                candidate_frame.iloc[0][
                    "_MappingScore"
                ]
            )


        if len(
            candidate_frame
        ) >= 2:

            second_score = float(
                candidate_frame.iloc[1][
                    "_MappingScore"
                ]
            )


        score_margin = (
            best_score
            - second_score
            if np.isfinite(
                second_score
            )
            else best_score
        )


        if (
            np.isfinite(
                best_score
            )
            and best_score >= 0.70
            and score_margin >= 0.10
        ):

            selected_row = (
                candidate_frame.iloc[0]
            )

            match_status = (
                "PROVISIONAL_RESOLVED"
            )

            match_method = (
                "unique_scored_mapping"
            )


    if selected_row is None:

        selected_source_path = None
        selected_relative_path = None
        selected_file_name = None
        selected_size = None
        selected_hash = None

    else:

        selected_source_path = (
            selected_row[
                "Project8SourcePath"
            ]
        )

        selected_relative_path = (
            selected_row[
                "Project8RelativeToThesisRoot"
            ]
        )

        selected_file_name = (
            selected_row[
                "Project8FileName"
            ]
        )

        selected_size = int(
            selected_row[
                "Project8SizeBytes"
            ]
        )

        selected_hash = (
            selected_row[
                "Project8SHA256"
            ]
        )


    mapping_records.append({
        "ReferenceInventoryRow":
            int(
                reference_row.ReferenceInventoryRow
            ),

        "ReferencePackageRelativePath":
            reference_package_relative,

        "ReferenceSourcePath":
            reference_source_path,

        "TransformedPackageRelativePath":
            transformed_package_relative,

        "TransformedSourcePath":
            transformed_source_path,

        "ReferenceSizeBytes":
            reference_row.ReferenceSizeBytes,

        "ReferenceSHA256":
            reference_row.ReferenceSHA256,

        "MatchStatus":
            match_status,

        "MatchMethod":
            match_method,

        "BestScore":
            best_score,

        "SecondScore":
            second_score,

        "Project8SourcePath":
            selected_source_path,

        "Project8RelativeToThesisRoot":
            selected_relative_path,

        "Project8FileName":
            selected_file_name,

        "Project8SizeBytes":
            selected_size,

        "Project8SHA256":
            selected_hash,
    })


package_mapping = pd.DataFrame(
    mapping_records
)


resolved_mapping_mask = (
    package_mapping[
        "MatchStatus"
    ].isin(
        [
            "RESOLVED",
            "PROVISIONAL_RESOLVED",
        ]
    )
)


resolved_mappings = int(
    resolved_mapping_mask.sum()
)


unresolved_mappings = int(
    (
        ~resolved_mapping_mask
    ).sum()
)


duplicate_source_assignments = int(
    package_mapping.loc[
        resolved_mapping_mask,
        "Project8SourcePath",
    ].duplicated().sum()
)


unique_mapped_source_files = int(
    package_mapping.loc[
        resolved_mapping_mask,
        "Project8SourcePath",
    ].nunique()
)


print("\nProject 7 → Project 8 package mapping:")

print(
    "Reference package entries:",
    len(
        package_mapping
    ),
)

print(
    "Resolved mappings:",
    resolved_mappings,
)

print(
    "Unresolved mappings:",
    unresolved_mappings,
)

print(
    "Duplicate source assignments:",
    duplicate_source_assignments,
)

print(
    "Unique Project 8 source files mapped:",
    unique_mapped_source_files,
)


mapping_method_summary = (
    package_mapping
    .groupby(
        [
            "MatchStatus",
            "MatchMethod",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size":
                "Rows",
        }
    )
)


display(
    mapping_method_summary
)


if unresolved_mappings:

    print("\nUnresolved mappings:")

    display(
        package_mapping[
            ~resolved_mapping_mask
        ][
            [
                "ReferencePackageRelativePath",
                "ReferenceSourcePath",
                "TransformedPackageRelativePath",
                "TransformedSourcePath",
                "BestScore",
                "SecondScore",
            ]
        ]
    )


if duplicate_source_assignments:

    print("\nDuplicate source assignments:")

    duplicate_paths = (
        package_mapping.loc[
            resolved_mapping_mask,
            "Project8SourcePath",
        ]
        .value_counts()
    )

    duplicate_paths = (
        duplicate_paths[
            duplicate_paths > 1
        ]
        .index
        .tolist()
    )

    display(
        package_mapping[
            package_mapping[
                "Project8SourcePath"
            ].isin(
                duplicate_paths
            )
        ]
    )


# ------------------------------------------------------------
# 11. STEP 11A V2 AUDIT
# ------------------------------------------------------------

audit_records = [
    {
        "Check":
            "Project 8 Step 10 passed",

        "Expected":
            EXPECTED_STEP10_STATUS,

        "Actual":
            step10_status[
                "Status"
            ],

        "Pass":
            step10_status[
                "Status"
            ]
            == EXPECTED_STEP10_STATUS,
    },

    {
        "Check":
            "Projects 1–7 remain frozen",

        "Expected":
            list(
                range(1, 8)
            ),

        "Actual":
            frozen_project_numbers,

        "Pass":
            frozen_project_numbers
            == list(
                range(1, 8)
            ),
    },

    {
        "Check":
            "Project 8 absent from registry",

        "Expected":
            0,

        "Actual":
            len(
                project8_registry_rows
            ),

        "Pass":
            len(
                project8_registry_rows
            )
            == 0,
    },

    {
        "Check":
            "Exact reference package directory exists",

        "Expected":
            True,

        "Actual":
            REFERENCE_PACKAGE_DIRECTORY.exists(),

        "Pass":
            REFERENCE_PACKAGE_DIRECTORY.exists(),
    },

    {
        "Check":
            "Exact reference package inventory exists",

        "Expected":
            True,

        "Actual":
            REFERENCE_PACKAGE_INVENTORY_PATH.exists(),

        "Pass":
            REFERENCE_PACKAGE_INVENTORY_PATH.exists(),
    },

    {
        "Check":
            "Reference final-package inventory rows",

        "Expected":
            EXPECTED_REFERENCE_PACKAGE_FILES,

        "Actual":
            len(
                reference_inventory
            ),

        "Pass":
            len(
                reference_inventory
            )
            == EXPECTED_REFERENCE_PACKAGE_FILES,
    },

    {
        "Check":
            "Reference actual package files",

        "Expected":
            EXPECTED_REFERENCE_PACKAGE_FILES,

        "Actual":
            len(
                reference_actual_inventory
            ),

        "Pass":
            len(
                reference_actual_inventory
            )
            == EXPECTED_REFERENCE_PACKAGE_FILES,
    },

    {
        "Check":
            "Reference inventory missing files",

        "Expected":
            0,

        "Actual":
            len(
                reference_inventory_missing_files
            ),

        "Pass":
            len(
                reference_inventory_missing_files
            )
            == 0,
    },

    {
        "Check":
            "Reference inventory unexpected files",

        "Expected":
            0,

        "Actual":
            len(
                reference_inventory_unexpected_files
            ),

        "Pass":
            len(
                reference_inventory_unexpected_files
            )
            == 0,
    },

    {
        "Check":
            "Reference inventory size mismatches",

        "Expected":
            0,

        "Actual":
            reference_size_mismatches,

        "Pass":
            reference_size_mismatches
            == 0,
    },

    {
        "Check":
            "Reference inventory hash mismatches",

        "Expected":
            0,

        "Actual":
            reference_hash_mismatches,

        "Pass":
            reference_hash_mismatches
            == 0,
    },

    {
        "Check":
            "Project 8 package source files",

        "Expected":
            EXPECTED_PROJECT8_SOURCE_FILES,

        "Actual":
            len(
                project8_source_inventory
            ),

        "Pass":
            len(
                project8_source_inventory
            )
            == EXPECTED_PROJECT8_SOURCE_FILES,
    },

    {
        "Check":
            "Package mapping rows",

        "Expected":
            EXPECTED_REFERENCE_PACKAGE_FILES,

        "Actual":
            len(
                package_mapping
            ),

        "Pass":
            len(
                package_mapping
            )
            == EXPECTED_REFERENCE_PACKAGE_FILES,
    },

    {
        "Check":
            "Resolved package mappings",

        "Expected":
            EXPECTED_REFERENCE_PACKAGE_FILES,

        "Actual":
            resolved_mappings,

        "Pass":
            resolved_mappings
            == EXPECTED_REFERENCE_PACKAGE_FILES,
    },

    {
        "Check":
            "Unresolved package mappings",

        "Expected":
            0,

        "Actual":
            unresolved_mappings,

        "Pass":
            unresolved_mappings
            == 0,
    },

    {
        "Check":
            "Duplicate source assignments",

        "Expected":
            0,

        "Actual":
            duplicate_source_assignments,

        "Pass":
            duplicate_source_assignments
            == 0,
    },

    {
        "Check":
            "Unique mapped Project 8 sources",

        "Expected":
            EXPECTED_PROJECT8_SOURCE_FILES,

        "Actual":
            unique_mapped_source_files,

        "Pass":
            unique_mapped_source_files
            == EXPECTED_PROJECT8_SOURCE_FILES,
    },
]


step11a_audit = pd.DataFrame(
    audit_records
)


failed_checks = (
    step11a_audit[
        ~step11a_audit[
            "Pass"
        ]
    ]
    .copy()
)


print("\nStep 11A V2 audit:")

display(
    step11a_audit
)


# ------------------------------------------------------------
# 12. WRITE STEP 11A V2 OUTPUTS
# ------------------------------------------------------------

STEP11A_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_json(
    REFERENCE_PATHS_PATH,
    reference_package_paths,
)


atomic_write_csv(
    REFERENCE_INVENTORY_NORMALISED_PATH,
    reference_inventory,
)


atomic_write_csv(
    REFERENCE_ACTUAL_PACKAGE_INVENTORY_PATH,
    reference_actual_inventory,
)


atomic_write_csv(
    PROJECT8_SOURCE_INVENTORY_PATH,
    project8_source_inventory,
    compression="gzip",
)


atomic_write_csv(
    PACKAGE_MAPPING_PATH,
    package_mapping,
)


atomic_write_csv(
    STEP11A_AUDIT_PATH,
    step11a_audit,
)


step11a_status_text = (
    "PASS_PROJECT_8_FINAL_PACKAGE_BLUEPRINT_AND_MAPPING_RESOLVED"
    if failed_checks.empty
    else
    "PROJECT_8_FINAL_PACKAGE_MAPPING_REQUIRES_REVIEW"
)


step11a_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ReferenceProject": {
        "ProjectNumber":
            REFERENCE_PROJECT_NUMBER,

        "Project":
            REFERENCE_PROJECT_NAME,

        "ProjectSlug":
            REFERENCE_PROJECT_SLUG,

        "CheckpointPath":
            str(
                REFERENCE_CHECKPOINT_PATH
            ),

        "CheckpointSHA256":
            calculate_sha256(
                REFERENCE_CHECKPOINT_PATH
            ),

        "FinalPackageDirectory":
            str(
                REFERENCE_PACKAGE_DIRECTORY
            ),

        "FinalPackageInventory":
            str(
                REFERENCE_PACKAGE_INVENTORY_PATH
            ),

        "FinalPackageValidation":
            str(
                REFERENCE_PACKAGE_VALIDATION_PATH
            ),

        "FinalPackageReport":
            str(
                REFERENCE_PACKAGE_REPORT_PATH
            ),

        "FinalPackageRootSHA256":
            reference_package_root_sha256,

        "FrozenFinalPackageRootSHA256":
            reference_frozen_package_root_sha256,

        "InventoryRows":
            len(
                reference_inventory
            ),

        "ActualPackageFiles":
            len(
                reference_actual_inventory
            ),

        "ActualPackageBytes":
            reference_actual_package_bytes,

        "InventoryMissingFiles":
            len(
                reference_inventory_missing_files
            ),

        "InventoryUnexpectedFiles":
            len(
                reference_inventory_unexpected_files
            ),

        "InventorySizeMismatches":
            reference_size_mismatches,

        "InventoryHashMismatches":
            reference_hash_mismatches,
    },

    "Project8SourceInventory": {
        "Files":
            len(
                project8_source_inventory
            ),

        "Bytes":
            int(
                project8_source_inventory[
                    "Project8SizeBytes"
                ].sum()
            ),
    },

    "Mapping": {
        "Rows":
            len(
                package_mapping
            ),

        "ResolvedMappings":
            resolved_mappings,

        "UnresolvedMappings":
            unresolved_mappings,

        "DuplicateSourceAssignments":
            duplicate_source_assignments,

        "UniqueMappedSourceFiles":
            unique_mapped_source_files,
    },

    "AuditChecks":
        len(
            step11a_audit
        ),

    "FailedAuditChecks":
        len(
            failed_checks
        ),

    "CompletionRegistryModified":
        False,

    "Projects1To7Modified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "Status":
        step11a_status_text,
}


atomic_write_json(
    STEP11A_REPORT_PATH,
    step11a_report,
)


step11a_status = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        step11a_status_text,

    "ReferenceInventoryRows":
        len(
            reference_inventory
        ),

    "ReferenceActualPackageFiles":
        len(
            reference_actual_inventory
        ),

    "Project8SourceFiles":
        len(
            project8_source_inventory
        ),

    "MappingRows":
        len(
            package_mapping
        ),

    "ResolvedMappings":
        resolved_mappings,

    "UnresolvedMappings":
        unresolved_mappings,

    "DuplicateSourceAssignments":
        duplicate_source_assignments,

    "UniqueMappedSourceFiles":
        unique_mapped_source_files,

    "FailedAuditChecks":
        len(
            failed_checks
        ),

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP11A_STATUS_PATH,
    step11a_status,
)


expected_outputs = [
    REFERENCE_PATHS_PATH,
    REFERENCE_INVENTORY_NORMALISED_PATH,
    REFERENCE_ACTUAL_PACKAGE_INVENTORY_PATH,
    PROJECT8_SOURCE_INVENTORY_PATH,
    PACKAGE_MAPPING_PATH,
    STEP11A_AUDIT_PATH,
    STEP11A_REPORT_PATH,
    STEP11A_STATUS_PATH,
]


missing_outputs = [
    str(path)
    for path in expected_outputs
    if not path.exists()
]


if missing_outputs:

    raise RuntimeError(
        "Step 11A V2 outputs are missing:\n"
        + "\n".join(
            missing_outputs
        )
    )


# ------------------------------------------------------------
# 13. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 94)
print("=== PROJECT 8 STEP 11A V2 RESULT ===")
print("=" * 94)

print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nReference final package:")

print(
    "Directory:",
    REFERENCE_PACKAGE_DIRECTORY,
)

print(
    "Inventory:",
    REFERENCE_PACKAGE_INVENTORY_PATH,
)

print(
    "Inventory rows:",
    len(
        reference_inventory
    ),
)

print(
    "Actual package files:",
    len(
        reference_actual_inventory
    ),
)

print(
    "Actual package bytes:",
    reference_actual_package_bytes,
)

print(
    "Inventory missing files:",
    len(
        reference_inventory_missing_files
    ),
)

print(
    "Inventory unexpected files:",
    len(
        reference_inventory_unexpected_files
    ),
)

print(
    "Size mismatches:",
    reference_size_mismatches,
)

print(
    "SHA-256 mismatches:",
    reference_hash_mismatches,
)


print("\nProject 8 package source inventory:")

print(
    "Source files:",
    len(
        project8_source_inventory
    ),
)

print(
    "Source bytes:",
    int(
        project8_source_inventory[
            "Project8SizeBytes"
        ].sum()
    ),
)


print("\nPackage mapping:")

print(
    "Mapping rows:",
    len(
        package_mapping
    ),
)

print(
    "Resolved mappings:",
    resolved_mappings,
)

print(
    "Unresolved mappings:",
    unresolved_mappings,
)

print(
    "Duplicate source assignments:",
    duplicate_source_assignments,
)

print(
    "Unique mapped Project 8 sources:",
    unique_mapped_source_files,
)


print("\nAudit:")

print(
    "Checks:",
    len(
        step11a_audit
    ),
)

print(
    "Failed checks:",
    len(
        failed_checks
    ),
)


if not failed_checks.empty:

    print("\nFailed checks:")

    display(
        failed_checks
    )


print("\nStep 11A V2 outputs:")

for output_path in expected_outputs:

    print(output_path)


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–7 modified:")
print(0)


print(
    "\nSTATUS:",
    step11a_status_text,
)

print("=" * 94)

=== PROJECT 8 STEP 11A V2: EXACT FINAL-PACKAGE BLUEPRINT DISCOVERY ===

Project 8 Step 10 validation:
Status: PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED
Raw files: 2160
Raw bytes: 179766494
Raw root SHA-256: 19ae21c5d8524c358796e28f23577fbf03bfd31f7d4b530372c6b4d8a492887c

Exact Project 7 final-package paths:
FinalPackageDirectory: /content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_30_seed_final
FinalPackageInventory: /content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_final_package_audit/beast2_final_package_file_inventory_sha256.csv
FinalPackageValidation: /content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_final_package_audit/beast2_final_package_validation.csv
FinalPackageReport: /content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_final_package_audit/beast2_final_package_report.json
FinalPackageRootSHA256: 93e9cc74fac017fef89b005752e1b7ede2a73f9665

,Project8ParentName,Project8FileType,Files,Bytes
0,Notes,configuration,1,7864
1,Notes,protocol,1,7006
2,ojalgo_preflight,audit,4,3698
3,ojalgo_preflight,chronology,5,83399
4,ojalgo_preflight,configuration,1,7864
5,ojalgo_preflight,other,21,1530815
6,ojalgo_preflight,predictor,1,7211
7,ojalgo_preflight,protocol,3,13067
8,ojalgo_preflight,report,5,67379
9,ojalgo_preflight,status,8,34054



Project 7 → Project 8 package mapping:
Reference package entries: 72
Resolved mappings: 5
Unresolved mappings: 67
Duplicate source assignments: 1
Unique Project 8 source files mapped: 4


,MatchStatus,MatchMethod,Rows
0,PROVISIONAL_RESOLVED,unique_scored_mapping,5
1,UNRESOLVED,none,67



Unresolved mappings:


...


Step 11A V2 audit:


,Check,Expected,Actual,Pass
0,Project 8 Step 10 passed,PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED,PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED,True
1,Projects 1–7 remain frozen,"[1, 2, 3, 4, 5, 6, 7]","[1, 2, 3, 4, 5, 6, 7]",True
2,Project 8 absent from registry,0,0,True
3,Exact reference package directory exists,True,True,True
4,Exact reference package inventory exists,True,True,True
5,Reference final-package inventory rows,72,72,True
6,Reference actual package files,72,72,True
7,Reference inventory missing files,0,0,True
8,Reference inventory unexpected files,0,0,True
9,Reference inventory size mismatches,0,0,True




=== PROJECT 8 STEP 11A V2 RESULT ===

Project identity:
Project number: 8
Project: optimatika@ojAlgo
Project slug: optimatika__ojAlgo

Reference final package:
Directory: /content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_30_seed_final
Inventory: /content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_final_package_audit/beast2_final_package_file_inventory_sha256.csv
Inventory rows: 72
Actual package files: 72
Actual package bytes: 1624833
Inventory missing files: 0
Inventory unexpected files: 0
Size mismatches: 0
SHA-256 mismatches: 0

Project 8 package source inventory:
Source files: 72
Source bytes: 3151274

Package mapping:
Mapping rows: 72
Resolved mappings: 5
Unresolved mappings: 67
Duplicate source assignments: 1
Unique mapped Project 8 sources: 4

Audit:
Checks: 17
Failed checks: 4

Failed checks:


,Check,Expected,Actual,Pass
13,Resolved package mappings,72,5,False
14,Unresolved package mappings,0,67,False
15,Duplicate source assignments,0,1,False
16,Unique mapped Project 8 sources,72,4,False



Step 11A V2 outputs:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_step11a_package_blueprint/ojalgo_reference_final_package_paths.json
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_step11a_package_blueprint/ojalgo_reference_final_package_inventory.csv
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_step11a_package_blueprint/ojalgo_reference_actual_package_inventory.csv
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_step11a_package_blueprint/ojalgo_package_source_inventory.csv.gz
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_step11a_package_blueprint/ojalgo_final_package_mapping.csv
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_step11a_package_blueprint/ojalgo_step11a_v2_audit.csv
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__o

In [ ]:
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd

RESULT_PATH = Path(
    "/content/drive/MyDrive/Thesis_Experiment/"
    "Results/Aggregated/optimatika__ojAlgo/"
    "ojalgo_raw_audit/ojalgo_all_project_run_metrics.csv"
)

results = pd.read_csv(RESULT_PATH)

required_columns = {
    "NoisePercent",
    "RepetitionSeed",
    "Technique",
    "MeanAPFD",
    "MeanAPFDc",
}

missing_columns = required_columns - set(results.columns)

if missing_columns:
    raise RuntimeError(
        f"Missing required columns: {sorted(missing_columns)}\n"
        f"Available columns: {list(results.columns)}"
    )

for column in [
    "NoisePercent",
    "RepetitionSeed",
    "MeanAPFD",
    "MeanAPFDc",
]:
    results[column] = pd.to_numeric(
        results[column],
        errors="raise",
    )

print("=" * 90)
print("PROJECT 8 — ML AND BASELINE RESULT INSPECTION")
print("=" * 90)

print("\nInput file:")
print(RESULT_PATH)

print("\nRaw project-run result dimensions:")
print(results.shape)

print("\nExpected rows: 1,890")
print("Actual rows:", len(results))

print("\nTechniques:")
print(sorted(results["Technique"].unique()))

print("\nNoise levels:")
print(sorted(results["NoisePercent"].unique()))

print("\nSeeds:")
print(
    int(results["RepetitionSeed"].min()),
    "to",
    int(results["RepetitionSeed"].max()),
)


# ------------------------------------------------------------
# 30-seed mean and variability
# ------------------------------------------------------------

summary = (
    results
    .groupby(
        [
            "NoisePercent",
            "Technique",
        ],
        as_index=False,
    )
    .agg(
        Seeds=("RepetitionSeed", "nunique"),

        Mean_APFD=("MeanAPFD", "mean"),
        SD_Across_Seeds_APFD=("MeanAPFD", "std"),

        Mean_APFDc=("MeanAPFDc", "mean"),
        SD_Across_Seeds_APFDc=("MeanAPFDc", "std"),
    )
)

summary["SE_APFD"] = (
    summary["SD_Across_Seeds_APFD"]
    / np.sqrt(summary["Seeds"])
)

summary["SE_APFDc"] = (
    summary["SD_Across_Seeds_APFDc"]
    / np.sqrt(summary["Seeds"])
)

summary["CI95_Lower_APFD"] = (
    summary["Mean_APFD"]
    - 1.96 * summary["SE_APFD"]
)

summary["CI95_Upper_APFD"] = (
    summary["Mean_APFD"]
    + 1.96 * summary["SE_APFD"]
)

summary["CI95_Lower_APFDc"] = (
    summary["Mean_APFDc"]
    - 1.96 * summary["SE_APFDc"]
)

summary["CI95_Upper_APFDc"] = (
    summary["Mean_APFDc"]
    + 1.96 * summary["SE_APFDc"]
)

summary = (
    summary
    .sort_values(
        [
            "NoisePercent",
            "Mean_APFDc",
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

print("\n30-seed APFD and APFDc summary:")
display(summary.round(6))


# ------------------------------------------------------------
# Wide APFDc table
# ------------------------------------------------------------

apfdc_wide = (
    summary
    .pivot(
        index="NoisePercent",
        columns="Technique",
        values="Mean_APFDc",
    )
    .sort_index()
)

print("\nMean APFDc across 30 seeds:")
display(apfdc_wide.round(6))


# ------------------------------------------------------------
# Wide APFD table
# ------------------------------------------------------------

apfd_wide = (
    summary
    .pivot(
        index="NoisePercent",
        columns="Technique",
        values="Mean_APFD",
    )
    .sort_index()
)

print("\nMean APFD across 30 seeds:")
display(apfd_wide.round(6))


# ------------------------------------------------------------
# Clean performance and 50% performance
# ------------------------------------------------------------

endpoint_comparison = (
    summary[
        summary["NoisePercent"].isin(
            [
                0,
                50,
            ]
        )
    ]
    [
        [
            "NoisePercent",
            "Technique",
            "Mean_APFD",
            "Mean_APFDc",
            "SD_Across_Seeds_APFD",
            "SD_Across_Seeds_APFDc",
        ]
    ]
    .sort_values(
        [
            "NoisePercent",
            "Mean_APFDc",
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

print("\n0% versus 50% noise:")
display(endpoint_comparison.round(6))


# ------------------------------------------------------------
# APFDc degradation from clean performance
# ------------------------------------------------------------

clean_apfdc = (
    summary[
        summary["NoisePercent"].eq(0)
    ]
    .set_index("Technique")[
        "Mean_APFDc"
    ]
)

summary["Clean_APFDc"] = (
    summary["Technique"].map(
        clean_apfdc
    )
)

summary["Absolute_APFDc_Change"] = (
    summary["Mean_APFDc"]
    - summary["Clean_APFDc"]
)

summary["APFDc_Retention_Percent"] = np.where(
    summary["Clean_APFDc"].ne(0),

    100.0
    * summary["Mean_APFDc"]
    / summary["Clean_APFDc"],

    np.nan,
)

print("\nAPFDc degradation and retention:")
display(
    summary[
        [
            "NoisePercent",
            "Technique",
            "Clean_APFDc",
            "Mean_APFDc",
            "Absolute_APFDc_Change",
            "APFDc_Retention_Percent",
        ]
    ].round(6)
)

print("\nSTATUS: PROJECT_8_RESULTS_LOADED_AND_DISPLAYED")

PROJECT 8 — ML AND BASELINE RESULT INSPECTION

Input file:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_raw_audit/ojalgo_all_project_run_metrics.csv

Raw project-run result dimensions:
(1890, 12)

Expected rows: 1,890
Actual rows: 1890

Techniques:
['LatestFail', 'LightGBM', 'NaiveBayes', 'QTF-Avg', 'Random', 'RandomForest', 'XGBoost']

Noise levels:
[np.float64(0.0), np.float64(5.0), np.float64(10.0), np.float64(15.0), np.float64(20.0), np.float64(25.0), np.float64(30.0), np.float64(40.0), np.float64(50.0)]

Seeds:
1 to 30

30-seed APFD and APFDc summary:


,NoisePercent,Technique,Seeds,Mean_APFD,SD_Across_Seeds_APFD,Mean_APFDc,SD_Across_Seeds_APFDc,SE_APFD,SE_APFDc,CI95_Lower_APFD,CI95_Upper_APFD,CI95_Lower_APFDc,CI95_Upper_APFDc
0,0.0,QTF-Avg,30,0.096178,0.000000,0.853237,0.000000,0.000000,0.000000,0.096178,0.096178,0.853237,0.853237
1,0.0,LatestFail,30,0.934893,0.000000,0.852401,0.000000,0.000000,0.000000,0.934893,0.934893,0.852401,0.852401
2,0.0,NaiveBayes,30,0.709531,0.000000,0.798309,0.000000,0.000000,0.000000,0.709531,0.709531,0.798309,0.798309
3,0.0,RandomForest,30,0.945877,0.009719,0.790071,0.016804,0.001774,0.003068,0.942399,0.949355,0.784058,0.796084
4,0.0,XGBoost,30,0.914499,0.000000,0.731460,0.000000,0.000000,0.000000,0.914499,0.914499,0.731460,0.731460
...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,50.0,NaiveBayes,30,0.443993,0.264067,0.587131,0.243573,0.048212,0.044470,0.349498,0.538488,0.499970,0.674292
59,50.0,Random,30,0.483751,0.069122,0.487324,0.088599,0.012620,0.016176,0.459016,0.508486,0.455620,0.519029
60,50.0,XGBoost,30,0.573385,0.189864,0.480360,0.168685,0.034664,0.030798,0.505443,0.641327,0.419997,0.540723
61,50.0,RandomForest,30,0.495820,0.121164,0.472036,0.134253,0.022121,0.024511,0.452462,0.539178,0.423994,0.520078



Mean APFDc across 30 seeds:


Technique,LatestFail,LightGBM,NaiveBayes,QTF-Avg,Random,RandomForest,XGBoost
NoisePercent,,,,,,,
0.0,0.852401,0.673642,0.798309,0.853237,0.487324,0.790071,0.731460
5.0,0.828070,0.676862,0.498227,0.853237,0.487324,0.713873,0.700453
10.0,0.823283,0.666596,0.429185,0.853237,0.487324,0.708649,0.632275
15.0,0.811102,0.647890,0.458774,0.853237,0.487324,0.672533,0.600499
20.0,0.807868,0.645039,0.489709,0.853237,0.487324,0.638181,0.595407
25.0,0.813777,0.624846,0.473633,0.853237,0.487324,0.616435,0.592518
30.0,0.816377,0.630758,0.497715,0.853237,0.487324,0.563225,0.578824
40.0,0.820406,0.540095,0.582220,0.853237,0.487324,0.500473,0.539359
50.0,0.833949,0.456600,0.587131,0.853237,0.487324,0.472036,0.480360



Mean APFD across 30 seeds:


...


0% versus 50% noise:


,NoisePercent,Technique,Mean_APFD,Mean_APFDc,SD_Across_Seeds_APFD,SD_Across_Seeds_APFDc
0,0.0,QTF-Avg,0.096178,0.853237,0.000000,0.000000
1,0.0,LatestFail,0.934893,0.852401,0.000000,0.000000
2,0.0,NaiveBayes,0.709531,0.798309,0.000000,0.000000
3,0.0,RandomForest,0.945877,0.790071,0.009719,0.016804
4,0.0,XGBoost,0.914499,0.731460,0.000000,0.000000
5,0.0,LightGBM,0.924470,0.673642,0.000000,0.000000
6,0.0,Random,0.483751,0.487324,0.069122,0.088599
7,50.0,QTF-Avg,0.096178,0.853237,0.000000,0.000000
8,50.0,LatestFail,0.847678,0.833949,0.040962,0.062038
9,50.0,NaiveBayes,0.443993,0.587131,0.264067,0.243573



APFDc degradation and retention:


,NoisePercent,Technique,Clean_APFDc,Mean_APFDc,Absolute_APFDc_Change,APFDc_Retention_Percent
0,0.0,QTF-Avg,0.853237,0.853237,0.000000,100.000000
1,0.0,LatestFail,0.852401,0.852401,0.000000,100.000000
2,0.0,NaiveBayes,0.798309,0.798309,0.000000,100.000000
3,0.0,RandomForest,0.790071,0.790071,0.000000,100.000000
4,0.0,XGBoost,0.731460,0.731460,0.000000,100.000000
...,...,...,...,...,...,...
58,50.0,NaiveBayes,0.798309,0.587131,-0.211178,73.546846
59,50.0,Random,0.487324,0.487324,0.000000,100.000000
60,50.0,XGBoost,0.731460,0.480360,-0.251100,65.671354
61,50.0,RandomForest,0.790071,0.472036,-0.318035,59.746001



STATUS: PROJECT_8_RESULTS_LOADED_AND_DISPLAYED


In [ ]:
# ============================================================
# PROJECT 8 — STEP 11B
# GENERATE AND VALIDATE THE FINAL 72-FILE RESULT PACKAGE
#
# PROJECT: optimatika@ojAlgo
#
# IMPORTANT:
# - Project 7 is used only for relative filenames and schemas.
# - Every value written into the new package is generated from
#   audited Project 8 results.
# - The failed Step 11A mapping is ignored completely.
#
# This cell creates:
#   9 top-level package files
#   63 technique × noise files
#   72 package files total
#
# This cell does NOT:
# - rerun any model
# - alter raw Project 8 results
# - alter Projects 1–7
# - update the completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import re
import shutil

import numpy as np
import pandas as pd

from scipy.stats import t as student_t


# ------------------------------------------------------------
# 1. PROJECT CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 8
PROJECT_NAME = "optimatika@ojAlgo"
PROJECT_SLUG = "optimatika__ojAlgo"
PROJECT_SHORT_NAME = "ojalgo"

REFERENCE_PROJECT_NAME = "CompEvol@beast2"
REFERENCE_PROJECT_SLUG = "CompEvol__beast2"
REFERENCE_PROJECT_SHORT_NAME = "beast2"

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

SEEDS = list(
    range(1, 31)
)

TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
    "Random",
    "LatestFail",
    "QTF-Avg",
]

TECHNIQUE_TO_SLUG = {
    "RandomForest":
        "random_forest",

    "XGBoost":
        "xgboost",

    "LightGBM":
        "lightgbm",

    "NaiveBayes":
        "naive_bayes",

    "Random":
        "random",

    "LatestFail":
        "latest_fail",

    "QTF-Avg":
        "qtf_avg",
}

SLUG_TO_TECHNIQUE = {
    value: key
    for key, value
    in TECHNIQUE_TO_SLUG.items()
}

EXPECTED_CONDITIONS = 270
EXPECTED_PROJECT_RUN_ROWS = 1890
EXPECTED_BUILD_METRIC_ROWS = 30240
EXPECTED_FIT_ROWS = 1080

EXPECTED_PACKAGE_FILES = 72
EXPECTED_TECHNIQUE_NOISE_FILES = 63
EXPECTED_TOP_LEVEL_FILES = 9

EXPECTED_RAW_FILES = 2160
EXPECTED_RAW_BYTES = 179766494

EXPECTED_RAW_ROOT_SHA256 = (
    "19ae21c5d8524c358796e28f23577fbf03bfd31f7d4b530372c6b4d8a492887c"
)

EXPECTED_STEP10_STATUS = (
    "PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED"
)

GENERATED_AT_UTC = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

AGGREGATED_ROOT = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
)

PROJECT_DIR = (
    AGGREGATED_ROOT
    / PROJECT_SLUG
)

RAW_AUDIT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_raw_audit"
)

STEP10_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step10_status.json"
)

REFERENCE_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_07_selection_checkpoint.json"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)


# Audited Project 8 inputs

PROJECT_RUN_INPUT = (
    RAW_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_all_project_run_metrics.csv"
)

BUILD_METRICS_INPUT = (
    RAW_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_all_build_metrics.csv.gz"
)

NOISE_SUMMARY_INPUT = (
    RAW_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_all_noise_summary.csv"
)

FIT_TIMES_INPUT = (
    RAW_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_all_fit_times.csv"
)


# Final package candidate

FINAL_PACKAGE_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_30_seed_final"
)

STAGING_PACKAGE_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_30_seed_final__staging"
)


# Final package audit outputs

FINAL_PACKAGE_AUDIT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_audit"
)

FINAL_PACKAGE_INVENTORY_PATH = (
    FINAL_PACKAGE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_file_inventory_sha256.csv"
)

FINAL_PACKAGE_VALIDATION_PATH = (
    FINAL_PACKAGE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_validation.csv"
)

FINAL_PACKAGE_GENERATION_MAP_PATH = (
    FINAL_PACKAGE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_generation_map.csv"
)

FINAL_PACKAGE_REPORT_PATH = (
    FINAL_PACKAGE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_report.json"
)

STEP11B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step11b_status.json"
)


print("=" * 96)
print("=== PROJECT 8 STEP 11B: GENERATE FINAL 72-FILE RESULT PACKAGE ===")
print("=" * 96)


# ------------------------------------------------------------
# 3. GENERAL HELPERS
# ------------------------------------------------------------

def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def json_safe(value):

    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:

        if pd.isna(value):
            return None

    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(path)


def recursive_key_lookup(
    value,
    requested_key,
):
    requested_key = str(
        requested_key
    ).strip().lower()

    if isinstance(value, dict):

        for key, child in value.items():

            if (
                str(key)
                .strip()
                .lower()
                == requested_key
            ):
                return child

        for child in value.values():

            result = recursive_key_lookup(
                child,
                requested_key,
            )

            if result is not None:
                return result

    elif isinstance(value, list):

        for child in value:

            result = recursive_key_lookup(
                child,
                requested_key,
            )

            if result is not None:
                return result

    return None


def normalise_column_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


def detect_column(
    dataframe,
    candidates,
    required=True,
):
    lookup = {
        normalise_column_name(column):
            column
        for column in dataframe.columns
    }

    for candidate in candidates:

        normalised_candidate = (
            normalise_column_name(
                candidate
            )
        )

        if normalised_candidate in lookup:

            return lookup[
                normalised_candidate
            ]

    if required:

        raise RuntimeError(
            "Could not detect a required column.\n"
            f"Candidates: {candidates}\n"
            f"Available: {list(dataframe.columns)}"
        )

    return None


def package_relative_path(
    value,
    package_directory,
):
    text = (
        str(value)
        .replace("\\", "/")
        .strip()
    )

    text = re.sub(
        r"/+",
        "/",
        text,
    )

    package_text = (
        str(package_directory)
        .replace("\\", "/")
        .rstrip("/")
    )

    if text.startswith(
        package_text + "/"
    ):

        return text[
            len(package_text) + 1:
        ]

    package_name = Path(
        package_directory
    ).name

    marker = (
        package_name
        + "/"
    )

    if marker in text:

        return text.split(
            marker,
            1,
        )[1]

    return text.lstrip("./")


def read_table(
    path,
):
    path = Path(path)

    suffixes = "".join(
        path.suffixes
    ).lower()

    if suffixes.endswith(
        ".parquet"
    ):

        return pd.read_parquet(
            path
        )

    if (
        suffixes.endswith(".csv")
        or suffixes.endswith(".csv.gz")
    ):

        return pd.read_csv(
            path,
            low_memory=False,
        )

    raise RuntimeError(
        "Unsupported reference package table format:\n"
        f"{path}"
    )


def write_table(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    suffixes = "".join(
        path.suffixes
    ).lower()

    if suffixes.endswith(
        ".parquet"
    ):

        temporary_path = path.with_name(
            path.name + ".tmp.parquet"
        )

        dataframe.to_parquet(
            temporary_path,
            index=False,
        )

        temporary_path.replace(path)

        return

    if suffixes.endswith(
        ".csv.gz"
    ):

        temporary_path = path.with_name(
            path.name + ".tmp.gz"
        )

        dataframe.to_csv(
            temporary_path,
            index=False,
            compression="gzip",
        )

        temporary_path.replace(path)

        return

    if suffixes.endswith(
        ".csv"
    ):

        temporary_path = path.with_name(
            path.name + ".tmp"
        )

        dataframe.to_csv(
            temporary_path,
            index=False,
        )

        temporary_path.replace(path)

        return

    raise RuntimeError(
        "Unsupported output table format:\n"
        f"{path}"
    )


def root_inventory_hash(
    inventory,
):
    digest = hashlib.sha256()

    ordered = inventory.sort_values(
        "RelativePath",
        kind="mergesort",
    )

    for row in ordered.itertuples(
        index=False
    ):

        digest.update(
            row.RelativePath.encode(
                "utf-8"
            )
        )

        digest.update(b"\0")

        digest.update(
            str(
                int(row.SizeBytes)
            ).encode(
                "utf-8"
            )
        )

        digest.update(b"\0")

        digest.update(
            bytes.fromhex(
                row.SHA256
            )
        )

        digest.update(b"\n")

    return digest.hexdigest()


def replace_project_identity(
    dataframe,
):
    dataframe = dataframe.copy()

    for column in dataframe.columns:

        if (
            pd.api.types.is_object_dtype(
                dataframe[column]
            )
            or pd.api.types.is_string_dtype(
                dataframe[column]
            )
        ):

            dataframe[column] = (
                dataframe[column]
                .astype(str)
                .str.replace(
                    REFERENCE_PROJECT_NAME,
                    PROJECT_NAME,
                    regex=False,
                )
                .str.replace(
                    REFERENCE_PROJECT_SLUG,
                    PROJECT_SLUG,
                    regex=False,
                )
                .str.replace(
                    REFERENCE_PROJECT_SHORT_NAME,
                    PROJECT_SHORT_NAME,
                    regex=False,
                )
            )

    return dataframe


# ------------------------------------------------------------
# 4. VALIDATE INPUTS AND PROJECT STATE
# ------------------------------------------------------------

required_inputs = [
    STEP10_STATUS_PATH,
    REFERENCE_CHECKPOINT_PATH,
    REGISTRY_PATH,
    PROJECT_RUN_INPUT,
    BUILD_METRICS_INPUT,
    NOISE_SUMMARY_INPUT,
    FIT_TIMES_INPUT,
]


missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.exists()
]


if missing_inputs:

    raise FileNotFoundError(
        "Required Step 11B inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
    )


step10_status = json.loads(
    STEP10_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    step10_status.get("Status")
    != EXPECTED_STEP10_STATUS
):

    raise AssertionError(
        "Project 8 Step 10 has not passed.\n"
        f"Detected status: "
        f"{step10_status.get('Status')}"
    )


if (
    int(
        step10_status.get(
            "RawFiles",
            -1,
        )
    )
    != EXPECTED_RAW_FILES
):

    raise AssertionError(
        "Project 8 raw-file count differs."
    )


if (
    int(
        step10_status.get(
            "RawBytes",
            -1,
        )
    )
    != EXPECTED_RAW_BYTES
):

    raise AssertionError(
        "Project 8 raw-byte count differs."
    )


if (
    step10_status.get(
        "RawRootSHA256"
    )
    != EXPECTED_RAW_ROOT_SHA256
):

    raise AssertionError(
        "Project 8 raw-root hash differs."
    )


registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
)


registry_project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="coerce",
)


project8_registry_rows = registry[
    registry_project_numbers.eq(
        PROJECT_NUMBER
    )
]


if len(
    project8_registry_rows
) != 0:

    raise AssertionError(
        "Project 8 is already present in the completion "
        "registry. The candidate package must not be rebuilt."
    )


print("\nInput validation:")

print(
    "Step 10 status:",
    step10_status["Status"],
)

print(
    "Raw files:",
    step10_status["RawFiles"],
)

print(
    "Raw bytes:",
    step10_status["RawBytes"],
)

print(
    "Raw SHA-256:",
    step10_status["RawRootSHA256"],
)

print(
    "Project 8 registry rows:",
    len(project8_registry_rows),
)


# ------------------------------------------------------------
# 5. LOAD THE PROJECT 7 PACKAGE BLUEPRINT
# ------------------------------------------------------------

reference_checkpoint = json.loads(
    REFERENCE_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)


reference_package_directory = Path(
    recursive_key_lookup(
        reference_checkpoint,
        "FinalPackageDirectory",
    )
)


reference_inventory_path = Path(
    recursive_key_lookup(
        reference_checkpoint,
        "FinalPackageInventory",
    )
)


if not reference_package_directory.exists():

    raise FileNotFoundError(
        "Project 7 reference package directory is missing:\n"
        f"{reference_package_directory}"
    )


if not reference_inventory_path.exists():

    raise FileNotFoundError(
        "Project 7 reference inventory is missing:\n"
        f"{reference_inventory_path}"
    )


reference_inventory_raw = pd.read_csv(
    reference_inventory_path,
    low_memory=False,
)


reference_relative_column = detect_column(
    reference_inventory_raw,
    [
        "RelativePath",
        "PackageRelativePath",
        "Path",
        "FilePath",
    ],
)


reference_relative_paths = (
    reference_inventory_raw[
        reference_relative_column
    ]
    .astype(str)
    .map(
        lambda value:
            package_relative_path(
                value,
                reference_package_directory,
            )
    )
    .tolist()
)


if len(
    reference_relative_paths
) != EXPECTED_PACKAGE_FILES:

    raise AssertionError(
        "Reference package does not contain 72 inventory rows.\n"
        f"Detected: {len(reference_relative_paths)}"
    )


if len(
    set(reference_relative_paths)
) != EXPECTED_PACKAGE_FILES:

    raise AssertionError(
        "Reference package relative paths are not unique."
    )


missing_reference_files = [
    relative_path
    for relative_path
    in reference_relative_paths
    if not (
        reference_package_directory
        / relative_path
    ).exists()
]


if missing_reference_files:

    raise FileNotFoundError(
        "Reference package files are missing:\n"
        + "\n".join(
            missing_reference_files
        )
    )


technique_noise_relative_paths = [
    relative_path
    for relative_path
    in reference_relative_paths
    if relative_path.replace(
        "\\",
        "/",
    ).startswith(
        "technique_noise/"
    )
]


top_level_relative_paths = [
    relative_path
    for relative_path
    in reference_relative_paths
    if relative_path
    not in technique_noise_relative_paths
]


if (
    len(
        technique_noise_relative_paths
    )
    != EXPECTED_TECHNIQUE_NOISE_FILES
):

    raise AssertionError(
        "Reference package must contain 63 technique-noise "
        "files.\n"
        f"Detected: "
        f"{len(technique_noise_relative_paths)}"
    )


if (
    len(
        top_level_relative_paths
    )
    != EXPECTED_TOP_LEVEL_FILES
):

    raise AssertionError(
        "Reference package must contain nine top-level files.\n"
        f"Detected: "
        f"{len(top_level_relative_paths)}"
    )


print("\nReference package blueprint:")

print(
    "Package directory:",
    reference_package_directory,
)

print(
    "Inventory rows:",
    len(reference_relative_paths),
)

print(
    "Top-level files:",
    len(top_level_relative_paths),
)

print(
    "Technique-noise files:",
    len(
        technique_noise_relative_paths
    ),
)

print("\nTop-level package files:")

for relative_path in sorted(
    top_level_relative_paths
):

    reference_frame = read_table(
        reference_package_directory
        / relative_path
    )

    print(
        f" - {relative_path}: "
        f"{len(reference_frame)} rows × "
        f"{len(reference_frame.columns)} columns"
    )


# ------------------------------------------------------------
# 6. LOAD AUDITED PROJECT 8 RESULTS
# ------------------------------------------------------------

project_run = pd.read_csv(
    PROJECT_RUN_INPUT,
    low_memory=False,
)

build_metrics = pd.read_csv(
    BUILD_METRICS_INPUT,
    low_memory=False,
)

noise_summary = pd.read_csv(
    NOISE_SUMMARY_INPUT,
    low_memory=False,
)

fit_times = pd.read_csv(
    FIT_TIMES_INPUT,
    low_memory=False,
)


# Normalise core project-run columns.

for column in [
    "NoisePercent",
    "RepetitionSeed",
    "MeanAPFD",
    "MeanAPFDc",
]:

    project_run[column] = pd.to_numeric(
        project_run[column],
        errors="raise",
    )


project_run[
    "NoisePercent"
] = project_run[
    "NoisePercent"
].astype(int)


project_run[
    "RepetitionSeed"
] = project_run[
    "RepetitionSeed"
].astype(int)


project_run[
    "Project"
] = PROJECT_NAME


for column in [
    "NoisePercent",
    "RepetitionSeed",
    "APFD",
    "APFDc",
]:

    build_metrics[column] = pd.to_numeric(
        build_metrics[column],
        errors="raise",
    )


build_metrics[
    "NoisePercent"
] = build_metrics[
    "NoisePercent"
].astype(int)


build_metrics[
    "RepetitionSeed"
] = build_metrics[
    "RepetitionSeed"
].astype(int)


build_metrics[
    "Project"
] = PROJECT_NAME


for column in [
    "NoisePercent",
    "RepetitionSeed",
    "FitSeconds",
]:

    fit_times[column] = pd.to_numeric(
        fit_times[column],
        errors="raise",
    )


fit_times[
    "NoisePercent"
] = fit_times[
    "NoisePercent"
].astype(int)


fit_times[
    "RepetitionSeed"
] = fit_times[
    "RepetitionSeed"
].astype(int)


fit_times[
    "Project"
] = PROJECT_NAME


# Noise summary may call the field NoisePercentRequested.

noise_percent_source_column = detect_column(
    noise_summary,
    [
        "NoisePercent",
        "NoisePercentRequested",
    ],
)


if (
    noise_percent_source_column
    != "NoisePercent"
):

    noise_summary[
        "NoisePercent"
    ] = pd.to_numeric(
        noise_summary[
            noise_percent_source_column
        ],
        errors="raise",
    ).astype(int)

else:

    noise_summary[
        "NoisePercent"
    ] = pd.to_numeric(
        noise_summary[
            "NoisePercent"
        ],
        errors="raise",
    ).astype(int)


noise_seed_column = detect_column(
    noise_summary,
    [
        "RepetitionSeed",
        "Seed",
    ],
)


if noise_seed_column != "RepetitionSeed":

    noise_summary[
        "RepetitionSeed"
    ] = pd.to_numeric(
        noise_summary[
            noise_seed_column
        ],
        errors="raise",
    ).astype(int)

else:

    noise_summary[
        "RepetitionSeed"
    ] = pd.to_numeric(
        noise_summary[
            "RepetitionSeed"
        ],
        errors="raise",
    ).astype(int)


noise_summary[
    "Project"
] = PROJECT_NAME


# ------------------------------------------------------------
# 7. VALIDATE AUDITED RESULT DIMENSIONS
# ------------------------------------------------------------

if len(project_run) != EXPECTED_PROJECT_RUN_ROWS:

    raise AssertionError(
        "Project-run metric row count differs.\n"
        f"Expected: {EXPECTED_PROJECT_RUN_ROWS}\n"
        f"Actual: {len(project_run)}"
    )


if len(build_metrics) != EXPECTED_BUILD_METRIC_ROWS:

    raise AssertionError(
        "Build-metric row count differs.\n"
        f"Expected: {EXPECTED_BUILD_METRIC_ROWS}\n"
        f"Actual: {len(build_metrics)}"
    )


if len(noise_summary) != EXPECTED_CONDITIONS:

    raise AssertionError(
        "Noise-summary condition count differs.\n"
        f"Expected: {EXPECTED_CONDITIONS}\n"
        f"Actual: {len(noise_summary)}"
    )


if len(fit_times) != EXPECTED_FIT_ROWS:

    raise AssertionError(
        "Fit-time row count differs.\n"
        f"Expected: {EXPECTED_FIT_ROWS}\n"
        f"Actual: {len(fit_times)}"
    )


if set(
    project_run[
        "Technique"
    ].unique()
) != set(TECHNIQUES):

    raise AssertionError(
        "Project-run technique set differs."
    )


if set(
    project_run[
        "NoisePercent"
    ].unique()
) != set(NOISE_LEVELS):

    raise AssertionError(
        "Project-run noise-level set differs."
    )


if set(
    project_run[
        "RepetitionSeed"
    ].unique()
) != set(SEEDS):

    raise AssertionError(
        "Project-run seed set differs."
    )


duplicate_project_run_rows = int(
    project_run.duplicated(
        subset=[
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ]
    ).sum()
)


if duplicate_project_run_rows != 0:

    raise AssertionError(
        "Duplicate project-run condition-technique rows exist."
    )


project_metric_values = project_run[
    [
        "MeanAPFD",
        "MeanAPFDc",
    ]
].to_numpy(dtype=float)


if not np.isfinite(
    project_metric_values
).all():

    raise AssertionError(
        "Project-run APFD/APFDc contains missing or infinite "
        "values."
    )


if (
    (project_metric_values < 0)
    | (project_metric_values > 1)
).any():

    raise AssertionError(
        "Project-run APFD/APFDc falls outside [0, 1]."
    )


print("\nAudited Project 8 result inputs:")

print(
    "Project-run rows:",
    len(project_run),
)

print(
    "Build-metric rows:",
    len(build_metrics),
)

print(
    "Condition/noise rows:",
    len(noise_summary),
)

print(
    "ML fit rows:",
    len(fit_times),
)

print(
    "Techniques:",
    sorted(
        project_run[
            "Technique"
        ].unique()
    ),
)


# ------------------------------------------------------------
# 8. BUILD CANONICAL DERIVED TABLES
# ------------------------------------------------------------

# 8.1 Condition-level summary

condition_metric_summary = (
    project_run
    .groupby(
        [
            "NoisePercent",
            "RepetitionSeed",
        ],
        as_index=False,
    )
    .agg(
        TechniqueCount=(
            "Technique",
            "nunique",
        ),

        MeanAPFD_AcrossTechniques=(
            "MeanAPFD",
            "mean",
        ),

        MeanAPFDc_AcrossTechniques=(
            "MeanAPFDc",
            "mean",
        ),

        MaximumAPFD=(
            "MeanAPFD",
            "max",
        ),

        MaximumAPFDc=(
            "MeanAPFDc",
            "max",
        ),

        MinimumAPFD=(
            "MeanAPFD",
            "min",
        ),

        MinimumAPFDc=(
            "MeanAPFDc",
            "min",
        ),
    )
)


best_apfd_rows = (
    project_run.loc[
        project_run.groupby(
            [
                "NoisePercent",
                "RepetitionSeed",
            ]
        )[
            "MeanAPFD"
        ].idxmax(),
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
    ]
    .rename(
        columns={
            "Technique":
                "BestAPFDTechnique",
        }
    )
)


best_apfdc_rows = (
    project_run.loc[
        project_run.groupby(
            [
                "NoisePercent",
                "RepetitionSeed",
            ]
        )[
            "MeanAPFDc"
        ].idxmax(),
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
    ]
    .rename(
        columns={
            "Technique":
                "BestAPFDcTechnique",
        }
    )
)


condition_summary = (
    noise_summary
    .merge(
        condition_metric_summary,
        on=[
            "NoisePercent",
            "RepetitionSeed",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        best_apfd_rows,
        on=[
            "NoisePercent",
            "RepetitionSeed",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        best_apfdc_rows,
        on=[
            "NoisePercent",
            "RepetitionSeed",
        ],
        how="left",
        validate="one_to_one",
    )
)


condition_summary[
    "Project"
] = PROJECT_NAME

condition_summary[
    "ProjectSlug"
] = PROJECT_SLUG

condition_summary[
    "ProjectNumber"
] = PROJECT_NUMBER


condition_summary = (
    condition_summary
    .sort_values(
        [
            "RepetitionSeed",
            "NoisePercent",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# 8.2 Noise injection summary

noise_numeric_fields = {
    "NumberFlipped":
        [
            "NumberFlipped",
            "FlippedRawRows",
        ],

    "RealisedNoisePercent":
        [
            "RealisedNoisePercent",
        ],

    "PassToFailure":
        [
            "PassToFailure",
        ],

    "FailureToPass":
        [
            "FailureToPass",
        ],

    "ModelTrainingLabelChanges":
        [
            "ModelTrainingLabelChanges",
            "LabelChanges",
        ],

    "ElapsedSeconds":
        [
            "ElapsedSeconds",
        ],
}


noise_aggregation_source = noise_summary[
    [
        "NoisePercent",
        "RepetitionSeed",
    ]
].copy()


for canonical_name, candidates in (
    noise_numeric_fields.items()
):

    detected = detect_column(
        noise_summary,
        candidates,
        required=False,
    )

    if detected is not None:

        noise_aggregation_source[
            canonical_name
        ] = pd.to_numeric(
            noise_summary[
                detected
            ],
            errors="coerce",
        )


noise_injection_summary = (
    noise_aggregation_source
    .groupby(
        "NoisePercent",
        as_index=False,
    )
    .agg(
        SeedCount=(
            "RepetitionSeed",
            "nunique",
        )
    )
)


for numeric_column in [
    column
    for column in noise_aggregation_source.columns
    if column
    not in {
        "NoisePercent",
        "RepetitionSeed",
    }
]:

    aggregated = (
        noise_aggregation_source
        .groupby(
            "NoisePercent"
        )[
            numeric_column
        ]
        .agg(
            [
                "mean",
                "std",
                "min",
                "max",
                "sum",
            ]
        )
        .reset_index()
        .rename(
            columns={
                "mean":
                    f"Mean{numeric_column}",

                "std":
                    f"SD{numeric_column}",

                "min":
                    f"Minimum{numeric_column}",

                "max":
                    f"Maximum{numeric_column}",

                "sum":
                    f"Total{numeric_column}",
            }
        )
    )

    noise_injection_summary = (
        noise_injection_summary
        .merge(
            aggregated,
            on="NoisePercent",
            how="left",
            validate="one_to_one",
        )
    )


noise_injection_summary[
    "Project"
] = PROJECT_NAME

noise_injection_summary[
    "ProjectSlug"
] = PROJECT_SLUG

noise_injection_summary[
    "ProjectNumber"
] = PROJECT_NUMBER


# 8.3 Project-level technique/noise summary

summary_group = (
    project_run
    .groupby(
        [
            "NoisePercent",
            "Technique",
        ]
    )
)


project_level_summary = (
    summary_group
    .agg(
        SeedCount=(
            "RepetitionSeed",
            "nunique",
        ),

        MeanAPFD=(
            "MeanAPFD",
            "mean",
        ),

        SDAPFD=(
            "MeanAPFD",
            "std",
        ),

        MedianAPFD=(
            "MeanAPFD",
            "median",
        ),

        MinimumAPFD=(
            "MeanAPFD",
            "min",
        ),

        MaximumAPFD=(
            "MeanAPFD",
            "max",
        ),

        MeanAPFDc=(
            "MeanAPFDc",
            "mean",
        ),

        SDAPFDc=(
            "MeanAPFDc",
            "std",
        ),

        MedianAPFDc=(
            "MeanAPFDc",
            "median",
        ),

        MinimumAPFDc=(
            "MeanAPFDc",
            "min",
        ),

        MaximumAPFDc=(
            "MeanAPFDc",
            "max",
        ),
    )
    .reset_index()
)


project_level_summary[
    "SEAPFD"
] = (
    project_level_summary[
        "SDAPFD"
    ]
    / np.sqrt(
        project_level_summary[
            "SeedCount"
        ]
    )
)


project_level_summary[
    "SEAPFDc"
] = (
    project_level_summary[
        "SDAPFDc"
    ]
    / np.sqrt(
        project_level_summary[
            "SeedCount"
        ]
    )
)


project_level_summary[
    "TCritical95"
] = project_level_summary[
    "SeedCount"
].map(
    lambda count:
        float(
            student_t.ppf(
                0.975,
                int(count) - 1,
            )
        )
)


project_level_summary[
    "CI95LowerAPFD"
] = (
    project_level_summary[
        "MeanAPFD"
    ]
    - project_level_summary[
        "TCritical95"
    ]
    * project_level_summary[
        "SEAPFD"
    ]
)


project_level_summary[
    "CI95UpperAPFD"
] = (
    project_level_summary[
        "MeanAPFD"
    ]
    + project_level_summary[
        "TCritical95"
    ]
    * project_level_summary[
        "SEAPFD"
    ]
)


project_level_summary[
    "CI95LowerAPFDc"
] = (
    project_level_summary[
        "MeanAPFDc"
    ]
    - project_level_summary[
        "TCritical95"
    ]
    * project_level_summary[
        "SEAPFDc"
    ]
)


project_level_summary[
    "CI95UpperAPFDc"
] = (
    project_level_summary[
        "MeanAPFDc"
    ]
    + project_level_summary[
        "TCritical95"
    ]
    * project_level_summary[
        "SEAPFDc"
    ]
)


project_level_summary[
    "Project"
] = PROJECT_NAME

project_level_summary[
    "ProjectSlug"
] = PROJECT_SLUG

project_level_summary[
    "ProjectNumber"
] = PROJECT_NUMBER


project_level_summary = (
    project_level_summary
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# 8.4 Degradation summary

clean_reference = (
    project_level_summary[
        project_level_summary[
            "NoisePercent"
        ].eq(0)
    ][
        [
            "Technique",
            "MeanAPFD",
            "MeanAPFDc",
        ]
    ]
    .rename(
        columns={
            "MeanAPFD":
                "CleanMeanAPFD",

            "MeanAPFDc":
                "CleanMeanAPFDc",
        }
    )
)


project_level_degradation = (
    project_level_summary
    .merge(
        clean_reference,
        on="Technique",
        how="left",
        validate="many_to_one",
    )
)


project_level_degradation[
    "APFDAbsoluteChange"
] = (
    project_level_degradation[
        "MeanAPFD"
    ]
    - project_level_degradation[
        "CleanMeanAPFD"
    ]
)


project_level_degradation[
    "APFDAbsoluteDegradation"
] = (
    project_level_degradation[
        "CleanMeanAPFD"
    ]
    - project_level_degradation[
        "MeanAPFD"
    ]
)


project_level_degradation[
    "APFDRelativeChangePercent"
] = np.where(
    project_level_degradation[
        "CleanMeanAPFD"
    ].ne(0),

    100.0
    * project_level_degradation[
        "APFDAbsoluteChange"
    ]
    / project_level_degradation[
        "CleanMeanAPFD"
    ],

    np.nan,
)


project_level_degradation[
    "APFDDegradationPercent"
] = np.where(
    project_level_degradation[
        "CleanMeanAPFD"
    ].ne(0),

    100.0
    * project_level_degradation[
        "APFDAbsoluteDegradation"
    ]
    / project_level_degradation[
        "CleanMeanAPFD"
    ],

    np.nan,
)


project_level_degradation[
    "APFDRetentionPercent"
] = np.where(
    project_level_degradation[
        "CleanMeanAPFD"
    ].ne(0),

    100.0
    * project_level_degradation[
        "MeanAPFD"
    ]
    / project_level_degradation[
        "CleanMeanAPFD"
    ],

    np.nan,
)


project_level_degradation[
    "APFDcAbsoluteChange"
] = (
    project_level_degradation[
        "MeanAPFDc"
    ]
    - project_level_degradation[
        "CleanMeanAPFDc"
    ]
)


project_level_degradation[
    "APFDcAbsoluteDegradation"
] = (
    project_level_degradation[
        "CleanMeanAPFDc"
    ]
    - project_level_degradation[
        "MeanAPFDc"
    ]
)


project_level_degradation[
    "APFDcRelativeChangePercent"
] = np.where(
    project_level_degradation[
        "CleanMeanAPFDc"
    ].ne(0),

    100.0
    * project_level_degradation[
        "APFDcAbsoluteChange"
    ]
    / project_level_degradation[
        "CleanMeanAPFDc"
    ],

    np.nan,
)


project_level_degradation[
    "APFDcDegradationPercent"
] = np.where(
    project_level_degradation[
        "CleanMeanAPFDc"
    ].ne(0),

    100.0
    * project_level_degradation[
        "APFDcAbsoluteDegradation"
    ]
    / project_level_degradation[
        "CleanMeanAPFDc"
    ],

    np.nan,
)


project_level_degradation[
    "APFDcRetentionPercent"
] = np.where(
    project_level_degradation[
        "CleanMeanAPFDc"
    ].ne(0),

    100.0
    * project_level_degradation[
        "MeanAPFDc"
    ]
    / project_level_degradation[
        "CleanMeanAPFDc"
    ],

    np.nan,
)


# 8.5 ML fit summary

fit_summary = (
    fit_times
    .groupby(
        [
            "NoisePercent",
            "Technique",
        ],
        as_index=False,
    )
    .agg(
        SeedCount=(
            "RepetitionSeed",
            "nunique",
        ),

        MeanFitSeconds=(
            "FitSeconds",
            "mean",
        ),

        SDFitSeconds=(
            "FitSeconds",
            "std",
        ),

        MedianFitSeconds=(
            "FitSeconds",
            "median",
        ),

        MinimumFitSeconds=(
            "FitSeconds",
            "min",
        ),

        MaximumFitSeconds=(
            "FitSeconds",
            "max",
        ),

        TotalFitSeconds=(
            "FitSeconds",
            "sum",
        ),

        MeanTrainingFailures=(
            "TrainingFailures",
            "mean",
        ),

        MeanTrainingPasses=(
            "TrainingPasses",
            "mean",
        ),

        TrainingRows=(
            "TrainingRows",
            "first",
        ),

        ActiveFeatures=(
            "ActiveFeatures",
            "first",
        ),
    )
)


fit_summary[
    "Project"
] = PROJECT_NAME

fit_summary[
    "ProjectSlug"
] = PROJECT_SLUG

fit_summary[
    "ProjectNumber"
] = PROJECT_NUMBER


# Canonical candidate tables.

canonical_tables = {
    "project_run_metrics":
        project_run.sort_values(
            [
                "NoisePercent",
                "RepetitionSeed",
                "Technique",
            ],
            kind="mergesort",
        ).reset_index(drop=True),

    "build_metrics":
        build_metrics.sort_values(
            [
                "NoisePercent",
                "RepetitionSeed",
                "Technique",
                "BuildOrder",
                "Build",
            ],
            kind="mergesort",
        ).reset_index(drop=True),

    "noise_summary_all":
        noise_summary.sort_values(
            [
                "RepetitionSeed",
                "NoisePercent",
            ],
            kind="mergesort",
        ).reset_index(drop=True),

    "fit_times":
        fit_times.sort_values(
            [
                "NoisePercent",
                "RepetitionSeed",
                "Technique",
            ],
            kind="mergesort",
        ).reset_index(drop=True),

    "condition_summary":
        condition_summary,

    "noise_injection_summary":
        noise_injection_summary,

    "project_level_summary":
        project_level_summary,

    "project_level_degradation":
        project_level_degradation,

    "fit_summary":
        fit_summary,
}


# ------------------------------------------------------------
# 9. COLUMN-ALIAS SUPPORT
# ------------------------------------------------------------

ALIAS_TO_SOURCE = {
    # Project identity
    "ProjectName":
        "Project",

    "Repository":
        "Project",

    "ProjectID":
        "Project",

    # Noise
    "Noise":
        "NoisePercent",

    "NoiseLevel":
        "NoisePercent",

    "NoisePct":
        "NoisePercent",

    "NoisePercentage":
        "NoisePercent",

    "RequestedNoisePercent":
        "NoisePercent",

    # Seed
    "Seed":
        "RepetitionSeed",

    "RunSeed":
        "RepetitionSeed",

    "RandomSeed":
        "RepetitionSeed",

    "Repetition":
        "RepetitionSeed",

    # Technique
    "Method":
        "Technique",

    "Model":
        "Technique",

    "TechniqueName":
        "Technique",

    # Seed count
    "Seeds":
        "SeedCount",

    "NumberOfSeeds":
        "SeedCount",

    "Repetitions":
        "SeedCount",

    "Runs":
        "SeedCount",

    "N":
        "SeedCount",

    # APFD
    "AverageAPFD":
        "MeanAPFD",

    "APFDMean":
        "MeanAPFD",

    "Mean_APFD":
        "MeanAPFD",

    "StdAPFD":
        "SDAPFD",

    "APFDStd":
        "SDAPFD",

    "APFDSD":
        "SDAPFD",

    "SD_APFD":
        "SDAPFD",

    "APFDMedian":
        "MedianAPFD",

    "APFDMin":
        "MinimumAPFD",

    "APFDMax":
        "MaximumAPFD",

    # APFDc
    "AverageAPFDc":
        "MeanAPFDc",

    "APFDcMean":
        "MeanAPFDc",

    "Mean_APFDc":
        "MeanAPFDc",

    "StdAPFDc":
        "SDAPFDc",

    "APFDcStd":
        "SDAPFDc",

    "APFDcSD":
        "SDAPFDc",

    "SD_APFDc":
        "SDAPFDc",

    "APFDcMedian":
        "MedianAPFDc",

    "APFDcMin":
        "MinimumAPFDc",

    "APFDcMax":
        "MaximumAPFDc",

    # Clean and degradation fields
    "CleanAPFD":
        "CleanMeanAPFD",

    "BaselineAPFD":
        "CleanMeanAPFD",

    "CleanAPFDc":
        "CleanMeanAPFDc",

    "BaselineAPFDc":
        "CleanMeanAPFDc",

    "APFDChange":
        "APFDAbsoluteChange",

    "APFDDegradation":
        "APFDAbsoluteDegradation",

    "RelativeAPFDChangePercent":
        "APFDRelativeChangePercent",

    "RelativeAPFDDegradationPercent":
        "APFDDegradationPercent",

    "APFDRetention":
        "APFDRetentionPercent",

    "APFDcChange":
        "APFDcAbsoluteChange",

    "APFDcDegradation":
        "APFDcAbsoluteDegradation",

    "RelativeAPFDcChangePercent":
        "APFDcRelativeChangePercent",

    "RelativeAPFDcDegradationPercent":
        "APFDcDegradationPercent",

    "APFDcRetention":
        "APFDcRetentionPercent",

    # Noise injection
    "FlippedRows":
        "NumberFlipped",

    "FlippedRawRows":
        "NumberFlipped",

    "RealizedNoisePercent":
        "RealisedNoisePercent",

    "MeanFlippedRows":
        "MeanNumberFlipped",

    "MeanFlippedRawRows":
        "MeanNumberFlipped",

    "MeanRealizedNoisePercent":
        "MeanRealisedNoisePercent",

    # Fit time
    "AverageFitSeconds":
        "MeanFitSeconds",

    "FitSecondsMean":
        "MeanFitSeconds",

    "FitSecondsSD":
        "SDFitSeconds",
}


def augment_aliases(
    dataframe,
):
    dataframe = dataframe.copy()

    for alias, source in (
        ALIAS_TO_SOURCE.items()
    ):

        if (
            alias not in dataframe.columns
            and source in dataframe.columns
        ):

            dataframe[
                alias
            ] = dataframe[
                source
            ]

    if (
        "Technique"
        in dataframe.columns
        and "TechniqueSlug"
        not in dataframe.columns
    ):

        dataframe[
            "TechniqueSlug"
        ] = dataframe[
            "Technique"
        ].map(
            TECHNIQUE_TO_SLUG
        )

    if (
        "NoisePercent"
        in dataframe.columns
        and "NoiseLabel"
        not in dataframe.columns
    ):

        dataframe[
            "NoiseLabel"
        ] = dataframe[
            "NoisePercent"
        ].map(
            lambda value:
                f"{int(value)}%"
        )

    if (
        "ConditionID"
        not in dataframe.columns
        and {
            "NoisePercent",
            "RepetitionSeed",
        }.issubset(
            dataframe.columns
        )
    ):

        dataframe[
            "ConditionID"
        ] = (
            dataframe[
                "NoisePercent"
            ]
            .astype(int)
            .map(
                lambda value:
                    f"noise_{value:03d}"
            )
            + "__"
            + dataframe[
                "RepetitionSeed"
            ]
            .astype(int)
            .map(
                lambda value:
                    f"seed_{value:03d}"
            )
        )

    return dataframe


def resolve_reference_column(
    reference_column,
    generated_dataframe,
):
    generated_lookup = {
        normalise_column_name(column):
            column
        for column in generated_dataframe.columns
    }

    reference_normalised = (
        normalise_column_name(
            reference_column
        )
    )

    if reference_normalised in generated_lookup:

        return (
            "column",
            generated_lookup[
                reference_normalised
            ],
        )

    # Constant project fields.

    constant_fields = {
        "projectnumber":
            PROJECT_NUMBER,

        "projectslug":
            PROJECT_SLUG,

        "project":
            PROJECT_NAME,

        "projectname":
            PROJECT_NAME,

        "generatedatutc":
            GENERATED_AT_UTC,

        "createdatutc":
            GENERATED_AT_UTC,

        "completedatutc":
            GENERATED_AT_UTC,
    }

    if reference_normalised in constant_fields:

        return (
            "constant",
            constant_fields[
                reference_normalised
            ],
        )

    if reference_normalised.startswith(
        "unnamed"
    ):

        return (
            "sequence",
            None,
        )

    return (
        None,
        None,
    )


def align_to_reference_schema(
    generated_dataframe,
    reference_dataframe,
    relative_path,
):
    generated_dataframe = augment_aliases(
        generated_dataframe
    )

    output = pd.DataFrame(
        index=np.arange(
            len(
                generated_dataframe
            )
        )
    )

    unresolved_columns = []


    for reference_column in (
        reference_dataframe.columns
    ):

        resolution_type, resolution_value = (
            resolve_reference_column(
                reference_column,
                generated_dataframe,
            )
        )

        if resolution_type == "column":

            output[
                reference_column
            ] = generated_dataframe[
                resolution_value
            ].reset_index(drop=True)

        elif resolution_type == "constant":

            output[
                reference_column
            ] = resolution_value

        elif resolution_type == "sequence":

            output[
                reference_column
            ] = np.arange(
                len(
                    generated_dataframe
                )
            )

        else:

            unresolved_columns.append(
                reference_column
            )


    if unresolved_columns:

        raise RuntimeError(
            "Could not map all reference-schema columns.\n"
            f"Package file: {relative_path}\n"
            f"Unresolved columns: {unresolved_columns}\n"
            f"Generated columns: "
            f"{list(generated_dataframe.columns)}"
        )


    # Match the broad reference dtypes where safe.

    for column in reference_dataframe.columns:

        reference_dtype = (
            reference_dataframe[
                column
            ].dtype
        )

        try:

            if pd.api.types.is_integer_dtype(
                reference_dtype
            ):

                numeric = pd.to_numeric(
                    output[column],
                    errors="raise",
                )

                if numeric.isna().any():
                    output[column] = (
                        numeric.astype("Int64")
                    )

                else:
                    output[column] = (
                        numeric.astype(
                            reference_dtype
                        )
                    )

            elif pd.api.types.is_float_dtype(
                reference_dtype
            ):

                output[column] = pd.to_numeric(
                    output[column],
                    errors="coerce",
                ).astype(float)

            elif pd.api.types.is_bool_dtype(
                reference_dtype
            ):

                output[column] = (
                    output[column]
                    .astype(bool)
                )

        except Exception as error:

            raise RuntimeError(
                "Failed to align a reference dtype.\n"
                f"Package file: {relative_path}\n"
                f"Column: {column}\n"
                f"Reference dtype: {reference_dtype}\n"
                f"Error: {error}"
            )


    return output


# ------------------------------------------------------------
# 10. SELECT THE CANONICAL TABLE FOR EACH TOP-LEVEL FILE
# ------------------------------------------------------------

def score_candidate_schema(
    candidate_dataframe,
    reference_dataframe,
):
    candidate_dataframe = augment_aliases(
        candidate_dataframe
    )

    resolved = 0

    for reference_column in (
        reference_dataframe.columns
    ):

        resolution_type, _ = (
            resolve_reference_column(
                reference_column,
                candidate_dataframe,
            )
        )

        if resolution_type is not None:
            resolved += 1

    schema_fraction = (
        resolved
        / max(
            len(
                reference_dataframe.columns
            ),
            1,
        )
    )

    row_match = (
        len(candidate_dataframe)
        == len(reference_dataframe)
    )

    return (
        schema_fraction
        + (
            1.0
            if row_match
            else 0.0
        )
    )


def choose_top_level_table(
    relative_path,
    reference_dataframe,
):
    file_name = Path(
        relative_path
    ).name.lower()


    # Strong filename rules first.

    direct_rules = [
        (
            (
                "project_run_metrics_all"
                in file_name
            ),
            "project_run_metrics",
        ),

        (
            (
                "build_metric"
                in file_name
            ),
            "build_metrics",
        ),

        (
            (
                "fit_time"
                in file_name
                and "summary"
                not in file_name
            ),
            "fit_times",
        ),

        (
            (
                "condition_summary"
                in file_name
            ),
            "condition_summary",
        ),

        (
            (
                "noise_injection_summary"
                in file_name
            ),
            "noise_injection_summary",
        ),

        (
            (
                "degradation"
                in file_name
            ),
            "project_level_degradation",
        ),

        (
            (
                "noise_technique_summary"
                in file_name
            ),
            "project_level_summary",
        ),

        (
            (
                "fit"
                in file_name
                and "summary"
                in file_name
            ),
            "fit_summary",
        ),

        (
            (
                "noise_summary"
                in file_name
            ),
            "noise_summary_all",
        ),
    ]


    for condition, table_name in (
        direct_rules
    ):

        if condition:

            return (
                table_name,
                canonical_tables[
                    table_name
                ],
                "filename_rule",
            )


    # Schema and row-count fallback.

    scored = []


    for table_name, table in (
        canonical_tables.items()
    ):

        score = score_candidate_schema(
            table,
            reference_dataframe,
        )

        scored.append({
            "TableName":
                table_name,

            "Score":
                score,

            "Rows":
                len(table),
        })


    scored_frame = (
        pd.DataFrame(
            scored
        )
        .sort_values(
            [
                "Score",
                "TableName",
            ],
            ascending=[
                False,
                True,
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    best = scored_frame.iloc[0]

    second_score = (
        float(
            scored_frame.iloc[1][
                "Score"
            ]
        )
        if len(scored_frame) > 1
        else -np.inf
    )


    if (
        float(
            best[
                "Score"
            ]
        )
        < 1.50
        or (
            float(
                best[
                    "Score"
                ]
            )
            - second_score
        )
        < 0.05
    ):

        print(
            "\nAmbiguous table classification:"
        )

        print(
            "Package file:",
            relative_path,
        )

        display(
            scored_frame
        )

        raise RuntimeError(
            "Could not classify a top-level package table "
            "unambiguously."
        )


    selected_name = str(
        best[
            "TableName"
        ]
    )


    return (
        selected_name,
        canonical_tables[
            selected_name
        ],
        "schema_and_row_count",
    )


# ------------------------------------------------------------
# 11. CREATE THE STAGING PACKAGE
# ------------------------------------------------------------

if STAGING_PACKAGE_DIR.exists():

    shutil.rmtree(
        STAGING_PACKAGE_DIR
    )


STAGING_PACKAGE_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


generation_records = []
technique_noise_rows_generated = 0
technique_noise_combinations = set()


for relative_path in sorted(
    reference_relative_paths
):

    reference_path = (
        reference_package_directory
        / relative_path
    )

    reference_dataframe = read_table(
        reference_path
    )


    normalised_relative_path = (
        relative_path
        .replace("\\", "/")
    )


    if normalised_relative_path.startswith(
        "technique_noise/"
    ):

        file_name = Path(
            normalised_relative_path
        ).name


        match = re.fullmatch(
            (
                r"([a-z0-9_\-]+)"
                r"__noise_(\d{3})"
                r"\.(csv|csv\.gz|parquet)"
            ),
            file_name.lower(),
        )


        if match is None:

            raise RuntimeError(
                "Could not parse a technique-noise filename:\n"
                f"{relative_path}"
            )


        technique_slug = (
            match.group(1)
            .replace("-", "_")
        )

        noise_percent = int(
            match.group(2)
        )


        if technique_slug not in (
            SLUG_TO_TECHNIQUE
        ):

            raise RuntimeError(
                "Unknown technique slug in reference package:\n"
                f"{technique_slug}"
            )


        technique = SLUG_TO_TECHNIQUE[
            technique_slug
        ]


        generated_source = (
            project_run[
                project_run[
                    "Technique"
                ].eq(
                    technique
                )
                &
                project_run[
                    "NoisePercent"
                ].eq(
                    noise_percent
                )
            ]
            .sort_values(
                "RepetitionSeed",
                kind="mergesort",
            )
            .reset_index(drop=True)
            .copy()
        )


        if len(
            generated_source
        ) != 30:

            raise AssertionError(
                "Each technique-noise file must contain "
                "30 seed rows.\n"
                f"Technique: {technique}\n"
                f"Noise: {noise_percent}\n"
                f"Rows: {len(generated_source)}"
            )


        table_class = (
            "technique_noise_project_run_metrics"
        )

        classification_method = (
            "filename_parse"
        )

        technique_noise_rows_generated += (
            len(generated_source)
        )

        technique_noise_combinations.add(
            (
                technique,
                noise_percent,
            )
        )


    else:

        (
            table_class,
            generated_source,
            classification_method,
        ) = choose_top_level_table(
            relative_path,
            reference_dataframe,
        )


    output_dataframe = (
        align_to_reference_schema(
            generated_dataframe=(
                generated_source
            ),

            reference_dataframe=(
                reference_dataframe
            ),

            relative_path=(
                relative_path
            ),
        )
    )


    output_dataframe = replace_project_identity(
        output_dataframe
    )


    # Confirm no reference-project identifier survives.

    reference_identity_cells = 0


    for column in output_dataframe.columns:

        if (
            pd.api.types.is_object_dtype(
                output_dataframe[column]
            )
            or pd.api.types.is_string_dtype(
                output_dataframe[column]
            )
        ):

            text_values = (
                output_dataframe[
                    column
                ]
                .fillna("")
                .astype(str)
                .str.lower()
            )

            reference_identity_cells += int(
                text_values.str.contains(
                    "compevol@beast2",
                    regex=False,
                ).sum()
            )

            reference_identity_cells += int(
                text_values.str.contains(
                    "compevol__beast2",
                    regex=False,
                ).sum()
            )

            reference_identity_cells += int(
                text_values.str.contains(
                    "beast2",
                    regex=False,
                ).sum()
            )


    if reference_identity_cells != 0:

        raise AssertionError(
            "A generated package table still contains "
            "Project 7 identity text.\n"
            f"File: {relative_path}"
        )


    output_path = (
        STAGING_PACKAGE_DIR
        / relative_path
    )


    write_table(
        output_path,
        output_dataframe,
    )


    generation_records.append({
        "RelativePath":
            relative_path,

        "ReferenceRows":
            len(
                reference_dataframe
            ),

        "GeneratedRows":
            len(
                output_dataframe
            ),

        "Columns":
            len(
                output_dataframe.columns
            ),

        "TableClass":
            table_class,

        "ClassificationMethod":
            classification_method,

        "OutputPath":
            str(
                output_path
            ),
    })


generation_map = pd.DataFrame(
    generation_records
)


# ------------------------------------------------------------
# 12. VALIDATE THE COMPLETE STAGING PACKAGE
# ------------------------------------------------------------

staging_files = sorted([
    path
    for path in STAGING_PACKAGE_DIR.rglob("*")
    if path.is_file()
])


staging_relative_paths = {
    path.relative_to(
        STAGING_PACKAGE_DIR
    ).as_posix()
    for path in staging_files
}


expected_relative_path_set = set(
    reference_relative_paths
)


missing_package_files = sorted(
    expected_relative_path_set
    - staging_relative_paths
)


unexpected_package_files = sorted(
    staging_relative_paths
    - expected_relative_path_set
)


if len(
    staging_files
) != EXPECTED_PACKAGE_FILES:

    raise AssertionError(
        "Generated package file count differs.\n"
        f"Expected: {EXPECTED_PACKAGE_FILES}\n"
        f"Actual: {len(staging_files)}"
    )


if missing_package_files:

    raise AssertionError(
        "Generated package is missing files:\n"
        + "\n".join(
            missing_package_files
        )
    )


if unexpected_package_files:

    raise AssertionError(
        "Generated package contains unexpected files:\n"
        + "\n".join(
            unexpected_package_files
        )
    )


if (
    len(
        technique_noise_combinations
    )
    != EXPECTED_TECHNIQUE_NOISE_FILES
):

    raise AssertionError(
        "Technique-noise combination count differs.\n"
        f"Expected: {EXPECTED_TECHNIQUE_NOISE_FILES}\n"
        f"Actual: {len(technique_noise_combinations)}"
    )


expected_technique_noise_combinations = {
    (
        technique,
        noise_percent,
    )
    for technique in TECHNIQUES
    for noise_percent in NOISE_LEVELS
}


if (
    technique_noise_combinations
    != expected_technique_noise_combinations
):

    missing_combinations = sorted(
        expected_technique_noise_combinations
        - technique_noise_combinations
    )

    unexpected_combinations = sorted(
        technique_noise_combinations
        - expected_technique_noise_combinations
    )

    raise AssertionError(
        "Technique-noise package combinations differ.\n"
        f"Missing: {missing_combinations}\n"
        f"Unexpected: {unexpected_combinations}"
    )


if (
    technique_noise_rows_generated
    != EXPECTED_PROJECT_RUN_ROWS
):

    raise AssertionError(
        "Technique-noise files do not reproduce all "
        "1,890 project-run rows.\n"
        f"Detected: {technique_noise_rows_generated}"
    )


# Validate every package table by reading it back.

readback_failures = []
project7_identity_files = []
invalid_metric_files = []
reference_hash_matches = []


reference_hash_lookup = {
    relative_path:
        calculate_sha256(
            reference_package_directory
            / relative_path
        )
    for relative_path
    in reference_relative_paths
}


for relative_path in sorted(
    reference_relative_paths
):

    package_file_path = (
        STAGING_PACKAGE_DIR
        / relative_path
    )


    try:

        package_frame = read_table(
            package_file_path
        )

    except Exception as error:

        readback_failures.append({
            "RelativePath":
                relative_path,

            "Error":
                str(error),
        })

        continue


    # Detect any copied Project 7 identifiers.

    contains_reference_identity = False


    for column in package_frame.columns:

        if (
            pd.api.types.is_object_dtype(
                package_frame[column]
            )
            or pd.api.types.is_string_dtype(
                package_frame[column]
            )
        ):

            text_values = (
                package_frame[
                    column
                ]
                .fillna("")
                .astype(str)
                .str.lower()
            )

            if (
                text_values.str.contains(
                    "beast2",
                    regex=False,
                ).any()
                or
                text_values.str.contains(
                    "compevol",
                    regex=False,
                ).any()
            ):

                contains_reference_identity = True
                break


    if contains_reference_identity:

        project7_identity_files.append(
            relative_path
        )


    # Validate actual APFD/APFDc metric columns.
    # Degradation and change columns are not constrained to [0,1].

    for column in package_frame.columns:

        normalised_column = (
            normalise_column_name(
                column
            )
        )


        direct_metric_column = (
            normalised_column
            in {
                "apfd",
                "apfdc",
                "meanapfd",
                "meanapfdc",
                "averageapfd",
                "averageapfdc",
                "medianapfd",
                "medianapfdc",
                "minimumapfd",
                "minimumapfdc",
                "maximumapfd",
                "maximumapfdc",
                "cleanmeanapfd",
                "cleanmeanapfdc",
            }
        )


        if not direct_metric_column:
            continue


        numeric_values = pd.to_numeric(
            package_frame[column],
            errors="coerce",
        ).to_numpy(dtype=float)


        finite_values = numeric_values[
            np.isfinite(
                numeric_values
            )
        ]


        if (
            len(finite_values) > 0
            and (
                (finite_values < 0)
                | (finite_values > 1)
            ).any()
        ):

            invalid_metric_files.append({
                "RelativePath":
                    relative_path,

                "Column":
                    column,
            })


    generated_hash = calculate_sha256(
        package_file_path
    )


    if (
        generated_hash
        == reference_hash_lookup[
            relative_path
        ]
    ):

        reference_hash_matches.append(
            relative_path
        )


if readback_failures:

    display(
        pd.DataFrame(
            readback_failures
        )
    )

    raise RuntimeError(
        "At least one generated package table could not "
        "be read back."
    )


if project7_identity_files:

    raise AssertionError(
        "Generated package files contain Project 7 identity:\n"
        + "\n".join(
            project7_identity_files
        )
    )


if invalid_metric_files:

    display(
        pd.DataFrame(
            invalid_metric_files
        )
    )

    raise AssertionError(
        "Generated package contains invalid APFD/APFDc "
        "metric values."
    )


if reference_hash_matches:

    raise AssertionError(
        "At least one Project 8 package file is byte-identical "
        "to its Project 7 reference file. This suggests copied "
        "reference content:\n"
        + "\n".join(
            reference_hash_matches
        )
    )


# ------------------------------------------------------------
# 13. PROMOTE STAGING PACKAGE TO FINAL CANDIDATE
# ------------------------------------------------------------

if FINAL_PACKAGE_DIR.exists():

    shutil.rmtree(
        FINAL_PACKAGE_DIR
    )


STAGING_PACKAGE_DIR.replace(
    FINAL_PACKAGE_DIR
)


# ------------------------------------------------------------
# 14. FINAL PACKAGE INVENTORY AND ROOT HASH
# ------------------------------------------------------------

final_package_files = sorted([
    path
    for path in FINAL_PACKAGE_DIR.rglob("*")
    if path.is_file()
])


inventory_records = []


for path in final_package_files:

    inventory_records.append({
        "RelativePath":
            path.relative_to(
                FINAL_PACKAGE_DIR
            ).as_posix(),

        "SizeBytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            calculate_sha256(
                path
            ),
    })


final_package_inventory = pd.DataFrame(
    inventory_records
)


final_package_files_count = len(
    final_package_inventory
)


final_package_bytes = int(
    final_package_inventory[
        "SizeBytes"
    ].sum()
)


final_package_root_sha256 = (
    root_inventory_hash(
        final_package_inventory
    )
)


# ------------------------------------------------------------
# 15. FINAL PACKAGE AUDIT
# ------------------------------------------------------------

final_relative_paths = set(
    final_package_inventory[
        "RelativePath"
    ]
)


final_missing_files = sorted(
    expected_relative_path_set
    - final_relative_paths
)


final_unexpected_files = sorted(
    final_relative_paths
    - expected_relative_path_set
)


top_level_generated_files = int(
    sum(
        1
        for relative_path
        in final_relative_paths
        if not relative_path.startswith(
            "technique_noise/"
        )
    )
)


technique_noise_generated_files = int(
    sum(
        1
        for relative_path
        in final_relative_paths
        if relative_path.startswith(
            "technique_noise/"
        )
    )
)


validation_records = [
    {
        "Check":
            "Project 8 Step 10 passed",

        "Expected":
            EXPECTED_STEP10_STATUS,

        "Actual":
            step10_status[
                "Status"
            ],

        "Pass":
            step10_status[
                "Status"
            ]
            == EXPECTED_STEP10_STATUS,
    },

    {
        "Check":
            "Reference package files",

        "Expected":
            EXPECTED_PACKAGE_FILES,

        "Actual":
            len(
                reference_relative_paths
            ),

        "Pass":
            len(
                reference_relative_paths
            )
            == EXPECTED_PACKAGE_FILES,
    },

    {
        "Check":
            "Final package files",

        "Expected":
            EXPECTED_PACKAGE_FILES,

        "Actual":
            final_package_files_count,

        "Pass":
            final_package_files_count
            == EXPECTED_PACKAGE_FILES,
    },

    {
        "Check":
            "Final top-level files",

        "Expected":
            EXPECTED_TOP_LEVEL_FILES,

        "Actual":
            top_level_generated_files,

        "Pass":
            top_level_generated_files
            == EXPECTED_TOP_LEVEL_FILES,
    },

    {
        "Check":
            "Final technique-noise files",

        "Expected":
            EXPECTED_TECHNIQUE_NOISE_FILES,

        "Actual":
            technique_noise_generated_files,

        "Pass":
            technique_noise_generated_files
            == EXPECTED_TECHNIQUE_NOISE_FILES,
    },

    {
        "Check":
            "Missing package files",

        "Expected":
            0,

        "Actual":
            len(
                final_missing_files
            ),

        "Pass":
            len(
                final_missing_files
            )
            == 0,
    },

    {
        "Check":
            "Unexpected package files",

        "Expected":
            0,

        "Actual":
            len(
                final_unexpected_files
            ),

        "Pass":
            len(
                final_unexpected_files
            )
            == 0,
    },

    {
        "Check":
            "Technique-noise combinations",

        "Expected":
            EXPECTED_TECHNIQUE_NOISE_FILES,

        "Actual":
            len(
                technique_noise_combinations
            ),

        "Pass":
            len(
                technique_noise_combinations
            )
            == EXPECTED_TECHNIQUE_NOISE_FILES,
    },

    {
        "Check":
            "Technique-noise project-run rows",

        "Expected":
            EXPECTED_PROJECT_RUN_ROWS,

        "Actual":
            technique_noise_rows_generated,

        "Pass":
            technique_noise_rows_generated
            == EXPECTED_PROJECT_RUN_ROWS,
    },

    {
        "Check":
            "Readback failures",

        "Expected":
            0,

        "Actual":
            len(
                readback_failures
            ),

        "Pass":
            len(
                readback_failures
            )
            == 0,
    },

    {
        "Check":
            "Project 7 identity files",

        "Expected":
            0,

        "Actual":
            len(
                project7_identity_files
            ),

        "Pass":
            len(
                project7_identity_files
            )
            == 0,
    },

    {
        "Check":
            "Invalid APFD/APFDc files",

        "Expected":
            0,

        "Actual":
            len(
                invalid_metric_files
            ),

        "Pass":
            len(
                invalid_metric_files
            )
            == 0,
    },

    {
        "Check":
            "Files identical to Project 7 reference",

        "Expected":
            0,

        "Actual":
            len(
                reference_hash_matches
            ),

        "Pass":
            len(
                reference_hash_matches
            )
            == 0,
    },

    {
        "Check":
            "Project 8 registry rows",

        "Expected":
            0,

        "Actual":
            len(
                project8_registry_rows
            ),

        "Pass":
            len(
                project8_registry_rows
            )
            == 0,
    },
]


final_package_validation = pd.DataFrame(
    validation_records
)


failed_validation_checks = (
    final_package_validation[
        ~final_package_validation[
            "Pass"
        ]
    ]
    .copy()
)


print("\nFinal package validation:")

display(
    final_package_validation
)


if not failed_validation_checks.empty:

    print("\nFailed package checks:")

    display(
        failed_validation_checks
    )

    raise RuntimeError(
        "PROJECT 8 STEP 11B DID NOT PASS.\n"
        "Do not update the completion registry."
    )


# ------------------------------------------------------------
# 16. WRITE PACKAGE AUDIT OUTPUTS
# ------------------------------------------------------------

FINAL_PACKAGE_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    FINAL_PACKAGE_INVENTORY_PATH,
    final_package_inventory,
)


atomic_write_csv(
    FINAL_PACKAGE_VALIDATION_PATH,
    final_package_validation,
)


atomic_write_csv(
    FINAL_PACKAGE_GENERATION_MAP_PATH,
    generation_map,
)


step11b_status_text = (
    "PASS_PROJECT_8_FINAL_72_FILE_PACKAGE_GENERATED_AND_VALIDATED"
)


final_package_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "GeneratedAtUTC":
        GENERATED_AT_UTC,

    "SourceResults": {
        "ProjectRunMetrics": {
            "Path":
                str(
                    PROJECT_RUN_INPUT
                ),

            "Rows":
                len(
                    project_run
                ),

            "SHA256":
                calculate_sha256(
                    PROJECT_RUN_INPUT
                ),
        },

        "BuildMetrics": {
            "Path":
                str(
                    BUILD_METRICS_INPUT
                ),

            "Rows":
                len(
                    build_metrics
                ),

            "SHA256":
                calculate_sha256(
                    BUILD_METRICS_INPUT
                ),
        },

        "NoiseSummary": {
            "Path":
                str(
                    NOISE_SUMMARY_INPUT
                ),

            "Rows":
                len(
                    noise_summary
                ),

            "SHA256":
                calculate_sha256(
                    NOISE_SUMMARY_INPUT
                ),
        },

        "FitTimes": {
            "Path":
                str(
                    FIT_TIMES_INPUT
                ),

            "Rows":
                len(
                    fit_times
                ),

            "SHA256":
                calculate_sha256(
                    FIT_TIMES_INPUT
                ),
        },
    },

    "ReferenceBlueprint": {
        "Project":
            REFERENCE_PROJECT_NAME,

        "PackageDirectory":
            str(
                reference_package_directory
            ),

        "InventoryPath":
            str(
                reference_inventory_path
            ),

        "RelativePathsUsed":
            len(
                reference_relative_paths
            ),

        "ReferenceValuesCopied":
            False,

        "ReferenceFilesWithMatchingSHA256":
            len(
                reference_hash_matches
            ),
    },

    "FinalPackage": {
        "Directory":
            str(
                FINAL_PACKAGE_DIR
            ),

        "Files":
            final_package_files_count,

        "TopLevelFiles":
            top_level_generated_files,

        "TechniqueNoiseFiles":
            technique_noise_generated_files,

        "Bytes":
            final_package_bytes,

        "RootSHA256":
            final_package_root_sha256,

        "MissingFiles":
            len(
                final_missing_files
            ),

        "UnexpectedFiles":
            len(
                final_unexpected_files
            ),
    },

    "ExperimentDimensions": {
        "NoiseLevels":
            len(
                NOISE_LEVELS
            ),

        "Seeds":
            len(
                SEEDS
            ),

        "Techniques":
            len(
                TECHNIQUES
            ),

        "Conditions":
            EXPECTED_CONDITIONS,

        "ProjectRunRows":
            EXPECTED_PROJECT_RUN_ROWS,

        "BuildMetricRows":
            EXPECTED_BUILD_METRIC_ROWS,

        "MLFits":
            EXPECTED_FIT_ROWS,
    },

    "Validation": {
        "Checks":
            len(
                final_package_validation
            ),

        "FailedChecks":
            len(
                failed_validation_checks
            ),

        "ReadbackFailures":
            len(
                readback_failures
            ),

        "Project7IdentityFiles":
            len(
                project7_identity_files
            ),

        "InvalidMetricFiles":
            len(
                invalid_metric_files
            ),
    },

    "CompletionRegistryModified":
        False,

    "Projects1To7Modified":
        False,

    "Status":
        step11b_status_text,
}


atomic_write_json(
    FINAL_PACKAGE_REPORT_PATH,
    final_package_report,
)


step11b_status = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        step11b_status_text,

    "RawFiles":
        EXPECTED_RAW_FILES,

    "RawBytes":
        EXPECTED_RAW_BYTES,

    "RawRootSHA256":
        EXPECTED_RAW_ROOT_SHA256,

    "PackageFiles":
        final_package_files_count,

    "PackageBytes":
        final_package_bytes,

    "PackageRootSHA256":
        final_package_root_sha256,

    "TopLevelFiles":
        top_level_generated_files,

    "TechniqueNoiseFiles":
        technique_noise_generated_files,

    "MissingPackageFiles":
        len(
            final_missing_files
        ),

    "UnexpectedPackageFiles":
        len(
            final_unexpected_files
        ),

    "FailedValidationChecks":
        len(
            failed_validation_checks
        ),

    "CompletionRegistryModified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP11B_STATUS_PATH,
    step11b_status,
)


expected_audit_outputs = [
    FINAL_PACKAGE_INVENTORY_PATH,
    FINAL_PACKAGE_VALIDATION_PATH,
    FINAL_PACKAGE_GENERATION_MAP_PATH,
    FINAL_PACKAGE_REPORT_PATH,
    STEP11B_STATUS_PATH,
]


missing_audit_outputs = [
    str(path)
    for path in expected_audit_outputs
    if not path.exists()
]


if missing_audit_outputs:

    raise RuntimeError(
        "Step 11B audit outputs are missing:\n"
        + "\n".join(
            missing_audit_outputs
        )
    )


# ------------------------------------------------------------
# 17. DISPLAY KEY PROJECT 8 RESULTS
# ------------------------------------------------------------

result_display = (
    project_level_summary[
        [
            "NoisePercent",
            "Technique",
            "SeedCount",
            "MeanAPFD",
            "SDAPFD",
            "MeanAPFDc",
            "SDAPFDc",
        ]
    ]
    .sort_values(
        [
            "NoisePercent",
            "MeanAPFDc",
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


print("\nProject 8 APFD/APFDc summary written into the package:")

display(
    result_display.round(6)
)


# ------------------------------------------------------------
# 18. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 96)
print("=== PROJECT 8 STEP 11B RESULT ===")
print("=" * 96)

print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nAudited source results:")

print(
    "Conditions:",
    EXPECTED_CONDITIONS,
)

print(
    "Project-run rows:",
    len(
        project_run
    ),
)

print(
    "Build-metric rows:",
    len(
        build_metrics
    ),
)

print(
    "ML fits:",
    len(
        fit_times
    ),
)

print(
    "Techniques:",
    len(
        TECHNIQUES
    ),
)


print("\nFinal package:")

print(
    "Directory:",
    FINAL_PACKAGE_DIR,
)

print(
    "Files:",
    final_package_files_count,
)

print(
    "Top-level files:",
    top_level_generated_files,
)

print(
    "Technique-noise files:",
    technique_noise_generated_files,
)

print(
    "Package bytes:",
    final_package_bytes,
)

print(
    "Package root SHA-256:",
    final_package_root_sha256,
)

print(
    "Missing files:",
    len(
        final_missing_files
    ),
)

print(
    "Unexpected files:",
    len(
        final_unexpected_files
    ),
)


print("\nContent validation:")

print(
    "Technique-noise combinations:",
    len(
        technique_noise_combinations
    ),
)

print(
    "Technique-noise rows:",
    technique_noise_rows_generated,
)

print(
    "Readback failures:",
    len(
        readback_failures
    ),
)

print(
    "Project 7 identity files:",
    len(
        project7_identity_files
    ),
)

print(
    "Files identical to Project 7:",
    len(
        reference_hash_matches
    ),
)

print(
    "Invalid APFD/APFDc files:",
    len(
        invalid_metric_files
    ),
)


print("\nValidation:")

print(
    "Checks:",
    len(
        final_package_validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation_checks
    ),
)


print("\nAudit outputs:")

for output_path in (
    expected_audit_outputs
):

    print(output_path)


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–7 modified:")
print(0)


print(
    "\nSTATUS:",
    step11b_status_text,
)

print("=" * 96)

=== PROJECT 8 STEP 11B: GENERATE FINAL 72-FILE RESULT PACKAGE ===

Input validation:
Step 10 status: PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED
Raw files: 2160
Raw bytes: 179766494
Raw SHA-256: 19ae21c5d8524c358796e28f23577fbf03bfd31f7d4b530372c6b4d8a492887c
Project 8 registry rows: 0

Reference package blueprint:
Package directory: /content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_30_seed_final
Inventory rows: 72
Top-level files: 9
Technique-noise files: 63

Top-level package files:
 - build_metrics_all.parquet: 98280 rows × 10 columns
 - condition_summary.csv: 270 rows × 16 columns
 - fit_times_all.csv: 1080 rows × 9 columns
 - noise_injection_summary.csv: 9 rows × 17 columns
 - prediction_summary_all.csv: 1080 rows × 9 columns
 - project_level_degradation_summary.csv: 63 rows × 33 columns
 - project_level_noise_technique_summary.csv: 63 rows × 23 columns
 - project_run_metrics_all.csv: 1890 rows × 11 columns
 - project_run_metrics_all.parquet

RuntimeError: Could not map all reference-schema columns.
Package file: condition_summary.csv
Unresolved columns: ['ConditionOrder', 'ConditionKey', 'RawRowsFlipped', 'RetainedLabelChanges', 'TrainingFailures', 'TrainingPasses', 'ConditionSeconds', 'Status']
Generated columns: ['ConditionID', 'Project', 'NoisePercentRequested', 'RepetitionSeed', 'TrainingExecutionRows', 'NumberFlipped', 'RealisedNoisePercent', 'PassToFailure', 'FailureToPass', 'ModelTrainingRows', 'ModelTrainingLabelChanges', 'ModelFits', 'RankingRows', 'BuildMetricRows', 'ProjectRunRows', 'EvaluationRows', 'EvaluationBuilds', 'EvaluationImmutable', 'ElapsedSeconds', 'NoisePercent', 'TechniqueCount', 'MeanAPFD_AcrossTechniques', 'MeanAPFDc_AcrossTechniques', 'MaximumAPFD', 'MaximumAPFDc', 'MinimumAPFD', 'MinimumAPFDc', 'BestAPFDTechnique', 'BestAPFDcTechnique', 'ProjectSlug', 'ProjectNumber', 'ProjectName', 'Repository', 'ProjectID', 'Noise', 'NoiseLevel', 'NoisePct', 'NoisePercentage', 'RequestedNoisePercent', 'Seed',

In [ ]:
noise_injection_summary = (
    condition_summary
    .groupby(
        "NoisePercent",
        as_index=False,
    )
    .agg(
        SeedCount=(
            "RepetitionSeed",
            "nunique",
        ),

        TrainingExecutionRowsPerSeed=(
            "TrainingExecutionRows",
            "first",
        ),

        MeanPassToFailure=(
            "PassToFailure",
            "mean",
        ),

        MeanFailureToPass=(
            "FailureToToPass",
            "mean",
        ),

        MeanRawRowsFlipped=(
            "RawRowsFlipped",
            "mean",
        ),

        SDRawRowsFlipped=(
            "RawRowsFlipped",
            "std",
        ),

        MinRawRowsFlipped=(
            "RawRowsFlipped",
            "min",
        ),

        MaxRawRowsFlipped=(
            "RawRowsFlipped",
            "max",
        ),

        TotalRawRowsFlipped=(
            "RawRowsFlipped",
            "sum",
        ),

        MeanRetainedLabelChanges=(
            "RetainedLabelChanges",
            "mean",
        ),

        SDRetainedLabelChanges=(
            "RetainedLabelChanges",
            "std",
        ),

        MinRetainedLabelChanges=(
            "RetainedLabelChanges",
            "min",
        ),

        MaxRetainedLabelChanges=(
            "RetainedLabelChanges",
            "max",
        ),

        TotalRetainedLabelChanges=(
            "RetainedLabelChanges",
            "sum",
        ),

        MeanTrainingFailures=(
            "TrainingFailures",
            "mean",
        ),

        SDTrainingFailures=(
            "TrainingFailures",
            "std",
        ),

        MeanTrainingPasses=(
            "TrainingPasses",
            "mean",
        ),

        SDTrainingPasses=(
            "TrainingPasses",
            "std",
        ),

        MeanConditionSeconds=(
            "ConditionSeconds",
            "mean",
        ),

        SDConditionSeconds=(
            "ConditionSeconds",
            "std",
        ),
    )
)

=== PROJECT 8 STEP 11B V2: GENERATE FINAL 72-FILE RESULT PACKAGE ===

Input validation:
Step 10 status: PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED
Raw files: 2160
Raw bytes: 179766494
Raw root SHA-256: 19ae21c5d8524c358796e28f23577fbf03bfd31f7d4b530372c6b4d8a492887c
Project 8 registry rows: 0

Reference package blueprint:
Directory: /content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_30_seed_final
Inventory rows: 72
Top-level files: 9
Technique-noise files: 63

Reference top-level schemas:

build_metrics_all.parquet
Rows: 98280
Columns: ['Project', 'NoisePercent', 'RepetitionSeed', 'Technique', 'Build', 'BuildOrder', 'NumberOfTests', 'NumberOfFailures', 'APFD', 'APFDc']

condition_summary.csv
Rows: 270
Columns: ['Project', 'ProjectSlug', 'ConditionOrder', 'ConditionKey', 'NoisePercent', 'RepetitionSeed', 'TrainingExecutionRows', 'RawRowsFlipped', 'RealisedNoisePercent', 'PassToFailure', 'FailureToPass', 'RetainedLabelChanges', 'TrainingFailures', 

RuntimeError: Could not map all reference-schema columns.
Package file: noise_injection_summary.csv
Unresolved columns: ['TrainingExecutionRowsPerSeed', 'MeanPassToFailure', 'MeanFailureToPass']

In [ ]:
# ============================================================
# PROJECT 8 — STEP 11B V3
# GENERATE AND VALIDATE FINAL 72-FILE RESULT PACKAGE
#
# PROJECT: optimatika@ojAlgo
#
# This replacement uses explicit schemas for every package
# file. It does not use the failed Step 11A mapping.
#
# Creates:
#   9 top-level result files
#   63 technique × noise result files
#   72 final package files
#
# It does NOT:
# - rerun any model
# - modify raw Project 8 experiment results
# - modify Projects 1–7
# - modify the completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import re
import shutil

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. PROJECT CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 8
PROJECT_NAME = "optimatika@ojAlgo"
PROJECT_SLUG = "optimatika__ojAlgo"
PROJECT_SHORT_NAME = "ojalgo"

REFERENCE_PROJECT_NAME = "CompEvol@beast2"

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

SEEDS = list(
    range(1, 31)
)

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)

TECHNIQUE_SLUG_LOOKUP = {
    "randomforest":
        "RandomForest",

    "xgboost":
        "XGBoost",

    "lightgbm":
        "LightGBM",

    "naivebayes":
        "NaiveBayes",

    "random":
        "Random",

    "latestfail":
        "LatestFail",

    "qtfavg":
        "QTF-Avg",
}

EXPECTED_CONDITIONS = 270
EXPECTED_ML_FITS = 1080
EXPECTED_PROJECT_RUN_ROWS = 1890
EXPECTED_BUILD_METRIC_ROWS = 30240
EXPECTED_PREDICTION_SUMMARIES = 1080

EXPECTED_PACKAGE_FILES = 72
EXPECTED_TOP_LEVEL_FILES = 9
EXPECTED_TECHNIQUE_NOISE_FILES = 63

EXPECTED_RAW_FILES = 2160
EXPECTED_RAW_BYTES = 179766494

EXPECTED_RAW_ROOT_SHA256 = (
    "19ae21c5d8524c358796e28f23577fbf03bfd31f7d4b530372c6b4d8a492887c"
)

EXPECTED_STEP10_STATUS = (
    "PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED"
)

# Fallback only. The code first derives the critical value
# from Project 7's frozen summary to preserve consistency.
FALLBACK_T_CRITICAL_95 = (
    2.045229642132703
)

GENERATED_AT_UTC = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

AGGREGATED_ROOT = (
    RESULTS_DIR
    / "Aggregated"
)

PROJECT_DIR = (
    AGGREGATED_ROOT
    / PROJECT_SLUG
)

RAW_RESULTS_ROOT = (
    RESULTS_DIR
    / "Raw"
    / PROJECT_SLUG
)

RAW_AUDIT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_raw_audit"
)

STEP10_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step10_status.json"
)

REFERENCE_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_07_selection_checkpoint.json"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)


# Audited Project 8 inputs

PROJECT_RUN_INPUT = (
    RAW_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_all_project_run_metrics.csv"
)

BUILD_METRICS_INPUT = (
    RAW_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_all_build_metrics.csv.gz"
)

NOISE_SUMMARY_INPUT = (
    RAW_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_all_noise_summary.csv"
)

FIT_TIMES_INPUT = (
    RAW_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_all_fit_times.csv"
)


# Final package

FINAL_PACKAGE_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_30_seed_final"
)

STAGING_PACKAGE_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_30_seed_final__staging"
)


# Package audit

FINAL_PACKAGE_AUDIT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_audit"
)

FINAL_PACKAGE_INVENTORY_PATH = (
    FINAL_PACKAGE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_file_inventory_sha256.csv"
)

FINAL_PACKAGE_VALIDATION_PATH = (
    FINAL_PACKAGE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_validation.csv"
)

FINAL_PACKAGE_GENERATION_MAP_PATH = (
    FINAL_PACKAGE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_generation_map.csv"
)

FINAL_PACKAGE_REPORT_PATH = (
    FINAL_PACKAGE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_report.json"
)

STEP11B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step11b_status.json"
)


print("=" * 100)
print("=== PROJECT 8 STEP 11B V3: GENERATE FINAL 72-FILE RESULT PACKAGE ===")
print("=" * 100)


# ------------------------------------------------------------
# 3. HELPERS
# ------------------------------------------------------------

def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def json_safe(value):

    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:

        if pd.isna(value):
            return None

    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(path)


def recursive_key_lookup(
    value,
    requested_key,
):
    requested_key = (
        str(requested_key)
        .strip()
        .lower()
    )

    if isinstance(value, dict):

        for key, child in value.items():

            if (
                str(key)
                .strip()
                .lower()
                == requested_key
            ):
                return child

        for child in value.values():

            result = recursive_key_lookup(
                child,
                requested_key,
            )

            if result is not None:
                return result

    elif isinstance(value, list):

        for child in value:

            result = recursive_key_lookup(
                child,
                requested_key,
            )

            if result is not None:
                return result

    return None


def normalise_column_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


def package_relative_path(
    value,
    package_directory,
):
    text = (
        str(value)
        .replace("\\", "/")
        .strip()
    )

    text = re.sub(
        r"/+",
        "/",
        text,
    )

    package_text = (
        str(package_directory)
        .replace("\\", "/")
        .rstrip("/")
    )

    if text.startswith(
        package_text + "/"
    ):

        return text[
            len(package_text) + 1:
        ]

    package_name = Path(
        package_directory
    ).name

    marker = (
        package_name
        + "/"
    )

    if marker in text:

        return text.split(
            marker,
            1,
        )[1]

    return text.lstrip("./")


def read_table(
    path,
    **kwargs,
):
    path = Path(path)

    suffixes = "".join(
        path.suffixes
    ).lower()

    if suffixes.endswith(
        ".parquet"
    ):

        return pd.read_parquet(
            path,
            **kwargs,
        )

    if (
        suffixes.endswith(".csv")
        or suffixes.endswith(".csv.gz")
    ):

        return pd.read_csv(
            path,
            low_memory=False,
            **kwargs,
        )

    raise RuntimeError(
        "Unsupported table format:\n"
        f"{path}"
    )


def write_table(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    suffixes = "".join(
        path.suffixes
    ).lower()

    if suffixes.endswith(
        ".parquet"
    ):

        temporary_path = path.with_name(
            path.name + ".tmp.parquet"
        )

        dataframe.to_parquet(
            temporary_path,
            index=False,
        )

        temporary_path.replace(path)

        return

    if suffixes.endswith(
        ".csv.gz"
    ):

        temporary_path = path.with_name(
            path.name + ".tmp.gz"
        )

        dataframe.to_csv(
            temporary_path,
            index=False,
            compression="gzip",
        )

        temporary_path.replace(path)

        return

    if suffixes.endswith(
        ".csv"
    ):

        temporary_path = path.with_name(
            path.name + ".tmp"
        )

        dataframe.to_csv(
            temporary_path,
            index=False,
        )

        temporary_path.replace(path)

        return

    raise RuntimeError(
        "Unsupported output format:\n"
        f"{path}"
    )


def root_inventory_hash(
    inventory,
):
    digest = hashlib.sha256()

    ordered = inventory.sort_values(
        "RelativePath",
        kind="mergesort",
    )

    for row in ordered.itertuples(
        index=False
    ):

        digest.update(
            row.RelativePath.encode(
                "utf-8"
            )
        )

        digest.update(b"\0")

        digest.update(
            str(
                int(row.SizeBytes)
            ).encode(
                "utf-8"
            )
        )

        digest.update(b"\0")

        digest.update(
            bytes.fromhex(
                row.SHA256
            )
        )

        digest.update(b"\n")

    return digest.hexdigest()


def find_column(
    dataframe,
    candidate_names,
):
    lookup = {
        normalise_column_name(column):
            column
        for column in dataframe.columns
    }

    for candidate_name in candidate_names:

        normalised_candidate = (
            normalise_column_name(
                candidate_name
            )
        )

        if normalised_candidate in lookup:

            return lookup[
                normalised_candidate
            ]

    raise RuntimeError(
        "Required column could not be found.\n"
        f"Candidates: {candidate_names}\n"
        f"Available: {list(dataframe.columns)}"
    )


def condition_key(
    noise_percent,
    repetition_seed,
):
    return (
        f"noise_{int(noise_percent):03d}"
        f"__seed_{int(repetition_seed):03d}"
    )


def cast_like_reference(
    generated_dataframe,
    reference_dataframe,
    relative_path,
):
    generated_dataframe = (
        generated_dataframe.copy()
    )

    if list(
        generated_dataframe.columns
    ) != list(
        reference_dataframe.columns
    ):

        raise RuntimeError(
            "Generated schema differs from the reference.\n"
            f"File: {relative_path}\n"
            f"Reference: {list(reference_dataframe.columns)}\n"
            f"Generated: {list(generated_dataframe.columns)}"
        )

    for column in generated_dataframe.columns:

        reference_dtype = (
            reference_dataframe[
                column
            ].dtype
        )

        try:

            if pd.api.types.is_integer_dtype(
                reference_dtype
            ):

                numeric = pd.to_numeric(
                    generated_dataframe[
                        column
                    ],
                    errors="raise",
                )

                if numeric.isna().any():

                    generated_dataframe[
                        column
                    ] = numeric.astype(
                        "Int64"
                    )

                else:

                    generated_dataframe[
                        column
                    ] = numeric.astype(
                        reference_dtype
                    )

            elif pd.api.types.is_float_dtype(
                reference_dtype
            ):

                generated_dataframe[
                    column
                ] = pd.to_numeric(
                    generated_dataframe[
                        column
                    ],
                    errors="coerce",
                ).astype(float)

            elif pd.api.types.is_bool_dtype(
                reference_dtype
            ):

                generated_dataframe[
                    column
                ] = generated_dataframe[
                    column
                ].astype(bool)

        except Exception as error:

            raise RuntimeError(
                "Failed to cast a generated column.\n"
                f"File: {relative_path}\n"
                f"Column: {column}\n"
                f"Reference dtype: {reference_dtype}\n"
                f"Error: {error}"
            )

    return generated_dataframe


def normalise_technique_slug(
    slug,
):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(slug).lower(),
    )


# ------------------------------------------------------------
# 4. VALIDATE PROJECT STATE
# ------------------------------------------------------------

required_paths = [
    STEP10_STATUS_PATH,
    REFERENCE_CHECKPOINT_PATH,
    REGISTRY_PATH,
    PROJECT_RUN_INPUT,
    BUILD_METRICS_INPUT,
    NOISE_SUMMARY_INPUT,
    FIT_TIMES_INPUT,
    RAW_RESULTS_ROOT,
]


missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]


if missing_paths:

    raise FileNotFoundError(
        "Required Step 11B V3 inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


step10_status = json.loads(
    STEP10_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    step10_status.get("Status")
    != EXPECTED_STEP10_STATUS
):

    raise AssertionError(
        "Project 8 Step 10 has not passed.\n"
        f"Detected: "
        f"{step10_status.get('Status')}"
    )


if (
    int(
        step10_status.get(
            "RawFiles",
            -1,
        )
    )
    != EXPECTED_RAW_FILES
):

    raise AssertionError(
        "Project 8 raw-file count differs."
    )


if (
    int(
        step10_status.get(
            "RawBytes",
            -1,
        )
    )
    != EXPECTED_RAW_BYTES
):

    raise AssertionError(
        "Project 8 raw-byte count differs."
    )


if (
    step10_status.get(
        "RawRootSHA256"
    )
    != EXPECTED_RAW_ROOT_SHA256
):

    raise AssertionError(
        "Project 8 raw-root SHA-256 differs."
    )


registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
)


registry_project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="coerce",
)


project8_registry_rows = registry[
    registry_project_numbers.eq(
        PROJECT_NUMBER
    )
]


if len(
    project8_registry_rows
) != 0:

    raise AssertionError(
        "Project 8 already exists in the completion registry."
    )


print("\nInput validation:")

print(
    "Step 10 status:",
    step10_status["Status"],
)

print(
    "Raw files:",
    step10_status["RawFiles"],
)

print(
    "Raw bytes:",
    step10_status["RawBytes"],
)

print(
    "Raw root SHA-256:",
    step10_status["RawRootSHA256"],
)

print(
    "Project 8 registry rows:",
    len(
        project8_registry_rows
    ),
)


# ------------------------------------------------------------
# 5. LOAD THE EXACT REFERENCE PACKAGE BLUEPRINT
# ------------------------------------------------------------

reference_checkpoint = json.loads(
    REFERENCE_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)


reference_package_directory_value = (
    recursive_key_lookup(
        reference_checkpoint,
        "FinalPackageDirectory",
    )
)


reference_inventory_value = (
    recursive_key_lookup(
        reference_checkpoint,
        "FinalPackageInventory",
    )
)


if (
    reference_package_directory_value is None
    or reference_inventory_value is None
):

    raise RuntimeError(
        "The reference checkpoint does not contain the "
        "final-package paths."
    )


REFERENCE_PACKAGE_DIR = Path(
    reference_package_directory_value
)

REFERENCE_INVENTORY_PATH = Path(
    reference_inventory_value
)


if not REFERENCE_PACKAGE_DIR.exists():

    raise FileNotFoundError(
        "Reference package directory is missing:\n"
        f"{REFERENCE_PACKAGE_DIR}"
    )


if not REFERENCE_INVENTORY_PATH.exists():

    raise FileNotFoundError(
        "Reference package inventory is missing:\n"
        f"{REFERENCE_INVENTORY_PATH}"
    )


reference_inventory = pd.read_csv(
    REFERENCE_INVENTORY_PATH,
    low_memory=False,
)


relative_path_column = find_column(
    reference_inventory,
    [
        "RelativePath",
        "PackageRelativePath",
        "Path",
        "FilePath",
    ],
)


reference_relative_paths = (
    reference_inventory[
        relative_path_column
    ]
    .astype(str)
    .map(
        lambda value:
            package_relative_path(
                value,
                REFERENCE_PACKAGE_DIR,
            )
    )
    .tolist()
)


if len(
    reference_relative_paths
) != EXPECTED_PACKAGE_FILES:

    raise AssertionError(
        "Reference inventory does not contain 72 rows."
    )


if len(
    set(reference_relative_paths)
) != EXPECTED_PACKAGE_FILES:

    raise AssertionError(
        "Reference package paths are not unique."
    )


missing_reference_files = [
    relative_path
    for relative_path
    in reference_relative_paths
    if not (
        REFERENCE_PACKAGE_DIR
        / relative_path
    ).exists()
]


if missing_reference_files:

    raise FileNotFoundError(
        "Reference package files are missing:\n"
        + "\n".join(
            missing_reference_files
        )
    )


technique_noise_paths = [
    relative_path
    for relative_path
    in reference_relative_paths
    if (
        relative_path
        .replace("\\", "/")
        .startswith(
            "technique_noise/"
        )
    )
]


top_level_paths = [
    relative_path
    for relative_path
    in reference_relative_paths
    if relative_path
    not in technique_noise_paths
]


if len(
    technique_noise_paths
) != EXPECTED_TECHNIQUE_NOISE_FILES:

    raise AssertionError(
        "Reference package technique-noise file count differs."
    )


if len(
    top_level_paths
) != EXPECTED_TOP_LEVEL_FILES:

    raise AssertionError(
        "Reference package top-level file count differs."
    )


REFERENCE_SCHEMAS = {
    Path(relative_path).name:
        list(
            read_table(
                REFERENCE_PACKAGE_DIR
                / relative_path
            ).columns
        )

    for relative_path in top_level_paths
}


print("\nReference package blueprint:")

print(
    "Directory:",
    REFERENCE_PACKAGE_DIR,
)

print(
    "Inventory rows:",
    len(
        reference_relative_paths
    ),
)

print(
    "Top-level files:",
    len(
        top_level_paths
    ),
)

print(
    "Technique-noise files:",
    len(
        technique_noise_paths
    ),
)


# ------------------------------------------------------------
# 6. DERIVE THE FROZEN CONFIDENCE-INTERVAL CONVENTION
# ------------------------------------------------------------

reference_summary_path = (
    REFERENCE_PACKAGE_DIR
    / "project_level_noise_technique_summary.csv"
)


reference_summary = pd.read_csv(
    reference_summary_path,
    low_memory=False,
)


critical_candidates = []


for (
    mean_column,
    se_column,
    upper_column,
) in [
    (
        "MeanAPFD",
        "SE_APFD",
        "CI95UpperAPFD",
    ),
    (
        "MeanAPFDc",
        "SE_APFDc",
        "CI95UpperAPFDc",
    ),
]:

    mean_values = pd.to_numeric(
        reference_summary[
            mean_column
        ],
        errors="coerce",
    )

    se_values = pd.to_numeric(
        reference_summary[
            se_column
        ],
        errors="coerce",
    )

    upper_values = pd.to_numeric(
        reference_summary[
            upper_column
        ],
        errors="coerce",
    )

    valid = (
        np.isfinite(mean_values)
        & np.isfinite(se_values)
        & np.isfinite(upper_values)
        & se_values.gt(0)
    )

    current_candidates = (
        (
            upper_values[
                valid
            ]
            - mean_values[
                valid
            ]
        )
        / se_values[
            valid
        ]
    )

    critical_candidates.extend(
        current_candidates.tolist()
    )


if critical_candidates:

    CI_CRITICAL_95 = float(
        np.median(
            critical_candidates
        )
    )

else:

    CI_CRITICAL_95 = (
        FALLBACK_T_CRITICAL_95
    )


if not (
    1.5
    <= CI_CRITICAL_95
    <= 3.0
):

    raise AssertionError(
        "The reference confidence-interval critical value "
        "could not be derived safely."
    )


reference_degradation = pd.read_csv(
    REFERENCE_PACKAGE_DIR
    / "project_level_degradation_summary.csv",
    low_memory=False,
)


clean_reference_retention = pd.to_numeric(
    reference_degradation.loc[
        pd.to_numeric(
            reference_degradation[
                "NoisePercent"
            ],
            errors="coerce",
        ).eq(0),
        "RetentionAPFD",
    ],
    errors="coerce",
).dropna()


if (
    not clean_reference_retention.empty
    and clean_reference_retention.median()
    > 10
):

    RETENTION_SCALE = 100.0

else:

    RETENTION_SCALE = 1.0


print("\nFrozen summary conventions:")

print(
    "CI critical value:",
    CI_CRITICAL_95,
)

print(
    "Retention scale:",
    RETENTION_SCALE,
)


# ------------------------------------------------------------
# 7. LOAD AUDITED PROJECT 8 RESULTS
# ------------------------------------------------------------

project_run_source = pd.read_csv(
    PROJECT_RUN_INPUT,
    low_memory=False,
)

build_metrics_source = pd.read_csv(
    BUILD_METRICS_INPUT,
    low_memory=False,
)

noise_summary_source = pd.read_csv(
    NOISE_SUMMARY_INPUT,
    low_memory=False,
)

fit_times_source = pd.read_csv(
    FIT_TIMES_INPUT,
    low_memory=False,
)


for dataframe in [
    project_run_source,
    build_metrics_source,
    noise_summary_source,
    fit_times_source,
]:

    dataframe[
        "Project"
    ] = PROJECT_NAME


for column in [
    "NoisePercent",
    "RepetitionSeed",
    "MeanAPFD",
    "MeanAPFDc",
    "SD_APFD",
    "SD_APFDc",
    "EvaluatedBuilds",
    "TotalRankedTests",
    "TotalFailures",
]:

    if column in project_run_source.columns:

        project_run_source[
            column
        ] = pd.to_numeric(
            project_run_source[
                column
            ],
            errors="raise",
        )


for column in [
    "NoisePercent",
    "RepetitionSeed",
    "Build",
    "BuildOrder",
    "NumberOfTests",
    "NumberOfFailures",
    "APFD",
    "APFDc",
]:

    if column in build_metrics_source.columns:

        build_metrics_source[
            column
        ] = pd.to_numeric(
            build_metrics_source[
                column
            ],
            errors="raise",
        )


for column in [
    "NoisePercent",
    "RepetitionSeed",
    "FitSeconds",
    "TrainingRows",
    "TrainingFailures",
    "TrainingPasses",
    "ActiveFeatures",
]:

    if column in fit_times_source.columns:

        fit_times_source[
            column
        ] = pd.to_numeric(
            fit_times_source[
                column
            ],
            errors="raise",
        )


noise_percent_source_column = find_column(
    noise_summary_source,
    [
        "NoisePercent",
        "NoisePercentRequested",
    ],
)


noise_seed_source_column = find_column(
    noise_summary_source,
    [
        "RepetitionSeed",
        "Seed",
    ],
)


noise_summary_source[
    "NoisePercent"
] = pd.to_numeric(
    noise_summary_source[
        noise_percent_source_column
    ],
    errors="raise",
).astype(int)


noise_summary_source[
    "RepetitionSeed"
] = pd.to_numeric(
    noise_summary_source[
        noise_seed_source_column
    ],
    errors="raise",
).astype(int)


for dataframe in [
    project_run_source,
    build_metrics_source,
    fit_times_source,
]:

    dataframe[
        "NoisePercent"
    ] = dataframe[
        "NoisePercent"
    ].astype(int)

    dataframe[
        "RepetitionSeed"
    ] = dataframe[
        "RepetitionSeed"
    ].astype(int)


if len(
    project_run_source
) != EXPECTED_PROJECT_RUN_ROWS:

    raise AssertionError(
        "Project-run row count differs."
    )


if len(
    build_metrics_source
) != EXPECTED_BUILD_METRIC_ROWS:

    raise AssertionError(
        "Build-metric row count differs."
    )


if len(
    noise_summary_source
) != EXPECTED_CONDITIONS:

    raise AssertionError(
        "Noise-summary row count differs."
    )


if len(
    fit_times_source
) != EXPECTED_ML_FITS:

    raise AssertionError(
        "Fit-time row count differs."
    )


if set(
    project_run_source[
        "Technique"
    ].unique()
) != set(
    TECHNIQUES
):

    raise AssertionError(
        "Project-run technique set differs."
    )


project_metric_values = (
    project_run_source[
        [
            "MeanAPFD",
            "MeanAPFDc",
        ]
    ]
    .to_numpy(
        dtype=float
    )
)


if (
    not np.isfinite(
        project_metric_values
    ).all()
    or (
        (project_metric_values < 0)
        | (project_metric_values > 1)
    ).any()
):

    raise AssertionError(
        "Project-run APFD/APFDc values are invalid."
    )


print("\nAudited Project 8 inputs:")

print(
    "Project-run rows:",
    len(
        project_run_source
    ),
)

print(
    "Build-metric rows:",
    len(
        build_metrics_source
    ),
)

print(
    "Condition rows:",
    len(
        noise_summary_source
    ),
)

print(
    "ML fit rows:",
    len(
        fit_times_source
    ),
)


# ------------------------------------------------------------
# 8. EXACT PROJECT-RUN METRICS TABLE
# ------------------------------------------------------------

PROJECT_RUN_COLUMNS = [
    "Project",
    "NoisePercent",
    "RepetitionSeed",
    "Technique",
    "MeanAPFD",
    "MeanAPFDc",
    "SD_APFD",
    "SD_APFDc",
    "EvaluatedBuilds",
    "TotalRankedTests",
    "TotalFailures",
]


project_run_metrics = (
    project_run_source[
        PROJECT_RUN_COLUMNS
    ]
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9. EXACT BUILD METRICS TABLE
# ------------------------------------------------------------

BUILD_METRIC_COLUMNS = [
    "Project",
    "NoisePercent",
    "RepetitionSeed",
    "Technique",
    "Build",
    "BuildOrder",
    "NumberOfTests",
    "NumberOfFailures",
    "APFD",
    "APFDc",
]


build_metrics = (
    build_metrics_source[
        BUILD_METRIC_COLUMNS
    ]
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "BuildOrder",
            "Build",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 10. EXACT FIT-TIME TABLE
# ------------------------------------------------------------

FIT_TIME_COLUMNS = [
    "Project",
    "NoisePercent",
    "RepetitionSeed",
    "Technique",
    "FitSeconds",
    "TrainingRows",
    "TrainingFailures",
    "TrainingPasses",
    "ActiveFeatures",
]


fit_times = (
    fit_times_source[
        FIT_TIME_COLUMNS
    ]
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 11. EXACT CONDITION SUMMARY
# ------------------------------------------------------------

training_rows_column = find_column(
    noise_summary_source,
    [
        "TrainingExecutionRows",
        "RawTrainingRows",
    ],
)


raw_flips_column = find_column(
    noise_summary_source,
    [
        "NumberFlipped",
        "FlippedRawRows",
        "RawRowsFlipped",
    ],
)


realised_noise_column = find_column(
    noise_summary_source,
    [
        "RealisedNoisePercent",
        "RealizedNoisePercent",
    ],
)


pass_to_failure_column = find_column(
    noise_summary_source,
    [
        "PassToFailure",
    ],
)


failure_to_pass_column = find_column(
    noise_summary_source,
    [
        "FailureToPass",
    ],
)


retained_changes_column = find_column(
    noise_summary_source,
    [
        "ModelTrainingLabelChanges",
        "RetainedLabelChanges",
        "LabelChanges",
    ],
)


condition_seconds_column = find_column(
    noise_summary_source,
    [
        "ElapsedSeconds",
        "ConditionSeconds",
    ],
)


condition_base = pd.DataFrame({
    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "NoisePercent":
        noise_summary_source[
            "NoisePercent"
        ].astype(int),

    "RepetitionSeed":
        noise_summary_source[
            "RepetitionSeed"
        ].astype(int),

    "TrainingExecutionRows":
        pd.to_numeric(
            noise_summary_source[
                training_rows_column
            ],
            errors="raise",
        ).astype(int),

    "RawRowsFlipped":
        pd.to_numeric(
            noise_summary_source[
                raw_flips_column
            ],
            errors="raise",
        ).astype(int),

    "RealisedNoisePercent":
        pd.to_numeric(
            noise_summary_source[
                realised_noise_column
            ],
            errors="raise",
        ).astype(float),

    "PassToFailure":
        pd.to_numeric(
            noise_summary_source[
                pass_to_failure_column
            ],
            errors="raise",
        ).astype(int),

    "FailureToPass":
        pd.to_numeric(
            noise_summary_source[
                failure_to_pass_column
            ],
            errors="raise",
        ).astype(int),

    "RetainedLabelChanges":
        pd.to_numeric(
            noise_summary_source[
                retained_changes_column
            ],
            errors="raise",
        ).astype(int),

    "ConditionSeconds":
        pd.to_numeric(
            noise_summary_source[
                condition_seconds_column
            ],
            errors="raise",
        ).astype(float),
})


condition_class_counts = (
    fit_times
    .groupby(
        [
            "NoisePercent",
            "RepetitionSeed",
        ],
        as_index=False,
    )
    .agg(
        TrainingFailures=(
            "TrainingFailures",
            "first",
        ),

        TrainingPasses=(
            "TrainingPasses",
            "first",
        ),

        FailureCountVariants=(
            "TrainingFailures",
            "nunique",
        ),

        PassCountVariants=(
            "TrainingPasses",
            "nunique",
        ),
    )
)


if (
    condition_class_counts[
        "FailureCountVariants"
    ].ne(1).any()
    or
    condition_class_counts[
        "PassCountVariants"
    ].ne(1).any()
):

    raise AssertionError(
        "ML models within the same condition have different "
        "training class counts."
    )


condition_class_counts = (
    condition_class_counts.drop(
        columns=[
            "FailureCountVariants",
            "PassCountVariants",
        ]
    )
)


condition_summary = (
    condition_base
    .merge(
        condition_class_counts,
        on=[
            "NoisePercent",
            "RepetitionSeed",
        ],
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        [
            "RepetitionSeed",
            "NoisePercent",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


condition_summary[
    "ConditionOrder"
] = np.arange(
    1,
    len(condition_summary) + 1,
)


condition_summary[
    "ConditionKey"
] = [
    condition_key(
        noise_percent,
        repetition_seed,
    )
    for noise_percent, repetition_seed
    in zip(
        condition_summary[
            "NoisePercent"
        ],
        condition_summary[
            "RepetitionSeed"
        ],
    )
]


condition_summary[
    "Status"
] = "COMPLETE"


CONDITION_SUMMARY_COLUMNS = [
    "Project",
    "ProjectSlug",
    "ConditionOrder",
    "ConditionKey",
    "NoisePercent",
    "RepetitionSeed",
    "TrainingExecutionRows",
    "RawRowsFlipped",
    "RealisedNoisePercent",
    "PassToFailure",
    "FailureToPass",
    "RetainedLabelChanges",
    "TrainingFailures",
    "TrainingPasses",
    "ConditionSeconds",
    "Status",
]


condition_summary = condition_summary[
    CONDITION_SUMMARY_COLUMNS
]


# ------------------------------------------------------------
# 12. EXACT NOISE-INJECTION SUMMARY
# ------------------------------------------------------------

noise_validation = (
    condition_summary
    .groupby(
        "NoisePercent"
    )
    .agg(
        TrainingRowsVariants=(
            "TrainingExecutionRows",
            "nunique",
        ),

        SeedCount=(
            "RepetitionSeed",
            "nunique",
        ),
    )
    .reset_index()
)


if (
    noise_validation[
        "TrainingRowsVariants"
    ].ne(1).any()
):

    raise AssertionError(
        "TrainingExecutionRows differs within a noise level."
    )


if (
    noise_validation[
        "SeedCount"
    ].ne(30).any()
):

    raise AssertionError(
        "A noise level does not contain all 30 seeds."
    )


noise_injection_summary = (
    condition_summary
    .groupby(
        "NoisePercent",
        as_index=False,
    )
    .agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        TrainingExecutionRowsPerSeed=(
            "TrainingExecutionRows",
            "first",
        ),

        MeanRawRowsFlipped=(
            "RawRowsFlipped",
            "mean",
        ),

        SDRawRowsFlipped=(
            "RawRowsFlipped",
            "std",
        ),

        MinimumRawRowsFlipped=(
            "RawRowsFlipped",
            "min",
        ),

        MaximumRawRowsFlipped=(
            "RawRowsFlipped",
            "max",
        ),

        MeanRealisedNoisePercent=(
            "RealisedNoisePercent",
            "mean",
        ),

        SDRealisedNoisePercent=(
            "RealisedNoisePercent",
            "std",
        ),

        MeanPassToFailure=(
            "PassToFailure",
            "mean",
        ),

        MeanFailureToPass=(
            "FailureToPass",
            "mean",
        ),

        MeanRetainedLabelChanges=(
            "RetainedLabelChanges",
            "mean",
        ),

        SDRetainedLabelChanges=(
            "RetainedLabelChanges",
            "std",
        ),

        MeanTrainingFailures=(
            "TrainingFailures",
            "mean",
        ),

        MeanTrainingPasses=(
            "TrainingPasses",
            "mean",
        ),

        MeanConditionSeconds=(
            "ConditionSeconds",
            "mean",
        ),

        SDConditionSeconds=(
            "ConditionSeconds",
            "std",
        ),
    )
)


NOISE_INJECTION_COLUMNS = [
    "NoisePercent",
    "Seeds",
    "TrainingExecutionRowsPerSeed",
    "MeanRawRowsFlipped",
    "SDRawRowsFlipped",
    "MinimumRawRowsFlipped",
    "MaximumRawRowsFlipped",
    "MeanRealisedNoisePercent",
    "SDRealisedNoisePercent",
    "MeanPassToFailure",
    "MeanFailureToPass",
    "MeanRetainedLabelChanges",
    "SDRetainedLabelChanges",
    "MeanTrainingFailures",
    "MeanTrainingPasses",
    "MeanConditionSeconds",
    "SDConditionSeconds",
]


noise_injection_summary = (
    noise_injection_summary[
        NOISE_INJECTION_COLUMNS
    ]
    .sort_values(
        "NoisePercent",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 13. EXACT PROJECT-LEVEL SUMMARY
# ------------------------------------------------------------

project_level_summary = (
    project_run_metrics
    .groupby(
        [
            "Project",
            "NoisePercent",
            "Technique",
        ],
        as_index=False,
    )
    .agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        MeanAPFD=(
            "MeanAPFD",
            "mean",
        ),

        SD_APFD_AcrossSeeds=(
            "MeanAPFD",
            "std",
        ),

        MedianAPFD=(
            "MeanAPFD",
            "median",
        ),

        MinimumAPFD=(
            "MeanAPFD",
            "min",
        ),

        MaximumAPFD=(
            "MeanAPFD",
            "max",
        ),

        MeanAPFDc=(
            "MeanAPFDc",
            "mean",
        ),

        SD_APFDc_AcrossSeeds=(
            "MeanAPFDc",
            "std",
        ),

        MedianAPFDc=(
            "MeanAPFDc",
            "median",
        ),

        MinimumAPFDc=(
            "MeanAPFDc",
            "min",
        ),

        MaximumAPFDc=(
            "MeanAPFDc",
            "max",
        ),

        EvaluatedBuildsPerSeed=(
            "EvaluatedBuilds",
            "mean",
        ),

        TotalRankedTestsPerSeed=(
            "TotalRankedTests",
            "mean",
        ),

        TotalFailuresPerSeed=(
            "TotalFailures",
            "mean",
        ),
    )
)


project_level_summary[
    "SE_APFD"
] = (
    project_level_summary[
        "SD_APFD_AcrossSeeds"
    ]
    / np.sqrt(
        project_level_summary[
            "Seeds"
        ]
    )
)


project_level_summary[
    "CI95LowerAPFD"
] = (
    project_level_summary[
        "MeanAPFD"
    ]
    - CI_CRITICAL_95
    * project_level_summary[
        "SE_APFD"
    ]
)


project_level_summary[
    "CI95UpperAPFD"
] = (
    project_level_summary[
        "MeanAPFD"
    ]
    + CI_CRITICAL_95
    * project_level_summary[
        "SE_APFD"
    ]
)


project_level_summary[
    "SE_APFDc"
] = (
    project_level_summary[
        "SD_APFDc_AcrossSeeds"
    ]
    / np.sqrt(
        project_level_summary[
            "Seeds"
        ]
    )
)


project_level_summary[
    "CI95LowerAPFDc"
] = (
    project_level_summary[
        "MeanAPFDc"
    ]
    - CI_CRITICAL_95
    * project_level_summary[
        "SE_APFDc"
    ]
)


project_level_summary[
    "CI95UpperAPFDc"
] = (
    project_level_summary[
        "MeanAPFDc"
    ]
    + CI_CRITICAL_95
    * project_level_summary[
        "SE_APFDc"
    ]
)


PROJECT_LEVEL_SUMMARY_COLUMNS = [
    "Project",
    "NoisePercent",
    "Technique",
    "Seeds",
    "MeanAPFD",
    "SD_APFD_AcrossSeeds",
    "MedianAPFD",
    "MinimumAPFD",
    "MaximumAPFD",
    "MeanAPFDc",
    "SD_APFDc_AcrossSeeds",
    "MedianAPFDc",
    "MinimumAPFDc",
    "MaximumAPFDc",
    "EvaluatedBuildsPerSeed",
    "TotalRankedTestsPerSeed",
    "TotalFailuresPerSeed",
    "SE_APFD",
    "CI95LowerAPFD",
    "CI95UpperAPFD",
    "SE_APFDc",
    "CI95LowerAPFDc",
    "CI95UpperAPFDc",
]


project_level_summary = (
    project_level_summary[
        PROJECT_LEVEL_SUMMARY_COLUMNS
    ]
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 14. EXACT DEGRADATION SUMMARY
# ------------------------------------------------------------

clean_means = (
    project_level_summary[
        project_level_summary[
            "NoisePercent"
        ].eq(0)
    ][
        [
            "Technique",
            "MeanAPFD",
            "MeanAPFDc",
        ]
    ]
    .rename(
        columns={
            "MeanAPFD":
                "CleanProjectMeanAPFD",

            "MeanAPFDc":
                "CleanProjectMeanAPFDc",
        }
    )
)


project_level_degradation = (
    project_level_summary
    .merge(
        clean_means,
        on="Technique",
        how="left",
        validate="many_to_one",
    )
)


project_level_degradation[
    "DeltaAPFD_NoiseMinusClean"
] = (
    project_level_degradation[
        "MeanAPFD"
    ]
    - project_level_degradation[
        "CleanProjectMeanAPFD"
    ]
)


project_level_degradation[
    "DeltaAPFDc_NoiseMinusClean"
] = (
    project_level_degradation[
        "MeanAPFDc"
    ]
    - project_level_degradation[
        "CleanProjectMeanAPFDc"
    ]
)


project_level_degradation[
    "AbsoluteLossAPFD"
] = (
    project_level_degradation[
        "CleanProjectMeanAPFD"
    ]
    - project_level_degradation[
        "MeanAPFD"
    ]
)


project_level_degradation[
    "AbsoluteLossAPFDc"
] = (
    project_level_degradation[
        "CleanProjectMeanAPFDc"
    ]
    - project_level_degradation[
        "MeanAPFDc"
    ]
)


project_level_degradation[
    "PercentageLossAPFD"
] = np.where(
    project_level_degradation[
        "CleanProjectMeanAPFD"
    ].ne(0),

    100.0
    * project_level_degradation[
        "AbsoluteLossAPFD"
    ]
    / project_level_degradation[
        "CleanProjectMeanAPFD"
    ],

    np.nan,
)


project_level_degradation[
    "PercentageLossAPFDc"
] = np.where(
    project_level_degradation[
        "CleanProjectMeanAPFDc"
    ].ne(0),

    100.0
    * project_level_degradation[
        "AbsoluteLossAPFDc"
    ]
    / project_level_degradation[
        "CleanProjectMeanAPFDc"
    ],

    np.nan,
)


project_level_degradation[
    "RetentionAPFD"
] = np.where(
    project_level_degradation[
        "CleanProjectMeanAPFD"
    ].ne(0),

    RETENTION_SCALE
    * project_level_degradation[
        "MeanAPFD"
    ]
    / project_level_degradation[
        "CleanProjectMeanAPFD"
    ],

    np.nan,
)


project_level_degradation[
    "RetentionAPFDc"
] = np.where(
    project_level_degradation[
        "CleanProjectMeanAPFDc"
    ].ne(0),

    RETENTION_SCALE
    * project_level_degradation[
        "MeanAPFDc"
    ]
    / project_level_degradation[
        "CleanProjectMeanAPFDc"
    ],

    np.nan,
)


PROJECT_LEVEL_DEGRADATION_COLUMNS = [
    *PROJECT_LEVEL_SUMMARY_COLUMNS,
    "CleanProjectMeanAPFD",
    "CleanProjectMeanAPFDc",
    "DeltaAPFD_NoiseMinusClean",
    "DeltaAPFDc_NoiseMinusClean",
    "AbsoluteLossAPFD",
    "AbsoluteLossAPFDc",
    "PercentageLossAPFD",
    "PercentageLossAPFDc",
    "RetentionAPFD",
    "RetentionAPFDc",
]


project_level_degradation = (
    project_level_degradation[
        PROJECT_LEVEL_DEGRADATION_COLUMNS
    ]
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 15. RECONSTRUCT EXACT PREDICTION SUMMARY
# ------------------------------------------------------------

print(
    "\nReconstructing 1,080 ML prediction summaries "
    "from raw Project 8 rankings..."
)


prediction_records = []


for seed_position, repetition_seed in enumerate(
    SEEDS,
    start=1,
):

    for noise_percent in NOISE_LEVELS:

        rankings_path = (
            RAW_RESULTS_ROOT
            / f"noise_{noise_percent:03d}"
            / f"seed_{repetition_seed:03d}"
            / "rankings.csv.gz"
        )


        if not rankings_path.exists():

            raise FileNotFoundError(
                "Raw ranking file is missing:\n"
                f"{rankings_path}"
            )


        ranking_scores = pd.read_csv(
            rankings_path,
            usecols=[
                "Technique",
                "Score",
            ],
            low_memory=False,
        )


        ranking_scores[
            "Score"
        ] = pd.to_numeric(
            ranking_scores[
                "Score"
            ],
            errors="coerce",
        )


        for technique in ML_TECHNIQUES:

            probabilities = (
                ranking_scores.loc[
                    ranking_scores[
                        "Technique"
                    ].eq(
                        technique
                    ),
                    "Score",
                ]
            )


            if len(
                probabilities
            ) != 2300:

                raise AssertionError(
                    "Unexpected model prediction count.\n"
                    f"Noise: {noise_percent}\n"
                    f"Seed: {repetition_seed}\n"
                    f"Technique: {technique}\n"
                    f"Rows: {len(probabilities)}"
                )


            if (
                probabilities.isna().any()
                or
                not np.isfinite(
                    probabilities.to_numpy(
                        dtype=float
                    )
                ).all()
            ):

                raise AssertionError(
                    "Model probabilities contain missing or "
                    "infinite values."
                )


            prediction_records.append({
                "Project":
                    PROJECT_NAME,

                "NoisePercent":
                    int(
                        noise_percent
                    ),

                "RepetitionSeed":
                    int(
                        repetition_seed
                    ),

                "Technique":
                    technique,

                "PredictionRows":
                    int(
                        len(
                            probabilities
                        )
                    ),

                "MinimumProbability":
                    float(
                        probabilities.min()
                    ),

                "MaximumProbability":
                    float(
                        probabilities.max()
                    ),

                "MeanProbability":
                    float(
                        probabilities.mean()
                    ),

                "UniqueProbabilities":
                    int(
                        probabilities.nunique()
                    ),
            })


    print(
        "Prediction summaries reconstructed for seed:",
        seed_position,
        "/",
        len(SEEDS),
    )


PREDICTION_SUMMARY_COLUMNS = [
    "Project",
    "NoisePercent",
    "RepetitionSeed",
    "Technique",
    "PredictionRows",
    "MinimumProbability",
    "MaximumProbability",
    "MeanProbability",
    "UniqueProbabilities",
]


prediction_summary = (
    pd.DataFrame(
        prediction_records
    )[
        PREDICTION_SUMMARY_COLUMNS
    ]
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if len(
    prediction_summary
) != EXPECTED_PREDICTION_SUMMARIES:

    raise AssertionError(
        "Prediction-summary row count differs."
    )


# ------------------------------------------------------------
# 16. EXPLICIT TOP-LEVEL SOURCE TABLES
# ------------------------------------------------------------

TOP_LEVEL_TABLES = {
    "build_metrics_all.parquet":
        build_metrics,

    "condition_summary.csv":
        condition_summary,

    "fit_times_all.csv":
        fit_times,

    "noise_injection_summary.csv":
        noise_injection_summary,

    "prediction_summary_all.csv":
        prediction_summary,

    "project_level_degradation_summary.csv":
        project_level_degradation,

    "project_level_noise_technique_summary.csv":
        project_level_summary,

    "project_run_metrics_all.csv":
        project_run_metrics,

    "project_run_metrics_all.parquet":
        project_run_metrics,
}


reference_top_level_names = {
    Path(relative_path).name
    for relative_path in top_level_paths
}


if (
    reference_top_level_names
    != set(
        TOP_LEVEL_TABLES.keys()
    )
):

    raise AssertionError(
        "Reference top-level filenames differ.\n"
        f"Reference: {sorted(reference_top_level_names)}\n"
        f"Generated: {sorted(TOP_LEVEL_TABLES.keys())}"
    )


# ------------------------------------------------------------
# 17. VALIDATE EXACT GENERATED SCHEMAS
# ------------------------------------------------------------

for file_name, generated_table in (
    TOP_LEVEL_TABLES.items()
):

    expected_columns = (
        REFERENCE_SCHEMAS[
            file_name
        ]
    )

    actual_columns = list(
        generated_table.columns
    )


    if actual_columns != expected_columns:

        raise RuntimeError(
            "Explicit Project 8 schema differs from the "
            "reference package schema.\n"
            f"File: {file_name}\n"
            f"Expected: {expected_columns}\n"
            f"Actual: {actual_columns}"
        )


# ------------------------------------------------------------
# 18. CLEAR ONLY INCOMPLETE PACKAGE OUTPUTS
# ------------------------------------------------------------

if STAGING_PACKAGE_DIR.exists():

    shutil.rmtree(
        STAGING_PACKAGE_DIR
    )


# The project is not registered, so any existing package here
# is an incomplete candidate from a failed Step 11B attempt.

if FINAL_PACKAGE_DIR.exists():

    shutil.rmtree(
        FINAL_PACKAGE_DIR
    )


STAGING_PACKAGE_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# ------------------------------------------------------------
# 19. GENERATE ALL 72 FILES
# ------------------------------------------------------------

generation_records = []

technique_noise_combinations = set()
technique_noise_rows = 0


for relative_path in sorted(
    reference_relative_paths
):

    reference_path = (
        REFERENCE_PACKAGE_DIR
        / relative_path
    )

    reference_frame = read_table(
        reference_path
    )


    normalised_relative_path = (
        relative_path
        .replace("\\", "/")
    )


    if normalised_relative_path.startswith(
        "technique_noise/"
    ):

        file_name = Path(
            normalised_relative_path
        ).name


        parsed = re.fullmatch(
            (
                r"(.+?)"
                r"__noise_(\d{3})"
                r"\.(csv|csv\.gz|parquet)"
            ),
            file_name.lower(),
        )


        if parsed is None:

            raise RuntimeError(
                "Could not parse technique-noise filename:\n"
                f"{relative_path}"
            )


        technique_slug = (
            normalise_technique_slug(
                parsed.group(1)
            )
        )

        noise_percent = int(
            parsed.group(2)
        )


        if technique_slug not in (
            TECHNIQUE_SLUG_LOOKUP
        ):

            raise RuntimeError(
                "Unknown technique slug:\n"
                f"{technique_slug}"
            )


        technique = (
            TECHNIQUE_SLUG_LOOKUP[
                technique_slug
            ]
        )


        generated_frame = (
            project_run_metrics[
                project_run_metrics[
                    "Technique"
                ].eq(
                    technique
                )
                &
                project_run_metrics[
                    "NoisePercent"
                ].eq(
                    noise_percent
                )
            ]
            .sort_values(
                "RepetitionSeed",
                kind="mergesort",
            )
            .reset_index(drop=True)
            .copy()
        )


        if len(
            generated_frame
        ) != 30:

            raise AssertionError(
                "Each technique-noise file must contain "
                "30 rows.\n"
                f"Technique: {technique}\n"
                f"Noise: {noise_percent}\n"
                f"Rows: {len(generated_frame)}"
            )


        if list(
            generated_frame.columns
        ) != list(
            reference_frame.columns
        ):

            raise RuntimeError(
                "Technique-noise schema differs.\n"
                f"File: {relative_path}\n"
                f"Reference: {list(reference_frame.columns)}\n"
                f"Generated: {list(generated_frame.columns)}"
            )


        source_table = (
            "project_run_metrics"
        )

        technique_noise_combinations.add(
            (
                technique,
                noise_percent,
            )
        )

        technique_noise_rows += len(
            generated_frame
        )


    else:

        file_name = Path(
            normalised_relative_path
        ).name


        if file_name not in TOP_LEVEL_TABLES:

            raise RuntimeError(
                "No explicit source table exists for:\n"
                f"{file_name}"
            )


        generated_frame = (
            TOP_LEVEL_TABLES[
                file_name
            ].copy()
        )

        source_table = file_name


    generated_frame = (
        cast_like_reference(
            generated_dataframe=(
                generated_frame
            ),

            reference_dataframe=(
                reference_frame
            ),

            relative_path=(
                relative_path
            ),
        )
    )


    output_path = (
        STAGING_PACKAGE_DIR
        / relative_path
    )


    write_table(
        output_path,
        generated_frame,
    )


    generation_records.append({
        "RelativePath":
            relative_path,

        "SourceTable":
            source_table,

        "ReferenceRows":
            int(
                len(
                    reference_frame
                )
            ),

        "GeneratedRows":
            int(
                len(
                    generated_frame
                )
            ),

        "Columns":
            int(
                len(
                    generated_frame.columns
                )
            ),

        "OutputPath":
            str(
                output_path
            ),
    })


generation_map = pd.DataFrame(
    generation_records
)


# ------------------------------------------------------------
# 20. STAGING-PACKAGE VALIDATION
# ------------------------------------------------------------

staging_files = sorted([
    path
    for path in STAGING_PACKAGE_DIR.rglob("*")
    if path.is_file()
])


staging_relative_paths = {
    path.relative_to(
        STAGING_PACKAGE_DIR
    ).as_posix()
    for path in staging_files
}


expected_relative_paths = set(
    reference_relative_paths
)


missing_package_files = sorted(
    expected_relative_paths
    - staging_relative_paths
)


unexpected_package_files = sorted(
    staging_relative_paths
    - expected_relative_paths
)


expected_technique_noise_combinations = {
    (
        technique,
        noise_percent,
    )
    for technique in TECHNIQUES
    for noise_percent in NOISE_LEVELS
}


readback_failures = []
schema_mismatches = []
row_count_mismatches = []
project7_identity_files = []
invalid_metric_files = []
reference_hash_matches = []


expected_top_level_rows = {
    "build_metrics_all.parquet":
        EXPECTED_BUILD_METRIC_ROWS,

    "condition_summary.csv":
        EXPECTED_CONDITIONS,

    "fit_times_all.csv":
        EXPECTED_ML_FITS,

    "noise_injection_summary.csv":
        9,

    "prediction_summary_all.csv":
        EXPECTED_PREDICTION_SUMMARIES,

    "project_level_degradation_summary.csv":
        63,

    "project_level_noise_technique_summary.csv":
        63,

    "project_run_metrics_all.csv":
        EXPECTED_PROJECT_RUN_ROWS,

    "project_run_metrics_all.parquet":
        EXPECTED_PROJECT_RUN_ROWS,
}


for relative_path in sorted(
    reference_relative_paths
):

    generated_path = (
        STAGING_PACKAGE_DIR
        / relative_path
    )

    reference_path = (
        REFERENCE_PACKAGE_DIR
        / relative_path
    )


    try:

        generated_frame = read_table(
            generated_path
        )

        reference_frame = read_table(
            reference_path
        )

    except Exception as error:

        readback_failures.append({
            "RelativePath":
                relative_path,

            "Error":
                str(error),
        })

        continue


    if list(
        generated_frame.columns
    ) != list(
        reference_frame.columns
    ):

        schema_mismatches.append({
            "RelativePath":
                relative_path,

            "ExpectedColumns":
                list(
                    reference_frame.columns
                ),

            "ActualColumns":
                list(
                    generated_frame.columns
                ),
        })


    if (
        relative_path
        .replace("\\", "/")
        .startswith(
            "technique_noise/"
        )
    ):

        expected_rows = 30

    else:

        expected_rows = (
            expected_top_level_rows[
                Path(relative_path).name
            ]
        )


    if len(
        generated_frame
    ) != expected_rows:

        row_count_mismatches.append({
            "RelativePath":
                relative_path,

            "ExpectedRows":
                expected_rows,

            "ActualRows":
                len(
                    generated_frame
                ),
        })


    # Ensure Project 7 identifiers were not copied.

    contains_project7_identity = False


    for column in generated_frame.columns:

        if (
            pd.api.types.is_object_dtype(
                generated_frame[column]
            )
            or pd.api.types.is_string_dtype(
                generated_frame[column]
            )
        ):

            values = (
                generated_frame[
                    column
                ]
                .fillna("")
                .astype(str)
                .str.lower()
            )


            if (
                values.str.contains(
                    "compevol",
                    regex=False,
                ).any()
                or
                values.str.contains(
                    "beast2",
                    regex=False,
                ).any()
            ):

                contains_project7_identity = True
                break


    if contains_project7_identity:

        project7_identity_files.append(
            relative_path
        )


    # Validate direct APFD/APFDc values only.
    # Delta/loss/percentage columns may legitimately be
    # negative or greater than one.

    direct_metric_columns = {
        "apfd",
        "apfdc",
        "meanapfd",
        "meanapfdc",
        "medianapfd",
        "medianapfdc",
        "minimumapfd",
        "minimumapfdc",
        "maximumapfd",
        "maximumapfdc",
        "cleanprojectmeanapfd",
        "cleanprojectmeanapfdc",
    }


    for column in generated_frame.columns:

        if (
            normalise_column_name(
                column
            )
            not in direct_metric_columns
        ):

            continue


        values = pd.to_numeric(
            generated_frame[
                column
            ],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )


        finite_values = values[
            np.isfinite(
                values
            )
        ]


        if (
            len(finite_values) > 0
            and (
                (finite_values < 0)
                | (finite_values > 1)
            ).any()
        ):

            invalid_metric_files.append({
                "RelativePath":
                    relative_path,

                "Column":
                    column,
            })


    if (
        calculate_sha256(
            generated_path
        )
        ==
        calculate_sha256(
            reference_path
        )
    ):

        reference_hash_matches.append(
            relative_path
        )


if len(
    staging_files
) != EXPECTED_PACKAGE_FILES:

    raise AssertionError(
        "Staging-package file count differs.\n"
        f"Expected: {EXPECTED_PACKAGE_FILES}\n"
        f"Actual: {len(staging_files)}"
    )


if missing_package_files:

    raise AssertionError(
        "Generated package is missing files:\n"
        + "\n".join(
            missing_package_files
        )
    )


if unexpected_package_files:

    raise AssertionError(
        "Generated package contains unexpected files:\n"
        + "\n".join(
            unexpected_package_files
        )
    )


if (
    technique_noise_combinations
    != expected_technique_noise_combinations
):

    raise AssertionError(
        "Technique-noise combinations differ.\n"
        f"Missing: "
        f"{sorted(expected_technique_noise_combinations - technique_noise_combinations)}\n"
        f"Unexpected: "
        f"{sorted(technique_noise_combinations - expected_technique_noise_combinations)}"
    )


if (
    technique_noise_rows
    != EXPECTED_PROJECT_RUN_ROWS
):

    raise AssertionError(
        "Technique-noise files do not reproduce all "
        "1,890 project-run rows."
    )


if readback_failures:

    display(
        pd.DataFrame(
            readback_failures
        )
    )

    raise RuntimeError(
        "At least one generated package file could not "
        "be read back."
    )


if schema_mismatches:

    display(
        pd.DataFrame(
            schema_mismatches
        )
    )

    raise RuntimeError(
        "Generated package schemas differ."
    )


if row_count_mismatches:

    display(
        pd.DataFrame(
            row_count_mismatches
        )
    )

    raise RuntimeError(
        "Generated package row counts differ."
    )


if project7_identity_files:

    raise AssertionError(
        "Generated package contains Project 7 identity:\n"
        + "\n".join(
            project7_identity_files
        )
    )


if invalid_metric_files:

    display(
        pd.DataFrame(
            invalid_metric_files
        )
    )

    raise AssertionError(
        "Generated package contains invalid APFD/APFDc "
        "values."
    )


if reference_hash_matches:

    raise AssertionError(
        "A generated Project 8 file is byte-identical to "
        "the corresponding Project 7 file:\n"
        + "\n".join(
            reference_hash_matches
        )
    )


# ------------------------------------------------------------
# 21. CROSS-FORMAT EQUIVALENCE CHECK
# ------------------------------------------------------------

project_run_csv_readback = pd.read_csv(
    STAGING_PACKAGE_DIR
    / "project_run_metrics_all.csv",
    low_memory=False,
)


project_run_parquet_readback = pd.read_parquet(
    STAGING_PACKAGE_DIR
    / "project_run_metrics_all.parquet"
)


pd.testing.assert_frame_equal(
    project_run_csv_readback,
    project_run_parquet_readback,
    check_dtype=False,
    check_like=False,
)


# ------------------------------------------------------------
# 22. PROMOTE STAGING PACKAGE
# ------------------------------------------------------------

STAGING_PACKAGE_DIR.replace(
    FINAL_PACKAGE_DIR
)


# ------------------------------------------------------------
# 23. FINAL PACKAGE INVENTORY
# ------------------------------------------------------------

final_package_files = sorted([
    path
    for path in FINAL_PACKAGE_DIR.rglob("*")
    if path.is_file()
])


inventory_records = []


for path in final_package_files:

    inventory_records.append({
        "RelativePath":
            path.relative_to(
                FINAL_PACKAGE_DIR
            ).as_posix(),

        "SizeBytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            calculate_sha256(
                path
            ),
    })


final_package_inventory = pd.DataFrame(
    inventory_records
)


final_package_bytes = int(
    final_package_inventory[
        "SizeBytes"
    ].sum()
)


final_package_root_sha256 = (
    root_inventory_hash(
        final_package_inventory
    )
)


final_relative_paths = set(
    final_package_inventory[
        "RelativePath"
    ]
)


final_missing_files = sorted(
    expected_relative_paths
    - final_relative_paths
)


final_unexpected_files = sorted(
    final_relative_paths
    - expected_relative_paths
)


top_level_file_count = int(
    sum(
        1
        for relative_path
        in final_relative_paths
        if not relative_path.startswith(
            "technique_noise/"
        )
    )
)


technique_noise_file_count = int(
    sum(
        1
        for relative_path
        in final_relative_paths
        if relative_path.startswith(
            "technique_noise/"
        )
    )
)


# ------------------------------------------------------------
# 24. FINAL VALIDATION TABLE
# ------------------------------------------------------------

validation_records = [
    {
        "Check":
            "Project 8 Step 10 passed",

        "Expected":
            EXPECTED_STEP10_STATUS,

        "Actual":
            step10_status[
                "Status"
            ],

        "Pass":
            step10_status[
                "Status"
            ]
            == EXPECTED_STEP10_STATUS,
    },

    {
        "Check":
            "Final package files",

        "Expected":
            EXPECTED_PACKAGE_FILES,

        "Actual":
            len(
                final_package_inventory
            ),

        "Pass":
            len(
                final_package_inventory
            )
            == EXPECTED_PACKAGE_FILES,
    },

    {
        "Check":
            "Top-level files",

        "Expected":
            EXPECTED_TOP_LEVEL_FILES,

        "Actual":
            top_level_file_count,

        "Pass":
            top_level_file_count
            == EXPECTED_TOP_LEVEL_FILES,
    },

    {
        "Check":
            "Technique-noise files",

        "Expected":
            EXPECTED_TECHNIQUE_NOISE_FILES,

        "Actual":
            technique_noise_file_count,

        "Pass":
            technique_noise_file_count
            == EXPECTED_TECHNIQUE_NOISE_FILES,
    },

    {
        "Check":
            "Technique-noise combinations",

        "Expected":
            EXPECTED_TECHNIQUE_NOISE_FILES,

        "Actual":
            len(
                technique_noise_combinations
            ),

        "Pass":
            technique_noise_combinations
            == expected_technique_noise_combinations,
    },

    {
        "Check":
            "Technique-noise rows",

        "Expected":
            EXPECTED_PROJECT_RUN_ROWS,

        "Actual":
            technique_noise_rows,

        "Pass":
            technique_noise_rows
            == EXPECTED_PROJECT_RUN_ROWS,
    },

    {
        "Check":
            "Build-metric rows",

        "Expected":
            EXPECTED_BUILD_METRIC_ROWS,

        "Actual":
            len(
                build_metrics
            ),

        "Pass":
            len(
                build_metrics
            )
            == EXPECTED_BUILD_METRIC_ROWS,
    },

    {
        "Check":
            "Condition-summary rows",

        "Expected":
            EXPECTED_CONDITIONS,

        "Actual":
            len(
                condition_summary
            ),

        "Pass":
            len(
                condition_summary
            )
            == EXPECTED_CONDITIONS,
    },

    {
        "Check":
            "Prediction-summary rows",

        "Expected":
            EXPECTED_PREDICTION_SUMMARIES,

        "Actual":
            len(
                prediction_summary
            ),

        "Pass":
            len(
                prediction_summary
            )
            == EXPECTED_PREDICTION_SUMMARIES,
    },

    {
        "Check":
            "Project-level summary rows",

        "Expected":
            63,

        "Actual":
            len(
                project_level_summary
            ),

        "Pass":
            len(
                project_level_summary
            )
            == 63,
    },

    {
        "Check":
            "Degradation-summary rows",

        "Expected":
            63,

        "Actual":
            len(
                project_level_degradation
            ),

        "Pass":
            len(
                project_level_degradation
            )
            == 63,
    },

    {
        "Check":
            "Missing package files",

        "Expected":
            0,

        "Actual":
            len(
                final_missing_files
            ),

        "Pass":
            len(
                final_missing_files
            )
            == 0,
    },

    {
        "Check":
            "Unexpected package files",

        "Expected":
            0,

        "Actual":
            len(
                final_unexpected_files
            ),

        "Pass":
            len(
                final_unexpected_files
            )
            == 0,
    },

    {
        "Check":
            "Schema mismatches",

        "Expected":
            0,

        "Actual":
            len(
                schema_mismatches
            ),

        "Pass":
            len(
                schema_mismatches
            )
            == 0,
    },

    {
        "Check":
            "Row-count mismatches",

        "Expected":
            0,

        "Actual":
            len(
                row_count_mismatches
            ),

        "Pass":
            len(
                row_count_mismatches
            )
            == 0,
    },

    {
        "Check":
            "Readback failures",

        "Expected":
            0,

        "Actual":
            len(
                readback_failures
            ),

        "Pass":
            len(
                readback_failures
            )
            == 0,
    },

    {
        "Check":
            "Project 7 identity files",

        "Expected":
            0,

        "Actual":
            len(
                project7_identity_files
            ),

        "Pass":
            len(
                project7_identity_files
            )
            == 0,
    },

    {
        "Check":
            "Invalid APFD/APFDc files",

        "Expected":
            0,

        "Actual":
            len(
                invalid_metric_files
            ),

        "Pass":
            len(
                invalid_metric_files
            )
            == 0,
    },

    {
        "Check":
            "Files identical to Project 7",

        "Expected":
            0,

        "Actual":
            len(
                reference_hash_matches
            ),

        "Pass":
            len(
                reference_hash_matches
            )
            == 0,
    },

    {
        "Check":
            "Project 8 registry rows",

        "Expected":
            0,

        "Actual":
            len(
                project8_registry_rows
            ),

        "Pass":
            len(
                project8_registry_rows
            )
            == 0,
    },
]


final_package_validation = pd.DataFrame(
    validation_records
)


failed_checks = (
    final_package_validation[
        ~final_package_validation[
            "Pass"
        ]
    ]
    .copy()
)


print("\nFinal package validation:")

display(
    final_package_validation
)


if not failed_checks.empty:

    print("\nFailed checks:")

    display(
        failed_checks
    )

    raise RuntimeError(
        "PROJECT 8 STEP 11B V3 DID NOT PASS.\n"
        "Do not modify the completion registry."
    )


# ------------------------------------------------------------
# 25. WRITE PACKAGE AUDIT OUTPUTS
# ------------------------------------------------------------

FINAL_PACKAGE_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    FINAL_PACKAGE_INVENTORY_PATH,
    final_package_inventory,
)


atomic_write_csv(
    FINAL_PACKAGE_VALIDATION_PATH,
    final_package_validation,
)


atomic_write_csv(
    FINAL_PACKAGE_GENERATION_MAP_PATH,
    generation_map,
)


step11b_status_text = (
    "PASS_PROJECT_8_FINAL_72_FILE_PACKAGE_GENERATED_AND_VALIDATED"
)


final_package_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "GeneratedAtUTC":
        GENERATED_AT_UTC,

    "AuditedSourceResults": {
        "Conditions":
            EXPECTED_CONDITIONS,

        "MLFits":
            EXPECTED_ML_FITS,

        "PredictionSummaries":
            EXPECTED_PREDICTION_SUMMARIES,

        "ProjectRunRows":
            EXPECTED_PROJECT_RUN_ROWS,

        "BuildMetricRows":
            EXPECTED_BUILD_METRIC_ROWS,
    },

    "StatisticalConvention": {
        "CI95CriticalValue":
            CI_CRITICAL_95,

        "RetentionScale":
            RETENTION_SCALE,

        "ConventionDerivedFromFrozenReferencePackage":
            True,
    },

    "FinalPackage": {
        "Directory":
            str(
                FINAL_PACKAGE_DIR
            ),

        "Files":
            len(
                final_package_inventory
            ),

        "TopLevelFiles":
            top_level_file_count,

        "TechniqueNoiseFiles":
            technique_noise_file_count,

        "Bytes":
            final_package_bytes,

        "RootSHA256":
            final_package_root_sha256,

        "MissingFiles":
            len(
                final_missing_files
            ),

        "UnexpectedFiles":
            len(
                final_unexpected_files
            ),
    },

    "Validation": {
        "Checks":
            len(
                final_package_validation
            ),

        "FailedChecks":
            len(
                failed_checks
            ),

        "SchemaMismatches":
            len(
                schema_mismatches
            ),

        "RowCountMismatches":
            len(
                row_count_mismatches
            ),

        "ReadbackFailures":
            len(
                readback_failures
            ),

        "Project7IdentityFiles":
            len(
                project7_identity_files
            ),

        "InvalidMetricFiles":
            len(
                invalid_metric_files
            ),

        "ReferenceHashMatches":
            len(
                reference_hash_matches
            ),
    },

    "CompletionRegistryModified":
        False,

    "Projects1To7Modified":
        False,

    "Status":
        step11b_status_text,
}


atomic_write_json(
    FINAL_PACKAGE_REPORT_PATH,
    final_package_report,
)


step11b_status = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        step11b_status_text,

    "RawFiles":
        EXPECTED_RAW_FILES,

    "RawBytes":
        EXPECTED_RAW_BYTES,

    "RawRootSHA256":
        EXPECTED_RAW_ROOT_SHA256,

    "PackageFiles":
        len(
            final_package_inventory
        ),

    "PackageBytes":
        final_package_bytes,

    "PackageRootSHA256":
        final_package_root_sha256,

    "TopLevelFiles":
        top_level_file_count,

    "TechniqueNoiseFiles":
        technique_noise_file_count,

    "MissingPackageFiles":
        len(
            final_missing_files
        ),

    "UnexpectedPackageFiles":
        len(
            final_unexpected_files
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletionRegistryModified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP11B_STATUS_PATH,
    step11b_status,
)


audit_outputs = [
    FINAL_PACKAGE_INVENTORY_PATH,
    FINAL_PACKAGE_VALIDATION_PATH,
    FINAL_PACKAGE_GENERATION_MAP_PATH,
    FINAL_PACKAGE_REPORT_PATH,
    STEP11B_STATUS_PATH,
]


missing_audit_outputs = [
    str(path)
    for path in audit_outputs
    if not path.exists()
]


if missing_audit_outputs:

    raise RuntimeError(
        "Step 11B V3 audit outputs are missing:\n"
        + "\n".join(
            missing_audit_outputs
        )
    )


# ------------------------------------------------------------
# 26. DISPLAY PROJECT 8 APFD/APFDC RESULTS
# ------------------------------------------------------------

result_display = (
    project_level_summary[
        [
            "NoisePercent",
            "Technique",
            "Seeds",
            "MeanAPFD",
            "SD_APFD_AcrossSeeds",
            "MeanAPFDc",
            "SD_APFDc_AcrossSeeds",
        ]
    ]
    .sort_values(
        [
            "NoisePercent",
            "MeanAPFDc",
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


print(
    "\nProject 8 APFD/APFDc summary included "
    "in the final package:"
)

display(
    result_display.round(6)
)


# ------------------------------------------------------------
# 27. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("=== PROJECT 8 STEP 11B V3 RESULT ===")
print("=" * 100)

print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nAudited source results:")

print(
    "Conditions:",
    EXPECTED_CONDITIONS,
)

print(
    "ML fits:",
    EXPECTED_ML_FITS,
)

print(
    "Prediction summaries:",
    len(
        prediction_summary
    ),
)

print(
    "Project-run rows:",
    len(
        project_run_metrics
    ),
)

print(
    "Build-metric rows:",
    len(
        build_metrics
    ),
)

print(
    "Techniques:",
    len(
        TECHNIQUES
    ),
)


print("\nFinal package:")

print(
    "Directory:",
    FINAL_PACKAGE_DIR,
)

print(
    "Files:",
    len(
        final_package_inventory
    ),
)

print(
    "Top-level files:",
    top_level_file_count,
)

print(
    "Technique-noise files:",
    technique_noise_file_count,
)

print(
    "Package bytes:",
    final_package_bytes,
)

print(
    "Package root SHA-256:",
    final_package_root_sha256,
)

print(
    "Missing files:",
    len(
        final_missing_files
    ),
)

print(
    "Unexpected files:",
    len(
        final_unexpected_files
    ),
)


print("\nContent validation:")

print(
    "Technique-noise combinations:",
    len(
        technique_noise_combinations
    ),
)

print(
    "Technique-noise rows:",
    technique_noise_rows,
)

print(
    "Schema mismatches:",
    len(
        schema_mismatches
    ),
)

print(
    "Row-count mismatches:",
    len(
        row_count_mismatches
    ),
)

print(
    "Readback failures:",
    len(
        readback_failures
    ),
)

print(
    "Project 7 identity files:",
    len(
        project7_identity_files
    ),
)

print(
    "Files identical to Project 7:",
    len(
        reference_hash_matches
    ),
)

print(
    "Invalid APFD/APFDc files:",
    len(
        invalid_metric_files
    ),
)


print("\nValidation:")

print(
    "Checks:",
    len(
        final_package_validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_checks
    ),
)


print("\nAudit outputs:")

for output_path in audit_outputs:

    print(output_path)


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–7 modified:")
print(0)


print(
    "\nSTATUS:",
    step11b_status_text,
)

print("=" * 100)

=== PROJECT 8 STEP 11B V3: GENERATE FINAL 72-FILE RESULT PACKAGE ===

Input validation:
Step 10 status: PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED
Raw files: 2160
Raw bytes: 179766494
Raw root SHA-256: 19ae21c5d8524c358796e28f23577fbf03bfd31f7d4b530372c6b4d8a492887c
Project 8 registry rows: 0

Reference package blueprint:
Directory: /content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_30_seed_final
Inventory rows: 72
Top-level files: 9
Technique-noise files: 63

Frozen summary conventions:
CI critical value: 2.045229642132711
Retention scale: 1.0

Audited Project 8 inputs:
Project-run rows: 1890
Build-metric rows: 30240
Condition rows: 270
ML fit rows: 1080

Reconstructing 1,080 ML prediction summaries from raw Project 8 rankings...
Prediction summaries reconstructed for seed: 1 / 30
Prediction summaries reconstructed for seed: 2 / 30
Prediction summaries reconstructed for seed: 3 / 30
Prediction summaries reconstructed for seed: 4 / 30
Prediction 

RuntimeError: Technique-noise schema differs.
File: technique_noise/latest_fail__noise_000.csv
Reference: ['Project', 'NoisePercent', 'RepetitionSeed', 'Technique', 'MeanAPFD', 'MeanAPFDc', 'SD_APFD', 'SD_APFDc', 'EvaluatedBuilds', 'TotalRankedTests', 'TotalFailures', 'CleanMeanAPFD', 'CleanMeanAPFDc', 'DeltaAPFD_NoiseMinusClean', 'DeltaAPFDc_NoiseMinusClean', 'AbsoluteLossAPFD', 'AbsoluteLossAPFDc', 'PercentageLossAPFD', 'PercentageLossAPFDc', 'RetentionAPFD', 'RetentionAPFDc']
Generated: ['Project', 'NoisePercent', 'RepetitionSeed', 'Technique', 'MeanAPFD', 'MeanAPFDc', 'SD_APFD', 'SD_APFDc', 'EvaluatedBuilds', 'TotalRankedTests', 'TotalFailures']

In [ ]:
# ============================================================
# PROJECT 8 — STEP 11B V4
# FINAL 72-FILE PACKAGE GENERATION AND VALIDATION
#
# PROJECT: optimatika@ojAlgo
#
# This version explicitly generates:
# - 9 top-level result files
# - 63 technique × noise files
# - per-seed clean references and degradation statistics
# - exactly 72 package files
#
# Project 7 is used only for:
# - frozen filenames
# - column order
# - broad column dtypes
# - CI and retention conventions
#
# No Project 7 result values are copied.
#
# This cell does NOT:
# - rerun models
# - modify raw Project 8 results
# - modify Projects 1–7
# - update the completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import re
import shutil

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 8
PROJECT_NAME = "optimatika@ojAlgo"
PROJECT_SLUG = "optimatika__ojAlgo"
PROJECT_SHORT_NAME = "ojalgo"

REFERENCE_PROJECT_NAME = "CompEvol@beast2"
REFERENCE_PROJECT_SLUG = "CompEvol__beast2"
REFERENCE_PROJECT_SHORT_NAME = "beast2"

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

SEEDS = list(range(1, 31))

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)

TECHNIQUE_SLUG_TO_NAME = {
    "randomforest":
        "RandomForest",

    "xgboost":
        "XGBoost",

    "lightgbm":
        "LightGBM",

    "naivebayes":
        "NaiveBayes",

    "random":
        "Random",

    "latestfail":
        "LatestFail",

    "qtfavg":
        "QTF-Avg",
}

EXPECTED_CONDITIONS = 270
EXPECTED_ML_FITS = 1080
EXPECTED_PROJECT_RUN_ROWS = 1890
EXPECTED_BUILD_METRIC_ROWS = 30240
EXPECTED_PREDICTION_ROWS = 1080

EXPECTED_PACKAGE_FILES = 72
EXPECTED_TOP_LEVEL_FILES = 9
EXPECTED_TECHNIQUE_NOISE_FILES = 63

EXPECTED_RAW_FILES = 2160
EXPECTED_RAW_BYTES = 179766494

EXPECTED_RAW_ROOT_SHA256 = (
    "19ae21c5d8524c358796e28f23577fbf03bfd31f7d4b530372c6b4d8a492887c"
)

EXPECTED_STEP10_STATUS = (
    "PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED"
)

FALLBACK_CI_CRITICAL_95 = (
    2.045229642132703
)

GENERATED_AT_UTC = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

AGGREGATED_ROOT = (
    RESULTS_DIR
    / "Aggregated"
)

PROJECT_DIR = (
    AGGREGATED_ROOT
    / PROJECT_SLUG
)

RAW_RESULTS_ROOT = (
    RESULTS_DIR
    / "Raw"
    / PROJECT_SLUG
)

RAW_AUDIT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_raw_audit"
)

STEP10_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step10_status.json"
)

REFERENCE_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_07_selection_checkpoint.json"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)


# Audited Project 8 inputs

PROJECT_RUN_INPUT = (
    RAW_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_all_project_run_metrics.csv"
)

BUILD_METRICS_INPUT = (
    RAW_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_all_build_metrics.csv.gz"
)

NOISE_SUMMARY_INPUT = (
    RAW_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_all_noise_summary.csv"
)

FIT_TIMES_INPUT = (
    RAW_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_all_fit_times.csv"
)


# Final package candidate

FINAL_PACKAGE_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_30_seed_final"
)

STAGING_PACKAGE_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_30_seed_final__staging"
)


# Package audit outputs

FINAL_PACKAGE_AUDIT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_audit"
)

FINAL_PACKAGE_INVENTORY_PATH = (
    FINAL_PACKAGE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_file_inventory_sha256.csv"
)

FINAL_PACKAGE_VALIDATION_PATH = (
    FINAL_PACKAGE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_validation.csv"
)

FINAL_PACKAGE_GENERATION_MAP_PATH = (
    FINAL_PACKAGE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_generation_map.csv"
)

FINAL_PACKAGE_REPORT_PATH = (
    FINAL_PACKAGE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_report.json"
)

STEP11B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step11b_status.json"
)


print("=" * 100)
print("=== PROJECT 8 STEP 11B V4: GENERATE FINAL 72-FILE RESULT PACKAGE ===")
print("=" * 100)


# ------------------------------------------------------------
# 3. HELPERS
# ------------------------------------------------------------

def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def json_safe(value):

    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:

        if pd.isna(value):
            return None

    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(path)


def recursive_key_lookup(
    value,
    requested_key,
):
    requested_key = (
        str(requested_key)
        .strip()
        .lower()
    )

    if isinstance(value, dict):

        for key, child in value.items():

            if (
                str(key)
                .strip()
                .lower()
                == requested_key
            ):
                return child

        for child in value.values():

            result = recursive_key_lookup(
                child,
                requested_key,
            )

            if result is not None:
                return result

    elif isinstance(value, list):

        for child in value:

            result = recursive_key_lookup(
                child,
                requested_key,
            )

            if result is not None:
                return result

    return None


def normalise_column_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


def package_relative_path(
    value,
    package_directory,
):
    text = (
        str(value)
        .replace("\\", "/")
        .strip()
    )

    text = re.sub(
        r"/+",
        "/",
        text,
    )

    package_text = (
        str(package_directory)
        .replace("\\", "/")
        .rstrip("/")
    )

    if text.startswith(
        package_text + "/"
    ):

        return text[
            len(package_text) + 1:
        ]

    package_name = Path(
        package_directory
    ).name

    marker = package_name + "/"

    if marker in text:

        return text.split(
            marker,
            1,
        )[1]

    return text.lstrip("./")


def read_table(
    path,
    **kwargs,
):
    path = Path(path)

    suffixes = "".join(
        path.suffixes
    ).lower()

    if suffixes.endswith(
        ".parquet"
    ):

        return pd.read_parquet(
            path,
            **kwargs,
        )

    if (
        suffixes.endswith(".csv")
        or suffixes.endswith(".csv.gz")
    ):

        return pd.read_csv(
            path,
            low_memory=False,
            **kwargs,
        )

    raise RuntimeError(
        "Unsupported table format:\n"
        f"{path}"
    )


def write_table(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    suffixes = "".join(
        path.suffixes
    ).lower()

    if suffixes.endswith(
        ".parquet"
    ):

        temporary_path = path.with_name(
            path.name + ".tmp.parquet"
        )

        dataframe.to_parquet(
            temporary_path,
            index=False,
        )

        temporary_path.replace(path)

        return

    if suffixes.endswith(
        ".csv.gz"
    ):

        temporary_path = path.with_name(
            path.name + ".tmp.gz"
        )

        dataframe.to_csv(
            temporary_path,
            index=False,
            compression="gzip",
        )

        temporary_path.replace(path)

        return

    if suffixes.endswith(
        ".csv"
    ):

        temporary_path = path.with_name(
            path.name + ".tmp"
        )

        dataframe.to_csv(
            temporary_path,
            index=False,
        )

        temporary_path.replace(path)

        return

    raise RuntimeError(
        "Unsupported output format:\n"
        f"{path}"
    )


def root_inventory_hash(
    inventory,
):
    digest = hashlib.sha256()

    ordered = inventory.sort_values(
        "RelativePath",
        kind="mergesort",
    )

    for row in ordered.itertuples(
        index=False
    ):

        digest.update(
            row.RelativePath.encode(
                "utf-8"
            )
        )

        digest.update(b"\0")

        digest.update(
            str(
                int(row.SizeBytes)
            ).encode(
                "utf-8"
            )
        )

        digest.update(b"\0")

        digest.update(
            bytes.fromhex(
                row.SHA256
            )
        )

        digest.update(b"\n")

    return digest.hexdigest()


def find_column(
    dataframe,
    candidate_names,
):
    lookup = {
        normalise_column_name(column):
            column
        for column in dataframe.columns
    }

    for candidate_name in candidate_names:

        normalised_candidate = (
            normalise_column_name(
                candidate_name
            )
        )

        if normalised_candidate in lookup:

            return lookup[
                normalised_candidate
            ]

    raise RuntimeError(
        "Required column could not be found.\n"
        f"Candidates: {candidate_names}\n"
        f"Available: {list(dataframe.columns)}"
    )


def condition_key(
    noise_percent,
    repetition_seed,
):
    return (
        f"noise_{int(noise_percent):03d}"
        f"__seed_{int(repetition_seed):03d}"
    )


def transform_reference_identity(
    value,
):
    transformed = str(value)

    replacements = [
        (
            REFERENCE_PROJECT_NAME,
            PROJECT_NAME,
        ),
        (
            REFERENCE_PROJECT_SLUG,
            PROJECT_SLUG,
        ),
        (
            REFERENCE_PROJECT_SHORT_NAME,
            PROJECT_SHORT_NAME,
        ),
        (
            "project_07",
            "project_08",
        ),
    ]

    for old_value, new_value in replacements:

        transformed = transformed.replace(
            old_value,
            new_value,
        )

    return transformed


def cast_like_reference(
    generated_dataframe,
    reference_dataframe,
    relative_path,
):
    generated_dataframe = (
        generated_dataframe.copy()
    )

    if list(
        generated_dataframe.columns
    ) != list(
        reference_dataframe.columns
    ):

        raise RuntimeError(
            "Generated schema differs from reference.\n"
            f"File: {relative_path}\n"
            f"Reference: {list(reference_dataframe.columns)}\n"
            f"Generated: {list(generated_dataframe.columns)}"
        )

    for column in generated_dataframe.columns:

        reference_dtype = (
            reference_dataframe[
                column
            ].dtype
        )

        try:

            if pd.api.types.is_integer_dtype(
                reference_dtype
            ):

                numeric = pd.to_numeric(
                    generated_dataframe[
                        column
                    ],
                    errors="raise",
                )

                if numeric.isna().any():

                    generated_dataframe[
                        column
                    ] = numeric.astype(
                        "Int64"
                    )

                else:

                    generated_dataframe[
                        column
                    ] = numeric.astype(
                        reference_dtype
                    )

            elif pd.api.types.is_float_dtype(
                reference_dtype
            ):

                generated_dataframe[
                    column
                ] = pd.to_numeric(
                    generated_dataframe[
                        column
                    ],
                    errors="coerce",
                ).astype(float)

            elif pd.api.types.is_bool_dtype(
                reference_dtype
            ):

                generated_dataframe[
                    column
                ] = generated_dataframe[
                    column
                ].astype(bool)

        except Exception as error:

            raise RuntimeError(
                "Failed to cast generated column.\n"
                f"File: {relative_path}\n"
                f"Column: {column}\n"
                f"Reference dtype: {reference_dtype}\n"
                f"Error: {error}"
            )

    return generated_dataframe


def normalise_technique_slug(
    value,
):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


# ------------------------------------------------------------
# 4. VALIDATE PROJECT STATE
# ------------------------------------------------------------

required_paths = [
    STEP10_STATUS_PATH,
    REFERENCE_CHECKPOINT_PATH,
    REGISTRY_PATH,
    PROJECT_RUN_INPUT,
    BUILD_METRICS_INPUT,
    NOISE_SUMMARY_INPUT,
    FIT_TIMES_INPUT,
    RAW_RESULTS_ROOT,
]


missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]


if missing_paths:

    raise FileNotFoundError(
        "Required Step 11B V4 inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


step10_status = json.loads(
    STEP10_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    step10_status.get("Status")
    != EXPECTED_STEP10_STATUS
):

    raise AssertionError(
        "Project 8 Step 10 has not passed.\n"
        f"Detected: "
        f"{step10_status.get('Status')}"
    )


if (
    int(
        step10_status.get(
            "RawFiles",
            -1,
        )
    )
    != EXPECTED_RAW_FILES
):

    raise AssertionError(
        "Project 8 raw-file count differs."
    )


if (
    int(
        step10_status.get(
            "RawBytes",
            -1,
        )
    )
    != EXPECTED_RAW_BYTES
):

    raise AssertionError(
        "Project 8 raw-byte count differs."
    )


if (
    step10_status.get(
        "RawRootSHA256"
    )
    != EXPECTED_RAW_ROOT_SHA256
):

    raise AssertionError(
        "Project 8 raw-root SHA-256 differs."
    )


registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
)


registry_project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="coerce",
)


project8_registry_rows = registry[
    registry_project_numbers.eq(
        PROJECT_NUMBER
    )
]


if len(
    project8_registry_rows
) != 0:

    raise AssertionError(
        "Project 8 is already present in the completion "
        "registry."
    )


print("\nInput validation:")

print(
    "Step 10 status:",
    step10_status["Status"],
)

print(
    "Raw files:",
    step10_status["RawFiles"],
)

print(
    "Raw bytes:",
    step10_status["RawBytes"],
)

print(
    "Raw root SHA-256:",
    step10_status["RawRootSHA256"],
)

print(
    "Project 8 registry rows:",
    len(
        project8_registry_rows
    ),
)


# ------------------------------------------------------------
# 5. LOAD EXACT PROJECT 7 PACKAGE BLUEPRINT
# ------------------------------------------------------------

reference_checkpoint = json.loads(
    REFERENCE_CHECKPOINT_PATH.read_text(
        encoding="utf-8"
    )
)


reference_package_directory_value = (
    recursive_key_lookup(
        reference_checkpoint,
        "FinalPackageDirectory",
    )
)


reference_inventory_value = (
    recursive_key_lookup(
        reference_checkpoint,
        "FinalPackageInventory",
    )
)


if (
    reference_package_directory_value is None
    or reference_inventory_value is None
):

    raise RuntimeError(
        "Reference checkpoint does not contain final-package "
        "paths."
    )


REFERENCE_PACKAGE_DIR = Path(
    reference_package_directory_value
)

REFERENCE_INVENTORY_PATH = Path(
    reference_inventory_value
)


if not REFERENCE_PACKAGE_DIR.exists():

    raise FileNotFoundError(
        "Reference package directory is missing:\n"
        f"{REFERENCE_PACKAGE_DIR}"
    )


if not REFERENCE_INVENTORY_PATH.exists():

    raise FileNotFoundError(
        "Reference package inventory is missing:\n"
        f"{REFERENCE_INVENTORY_PATH}"
    )


reference_inventory = pd.read_csv(
    REFERENCE_INVENTORY_PATH,
    low_memory=False,
)


relative_path_column = find_column(
    reference_inventory,
    [
        "RelativePath",
        "PackageRelativePath",
        "Path",
        "FilePath",
    ],
)


reference_relative_paths = (
    reference_inventory[
        relative_path_column
    ]
    .astype(str)
    .map(
        lambda value:
            package_relative_path(
                value,
                REFERENCE_PACKAGE_DIR,
            )
    )
    .tolist()
)


if len(
    reference_relative_paths
) != EXPECTED_PACKAGE_FILES:

    raise AssertionError(
        "Reference package inventory does not contain "
        "72 rows."
    )


if len(
    set(reference_relative_paths)
) != EXPECTED_PACKAGE_FILES:

    raise AssertionError(
        "Reference package paths are not unique."
    )


missing_reference_files = [
    relative_path
    for relative_path
    in reference_relative_paths
    if not (
        REFERENCE_PACKAGE_DIR
        / relative_path
    ).exists()
]


if missing_reference_files:

    raise FileNotFoundError(
        "Reference package files are missing:\n"
        + "\n".join(
            missing_reference_files
        )
    )


technique_noise_paths = [
    relative_path
    for relative_path
    in reference_relative_paths
    if (
        relative_path
        .replace("\\", "/")
        .startswith(
            "technique_noise/"
        )
    )
]


top_level_paths = [
    relative_path
    for relative_path
    in reference_relative_paths
    if relative_path
    not in technique_noise_paths
]


if len(
    technique_noise_paths
) != EXPECTED_TECHNIQUE_NOISE_FILES:

    raise AssertionError(
        "Reference technique-noise file count differs."
    )


if len(
    top_level_paths
) != EXPECTED_TOP_LEVEL_FILES:

    raise AssertionError(
        "Reference top-level file count differs."
    )


print("\nReference package blueprint:")

print(
    "Directory:",
    REFERENCE_PACKAGE_DIR,
)

print(
    "Inventory rows:",
    len(
        reference_relative_paths
    ),
)

print(
    "Top-level files:",
    len(
        top_level_paths
    ),
)

print(
    "Technique-noise files:",
    len(
        technique_noise_paths
    ),
)


# ------------------------------------------------------------
# 6. VALIDATE REFERENCE SCHEMAS
# ------------------------------------------------------------

EXPECTED_TOP_LEVEL_SCHEMAS = {
    "build_metrics_all.parquet": [
        "Project",
        "NoisePercent",
        "RepetitionSeed",
        "Technique",
        "Build",
        "BuildOrder",
        "NumberOfTests",
        "NumberOfFailures",
        "APFD",
        "APFDc",
    ],

    "condition_summary.csv": [
        "Project",
        "ProjectSlug",
        "ConditionOrder",
        "ConditionKey",
        "NoisePercent",
        "RepetitionSeed",
        "TrainingExecutionRows",
        "RawRowsFlipped",
        "RealisedNoisePercent",
        "PassToFailure",
        "FailureToPass",
        "RetainedLabelChanges",
        "TrainingFailures",
        "TrainingPasses",
        "ConditionSeconds",
        "Status",
    ],

    "fit_times_all.csv": [
        "Project",
        "NoisePercent",
        "RepetitionSeed",
        "Technique",
        "FitSeconds",
        "TrainingRows",
        "TrainingFailures",
        "TrainingPasses",
        "ActiveFeatures",
    ],

    "noise_injection_summary.csv": [
        "NoisePercent",
        "Seeds",
        "TrainingExecutionRowsPerSeed",
        "MeanRawRowsFlipped",
        "SDRawRowsFlipped",
        "MinimumRawRowsFlipped",
        "MaximumRawRowsFlipped",
        "MeanRealisedNoisePercent",
        "SDRealisedNoisePercent",
        "MeanPassToFailure",
        "MeanFailureToPass",
        "MeanRetainedLabelChanges",
        "SDRetainedLabelChanges",
        "MeanTrainingFailures",
        "MeanTrainingPasses",
        "MeanConditionSeconds",
        "SDConditionSeconds",
    ],

    "prediction_summary_all.csv": [
        "Project",
        "NoisePercent",
        "RepetitionSeed",
        "Technique",
        "PredictionRows",
        "MinimumProbability",
        "MaximumProbability",
        "MeanProbability",
        "UniqueProbabilities",
    ],

    "project_level_degradation_summary.csv": [
        "Project",
        "NoisePercent",
        "Technique",
        "Seeds",
        "MeanAPFD",
        "SD_APFD_AcrossSeeds",
        "MedianAPFD",
        "MinimumAPFD",
        "MaximumAPFD",
        "MeanAPFDc",
        "SD_APFDc_AcrossSeeds",
        "MedianAPFDc",
        "MinimumAPFDc",
        "MaximumAPFDc",
        "EvaluatedBuildsPerSeed",
        "TotalRankedTestsPerSeed",
        "TotalFailuresPerSeed",
        "SE_APFD",
        "CI95LowerAPFD",
        "CI95UpperAPFD",
        "SE_APFDc",
        "CI95LowerAPFDc",
        "CI95UpperAPFDc",
        "CleanProjectMeanAPFD",
        "CleanProjectMeanAPFDc",
        "DeltaAPFD_NoiseMinusClean",
        "DeltaAPFDc_NoiseMinusClean",
        "AbsoluteLossAPFD",
        "AbsoluteLossAPFDc",
        "PercentageLossAPFD",
        "PercentageLossAPFDc",
        "RetentionAPFD",
        "RetentionAPFDc",
    ],

    "project_level_noise_technique_summary.csv": [
        "Project",
        "NoisePercent",
        "Technique",
        "Seeds",
        "MeanAPFD",
        "SD_APFD_AcrossSeeds",
        "MedianAPFD",
        "MinimumAPFD",
        "MaximumAPFD",
        "MeanAPFDc",
        "SD_APFDc_AcrossSeeds",
        "MedianAPFDc",
        "MinimumAPFDc",
        "MaximumAPFDc",
        "EvaluatedBuildsPerSeed",
        "TotalRankedTestsPerSeed",
        "TotalFailuresPerSeed",
        "SE_APFD",
        "CI95LowerAPFD",
        "CI95UpperAPFD",
        "SE_APFDc",
        "CI95LowerAPFDc",
        "CI95UpperAPFDc",
    ],

    "project_run_metrics_all.csv": [
        "Project",
        "NoisePercent",
        "RepetitionSeed",
        "Technique",
        "MeanAPFD",
        "MeanAPFDc",
        "SD_APFD",
        "SD_APFDc",
        "EvaluatedBuilds",
        "TotalRankedTests",
        "TotalFailures",
    ],

    "project_run_metrics_all.parquet": [
        "Project",
        "NoisePercent",
        "RepetitionSeed",
        "Technique",
        "MeanAPFD",
        "MeanAPFDc",
        "SD_APFD",
        "SD_APFDc",
        "EvaluatedBuilds",
        "TotalRankedTests",
        "TotalFailures",
    ],
}


EXPECTED_TECHNIQUE_NOISE_SCHEMA = [
    "Project",
    "NoisePercent",
    "RepetitionSeed",
    "Technique",
    "MeanAPFD",
    "MeanAPFDc",
    "SD_APFD",
    "SD_APFDc",
    "EvaluatedBuilds",
    "TotalRankedTests",
    "TotalFailures",
    "CleanMeanAPFD",
    "CleanMeanAPFDc",
    "DeltaAPFD_NoiseMinusClean",
    "DeltaAPFDc_NoiseMinusClean",
    "AbsoluteLossAPFD",
    "AbsoluteLossAPFDc",
    "PercentageLossAPFD",
    "PercentageLossAPFDc",
    "RetentionAPFD",
    "RetentionAPFDc",
]


for relative_path in top_level_paths:

    file_name = Path(
        relative_path
    ).name

    if file_name not in (
        EXPECTED_TOP_LEVEL_SCHEMAS
    ):

        raise RuntimeError(
            "Unexpected top-level reference file:\n"
            f"{file_name}"
        )

    reference_frame = read_table(
        REFERENCE_PACKAGE_DIR
        / relative_path
    )

    expected_columns = (
        EXPECTED_TOP_LEVEL_SCHEMAS[
            file_name
        ]
    )

    if list(
        reference_frame.columns
    ) != expected_columns:

        raise RuntimeError(
            "Reference top-level schema changed.\n"
            f"File: {file_name}\n"
            f"Expected: {expected_columns}\n"
            f"Actual: {list(reference_frame.columns)}"
        )


for relative_path in technique_noise_paths:

    reference_frame = read_table(
        REFERENCE_PACKAGE_DIR
        / relative_path
    )

    if list(
        reference_frame.columns
    ) != EXPECTED_TECHNIQUE_NOISE_SCHEMA:

        raise RuntimeError(
            "Reference technique-noise schema changed.\n"
            f"File: {relative_path}\n"
            f"Expected: {EXPECTED_TECHNIQUE_NOISE_SCHEMA}\n"
            f"Actual: {list(reference_frame.columns)}"
        )


print("\nReference schemas validated:")

print(
    "Top-level schemas:",
    len(
        EXPECTED_TOP_LEVEL_SCHEMAS
    ),
)

print(
    "Technique-noise schema columns:",
    len(
        EXPECTED_TECHNIQUE_NOISE_SCHEMA
    ),
)


# ------------------------------------------------------------
# 7. DERIVE FROZEN STATISTICAL CONVENTIONS
# ------------------------------------------------------------

reference_project_summary = pd.read_csv(
    REFERENCE_PACKAGE_DIR
    / "project_level_noise_technique_summary.csv",
    low_memory=False,
)


critical_values = []


for (
    mean_column,
    se_column,
    upper_column,
) in [
    (
        "MeanAPFD",
        "SE_APFD",
        "CI95UpperAPFD",
    ),
    (
        "MeanAPFDc",
        "SE_APFDc",
        "CI95UpperAPFDc",
    ),
]:

    means = pd.to_numeric(
        reference_project_summary[
            mean_column
        ],
        errors="coerce",
    )

    standard_errors = pd.to_numeric(
        reference_project_summary[
            se_column
        ],
        errors="coerce",
    )

    upper_bounds = pd.to_numeric(
        reference_project_summary[
            upper_column
        ],
        errors="coerce",
    )

    valid = (
        np.isfinite(means)
        & np.isfinite(standard_errors)
        & np.isfinite(upper_bounds)
        & standard_errors.gt(0)
    )

    calculated = (
        (
            upper_bounds[
                valid
            ]
            - means[
                valid
            ]
        )
        / standard_errors[
            valid
        ]
    )

    critical_values.extend(
        calculated.tolist()
    )


if critical_values:

    CI_CRITICAL_95 = float(
        np.median(
            critical_values
        )
    )

else:

    CI_CRITICAL_95 = (
        FALLBACK_CI_CRITICAL_95
    )


if not (
    1.5
    <= CI_CRITICAL_95
    <= 3.0
):

    raise AssertionError(
        "Derived CI critical value is invalid."
    )


reference_project_degradation = pd.read_csv(
    REFERENCE_PACKAGE_DIR
    / "project_level_degradation_summary.csv",
    low_memory=False,
)


reference_clean_retention = pd.to_numeric(
    reference_project_degradation.loc[
        pd.to_numeric(
            reference_project_degradation[
                "NoisePercent"
            ],
            errors="coerce",
        ).eq(0),
        "RetentionAPFD",
    ],
    errors="coerce",
).dropna()


if (
    not reference_clean_retention.empty
    and reference_clean_retention.median() > 10
):

    RETENTION_SCALE = 100.0

else:

    RETENTION_SCALE = 1.0


print("\nFrozen statistical conventions:")

print(
    "CI critical value:",
    CI_CRITICAL_95,
)

print(
    "Retention scale:",
    RETENTION_SCALE,
)


# ------------------------------------------------------------
# 8. LOAD AUDITED PROJECT 8 RESULTS
# ------------------------------------------------------------

project_run_source = pd.read_csv(
    PROJECT_RUN_INPUT,
    low_memory=False,
)

build_metrics_source = pd.read_csv(
    BUILD_METRICS_INPUT,
    low_memory=False,
)

noise_summary_source = pd.read_csv(
    NOISE_SUMMARY_INPUT,
    low_memory=False,
)

fit_times_source = pd.read_csv(
    FIT_TIMES_INPUT,
    low_memory=False,
)


for dataframe in [
    project_run_source,
    build_metrics_source,
    noise_summary_source,
    fit_times_source,
]:

    dataframe[
        "Project"
    ] = PROJECT_NAME


for column in [
    "NoisePercent",
    "RepetitionSeed",
    "MeanAPFD",
    "MeanAPFDc",
    "SD_APFD",
    "SD_APFDc",
    "EvaluatedBuilds",
    "TotalRankedTests",
    "TotalFailures",
]:

    if column in project_run_source.columns:

        project_run_source[
            column
        ] = pd.to_numeric(
            project_run_source[
                column
            ],
            errors="raise",
        )


for column in [
    "NoisePercent",
    "RepetitionSeed",
    "Build",
    "BuildOrder",
    "NumberOfTests",
    "NumberOfFailures",
    "APFD",
    "APFDc",
]:

    if column in build_metrics_source.columns:

        build_metrics_source[
            column
        ] = pd.to_numeric(
            build_metrics_source[
                column
            ],
            errors="raise",
        )


for column in [
    "NoisePercent",
    "RepetitionSeed",
    "FitSeconds",
    "TrainingRows",
    "TrainingFailures",
    "TrainingPasses",
    "ActiveFeatures",
]:

    if column in fit_times_source.columns:

        fit_times_source[
            column
        ] = pd.to_numeric(
            fit_times_source[
                column
            ],
            errors="raise",
        )


noise_percent_column = find_column(
    noise_summary_source,
    [
        "NoisePercent",
        "NoisePercentRequested",
    ],
)


noise_seed_column = find_column(
    noise_summary_source,
    [
        "RepetitionSeed",
        "Seed",
    ],
)


noise_summary_source[
    "NoisePercent"
] = pd.to_numeric(
    noise_summary_source[
        noise_percent_column
    ],
    errors="raise",
).astype(int)


noise_summary_source[
    "RepetitionSeed"
] = pd.to_numeric(
    noise_summary_source[
        noise_seed_column
    ],
    errors="raise",
).astype(int)


for dataframe in [
    project_run_source,
    build_metrics_source,
    fit_times_source,
]:

    dataframe[
        "NoisePercent"
    ] = dataframe[
        "NoisePercent"
    ].astype(int)

    dataframe[
        "RepetitionSeed"
    ] = dataframe[
        "RepetitionSeed"
    ].astype(int)


if len(
    project_run_source
) != EXPECTED_PROJECT_RUN_ROWS:

    raise AssertionError(
        "Project-run row count differs."
    )


if len(
    build_metrics_source
) != EXPECTED_BUILD_METRIC_ROWS:

    raise AssertionError(
        "Build-metric row count differs."
    )


if len(
    noise_summary_source
) != EXPECTED_CONDITIONS:

    raise AssertionError(
        "Noise-summary row count differs."
    )


if len(
    fit_times_source
) != EXPECTED_ML_FITS:

    raise AssertionError(
        "Fit-time row count differs."
    )


if set(
    project_run_source[
        "Technique"
    ].unique()
) != set(
    TECHNIQUES
):

    raise AssertionError(
        "Project-run technique set differs."
    )


metric_values = (
    project_run_source[
        [
            "MeanAPFD",
            "MeanAPFDc",
        ]
    ]
    .to_numpy(
        dtype=float
    )
)


if (
    not np.isfinite(
        metric_values
    ).all()
    or (
        (metric_values < 0)
        | (metric_values > 1)
    ).any()
):

    raise AssertionError(
        "Project-run APFD/APFDc values are invalid."
    )


print("\nAudited Project 8 inputs:")

print(
    "Project-run rows:",
    len(
        project_run_source
    ),
)

print(
    "Build-metric rows:",
    len(
        build_metrics_source
    ),
)

print(
    "Condition rows:",
    len(
        noise_summary_source
    ),
)

print(
    "ML fit rows:",
    len(
        fit_times_source
    ),
)


# ------------------------------------------------------------
# 9. PROJECT-RUN METRICS
# ------------------------------------------------------------

PROJECT_RUN_COLUMNS = (
    EXPECTED_TOP_LEVEL_SCHEMAS[
        "project_run_metrics_all.csv"
    ]
)


project_run_metrics = (
    project_run_source[
        PROJECT_RUN_COLUMNS
    ]
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 10. BUILD METRICS
# ------------------------------------------------------------

BUILD_METRIC_COLUMNS = (
    EXPECTED_TOP_LEVEL_SCHEMAS[
        "build_metrics_all.parquet"
    ]
)


build_metrics = (
    build_metrics_source[
        BUILD_METRIC_COLUMNS
    ]
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "BuildOrder",
            "Build",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 11. FIT TIMES
# ------------------------------------------------------------

FIT_TIME_COLUMNS = (
    EXPECTED_TOP_LEVEL_SCHEMAS[
        "fit_times_all.csv"
    ]
)


fit_times = (
    fit_times_source[
        FIT_TIME_COLUMNS
    ]
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 12. CONDITION SUMMARY
# ------------------------------------------------------------

reference_condition_summary = pd.read_csv(
    REFERENCE_PACKAGE_DIR
    / "condition_summary.csv",
    low_memory=False,
)


reference_condition_order = {
    (
        int(row.NoisePercent),
        int(row.RepetitionSeed),
    ):
        int(row.ConditionOrder)

    for row in (
        reference_condition_summary
        .itertuples(
            index=False
        )
    )
}


reference_condition_keys = {
    (
        int(row.NoisePercent),
        int(row.RepetitionSeed),
    ):
        transform_reference_identity(
            row.ConditionKey
        )

    for row in (
        reference_condition_summary
        .itertuples(
            index=False
        )
    )
}


training_rows_column = find_column(
    noise_summary_source,
    [
        "TrainingExecutionRows",
        "RawTrainingRows",
    ],
)


raw_flips_column = find_column(
    noise_summary_source,
    [
        "NumberFlipped",
        "FlippedRawRows",
        "RawRowsFlipped",
    ],
)


realised_noise_column = find_column(
    noise_summary_source,
    [
        "RealisedNoisePercent",
        "RealizedNoisePercent",
    ],
)


pass_to_failure_column = find_column(
    noise_summary_source,
    [
        "PassToFailure",
    ],
)


failure_to_pass_column = find_column(
    noise_summary_source,
    [
        "FailureToPass",
    ],
)


retained_changes_column = find_column(
    noise_summary_source,
    [
        "ModelTrainingLabelChanges",
        "RetainedLabelChanges",
        "LabelChanges",
    ],
)


condition_seconds_column = find_column(
    noise_summary_source,
    [
        "ElapsedSeconds",
        "ConditionSeconds",
    ],
)


condition_base = pd.DataFrame({
    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "NoisePercent":
        noise_summary_source[
            "NoisePercent"
        ].astype(int),

    "RepetitionSeed":
        noise_summary_source[
            "RepetitionSeed"
        ].astype(int),

    "TrainingExecutionRows":
        pd.to_numeric(
            noise_summary_source[
                training_rows_column
            ],
            errors="raise",
        ).astype(int),

    "RawRowsFlipped":
        pd.to_numeric(
            noise_summary_source[
                raw_flips_column
            ],
            errors="raise",
        ).astype(int),

    "RealisedNoisePercent":
        pd.to_numeric(
            noise_summary_source[
                realised_noise_column
            ],
            errors="raise",
        ).astype(float),

    "PassToFailure":
        pd.to_numeric(
            noise_summary_source[
                pass_to_failure_column
            ],
            errors="raise",
        ).astype(int),

    "FailureToPass":
        pd.to_numeric(
            noise_summary_source[
                failure_to_pass_column
            ],
            errors="raise",
        ).astype(int),

    "RetainedLabelChanges":
        pd.to_numeric(
            noise_summary_source[
                retained_changes_column
            ],
            errors="raise",
        ).astype(int),

    "ConditionSeconds":
        pd.to_numeric(
            noise_summary_source[
                condition_seconds_column
            ],
            errors="raise",
        ).astype(float),
})


condition_class_counts = (
    fit_times
    .groupby(
        [
            "NoisePercent",
            "RepetitionSeed",
        ],
        as_index=False,
    )
    .agg(
        TrainingFailures=(
            "TrainingFailures",
            "first",
        ),

        TrainingPasses=(
            "TrainingPasses",
            "first",
        ),

        FailureVariants=(
            "TrainingFailures",
            "nunique",
        ),

        PassVariants=(
            "TrainingPasses",
            "nunique",
        ),
    )
)


if (
    condition_class_counts[
        "FailureVariants"
    ].ne(1).any()
    or
    condition_class_counts[
        "PassVariants"
    ].ne(1).any()
):

    raise AssertionError(
        "ML models in a condition have inconsistent "
        "training class counts."
    )


condition_class_counts = (
    condition_class_counts.drop(
        columns=[
            "FailureVariants",
            "PassVariants",
        ]
    )
)


condition_summary = (
    condition_base
    .merge(
        condition_class_counts,
        on=[
            "NoisePercent",
            "RepetitionSeed",
        ],
        how="left",
        validate="one_to_one",
    )
)


condition_summary[
    "ConditionOrder"
] = [
    reference_condition_order[
        (
            int(noise_percent),
            int(repetition_seed),
        )
    ]
    for noise_percent, repetition_seed
    in zip(
        condition_summary[
            "NoisePercent"
        ],
        condition_summary[
            "RepetitionSeed"
        ],
    )
]


condition_summary[
    "ConditionKey"
] = [
    reference_condition_keys.get(
        (
            int(noise_percent),
            int(repetition_seed),
        ),
        condition_key(
            noise_percent,
            repetition_seed,
        ),
    )
    for noise_percent, repetition_seed
    in zip(
        condition_summary[
            "NoisePercent"
        ],
        condition_summary[
            "RepetitionSeed"
        ],
    )
]


condition_summary[
    "Status"
] = "COMPLETE"


condition_summary = (
    condition_summary[
        EXPECTED_TOP_LEVEL_SCHEMAS[
            "condition_summary.csv"
        ]
    ]
    .sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 13. NOISE-INJECTION SUMMARY
# ------------------------------------------------------------

noise_injection_summary = (
    condition_summary
    .groupby(
        "NoisePercent",
        as_index=False,
    )
    .agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        TrainingExecutionRowsPerSeed=(
            "TrainingExecutionRows",
            "first",
        ),

        MeanRawRowsFlipped=(
            "RawRowsFlipped",
            "mean",
        ),

        SDRawRowsFlipped=(
            "RawRowsFlipped",
            "std",
        ),

        MinimumRawRowsFlipped=(
            "RawRowsFlipped",
            "min",
        ),

        MaximumRawRowsFlipped=(
            "RawRowsFlipped",
            "max",
        ),

        MeanRealisedNoisePercent=(
            "RealisedNoisePercent",
            "mean",
        ),

        SDRealisedNoisePercent=(
            "RealisedNoisePercent",
            "std",
        ),

        MeanPassToFailure=(
            "PassToFailure",
            "mean",
        ),

        MeanFailureToPass=(
            "FailureToPass",
            "mean",
        ),

        MeanRetainedLabelChanges=(
            "RetainedLabelChanges",
            "mean",
        ),

        SDRetainedLabelChanges=(
            "RetainedLabelChanges",
            "std",
        ),

        MeanTrainingFailures=(
            "TrainingFailures",
            "mean",
        ),

        MeanTrainingPasses=(
            "TrainingPasses",
            "mean",
        ),

        MeanConditionSeconds=(
            "ConditionSeconds",
            "mean",
        ),

        SDConditionSeconds=(
            "ConditionSeconds",
            "std",
        ),
    )
)


noise_injection_summary = (
    noise_injection_summary[
        EXPECTED_TOP_LEVEL_SCHEMAS[
            "noise_injection_summary.csv"
        ]
    ]
    .sort_values(
        "NoisePercent",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 14. PROJECT-LEVEL TECHNIQUE SUMMARY
# ------------------------------------------------------------

project_level_summary = (
    project_run_metrics
    .groupby(
        [
            "Project",
            "NoisePercent",
            "Technique",
        ],
        as_index=False,
    )
    .agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        MeanAPFD=(
            "MeanAPFD",
            "mean",
        ),

        SD_APFD_AcrossSeeds=(
            "MeanAPFD",
            "std",
        ),

        MedianAPFD=(
            "MeanAPFD",
            "median",
        ),

        MinimumAPFD=(
            "MeanAPFD",
            "min",
        ),

        MaximumAPFD=(
            "MeanAPFD",
            "max",
        ),

        MeanAPFDc=(
            "MeanAPFDc",
            "mean",
        ),

        SD_APFDc_AcrossSeeds=(
            "MeanAPFDc",
            "std",
        ),

        MedianAPFDc=(
            "MeanAPFDc",
            "median",
        ),

        MinimumAPFDc=(
            "MeanAPFDc",
            "min",
        ),

        MaximumAPFDc=(
            "MeanAPFDc",
            "max",
        ),

        EvaluatedBuildsPerSeed=(
            "EvaluatedBuilds",
            "mean",
        ),

        TotalRankedTestsPerSeed=(
            "TotalRankedTests",
            "mean",
        ),

        TotalFailuresPerSeed=(
            "TotalFailures",
            "mean",
        ),
    )
)


project_level_summary[
    "SE_APFD"
] = (
    project_level_summary[
        "SD_APFD_AcrossSeeds"
    ]
    / np.sqrt(
        project_level_summary[
            "Seeds"
        ]
    )
)


project_level_summary[
    "CI95LowerAPFD"
] = (
    project_level_summary[
        "MeanAPFD"
    ]
    - CI_CRITICAL_95
    * project_level_summary[
        "SE_APFD"
    ]
)


project_level_summary[
    "CI95UpperAPFD"
] = (
    project_level_summary[
        "MeanAPFD"
    ]
    + CI_CRITICAL_95
    * project_level_summary[
        "SE_APFD"
    ]
)


project_level_summary[
    "SE_APFDc"
] = (
    project_level_summary[
        "SD_APFDc_AcrossSeeds"
    ]
    / np.sqrt(
        project_level_summary[
            "Seeds"
        ]
    )
)


project_level_summary[
    "CI95LowerAPFDc"
] = (
    project_level_summary[
        "MeanAPFDc"
    ]
    - CI_CRITICAL_95
    * project_level_summary[
        "SE_APFDc"
    ]
)


project_level_summary[
    "CI95UpperAPFDc"
] = (
    project_level_summary[
        "MeanAPFDc"
    ]
    + CI_CRITICAL_95
    * project_level_summary[
        "SE_APFDc"
    ]
)


project_level_summary = (
    project_level_summary[
        EXPECTED_TOP_LEVEL_SCHEMAS[
            "project_level_noise_technique_summary.csv"
        ]
    ]
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 15. PROJECT-LEVEL DEGRADATION SUMMARY
# ------------------------------------------------------------

clean_project_means = (
    project_level_summary[
        project_level_summary[
            "NoisePercent"
        ].eq(0)
    ][
        [
            "Technique",
            "MeanAPFD",
            "MeanAPFDc",
        ]
    ]
    .rename(
        columns={
            "MeanAPFD":
                "CleanProjectMeanAPFD",

            "MeanAPFDc":
                "CleanProjectMeanAPFDc",
        }
    )
)


project_level_degradation = (
    project_level_summary
    .merge(
        clean_project_means,
        on="Technique",
        how="left",
        validate="many_to_one",
    )
)


project_level_degradation[
    "DeltaAPFD_NoiseMinusClean"
] = (
    project_level_degradation[
        "MeanAPFD"
    ]
    - project_level_degradation[
        "CleanProjectMeanAPFD"
    ]
)


project_level_degradation[
    "DeltaAPFDc_NoiseMinusClean"
] = (
    project_level_degradation[
        "MeanAPFDc"
    ]
    - project_level_degradation[
        "CleanProjectMeanAPFDc"
    ]
)


project_level_degradation[
    "AbsoluteLossAPFD"
] = (
    project_level_degradation[
        "CleanProjectMeanAPFD"
    ]
    - project_level_degradation[
        "MeanAPFD"
    ]
)


project_level_degradation[
    "AbsoluteLossAPFDc"
] = (
    project_level_degradation[
        "CleanProjectMeanAPFDc"
    ]
    - project_level_degradation[
        "MeanAPFDc"
    ]
)


project_level_degradation[
    "PercentageLossAPFD"
] = np.where(
    project_level_degradation[
        "CleanProjectMeanAPFD"
    ].ne(0),

    100.0
    * project_level_degradation[
        "AbsoluteLossAPFD"
    ]
    / project_level_degradation[
        "CleanProjectMeanAPFD"
    ],

    np.nan,
)


project_level_degradation[
    "PercentageLossAPFDc"
] = np.where(
    project_level_degradation[
        "CleanProjectMeanAPFDc"
    ].ne(0),

    100.0
    * project_level_degradation[
        "AbsoluteLossAPFDc"
    ]
    / project_level_degradation[
        "CleanProjectMeanAPFDc"
    ],

    np.nan,
)


project_level_degradation[
    "RetentionAPFD"
] = np.where(
    project_level_degradation[
        "CleanProjectMeanAPFD"
    ].ne(0),

    RETENTION_SCALE
    * project_level_degradation[
        "MeanAPFD"
    ]
    / project_level_degradation[
        "CleanProjectMeanAPFD"
    ],

    np.nan,
)


project_level_degradation[
    "RetentionAPFDc"
] = np.where(
    project_level_degradation[
        "CleanProjectMeanAPFDc"
    ].ne(0),

    RETENTION_SCALE
    * project_level_degradation[
        "MeanAPFDc"
    ]
    / project_level_degradation[
        "CleanProjectMeanAPFDc"
    ],

    np.nan,
)


project_level_degradation = (
    project_level_degradation[
        EXPECTED_TOP_LEVEL_SCHEMAS[
            "project_level_degradation_summary.csv"
        ]
    ]
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 16. PER-SEED TECHNIQUE-NOISE DEGRADATION TABLE
# ------------------------------------------------------------

clean_seed_metrics = (
    project_run_metrics[
        project_run_metrics[
            "NoisePercent"
        ].eq(0)
    ][
        [
            "RepetitionSeed",
            "Technique",
            "MeanAPFD",
            "MeanAPFDc",
        ]
    ]
    .rename(
        columns={
            "MeanAPFD":
                "CleanMeanAPFD",

            "MeanAPFDc":
                "CleanMeanAPFDc",
        }
    )
)


if len(
    clean_seed_metrics
) != (
    len(SEEDS)
    * len(TECHNIQUES)
):

    raise AssertionError(
        "Clean per-seed reference table does not contain "
        "210 rows."
    )


technique_noise_all = (
    project_run_metrics
    .merge(
        clean_seed_metrics,
        on=[
            "RepetitionSeed",
            "Technique",
        ],
        how="left",
        validate="many_to_one",
    )
)


if (
    technique_noise_all[
        [
            "CleanMeanAPFD",
            "CleanMeanAPFDc",
        ]
    ]
    .isna()
    .any()
    .any()
):

    raise AssertionError(
        "Clean per-seed metric references are missing."
    )


technique_noise_all[
    "DeltaAPFD_NoiseMinusClean"
] = (
    technique_noise_all[
        "MeanAPFD"
    ]
    - technique_noise_all[
        "CleanMeanAPFD"
    ]
)


technique_noise_all[
    "DeltaAPFDc_NoiseMinusClean"
] = (
    technique_noise_all[
        "MeanAPFDc"
    ]
    - technique_noise_all[
        "CleanMeanAPFDc"
    ]
)


technique_noise_all[
    "AbsoluteLossAPFD"
] = (
    technique_noise_all[
        "CleanMeanAPFD"
    ]
    - technique_noise_all[
        "MeanAPFD"
    ]
)


technique_noise_all[
    "AbsoluteLossAPFDc"
] = (
    technique_noise_all[
        "CleanMeanAPFDc"
    ]
    - technique_noise_all[
        "MeanAPFDc"
    ]
)


technique_noise_all[
    "PercentageLossAPFD"
] = np.where(
    technique_noise_all[
        "CleanMeanAPFD"
    ].ne(0),

    100.0
    * technique_noise_all[
        "AbsoluteLossAPFD"
    ]
    / technique_noise_all[
        "CleanMeanAPFD"
    ],

    np.nan,
)


technique_noise_all[
    "PercentageLossAPFDc"
] = np.where(
    technique_noise_all[
        "CleanMeanAPFDc"
    ].ne(0),

    100.0
    * technique_noise_all[
        "AbsoluteLossAPFDc"
    ]
    / technique_noise_all[
        "CleanMeanAPFDc"
    ],

    np.nan,
)


technique_noise_all[
    "RetentionAPFD"
] = np.where(
    technique_noise_all[
        "CleanMeanAPFD"
    ].ne(0),

    RETENTION_SCALE
    * technique_noise_all[
        "MeanAPFD"
    ]
    / technique_noise_all[
        "CleanMeanAPFD"
    ],

    np.nan,
)


technique_noise_all[
    "RetentionAPFDc"
] = np.where(
    technique_noise_all[
        "CleanMeanAPFDc"
    ].ne(0),

    RETENTION_SCALE
    * technique_noise_all[
        "MeanAPFDc"
    ]
    / technique_noise_all[
        "CleanMeanAPFDc"
    ],

    np.nan,
)


technique_noise_all = (
    technique_noise_all[
        EXPECTED_TECHNIQUE_NOISE_SCHEMA
    ]
    .sort_values(
        [
            "NoisePercent",
            "Technique",
            "RepetitionSeed",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if len(
    technique_noise_all
) != EXPECTED_PROJECT_RUN_ROWS:

    raise AssertionError(
        "Technique-noise combined table does not contain "
        "1,890 rows."
    )


zero_noise_technique_rows = (
    technique_noise_all[
        technique_noise_all[
            "NoisePercent"
        ].eq(0)
    ]
)


if not np.allclose(
    zero_noise_technique_rows[
        "DeltaAPFD_NoiseMinusClean"
    ],
    0.0,
    rtol=0,
    atol=1e-12,
):

    raise AssertionError(
        "Zero-noise APFD deltas are not zero."
    )


if not np.allclose(
    zero_noise_technique_rows[
        "DeltaAPFDc_NoiseMinusClean"
    ],
    0.0,
    rtol=0,
    atol=1e-12,
):

    raise AssertionError(
        "Zero-noise APFDc deltas are not zero."
    )


# ------------------------------------------------------------
# 17. RECONSTRUCT PREDICTION SUMMARY
# ------------------------------------------------------------

print(
    "\nReconstructing 1,080 ML prediction summaries "
    "from raw Project 8 rankings..."
)


prediction_records = []


for seed_position, repetition_seed in enumerate(
    SEEDS,
    start=1,
):

    for noise_percent in NOISE_LEVELS:

        rankings_path = (
            RAW_RESULTS_ROOT
            / f"noise_{noise_percent:03d}"
            / f"seed_{repetition_seed:03d}"
            / "rankings.csv.gz"
        )


        if not rankings_path.exists():

            raise FileNotFoundError(
                "Raw rankings file is missing:\n"
                f"{rankings_path}"
            )


        ranking_scores = pd.read_csv(
            rankings_path,
            usecols=[
                "Technique",
                "Score",
            ],
            low_memory=False,
        )


        ranking_scores[
            "Score"
        ] = pd.to_numeric(
            ranking_scores[
                "Score"
            ],
            errors="coerce",
        )


        for technique in ML_TECHNIQUES:

            probabilities = (
                ranking_scores.loc[
                    ranking_scores[
                        "Technique"
                    ].eq(
                        technique
                    ),
                    "Score",
                ]
            )


            if len(
                probabilities
            ) != 2300:

                raise AssertionError(
                    "Unexpected prediction count.\n"
                    f"Noise: {noise_percent}\n"
                    f"Seed: {repetition_seed}\n"
                    f"Technique: {technique}\n"
                    f"Rows: {len(probabilities)}"
                )


            values = probabilities.to_numpy(
                dtype=float
            )


            if (
                probabilities.isna().any()
                or not np.isfinite(
                    values
                ).all()
            ):

                raise AssertionError(
                    "Prediction probabilities contain "
                    "missing or infinite values."
                )


            prediction_records.append({
                "Project":
                    PROJECT_NAME,

                "NoisePercent":
                    int(
                        noise_percent
                    ),

                "RepetitionSeed":
                    int(
                        repetition_seed
                    ),

                "Technique":
                    technique,

                "PredictionRows":
                    int(
                        len(
                            probabilities
                        )
                    ),

                "MinimumProbability":
                    float(
                        probabilities.min()
                    ),

                "MaximumProbability":
                    float(
                        probabilities.max()
                    ),

                "MeanProbability":
                    float(
                        probabilities.mean()
                    ),

                "UniqueProbabilities":
                    int(
                        probabilities.nunique()
                    ),
            })


    print(
        "Prediction summaries reconstructed for seed:",
        seed_position,
        "/",
        len(SEEDS),
    )


prediction_summary = (
    pd.DataFrame(
        prediction_records
    )[
        EXPECTED_TOP_LEVEL_SCHEMAS[
            "prediction_summary_all.csv"
        ]
    ]
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


if len(
    prediction_summary
) != EXPECTED_PREDICTION_ROWS:

    raise AssertionError(
        "Prediction-summary row count differs."
    )


# ------------------------------------------------------------
# 18. EXPLICIT TOP-LEVEL TABLES
# ------------------------------------------------------------

TOP_LEVEL_TABLES = {
    "build_metrics_all.parquet":
        build_metrics,

    "condition_summary.csv":
        condition_summary,

    "fit_times_all.csv":
        fit_times,

    "noise_injection_summary.csv":
        noise_injection_summary,

    "prediction_summary_all.csv":
        prediction_summary,

    "project_level_degradation_summary.csv":
        project_level_degradation,

    "project_level_noise_technique_summary.csv":
        project_level_summary,

    "project_run_metrics_all.csv":
        project_run_metrics,

    "project_run_metrics_all.parquet":
        project_run_metrics,
}


reference_top_level_names = {
    Path(relative_path).name
    for relative_path in top_level_paths
}


if (
    reference_top_level_names
    != set(
        TOP_LEVEL_TABLES.keys()
    )
):

    raise AssertionError(
        "Top-level package filenames differ.\n"
        f"Reference: {sorted(reference_top_level_names)}\n"
        f"Generated: {sorted(TOP_LEVEL_TABLES.keys())}"
    )


for file_name, table in (
    TOP_LEVEL_TABLES.items()
):

    expected_columns = (
        EXPECTED_TOP_LEVEL_SCHEMAS[
            file_name
        ]
    )

    if list(
        table.columns
    ) != expected_columns:

        raise RuntimeError(
            "Generated top-level schema differs.\n"
            f"File: {file_name}\n"
            f"Expected: {expected_columns}\n"
            f"Actual: {list(table.columns)}"
        )


# ------------------------------------------------------------
# 19. REMOVE ONLY INCOMPLETE STEP 11B OUTPUTS
# ------------------------------------------------------------

if STAGING_PACKAGE_DIR.exists():

    shutil.rmtree(
        STAGING_PACKAGE_DIR
    )


# Project 8 is not registered, therefore any existing package
# at this path is an incomplete Step 11B candidate.

if FINAL_PACKAGE_DIR.exists():

    shutil.rmtree(
        FINAL_PACKAGE_DIR
    )


STAGING_PACKAGE_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


# ------------------------------------------------------------
# 20. GENERATE ALL 72 PACKAGE FILES
# ------------------------------------------------------------

generation_records = []

technique_noise_combinations = set()
technique_noise_rows_written = 0


for relative_path in sorted(
    reference_relative_paths
):

    reference_path = (
        REFERENCE_PACKAGE_DIR
        / relative_path
    )

    reference_frame = read_table(
        reference_path
    )


    normalised_relative_path = (
        relative_path
        .replace("\\", "/")
    )


    if normalised_relative_path.startswith(
        "technique_noise/"
    ):

        file_name = Path(
            normalised_relative_path
        ).name.lower()


        parsed = re.fullmatch(
            (
                r"(.+?)"
                r"__noise_(\d{3})"
                r"\.(csv|csv\.gz|parquet)"
            ),
            file_name,
        )


        if parsed is None:

            raise RuntimeError(
                "Could not parse technique-noise filename:\n"
                f"{relative_path}"
            )


        technique_slug = (
            normalise_technique_slug(
                parsed.group(1)
            )
        )

        noise_percent = int(
            parsed.group(2)
        )


        if technique_slug not in (
            TECHNIQUE_SLUG_TO_NAME
        ):

            raise RuntimeError(
                "Unknown technique slug:\n"
                f"{technique_slug}"
            )


        technique = (
            TECHNIQUE_SLUG_TO_NAME[
                technique_slug
            ]
        )


        generated_frame = (
            technique_noise_all[
                technique_noise_all[
                    "Technique"
                ].eq(
                    technique
                )
                &
                technique_noise_all[
                    "NoisePercent"
                ].eq(
                    noise_percent
                )
            ]
            .sort_values(
                "RepetitionSeed",
                kind="mergesort",
            )
            .reset_index(drop=True)
            .copy()
        )


        if len(
            generated_frame
        ) != 30:

            raise AssertionError(
                "Technique-noise file does not contain "
                "30 seed rows.\n"
                f"Technique: {technique}\n"
                f"Noise: {noise_percent}\n"
                f"Rows: {len(generated_frame)}"
            )


        source_name = (
            "technique_noise_per_seed_degradation"
        )

        technique_noise_combinations.add(
            (
                technique,
                noise_percent,
            )
        )

        technique_noise_rows_written += (
            len(
                generated_frame
            )
        )


    else:

        file_name = Path(
            normalised_relative_path
        ).name


        if file_name not in (
            TOP_LEVEL_TABLES
        ):

            raise RuntimeError(
                "No explicit Project 8 table exists for:\n"
                f"{file_name}"
            )


        generated_frame = (
            TOP_LEVEL_TABLES[
                file_name
            ].copy()
        )

        source_name = file_name


    generated_frame = (
        cast_like_reference(
            generated_dataframe=(
                generated_frame
            ),

            reference_dataframe=(
                reference_frame
            ),

            relative_path=(
                relative_path
            ),
        )
    )


    output_path = (
        STAGING_PACKAGE_DIR
        / relative_path
    )


    write_table(
        output_path,
        generated_frame,
    )


    generation_records.append({
        "RelativePath":
            relative_path,

        "SourceTable":
            source_name,

        "ReferenceRows":
            int(
                len(
                    reference_frame
                )
            ),

        "GeneratedRows":
            int(
                len(
                    generated_frame
                )
            ),

        "Columns":
            int(
                len(
                    generated_frame.columns
                )
            ),

        "OutputPath":
            str(
                output_path
            ),
    })


generation_map = pd.DataFrame(
    generation_records
)


# ------------------------------------------------------------
# 21. VALIDATE STAGING PACKAGE
# ------------------------------------------------------------

staging_files = sorted([
    path
    for path in STAGING_PACKAGE_DIR.rglob("*")
    if path.is_file()
])


staging_relative_paths = {
    path.relative_to(
        STAGING_PACKAGE_DIR
    ).as_posix()
    for path in staging_files
}


expected_relative_paths = set(
    reference_relative_paths
)


missing_package_files = sorted(
    expected_relative_paths
    - staging_relative_paths
)


unexpected_package_files = sorted(
    staging_relative_paths
    - expected_relative_paths
)


expected_technique_noise_combinations = {
    (
        technique,
        noise_percent,
    )
    for technique in TECHNIQUES
    for noise_percent in NOISE_LEVELS
}


expected_top_level_rows = {
    "build_metrics_all.parquet":
        EXPECTED_BUILD_METRIC_ROWS,

    "condition_summary.csv":
        EXPECTED_CONDITIONS,

    "fit_times_all.csv":
        EXPECTED_ML_FITS,

    "noise_injection_summary.csv":
        9,

    "prediction_summary_all.csv":
        EXPECTED_PREDICTION_ROWS,

    "project_level_degradation_summary.csv":
        63,

    "project_level_noise_technique_summary.csv":
        63,

    "project_run_metrics_all.csv":
        EXPECTED_PROJECT_RUN_ROWS,

    "project_run_metrics_all.parquet":
        EXPECTED_PROJECT_RUN_ROWS,
}


readback_failures = []
schema_mismatches = []
row_count_mismatches = []
project7_identity_files = []
invalid_metric_files = []
reference_hash_matches = []


for relative_path in sorted(
    reference_relative_paths
):

    generated_path = (
        STAGING_PACKAGE_DIR
        / relative_path
    )

    reference_path = (
        REFERENCE_PACKAGE_DIR
        / relative_path
    )


    try:

        generated_frame = read_table(
            generated_path
        )

        reference_frame = read_table(
            reference_path
        )

    except Exception as error:

        readback_failures.append({
            "RelativePath":
                relative_path,

            "Error":
                str(error),
        })

        continue


    if list(
        generated_frame.columns
    ) != list(
        reference_frame.columns
    ):

        schema_mismatches.append({
            "RelativePath":
                relative_path,

            "ExpectedColumns":
                list(
                    reference_frame.columns
                ),

            "ActualColumns":
                list(
                    generated_frame.columns
                ),
        })


    if (
        relative_path
        .replace("\\", "/")
        .startswith(
            "technique_noise/"
        )
    ):

        expected_rows = 30

    else:

        expected_rows = (
            expected_top_level_rows[
                Path(
                    relative_path
                ).name
            ]
        )


    if len(
        generated_frame
    ) != expected_rows:

        row_count_mismatches.append({
            "RelativePath":
                relative_path,

            "ExpectedRows":
                expected_rows,

            "ActualRows":
                len(
                    generated_frame
                ),
        })


    contains_project7_identity = False


    for column in generated_frame.columns:

        if (
            pd.api.types.is_object_dtype(
                generated_frame[column]
            )
            or pd.api.types.is_string_dtype(
                generated_frame[column]
            )
        ):

            values = (
                generated_frame[
                    column
                ]
                .fillna("")
                .astype(str)
                .str.lower()
            )


            if (
                values.str.contains(
                    "compevol",
                    regex=False,
                ).any()
                or
                values.str.contains(
                    "beast2",
                    regex=False,
                ).any()
            ):

                contains_project7_identity = True
                break


    if contains_project7_identity:

        project7_identity_files.append(
            relative_path
        )


    direct_metric_columns = {
        "apfd",
        "apfdc",
        "meanapfd",
        "meanapfdc",
        "medianapfd",
        "medianapfdc",
        "minimumapfd",
        "minimumapfdc",
        "maximumapfd",
        "maximumapfdc",
        "cleanmeanapfd",
        "cleanmeanapfdc",
        "cleanprojectmeanapfd",
        "cleanprojectmeanapfdc",
    }


    for column in generated_frame.columns:

        if (
            normalise_column_name(
                column
            )
            not in direct_metric_columns
        ):

            continue


        values = pd.to_numeric(
            generated_frame[
                column
            ],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )


        finite_values = values[
            np.isfinite(
                values
            )
        ]


        if (
            len(finite_values) > 0
            and (
                (finite_values < 0)
                | (finite_values > 1)
            ).any()
        ):

            invalid_metric_files.append({
                "RelativePath":
                    relative_path,

                "Column":
                    column,
            })


    if (
        calculate_sha256(
            generated_path
        )
        ==
        calculate_sha256(
            reference_path
        )
    ):

        reference_hash_matches.append(
            relative_path
        )


if len(
    staging_files
) != EXPECTED_PACKAGE_FILES:

    raise AssertionError(
        "Staging package file count differs.\n"
        f"Expected: {EXPECTED_PACKAGE_FILES}\n"
        f"Actual: {len(staging_files)}"
    )


if missing_package_files:

    raise AssertionError(
        "Generated package is missing files:\n"
        + "\n".join(
            missing_package_files
        )
    )


if unexpected_package_files:

    raise AssertionError(
        "Generated package contains unexpected files:\n"
        + "\n".join(
            unexpected_package_files
        )
    )


if (
    technique_noise_combinations
    != expected_technique_noise_combinations
):

    raise AssertionError(
        "Technique-noise combinations differ.\n"
        f"Missing: "
        f"{sorted(expected_technique_noise_combinations - technique_noise_combinations)}\n"
        f"Unexpected: "
        f"{sorted(technique_noise_combinations - expected_technique_noise_combinations)}"
    )


if (
    technique_noise_rows_written
    != EXPECTED_PROJECT_RUN_ROWS
):

    raise AssertionError(
        "Technique-noise files do not contain exactly "
        "1,890 rows."
    )


if readback_failures:

    display(
        pd.DataFrame(
            readback_failures
        )
    )

    raise RuntimeError(
        "At least one generated package file could not "
        "be read back."
    )


if schema_mismatches:

    display(
        pd.DataFrame(
            schema_mismatches
        )
    )

    raise RuntimeError(
        "Generated package schemas differ."
    )


if row_count_mismatches:

    display(
        pd.DataFrame(
            row_count_mismatches
        )
    )

    raise RuntimeError(
        "Generated package row counts differ."
    )


if project7_identity_files:

    raise AssertionError(
        "Generated package contains Project 7 identity:\n"
        + "\n".join(
            project7_identity_files
        )
    )


if invalid_metric_files:

    display(
        pd.DataFrame(
            invalid_metric_files
        )
    )

    raise AssertionError(
        "Generated package contains invalid APFD/APFDc "
        "values."
    )


if reference_hash_matches:

    raise AssertionError(
        "A generated Project 8 file is byte-identical to "
        "the corresponding Project 7 file:\n"
        + "\n".join(
            reference_hash_matches
        )
    )


# ------------------------------------------------------------
# 22. CROSS-FORMAT EQUIVALENCE
# ------------------------------------------------------------

project_run_csv_readback = pd.read_csv(
    STAGING_PACKAGE_DIR
    / "project_run_metrics_all.csv",
    low_memory=False,
)


project_run_parquet_readback = pd.read_parquet(
    STAGING_PACKAGE_DIR
    / "project_run_metrics_all.parquet"
)


pd.testing.assert_frame_equal(
    project_run_csv_readback,
    project_run_parquet_readback,
    check_dtype=False,
    check_like=False,
)


# ------------------------------------------------------------
# 23. PROMOTE STAGING PACKAGE
# ------------------------------------------------------------

STAGING_PACKAGE_DIR.replace(
    FINAL_PACKAGE_DIR
)


# ------------------------------------------------------------
# 24. FINAL INVENTORY AND ROOT HASH
# ------------------------------------------------------------

final_package_files = sorted([
    path
    for path in FINAL_PACKAGE_DIR.rglob("*")
    if path.is_file()
])


inventory_records = []


for path in final_package_files:

    inventory_records.append({
        "RelativePath":
            path.relative_to(
                FINAL_PACKAGE_DIR
            ).as_posix(),

        "SizeBytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            calculate_sha256(
                path
            ),
    })


final_package_inventory = pd.DataFrame(
    inventory_records
)


final_package_bytes = int(
    final_package_inventory[
        "SizeBytes"
    ].sum()
)


final_package_root_sha256 = (
    root_inventory_hash(
        final_package_inventory
    )
)


final_relative_paths = set(
    final_package_inventory[
        "RelativePath"
    ]
)


final_missing_files = sorted(
    expected_relative_paths
    - final_relative_paths
)


final_unexpected_files = sorted(
    final_relative_paths
    - expected_relative_paths
)


top_level_file_count = int(
    sum(
        1
        for relative_path
        in final_relative_paths
        if not relative_path.startswith(
            "technique_noise/"
        )
    )
)


technique_noise_file_count = int(
    sum(
        1
        for relative_path
        in final_relative_paths
        if relative_path.startswith(
            "technique_noise/"
        )
    )
)


# ------------------------------------------------------------
# 25. FINAL VALIDATION
# ------------------------------------------------------------

validation_records = [
    {
        "Check":
            "Project 8 Step 10 passed",

        "Expected":
            EXPECTED_STEP10_STATUS,

        "Actual":
            step10_status[
                "Status"
            ],

        "Pass":
            step10_status[
                "Status"
            ]
            == EXPECTED_STEP10_STATUS,
    },

    {
        "Check":
            "Final package files",

        "Expected":
            EXPECTED_PACKAGE_FILES,

        "Actual":
            len(
                final_package_inventory
            ),

        "Pass":
            len(
                final_package_inventory
            )
            == EXPECTED_PACKAGE_FILES,
    },

    {
        "Check":
            "Top-level files",

        "Expected":
            EXPECTED_TOP_LEVEL_FILES,

        "Actual":
            top_level_file_count,

        "Pass":
            top_level_file_count
            == EXPECTED_TOP_LEVEL_FILES,
    },

    {
        "Check":
            "Technique-noise files",

        "Expected":
            EXPECTED_TECHNIQUE_NOISE_FILES,

        "Actual":
            technique_noise_file_count,

        "Pass":
            technique_noise_file_count
            == EXPECTED_TECHNIQUE_NOISE_FILES,
    },

    {
        "Check":
            "Technique-noise combinations",

        "Expected":
            EXPECTED_TECHNIQUE_NOISE_FILES,

        "Actual":
            len(
                technique_noise_combinations
            ),

        "Pass":
            technique_noise_combinations
            == expected_technique_noise_combinations,
    },

    {
        "Check":
            "Technique-noise rows",

        "Expected":
            EXPECTED_PROJECT_RUN_ROWS,

        "Actual":
            technique_noise_rows_written,

        "Pass":
            technique_noise_rows_written
            == EXPECTED_PROJECT_RUN_ROWS,
    },

    {
        "Check":
            "Build-metric rows",

        "Expected":
            EXPECTED_BUILD_METRIC_ROWS,

        "Actual":
            len(
                build_metrics
            ),

        "Pass":
            len(
                build_metrics
            )
            == EXPECTED_BUILD_METRIC_ROWS,
    },

    {
        "Check":
            "Condition-summary rows",

        "Expected":
            EXPECTED_CONDITIONS,

        "Actual":
            len(
                condition_summary
            ),

        "Pass":
            len(
                condition_summary
            )
            == EXPECTED_CONDITIONS,
    },

    {
        "Check":
            "Prediction-summary rows",

        "Expected":
            EXPECTED_PREDICTION_ROWS,

        "Actual":
            len(
                prediction_summary
            ),

        "Pass":
            len(
                prediction_summary
            )
            == EXPECTED_PREDICTION_ROWS,
    },

    {
        "Check":
            "Project-level summary rows",

        "Expected":
            63,

        "Actual":
            len(
                project_level_summary
            ),

        "Pass":
            len(
                project_level_summary
            )
            == 63,
    },

    {
        "Check":
            "Project-level degradation rows",

        "Expected":
            63,

        "Actual":
            len(
                project_level_degradation
            ),

        "Pass":
            len(
                project_level_degradation
            )
            == 63,
    },

    {
        "Check":
            "Missing package files",

        "Expected":
            0,

        "Actual":
            len(
                final_missing_files
            ),

        "Pass":
            len(
                final_missing_files
            )
            == 0,
    },

    {
        "Check":
            "Unexpected package files",

        "Expected":
            0,

        "Actual":
            len(
                final_unexpected_files
            ),

        "Pass":
            len(
                final_unexpected_files
            )
            == 0,
    },

    {
        "Check":
            "Schema mismatches",

        "Expected":
            0,

        "Actual":
            len(
                schema_mismatches
            ),

        "Pass":
            len(
                schema_mismatches
            )
            == 0,
    },

    {
        "Check":
            "Row-count mismatches",

        "Expected":
            0,

        "Actual":
            len(
                row_count_mismatches
            ),

        "Pass":
            len(
                row_count_mismatches
            )
            == 0,
    },

    {
        "Check":
            "Readback failures",

        "Expected":
            0,

        "Actual":
            len(
                readback_failures
            ),

        "Pass":
            len(
                readback_failures
            )
            == 0,
    },

    {
        "Check":
            "Project 7 identity files",

        "Expected":
            0,

        "Actual":
            len(
                project7_identity_files
            ),

        "Pass":
            len(
                project7_identity_files
            )
            == 0,
    },

    {
        "Check":
            "Invalid APFD/APFDc files",

        "Expected":
            0,

        "Actual":
            len(
                invalid_metric_files
            ),

        "Pass":
            len(
                invalid_metric_files
            )
            == 0,
    },

    {
        "Check":
            "Files identical to Project 7",

        "Expected":
            0,

        "Actual":
            len(
                reference_hash_matches
            ),

        "Pass":
            len(
                reference_hash_matches
            )
            == 0,
    },

    {
        "Check":
            "Project 8 registry rows",

        "Expected":
            0,

        "Actual":
            len(
                project8_registry_rows
            ),

        "Pass":
            len(
                project8_registry_rows
            )
            == 0,
    },
]


final_package_validation = pd.DataFrame(
    validation_records
)


failed_checks = (
    final_package_validation[
        ~final_package_validation[
            "Pass"
        ]
    ]
    .copy()
)


print("\nFinal package validation:")

display(
    final_package_validation
)


if not failed_checks.empty:

    print("\nFailed checks:")

    display(
        failed_checks
    )

    raise RuntimeError(
        "PROJECT 8 STEP 11B V4 DID NOT PASS.\n"
        "Do not modify the completion registry."
    )


# ------------------------------------------------------------
# 26. WRITE PACKAGE AUDIT OUTPUTS
# ------------------------------------------------------------

FINAL_PACKAGE_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    FINAL_PACKAGE_INVENTORY_PATH,
    final_package_inventory,
)


atomic_write_csv(
    FINAL_PACKAGE_VALIDATION_PATH,
    final_package_validation,
)


atomic_write_csv(
    FINAL_PACKAGE_GENERATION_MAP_PATH,
    generation_map,
)


step11b_status_text = (
    "PASS_PROJECT_8_FINAL_72_FILE_PACKAGE_GENERATED_AND_VALIDATED"
)


final_package_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "GeneratedAtUTC":
        GENERATED_AT_UTC,

    "RawResultFreeze": {
        "Files":
            EXPECTED_RAW_FILES,

        "Bytes":
            EXPECTED_RAW_BYTES,

        "RootSHA256":
            EXPECTED_RAW_ROOT_SHA256,
    },

    "ExperimentDimensions": {
        "Conditions":
            EXPECTED_CONDITIONS,

        "MLFits":
            EXPECTED_ML_FITS,

        "PredictionSummaries":
            EXPECTED_PREDICTION_ROWS,

        "ProjectRunRows":
            EXPECTED_PROJECT_RUN_ROWS,

        "BuildMetricRows":
            EXPECTED_BUILD_METRIC_ROWS,

        "NoiseLevels":
            len(
                NOISE_LEVELS
            ),

        "Seeds":
            len(
                SEEDS
            ),

        "Techniques":
            len(
                TECHNIQUES
            ),
    },

    "StatisticalConvention": {
        "CI95CriticalValue":
            CI_CRITICAL_95,

        "RetentionScale":
            RETENTION_SCALE,

        "DerivedFromFrozenReferencePackage":
            True,
    },

    "FinalPackage": {
        "Directory":
            str(
                FINAL_PACKAGE_DIR
            ),

        "Files":
            len(
                final_package_inventory
            ),

        "TopLevelFiles":
            top_level_file_count,

        "TechniqueNoiseFiles":
            technique_noise_file_count,

        "Bytes":
            final_package_bytes,

        "RootSHA256":
            final_package_root_sha256,

        "MissingFiles":
            len(
                final_missing_files
            ),

        "UnexpectedFiles":
            len(
                final_unexpected_files
            ),
    },

    "Validation": {
        "Checks":
            len(
                final_package_validation
            ),

        "FailedChecks":
            len(
                failed_checks
            ),

        "SchemaMismatches":
            len(
                schema_mismatches
            ),

        "RowCountMismatches":
            len(
                row_count_mismatches
            ),

        "ReadbackFailures":
            len(
                readback_failures
            ),

        "Project7IdentityFiles":
            len(
                project7_identity_files
            ),

        "ReferenceHashMatches":
            len(
                reference_hash_matches
            ),

        "InvalidMetricFiles":
            len(
                invalid_metric_files
            ),
    },

    "CompletionRegistryModified":
        False,

    "Projects1To7Modified":
        False,

    "Status":
        step11b_status_text,
}


atomic_write_json(
    FINAL_PACKAGE_REPORT_PATH,
    final_package_report,
)


step11b_status = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        step11b_status_text,

    "RawFiles":
        EXPECTED_RAW_FILES,

    "RawBytes":
        EXPECTED_RAW_BYTES,

    "RawRootSHA256":
        EXPECTED_RAW_ROOT_SHA256,

    "PackageFiles":
        len(
            final_package_inventory
        ),

    "PackageBytes":
        final_package_bytes,

    "PackageRootSHA256":
        final_package_root_sha256,

    "TopLevelFiles":
        top_level_file_count,

    "TechniqueNoiseFiles":
        technique_noise_file_count,

    "MissingPackageFiles":
        len(
            final_missing_files
        ),

    "UnexpectedPackageFiles":
        len(
            final_unexpected_files
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletionRegistryModified":
        False,

    "CompletedAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP11B_STATUS_PATH,
    step11b_status,
)


audit_outputs = [
    FINAL_PACKAGE_INVENTORY_PATH,
    FINAL_PACKAGE_VALIDATION_PATH,
    FINAL_PACKAGE_GENERATION_MAP_PATH,
    FINAL_PACKAGE_REPORT_PATH,
    STEP11B_STATUS_PATH,
]


missing_audit_outputs = [
    str(path)
    for path in audit_outputs
    if not path.exists()
]


if missing_audit_outputs:

    raise RuntimeError(
        "Step 11B V4 audit outputs are missing:\n"
        + "\n".join(
            missing_audit_outputs
        )
    )


# ------------------------------------------------------------
# 27. DISPLAY PROJECT 8 RESULT SUMMARY
# ------------------------------------------------------------

result_display = (
    project_level_summary[
        [
            "NoisePercent",
            "Technique",
            "Seeds",
            "MeanAPFD",
            "SD_APFD_AcrossSeeds",
            "MeanAPFDc",
            "SD_APFDc_AcrossSeeds",
        ]
    ]
    .sort_values(
        [
            "NoisePercent",
            "MeanAPFDc",
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


print(
    "\nProject 8 APFD/APFDc summary included "
    "in the final package:"
)

display(
    result_display.round(6)
)


# ------------------------------------------------------------
# 28. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("=== PROJECT 8 STEP 11B V4 RESULT ===")
print("=" * 100)

print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nAudited source results:")

print(
    "Conditions:",
    EXPECTED_CONDITIONS,
)

print(
    "ML fits:",
    EXPECTED_ML_FITS,
)

print(
    "Prediction summaries:",
    len(
        prediction_summary
    ),
)

print(
    "Project-run rows:",
    len(
        project_run_metrics
    ),
)

print(
    "Build-metric rows:",
    len(
        build_metrics
    ),
)

print(
    "Techniques:",
    len(
        TECHNIQUES
    ),
)


print("\nFinal package:")

print(
    "Directory:",
    FINAL_PACKAGE_DIR,
)

print(
    "Files:",
    len(
        final_package_inventory
    ),
)

print(
    "Top-level files:",
    top_level_file_count,
)

print(
    "Technique-noise files:",
    technique_noise_file_count,
)

print(
    "Package bytes:",
    final_package_bytes,
)

print(
    "Package root SHA-256:",
    final_package_root_sha256,
)

print(
    "Missing files:",
    len(
        final_missing_files
    ),
)

print(
    "Unexpected files:",
    len(
        final_unexpected_files
    ),
)


print("\nContent validation:")

print(
    "Technique-noise combinations:",
    len(
        technique_noise_combinations
    ),
)

print(
    "Technique-noise rows:",
    technique_noise_rows_written,
)

print(
    "Schema mismatches:",
    len(
        schema_mismatches
    ),
)

print(
    "Row-count mismatches:",
    len(
        row_count_mismatches
    ),
)

print(
    "Readback failures:",
    len(
        readback_failures
    ),
)

print(
    "Project 7 identity files:",
    len(
        project7_identity_files
    ),
)

print(
    "Files identical to Project 7:",
    len(
        reference_hash_matches
    ),
)

print(
    "Invalid APFD/APFDc files:",
    len(
        invalid_metric_files
    ),
)


print("\nValidation:")

print(
    "Checks:",
    len(
        final_package_validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_checks
    ),
)


print("\nAudit outputs:")

for output_path in audit_outputs:

    print(output_path)


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–7 modified:")
print(0)


print(
    "\nSTATUS:",
    step11b_status_text,
)

print("=" * 100)

=== PROJECT 8 STEP 11B V4: GENERATE FINAL 72-FILE RESULT PACKAGE ===

Input validation:
Step 10 status: PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED
Raw files: 2160
Raw bytes: 179766494
Raw root SHA-256: 19ae21c5d8524c358796e28f23577fbf03bfd31f7d4b530372c6b4d8a492887c
Project 8 registry rows: 0

Reference package blueprint:
Directory: /content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/CompEvol__beast2/beast2_30_seed_final
Inventory rows: 72
Top-level files: 9
Technique-noise files: 63

Reference schemas validated:
Top-level schemas: 9
Technique-noise schema columns: 21

Frozen statistical conventions:
CI critical value: 2.045229642132711
Retention scale: 1.0

Audited Project 8 inputs:
Project-run rows: 1890
Build-metric rows: 30240
Condition rows: 270
ML fit rows: 1080

Reconstructing 1,080 ML prediction summaries from raw Project 8 rankings...
Prediction summaries reconstructed for seed: 1 / 30
Prediction summaries reconstructed for seed: 2 / 30
Prediction summaries reco

,Check,Expected,Actual,Pass
0,Project 8 Step 10 passed,PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED,PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED,True
1,Final package files,72,72,True
2,Top-level files,9,9,True
3,Technique-noise files,63,63,True
4,Technique-noise combinations,63,63,True
5,Technique-noise rows,1890,1890,True
6,Build-metric rows,30240,30240,True
7,Condition-summary rows,270,270,True
8,Prediction-summary rows,1080,1080,True
9,Project-level summary rows,63,63,True



Project 8 APFD/APFDc summary included in the final package:


,NoisePercent,Technique,Seeds,MeanAPFD,SD_APFD_AcrossSeeds,MeanAPFDc,SD_APFDc_AcrossSeeds
0,0,QTF-Avg,30,0.096178,0.000000,0.853237,0.000000
1,0,LatestFail,30,0.934893,0.000000,0.852401,0.000000
2,0,NaiveBayes,30,0.709531,0.000000,0.798309,0.000000
3,0,RandomForest,30,0.945877,0.009719,0.790071,0.016804
4,0,XGBoost,30,0.914499,0.000000,0.731460,0.000000
...,...,...,...,...,...,...,...
58,50,NaiveBayes,30,0.443993,0.264067,0.587131,0.243573
59,50,Random,30,0.483751,0.069122,0.487324,0.088599
60,50,XGBoost,30,0.573385,0.189864,0.480360,0.168685
61,50,RandomForest,30,0.495820,0.121164,0.472036,0.134253




=== PROJECT 8 STEP 11B V4 RESULT ===

Project identity:
Project number: 8
Project: optimatika@ojAlgo
Project slug: optimatika__ojAlgo

Audited source results:
Conditions: 270
ML fits: 1080
Prediction summaries: 1080
Project-run rows: 1890
Build-metric rows: 30240
Techniques: 7

Final package:
Directory: /content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_30_seed_final
Files: 72
Top-level files: 9
Technique-noise files: 63
Package bytes: 1242844
Package root SHA-256: 1e27d6f57526d3ce819b7358867c0c58411cc4eaddcdcb2e5576b2282a6cd607
Missing files: 0
Unexpected files: 0

Content validation:
Technique-noise combinations: 63
Technique-noise rows: 1890
Schema mismatches: 0
Row-count mismatches: 0
Readback failures: 0
Project 7 identity files: 0
Files identical to Project 7: 0
Invalid APFD/APFDc files: 0

Validation:
Checks: 20
Failed checks: 0

Audit outputs:
/content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_final_package

In [ ]:
# ============================================================
# PROJECT 8 — STEP 11C
# INDEPENDENT FINAL-PACKAGE AUDIT AND FREEZE
#
# PROJECT: optimatika@ojAlgo
#
# This cell independently verifies the package generated by
# Step 11B. It does not regenerate or modify package contents.
#
# It verifies:
# - Step 10 and Step 11B passed
# - exactly 72 final-package files
# - exactly 9 top-level files
# - exactly 63 technique-noise files
# - all file sizes and SHA-256 hashes
# - package-root SHA-256
# - CSV/Parquet equivalence
# - all schemas and row counts
# - all seven techniques, nine noise levels and 30 seeds
# - APFD/APFDc values against audited source results
# - clean-reference and degradation calculations
# - project-level summary calculations
# - no Project 7 identity contamination
# - completion registry remains unchanged
#
# This cell does NOT:
# - rerun models
# - regenerate package files
# - alter raw results
# - modify Projects 1–7
# - update the completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import re

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 8
PROJECT_NAME = "optimatika@ojAlgo"
PROJECT_SLUG = "optimatika__ojAlgo"
PROJECT_SHORT_NAME = "ojalgo"

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

SEEDS = list(range(1, 31))

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)

TECHNIQUE_SLUG_TO_NAME = {
    "randomforest":
        "RandomForest",

    "xgboost":
        "XGBoost",

    "lightgbm":
        "LightGBM",

    "naivebayes":
        "NaiveBayes",

    "random":
        "Random",

    "latestfail":
        "LatestFail",

    "qtfavg":
        "QTF-Avg",
}

EXPECTED_CONDITIONS = 270
EXPECTED_ML_FITS = 1080
EXPECTED_PROJECT_RUN_ROWS = 1890
EXPECTED_BUILD_METRIC_ROWS = 30240
EXPECTED_PREDICTION_ROWS = 1080

EXPECTED_PACKAGE_FILES = 72
EXPECTED_TOP_LEVEL_FILES = 9
EXPECTED_TECHNIQUE_NOISE_FILES = 63

EXPECTED_RAW_FILES = 2160
EXPECTED_RAW_BYTES = 179766494

EXPECTED_RAW_ROOT_SHA256 = (
    "19ae21c5d8524c358796e28f23577fbf03bfd31f7d4b530372c6b4d8a492887c"
)

EXPECTED_PACKAGE_BYTES = 1242844

EXPECTED_PACKAGE_ROOT_SHA256 = (
    "1e27d6f57526d3ce819b7358867c0c58411cc4eaddcdcb2e5576b2282a6cd607"
)

EXPECTED_STEP10_STATUS = (
    "PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED"
)

EXPECTED_STEP11B_STATUS = (
    "PASS_PROJECT_8_FINAL_72_FILE_PACKAGE_GENERATED_AND_VALIDATED"
)

STEP11C_PASS_STATUS = (
    "PASS_PROJECT_8_FINAL_PACKAGE_INDEPENDENTLY_AUDITED_AND_FROZEN"
)

FLOAT_ATOL = 1e-12


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)

RAW_AUDIT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_raw_audit"
)

FINAL_PACKAGE_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_30_seed_final"
)

FINAL_PACKAGE_AUDIT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_audit"
)

STEP10_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step10_status.json"
)

STEP11B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step11b_status.json"
)

STEP11B_INVENTORY_PATH = (
    FINAL_PACKAGE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_file_inventory_sha256.csv"
)

PROJECT_RUN_SOURCE_PATH = (
    RAW_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_all_project_run_metrics.csv"
)

BUILD_METRICS_SOURCE_PATH = (
    RAW_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_all_build_metrics.csv.gz"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)


# Step 11C outputs

FREEZE_AUDIT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_final_freeze_audit"
)

FREEZE_INVENTORY_PATH = (
    FREEZE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_freeze_file_inventory_sha256.csv"
)

FREEZE_VALIDATION_PATH = (
    FREEZE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_freeze_validation.csv"
)

TECHNIQUE_NOISE_AUDIT_PATH = (
    FREEZE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_technique_noise_content_audit.csv"
)

SUMMARY_RECALCULATION_AUDIT_PATH = (
    FREEZE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_summary_recalculation_audit.csv"
)

FREEZE_REPORT_PATH = (
    FREEZE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_freeze_report.json"
)

STEP11C_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step11c_status.json"
)


print("=" * 100)
print("=== PROJECT 8 STEP 11C: INDEPENDENT FINAL-PACKAGE AUDIT AND FREEZE ===")
print("=" * 100)


# ------------------------------------------------------------
# 3. HELPERS
# ------------------------------------------------------------

def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def root_inventory_hash(
    inventory,
):
    digest = hashlib.sha256()

    ordered = inventory.sort_values(
        "RelativePath",
        kind="mergesort",
    )

    for row in ordered.itertuples(
        index=False
    ):

        digest.update(
            row.RelativePath.encode(
                "utf-8"
            )
        )

        digest.update(b"\0")

        digest.update(
            str(
                int(row.SizeBytes)
            ).encode(
                "utf-8"
            )
        )

        digest.update(b"\0")

        digest.update(
            bytes.fromhex(
                row.SHA256
            )
        )

        digest.update(b"\n")

    return digest.hexdigest()


def json_safe(value):

    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:

        if pd.isna(value):
            return None

    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(path)


def read_table(
    path,
):
    path = Path(path)

    suffixes = "".join(
        path.suffixes
    ).lower()

    if suffixes.endswith(
        ".parquet"
    ):

        return pd.read_parquet(
            path
        )

    if (
        suffixes.endswith(".csv")
        or suffixes.endswith(".csv.gz")
    ):

        return pd.read_csv(
            path,
            low_memory=False,
        )

    raise RuntimeError(
        "Unsupported package file:\n"
        f"{path}"
    )


def normalise_column_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


def normalise_technique_slug(
    value,
):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


def dataframe_numeric_equal(
    left,
    right,
    columns,
    atol=FLOAT_ATOL,
):
    for column in columns:

        left_values = pd.to_numeric(
            left[
                column
            ],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )

        right_values = pd.to_numeric(
            right[
                column
            ],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )

        if not np.allclose(
            left_values,
            right_values,
            rtol=0,
            atol=atol,
            equal_nan=True,
        ):

            return False

    return True


# ------------------------------------------------------------
# 4. VALIDATE REQUIRED INPUTS
# ------------------------------------------------------------

required_paths = [
    STEP10_STATUS_PATH,
    STEP11B_STATUS_PATH,
    STEP11B_INVENTORY_PATH,
    PROJECT_RUN_SOURCE_PATH,
    BUILD_METRICS_SOURCE_PATH,
    FINAL_PACKAGE_DIR,
    REGISTRY_PATH,
]


missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]


if missing_paths:

    raise FileNotFoundError(
        "Required Step 11C inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


step10_status = json.loads(
    STEP10_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


step11b_status = json.loads(
    STEP11B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    step10_status.get("Status")
    != EXPECTED_STEP10_STATUS
):

    raise AssertionError(
        "Step 10 status differs.\n"
        f"Actual: {step10_status.get('Status')}"
    )


if (
    step11b_status.get("Status")
    != EXPECTED_STEP11B_STATUS
):

    raise AssertionError(
        "Step 11B status differs.\n"
        f"Actual: {step11b_status.get('Status')}"
    )


if (
    step11b_status.get(
        "PackageRootSHA256"
    )
    != EXPECTED_PACKAGE_ROOT_SHA256
):

    raise AssertionError(
        "Step 11B package-root SHA-256 differs."
    )


registry_sha256_before = (
    calculate_sha256(
        REGISTRY_PATH
    )
)


registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
)


registry_project_numbers = pd.to_numeric(
    registry[
        "ProjectNumber"
    ],
    errors="coerce",
)


project8_registry_rows_before = int(
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).sum()
)


if project8_registry_rows_before != 0:

    raise AssertionError(
        "Project 8 is already present in the completion "
        "registry."
    )


print("\nPrerequisite validation:")

print(
    "Step 10:",
    step10_status["Status"],
)

print(
    "Step 11B:",
    step11b_status["Status"],
)

print(
    "Recorded package files:",
    step11b_status["PackageFiles"],
)

print(
    "Recorded package bytes:",
    step11b_status["PackageBytes"],
)

print(
    "Recorded package root SHA-256:",
    step11b_status["PackageRootSHA256"],
)

print(
    "Project 8 registry rows:",
    project8_registry_rows_before,
)


# ------------------------------------------------------------
# 5. INDEPENDENT FILE INVENTORY
# ------------------------------------------------------------

package_files = sorted([
    path
    for path in FINAL_PACKAGE_DIR.rglob("*")
    if path.is_file()
])


inventory_records = []


print("\nHashing final-package files...")


for index, path in enumerate(
    package_files,
    start=1,
):

    inventory_records.append({
        "RelativePath":
            path.relative_to(
                FINAL_PACKAGE_DIR
            ).as_posix(),

        "SizeBytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            calculate_sha256(
                path
            ),
    })

    if (
        index % 20 == 0
        or index == len(package_files)
    ):

        print(
            "Hashed:",
            index,
            "/",
            len(package_files),
        )


freeze_inventory = pd.DataFrame(
    inventory_records
)


package_file_count = int(
    len(
        freeze_inventory
    )
)


package_bytes = int(
    freeze_inventory[
        "SizeBytes"
    ].sum()
)


package_root_sha256 = (
    root_inventory_hash(
        freeze_inventory
    )
)


top_level_paths = sorted([
    relative_path
    for relative_path in (
        freeze_inventory[
            "RelativePath"
        ].tolist()
    )
    if not relative_path.startswith(
        "technique_noise/"
    )
])


technique_noise_paths = sorted([
    relative_path
    for relative_path in (
        freeze_inventory[
            "RelativePath"
        ].tolist()
    )
    if relative_path.startswith(
        "technique_noise/"
    )
])


print("\nIndependent package inventory:")

print(
    "Files:",
    package_file_count,
)

print(
    "Bytes:",
    package_bytes,
)

print(
    "Top-level files:",
    len(
        top_level_paths
    ),
)

print(
    "Technique-noise files:",
    len(
        technique_noise_paths
    ),
)

print(
    "Package root SHA-256:",
    package_root_sha256,
)


# ------------------------------------------------------------
# 6. COMPARE AGAINST STEP 11B INVENTORY
# ------------------------------------------------------------

step11b_inventory = pd.read_csv(
    STEP11B_INVENTORY_PATH,
    low_memory=False,
)


inventory_comparison = (
    step11b_inventory
    .merge(
        freeze_inventory,
        on="RelativePath",
        how="outer",
        suffixes=(
            "_Step11B",
            "_Step11C",
        ),
        indicator=True,
    )
)


inventory_missing_from_package = int(
    inventory_comparison[
        "_merge"
    ].eq(
        "left_only"
    ).sum()
)


inventory_unexpected_in_package = int(
    inventory_comparison[
        "_merge"
    ].eq(
        "right_only"
    ).sum()
)


comparable_inventory = (
    inventory_comparison[
        inventory_comparison[
            "_merge"
        ].eq(
            "both"
        )
    ]
)


inventory_size_mismatches = int(
    (
        pd.to_numeric(
            comparable_inventory[
                "SizeBytes_Step11B"
            ],
            errors="coerce",
        )
        !=
        pd.to_numeric(
            comparable_inventory[
                "SizeBytes_Step11C"
            ],
            errors="coerce",
        )
    ).sum()
)


inventory_hash_mismatches = int(
    (
        comparable_inventory[
            "SHA256_Step11B"
        ]
        .astype(str)
        .str.lower()
        !=
        comparable_inventory[
            "SHA256_Step11C"
        ]
        .astype(str)
        .str.lower()
    ).sum()
)


# ------------------------------------------------------------
# 7. READ AND VALIDATE TOP-LEVEL TABLES
# ------------------------------------------------------------

expected_top_level_rows = {
    "build_metrics_all.parquet":
        EXPECTED_BUILD_METRIC_ROWS,

    "condition_summary.csv":
        EXPECTED_CONDITIONS,

    "fit_times_all.csv":
        EXPECTED_ML_FITS,

    "noise_injection_summary.csv":
        9,

    "prediction_summary_all.csv":
        EXPECTED_PREDICTION_ROWS,

    "project_level_degradation_summary.csv":
        63,

    "project_level_noise_technique_summary.csv":
        63,

    "project_run_metrics_all.csv":
        EXPECTED_PROJECT_RUN_ROWS,

    "project_run_metrics_all.parquet":
        EXPECTED_PROJECT_RUN_ROWS,
}


readback_failures = []
top_level_row_mismatches = []
project7_identity_files = []
invalid_metric_files = []


top_level_tables = {}


for relative_path in top_level_paths:

    path = (
        FINAL_PACKAGE_DIR
        / relative_path
    )


    try:

        frame = read_table(
            path
        )

        top_level_tables[
            Path(relative_path).name
        ] = frame

    except Exception as error:

        readback_failures.append({
            "RelativePath":
                relative_path,

            "Error":
                str(error),
        })

        continue


    file_name = Path(
        relative_path
    ).name


    if file_name not in (
        expected_top_level_rows
    ):

        top_level_row_mismatches.append({
            "RelativePath":
                relative_path,

            "ExpectedRows":
                "KNOWN_TOP_LEVEL_FILE",

            "ActualRows":
                len(frame),
        })

    elif len(
        frame
    ) != expected_top_level_rows[
        file_name
    ]:

        top_level_row_mismatches.append({
            "RelativePath":
                relative_path,

            "ExpectedRows":
                expected_top_level_rows[
                    file_name
                ],

            "ActualRows":
                len(frame),
        })


    contains_project7_identity = False


    for column in frame.columns:

        if (
            pd.api.types.is_object_dtype(
                frame[column]
            )
            or pd.api.types.is_string_dtype(
                frame[column]
            )
        ):

            values = (
                frame[
                    column
                ]
                .fillna("")
                .astype(str)
                .str.lower()
            )


            if (
                values.str.contains(
                    "compevol",
                    regex=False,
                ).any()
                or
                values.str.contains(
                    "beast2",
                    regex=False,
                ).any()
            ):

                contains_project7_identity = True
                break


    if contains_project7_identity:

        project7_identity_files.append(
            relative_path
        )


    direct_metric_columns = {
        "apfd",
        "apfdc",
        "meanapfd",
        "meanapfdc",
        "medianapfd",
        "medianapfdc",
        "minimumapfd",
        "minimumapfdc",
        "maximumapfd",
        "maximumapfdc",
        "cleanmeanapfd",
        "cleanmeanapfdc",
        "cleanprojectmeanapfd",
        "cleanprojectmeanapfdc",
    }


    for column in frame.columns:

        if (
            normalise_column_name(
                column
            )
            not in direct_metric_columns
        ):

            continue


        values = pd.to_numeric(
            frame[
                column
            ],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )


        finite_values = values[
            np.isfinite(
                values
            )
        ]


        if (
            len(finite_values) > 0
            and (
                (finite_values < 0)
                | (finite_values > 1)
            ).any()
        ):

            invalid_metric_files.append({
                "RelativePath":
                    relative_path,

                "Column":
                    column,
            })


if readback_failures:

    display(
        pd.DataFrame(
            readback_failures
        )
    )

    raise RuntimeError(
        "A top-level package file could not be read."
    )


# ------------------------------------------------------------
# 8. CSV/PARQUET EQUIVALENCE
# ------------------------------------------------------------

project_run_csv = (
    top_level_tables[
        "project_run_metrics_all.csv"
    ]
)


project_run_parquet = (
    top_level_tables[
        "project_run_metrics_all.parquet"
    ]
)


pd.testing.assert_frame_equal(
    project_run_csv,
    project_run_parquet,
    check_dtype=False,
    check_like=False,
)


# ------------------------------------------------------------
# 9. COMPARE PACKAGE RESULTS TO AUDITED SOURCES
# ------------------------------------------------------------

project_run_source = pd.read_csv(
    PROJECT_RUN_SOURCE_PATH,
    low_memory=False,
)


project_run_columns = list(
    project_run_csv.columns
)


project_run_source = (
    project_run_source[
        project_run_columns
    ]
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


project_run_package = (
    project_run_csv
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


project_run_identity_columns = [
    "Project",
    "NoisePercent",
    "RepetitionSeed",
    "Technique",
]


project_run_numeric_columns = [
    "MeanAPFD",
    "MeanAPFDc",
    "SD_APFD",
    "SD_APFDc",
    "EvaluatedBuilds",
    "TotalRankedTests",
    "TotalFailures",
]


project_run_identity_match = bool(
    project_run_package[
        project_run_identity_columns
    ].astype(str).equals(
        project_run_source[
            project_run_identity_columns
        ].astype(str)
    )
)


project_run_numeric_match = (
    dataframe_numeric_equal(
        project_run_package,
        project_run_source,
        project_run_numeric_columns,
    )
)


build_metrics_source = pd.read_csv(
    BUILD_METRICS_SOURCE_PATH,
    low_memory=False,
)


build_metrics_package = (
    top_level_tables[
        "build_metrics_all.parquet"
    ]
)


build_columns = list(
    build_metrics_package.columns
)


build_metrics_source = (
    build_metrics_source[
        build_columns
    ]
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "BuildOrder",
            "Build",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


build_metrics_package = (
    build_metrics_package
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "BuildOrder",
            "Build",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


build_identity_columns = [
    "Project",
    "NoisePercent",
    "RepetitionSeed",
    "Technique",
    "Build",
    "BuildOrder",
    "NumberOfTests",
    "NumberOfFailures",
]


build_numeric_columns = [
    "APFD",
    "APFDc",
]


build_identity_match = bool(
    build_metrics_package[
        build_identity_columns
    ].astype(str).equals(
        build_metrics_source[
            build_identity_columns
        ].astype(str)
    )
)


build_numeric_match = (
    dataframe_numeric_equal(
        build_metrics_package,
        build_metrics_source,
        build_numeric_columns,
    )
)


# ------------------------------------------------------------
# 10. TECHNIQUE-NOISE FILE CONTENT AUDIT
# ------------------------------------------------------------

clean_seed_metrics = (
    project_run_package[
        project_run_package[
            "NoisePercent"
        ].eq(0)
    ][
        [
            "RepetitionSeed",
            "Technique",
            "MeanAPFD",
            "MeanAPFDc",
        ]
    ]
    .rename(
        columns={
            "MeanAPFD":
                "CleanMeanAPFD",

            "MeanAPFDc":
                "CleanMeanAPFDc",
        }
    )
)


technique_noise_audit_records = []


for relative_path in technique_noise_paths:

    file_name = Path(
        relative_path
    ).name.lower()


    parsed = re.fullmatch(
        (
            r"(.+?)"
            r"__noise_(\d{3})"
            r"\.(csv|csv\.gz|parquet)"
        ),
        file_name,
    )


    if parsed is None:

        raise RuntimeError(
            "Could not parse technique-noise filename:\n"
            f"{relative_path}"
        )


    technique_slug = (
        normalise_technique_slug(
            parsed.group(1)
        )
    )

    noise_percent = int(
        parsed.group(2)
    )


    if technique_slug not in (
        TECHNIQUE_SLUG_TO_NAME
    ):

        raise RuntimeError(
            "Unknown technique slug:\n"
            f"{technique_slug}"
        )


    technique = (
        TECHNIQUE_SLUG_TO_NAME[
            technique_slug
        ]
    )


    frame = read_table(
        FINAL_PACKAGE_DIR
        / relative_path
    )


    expected_base = (
        project_run_package[
            project_run_package[
                "Technique"
            ].eq(
                technique
            )
            &
            project_run_package[
                "NoisePercent"
            ].eq(
                noise_percent
            )
        ]
        .merge(
            clean_seed_metrics,
            on=[
                "RepetitionSeed",
                "Technique",
            ],
            how="left",
            validate="one_to_one",
        )
        .sort_values(
            "RepetitionSeed",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    expected_base[
        "DeltaAPFD_NoiseMinusClean"
    ] = (
        expected_base[
            "MeanAPFD"
        ]
        - expected_base[
            "CleanMeanAPFD"
        ]
    )


    expected_base[
        "DeltaAPFDc_NoiseMinusClean"
    ] = (
        expected_base[
            "MeanAPFDc"
        ]
        - expected_base[
            "CleanMeanAPFDc"
        ]
    )


    expected_base[
        "AbsoluteLossAPFD"
    ] = (
        expected_base[
            "CleanMeanAPFD"
        ]
        - expected_base[
            "MeanAPFD"
        ]
    )


    expected_base[
        "AbsoluteLossAPFDc"
    ] = (
        expected_base[
            "CleanMeanAPFDc"
        ]
        - expected_base[
            "MeanAPFDc"
        ]
    )


    expected_base[
        "PercentageLossAPFD"
    ] = np.where(
        expected_base[
            "CleanMeanAPFD"
        ].ne(0),

        100.0
        * expected_base[
            "AbsoluteLossAPFD"
        ]
        / expected_base[
            "CleanMeanAPFD"
        ],

        np.nan,
    )


    expected_base[
        "PercentageLossAPFDc"
    ] = np.where(
        expected_base[
            "CleanMeanAPFDc"
        ].ne(0),

        100.0
        * expected_base[
            "AbsoluteLossAPFDc"
        ]
        / expected_base[
            "CleanMeanAPFDc"
        ],

        np.nan,
    )


    # Retention scale is determined from the generated file.
    # At 0% noise it must be consistently 1 or consistently 100.

    if noise_percent == 0:

        median_retention = float(
            pd.to_numeric(
                frame[
                    "RetentionAPFD"
                ],
                errors="coerce",
            ).median()
        )

        retention_scale = (
            100.0
            if median_retention > 10
            else 1.0
        )

    else:

        clean_same_technique_path = (
            FINAL_PACKAGE_DIR
            / "technique_noise"
            / (
                re.sub(
                    r"__noise_\d{3}",
                    "__noise_000",
                    Path(
                        relative_path
                    ).name,
                )
            )
        )

        clean_file = read_table(
            clean_same_technique_path
        )

        clean_retention_median = float(
            pd.to_numeric(
                clean_file[
                    "RetentionAPFD"
                ],
                errors="coerce",
            ).median()
        )

        retention_scale = (
            100.0
            if clean_retention_median > 10
            else 1.0
        )


    expected_base[
        "RetentionAPFD"
    ] = np.where(
        expected_base[
            "CleanMeanAPFD"
        ].ne(0),

        retention_scale
        * expected_base[
            "MeanAPFD"
        ]
        / expected_base[
            "CleanMeanAPFD"
        ],

        np.nan,
    )


    expected_base[
        "RetentionAPFDc"
    ] = np.where(
        expected_base[
            "CleanMeanAPFDc"
        ].ne(0),

        retention_scale
        * expected_base[
            "MeanAPFDc"
        ]
        / expected_base[
            "CleanMeanAPFDc"
        ],

        np.nan,
    )


    expected_base = expected_base[
        frame.columns
    ]


    frame = (
        frame
        .sort_values(
            "RepetitionSeed",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    identity_columns = [
        "Project",
        "NoisePercent",
        "RepetitionSeed",
        "Technique",
    ]


    numeric_columns = [
        column
        for column in frame.columns
        if column not in identity_columns
    ]


    identity_match = bool(
        frame[
            identity_columns
        ].astype(str).equals(
            expected_base[
                identity_columns
            ].astype(str)
        )
    )


    numeric_match = (
        dataframe_numeric_equal(
            frame,
            expected_base,
            numeric_columns,
        )
    )


    technique_noise_audit_records.append({
        "RelativePath":
            relative_path,

        "Technique":
            technique,

        "NoisePercent":
            noise_percent,

        "Rows":
            len(frame),

        "IdentityMatch":
            identity_match,

        "NumericContentMatch":
            numeric_match,

        "Pass":
            (
                len(frame) == 30
                and identity_match
                and numeric_match
            ),
    })


technique_noise_audit = pd.DataFrame(
    technique_noise_audit_records
)


failed_technique_noise_files = int(
    (
        ~technique_noise_audit[
            "Pass"
        ]
    ).sum()
)


# ------------------------------------------------------------
# 11. RECOMPUTE PROJECT-LEVEL SUMMARY
# ------------------------------------------------------------

project_summary_package = (
    top_level_tables[
        "project_level_noise_technique_summary.csv"
    ]
)


recalculated_summary = (
    project_run_package
    .groupby(
        [
            "Project",
            "NoisePercent",
            "Technique",
        ],
        as_index=False,
    )
    .agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        MeanAPFD=(
            "MeanAPFD",
            "mean",
        ),

        SD_APFD_AcrossSeeds=(
            "MeanAPFD",
            "std",
        ),

        MedianAPFD=(
            "MeanAPFD",
            "median",
        ),

        MinimumAPFD=(
            "MeanAPFD",
            "min",
        ),

        MaximumAPFD=(
            "MeanAPFD",
            "max",
        ),

        MeanAPFDc=(
            "MeanAPFDc",
            "mean",
        ),

        SD_APFDc_AcrossSeeds=(
            "MeanAPFDc",
            "std",
        ),

        MedianAPFDc=(
            "MeanAPFDc",
            "median",
        ),

        MinimumAPFDc=(
            "MeanAPFDc",
            "min",
        ),

        MaximumAPFDc=(
            "MeanAPFDc",
            "max",
        ),

        EvaluatedBuildsPerSeed=(
            "EvaluatedBuilds",
            "mean",
        ),

        TotalRankedTestsPerSeed=(
            "TotalRankedTests",
            "mean",
        ),

        TotalFailuresPerSeed=(
            "TotalFailures",
            "mean",
        ),
    )
)


recalculated_summary[
    "SE_APFD"
] = (
    recalculated_summary[
        "SD_APFD_AcrossSeeds"
    ]
    / np.sqrt(
        recalculated_summary[
            "Seeds"
        ]
    )
)


recalculated_summary[
    "SE_APFDc"
] = (
    recalculated_summary[
        "SD_APFDc_AcrossSeeds"
    ]
    / np.sqrt(
        recalculated_summary[
            "Seeds"
        ]
    )
)


# Derive the critical value directly from the package.
valid_ci_rows = (
    pd.to_numeric(
        project_summary_package[
            "SE_APFD"
        ],
        errors="coerce",
    ).gt(0)
)


critical_values = (
    (
        pd.to_numeric(
            project_summary_package.loc[
                valid_ci_rows,
                "CI95UpperAPFD",
            ],
            errors="coerce",
        )
        -
        pd.to_numeric(
            project_summary_package.loc[
                valid_ci_rows,
                "MeanAPFD",
            ],
            errors="coerce",
        )
    )
    /
    pd.to_numeric(
        project_summary_package.loc[
            valid_ci_rows,
            "SE_APFD",
        ],
        errors="coerce",
    )
)


ci_critical_value = float(
    critical_values.median()
)


recalculated_summary[
    "CI95LowerAPFD"
] = (
    recalculated_summary[
        "MeanAPFD"
    ]
    - ci_critical_value
    * recalculated_summary[
        "SE_APFD"
    ]
)


recalculated_summary[
    "CI95UpperAPFD"
] = (
    recalculated_summary[
        "MeanAPFD"
    ]
    + ci_critical_value
    * recalculated_summary[
        "SE_APFD"
    ]
)


recalculated_summary[
    "CI95LowerAPFDc"
] = (
    recalculated_summary[
        "MeanAPFDc"
    ]
    - ci_critical_value
    * recalculated_summary[
        "SE_APFDc"
    ]
)


recalculated_summary[
    "CI95UpperAPFDc"
] = (
    recalculated_summary[
        "MeanAPFDc"
    ]
    + ci_critical_value
    * recalculated_summary[
        "SE_APFDc"
    ]
)


recalculated_summary = (
    recalculated_summary[
        project_summary_package.columns
    ]
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


project_summary_package = (
    project_summary_package
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


summary_identity_columns = [
    "Project",
    "NoisePercent",
    "Technique",
]


summary_numeric_columns = [
    column
    for column in (
        project_summary_package.columns
    )
    if column not in summary_identity_columns
]


summary_identity_match = bool(
    project_summary_package[
        summary_identity_columns
    ].astype(str).equals(
        recalculated_summary[
            summary_identity_columns
        ].astype(str)
    )
)


summary_numeric_match = (
    dataframe_numeric_equal(
        project_summary_package,
        recalculated_summary,
        summary_numeric_columns,
    )
)


summary_recalculation_audit = pd.DataFrame([
    {
        "Table":
            "project_level_noise_technique_summary.csv",

        "Rows":
            len(
                project_summary_package
            ),

        "IdentityMatch":
            summary_identity_match,

        "NumericContentMatch":
            summary_numeric_match,

        "DerivedCI95CriticalValue":
            ci_critical_value,

        "Pass":
            (
                summary_identity_match
                and summary_numeric_match
            ),
    },
])


# ------------------------------------------------------------
# 12. REGISTRY IMMUTABILITY CHECK
# ------------------------------------------------------------

registry_sha256_after = (
    calculate_sha256(
        REGISTRY_PATH
    )
)


registry_after = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
)


registry_project_numbers_after = pd.to_numeric(
    registry_after[
        "ProjectNumber"
    ],
    errors="coerce",
)


project8_registry_rows_after = int(
    registry_project_numbers_after.eq(
        PROJECT_NUMBER
    ).sum()
)


registry_unchanged = bool(
    registry_sha256_before
    == registry_sha256_after
)


# ------------------------------------------------------------
# 13. OVERALL VALIDATION
# ------------------------------------------------------------

validation_records = [
    {
        "Check":
            "Step 10 passed",

        "Expected":
            EXPECTED_STEP10_STATUS,

        "Actual":
            step10_status[
                "Status"
            ],

        "Pass":
            step10_status[
                "Status"
            ]
            == EXPECTED_STEP10_STATUS,
    },

    {
        "Check":
            "Step 11B passed",

        "Expected":
            EXPECTED_STEP11B_STATUS,

        "Actual":
            step11b_status[
                "Status"
            ],

        "Pass":
            step11b_status[
                "Status"
            ]
            == EXPECTED_STEP11B_STATUS,
    },

    {
        "Check":
            "Package files",

        "Expected":
            EXPECTED_PACKAGE_FILES,

        "Actual":
            package_file_count,

        "Pass":
            package_file_count
            == EXPECTED_PACKAGE_FILES,
    },

    {
        "Check":
            "Package bytes",

        "Expected":
            EXPECTED_PACKAGE_BYTES,

        "Actual":
            package_bytes,

        "Pass":
            package_bytes
            == EXPECTED_PACKAGE_BYTES,
    },

    {
        "Check":
            "Package root SHA-256",

        "Expected":
            EXPECTED_PACKAGE_ROOT_SHA256,

        "Actual":
            package_root_sha256,

        "Pass":
            package_root_sha256
            == EXPECTED_PACKAGE_ROOT_SHA256,
    },

    {
        "Check":
            "Top-level files",

        "Expected":
            EXPECTED_TOP_LEVEL_FILES,

        "Actual":
            len(
                top_level_paths
            ),

        "Pass":
            len(
                top_level_paths
            )
            == EXPECTED_TOP_LEVEL_FILES,
    },

    {
        "Check":
            "Technique-noise files",

        "Expected":
            EXPECTED_TECHNIQUE_NOISE_FILES,

        "Actual":
            len(
                technique_noise_paths
            ),

        "Pass":
            len(
                technique_noise_paths
            )
            == EXPECTED_TECHNIQUE_NOISE_FILES,
    },

    {
        "Check":
            "Step 11B inventory missing files",

        "Expected":
            0,

        "Actual":
            inventory_missing_from_package,

        "Pass":
            inventory_missing_from_package
            == 0,
    },

    {
        "Check":
            "Step 11B inventory unexpected files",

        "Expected":
            0,

        "Actual":
            inventory_unexpected_in_package,

        "Pass":
            inventory_unexpected_in_package
            == 0,
    },

    {
        "Check":
            "Inventory size mismatches",

        "Expected":
            0,

        "Actual":
            inventory_size_mismatches,

        "Pass":
            inventory_size_mismatches
            == 0,
    },

    {
        "Check":
            "Inventory SHA-256 mismatches",

        "Expected":
            0,

        "Actual":
            inventory_hash_mismatches,

        "Pass":
            inventory_hash_mismatches
            == 0,
    },

    {
        "Check":
            "Top-level readback failures",

        "Expected":
            0,

        "Actual":
            len(
                readback_failures
            ),

        "Pass":
            len(
                readback_failures
            )
            == 0,
    },

    {
        "Check":
            "Top-level row mismatches",

        "Expected":
            0,

        "Actual":
            len(
                top_level_row_mismatches
            ),

        "Pass":
            len(
                top_level_row_mismatches
            )
            == 0,
    },

    {
        "Check":
            "Project-run identity matches audited source",

        "Expected":
            True,

        "Actual":
            project_run_identity_match,

        "Pass":
            project_run_identity_match,
    },

    {
        "Check":
            "Project-run metrics match audited source",

        "Expected":
            True,

        "Actual":
            project_run_numeric_match,

        "Pass":
            project_run_numeric_match,
    },

    {
        "Check":
            "Build identity matches audited source",

        "Expected":
            True,

        "Actual":
            build_identity_match,

        "Pass":
            build_identity_match,
    },

    {
        "Check":
            "Build APFD/APFDc match audited source",

        "Expected":
            True,

        "Actual":
            build_numeric_match,

        "Pass":
            build_numeric_match,
    },

    {
        "Check":
            "Failed technique-noise files",

        "Expected":
            0,

        "Actual":
            failed_technique_noise_files,

        "Pass":
            failed_technique_noise_files
            == 0,
    },

    {
        "Check":
            "Project summary recalculation",

        "Expected":
            True,

        "Actual":
            (
                summary_identity_match
                and summary_numeric_match
            ),

        "Pass":
            (
                summary_identity_match
                and summary_numeric_match
            ),
    },

    {
        "Check":
            "Project 7 identity files",

        "Expected":
            0,

        "Actual":
            len(
                project7_identity_files
            ),

        "Pass":
            len(
                project7_identity_files
            )
            == 0,
    },

    {
        "Check":
            "Invalid APFD/APFDc files",

        "Expected":
            0,

        "Actual":
            len(
                invalid_metric_files
            ),

        "Pass":
            len(
                invalid_metric_files
            )
            == 0,
    },

    {
        "Check":
            "Completion registry unchanged",

        "Expected":
            True,

        "Actual":
            registry_unchanged,

        "Pass":
            registry_unchanged,
    },

    {
        "Check":
            "Project 8 registry rows",

        "Expected":
            0,

        "Actual":
            project8_registry_rows_after,

        "Pass":
            project8_registry_rows_after
            == 0,
    },
]


freeze_validation = pd.DataFrame(
    validation_records
)


failed_checks = (
    freeze_validation[
        ~freeze_validation[
            "Pass"
        ]
    ]
    .copy()
)


print("\nIndependent freeze validation:")

display(
    freeze_validation
)


if not failed_checks.empty:

    print("\nFailed checks:")

    display(
        failed_checks
    )

    raise RuntimeError(
        "PROJECT 8 STEP 11C DID NOT PASS.\n"
        "Do not update the completion registry."
    )


# ------------------------------------------------------------
# 14. WRITE STEP 11C OUTPUTS
# ------------------------------------------------------------

FREEZE_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    FREEZE_INVENTORY_PATH,
    freeze_inventory,
)


atomic_write_csv(
    FREEZE_VALIDATION_PATH,
    freeze_validation,
)


atomic_write_csv(
    TECHNIQUE_NOISE_AUDIT_PATH,
    technique_noise_audit,
)


atomic_write_csv(
    SUMMARY_RECALCULATION_AUDIT_PATH,
    summary_recalculation_audit,
)


freeze_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "RawResultFreeze": {
        "Files":
            EXPECTED_RAW_FILES,

        "Bytes":
            EXPECTED_RAW_BYTES,

        "RootSHA256":
            EXPECTED_RAW_ROOT_SHA256,
    },

    "FinalPackageFreeze": {
        "Directory":
            str(
                FINAL_PACKAGE_DIR
            ),

        "Files":
            package_file_count,

        "Bytes":
            package_bytes,

        "RootSHA256":
            package_root_sha256,

        "TopLevelFiles":
            len(
                top_level_paths
            ),

        "TechniqueNoiseFiles":
            len(
                technique_noise_paths
            ),

        "InventorySizeMismatches":
            inventory_size_mismatches,

        "InventoryHashMismatches":
            inventory_hash_mismatches,
    },

    "ContentValidation": {
        "ProjectRunSourceMatch":
            (
                project_run_identity_match
                and project_run_numeric_match
            ),

        "BuildMetricSourceMatch":
            (
                build_identity_match
                and build_numeric_match
            ),

        "FailedTechniqueNoiseFiles":
            failed_technique_noise_files,

        "ProjectSummaryRecalculationPass":
            (
                summary_identity_match
                and summary_numeric_match
            ),

        "CI95CriticalValue":
            ci_critical_value,

        "Project7IdentityFiles":
            len(
                project7_identity_files
            ),

        "InvalidMetricFiles":
            len(
                invalid_metric_files
            ),
    },

    "ValidationChecks":
        len(
            freeze_validation
        ),

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletionRegistryModified":
        False,

    "Projects1To7Modified":
        False,

    "Status":
        STEP11C_PASS_STATUS,
}


atomic_write_json(
    FREEZE_REPORT_PATH,
    freeze_report,
)


step11c_status = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP11C_PASS_STATUS,

    "RawFiles":
        EXPECTED_RAW_FILES,

    "RawBytes":
        EXPECTED_RAW_BYTES,

    "RawRootSHA256":
        EXPECTED_RAW_ROOT_SHA256,

    "PackageFiles":
        package_file_count,

    "PackageBytes":
        package_bytes,

    "PackageRootSHA256":
        package_root_sha256,

    "FailedValidationChecks":
        len(
            failed_checks
        ),

    "CompletionRegistryModified":
        False,

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP11C_STATUS_PATH,
    step11c_status,
)


expected_outputs = [
    FREEZE_INVENTORY_PATH,
    FREEZE_VALIDATION_PATH,
    TECHNIQUE_NOISE_AUDIT_PATH,
    SUMMARY_RECALCULATION_AUDIT_PATH,
    FREEZE_REPORT_PATH,
    STEP11C_STATUS_PATH,
]


missing_outputs = [
    str(path)
    for path in expected_outputs
    if not path.exists()
]


if missing_outputs:

    raise RuntimeError(
        "Step 11C outputs are missing:\n"
        + "\n".join(
            missing_outputs
        )
    )


# ------------------------------------------------------------
# 15. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("=== PROJECT 8 STEP 11C RESULT ===")
print("=" * 100)

print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nRaw result freeze:")

print(
    "Raw files:",
    EXPECTED_RAW_FILES,
)

print(
    "Raw bytes:",
    EXPECTED_RAW_BYTES,
)

print(
    "Raw root SHA-256:",
    EXPECTED_RAW_ROOT_SHA256,
)


print("\nFinal package freeze:")

print(
    "Package directory:",
    FINAL_PACKAGE_DIR,
)

print(
    "Package files:",
    package_file_count,
)

print(
    "Package bytes:",
    package_bytes,
)

print(
    "Package root SHA-256:",
    package_root_sha256,
)

print(
    "Top-level files:",
    len(
        top_level_paths
    ),
)

print(
    "Technique-noise files:",
    len(
        technique_noise_paths
    ),
)

print(
    "Inventory size mismatches:",
    inventory_size_mismatches,
)

print(
    "Inventory SHA-256 mismatches:",
    inventory_hash_mismatches,
)


print("\nIndependent content validation:")

print(
    "Project-run source match:",
    (
        project_run_identity_match
        and project_run_numeric_match
    ),
)

print(
    "Build-metric source match:",
    (
        build_identity_match
        and build_numeric_match
    ),
)

print(
    "Failed technique-noise files:",
    failed_technique_noise_files,
)

print(
    "Project summary recalculation pass:",
    (
        summary_identity_match
        and summary_numeric_match
    ),
)

print(
    "Project 7 identity files:",
    len(
        project7_identity_files
    ),
)

print(
    "Invalid APFD/APFDc files:",
    len(
        invalid_metric_files
    ),
)


print("\nValidation:")

print(
    "Checks:",
    len(
        freeze_validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_checks
    ),
)


print("\nStep 11C outputs:")

for output_path in expected_outputs:

    print(output_path)


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–7 modified:")
print(0)


print(
    "\nSTATUS:",
    STEP11C_PASS_STATUS,
)

print("=" * 100)

=== PROJECT 8 STEP 11C: INDEPENDENT FINAL-PACKAGE AUDIT AND FREEZE ===

Prerequisite validation:
Step 10: PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED
Step 11B: PASS_PROJECT_8_FINAL_72_FILE_PACKAGE_GENERATED_AND_VALIDATED
Recorded package files: 72
Recorded package bytes: 1242844
Recorded package root SHA-256: 1e27d6f57526d3ce819b7358867c0c58411cc4eaddcdcb2e5576b2282a6cd607
Project 8 registry rows: 0

Hashing final-package files...
Hashed: 20 / 72
Hashed: 40 / 72
Hashed: 60 / 72
Hashed: 72 / 72

Independent package inventory:
Files: 72
Bytes: 1242844
Top-level files: 9
Technique-noise files: 63
Package root SHA-256: 1e27d6f57526d3ce819b7358867c0c58411cc4eaddcdcb2e5576b2282a6cd607

Independent freeze validation:


,Check,Expected,Actual,Pass
0,Step 10 passed,PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED,PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED,True
1,Step 11B passed,PASS_PROJECT_8_FINAL_72_FILE_PACKAGE_GENERATED...,PASS_PROJECT_8_FINAL_72_FILE_PACKAGE_GENERATED...,True
2,Package files,72,72,True
3,Package bytes,1242844,1242844,True
4,Package root SHA-256,1e27d6f57526d3ce819b7358867c0c58411cc4eaddcdcb...,1e27d6f57526d3ce819b7358867c0c58411cc4eaddcdcb...,True
5,Top-level files,9,9,True
6,Technique-noise files,63,63,True
7,Step 11B inventory missing files,0,0,True
8,Step 11B inventory unexpected files,0,0,True
9,Inventory size mismatches,0,0,True



Failed checks:


,Check,Expected,Actual,Pass
13,Project-run identity matches audited source,True,False,False


RuntimeError: PROJECT 8 STEP 11C DID NOT PASS.
Do not update the completion registry.

In [ ]:
# ============================================================
# PROJECT 8 — STEP 11C V2
# INDEPENDENT FINAL-PACKAGE AUDIT AND FREEZE
#
# Fix:
# - compares text identity fields as text
# - compares numeric identity fields numerically
# - avoids false mismatches such as "0" versus "0.0"
#
# This cell does NOT:
# - regenerate package files
# - rerun models
# - alter raw results
# - alter Projects 1–7
# - modify the completion registry
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import re

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 8
PROJECT_NAME = "optimatika@ojAlgo"
PROJECT_SLUG = "optimatika__ojAlgo"
PROJECT_SHORT_NAME = "ojalgo"

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

SEEDS = list(range(1, 31))

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)

TECHNIQUE_SLUG_TO_NAME = {
    "randomforest": "RandomForest",
    "xgboost": "XGBoost",
    "lightgbm": "LightGBM",
    "naivebayes": "NaiveBayes",
    "random": "Random",
    "latestfail": "LatestFail",
    "qtfavg": "QTF-Avg",
}

EXPECTED_CONDITIONS = 270
EXPECTED_ML_FITS = 1080
EXPECTED_PROJECT_RUN_ROWS = 1890
EXPECTED_BUILD_METRIC_ROWS = 30240
EXPECTED_PREDICTION_ROWS = 1080

EXPECTED_PACKAGE_FILES = 72
EXPECTED_TOP_LEVEL_FILES = 9
EXPECTED_TECHNIQUE_NOISE_FILES = 63

EXPECTED_RAW_FILES = 2160
EXPECTED_RAW_BYTES = 179766494

EXPECTED_RAW_ROOT_SHA256 = (
    "19ae21c5d8524c358796e28f23577fbf03bfd31f7d4b530372c6b4d8a492887c"
)

EXPECTED_PACKAGE_BYTES = 1242844

EXPECTED_PACKAGE_ROOT_SHA256 = (
    "1e27d6f57526d3ce819b7358867c0c58411cc4eaddcdcb2e5576b2282a6cd607"
)

EXPECTED_STEP10_STATUS = (
    "PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED"
)

EXPECTED_STEP11B_STATUS = (
    "PASS_PROJECT_8_FINAL_72_FILE_PACKAGE_GENERATED_AND_VALIDATED"
)

STEP11C_PASS_STATUS = (
    "PASS_PROJECT_8_FINAL_PACKAGE_INDEPENDENTLY_AUDITED_AND_FROZEN"
)

FLOAT_ATOL = 1e-12


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)

RAW_AUDIT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_raw_audit"
)

FINAL_PACKAGE_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_30_seed_final"
)

FINAL_PACKAGE_AUDIT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_audit"
)

STEP10_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step10_status.json"
)

STEP11B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step11b_status.json"
)

STEP11B_REPORT_PATH = (
    FINAL_PACKAGE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_report.json"
)

STEP11B_INVENTORY_PATH = (
    FINAL_PACKAGE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_file_inventory_sha256.csv"
)

PROJECT_RUN_SOURCE_PATH = (
    RAW_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_all_project_run_metrics.csv"
)

BUILD_METRICS_SOURCE_PATH = (
    RAW_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_all_build_metrics.csv.gz"
)

FIT_TIMES_SOURCE_PATH = (
    RAW_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_all_fit_times.csv"
)

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)


# Step 11C outputs

FREEZE_AUDIT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_final_freeze_audit"
)

FREEZE_INVENTORY_PATH = (
    FREEZE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_freeze_file_inventory_sha256.csv"
)

FREEZE_VALIDATION_PATH = (
    FREEZE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_freeze_validation.csv"
)

TECHNIQUE_NOISE_AUDIT_PATH = (
    FREEZE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_technique_noise_content_audit.csv"
)

SUMMARY_RECALCULATION_AUDIT_PATH = (
    FREEZE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_summary_recalculation_audit.csv"
)

IDENTITY_DIAGNOSTICS_PATH = (
    FREEZE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_identity_comparison_diagnostics.csv"
)

FREEZE_REPORT_PATH = (
    FREEZE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_freeze_report.json"
)

STEP11C_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step11c_status.json"
)


print("=" * 102)
print("=== PROJECT 8 STEP 11C V2: INDEPENDENT FINAL-PACKAGE AUDIT AND FREEZE ===")
print("=" * 102)


# ------------------------------------------------------------
# 3. GENERAL HELPERS
# ------------------------------------------------------------

def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def root_inventory_hash(inventory):
    digest = hashlib.sha256()

    ordered = inventory.sort_values(
        "RelativePath",
        kind="mergesort",
    )

    for row in ordered.itertuples(index=False):
        digest.update(
            row.RelativePath.encode("utf-8")
        )
        digest.update(b"\0")
        digest.update(
            str(int(row.SizeBytes)).encode("utf-8")
        )
        digest.update(b"\0")
        digest.update(
            bytes.fromhex(row.SHA256)
        )
        digest.update(b"\n")

    return digest.hexdigest()


def json_safe(value):
    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return value


def atomic_write_json(path, payload):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def atomic_write_csv(path, dataframe):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(path)


def read_table(path):
    path = Path(path)

    suffixes = "".join(
        path.suffixes
    ).lower()

    if suffixes.endswith(".parquet"):
        return pd.read_parquet(path)

    if (
        suffixes.endswith(".csv")
        or suffixes.endswith(".csv.gz")
    ):
        return pd.read_csv(
            path,
            low_memory=False,
        )

    raise RuntimeError(
        "Unsupported package file:\n"
        f"{path}"
    )


def normalise_column_name(value):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


def normalise_technique_slug(value):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


def dataframe_numeric_equal(
    left,
    right,
    columns,
    atol=FLOAT_ATOL,
):
    if len(left) != len(right):
        return False

    for column in columns:
        left_values = pd.to_numeric(
            left[column],
            errors="coerce",
        ).to_numpy(dtype=float)

        right_values = pd.to_numeric(
            right[column],
            errors="coerce",
        ).to_numpy(dtype=float)

        if not np.allclose(
            left_values,
            right_values,
            rtol=0,
            atol=atol,
            equal_nan=True,
        ):
            return False

    return True


def semantic_identity_compare(
    left,
    right,
    text_columns,
    numeric_columns,
    comparison_name,
):
    """
    Compare identifiers according to their actual type.

    Text identifiers:
      compared as stripped text.

    Numeric identifiers:
      converted to numbers and compared numerically.

    This avoids false mismatches such as:
      0 versus 0.0
      1 versus 1.0
    """

    diagnostics = []

    if len(left) != len(right):
        diagnostics.append({
            "Comparison":
                comparison_name,

            "Column":
                "__ROW_COUNT__",

            "ColumnType":
                "row_count",

            "MismatchCount":
                abs(
                    len(left)
                    - len(right)
                ),

            "LeftExample":
                len(left),

            "RightExample":
                len(right),
        })

        return False, diagnostics


    all_columns_pass = True


    for column in text_columns:
        left_values = (
            left[column]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        right_values = (
            right[column]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        mismatch_mask = (
            left_values
            != right_values
        )

        mismatch_count = int(
            mismatch_mask.sum()
        )

        if mismatch_count:
            all_columns_pass = False

            first_index = mismatch_mask[
                mismatch_mask
            ].index[0]

            diagnostics.append({
                "Comparison":
                    comparison_name,

                "Column":
                    column,

                "ColumnType":
                    "text",

                "MismatchCount":
                    mismatch_count,

                "LeftExample":
                    left_values.loc[
                        first_index
                    ],

                "RightExample":
                    right_values.loc[
                        first_index
                    ],
            })


    for column in numeric_columns:
        left_values = pd.to_numeric(
            left[column],
            errors="coerce",
        ).to_numpy(dtype=float)

        right_values = pd.to_numeric(
            right[column],
            errors="coerce",
        ).to_numpy(dtype=float)

        equal_mask = np.isclose(
            left_values,
            right_values,
            rtol=0,
            atol=0,
            equal_nan=True,
        )

        mismatch_count = int(
            (~equal_mask).sum()
        )

        if mismatch_count:
            all_columns_pass = False

            first_position = int(
                np.flatnonzero(
                    ~equal_mask
                )[0]
            )

            diagnostics.append({
                "Comparison":
                    comparison_name,

                "Column":
                    column,

                "ColumnType":
                    "numeric",

                "MismatchCount":
                    mismatch_count,

                "LeftExample":
                    left_values[
                        first_position
                    ],

                "RightExample":
                    right_values[
                        first_position
                    ],
            })


    if all_columns_pass:
        diagnostics.append({
            "Comparison":
                comparison_name,

            "Column":
                "__ALL_IDENTITY_COLUMNS__",

            "ColumnType":
                "summary",

            "MismatchCount":
                0,

            "LeftExample":
                None,

            "RightExample":
                None,
        })


    return all_columns_pass, diagnostics


def scan_frame_for_reference_identity(frame):
    for column in frame.columns:
        if not (
            pd.api.types.is_object_dtype(
                frame[column]
            )
            or pd.api.types.is_string_dtype(
                frame[column]
            )
        ):
            continue

        values = (
            frame[column]
            .fillna("")
            .astype(str)
            .str.lower()
        )

        if (
            values.str.contains(
                "compevol",
                regex=False,
            ).any()
            or
            values.str.contains(
                "beast2",
                regex=False,
            ).any()
        ):
            return True

    return False


def find_invalid_direct_metrics(
    frame,
    relative_path,
):
    invalid_records = []

    direct_metric_columns = {
        "apfd",
        "apfdc",
        "meanapfd",
        "meanapfdc",
        "medianapfd",
        "medianapfdc",
        "minimumapfd",
        "minimumapfdc",
        "maximumapfd",
        "maximumapfdc",
        "cleanmeanapfd",
        "cleanmeanapfdc",
        "cleanprojectmeanapfd",
        "cleanprojectmeanapfdc",
    }

    for column in frame.columns:
        if (
            normalise_column_name(column)
            not in direct_metric_columns
        ):
            continue

        values = pd.to_numeric(
            frame[column],
            errors="coerce",
        ).to_numpy(dtype=float)

        finite_values = values[
            np.isfinite(values)
        ]

        if (
            len(finite_values) > 0
            and (
                (finite_values < 0)
                | (finite_values > 1)
            ).any()
        ):
            invalid_records.append({
                "RelativePath":
                    relative_path,

                "Column":
                    column,
            })

    return invalid_records


# ------------------------------------------------------------
# 4. VALIDATE PREREQUISITES
# ------------------------------------------------------------

required_paths = [
    STEP10_STATUS_PATH,
    STEP11B_STATUS_PATH,
    STEP11B_REPORT_PATH,
    STEP11B_INVENTORY_PATH,
    PROJECT_RUN_SOURCE_PATH,
    BUILD_METRICS_SOURCE_PATH,
    FIT_TIMES_SOURCE_PATH,
    FINAL_PACKAGE_DIR,
    REGISTRY_PATH,
]


missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]


if missing_paths:
    raise FileNotFoundError(
        "Required Step 11C V2 inputs are missing:\n"
        + "\n".join(missing_paths)
    )


step10_status = json.loads(
    STEP10_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


step11b_status = json.loads(
    STEP11B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


step11b_report = json.loads(
    STEP11B_REPORT_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    step10_status.get("Status")
    != EXPECTED_STEP10_STATUS
):
    raise AssertionError(
        "Step 10 status differs.\n"
        f"Actual: "
        f"{step10_status.get('Status')}"
    )


if (
    step11b_status.get("Status")
    != EXPECTED_STEP11B_STATUS
):
    raise AssertionError(
        "Step 11B status differs.\n"
        f"Actual: "
        f"{step11b_status.get('Status')}"
    )


if (
    int(
        step11b_status.get(
            "PackageFiles",
            -1,
        )
    )
    != EXPECTED_PACKAGE_FILES
):
    raise AssertionError(
        "Step 11B package-file count differs."
    )


if (
    int(
        step11b_status.get(
            "PackageBytes",
            -1,
        )
    )
    != EXPECTED_PACKAGE_BYTES
):
    raise AssertionError(
        "Step 11B package-byte count differs."
    )


if (
    step11b_status.get(
        "PackageRootSHA256"
    )
    != EXPECTED_PACKAGE_ROOT_SHA256
):
    raise AssertionError(
        "Step 11B package-root SHA-256 differs."
    )


CI_CRITICAL_95 = float(
    step11b_report[
        "StatisticalConvention"
    ][
        "CI95CriticalValue"
    ]
)


RETENTION_SCALE = float(
    step11b_report[
        "StatisticalConvention"
    ][
        "RetentionScale"
    ]
)


registry_sha256_before = (
    calculate_sha256(
        REGISTRY_PATH
    )
)


registry_before = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
)


registry_project_numbers_before = (
    pd.to_numeric(
        registry_before[
            "ProjectNumber"
        ],
        errors="coerce",
    )
)


project8_registry_rows_before = int(
    registry_project_numbers_before.eq(
        PROJECT_NUMBER
    ).sum()
)


if project8_registry_rows_before != 0:
    raise AssertionError(
        "Project 8 is already present in the "
        "completion registry."
    )


print("\nPrerequisite validation:")

print(
    "Step 10:",
    step10_status["Status"],
)

print(
    "Step 11B:",
    step11b_status["Status"],
)

print(
    "Recorded package files:",
    step11b_status["PackageFiles"],
)

print(
    "Recorded package bytes:",
    step11b_status["PackageBytes"],
)

print(
    "Recorded package root SHA-256:",
    step11b_status[
        "PackageRootSHA256"
    ],
)

print(
    "CI critical value:",
    CI_CRITICAL_95,
)

print(
    "Retention scale:",
    RETENTION_SCALE,
)

print(
    "Project 8 registry rows:",
    project8_registry_rows_before,
)


# ------------------------------------------------------------
# 5. INDEPENDENT PACKAGE INVENTORY
# ------------------------------------------------------------

package_files = sorted([
    path
    for path in FINAL_PACKAGE_DIR.rglob("*")
    if path.is_file()
])


inventory_records = []


print("\nHashing final-package files...")


for index, path in enumerate(
    package_files,
    start=1,
):
    inventory_records.append({
        "RelativePath":
            path.relative_to(
                FINAL_PACKAGE_DIR
            ).as_posix(),

        "SizeBytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            calculate_sha256(path),
    })

    if (
        index % 20 == 0
        or index == len(package_files)
    ):
        print(
            "Hashed:",
            index,
            "/",
            len(package_files),
        )


freeze_inventory = pd.DataFrame(
    inventory_records
)


package_file_count = int(
    len(freeze_inventory)
)


package_bytes = int(
    freeze_inventory[
        "SizeBytes"
    ].sum()
)


package_root_sha256 = (
    root_inventory_hash(
        freeze_inventory
    )
)


top_level_paths = sorted([
    relative_path
    for relative_path in (
        freeze_inventory[
            "RelativePath"
        ].tolist()
    )
    if not relative_path.startswith(
        "technique_noise/"
    )
])


technique_noise_paths = sorted([
    relative_path
    for relative_path in (
        freeze_inventory[
            "RelativePath"
        ].tolist()
    )
    if relative_path.startswith(
        "technique_noise/"
    )
])


print("\nIndependent package inventory:")

print(
    "Files:",
    package_file_count,
)

print(
    "Bytes:",
    package_bytes,
)

print(
    "Top-level files:",
    len(top_level_paths),
)

print(
    "Technique-noise files:",
    len(technique_noise_paths),
)

print(
    "Package root SHA-256:",
    package_root_sha256,
)


# ------------------------------------------------------------
# 6. COMPARE WITH STEP 11B INVENTORY
# ------------------------------------------------------------

step11b_inventory = pd.read_csv(
    STEP11B_INVENTORY_PATH,
    low_memory=False,
)


inventory_comparison = (
    step11b_inventory
    .merge(
        freeze_inventory,
        on="RelativePath",
        how="outer",
        suffixes=(
            "_Step11B",
            "_Step11C",
        ),
        indicator=True,
    )
)


inventory_missing_from_package = int(
    inventory_comparison[
        "_merge"
    ].eq("left_only").sum()
)


inventory_unexpected_in_package = int(
    inventory_comparison[
        "_merge"
    ].eq("right_only").sum()
)


comparable_inventory = (
    inventory_comparison[
        inventory_comparison[
            "_merge"
        ].eq("both")
    ]
)


inventory_size_mismatches = int(
    (
        pd.to_numeric(
            comparable_inventory[
                "SizeBytes_Step11B"
            ],
            errors="coerce",
        )
        !=
        pd.to_numeric(
            comparable_inventory[
                "SizeBytes_Step11C"
            ],
            errors="coerce",
        )
    ).sum()
)


inventory_hash_mismatches = int(
    (
        comparable_inventory[
            "SHA256_Step11B"
        ]
        .astype(str)
        .str.lower()
        !=
        comparable_inventory[
            "SHA256_Step11C"
        ]
        .astype(str)
        .str.lower()
    ).sum()
)


# ------------------------------------------------------------
# 7. READ AND CHECK ALL PACKAGE TABLES
# ------------------------------------------------------------

expected_top_level_rows = {
    "build_metrics_all.parquet":
        EXPECTED_BUILD_METRIC_ROWS,

    "condition_summary.csv":
        EXPECTED_CONDITIONS,

    "fit_times_all.csv":
        EXPECTED_ML_FITS,

    "noise_injection_summary.csv":
        9,

    "prediction_summary_all.csv":
        EXPECTED_PREDICTION_ROWS,

    "project_level_degradation_summary.csv":
        63,

    "project_level_noise_technique_summary.csv":
        63,

    "project_run_metrics_all.csv":
        EXPECTED_PROJECT_RUN_ROWS,

    "project_run_metrics_all.parquet":
        EXPECTED_PROJECT_RUN_ROWS,
}


readback_failures = []
top_level_row_mismatches = []
project7_identity_files = []
invalid_metric_files = []

top_level_tables = {}


for relative_path in top_level_paths:
    package_path = (
        FINAL_PACKAGE_DIR
        / relative_path
    )

    try:
        frame = read_table(
            package_path
        )

    except Exception as error:
        readback_failures.append({
            "RelativePath":
                relative_path,

            "Error":
                str(error),
        })

        continue


    file_name = Path(
        relative_path
    ).name

    top_level_tables[
        file_name
    ] = frame


    if file_name not in expected_top_level_rows:
        top_level_row_mismatches.append({
            "RelativePath":
                relative_path,

            "ExpectedRows":
                "RECOGNISED_TOP_LEVEL_FILE",

            "ActualRows":
                len(frame),
        })

    elif (
        len(frame)
        != expected_top_level_rows[
            file_name
        ]
    ):
        top_level_row_mismatches.append({
            "RelativePath":
                relative_path,

            "ExpectedRows":
                expected_top_level_rows[
                    file_name
                ],

            "ActualRows":
                len(frame),
        })


    if scan_frame_for_reference_identity(
        frame
    ):
        project7_identity_files.append(
            relative_path
        )


    invalid_metric_files.extend(
        find_invalid_direct_metrics(
            frame,
            relative_path,
        )
    )


if readback_failures:
    display(
        pd.DataFrame(
            readback_failures
        )
    )

    raise RuntimeError(
        "A top-level package file could not "
        "be read."
    )


# ------------------------------------------------------------
# 8. CSV/PARQUET EQUIVALENCE
# ------------------------------------------------------------

project_run_csv = (
    top_level_tables[
        "project_run_metrics_all.csv"
    ]
)


project_run_parquet = (
    top_level_tables[
        "project_run_metrics_all.parquet"
    ]
)


pd.testing.assert_frame_equal(
    project_run_csv,
    project_run_parquet,
    check_dtype=False,
    check_like=False,
)


# ------------------------------------------------------------
# 9. COMPARE PROJECT-RUN TABLE TO AUDITED SOURCE
# ------------------------------------------------------------

project_run_source = pd.read_csv(
    PROJECT_RUN_SOURCE_PATH,
    low_memory=False,
)


project_run_columns = list(
    project_run_csv.columns
)


project_run_source = (
    project_run_source[
        project_run_columns
    ]
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


project_run_package = (
    project_run_csv
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


(
    project_run_identity_match,
    project_run_identity_diagnostics,
) = semantic_identity_compare(
    left=project_run_package,
    right=project_run_source,

    text_columns=[
        "Project",
        "Technique",
    ],

    numeric_columns=[
        "NoisePercent",
        "RepetitionSeed",
    ],

    comparison_name=(
        "project_run_package_vs_audited_source"
    ),
)


project_run_numeric_match = (
    dataframe_numeric_equal(
        project_run_package,
        project_run_source,
        [
            "MeanAPFD",
            "MeanAPFDc",
            "SD_APFD",
            "SD_APFDc",
            "EvaluatedBuilds",
            "TotalRankedTests",
            "TotalFailures",
        ],
    )
)


# ------------------------------------------------------------
# 10. COMPARE BUILD TABLE TO AUDITED SOURCE
# ------------------------------------------------------------

build_metrics_source = pd.read_csv(
    BUILD_METRICS_SOURCE_PATH,
    low_memory=False,
)


build_metrics_package = (
    top_level_tables[
        "build_metrics_all.parquet"
    ]
)


build_columns = list(
    build_metrics_package.columns
)


build_metrics_source = (
    build_metrics_source[
        build_columns
    ]
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "BuildOrder",
            "Build",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


build_metrics_package = (
    build_metrics_package
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "BuildOrder",
            "Build",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


(
    build_identity_match,
    build_identity_diagnostics,
) = semantic_identity_compare(
    left=build_metrics_package,
    right=build_metrics_source,

    text_columns=[
        "Project",
        "Technique",
    ],

    numeric_columns=[
        "NoisePercent",
        "RepetitionSeed",
        "Build",
        "BuildOrder",
        "NumberOfTests",
        "NumberOfFailures",
    ],

    comparison_name=(
        "build_metrics_package_vs_audited_source"
    ),
)


build_numeric_match = (
    dataframe_numeric_equal(
        build_metrics_package,
        build_metrics_source,
        [
            "APFD",
            "APFDc",
        ],
    )
)


# ------------------------------------------------------------
# 11. COMPARE FIT-TIME TABLE TO AUDITED SOURCE
# ------------------------------------------------------------

fit_times_source = pd.read_csv(
    FIT_TIMES_SOURCE_PATH,
    low_memory=False,
)


fit_times_package = (
    top_level_tables[
        "fit_times_all.csv"
    ]
)


fit_times_columns = list(
    fit_times_package.columns
)


fit_times_source = (
    fit_times_source[
        fit_times_columns
    ]
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


fit_times_package = (
    fit_times_package
    .sort_values(
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


(
    fit_identity_match,
    fit_identity_diagnostics,
) = semantic_identity_compare(
    left=fit_times_package,
    right=fit_times_source,

    text_columns=[
        "Project",
        "Technique",
    ],

    numeric_columns=[
        "NoisePercent",
        "RepetitionSeed",
    ],

    comparison_name=(
        "fit_times_package_vs_audited_source"
    ),
)


fit_numeric_match = (
    dataframe_numeric_equal(
        fit_times_package,
        fit_times_source,
        [
            "FitSeconds",
            "TrainingRows",
            "TrainingFailures",
            "TrainingPasses",
            "ActiveFeatures",
        ],
    )
)


identity_diagnostics = pd.DataFrame(
    project_run_identity_diagnostics
    + build_identity_diagnostics
    + fit_identity_diagnostics
)


print("\nIdentity comparison diagnostics:")

display(
    identity_diagnostics
)


# ------------------------------------------------------------
# 12. TECHNIQUE-NOISE FILE AUDIT
# ------------------------------------------------------------

clean_seed_metrics = (
    project_run_package[
        project_run_package[
            "NoisePercent"
        ].eq(0)
    ][
        [
            "RepetitionSeed",
            "Technique",
            "MeanAPFD",
            "MeanAPFDc",
        ]
    ]
    .rename(
        columns={
            "MeanAPFD":
                "CleanMeanAPFD",

            "MeanAPFDc":
                "CleanMeanAPFDc",
        }
    )
)


technique_noise_audit_records = []


for relative_path in technique_noise_paths:
    file_name = Path(
        relative_path
    ).name.lower()


    parsed = re.fullmatch(
        (
            r"(.+?)"
            r"__noise_(\d{3})"
            r"\.(csv|csv\.gz|parquet)"
        ),
        file_name,
    )


    if parsed is None:
        raise RuntimeError(
            "Could not parse technique-noise file:\n"
            f"{relative_path}"
        )


    technique_slug = (
        normalise_technique_slug(
            parsed.group(1)
        )
    )

    noise_percent = int(
        parsed.group(2)
    )


    if technique_slug not in (
        TECHNIQUE_SLUG_TO_NAME
    ):
        raise RuntimeError(
            "Unknown technique slug:\n"
            f"{technique_slug}"
        )


    technique = (
        TECHNIQUE_SLUG_TO_NAME[
            technique_slug
        ]
    )


    frame = (
        read_table(
            FINAL_PACKAGE_DIR
            / relative_path
        )
        .sort_values(
            "RepetitionSeed",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    if scan_frame_for_reference_identity(
        frame
    ):
        project7_identity_files.append(
            relative_path
        )


    invalid_metric_files.extend(
        find_invalid_direct_metrics(
            frame,
            relative_path,
        )
    )


    expected_frame = (
        project_run_package[
            project_run_package[
                "Technique"
            ].eq(technique)
            &
            project_run_package[
                "NoisePercent"
            ].eq(noise_percent)
        ]
        .merge(
            clean_seed_metrics,
            on=[
                "RepetitionSeed",
                "Technique",
            ],
            how="left",
            validate="one_to_one",
        )
        .sort_values(
            "RepetitionSeed",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    expected_frame[
        "DeltaAPFD_NoiseMinusClean"
    ] = (
        expected_frame[
            "MeanAPFD"
        ]
        - expected_frame[
            "CleanMeanAPFD"
        ]
    )


    expected_frame[
        "DeltaAPFDc_NoiseMinusClean"
    ] = (
        expected_frame[
            "MeanAPFDc"
        ]
        - expected_frame[
            "CleanMeanAPFDc"
        ]
    )


    expected_frame[
        "AbsoluteLossAPFD"
    ] = (
        expected_frame[
            "CleanMeanAPFD"
        ]
        - expected_frame[
            "MeanAPFD"
        ]
    )


    expected_frame[
        "AbsoluteLossAPFDc"
    ] = (
        expected_frame[
            "CleanMeanAPFDc"
        ]
        - expected_frame[
            "MeanAPFDc"
        ]
    )


    expected_frame[
        "PercentageLossAPFD"
    ] = np.where(
        expected_frame[
            "CleanMeanAPFD"
        ].ne(0),

        100.0
        * expected_frame[
            "AbsoluteLossAPFD"
        ]
        / expected_frame[
            "CleanMeanAPFD"
        ],

        np.nan,
    )


    expected_frame[
        "PercentageLossAPFDc"
    ] = np.where(
        expected_frame[
            "CleanMeanAPFDc"
        ].ne(0),

        100.0
        * expected_frame[
            "AbsoluteLossAPFDc"
        ]
        / expected_frame[
            "CleanMeanAPFDc"
        ],

        np.nan,
    )


    expected_frame[
        "RetentionAPFD"
    ] = np.where(
        expected_frame[
            "CleanMeanAPFD"
        ].ne(0),

        RETENTION_SCALE
        * expected_frame[
            "MeanAPFD"
        ]
        / expected_frame[
            "CleanMeanAPFD"
        ],

        np.nan,
    )


    expected_frame[
        "RetentionAPFDc"
    ] = np.where(
        expected_frame[
            "CleanMeanAPFDc"
        ].ne(0),

        RETENTION_SCALE
        * expected_frame[
            "MeanAPFDc"
        ]
        / expected_frame[
            "CleanMeanAPFDc"
        ],

        np.nan,
    )


    expected_frame = expected_frame[
        frame.columns
    ]


    (
        identity_match,
        _
    ) = semantic_identity_compare(
        left=frame,
        right=expected_frame,

        text_columns=[
            "Project",
            "Technique",
        ],

        numeric_columns=[
            "NoisePercent",
            "RepetitionSeed",
        ],

        comparison_name=relative_path,
    )


    numeric_columns = [
        column
        for column in frame.columns
        if column not in {
            "Project",
            "Technique",
            "NoisePercent",
            "RepetitionSeed",
        }
    ]


    numeric_match = (
        dataframe_numeric_equal(
            frame,
            expected_frame,
            numeric_columns,
        )
    )


    technique_noise_audit_records.append({
        "RelativePath":
            relative_path,

        "Technique":
            technique,

        "NoisePercent":
            noise_percent,

        "Rows":
            len(frame),

        "IdentityMatch":
            identity_match,

        "NumericContentMatch":
            numeric_match,

        "Pass":
            (
                len(frame) == 30
                and identity_match
                and numeric_match
            ),
    })


technique_noise_audit = pd.DataFrame(
    technique_noise_audit_records
)


failed_technique_noise_files = int(
    (
        ~technique_noise_audit[
            "Pass"
        ]
    ).sum()
)


# Remove duplicates caused by scanning all tables.

project7_identity_files = sorted(
    set(
        project7_identity_files
    )
)


if invalid_metric_files:
    invalid_metric_files_frame = (
        pd.DataFrame(
            invalid_metric_files
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

else:
    invalid_metric_files_frame = pd.DataFrame(
        columns=[
            "RelativePath",
            "Column",
        ]
    )


# ------------------------------------------------------------
# 13. RECOMPUTE PROJECT-LEVEL SUMMARY
# ------------------------------------------------------------

summary_package = (
    top_level_tables[
        "project_level_noise_technique_summary.csv"
    ]
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


summary_recalculated = (
    project_run_package
    .groupby(
        [
            "Project",
            "NoisePercent",
            "Technique",
        ],
        as_index=False,
    )
    .agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        MeanAPFD=(
            "MeanAPFD",
            "mean",
        ),

        SD_APFD_AcrossSeeds=(
            "MeanAPFD",
            "std",
        ),

        MedianAPFD=(
            "MeanAPFD",
            "median",
        ),

        MinimumAPFD=(
            "MeanAPFD",
            "min",
        ),

        MaximumAPFD=(
            "MeanAPFD",
            "max",
        ),

        MeanAPFDc=(
            "MeanAPFDc",
            "mean",
        ),

        SD_APFDc_AcrossSeeds=(
            "MeanAPFDc",
            "std",
        ),

        MedianAPFDc=(
            "MeanAPFDc",
            "median",
        ),

        MinimumAPFDc=(
            "MeanAPFDc",
            "min",
        ),

        MaximumAPFDc=(
            "MeanAPFDc",
            "max",
        ),

        EvaluatedBuildsPerSeed=(
            "EvaluatedBuilds",
            "mean",
        ),

        TotalRankedTestsPerSeed=(
            "TotalRankedTests",
            "mean",
        ),

        TotalFailuresPerSeed=(
            "TotalFailures",
            "mean",
        ),
    )
)


summary_recalculated[
    "SE_APFD"
] = (
    summary_recalculated[
        "SD_APFD_AcrossSeeds"
    ]
    / np.sqrt(
        summary_recalculated[
            "Seeds"
        ]
    )
)


summary_recalculated[
    "CI95LowerAPFD"
] = (
    summary_recalculated[
        "MeanAPFD"
    ]
    - CI_CRITICAL_95
    * summary_recalculated[
        "SE_APFD"
    ]
)


summary_recalculated[
    "CI95UpperAPFD"
] = (
    summary_recalculated[
        "MeanAPFD"
    ]
    + CI_CRITICAL_95
    * summary_recalculated[
        "SE_APFD"
    ]
)


summary_recalculated[
    "SE_APFDc"
] = (
    summary_recalculated[
        "SD_APFDc_AcrossSeeds"
    ]
    / np.sqrt(
        summary_recalculated[
            "Seeds"
        ]
    )
)


summary_recalculated[
    "CI95LowerAPFDc"
] = (
    summary_recalculated[
        "MeanAPFDc"
    ]
    - CI_CRITICAL_95
    * summary_recalculated[
        "SE_APFDc"
    ]
)


summary_recalculated[
    "CI95UpperAPFDc"
] = (
    summary_recalculated[
        "MeanAPFDc"
    ]
    + CI_CRITICAL_95
    * summary_recalculated[
        "SE_APFDc"
    ]
)


summary_recalculated = (
    summary_recalculated[
        summary_package.columns
    ]
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


(
    summary_identity_match,
    summary_identity_diagnostics,
) = semantic_identity_compare(
    left=summary_package,
    right=summary_recalculated,

    text_columns=[
        "Project",
        "Technique",
    ],

    numeric_columns=[
        "NoisePercent",
    ],

    comparison_name=(
        "project_level_summary_recalculation"
    ),
)


summary_numeric_columns = [
    column
    for column in summary_package.columns
    if column not in {
        "Project",
        "Technique",
        "NoisePercent",
    }
]


summary_numeric_match = (
    dataframe_numeric_equal(
        summary_package,
        summary_recalculated,
        summary_numeric_columns,
    )
)


# ------------------------------------------------------------
# 14. RECOMPUTE PROJECT-LEVEL DEGRADATION
# ------------------------------------------------------------

degradation_package = (
    top_level_tables[
        "project_level_degradation_summary.csv"
    ]
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


clean_project_means = (
    summary_recalculated[
        summary_recalculated[
            "NoisePercent"
        ].eq(0)
    ][
        [
            "Technique",
            "MeanAPFD",
            "MeanAPFDc",
        ]
    ]
    .rename(
        columns={
            "MeanAPFD":
                "CleanProjectMeanAPFD",

            "MeanAPFDc":
                "CleanProjectMeanAPFDc",
        }
    )
)


degradation_recalculated = (
    summary_recalculated
    .merge(
        clean_project_means,
        on="Technique",
        how="left",
        validate="many_to_one",
    )
)


degradation_recalculated[
    "DeltaAPFD_NoiseMinusClean"
] = (
    degradation_recalculated[
        "MeanAPFD"
    ]
    - degradation_recalculated[
        "CleanProjectMeanAPFD"
    ]
)


degradation_recalculated[
    "DeltaAPFDc_NoiseMinusClean"
] = (
    degradation_recalculated[
        "MeanAPFDc"
    ]
    - degradation_recalculated[
        "CleanProjectMeanAPFDc"
    ]
)


degradation_recalculated[
    "AbsoluteLossAPFD"
] = (
    degradation_recalculated[
        "CleanProjectMeanAPFD"
    ]
    - degradation_recalculated[
        "MeanAPFD"
    ]
)


degradation_recalculated[
    "AbsoluteLossAPFDc"
] = (
    degradation_recalculated[
        "CleanProjectMeanAPFDc"
    ]
    - degradation_recalculated[
        "MeanAPFDc"
    ]
)


degradation_recalculated[
    "PercentageLossAPFD"
] = np.where(
    degradation_recalculated[
        "CleanProjectMeanAPFD"
    ].ne(0),

    100.0
    * degradation_recalculated[
        "AbsoluteLossAPFD"
    ]
    / degradation_recalculated[
        "CleanProjectMeanAPFD"
    ],

    np.nan,
)


degradation_recalculated[
    "PercentageLossAPFDc"
] = np.where(
    degradation_recalculated[
        "CleanProjectMeanAPFDc"
    ].ne(0),

    100.0
    * degradation_recalculated[
        "AbsoluteLossAPFDc"
    ]
    / degradation_recalculated[
        "CleanProjectMeanAPFDc"
    ],

    np.nan,
)


degradation_recalculated[
    "RetentionAPFD"
] = np.where(
    degradation_recalculated[
        "CleanProjectMeanAPFD"
    ].ne(0),

    RETENTION_SCALE
    * degradation_recalculated[
        "MeanAPFD"
    ]
    / degradation_recalculated[
        "CleanProjectMeanAPFD"
    ],

    np.nan,
)


degradation_recalculated[
    "RetentionAPFDc"
] = np.where(
    degradation_recalculated[
        "CleanProjectMeanAPFDc"
    ].ne(0),

    RETENTION_SCALE
    * degradation_recalculated[
        "MeanAPFDc"
    ]
    / degradation_recalculated[
        "CleanProjectMeanAPFDc"
    ],

    np.nan,
)


degradation_recalculated = (
    degradation_recalculated[
        degradation_package.columns
    ]
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


(
    degradation_identity_match,
    degradation_identity_diagnostics,
) = semantic_identity_compare(
    left=degradation_package,
    right=degradation_recalculated,

    text_columns=[
        "Project",
        "Technique",
    ],

    numeric_columns=[
        "NoisePercent",
    ],

    comparison_name=(
        "project_level_degradation_recalculation"
    ),
)


degradation_numeric_columns = [
    column
    for column in degradation_package.columns
    if column not in {
        "Project",
        "Technique",
        "NoisePercent",
    }
]


degradation_numeric_match = (
    dataframe_numeric_equal(
        degradation_package,
        degradation_recalculated,
        degradation_numeric_columns,
    )
)


summary_recalculation_audit = pd.DataFrame([
    {
        "Table":
            "project_level_noise_technique_summary.csv",

        "Rows":
            len(summary_package),

        "IdentityMatch":
            summary_identity_match,

        "NumericContentMatch":
            summary_numeric_match,

        "CI95CriticalValue":
            CI_CRITICAL_95,

        "RetentionScale":
            RETENTION_SCALE,

        "Pass":
            (
                summary_identity_match
                and summary_numeric_match
            ),
    },

    {
        "Table":
            "project_level_degradation_summary.csv",

        "Rows":
            len(degradation_package),

        "IdentityMatch":
            degradation_identity_match,

        "NumericContentMatch":
            degradation_numeric_match,

        "CI95CriticalValue":
            CI_CRITICAL_95,

        "RetentionScale":
            RETENTION_SCALE,

        "Pass":
            (
                degradation_identity_match
                and degradation_numeric_match
            ),
    },
])


identity_diagnostics = pd.concat(
    [
        identity_diagnostics,

        pd.DataFrame(
            summary_identity_diagnostics
        ),

        pd.DataFrame(
            degradation_identity_diagnostics
        ),
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# 15. REGISTRY IMMUTABILITY
# ------------------------------------------------------------

registry_sha256_after = (
    calculate_sha256(
        REGISTRY_PATH
    )
)


registry_after = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
)


registry_project_numbers_after = (
    pd.to_numeric(
        registry_after[
            "ProjectNumber"
        ],
        errors="coerce",
    )
)


project8_registry_rows_after = int(
    registry_project_numbers_after.eq(
        PROJECT_NUMBER
    ).sum()
)


registry_unchanged = bool(
    registry_sha256_before
    == registry_sha256_after
)


# ------------------------------------------------------------
# 16. OVERALL VALIDATION
# ------------------------------------------------------------

validation_records = [
    {
        "Check":
            "Step 10 passed",

        "Expected":
            EXPECTED_STEP10_STATUS,

        "Actual":
            step10_status["Status"],

        "Pass":
            step10_status["Status"]
            == EXPECTED_STEP10_STATUS,
    },

    {
        "Check":
            "Step 11B passed",

        "Expected":
            EXPECTED_STEP11B_STATUS,

        "Actual":
            step11b_status["Status"],

        "Pass":
            step11b_status["Status"]
            == EXPECTED_STEP11B_STATUS,
    },

    {
        "Check":
            "Package files",

        "Expected":
            EXPECTED_PACKAGE_FILES,

        "Actual":
            package_file_count,

        "Pass":
            package_file_count
            == EXPECTED_PACKAGE_FILES,
    },

    {
        "Check":
            "Package bytes",

        "Expected":
            EXPECTED_PACKAGE_BYTES,

        "Actual":
            package_bytes,

        "Pass":
            package_bytes
            == EXPECTED_PACKAGE_BYTES,
    },

    {
        "Check":
            "Package root SHA-256",

        "Expected":
            EXPECTED_PACKAGE_ROOT_SHA256,

        "Actual":
            package_root_sha256,

        "Pass":
            package_root_sha256
            == EXPECTED_PACKAGE_ROOT_SHA256,
    },

    {
        "Check":
            "Top-level files",

        "Expected":
            EXPECTED_TOP_LEVEL_FILES,

        "Actual":
            len(top_level_paths),

        "Pass":
            len(top_level_paths)
            == EXPECTED_TOP_LEVEL_FILES,
    },

    {
        "Check":
            "Technique-noise files",

        "Expected":
            EXPECTED_TECHNIQUE_NOISE_FILES,

        "Actual":
            len(technique_noise_paths),

        "Pass":
            len(technique_noise_paths)
            == EXPECTED_TECHNIQUE_NOISE_FILES,
    },

    {
        "Check":
            "Step 11B inventory missing files",

        "Expected":
            0,

        "Actual":
            inventory_missing_from_package,

        "Pass":
            inventory_missing_from_package
            == 0,
    },

    {
        "Check":
            "Step 11B inventory unexpected files",

        "Expected":
            0,

        "Actual":
            inventory_unexpected_in_package,

        "Pass":
            inventory_unexpected_in_package
            == 0,
    },

    {
        "Check":
            "Inventory size mismatches",

        "Expected":
            0,

        "Actual":
            inventory_size_mismatches,

        "Pass":
            inventory_size_mismatches
            == 0,
    },

    {
        "Check":
            "Inventory SHA-256 mismatches",

        "Expected":
            0,

        "Actual":
            inventory_hash_mismatches,

        "Pass":
            inventory_hash_mismatches
            == 0,
    },

    {
        "Check":
            "Top-level readback failures",

        "Expected":
            0,

        "Actual":
            len(readback_failures),

        "Pass":
            len(readback_failures)
            == 0,
    },

    {
        "Check":
            "Top-level row mismatches",

        "Expected":
            0,

        "Actual":
            len(top_level_row_mismatches),

        "Pass":
            len(top_level_row_mismatches)
            == 0,
    },

    {
        "Check":
            "Project-run identity matches source",

        "Expected":
            True,

        "Actual":
            project_run_identity_match,

        "Pass":
            project_run_identity_match,
    },

    {
        "Check":
            "Project-run metrics match source",

        "Expected":
            True,

        "Actual":
            project_run_numeric_match,

        "Pass":
            project_run_numeric_match,
    },

    {
        "Check":
            "Build identity matches source",

        "Expected":
            True,

        "Actual":
            build_identity_match,

        "Pass":
            build_identity_match,
    },

    {
        "Check":
            "Build APFD/APFDc match source",

        "Expected":
            True,

        "Actual":
            build_numeric_match,

        "Pass":
            build_numeric_match,
    },

    {
        "Check":
            "Fit-time identity matches source",

        "Expected":
            True,

        "Actual":
            fit_identity_match,

        "Pass":
            fit_identity_match,
    },

    {
        "Check":
            "Fit-time values match source",

        "Expected":
            True,

        "Actual":
            fit_numeric_match,

        "Pass":
            fit_numeric_match,
    },

    {
        "Check":
            "Failed technique-noise files",

        "Expected":
            0,

        "Actual":
            failed_technique_noise_files,

        "Pass":
            failed_technique_noise_files
            == 0,
    },

    {
        "Check":
            "Project summary recalculation",

        "Expected":
            True,

        "Actual":
            (
                summary_identity_match
                and summary_numeric_match
            ),

        "Pass":
            (
                summary_identity_match
                and summary_numeric_match
            ),
    },

    {
        "Check":
            "Degradation summary recalculation",

        "Expected":
            True,

        "Actual":
            (
                degradation_identity_match
                and degradation_numeric_match
            ),

        "Pass":
            (
                degradation_identity_match
                and degradation_numeric_match
            ),
    },

    {
        "Check":
            "Project 7 identity files",

        "Expected":
            0,

        "Actual":
            len(project7_identity_files),

        "Pass":
            len(project7_identity_files)
            == 0,
    },

    {
        "Check":
            "Invalid APFD/APFDc cells",

        "Expected":
            0,

        "Actual":
            len(
                invalid_metric_files_frame
            ),

        "Pass":
            len(
                invalid_metric_files_frame
            )
            == 0,
    },

    {
        "Check":
            "Completion registry unchanged",

        "Expected":
            True,

        "Actual":
            registry_unchanged,

        "Pass":
            registry_unchanged,
    },

    {
        "Check":
            "Project 8 registry rows",

        "Expected":
            0,

        "Actual":
            project8_registry_rows_after,

        "Pass":
            project8_registry_rows_after
            == 0,
    },
]


freeze_validation = pd.DataFrame(
    validation_records
)


failed_checks = (
    freeze_validation[
        ~freeze_validation[
            "Pass"
        ]
    ]
    .copy()
)


print("\nIndependent freeze validation:")

display(
    freeze_validation
)


if not failed_checks.empty:
    print("\nFailed checks:")

    display(
        failed_checks
    )

    raise RuntimeError(
        "PROJECT 8 STEP 11C V2 DID NOT PASS.\n"
        "Do not update the completion registry."
    )


# ------------------------------------------------------------
# 17. WRITE STEP 11C OUTPUTS
# ------------------------------------------------------------

FREEZE_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_write_csv(
    FREEZE_INVENTORY_PATH,
    freeze_inventory,
)


atomic_write_csv(
    FREEZE_VALIDATION_PATH,
    freeze_validation,
)


atomic_write_csv(
    TECHNIQUE_NOISE_AUDIT_PATH,
    technique_noise_audit,
)


atomic_write_csv(
    SUMMARY_RECALCULATION_AUDIT_PATH,
    summary_recalculation_audit,
)


atomic_write_csv(
    IDENTITY_DIAGNOSTICS_PATH,
    identity_diagnostics,
)


freeze_report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "RawResultFreeze": {
        "Files":
            EXPECTED_RAW_FILES,

        "Bytes":
            EXPECTED_RAW_BYTES,

        "RootSHA256":
            EXPECTED_RAW_ROOT_SHA256,
    },

    "FinalPackageFreeze": {
        "Directory":
            str(FINAL_PACKAGE_DIR),

        "Files":
            package_file_count,

        "Bytes":
            package_bytes,

        "RootSHA256":
            package_root_sha256,

        "TopLevelFiles":
            len(top_level_paths),

        "TechniqueNoiseFiles":
            len(technique_noise_paths),

        "InventorySizeMismatches":
            inventory_size_mismatches,

        "InventoryHashMismatches":
            inventory_hash_mismatches,
    },

    "IndependentContentValidation": {
        "ProjectRunIdentityMatch":
            project_run_identity_match,

        "ProjectRunNumericMatch":
            project_run_numeric_match,

        "BuildIdentityMatch":
            build_identity_match,

        "BuildNumericMatch":
            build_numeric_match,

        "FitIdentityMatch":
            fit_identity_match,

        "FitNumericMatch":
            fit_numeric_match,

        "FailedTechniqueNoiseFiles":
            failed_technique_noise_files,

        "ProjectSummaryRecalculationPass":
            (
                summary_identity_match
                and summary_numeric_match
            ),

        "DegradationSummaryRecalculationPass":
            (
                degradation_identity_match
                and degradation_numeric_match
            ),

        "CI95CriticalValue":
            CI_CRITICAL_95,

        "RetentionScale":
            RETENTION_SCALE,

        "Project7IdentityFiles":
            len(project7_identity_files),

        "InvalidMetricCells":
            len(
                invalid_metric_files_frame
            ),
    },

    "ValidationChecks":
        len(freeze_validation),

    "FailedValidationChecks":
        len(failed_checks),

    "CompletionRegistryModified":
        False,

    "Projects1To7Modified":
        False,

    "Status":
        STEP11C_PASS_STATUS,
}


atomic_write_json(
    FREEZE_REPORT_PATH,
    freeze_report,
)


step11c_status = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP11C_PASS_STATUS,

    "RawFiles":
        EXPECTED_RAW_FILES,

    "RawBytes":
        EXPECTED_RAW_BYTES,

    "RawRootSHA256":
        EXPECTED_RAW_ROOT_SHA256,

    "PackageFiles":
        package_file_count,

    "PackageBytes":
        package_bytes,

    "PackageRootSHA256":
        package_root_sha256,

    "FailedValidationChecks":
        len(failed_checks),

    "CompletionRegistryModified":
        False,

    "FrozenAtUTC":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_write_json(
    STEP11C_STATUS_PATH,
    step11c_status,
)


expected_outputs = [
    FREEZE_INVENTORY_PATH,
    FREEZE_VALIDATION_PATH,
    TECHNIQUE_NOISE_AUDIT_PATH,
    SUMMARY_RECALCULATION_AUDIT_PATH,
    IDENTITY_DIAGNOSTICS_PATH,
    FREEZE_REPORT_PATH,
    STEP11C_STATUS_PATH,
]


missing_outputs = [
    str(path)
    for path in expected_outputs
    if not path.exists()
]


if missing_outputs:
    raise RuntimeError(
        "Step 11C V2 outputs are missing:\n"
        + "\n".join(missing_outputs)
    )


# ------------------------------------------------------------
# 18. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 102)
print("=== PROJECT 8 STEP 11C V2 RESULT ===")
print("=" * 102)

print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nRaw result freeze:")

print(
    "Raw files:",
    EXPECTED_RAW_FILES,
)

print(
    "Raw bytes:",
    EXPECTED_RAW_BYTES,
)

print(
    "Raw root SHA-256:",
    EXPECTED_RAW_ROOT_SHA256,
)


print("\nFinal package freeze:")

print(
    "Package directory:",
    FINAL_PACKAGE_DIR,
)

print(
    "Package files:",
    package_file_count,
)

print(
    "Package bytes:",
    package_bytes,
)

print(
    "Package root SHA-256:",
    package_root_sha256,
)

print(
    "Top-level files:",
    len(top_level_paths),
)

print(
    "Technique-noise files:",
    len(technique_noise_paths),
)

print(
    "Inventory size mismatches:",
    inventory_size_mismatches,
)

print(
    "Inventory SHA-256 mismatches:",
    inventory_hash_mismatches,
)


print("\nIndependent content validation:")

print(
    "Project-run identity match:",
    project_run_identity_match,
)

print(
    "Project-run metric match:",
    project_run_numeric_match,
)

print(
    "Build identity match:",
    build_identity_match,
)

print(
    "Build APFD/APFDc match:",
    build_numeric_match,
)

print(
    "Fit-time identity match:",
    fit_identity_match,
)

print(
    "Fit-time value match:",
    fit_numeric_match,
)

print(
    "Failed technique-noise files:",
    failed_technique_noise_files,
)

print(
    "Project summary recalculation pass:",
    (
        summary_identity_match
        and summary_numeric_match
    ),
)

print(
    "Degradation summary recalculation pass:",
    (
        degradation_identity_match
        and degradation_numeric_match
    ),
)

print(
    "Project 7 identity files:",
    len(project7_identity_files),
)

print(
    "Invalid APFD/APFDc cells:",
    len(
        invalid_metric_files_frame
    ),
)


print("\nValidation:")

print(
    "Checks:",
    len(freeze_validation),
)

print(
    "Failed checks:",
    len(failed_checks),
)


print("\nStep 11C V2 outputs:")

for output_path in expected_outputs:
    print(output_path)


print("\nCompletion registry modified:")
print(0)

print("\nProjects 1–7 modified:")
print(0)


print(
    "\nSTATUS:",
    STEP11C_PASS_STATUS,
)

print("=" * 102)

=== PROJECT 8 STEP 11C V2: INDEPENDENT FINAL-PACKAGE AUDIT AND FREEZE ===

Prerequisite validation:
Step 10: PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED
Step 11B: PASS_PROJECT_8_FINAL_72_FILE_PACKAGE_GENERATED_AND_VALIDATED
Recorded package files: 72
Recorded package bytes: 1242844
Recorded package root SHA-256: 1e27d6f57526d3ce819b7358867c0c58411cc4eaddcdcb2e5576b2282a6cd607
CI critical value: 2.045229642132711
Retention scale: 1.0
Project 8 registry rows: 0

Hashing final-package files...
Hashed: 20 / 72
Hashed: 40 / 72
Hashed: 60 / 72
Hashed: 72 / 72

Independent package inventory:
Files: 72
Bytes: 1242844
Top-level files: 9
Technique-noise files: 63
Package root SHA-256: 1e27d6f57526d3ce819b7358867c0c58411cc4eaddcdcb2e5576b2282a6cd607

Identity comparison diagnostics:


,Comparison,Column,ColumnType,MismatchCount,LeftExample,RightExample
0,project_run_package_vs_audited_source,__ALL_IDENTITY_COLUMNS__,summary,0,None,None
1,build_metrics_package_vs_audited_source,__ALL_IDENTITY_COLUMNS__,summary,0,None,None
2,fit_times_package_vs_audited_source,__ALL_IDENTITY_COLUMNS__,summary,0,None,None



Independent freeze validation:


,Check,Expected,Actual,Pass
0,Step 10 passed,PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED,PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED,True
1,Step 11B passed,PASS_PROJECT_8_FINAL_72_FILE_PACKAGE_GENERATED...,PASS_PROJECT_8_FINAL_72_FILE_PACKAGE_GENERATED...,True
2,Package files,72,72,True
3,Package bytes,1242844,1242844,True
4,Package root SHA-256,1e27d6f57526d3ce819b7358867c0c58411cc4eaddcdcb...,1e27d6f57526d3ce819b7358867c0c58411cc4eaddcdcb...,True
5,Top-level files,9,9,True
6,Technique-noise files,63,63,True
7,Step 11B inventory missing files,0,0,True
8,Step 11B inventory unexpected files,0,0,True
9,Inventory size mismatches,0,0,True




=== PROJECT 8 STEP 11C V2 RESULT ===

Project identity:
Project number: 8
Project: optimatika@ojAlgo
Project slug: optimatika__ojAlgo

Raw result freeze:
Raw files: 2160
Raw bytes: 179766494
Raw root SHA-256: 19ae21c5d8524c358796e28f23577fbf03bfd31f7d4b530372c6b4d8a492887c

Final package freeze:
Package directory: /content/drive/MyDrive/Thesis_Experiment/Results/Aggregated/optimatika__ojAlgo/ojalgo_30_seed_final
Package files: 72
Package bytes: 1242844
Package root SHA-256: 1e27d6f57526d3ce819b7358867c0c58411cc4eaddcdcb2e5576b2282a6cd607
Top-level files: 9
Technique-noise files: 63
Inventory size mismatches: 0
Inventory SHA-256 mismatches: 0

Independent content validation:
Project-run identity match: True
Project-run metric match: True
Build identity match: True
Build APFD/APFDc match: True
Fit-time identity match: True
Fit-time value match: True
Failed technique-noise files: 0
Project summary recalculation pass: True
Degradation summary recalculation pass: True
Project 7 identity f

In [ ]:
# ============================================================
# PROJECT 8 — STEP 11D
# FINAL COMPLETION-REGISTRY UPDATE AND CHECKPOINT FREEZE
#
# PROJECT: optimatika@ojAlgo
#
# Prerequisites:
# - Step 10 passed
# - Step 11B passed
# - Step 11C passed
# - Project 8 is absent from the completion registry
#
# This cell:
# - independently verifies the frozen package again
# - creates project_08_selection_checkpoint.json
# - appends Project 8 to the completion registry
# - validates that Projects 1–7 registry rows are unchanged
# - marks Project 8 COMPLETE_AND_FROZEN
#
# It does NOT:
# - rerun models
# - regenerate results
# - modify the 72-file final package
# - modify raw Project 8 results
# - modify Projects 1–7 result directories
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import re
import shutil

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. CONSTANTS
# ------------------------------------------------------------

PROJECT_NUMBER = 8
PROJECT_NAME = "optimatika@ojAlgo"
PROJECT_SLUG = "optimatika__ojAlgo"
PROJECT_SHORT_NAME = "ojalgo"

REFERENCE_PROJECT_NUMBER = 7
REFERENCE_PROJECT_NAME = "CompEvol@beast2"
REFERENCE_PROJECT_SLUG = "CompEvol__beast2"
REFERENCE_PROJECT_SHORT_NAME = "beast2"

COMPLETION_STATUS = "COMPLETE_AND_FROZEN"

EXPECTED_STEP10_STATUS = (
    "PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED"
)

EXPECTED_STEP11B_STATUS = (
    "PASS_PROJECT_8_FINAL_72_FILE_PACKAGE_GENERATED_AND_VALIDATED"
)

EXPECTED_STEP11C_STATUS = (
    "PASS_PROJECT_8_FINAL_PACKAGE_INDEPENDENTLY_AUDITED_AND_FROZEN"
)

STEP11D_PASS_STATUS = (
    "PASS_PROJECT_8_COMPLETION_REGISTRY_UPDATED_AND_FINAL_FREEZE_RECORDED"
)

EXPECTED_RAW_FILES = 2160
EXPECTED_RAW_BYTES = 179766494

EXPECTED_RAW_ROOT_SHA256 = (
    "19ae21c5d8524c358796e28f23577fbf03bfd31f7d4b530372c6b4d8a492887c"
)

EXPECTED_PACKAGE_FILES = 72
EXPECTED_PACKAGE_BYTES = 1242844

EXPECTED_PACKAGE_ROOT_SHA256 = (
    "1e27d6f57526d3ce819b7358867c0c58411cc4eaddcdcb2e5576b2282a6cd607"
)

EXPECTED_CONDITIONS = 270
EXPECTED_ML_FITS = 1080
EXPECTED_PROJECT_RUN_ROWS = 1890
EXPECTED_BUILD_METRIC_ROWS = 30240
EXPECTED_PREDICTION_SUMMARIES = 1080

EXPECTED_TOP_LEVEL_FILES = 9
EXPECTED_TECHNIQUE_NOISE_FILES = 63

EXPECTED_REGISTRY_ROWS_BEFORE = 7
EXPECTED_REGISTRY_ROWS_AFTER = 8

COMPLETED_AT_UTC = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


# Project 7 hashes that must never survive in Project 8's row.

PROJECT7_RAW_ROOT_SHA256 = (
    "93a6581d90895fd011b48490aa66f55a91b0078818bfd6966b3cf5b9e0f86373"
)

PROJECT7_PACKAGE_ROOT_SHA256 = (
    "93e9cc74fac017fef89b005752e1b7ede2a73f9665dd1840891e7f6d9c341c13"
)


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_DIR = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_DIR = (
    THESIS_ROOT
    / "Results"
)

PROJECT_DIR = (
    RESULTS_DIR
    / "Aggregated"
    / PROJECT_SLUG
)

RAW_RESULTS_DIR = (
    RESULTS_DIR
    / "Raw"
    / PROJECT_SLUG
)

RAW_AUDIT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_raw_audit"
)

FINAL_PACKAGE_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_30_seed_final"
)

FINAL_PACKAGE_AUDIT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_audit"
)

FINAL_FREEZE_AUDIT_DIR = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_final_freeze_audit"
)


# Status inputs

STEP10_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step10_status.json"
)

STEP11B_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step11b_status.json"
)

STEP11C_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step11c_status.json"
)


# Step 10 outputs

RAW_AUDIT_REPORT_PATH = (
    RAW_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_raw_audit_report.json"
)

RAW_FILE_INVENTORY_PATH = (
    RAW_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_raw_file_manifest.csv.gz"
)


# Step 11B outputs

FINAL_PACKAGE_GENERATION_REPORT_PATH = (
    FINAL_PACKAGE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_report.json"
)

FINAL_PACKAGE_GENERATION_INVENTORY_PATH = (
    FINAL_PACKAGE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_file_inventory_sha256.csv"
)

FINAL_PACKAGE_GENERATION_VALIDATION_PATH = (
    FINAL_PACKAGE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_package_validation.csv"
)


# Step 11C outputs

FINAL_FREEZE_REPORT_PATH = (
    FINAL_FREEZE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_freeze_report.json"
)

FINAL_FREEZE_INVENTORY_PATH = (
    FINAL_FREEZE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_freeze_file_inventory_sha256.csv"
)

FINAL_FREEZE_VALIDATION_PATH = (
    FINAL_FREEZE_AUDIT_DIR
    / f"{PROJECT_SHORT_NAME}_final_freeze_validation.csv"
)


# Registry and checkpoint

REGISTRY_PATH = (
    NOTES_DIR
    / "completed_project_registry.csv"
)

PROJECT8_CHECKPOINT_PATH = (
    NOTES_DIR
    / "project_08_selection_checkpoint.json"
)

REGISTRY_BACKUP_DIR = (
    NOTES_DIR
    / "completion_registry_backups"
)

STEP11D_AUDIT_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step11d_registry_audit.csv"
)

STEP11D_STATUS_PATH = (
    PROJECT_DIR
    / f"{PROJECT_SHORT_NAME}_step11d_status.json"
)


print("=" * 104)
print("=== PROJECT 8 STEP 11D: COMPLETION REGISTRY UPDATE AND FINAL FREEZE RECORD ===")
print("=" * 104)


# ------------------------------------------------------------
# 3. HELPERS
# ------------------------------------------------------------

def calculate_sha256(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def root_inventory_hash(
    inventory,
):
    digest = hashlib.sha256()

    ordered = inventory.sort_values(
        "RelativePath",
        kind="mergesort",
    )

    for row in ordered.itertuples(
        index=False
    ):

        digest.update(
            row.RelativePath.encode(
                "utf-8"
            )
        )

        digest.update(b"\0")

        digest.update(
            str(
                int(row.SizeBytes)
            ).encode(
                "utf-8"
            )
        )

        digest.update(b"\0")

        digest.update(
            bytes.fromhex(
                row.SHA256
            )
        )

        digest.update(b"\n")

    return digest.hexdigest()


def normalise_column_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


def json_safe(value):

    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    try:

        if pd.isna(value):
            return None

    except Exception:
        pass

    return value


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            default=json_safe,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def atomic_write_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(path)


def atomic_restore_bytes(
    path,
    content,
):
    path = Path(path)

    temporary_path = path.with_name(
        path.name + ".restore.tmp"
    )

    temporary_path.write_bytes(
        content
    )

    temporary_path.replace(path)


def find_column(
    dataframe,
    candidates,
    required=True,
):
    lookup = {
        normalise_column_name(column):
            column
        for column in dataframe.columns
    }

    for candidate in candidates:

        normalised_candidate = (
            normalise_column_name(
                candidate
            )
        )

        if normalised_candidate in lookup:

            return lookup[
                normalised_candidate
            ]

    if required:

        raise RuntimeError(
            "Required registry column was not found.\n"
            f"Candidates: {candidates}\n"
            f"Available: {list(dataframe.columns)}"
        )

    return None


def transform_project7_value(
    value,
):
    if pd.isna(value):
        return ""

    text = str(value)

    replacements = [
        (
            REFERENCE_PROJECT_NAME,
            PROJECT_NAME,
        ),
        (
            REFERENCE_PROJECT_SLUG,
            PROJECT_SLUG,
        ),
        (
            REFERENCE_PROJECT_SHORT_NAME,
            PROJECT_SHORT_NAME,
        ),
        (
            "project_07",
            "project_08",
        ),
        (
            "Project 7",
            "Project 8",
        ),
        (
            PROJECT7_RAW_ROOT_SHA256,
            EXPECTED_RAW_ROOT_SHA256,
        ),
        (
            PROJECT7_PACKAGE_ROOT_SHA256,
            EXPECTED_PACKAGE_ROOT_SHA256,
        ),
        (
            "159516190",
            str(
                EXPECTED_RAW_BYTES
            ),
        ),
        (
            "1624833",
            str(
                EXPECTED_PACKAGE_BYTES
            ),
        ),
    ]

    for old_value, new_value in replacements:

        text = text.replace(
            old_value,
            new_value,
        )

    return text


def project_number_series(
    dataframe,
    project_number_column,
):
    return pd.to_numeric(
        dataframe[
            project_number_column
        ],
        errors="coerce",
    )


# ------------------------------------------------------------
# 4. VALIDATE ALL PREREQUISITES
# ------------------------------------------------------------

required_paths = [
    STEP10_STATUS_PATH,
    STEP11B_STATUS_PATH,
    STEP11C_STATUS_PATH,
    RAW_AUDIT_REPORT_PATH,
    RAW_FILE_INVENTORY_PATH,
    FINAL_PACKAGE_DIR,
    FINAL_PACKAGE_GENERATION_REPORT_PATH,
    FINAL_PACKAGE_GENERATION_INVENTORY_PATH,
    FINAL_PACKAGE_GENERATION_VALIDATION_PATH,
    FINAL_FREEZE_REPORT_PATH,
    FINAL_FREEZE_INVENTORY_PATH,
    FINAL_FREEZE_VALIDATION_PATH,
    REGISTRY_PATH,
]


missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]


if missing_paths:

    raise FileNotFoundError(
        "Required Step 11D inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


step10_status = json.loads(
    STEP10_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


step11b_status = json.loads(
    STEP11B_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


step11c_status = json.loads(
    STEP11C_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    step10_status.get("Status")
    != EXPECTED_STEP10_STATUS
):

    raise AssertionError(
        "Step 10 status differs.\n"
        f"Actual: {step10_status.get('Status')}"
    )


if (
    step11b_status.get("Status")
    != EXPECTED_STEP11B_STATUS
):

    raise AssertionError(
        "Step 11B status differs.\n"
        f"Actual: {step11b_status.get('Status')}"
    )


if (
    step11c_status.get("Status")
    != EXPECTED_STEP11C_STATUS
):

    raise AssertionError(
        "Step 11C status differs.\n"
        f"Actual: {step11c_status.get('Status')}"
    )


status_value_checks = {
    "Step10 raw files":
        (
            int(
                step10_status.get(
                    "RawFiles",
                    -1,
                )
            ),
            EXPECTED_RAW_FILES,
        ),

    "Step10 raw bytes":
        (
            int(
                step10_status.get(
                    "RawBytes",
                    -1,
                )
            ),
            EXPECTED_RAW_BYTES,
        ),

    "Step11B package files":
        (
            int(
                step11b_status.get(
                    "PackageFiles",
                    -1,
                )
            ),
            EXPECTED_PACKAGE_FILES,
        ),

    "Step11B package bytes":
        (
            int(
                step11b_status.get(
                    "PackageBytes",
                    -1,
                )
            ),
            EXPECTED_PACKAGE_BYTES,
        ),

    "Step11C package files":
        (
            int(
                step11c_status.get(
                    "PackageFiles",
                    -1,
                )
            ),
            EXPECTED_PACKAGE_FILES,
        ),

    "Step11C package bytes":
        (
            int(
                step11c_status.get(
                    "PackageBytes",
                    -1,
                )
            ),
            EXPECTED_PACKAGE_BYTES,
        ),
}


for check_name, (
    actual_value,
    expected_value,
) in status_value_checks.items():

    if actual_value != expected_value:

        raise AssertionError(
            f"{check_name} differs.\n"
            f"Expected: {expected_value}\n"
            f"Actual: {actual_value}"
        )


if (
    step10_status.get(
        "RawRootSHA256"
    )
    != EXPECTED_RAW_ROOT_SHA256
):

    raise AssertionError(
        "Step 10 raw-root SHA-256 differs."
    )


if (
    step11b_status.get(
        "PackageRootSHA256"
    )
    != EXPECTED_PACKAGE_ROOT_SHA256
):

    raise AssertionError(
        "Step 11B package-root SHA-256 differs."
    )


if (
    step11c_status.get(
        "PackageRootSHA256"
    )
    != EXPECTED_PACKAGE_ROOT_SHA256
):

    raise AssertionError(
        "Step 11C package-root SHA-256 differs."
    )


print("\nPrerequisite validation:")

print(
    "Step 10:",
    step10_status["Status"],
)

print(
    "Step 11B:",
    step11b_status["Status"],
)

print(
    "Step 11C:",
    step11c_status["Status"],
)

print(
    "Raw root SHA-256:",
    EXPECTED_RAW_ROOT_SHA256,
)

print(
    "Final package root SHA-256:",
    EXPECTED_PACKAGE_ROOT_SHA256,
)


# ------------------------------------------------------------
# 5. REHASH THE FROZEN PACKAGE BEFORE REGISTRY UPDATE
# ------------------------------------------------------------

package_files = sorted([
    path
    for path in FINAL_PACKAGE_DIR.rglob("*")
    if path.is_file()
])


package_inventory_records = []


for path in package_files:

    package_inventory_records.append({
        "RelativePath":
            path.relative_to(
                FINAL_PACKAGE_DIR
            ).as_posix(),

        "SizeBytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            calculate_sha256(
                path
            ),
    })


current_package_inventory = pd.DataFrame(
    package_inventory_records
)


current_package_files = int(
    len(
        current_package_inventory
    )
)


current_package_bytes = int(
    current_package_inventory[
        "SizeBytes"
    ].sum()
)


current_package_root_sha256 = (
    root_inventory_hash(
        current_package_inventory
    )
)


current_top_level_files = int(
    sum(
        1
        for relative_path
        in current_package_inventory[
            "RelativePath"
        ]
        if not relative_path.startswith(
            "technique_noise/"
        )
    )
)


current_technique_noise_files = int(
    sum(
        1
        for relative_path
        in current_package_inventory[
            "RelativePath"
        ]
        if relative_path.startswith(
            "technique_noise/"
        )
    )
)


if current_package_files != EXPECTED_PACKAGE_FILES:

    raise AssertionError(
        "Current final-package file count differs."
    )


if current_package_bytes != EXPECTED_PACKAGE_BYTES:

    raise AssertionError(
        "Current final-package byte count differs."
    )


if (
    current_package_root_sha256
    != EXPECTED_PACKAGE_ROOT_SHA256
):

    raise AssertionError(
        "Current final-package root SHA-256 differs."
    )


if (
    current_top_level_files
    != EXPECTED_TOP_LEVEL_FILES
):

    raise AssertionError(
        "Current top-level package file count differs."
    )


if (
    current_technique_noise_files
    != EXPECTED_TECHNIQUE_NOISE_FILES
):

    raise AssertionError(
        "Current technique-noise file count differs."
    )


print("\nFrozen package revalidation:")

print(
    "Files:",
    current_package_files,
)

print(
    "Bytes:",
    current_package_bytes,
)

print(
    "Top-level files:",
    current_top_level_files,
)

print(
    "Technique-noise files:",
    current_technique_noise_files,
)

print(
    "Root SHA-256:",
    current_package_root_sha256,
)


# ------------------------------------------------------------
# 6. LOAD AND VALIDATE REGISTRY BEFORE UPDATE
# ------------------------------------------------------------

registry_original_bytes = (
    REGISTRY_PATH.read_bytes()
)


registry_sha256_before = (
    calculate_sha256(
        REGISTRY_PATH
    )
)


registry_before = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = find_column(
    registry_before,
    [
        "ProjectNumber",
        "ProjectNo",
        "ProjectIndex",
    ],
)


status_column = find_column(
    registry_before,
    [
        "Status",
        "CompletionStatus",
        "State",
    ],
)


project_numbers_before = (
    project_number_series(
        registry_before,
        project_number_column,
    )
)


if len(
    registry_before
) != EXPECTED_REGISTRY_ROWS_BEFORE:

    raise AssertionError(
        "Registry row count before update differs.\n"
        f"Expected: {EXPECTED_REGISTRY_ROWS_BEFORE}\n"
        f"Actual: {len(registry_before)}"
    )


if set(
    project_numbers_before.dropna().astype(int)
) != set(
    range(1, 8)
):

    raise AssertionError(
        "Registry does not contain exactly Projects 1–7."
    )


if project_numbers_before.duplicated().any():

    raise AssertionError(
        "Duplicate project numbers exist before update."
    )


project8_rows_before = int(
    project_numbers_before.eq(
        PROJECT_NUMBER
    ).sum()
)


if project8_rows_before != 0:

    raise AssertionError(
        "Project 8 is already present in the registry."
    )


complete_rows_before = int(
    registry_before[
        status_column
    ].eq(
        COMPLETION_STATUS
    ).sum()
)


if complete_rows_before != 7:

    raise AssertionError(
        "Registry does not contain seven "
        "COMPLETE_AND_FROZEN projects."
    )


project7_template_rows = registry_before[
    project_numbers_before.eq(
        REFERENCE_PROJECT_NUMBER
    )
]


if len(
    project7_template_rows
) != 1:

    raise AssertionError(
        "Exactly one Project 7 template row is required."
    )


project7_template = (
    project7_template_rows.iloc[0]
)


print("\nRegistry before update:")

print(
    "Rows:",
    len(
        registry_before
    ),
)

print(
    "Projects:",
    sorted(
        project_numbers_before
        .dropna()
        .astype(int)
        .tolist()
    ),
)

print(
    "COMPLETE_AND_FROZEN rows:",
    complete_rows_before,
)

print(
    "Registry SHA-256:",
    registry_sha256_before,
)


# ------------------------------------------------------------
# 7. CONSTRUCT PROJECT 8 REGISTRY ROW
# ------------------------------------------------------------

new_row = {
    column:
        transform_project7_value(
            project7_template[
                column
            ]
        )
    for column in registry_before.columns
}


# Explicit values keyed by normalised registry-column names.
# These override the transformed Project 7 template.

explicit_registry_values = {
    # Identity
    "projectnumber":
        PROJECT_NUMBER,

    "projectno":
        PROJECT_NUMBER,

    "projectindex":
        PROJECT_NUMBER,

    "project":
        PROJECT_NAME,

    "projectname":
        PROJECT_NAME,

    "repository":
        PROJECT_NAME,

    "repositoryname":
        PROJECT_NAME,

    "projectslug":
        PROJECT_SLUG,

    "slug":
        PROJECT_SLUG,

    "shortname":
        PROJECT_SHORT_NAME,

    "projectshortname":
        PROJECT_SHORT_NAME,

    # Status
    "status":
        COMPLETION_STATUS,

    "completionstatus":
        COMPLETION_STATUS,

    "state":
        COMPLETION_STATUS,

    # Experiment dimensions
    "conditions":
        EXPECTED_CONDITIONS,

    "conditioncount":
        EXPECTED_CONDITIONS,

    "totalconditions":
        EXPECTED_CONDITIONS,

    "noiselevels":
        len(
            [
                0,
                5,
                10,
                15,
                20,
                25,
                30,
                40,
                50,
            ]
        ),

    "noisecount":
        9,

    "seeds":
        30,

    "seedcount":
        30,

    "techniques":
        7,

    "techniquecount":
        7,

    "mlfits":
        EXPECTED_ML_FITS,

    "modelfits":
        EXPECTED_ML_FITS,

    "predictionsummaries":
        EXPECTED_PREDICTION_SUMMARIES,

    "predictionsummaryrows":
        EXPECTED_PREDICTION_SUMMARIES,

    "projectrunrows":
        EXPECTED_PROJECT_RUN_ROWS,

    "projectrunmetricrows":
        EXPECTED_PROJECT_RUN_ROWS,

    "buildmetricrows":
        EXPECTED_BUILD_METRIC_ROWS,

    # Raw freeze
    "rawfiles":
        EXPECTED_RAW_FILES,

    "rawfilecount":
        EXPECTED_RAW_FILES,

    "rawoutputfiles":
        EXPECTED_RAW_FILES,

    "rawbytes":
        EXPECTED_RAW_BYTES,

    "rawoutputbytes":
        EXPECTED_RAW_BYTES,

    "rawrootsha256":
        EXPECTED_RAW_ROOT_SHA256,

    "rawsha256":
        EXPECTED_RAW_ROOT_SHA256,

    "rawauditrootsha256":
        EXPECTED_RAW_ROOT_SHA256,

    "rawoutputrootsha256":
        EXPECTED_RAW_ROOT_SHA256,

    "rawoutputdirectory":
        str(
            RAW_RESULTS_DIR
        ),

    "rawoutputauditdirectory":
        str(
            RAW_AUDIT_DIR
        ),

    "rawoutputauditreport":
        str(
            RAW_AUDIT_REPORT_PATH
        ),

    "rawfileinventory":
        str(
            RAW_FILE_INVENTORY_PATH
        ),

    # Package freeze
    "packagefiles":
        EXPECTED_PACKAGE_FILES,

    "packagefilecount":
        EXPECTED_PACKAGE_FILES,

    "finalpackagefiles":
        EXPECTED_PACKAGE_FILES,

    "packagebytes":
        EXPECTED_PACKAGE_BYTES,

    "finalpackagebytes":
        EXPECTED_PACKAGE_BYTES,

    "packagerootsha256":
        EXPECTED_PACKAGE_ROOT_SHA256,

    "packagesha256":
        EXPECTED_PACKAGE_ROOT_SHA256,

    "finalpackagerootsha256":
        EXPECTED_PACKAGE_ROOT_SHA256,

    "frozenfinalpackagerootsha256":
        EXPECTED_PACKAGE_ROOT_SHA256,

    "finalpackagedirectory":
        str(
            FINAL_PACKAGE_DIR
        ),

    "finalpackageinventory":
        str(
            FINAL_FREEZE_INVENTORY_PATH
        ),

    "finalpackagevalidation":
        str(
            FINAL_FREEZE_VALIDATION_PATH
        ),

    "finalpackagereport":
        str(
            FINAL_FREEZE_REPORT_PATH
        ),

    "toplevelfiles":
        EXPECTED_TOP_LEVEL_FILES,

    "techniquenoisefiles":
        EXPECTED_TECHNIQUE_NOISE_FILES,

    # Checkpoint
    "checkpoint":
        str(
            PROJECT8_CHECKPOINT_PATH
        ),

    "checkpointpath":
        str(
            PROJECT8_CHECKPOINT_PATH
        ),

    "selectioncheckpoint":
        str(
            PROJECT8_CHECKPOINT_PATH
        ),

    "selectioncheckpointpath":
        str(
            PROJECT8_CHECKPOINT_PATH
        ),

    # Registry path
    "completionregistry":
        str(
            REGISTRY_PATH
        ),

    "completionregistrypath":
        str(
            REGISTRY_PATH
        ),

    # A registry cannot contain its own final hash
    # without creating a self-reference.
    "completionregistrysha256":
        "",

    "registrysha256":
        "",

    # Time fields
    "completedatutc":
        COMPLETED_AT_UTC,

    "completiontimestamp":
        COMPLETED_AT_UTC,

    "frozenatutc":
        COMPLETED_AT_UTC,

    "updatedatutc":
        COMPLETED_AT_UTC,

    "createdatutc":
        COMPLETED_AT_UTC,
}


for column in registry_before.columns:

    normalised_column = (
        normalise_column_name(
            column
        )
    )

    if normalised_column in (
        explicit_registry_values
    ):

        new_row[
            column
        ] = str(
            explicit_registry_values[
                normalised_column
            ]
        )


# Ensure the exact detected key fields are set.

new_row[
    project_number_column
] = str(
    PROJECT_NUMBER
)


new_row[
    status_column
] = COMPLETION_STATUS


# Reject Project 7 identity or hash contamination.

for column, value in new_row.items():

    text = str(value).lower()

    forbidden_values = [
        "compevol@beast2",
        "compevol__beast2",
        PROJECT7_RAW_ROOT_SHA256.lower(),
        PROJECT7_PACKAGE_ROOT_SHA256.lower(),
    ]

    for forbidden_value in forbidden_values:

        if forbidden_value in text:

            raise AssertionError(
                "Project 7 value survived in the "
                "Project 8 registry row.\n"
                f"Column: {column}\n"
                f"Value: {value}"
            )


# Any unknown SHA/hash field must not retain an unrelated hash.

approved_hashes = {
    "",
    EXPECTED_RAW_ROOT_SHA256.lower(),
    EXPECTED_PACKAGE_ROOT_SHA256.lower(),
}


for column, value in new_row.items():

    normalised_column = (
        normalise_column_name(
            column
        )
    )

    text = str(value).strip().lower()

    looks_like_hash = bool(
        re.fullmatch(
            r"[0-9a-f]{64}",
            text,
        )
    )

    if (
        looks_like_hash
        and (
            "sha256" in normalised_column
            or "hash" in normalised_column
        )
        and text not in approved_hashes
    ):

        raise AssertionError(
            "An unrecognised hash remains in the "
            "Project 8 registry row.\n"
            f"Column: {column}\n"
            f"Value: {value}"
        )


project8_row = pd.DataFrame(
    [
        new_row
    ],
    columns=registry_before.columns,
)


registry_candidate = pd.concat(
    [
        registry_before,
        project8_row,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# 8. CREATE REGISTRY BACKUP
# ------------------------------------------------------------

REGISTRY_BACKUP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


backup_timestamp = (
    datetime.now(
        timezone.utc
    )
    .strftime(
        "%Y%m%dT%H%M%S%fZ"
    )
)


registry_backup_path = (
    REGISTRY_BACKUP_DIR
    / (
        "completed_project_registry"
        f"__before_project_08__{backup_timestamp}.csv"
    )
)


registry_backup_path.write_bytes(
    registry_original_bytes
)


if (
    calculate_sha256(
        registry_backup_path
    )
    != registry_sha256_before
):

    raise AssertionError(
        "Registry backup SHA-256 differs from the "
        "original registry."
    )


print("\nRegistry backup created:")

print(
    registry_backup_path
)

print(
    "Backup SHA-256:",
    calculate_sha256(
        registry_backup_path
    ),
)


# ------------------------------------------------------------
# 9. ATOMIC REGISTRY UPDATE WITH AUTOMATIC ROLLBACK
# ------------------------------------------------------------

checkpoint_written = False
status_written = False
audit_written = False


try:

    atomic_write_csv(
        REGISTRY_PATH,
        registry_candidate,
    )


    registry_after = (
        pd.read_csv(
            REGISTRY_PATH,
            dtype=str,
        )
        .fillna("")
    )


    project_numbers_after = (
        project_number_series(
            registry_after,
            project_number_column,
        )
    )


    project8_rows_after = registry_after[
        project_numbers_after.eq(
            PROJECT_NUMBER
        )
    ]


    complete_rows_after = int(
        registry_after[
            status_column
        ].eq(
            COMPLETION_STATUS
        ).sum()
    )


    # Verify registry dimensions.

    if len(
        registry_after
    ) != EXPECTED_REGISTRY_ROWS_AFTER:

        raise AssertionError(
            "Registry row count after update differs.\n"
            f"Expected: {EXPECTED_REGISTRY_ROWS_AFTER}\n"
            f"Actual: {len(registry_after)}"
        )


    if (
        project_numbers_after.isna().any()
        or project_numbers_after.duplicated().any()
    ):

        raise AssertionError(
            "Project numbers are missing or duplicated "
            "after the registry update."
        )


    if set(
        project_numbers_after.astype(int)
    ) != set(
        range(1, 9)
    ):

        raise AssertionError(
            "Registry does not contain exactly Projects 1–8."
        )


    if len(
        project8_rows_after
    ) != 1:

        raise AssertionError(
            "Registry does not contain exactly one "
            "Project 8 row."
        )


    if complete_rows_after != 8:

        raise AssertionError(
            "Registry does not contain eight "
            "COMPLETE_AND_FROZEN rows."
        )


    # Verify Projects 1–7 registry rows are unchanged.

    registry_first_seven_after = (
        registry_after.iloc[
            :len(
                registry_before
            )
        ]
        .reset_index(drop=True)
    )


    registry_first_seven_before = (
        registry_before
        .reset_index(drop=True)
    )


    pd.testing.assert_frame_equal(
        registry_first_seven_after,
        registry_first_seven_before,
        check_dtype=False,
        check_like=False,
    )


    project8_registry_row = (
        project8_rows_after.iloc[0]
    )


    if (
        project8_registry_row[
            status_column
        ]
        != COMPLETION_STATUS
    ):

        raise AssertionError(
            "Project 8 registry status is not "
            "COMPLETE_AND_FROZEN."
        )


    registry_sha256_after = (
        calculate_sha256(
            REGISTRY_PATH
        )
    )


    # Rehash the final package after registry modification.
    # The package must remain exactly unchanged.

    package_inventory_after_records = []


    for path in sorted([
        path
        for path in FINAL_PACKAGE_DIR.rglob("*")
        if path.is_file()
    ]):

        package_inventory_after_records.append({
            "RelativePath":
                path.relative_to(
                    FINAL_PACKAGE_DIR
                ).as_posix(),

            "SizeBytes":
                int(
                    path.stat().st_size
                ),

            "SHA256":
                calculate_sha256(
                    path
                ),
        })


    package_inventory_after = pd.DataFrame(
        package_inventory_after_records
    )


    package_root_sha256_after = (
        root_inventory_hash(
            package_inventory_after
        )
    )


    package_unchanged = bool(
        package_root_sha256_after
        == EXPECTED_PACKAGE_ROOT_SHA256
    )


    if not package_unchanged:

        raise AssertionError(
            "The final package changed during the "
            "registry update."
        )


    # --------------------------------------------------------
    # 10. CREATE PERMANENT PROJECT 8 CHECKPOINT
    # --------------------------------------------------------

    checkpoint_payload = {
        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "ProjectShortName":
            PROJECT_SHORT_NAME,

        "Status":
            COMPLETION_STATUS,

        "SourceManifest":
            str(
                RAW_FILE_INVENTORY_PATH
            ),

        "RawAuditSHA256Algorithm":
            "SHA-256",

        "RawAuditRootSHA256":
            EXPECTED_RAW_ROOT_SHA256,

        "RawFiles":
            EXPECTED_RAW_FILES,

        "RawBytes":
            EXPECTED_RAW_BYTES,

        "RawOutputDirectory":
            str(
                RAW_RESULTS_DIR
            ),

        "RawOutputAuditDirectory":
            str(
                RAW_AUDIT_DIR
            ),

        "RawOutputAuditReport":
            str(
                RAW_AUDIT_REPORT_PATH
            ),

        "RawFileInventory":
            str(
                RAW_FILE_INVENTORY_PATH
            ),

        "FinalPackageDirectory":
            str(
                FINAL_PACKAGE_DIR
            ),

        "FinalPackageHashAlgorithm":
            "SHA-256",

        "FinalPackageRootSHA256":
            EXPECTED_PACKAGE_ROOT_SHA256,

        "FrozenFinalPackageRootSHA256":
            EXPECTED_PACKAGE_ROOT_SHA256,

        "FinalPackageFiles":
            EXPECTED_PACKAGE_FILES,

        "FinalPackageBytes":
            EXPECTED_PACKAGE_BYTES,

        "FinalPackageTopLevelFiles":
            EXPECTED_TOP_LEVEL_FILES,

        "FinalPackageTechniqueNoiseFiles":
            EXPECTED_TECHNIQUE_NOISE_FILES,

        "FinalPackageInventory":
            str(
                FINAL_FREEZE_INVENTORY_PATH
            ),

        "FinalPackageValidation":
            str(
                FINAL_FREEZE_VALIDATION_PATH
            ),

        "FinalPackageReport":
            str(
                FINAL_FREEZE_REPORT_PATH
            ),

        "FinalPackageGenerationInventory":
            str(
                FINAL_PACKAGE_GENERATION_INVENTORY_PATH
            ),

        "FinalPackageGenerationValidation":
            str(
                FINAL_PACKAGE_GENERATION_VALIDATION_PATH
            ),

        "FinalPackageGenerationReport":
            str(
                FINAL_PACKAGE_GENERATION_REPORT_PATH
            ),

        "ExperimentDimensions": {
            "Conditions":
                EXPECTED_CONDITIONS,

            "NoiseLevels":
                9,

            "Seeds":
                30,

            "Techniques":
                7,

            "MLFits":
                EXPECTED_ML_FITS,

            "PredictionSummaries":
                EXPECTED_PREDICTION_SUMMARIES,

            "ProjectRunRows":
                EXPECTED_PROJECT_RUN_ROWS,

            "BuildMetricRows":
                EXPECTED_BUILD_METRIC_ROWS,
        },

        "Step10Status":
            str(
                STEP10_STATUS_PATH
            ),

        "Step11BStatus":
            str(
                STEP11B_STATUS_PATH
            ),

        "Step11CStatus":
            str(
                STEP11C_STATUS_PATH
            ),

        "FinalAggregationCompletedAtUTC":
            step11b_status.get(
                "CompletedAtUTC"
            ),

        "FinalFreezeCompletedAtUTC":
            step11c_status.get(
                "FrozenAtUTC"
            ),

        "CompletionRegistry":
            str(
                REGISTRY_PATH
            ),

        "CompletionRegistrySHA256":
            registry_sha256_after,

        "CompletionRegistryRows":
            len(
                registry_after
            ),

        "CompleteAndFrozenProjects":
            complete_rows_after,

        "RegistryBackup":
            str(
                registry_backup_path
            ),

        "RegistryBackupSHA256":
            calculate_sha256(
                registry_backup_path
            ),

        "Project1To7RegistryRowsUnchanged":
            True,

        "FinalPackageUnchangedDuringRegistryUpdate":
            package_unchanged,

        "CompletedAtUTC":
            COMPLETED_AT_UTC,
    }


    atomic_write_json(
        PROJECT8_CHECKPOINT_PATH,
        checkpoint_payload,
    )

    checkpoint_written = True


    checkpoint_sha256 = (
        calculate_sha256(
            PROJECT8_CHECKPOINT_PATH
        )
    )


    # --------------------------------------------------------
    # 11. FINAL AUDIT TABLE
    # --------------------------------------------------------

    audit_records = [
        {
            "Check":
                "Step 10 passed",

            "Expected":
                EXPECTED_STEP10_STATUS,

            "Actual":
                step10_status[
                    "Status"
                ],

            "Pass":
                step10_status[
                    "Status"
                ]
                == EXPECTED_STEP10_STATUS,
        },

        {
            "Check":
                "Step 11B passed",

            "Expected":
                EXPECTED_STEP11B_STATUS,

            "Actual":
                step11b_status[
                    "Status"
                ],

            "Pass":
                step11b_status[
                    "Status"
                ]
                == EXPECTED_STEP11B_STATUS,
        },

        {
            "Check":
                "Step 11C passed",

            "Expected":
                EXPECTED_STEP11C_STATUS,

            "Actual":
                step11c_status[
                    "Status"
                ],

            "Pass":
                step11c_status[
                    "Status"
                ]
                == EXPECTED_STEP11C_STATUS,
        },

        {
            "Check":
                "Registry rows before",

            "Expected":
                EXPECTED_REGISTRY_ROWS_BEFORE,

            "Actual":
                len(
                    registry_before
                ),

            "Pass":
                len(
                    registry_before
                )
                == EXPECTED_REGISTRY_ROWS_BEFORE,
        },

        {
            "Check":
                "Registry rows after",

            "Expected":
                EXPECTED_REGISTRY_ROWS_AFTER,

            "Actual":
                len(
                    registry_after
                ),

            "Pass":
                len(
                    registry_after
                )
                == EXPECTED_REGISTRY_ROWS_AFTER,
        },

        {
            "Check":
                "Unique projects after",

            "Expected":
                list(
                    range(1, 9)
                ),

            "Actual":
                sorted(
                    project_numbers_after
                    .astype(int)
                    .tolist()
                ),

            "Pass":
                set(
                    project_numbers_after
                    .astype(int)
                )
                == set(
                    range(1, 9)
                ),
        },

        {
            "Check":
                "Project 8 registry rows",

            "Expected":
                1,

            "Actual":
                len(
                    project8_rows_after
                ),

            "Pass":
                len(
                    project8_rows_after
                )
                == 1,
        },

        {
            "Check":
                "COMPLETE_AND_FROZEN rows",

            "Expected":
                8,

            "Actual":
                complete_rows_after,

            "Pass":
                complete_rows_after
                == 8,
        },

        {
            "Check":
                "Projects 1–7 registry rows unchanged",

            "Expected":
                True,

            "Actual":
                True,

            "Pass":
                True,
        },

        {
            "Check":
                "Project 8 status",

            "Expected":
                COMPLETION_STATUS,

            "Actual":
                project8_registry_row[
                    status_column
                ],

            "Pass":
                project8_registry_row[
                    status_column
                ]
                == COMPLETION_STATUS,
        },

        {
            "Check":
                "Package files unchanged",

            "Expected":
                EXPECTED_PACKAGE_FILES,

            "Actual":
                len(
                    package_inventory_after
                ),

            "Pass":
                len(
                    package_inventory_after
                )
                == EXPECTED_PACKAGE_FILES,
        },

        {
            "Check":
                "Package root unchanged",

            "Expected":
                EXPECTED_PACKAGE_ROOT_SHA256,

            "Actual":
                package_root_sha256_after,

            "Pass":
                package_root_sha256_after
                == EXPECTED_PACKAGE_ROOT_SHA256,
        },

        {
            "Check":
                "Checkpoint created",

            "Expected":
                True,

            "Actual":
                PROJECT8_CHECKPOINT_PATH.exists(),

            "Pass":
                PROJECT8_CHECKPOINT_PATH.exists(),
        },

        {
            "Check":
                "Registry backup created",

            "Expected":
                True,

            "Actual":
                registry_backup_path.exists(),

            "Pass":
                registry_backup_path.exists(),
        },
    ]


    step11d_audit = pd.DataFrame(
        audit_records
    )


    failed_audit_checks = (
        step11d_audit[
            ~step11d_audit[
                "Pass"
            ]
        ]
        .copy()
    )


    if not failed_audit_checks.empty:

        display(
            failed_audit_checks
        )

        raise RuntimeError(
            "Step 11D validation failed."
        )


    atomic_write_csv(
        STEP11D_AUDIT_PATH,
        step11d_audit,
    )

    audit_written = True


    # --------------------------------------------------------
    # 12. WRITE STEP 11D STATUS
    # --------------------------------------------------------

    step11d_status = {
        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "Status":
            STEP11D_PASS_STATUS,

        "CompletionStatus":
            COMPLETION_STATUS,

        "RawFiles":
            EXPECTED_RAW_FILES,

        "RawBytes":
            EXPECTED_RAW_BYTES,

        "RawRootSHA256":
            EXPECTED_RAW_ROOT_SHA256,

        "PackageFiles":
            EXPECTED_PACKAGE_FILES,

        "PackageBytes":
            EXPECTED_PACKAGE_BYTES,

        "PackageRootSHA256":
            EXPECTED_PACKAGE_ROOT_SHA256,

        "RegistryRowsBefore":
            len(
                registry_before
            ),

        "RegistryRowsAfter":
            len(
                registry_after
            ),

        "CompleteAndFrozenProjects":
            complete_rows_after,

        "Project8RegistryRows":
            len(
                project8_rows_after
            ),

        "Projects1To7RegistryRowsUnchanged":
            True,

        "RegistrySHA256Before":
            registry_sha256_before,

        "RegistrySHA256After":
            registry_sha256_after,

        "RegistryBackup":
            str(
                registry_backup_path
            ),

        "Checkpoint":
            str(
                PROJECT8_CHECKPOINT_PATH
            ),

        "CheckpointSHA256":
            checkpoint_sha256,

        "PackageUnchanged":
            package_unchanged,

        "FailedValidationChecks":
            len(
                failed_audit_checks
            ),

        "CompletedAtUTC":
            COMPLETED_AT_UTC,
    }


    atomic_write_json(
        STEP11D_STATUS_PATH,
        step11d_status,
    )

    status_written = True


except Exception:

    # Restore the exact original registry bytes.

    atomic_restore_bytes(
        REGISTRY_PATH,
        registry_original_bytes,
    )


    # Remove incomplete newly created outputs.

    if checkpoint_written and PROJECT8_CHECKPOINT_PATH.exists():

        PROJECT8_CHECKPOINT_PATH.unlink()


    if audit_written and STEP11D_AUDIT_PATH.exists():

        STEP11D_AUDIT_PATH.unlink()


    if status_written and STEP11D_STATUS_PATH.exists():

        STEP11D_STATUS_PATH.unlink()


    restored_sha256 = (
        calculate_sha256(
            REGISTRY_PATH
        )
    )


    if restored_sha256 != registry_sha256_before:

        raise RuntimeError(
            "Step 11D failed and the registry rollback "
            "also failed."
        )


    print(
        "\nSTEP 11D FAILED — ORIGINAL REGISTRY RESTORED"
    )

    print(
        "Restored registry SHA-256:",
        restored_sha256,
    )


    raise


# ------------------------------------------------------------
# 13. FINAL INDEPENDENT READBACK
# ------------------------------------------------------------

final_registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


final_project_numbers = (
    project_number_series(
        final_registry,
        project_number_column,
    )
)


final_project8_rows = final_registry[
    final_project_numbers.eq(
        PROJECT_NUMBER
    )
]


final_complete_rows = int(
    final_registry[
        status_column
    ].eq(
        COMPLETION_STATUS
    ).sum()
)


final_registry_sha256 = (
    calculate_sha256(
        REGISTRY_PATH
    )
)


if len(
    final_registry
) != EXPECTED_REGISTRY_ROWS_AFTER:

    raise AssertionError(
        "Final registry row count differs."
    )


if len(
    final_project8_rows
) != 1:

    raise AssertionError(
        "Final registry does not contain exactly one "
        "Project 8 row."
    )


if final_complete_rows != 8:

    raise AssertionError(
        "Final registry does not contain eight "
        "COMPLETE_AND_FROZEN projects."
    )


if not PROJECT8_CHECKPOINT_PATH.exists():

    raise AssertionError(
        "Project 8 checkpoint is missing."
    )


if not STEP11D_STATUS_PATH.exists():

    raise AssertionError(
        "Step 11D status is missing."
    )


final_step11d_status = json.loads(
    STEP11D_STATUS_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    final_step11d_status.get(
        "Status"
    )
    != STEP11D_PASS_STATUS
):

    raise AssertionError(
        "Final Step 11D status differs."
    )


# ------------------------------------------------------------
# 14. DISPLAY UPDATED REGISTRY
# ------------------------------------------------------------

print("\nUpdated completion registry:")

display(
    final_registry
)


print("\nProject 8 registry row:")

display(
    final_project8_rows
)


# ------------------------------------------------------------
# 15. FINAL RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 104)
print("=== PROJECT 8 STEP 11D RESULT ===")
print("=" * 104)

print("\nProject identity:")

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)


print("\nRaw result freeze:")

print(
    "Raw files:",
    EXPECTED_RAW_FILES,
)

print(
    "Raw bytes:",
    EXPECTED_RAW_BYTES,
)

print(
    "Raw root SHA-256:",
    EXPECTED_RAW_ROOT_SHA256,
)


print("\nFinal package freeze:")

print(
    "Package files:",
    EXPECTED_PACKAGE_FILES,
)

print(
    "Package bytes:",
    EXPECTED_PACKAGE_BYTES,
)

print(
    "Package root SHA-256:",
    EXPECTED_PACKAGE_ROOT_SHA256,
)


print("\nCompletion registry:")

print(
    "Registry rows before:",
    EXPECTED_REGISTRY_ROWS_BEFORE,
)

print(
    "Registry rows after:",
    len(
        final_registry
    ),
)

print(
    "Projects in registry:",
    sorted(
        final_project_numbers
        .astype(int)
        .tolist()
    ),
)

print(
    "COMPLETE_AND_FROZEN projects:",
    final_complete_rows,
)

print(
    "Project 8 registry rows:",
    len(
        final_project8_rows
    ),
)

print(
    "Projects 1–7 registry rows unchanged:",
    True,
)

print(
    "Registry SHA-256 before:",
    registry_sha256_before,
)

print(
    "Registry SHA-256 after:",
    final_registry_sha256,
)


print("\nPermanent checkpoint:")

print(
    "Checkpoint:",
    PROJECT8_CHECKPOINT_PATH,
)

print(
    "Checkpoint SHA-256:",
    calculate_sha256(
        PROJECT8_CHECKPOINT_PATH
    ),
)


print("\nRegistry backup:")

print(
    registry_backup_path
)


print("\nStep 11D outputs:")

print(
    STEP11D_AUDIT_PATH
)

print(
    STEP11D_STATUS_PATH
)


print("\nValidation:")

print(
    "Checks:",
    len(
        step11d_audit
    ),
)

print(
    "Failed checks:",
    len(
        failed_audit_checks
    ),
)


print(
    "\nSTATUS:",
    STEP11D_PASS_STATUS,
)

print("=" * 104)

=== PROJECT 8 STEP 11D: COMPLETION REGISTRY UPDATE AND FINAL FREEZE RECORD ===

Prerequisite validation:
Step 10: PASS_PROJECT_8_RAW_RESULTS_AUDITED_AND_AGGREGATED
Step 11B: PASS_PROJECT_8_FINAL_72_FILE_PACKAGE_GENERATED_AND_VALIDATED
Step 11C: PASS_PROJECT_8_FINAL_PACKAGE_INDEPENDENTLY_AUDITED_AND_FROZEN
Raw root SHA-256: 19ae21c5d8524c358796e28f23577fbf03bfd31f7d4b530372c6b4d8a492887c
Final package root SHA-256: 1e27d6f57526d3ce819b7358867c0c58411cc4eaddcdcb2e5576b2282a6cd607

Frozen package revalidation:
Files: 72
Bytes: 1242844
Top-level files: 9
Technique-noise files: 63
Root SHA-256: 1e27d6f57526d3ce819b7358867c0c58411cc4eaddcdcb2e5576b2282a6cd607

Registry before update:
Rows: 7
Projects: [1, 2, 3, 4, 5, 6, 7]
COMPLETE_AND_FROZEN rows: 7
Registry SHA-256: 5f494b829fe17297f37979daf2022f012145c0783460d241752325be1b654250

Registry backup created:
/content/drive/MyDrive/Thesis_Experiment/Notes/completion_registry_backups/completed_project_registry__before_project_08__20260728T00191

,ProjectNumber,Project,ProjectSlug,Status,Conditions,Seeds,NoiseLevels,Techniques,EvaluationBuilds,EvaluationRows,...,FreezeRecord,ChecksumManifest,LastFreezeValidationAtUTC,RawResultsManifest,FinalPackageManifest,RawResultsRootSHA256,FinalPackageRootSHA256,ModelFits,ManifestRowsAudited,FinalAuditStatus
0,1,Angel-ML@angel,Angel-ML__angel,COMPLETE_AND_FROZEN,270,30.0,9.0,7.0,,,...,,,,,,,,,,
1,2,apache@airavata,apache__airavata,COMPLETE_AND_FROZEN,270,30.0,9.0,7.0,,,...,,,,,,,,,,
2,3,b2ihealthcare@snow-owl,b2ihealthcare__snow-owl,COMPLETE_AND_FROZEN,270,30.0,9.0,7.0,,,...,,,,,,,,,,
3,4,eclipse@paho.mqtt.java,eclipse__paho.mqtt.java,COMPLETE_AND_FROZEN,270,30.0,9.0,7.0,20.0,980.0,...,,,,,,,,,,
4,5,thinkaurelius@titan,thinkaurelius__titan,COMPLETE_AND_FROZEN,270,30.0,9.0,,10.0,363.0,...,/content/drive/MyDrive/Thesis_Experiment/Resul...,/content/drive/MyDrive/Thesis_Experiment/Resul...,2026-07-25T03:49:02.436302+00:00,,,,,,,
5,6,eclipse@jetty.project,eclipse__jetty.project,COMPLETE_AND_FROZEN,270,,,,,,...,/content/drive/MyDrive/Thesis_Experiment/Resul...,,,/content/drive/MyDrive/Thesis_Experiment/Resul...,/content/drive/MyDrive/Thesis_Experiment/Resul...,255be2222a28cc9cef226156605244e55f10d99526e3a2...,c2512bf9c72cd41dcb486b625d77e0331fdf1960296111...,1080.0,5358150.0,PASS
6,7,CompEvol@beast2,CompEvol__beast2,COMPLETE_AND_FROZEN,270,1-30,"0,5,10,15,20,25,30,40,50",7.0,52.0,,...,,,,,,,93e9cc74fac017fef89b005752e1b7ede2a73f9665dd18...,1080.0,,
7,8,optimatika@ojAlgo,optimatika__ojAlgo,COMPLETE_AND_FROZEN,270,30,9,7,52.0,,...,,,,,,,1e27d6f57526d3ce819b7358867c0c58411cc4eaddcdcb...,1080,,



Project 8 registry row:


,ProjectNumber,Project,ProjectSlug,Status,Conditions,Seeds,NoiseLevels,Techniques,EvaluationBuilds,EvaluationRows,...,FreezeRecord,ChecksumManifest,LastFreezeValidationAtUTC,RawResultsManifest,FinalPackageManifest,RawResultsRootSHA256,FinalPackageRootSHA256,ModelFits,ManifestRowsAudited,FinalAuditStatus
7,8,optimatika@ojAlgo,optimatika__ojAlgo,COMPLETE_AND_FROZEN,270,30,9,7,52.0,,...,,,,,,,1e27d6f57526d3ce819b7358867c0c58411cc4eaddcdcb...,1080,,




=== PROJECT 8 STEP 11D RESULT ===

Project identity:
Project number: 8
Project: optimatika@ojAlgo
Project slug: optimatika__ojAlgo

Raw result freeze:
Raw files: 2160
Raw bytes: 179766494
Raw root SHA-256: 19ae21c5d8524c358796e28f23577fbf03bfd31f7d4b530372c6b4d8a492887c

Final package freeze:
Package files: 72
Package bytes: 1242844
Package root SHA-256: 1e27d6f57526d3ce819b7358867c0c58411cc4eaddcdcb2e5576b2282a6cd607

Completion registry:
Registry rows before: 7
Registry rows after: 8
Projects in registry: [1, 2, 3, 4, 5, 6, 7, 8]
COMPLETE_AND_FROZEN projects: 8
Project 8 registry rows: 1
Projects 1–7 registry rows unchanged: True
Registry SHA-256 before: 5f494b829fe17297f37979daf2022f012145c0783460d241752325be1b654250
Registry SHA-256 after: a442446f3ca6207b31213fc422fb0c883c96608a8e6a3907da969e326808bee7

Permanent checkpoint:
Checkpoint: /content/drive/MyDrive/Thesis_Experiment/Notes/project_08_selection_checkpoint.json
Checkpoint SHA-256: e43683b39e8ab77b46b16a76c38f8b9d3a3ef9f3